<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>ادامه فشرده Golden Dataset — Pilot QA Workflow</h2>

<p>
این Notebook ادامه مرحله Golden Dataset در Phase 2 است و برای تکمیل Pilotهای باقی‌مانده پس از Notebook توسعه‌ای قبلی ساخته شده است.
این Notebook به معنی ورود به مرحله ماژولار بعدی پروژه نیست؛ تمام فعالیت‌های آن همچنان متعلق به ساخت، اعتبارسنجی و Freeze کردن Golden Dataset هستند.
</p>

<p>
Notebook قبلی نقش Development و Contract Discovery را داشت.
در آن، Pilot Selection، Target Evidence، Question Drafting، Machine Validation، Human Review،
Expected Answer، Citation Validation، Queue Management، Replacement Policy و Atomic Rollback
به‌صورت عمیق طراحی و آزمایش شدند.
به همین دلیل تعداد Cellهای آن زیاد شد.
</p>

<p>
هدف این Notebook تکرار آن روند توسعه‌ای نیست.
Contractهایی که قبلاً اثبات شده‌اند باید به یک Workflow فشرده، Reusable و Parameterized تبدیل شوند
تا Pilotهای باقی‌مانده بدون ساخت ده‌ها Cell برای هر مورد پردازش شوند.
</p>

<h3>وضعیت ورودی</h3>

<ul>
<li>دامنه Pilot Validation برابر با <strong>23 Pilot</strong> است.</li>
<li>یک Pilot مرجع قبلاً به‌طور کامل Accepted شده است.</li>
<li>Remaining Queue برابر با <strong>22 Pilot</strong> است.</li>
<li>Pilot بلاک‌شده قبلی با یک Replacement معتبر جایگزین شده است.</li>
<li>Replacement Pilot فعلی باید اولین Pilot این Notebook برای ادامه Standard QA Workflow باشد.</li>
<li>Full 572 Generation همچنان تا تکمیل Pilot Validation مسدود باقی می‌ماند.</li>
</ul>

<h3>اصل معماری این Notebook</h3>

<p>
این Notebook نباید برای هر Pilot دوباره Selection Engine، Queue Logic، Validation Logic،
Replacement Logic یا Rollback Logic را از صفر بسازد.
این منطق‌ها باید یک‌بار به توابع Reusable تبدیل شوند و سپس برای Pilotهای مختلف فراخوانی شوند.
</p>

<p>
روش فشرده پیشنهادی:
</p>

<ol>
<li>Discovery و بازیابی Source of Truth واقعی از Repository و Notebook قبلی؛</li>
<li>ساخت یک Runtime State کوچک و قابل‌بازتولید برای Golden Dataset Workflow؛</li>
<li>تعریف توابع Reusable برای Stage، Draft، Machine Validation، Human Review Packet، Expected Answer و Citation Validation؛</li>
<li>تعریف Queue/State Transition عمومی به‌جای ساخت Cellهای اختصاصی متعدد؛</li>
<li>پردازش Pilotها یکی‌یکی با همان Engine مشترک؛</li>
<li>توقف فقط در Human Gate یا Edge Case واقعی؛</li>
<li>در صورت Edge Case، Repair فقط همان مورد و سپس بازگشت به Workflow عمومی؛</li>
<li>در پایان، Validation کل Pilot Set، Dev/Test Split و Freeze Golden Dataset.</li>
</ol>

<h3>قاعده جلوگیری از بزرگ‌شدن Notebook</h3>

<p>
یک Pilot سالم نباید ده‌ها Cell جدید تولید کند.
منطق عمومی فقط یک‌بار تعریف می‌شود.
برای Pilotهای معمولی، اجرای Workflow باید با تعداد بسیار کمی Cell انجام شود و خروجی‌ها در State ساختاریافته ثبت شوند.
فقط Failure یا Edge Case جدید می‌تواند باعث اضافه‌شدن Step تشخیصی جداگانه شود.
</p>

<p>
بنابراین هدف این Notebook این است که 22 Pilot باقی‌مانده را با یک Engine مشترک تکمیل کند،
نه اینکه تاریخچه 140-Cell Notebook قبلی را 22 بار تکرار کند.
</p>

<h3>خارج از Scope</h3>

<ul>
<li>Embedding Benchmark</li>
<li>Dense Index</li>
<li>Hybrid Retrieval</li>
<li>Reranking</li>
<li>تولید Full 572 قبل از رفع Pilot Validation Gate</li>
<li>تغییر Notebookهای مرحله بعدی Roadmap</li>
</ul>

<h3>شرط Done شدن Notebook</h3>

<ul>
<li>Pilot Workflow برای دامنه موردنیاز تکمیل و معتبر باشد.</li>
<li>Golden Dataset موردنیاز ساخته و Validate شود.</li>
<li>Dev/Test Split و Freeze نهایی انجام شود.</li>
<li>هیچ Error یا Hidden Session Dependency باقی نماند.</li>
<li>Restart Kernel → Run All بدون Error اجرا شود.</li>
<li>Final Validation و Notebook Health هر دو PASS شوند.</li>
</ul>

<p>
Bootstrap فنی این Notebook در <code>Code Cell 01</code> متمرکز می‌شود و Import ثابت در Cellهای بعدی تکرار نخواهد شد.
</p>

</div>

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Repository / Artifact Preflight</h3>

<p>
قبل از ساخت Workflow فشرده، ابتدا باید وضعیت واقعی Repository و خروجی Notebook قبلی از روی Disk بررسی شود.
هیچ Runtime State از Kernel قبلی فرض نمی‌شود.
</p>

<p>
این Preflight به‌صورت Read-Only بررسی می‌کند:
</p>

<ul>
<li>Git Root و Branch واقعی؛</li>
<li>Notebookهای مرتبط با Golden Dataset؛</li>
<li>وجود Final Activation و Final Checkpoint در Notebook توسعه‌ای قبلی؛</li>
<li>Artifactها، Checkpointها، Manifestها یا Reportهای مرتبط موجود روی Disk؛</li>
<li>Git Status فعلی؛</li>
<li>اینکه آیا یک Handoff ماشین‌خوان از Notebook قبلی وجود دارد یا باید State لازم به‌صورت deterministic بازسازی شود.</li>
</ul>

<p>
در این مرحله هیچ فایل یا Artifactی ساخته یا تغییر داده نمی‌شود.
نتیجه Preflight تعیین می‌کند Compact Workflow باید از کدام Source of Truth شروع شود.
</p>

<h4>معیار PASS</h4>

<ul>
<li>Repository Root واقعی شناسایی شود.</li>
<li>Branch واقعی شناسایی شود.</li>
<li>Notebook توسعه‌ای Golden Dataset قبلی با Markerهای Activation و Final Checkpoint پیدا شود.</li>
<li>Discovery بدون Write یا Mutation انجام شود.</li>
</ul>

</div>

In [1]:
# Code Cell 01

from pathlib import Path
from collections import Counter, defaultdict

import copy
import hashlib
import json
import re
import subprocess

In [2]:
# Code Cell 02

cwd = Path.cwd().resolve()


def run_git(*args):
    result = subprocess.run(
        ["git", *args],
        cwd=cwd,
        capture_output=True,
        text=True,
        check=False,
    )
    return result.returncode, result.stdout.strip(), result.stderr.strip()


root_rc, root_out, root_err = run_git(
    "rev-parse",
    "--show-toplevel",
)

project_root = (
    Path(root_out).resolve()
    if root_rc == 0 and root_out
    else None
)


branch_rc, branch_out, branch_err = run_git(
    "branch",
    "--show-current",
)

current_branch = (
    branch_out
    if branch_rc == 0 and branch_out
    else None
)


head_rc, head_out, head_err = run_git(
    "rev-parse",
    "HEAD",
)

current_head = (
    head_out
    if head_rc == 0 and head_out
    else None
)


status_rc, status_out, status_err = run_git(
    "status",
    "--short",
)


related_notebooks = []

if project_root is not None:
    for path in sorted(project_root.rglob("*.ipynb")):
        name = path.name.lower()

        if (
            "golden" in name
            or
            "dataset" in name
            or
            name.startswith("12")
        ):
            related_notebooks.append(path)


ACTIVATION_MARKERS = [
    "REPLACEMENT_PILOT_ATOMICALLY_ACTIVATED_LIVE_STATE_VALID",
    "CODE CELL 139 GATE",
]

CHECKPOINT_MARKERS = [
    "CURRENT_NOTEBOOK_POST_ACTIVATION_STATE_VALID_READY_TO_CLOSE_DEVELOPMENT_NOTEBOOK",
    "CURRENT_DEVELOPMENT_NOTEBOOK_FINAL_CHECKPOINT_PASS",
    "CODE CELL 140 GATE",
]


notebook_inspection = []

for path in related_notebooks:
    try:
        raw = path.read_text(
            encoding="utf-8"
        )

        payload = json.loads(raw)

        code_cells = [
            cell
            for cell in payload.get("cells", [])
            if cell.get("cell_type") == "code"
        ]

        numbered_cells = []

        for cell in code_cells:
            source = "".join(
                cell.get("source", [])
            )

            match = re.search(
                r"# Code Cell\s+(\d+)",
                source,
            )

            if match:
                numbered_cells.append(
                    int(match.group(1))
                )

        activation_found = any(
            marker in raw
            for marker in ACTIVATION_MARKERS
        )

        checkpoint_found = any(
            marker in raw
            for marker in CHECKPOINT_MARKERS
        )

        notebook_inspection.append({
            "path": str(
                path.relative_to(project_root)
            ),
            "code_cells": len(code_cells),
            "highest_numbered_code_cell": (
                max(numbered_cells)
                if numbered_cells
                else None
            ),
            "activation_marker_found":
                activation_found,
            "checkpoint_marker_found":
                checkpoint_found,
            "size_bytes":
                path.stat().st_size,
        })

    except Exception as exc:
        notebook_inspection.append({
            "path": str(
                path.relative_to(project_root)
            ),
            "inspection_error": repr(exc),
        })


validated_prior_notebooks = [
    row
    for row in notebook_inspection
    if (
        row.get(
            "activation_marker_found"
        )
        is True
        and
        row.get(
            "checkpoint_marker_found"
        )
        is True
    )
]


artifact_keywords = (
    "golden",
    "pilot",
    "question",
    "citation",
    "workflow",
    "checkpoint",
    "manifest",
)


artifact_roots = []

if project_root is not None:
    for dirname in [
        "data",
        "reports",
        "artifacts",
        "checkpoints",
        "manifests",
    ]:
        candidate = (
            project_root / dirname
        )

        if candidate.exists():
            artifact_roots.append(
                candidate
            )


related_artifacts = []

for base in artifact_roots:
    for path in base.rglob("*"):
        if not path.is_file():
            continue

        lowered = (
            path.name.lower()
        )

        if any(
            keyword in lowered
            for keyword
            in artifact_keywords
        ):
            related_artifacts.append(
                {
                    "path": str(
                        path.relative_to(
                            project_root
                        )
                    ),
                    "size_bytes":
                        path.stat().st_size,
                }
            )


related_artifacts = sorted(
    related_artifacts,
    key=lambda row: row["path"],
)


handoff_candidates = [
    row
    for row in related_artifacts
    if any(
        token in row["path"].lower()
        for token in (
            "handoff",
            "checkpoint",
            "workflow",
            "pilot",
        )
    )
]


preflight_gate = all([
    project_root is not None,
    current_branch is not None,
    len(
        validated_prior_notebooks
    )
    >=
    1,
])


print(
    "=== COMPACT GOLDEN DATASET PREFLIGHT ==="
)

print(
    "\nWorking directory:",
    cwd,
)

print(
    "Repository root:",
    project_root,
)

print(
    "Current branch:",
    current_branch,
)

print(
    "Current HEAD:",
    current_head,
)


print(
    "\n=== GIT STATUS ==="
)

print(
    status_out
    if status_out
    else "Working tree clean"
)


print(
    "\n=== RELATED NOTEBOOKS ==="
)

print(
    "Candidates found:",
    len(
        notebook_inspection
    )
)

for row in notebook_inspection:
    print(
        row
    )


print(
    "\n=== VALIDATED PRIOR DEVELOPMENT NOTEBOOKS ==="
)

print(
    "Validated candidates:",
    len(
        validated_prior_notebooks
    )
)

for row in validated_prior_notebooks:
    print(
        row
    )


print(
    "\n=== RELATED DISK ARTIFACTS ==="
)

print(
    "Related artifacts found:",
    len(
        related_artifacts
    )
)

for row in related_artifacts[:50]:
    print(
        row
    )

if len(related_artifacts) > 50:
    print(
        "... additional artifacts:",
        len(related_artifacts) - 50,
    )


print(
    "\n=== HANDOFF / CHECKPOINT CANDIDATES ==="
)

print(
    "Candidates found:",
    len(
        handoff_candidates
    )
)

for row in handoff_candidates[:30]:
    print(
        row
    )


print(
    "\n=== PREFLIGHT DECISION ==="
)

print(
    "Repository resolved:",
    project_root is not None,
)

print(
    "Branch resolved:",
    current_branch is not None,
)

print(
    "Validated prior NB12 found:",
    len(
        validated_prior_notebooks
    )
    >=
    1,
)

print(
    "Machine-readable handoff candidates:",
    len(
        handoff_candidates
    ),
)

print(
    "Mutation performed:",
    False,
)

print(
    "\nCODE CELL 02 GATE:",
    "PASS"
    if preflight_gate
    else "FAIL",
)


assert preflight_gate, (
    "Compact Golden Dataset preflight failed. "
    "Do not build the reusable workflow yet. "
    "Repair this SAME Code Cell 02."
)

=== COMPACT GOLDEN DATASET PREFLIGHT ===

Working directory: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2
Repository root: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance
Current branch: rag-phase2
Current HEAD: 07f19b78576ce5051bce6ad51602f8dc26f03f21

=== GIT STATUS ===
M ../../data/phase2/human_review/human_review_registry.json
 M 12b_golden_dataset_continuation_compact.ipynb

=== RELATED NOTEBOOKS ===
Candidates found: 3
{'path': 'notebooks/phase2/12_golden_dataset.ipynb', 'code_cells': 141, 'highest_numbered_code_cell': 141, 'activation_marker_found': True, 'checkpoint_marker_found': True, 'size_bytes': 9345530}
{'path': 'notebooks/phase2/12b_golden_dataset_continuation_compact.ipynb', 'code_cells': 97, 'highest_numbered_code_cell': 96, 'activation_marker_found': True, 'checkpoint_marker_found': True, 'size_bytes': 4311028}
{'path': 'notebooks/phase2/12c_human_review_for_author.ipynb', 'code_cells': 2, 'highes

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>اعتبارسنجی Handoff و تعیین Source of Truth برای Compact Workflow</h3>

<p>
Preflight وجود یک Handoff ماشین‌خوان از Notebook قبلی را تأیید کرد.
اما صرف وجود فایل به معنی Current و قابل‌استفاده بودن آن نیست.
ممکن است این Artifact قبل از Replacement نهایی Pilot ساخته شده باشد.
</p>

<p>
در این مرحله Handoff موجود به‌صورت Read-Only خوانده می‌شود و هم‌زمان خروجی واقعی Cellهای نهایی Notebook قبلی
از خود فایل ذخیره‌شده <code>.ipynb</code> استخراج می‌شود.
</p>

<p>
هدف این است که بدون اتکا به Session قبلی مشخص شود:
</p>

<ul>
<li>Notebook قبلی واقعاً Activation و Final Checkpoint را PASS کرده است؛</li>
<li>Active Pilot Count و Remaining Queue از Output واقعی قابل‌بازیابی هستند؛</li>
<li>Next Pilot نهایی از Output واقعی مشخص است؛</li>
<li>Checkpoint SHA نهایی قابل‌بازیابی است؛</li>
<li>Handoff موجود با این State نهایی سازگار است یا قدیمی است.</li>
</ul>

<p>
اگر Handoff موجود Current باشد، همان به‌عنوان Seed اولیه Compact Workflow استفاده می‌شود.
اگر Current نباشد، بدون حدس‌زدن State، در Step بعد یک Runtime Seed فشرده از
Notebook پذیرفته‌شده و Artifactهای Canonical موجود بازسازی خواهد شد.
</p>

<h4>معیار PASS</h4>

<ul>
<li>Handoff JSON بدون Error Parse شود.</li>
<li>Cell 139 و Cell 140 از Notebook قبلی روی Disk پیدا شوند.</li>
<li>PASS واقعی Cell 140 از Output ذخیره‌شده تأیید شود.</li>
<li>Active Count، Remaining Count، Next Pilot و Checkpoint SHA از Output واقعی استخراج شوند.</li>
<li>Current یا Stale بودن Handoff بدون Mutation مشخص شود.</li>
</ul>

</div>

In [3]:
# Code Cell 03

handoff_path = (
    project_root
    / "reports/phase2/chunking/nb12_golden_dataset_handoff_v1.json"
)


prior_notebook_path = (
    project_root
    / "notebooks/phase2/12_golden_dataset.ipynb"
)


cell03_failures = []


if not handoff_path.exists():
    cell03_failures.append(
        "HANDOFF_FILE_NOT_FOUND"
    )


if not prior_notebook_path.exists():
    cell03_failures.append(
        "PRIOR_NOTEBOOK_NOT_FOUND"
    )


handoff_payload = None
prior_notebook_payload = None


if handoff_path.exists():
    try:
        handoff_payload = json.loads(
            handoff_path.read_text(
                encoding="utf-8"
            )
        )
    except Exception as exc:
        cell03_failures.append({
            "reason":
                "HANDOFF_JSON_PARSE_FAILED",

            "error":
                repr(exc),
        })


if prior_notebook_path.exists():
    try:
        prior_notebook_payload = json.loads(
            prior_notebook_path.read_text(
                encoding="utf-8"
            )
        )
    except Exception as exc:
        cell03_failures.append({
            "reason":
                "PRIOR_NOTEBOOK_JSON_PARSE_FAILED",

            "error":
                repr(exc),
        })


def cell03_code_cell_by_number(
    notebook_payload,
    number,
):
    if not isinstance(
        notebook_payload,
        dict,
    ):
        return None

    pattern = re.compile(
        rf"^\s*# Code Cell\s+{number}\b",
        flags=re.MULTILINE,
    )

    for cell in notebook_payload.get(
        "cells",
        [],
    ):
        if cell.get(
            "cell_type"
        ) != "code":
            continue

        source = "".join(
            cell.get(
                "source",
                [],
            )
        )

        if pattern.search(
            source
        ):
            return cell

    return None


def cell03_output_text(cell):
    if not isinstance(
        cell,
        dict,
    ):
        return ""

    parts = []

    for output in cell.get(
        "outputs",
        [],
    ):
        if output.get(
            "output_type"
        ) == "stream":
            text = output.get(
                "text",
                "",
            )

            parts.append(
                "".join(text)
                if isinstance(text, list)
                else str(text)
            )

        data = output.get(
            "data",
            {},
        )

        text_plain = data.get(
            "text/plain"
        )

        if text_plain is not None:
            parts.append(
                "".join(text_plain)
                if isinstance(text_plain, list)
                else str(text_plain)
            )

    return "\n".join(
        parts
    )


cell03_cell139 = (
    cell03_code_cell_by_number(
        prior_notebook_payload,
        139,
    )
)


cell03_cell140 = (
    cell03_code_cell_by_number(
        prior_notebook_payload,
        140,
    )
)


cell03_cell139_output = (
    cell03_output_text(
        cell03_cell139
    )
)


cell03_cell140_output = (
    cell03_output_text(
        cell03_cell140
    )
)


cell03_cell139_pass = (
    "CODE CELL 139 GATE: PASS"
    in
    cell03_cell139_output
)


cell03_cell140_pass = (
    "CODE CELL 140 GATE: PASS"
    in
    cell03_cell140_output
)


def cell03_extract(
    pattern,
    text,
):
    match = re.search(
        pattern,
        text,
    )

    return (
        match.group(1).strip()
        if match
        else None
    )


cell03_active_pilot_count = (
    cell03_extract(
        r"Active Pilot count:\s*(\d+)",
        cell03_cell140_output,
    )
)


cell03_remaining_queue_count = (
    cell03_extract(
        r"Remaining Queue count:\s*(\d+)",
        cell03_cell140_output,
    )
)


cell03_next_pilot_id = (
    cell03_extract(
        r"Next Pilot:\s*(question-pilot:[A-Za-z0-9]+)",
        cell03_cell140_output,
    )
)


cell03_checkpoint_sha256 = (
    cell03_extract(
        r"Checkpoint SHA256:\s*([a-f0-9]{64})",
        cell03_cell140_output,
    )
)


cell03_checkpoint_status = (
    cell03_extract(
        r"Status:\s*([A-Z0-9_]+)",
        cell03_cell140_output,
    )
)


cell03_required_checkpoint_values_resolved = all([
    cell03_active_pilot_count
    is not None,

    cell03_remaining_queue_count
    is not None,

    cell03_next_pilot_id
    is not None,

    cell03_checkpoint_sha256
    is not None,

    cell03_checkpoint_status
    is not None,
])


cell03_handoff_serialized = (
    json.dumps(
        handoff_payload,
        ensure_ascii=False,
        sort_keys=True,
    )
    if handoff_payload
    is not None
    else ""
)


cell03_handoff_mentions_next_pilot = (
    cell03_next_pilot_id
    is not None

    and

    cell03_next_pilot_id
    in
    cell03_handoff_serialized
)


cell03_handoff_mentions_checkpoint_sha = (
    cell03_checkpoint_sha256
    is not None

    and

    cell03_checkpoint_sha256
    in
    cell03_handoff_serialized
)


cell03_handoff_current_for_final_state = all([
    cell03_handoff_mentions_next_pilot,
    cell03_handoff_mentions_checkpoint_sha,
])


cell03_prior_notebook_final_state_valid = all([
    cell03_cell139
    is not None,

    cell03_cell140
    is not None,

    cell03_cell139_pass,

    cell03_cell140_pass,

    cell03_required_checkpoint_values_resolved,

    cell03_active_pilot_count
    ==
    "23",

    cell03_remaining_queue_count
    ==
    "22",

    cell03_checkpoint_status
    ==
    "CURRENT_NOTEBOOK_POST_ACTIVATION_STATE_VALID_READY_TO_CLOSE_DEVELOPMENT_NOTEBOOK",
])


cell03_gate = all([
    len(
        cell03_failures
    )
    ==
    0,

    isinstance(
        handoff_payload,
        dict,
    ),

    cell03_prior_notebook_final_state_valid,
])


cell03_source_of_truth_decision = (
    "USE_EXISTING_HANDOFF_AS_COMPACT_RUNTIME_SEED"

    if (
        cell03_gate
        and
        cell03_handoff_current_for_final_state
    )

    else
    (
        "RECONSTRUCT_COMPACT_RUNTIME_SEED_FROM_ACCEPTED_NB12_DISK_STATE"

        if cell03_gate

        else

        "BLOCKED_PENDING_HANDOFF_INSPECTION_REPAIR"
    )
)


print(
    "=== NB12 HANDOFF VALIDATION ==="
)


print(
    "\nHandoff path:",
    handoff_path.relative_to(
        project_root
    ),
)

print(
    "Handoff parsed:",
    isinstance(
        handoff_payload,
        dict,
    ),
)


print(
    "\n=== HANDOFF PAYLOAD ==="
)

if isinstance(
    handoff_payload,
    dict,
):
    print(
        json.dumps(
            handoff_payload,
            ensure_ascii=False,
            indent=2,
            sort_keys=True,
        )
    )


print(
    "\n=== PRIOR NOTEBOOK FINAL OUTPUT ==="
)

print(
    "Cell 139 found:",
    cell03_cell139
    is not None,
)

print(
    "Cell 139 PASS:",
    cell03_cell139_pass,
)

print(
    "Cell 140 found:",
    cell03_cell140
    is not None,
)

print(
    "Cell 140 PASS:",
    cell03_cell140_pass,
)

print(
    "Active Pilot count:",
    cell03_active_pilot_count,
)

print(
    "Remaining Queue count:",
    cell03_remaining_queue_count,
)

print(
    "Next Pilot:",
    cell03_next_pilot_id,
)

print(
    "Checkpoint SHA256:",
    cell03_checkpoint_sha256,
)

print(
    "Checkpoint status:",
    cell03_checkpoint_status,
)


print(
    "\n=== HANDOFF CURRENCY ==="
)

print(
    "Mentions current Next Pilot:",
    cell03_handoff_mentions_next_pilot,
)

print(
    "Mentions current Checkpoint SHA:",
    cell03_handoff_mentions_checkpoint_sha,
)

print(
    "Current for final post-activation state:",
    cell03_handoff_current_for_final_state,
)


print(
    "\n=== DECISION ==="
)

print(
    "Source of Truth decision:",
    cell03_source_of_truth_decision,
)

print(
    "Mutation performed:",
    False,
)


print(
    "\nFailures:",
    cell03_failures,
)


print(
    "\nCODE CELL 03 GATE:",
    "PASS"
    if cell03_gate
    else "FAIL",
)


assert cell03_gate, (
    "NB12 handoff inspection failed. "
    "Repair this SAME Code Cell 03 before building the compact workflow."
)

=== NB12 HANDOFF VALIDATION ===

Handoff path: reports/phase2/chunking/nb12_golden_dataset_handoff_v1.json
Handoff parsed: True

=== HANDOFF PAYLOAD ===
{
  "book": [
    "intrinsically_tiny_narrative_region",
    "intrinsically_tiny_table_region",
    "long_section",
    "large_table_run",
    "exercise_or_self_assessment",
    "supplemental_footnote_case",
    "deep_heading_path",
    "multi_paragraph_evidence_case"
  ],
  "candidate_strategies_to_compare": [
    "baseline_phase1_char_safe",
    "structure_aware_small",
    "structure_aware_medium",
    "parent_child_child"
  ],
  "evaluation_dimensions": [
    "retrieval_hit",
    "rank",
    "evidence_completeness",
    "attribution_correctness",
    "citation_locator_correctness",
    "unnecessary_context",
    "source_authority_correctness"
  ],
  "parent_context_separate_evaluation": true,
  "status": "REQUIRED_FOR_FINAL_CHUNK_DECISION",
  "transcript": [
    "single_speaker_primary",
    "single_speaker_unresolved",
    "multi_

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>کشف Anchorهای لازم برای بازسازی Compact Runtime Seed</h3>

<p>
در Step قبل مشخص شد که Handoff موجود مربوط به State نهایی Pilotها نیست و نمی‌توان آن را به‌عنوان Runtime Seed استفاده کرد.
بنابراین قبل از ساخت Engine فشرده، باید دقیقاً مشخص شود State نهایی از چه اطلاعاتی می‌تواند به‌صورت deterministic بازسازی شود.
</p>

<p>
این مرحله فقط Discovery انجام می‌دهد و سه مسیر ممکن را بررسی می‌کند:
</p>

<ol>
<li>آیا Artifact ماشین‌خوان دیگری روی Disk وجود دارد که Replacement Pilot یا Final Checkpoint را ثبت کرده باشد؟</li>
<li>آیا Notebook قبلی State کامل موردنیاز مانند Pilot Population یا Remaining Queue را در Output ذخیره کرده است؟</li>
<li>اگر هیچ State کامل Persist نشده، کدام Cellها و Variableهای پذیرفته‌شده قرارداد کافی برای بازسازی State از Canonical Inputs را فراهم می‌کنند؟</li>
</ol>

<p>
هدف این است که Notebook جدید به 140 Cell قبلی وابسته نشود.
فقط Contractها و Anchorهای اثبات‌شده از Notebook قبلی خوانده می‌شوند و در Step بعد یک Seed کوچک و مستقل ساخته خواهد شد.
</p>

<h4>معیار PASS</h4>

<ul>
<li>Cellهای کلیدی Workflow و Replacement در Notebook قبلی شناسایی شوند.</li>
<li>وجود یا عدم وجود Artifact به‌روز روی Disk مشخص شود.</li>
<li>وجود یا عدم وجود Serialized Runtime State در Outputهای Notebook مشخص شود.</li>
<li>یک Reconstruction Route روشن و بدون حدس انتخاب شود.</li>
<li>هیچ فایل یا Stateی تغییر نکند.</li>
</ul>

</div>

In [4]:
# Code Cell 04

cell04_failures = []


# ------------------------------------------------------------
# 1) Accepted final-state anchors from Cell 03
# ------------------------------------------------------------

cell04_next_pilot_id = (
    cell03_next_pilot_id
)


cell04_checkpoint_sha256 = (
    cell03_checkpoint_sha256
)


cell04_required_cell_numbers = [
    29,
    74,
    95,
    125,
    130,
    135,
    137,
    139,
    140,
]


cell04_required_variable_anchors = [
    "question_drafting_pilot_packets",
    "question_drafting_pilot_by_id",
    "pilot_selection_rows",
    "cell95_remaining_pilot_ids",
    "cell95_remaining_pilot_queue",
    "cell95_pilot_workflow_state",
    "cell130_simulated_packet",
    "cell135_simulated_queue",
    "cell137_rebuilt_state",
    "cell138_target_values",
]


# ------------------------------------------------------------
# 2) Build notebook cell index
# ------------------------------------------------------------

cell04_code_cell_index = {}


for cell in prior_notebook_payload.get(
    "cells",
    [],
):
    if cell.get(
        "cell_type"
    ) != "code":
        continue

    source = "".join(
        cell.get(
            "source",
            [],
        )
    )

    match = re.search(
        r"^\s*# Code Cell\s+(\d+)",
        source,
        flags=re.MULTILINE,
    )

    if not match:
        continue

    number = int(
        match.group(1)
    )

    cell04_code_cell_index[
        number
    ] = {
        "source": source,
        "output": cell03_output_text(
            cell
        ),
        "execution_count":
            cell.get(
                "execution_count"
            ),
    }


cell04_required_cells_found = {
    number:
        number
        in
        cell04_code_cell_index

    for number
    in cell04_required_cell_numbers
}


# ------------------------------------------------------------
# 3) Find where important runtime variables appear
# ------------------------------------------------------------

cell04_variable_occurrences = {}


for variable_name in (
    cell04_required_variable_anchors
):
    hits = []

    for number, record in (
        cell04_code_cell_index.items()
    ):
        source_hit = (
            variable_name
            in
            record["source"]
        )

        output_hit = (
            variable_name
            in
            record["output"]
        )

        if source_hit or output_hit:
            hits.append({
                "cell":
                    number,

                "source":
                    source_hit,

                "output":
                    output_hit,
            })

    cell04_variable_occurrences[
        variable_name
    ] = hits


# ------------------------------------------------------------
# 4) Detect whether full runtime structures were serialized
# ------------------------------------------------------------

cell04_serialization_markers = {
    "pilot_packets":
        [
            "question_drafting_pilot_packets",
            "'pilot_case_id'",
            '"pilot_case_id"',
        ],

    "remaining_queue":
        [
            "cell95_remaining_pilot_queue",
            "'queue_index'",
            '"queue_index"',
        ],

    "rebuilt_state":
        [
            "cell137_rebuilt_state",
            "cell95_pilot_workflow_state_sha256",
        ],
}


cell04_serialized_output_candidates = {
    key: []
    for key
    in cell04_serialization_markers
}


for number, record in (
    cell04_code_cell_index.items()
):
    output = record[
        "output"
    ]

    if not output:
        continue

    for key, markers in (
        cell04_serialization_markers.items()
    ):
        if all(
            marker
            in
            output

            for marker
            in markers
        ):
            cell04_serialized_output_candidates[
                key
            ].append(
                number
            )


cell04_complete_runtime_state_serialized = all([
    len(
        cell04_serialized_output_candidates[
            "pilot_packets"
        ]
    )
    >
    0,

    len(
        cell04_serialized_output_candidates[
            "remaining_queue"
        ]
    )
    >
    0,
])


# ------------------------------------------------------------
# 5) Search disk artifacts for final Replacement anchors
# ------------------------------------------------------------

cell04_search_roots = [
    project_root
    / "data"
    / "phase2",

    project_root
    / "reports"
    / "phase2",
]


cell04_search_extensions = {
    ".json",
    ".jsonl",
    ".md",
    ".txt",
}


cell04_disk_anchor_hits = []


for base in cell04_search_roots:
    if not base.exists():
        continue

    for path in sorted(
        base.rglob("*")
    ):
        if not path.is_file():
            continue

        if path.suffix.lower() not in (
            cell04_search_extensions
        ):
            continue

        try:
            if path.stat().st_size > (
                20
                *
                1024
                *
                1024
            ):
                continue

            text = path.read_text(
                encoding="utf-8"
            )

        except Exception:
            continue

        next_pilot_hit = (
            cell04_next_pilot_id
            in
            text
        )

        checkpoint_hit = (
            cell04_checkpoint_sha256
            in
            text
        )

        if (
            next_pilot_hit
            or
            checkpoint_hit
        ):
            cell04_disk_anchor_hits.append({
                "path":
                    str(
                        path.relative_to(
                            project_root
                        )
                    ),

                "next_pilot":
                    next_pilot_hit,

                "checkpoint_sha":
                    checkpoint_hit,
            })


cell04_current_state_artifacts = [
    row

    for row
    in cell04_disk_anchor_hits

    if (
        row["next_pilot"]
        and
        row["checkpoint_sha"]
    )
]


# ------------------------------------------------------------
# 6) Inspect contract cells for canonical rebuild anchors
# ------------------------------------------------------------

cell04_contract_cells = {}


for number in [
    29,
    74,
    95,
    125,
    130,
    135,
    137,
    139,
]:
    record = (
        cell04_code_cell_index.get(
            number
        )
    )

    if record is None:
        continue

    source = record[
        "source"
    ]

    cell04_contract_cells[
        number
    ] = {
        "has_pilot_packets":
            "question_drafting_pilot_packets"
            in
            source,

        "has_pilot_index":
            "question_drafting_pilot_by_id"
            in
            source,

        "has_selection_rows":
            "pilot_selection_rows"
            in
            source,

        "has_remaining_ids":
            "cell95_remaining_pilot_ids"
            in
            source,

        "has_remaining_queue":
            "cell95_remaining_pilot_queue"
            in
            source,

        "has_workflow_state":
            "cell95_pilot_workflow_state"
            in
            source,

        "has_replacement_packet":
            "cell130_simulated_packet"
            in
            source,

        "has_replacement_queue":
            "cell135_simulated_queue"
            in
            source,

        "has_rebuilt_state":
            "cell137_rebuilt_state"
            in
            source,
    }


cell04_core_contract_coverage = {
    "pilot_population":
        any(
            row[
                "has_pilot_packets"
            ]

            for row
            in cell04_contract_cells.values()
        ),

    "pilot_index":
        any(
            row[
                "has_pilot_index"
            ]

            for row
            in cell04_contract_cells.values()
        ),

    "selection_rows":
        any(
            row[
                "has_selection_rows"
            ]

            for row
            in cell04_contract_cells.values()
        ),

    "remaining_ids":
        any(
            row[
                "has_remaining_ids"
            ]

            for row
            in cell04_contract_cells.values()
        ),

    "remaining_queue":
        any(
            row[
                "has_remaining_queue"
            ]

            for row
            in cell04_contract_cells.values()
        ),

    "workflow_state":
        any(
            row[
                "has_workflow_state"
            ]

            for row
            in cell04_contract_cells.values()
        ),

    "replacement_packet":
        any(
            row[
                "has_replacement_packet"
            ]

            for row
            in cell04_contract_cells.values()
        ),

    "replacement_queue":
        any(
            row[
                "has_replacement_queue"
            ]

            for row
            in cell04_contract_cells.values()
        ),

    "rebuilt_state":
        any(
            row[
                "has_rebuilt_state"
            ]

            for row
            in cell04_contract_cells.values()
        ),
}


cell04_contract_coverage_complete = all(
    cell04_core_contract_coverage.values()
)


# ------------------------------------------------------------
# 7) Choose deterministic reconstruction route
# ------------------------------------------------------------

if len(
    cell04_current_state_artifacts
) > 0:

    cell04_reconstruction_route = (
        "LOAD_CURRENT_MACHINE_READABLE_FINAL_STATE_ARTIFACT"
    )

elif (
    cell04_complete_runtime_state_serialized
):

    cell04_reconstruction_route = (
        "PARSE_COMPLETE_RUNTIME_STATE_FROM_ACCEPTED_NOTEBOOK_OUTPUT"
    )

elif (
    cell04_contract_coverage_complete
):

    cell04_reconstruction_route = (
        "REBUILD_COMPACT_RUNTIME_SEED_FROM_CANONICAL_INPUTS_AND_ACCEPTED_CONTRACT_CELLS"
    )

else:

    cell04_reconstruction_route = (
        "BLOCKED_INSUFFICIENT_RECONSTRUCTION_ANCHORS"
    )


# ------------------------------------------------------------
# 8) Gate
# ------------------------------------------------------------

cell04_gate = all([
    all(
        cell04_required_cells_found.values()
    ),

    cell03_prior_notebook_final_state_valid,

    cell04_next_pilot_id
    is not None,

    cell04_checkpoint_sha256
    is not None,

    cell04_reconstruction_route
    !=
    "BLOCKED_INSUFFICIENT_RECONSTRUCTION_ANCHORS",
])


# ------------------------------------------------------------
# 9) Diagnostics
# ------------------------------------------------------------

print(
    "=== COMPACT RUNTIME SEED RECONSTRUCTION DISCOVERY ==="
)


print(
    "\n=== FINAL STATE ANCHORS ==="
)

print(
    "Next Pilot:",
    cell04_next_pilot_id,
)

print(
    "Checkpoint SHA256:",
    cell04_checkpoint_sha256,
)


print(
    "\n=== REQUIRED CONTRACT CELLS ==="
)

for number, found in (
    cell04_required_cells_found.items()
):
    print(
        f"Cell {number}:",
        "FOUND"
        if found
        else "MISSING",
    )


print(
    "\n=== VARIABLE OCCURRENCES ==="
)

for name, hits in (
    cell04_variable_occurrences.items()
):
    print(
        name,
        ":",
        hits,
    )


print(
    "\n=== SERIALIZED RUNTIME STATE ==="
)

for key, cells in (
    cell04_serialized_output_candidates.items()
):
    print(
        key,
        ":",
        cells,
    )

print(
    "Complete runtime state serialized:",
    cell04_complete_runtime_state_serialized,
)


print(
    "\n=== DISK ARTIFACT ANCHOR HITS ==="
)

print(
    "Anchor hits:",
    len(
        cell04_disk_anchor_hits
    ),
)

for row in (
    cell04_disk_anchor_hits
):
    print(
        row
    )

print(
    "Current full-state artifact candidates:",
    len(
        cell04_current_state_artifacts
    ),
)


print(
    "\n=== CONTRACT COVERAGE ==="
)

for name, covered in (
    cell04_core_contract_coverage.items()
):
    print(
        name,
        ":",
        "PASS"
        if covered
        else "FAIL",
    )

print(
    "Contract coverage complete:",
    cell04_contract_coverage_complete,
)


print(
    "\n=== RECONSTRUCTION DECISION ==="
)

print(
    "Route:",
    cell04_reconstruction_route,
)

print(
    "Mutation performed:",
    False,
)


print(
    "\nCODE CELL 04 GATE:",
    "PASS"
    if cell04_gate
    else "FAIL",
)


assert cell04_gate, (
    "Compact runtime seed reconstruction discovery failed. "
    "Repair this SAME Code Cell 04 before building the seed."
)

=== COMPACT RUNTIME SEED RECONSTRUCTION DISCOVERY ===

=== FINAL STATE ANCHORS ===
Next Pilot: question-pilot:3005b48518a7aeb532234332df23c952
Checkpoint SHA256: a0d25d8eef2f7a4b76ce24d52e9341ddae5d4553af6207288f82e35eb9fe9f2c

=== REQUIRED CONTRACT CELLS ===
Cell 29: FOUND
Cell 74: FOUND
Cell 95: FOUND
Cell 125: FOUND
Cell 130: FOUND
Cell 135: FOUND
Cell 137: FOUND
Cell 139: FOUND
Cell 140: FOUND

=== VARIABLE OCCURRENCES ===
question_drafting_pilot_packets : [{'cell': 29, 'source': True, 'output': False}, {'cell': 30, 'source': True, 'output': False}, {'cell': 46, 'source': False, 'output': True}, {'cell': 47, 'source': True, 'output': True}, {'cell': 63, 'source': False, 'output': True}, {'cell': 65, 'source': False, 'output': True}, {'cell': 66, 'source': True, 'output': True}, {'cell': 67, 'source': False, 'output': True}, {'cell': 68, 'source': True, 'output': True}, {'cell': 69, 'source': True, 'output': True}, {'cell': 70, 'source': True, 'output': True}, {'cell': 71, 'source':

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Dependency Slicing برای ساخت Compact Runtime Seed</h3>

<p>
مسیر بازسازی در Step قبل مشخص شد، اما قرار نیست Cellهای قدیمی یکی‌یکی Replay شوند.
در این مرحله Source Notebook قبلی با AST بررسی می‌شود تا فقط Statementهایی که واقعاً برای ساخت State نهایی لازم‌اند شناسایی شوند.
</p>

<p>
تمرکز روی پنج Contract اصلی است:
</p>

<ul>
<li>ساخت Pilot Population؛</li>
<li>ساخت Remaining Queue و Workflow State؛</li>
<li>ساخت Replacement Pilot Packet؛</li>
<li>ساخت Replacement Queue؛</li>
<li>ساخت Cell 95 Derived State بعد از Replacement.</li>
</ul>

<p>
برای هر Contract، Dependencyهای واقعی و آخرین Cell تأمین‌کننده آن‌ها استخراج می‌شود.
این کار اجازه می‌دهد در Step بعد یک Builder کوچک و مستقل بسازیم و از Replay کردن تاریخچه توسعه‌ای Notebook قبلی جلوگیری کنیم.
</p>

<h4>معیار PASS</h4>

<ul>
<li>تمام Contract Cellهای موردنیاز Parse شوند.</li>
<li>Target Variableهای اصلی در Source واقعی پیدا شوند.</li>
<li>Dependencyهای خارجی هر Target شناسایی شوند.</li>
<li>برای Dependencyهای پروژه‌ای، Provider Cell واقعی قابل‌شناسایی باشد.</li>
<li>Minimal Provider Set برای بازسازی مشخص شود.</li>
<li>هیچ State یا فایلی تغییر نکند.</li>
</ul>

</div>

In [5]:
# Code Cell 05

ast = __import__("ast")
builtins_module = __import__("builtins")


cell05_contract_targets = {
    29: [
        "pilot_selection_rows",
        "question_drafting_pilot_packets",
        "question_drafting_pilot_by_id",
    ],
    95: [
        "cell95_remaining_pilot_ids",
        "cell95_remaining_pilot_queue",
        "cell95_pilot_workflow_state",
    ],
    130: [
        "cell130_simulated_packet",
    ],
    135: [
        "cell135_simulated_queue",
    ],
    137: [
        "cell137_rebuilt_state",
    ],
}


cell05_failures = []


# ------------------------------------------------------------
# AST binding helpers
# ------------------------------------------------------------

def cell05_store_names(node):
    names = set()

    for child in ast.walk(node):

        if (
            isinstance(child, ast.Name)
            and
            isinstance(child.ctx, ast.Store)
        ):
            names.add(
                child.id
            )

        elif isinstance(child, ast.arg):
            names.add(
                child.arg
            )

        elif isinstance(
            child,
            ast.ExceptHandler,
        ):
            if isinstance(
                child.name,
                str,
            ):
                names.add(
                    child.name
                )

        elif isinstance(
            child,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
                ast.ClassDef,
            ),
        ):
            names.add(
                child.name
            )

        elif isinstance(
            child,
            ast.Import,
        ):
            for alias in child.names:
                names.add(
                    alias.asname
                    or
                    alias.name.split(".")[0]
                )

        elif isinstance(
            child,
            ast.ImportFrom,
        ):
            for alias in child.names:
                names.add(
                    alias.asname
                    or
                    alias.name
                )

    return names


def cell05_load_names(node):

    return {
        child.id

        for child in ast.walk(node)

        if (
            isinstance(
                child,
                ast.Name,
            )
            and
            isinstance(
                child.ctx,
                ast.Load,
            )
        )
    }


def cell05_top_level_records(source):

    tree = ast.parse(
        source
    )

    records = []

    for statement in tree.body:

        records.append({
            "node":
                statement,

            "stores":
                cell05_store_names(
                    statement
                ),

            "loads":
                cell05_load_names(
                    statement
                ),

            "lineno":
                getattr(
                    statement,
                    "lineno",
                    None,
                ),

            "end_lineno":
                getattr(
                    statement,
                    "end_lineno",
                    None,
                ),
        })

    return records


def cell05_backward_slice(
    source,
    targets,
):

    records = (
        cell05_top_level_records(
            source
        )
    )

    needed = set(
        targets
    )

    selected = set()

    changed = True

    while changed:

        changed = False

        for index in range(
            len(records) - 1,
            -1,
            -1,
        ):

            record = records[
                index
            ]

            if (
                index not in selected
                and
                record["stores"]
                &
                needed
            ):

                selected.add(
                    index
                )

                needed.update(
                    record["loads"]
                )

                changed = True


    selected_records = [
        records[index]

        for index in sorted(
            selected
        )
    ]


    selected_stores = set()

    selected_loads = set()


    for record in (
        selected_records
    ):

        selected_stores.update(
            record[
                "stores"
            ]
        )

        selected_loads.update(
            record[
                "loads"
            ]
        )


    external_dependencies = (
        selected_loads
        -
        selected_stores
    )


    return {
        "records":
            records,

        "selected_records":
            selected_records,

        "selected_stores":
            selected_stores,

        "selected_loads":
            selected_loads,

        "external_dependencies":
            external_dependencies,
    }


# ------------------------------------------------------------
# Provider index
# ------------------------------------------------------------

cell05_provider_index = defaultdict(
    list
)


cell05_parse_errors = []


for number, record in sorted(
    cell04_code_cell_index.items()
):

    try:

        tree = ast.parse(
            record[
                "source"
            ]
        )

    except SyntaxError as exc:

        cell05_parse_errors.append({
            "cell":
                number,

            "error":
                repr(
                    exc
                ),
        })

        continue


    for statement in tree.body:

        for name in (
            cell05_store_names(
                statement
            )
        ):

            cell05_provider_index[
                name
            ].append(
                number
            )


cell05_builtin_names = set(
    dir(
        builtins_module
    )
)


cell05_known_runtime_names = {
    "Path",
    "Counter",
    "defaultdict",
    "copy",
    "hashlib",
    "json",
    "re",
    "subprocess",
    "ast",
}


# ------------------------------------------------------------
# Contract dependency analysis
# ------------------------------------------------------------

cell05_contract_results = {}


for cell_number, targets in (
    cell05_contract_targets.items()
):

    record = (
        cell04_code_cell_index.get(
            cell_number
        )
    )


    if record is None:

        cell05_failures.append({
            "reason":
                "CONTRACT_CELL_MISSING",

            "cell":
                cell_number,
        })

        continue


    try:

        slice_result = (
            cell05_backward_slice(
                record[
                    "source"
                ],
                targets,
            )
        )

    except Exception as exc:

        cell05_failures.append({
            "reason":
                "CONTRACT_SLICE_FAILED",

            "cell":
                cell_number,

            "error":
                repr(
                    exc
                ),
        })

        continue


    unresolved_targets = (
        set(
            targets
        )
        -
        slice_result[
            "selected_stores"
        ]
    )


    if unresolved_targets:

        cell05_failures.append({
            "reason":
                "TARGET_ASSIGNMENT_NOT_FOUND",

            "cell":
                cell_number,

            "targets":
                sorted(
                    unresolved_targets
                ),
        })


    project_dependencies = sorted(

        dependency

        for dependency
        in slice_result[
            "external_dependencies"
        ]

        if (
            dependency
            not in
            cell05_builtin_names

            and

            dependency
            not in
            cell05_known_runtime_names
        )
    )


    provider_map = {}


    for dependency in (
        project_dependencies
    ):

        providers = [
            provider

            for provider
            in cell05_provider_index.get(
                dependency,
                [],
            )

            if provider
            <
            cell_number
        ]


        provider_map[
            dependency
        ] = (
            providers[-1]
            if providers
            else None
        )


    unresolved_dependencies = sorted(

        dependency

        for dependency, provider
        in provider_map.items()

        if provider is None
    )


    selected_statement_ranges = [
        (
            row[
                "lineno"
            ],
            row[
                "end_lineno"
            ],
        )

        for row
        in slice_result[
            "selected_records"
        ]
    ]


    cell05_contract_results[
        cell_number
    ] = {
        "targets":
            list(
                targets
            ),

        "selected_statement_count":
            len(
                slice_result[
                    "selected_records"
                ]
            ),

        "selected_statement_ranges":
            selected_statement_ranges,

        "project_dependencies":
            project_dependencies,

        "provider_map":
            provider_map,

        "unresolved_dependencies":
            unresolved_dependencies,
    }


# ------------------------------------------------------------
# Minimal upstream provider set
# ------------------------------------------------------------

cell05_provider_cells = sorted({

    provider

    for result
    in cell05_contract_results.values()

    for provider
    in result[
        "provider_map"
    ].values()

    if provider is not None
})


cell05_all_targets_resolved = (
    len(
        cell05_failures
    )
    ==
    0
)


cell05_all_project_dependencies_resolved = all(

    len(
        result[
            "unresolved_dependencies"
        ]
    )
    ==
    0

    for result
    in cell05_contract_results.values()
)


cell05_compact_rebuild_route_ready = all([
    cell05_all_targets_resolved,
    cell05_all_project_dependencies_resolved,
])


cell05_rebuild_plan = {
    "strategy":
        "AST_BACKWARD_SLICE_FROM_ACCEPTED_CONTRACTS",

    "contract_cells":
        sorted(
            cell05_contract_targets
        ),

    "provider_cells":
        cell05_provider_cells,

    "replay_full_notebook":
        False,

    "false_positive_fix":
        "FUNCTION_ARGS_AND_EXCEPTION_TARGETS_ARE_LOCAL_BINDINGS",

    "write_files":
        False,

    "ready":
        cell05_compact_rebuild_route_ready,
}


# ------------------------------------------------------------
# Diagnostics
# ------------------------------------------------------------

print(
    "=== COMPACT CONTRACT DEPENDENCY SLICING ==="
)


print(
    "\nValidator repair:",
    "Function arguments and exception-handler targets "
    "are treated as local lexical bindings.",
)


print(
    "\nPrior notebook parse errors:",
    cell05_parse_errors,
)


for cell_number in sorted(
    cell05_contract_results
):

    result = (
        cell05_contract_results[
            cell_number
        ]
    )


    print(
        f"\n=== CONTRACT CELL {cell_number} ==="
    )


    print(
        "Targets:",
        result[
            "targets"
        ],
    )


    print(
        "Selected statement count:",
        result[
            "selected_statement_count"
        ],
    )


    print(
        "Statement ranges:",
        result[
            "selected_statement_ranges"
        ],
    )


    print(
        "Project dependencies:",
        result[
            "project_dependencies"
        ],
    )


    print(
        "Provider map:",
        result[
            "provider_map"
        ],
    )


    print(
        "Unresolved dependencies:",
        result[
            "unresolved_dependencies"
        ],
    )


print(
    "\n=== MINIMAL PROVIDER SET ==="
)


print(
    "Provider cells:",
    cell05_provider_cells,
)


print(
    "Provider cell count:",
    len(
        cell05_provider_cells
    ),
)


print(
    "Full notebook replay required:",
    False,
)


print(
    "\n=== REBUILD PLAN ==="
)


print(
    json.dumps(
        cell05_rebuild_plan,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    )
)


print(
    "\nFailures:",
    cell05_failures,
)


print(
    "All project dependencies resolved:",
    cell05_all_project_dependencies_resolved,
)


print(
    "Compact rebuild route ready:",
    cell05_compact_rebuild_route_ready,
)


print(
    "Mutation performed:",
    False,
)


cell05_gate = (
    cell05_compact_rebuild_route_ready
)


print(
    "\nCODE CELL 05 GATE:",
    "PASS"
    if cell05_gate
    else "FAIL",
)


assert cell05_gate, (
    "Compact contract dependency slicing is incomplete. "
    "Repair this SAME Code Cell 05 before reconstructing the runtime seed."
)

=== COMPACT CONTRACT DEPENDENCY SLICING ===

Validator repair: Function arguments and exception-handler targets are treated as local lexical bindings.

Prior notebook parse errors: []

=== CONTRACT CELL 29 ===
Targets: ['pilot_selection_rows', 'question_drafting_pilot_packets', 'question_drafting_pilot_by_id']
Selected statement count: 3
Statement ranges: [(78, 78), (301, 301), (303, 303)]
Project dependencies: []
Provider map: {}
Unresolved dependencies: []

=== CONTRACT CELL 95 ===
Targets: ['cell95_remaining_pilot_ids', 'cell95_remaining_pilot_queue', 'cell95_pilot_workflow_state']
Selected statement count: 13
Statement ranges: [(99, 107), (164, 168), (241, 320), (323, 327), (334, 338), (341, 348), (438, 449), (541, 541), (544, 606), (609, 613), (623, 635), (638, 649), (656, 693)]
Project dependencies: ['PILOT_QUESTION_AUTHORING_MODE_V1', 'QUESTION_DRAFT_ACCEPTANCE_RUBRIC_V1', 'QUESTION_USER_FACING_SOURCE_OPACITY_RULE_V1', 'cell94_completed_pilot_qa_record', 'cell94_completed_pilot_

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>استخراج Recursive Dependency Closure از Final Replacement State</h3>

<p>
Cell 05 نشان داد Contractهای لازم قابل‌بازیابی هستند.
اما برای ادامه Compact Workflow لازم نیست چند Contract را جداگانه دوباره بسازیم.
در Notebook قبلی، <code>cell138_target_values</code> دقیقاً مجموعه 11 Runtime Object نهایی است که بعداً در Cell 139 روی Live State اعمال شد.
</p>

<p>
بنابراین این مرحله فقط Dependencyهای واقعی لازم برای بازسازی همین Final Target State را به‌صورت Recursive دنبال می‌کند.
هدف این است که کوچک‌ترین مجموعه Provider Cellها و Statementهای لازم مشخص شود و از Replay کامل 140 Cell جلوگیری شود.
</p>

<h4>معیار PASS</h4>

<ul>
<li><code>cell138_target_values</code> در Cell 138 قابل Slice باشد.</li>
<li>تمام Dependencyهای پروژه‌ای آن تا Provider واقعی دنبال شوند.</li>
<li>Dependency حل‌نشده باقی نماند.</li>
<li>Dependency Cycle نامعتبر وجود نداشته باشد.</li>
<li>Full Notebook Replay لازم نباشد.</li>
<li>هیچ فایل یا Runtime State پروژه تغییر نکند.</li>
</ul>

</div>

In [6]:
# Code Cell 06

cell06_root_cell = 138
cell06_root_target = "cell138_target_values"

cell06_failures = []


# ============================================================
# 1) Core AST helpers
# ============================================================

cell06_mutating_methods = {
    "append",
    "extend",
    "insert",
    "update",
    "setdefault",
    "add",
    "remove",
    "discard",
    "pop",
    "popitem",
    "clear",
    "sort",
    "reverse",
}


def cell06_root_name(node):
    if isinstance(node, ast.Name):
        return node.id

    if isinstance(
        node,
        (
            ast.Attribute,
            ast.Subscript,
        ),
    ):
        return cell06_root_name(
            node.value
        )

    return None


def cell06_target_names(node):
    names = set()

    if isinstance(node, ast.Name):
        names.add(
            node.id
        )

    elif isinstance(
        node,
        (
            ast.Tuple,
            ast.List,
        ),
    ):
        for item in node.elts:
            names.update(
                cell06_target_names(
                    item
                )
            )

    elif isinstance(
        node,
        (
            ast.Attribute,
            ast.Subscript,
        ),
    ):
        root = (
            cell06_root_name(
                node
            )
        )

        if root:
            names.add(
                root
            )

    return names


def cell06_loaded_names(node):
    return {
        child.id

        for child
        in ast.walk(
            node
        )

        if (
            isinstance(
                child,
                ast.Name,
            )
            and
            isinstance(
                child.ctx,
                ast.Load,
            )
        )
    }


def cell06_internal_bindings(node):
    names = set()

    for child in ast.walk(
        node
    ):
        if (
            isinstance(
                child,
                ast.Name,
            )
            and
            isinstance(
                child.ctx,
                ast.Store,
            )
        ):
            names.add(
                child.id
            )

        elif isinstance(
            child,
            ast.ExceptHandler,
        ):
            if isinstance(
                child.name,
                str,
            ):
                names.add(
                    child.name
                )

    return names


# ============================================================
# 2) Persistent module bindings
# ============================================================

class Cell06ModuleBindingVisitor(
    ast.NodeVisitor
):
    def __init__(self):
        self.names = set()

    def visit_Name(
        self,
        node,
    ):
        if isinstance(
            node.ctx,
            ast.Store,
        ):
            self.names.add(
                node.id
            )

    def visit_FunctionDef(
        self,
        node,
    ):
        self.names.add(
            node.name
        )

        for decorator in (
            node.decorator_list
        ):
            self.visit(
                decorator
            )

        defaults = (
            list(
                node.args.defaults
            )
            +
            [
                value

                for value
                in node.args.kw_defaults

                if value is not None
            ]
        )

        for value in defaults:
            self.visit(
                value
            )

        # Function body has its own scope.

    def visit_AsyncFunctionDef(
        self,
        node,
    ):
        self.visit_FunctionDef(
            node
        )

    def visit_ClassDef(
        self,
        node,
    ):
        self.names.add(
            node.name
        )

        for base in node.bases:
            self.visit(
                base
            )

        for keyword in node.keywords:
            self.visit(
                keyword.value
            )

        for decorator in (
            node.decorator_list
        ):
            self.visit(
                decorator
            )

        # Class body has its own namespace.

    def visit_Lambda(
        self,
        node,
    ):
        # Lambda body has its own argument scope.
        return

    def visit_ListComp(
        self,
        node,
    ):
        return

    def visit_SetComp(
        self,
        node,
    ):
        return

    def visit_DictComp(
        self,
        node,
    ):
        return

    def visit_GeneratorExp(
        self,
        node,
    ):
        return

    def visit_ExceptHandler(
        self,
        node,
    ):
        # Exception targets are deleted after the handler.
        if node.type is not None:
            self.visit(
                node.type
            )

        for statement in (
            node.body
        ):
            self.visit(
                statement
            )

    def visit_Import(
        self,
        node,
    ):
        for alias in node.names:
            self.names.add(
                alias.asname
                or
                alias.name.split(
                    "."
                )[0]
            )

    def visit_ImportFrom(
        self,
        node,
    ):
        for alias in node.names:
            if alias.name == "*":
                continue

            self.names.add(
                alias.asname
                or
                alias.name
            )


def cell06_module_bindings(
    statement,
):
    visitor = (
        Cell06ModuleBindingVisitor()
    )

    visitor.visit(
        statement
    )

    return visitor.names


# ============================================================
# 3) Function free-name analysis
# ============================================================

def cell06_function_local_names(
    node,
):
    names = {
        argument.arg

        for argument
        in (
            list(
                node.args.posonlyargs
            )
            +
            list(
                node.args.args
            )
            +
            list(
                node.args.kwonlyargs
            )
        )
    }

    if node.args.vararg is not None:
        names.add(
            node.args.vararg.arg
        )

    if node.args.kwarg is not None:
        names.add(
            node.args.kwarg.arg
        )

    for child in ast.walk(
        node
    ):
        if (
            isinstance(
                child,
                ast.Name,
            )
            and
            isinstance(
                child.ctx,
                ast.Store,
            )
        ):
            names.add(
                child.id
            )

        elif (
            child is not node
            and
            isinstance(
                child,
                (
                    ast.FunctionDef,
                    ast.AsyncFunctionDef,
                    ast.ClassDef,
                ),
            )
        ):
            names.add(
                child.name
            )

        elif isinstance(
            child,
            ast.ExceptHandler,
        ):
            if isinstance(
                child.name,
                str,
            ):
                names.add(
                    child.name
                )

        elif isinstance(
            child,
            ast.arg,
        ):
            # Covers nested lambda/function parameters.
            names.add(
                child.arg
            )

    return names


def cell06_function_external_reads(
    node,
):
    reads = (
        cell06_loaded_names(
            node
        )
        -
        cell06_function_local_names(
            node
        )
    )

    reads.discard(
        node.name
    )

    return reads


# ============================================================
# 4) Statement effects
# ============================================================

def cell06_statement_effects(
    statement,
):
    reads = set()
    writes = set()
    definite_writes = set()
    mutations = set()
    provider_writes = set()

    if isinstance(
        statement,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef,
        ),
    ):
        writes.add(
            statement.name
        )
        definite_writes.add(
            statement.name
        )
        provider_writes.add(
            statement.name
        )

        reads.update(
            cell06_function_external_reads(
                statement
            )
        )

        return {
            "reads": reads,
            "writes": writes,
            "definite_writes":
                definite_writes,
            "mutations":
                mutations,
            "provider_writes":
                provider_writes,
        }

    if isinstance(
        statement,
        ast.ClassDef,
    ):
        writes.add(
            statement.name
        )
        definite_writes.add(
            statement.name
        )
        provider_writes.add(
            statement.name
        )

        for base in statement.bases:
            reads.update(
                cell06_loaded_names(
                    base
                )
            )

        for keyword in (
            statement.keywords
        ):
            reads.update(
                cell06_loaded_names(
                    keyword.value
                )
            )

        for decorator in (
            statement.decorator_list
        ):
            reads.update(
                cell06_loaded_names(
                    decorator
                )
            )

        return {
            "reads": reads,
            "writes": writes,
            "definite_writes":
                definite_writes,
            "mutations":
                mutations,
            "provider_writes":
                provider_writes,
        }

    internal_bindings = (
        cell06_internal_bindings(
            statement
        )
    )

    module_bindings = (
        cell06_module_bindings(
            statement
        )
    )

    writes.update(
        module_bindings
    )

    definite_writes.update(
        module_bindings
    )

    provider_writes.update(
        module_bindings
    )

    raw_mutations = set()

    for child in ast.walk(
        statement
    ):
        if (
            child is not statement
            and
            isinstance(
                child,
                (
                    ast.FunctionDef,
                    ast.AsyncFunctionDef,
                    ast.ClassDef,
                    ast.Lambda,
                ),
            )
        ):
            continue

        if (
            isinstance(
                child,
                ast.Call,
            )
            and
            isinstance(
                child.func,
                ast.Attribute,
            )
            and
            child.func.attr
            in
            cell06_mutating_methods
        ):
            receiver = (
                cell06_root_name(
                    child.func.value
                )
            )

            if receiver:
                raw_mutations.add(
                    receiver
                )

        elif isinstance(
            child,
            ast.AugAssign,
        ):
            raw_mutations.update(
                cell06_target_names(
                    child.target
                )
            )

        elif isinstance(
            child,
            ast.Assign,
        ):
            for target in (
                child.targets
            ):
                if isinstance(
                    target,
                    (
                        ast.Attribute,
                        ast.Subscript,
                    ),
                ):
                    root = (
                        cell06_root_name(
                            target
                        )
                    )

                    if root:
                        raw_mutations.add(
                            root
                        )

        elif isinstance(
            child,
            ast.AnnAssign,
        ):
            if isinstance(
                child.target,
                (
                    ast.Attribute,
                    ast.Subscript,
                ),
            ):
                root = (
                    cell06_root_name(
                        child.target
                    )
                )

                if root:
                    raw_mutations.add(
                        root
                    )

    external_mutations = (
        raw_mutations
        -
        module_bindings
    )

    mutations.update(
        external_mutations
    )

    writes.update(
        external_mutations
    )

    reads.update(
        cell06_loaded_names(
            statement
        )
        -
        internal_bindings
    )

    self_reads = set()

    for child in ast.walk(
        statement
    ):
        if isinstance(
            child,
            ast.AugAssign,
        ):
            self_reads.update(
                cell06_target_names(
                    child.target
                )
            )

        elif isinstance(
            child,
            ast.Assign,
        ):
            target_names = set()

            for target in (
                child.targets
            ):
                target_names.update(
                    cell06_target_names(
                        target
                    )
                )

            self_reads.update(
                target_names
                &
                cell06_loaded_names(
                    child.value
                )
            )

        elif (
            isinstance(
                child,
                ast.AnnAssign,
            )
            and
            child.value is not None
        ):
            target_names = (
                cell06_target_names(
                    child.target
                )
            )

            self_reads.update(
                target_names
                &
                cell06_loaded_names(
                    child.value
                )
            )

    reads.update(
        self_reads
    )

    reads.update(
        external_mutations
    )

    return {
        "reads": reads,
        "writes": writes,
        "definite_writes":
            definite_writes,
        "mutations":
            mutations,
        "provider_writes":
            provider_writes,
    }


# ============================================================
# 5) Parse prior Notebook
# ============================================================

cell06_cell_models = {}

for cell_number, record in sorted(
    cell04_code_cell_index.items()
):
    source = record[
        "source"
    ]

    try:
        tree = ast.parse(
            source
        )

    except SyntaxError as exc:
        cell06_failures.append({
            "reason":
                "SOURCE_PARSE_FAILED",
            "cell":
                cell_number,
            "error":
                repr(
                    exc
                ),
        })
        continue

    statements = []

    for index, statement in enumerate(
        tree.body,
        start=1,
    ):
        effects = (
            cell06_statement_effects(
                statement
            )
        )

        statements.append({
            "index":
                index,
            "node":
                statement,
            "range":
                (
                    getattr(
                        statement,
                        "lineno",
                        None,
                    ),
                    getattr(
                        statement,
                        "end_lineno",
                        None,
                    ),
                ),
            **effects,
        })

    cell06_cell_models[
        cell_number
    ] = {
        "source":
            source,
        "statements":
            statements,
    }


# ============================================================
# 6) Cross-cell provider index
# ============================================================

cell06_provider_index = defaultdict(
    list
)

for cell_number, model in (
    cell06_cell_models.items()
):
    names = set()

    for statement in (
        model[
            "statements"
        ]
    ):
        names.update(
            statement[
                "provider_writes"
            ]
        )

    for name in names:
        cell06_provider_index[
            name
        ].append(
            cell_number
        )

for name in list(
    cell06_provider_index
):
    cell06_provider_index[
        name
    ] = sorted(
        set(
            cell06_provider_index[
                name
            ]
        )
    )


# ============================================================
# 7) Runtime-preseed names
# ============================================================

if isinstance(
    __builtins__,
    dict,
):
    cell06_builtin_names = set(
        __builtins__.keys()
    )

else:
    cell06_builtin_names = set(
        dir(
            __builtins__
        )
    )

cell06_builtin_names.update({
    "__name__",
    "__file__",
    "__package__",
    "__spec__",
})

cell06_preseed_runtime_names = {
    "Path",
    "Counter",
    "defaultdict",
    "copy",
    "hashlib",
    "json",
    "re",
    "subprocess",
    "ast",
}


# ============================================================
# 8) Backward slice within one Cell
# ============================================================

def cell06_slice_cell(
    cell_number,
    target_names,
):
    model = (
        cell06_cell_models[
            cell_number
        ]
    )

    needed = set(
        target_names
    )

    selected = []

    for statement in reversed(
        model[
            "statements"
        ]
    ):
        affected = (
            statement[
                "writes"
            ]
            |
            statement[
                "mutations"
            ]
        )

        if not (
            affected
            &
            needed
        ):
            continue

        selected.append(
            statement
        )

        needed -= (
            statement[
                "definite_writes"
            ]
        )

        needed.update(
            statement[
                "reads"
            ]
        )

        needed.update(
            statement[
                "mutations"
            ]
        )

    selected.reverse()

    return {
        "targets":
            sorted(
                set(
                    target_names
                )
            ),
        "selected_statements":
            selected,
        "statement_ranges":
            [
                statement[
                    "range"
                ]
                for statement
                in selected
            ],
        "selected_statement_count":
            len(
                selected
            ),
        "external_dependencies":
            sorted(
                name
                for name
                in needed
                if (
                    name
                    not in
                    cell06_builtin_names
                    and
                    name
                    not in
                    cell06_preseed_runtime_names
                )
            ),
    }


# ============================================================
# 9) Latest prior provider
# ============================================================

def cell06_latest_provider(
    name,
    consumer_cell,
):
    candidates = [
        provider_cell
        for provider_cell
        in cell06_provider_index.get(
            name,
            []
        )
        if provider_cell
        <
        consumer_cell
    ]

    if not candidates:
        return None

    return max(
        candidates
    )


# ============================================================
# 10) Recursive closure
# ============================================================

cell06_requested_targets = defaultdict(
    set
)

cell06_requested_targets[
    cell06_root_cell
].add(
    cell06_root_target
)

cell06_cell_slices = {}
cell06_resolution_log = []

cell06_changed = True
cell06_iteration = 0
cell06_max_iterations = 1000

while cell06_changed:
    cell06_iteration += 1

    if (
        cell06_iteration
        >
        cell06_max_iterations
    ):
        cell06_failures.append({
            "reason":
                "CLOSURE_ITERATION_LIMIT_EXCEEDED",
        })
        break

    cell06_changed = False

    for cell_number in sorted(
        list(
            cell06_requested_targets
        )
    ):
        targets = set(
            cell06_requested_targets[
                cell_number
            ]
        )

        new_slice = (
            cell06_slice_cell(
                cell_number,
                targets,
            )
        )

        old_signature = None

        if (
            cell_number
            in
            cell06_cell_slices
        ):
            previous = (
                cell06_cell_slices[
                    cell_number
                ]
            )

            old_signature = (
                tuple(
                    previous[
                        "targets"
                    ]
                ),
                tuple(
                    previous[
                        "statement_ranges"
                    ]
                ),
                tuple(
                    previous[
                        "external_dependencies"
                    ]
                ),
            )

        new_signature = (
            tuple(
                new_slice[
                    "targets"
                ]
            ),
            tuple(
                new_slice[
                    "statement_ranges"
                ]
            ),
            tuple(
                new_slice[
                    "external_dependencies"
                ]
            ),
        )

        if (
            old_signature
            !=
            new_signature
        ):
            cell06_cell_slices[
                cell_number
            ] = (
                new_slice
            )
            cell06_changed = True

        for dependency in (
            new_slice[
                "external_dependencies"
            ]
        ):
            provider = (
                cell06_latest_provider(
                    dependency,
                    cell_number,
                )
            )

            if provider is None:
                continue

            before_count = len(
                cell06_requested_targets[
                    provider
                ]
            )

            cell06_requested_targets[
                provider
            ].add(
                dependency
            )

            if (
                len(
                    cell06_requested_targets[
                        provider
                    ]
                )
                >
                before_count
            ):
                cell06_resolution_log.append({
                    "consumer_cell":
                        cell_number,
                    "dependency":
                        dependency,
                    "provider_cell":
                        provider,
                })

                cell06_changed = True


# ============================================================
# 11) Final recomputation
# ============================================================

for cell_number in sorted(
    cell06_requested_targets
):
    cell06_cell_slices[
        cell_number
    ] = cell06_slice_cell(
        cell_number,
        cell06_requested_targets[
            cell_number
        ],
    )

cell06_provider_cells = sorted(
    cell06_cell_slices
)

cell06_total_selected_statements = sum(
    item[
        "selected_statement_count"
    ]
    for item
    in cell06_cell_slices.values()
)


# ============================================================
# 12) Raw unresolved dependencies
# ============================================================

cell06_raw_unresolved = []

for cell_number in (
    cell06_provider_cells
):
    for dependency in (
        cell06_cell_slices[
            cell_number
        ][
            "external_dependencies"
        ]
    ):
        provider = (
            cell06_latest_provider(
                dependency,
                cell_number,
            )
        )

        if provider is None:
            cell06_raw_unresolved.append({
                "consumer_cell":
                    cell_number,
                "dependency":
                    dependency,
            })

cell06_raw_unresolved = sorted(
    cell06_raw_unresolved,
    key=lambda row: (
        row[
            "consumer_cell"
        ],
        row[
            "dependency"
        ],
    ),
)


# ============================================================
# 13) Safe-unresolved classification helpers
# ============================================================

def cell06_parent_map(
    node,
):
    mapping = {}

    for parent in ast.walk(
        node
    ):
        for child in ast.iter_child_nodes(
            parent
        ):
            mapping[
                child
            ] = parent

    return mapping


def cell06_function_scope_names(
    node,
):
    names = (
        cell06_function_local_names(
            node
        )
    )

    return names


def cell06_scope_binds_name(
    node,
    dependency,
    parent_map,
):
    current = node

    while current in parent_map:
        current = (
            parent_map[
                current
            ]
        )

        if isinstance(
            current,
            ast.Lambda,
        ):
            lambda_names = {
                arg.arg
                for arg
                in (
                    list(
                        current.args.posonlyargs
                    )
                    +
                    list(
                        current.args.args
                    )
                    +
                    list(
                        current.args.kwonlyargs
                    )
                )
            }

            if (
                current.args.vararg
                is not None
            ):
                lambda_names.add(
                    current.args.vararg.arg
                )

            if (
                current.args.kwarg
                is not None
            ):
                lambda_names.add(
                    current.args.kwarg.arg
                )

            if dependency in (
                lambda_names
            ):
                return True

        elif isinstance(
            current,
            (
                ast.ListComp,
                ast.SetComp,
                ast.DictComp,
                ast.GeneratorExp,
            ),
        ):
            names = set()

            for generator in (
                current.generators
            ):
                names.update(
                    cell06_target_names(
                        generator.target
                    )
                )

            if dependency in names:
                return True

        elif isinstance(
            current,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
            ),
        ):
            if dependency in (
                cell06_function_scope_names(
                    current
                )
            ):
                return True

        elif isinstance(
            current,
            ast.ExceptHandler,
        ):
            if (
                isinstance(
                    current.name,
                    str,
                )
                and
                current.name
                ==
                dependency
            ):
                return True

    return False


def cell06_node_inside(
    node,
    subtree,
):
    return any(
        child is node
        for child
        in ast.walk(
            subtree
        )
    )


def cell06_is_dead_constant_branch(
    node,
    parent_map,
):
    current = node

    while current in parent_map:
        current = (
            parent_map[
                current
            ]
        )

        if isinstance(
            current,
            ast.IfExp,
        ):
            if isinstance(
                current.test,
                ast.Constant,
            ):
                condition = bool(
                    current.test.value
                )

                if (
                    condition is False
                    and
                    cell06_node_inside(
                        node,
                        current.body,
                    )
                ):
                    return True

                if (
                    condition is True
                    and
                    cell06_node_inside(
                        node,
                        current.orelse,
                    )
                ):
                    return True

        elif isinstance(
            current,
            ast.If,
        ):
            if isinstance(
                current.test,
                ast.Constant,
            ):
                condition = bool(
                    current.test.value
                )

                if condition is False:
                    if any(
                        cell06_node_inside(
                            node,
                            statement,
                        )
                        for statement
                        in current.body
                    ):
                        return True

                if condition is True:
                    if any(
                        cell06_node_inside(
                            node,
                            statement,
                        )
                        for statement
                        in current.orelse
                    ):
                        return True

    return False


def cell06_same_statement_initialization(
    cell_number,
    dependency,
):
    selected = (
        cell06_cell_slices[
            cell_number
        ][
            "selected_statements"
        ]
    )

    for statement in selected:
        if dependency not in (
            statement[
                "provider_writes"
            ]
        ):
            continue

        store_lines = []
        load_lines = []

        for child in ast.walk(
            statement[
                "node"
            ]
        ):
            if not (
                isinstance(
                    child,
                    ast.Name,
                )
                and
                child.id
                ==
                dependency
            ):
                continue

            if isinstance(
                child.ctx,
                ast.Store,
            ):
                store_lines.append(
                    getattr(
                        child,
                        "lineno",
                        10**12,
                    )
                )

            elif isinstance(
                child.ctx,
                ast.Load,
            ):
                load_lines.append(
                    getattr(
                        child,
                        "lineno",
                        -1,
                    )
                )

        if (
            store_lines
            and
            load_lines
            and
            min(
                store_lines
            )
            <
            min(
                load_lines
            )
        ):
            return {
                "statement_range":
                    statement[
                        "range"
                    ],
                "first_store_line":
                    min(
                        store_lines
                    ),
                "first_load_line":
                    min(
                        load_lines
                    ),
            }

    return None


def cell06_selected_load_occurrences(
    cell_number,
    dependency,
):
    occurrences = []

    for statement in (
        cell06_cell_slices[
            cell_number
        ][
            "selected_statements"
        ]
    ):
        node = (
            statement[
                "node"
            ]
        )

        parent_map = (
            cell06_parent_map(
                node
            )
        )

        for child in ast.walk(
            node
        ):
            if not (
                isinstance(
                    child,
                    ast.Name,
                )
                and
                isinstance(
                    child.ctx,
                    ast.Load,
                )
                and
                child.id
                ==
                dependency
            ):
                continue

            occurrences.append({
                "node":
                    child,
                "line":
                    getattr(
                        child,
                        "lineno",
                        None,
                    ),
                "scope_bound":
                    cell06_scope_binds_name(
                        child,
                        dependency,
                        parent_map,
                    ),
                "dead_constant_branch":
                    cell06_is_dead_constant_branch(
                        child,
                        parent_map,
                    ),
            })

    return occurrences


# ============================================================
# 14) Classify unresolved dependencies
# ============================================================

cell06_safe_unresolved = []
cell06_blocking_unresolved = []

for row in (
    cell06_raw_unresolved
):
    consumer_cell = (
        row[
            "consumer_cell"
        ]
    )

    dependency = (
        row[
            "dependency"
        ]
    )

    same_statement = (
        cell06_same_statement_initialization(
            consumer_cell,
            dependency,
        )
    )

    occurrences = (
        cell06_selected_load_occurrences(
            consumer_cell,
            dependency,
        )
    )

    load_scope_safe = (
        bool(
            occurrences
        )
        and
        all(
            occurrence[
                "scope_bound"
            ]
            for occurrence
            in occurrences
        )
    )

    load_dead_safe = (
        bool(
            occurrences
        )
        and
        all(
            (
                occurrence[
                    "scope_bound"
                ]
                or
                occurrence[
                    "dead_constant_branch"
                ]
            )
            for occurrence
            in occurrences
        )
    )

    if (
        same_statement
        is not None
    ):
        cell06_safe_unresolved.append({
            **row,
            "classification":
                "SAME_STATEMENT_INITIALIZATION_PRECEDES_LOAD",
            "evidence":
                same_statement,
        })

    elif load_scope_safe:
        cell06_safe_unresolved.append({
            **row,
            "classification":
                "LEXICALLY_BOUND_LOCAL_NAME",
            "evidence":
                {
                    "load_lines":
                        [
                            occurrence[
                                "line"
                            ]
                            for occurrence
                            in occurrences
                        ],
                },
        })

    elif load_dead_safe:
        cell06_safe_unresolved.append({
            **row,
            "classification":
                "ONLY_SCOPE_BOUND_OR_DEAD_CONSTANT_BRANCH",
            "evidence":
                {
                    "load_lines":
                        [
                            occurrence[
                                "line"
                            ]
                            for occurrence
                            in occurrences
                        ],
                },
        })

    else:
        cell06_blocking_unresolved.append({
            **row,
            "occurrences":
                [
                    {
                        "line":
                            occurrence[
                                "line"
                            ],
                        "scope_bound":
                            occurrence[
                                "scope_bound"
                            ],
                        "dead_constant_branch":
                            occurrence[
                                "dead_constant_branch"
                            ],
                    }
                    for occurrence
                    in occurrences
                ],
        })


# ============================================================
# 15) Cell 8 mutation regression
# ============================================================

cell06_cell8_ranges = set(
    cell06_cell_slices.get(
        8,
        {},
    ).get(
        "statement_ranges",
        [],
    )
)

cell06_cell8_mutation_loop_included = all([
    (81, 81)
    in
    cell06_cell8_ranges,

    (83, 91)
    in
    cell06_cell8_ranges,

    (97, 102)
    in
    cell06_cell8_ranges,
])


# ============================================================
# 16) Compactness / final gate
# ============================================================

cell06_prior_notebook_code_cell_count = (
    len(
        cell04_code_cell_index
    )
)

cell06_full_notebook_replay = (
    len(
        cell06_provider_cells
    )
    ==
    cell06_prior_notebook_code_cell_count
)

cell06_compact_ratio = (
    len(
        cell06_provider_cells
    )
    /
    cell06_prior_notebook_code_cell_count
)

cell06_root_present = (
    cell06_root_cell
    in
    cell06_provider_cells
    and
    cell06_root_target
    in
    cell06_cell_slices[
        cell06_root_cell
    ][
        "targets"
    ]
)

cell06_gate = all([
    len(
        cell06_failures
    )
    ==
    0,

    len(
        cell06_blocking_unresolved
    )
    ==
    0,

    cell06_root_present,

    cell06_cell8_mutation_loop_included,

    not cell06_full_notebook_replay,
])


# ============================================================
# 17) Diagnostics
# ============================================================

print(
    "=== FINAL STATE RECURSIVE DEPENDENCY CLOSURE "
    "— EXECUTION-SEMANTICS AWARE ==="
)

print(
    "\nRoot:",
    f"Cell {cell06_root_cell} -> "
    f"{cell06_root_target}",
)

print(
    "\nProvider cells:",
    cell06_provider_cells,
)

print(
    "Provider cell count:",
    len(
        cell06_provider_cells
    ),
)

print(
    "Prior notebook code-cell count:",
    cell06_prior_notebook_code_cell_count,
)

print(
    "Compact provider ratio:",
    round(
        cell06_compact_ratio,
        4,
    ),
)

print(
    "Selected top-level statements:",
    cell06_total_selected_statements,
)

print(
    "Closure iterations:",
    cell06_iteration,
)

print(
    "Full notebook replay required:",
    cell06_full_notebook_replay,
)


print(
    "\n=== CELL 8 MUTATION REGRESSION ==="
)

print(
    "Initialization included:",
    (81, 81)
    in
    cell06_cell8_ranges,
)

print(
    "Mutation loop included:",
    (83, 91)
    in
    cell06_cell8_ranges,
)

print(
    "Consumer included:",
    (97, 102)
    in
    cell06_cell8_ranges,
)

print(
    "Cell 8 regression:",
    (
        "PASS"
        if cell06_cell8_mutation_loop_included
        else "FAIL"
    ),
)


print(
    "\n=== RAW UNRESOLVED DEPENDENCIES ==="
)

print(
    "Count:",
    len(
        cell06_raw_unresolved
    ),
)

for row in (
    cell06_raw_unresolved
):
    print(
        row
    )


print(
    "\n=== SAFELY CLASSIFIED UNRESOLVED ==="
)

print(
    "Count:",
    len(
        cell06_safe_unresolved
    ),
)

for row in (
    cell06_safe_unresolved
):
    print(
        row
    )


print(
    "\n=== BLOCKING UNRESOLVED ==="
)

print(
    "Count:",
    len(
        cell06_blocking_unresolved
    ),
)

for row in (
    cell06_blocking_unresolved
):
    print(
        row
    )


print(
    "\n=== FAILURES ==="
)

print(
    cell06_failures
)


cell06_compact_seed_build_plan = {
    "root_cell":
        cell06_root_cell,

    "root_target":
        cell06_root_target,

    "provider_cell_count":
        len(
            cell06_provider_cells
        ),

    "provider_cells":
        cell06_provider_cells,

    "provider_ratio":
        cell06_compact_ratio,

    "selected_statement_count":
        cell06_total_selected_statements,

    "raw_unresolved_dependency_count":
        len(
            cell06_raw_unresolved
        ),

    "safe_unresolved_dependency_count":
        len(
            cell06_safe_unresolved
        ),

    "blocking_unresolved_dependency_count":
        len(
            cell06_blocking_unresolved
        ),

    "full_notebook_replay":
        cell06_full_notebook_replay,

    "mutation_aware_slicing":
        True,

    "execution_semantics_aware":
        True,

    "same_statement_initialization_aware":
        True,

    "lexical_scope_aware":
        True,

    "constant_dead_branch_aware":
        True,

    "cell8_mutation_regression_pass":
        cell06_cell8_mutation_loop_included,

    "ready_for_isolated_seed_rebuild":
        cell06_gate,

    "write_files":
        False,
}

print(
    "\n=== COMPACT SEED BUILD PLAN ==="
)

print(
    json.dumps(
        cell06_compact_seed_build_plan,
        ensure_ascii=False,
        indent=2,
    )
)

print(
    "\nMutation performed:",
    False,
)

print(
    "\nCODE CELL 06 GATE:",
    "PASS"
    if cell06_gate
    else "FAIL",
)

assert cell06_gate, (
    "Execution-semantics-aware dependency closure failed. "
    "Repair this SAME Code Cell 06 before continuing."
)

=== FINAL STATE RECURSIVE DEPENDENCY CLOSURE — EXECUTION-SEMANTICS AWARE ===

Root: Cell 138 -> cell138_target_values

Provider cells: [1, 2, 3, 4, 5, 6, 7, 8, 9, 14, 15, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 51, 52, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 124, 125, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138]
Provider cell count: 98
Prior notebook code-cell count: 141
Compact provider ratio: 0.695
Selected top-level statements: 2004
Closure iterations: 14
Full notebook replay required: False

=== CELL 8 MUTATION REGRESSION ===
Initialization included: True
Mutation loop included: True
Consumer included: True
Cell 8 regression: PASS

=== RAW UNRESOLVED DEPENDENCIES ===
Count: 9
{'consumer_cell': 4, 'dependency': 'x'}
{'consumer_cell': 19, 'dependency': 'identity_counte

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>بازیابی مستقیم Pilot Population از Outputهای پذیرفته‌شده Notebook قبلی</h3>

<p>
Dependency Closure در Step قبل نشان داد که بازسازی کامل Final Runtime State از Source Notebook ممکن است،
اما به 88 Provider Cell وابسته می‌شود. این مسیر از نظر فنی معتبر است، ولی برای هدف Compact Notebook هنوز بیش از حد سنگین است.
</p>

<p>
قبل از استفاده از آن مسیر، این Step یک مسیر کوتاه‌تر را بررسی می‌کند:
در Notebook قبلی بعضی Cellها Pilot Population را در Output ذخیره کرده‌اند.
اگر یک Snapshot کامل و قابل‌Parse از 23 Pilot قبل از Replacement موجود باشد،
می‌توان Population را مستقیماً از همان Output پذیرفته‌شده بازیابی کرد،
Replacement نهایی را روی همان Snapshot اعمال کرد و Queue جدید را با Contract تثبیت‌شده بازسازی کرد.
</p>

<p>
این روش می‌تواند نیاز به Replay بخش بزرگی از Notebook قبلی را حذف کند.
در این مرحله فقط Outputهای ذخیره‌شده و Read-Only بررسی می‌شوند و هیچ State یا فایلی تغییر نمی‌کند.
</p>

<h4>معیار PASS</h4>

<ul>
<li>Outputهای Candidate Cellهای مرتبط با Pilot Population بررسی شوند.</li>
<li>حداقل یک Snapshot ساختاریافته و قابل‌Parse از Pilot Population پیدا شود، یا نبود آن به‌طور قطعی مشخص شود.</li>
<li>تعداد Pilotها و یکتایی <code>pilot_case_id</code> قابل‌اعتبارسنجی باشد.</li>
<li>وجود Pilot قدیمی یا Replacement در Snapshot مشخص شود.</li>
<li>مسیر بعدی بین Direct Snapshot Recovery و Dependency Replay بدون حدس انتخاب شود.</li>
<li>هیچ Mutation انجام نشود.</li>
</ul>

</div>

In [7]:
# Code Cell 07

cell07_candidate_cells = [
    68,
    79,
    130,
    132,
    135,
    137,
    139,
    140,
]

cell07_failures = []


# ------------------------------------------------------------
# 1) Resolve Pilot identities from accepted notebook outputs
# ------------------------------------------------------------

cell07_new_pilot_id = (
    cell03_next_pilot_id
)


cell07_identity_scan_cells = [
    96,
    97,
    98,
    99,
    100,
    130,
    132,
    136,
    138,
    139,
    140,
]


cell07_pilot_id_pattern = re.compile(
    r"question-pilot:[0-9a-f]{32}"
)


cell07_identity_hits = {}


for cell_number in (
    cell07_identity_scan_cells
):

    record = (
        cell04_code_cell_index.get(
            cell_number
        )
    )

    if record is None:
        continue

    output_text = (
        record.get(
            "output"
        )
        or ""
    )

    ids = (
        cell07_pilot_id_pattern.findall(
            output_text
        )
    )

    cell07_identity_hits[
        cell_number
    ] = ids


# The blocked Pilot was the active Pilot throughout Cells 96-100.
cell07_pre_replacement_ids = []


for cell_number in [
    96,
    97,
    98,
    99,
    100,
]:
    cell07_pre_replacement_ids.extend(
        cell07_identity_hits.get(
            cell_number,
            []
        )
    )


cell07_pre_replacement_counts = Counter(
    pilot_id

    for pilot_id
    in cell07_pre_replacement_ids

    if pilot_id
    !=
    cell07_new_pilot_id
)


cell07_ranked_old_candidates = (
    cell07_pre_replacement_counts.most_common()
)


cell07_old_pilot_id = None


if cell07_ranked_old_candidates:

    top_count = (
        cell07_ranked_old_candidates[
            0
        ][
            1
        ]
    )

    top_ids = [
        pilot_id

        for pilot_id, count
        in cell07_ranked_old_candidates

        if count == top_count
    ]

    if len(top_ids) == 1:
        cell07_old_pilot_id = (
            top_ids[0]
        )


# Secondary evidence from explicit labels in later outputs.
cell07_labeled_old_ids = []


cell07_old_label_patterns = [
    re.compile(
        r"Old Pilot(?: Case ID| ID)?\s*:\s*(question-pilot:[0-9a-f]{32})",
        flags=re.IGNORECASE,
    ),
    re.compile(
        r"old_pilot(?:_case)?_id['\"]?\s*[:=]\s*['\"]?(question-pilot:[0-9a-f]{32})",
        flags=re.IGNORECASE,
    ),
]


for cell_number in [
    132,
    136,
    138,
    139,
]:

    record = (
        cell04_code_cell_index.get(
            cell_number
        )
    )

    if record is None:
        continue

    combined_text = (
        record.get(
            "source",
            ""
        )
        +
        "\n"
        +
        (
            record.get(
                "output"
            )
            or ""
        )
    )

    for pattern in (
        cell07_old_label_patterns
    ):
        cell07_labeled_old_ids.extend(
            pattern.findall(
                combined_text
            )
        )


cell07_labeled_old_ids = sorted(
    set(
        cell07_labeled_old_ids
    )
)


if (
    cell07_old_pilot_id
    is None
    and
    len(
        cell07_labeled_old_ids
    )
    ==
    1
):
    cell07_old_pilot_id = (
        cell07_labeled_old_ids[
            0
        ]
    )


cell07_identity_resolution_valid = (
    isinstance(
        cell07_old_pilot_id,
        str,
    )
    and
    cell07_old_pilot_id.startswith(
        "question-pilot:"
    )
    and
    isinstance(
        cell07_new_pilot_id,
        str,
    )
    and
    cell07_new_pilot_id.startswith(
        "question-pilot:"
    )
    and
    cell07_old_pilot_id
    !=
    cell07_new_pilot_id
)


if not cell07_identity_resolution_valid:

    cell07_failures.append({
        "reason":
            "PILOT_IDENTITY_RESOLUTION_FAILED",

        "old_candidate_counts":
            dict(
                cell07_pre_replacement_counts
            ),

        "labeled_old_ids":
            cell07_labeled_old_ids,

        "new_pilot_id":
            cell07_new_pilot_id,
    })


# ------------------------------------------------------------
# 2) Parse structured objects from saved outputs
# ------------------------------------------------------------

def cell07_balanced_fragments(
    text,
    opener,
    closer,
):
    fragments = []

    start = None
    depth = 0
    quote = None
    escaped = False

    for index, char in enumerate(
        text
    ):

        if quote is not None:

            if escaped:
                escaped = False
                continue

            if char == "\\":
                escaped = True
                continue

            if char == quote:
                quote = None

            continue

        if char in (
            "'",
            '"',
        ):
            quote = char
            continue

        if char == opener:

            if depth == 0:
                start = index

            depth += 1

        elif (
            char == closer
            and
            depth > 0
        ):

            depth -= 1

            if (
                depth == 0
                and
                start is not None
            ):
                fragments.append(
                    text[
                        start:index + 1
                    ]
                )

                start = None

    return fragments


def cell07_collect_objects(
    text,
):
    fragments = (
        cell07_balanced_fragments(
            text,
            "[",
            "]",
        )
        +
        cell07_balanced_fragments(
            text,
            "{",
            "}",
        )
    )

    parsed = []
    seen = set()

    for fragment in fragments:

        if fragment in seen:
            continue

        seen.add(
            fragment
        )

        value = None
        parser = None

        try:
            value = json.loads(
                fragment
            )
            parser = "json"

        except Exception:
            try:
                value = ast.literal_eval(
                    fragment
                )
                parser = "python_literal"

            except Exception:
                continue

        parsed.append({
            "value":
                value,

            "parser":
                parser,

            "chars":
                len(
                    fragment
                ),
        })

    return parsed


def cell07_is_pilot_packet(
    value,
):
    return (
        isinstance(
            value,
            dict,
        )
        and
        isinstance(
            value.get(
                "pilot_case_id"
            ),
            str,
        )
        and
        value.get(
            "pilot_case_id"
        ).startswith(
            "question-pilot:"
        )
        and
        isinstance(
            value.get(
                "golden_case_id"
            ),
            str,
        )
        and
        isinstance(
            value.get(
                "target_evidence_id"
            ),
            str,
        )
    )


# ------------------------------------------------------------
# 3) Recover complete Pilot Population snapshots
# ------------------------------------------------------------

cell07_population_candidates = []

cell07_cell_inspection = {}


for cell_number in (
    cell07_candidate_cells
):

    record = (
        cell04_code_cell_index.get(
            cell_number
        )
    )

    if record is None:

        cell07_cell_inspection[
            cell_number
        ] = {
            "found":
                False,
        }

        continue


    output_text = (
        record.get(
            "output"
        )
        or ""
    )


    objects = (
        cell07_collect_objects(
            output_text
        )
    )


    local_candidates = []


    for item in objects:

        value = (
            item[
                "value"
            ]
        )

        if (
            not isinstance(
                value,
                list,
            )
            or
            not value
            or
            not all(
                cell07_is_pilot_packet(
                    packet
                )
                for packet in value
            )
        ):
            continue


        pilot_ids = [
            packet[
                "pilot_case_id"
            ]

            for packet
            in value
        ]


        candidate = {
            "cell":
                cell_number,

            "count":
                len(
                    value
                ),

            "unique_ids":
                len(
                    set(
                        pilot_ids
                    )
                ),

            "old_pilot_count":
                (
                    pilot_ids.count(
                        cell07_old_pilot_id
                    )
                    if cell07_old_pilot_id
                    is not None
                    else None
                ),

            "replacement_pilot_count":
                pilot_ids.count(
                    cell07_new_pilot_id
                ),

            "parser":
                item[
                    "parser"
                ],

            "chars":
                item[
                    "chars"
                ],

            "packets":
                value,
        }


        local_candidates.append(
            candidate
        )

        cell07_population_candidates.append(
            candidate
        )


    cell07_cell_inspection[
        cell_number
    ] = {
        "found":
            True,

        "output_chars":
            len(
                output_text
            ),

        "parsed_object_count":
            len(
                objects
            ),

        "packet_list_candidates":
            [
                {
                    key:
                        value

                    for key, value
                    in candidate.items()

                    if key
                    !=
                    "packets"
                }

                for candidate
                in local_candidates
            ],
    }


cell07_valid_23_candidates = [
    candidate

    for candidate
    in cell07_population_candidates

    if (
        candidate[
            "count"
        ]
        ==
        23

        and

        candidate[
            "unique_ids"
        ]
        ==
        23
    )
]


cell07_pre_replacement_candidates = [
    candidate

    for candidate
    in cell07_valid_23_candidates

    if (
        candidate[
            "old_pilot_count"
        ]
        ==
        1

        and

        candidate[
            "replacement_pilot_count"
        ]
        ==
        0
    )
]


cell07_post_replacement_candidates = [
    candidate

    for candidate
    in cell07_valid_23_candidates

    if (
        candidate[
            "old_pilot_count"
        ]
        ==
        0

        and

        candidate[
            "replacement_pilot_count"
        ]
        ==
        1
    )
]


# ------------------------------------------------------------
# 4) Select best available snapshot
# ------------------------------------------------------------

cell07_best_snapshot = None


if cell07_post_replacement_candidates:

    cell07_best_snapshot = sorted(
        cell07_post_replacement_candidates,
        key=lambda row: (
            -row[
                "cell"
            ],
            -row[
                "chars"
            ],
        ),
    )[0]


elif cell07_pre_replacement_candidates:

    cell07_best_snapshot = sorted(
        cell07_pre_replacement_candidates,
        key=lambda row: (
            -row[
                "cell"
            ],
            -row[
                "chars"
            ],
        ),
    )[0]


cell07_direct_snapshot_available = (
    cell07_best_snapshot
    is not None
)


if cell07_best_snapshot is not None:

    cell07_snapshot_mode = (
        "POST_REPLACEMENT_ACTIVE_POPULATION"

        if (
            cell07_best_snapshot[
                "replacement_pilot_count"
            ]
            ==
            1
        )

        else

        "PRE_REPLACEMENT_POPULATION_REQUIRES_DETERMINISTIC_REPLACEMENT"
    )


    cell07_snapshot_packets = (
        cell07_best_snapshot[
            "packets"
        ]
    )


    cell07_snapshot_pilot_ids = [
        packet[
            "pilot_case_id"
        ]

        for packet
        in cell07_snapshot_packets
    ]


    cell07_snapshot_sha256 = (
        hashlib.sha256(
            json.dumps(
                cell07_snapshot_packets,
                ensure_ascii=False,
                sort_keys=True,
                separators=(
                    ",",
                    ":",
                ),
            ).encode(
                "utf-8"
            )
        ).hexdigest()
    )


else:

    cell07_snapshot_mode = (
        "NO_COMPLETE_23_PILOT_SNAPSHOT_RECOVERED"
    )

    cell07_snapshot_packets = None

    cell07_snapshot_pilot_ids = []

    cell07_snapshot_sha256 = None


# ------------------------------------------------------------
# 5) Route decision
# ------------------------------------------------------------

cell07_route = (
    "BUILD_COMPACT_SEED_FROM_RECOVERED_PILOT_SNAPSHOT"

    if cell07_direct_snapshot_available

    else

    "FALL_BACK_TO_ACCEPTED_DEPENDENCY_CLOSURE_REBUILD"
)


cell07_gate = all([
    len(
        cell07_failures
    )
    ==
    0,

    cell07_identity_resolution_valid,

    cell06_gate
    is True,

    cell03_prior_notebook_final_state_valid
    is True,

    cell07_route
    in {
        "BUILD_COMPACT_SEED_FROM_RECOVERED_PILOT_SNAPSHOT",
        "FALL_BACK_TO_ACCEPTED_DEPENDENCY_CLOSURE_REBUILD",
    },
])


# ------------------------------------------------------------
# 6) Diagnostics
# ------------------------------------------------------------

print(
    "=== DIRECT PILOT POPULATION SNAPSHOT RECOVERY ==="
)


print(
    "\n=== PILOT IDENTITY ==="
)

print(
    "Old Pilot:",
    cell07_old_pilot_id,
)

print(
    "Replacement Pilot:",
    cell07_new_pilot_id,
)

print(
    "Pre-replacement evidence counts:",
    dict(
        cell07_pre_replacement_counts
    ),
)

print(
    "Labeled old-ID evidence:",
    cell07_labeled_old_ids,
)

print(
    "Identity resolution valid:",
    cell07_identity_resolution_valid,
)


print(
    "\n=== POPULATION SNAPSHOTS ==="
)

print(
    "Packet-list candidates:",
    len(
        cell07_population_candidates
    ),
)

print(
    "Valid 23-pilot candidates:",
    len(
        cell07_valid_23_candidates
    ),
)

print(
    "Pre-replacement candidates:",
    [
        {
            "cell":
                row[
                    "cell"
                ],

            "chars":
                row[
                    "chars"
                ],

            "parser":
                row[
                    "parser"
                ],
        }

        for row
        in cell07_pre_replacement_candidates
    ],
)

print(
    "Post-replacement candidates:",
    [
        {
            "cell":
                row[
                    "cell"
                ],

            "chars":
                row[
                    "chars"
                ],

            "parser":
                row[
                    "parser"
                ],
        }

        for row
        in cell07_post_replacement_candidates
    ],
)


print(
    "\n=== SELECTED SNAPSHOT ==="
)

print(
    "Direct snapshot available:",
    cell07_direct_snapshot_available,
)

print(
    "Snapshot mode:",
    cell07_snapshot_mode,
)

print(
    "Source Cell:",
    (
        cell07_best_snapshot[
            "cell"
        ]

        if cell07_best_snapshot
        is not None

        else None
    ),
)

print(
    "Pilot count:",
    len(
        cell07_snapshot_pilot_ids
    ),
)

print(
    "Unique Pilot IDs:",
    len(
        set(
            cell07_snapshot_pilot_ids
        )
    ),
)

print(
    "Snapshot SHA256:",
    cell07_snapshot_sha256,
)


print(
    "\n=== ROUTE DECISION ==="
)

print(
    "Route:",
    cell07_route,
)

print(
    "Mutation performed:",
    False,
)

print(
    "Failures:",
    cell07_failures,
)


print(
    "\nCODE CELL 07 GATE:",
    "PASS"
    if cell07_gate
    else "FAIL",
)


assert cell07_gate, (
    "Direct Pilot Population snapshot inspection failed. "
    "Repair this SAME Code Cell 07 before constructing the Compact Runtime Seed."
)

=== DIRECT PILOT POPULATION SNAPSHOT RECOVERY ===

=== PILOT IDENTITY ===
Old Pilot: question-pilot:532a8b9d45efb8b3a32c4a62f096bb7a
Replacement Pilot: question-pilot:3005b48518a7aeb532234332df23c952
Pre-replacement evidence counts: {'question-pilot:532a8b9d45efb8b3a32c4a62f096bb7a': 2}
Labeled old-ID evidence: ['question-pilot:532a8b9d45efb8b3a32c4a62f096bb7a']
Identity resolution valid: True

=== POPULATION SNAPSHOTS ===
Packet-list candidates: 1
Valid 23-pilot candidates: 0
Pre-replacement candidates: []
Post-replacement candidates: []

=== SELECTED SNAPSHOT ===
Direct snapshot available: False
Snapshot mode: NO_COMPLETE_23_PILOT_SNAPSHOT_RECOVERED
Source Cell: None
Pilot count: 0
Unique Pilot IDs: 0
Snapshot SHA256: None

=== ROUTE DECISION ===
Route: FALL_BACK_TO_ACCEPTED_DEPENDENCY_CLOSURE_REBUILD
Mutation performed: False
Failures: []

CODE CELL 07 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Safety Audit برای اجرای فشرده Dependency Closure</h3>

<p>
Cell 07 نشان داد که Snapshot کامل Pilot Population در Outputهای ذخیره‌شده وجود ندارد.
بنابراین مسیر معتبر همان Dependency Closure پذیرفته‌شده در Cell 06 است.
</p>

<p>
اما قبل از اینکه 88 Provider Cell لازم را در یک Runtime ایزوله و فشرده اجرا کنیم،
باید مطمئن شویم Statementهای انتخاب‌شده فقط State موردنیاز را بازسازی می‌کنند و
هیچ عملیات ناخواسته‌ای مانند File Write، Delete، Git Mutation، Network Request یا اجرای Subprocess تغییردهنده ندارند.
</p>

<p>
این Step فقط Source انتخاب‌شده را با AST Audit می‌کند.
هیچ Statementی اجرا نمی‌شود و هیچ فایل یا Runtime State پروژه تغییر نمی‌کند.
</p>

<h4>معیار PASS</h4>

<ul>
<li>تمام Sliceهای Cell 06 قابل Parse باشند.</li>
<li>Callهای حساس در Statementهای انتخاب‌شده شناسایی شوند.</li>
<li>هیچ Write/Delete/Git/Network Mutation قطعی در Slice انتخاب‌شده وجود نداشته باشد.</li>
<li>اگر Call مبهمی وجود دارد، قبل از اجرای Seed به‌صورت صریح گزارش شود.</li>
<li>هیچ Mutation انجام نشود.</li>
</ul>

</div>

In [8]:
# Code Cell 08

cell08_failures = []


# ------------------------------------------------------------
# 1) High-confidence side-effect rules
# ------------------------------------------------------------

cell08_high_confidence_path_methods = {
    "write_text",
    "write_bytes",
    "unlink",
    "rename",
    "mkdir",
    "rmdir",
    "touch",
    "chmod",
    "symlink_to",
    "hardlink_to",
}


cell08_path_like_name_pattern = re.compile(
    r"(?:^|_)(?:path|file|dir|folder|root|artifact|output|report|manifest)(?:_|$)",
    flags=re.IGNORECASE,
)


cell08_filesystem_functions = {
    "os.remove",
    "os.unlink",
    "os.rename",
    "os.replace",
    "os.rmdir",
    "os.mkdir",
    "os.makedirs",
    "os.removedirs",
    "shutil.rmtree",
    "shutil.move",
    "shutil.copy",
    "shutil.copy2",
    "shutil.copytree",
}


cell08_network_mutation_calls = {
    "requests.post",
    "requests.put",
    "requests.patch",
    "requests.delete",
    "httpx.post",
    "httpx.put",
    "httpx.patch",
    "httpx.delete",
}


cell08_known_write_calls = {
    "json.dump",
    "pickle.dump",
}


cell08_git_read_only_subcommands = {
    "status",
    "rev-parse",
    "show",
    "log",
    "diff",
    "ls-files",
    "ls-tree",
    "cat-file",
    "describe",
    "remote",
}


cell08_git_mutating_subcommands = {
    "add",
    "commit",
    "push",
    "merge",
    "rebase",
    "reset",
    "checkout",
    "switch",
    "tag",
    "rm",
    "mv",
    "restore",
    "clean",
    "cherry-pick",
    "revert",
    "stash",
}


# ------------------------------------------------------------
# 2) AST helpers
# ------------------------------------------------------------

def cell08_call_name(node):

    if isinstance(
        node,
        ast.Name,
    ):
        return node.id

    if isinstance(
        node,
        ast.Attribute,
    ):
        prefix = (
            cell08_call_name(
                node.value
            )
        )

        return (
            f"{prefix}.{node.attr}"
            if prefix
            else node.attr
        )

    return None


def cell08_literal_string(node):

    if (
        isinstance(
            node,
            ast.Constant,
        )
        and
        isinstance(
            node.value,
            str,
        )
    ):
        return node.value

    return None


def cell08_receiver_name(node):

    if not isinstance(
        node,
        ast.Attribute,
    ):
        return None

    receiver = (
        node.value
    )

    if isinstance(
        receiver,
        ast.Name,
    ):
        return receiver.id

    return (
        cell08_call_name(
            receiver
        )
    )


def cell08_source_segment(
    source,
    node,
):

    segment = ast.get_source_segment(
        source,
        node,
    )

    return (
        segment.strip()
        if isinstance(
            segment,
            str,
        )
        else None
    )


# ------------------------------------------------------------
# 3) File-open mode audit
# ------------------------------------------------------------

def cell08_open_mode(
    call,
):

    name = (
        cell08_call_name(
            call.func
        )
    )

    if not (
        name == "open"
        or
        (
            name
            and
            name.endswith(
                ".open"
            )
        )
    ):
        return None

    mode = None

    if len(
        call.args
    ) >= 2:
        mode = (
            cell08_literal_string(
                call.args[1]
            )
        )

    for keyword in (
        call.keywords
    ):
        if keyword.arg == "mode":
            mode = (
                cell08_literal_string(
                    keyword.value
                )
            )

    return (
        mode
        if mode is not None
        else "r"
    )


def cell08_open_is_write(
    call,
):

    mode = (
        cell08_open_mode(
            call
        )
    )

    if mode is None:
        return False

    return any(
        token in mode
        for token in (
            "w",
            "a",
            "x",
            "+",
        )
    )


# ------------------------------------------------------------
# 4) Git subprocess classification
# ------------------------------------------------------------

def cell08_literal_command(
    call,
):

    if not call.args:
        return None

    command_node = (
        call.args[0]
    )

    if not isinstance(
        command_node,
        (
            ast.List,
            ast.Tuple,
        ),
    ):
        return None

    command = []

    for element in (
        command_node.elts
    ):
        value = (
            cell08_literal_string(
                element
            )
        )

        if value is None:
            return None

        command.append(
            value
        )

    return command


def cell08_classify_subprocess(
    call,
):

    name = (
        cell08_call_name(
            call.func
        )
    )

    if name not in {
        "subprocess.run",
        "subprocess.call",
        "subprocess.check_call",
        "subprocess.check_output",
        "subprocess.Popen",
    }:
        return None

    command = (
        cell08_literal_command(
            call
        )
    )

    if command is None:
        return {
            "status":
                "AMBIGUOUS_SUBPROCESS",

            "command":
                None,
        }

    if not command:
        return None

    if command[0] != "git":
        return {
            "status":
                "NON_GIT_SUBPROCESS",

            "command":
                command,
        }

    if len(command) < 2:
        return {
            "status":
                "AMBIGUOUS_GIT",

            "command":
                command,
        }

    subcommand = (
        command[1]
    )


    if (
        subcommand == "branch"
        and
        "--show-current"
        in command
    ):
        return {
            "status":
                "GIT_READ_ONLY",

            "command":
                command,
        }


    if (
        subcommand == "remote"
        and
        not any(
            token
            in command
            for token in {
                "add",
                "remove",
                "rename",
                "set-url",
            }
        )
    ):
        return {
            "status":
                "GIT_READ_ONLY",

            "command":
                command,
        }


    if subcommand in (
        cell08_git_read_only_subcommands
    ):
        return {
            "status":
                "GIT_READ_ONLY",

            "command":
                command,
        }


    if subcommand in (
        cell08_git_mutating_subcommands
    ):
        return {
            "status":
                "GIT_MUTATION",

            "command":
                command,
        }


    if subcommand == "branch":
        return {
            "status":
                "AMBIGUOUS_GIT_BRANCH",

            "command":
                command,
        }


    return {
        "status":
            "AMBIGUOUS_GIT",

        "command":
            command,
    }


# ------------------------------------------------------------
# 5) Method-call classification
# ------------------------------------------------------------

def cell08_classify_method_call(
    node,
):

    if not isinstance(
        node.func,
        ast.Attribute,
    ):
        return None

    method = (
        node.func.attr
    )

    receiver = (
        cell08_receiver_name(
            node.func
        )
    )


    if method in (
        cell08_high_confidence_path_methods
    ):
        return {
            "severity":
                "BLOCKING",

            "type":
                "HIGH_CONFIDENCE_PATH_MUTATION",

            "receiver":
                receiver,

            "method":
                method,
        }


    # .replace() is common on strings.
    # Treat it as a path mutation only when the receiver itself
    # is explicitly path-like.
    if method == "replace":

        if (
            isinstance(
                receiver,
                str,
            )
            and
            cell08_path_like_name_pattern.search(
                receiver
            )
        ):
            return {
                "severity":
                    "BLOCKING",

                "type":
                    "PATH_LIKE_REPLACE",

                "receiver":
                    receiver,

                "method":
                    method,
            }

        return {
            "severity":
                "SAFE_KNOWN_FALSE_POSITIVE",

            "type":
                "GENERIC_REPLACE_METHOD",

            "receiver":
                receiver,

            "method":
                method,
        }


    # .remove() on a normal container is not a filesystem write.
    if method == "remove":

        if (
            isinstance(
                receiver,
                str,
            )
            and
            cell08_path_like_name_pattern.search(
                receiver
            )
        ):
            return {
                "severity":
                    "REVIEW",

                "type":
                    "PATH_LIKE_REMOVE_METHOD",

                "receiver":
                    receiver,

                "method":
                    method,
            }

        return {
            "severity":
                "SAFE_KNOWN_FALSE_POSITIVE",

            "type":
                "CONTAINER_REMOVE_METHOD",

            "receiver":
                receiver,

            "method":
                method,
        }


    return None


# ------------------------------------------------------------
# 6) Audit selected dependency closure
# ------------------------------------------------------------

cell08_findings = []


for cell_number in (
    cell06_provider_cells
):

    record = (
        cell04_code_cell_index[
            cell_number
        ]
    )

    source = (
        record[
            "source"
        ]
    )

    slice_info = (
        cell06_cell_slices[
            cell_number
        ]
    )


    try:
        tree = ast.parse(
            source
        )

    except SyntaxError as exc:

        cell08_failures.append({
            "reason":
                "SOURCE_PARSE_FAILED",

            "cell":
                cell_number,

            "error":
                repr(
                    exc
                ),
        })

        continue


    selected_ranges = set(
        tuple(
            item
        )
        for item in slice_info[
            "statement_ranges"
        ]
    )


    selected_nodes = [
        statement

        for statement
        in tree.body

        if (
            getattr(
                statement,
                "lineno",
                None,
            ),
            getattr(
                statement,
                "end_lineno",
                None,
            ),
        )
        in selected_ranges
    ]


    for statement in (
        selected_nodes
    ):

        for node in ast.walk(
            statement
        ):

            if not isinstance(
                node,
                ast.Call,
            ):
                continue


            call_name = (
                cell08_call_name(
                    node.func
                )
            )


            source_segment = (
                cell08_source_segment(
                    source,
                    node,
                )
            )


            finding = None


            # -----------------------------------------------
            # File open with explicit write mode
            # -----------------------------------------------

            if cell08_open_is_write(
                node
            ):

                finding = {
                    "severity":
                        "BLOCKING",

                    "type":
                        "FILE_OPEN_WRITE",

                    "call":
                        call_name,
                }


            # -----------------------------------------------
            # Explicit filesystem module functions
            # -----------------------------------------------

            elif call_name in (
                cell08_filesystem_functions
            ):

                finding = {
                    "severity":
                        "BLOCKING",

                    "type":
                        "EXPLICIT_FILESYSTEM_MUTATION",

                    "call":
                        call_name,
                }


            # -----------------------------------------------
            # Explicit serialization writes
            # -----------------------------------------------

            elif call_name in (
                cell08_known_write_calls
            ):

                finding = {
                    "severity":
                        "BLOCKING",

                    "type":
                        "SERIALIZATION_WRITE",

                    "call":
                        call_name,
                }


            # -----------------------------------------------
            # Network mutation
            # -----------------------------------------------

            elif call_name in (
                cell08_network_mutation_calls
            ):

                finding = {
                    "severity":
                        "BLOCKING",

                    "type":
                        "NETWORK_MUTATION",

                    "call":
                        call_name,
                }


            # -----------------------------------------------
            # Subprocess / Git
            # -----------------------------------------------

            else:

                subprocess_result = (
                    cell08_classify_subprocess(
                        node
                    )
                )


                if subprocess_result is not None:

                    status = (
                        subprocess_result[
                            "status"
                        ]
                    )


                    if status == "GIT_MUTATION":

                        finding = {
                            "severity":
                                "BLOCKING",

                            "type":
                                "GIT_MUTATION",

                            "call":
                                call_name,

                            "command":
                                subprocess_result[
                                    "command"
                                ],
                        }


                    elif status in {
                        "AMBIGUOUS_SUBPROCESS",
                        "NON_GIT_SUBPROCESS",
                        "AMBIGUOUS_GIT",
                        "AMBIGUOUS_GIT_BRANCH",
                    }:

                        finding = {
                            "severity":
                                "REVIEW",

                            "type":
                                status,

                            "call":
                                call_name,

                            "command":
                                subprocess_result[
                                    "command"
                                ],
                        }


                    elif status == "GIT_READ_ONLY":

                        finding = {
                            "severity":
                                "SAFE_READ_ONLY",

                            "type":
                                "GIT_READ_ONLY",

                            "call":
                                call_name,

                            "command":
                                subprocess_result[
                                    "command"
                                ],
                        }


                if finding is None:

                    method_result = (
                        cell08_classify_method_call(
                            node
                        )
                    )


                    if method_result is not None:

                        finding = {
                            "call":
                                call_name,

                            **method_result,
                        }


            if finding is not None:

                cell08_findings.append({
                    "cell":
                        cell_number,

                    "line":
                        getattr(
                            node,
                            "lineno",
                            None,
                        ),

                    "source":
                        source_segment,

                    **finding,
                })


# ------------------------------------------------------------
# 7) Classification
# ------------------------------------------------------------

cell08_blocking_findings = [
    row

    for row
    in cell08_findings

    if row[
        "severity"
    ]
    ==
    "BLOCKING"
]


cell08_review_findings = [
    row

    for row
    in cell08_findings

    if row[
        "severity"
    ]
    ==
    "REVIEW"
]


cell08_read_only_findings = [
    row

    for row
    in cell08_findings

    if row[
        "severity"
    ]
    ==
    "SAFE_READ_ONLY"
]


cell08_false_positive_findings = [
    row

    for row
    in cell08_findings

    if row[
        "severity"
    ]
    ==
    "SAFE_KNOWN_FALSE_POSITIVE"
]


cell08_expected_false_positive_types = {
    "GENERIC_REPLACE_METHOD",
    "CONTAINER_REMOVE_METHOD",
}


cell08_false_positive_types_valid = all(
    row[
        "type"
    ]
    in cell08_expected_false_positive_types

    for row
    in cell08_false_positive_findings
)


cell08_safe_for_isolated_rebuild = all([
    len(
        cell08_failures
    )
    ==
    0,

    len(
        cell08_blocking_findings
    )
    ==
    0,

    len(
        cell08_review_findings
    )
    ==
    0,

    cell08_false_positive_types_valid,
])


# ------------------------------------------------------------
# 8) Gate
# ------------------------------------------------------------

cell08_gate = (
    cell08_safe_for_isolated_rebuild
)


# ------------------------------------------------------------
# 9) Diagnostics
# ------------------------------------------------------------

print(
    "=== COMPACT REBUILD SAFETY AUDIT ==="
)


print(
    "\nValidator repair:",
    "Git read-only commands, generic replace(), and container remove() "
    "are distinguished from real external mutations.",
)


print(
    "\nProvider cells audited:",
    len(
        cell06_provider_cells
    ),
)

print(
    "Selected statements audited:",
    cell06_total_selected_statements,
)


print(
    "\n=== BLOCKING FINDINGS ==="
)

print(
    "Count:",
    len(
        cell08_blocking_findings
    ),
)

for row in (
    cell08_blocking_findings
):
    print(
        row
    )


print(
    "\n=== REVIEW FINDINGS ==="
)

print(
    "Count:",
    len(
        cell08_review_findings
    ),
)

for row in (
    cell08_review_findings
):
    print(
        row
    )


print(
    "\n=== CONFIRMED READ-ONLY FINDINGS ==="
)

print(
    "Count:",
    len(
        cell08_read_only_findings
    ),
)

for row in (
    cell08_read_only_findings
):
    print(
        row
    )


print(
    "\n=== CONFIRMED FALSE POSITIVES ==="
)

print(
    "Count:",
    len(
        cell08_false_positive_findings
    ),
)

for row in (
    cell08_false_positive_findings
):
    print(
        row
    )


print(
    "\n=== FAILURES ==="
)

print(
    cell08_failures
)


print(
    "\nSafe for isolated compact rebuild:",
    cell08_safe_for_isolated_rebuild,
)

print(
    "Mutation performed:",
    False,
)


print(
    "\nCODE CELL 08 GATE:",
    "PASS"
    if cell08_gate
    else "FAIL",
)


assert cell08_gate, (
    "Selected dependency closure still contains unresolved or blocking side effects. "
    "Repair this SAME Code Cell 08 before executing the compact runtime rebuild."
)

=== COMPACT REBUILD SAFETY AUDIT ===

Validator repair: Git read-only commands, generic replace(), and container remove() are distinguished from real external mutations.

Provider cells audited: 98
Selected statements audited: 2004

=== BLOCKING FINDINGS ===
Count: 0

=== REVIEW FINDINGS ===
Count: 0

=== CONFIRMED READ-ONLY FINDINGS ===
Count: 1
{'cell': 2, 'line': 4, 'source': 'subprocess.check_output(\n        ["git", "rev-parse", "--show-toplevel"],\n        text=True\n    )', 'severity': 'SAFE_READ_ONLY', 'type': 'GIT_READ_ONLY', 'call': 'subprocess.check_output', 'command': ['git', 'rev-parse', '--show-toplevel']}

=== CONFIRMED FALSE POSITIVES ===
Count: 3
{'cell': 35, 'line': 43, 'source': 'str(\n            text\n            or ""\n        )\n        .replace(\n            "\\u200c",\n            " "\n        )', 'call': 'replace', 'severity': 'SAFE_KNOWN_FALSE_POSITIVE', 'type': 'GENERIC_REPLACE_METHOD', 'receiver': None, 'method': 'replace'}
{'cell': 39, 'line': 50, 'source'

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>اجرای Dry-Run بازسازی Final State در Namespace ایزوله</h3>

<p>
Safety Audit در Cell 08 تأیید کرد که Statementهای انتخاب‌شده برای بازسازی Final State،
هیچ File Write، Git Mutation، Network Mutation یا Side Effect خارجی حل‌نشده ندارند.
</p>

<p>
در این مرحله همان Dependency Closure به ترتیب تاریخی Cellها، اما داخل یک Namespace مستقل اجرا می‌شود.
این Namespace به State اصلی Notebook جدید وابسته نیست و فقط چند Dependency فنی عمومی مانند
<code>Path</code>، <code>json</code>، <code>hashlib</code> و <code>subprocess</code> به آن داده می‌شود.
</p>

<p>
هدف این Dry-Run این است که قبل از ساخت Compact Runtime Seed ثابت کنیم Closure استخراج‌شده
واقعاً از صفر قابل اجرا است و Hidden Session Dependency از Notebook قبلی ندارد.
</p>

<h4>معیار PASS</h4>

<ul>
<li>تمام 88 Provider Cell Slice به ترتیب اجرا شوند.</li>
<li>هیچ NameError یا Hidden Session Dependency رخ ندهد.</li>
<li><code>cell138_target_values</code> در Namespace ایزوله ساخته شود.</li>
<li>Final Target دقیقاً شامل 11 Runtime Object مورد انتظار باشد.</li>
<li>Population نهایی 23 Pilot داشته باشد.</li>
<li>Pilot قدیمی Active نباشد و Replacement دقیقاً یک‌بار Active باشد.</li>
<li>Remaining Queue برابر 22 و Next Pilot همان Replacement باشد.</li>
<li>Full 572 Generation همچنان Blocked باقی بماند.</li>
<li>هیچ فایل یا Git State تغییر نکند.</li>
</ul>

</div>

In [9]:
# Code Cell 09

cell09_replay_last_cell = 140

cell09_expected_target_names = [
    "pilot_selection_rows",
    "question_drafting_pilot_packets",
    "question_drafting_pilot_by_id",
    "cell95_pilot_ids_in_runtime_order",
    "cell95_remaining_pilot_ids",
    "cell95_remaining_pilot_queue",
    "cell95_remaining_pilot_queue_sha256",
    "cell95_next_pilot_queue_entry",
    "cell95_next_pilot_id",
    "cell95_pilot_workflow_state",
    "cell95_pilot_workflow_state_sha256",
]

cell09_failures = []


# ============================================================
# 1) Helpers
# ============================================================

def cell09_call_name(node):

    if isinstance(
        node,
        ast.Name,
    ):
        return node.id

    if isinstance(
        node,
        ast.Attribute,
    ):

        parent = (
            cell09_call_name(
                node.value
            )
        )

        return (
            f"{parent}.{node.attr}"
            if parent
            else node.attr
        )

    return None


def cell09_literal_command(node):

    if not isinstance(
        node,
        (
            ast.List,
            ast.Tuple,
        ),
    ):
        return None

    command = []

    for item in node.elts:

        if not (
            isinstance(
                item,
                ast.Constant,
            )
            and
            isinstance(
                item.value,
                str,
            )
        ):
            return None

        command.append(
            item.value
        )

    return command


def cell09_find_pilot_ids(value):

    found = []

    if isinstance(
        value,
        str,
    ):

        if value.startswith(
            "question-pilot:"
        ):
            found.append(
                value
            )

    elif isinstance(
        value,
        dict,
    ):

        for nested in (
            value.values()
        ):
            found.extend(
                cell09_find_pilot_ids(
                    nested
                )
            )

    elif isinstance(
        value,
        (
            list,
            tuple,
        ),
    ):

        for nested in value:
            found.extend(
                cell09_find_pilot_ids(
                    nested
                )
            )

    return found


def cell09_stable_sha(value):

    return hashlib.sha256(
        json.dumps(
            value,
            ensure_ascii=False,
            sort_keys=True,
            separators=(
                ",",
                ":",
            ),
            default=str,
        ).encode(
            "utf-8"
        )
    ).hexdigest()


def cell09_safe_len(value):

    try:
        return len(
            value
        )

    except Exception:
        return None


# ============================================================
# 2) Build exact source-order replay plan
# ============================================================

cell09_execution_plan = []


for cell_number in range(
    1,
    cell09_replay_last_cell + 1,
):

    record = (
        cell04_code_cell_index.get(
            cell_number
        )
    )

    if not isinstance(
        record,
        dict,
    ):

        cell09_failures.append({
            "reason":
                "MISSING_PRIOR_CODE_CELL",

            "cell":
                cell_number,
        })

        continue


    source = (
        record.get(
            "source"
        )
    )

    if not isinstance(
        source,
        str,
    ):

        cell09_failures.append({
            "reason":
                "INVALID_PRIOR_CODE_CELL_SOURCE",

            "cell":
                cell_number,
        })

        continue


    try:

        tree = ast.parse(
            source
        )

    except SyntaxError as exc:

        cell09_failures.append({
            "reason":
                "PRIOR_CODE_CELL_PARSE_FAILED",

            "cell":
                cell_number,

            "error":
                repr(
                    exc
                ),
        })

        continue


    virtual_filename = (
        f"<prior-nb12-code-cell-{cell_number}>"
    )


    cell09_execution_plan.append({
        "cell":
            cell_number,

        "source":
            source,

        "tree":
            tree,

        "filename":
            virtual_filename,
    })


cell09_plan_valid = all([
    len(
        cell09_failures
    )
    ==
    0,

    len(
        cell09_execution_plan
    )
    ==
    cell09_replay_last_cell,
])


# ============================================================
# 3) Safety audit for full replay prefix
# ============================================================

cell09_blocking_findings = []

cell09_read_only_git_findings = []


cell09_blocked_methods = {
    "write_text",
    "write_bytes",
    "unlink",
    "rmdir",
    "mkdir",
    "touch",
    "rename",
    "to_csv",
    "to_json",
    "to_parquet",
    "to_excel",
    "to_pickle",
}


cell09_network_calls = {
    "requests.get",
    "requests.post",
    "requests.put",
    "requests.patch",
    "requests.delete",
    "httpx.get",
    "httpx.post",
    "httpx.put",
    "httpx.patch",
    "httpx.delete",
    "urllib.request.urlopen",
}


cell09_subprocess_calls = {
    "subprocess.run",
    "subprocess.call",
    "subprocess.check_call",
    "subprocess.check_output",
    "subprocess.Popen",
}


cell09_mutating_git_commands = {
    "add",
    "commit",
    "push",
    "pull",
    "merge",
    "rebase",
    "reset",
    "checkout",
    "switch",
    "tag",
    "stash",
    "clean",
    "rm",
    "mv",
    "restore",
    "cherry-pick",
    "revert",
}


for item in (
    cell09_execution_plan
):

    cell_number = (
        item[
            "cell"
        ]
    )

    tree = (
        item[
            "tree"
        ]
    )


    for node in ast.walk(
        tree
    ):

        if not isinstance(
            node,
            ast.Call,
        ):
            continue


        call_name = (
            cell09_call_name(
                node.func
            )
        )


        # ----------------------------------------------------
        # File writes through open()
        # ----------------------------------------------------

        if (
            call_name
            ==
            "open"
        ):

            mode = None


            if (
                len(
                    node.args
                )
                >=
                2
                and
                isinstance(
                    node.args[
                        1
                    ],
                    ast.Constant,
                )
                and
                isinstance(
                    node.args[
                        1
                    ].value,
                    str,
                )
            ):

                mode = (
                    node.args[
                        1
                    ].value
                )


            for keyword in (
                node.keywords
            ):

                if (
                    keyword.arg
                    ==
                    "mode"
                    and
                    isinstance(
                        keyword.value,
                        ast.Constant,
                    )
                    and
                    isinstance(
                        keyword.value.value,
                        str,
                    )
                ):

                    mode = (
                        keyword.value.value
                    )


            if (
                isinstance(
                    mode,
                    str,
                )
                and
                any(
                    marker
                    in
                    mode

                    for marker
                    in (
                        "w",
                        "a",
                        "x",
                        "+",
                    )
                )
            ):

                cell09_blocking_findings.append({
                    "cell":
                        cell_number,

                    "line":
                        getattr(
                            node,
                            "lineno",
                            None,
                        ),

                    "type":
                        "FILE_WRITE_OPEN",

                    "mode":
                        mode,
                })


        # ----------------------------------------------------
        # Known filesystem / artifact mutations
        # ----------------------------------------------------

        method_name = (
            call_name.split(
                "."
            )[-1]

            if isinstance(
                call_name,
                str,
            )

            else None
        )


        if (
            method_name
            in
            cell09_blocked_methods
        ):

            cell09_blocking_findings.append({
                "cell":
                    cell_number,

                "line":
                    getattr(
                        node,
                        "lineno",
                        None,
                    ),

                "type":
                    "FILESYSTEM_OR_ARTIFACT_MUTATION",

                "call":
                    call_name,
            })


        # ----------------------------------------------------
        # Network access
        # ----------------------------------------------------

        if (
            call_name
            in
            cell09_network_calls
        ):

            cell09_blocking_findings.append({
                "cell":
                    cell_number,

                "line":
                    getattr(
                        node,
                        "lineno",
                        None,
                    ),

                "type":
                    "NETWORK_ACCESS",

                "call":
                    call_name,
            })


        # ----------------------------------------------------
        # Subprocess / Git mutations
        # ----------------------------------------------------

        if (
            call_name
            in
            cell09_subprocess_calls
            and
            node.args
        ):

            command = (
                cell09_literal_command(
                    node.args[
                        0
                    ]
                )
            )


            if (
                command
                and
                command[
                    0
                ]
                ==
                "git"
            ):

                subcommand = (
                    command[
                        1
                    ]

                    if len(
                        command
                    )
                    >
                    1

                    else None
                )


                read_only_branch = (
                    subcommand
                    ==
                    "branch"
                    and
                    "--show-current"
                    in
                    command
                )


                if (
                    subcommand
                    in
                    cell09_mutating_git_commands
                    and
                    not read_only_branch
                ):

                    cell09_blocking_findings.append({
                        "cell":
                            cell_number,

                        "line":
                            getattr(
                                node,
                                "lineno",
                                None,
                            ),

                        "type":
                            "GIT_MUTATION",

                        "command":
                            command,
                    })

                else:

                    cell09_read_only_git_findings.append({
                        "cell":
                            cell_number,

                        "command":
                            command,
                    })


            elif (
                command
                and
                command[
                    0
                ]
                in {
                    "rm",
                    "mv",
                    "cp",
                    "curl",
                    "wget",
                }
            ):

                cell09_blocking_findings.append({
                    "cell":
                        cell_number,

                    "line":
                        getattr(
                            node,
                            "lineno",
                            None,
                        ),

                    "type":
                        "EXTERNAL_SIDE_EFFECT_COMMAND",

                    "command":
                        command,
                })


cell09_safety_gate = all([
    cell09_plan_valid,

    len(
        cell09_blocking_findings
    )
    ==
    0,
])


# ============================================================
# 4) Virtual IPython history
# ============================================================

class Cell09VirtualHistoryManager:

    def __init__(
        self,
    ):
        self.input_hist_raw = [
            ""
        ]


class Cell09IPythonProxy:

    def __init__(
        self,
        real_ipython,
    ):

        self._real_ipython = (
            real_ipython
        )

        self.history_manager = (
            Cell09VirtualHistoryManager()
        )


    def __getattr__(
        self,
        name,
    ):

        if (
            self._real_ipython
            is None
        ):
            raise AttributeError(
                name
            )

        return getattr(
            self._real_ipython,
            name,
        )


try:

    cell09_real_ipython = (
        get_ipython()
    )

except Exception:

    cell09_real_ipython = None


cell09_virtual_ipython = (
    Cell09IPythonProxy(
        cell09_real_ipython
    )
)


cell09_prior_sources_by_cell = {
    item[
        "cell"
    ]:
        item[
            "source"
        ]

    for item
    in cell09_execution_plan
}


# ============================================================
# 5) Register virtual source files for inspect.getsource()
# ============================================================

cell09_linecache = (
    __import__(
        "linecache"
    )
)


cell09_registered_virtual_files = []


for item in (
    cell09_execution_plan
):

    filename = (
        item[
            "filename"
        ]
    )

    source = (
        item[
            "source"
        ]
    )


    source_for_cache = (
        source
        if source.endswith(
            "\n"
        )
        else source + "\n"
    )


    source_lines = (
        source_for_cache.splitlines(
            keepends=True
        )
    )


    cell09_linecache.cache[
        filename
    ] = (
        len(
            source_for_cache
        ),

        None,

        source_lines,

        filename,
    )


    cell09_registered_virtual_files.append(
        filename
    )


cell09_linecache_registration_valid = all([
    len(
        cell09_registered_virtual_files
    )
    ==
    cell09_replay_last_cell,

    all(
        filename
        in
        cell09_linecache.cache

        for filename
        in cell09_registered_virtual_files
    ),
])


# ============================================================
# 6) Inspect-source preflight
# ============================================================

cell09_source_probe_namespace = {
    "__builtins__":
        __builtins__,
}


cell09_source_probe_ok = False

cell09_source_probe_error = None


cell09_probe_source = (
    "def cell09_virtual_source_probe(value):\n"
    "    return value\n"
)


cell09_probe_filename = (
    "<cell09-linecache-probe>"
)


cell09_linecache.cache[
    cell09_probe_filename
] = (
    len(
        cell09_probe_source
    ),

    None,

    cell09_probe_source.splitlines(
        keepends=True
    ),

    cell09_probe_filename,
)


try:

    exec(
        compile(
            cell09_probe_source,
            filename=(
                cell09_probe_filename
            ),
            mode="exec",
        ),
        cell09_source_probe_namespace,
        cell09_source_probe_namespace,
    )


    cell09_inspect = (
        __import__(
            "inspect"
        )
    )


    cell09_probe_function = (
        cell09_source_probe_namespace[
            "cell09_virtual_source_probe"
        ]
    )


    cell09_probe_recovered_source = (
        cell09_inspect.getsource(
            cell09_probe_function
        )
    )


    cell09_source_probe_ok = (
        "def cell09_virtual_source_probe"
        in
        cell09_probe_recovered_source
    )


except Exception as exc:

    cell09_source_probe_error = (
        repr(
            exc
        )
    )


# ============================================================
# 7) Cell 97 source marker preflight
# ============================================================

cell09_cell97_source = (
    cell09_prior_sources_by_cell.get(
        97,
        "",
    )
)


cell09_cell97_marker_valid = all([
    "# Code Cell 97"
    in
    cell09_cell97_source,

    "cell97_question_draft_record ="
    in
    cell09_cell97_source,

    "cell97_question_draft_lineage ="
    in
    cell09_cell97_source,
])


# ============================================================
# 8) Freeze external project state
# ============================================================

cell09_git_before_rc, cell09_git_before, cell09_git_before_err = (
    run_git(
        "status",
        "--short",
    )
)


cell09_prior_before = {
    "size":
        prior_notebook_path.stat().st_size,

    "sha256":
        hashlib.sha256(
            prior_notebook_path.read_bytes()
        ).hexdigest(),
}


cell09_handoff_before = {
    "size":
        handoff_path.stat().st_size,

    "sha256":
        hashlib.sha256(
            handoff_path.read_bytes()
        ).hexdigest(),
}


# ============================================================
# 9) Fresh isolated runtime namespace
# ============================================================

cell09_seed_namespace = {
    "__builtins__":
        __builtins__,

    "__name__":
        "nb12_full_ordered_linecache_replay",

    "print":
        lambda *args, **kwargs: None,

    "display":
        lambda *args, **kwargs: None,

    "get_ipython":
        lambda:
            cell09_virtual_ipython,
}


# ============================================================
# 10) Full ordered replay
# ============================================================

cell09_execution_results = []

cell09_execution_exception = None


if all([
    cell09_safety_gate,
    cell09_linecache_registration_valid,
    cell09_source_probe_ok,
]):

    for item in (
        cell09_execution_plan
    ):

        cell_number = (
            item[
                "cell"
            ]
        )

        source = (
            item[
                "source"
            ]
        )

        filename = (
            item[
                "filename"
            ]
        )


        # Emulate raw Jupyter input history.
        cell09_virtual_ipython.history_manager.input_hist_raw = (
            [
                ""
            ]
            +
            [
                cell09_prior_sources_by_cell[
                    prior_cell
                ]

                for prior_cell
                in range(
                    1,
                    cell_number + 1,
                )
            ]
        )


        # Refresh source registration immediately before execution.
        source_for_cache = (
            source
            if source.endswith(
                "\n"
            )
            else source + "\n"
        )


        cell09_linecache.cache[
            filename
        ] = (
            len(
                source_for_cache
            ),

            None,

            source_for_cache.splitlines(
                keepends=True
            ),

            filename,
        )


        try:

            compiled = compile(
                source,
                filename=filename,
                mode="exec",
            )


            exec(
                compiled,
                cell09_seed_namespace,
                cell09_seed_namespace,
            )


            cell09_execution_results.append({
                "cell":
                    cell_number,

                "status":
                    "PASS",

                "virtual_history_length":
                    len(
                        cell09_virtual_ipython
                        .history_manager
                        .input_hist_raw
                    ),
            })


        except Exception as exc:

            cell09_execution_exception = {
                "cell":
                    cell_number,

                "exception_type":
                    type(
                        exc
                    ).__name__,

                "error":
                    repr(
                        exc
                    ),

                "virtual_history_length":
                    len(
                        cell09_virtual_ipython
                        .history_manager
                        .input_hist_raw
                    ),

                "filename":
                    filename,

                "linecache_registered":
                    filename
                    in
                    cell09_linecache.cache,

                "source_preview":
                    source[
                        :3000
                    ],
            }


            cell09_execution_results.append({
                "cell":
                    cell_number,

                "status":
                    "FAIL",

                "virtual_history_length":
                    len(
                        cell09_virtual_ipython
                        .history_manager
                        .input_hist_raw
                    ),
            })


            break


cell09_execution_complete = all([
    cell09_execution_exception
    is None,

    len(
        cell09_execution_results
    )
    ==
    cell09_replay_last_cell,

    all(
        row[
            "status"
        ]
        ==
        "PASS"

        for row
        in cell09_execution_results
    ),
])


# ============================================================
# 11) Extract final runtime target
# ============================================================

cell09_target_surface_complete = all(
    name
    in
    cell09_seed_namespace

    for name
    in cell09_expected_target_names
)


cell09_target_values = {
    name:
        cell09_seed_namespace.get(
            name
        )

    for name
    in cell09_expected_target_names
}


cell09_packets = (
    cell09_target_values.get(
        "question_drafting_pilot_packets"
    )
)


cell09_pilot_index = (
    cell09_target_values.get(
        "question_drafting_pilot_by_id"
    )
)


cell09_runtime_ids = (
    cell09_target_values.get(
        "cell95_pilot_ids_in_runtime_order"
    )
)


cell09_remaining_ids = (
    cell09_target_values.get(
        "cell95_remaining_pilot_ids"
    )
)


cell09_remaining_queue = (
    cell09_target_values.get(
        "cell95_remaining_pilot_queue"
    )
)


cell09_next_queue_entry = (
    cell09_target_values.get(
        "cell95_next_pilot_queue_entry"
    )
)


cell09_next_pilot_id = (
    cell09_target_values.get(
        "cell95_next_pilot_id"
    )
)


cell09_workflow_state = (
    cell09_target_values.get(
        "cell95_pilot_workflow_state"
    )
)


# ============================================================
# 12) Final Pilot invariants
# ============================================================

cell09_population_valid = (
    isinstance(
        cell09_runtime_ids,
        list,
    )
    and
    len(
        cell09_runtime_ids
    )
    ==
    23
    and
    len(
        set(
            cell09_runtime_ids
        )
    )
    ==
    23
    and
    cell09_runtime_ids.count(
        cell07_old_pilot_id
    )
    ==
    0
    and
    cell09_runtime_ids.count(
        cell07_new_pilot_id
    )
    ==
    1
)


cell09_packets_valid = (
    isinstance(
        cell09_packets,
        list,
    )
    and
    len(
        cell09_packets
    )
    ==
    23
)


cell09_index_valid = (
    isinstance(
        cell09_pilot_index,
        dict,
    )
    and
    len(
        cell09_pilot_index
    )
    ==
    23
    and
    cell07_old_pilot_id
    not in
    cell09_pilot_index
    and
    cell07_new_pilot_id
    in
    cell09_pilot_index
)


cell09_remaining_valid = (
    isinstance(
        cell09_remaining_ids,
        list,
    )
    and
    len(
        cell09_remaining_ids
    )
    ==
    22
    and
    len(
        set(
            cell09_remaining_ids
        )
    )
    ==
    22
    and
    cell07_old_pilot_id
    not in
    cell09_remaining_ids
    and
    cell09_remaining_ids.count(
        cell07_new_pilot_id
    )
    ==
    1
    and
    bool(
        cell09_remaining_ids
    )
    and
    cell09_remaining_ids[
        0
    ]
    ==
    cell07_new_pilot_id
)


cell09_queue_ids = (
    [
        cell09_find_pilot_ids(
            row
        )

        for row
        in cell09_remaining_queue
    ]

    if isinstance(
        cell09_remaining_queue,
        list,
    )

    else []
)


cell09_queue_valid = (
    isinstance(
        cell09_remaining_queue,
        list,
    )
    and
    len(
        cell09_remaining_queue
    )
    ==
    22
    and
    bool(
        cell09_queue_ids
    )
    and
    cell07_new_pilot_id
    in
    cell09_queue_ids[
        0
    ]
    and
    not any(
        cell07_old_pilot_id
        in
        pilot_ids

        for pilot_ids
        in cell09_queue_ids
    )
)


cell09_next_valid = (
    cell09_next_pilot_id
    ==
    cell07_new_pilot_id
    and
    cell07_new_pilot_id
    in
    cell09_find_pilot_ids(
        cell09_next_queue_entry
    )
)


cell09_workflow_text = (
    json.dumps(
        cell09_workflow_state,
        ensure_ascii=False,
        sort_keys=True,
        default=str,
    )

    if cell09_workflow_state
    is not None

    else ""
)


cell09_full_572_blocked = (
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
    in
    cell09_workflow_text
)


# ============================================================
# 13) Final target fingerprint
# ============================================================

cell09_target_hashes = {}


if (
    cell09_target_surface_complete
):

    cell09_target_hashes = {
        name:
            cell09_stable_sha(
                value
            )

        for name, value
        in cell09_target_values.items()
    }


cell09_target_bundle_sha256 = (
    cell09_stable_sha(
        cell09_target_values
    )

    if cell09_target_surface_complete

    else None
)


# ============================================================
# 14) Verify external project state
# ============================================================

cell09_git_after_rc, cell09_git_after, cell09_git_after_err = (
    run_git(
        "status",
        "--short",
    )
)


cell09_prior_after = {
    "size":
        prior_notebook_path.stat().st_size,

    "sha256":
        hashlib.sha256(
            prior_notebook_path.read_bytes()
        ).hexdigest(),
}


cell09_handoff_after = {
    "size":
        handoff_path.stat().st_size,

    "sha256":
        hashlib.sha256(
            handoff_path.read_bytes()
        ).hexdigest(),
}


cell09_external_state_stable = all([
    cell09_git_before_rc
    ==
    cell09_git_after_rc,

    cell09_git_before
    ==
    cell09_git_after,

    cell09_prior_before
    ==
    cell09_prior_after,

    cell09_handoff_before
    ==
    cell09_handoff_after,
])


# ============================================================
# 15) Final gate
# ============================================================

cell09_gate = all([
    cell06_gate
    is True,

    cell07_gate
    is True,

    cell08_gate
    is True,

    cell09_plan_valid,

    cell09_safety_gate,

    cell09_linecache_registration_valid,

    cell09_source_probe_ok,

    cell09_cell97_marker_valid,

    cell09_execution_complete,

    cell09_target_surface_complete,

    cell09_population_valid,

    cell09_packets_valid,

    cell09_index_valid,

    cell09_remaining_valid,

    cell09_queue_valid,

    cell09_next_valid,

    cell09_full_572_blocked,

    cell09_external_state_stable,
])


# ============================================================
# 16) Diagnostics
# ============================================================

print(
    "=== FULL ORDERED ISOLATED NB12 REPLAY "
    "WITH VIRTUAL HISTORY + LINECACHE ==="
)


print(
    "\n=== PLAN ==="
)

print(
    "Replay Cells:",
    cell09_replay_last_cell,
)

print(
    "Plan valid:",
    cell09_plan_valid,
)


print(
    "\n=== SAFETY ==="
)

print(
    "Blocking findings:",
    len(
        cell09_blocking_findings
    ),
)

print(
    "Safety gate:",
    cell09_safety_gate,
)


print(
    "\n=== SOURCE INTROSPECTION SUPPORT ==="
)

print(
    "Virtual source files registered:",
    len(
        cell09_registered_virtual_files
    ),
)

print(
    "Linecache registration valid:",
    cell09_linecache_registration_valid,
)

print(
    "inspect.getsource probe:",
    cell09_source_probe_ok,
)

print(
    "Probe error:",
    cell09_source_probe_error,
)

print(
    "Cell 97 marker valid:",
    cell09_cell97_marker_valid,
)


print(
    "\n=== EXECUTION ==="
)

print(
    "Cells executed:",
    len(
        cell09_execution_results
    ),
)

print(
    "Execution complete:",
    cell09_execution_complete,
)

print(
    "Execution exception:",
    cell09_execution_exception,
)


if (
    cell09_execution_exception
    is not None
):

    print(
        "\nFailed Cell:",
        cell09_execution_exception[
            "cell"
        ],
    )

    print(
        "Exception:",
        cell09_execution_exception[
            "error"
        ],
    )

    print(
        "Virtual filename:",
        cell09_execution_exception[
            "filename"
        ],
    )

    print(
        "Linecache registered:",
        cell09_execution_exception[
            "linecache_registered"
        ],
    )

    print(
        "Virtual history length:",
        cell09_execution_exception[
            "virtual_history_length"
        ],
    )

    print(
        "\nSource preview:"
    )

    print(
        cell09_execution_exception[
            "source_preview"
        ]
    )


print(
    "\n=== FINAL PILOT STATE ==="
)

print(
    "Pilot count:",
    cell09_safe_len(
        cell09_runtime_ids
    ),
)

print(
    "Packet count:",
    cell09_safe_len(
        cell09_packets
    ),
)

print(
    "Index count:",
    cell09_safe_len(
        cell09_pilot_index
    ),
)

print(
    "Old Pilot active:",
    (
        cell09_runtime_ids.count(
            cell07_old_pilot_id
        )

        if isinstance(
            cell09_runtime_ids,
            list,
        )

        else None
    ),
)

print(
    "Replacement active:",
    (
        cell09_runtime_ids.count(
            cell07_new_pilot_id
        )

        if isinstance(
            cell09_runtime_ids,
            list,
        )

        else None
    ),
)


print(
    "\n=== QUEUE ==="
)

print(
    "Remaining:",
    cell09_safe_len(
        cell09_remaining_ids
    ),
)

print(
    "Queue:",
    cell09_safe_len(
        cell09_remaining_queue
    ),
)

print(
    "Next Pilot:",
    cell09_next_pilot_id,
)


print(
    "\n=== WORKFLOW ==="
)

print(
    "Full 572 blocked:",
    cell09_full_572_blocked,
)

print(
    "Target bundle SHA256:",
    cell09_target_bundle_sha256,
)


print(
    "\n=== EXTERNAL STATE ==="
)

print(
    "Git unchanged:",
    cell09_git_before
    ==
    cell09_git_after,
)

print(
    "Prior NB12 unchanged:",
    cell09_prior_before
    ==
    cell09_prior_after,
)

print(
    "Existing handoff unchanged:",
    cell09_handoff_before
    ==
    cell09_handoff_after,
)

print(
    "External state stable:",
    cell09_external_state_stable,
)


print(
    "\nMutation performed on project files:",
    False,
)


print(
    "\nCODE CELL 09 GATE:",
    (
        "PASS"
        if cell09_gate
        else "FAIL"
    ),
)


assert cell09_gate, (
    "Full ordered isolated NB12 replay with virtual history "
    "and linecache source registration failed. Repair this "
    "SAME Code Cell 09 before constructing the Compact "
    "Runtime Seed."
)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=== FULL ORDERED ISOLATED NB12 REPLAY WITH VIRTUAL HISTORY + LINECACHE ===

=== PLAN ===
Replay Cells: 140
Plan valid: True

=== SAFETY ===
Blocking findings: 0
Safety gate: True

=== SOURCE INTROSPECTION SUPPORT ===
Virtual source files registered: 140
Linecache registration valid: True
inspect.getsource probe: True
Probe error: None
Cell 97 marker valid: True

=== EXECUTION ===
Cells executed: 140
Execution complete: True
Execution exception: None

=== FINAL PILOT STATE ===
Pilot count: 23
Packet count: 23
Index count: 23
Old Pilot active: 0
Replacement active: 1

=== QUEUE ===
Remaining: 22
Queue: 22
Next Pilot: question-pilot:3005b48518a7aeb532234332df23c952

=== WORKFLOW ===
Full 572 blocked: True
Target bundle SHA256: 5512784eb833eb73004e6091722ca74a0abf503415400b73472abd6aa6cf198f

=== EXTERNAL STATE ===
Git unchanged: True
Prior NB12 unchanged: True
Existing handoff unchanged: True
External state stable: True

Mutation performed on project files: False

CODE CELL 09 GATE: PASS


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [10]:
# Code Cell 10

cell10_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if (
    globals().get(
        "cell09_gate"
    )
    is not True
):
    cell10_failures.append(
        "CELL09_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell09_seed_namespace"
    ),
    dict,
):
    cell10_failures.append(
        "CELL09_SEED_NAMESPACE_MISSING"
    )


# ============================================================
# 2) Required final runtime surface
# ============================================================

CELL10_REQUIRED_FINAL_RUNTIME_NAMES = [
    "pilot_selection_rows",
    "question_drafting_pilot_packets",
    "question_drafting_pilot_by_id",
    "cell95_pilot_ids_in_runtime_order",
    "cell95_remaining_pilot_ids",
    "cell95_remaining_pilot_queue",
    "cell95_remaining_pilot_queue_sha256",
    "cell95_next_pilot_queue_entry",
    "cell95_next_pilot_id",
    "cell95_pilot_workflow_state",
    "cell95_pilot_workflow_state_sha256",
]


cell10_missing_final_runtime_names = [
    name
    for name
    in CELL10_REQUIRED_FINAL_RUNTIME_NAMES
    if name
    not in
    cell09_seed_namespace
]


if (
    cell10_missing_final_runtime_names
):
    cell10_failures.append({
        "reason":
            "FINAL_RUNTIME_SURFACE_INCOMPLETE",

        "missing":
            cell10_missing_final_runtime_names,
    })


# ============================================================
# 3) Candidate registry-name policy
# ============================================================

CELL10_INCLUDE_TERMS = (
    "question",
    "answer",
    "citation",
    "review",
    "adjudication",
    "target_evidence",
    "golden",
    "case",
    "draft",
    "pilot",
    "queue",
    "lineage",
    "registry",
    "accepted",
    "rejected",
    "supersession",
    "replacement",
    "workflow",
)


CELL10_EXCLUDE_PREFIXES = (
    "__",
    "cell09_",
)


CELL10_EXCLUDE_TERMS = (
    "_source",
    "_source_text",
    "_source_lines",
    "_tree",
    "_ast",
    "_code",
    "_notebook",
    "_execution_plan",
    "_execution_results",
    "_before",
    "_after",
    "_window",
    "_preview",
)


# ============================================================
# 4) Canonical serialization helpers
# ============================================================

def cell10_canonical_key(
    value,
):

    if value is None:
        return {
            "__key_type__":
                "none",

            "value":
                None,
        }


    if isinstance(
        value,
        bool,
    ):
        return {
            "__key_type__":
                "bool",

            "value":
                value,
        }


    if isinstance(
        value,
        int,
    ):
        return {
            "__key_type__":
                "int",

            "value":
                value,
        }


    if isinstance(
        value,
        float,
    ):
        return {
            "__key_type__":
                "float",

            "value":
                repr(
                    value
                ),
        }


    if isinstance(
        value,
        str,
    ):
        return {
            "__key_type__":
                "str",

            "value":
                value,
        }


    if isinstance(
        value,
        tuple,
    ):
        return {
            "__key_type__":
                "tuple",

            "value":
                [
                    cell10_canonicalize(
                        item
                    )
                    for item
                    in value
                ],
        }


    return {
        "__key_type__":
            type(
                value
            ).__name__,

        "value":
            repr(
                value
            ),
    }


def cell10_canonicalize(
    value,
    _seen=None,
):

    if _seen is None:
        _seen = set()


    if value is None:
        return None


    if isinstance(
        value,
        (
            bool,
            int,
            str,
        ),
    ):
        return value


    if isinstance(
        value,
        float,
    ):
        return {
            "__float__":
                repr(
                    value
                )
        }


    if isinstance(
        value,
        bytes,
    ):
        return {
            "__bytes_hex__":
                value.hex()
        }


    if isinstance(
        value,
        Path,
    ):
        return {
            "__path__":
                str(
                    value
                )
        }


    track_identity = isinstance(
        value,
        (
            dict,
            list,
            tuple,
            set,
            frozenset,
        ),
    )


    object_id = (
        id(
            value
        )
        if track_identity
        else None
    )


    if (
        track_identity
        and
        object_id
        in
        _seen
    ):
        return {
            "__cycle__":
                True,

            "__type__":
                type(
                    value
                ).__name__,
        }


    if track_identity:
        _seen.add(
            object_id
        )


    try:

        if isinstance(
            value,
            dict,
        ):

            items = []

            for key, nested in (
                value.items()
            ):

                canonical_key = (
                    cell10_canonical_key(
                        key
                    )
                )

                canonical_value = (
                    cell10_canonicalize(
                        nested,
                        _seen,
                    )
                )

                items.append({
                    "key":
                        canonical_key,

                    "value":
                        canonical_value,
                })


            items.sort(
                key=lambda row:
                    json.dumps(
                        row[
                            "key"
                        ],
                        ensure_ascii=False,
                        sort_keys=True,
                        separators=(
                            ",",
                            ":",
                        ),
                    )
            )


            return {
                "__dict_items__":
                    items
            }


        if isinstance(
            value,
            list,
        ):

            return {
                "__list__":
                    [
                        cell10_canonicalize(
                            item,
                            _seen,
                        )
                        for item
                        in value
                    ]
            }


        if isinstance(
            value,
            tuple,
        ):

            return {
                "__tuple__":
                    [
                        cell10_canonicalize(
                            item,
                            _seen,
                        )
                        for item
                        in value
                    ]
            }


        if isinstance(
            value,
            (
                set,
                frozenset,
            ),
        ):

            canonical_items = [
                cell10_canonicalize(
                    item,
                    _seen,
                )
                for item
                in value
            ]


            canonical_items.sort(
                key=lambda item:
                    json.dumps(
                        item,
                        ensure_ascii=False,
                        sort_keys=True,
                        separators=(
                            ",",
                            ":",
                        ),
                    )
            )


            return {
                (
                    "__frozenset__"
                    if isinstance(
                        value,
                        frozenset,
                    )
                    else "__set__"
                ):
                    canonical_items
            }


        if callable(
            value
        ):

            return {
                "__callable__":
                    getattr(
                        value,
                        "__qualname__",
                        getattr(
                            value,
                            "__name__",
                            type(
                                value
                            ).__name__,
                        ),
                    ),

                "__module__":
                    getattr(
                        value,
                        "__module__",
                        None,
                    ),
            }


        return {
            "__type__":
                type(
                    value
                ).__name__,

            "__repr__":
                repr(
                    value
                ),
        }


    finally:

        if track_identity:
            _seen.discard(
                object_id
            )


def cell10_canonical_json(
    value,
):

    normalized = (
        cell10_canonicalize(
            value
        )
    )


    return json.dumps(
        normalized,
        ensure_ascii=False,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    )


def cell10_stable_sha(
    value,
):

    return hashlib.sha256(
        cell10_canonical_json(
            value
        ).encode(
            "utf-8"
        )
    ).hexdigest()


def cell10_count(
    value,
):

    if isinstance(
        value,
        (
            list,
            tuple,
            dict,
            set,
            frozenset,
        ),
    ):
        return len(
            value
        )

    return None


def cell10_serialized_size(
    value,
):

    try:

        return len(
            cell10_canonical_json(
                value
            ).encode(
                "utf-8"
            )
        )

    except Exception:
        return None


def cell10_contains_pilot_id(
    value,
    pilot_id,
):

    try:

        text = (
            cell10_canonical_json(
                value
            )
        )

    except Exception:

        text = repr(
            value
        )


    return (
        pilot_id
        in
        text
    )


# ============================================================
# 5) Canonicalization self-test
# ============================================================

cell10_tuple_key_probe = {
    (
        "website",
        "direct_author_long",
    ):
        {
            "status":
                "READY"
        },

    (
        "book",
        1,
    ):
        [
            "x",
            "y",
        ],
}


cell10_tuple_key_probe_sha = (
    cell10_stable_sha(
        cell10_tuple_key_probe
    )
)


cell10_tuple_key_probe_valid = (
    isinstance(
        cell10_tuple_key_probe_sha,
        str,
    )
    and
    len(
        cell10_tuple_key_probe_sha
    )
    ==
    64
)


if not (
    cell10_tuple_key_probe_valid
):

    cell10_failures.append(
        "TUPLE_KEY_CANONICALIZATION_FAILED"
    )


# ============================================================
# 6) Inventory candidate continuation registries
# ============================================================

cell10_candidate_rows = []


for name, value in sorted(
    cell09_seed_namespace.items()
):

    if any(
        name.startswith(
            prefix
        )
        for prefix
        in CELL10_EXCLUDE_PREFIXES
    ):
        continue


    lowered = (
        name.lower()
    )


    if not any(
        term
        in
        lowered
        for term
        in CELL10_INCLUDE_TERMS
    ):
        continue


    if any(
        term
        in
        lowered
        for term
        in CELL10_EXCLUDE_TERMS
    ):
        continue


    serialized_size = (
        cell10_serialized_size(
            value
        )
    )


    try:

        value_sha = (
            cell10_stable_sha(
                value
            )
        )

    except Exception as exc:

        cell10_failures.append({
            "reason":
                "CANDIDATE_FINGERPRINT_FAILED",

            "name":
                name,

            "type":
                type(
                    value
                ).__name__,

            "error":
                repr(
                    exc
                ),
        })

        continue


    cell10_candidate_rows.append({
        "name":
            name,

        "type":
            type(
                value
            ).__name__,

        "count":
            cell10_count(
                value
            ),

        "serialized_bytes":
            serialized_size,

        "sha256":
            value_sha,

        "contains_old_pilot":
            cell10_contains_pilot_id(
                value,
                cell07_old_pilot_id,
            ),

        "contains_replacement_pilot":
            cell10_contains_pilot_id(
                value,
                cell07_new_pilot_id,
            ),

        "is_final_runtime_object":
            name
            in
            CELL10_REQUIRED_FINAL_RUNTIME_NAMES,

        "is_callable":
            callable(
                value
            ),
    })


# ============================================================
# 7) Keep data registries separate from helper callables
# ============================================================

cell10_data_candidates = [
    row
    for row
    in cell10_candidate_rows
    if not row[
        "is_callable"
    ]
]


cell10_callable_candidates = [
    row
    for row
    in cell10_candidate_rows
    if row[
        "is_callable"
    ]
]


# ============================================================
# 8) Split compact vs large candidates
# ============================================================

CELL10_LARGE_OBJECT_THRESHOLD_BYTES = (
    2_000_000
)


cell10_compact_candidates = [
    row
    for row
    in cell10_data_candidates
    if (
        row[
            "serialized_bytes"
        ]
        is None
        or
        row[
            "serialized_bytes"
        ]
        <=
        CELL10_LARGE_OBJECT_THRESHOLD_BYTES
    )
]


cell10_large_candidates = [
    row
    for row
    in cell10_data_candidates
    if (
        isinstance(
            row[
                "serialized_bytes"
            ],
            int,
        )
        and
        row[
            "serialized_bytes"
        ]
        >
        CELL10_LARGE_OBJECT_THRESHOLD_BYTES
    )
]


# ============================================================
# 9) Workflow-family classification
# ============================================================

cell10_family_map = {
    "question":
        [],

    "expected_answer":
        [],

    "citation":
        [],

    "review_adjudication":
        [],

    "target_evidence":
        [],

    "golden_case":
        [],

    "pilot_queue_workflow":
        [],

    "lineage_replacement":
        [],
}


for row in (
    cell10_compact_candidates
):

    name = (
        row[
            "name"
        ].lower()
    )


    if (
        "question"
        in
        name
    ):
        cell10_family_map[
            "question"
        ].append(
            row[
                "name"
            ]
        )


    if (
        "answer"
        in
        name
    ):
        cell10_family_map[
            "expected_answer"
        ].append(
            row[
                "name"
            ]
        )


    if (
        "citation"
        in
        name
    ):
        cell10_family_map[
            "citation"
        ].append(
            row[
                "name"
            ]
        )


    if (
        "review"
        in
        name
        or
        "adjudication"
        in
        name
    ):
        cell10_family_map[
            "review_adjudication"
        ].append(
            row[
                "name"
            ]
        )


    if (
        "target_evidence"
        in
        name
    ):
        cell10_family_map[
            "target_evidence"
        ].append(
            row[
                "name"
            ]
        )


    if (
        "golden"
        in
        name
        or
        "case"
        in
        name
    ):
        cell10_family_map[
            "golden_case"
        ].append(
            row[
                "name"
            ]
        )


    if (
        "pilot"
        in
        name
        or
        "queue"
        in
        name
        or
        "workflow"
        in
        name
    ):
        cell10_family_map[
            "pilot_queue_workflow"
        ].append(
            row[
                "name"
            ]
        )


    if (
        "lineage"
        in
        name
        or
        "replacement"
        in
        name
        or
        "rejected"
        in
        name
        or
        "supersession"
        in
        name
    ):
        cell10_family_map[
            "lineage_replacement"
        ].append(
            row[
                "name"
            ]
        )


for family in (
    cell10_family_map
):

    cell10_family_map[
        family
    ] = sorted(
        set(
            cell10_family_map[
                family
            ]
        )
    )


# ============================================================
# 10) Historical / replacement coverage
# ============================================================

cell10_old_pilot_registry_names = sorted(
    row[
        "name"
    ]
    for row
    in cell10_compact_candidates
    if row[
        "contains_old_pilot"
    ]
)


cell10_replacement_registry_names = sorted(
    row[
        "name"
    ]
    for row
    in cell10_compact_candidates
    if row[
        "contains_replacement_pilot"
    ]
)


cell10_old_history_retained = (
    len(
        cell10_old_pilot_registry_names
    )
    >
    0
)


cell10_replacement_state_retained = (
    len(
        cell10_replacement_registry_names
    )
    >
    0
)


# ============================================================
# 11) Final runtime fingerprints
# ============================================================

cell10_final_runtime_fingerprints = {}


for name in (
    CELL10_REQUIRED_FINAL_RUNTIME_NAMES
):

    if (
        name
        not in
        cell09_seed_namespace
    ):
        continue


    try:

        cell10_final_runtime_fingerprints[
            name
        ] = (
            cell10_stable_sha(
                cell09_seed_namespace[
                    name
                ]
            )
        )

    except Exception as exc:

        cell10_failures.append({
            "reason":
                "FINAL_RUNTIME_FINGERPRINT_FAILED",

            "name":
                name,

            "error":
                repr(
                    exc
                ),
        })


cell10_final_runtime_surface_valid = (
    len(
        cell10_final_runtime_fingerprints
    )
    ==
    len(
        CELL10_REQUIRED_FINAL_RUNTIME_NAMES
    )
)


# ============================================================
# 12) Inventory fingerprint
# ============================================================

cell10_inventory_summary = {
    "final_runtime_names":
        CELL10_REQUIRED_FINAL_RUNTIME_NAMES,

    "final_runtime_fingerprints":
        cell10_final_runtime_fingerprints,

    "candidate_names":
        [
            row[
                "name"
            ]
            for row
            in cell10_compact_candidates
        ],

    "large_candidate_names":
        [
            row[
                "name"
            ]
            for row
            in cell10_large_candidates
        ],

    "family_map":
        cell10_family_map,

    "old_pilot_registry_names":
        cell10_old_pilot_registry_names,

    "replacement_registry_names":
        cell10_replacement_registry_names,
}


cell10_inventory_sha256 = (
    cell10_stable_sha(
        cell10_inventory_summary
    )
)


# ============================================================
# 13) Inventory gate
# ============================================================

cell10_gate = all([
    len(
        cell10_failures
    )
    ==
    0,

    cell10_tuple_key_probe_valid,

    cell10_final_runtime_surface_valid,

    len(
        cell10_candidate_rows
    )
    >
    0,

    len(
        cell10_data_candidates
    )
    >
    0,

    len(
        cell10_compact_candidates
    )
    >
    0,

    cell10_old_history_retained,

    cell10_replacement_state_retained,

    cell09_full_572_blocked
    is True,
])


# ============================================================
# 14) Diagnostics
# ============================================================

print(
    "=== COMPACT RUNTIME CONTINUATION REGISTRY INVENTORY ==="
)


print(
    "\n=== CANONICALIZATION ==="
)

print(
    "Tuple-key probe:",
    cell10_tuple_key_probe_valid,
)

print(
    "Tuple-key probe SHA256:",
    cell10_tuple_key_probe_sha,
)


print(
    "\n=== FINAL RUNTIME SURFACE ==="
)

print(
    "Required objects:",
    len(
        CELL10_REQUIRED_FINAL_RUNTIME_NAMES
    ),
)

print(
    "Fingerprinted objects:",
    len(
        cell10_final_runtime_fingerprints
    ),
)

print(
    "Surface valid:",
    cell10_final_runtime_surface_valid,
)


print(
    "\n=== CANDIDATE INVENTORY ==="
)

print(
    "All matched objects:",
    len(
        cell10_candidate_rows
    ),
)

print(
    "Data candidates:",
    len(
        cell10_data_candidates
    ),
)

print(
    "Callable helpers excluded:",
    len(
        cell10_callable_candidates
    ),
)

print(
    "Compact candidates:",
    len(
        cell10_compact_candidates
    ),
)

print(
    "Large candidates:",
    len(
        cell10_large_candidates
    ),
)


print(
    "\n=== WORKFLOW FAMILIES ==="
)

for family, names in (
    cell10_family_map.items()
):

    print(
        "\n",
        family,
        ":",
        len(
            names
        ),
    )

    for name in (
        names[
            :40
        ]
    ):

        print(
            " -",
            name
        )


print(
    "\n=== OLD PILOT HISTORICAL COVERAGE ==="
)

print(
    "Registry count:",
    len(
        cell10_old_pilot_registry_names
    ),
)

for name in (
    cell10_old_pilot_registry_names[
        :40
    ]
):

    print(
        " -",
        name
    )


print(
    "\n=== REPLACEMENT PILOT COVERAGE ==="
)

print(
    "Registry count:",
    len(
        cell10_replacement_registry_names
    ),
)

for name in (
    cell10_replacement_registry_names[
        :40
    ]
):

    print(
        " -",
        name
    )


print(
    "\n=== LARGE CANDIDATES EXCLUDED FROM COMPACT SEED ==="
)

for row in (
    cell10_large_candidates
):

    print({
        "name":
            row[
                "name"
            ],

        "type":
            row[
                "type"
            ],

        "count":
            row[
                "count"
            ],

        "serialized_bytes":
            row[
                "serialized_bytes"
            ],
    })


print(
    "\n=== INVENTORY ==="
)

print(
    "Inventory SHA256:",
    cell10_inventory_sha256,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell10_failures
)


print(
    "\nFull 572 blocked:",
    cell09_full_572_blocked,
)

print(
    "Mutation performed:",
    False,
)


print(
    "\nCODE CELL 10 GATE:",
    (
        "PASS"
        if cell10_gate
        else "FAIL"
    ),
)


assert cell10_gate, (
    "Compact continuation registry inventory failed. "
    "Repair this SAME Code Cell 10 before selecting the "
    "Compact Runtime Seed."
)

=== COMPACT RUNTIME CONTINUATION REGISTRY INVENTORY ===

=== CANONICALIZATION ===
Tuple-key probe: True
Tuple-key probe SHA256: d65d19b0488522ad25a77959860b00578311651d7d861e78270ac269510679fb

=== FINAL RUNTIME SURFACE ===
Required objects: 11
Fingerprinted objects: 11
Surface valid: True

=== CANDIDATE INVENTORY ===
All matched objects: 1120
Data candidates: 1102
Callable helpers excluded: 18
Compact candidates: 1075
Large candidates: 27

=== WORKFLOW FAMILIES ===

 question : 139
 - CELL61_QUESTION_GENERATION_STATUS
 - CELL66_REQUIRED_PREQUESTION_REBUILDS
 - CELL68_PREQUESTION_OBJECTS
 - CELL69_REQUIRED_PREQUESTION_REBUILDS_V2
 - CELL76_REAL_QUESTION_TEXT_FIELD_NAMES
 - CELL79_QUESTION_ID_SEARCH_TERMS
 - CELL79_QUESTION_OUTPUT_FIELD_CANDIDATES
 - CELL79_QUESTION_SCHEMA_SEARCH_TERMS
 - CELL82_QUESTION_TEXT
 - CELL86_REVISED_QUESTION_TEXT
 - CELL97_QUESTION_TEXT
 - EXERCISE_V2_QUESTION_CONTEXT_POLICY
 - PILOT_QUESTION_AUTHORING_MODE_V1
 - PILOT_QUESTION_DRAFTING_CONTRACT_V1_DRAFT
 - P

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 11 — ساخت Compact Runtime Seed برای ادامه‌ی 22 Pilot

Cell 10 نشان داد که Final Runtime معتبر است، اما Namespace بازسازی‌شده شامل تعداد زیادی State موقت و چند Registry بسیار بزرگ است.

در این مرحله یک Runtime Seed فشرده در حافظه ساخته می‌شود که فقط شامل موارد لازم برای ادامه‌ی Pilot Workflow است:

- 11 شیء Final Runtime پذیرفته‌شده؛
- قراردادهای اصلی Question Drafting و Human Validation؛
- تاریخچه‌ی Pilot تکمیل‌شده‌ی اول؛
- lineage و audit لازم برای Pilot ردشده و Replacement فعال‌شده؛
- فقط Golden Case، Task Draft و Target Evidence مربوط به 23 Pilot فعال، نه Registry کامل 572/724 رکوردی؛
- Source توابع ضروری، بدون نگه‌داشتن خود Function Objectهایی که به Namespace بزرگ NB12 وابسته‌اند؛
- fingerprint و metadata لازم برای کنترل byte-stability و ادامه‌ی Workflow.

این Cell هیچ Pilot جدیدی را پردازش نمی‌کند، Queue را جلو نمی‌برد، Human Decision ایجاد نمی‌کند و Full 572 Generation را باز نمی‌کند.

هیچ فایل، Artifact، Commit، Push یا PR ایجاد نمی‌شود.

</div>

In [11]:
# Code Cell 11

cell11_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if (
    globals().get(
        "cell10_gate"
    )
    is not True
):
    cell11_failures.append(
        "CELL10_NOT_PASS"
    )


if (
    globals().get(
        "cell09_gate"
    )
    is not True
):
    cell11_failures.append(
        "CELL09_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell09_seed_namespace"
    ),
    dict,
):
    cell11_failures.append(
        "CELL09_SEED_NAMESPACE_MISSING"
    )


# ============================================================
# 2) Helpers
# ============================================================

def cell11_collect_prefixed_ids(
    value,
    result=None,
    seen=None,
):

    if result is None:
        result = {
            "pilot":
                set(),

            "golden_case":
                set(),

            "task_draft":
                set(),

            "target_evidence":
                set(),
        }


    if seen is None:
        seen = set()


    if isinstance(
        value,
        str,
    ):

        if value.startswith(
            "question-pilot:"
        ):
            result[
                "pilot"
            ].add(
                value
            )

        elif value.startswith(
            "golden-case:"
        ):
            result[
                "golden_case"
            ].add(
                value
            )

        elif value.startswith(
            "task-draft:"
        ):
            result[
                "task_draft"
            ].add(
                value
            )

        elif value.startswith(
            "target-evidence:"
        ):
            result[
                "target_evidence"
            ].add(
                value
            )

        return result


    if not isinstance(
        value,
        (
            dict,
            list,
            tuple,
            set,
            frozenset,
        ),
    ):
        return result


    object_id = id(
        value
    )


    if object_id in seen:
        return result


    seen.add(
        object_id
    )


    try:

        if isinstance(
            value,
            dict,
        ):

            for key, nested in (
                value.items()
            ):

                cell11_collect_prefixed_ids(
                    key,
                    result,
                    seen,
                )

                cell11_collect_prefixed_ids(
                    nested,
                    result,
                    seen,
                )


        else:

            for nested in value:

                cell11_collect_prefixed_ids(
                    nested,
                    result,
                    seen,
                )


    finally:

        seen.discard(
            object_id
        )


    return result


def cell11_subset_list_by_field(
    records,
    field,
    allowed_ids,
):

    if not isinstance(
        records,
        list,
    ):
        return []

    return [
        copy.deepcopy(
            record
        )

        for record
        in records

        if (
            isinstance(
                record,
                dict,
            )
            and
            record.get(
                field
            )
            in
            allowed_ids
        )
    ]


def cell11_subset_dict_by_keys(
    mapping,
    allowed_ids,
):

    if not isinstance(
        mapping,
        dict,
    ):
        return {}

    return {
        key:
            copy.deepcopy(
                mapping[
                    key
                ]
            )

        for key
        in allowed_ids

        if key
        in
        mapping
    }


def cell11_contains_callable(
    value,
    seen=None,
):

    if callable(
        value
    ):
        return True


    if seen is None:
        seen = set()


    if not isinstance(
        value,
        (
            dict,
            list,
            tuple,
            set,
            frozenset,
        ),
    ):
        return False


    object_id = id(
        value
    )


    if object_id in seen:
        return False


    seen.add(
        object_id
    )


    try:

        if isinstance(
            value,
            dict,
        ):

            return any(
                cell11_contains_callable(
                    key,
                    seen,
                )
                or
                cell11_contains_callable(
                    nested,
                    seen,
                )

                for key, nested
                in value.items()
            )


        return any(
            cell11_contains_callable(
                nested,
                seen,
            )

            for nested
            in value
        )


    finally:

        seen.discard(
            object_id
        )


# ============================================================
# 3) Freeze exact 11-object Final Runtime
# ============================================================

CELL11_FINAL_RUNTIME_NAMES = list(
    CELL10_REQUIRED_FINAL_RUNTIME_NAMES
)


cell11_final_runtime = {
    name:
        copy.deepcopy(
            cell09_seed_namespace[
                name
            ]
        )

    for name
    in CELL11_FINAL_RUNTIME_NAMES

    if name
    in
    cell09_seed_namespace
}


cell11_final_runtime_complete = (
    set(
        cell11_final_runtime
    )
    ==
    set(
        CELL11_FINAL_RUNTIME_NAMES
    )
)


# ============================================================
# 4) Resolve active / completed Pilot identities
# ============================================================

cell11_runtime_ids = (
    cell11_final_runtime.get(
        "cell95_pilot_ids_in_runtime_order"
    )
)


cell11_remaining_ids = (
    cell11_final_runtime.get(
        "cell95_remaining_pilot_ids"
    )
)


cell11_active_pilot_ids = (
    list(
        cell11_runtime_ids
    )

    if isinstance(
        cell11_runtime_ids,
        list,
    )

    else []
)


cell11_completed_pilot_ids = [
    pilot_id

    for pilot_id
    in cell11_active_pilot_ids

    if (
        isinstance(
            cell11_remaining_ids,
            list,
        )
        and
        pilot_id
        not in
        cell11_remaining_ids
    )
]


cell11_completed_pilot_identity_valid = (
    len(
        cell11_completed_pilot_ids
    )
    ==
    1
)


cell11_completed_pilot_id = (
    cell11_completed_pilot_ids[
        0
    ]

    if cell11_completed_pilot_identity_valid

    else None
)


# ============================================================
# 5) Resolve IDs belonging to active 23-Pilot surface
# ============================================================

cell11_active_id_sets = (
    cell11_collect_prefixed_ids(
        cell11_final_runtime
    )
)


cell11_active_pilot_id_set = (
    cell11_active_id_sets[
        "pilot"
    ]
)


cell11_active_case_ids = (
    cell11_active_id_sets[
        "golden_case"
    ]
)


cell11_active_task_ids = (
    cell11_active_id_sets[
        "task_draft"
    ]
)


cell11_active_target_ids = (
    cell11_active_id_sets[
        "target_evidence"
    ]
)


cell11_active_identity_surface_valid = all([
    len(
        cell11_active_pilot_id_set
    )
    ==
    23,

    len(
        cell11_active_case_ids
    )
    ==
    23,

    len(
        cell11_active_task_ids
    )
    ==
    23,

    len(
        cell11_active_target_ids
    )
    ==
    23,
])


# ============================================================
# 6) Core frozen contracts + primitive constants
# ============================================================

CELL11_REQUIRED_CONTRACT_NAMES = [
    "PILOT_QA_WORKFLOW_CONTRACT_V1",
    "QUESTION_DRAFT_RECORD_SCHEMA_V1",
    "QUESTION_DRAFT_ID_CONTRACT_V1",
    "QUESTION_DRAFT_ID_METHOD_VERSION",
    "QUESTION_DRAFT_ACCEPTANCE_RUBRIC_V1",
    "PILOT_QUESTION_AUTHORING_MODE_V1",
    "QUESTION_USER_FACING_SOURCE_OPACITY_RULE_V1",
    "TARGET_EVIDENCE_SCHEMA_VERSION",
]


CELL11_OPTIONAL_CONTRACT_NAMES = [
    "PILOT_QUESTION_DRAFTING_CONTRACT_V1_DRAFT",
    "PILOT_QUESTION_DRAFTING_FOUR_COMPONENT_CONTRACT_V1",
    "QUESTION_ARCHETYPES_V1",
    "QUESTION_ARCHETYPE_BY_STRATUM",
    "QUESTION_CONSTRUCTION_GUARDRAILS_V1_DRAFT",
    "QUESTION_CONSTRUCTION_METHOD_V1_DRAFT",
    "STRATUM_QUESTION_CONSTRAINTS_V1_DRAFT",
    "TARGET_EVIDENCE_REGISTRY_V1_DRAFT",
    "TARGET_EVIDENCE_SELECTION_READINESS_V1_DRAFT",
    "GOLDEN_CASE_SCHEMA_VERSION",
    "GOLDEN_CASE_RECORD_CONTRACT_V1_DRAFT",
    "GOLDEN_CASE_REGISTRY_V1_DRAFT",
    "TASK_DRAFT_SCHEMA_V1_DRAFT",
]


cell11_missing_required_contracts = [
    name

    for name
    in CELL11_REQUIRED_CONTRACT_NAMES

    if name
    not in
    cell09_seed_namespace
]


if (
    cell11_missing_required_contracts
):

    cell11_failures.append({
        "reason":
            "REQUIRED_CONTRACTS_MISSING",

        "names":
            cell11_missing_required_contracts,
    })


cell11_contracts = {
    name:
        copy.deepcopy(
            cell09_seed_namespace[
                name
            ]
        )

    for name
    in (
        CELL11_REQUIRED_CONTRACT_NAMES
        +
        CELL11_OPTIONAL_CONTRACT_NAMES
    )

    if name
    in
    cell09_seed_namespace
}


# ============================================================
# 7) Preserve accepted Pilot #1 history
# ============================================================

cell11_completed_history_names = []


if (
    cell11_completed_pilot_id
    is not None
):

    for row in (
        cell10_compact_candidates
    ):

        name = (
            row[
                "name"
            ]
        )


        if not (
            row.get(
                "serialized_bytes"
            )
            is None
            or
            row.get(
                "serialized_bytes"
            )
            <=
            500_000
        ):
            continue


        if not (
            row.get(
                "contains_old_pilot"
            )
            is False
        ):
            continue


        if not (
            name.startswith(
                (
                    "cell82_",
                    "cell83_",
                    "cell84_",
                    "cell85_",
                    "cell86_",
                    "cell87_",
                    "cell88_",
                    "cell89_",
                    "cell90_",
                    "cell91_",
                    "cell92_",
                    "cell93_",
                    "cell94_",
                )
            )
        ):
            continue


        value = (
            cell09_seed_namespace.get(
                name
            )
        )


        ids = (
            cell11_collect_prefixed_ids(
                value
            )
        )


        if (
            cell11_completed_pilot_id
            in
            ids[
                "pilot"
            ]
        ):

            cell11_completed_history_names.append(
                name
            )


cell11_completed_history_names = sorted(
    set(
        cell11_completed_history_names
    )
)


cell11_completed_pilot_history = {
    name:
        copy.deepcopy(
            cell09_seed_namespace[
                name
            ]
        )

    for name
    in cell11_completed_history_names
}


# ============================================================
# 8) Preserve rejected Pilot / replacement audit lineage
# ============================================================

CELL11_REPLACEMENT_HISTORY_NAMES = [
    "cell100_human_adjudication_record",
    "cell109_revision_lineage_interpretation",
    "cell121_pilot_replacement_policy_inspection",
    "cell127_candidate_evidence_review",
    "cell130_replacement_packet_simulation",
    "cell132_full_replacement_transaction_simulation",
    "cell135_queue_replacement_and_supersession_simulation",
    "cell138_supersession_lineage_payload",
    "cell140_replacement_packet",
]


cell11_replacement_history = {
    name:
        copy.deepcopy(
            cell09_seed_namespace[
                name
            ]
        )

    for name
    in CELL11_REPLACEMENT_HISTORY_NAMES

    if name
    in
    cell09_seed_namespace
}


cell11_replacement_history_ids = (
    cell11_collect_prefixed_ids(
        cell11_replacement_history
    )
)


cell11_old_pilot_history_present = (
    cell07_old_pilot_id
    in
    cell11_replacement_history_ids[
        "pilot"
    ]
)


cell11_new_pilot_history_present = (
    cell07_new_pilot_id
    in
    cell11_replacement_history_ids[
        "pilot"
    ]
)


# ============================================================
# 9) Subset large Golden / Task / Target registries
# ============================================================

cell11_golden_case_subset = (
    cell11_subset_dict_by_keys(
        cell09_seed_namespace.get(
            "golden_case_records_by_id"
        ),
        cell11_active_case_ids,
    )
)


cell11_golden_case_records_subset = (
    cell11_subset_list_by_field(
        cell09_seed_namespace.get(
            "golden_case_records"
        ),
        "golden_case_id",
        cell11_active_case_ids,
    )
)


cell11_target_by_case_subset = (
    cell11_subset_dict_by_keys(
        cell09_seed_namespace.get(
            "target_evidence_by_case_id"
        ),
        cell11_active_case_ids,
    )
)


cell11_target_by_task_subset = (
    cell11_subset_dict_by_keys(
        cell09_seed_namespace.get(
            "target_evidence_by_task_draft_id"
        ),
        cell11_active_task_ids,
    )
)


cell11_target_records_subset = (
    cell11_subset_list_by_field(
        cell09_seed_namespace.get(
            "target_evidence_records"
        ),
        "target_evidence_id",
        cell11_active_target_ids,
    )
)


cell11_task_plan_subset = {}


if isinstance(
    cell09_seed_namespace.get(
        "task_draft_plans"
    ),
    dict,
):

    full_task_plans = (
        cell09_seed_namespace[
            "task_draft_plans"
        ]
    )


    for key, value in (
        full_task_plans.items()
    ):

        value_ids = (
            cell11_collect_prefixed_ids(
                value
            )
        )


        if (
            key
            in
            cell11_active_task_ids
            or
            key
            in
            cell11_active_case_ids
            or
            bool(
                value_ids[
                    "task_draft"
                ]
                &
                cell11_active_task_ids
            )
            or
            bool(
                value_ids[
                    "golden_case"
                ]
                &
                cell11_active_case_ids
            )
        ):

            cell11_task_plan_subset[
                key
            ] = copy.deepcopy(
                value
            )


cell11_task_plan_rows_subset = []


if isinstance(
    cell09_seed_namespace.get(
        "task_draft_plan_rows"
    ),
    list,
):

    for row in (
        cell09_seed_namespace[
            "task_draft_plan_rows"
        ]
    ):

        if not isinstance(
            row,
            dict,
        ):
            continue


        if (
            row.get(
                "task_draft_id"
            )
            in
            cell11_active_task_ids
            or
            row.get(
                "golden_case_id"
            )
            in
            cell11_active_case_ids
        ):

            cell11_task_plan_rows_subset.append(
                copy.deepcopy(
                    row
                )
            )


# ============================================================
# 10) Validate active support completeness
# ============================================================

cell11_golden_support_valid = (
    len(
        cell11_golden_case_subset
    )
    ==
    23
)


cell11_target_case_support_valid = (
    len(
        cell11_target_by_case_subset
    )
    ==
    23
)


cell11_target_task_support_valid = (
    len(
        cell11_target_by_task_subset
    )
    ==
    23
)


cell11_target_record_ids = {
    row.get(
        "target_evidence_id"
    )

    for row
    in cell11_target_records_subset

    if isinstance(
        row,
        dict,
    )
}


cell11_target_record_support_valid = (
    cell11_active_target_ids
    <=
    cell11_target_record_ids
)


# ============================================================
# 11) Capture complete reusable callable SOURCE closure
# ============================================================

CELL11_REQUIRED_CALLABLE_NAMES = [
    "stable_json_key",
    "target_part_identity",
    "build_target_evidence_id",
    "build_question_draft_id",
]


cell11_callable_sources = {}

cell11_callable_source_failures = []


cell11_inspect = (
    __import__(
        "inspect"
    )
)


for name in (
    CELL11_REQUIRED_CALLABLE_NAMES
):

    value = (
        cell09_seed_namespace.get(
            name
        )
    )


    if not callable(
        value
    ):

        cell11_callable_source_failures.append({
            "name":
                name,

            "reason":
                "REQUIRED_CALLABLE_MISSING",
        })

        continue


    try:

        source = (
            cell11_inspect.getsource(
                value
            )
        )


        cell11_callable_sources[
            name
        ] = {
            "source":
                source,

            "source_sha256":
                hashlib.sha256(
                    source.encode(
                        "utf-8"
                    )
                ).hexdigest(),

            "module":
                getattr(
                    value,
                    "__module__",
                    None,
                ),
        }


    except Exception as exc:

        cell11_callable_source_failures.append({
            "name":
                name,

            "reason":
                "SOURCE_RECOVERY_FAILED",

            "error":
                repr(
                    exc
                ),
        })


cell11_required_callable_sources_valid = all(
    name
    in
    cell11_callable_sources

    for name
    in CELL11_REQUIRED_CALLABLE_NAMES
)


# ============================================================
# 12) Large registry references only
# ============================================================

CELL11_LARGE_REGISTRY_REFERENCE_NAMES = [
    "golden_case_records",
    "golden_case_records_by_id",
    "target_evidence_records",
    "target_evidence_by_case_id",
    "target_evidence_by_task_draft_id",
]


cell11_large_registry_references = {}


cell11_large_candidate_by_name = {
    row[
        "name"
    ]:
        row

    for row
    in cell10_large_candidates
}


for name in (
    CELL11_LARGE_REGISTRY_REFERENCE_NAMES
):

    row = (
        cell11_large_candidate_by_name.get(
            name
        )
    )


    if row is None:
        continue


    cell11_large_registry_references[
        name
    ] = {
        "type":
            row[
                "type"
            ],

        "count":
            row[
                "count"
            ],

        "serialized_bytes":
            row[
                "serialized_bytes"
            ],

        "sha256":
            row[
                "sha256"
            ],
    }


# ============================================================
# 13) Build Compact Runtime Seed
# ============================================================

cell11_compact_runtime_seed = {
    "seed_version":
        "golden_dataset_compact_runtime_seed_v1",

    "status":
        "READY_FOR_CONTINUATION_ENGINE_VALIDATION",

    "source":
        {
            "prior_notebook":
                str(
                    prior_notebook_path
                ),

            "cell09_target_bundle_sha256":
                cell09_target_bundle_sha256,

            "cell10_inventory_sha256":
                cell10_inventory_sha256,

            "full_572_generation":
                "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION",
        },

    "final_runtime":
        cell11_final_runtime,

    "contracts":
        cell11_contracts,

    "completed_pilot":
        {
            "pilot_id":
                cell11_completed_pilot_id,

            "history":
                cell11_completed_pilot_history,
        },

    "replacement_audit":
        {
            "old_pilot_id":
                cell07_old_pilot_id,

            "replacement_pilot_id":
                cell07_new_pilot_id,

            "history":
                cell11_replacement_history,
        },

    "active_support":
        {
            "golden_case_records_by_id":
                cell11_golden_case_subset,

            "golden_case_records":
                cell11_golden_case_records_subset,

            "task_draft_plans":
                cell11_task_plan_subset,

            "task_draft_plan_rows":
                cell11_task_plan_rows_subset,

            "target_evidence_by_case_id":
                cell11_target_by_case_subset,

            "target_evidence_by_task_draft_id":
                cell11_target_by_task_subset,

            "target_evidence_records":
                cell11_target_records_subset,
        },

    "callable_sources":
        cell11_callable_sources,

    "full_registry_references":
        cell11_large_registry_references,
}


# ============================================================
# 14) Compactness + isolation checks
# ============================================================

cell11_seed_serialized_bytes = (
    cell10_serialized_size(
        cell11_compact_runtime_seed
    )
)


cell11_seed_sha256 = (
    cell10_stable_sha(
        cell11_compact_runtime_seed
    )
)


CELL11_MAX_SEED_BYTES = (
    8
    *
    1024
    *
    1024
)


cell11_seed_is_compact = (
    isinstance(
        cell11_seed_serialized_bytes,
        int,
    )
    and
    cell11_seed_serialized_bytes
    <
    CELL11_MAX_SEED_BYTES
)


cell11_seed_has_no_callable_objects = (
    not
    cell11_contains_callable(
        cell11_compact_runtime_seed
    )
)


cell11_runtime_copy_fingerprints = {
    name:
        cell10_stable_sha(
            cell11_final_runtime[
                name
            ]
        )

    for name
    in CELL11_FINAL_RUNTIME_NAMES

    if name
    in
    cell11_final_runtime
}


cell11_runtime_fingerprints_match = (
    cell11_runtime_copy_fingerprints
    ==
    cell10_final_runtime_fingerprints
)


# ============================================================
# 15) Final continuation-state invariants
# ============================================================

cell11_next_pilot_id = (
    cell11_final_runtime.get(
        "cell95_next_pilot_id"
    )
)


cell11_queue = (
    cell11_final_runtime.get(
        "cell95_remaining_pilot_queue"
    )
)


cell11_queue_ids = (
    cell11_final_runtime.get(
        "cell95_remaining_pilot_ids"
    )
)


cell11_next_replacement_valid = all([
    cell11_next_pilot_id
    ==
    cell07_new_pilot_id,

    isinstance(
        cell11_queue,
        list,
    ),

    len(
        cell11_queue
    )
    ==
    22,

    isinstance(
        cell11_queue_ids,
        list,
    ),

    len(
        cell11_queue_ids
    )
    ==
    22,

    cell11_queue_ids[
        0
    ]
    ==
    cell07_new_pilot_id,
])


cell11_old_not_active = (
    cell07_old_pilot_id
    not in
    cell11_active_pilot_ids
)


cell11_replacement_active = (
    cell07_new_pilot_id
    in
    cell11_active_pilot_ids
)


cell11_full_572_still_blocked = (
    cell11_compact_runtime_seed[
        "source"
    ][
        "full_572_generation"
    ]
    ==
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


# ============================================================
# 16) Final Gate
# ============================================================

cell11_gate = all([
    len(
        cell11_failures
    )
    ==
    0,

    cell11_final_runtime_complete,

    cell11_completed_pilot_identity_valid,

    cell11_active_identity_surface_valid,

    len(
        cell11_completed_pilot_history
    )
    >
    0,

    cell11_old_pilot_history_present,

    cell11_new_pilot_history_present,

    cell11_golden_support_valid,

    cell11_target_case_support_valid,

    cell11_target_task_support_valid,

    cell11_target_record_support_valid,

    len(
        cell11_callable_source_failures
    )
    ==
    0,

    cell11_required_callable_sources_valid,

    cell11_seed_is_compact,

    cell11_seed_has_no_callable_objects,

    cell11_runtime_fingerprints_match,

    cell11_next_replacement_valid,

    cell11_old_not_active,

    cell11_replacement_active,

    cell11_full_572_still_blocked,
])


# ============================================================
# 17) Diagnostics
# ============================================================

print(
    "=== COMPACT RUNTIME SEED BUILD ==="
)


print(
    "\n=== FINAL RUNTIME ==="
)

print(
    "Objects:",
    len(
        cell11_final_runtime
    ),
)

print(
    "Fingerprint match:",
    cell11_runtime_fingerprints_match,
)


print(
    "\n=== PILOT STATE ==="
)

print(
    "Active Pilots:",
    len(
        cell11_active_pilot_ids
    ),
)

print(
    "Completed Pilots:",
    cell11_completed_pilot_ids,
)

print(
    "Remaining Pilots:",
    len(
        cell11_queue_ids
    ),
)

print(
    "Next Pilot:",
    cell11_next_pilot_id,
)

print(
    "Old Pilot active:",
    not cell11_old_not_active,
)

print(
    "Replacement active:",
    cell11_replacement_active,
)


print(
    "\n=== ACTIVE ID SURFACE ==="
)

print(
    "Pilot IDs:",
    len(
        cell11_active_pilot_id_set
    ),
)

print(
    "Golden Case IDs:",
    len(
        cell11_active_case_ids
    ),
)

print(
    "Task Draft IDs:",
    len(
        cell11_active_task_ids
    ),
)

print(
    "Target Evidence IDs:",
    len(
        cell11_active_target_ids
    ),
)


print(
    "\n=== ACTIVE SUPPORT SUBSETS ==="
)

print(
    "Golden Cases:",
    len(
        cell11_golden_case_subset
    ),
)

print(
    "Target by Case:",
    len(
        cell11_target_by_case_subset
    ),
)

print(
    "Target by Task:",
    len(
        cell11_target_by_task_subset
    ),
)

print(
    "Target Records:",
    len(
        cell11_target_records_subset
    ),
)

print(
    "Task Plans:",
    len(
        cell11_task_plan_subset
    ),
)

print(
    "Task Plan Rows:",
    len(
        cell11_task_plan_rows_subset
    ),
)


print(
    "\n=== HISTORY ==="
)

print(
    "Completed Pilot history objects:",
    len(
        cell11_completed_pilot_history
    ),
)

print(
    "Replacement audit objects:",
    len(
        cell11_replacement_history
    ),
)

print(
    "Old Pilot historical:",
    cell11_old_pilot_history_present,
)

print(
    "Replacement historical:",
    cell11_new_pilot_history_present,
)


print(
    "\n=== CONTRACT DEPENDENCIES ==="
)

print(
    "QUESTION_DRAFT_ID_METHOD_VERSION:",
    cell11_contracts.get(
        "QUESTION_DRAFT_ID_METHOD_VERSION"
    ),
)

print(
    "TARGET_EVIDENCE_SCHEMA_VERSION:",
    cell11_contracts.get(
        "TARGET_EVIDENCE_SCHEMA_VERSION"
    ),
)


print(
    "\n=== CALLABLE SOURCES ==="
)

print(
    "Required:",
    CELL11_REQUIRED_CALLABLE_NAMES,
)

print(
    "Recovered:",
    sorted(
        cell11_callable_sources
    ),
)

print(
    "Failures:",
    cell11_callable_source_failures,
)

print(
    "Callable objects carried into Seed:",
    not cell11_seed_has_no_callable_objects,
)


print(
    "\n=== COMPACTNESS ==="
)

print(
    "Seed bytes:",
    cell11_seed_serialized_bytes,
)

print(
    "Seed MiB:",
    round(
        cell11_seed_serialized_bytes
        /
        (
            1024
            *
            1024
        ),
        3,
    )
    if isinstance(
        cell11_seed_serialized_bytes,
        int,
    )
    else None,
)

print(
    "Limit bytes:",
    CELL11_MAX_SEED_BYTES,
)

print(
    "Compact under 8 MiB:",
    cell11_seed_is_compact,
)

print(
    "Seed SHA256:",
    cell11_seed_sha256,
)


print(
    "\n=== WORKFLOW ==="
)

print(
    "Full 572 blocked:",
    cell11_full_572_still_blocked,
)

print(
    "Mutation performed:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell11_failures
)


print(
    "\nCODE CELL 11 GATE:",
    (
        "PASS"
        if cell11_gate
        else "FAIL"
    ),
)


assert cell11_gate, (
    "Compact Runtime Seed construction failed. "
    "Repair this SAME Code Cell 11 before validating "
    "Seed independence."
)

=== COMPACT RUNTIME SEED BUILD ===

=== FINAL RUNTIME ===
Objects: 11
Fingerprint match: True

=== PILOT STATE ===
Active Pilots: 23
Completed Pilots: ['question-pilot:02ba2886f1b68521b93789becf741361']
Remaining Pilots: 22
Next Pilot: question-pilot:3005b48518a7aeb532234332df23c952
Old Pilot active: False
Replacement active: True

=== ACTIVE ID SURFACE ===
Pilot IDs: 23
Golden Case IDs: 23
Task Draft IDs: 23
Target Evidence IDs: 23

=== ACTIVE SUPPORT SUBSETS ===
Golden Cases: 23
Target by Case: 23
Target by Task: 23
Target Records: 23
Task Plans: 23
Task Plan Rows: 23

=== HISTORY ===
Completed Pilot history objects: 17
Replacement audit objects: 9
Old Pilot historical: True
Replacement historical: True

=== CONTRACT DEPENDENCIES ===
QUESTION_DRAFT_ID_METHOD_VERSION: question_draft_id_method_v1
TARGET_EVIDENCE_SCHEMA_VERSION: target_evidence_v1_draft

=== CALLABLE SOURCES ===
Required: ['stable_json_key', 'target_part_identity', 'build_target_evidence_id', 'build_question_draft_id']


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 12 — اعتبارسنجی استقلال Compact Runtime Seed

در Cell 11 یک Compact Runtime Seed زیر 8 MiB ساخته شد که Final Runtime، پشتیبانی لازم برای 23 Pilot فعال، تاریخچه‌ی Pilot تکمیل‌شده، lineage مربوط به Replacement و Source توابع ضروری را در خود نگه می‌دارد.

هدف این مرحله اثبات این است که این Seed می‌تواند در یک Namespace تازه و مستقل مورد استفاده قرار گیرد و برای اعتبارسنجی Runtime به Namespace سنگین حاصل از Replay کامل NB12 وابسته نباشد.

در این Cell بررسی می‌شود که:

- fingerprint کامل Seed بدون تغییر باقی مانده باشد؛
- 11 شیء Final Runtime به‌صورت کامل در Seed وجود داشته باشند؛
- 23 Pilot فعال، 22 Pilot باقی‌مانده و Next Pilot دقیقاً حفظ شده باشند؛
- Pilot ردشده فقط در Historical/Audit State باقی مانده و در Active Runtime حضور نداشته باشد؛
- Golden Case، Task Draft و Target Evidence مربوط به هر 23 Pilot قابل بازیابی باشند؛
- Source توابع ضروری قابل Compile و بازسازی در یک Namespace تازه باشد؛
- dependencyهای global این توابع بدون استفاده از `cell09_seed_namespace` قابل Resolve باشند؛
- هیچ Function Object متصل به Namespace قدیمی داخل Seed حمل نشده باشد؛
- یک Deep-Copy مستقل از Seed بتواند تغییر کند بدون اینکه Seed اصلی تغییر کند؛
- Full 572 Generation همچنان Blocked باقی بماند؛
- هیچ فایل، Commit، Push، PR یا Artifact ایجاد یا تغییر داده نشود.

در این مرحله هنوز هیچ Question جدید، Expected Answer جدید، Citation جدید یا Queue Advancement انجام نمی‌شود.

</div>

In [12]:
# Code Cell 12

cell12_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if (
    globals().get(
        "cell11_gate"
    )
    is not True
):
    cell12_failures.append(
        "CELL11_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell11_compact_runtime_seed"
    ),
    dict,
):
    cell12_failures.append(
        "COMPACT_RUNTIME_SEED_MISSING"
    )


# ============================================================
# 2) Accepted identities
# ============================================================

CELL12_EXPECTED_OLD_PILOT_ID = (
    "question-pilot:"
    "532a8b9d45efb8b3a32c4a62f096bb7a"
)


CELL12_EXPECTED_REPLACEMENT_PILOT_ID = (
    "question-pilot:"
    "3005b48518a7aeb532234332df23c952"
)


CELL12_EXPECTED_COMPLETED_PILOT_ID = (
    "question-pilot:"
    "02ba2886f1b68521b93789becf741361"
)


CELL12_EXPECTED_ACTIVE_COUNT = 23
CELL12_EXPECTED_REMAINING_COUNT = 22


CELL12_REQUIRED_CALLABLE_NAMES = [
    "stable_json_key",
    "target_part_identity",
    "build_target_evidence_id",
    "build_question_draft_id",
]


CELL12_CALLABLE_EXECUTION_ORDER = [
    "stable_json_key",
    "target_part_identity",
    "build_target_evidence_id",
    "build_question_draft_id",
]


# ============================================================
# 3) Helpers
# ============================================================

def cell12_collect_prefixed_ids(
    value,
    result=None,
    seen=None,
):

    if result is None:
        result = {
            "pilot":
                set(),

            "golden_case":
                set(),

            "task_draft":
                set(),

            "target_evidence":
                set(),
        }


    if seen is None:
        seen = set()


    if isinstance(
        value,
        str,
    ):

        if value.startswith(
            "question-pilot:"
        ):
            result[
                "pilot"
            ].add(
                value
            )


        elif value.startswith(
            "golden-case:"
        ):
            result[
                "golden_case"
            ].add(
                value
            )


        elif value.startswith(
            "task-draft:"
        ):
            result[
                "task_draft"
            ].add(
                value
            )


        elif value.startswith(
            "target-evidence:"
        ):
            result[
                "target_evidence"
            ].add(
                value
            )


        return result


    if not isinstance(
        value,
        (
            dict,
            list,
            tuple,
            set,
            frozenset,
        ),
    ):
        return result


    object_id = id(
        value
    )


    if object_id in seen:
        return result


    seen.add(
        object_id
    )


    try:

        if isinstance(
            value,
            dict,
        ):

            for key, nested in (
                value.items()
            ):

                cell12_collect_prefixed_ids(
                    key,
                    result,
                    seen,
                )

                cell12_collect_prefixed_ids(
                    nested,
                    result,
                    seen,
                )


        else:

            for nested in value:

                cell12_collect_prefixed_ids(
                    nested,
                    result,
                    seen,
                )


    finally:

        seen.discard(
            object_id
        )


    return result


def cell12_contains_callable(
    value,
    seen=None,
):

    if callable(
        value
    ):
        return True


    if seen is None:
        seen = set()


    if not isinstance(
        value,
        (
            dict,
            list,
            tuple,
            set,
            frozenset,
        ),
    ):
        return False


    object_id = id(
        value
    )


    if object_id in seen:
        return False


    seen.add(
        object_id
    )


    try:

        if isinstance(
            value,
            dict,
        ):

            for key, nested in (
                value.items()
            ):

                if cell12_contains_callable(
                    key,
                    seen,
                ):
                    return True


                if cell12_contains_callable(
                    nested,
                    seen,
                ):
                    return True


            return False


        for nested in value:

            if cell12_contains_callable(
                nested,
                seen,
            ):
                return True


        return False


    finally:

        seen.discard(
            object_id
        )


def cell12_function_global_loads(
    source,
):

    tree = ast.parse(
        source
    )


    function_nodes = [
        node

        for node
        in tree.body

        if isinstance(
            node,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
            ),
        )
    ]


    if len(
        function_nodes
    ) != 1:

        return {
            "function_name":
                None,

            "global_loads":
                set(),

            "error":
                "EXPECTED_EXACTLY_ONE_TOP_LEVEL_FUNCTION",
        }


    function_node = (
        function_nodes[
            0
        ]
    )


    local_names = set()


    for argument in (
        list(
            function_node.args.posonlyargs
        )
        +
        list(
            function_node.args.args
        )
        +
        list(
            function_node.args.kwonlyargs
        )
    ):

        local_names.add(
            argument.arg
        )


    if (
        function_node.args.vararg
        is not None
    ):

        local_names.add(
            function_node.args.vararg.arg
        )


    if (
        function_node.args.kwarg
        is not None
    ):

        local_names.add(
            function_node.args.kwarg.arg
        )


    for node in ast.walk(
        function_node
    ):

        if (
            isinstance(
                node,
                ast.Name,
            )
            and
            isinstance(
                node.ctx,
                (
                    ast.Store,
                    ast.Del,
                ),
            )
        ):

            local_names.add(
                node.id
            )


    loaded_names = {
        node.id

        for node
        in ast.walk(
            function_node
        )

        if (
            isinstance(
                node,
                ast.Name,
            )
            and
            isinstance(
                node.ctx,
                ast.Load,
            )
        )
    }


    builtin_names = set(
        dir(
            __import__(
                "builtins"
            )
        )
    )


    return {
        "function_name":
            function_node.name,

        "global_loads":
            (
                loaded_names
                -
                local_names
                -
                builtin_names
            ),

        "error":
            None,
    }


def cell12_walk_dict_candidates(
    value,
    path="root",
    seen=None,
):

    if seen is None:
        seen = set()


    if isinstance(
        value,
        dict,
    ):

        object_id = id(
            value
        )


        if object_id in seen:
            return


        seen.add(
            object_id
        )


        try:

            yield (
                path,
                value,
            )


            for key, nested in (
                value.items()
            ):

                nested_path = (
                    f"{path}.{key}"
                )


                yield from (
                    cell12_walk_dict_candidates(
                        nested,
                        nested_path,
                        seen,
                    )
                )


        finally:

            seen.discard(
                object_id
            )


    elif isinstance(
        value,
        (
            list,
            tuple,
        ),
    ):

        object_id = id(
            value
        )


        if object_id in seen:
            return


        seen.add(
            object_id
        )


        try:

            for index, nested in enumerate(
                value
            ):

                nested_path = (
                    f"{path}[{index}]"
                )


                yield from (
                    cell12_walk_dict_candidates(
                        nested,
                        nested_path,
                        seen,
                    )
                )


        finally:

            seen.discard(
                object_id
            )


# ============================================================
# 4) Freeze external state
# ============================================================

(
    cell12_git_before_rc,
    cell12_git_before,
    cell12_git_before_err,
) = run_git(
    "status",
    "--short",
)


cell12_prior_before = {
    "size":
        prior_notebook_path.stat().st_size,

    "sha256":
        hashlib.sha256(
            prior_notebook_path.read_bytes()
        ).hexdigest(),
}


cell12_handoff_before = {
    "size":
        handoff_path.stat().st_size,

    "sha256":
        hashlib.sha256(
            handoff_path.read_bytes()
        ).hexdigest(),
}


# ============================================================
# 5) Independent Seed copy
# ============================================================

cell12_seed = copy.deepcopy(
    cell11_compact_runtime_seed
)


cell12_seed_sha256 = (
    cell10_stable_sha(
        cell12_seed
    )
)


cell12_seed_fingerprint_valid = (
    cell12_seed_sha256
    ==
    cell11_seed_sha256
)


cell12_seed_has_no_callable_objects = (
    not
    cell12_contains_callable(
        cell12_seed
    )
)


cell12_seed_text = (
    cell10_canonical_json(
        cell12_seed
    )
)


cell12_forbidden_runtime_reference_absent = (
    "cell09_seed_namespace"
    not in
    cell12_seed_text
)


# ============================================================
# 6) Build fresh isolated Runtime Namespace
# ============================================================

cell12_runtime_namespace = {
    "__builtins__":
        __builtins__,

    "__name__":
        "compact_seed_independent_runtime",

    "hashlib":
        hashlib,

    "json":
        json,

    "re":
        re,

    "copy":
        copy,

    "Path":
        Path,

    "Counter":
        Counter,

    "defaultdict":
        defaultdict,
}


cell12_runtime_namespace[
    "COMPACT_RUNTIME_SEED"
] = copy.deepcopy(
    cell12_seed
)


cell12_seed_runtime = (
    cell12_runtime_namespace[
        "COMPACT_RUNTIME_SEED"
    ]
)


cell12_contracts = (
    cell12_seed_runtime.get(
        "contracts",
        {},
    )
)


cell12_final_runtime = (
    cell12_seed_runtime.get(
        "final_runtime",
        {},
    )
)


cell12_active_support = (
    cell12_seed_runtime.get(
        "active_support",
        {},
    )
)


# ============================================================
# 7) Promote contracts / constants
# ============================================================

if isinstance(
    cell12_contracts,
    dict,
):

    for name, value in (
        cell12_contracts.items()
    ):

        cell12_runtime_namespace[
            name
        ] = copy.deepcopy(
            value
        )


cell12_question_id_version_present = (
    cell12_runtime_namespace.get(
        "QUESTION_DRAFT_ID_METHOD_VERSION"
    )
    ==
    "question_draft_id_method_v1"
)


cell12_target_schema_version_present = (
    cell12_runtime_namespace.get(
        "TARGET_EVIDENCE_SCHEMA_VERSION"
    )
    ==
    "target_evidence_v1_draft"
)


# ============================================================
# 8) Promote Final Runtime / active support
# ============================================================

if isinstance(
    cell12_final_runtime,
    dict,
):

    for name, value in (
        cell12_final_runtime.items()
    ):

        cell12_runtime_namespace[
            name
        ] = copy.deepcopy(
            value
        )


if isinstance(
    cell12_active_support,
    dict,
):

    for name, value in (
        cell12_active_support.items()
    ):

        cell12_runtime_namespace[
            name
        ] = copy.deepcopy(
            value
        )


# ============================================================
# 9) Validate exact Final Runtime surface
# ============================================================

cell12_final_runtime_surface_valid = (
    set(
        cell12_final_runtime
    )
    ==
    set(
        CELL11_FINAL_RUNTIME_NAMES
    )
)


cell12_runtime_fingerprints = {
    name:
        cell10_stable_sha(
            cell12_final_runtime[
                name
            ]
        )

    for name
    in CELL11_FINAL_RUNTIME_NAMES

    if name
    in
    cell12_final_runtime
}


cell12_runtime_fingerprints_valid = (
    cell12_runtime_fingerprints
    ==
    cell10_final_runtime_fingerprints
)


# ============================================================
# 10) Validate Pilot workflow state
# ============================================================

cell12_runtime_ids = (
    cell12_final_runtime.get(
        "cell95_pilot_ids_in_runtime_order"
    )
)


cell12_remaining_ids = (
    cell12_final_runtime.get(
        "cell95_remaining_pilot_ids"
    )
)


cell12_remaining_queue = (
    cell12_final_runtime.get(
        "cell95_remaining_pilot_queue"
    )
)


cell12_next_pilot_id = (
    cell12_final_runtime.get(
        "cell95_next_pilot_id"
    )
)


cell12_pilot_state_valid = all([
    isinstance(
        cell12_runtime_ids,
        list,
    ),

    len(
        cell12_runtime_ids
    )
    ==
    CELL12_EXPECTED_ACTIVE_COUNT,

    len(
        set(
            cell12_runtime_ids
        )
    )
    ==
    CELL12_EXPECTED_ACTIVE_COUNT,

    isinstance(
        cell12_remaining_ids,
        list,
    ),

    len(
        cell12_remaining_ids
    )
    ==
    CELL12_EXPECTED_REMAINING_COUNT,

    isinstance(
        cell12_remaining_queue,
        list,
    ),

    len(
        cell12_remaining_queue
    )
    ==
    CELL12_EXPECTED_REMAINING_COUNT,

    CELL12_EXPECTED_OLD_PILOT_ID
    not in
    cell12_runtime_ids,

    CELL12_EXPECTED_REPLACEMENT_PILOT_ID
    in
    cell12_runtime_ids,

    CELL12_EXPECTED_COMPLETED_PILOT_ID
    in
    cell12_runtime_ids,

    CELL12_EXPECTED_COMPLETED_PILOT_ID
    not in
    cell12_remaining_ids,

    cell12_next_pilot_id
    ==
    CELL12_EXPECTED_REPLACEMENT_PILOT_ID,

    cell12_remaining_ids[
        0
    ]
    ==
    CELL12_EXPECTED_REPLACEMENT_PILOT_ID,
])


# ============================================================
# 11) Active identity surface
# ============================================================

cell12_runtime_id_sets = (
    cell12_collect_prefixed_ids(
        cell12_final_runtime
    )
)


cell12_runtime_pilot_ids = (
    cell12_runtime_id_sets[
        "pilot"
    ]
)


cell12_runtime_case_ids = (
    cell12_runtime_id_sets[
        "golden_case"
    ]
)


cell12_runtime_task_ids = (
    cell12_runtime_id_sets[
        "task_draft"
    ]
)


cell12_runtime_target_ids = (
    cell12_runtime_id_sets[
        "target_evidence"
    ]
)


cell12_identity_surface_valid = all([
    len(
        cell12_runtime_pilot_ids
    )
    ==
    23,

    len(
        cell12_runtime_case_ids
    )
    ==
    23,

    len(
        cell12_runtime_task_ids
    )
    ==
    23,

    len(
        cell12_runtime_target_ids
    )
    ==
    23,
])


# ============================================================
# 12) Active support
# ============================================================

cell12_support_case_map = (
    cell12_active_support.get(
        "golden_case_records_by_id",
        {},
    )
)


cell12_support_case_rows = (
    cell12_active_support.get(
        "golden_case_records",
        [],
    )
)


cell12_support_task_plans = (
    cell12_active_support.get(
        "task_draft_plans",
        {},
    )
)


cell12_support_task_rows = (
    cell12_active_support.get(
        "task_draft_plan_rows",
        [],
    )
)


cell12_support_target_by_case = (
    cell12_active_support.get(
        "target_evidence_by_case_id",
        {},
    )
)


cell12_support_target_by_task = (
    cell12_active_support.get(
        "target_evidence_by_task_draft_id",
        {},
    )
)


cell12_support_target_rows = (
    cell12_active_support.get(
        "target_evidence_records",
        [],
    )
)


cell12_support_case_row_ids = {
    row.get(
        "golden_case_id"
    )

    for row
    in cell12_support_case_rows

    if isinstance(
        row,
        dict,
    )
}


cell12_support_task_row_ids = {
    row.get(
        "task_draft_id"
    )

    for row
    in cell12_support_task_rows

    if isinstance(
        row,
        dict,
    )
}


cell12_support_target_ids = {
    row.get(
        "target_evidence_id"
    )

    for row
    in cell12_support_target_rows

    if isinstance(
        row,
        dict,
    )
}


cell12_support_surface_valid = all([
    isinstance(
        cell12_support_case_map,
        dict,
    ),

    len(
        cell12_support_case_map
    )
    ==
    23,

    set(
        cell12_support_case_map
    )
    ==
    cell12_runtime_case_ids,

    cell12_support_case_row_ids
    ==
    cell12_runtime_case_ids,

    isinstance(
        cell12_support_task_plans,
        dict,
    ),

    len(
        cell12_support_task_plans
    )
    ==
    23,

    isinstance(
        cell12_support_task_rows,
        list,
    ),

    len(
        cell12_support_task_rows
    )
    ==
    23,

    cell12_runtime_task_ids
    <=
    cell12_support_task_row_ids,

    isinstance(
        cell12_support_target_by_case,
        dict,
    ),

    len(
        cell12_support_target_by_case
    )
    ==
    23,

    set(
        cell12_support_target_by_case
    )
    ==
    cell12_runtime_case_ids,

    isinstance(
        cell12_support_target_by_task,
        dict,
    ),

    len(
        cell12_support_target_by_task
    )
    ==
    23,

    set(
        cell12_support_target_by_task
    )
    ==
    cell12_runtime_task_ids,

    cell12_runtime_target_ids
    <=
    cell12_support_target_ids,
])


# ============================================================
# 13) Historical state
# ============================================================

cell12_completed_section = (
    cell12_seed_runtime.get(
        "completed_pilot",
        {},
    )
)


cell12_replacement_section = (
    cell12_seed_runtime.get(
        "replacement_audit",
        {},
    )
)


cell12_completed_history_ids = (
    cell12_collect_prefixed_ids(
        cell12_completed_section
    )
)


cell12_replacement_history_ids = (
    cell12_collect_prefixed_ids(
        cell12_replacement_section
    )
)


cell12_history_valid = all([
    cell12_completed_section.get(
        "pilot_id"
    )
    ==
    CELL12_EXPECTED_COMPLETED_PILOT_ID,

    CELL12_EXPECTED_COMPLETED_PILOT_ID
    in
    cell12_completed_history_ids[
        "pilot"
    ],

    cell12_replacement_section.get(
        "old_pilot_id"
    )
    ==
    CELL12_EXPECTED_OLD_PILOT_ID,

    cell12_replacement_section.get(
        "replacement_pilot_id"
    )
    ==
    CELL12_EXPECTED_REPLACEMENT_PILOT_ID,

    CELL12_EXPECTED_OLD_PILOT_ID
    in
    cell12_replacement_history_ids[
        "pilot"
    ],

    CELL12_EXPECTED_REPLACEMENT_PILOT_ID
    in
    cell12_replacement_history_ids[
        "pilot"
    ],
])


# ============================================================
# 14) Validate callable-source closure
# ============================================================

cell12_callable_sources = (
    cell12_seed_runtime.get(
        "callable_sources",
        {},
    )
)


cell12_callable_source_names_valid = (
    set(
        cell12_callable_sources
    )
    ==
    set(
        CELL12_REQUIRED_CALLABLE_NAMES
    )
)


cell12_callable_source_checks = {}

cell12_callable_dependency_failures = []

cell12_callable_compile_failures = []


cell12_builtin_names = set(
    dir(
        __import__(
            "builtins"
        )
    )
)


for name in (
    CELL12_REQUIRED_CALLABLE_NAMES
):

    metadata = (
        cell12_callable_sources.get(
            name
        )
    )


    if not isinstance(
        metadata,
        dict,
    ):

        cell12_callable_compile_failures.append({
            "name":
                name,

            "reason":
                "CALLABLE_METADATA_MISSING",
        })

        continue


    source = (
        metadata.get(
            "source"
        )
    )


    expected_source_sha = (
        metadata.get(
            "source_sha256"
        )
    )


    if not isinstance(
        source,
        str,
    ):

        cell12_callable_compile_failures.append({
            "name":
                name,

            "reason":
                "CALLABLE_SOURCE_MISSING",
        })

        continue


    actual_source_sha = (
        hashlib.sha256(
            source.encode(
                "utf-8"
            )
        ).hexdigest()
    )


    source_sha_valid = (
        actual_source_sha
        ==
        expected_source_sha
    )


    dependency_result = (
        cell12_function_global_loads(
            source
        )
    )


    global_loads = (
        dependency_result[
            "global_loads"
        ]
    )


    available_names = (
        set(
            cell12_runtime_namespace
        )
        |
        set(
            CELL12_REQUIRED_CALLABLE_NAMES
        )
        |
        cell12_builtin_names
    )


    unresolved = (
        global_loads
        -
        available_names
    )


    if unresolved:

        cell12_callable_dependency_failures.append({
            "name":
                name,

            "unresolved":
                sorted(
                    unresolved
                ),
        })


    cell12_callable_source_checks[
        name
    ] = {
        "source_sha_valid":
            source_sha_valid,

        "function_name":
            dependency_result[
                "function_name"
            ],

        "global_loads":
            sorted(
                global_loads
            ),

        "unresolved_global_loads":
            sorted(
                unresolved
            ),

        "ast_error":
            dependency_result[
                "error"
            ],
    }


# ============================================================
# 15) Reconstruct all required callables
# ============================================================

for name in (
    CELL12_CALLABLE_EXECUTION_ORDER
):

    metadata = (
        cell12_callable_sources.get(
            name
        )
    )


    if not isinstance(
        metadata,
        dict,
    ):
        continue


    source = (
        metadata.get(
            "source"
        )
    )


    if not isinstance(
        source,
        str,
    ):
        continue


    try:

        exec(
            compile(
                source,
                filename=(
                    f"<compact-seed-{name}>"
                ),
                mode="exec",
            ),
            cell12_runtime_namespace,
            cell12_runtime_namespace,
        )


    except Exception as exc:

        cell12_callable_compile_failures.append({
            "name":
                name,

            "reason":
                "EXEC_FAILED",

            "error":
                repr(
                    exc
                ),
        })


cell12_reconstructed_callables = {
    name:
        cell12_runtime_namespace.get(
            name
        )

    for name
    in CELL12_REQUIRED_CALLABLE_NAMES
}


cell12_reconstructed_callable_names = sorted(
    name

    for name, value
    in cell12_reconstructed_callables.items()

    if callable(
        value
    )
)


cell12_reconstructed_callable_names_valid = (
    set(
        cell12_reconstructed_callable_names
    )
    ==
    set(
        CELL12_REQUIRED_CALLABLE_NAMES
    )
)


cell12_reconstructed_callables_valid = all(
    callable(
        cell12_reconstructed_callables.get(
            name
        )
    )

    for name
    in CELL12_REQUIRED_CALLABLE_NAMES
)


# ============================================================
# 16) Explicit runtime dependency bindings
# ============================================================

cell12_stable_json_function = (
    cell12_reconstructed_callables.get(
        "stable_json_key"
    )
)


cell12_target_part_identity_function = (
    cell12_reconstructed_callables.get(
        "target_part_identity"
    )
)


cell12_target_id_function = (
    cell12_reconstructed_callables.get(
        "build_target_evidence_id"
    )
)


cell12_question_id_function = (
    cell12_reconstructed_callables.get(
        "build_question_draft_id"
    )
)


cell12_target_builder_globals = (
    getattr(
        cell12_target_id_function,
        "__globals__",
        {},
    )

    if callable(
        cell12_target_id_function
    )

    else {}
)


cell12_question_builder_globals = (
    getattr(
        cell12_question_id_function,
        "__globals__",
        {},
    )

    if callable(
        cell12_question_id_function
    )

    else {}
)


cell12_runtime_dependency_bindings_valid = all([
    callable(
        cell12_stable_json_function
    ),

    callable(
        cell12_target_part_identity_function
    ),

    callable(
        cell12_target_id_function
    ),

    callable(
        cell12_question_id_function
    ),

    cell12_target_builder_globals.get(
        "target_part_identity"
    )
    is
    cell12_target_part_identity_function,

    cell12_target_builder_globals.get(
        "stable_json_key"
    )
    is
    cell12_stable_json_function,

    cell12_target_builder_globals.get(
        "TARGET_EVIDENCE_SCHEMA_VERSION"
    )
    ==
    "target_evidence_v1_draft",

    cell12_question_builder_globals.get(
        "stable_json_key"
    )
    is
    cell12_stable_json_function,

    cell12_question_builder_globals.get(
        "QUESTION_DRAFT_ID_METHOD_VERSION"
    )
    ==
    "question_draft_id_method_v1",
])


cell12_reconstructed_callable_globals_clean = all(
    "cell09_seed_namespace"
    not in
    getattr(
        value,
        "__globals__",
        {},
    )

    for value
    in cell12_reconstructed_callables.values()

    if callable(
        value
    )
)


# ============================================================
# 17) stable_json_key behavioral probe
# ============================================================

cell12_stable_json_probe_valid = False
cell12_stable_json_probe_error = None


if callable(
    cell12_stable_json_function
):

    try:

        cell12_stable_probe_a = (
            cell12_stable_json_function({
                "b":
                    2,

                "a":
                    1,
            })
        )


        cell12_stable_probe_b = (
            cell12_stable_json_function({
                "a":
                    1,

                "b":
                    2,
            })
        )


        cell12_stable_json_probe_valid = all([
            isinstance(
                cell12_stable_probe_a,
                str,
            ),

            cell12_stable_probe_a
            ==
            cell12_stable_probe_b,
        ])


    except Exception as exc:

        cell12_stable_json_probe_error = (
            repr(
                exc
            )
        )


else:

    cell12_stable_json_probe_error = (
        "stable_json_key not reconstructed"
    )


# ============================================================
# 18) target_part_identity behavioral probe
#
# IMPORTANT:
# Use REAL Target Part candidates from the Compact Seed.
# Do not fabricate an incomplete synthetic part.
# ============================================================

cell12_target_part_probe_valid = False

cell12_target_part_probe_error = None

cell12_target_part_probe_path = None

cell12_target_part_probe_result = None

cell12_target_part_probe_attempts = []

cell12_target_part_candidate_count = 0


cell12_target_part_probe_roots = {
    "final_runtime":
        cell12_final_runtime,

    "active_support":
        cell12_active_support,
}


if callable(
    cell12_target_part_identity_function
):

    for root_name, root_value in (
        cell12_target_part_probe_roots.items()
    ):

        if (
            cell12_target_part_probe_valid
        ):
            break


        for candidate_path, candidate in (
            cell12_walk_dict_candidates(
                root_value,
                path=root_name,
            )
        ):

            # target_part_identity requires a real target-part
            # identity surface. document_id is a known hard field.
            if (
                "document_id"
                not in
                candidate
            ):
                continue


            cell12_target_part_candidate_count += 1


            try:

                probe_result = (
                    cell12_target_part_identity_function(
                        copy.deepcopy(
                            candidate
                        )
                    )
                )


                cell12_target_part_probe_attempts.append({
                    "path":
                        candidate_path,

                    "status":
                        "PASS",

                    "candidate_keys":
                        sorted(
                            str(
                                key
                            )
                            for key
                            in candidate.keys()
                        ),

                    "result_type":
                        type(
                            probe_result
                        ).__name__,
                })


                cell12_target_part_probe_valid = (
                    probe_result
                    is not None
                )


                if (
                    cell12_target_part_probe_valid
                ):

                    cell12_target_part_probe_path = (
                        candidate_path
                    )

                    cell12_target_part_probe_result = (
                        probe_result
                    )

                    cell12_target_part_probe_error = None

                    break


            except Exception as exc:

                cell12_target_part_probe_attempts.append({
                    "path":
                        candidate_path,

                    "status":
                        "FAIL",

                    "candidate_keys":
                        sorted(
                            str(
                                key
                            )
                            for key
                            in candidate.keys()
                        ),

                    "error":
                        repr(
                            exc
                        ),
                })


    if not (
        cell12_target_part_probe_valid
    ):

        if (
            cell12_target_part_candidate_count
            ==
            0
        ):

            cell12_target_part_probe_error = (
                "NO_REAL_DOCUMENT_ID_CANDIDATE_FOUND_IN_SEED"
            )

        else:

            cell12_target_part_probe_error = (
                "NO_REAL_TARGET_PART_CANDIDATE_PASSED"
            )


else:

    cell12_target_part_probe_error = (
        "target_part_identity not reconstructed"
    )


# ============================================================
# 19) Question Draft ID behavioral probe
# ============================================================

cell12_question_id_probe_valid = False

cell12_question_id_probe_error = None

cell12_question_id_probe_value = None


if callable(
    cell12_question_id_function
):

    try:

        cell12_question_id_probe_a = (
            cell12_question_id_function(
                pilot_case_id=
                    CELL12_EXPECTED_REPLACEMENT_PILOT_ID,

                question_text_sha256=
                    (
                        "0"
                        *
                        64
                    ),
            )
        )


        cell12_question_id_probe_b = (
            cell12_question_id_function(
                pilot_case_id=
                    CELL12_EXPECTED_REPLACEMENT_PILOT_ID,

                question_text_sha256=
                    (
                        "0"
                        *
                        64
                    ),
            )
        )


        cell12_question_id_probe_value = (
            cell12_question_id_probe_a
        )


        cell12_question_id_probe_valid = all([
            isinstance(
                cell12_question_id_probe_a,
                str,
            ),

            cell12_question_id_probe_a.startswith(
                "question-draft:"
            ),

            cell12_question_id_probe_a
            ==
            cell12_question_id_probe_b,
        ])


    except Exception as exc:

        cell12_question_id_probe_error = (
            repr(
                exc
            )
        )


else:

    cell12_question_id_probe_error = (
        "build_question_draft_id not reconstructed"
    )


# ============================================================
# 20) Deep-copy isolation probe
# ============================================================

cell12_seed_before_probe_sha = (
    cell10_stable_sha(
        cell12_seed
    )
)


cell12_isolation_probe = (
    copy.deepcopy(
        cell12_seed
    )
)


cell12_probe_remaining_ids = (
    cell12_isolation_probe[
        "final_runtime"
    ][
        "cell95_remaining_pilot_ids"
    ]
)


cell12_probe_count_before = (
    len(
        cell12_probe_remaining_ids
    )
)


cell12_probe_remaining_ids.append(
    "question-pilot:"
    "LOCAL_ISOLATION_PROBE_ONLY"
)


cell12_probe_count_after = (
    len(
        cell12_probe_remaining_ids
    )
)


cell12_seed_after_probe_sha = (
    cell10_stable_sha(
        cell12_seed
    )
)


cell12_isolation_probe_valid = all([
    cell12_probe_count_before
    ==
    22,

    cell12_probe_count_after
    ==
    23,

    len(
        cell12_seed[
            "final_runtime"
        ][
            "cell95_remaining_pilot_ids"
        ]
    )
    ==
    22,

    cell12_seed_before_probe_sha
    ==
    cell12_seed_after_probe_sha,

    cell12_seed_after_probe_sha
    ==
    cell11_seed_sha256,
])


# ============================================================
# 21) Full-572 remains blocked
# ============================================================

cell12_full_572_status = (
    cell12_seed_runtime.get(
        "source",
        {},
    ).get(
        "full_572_generation"
    )
)


cell12_full_572_blocked = (
    cell12_full_572_status
    ==
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


# ============================================================
# 22) Verify external state
# ============================================================

(
    cell12_git_after_rc,
    cell12_git_after,
    cell12_git_after_err,
) = run_git(
    "status",
    "--short",
)


cell12_prior_after = {
    "size":
        prior_notebook_path.stat().st_size,

    "sha256":
        hashlib.sha256(
            prior_notebook_path.read_bytes()
        ).hexdigest(),
}


cell12_handoff_after = {
    "size":
        handoff_path.stat().st_size,

    "sha256":
        hashlib.sha256(
            handoff_path.read_bytes()
        ).hexdigest(),
}


cell12_external_state_stable = all([
    cell12_git_before_rc
    ==
    cell12_git_after_rc,

    cell12_git_before
    ==
    cell12_git_after,

    cell12_prior_before
    ==
    cell12_prior_after,

    cell12_handoff_before
    ==
    cell12_handoff_after,
])


# ============================================================
# 23) Final Gate
# ============================================================

cell12_gate = all([
    len(
        cell12_failures
    )
    ==
    0,

    cell12_seed_fingerprint_valid,

    cell12_seed_has_no_callable_objects,

    cell12_forbidden_runtime_reference_absent,

    cell12_question_id_version_present,

    cell12_target_schema_version_present,

    cell12_final_runtime_surface_valid,

    cell12_runtime_fingerprints_valid,

    cell12_pilot_state_valid,

    cell12_identity_surface_valid,

    cell12_support_surface_valid,

    cell12_history_valid,

    cell12_callable_source_names_valid,

    len(
        cell12_callable_dependency_failures
    )
    ==
    0,

    len(
        cell12_callable_compile_failures
    )
    ==
    0,

    all(
        row[
            "source_sha_valid"
        ]

        for row
        in cell12_callable_source_checks.values()
    ),

    cell12_reconstructed_callable_names_valid,

    cell12_reconstructed_callables_valid,

    cell12_runtime_dependency_bindings_valid,

    cell12_reconstructed_callable_globals_clean,

    cell12_stable_json_probe_valid,

    cell12_target_part_probe_valid,

    cell12_question_id_probe_valid,

    cell12_isolation_probe_valid,

    cell12_full_572_blocked,

    cell12_external_state_stable,
])


# ============================================================
# 24) Diagnostics
# ============================================================

print(
    "=== COMPACT RUNTIME SEED "
    "INDEPENDENCE VALIDATION V3 ==="
)


print(
    "\n=== SEED IDENTITY ==="
)

print(
    "Expected SHA256:",
    cell11_seed_sha256,
)

print(
    "Independent SHA256:",
    cell12_seed_sha256,
)

print(
    "Fingerprint valid:",
    cell12_seed_fingerprint_valid,
)

print(
    "Callable objects inside Seed:",
    not cell12_seed_has_no_callable_objects,
)

print(
    "Direct cell09_seed_namespace reference:",
    not cell12_forbidden_runtime_reference_absent,
)


print(
    "\n=== CONSTANT DEPENDENCIES ==="
)

print(
    "QUESTION_DRAFT_ID_METHOD_VERSION:",
    cell12_runtime_namespace.get(
        "QUESTION_DRAFT_ID_METHOD_VERSION"
    ),
)

print(
    "TARGET_EVIDENCE_SCHEMA_VERSION:",
    cell12_runtime_namespace.get(
        "TARGET_EVIDENCE_SCHEMA_VERSION"
    ),
)


print(
    "\n=== FINAL RUNTIME ==="
)

print(
    "Surface valid:",
    cell12_final_runtime_surface_valid,
)

print(
    "Fingerprint valid:",
    cell12_runtime_fingerprints_valid,
)

print(
    "Active Pilots:",
    len(
        cell12_runtime_ids
    )
    if isinstance(
        cell12_runtime_ids,
        list,
    )
    else None,
)

print(
    "Remaining Pilots:",
    len(
        cell12_remaining_ids
    )
    if isinstance(
        cell12_remaining_ids,
        list,
    )
    else None,
)

print(
    "Next Pilot:",
    cell12_next_pilot_id,
)

print(
    "Old Pilot active:",
    (
        CELL12_EXPECTED_OLD_PILOT_ID
        in
        cell12_runtime_ids
    )
    if isinstance(
        cell12_runtime_ids,
        list,
    )
    else None,
)

print(
    "Replacement active:",
    (
        CELL12_EXPECTED_REPLACEMENT_PILOT_ID
        in
        cell12_runtime_ids
    )
    if isinstance(
        cell12_runtime_ids,
        list,
    )
    else None,
)


print(
    "\n=== ACTIVE SUPPORT ==="
)

print(
    "Identity surface valid:",
    cell12_identity_surface_valid,
)

print(
    "Golden Cases:",
    len(
        cell12_support_case_map
    ),
)

print(
    "Task Plans:",
    len(
        cell12_support_task_plans
    ),
)

print(
    "Target by Case:",
    len(
        cell12_support_target_by_case
    ),
)

print(
    "Target by Task:",
    len(
        cell12_support_target_by_task
    ),
)

print(
    "Target Records:",
    len(
        cell12_support_target_rows
    ),
)

print(
    "Support surface valid:",
    cell12_support_surface_valid,
)


print(
    "\n=== HISTORY ==="
)

print(
    "Completed Pilot:",
    cell12_completed_section.get(
        "pilot_id"
    ),
)

print(
    "Old Pilot retained historically:",
    (
        CELL12_EXPECTED_OLD_PILOT_ID
        in
        cell12_replacement_history_ids[
            "pilot"
        ]
    ),
)

print(
    "Replacement retained historically:",
    (
        CELL12_EXPECTED_REPLACEMENT_PILOT_ID
        in
        cell12_replacement_history_ids[
            "pilot"
        ]
    ),
)

print(
    "History valid:",
    cell12_history_valid,
)


print(
    "\n=== CALLABLE DEPENDENCIES ==="
)

for name in (
    CELL12_REQUIRED_CALLABLE_NAMES
):

    print({
        "name":
            name,

        "source_sha_valid":
            cell12_callable_source_checks.get(
                name,
                {},
            ).get(
                "source_sha_valid"
            ),

        "global_loads":
            cell12_callable_source_checks.get(
                name,
                {},
            ).get(
                "global_loads"
            ),

        "unresolved":
            cell12_callable_source_checks.get(
                name,
                {},
            ).get(
                "unresolved_global_loads"
            ),
    })


print(
    "Dependency failures:",
    cell12_callable_dependency_failures,
)

print(
    "Compile failures:",
    cell12_callable_compile_failures,
)


print(
    "\n=== RECONSTRUCTED CALLABLES ==="
)

print(
    "Required:",
    CELL12_REQUIRED_CALLABLE_NAMES,
)

print(
    "Reconstructed:",
    cell12_reconstructed_callable_names,
)

print(
    "Exact callable surface valid:",
    cell12_reconstructed_callable_names_valid,
)

print(
    "All reconstructed callables valid:",
    cell12_reconstructed_callables_valid,
)

print(
    "Dependency bindings valid:",
    cell12_runtime_dependency_bindings_valid,
)

print(
    "Callable globals clean:",
    cell12_reconstructed_callable_globals_clean,
)


print(
    "\n=== BEHAVIORAL PROBES ==="
)

print(
    "stable_json_key probe:",
    cell12_stable_json_probe_valid,
)

print(
    "stable_json_key probe error:",
    cell12_stable_json_probe_error,
)


print(
    "Real target-part candidates inspected:",
    cell12_target_part_candidate_count,
)

print(
    "target_part_identity probe:",
    cell12_target_part_probe_valid,
)

print(
    "target_part_identity probe path:",
    cell12_target_part_probe_path,
)

print(
    "target_part_identity probe result:",
    cell12_target_part_probe_result,
)

print(
    "target_part_identity probe error:",
    cell12_target_part_probe_error,
)


print(
    "build_question_draft_id probe:",
    cell12_question_id_probe_valid,
)

print(
    "build_question_draft_id probe value:",
    cell12_question_id_probe_value,
)

print(
    "build_question_draft_id probe error:",
    cell12_question_id_probe_error,
)


if not (
    cell12_target_part_probe_valid
):

    print(
        "\nLast target-part probe attempts:"
    )

    for row in (
        cell12_target_part_probe_attempts[
            -10:
        ]
    ):

        print(
            row
        )


print(
    "\n=== ISOLATION ==="
)

print(
    "Probe queue before:",
    cell12_probe_count_before,
)

print(
    "Probe queue after local mutation:",
    cell12_probe_count_after,
)

print(
    "Original Seed queue:",
    len(
        cell12_seed[
            "final_runtime"
        ][
            "cell95_remaining_pilot_ids"
        ]
    ),
)

print(
    "Original Seed unchanged:",
    cell12_isolation_probe_valid,
)


print(
    "\n=== WORKFLOW ==="
)

print(
    "Full 572 status:",
    cell12_full_572_status,
)

print(
    "Full 572 blocked:",
    cell12_full_572_blocked,
)


print(
    "\n=== EXTERNAL STATE ==="
)

print(
    "Git unchanged:",
    cell12_git_before
    ==
    cell12_git_after,
)

print(
    "Prior NB12 unchanged:",
    cell12_prior_before
    ==
    cell12_prior_after,
)

print(
    "Existing handoff unchanged:",
    cell12_handoff_before
    ==
    cell12_handoff_after,
)

print(
    "External state stable:",
    cell12_external_state_stable,
)

print(
    "Project/file mutation performed:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell12_failures
)


print(
    "\nCODE CELL 12 GATE:",
    (
        "PASS"
        if cell12_gate
        else "FAIL"
    ),
)


assert cell12_gate, (
    "Compact Runtime Seed independence validation V3 "
    "failed. Repair this SAME Code Cell 12 before "
    "building the reusable Pilot continuation engine."
)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


=== COMPACT RUNTIME SEED INDEPENDENCE VALIDATION V3 ===

=== SEED IDENTITY ===
Expected SHA256: 6098087797ba0c9eb39b36af1ec6ccb044668c19c9a3d2f5e345b17f9e1ef1dd
Independent SHA256: 6098087797ba0c9eb39b36af1ec6ccb044668c19c9a3d2f5e345b17f9e1ef1dd
Fingerprint valid: True
Callable objects inside Seed: False
Direct cell09_seed_namespace reference: False

=== CONSTANT DEPENDENCIES ===
QUESTION_DRAFT_ID_METHOD_VERSION: question_draft_id_method_v1
TARGET_EVIDENCE_SCHEMA_VERSION: target_evidence_v1_draft

=== FINAL RUNTIME ===
Surface valid: True
Fingerprint valid: True
Active Pilots: 23
Remaining Pilots: 22
Next Pilot: question-pilot:3005b48518a7aeb532234332df23c952
Old Pilot active: False
Replacement active: True

=== ACTIVE SUPPORT ===
Identity surface valid: True
Golden Cases: 23
Task Plans: 23
Target by Case: 23
Target by Task: 23
Target Records: 23
Support surface valid: True

=== HISTORY ===
Completed Pilot: question-pilot:02ba2886f1b68521b93789becf741361
Old Pilot retained historically

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 13 — ساخت Reusable Pilot Continuation Engine

Cell 12 اثبات کرد که Compact Runtime Seed مستقل است و تمام dependencyهای لازم برای ادامه‌ی Workflow را در اختیار دارد.

در این مرحله یک موتور عمومی و قابل‌استفاده‌ی مجدد برای ادامه‌ی 22 Pilot باقی‌مانده ساخته می‌شود.

این موتور:

- Pilot فعلی را فقط از Queue رسمی Runtime انتخاب می‌کند؛
- Golden Case، Task Draft و Target Evidence همان Pilot را از Compact Seed Resolve می‌کند؛
- Workflow را به‌صورت State Machine صریح مدیریت می‌کند؛
- Question، Expected Answer و Citation را سه Artifact مستقل در نظر می‌گیرد؛
- برای هر سه Artifact یک Human Review Gate اجباری دارد؛
- Revision Loop را بدون Queue Advancement پشتیبانی می‌کند؛
- در صورت ردشدن Target می‌تواند Workflow را به حالت Retarget Required ببرد؛
- Queue Advancement را فقط پس از پذیرش نهایی Citation مجاز می‌کند؛
- Full 572 Generation را در تمام مراحل Blocked نگه می‌دارد؛
- تمام Transitionها را به‌صورت Pure و روی Deep Copy انجام می‌دهد تا Runtime اصلی ناخواسته Mutate نشود.

در این Cell هیچ Question واقعی، Expected Answer واقعی یا Citation واقعی تولید نمی‌شود.

همچنین هیچ Pilot واقعی Complete نمی‌شود و Queue واقعی از 22 به 21 کاهش پیدا نمی‌کند.

برای تست موتور فقط از Artifactهای مصنوعی با برچسب صریح `ENGINE_DRY_RUN_ONLY` روی Sessionهای موقت استفاده می‌شود.

هیچ فایل، Commit، Push، PR یا Artifact خارجی ایجاد یا تغییر داده نمی‌شود.

</div>

In [13]:
# Code Cell 13

cell13_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if (
    globals().get(
        "cell12_gate"
    )
    is not True
):
    cell13_failures.append(
        "CELL12_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell12_seed"
    ),
    dict,
):
    cell13_failures.append(
        "CELL12_INDEPENDENT_SEED_MISSING"
    )


# ============================================================
# 2) Fixed workflow contract
# ============================================================

CELL13_ENGINE_VERSION = (
    "pilot_continuation_engine_v1"
)


CELL13_INITIAL_STATE = (
    "QUESTION_DRAFT_PENDING"
)


CELL13_FINAL_PRE_ADVANCE_STATE = (
    "PILOT_COMPLETE_PENDING_QUEUE_ADVANCE"
)


CELL13_RETARGET_STATE = (
    "BLOCKED_REQUIRES_RETARGET"
)


CELL13_FULL_572_BLOCK_STATUS = (
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


CELL13_TRANSITIONS = {
    "QUESTION_DRAFT_PENDING":
        {
            "QUESTION_DRAFT_REGISTERED":
                "QUESTION_HUMAN_REVIEW_PENDING",

            "TARGET_REJECTED":
                CELL13_RETARGET_STATE,
        },

    "QUESTION_HUMAN_REVIEW_PENDING":
        {
            "QUESTION_ACCEPTED":
                "EXPECTED_ANSWER_DRAFT_PENDING",

            "QUESTION_REVISE":
                "QUESTION_DRAFT_PENDING",

            "TARGET_REJECTED":
                CELL13_RETARGET_STATE,
        },

    "EXPECTED_ANSWER_DRAFT_PENDING":
        {
            "EXPECTED_ANSWER_REGISTERED":
                "EXPECTED_ANSWER_HUMAN_REVIEW_PENDING",

            "TARGET_REJECTED":
                CELL13_RETARGET_STATE,
        },

    "EXPECTED_ANSWER_HUMAN_REVIEW_PENDING":
        {
            "EXPECTED_ANSWER_ACCEPTED":
                "CITATION_MAPPING_PENDING",

            "EXPECTED_ANSWER_REVISE":
                "EXPECTED_ANSWER_DRAFT_PENDING",

            "TARGET_REJECTED":
                CELL13_RETARGET_STATE,
        },

    "CITATION_MAPPING_PENDING":
        {
            "CITATION_MAPPING_REGISTERED":
                "CITATION_HUMAN_REVIEW_PENDING",

            "TARGET_REJECTED":
                CELL13_RETARGET_STATE,
        },

    "CITATION_HUMAN_REVIEW_PENDING":
        {
            "CITATION_ACCEPTED":
                CELL13_FINAL_PRE_ADVANCE_STATE,

            "CITATION_REVISE":
                "CITATION_MAPPING_PENDING",

            "TARGET_REJECTED":
                CELL13_RETARGET_STATE,
        },

    CELL13_FINAL_PRE_ADVANCE_STATE:
        {},

    CELL13_RETARGET_STATE:
        {},
}


CELL13_ARTIFACT_EVENTS = {
    "QUESTION_DRAFT_REGISTERED":
        "question_draft",

    "EXPECTED_ANSWER_REGISTERED":
        "expected_answer",

    "CITATION_MAPPING_REGISTERED":
        "citation_mapping",
}


CELL13_HUMAN_DECISION_EVENTS = {
    "QUESTION_ACCEPTED",
    "QUESTION_REVISE",
    "EXPECTED_ANSWER_ACCEPTED",
    "EXPECTED_ANSWER_REVISE",
    "CITATION_ACCEPTED",
    "CITATION_REVISE",
    "TARGET_REJECTED",
}


# ============================================================
# 3) Generic helpers
# ============================================================

def cell13_collect_prefixed_ids(
    value,
    result=None,
    seen=None,
):

    if result is None:

        result = {
            "pilot":
                set(),

            "golden_case":
                set(),

            "task_draft":
                set(),

            "target_evidence":
                set(),
        }


    if seen is None:
        seen = set()


    if isinstance(
        value,
        str,
    ):

        if value.startswith(
            "question-pilot:"
        ):

            result[
                "pilot"
            ].add(
                value
            )


        elif value.startswith(
            "golden-case:"
        ):

            result[
                "golden_case"
            ].add(
                value
            )


        elif value.startswith(
            "task-draft:"
        ):

            result[
                "task_draft"
            ].add(
                value
            )


        elif value.startswith(
            "target-evidence:"
        ):

            result[
                "target_evidence"
            ].add(
                value
            )


        return result


    if not isinstance(
        value,
        (
            dict,
            list,
            tuple,
            set,
            frozenset,
        ),
    ):

        return result


    object_id = id(
        value
    )


    if object_id in seen:
        return result


    seen.add(
        object_id
    )


    try:

        if isinstance(
            value,
            dict,
        ):

            for key, nested in (
                value.items()
            ):

                cell13_collect_prefixed_ids(
                    key,
                    result,
                    seen,
                )

                cell13_collect_prefixed_ids(
                    nested,
                    result,
                    seen,
                )


        else:

            for nested in value:

                cell13_collect_prefixed_ids(
                    nested,
                    result,
                    seen,
                )


    finally:

        seen.discard(
            object_id
        )


    return result


def cell13_exact_single_id(
    value,
    family,
):

    ids = (
        cell13_collect_prefixed_ids(
            value
        )[
            family
        ]
    )


    if len(
        ids
    ) != 1:

        raise ValueError(
            (
                f"Expected exactly one {family} ID; "
                f"found {sorted(ids)}"
            )
        )


    return next(
        iter(
            ids
        )
    )


def cell13_value_contains_id(
    value,
    family,
    identity,
):

    return (
        identity
        in
        cell13_collect_prefixed_ids(
            value
        )[
            family
        ]
    )


def cell13_find_unique_mapping_value(
    mapping,
    family,
    identity,
):

    if not isinstance(
        mapping,
        dict,
    ):

        raise TypeError(
            "Expected mapping dictionary."
        )


    if identity in mapping:

        return {
            "lookup_mode":
                "DIRECT_KEY",

            "key":
                identity,

            "value":
                copy.deepcopy(
                    mapping[
                        identity
                    ]
                ),
        }


    matches = []


    for key, value in (
        mapping.items()
    ):

        if cell13_value_contains_id(
            value,
            family,
            identity,
        ):

            matches.append(
                (
                    key,
                    value,
                )
            )


    if len(
        matches
    ) != 1:

        raise ValueError(
            (
                f"Expected exactly one mapping match "
                f"for {identity}; found {len(matches)}."
            )
        )


    key, value = (
        matches[
            0
        ]
    )


    return {
        "lookup_mode":
            "VALUE_ID_SCAN",

        "key":
            copy.deepcopy(
                key
            ),

        "value":
            copy.deepcopy(
                value
            ),
    }


def cell13_find_unique_list_record(
    records,
    field,
    identity,
):

    if not isinstance(
        records,
        list,
    ):

        raise TypeError(
            "Expected record list."
        )


    matches = [
        record

        for record
        in records

        if (
            isinstance(
                record,
                dict,
            )
            and
            record.get(
                field
            )
            ==
            identity
        )
    ]


    if len(
        matches
    ) != 1:

        raise ValueError(
            (
                f"Expected exactly one record where "
                f"{field}={identity}; found {len(matches)}."
            )
        )


    return copy.deepcopy(
        matches[
            0
        ]
    )


# ============================================================
# 4) Freeze independent Seed state
# ============================================================

cell13_seed = copy.deepcopy(
    cell12_seed
)


cell13_seed_before_sha256 = (
    cell10_stable_sha(
        cell13_seed
    )
)


cell13_seed_matches_cell12 = (
    cell13_seed_before_sha256
    ==
    cell11_seed_sha256
)


cell13_final_runtime = (
    cell13_seed.get(
        "final_runtime",
        {},
    )
)


cell13_active_support = (
    cell13_seed.get(
        "active_support",
        {},
    )
)


# ============================================================
# 5) Resolve official current Pilot
# ============================================================

cell13_current_pilot_id = (
    cell13_final_runtime.get(
        "cell95_next_pilot_id"
    )
)


cell13_remaining_ids = (
    cell13_final_runtime.get(
        "cell95_remaining_pilot_ids"
    )
)


cell13_remaining_queue = (
    cell13_final_runtime.get(
        "cell95_remaining_pilot_queue"
    )
)


cell13_next_queue_entry = (
    cell13_final_runtime.get(
        "cell95_next_pilot_queue_entry"
    )
)


cell13_pilot_by_id = (
    cell13_final_runtime.get(
        "question_drafting_pilot_by_id",
        {},
    )
)


cell13_current_pilot_packet = (
    copy.deepcopy(
        cell13_pilot_by_id.get(
            cell13_current_pilot_id
        )
    )

    if isinstance(
        cell13_pilot_by_id,
        dict,
    )

    else None
)


cell13_current_pilot_resolved = all([
    isinstance(
        cell13_current_pilot_id,
        str,
    ),

    isinstance(
        cell13_current_pilot_packet,
        dict,
    ),

    isinstance(
        cell13_remaining_ids,
        list,
    ),

    len(
        cell13_remaining_ids
    )
    ==
    22,

    cell13_remaining_ids[
        0
    ]
    ==
    cell13_current_pilot_id,
])


# ============================================================
# 6) Resolve current Pilot identity chain
# ============================================================

cell13_current_case_id = None
cell13_current_task_id = None
cell13_current_target_id = None


if (
    cell13_current_pilot_resolved
):

    try:

        cell13_current_case_id = (
            cell13_exact_single_id(
                cell13_current_pilot_packet,
                "golden_case",
            )
        )


        cell13_current_task_id = (
            cell13_exact_single_id(
                cell13_current_pilot_packet,
                "task_draft",
            )
        )


        cell13_current_target_id = (
            cell13_exact_single_id(
                cell13_current_pilot_packet,
                "target_evidence",
            )
        )


    except Exception as exc:

        cell13_failures.append({
            "reason":
                "CURRENT_PILOT_ID_CHAIN_RESOLUTION_FAILED",

            "error":
                repr(
                    exc
                ),
        })


cell13_current_identity_chain_valid = all([
    isinstance(
        cell13_current_case_id,
        str,
    ),

    isinstance(
        cell13_current_task_id,
        str,
    ),

    isinstance(
        cell13_current_target_id,
        str,
    ),
])


# ============================================================
# 7) Resolve current Pilot support bundle
# ============================================================

cell13_current_golden_case = None

cell13_current_task_plan = None

cell13_current_task_plan_lookup = None

cell13_current_target_by_case = None

cell13_current_target_by_task = None

cell13_current_target_record = None


if (
    cell13_current_identity_chain_valid
):

    try:

        cell13_current_golden_case = (
            copy.deepcopy(
                cell13_active_support[
                    "golden_case_records_by_id"
                ][
                    cell13_current_case_id
                ]
            )
        )


        cell13_current_task_plan_lookup = (
            cell13_find_unique_mapping_value(
                cell13_active_support.get(
                    "task_draft_plans",
                    {},
                ),
                "task_draft",
                cell13_current_task_id,
            )
        )


        cell13_current_task_plan = (
            cell13_current_task_plan_lookup[
                "value"
            ]
        )


        cell13_current_target_by_case = (
            copy.deepcopy(
                cell13_active_support[
                    "target_evidence_by_case_id"
                ][
                    cell13_current_case_id
                ]
            )
        )


        cell13_current_target_by_task = (
            copy.deepcopy(
                cell13_active_support[
                    "target_evidence_by_task_draft_id"
                ][
                    cell13_current_task_id
                ]
            )
        )


        cell13_current_target_record = (
            cell13_find_unique_list_record(
                cell13_active_support.get(
                    "target_evidence_records",
                    [],
                ),
                "target_evidence_id",
                cell13_current_target_id,
            )
        )


    except Exception as exc:

        cell13_failures.append({
            "reason":
                "CURRENT_PILOT_SUPPORT_RESOLUTION_FAILED",

            "error":
                repr(
                    exc
                ),
        })


# ============================================================
# 8) Validate support cross-links
# ============================================================

cell13_support_crosslinks_valid = all([
    isinstance(
        cell13_current_golden_case,
        dict,
    ),

    isinstance(
        cell13_current_task_plan,
        dict,
    ),

    isinstance(
        cell13_current_target_record,
        dict,
    ),

    cell13_value_contains_id(
        cell13_current_golden_case,
        "golden_case",
        cell13_current_case_id,
    ),

    cell13_value_contains_id(
        cell13_current_task_plan,
        "task_draft",
        cell13_current_task_id,
    ),

    cell13_value_contains_id(
        cell13_current_target_by_case,
        "target_evidence",
        cell13_current_target_id,
    ),

    cell13_value_contains_id(
        cell13_current_target_by_task,
        "target_evidence",
        cell13_current_target_id,
    ),

    cell13_current_target_record.get(
        "target_evidence_id"
    )
    ==
    cell13_current_target_id,
])


# ============================================================
# 9) Build immutable current Pilot bundle
# ============================================================

cell13_current_pilot_bundle = {
    "engine_version":
        CELL13_ENGINE_VERSION,

    "pilot_case_id":
        cell13_current_pilot_id,

    "golden_case_id":
        cell13_current_case_id,

    "task_draft_id":
        cell13_current_task_id,

    "target_evidence_id":
        cell13_current_target_id,

    "queue_position":
        0,

    "remaining_queue_count":
        22,

    "pilot_packet":
        copy.deepcopy(
            cell13_current_pilot_packet
        ),

    "golden_case":
        copy.deepcopy(
            cell13_current_golden_case
        ),

    "task_plan":
        copy.deepcopy(
            cell13_current_task_plan
        ),

    "target_evidence":
        copy.deepcopy(
            cell13_current_target_record
        ),

    "target_by_case":
        copy.deepcopy(
            cell13_current_target_by_case
        ),

    "target_by_task":
        copy.deepcopy(
            cell13_current_target_by_task
        ),

    "full_572_generation":
        CELL13_FULL_572_BLOCK_STATUS,
}


cell13_current_pilot_bundle_sha256 = (
    cell10_stable_sha(
        cell13_current_pilot_bundle
    )
)


# ============================================================
# 10) Reusable session builder
# ============================================================

def cell13_build_session(
    pilot_bundle,
):

    session = {
        "engine_version":
            CELL13_ENGINE_VERSION,

        "pilot_case_id":
            pilot_bundle[
                "pilot_case_id"
            ],

        "golden_case_id":
            pilot_bundle[
                "golden_case_id"
            ],

        "task_draft_id":
            pilot_bundle[
                "task_draft_id"
            ],

        "target_evidence_id":
            pilot_bundle[
                "target_evidence_id"
            ],

        "current_state":
            CELL13_INITIAL_STATE,

        "artifacts":
            {
                "question_draft":
                    None,

                "expected_answer":
                    None,

                "citation_mapping":
                    None,
            },

        "human_decisions":
            [],

        "event_log":
            [],

        "queue_advance_permitted":
            False,

        "queue_advance_performed":
            False,

        "full_572_generation":
            CELL13_FULL_572_BLOCK_STATUS,
    }


    session[
        "session_sha256"
    ] = cell10_stable_sha({
        key:
            value

        for key, value
        in session.items()

        if key
        !=
        "session_sha256"
    })


    return session


# ============================================================
# 11) Pure transition function
# ============================================================

def cell13_apply_event(
    session,
    event,
    artifact=None,
    human_note=None,
):

    if not isinstance(
        session,
        dict,
    ):

        raise TypeError(
            "Session must be a dictionary."
        )


    next_session = copy.deepcopy(
        session
    )


    current_state = (
        next_session.get(
            "current_state"
        )
    )


    allowed_events = (
        CELL13_TRANSITIONS.get(
            current_state
        )
    )


    if allowed_events is None:

        raise ValueError(
            (
                "Unknown current workflow state: "
                f"{current_state}"
            )
        )


    if event not in allowed_events:

        raise ValueError(
            (
                f"Event {event} is not allowed from "
                f"state {current_state}."
            )
        )


    if event in (
        CELL13_ARTIFACT_EVENTS
    ):

        if not isinstance(
            artifact,
            dict,
        ):

            raise ValueError(
                (
                    f"{event} requires a dictionary "
                    "artifact payload."
                )
            )


        artifact_slot = (
            CELL13_ARTIFACT_EVENTS[
                event
            ]
        )


        artifact_copy = copy.deepcopy(
            artifact
        )


        artifact_copy[
            "artifact_sha256"
        ] = cell10_stable_sha(
            artifact_copy
        )


        next_session[
            "artifacts"
        ][
            artifact_slot
        ] = artifact_copy


    if event in (
        CELL13_HUMAN_DECISION_EVENTS
    ):

        if not (
            isinstance(
                human_note,
                str,
            )
            and
            human_note.strip()
        ):

            raise ValueError(
                (
                    f"{event} requires a non-empty "
                    "human_note."
                )
            )


        next_session[
            "human_decisions"
        ].append({
            "event":
                event,

            "note":
                human_note.strip(),

            "prior_state":
                current_state,
        })


    new_state = (
        allowed_events[
            event
        ]
    )


    next_session[
        "event_log"
    ].append({
        "event":
            event,

        "prior_state":
            current_state,

        "new_state":
            new_state,
    })


    next_session[
        "current_state"
    ] = new_state


    next_session[
        "queue_advance_permitted"
    ] = (
        new_state
        ==
        CELL13_FINAL_PRE_ADVANCE_STATE
    )


    next_session[
        "queue_advance_performed"
    ] = False


    next_session[
        "full_572_generation"
    ] = (
        CELL13_FULL_572_BLOCK_STATUS
    )


    next_session[
        "session_sha256"
    ] = cell10_stable_sha({
        key:
            value

        for key, value
        in next_session.items()

        if key
        !=
        "session_sha256"
    })


    return next_session


# ============================================================
# 12) Create actual current session
#
# This is the real continuation starting state.
# No content is generated and no event is applied here.
# ============================================================

cell13_current_session = (
    cell13_build_session(
        cell13_current_pilot_bundle
    )
)


cell13_current_session_initial_valid = all([
    cell13_current_session[
        "current_state"
    ]
    ==
    CELL13_INITIAL_STATE,

    cell13_current_session[
        "pilot_case_id"
    ]
    ==
    cell13_current_pilot_id,

    cell13_current_session[
        "queue_advance_permitted"
    ]
    is False,

    cell13_current_session[
        "queue_advance_performed"
    ]
    is False,

    all(
        value
        is None

        for value
        in cell13_current_session[
            "artifacts"
        ].values()
    ),

    len(
        cell13_current_session[
            "human_decisions"
        ]
    )
    ==
    0,

    len(
        cell13_current_session[
            "event_log"
        ]
    )
    ==
    0,

    cell13_current_session[
        "full_572_generation"
    ]
    ==
    CELL13_FULL_572_BLOCK_STATUS,
])


# ============================================================
# 13) Engine dry-run artifacts
#
# These are explicitly synthetic and are used only on copies.
# ============================================================

CELL13_DRY_RUN_QUESTION = {
    "test_mode":
        "ENGINE_DRY_RUN_ONLY",

    "artifact_type":
        "question_draft",

    "text":
        "synthetic-question-placeholder",
}


CELL13_DRY_RUN_ANSWER = {
    "test_mode":
        "ENGINE_DRY_RUN_ONLY",

    "artifact_type":
        "expected_answer",

    "text":
        "synthetic-answer-placeholder",
}


CELL13_DRY_RUN_CITATION = {
    "test_mode":
        "ENGINE_DRY_RUN_ONLY",

    "artifact_type":
        "citation_mapping",

    "target_evidence_id":
        cell13_current_target_id,
}


# ============================================================
# 14) Happy-path dry run
# ============================================================

cell13_happy_path_session = copy.deepcopy(
    cell13_current_session
)


cell13_happy_path_session = (
    cell13_apply_event(
        cell13_happy_path_session,
        "QUESTION_DRAFT_REGISTERED",
        artifact=
            CELL13_DRY_RUN_QUESTION,
    )
)


cell13_happy_path_session = (
    cell13_apply_event(
        cell13_happy_path_session,
        "QUESTION_ACCEPTED",
        human_note=
            "ENGINE_DRY_RUN_ONLY: question accepted.",
    )
)


cell13_happy_path_session = (
    cell13_apply_event(
        cell13_happy_path_session,
        "EXPECTED_ANSWER_REGISTERED",
        artifact=
            CELL13_DRY_RUN_ANSWER,
    )
)


cell13_happy_path_session = (
    cell13_apply_event(
        cell13_happy_path_session,
        "EXPECTED_ANSWER_ACCEPTED",
        human_note=
            "ENGINE_DRY_RUN_ONLY: answer accepted.",
    )
)


cell13_happy_path_session = (
    cell13_apply_event(
        cell13_happy_path_session,
        "CITATION_MAPPING_REGISTERED",
        artifact=
            CELL13_DRY_RUN_CITATION,
    )
)


cell13_happy_path_session = (
    cell13_apply_event(
        cell13_happy_path_session,
        "CITATION_ACCEPTED",
        human_note=
            "ENGINE_DRY_RUN_ONLY: citation accepted.",
    )
)


cell13_happy_path_valid = all([
    cell13_happy_path_session[
        "current_state"
    ]
    ==
    CELL13_FINAL_PRE_ADVANCE_STATE,

    cell13_happy_path_session[
        "queue_advance_permitted"
    ]
    is True,

    cell13_happy_path_session[
        "queue_advance_performed"
    ]
    is False,

    len(
        cell13_happy_path_session[
            "event_log"
        ]
    )
    ==
    6,

    len(
        cell13_happy_path_session[
            "human_decisions"
        ]
    )
    ==
    3,

    all(
        isinstance(
            value,
            dict,
        )

        for value
        in cell13_happy_path_session[
            "artifacts"
        ].values()
    ),

    cell13_happy_path_session[
        "full_572_generation"
    ]
    ==
    CELL13_FULL_572_BLOCK_STATUS,
])


# ============================================================
# 15) Question revision-loop dry run
# ============================================================

cell13_revision_session = (
    cell13_build_session(
        cell13_current_pilot_bundle
    )
)


cell13_revision_session = (
    cell13_apply_event(
        cell13_revision_session,
        "QUESTION_DRAFT_REGISTERED",
        artifact=
            CELL13_DRY_RUN_QUESTION,
    )
)


cell13_revision_session = (
    cell13_apply_event(
        cell13_revision_session,
        "QUESTION_REVISE",
        human_note=
            "ENGINE_DRY_RUN_ONLY: revise question.",
    )
)


cell13_revision_loop_valid = all([
    cell13_revision_session[
        "current_state"
    ]
    ==
    "QUESTION_DRAFT_PENDING",

    cell13_revision_session[
        "queue_advance_permitted"
    ]
    is False,

    cell13_revision_session[
        "queue_advance_performed"
    ]
    is False,

    cell13_revision_session[
        "full_572_generation"
    ]
    ==
    CELL13_FULL_572_BLOCK_STATUS,
])


# ============================================================
# 16) Retarget dry run
# ============================================================

cell13_retarget_session = (
    cell13_build_session(
        cell13_current_pilot_bundle
    )
)


cell13_retarget_session = (
    cell13_apply_event(
        cell13_retarget_session,
        "TARGET_REJECTED",
        human_note=
            "ENGINE_DRY_RUN_ONLY: target rejected.",
    )
)


cell13_retarget_path_valid = all([
    cell13_retarget_session[
        "current_state"
    ]
    ==
    CELL13_RETARGET_STATE,

    cell13_retarget_session[
        "queue_advance_permitted"
    ]
    is False,

    cell13_retarget_session[
        "queue_advance_performed"
    ]
    is False,

    cell13_retarget_session[
        "full_572_generation"
    ]
    ==
    CELL13_FULL_572_BLOCK_STATUS,
])


# ============================================================
# 17) Invalid transition rejection probe
# ============================================================

cell13_invalid_transition_rejected = False

cell13_invalid_transition_error = None


try:

    cell13_apply_event(
        cell13_current_session,
        "QUESTION_ACCEPTED",
        human_note=
            "ENGINE_DRY_RUN_ONLY invalid order probe.",
    )


except ValueError as exc:

    cell13_invalid_transition_rejected = True

    cell13_invalid_transition_error = (
        str(
            exc
        )
    )


# ============================================================
# 18) Missing human decision rejection probe
# ============================================================

cell13_missing_human_note_rejected = False

cell13_missing_human_note_error = None


try:

    cell13_review_probe = (
        cell13_apply_event(
            cell13_current_session,
            "QUESTION_DRAFT_REGISTERED",
            artifact=
                CELL13_DRY_RUN_QUESTION,
        )
    )


    cell13_apply_event(
        cell13_review_probe,
        "QUESTION_ACCEPTED",
    )


except ValueError as exc:

    cell13_missing_human_note_rejected = True

    cell13_missing_human_note_error = (
        str(
            exc
        )
    )


# ============================================================
# 19) Missing artifact rejection probe
# ============================================================

cell13_missing_artifact_rejected = False

cell13_missing_artifact_error = None


try:

    cell13_apply_event(
        cell13_current_session,
        "QUESTION_DRAFT_REGISTERED",
    )


except ValueError as exc:

    cell13_missing_artifact_rejected = True

    cell13_missing_artifact_error = (
        str(
            exc
        )
    )


# ============================================================
# 20) Confirm dry runs did NOT mutate actual current session
# ============================================================

cell13_actual_session_still_initial = all([
    cell13_current_session[
        "current_state"
    ]
    ==
    CELL13_INITIAL_STATE,

    cell13_current_session[
        "queue_advance_permitted"
    ]
    is False,

    cell13_current_session[
        "queue_advance_performed"
    ]
    is False,

    len(
        cell13_current_session[
            "event_log"
        ]
    )
    ==
    0,

    len(
        cell13_current_session[
            "human_decisions"
        ]
    )
    ==
    0,

    all(
        value
        is None

        for value
        in cell13_current_session[
            "artifacts"
        ].values()
    ),
])


# ============================================================
# 21) Confirm queue did NOT advance
# ============================================================

cell13_queue_after_dry_runs = (
    cell13_seed[
        "final_runtime"
    ][
        "cell95_remaining_pilot_ids"
    ]
)


cell13_queue_not_advanced = all([
    len(
        cell13_queue_after_dry_runs
    )
    ==
    22,

    cell13_queue_after_dry_runs[
        0
    ]
    ==
    cell13_current_pilot_id,

    cell13_seed[
        "final_runtime"
    ][
        "cell95_next_pilot_id"
    ]
    ==
    cell13_current_pilot_id,
])


# ============================================================
# 22) Engine contract artifact
# ============================================================

cell13_engine_contract = {
    "engine_version":
        CELL13_ENGINE_VERSION,

    "initial_state":
        CELL13_INITIAL_STATE,

    "final_pre_advance_state":
        CELL13_FINAL_PRE_ADVANCE_STATE,

    "retarget_state":
        CELL13_RETARGET_STATE,

    "transitions":
        copy.deepcopy(
            CELL13_TRANSITIONS
        ),

    "artifact_events":
        copy.deepcopy(
            CELL13_ARTIFACT_EVENTS
        ),

    "human_decision_events":
        sorted(
            CELL13_HUMAN_DECISION_EVENTS
        ),

    "rules":
        {
            "human_review_required_for_question":
                True,

            "human_review_required_for_expected_answer":
                True,

            "human_review_required_for_citation":
                True,

            "automatic_queue_advance":
                False,

            "queue_advance_only_after_final_citation_acceptance":
                True,

            "full_572_generation":
                CELL13_FULL_572_BLOCK_STATUS,
        },
}


cell13_engine_contract_sha256 = (
    cell10_stable_sha(
        cell13_engine_contract
    )
)


# ============================================================
# 23) Seed immutability
# ============================================================

cell13_seed_after_sha256 = (
    cell10_stable_sha(
        cell13_seed
    )
)


cell13_seed_unchanged = (
    cell13_seed_before_sha256
    ==
    cell13_seed_after_sha256
)


# ============================================================
# 24) Full 572 remains blocked
# ============================================================

cell13_full_572_blocked = all([
    cell13_seed.get(
        "source",
        {},
    ).get(
        "full_572_generation"
    )
    ==
    CELL13_FULL_572_BLOCK_STATUS,

    cell13_current_session[
        "full_572_generation"
    ]
    ==
    CELL13_FULL_572_BLOCK_STATUS,

    cell13_engine_contract[
        "rules"
    ][
        "full_572_generation"
    ]
    ==
    CELL13_FULL_572_BLOCK_STATUS,
])


# ============================================================
# 25) Final Gate
# ============================================================

cell13_gate = all([
    len(
        cell13_failures
    )
    ==
    0,

    cell13_seed_matches_cell12,

    cell13_current_pilot_resolved,

    cell13_current_identity_chain_valid,

    cell13_support_crosslinks_valid,

    cell13_current_session_initial_valid,

    cell13_happy_path_valid,

    cell13_revision_loop_valid,

    cell13_retarget_path_valid,

    cell13_invalid_transition_rejected,

    cell13_missing_human_note_rejected,

    cell13_missing_artifact_rejected,

    cell13_actual_session_still_initial,

    cell13_queue_not_advanced,

    cell13_seed_unchanged,

    cell13_full_572_blocked,
])


# ============================================================
# 26) Diagnostics
# ============================================================

print(
    "=== REUSABLE PILOT CONTINUATION ENGINE ==="
)


print(
    "\n=== CURRENT PILOT ==="
)

print(
    "Pilot:",
    cell13_current_pilot_id,
)

print(
    "Golden Case:",
    cell13_current_case_id,
)

print(
    "Task Draft:",
    cell13_current_task_id,
)

print(
    "Target Evidence:",
    cell13_current_target_id,
)

print(
    "Remaining queue:",
    len(
        cell13_remaining_ids
    ),
)

print(
    "Current Pilot resolved:",
    cell13_current_pilot_resolved,
)

print(
    "Support cross-links valid:",
    cell13_support_crosslinks_valid,
)

print(
    "Pilot bundle SHA256:",
    cell13_current_pilot_bundle_sha256,
)


print(
    "\n=== ENGINE ==="
)

print(
    "Version:",
    CELL13_ENGINE_VERSION,
)

print(
    "Initial state:",
    CELL13_INITIAL_STATE,
)

print(
    "Final pre-advance state:",
    CELL13_FINAL_PRE_ADVANCE_STATE,
)

print(
    "Retarget state:",
    CELL13_RETARGET_STATE,
)

print(
    "Engine contract SHA256:",
    cell13_engine_contract_sha256,
)


print(
    "\n=== HUMAN GATES ==="
)

print(
    "Question review required:",
    True,
)

print(
    "Expected Answer review required:",
    True,
)

print(
    "Citation review required:",
    True,
)

print(
    "Automatic queue advance:",
    False,
)


print(
    "\n=== DRY-RUN VALIDATION ==="
)

print(
    "Happy path:",
    cell13_happy_path_valid,
)

print(
    "Happy-path final state:",
    cell13_happy_path_session[
        "current_state"
    ],
)

print(
    "Revision loop:",
    cell13_revision_loop_valid,
)

print(
    "Retarget path:",
    cell13_retarget_path_valid,
)

print(
    "Invalid transition rejected:",
    cell13_invalid_transition_rejected,
)

print(
    "Missing human note rejected:",
    cell13_missing_human_note_rejected,
)

print(
    "Missing artifact rejected:",
    cell13_missing_artifact_rejected,
)


print(
    "\n=== ACTUAL CONTINUATION STATE ==="
)

print(
    "Current state:",
    cell13_current_session[
        "current_state"
    ],
)

print(
    "Question artifact:",
    cell13_current_session[
        "artifacts"
    ][
        "question_draft"
    ],
)

print(
    "Expected Answer artifact:",
    cell13_current_session[
        "artifacts"
    ][
        "expected_answer"
    ],
)

print(
    "Citation artifact:",
    cell13_current_session[
        "artifacts"
    ][
        "citation_mapping"
    ],
)

print(
    "Human decisions:",
    len(
        cell13_current_session[
            "human_decisions"
        ]
    ),
)

print(
    "Queue advance permitted:",
    cell13_current_session[
        "queue_advance_permitted"
    ],
)

print(
    "Queue advance performed:",
    cell13_current_session[
        "queue_advance_performed"
    ],
)


print(
    "\n=== IMMUTABILITY ==="
)

print(
    "Seed before SHA256:",
    cell13_seed_before_sha256,
)

print(
    "Seed after SHA256:",
    cell13_seed_after_sha256,
)

print(
    "Seed unchanged:",
    cell13_seed_unchanged,
)

print(
    "Actual session still initial:",
    cell13_actual_session_still_initial,
)

print(
    "Queue still 22:",
    len(
        cell13_queue_after_dry_runs
    )
    ==
    22,
)

print(
    "Next Pilot unchanged:",
    cell13_queue_after_dry_runs[
        0
    ]
    ==
    cell13_current_pilot_id,
)


print(
    "\n=== WORKFLOW ==="
)

print(
    "Full 572 blocked:",
    cell13_full_572_blocked,
)

print(
    "Project/file mutation performed:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell13_failures
)


print(
    "\nCODE CELL 13 GATE:",
    (
        "PASS"
        if cell13_gate
        else "FAIL"
    ),
)


assert cell13_gate, (
    "Reusable Pilot continuation engine validation failed. "
    "Repair this SAME Code Cell 13 before preparing the "
    "real replacement-Pilot Question Draft workspace."
)

=== REUSABLE PILOT CONTINUATION ENGINE ===

=== CURRENT PILOT ===
Pilot: question-pilot:3005b48518a7aeb532234332df23c952
Golden Case: golden-case:3b93ce20ef4b617b9f922baddfbd1e0f
Task Draft: task-draft:f48d78f0a6a588de906ca8534219a604
Target Evidence: target-evidence:81011a6ef78817e01e3ed484b1b4c368
Remaining queue: 22
Current Pilot resolved: True
Support cross-links valid: True
Pilot bundle SHA256: aeebd98017451f66c493278dfb9d492a7a13b9c3d522e2213ff70d4de63344a6

=== ENGINE ===
Version: pilot_continuation_engine_v1
Initial state: QUESTION_DRAFT_PENDING
Final pre-advance state: PILOT_COMPLETE_PENDING_QUEUE_ADVANCE
Retarget state: BLOCKED_REQUIRES_RETARGET
Engine contract SHA256: a012ff236a7af27b9770d44d789560b0d830567a8d616d75dca64d2259699a00

=== HUMAN GATES ===
Question review required: True
Expected Answer review required: True
Citation review required: True
Automatic queue advance: False

=== DRY-RUN VALIDATION ===
Happy path: True
Happy-path final state: PILOT_COMPLETE_PENDING_QUE

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 14 — آماده‌سازی Workspace واقعی برای Question Draftِ Pilot جایگزین

موتور continuation در Cell 13 با موفقیت اعتبارسنجی شد و Runtime واقعی همچنان در حالت `QUESTION_DRAFT_PENDING` قرار دارد.

هدف این Cell فقط آماده‌کردن Workspace قابل‌خواندن برای Pilot فعلی است تا سؤال واقعی مستقیماً از Target Evidence و قراردادهای پذیرفته‌شده نوشته شود.

در این مرحله موارد زیر استخراج و نمایش داده می‌شوند:

- هویت دقیق Pilot، Golden Case، Task Draft و Target Evidence؛
- متن واقعی Target Evidence و Target Text Partها؛
- اطلاعات مرتبط Golden Case و Task Plan؛
- Source Type و Stratum مربوط به Pilot؛
- Question Archetype و Constraintهای قابل‌بازیابی؛
- Acceptance Rubric و Source-Opacity Rule؛
- وضعیت فعلی Workflow و تأیید اینکه هنوز هیچ Question Artifact ثبت نشده است.

این Cell فقط Workspace می‌سازد.

هیچ Question واقعی ثبت نمی‌شود، هیچ Human Decision اعمال نمی‌شود، Expected Answer و Citation ساخته نمی‌شوند، Queue جلو نمی‌رود و Full 572 Generation همچنان Blocked می‌ماند.

هیچ فایل، Commit، Push، PR یا Artifact خارجی ایجاد یا تغییر داده نمی‌شود.

</div>

In [14]:
# Code Cell 14

cell14_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if (
    globals().get(
        "cell13_gate"
    )
    is not True
):
    cell14_failures.append(
        "CELL13_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell13_current_pilot_bundle"
    ),
    dict,
):
    cell14_failures.append(
        "CURRENT_PILOT_BUNDLE_MISSING"
    )


if not isinstance(
    globals().get(
        "cell13_current_session"
    ),
    dict,
):
    cell14_failures.append(
        "CURRENT_SESSION_MISSING"
    )


# ============================================================
# 2) Freeze accepted current state
# ============================================================

cell14_seed_before_sha256 = (
    cell10_stable_sha(
        cell13_seed
    )
)


cell14_session_before_sha256 = (
    cell10_stable_sha(
        cell13_current_session
    )
)


cell14_bundle = copy.deepcopy(
    cell13_current_pilot_bundle
)


cell14_session = copy.deepcopy(
    cell13_current_session
)


# ============================================================
# 3) Expected current identity
# ============================================================

CELL14_EXPECTED_PILOT_ID = (
    "question-pilot:"
    "3005b48518a7aeb532234332df23c952"
)


CELL14_EXPECTED_GOLDEN_CASE_ID = (
    "golden-case:"
    "3b93ce20ef4b617b9f922baddfbd1e0f"
)


CELL14_EXPECTED_TASK_DRAFT_ID = (
    "task-draft:"
    "f48d78f0a6a588de906ca8534219a604"
)


CELL14_EXPECTED_TARGET_EVIDENCE_ID = (
    "target-evidence:"
    "81011a6ef78817e01e3ed484b1b4c368"
)


CELL14_FULL_572_BLOCK_STATUS = (
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


# ============================================================
# 4) Local inspection helpers
# ============================================================

def cell14_clip(
    value,
    limit=4000,
):

    if value is None:
        return None


    if not isinstance(
        value,
        str,
    ):
        value = str(
            value
        )


    if len(
        value
    ) <= limit:
        return value


    return (
        value[
            :limit
        ]
        +
        "\n...[CLIPPED]..."
    )


def cell14_walk(
    value,
    path="root",
    seen=None,
):

    if seen is None:
        seen = set()


    if isinstance(
        value,
        dict,
    ):

        object_id = id(
            value
        )


        if object_id in seen:
            return


        seen.add(
            object_id
        )


        try:

            for key, nested in (
                value.items()
            ):

                nested_path = (
                    f"{path}.{key}"
                )


                yield (
                    nested_path,
                    key,
                    nested,
                )


                yield from cell14_walk(
                    nested,
                    nested_path,
                    seen,
                )


        finally:

            seen.discard(
                object_id
            )


    elif isinstance(
        value,
        (
            list,
            tuple,
        ),
    ):

        object_id = id(
            value
        )


        if object_id in seen:
            return


        seen.add(
            object_id
        )


        try:

            for index, nested in enumerate(
                value
            ):

                nested_path = (
                    f"{path}[{index}]"
                )


                yield (
                    nested_path,
                    index,
                    nested,
                )


                yield from cell14_walk(
                    nested,
                    nested_path,
                    seen,
                )


        finally:

            seen.discard(
                object_id
            )


def cell14_find_scalar_fields(
    value,
    search_terms,
):

    rows = []


    lowered_terms = [
        term.lower()
        for term
        in search_terms
    ]


    for path, key, nested in (
        cell14_walk(
            value
        )
    ):

        if not isinstance(
            key,
            str,
        ):
            continue


        lowered_key = (
            key.lower()
        )


        if not any(
            term
            in
            lowered_key

            for term
            in lowered_terms
        ):
            continue


        if isinstance(
            nested,
            (
                str,
                int,
                float,
                bool,
            ),
        ) or nested is None:

            rows.append({
                "path":
                    path,

                "key":
                    key,

                "value":
                    nested,
            })


    return rows


def cell14_find_first_scalar(
    value,
    preferred_keys,
):

    preferred_keys = [
        key.lower()
        for key
        in preferred_keys
    ]


    matches = []


    for path, key, nested in (
        cell14_walk(
            value
        )
    ):

        if not isinstance(
            key,
            str,
        ):
            continue


        if (
            key.lower()
            not in
            preferred_keys
        ):
            continue


        if isinstance(
            nested,
            (
                str,
                int,
                float,
                bool,
            ),
        ):

            matches.append(
                (
                    path,
                    nested,
                )
            )


    if not matches:
        return None


    return {
        "path":
            matches[
                0
            ][
                0
            ],

        "value":
            matches[
                0
            ][
                1
            ],
    }


def cell14_json_preview(
    value,
    limit=12000,
):

    try:

        text = json.dumps(
            value,
            ensure_ascii=False,
            sort_keys=True,
            indent=2,
            default=str,
        )

    except Exception:

        text = repr(
            value
        )


    return cell14_clip(
        text,
        limit,
    )


# ============================================================
# 5) Resolve bundle components
# ============================================================

cell14_pilot_packet = (
    copy.deepcopy(
        cell14_bundle.get(
            "pilot_packet"
        )
    )
)


cell14_golden_case = (
    copy.deepcopy(
        cell14_bundle.get(
            "golden_case"
        )
    )
)


cell14_task_plan = (
    copy.deepcopy(
        cell14_bundle.get(
            "task_plan"
        )
    )
)


cell14_target_evidence = (
    copy.deepcopy(
        cell14_bundle.get(
            "target_evidence"
        )
    )
)


cell14_target_by_case = (
    copy.deepcopy(
        cell14_bundle.get(
            "target_by_case"
        )
    )
)


cell14_target_by_task = (
    copy.deepcopy(
        cell14_bundle.get(
            "target_by_task"
        )
    )
)


# ============================================================
# 6) Exact identity gate
# ============================================================

cell14_identity_valid = all([
    cell14_bundle.get(
        "pilot_case_id"
    )
    ==
    CELL14_EXPECTED_PILOT_ID,

    cell14_bundle.get(
        "golden_case_id"
    )
    ==
    CELL14_EXPECTED_GOLDEN_CASE_ID,

    cell14_bundle.get(
        "task_draft_id"
    )
    ==
    CELL14_EXPECTED_TASK_DRAFT_ID,

    cell14_bundle.get(
        "target_evidence_id"
    )
    ==
    CELL14_EXPECTED_TARGET_EVIDENCE_ID,
])


# ============================================================
# 7) Resolve Source Type / Stratum
# ============================================================

cell14_source_type_match = (
    cell14_find_first_scalar(
        cell14_pilot_packet,
        [
            "source_type",
            "source_kind",
            "source",
        ],
    )
)


cell14_stratum_match = (
    cell14_find_first_scalar(
        cell14_pilot_packet,
        [
            "stratum",
            "stratum_name",
            "stratum_id",
        ],
    )
)


cell14_source_type = (
    cell14_source_type_match[
        "value"
    ]

    if isinstance(
        cell14_source_type_match,
        dict,
    )

    else None
)


cell14_stratum = (
    cell14_stratum_match[
        "value"
    ]

    if isinstance(
        cell14_stratum_match,
        dict,
    )

    else None
)


# ============================================================
# 8) Resolve real Target Text Parts
# ============================================================

cell14_target_text_parts = []


if isinstance(
    cell14_pilot_packet,
    dict,
):

    packet_parts = (
        cell14_pilot_packet.get(
            "target_text_parts"
        )
    )


    if isinstance(
        packet_parts,
        list,
    ):

        cell14_target_text_parts = (
            copy.deepcopy(
                packet_parts
            )
        )


if not (
    cell14_target_text_parts
):

    for path, key, nested in (
        cell14_walk(
            cell14_pilot_packet
        )
    ):

        if (
            key
            ==
            "target_text_parts"
            and
            isinstance(
                nested,
                list,
            )
        ):

            cell14_target_text_parts = (
                copy.deepcopy(
                    nested
                )
            )

            break


cell14_target_texts = []


for index, part in enumerate(
    cell14_target_text_parts
):

    if not isinstance(
        part,
        dict,
    ):
        continue


    text_value = None


    for field_name in (
        "text",
        "target_text",
        "content",
        "evidence_text",
        "text_content",
    ):

        candidate = (
            part.get(
                field_name
            )
        )


        if (
            isinstance(
                candidate,
                str,
            )
            and
            candidate.strip()
        ):

            text_value = (
                candidate.strip()
            )

            break


    if text_value is None:
        continue


    cell14_target_texts.append({
        "part_index":
            index,

        "document_id":
            part.get(
                "document_id"
            ),

        "role":
            part.get(
                "role"
            ),

        "char_start":
            part.get(
                "char_start"
            ),

        "char_end":
            part.get(
                "char_end"
            ),

        "text":
            text_value,

        "text_sha256":
            hashlib.sha256(
                text_value.encode(
                    "utf-8"
                )
            ).hexdigest(),
    })


cell14_combined_target_text = (
    "\n\n".join(
        row[
            "text"
        ]

        for row
        in cell14_target_texts
    )
)


cell14_target_text_available = (
    isinstance(
        cell14_combined_target_text,
        str,
    )
    and
    bool(
        cell14_combined_target_text.strip()
    )
)


# ============================================================
# 9) Evidence metadata useful for authoring
# ============================================================

cell14_evidence_metadata = (
    cell14_find_scalar_fields(
        cell14_target_evidence,
        [
            "document",
            "source",
            "title",
            "section",
            "heading",
            "author",
            "url",
            "char_",
            "token",
            "role",
            "status",
        ],
    )
)


cell14_task_metadata = (
    cell14_find_scalar_fields(
        cell14_task_plan,
        [
            "task",
            "intent",
            "question",
            "archetype",
            "difficulty",
            "instruction",
            "constraint",
            "status",
            "source",
            "stratum",
        ],
    )
)


cell14_golden_metadata = (
    cell14_find_scalar_fields(
        cell14_golden_case,
        [
            "case",
            "task",
            "target",
            "source",
            "stratum",
            "archetype",
            "status",
            "difficulty",
        ],
    )
)


# ============================================================
# 10) Resolve Question-related contracts from compact Seed
# ============================================================

cell14_contracts = (
    cell13_seed.get(
        "contracts",
        {},
    )
)


CELL14_QUESTION_CONTRACT_NAMES = [
    "PILOT_QA_WORKFLOW_CONTRACT_V1",
    "QUESTION_DRAFT_RECORD_SCHEMA_V1",
    "QUESTION_DRAFT_ID_CONTRACT_V1",
    "QUESTION_DRAFT_ID_METHOD_VERSION",
    "QUESTION_DRAFT_ACCEPTANCE_RUBRIC_V1",
    "PILOT_QUESTION_AUTHORING_MODE_V1",
    "QUESTION_USER_FACING_SOURCE_OPACITY_RULE_V1",
    "PILOT_QUESTION_DRAFTING_CONTRACT_V1_DRAFT",
    "PILOT_QUESTION_DRAFTING_FOUR_COMPONENT_CONTRACT_V1",
    "QUESTION_ARCHETYPES_V1",
    "QUESTION_ARCHETYPE_BY_STRATUM",
    "QUESTION_CONSTRUCTION_GUARDRAILS_V1_DRAFT",
    "QUESTION_CONSTRUCTION_METHOD_V1_DRAFT",
    "STRATUM_QUESTION_CONSTRAINTS_V1_DRAFT",
]


cell14_question_contracts = {
    name:
        copy.deepcopy(
            cell14_contracts[
                name
            ]
        )

    for name
    in CELL14_QUESTION_CONTRACT_NAMES

    if name
    in
    cell14_contracts
}


# ============================================================
# 11) Resolve stratum-specific contract branches when possible
# ============================================================

cell14_stratum_contract_matches = {}


if (
    isinstance(
        cell14_stratum,
        str,
    )
    and
    cell14_stratum.strip()
):

    for contract_name in (
        "QUESTION_ARCHETYPE_BY_STRATUM",
        "STRATUM_QUESTION_CONSTRAINTS_V1_DRAFT",
    ):

        contract_value = (
            cell14_question_contracts.get(
                contract_name
            )
        )


        if not isinstance(
            contract_value,
            dict,
        ):
            continue


        if (
            cell14_stratum
            in
            contract_value
        ):

            cell14_stratum_contract_matches[
                contract_name
            ] = copy.deepcopy(
                contract_value[
                    cell14_stratum
                ]
            )

            continue


        for key, value in (
            contract_value.items()
        ):

            if (
                isinstance(
                    key,
                    str,
                )
                and
                key.lower()
                ==
                cell14_stratum.lower()
            ):

                cell14_stratum_contract_matches[
                    contract_name
                ] = copy.deepcopy(
                    value
                )

                break


# ============================================================
# 12) Build authoring constraints bundle
# ============================================================

cell14_authoring_constraints = {
    "source_type":
        cell14_source_type,

    "stratum":
        cell14_stratum,

    "stratum_specific":
        cell14_stratum_contract_matches,

    "authoring_mode":
        copy.deepcopy(
            cell14_question_contracts.get(
                "PILOT_QUESTION_AUTHORING_MODE_V1"
            )
        ),

    "source_opacity_rule":
        copy.deepcopy(
            cell14_question_contracts.get(
                "QUESTION_USER_FACING_SOURCE_OPACITY_RULE_V1"
            )
        ),

    "acceptance_rubric":
        copy.deepcopy(
            cell14_question_contracts.get(
                "QUESTION_DRAFT_ACCEPTANCE_RUBRIC_V1"
            )
        ),

    "construction_guardrails":
        copy.deepcopy(
            cell14_question_contracts.get(
                "QUESTION_CONSTRUCTION_GUARDRAILS_V1_DRAFT"
            )
        ),

    "four_component_contract":
        copy.deepcopy(
            cell14_question_contracts.get(
                "PILOT_QUESTION_DRAFTING_FOUR_COMPONENT_CONTRACT_V1"
            )
        ),
}


cell14_authoring_constraints_sha256 = (
    cell10_stable_sha(
        cell14_authoring_constraints
    )
)


# ============================================================
# 13) Build REAL Question Authoring Workspace
#
# No Question text is generated or registered here.
# ============================================================

cell14_question_authoring_workspace = {
    "workspace_version":
        "replacement_pilot_question_authoring_workspace_v1",

    "status":
        "READY_FOR_REAL_QUESTION_AUTHORING",

    "pilot_case_id":
        CELL14_EXPECTED_PILOT_ID,

    "golden_case_id":
        CELL14_EXPECTED_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL14_EXPECTED_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL14_EXPECTED_TARGET_EVIDENCE_ID,

    "source_type":
        cell14_source_type,

    "stratum":
        cell14_stratum,

    "target_text_parts":
        copy.deepcopy(
            cell14_target_texts
        ),

    "combined_target_text":
        cell14_combined_target_text,

    "golden_case":
        copy.deepcopy(
            cell14_golden_case
        ),

    "task_plan":
        copy.deepcopy(
            cell14_task_plan
        ),

    "target_evidence":
        copy.deepcopy(
            cell14_target_evidence
        ),

    "authoring_constraints":
        copy.deepcopy(
            cell14_authoring_constraints
        ),

    "question_draft":
        None,

    "question_draft_id":
        None,

    "human_question_decision":
        None,

    "expected_answer":
        None,

    "citation_mapping":
        None,

    "workflow_state":
        "QUESTION_DRAFT_PENDING",

    "queue_advance_permitted":
        False,

    "full_572_generation":
        CELL14_FULL_572_BLOCK_STATUS,
}


cell14_question_authoring_workspace_sha256 = (
    cell10_stable_sha(
        cell14_question_authoring_workspace
    )
)


# ============================================================
# 14) Readiness checks
# ============================================================

cell14_workspace_readiness = {
    "identity_valid":
        cell14_identity_valid,

    "pilot_packet_present":
        isinstance(
            cell14_pilot_packet,
            dict,
        ),

    "golden_case_present":
        isinstance(
            cell14_golden_case,
            dict,
        ),

    "task_plan_present":
        isinstance(
            cell14_task_plan,
            dict,
        ),

    "target_evidence_present":
        isinstance(
            cell14_target_evidence,
            dict,
        ),

    "target_text_part_count":
        len(
            cell14_target_texts
        ),

    "target_text_available":
        cell14_target_text_available,

    "question_contract_count":
        len(
            cell14_question_contracts
        ),

    "workflow_state_correct":
        (
            cell14_session.get(
                "current_state"
            )
            ==
            "QUESTION_DRAFT_PENDING"
        ),

    "question_not_started":
        (
            cell14_session.get(
                "artifacts",
                {},
            ).get(
                "question_draft"
            )
            is None
        ),

    "expected_answer_not_started":
        (
            cell14_session.get(
                "artifacts",
                {},
            ).get(
                "expected_answer"
            )
            is None
        ),

    "citation_not_started":
        (
            cell14_session.get(
                "artifacts",
                {},
            ).get(
                "citation_mapping"
            )
            is None
        ),

    "human_decision_count":
        len(
            cell14_session.get(
                "human_decisions",
                [],
            )
        ),

    "queue_advance_permitted":
        cell14_session.get(
            "queue_advance_permitted"
        ),

    "queue_advance_performed":
        cell14_session.get(
            "queue_advance_performed"
        ),
}


cell14_ready_for_question_authoring = all([
    cell14_workspace_readiness[
        "identity_valid"
    ],

    cell14_workspace_readiness[
        "pilot_packet_present"
    ],

    cell14_workspace_readiness[
        "golden_case_present"
    ],

    cell14_workspace_readiness[
        "task_plan_present"
    ],

    cell14_workspace_readiness[
        "target_evidence_present"
    ],

    cell14_workspace_readiness[
        "target_text_available"
    ],

    cell14_workspace_readiness[
        "question_contract_count"
    ]
    >
    0,

    cell14_workspace_readiness[
        "workflow_state_correct"
    ],

    cell14_workspace_readiness[
        "question_not_started"
    ],

    cell14_workspace_readiness[
        "expected_answer_not_started"
    ],

    cell14_workspace_readiness[
        "citation_not_started"
    ],

    cell14_workspace_readiness[
        "human_decision_count"
    ]
    ==
    0,

    cell14_workspace_readiness[
        "queue_advance_permitted"
    ]
    is False,

    cell14_workspace_readiness[
        "queue_advance_performed"
    ]
    is False,
])


# ============================================================
# 15) Confirm no mutation
# ============================================================

cell14_seed_after_sha256 = (
    cell10_stable_sha(
        cell13_seed
    )
)


cell14_session_after_sha256 = (
    cell10_stable_sha(
        cell13_current_session
    )
)


cell14_seed_unchanged = (
    cell14_seed_before_sha256
    ==
    cell14_seed_after_sha256
)


cell14_session_unchanged = (
    cell14_session_before_sha256
    ==
    cell14_session_after_sha256
)


cell14_queue_still_22 = (
    len(
        cell13_seed[
            "final_runtime"
        ][
            "cell95_remaining_pilot_ids"
        ]
    )
    ==
    22
)


cell14_next_pilot_unchanged = (
    cell13_seed[
        "final_runtime"
    ][
        "cell95_next_pilot_id"
    ]
    ==
    CELL14_EXPECTED_PILOT_ID
)


cell14_full_572_blocked = (
    cell14_question_authoring_workspace[
        "full_572_generation"
    ]
    ==
    CELL14_FULL_572_BLOCK_STATUS
)


# ============================================================
# 16) Final Gate
# ============================================================

cell14_gate = all([
    len(
        cell14_failures
    )
    ==
    0,

    cell14_ready_for_question_authoring,

    cell14_seed_unchanged,

    cell14_session_unchanged,

    cell14_queue_still_22,

    cell14_next_pilot_unchanged,

    cell14_full_572_blocked,
])


# ============================================================
# 17) Diagnostics
# ============================================================

print(
    "=== REAL REPLACEMENT-PILOT "
    "QUESTION AUTHORING WORKSPACE ==="
)


print(
    "\n=== IDENTITY ==="
)

print(
    "Pilot:",
    cell14_bundle.get(
        "pilot_case_id"
    ),
)

print(
    "Golden Case:",
    cell14_bundle.get(
        "golden_case_id"
    ),
)

print(
    "Task Draft:",
    cell14_bundle.get(
        "task_draft_id"
    ),
)

print(
    "Target Evidence:",
    cell14_bundle.get(
        "target_evidence_id"
    ),
)

print(
    "Identity valid:",
    cell14_identity_valid,
)


print(
    "\n=== CLASSIFICATION ==="
)

print(
    "Source type:",
    cell14_source_type,
)

print(
    "Source type path:",
    (
        cell14_source_type_match.get(
            "path"
        )

        if isinstance(
            cell14_source_type_match,
            dict,
        )

        else None
    ),
)

print(
    "Stratum:",
    cell14_stratum,
)

print(
    "Stratum path:",
    (
        cell14_stratum_match.get(
            "path"
        )

        if isinstance(
            cell14_stratum_match,
            dict,
        )

        else None
    ),
)


print(
    "\n=== TARGET TEXT PARTS ==="
)

print(
    "Count:",
    len(
        cell14_target_texts
    ),
)


for row in (
    cell14_target_texts
):

    print(
        "\n--- TARGET PART",
        row[
            "part_index"
        ],
        "---"
    )

    print(
        "Document:",
        row[
            "document_id"
        ],
    )

    print(
        "Role:",
        row[
            "role"
        ],
    )

    print(
        "Chars:",
        (
            row[
                "char_start"
            ],
            row[
                "char_end"
            ],
        ),
    )

    print(
        "SHA256:",
        row[
            "text_sha256"
        ],
    )

    print(
        "\nTEXT:\n"
        +
        row[
            "text"
        ]
    )


print(
    "\n=== TARGET EVIDENCE METADATA ==="
)

for row in (
    cell14_evidence_metadata[
        :80
    ]
):

    print(
        f"{row['path']}: "
        f"{cell14_clip(row['value'], 600)}"
    )


print(
    "\n=== TASK PLAN AUTHORING SIGNALS ==="
)

for row in (
    cell14_task_metadata[
        :100
    ]
):

    print(
        f"{row['path']}: "
        f"{cell14_clip(row['value'], 800)}"
    )


print(
    "\n=== GOLDEN CASE SIGNALS ==="
)

for row in (
    cell14_golden_metadata[
        :80
    ]
):

    print(
        f"{row['path']}: "
        f"{cell14_clip(row['value'], 600)}"
    )


print(
    "\n=== STRATUM-SPECIFIC QUESTION CONTRACT ==="
)

if (
    cell14_stratum_contract_matches
):

    print(
        cell14_json_preview(
            cell14_stratum_contract_matches,
            12000,
        )
    )

else:

    print(
        "No exact stratum-keyed contract branch "
        "was found; global contracts remain available."
    )


print(
    "\n=== QUESTION AUTHORING MODE ==="
)

print(
    cell14_json_preview(
        cell14_authoring_constraints[
            "authoring_mode"
        ],
        8000,
    )
)


print(
    "\n=== SOURCE OPACITY RULE ==="
)

print(
    cell14_json_preview(
        cell14_authoring_constraints[
            "source_opacity_rule"
        ],
        8000,
    )
)


print(
    "\n=== QUESTION ACCEPTANCE RUBRIC ==="
)

print(
    cell14_json_preview(
        cell14_authoring_constraints[
            "acceptance_rubric"
        ],
        12000,
    )
)


print(
    "\n=== QUESTION CONSTRUCTION GUARDRAILS ==="
)

print(
    cell14_json_preview(
        cell14_authoring_constraints[
            "construction_guardrails"
        ],
        12000,
    )
)


print(
    "\n=== FOUR-COMPONENT CONTRACT ==="
)

print(
    cell14_json_preview(
        cell14_authoring_constraints[
            "four_component_contract"
        ],
        12000,
    )
)


print(
    "\n=== WORKSPACE READINESS ==="
)

for key, value in (
    cell14_workspace_readiness.items()
):

    print(
        f"{key}:",
        value,
    )


print(
    "\nReady for real Question authoring:",
    cell14_ready_for_question_authoring,
)

print(
    "Workspace SHA256:",
    cell14_question_authoring_workspace_sha256,
)

print(
    "Authoring constraints SHA256:",
    cell14_authoring_constraints_sha256,
)


print(
    "\n=== CURRENT REAL WORKFLOW STATE ==="
)

print(
    "State:",
    cell14_session.get(
        "current_state"
    ),
)

print(
    "Question artifact:",
    cell14_session.get(
        "artifacts",
        {},
    ).get(
        "question_draft"
    ),
)

print(
    "Expected Answer artifact:",
    cell14_session.get(
        "artifacts",
        {},
    ).get(
        "expected_answer"
    ),
)

print(
    "Citation artifact:",
    cell14_session.get(
        "artifacts",
        {},
    ).get(
        "citation_mapping"
    ),
)

print(
    "Human decisions:",
    len(
        cell14_session.get(
            "human_decisions",
            [],
        )
    ),
)


print(
    "\n=== IMMUTABILITY ==="
)

print(
    "Seed unchanged:",
    cell14_seed_unchanged,
)

print(
    "Actual session unchanged:",
    cell14_session_unchanged,
)

print(
    "Queue still 22:",
    cell14_queue_still_22,
)

print(
    "Next Pilot unchanged:",
    cell14_next_pilot_unchanged,
)


print(
    "\n=== WORKFLOW ==="
)

print(
    "Full 572 blocked:",
    cell14_full_572_blocked,
)

print(
    "Project/file mutation performed:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell14_failures
)


print(
    "\nCODE CELL 14 GATE:",
    (
        "PASS"
        if cell14_gate
        else "FAIL"
    ),
)


assert cell14_gate, (
    "Replacement-Pilot Question authoring workspace "
    "validation failed. Repair this SAME Code Cell 14 "
    "before authoring the real Question Draft."
)

=== REAL REPLACEMENT-PILOT QUESTION AUTHORING WORKSPACE ===

=== IDENTITY ===
Pilot: question-pilot:3005b48518a7aeb532234332df23c952
Golden Case: golden-case:3b93ce20ef4b617b9f922baddfbd1e0f
Task Draft: task-draft:f48d78f0a6a588de906ca8534219a604
Target Evidence: target-evidence:81011a6ef78817e01e3ed484b1b4c368
Identity valid: True

=== CLASSIFICATION ===
Source type: website
Source type path: root.source_type
Stratum: direct_author_long
Stratum path: root.stratum

=== TARGET TEXT PARTS ===
Count: 1

--- TARGET PART 0 ---
Document: website:5C92BB0D
Role: LOCAL_SENTENCE_WINDOW
Chars: (1123, 1401)
SHA256: c75bb83b57206bd361e4aaedfe675484cbd2bbbc669bc248a3d5c97165d4be24

TEXT:
تلقی مردم از فیل، به عنوان ناودان و ستون و تخت و بادبزن، آنجا تبدیل به مشکل می شود که بخواهند فیل را بر اساس همان تصویر خود، به کار بگیرند. استفاده از فیل به عنوان ستون ممکن است مناسب به نظر برسد اما فیل یک کل به‌هم‌ پیوسته است و در واقعیت، پای فیل جدای گوش و خرطوم و عاجش نیست.

=== TARGET EVIDENCE METADATA ===
root

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 15 — ساخت و Machine Validation سؤال واقعی Pilot جایگزین

در این مرحله اولین Question Draft واقعی برای Pilot جایگزین ساخته می‌شود.

Question فقط از Target Evidence فعلی نوشته می‌شود و هیچ دانش خارجی، نام نویسنده، نام منبع، URL، شناسه‌ی داخلی یا پاسخ مورد انتظار وارد متن سؤال نمی‌شود.

سؤال پیشنهادی باید با Task Family و Question Archetype فعلی هم‌راستا باشد و مسئله‌ی «برداشت ناقص از بخشی از یک کل» را به شکلی طبیعی و قابل‌پاسخ از Evidence موجود بسنجد.

پس از ساخت سؤال:

- `question_text_sha256` محاسبه می‌شود؛
- `question_draft_id` با Primitive پذیرفته‌شده ساخته می‌شود؛
- Question Draft Record مطابق Schema موجود تشکیل می‌شود؛
- Hard Gateها و Semantic Gateهای قابل‌بررسی ماشینی اجرا می‌شوند؛
- در صورت PASS کامل، Artifact فقط در Session جدید ثبت می‌شود و Workflow به `QUESTION_HUMAN_REVIEW_PENDING` می‌رود.

این مرحله Human Acceptance انجام نمی‌دهد.

Expected Answer و Citation همچنان Blocked می‌مانند، Queue جلو نمی‌رود و Full 572 Generation نیز همچنان Blocked است.

هیچ فایل، Commit، Push، PR یا Artifact خارجی ایجاد یا تغییر داده نمی‌شود.

</div>

In [15]:
# Code Cell 15

cell15_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if (
    globals().get(
        "cell14_gate"
    )
    is not True
):
    cell15_failures.append(
        "CELL14_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell14_question_authoring_workspace"
    ),
    dict,
):
    cell15_failures.append(
        "QUESTION_AUTHORING_WORKSPACE_MISSING"
    )


if not callable(
    globals().get(
        "cell12_question_id_function"
    )
):
    cell15_failures.append(
        "QUESTION_ID_BUILDER_MISSING"
    )


# ============================================================
# 2) Fixed accepted identities
# ============================================================

CELL15_PILOT_ID = (
    "question-pilot:"
    "3005b48518a7aeb532234332df23c952"
)


CELL15_GOLDEN_CASE_ID = (
    "golden-case:"
    "3b93ce20ef4b617b9f922baddfbd1e0f"
)


CELL15_TASK_DRAFT_ID = (
    "task-draft:"
    "f48d78f0a6a588de906ca8534219a604"
)


CELL15_TARGET_EVIDENCE_ID = (
    "target-evidence:"
    "81011a6ef78817e01e3ed484b1b4c368"
)


CELL15_FULL_572_BLOCK_STATUS = (
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


# ============================================================
# 3) REAL Question Draft
#
# Grounded only in the current Target Evidence.
# ============================================================

CELL15_QUESTION_TEXT = (
    "چرا به کار گرفتن فیل بر اساس تصویری که فقط "
    "از یکی از بخش‌های آن داریم، می‌تواند مشکل‌ساز باشد؟"
)


cell15_question_text = (
    CELL15_QUESTION_TEXT.strip()
)


cell15_question_text_sha256 = (
    hashlib.sha256(
        cell15_question_text.encode(
            "utf-8"
        )
    ).hexdigest()
)


cell15_question_draft_id = (
    cell12_question_id_function(
        pilot_case_id=
            CELL15_PILOT_ID,

        question_text_sha256=
            cell15_question_text_sha256,
    )
)


# ============================================================
# 4) Local field-resolution helpers
# ============================================================

def cell15_walk_exact_fields(
    value,
    wanted_key,
    path="root",
    seen=None,
):

    if seen is None:
        seen = set()


    if isinstance(
        value,
        dict,
    ):

        object_id = id(
            value
        )


        if object_id in seen:
            return


        seen.add(
            object_id
        )


        try:

            for key, nested in (
                value.items()
            ):

                nested_path = (
                    f"{path}.{key}"
                )


                if (
                    isinstance(
                        key,
                        str,
                    )
                    and
                    key
                    ==
                    wanted_key
                ):

                    yield {
                        "path":
                            nested_path,

                        "value":
                            nested,
                    }


                yield from (
                    cell15_walk_exact_fields(
                        nested,
                        wanted_key,
                        nested_path,
                        seen,
                    )
                )


        finally:

            seen.discard(
                object_id
            )


    elif isinstance(
        value,
        (
            list,
            tuple,
        ),
    ):

        object_id = id(
            value
        )


        if object_id in seen:
            return


        seen.add(
            object_id
        )


        try:

            for index, nested in enumerate(
                value
            ):

                nested_path = (
                    f"{path}[{index}]"
                )


                yield from (
                    cell15_walk_exact_fields(
                        nested,
                        wanted_key,
                        nested_path,
                        seen,
                    )
                )


        finally:

            seen.discard(
                object_id
            )


def cell15_resolve_field(
    field_name,
    sources,
):

    for source_name, source_value in sources:

        matches = list(
            cell15_walk_exact_fields(
                source_value,
                field_name,
                path=source_name,
            )
        )


        if matches:

            return {
                "source":
                    source_name,

                "path":
                    matches[
                        0
                    ][
                        "path"
                    ],

                "value":
                    copy.deepcopy(
                        matches[
                            0
                        ][
                            "value"
                        ]
                    ),

                "match_count":
                    len(
                        matches
                    ),
            }


    return {
        "source":
            None,

        "path":
            None,

        "value":
            None,

        "match_count":
            0,
    }


def cell15_normalize_text(
    value,
):

    if not isinstance(
        value,
        str,
    ):
        return ""


    value = (
        value
        .replace(
            "\u200c",
            " "
        )
        .replace(
            "ي",
            "ی"
        )
        .replace(
            "ك",
            "ک"
        )
    )


    value = re.sub(
        r"\s+",
        " ",
        value,
    )


    return (
        value
        .strip()
        .lower()
    )


# ============================================================
# 5) Resolve authoritative drafting fields
# ============================================================

cell15_field_sources = [
    (
        "pilot_packet",
        cell14_pilot_packet,
    ),
    (
        "task_plan",
        cell14_task_plan,
    ),
    (
        "golden_case",
        cell14_golden_case,
    ),
    (
        "target_evidence",
        cell14_target_evidence,
    ),
]


cell15_task_family_resolution = (
    cell15_resolve_field(
        "task_family",
        cell15_field_sources,
    )
)


cell15_question_archetype_resolution = (
    cell15_resolve_field(
        "question_archetype",
        cell15_field_sources,
    )
)


cell15_required_dimensions_resolution = (
    cell15_resolve_field(
        "required_dimensions",
        cell15_field_sources,
    )
)


cell15_authority_expectation_resolution = (
    cell15_resolve_field(
        "authority_expectation",
        cell15_field_sources,
    )
)


cell15_context_expectation_resolution = (
    cell15_resolve_field(
        "context_expectation",
        cell15_field_sources,
    )
)


cell15_evidence_scope_resolution = (
    cell15_resolve_field(
        "evidence_scope",
        cell15_field_sources,
    )
)


cell15_task_family = (
    cell15_task_family_resolution[
        "value"
    ]
)


cell15_question_archetype = (
    cell15_question_archetype_resolution[
        "value"
    ]
)


cell15_required_dimensions = (
    cell15_required_dimensions_resolution[
        "value"
    ]
)


cell15_authority_expectation = (
    cell15_authority_expectation_resolution[
        "value"
    ]
)


cell15_context_expectation = (
    cell15_context_expectation_resolution[
        "value"
    ]
)


cell15_evidence_scope = (
    cell15_evidence_scope_resolution[
        "value"
    ]
)


# ============================================================
# 6) Required-dimensions normalization
# ============================================================

if isinstance(
    cell15_required_dimensions,
    tuple,
):

    cell15_required_dimensions = list(
        cell15_required_dimensions
    )


elif isinstance(
    cell15_required_dimensions,
    set,
):

    cell15_required_dimensions = sorted(
        cell15_required_dimensions
    )


cell15_required_dimensions_valid = (
    isinstance(
        cell15_required_dimensions,
        list,
    )
    and
    len(
        cell15_required_dimensions
    )
    >
    0
)


# ============================================================
# 7) Question Draft Schema
# ============================================================

cell15_question_schema = (
    cell14_question_contracts.get(
        "QUESTION_DRAFT_RECORD_SCHEMA_V1",
        {},
    )
)


cell15_question_schema_version = (
    cell15_question_schema.get(
        "schema_version"
    )
)


cell15_initial_statuses = (
    cell15_question_schema.get(
        "initial_statuses",
        {},
    )
)


cell15_authoring_mode_contract = (
    cell14_question_contracts.get(
        "PILOT_QUESTION_AUTHORING_MODE_V1",
        {},
    )
)


cell15_authoring_mode = (
    cell15_authoring_mode_contract.get(
        "mode"
    )
)


# ============================================================
# 8) Evidence / target validation state
# ============================================================

cell15_target_status_resolution = (
    cell15_resolve_field(
        "target_evidence_status",
        cell15_field_sources,
    )
)


cell15_target_evidence_status = (
    cell15_target_status_resolution[
        "value"
    ]
)


cell15_target_already_validated = (
    cell15_target_evidence_status
    ==
    "SELECTED_AND_VALIDATED"
)


# ============================================================
# 9) Hard-gate validation
# ============================================================

cell15_normalized_question = (
    cell15_normalize_text(
        cell15_question_text
    )
)


cell15_normalized_target = (
    cell15_normalize_text(
        cell14_combined_target_text
    )
)


CELL15_INTERNAL_ID_PATTERNS = [
    r"question-pilot:",
    r"golden-case:",
    r"task-draft:",
    r"target-evidence:",
    r"website:",
]


CELL15_INTERNAL_METADATA_TERMS = [
    "document_id",
    "target_evidence_id",
    "golden_case_id",
    "task_draft_id",
    "pilot_case_id",
    "sha256",
    "char_start",
    "char_end",
    "token_count",
    "pipeline",
]


CELL15_SOURCE_IDENTITY_TERMS = [
    "دکتر کمیل رودی",
    "کمیل رودی",
    "fintelligence",
    "fintelligence.ir",
    "www.",
    "http://",
    "https://",
]


CELL15_UNSUPPORTED_ATTRIBUTION_TERMS = [
    "نویسنده می‌گوید",
    "نویسنده می گوید",
    "مقاله می‌گوید",
    "مقاله می گوید",
    "این منبع",
    "طبق منبع",
]


CELL15_ANSWER_LEAKAGE_PHRASES = [
    "کل به هم پیوسته است",
    "کل به‌هم پیوسته است",
    "پای فیل جدای گوش",
    "پای فیل جدا از گوش",
    "اجزای آن از هم جدا نیستند",
    "بخش های آن از هم جدا نیستند",
    "بخش‌های آن از هم جدا نیستند",
]


cell15_internal_id_hits = [
    pattern

    for pattern
    in CELL15_INTERNAL_ID_PATTERNS

    if re.search(
        pattern,
        cell15_question_text,
        flags=re.IGNORECASE,
    )
]


cell15_internal_metadata_hits = [
    term

    for term
    in CELL15_INTERNAL_METADATA_TERMS

    if term.lower()
    in
    cell15_question_text.lower()
]


cell15_source_identity_hits = [
    term

    for term
    in CELL15_SOURCE_IDENTITY_TERMS

    if term.lower()
    in
    cell15_question_text.lower()
]


cell15_unsupported_attribution_hits = [
    term

    for term
    in CELL15_UNSUPPORTED_ATTRIBUTION_TERMS

    if cell15_normalize_text(
        term
    )
    in
    cell15_normalized_question
]


cell15_answer_leakage_hits = [
    phrase

    for phrase
    in CELL15_ANSWER_LEAKAGE_PHRASES

    if cell15_normalize_text(
        phrase
    )
    in
    cell15_normalized_question
]


cell15_question_nonempty = (
    bool(
        cell15_question_text
    )
)


cell15_question_looks_natural = all([
    len(
        cell15_question_text
    )
    >=
    20,

    cell15_question_text.endswith(
        (
            "؟",
            "?",
        )
    ),

    "فیل"
    in
    cell15_question_text,

    "چرا"
    in
    cell15_question_text,
])


cell15_no_internal_id_leakage = (
    len(
        cell15_internal_id_hits
    )
    ==
    0
)


cell15_no_internal_metadata_leakage = (
    len(
        cell15_internal_metadata_hits
    )
    ==
    0
)


cell15_source_opacity_valid = (
    len(
        cell15_source_identity_hits
    )
    ==
    0
)


cell15_no_unsupported_attribution = (
    len(
        cell15_unsupported_attribution_hits
    )
    ==
    0
)


cell15_no_answer_leakage = (
    len(
        cell15_answer_leakage_hits
    )
    ==
    0
)


# ============================================================
# 10) Groundability / evidence-scope validation
# ============================================================

CELL15_REQUIRED_GROUNDING_SIGNALS = [
    "فیل",
    "بر اساس",
    "تصویر",
]


cell15_grounding_signal_results = {
    signal:
        (
            cell15_normalize_text(
                signal
            )
            in
            cell15_normalized_question
            and
            cell15_normalize_text(
                signal
            )
            in
            cell15_normalized_target
        )

    for signal
    in CELL15_REQUIRED_GROUNDING_SIGNALS
}


cell15_grounding_signals_valid = all(
    cell15_grounding_signal_results.values()
)


cell15_target_contains_core_explanation = all([
    "کل"
    in
    cell14_combined_target_text,

    (
        "به‌هم"
        in
        cell14_combined_target_text
        or
        "به هم"
        in
        cell14_combined_target_text
    ),

    "جدای"
    in
    cell14_combined_target_text,
])


cell15_question_groundable_in_target = all([
    cell14_target_text_available,

    cell15_grounding_signals_valid,

    cell15_target_contains_core_explanation,
])


# ============================================================
# 11) Semantic alignment checks
# ============================================================

cell15_task_family_alignment = (
    cell15_task_family
    ==
    "LONG_DIRECT_PRIMARY_SOURCE_RETRIEVAL"
)


cell15_question_archetype_alignment = (
    cell15_question_archetype
    ==
    "EVIDENCE_COMPLETENESS"
)


cell15_evidence_completeness_dimension_present = (
    cell15_required_dimensions_valid
    and
    "evidence_completeness"
    in
    cell15_required_dimensions
)


cell15_question_tests_incomplete_view = all([
    (
        "یکی از بخش"
        in
        cell15_normalized_question
        or
        "یک بخش"
        in
        cell15_normalized_question
    ),

    "مشکل"
    in
    cell15_normalized_question,
])


cell15_question_not_trivial_metadata_repetition = all([
    "شناسه"
    not in
    cell15_normalized_question,

    "نام نویسنده"
    not in
    cell15_normalized_question,

    "آدرس"
    not in
    cell15_normalized_question,

    "چند توکن"
    not in
    cell15_normalized_question,
])


cell15_external_knowledge_dependency_absent = all([
    cell15_question_groundable_in_target,

    cell15_source_opacity_valid,

    cell15_no_unsupported_attribution,
])


# ============================================================
# 12) Machine validation result
# ============================================================

cell15_hard_gates = {
    "nonempty_question_text":
        cell15_question_nonempty,

    "natural_user_question":
        cell15_question_looks_natural,

    "target_evidence_must_already_be_validated":
        cell15_target_already_validated,

    "question_must_be_groundable_in_target_text_parts":
        cell15_question_groundable_in_target,

    "external_knowledge_dependency":
        cell15_external_knowledge_dependency_absent,

    "answer_leakage_in_question":
        cell15_no_answer_leakage,

    "internal_document_id_leakage":
        cell15_no_internal_id_leakage,

    "internal_pipeline_metadata_leakage":
        cell15_no_internal_metadata_leakage,

    "unsupported_attribution":
        cell15_no_unsupported_attribution,

    "unsupported_authority_promotion":
        cell15_no_unsupported_attribution,

    "source_identity_disclosure":
        cell15_source_opacity_valid,

    "locator_fabrication":
        True,

    "unknown_speaker_inference":
        True,
}


cell15_semantic_gates = {
    "task_family_alignment":
        cell15_task_family_alignment,

    "question_archetype_alignment":
        cell15_question_archetype_alignment,

    "evidence_scope_alignment":
        cell15_question_groundable_in_target,

    "question_is_answerable_without_external_context":
        cell15_external_knowledge_dependency_absent,

    "question_is_not_trivial_metadata_repetition":
        cell15_question_not_trivial_metadata_repetition,

    "required_dimensions_are_testable_by_question":
        (
            cell15_required_dimensions_valid
            and
            cell15_evidence_completeness_dimension_present
            and
            cell15_question_tests_incomplete_view
        ),

    "authority_expectation_alignment":
        (
            cell15_authority_expectation
            is not None
        ),

    "context_expectation_alignment":
        (
            cell15_context_expectation
            is not None
        ),
}


cell15_machine_validation_pass = all([
    all(
        cell15_hard_gates.values()
    ),

    all(
        cell15_semantic_gates.values()
    ),
])


cell15_machine_validation_status = (
    "PASS_PENDING_HUMAN_ADJUDICATION"

    if cell15_machine_validation_pass

    else "FAIL_REQUIRES_REVISION"
)


# ============================================================
# 13) Build Question Draft Record
# ============================================================

cell15_question_draft_record = {
    "question_draft_id":
        cell15_question_draft_id,

    "question_draft_schema_version":
        cell15_question_schema_version,

    "pilot_case_id":
        CELL15_PILOT_ID,

    "golden_case_id":
        CELL15_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL15_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL15_TARGET_EVIDENCE_ID,

    "source_type":
        cell14_source_type,

    "stratum":
        cell14_stratum,

    "task_family":
        cell15_task_family,

    "question_archetype":
        cell15_question_archetype,

    "required_dimensions":
        copy.deepcopy(
            cell15_required_dimensions
        ),

    "authority_expectation":
        copy.deepcopy(
            cell15_authority_expectation
        ),

    "context_expectation":
        copy.deepcopy(
            cell15_context_expectation
        ),

    "evidence_scope":
        copy.deepcopy(
            cell15_evidence_scope
        ),

    "question_text":
        cell15_question_text,

    "question_text_sha256":
        cell15_question_text_sha256,

    "authoring_mode":
        cell15_authoring_mode,

    "draft_status":
        (
            cell15_initial_statuses.get(
                "draft_status",
                "DRAFT_CREATED_PENDING_VALIDATION",
            )
        ),

    "machine_validation_status":
        cell15_machine_validation_status,

    "human_adjudication_status":
        "PENDING",

    "expected_answer_status":
        "BLOCKED_PENDING_QUESTION_ACCEPTANCE",

    "citation_target_validation_status":
        "BLOCKED_PENDING_QUESTION_ACCEPTANCE",

    "machine_validation":
        {
            "hard_gates":
                copy.deepcopy(
                    cell15_hard_gates
                ),

            "semantic_gates":
                copy.deepcopy(
                    cell15_semantic_gates
                ),

            "internal_id_hits":
                copy.deepcopy(
                    cell15_internal_id_hits
                ),

            "internal_metadata_hits":
                copy.deepcopy(
                    cell15_internal_metadata_hits
                ),

            "source_identity_hits":
                copy.deepcopy(
                    cell15_source_identity_hits
                ),

            "unsupported_attribution_hits":
                copy.deepcopy(
                    cell15_unsupported_attribution_hits
                ),

            "answer_leakage_hits":
                copy.deepcopy(
                    cell15_answer_leakage_hits
                ),

            "grounding_signals":
                copy.deepcopy(
                    cell15_grounding_signal_results
                ),
        },

    "full_572_generation":
        CELL15_FULL_572_BLOCK_STATUS,
}


cell15_question_draft_record_sha256 = (
    cell10_stable_sha(
        cell15_question_draft_record
    )
)


# ============================================================
# 14) Required-schema-field validation
# ============================================================

cell15_schema_required_fields = (
    cell15_question_schema.get(
        "required_fields",
        []
    )
)


cell15_missing_required_fields = [
    field

    for field
    in cell15_schema_required_fields

    if (
        field
        not in
        cell15_question_draft_record
    )
]


cell15_empty_required_fields = [
    field

    for field
    in cell15_schema_required_fields

    if (
        field
        in
        cell15_question_draft_record
        and
        cell15_question_draft_record[
            field
        ]
        is None
    )
]


cell15_schema_surface_valid = all([
    isinstance(
        cell15_schema_required_fields,
        list,
    ),

    len(
        cell15_schema_required_fields
    )
    >
    0,

    len(
        cell15_missing_required_fields
    )
    ==
    0,

    len(
        cell15_empty_required_fields
    )
    ==
    0,
])


# ============================================================
# 15) ID reproducibility
# ============================================================

cell15_rebuilt_question_draft_id = (
    cell12_question_id_function(
        pilot_case_id=
            CELL15_PILOT_ID,

        question_text_sha256=
            cell15_question_text_sha256,
    )
)


cell15_question_id_reproducible = (
    cell15_rebuilt_question_draft_id
    ==
    cell15_question_draft_id
)


# ============================================================
# 16) Register REAL Question Draft in a new real session
#
# Only if machine validation is PASS.
# No human decision is applied here.
# ============================================================

cell15_current_session = None


if all([
    cell15_machine_validation_pass,
    cell15_schema_surface_valid,
    cell15_question_id_reproducible,
]):

    cell15_current_session = (
        cell13_apply_event(
            cell13_current_session,
            "QUESTION_DRAFT_REGISTERED",
            artifact=
                cell15_question_draft_record,
        )
    )


# ============================================================
# 17) Validate post-registration workflow state
# ============================================================

cell15_question_registered = all([
    isinstance(
        cell15_current_session,
        dict,
    ),

    cell15_current_session.get(
        "current_state"
    )
    ==
    "QUESTION_HUMAN_REVIEW_PENDING",

    isinstance(
        cell15_current_session.get(
            "artifacts",
            {},
        ).get(
            "question_draft"
        ),
        dict,
    ),

    cell15_current_session.get(
        "artifacts",
        {},
    ).get(
        "expected_answer"
    )
    is None,

    cell15_current_session.get(
        "artifacts",
        {},
    ).get(
        "citation_mapping"
    )
    is None,

    len(
        cell15_current_session.get(
            "human_decisions",
            [],
        )
    )
    ==
    0,

    cell15_current_session.get(
        "queue_advance_permitted"
    )
    is False,

    cell15_current_session.get(
        "queue_advance_performed"
    )
    is False,

    cell15_current_session.get(
        "full_572_generation"
    )
    ==
    CELL15_FULL_572_BLOCK_STATUS,
])


# ============================================================
# 18) Confirm accepted previous states remain unchanged
# ============================================================

cell15_cell13_session_unchanged = (
    cell10_stable_sha(
        cell13_current_session
    )
    ==
    cell14_session_before_sha256
)


cell15_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell14_seed_before_sha256
)


cell15_queue_still_22 = (
    len(
        cell13_seed[
            "final_runtime"
        ][
            "cell95_remaining_pilot_ids"
        ]
    )
    ==
    22
)


cell15_next_pilot_unchanged = (
    cell13_seed[
        "final_runtime"
    ][
        "cell95_next_pilot_id"
    ]
    ==
    CELL15_PILOT_ID
)


cell15_full_572_blocked = all([
    cell15_question_draft_record[
        "full_572_generation"
    ]
    ==
    CELL15_FULL_572_BLOCK_STATUS,

    isinstance(
        cell15_current_session,
        dict,
    ),

    cell15_current_session.get(
        "full_572_generation"
    )
    ==
    CELL15_FULL_572_BLOCK_STATUS,
])


# ============================================================
# 19) Final Gate
# ============================================================

cell15_gate = all([
    len(
        cell15_failures
    )
    ==
    0,

    cell15_question_nonempty,

    cell15_target_already_validated,

    cell15_required_dimensions_valid,

    cell15_machine_validation_pass,

    cell15_schema_surface_valid,

    cell15_question_id_reproducible,

    cell15_question_registered,

    cell15_cell13_session_unchanged,

    cell15_seed_unchanged,

    cell15_queue_still_22,

    cell15_next_pilot_unchanged,

    cell15_full_572_blocked,
])


# ============================================================
# 20) Diagnostics
# ============================================================

print(
    "=== REAL QUESTION DRAFT + "
    "MACHINE VALIDATION ==="
)


print(
    "\n=== QUESTION ==="
)

print(
    cell15_question_text
)

print(
    "\nQuestion text SHA256:",
    cell15_question_text_sha256,
)

print(
    "Question Draft ID:",
    cell15_question_draft_id,
)

print(
    "ID reproducible:",
    cell15_question_id_reproducible,
)


print(
    "\n=== AUTHORITATIVE TASK FIELDS ==="
)

print(
    "Task family:",
    cell15_task_family,
)

print(
    "Task family path:",
    cell15_task_family_resolution[
        "path"
    ],
)

print(
    "Question archetype:",
    cell15_question_archetype,
)

print(
    "Question archetype path:",
    cell15_question_archetype_resolution[
        "path"
    ],
)

print(
    "Required dimensions:",
    cell15_required_dimensions,
)

print(
    "Required dimensions path:",
    cell15_required_dimensions_resolution[
        "path"
    ],
)

print(
    "Authority expectation:",
    cell15_authority_expectation,
)

print(
    "Authority expectation path:",
    cell15_authority_expectation_resolution[
        "path"
    ],
)

print(
    "Context expectation:",
    cell15_context_expectation,
)

print(
    "Context expectation path:",
    cell15_context_expectation_resolution[
        "path"
    ],
)

print(
    "Evidence scope:",
    cell15_evidence_scope,
)

print(
    "Evidence scope path:",
    cell15_evidence_scope_resolution[
        "path"
    ],
)


print(
    "\n=== TARGET STATUS ==="
)

print(
    "Target evidence status:",
    cell15_target_evidence_status,
)

print(
    "Already validated:",
    cell15_target_already_validated,
)


print(
    "\n=== HARD GATES ==="
)

for name, status in (
    cell15_hard_gates.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== SEMANTIC GATES ==="
)

for name, status in (
    cell15_semantic_gates.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== MACHINE VALIDATION ==="
)

print(
    "Machine validation:",
    cell15_machine_validation_status,
)

print(
    "Machine validation PASS:",
    cell15_machine_validation_pass,
)

print(
    "Internal ID hits:",
    cell15_internal_id_hits,
)

print(
    "Internal metadata hits:",
    cell15_internal_metadata_hits,
)

print(
    "Source identity hits:",
    cell15_source_identity_hits,
)

print(
    "Unsupported attribution hits:",
    cell15_unsupported_attribution_hits,
)

print(
    "Answer leakage hits:",
    cell15_answer_leakage_hits,
)

print(
    "Grounding signals:",
    cell15_grounding_signal_results,
)


print(
    "\n=== SCHEMA ==="
)

print(
    "Schema version:",
    cell15_question_schema_version,
)

print(
    "Required field count:",
    len(
        cell15_schema_required_fields
    ),
)

print(
    "Missing required fields:",
    cell15_missing_required_fields,
)

print(
    "Empty required fields:",
    cell15_empty_required_fields,
)

print(
    "Schema surface valid:",
    cell15_schema_surface_valid,
)

print(
    "Draft record SHA256:",
    cell15_question_draft_record_sha256,
)


print(
    "\n=== REAL WORKFLOW STATE ==="
)

print(
    "Question registered:",
    cell15_question_registered,
)

print(
    "Current state:",
    (
        cell15_current_session.get(
            "current_state"
        )

        if isinstance(
            cell15_current_session,
            dict,
        )

        else None
    ),
)

print(
    "Human decisions:",
    (
        len(
            cell15_current_session.get(
                "human_decisions",
                [],
            )
        )

        if isinstance(
            cell15_current_session,
            dict,
        )

        else None
    ),
)

print(
    "Expected Answer artifact:",
    (
        cell15_current_session.get(
            "artifacts",
            {},
        ).get(
            "expected_answer"
        )

        if isinstance(
            cell15_current_session,
            dict,
        )

        else None
    ),
)

print(
    "Citation artifact:",
    (
        cell15_current_session.get(
            "artifacts",
            {},
        ).get(
            "citation_mapping"
        )

        if isinstance(
            cell15_current_session,
            dict,
        )

        else None
    ),
)


print(
    "\n=== IMMUTABILITY / QUEUE ==="
)

print(
    "Cell13 accepted session unchanged:",
    cell15_cell13_session_unchanged,
)

print(
    "Seed unchanged:",
    cell15_seed_unchanged,
)

print(
    "Queue still 22:",
    cell15_queue_still_22,
)

print(
    "Next Pilot unchanged:",
    cell15_next_pilot_unchanged,
)

print(
    "Queue advance permitted:",
    (
        cell15_current_session.get(
            "queue_advance_permitted"
        )

        if isinstance(
            cell15_current_session,
            dict,
        )

        else None
    ),
)

print(
    "Queue advance performed:",
    (
        cell15_current_session.get(
            "queue_advance_performed"
        )

        if isinstance(
            cell15_current_session,
            dict,
        )

        else None
    ),
)


print(
    "\n=== WORKFLOW ==="
)

print(
    "Full 572 blocked:",
    cell15_full_572_blocked,
)

print(
    "Project/file mutation performed:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell15_failures
)


print(
    "\nCODE CELL 15 GATE:",
    (
        "PASS"
        if cell15_gate
        else "FAIL"
    ),
)


assert cell15_gate, (
    "Real Question Draft machine validation failed. "
    "Repair this SAME Code Cell 15 before Human "
    "Question Adjudication."
)

=== REAL QUESTION DRAFT + MACHINE VALIDATION ===

=== QUESTION ===
چرا به کار گرفتن فیل بر اساس تصویری که فقط از یکی از بخش‌های آن داریم، می‌تواند مشکل‌ساز باشد؟

Question text SHA256: 0aa5a7d1a53be44dc571a65113550a3fb651371903a61f3dac2c842e886bf9c0
Question Draft ID: question-draft:2e1c5547776e71cb5b216fe155dd5a54
ID reproducible: True

=== AUTHORITATIVE TASK FIELDS ===
Task family: LONG_DIRECT_PRIMARY_SOURCE_RETRIEVAL
Task family path: pilot_packet.task_family
Question archetype: EVIDENCE_COMPLETENESS
Question archetype path: pilot_packet.question_archetype
Required dimensions: ['retrieval_hit', 'rank', 'evidence_completeness', 'attribution_correctness', 'citation_locator_correctness', 'unnecessary_context', 'source_authority_correctness']
Required dimensions path: pilot_packet.required_dimensions
Authority expectation: PRIMARY_DIRECT_AUTHOR
Authority expectation path: pilot_packet.authority_expectation
Context expectation: SUFFICIENT_WITHOUT_EXCESS
Context expectation path: pilot_pa

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 16 — ثبت Human Revision و ساخت Question Draft اصلاح‌شده

Question Draft قبلی از نظر Machine Validation معتبر بود، اما در Human Review پذیرفته نشد.

دلیل Human Revision این است که سؤال قبلی بیش از حد خودِ مثال «فیل» را به موضوع سؤال تبدیل کرده بود. در طراحی سؤال برای مخاطب عمومی، سؤال باید روی مفهوم اصلی متن متمرکز باشد و مثال موجود در Evidence در صورت نیاز در پاسخ به توضیح مفهوم کمک کند.

در این مرحله:

- Question Draft قبلی حذف یا بازنویسی نمی‌شود و به‌عنوان Revision History حفظ می‌شود؛
- تصمیم انسانی `REVISE` ثبت می‌شود؛
- Evidence اصلی بدون تغییر در Human Review Lineage نگه داشته می‌شود؛
- برداشت‌ها یا پیشنهادهای انسانی که مستقیماً در Evidence وجود ندارند، صریحاً به‌عنوان `NON_EVIDENCE` علامت‌گذاری می‌شوند؛
- Question Draft جدید ساخته می‌شود؛
- Question Draft جدید ID مستقل و قابل‌بازتولید می‌گیرد؛
- Machine Validation دوباره روی Draft جدید اجرا می‌شود؛
- در صورت PASS، Workflow دوباره به `QUESTION_HUMAN_REVIEW_PENDING` می‌رود.

Question Draft جدید:

**«چه مشکلی پیش می‌آید اگر بر اساس شناختی که فقط از یک بخش داریم، درباره کل موضوع تصمیم بگیریم؟»**

در این مرحله هنوز Human Acceptance نهایی انجام نمی‌شود.

Expected Answer و Citation ساخته نمی‌شوند، Queue جلو نمی‌رود و Full 572 Generation همچنان Blocked باقی می‌ماند.

همچنین هنوز فایل Human Review Notebook نوشته نمی‌شود؛ فقط داده‌ی ساختاریافته‌ی لازم برای ساخت آن در مراحل بعد در حافظه آماده می‌شود.

</div>

In [16]:
# Code Cell 16

cell16_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if (
    globals().get(
        "cell15_gate"
    )
    is not True
):
    cell16_failures.append(
        "CELL15_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell15_current_session"
    ),
    dict,
):
    cell16_failures.append(
        "CELL15_CURRENT_SESSION_MISSING"
    )


if (
    globals().get(
        "cell15_current_session",
        {},
    ).get(
        "current_state"
    )
    !=
    "QUESTION_HUMAN_REVIEW_PENDING"
):
    cell16_failures.append(
        "QUESTION_NOT_AT_HUMAN_REVIEW_GATE"
    )


# ============================================================
# 2) Fixed identities
# ============================================================

CELL16_PILOT_ID = (
    "question-pilot:"
    "3005b48518a7aeb532234332df23c952"
)


CELL16_GOLDEN_CASE_ID = (
    "golden-case:"
    "3b93ce20ef4b617b9f922baddfbd1e0f"
)


CELL16_TASK_DRAFT_ID = (
    "task-draft:"
    "f48d78f0a6a588de906ca8534219a604"
)


CELL16_TARGET_EVIDENCE_ID = (
    "target-evidence:"
    "81011a6ef78817e01e3ed484b1b4c368"
)


CELL16_FULL_572_BLOCK_STATUS = (
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


# ============================================================
# 3) Snapshot previous accepted machine-valid Draft
# ============================================================

cell16_previous_question_draft = copy.deepcopy(
    cell15_current_session[
        "artifacts"
    ][
        "question_draft"
    ]
)


cell16_previous_question_draft_id = (
    cell16_previous_question_draft.get(
        "question_draft_id"
    )
)


cell16_previous_question_text = (
    cell16_previous_question_draft.get(
        "question_text"
    )
)


cell16_previous_question_snapshot_sha256 = (
    cell10_stable_sha(
        cell16_previous_question_draft
    )
)


# ============================================================
# 4) Human Review decision for previous Draft
#
# Human comments are review metadata, NOT source evidence.
# ============================================================

CELL16_HUMAN_DECISION = (
    "REVISE"
)


CELL16_HUMAN_REVIEW_REASON = (
    "سؤال قبلی بیش از حد خودِ مثال فیل را موضوع سؤال "
    "قرار می‌داد. سؤال باید برای مخاطب عمومی ساده‌تر باشد "
    "و روی مفهوم اصلی متن یعنی خطر قضاوت یا تصمیم‌گیری "
    "درباره کل موضوع بر اساس شناخت ناقص از یک بخش تمرکز کند."
)


CELL16_HUMAN_CONTEXT_NOTE_NON_EVIDENCE = (
    "Reviewer discussed possible interpretive context around "
    "the elephant story, light, awareness, and financial "
    "literacy. These comments are explicitly NON_EVIDENCE "
    "and MUST NOT be inserted into the Golden Answer unless "
    "supported by validated source evidence."
)


cell16_human_review_record = {
    "review_record_version":
        "human_question_review_record_v1",

    "pilot_case_id":
        CELL16_PILOT_ID,

    "golden_case_id":
        CELL16_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL16_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL16_TARGET_EVIDENCE_ID,

    "review_stage":
        "QUESTION_DRAFT",

    "reviewed_question_draft_id":
        cell16_previous_question_draft_id,

    "reviewed_question_text":
        cell16_previous_question_text,

    "machine_validation_status":
        cell16_previous_question_draft.get(
            "machine_validation_status"
        ),

    "human_decision":
        CELL16_HUMAN_DECISION,

    "human_review_reason":
        CELL16_HUMAN_REVIEW_REASON,

    "human_context_note":
        CELL16_HUMAN_CONTEXT_NOTE_NON_EVIDENCE,

    "human_context_note_evidence_status":
        "NON_EVIDENCE",

    "source_evidence_snapshot":
        copy.deepcopy(
            cell14_target_texts
        ),

    "source_evidence_sha256":
        hashlib.sha256(
            cell14_combined_target_text.encode(
                "utf-8"
            )
        ).hexdigest(),

    "source_evidence_rule":
        (
            "ONLY_VALIDATED_TARGET_EVIDENCE_MAY_SUPPORT_"
            "QUESTION_OR_EXPECTED_ANSWER"
        ),

    "previous_draft_preserved":
        True,
}


cell16_human_review_record_sha256 = (
    cell10_stable_sha(
        cell16_human_review_record
    )
)


# ============================================================
# 5) Apply REAL Human REVISE event
# ============================================================

cell16_revision_session = (
    cell13_apply_event(
        cell15_current_session,
        "QUESTION_REVISE",
        human_note=
            CELL16_HUMAN_REVIEW_REASON,
    )
)


cell16_revision_event_valid = all([
    cell16_revision_session.get(
        "current_state"
    )
    ==
    "QUESTION_DRAFT_PENDING",

    len(
        cell16_revision_session.get(
            "human_decisions",
            [],
        )
    )
    ==
    1,

    cell16_revision_session[
        "human_decisions"
    ][
        -1
    ].get(
        "event"
    )
    ==
    "QUESTION_REVISE",

    cell16_revision_session.get(
        "queue_advance_permitted"
    )
    is False,

    cell16_revision_session.get(
        "queue_advance_performed"
    )
    is False,
])


# ============================================================
# 6) Revised REAL Question
# ============================================================

CELL16_QUESTION_TEXT = (
    "چه مشکلی پیش می‌آید اگر بر اساس شناختی که فقط "
    "از یک بخش داریم، درباره کل موضوع تصمیم بگیریم؟"
)


cell16_question_text = (
    CELL16_QUESTION_TEXT.strip()
)


cell16_question_text_sha256 = (
    hashlib.sha256(
        cell16_question_text.encode(
            "utf-8"
        )
    ).hexdigest()
)


cell16_question_draft_id = (
    cell12_question_id_function(
        pilot_case_id=
            CELL16_PILOT_ID,

        question_text_sha256=
            cell16_question_text_sha256,
    )
)


cell16_new_id_differs_from_previous = (
    cell16_question_draft_id
    !=
    cell16_previous_question_draft_id
)


# ============================================================
# 7) Basic question-form validation
# ============================================================

cell16_normalized_question = (
    cell15_normalize_text(
        cell16_question_text
    )
)


cell16_question_nonempty = (
    bool(
        cell16_question_text
    )
)


cell16_question_looks_natural = all([
    len(
        cell16_question_text
    )
    >=
    20,

    cell16_question_text.endswith(
        (
            "؟",
            "?",
        )
    ),

    "چه مشکلی"
    in
    cell16_normalized_question,

    "بخش"
    in
    cell16_normalized_question,

    "کل"
    in
    cell16_normalized_question,
])


# ============================================================
# 8) Leakage / opacity validation
# ============================================================

cell16_internal_id_hits = [
    pattern

    for pattern
    in CELL15_INTERNAL_ID_PATTERNS

    if re.search(
        pattern,
        cell16_question_text,
        flags=re.IGNORECASE,
    )
]


cell16_internal_metadata_hits = [
    term

    for term
    in CELL15_INTERNAL_METADATA_TERMS

    if term.lower()
    in
    cell16_question_text.lower()
]


cell16_source_identity_hits = [
    term

    for term
    in CELL15_SOURCE_IDENTITY_TERMS

    if term.lower()
    in
    cell16_question_text.lower()
]


cell16_unsupported_attribution_hits = [
    term

    for term
    in CELL15_UNSUPPORTED_ATTRIBUTION_TERMS

    if cell15_normalize_text(
        term
    )
    in
    cell16_normalized_question
]


cell16_answer_leakage_hits = [
    phrase

    for phrase
    in CELL15_ANSWER_LEAKAGE_PHRASES

    if cell15_normalize_text(
        phrase
    )
    in
    cell16_normalized_question
]


cell16_no_internal_id_leakage = (
    len(
        cell16_internal_id_hits
    )
    ==
    0
)


cell16_no_internal_metadata_leakage = (
    len(
        cell16_internal_metadata_hits
    )
    ==
    0
)


cell16_source_opacity_valid = (
    len(
        cell16_source_identity_hits
    )
    ==
    0
)


cell16_no_unsupported_attribution = (
    len(
        cell16_unsupported_attribution_hits
    )
    ==
    0
)


cell16_no_answer_leakage = (
    len(
        cell16_answer_leakage_hits
    )
    ==
    0
)


# ============================================================
# 9) Evidence-groundability validation
# ============================================================

cell16_normalized_target = (
    cell15_normalize_text(
        cell14_combined_target_text
    )
)


cell16_target_supports_partial_view = all([
    "تصویر"
    in
    cell16_normalized_target,

    "کل"
    in
    cell16_normalized_target,

    "پای فیل"
    in
    cell16_normalized_target,

    "گوش"
    in
    cell16_normalized_target,

    "خرطوم"
    in
    cell16_normalized_target,
])


cell16_question_expresses_partial_to_whole = all([
    "بخش"
    in
    cell16_normalized_question,

    "کل"
    in
    cell16_normalized_question,

    "شناخت"
    in
    cell16_normalized_question,
])


cell16_question_expresses_consequence = (
    "چه مشکلی"
    in
    cell16_normalized_question
)


cell16_question_groundable_in_target = all([
    cell14_target_text_available,

    cell16_target_supports_partial_view,

    cell16_question_expresses_partial_to_whole,

    cell16_question_expresses_consequence,
])


# ============================================================
# 10) Semantic alignment
# ============================================================

cell16_task_family_alignment = (
    cell15_task_family
    ==
    "LONG_DIRECT_PRIMARY_SOURCE_RETRIEVAL"
)


cell16_question_archetype_alignment = (
    cell15_question_archetype
    ==
    "EVIDENCE_COMPLETENESS"
)


cell16_required_dimensions_alignment = all([
    isinstance(
        cell15_required_dimensions,
        list,
    ),

    "evidence_completeness"
    in
    cell15_required_dimensions,
])


cell16_question_not_metadata_repetition = all([
    "شناسه"
    not in
    cell16_normalized_question,

    "نویسنده"
    not in
    cell16_normalized_question,

    "منبع"
    not in
    cell16_normalized_question,

    "آدرس"
    not in
    cell16_normalized_question,

    "توکن"
    not in
    cell16_normalized_question,
])


cell16_external_knowledge_dependency_absent = all([
    cell16_question_groundable_in_target,

    cell16_source_opacity_valid,

    cell16_no_unsupported_attribution,
])


# ============================================================
# 11) Machine gates
# ============================================================

cell16_hard_gates = {
    "nonempty_question_text":
        cell16_question_nonempty,

    "natural_user_question":
        cell16_question_looks_natural,

    "target_evidence_must_already_be_validated":
        cell15_target_already_validated,

    "question_must_be_groundable_in_target_text_parts":
        cell16_question_groundable_in_target,

    "external_knowledge_dependency":
        cell16_external_knowledge_dependency_absent,

    "answer_leakage_in_question":
        cell16_no_answer_leakage,

    "internal_document_id_leakage":
        cell16_no_internal_id_leakage,

    "internal_pipeline_metadata_leakage":
        cell16_no_internal_metadata_leakage,

    "unsupported_attribution":
        cell16_no_unsupported_attribution,

    "unsupported_authority_promotion":
        cell16_no_unsupported_attribution,

    "source_identity_disclosure":
        cell16_source_opacity_valid,

    "locator_fabrication":
        True,

    "unknown_speaker_inference":
        True,
}


cell16_semantic_gates = {
    "task_family_alignment":
        cell16_task_family_alignment,

    "question_archetype_alignment":
        cell16_question_archetype_alignment,

    "evidence_scope_alignment":
        cell16_question_groundable_in_target,

    "question_is_answerable_without_external_context":
        cell16_external_knowledge_dependency_absent,

    "question_is_not_trivial_metadata_repetition":
        cell16_question_not_metadata_repetition,

    "required_dimensions_are_testable_by_question":
        cell16_required_dimensions_alignment,

    "authority_expectation_alignment":
        (
            cell15_authority_expectation
            is not None
        ),

    "context_expectation_alignment":
        (
            cell15_context_expectation
            is not None
        ),
}


cell16_machine_validation_pass = all([
    all(
        cell16_hard_gates.values()
    ),

    all(
        cell16_semantic_gates.values()
    ),
])


cell16_machine_validation_status = (
    "PASS_PENDING_HUMAN_ADJUDICATION"

    if cell16_machine_validation_pass

    else "FAIL_REQUIRES_REVISION"
)


# ============================================================
# 12) Build revised Question Draft Record
# ============================================================

cell16_question_draft_record = {
    "question_draft_id":
        cell16_question_draft_id,

    "question_draft_schema_version":
        cell15_question_schema_version,

    "pilot_case_id":
        CELL16_PILOT_ID,

    "golden_case_id":
        CELL16_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL16_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL16_TARGET_EVIDENCE_ID,

    "source_type":
        cell14_source_type,

    "stratum":
        cell14_stratum,

    "task_family":
        cell15_task_family,

    "question_archetype":
        cell15_question_archetype,

    "required_dimensions":
        copy.deepcopy(
            cell15_required_dimensions
        ),

    "authority_expectation":
        copy.deepcopy(
            cell15_authority_expectation
        ),

    "context_expectation":
        copy.deepcopy(
            cell15_context_expectation
        ),

    "evidence_scope":
        copy.deepcopy(
            cell15_evidence_scope
        ),

    "question_text":
        cell16_question_text,

    "question_text_sha256":
        cell16_question_text_sha256,

    "authoring_mode":
        cell15_authoring_mode,

    "draft_status":
        "DRAFT_CREATED_PENDING_VALIDATION",

    "machine_validation_status":
        cell16_machine_validation_status,

    "human_adjudication_status":
        "PENDING",

    "expected_answer_status":
        "BLOCKED_PENDING_QUESTION_ACCEPTANCE",

    "citation_target_validation_status":
        "BLOCKED_PENDING_QUESTION_ACCEPTANCE",

    "revision_lineage":
        {
            "revision_number":
                2,

            "revises_question_draft_id":
                cell16_previous_question_draft_id,

            "revision_trigger":
                "HUMAN_REVIEW_REVISE",

            "human_review_record_sha256":
                cell16_human_review_record_sha256,
        },

    "machine_validation":
        {
            "hard_gates":
                copy.deepcopy(
                    cell16_hard_gates
                ),

            "semantic_gates":
                copy.deepcopy(
                    cell16_semantic_gates
                ),

            "internal_id_hits":
                copy.deepcopy(
                    cell16_internal_id_hits
                ),

            "internal_metadata_hits":
                copy.deepcopy(
                    cell16_internal_metadata_hits
                ),

            "source_identity_hits":
                copy.deepcopy(
                    cell16_source_identity_hits
                ),

            "unsupported_attribution_hits":
                copy.deepcopy(
                    cell16_unsupported_attribution_hits
                ),

            "answer_leakage_hits":
                copy.deepcopy(
                    cell16_answer_leakage_hits
                ),
        },

    "full_572_generation":
        CELL16_FULL_572_BLOCK_STATUS,
}


cell16_question_draft_record_sha256 = (
    cell10_stable_sha(
        cell16_question_draft_record
    )
)


# ============================================================
# 13) Schema validation
# ============================================================

cell16_missing_required_fields = [
    field

    for field
    in cell15_schema_required_fields

    if field
    not in
    cell16_question_draft_record
]


cell16_empty_required_fields = [
    field

    for field
    in cell15_schema_required_fields

    if (
        field
        in
        cell16_question_draft_record
        and
        cell16_question_draft_record[
            field
        ]
        is None
    )
]


cell16_schema_surface_valid = all([
    len(
        cell16_missing_required_fields
    )
    ==
    0,

    len(
        cell16_empty_required_fields
    )
    ==
    0,
])


# ============================================================
# 14) ID reproducibility
# ============================================================

cell16_rebuilt_question_draft_id = (
    cell12_question_id_function(
        pilot_case_id=
            CELL16_PILOT_ID,

        question_text_sha256=
            cell16_question_text_sha256,
    )
)


cell16_question_id_reproducible = (
    cell16_rebuilt_question_draft_id
    ==
    cell16_question_draft_id
)


# ============================================================
# 15) Register revised Draft
#
# Session returns to Human Review.
# ============================================================

cell16_current_session = None


if all([
    cell16_revision_event_valid,
    cell16_machine_validation_pass,
    cell16_schema_surface_valid,
    cell16_question_id_reproducible,
    cell16_new_id_differs_from_previous,
]):

    cell16_current_session = (
        cell13_apply_event(
            cell16_revision_session,
            "QUESTION_DRAFT_REGISTERED",
            artifact=
                cell16_question_draft_record,
        )
    )


# ============================================================
# 16) Human Review lineage package
#
# This remains in-memory for later Human Review Notebook export.
# ============================================================

cell16_human_review_case = {
    "human_review_case_version":
        "human_review_case_v1",

    "pilot_case_id":
        CELL16_PILOT_ID,

    "golden_case_id":
        CELL16_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL16_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL16_TARGET_EVIDENCE_ID,

    "source_type":
        cell14_source_type,

    "stratum":
        cell14_stratum,

    "source_evidence":
        copy.deepcopy(
            cell14_target_texts
        ),

    "model_drafts":
        [
            {
                "revision":
                    1,

                "question_draft_id":
                    cell16_previous_question_draft_id,

                "question_text":
                    cell16_previous_question_text,

                "machine_validation":
                    cell16_previous_question_draft.get(
                        "machine_validation_status"
                    ),

                "human_decision":
                    "REVISE",
            },
            {
                "revision":
                    2,

                "question_draft_id":
                    cell16_question_draft_id,

                "question_text":
                    cell16_question_text,

                "machine_validation":
                    cell16_machine_validation_status,

                "human_decision":
                    "PENDING",
            },
        ],

    "human_reviews":
        [
            copy.deepcopy(
                cell16_human_review_record
            )
        ],

    "author_final_review":
        {
            "reviewer":
                "AUTHOR",

            "status":
                "NOT_STARTED",

            "decision":
                None,

            "notes":
                None,
        },

    "final_question_draft_id":
        None,

    "final_status":
        "PENDING_HUMAN_QUESTION_ACCEPTANCE",
}


cell16_human_review_case_sha256 = (
    cell10_stable_sha(
        cell16_human_review_case
    )
)


# ============================================================
# 17) Post-registration validation
# ============================================================

cell16_revised_question_registered = all([
    isinstance(
        cell16_current_session,
        dict,
    ),

    cell16_current_session.get(
        "current_state"
    )
    ==
    "QUESTION_HUMAN_REVIEW_PENDING",

    cell16_current_session[
        "artifacts"
    ][
        "question_draft"
    ].get(
        "question_draft_id"
    )
    ==
    cell16_question_draft_id,

    cell16_current_session[
        "artifacts"
    ][
        "question_draft"
    ].get(
        "question_text"
    )
    ==
    cell16_question_text,

    cell16_current_session[
        "artifacts"
    ][
        "expected_answer"
    ]
    is None,

    cell16_current_session[
        "artifacts"
    ][
        "citation_mapping"
    ]
    is None,

    len(
        cell16_current_session.get(
            "human_decisions",
            [],
        )
    )
    ==
    1,

    cell16_current_session[
        "human_decisions"
    ][
        0
    ].get(
        "event"
    )
    ==
    "QUESTION_REVISE",

    cell16_current_session.get(
        "queue_advance_permitted"
    )
    is False,

    cell16_current_session.get(
        "queue_advance_performed"
    )
    is False,
])


# ============================================================
# 18) Preserve original accepted states
# ============================================================

cell16_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell14_seed_before_sha256
)


cell16_cell15_session_unchanged = (
    cell10_stable_sha(
        cell15_current_session
    )
    ==
    cell10_stable_sha(
        copy.deepcopy(
            cell15_current_session
        )
    )
)


cell16_queue_still_22 = (
    len(
        cell13_seed[
            "final_runtime"
        ][
            "cell95_remaining_pilot_ids"
        ]
    )
    ==
    22
)


cell16_next_pilot_unchanged = (
    cell13_seed[
        "final_runtime"
    ][
        "cell95_next_pilot_id"
    ]
    ==
    CELL16_PILOT_ID
)


cell16_full_572_blocked = all([
    isinstance(
        cell16_current_session,
        dict,
    ),

    cell16_current_session.get(
        "full_572_generation"
    )
    ==
    CELL16_FULL_572_BLOCK_STATUS,

    cell16_question_draft_record[
        "full_572_generation"
    ]
    ==
    CELL16_FULL_572_BLOCK_STATUS,
])


# ============================================================
# 19) Final Gate
# ============================================================

cell16_gate = all([
    len(
        cell16_failures
    )
    ==
    0,

    cell16_revision_event_valid,

    cell16_new_id_differs_from_previous,

    cell16_question_nonempty,

    cell16_question_looks_natural,

    cell16_question_groundable_in_target,

    cell16_machine_validation_pass,

    cell16_schema_surface_valid,

    cell16_question_id_reproducible,

    cell16_revised_question_registered,

    cell16_seed_unchanged,

    cell16_cell15_session_unchanged,

    cell16_queue_still_22,

    cell16_next_pilot_unchanged,

    cell16_full_572_blocked,
])


# ============================================================
# 20) Diagnostics
# ============================================================

print(
    "=== HUMAN REVISION + "
    "REVISED REAL QUESTION DRAFT ==="
)


print(
    "\n=== SOURCE EVIDENCE ==="
)

print(
    cell14_combined_target_text
)


print(
    "\n=== PREVIOUS MODEL DRAFT ==="
)

print(
    "Question Draft ID:",
    cell16_previous_question_draft_id,
)

print(
    "Question:",
    cell16_previous_question_text,
)


print(
    "\n=== HUMAN REVIEW ==="
)

print(
    "Decision:",
    CELL16_HUMAN_DECISION,
)

print(
    "Reason:",
    CELL16_HUMAN_REVIEW_REASON,
)

print(
    "Human context evidence status:",
    "NON_EVIDENCE",
)

print(
    "Human Review SHA256:",
    cell16_human_review_record_sha256,
)


print(
    "\n=== REVISED MODEL DRAFT ==="
)

print(
    cell16_question_text
)

print(
    "\nPrevious Draft ID:",
    cell16_previous_question_draft_id,
)

print(
    "Revised Draft ID:",
    cell16_question_draft_id,
)

print(
    "New ID differs:",
    cell16_new_id_differs_from_previous,
)

print(
    "ID reproducible:",
    cell16_question_id_reproducible,
)


print(
    "\n=== HARD GATES ==="
)

for name, status in (
    cell16_hard_gates.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== SEMANTIC GATES ==="
)

for name, status in (
    cell16_semantic_gates.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== MACHINE VALIDATION ==="
)

print(
    "Status:",
    cell16_machine_validation_status,
)

print(
    "PASS:",
    cell16_machine_validation_pass,
)

print(
    "Internal ID hits:",
    cell16_internal_id_hits,
)

print(
    "Internal metadata hits:",
    cell16_internal_metadata_hits,
)

print(
    "Source identity hits:",
    cell16_source_identity_hits,
)

print(
    "Unsupported attribution hits:",
    cell16_unsupported_attribution_hits,
)

print(
    "Answer leakage hits:",
    cell16_answer_leakage_hits,
)


print(
    "\n=== REVISION LINEAGE ==="
)

print(
    "Revision:",
    2,
)

print(
    "Revises:",
    cell16_previous_question_draft_id,
)

print(
    "New Draft:",
    cell16_question_draft_id,
)

print(
    "Draft record SHA256:",
    cell16_question_draft_record_sha256,
)

print(
    "Human Review Case SHA256:",
    cell16_human_review_case_sha256,
)


print(
    "\n=== REAL WORKFLOW STATE ==="
)

print(
    "Revision event valid:",
    cell16_revision_event_valid,
)

print(
    "Revised Question registered:",
    cell16_revised_question_registered,
)

print(
    "Current state:",
    (
        cell16_current_session.get(
            "current_state"
        )

        if isinstance(
            cell16_current_session,
            dict,
        )

        else None
    ),
)

print(
    "Human decisions recorded:",
    (
        len(
            cell16_current_session.get(
                "human_decisions",
                [],
            )
        )

        if isinstance(
            cell16_current_session,
            dict,
        )

        else None
    ),
)

print(
    "Expected Answer:",
    (
        cell16_current_session[
            "artifacts"
        ][
            "expected_answer"
        ]

        if isinstance(
            cell16_current_session,
            dict,
        )

        else None
    ),
)

print(
    "Citation:",
    (
        cell16_current_session[
            "artifacts"
        ][
            "citation_mapping"
        ]

        if isinstance(
            cell16_current_session,
            dict,
        )

        else None
    ),
)


print(
    "\n=== QUEUE / SAFETY ==="
)

print(
    "Seed unchanged:",
    cell16_seed_unchanged,
)

print(
    "Queue still 22:",
    cell16_queue_still_22,
)

print(
    "Next Pilot unchanged:",
    cell16_next_pilot_unchanged,
)

print(
    "Queue advance permitted:",
    (
        cell16_current_session.get(
            "queue_advance_permitted"
        )

        if isinstance(
            cell16_current_session,
            dict,
        )

        else None
    ),
)

print(
    "Full 572 blocked:",
    cell16_full_572_blocked,
)

print(
    "Human Review Notebook file written:",
    False,
)

print(
    "Project/file mutation performed:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell16_failures
)


print(
    "\nCODE CELL 16 GATE:",
    (
        "PASS"
        if cell16_gate
        else "FAIL"
    ),
)


assert cell16_gate, (
    "Human Revision / revised Question Draft validation "
    "failed. Repair this SAME Code Cell 16 before final "
    "Human Question Acceptance."
)

=== HUMAN REVISION + REVISED REAL QUESTION DRAFT ===

=== SOURCE EVIDENCE ===
تلقی مردم از فیل، به عنوان ناودان و ستون و تخت و بادبزن، آنجا تبدیل به مشکل می شود که بخواهند فیل را بر اساس همان تصویر خود، به کار بگیرند. استفاده از فیل به عنوان ستون ممکن است مناسب به نظر برسد اما فیل یک کل به‌هم‌ پیوسته است و در واقعیت، پای فیل جدای گوش و خرطوم و عاجش نیست.

=== PREVIOUS MODEL DRAFT ===
Question Draft ID: question-draft:2e1c5547776e71cb5b216fe155dd5a54
Question: چرا به کار گرفتن فیل بر اساس تصویری که فقط از یکی از بخش‌های آن داریم، می‌تواند مشکل‌ساز باشد؟

=== HUMAN REVIEW ===
Decision: REVISE
Reason: سؤال قبلی بیش از حد خودِ مثال فیل را موضوع سؤال قرار می‌داد. سؤال باید برای مخاطب عمومی ساده‌تر باشد و روی مفهوم اصلی متن یعنی خطر قضاوت یا تصمیم‌گیری درباره کل موضوع بر اساس شناخت ناقص از یک بخش تمرکز کند.
Human context evidence status: NON_EVIDENCE
Human Review SHA256: d96ffd1d2e7e1806a6719ece002a5885ea5f15d418335ff9f2c25fbc35672477

=== REVISED MODEL DRAFT ===
چه مشکلی پیش می‌آید اگر بر ا

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 17 — ثبت Human ACCEPT سؤال و ساخت Expected Answer Draft

Question Draft اصلاح‌شده توسط Human Reviewer پذیرفته شد.

Question نهایی این مرحله:

**«چه مشکلی پیش می‌آید اگر بر اساس شناختی که فقط از یک بخش داریم، درباره کل موضوع تصمیم بگیریم؟»**

در این Cell دو کار انجام می‌شود:

1. تصمیم انسانی `ACCEPT` برای Question Draft ثبت می‌شود؛
2. Expected Answer Draft فقط بر اساس Target Evidence معتبر فعلی ساخته می‌شود.

Expected Answer اجازه ندارد از بحث‌ها، برداشت‌ها یا مثال‌های اضافه‌ی Human Reviewer یا مدل استفاده کند، مگر آنکه آن مطالب در Evidence معتبر وجود داشته باشند.

بنابراین مطالبی مانند نسبت‌دادن داستان به مثنوی، مفهوم نور، یا توسعه‌ی آن به سواد مالی در این Expected Answer وارد نمی‌شوند، چون در Target Evidence فعلی وجود ندارند.

Expected Answer پیشنهادی:

**«مشکل این است که ممکن است همان شناخت ناقص را به کل موضوع تعمیم بدهیم و بخواهیم بر اساس آن تصمیم بگیریم یا از کل موضوع استفاده کنیم. چنین کاری شاید در ظاهر مناسب به نظر برسد، اما بخش‌های یک موضوع می‌توانند به هم وابسته باشند و نمی‌شود یک بخش را کاملاً جدا از بقیه در نظر گرفت. در مثال متن نیز فیل یک کل به‌هم‌پیوسته است و شناخت یک قسمت از آن، به‌تنهایی تصویر کاملی از فیل نمی‌دهد.»**

این Answer پس از Machine Validation فقط وارد `EXPECTED_ANSWER_HUMAN_REVIEW_PENDING` می‌شود.

در این مرحله Citation نهایی تأیید نمی‌شود، Queue جلو نمی‌رود و Full 572 Generation همچنان Blocked باقی می‌ماند.

</div>

In [17]:
# Code Cell 17

cell17_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if (
    globals().get(
        "cell16_gate"
    )
    is not True
):
    cell17_failures.append(
        "CELL16_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell16_current_session"
    ),
    dict,
):
    cell17_failures.append(
        "CELL16_CURRENT_SESSION_MISSING"
    )


if (
    globals().get(
        "cell16_current_session",
        {},
    ).get(
        "current_state"
    )
    !=
    "QUESTION_HUMAN_REVIEW_PENDING"
):
    cell17_failures.append(
        "QUESTION_NOT_AT_HUMAN_REVIEW_GATE"
    )


# ============================================================
# 2) Fixed identities
# ============================================================

CELL17_PILOT_ID = (
    "question-pilot:"
    "3005b48518a7aeb532234332df23c952"
)


CELL17_GOLDEN_CASE_ID = (
    "golden-case:"
    "3b93ce20ef4b617b9f922baddfbd1e0f"
)


CELL17_TASK_DRAFT_ID = (
    "task-draft:"
    "f48d78f0a6a588de906ca8534219a604"
)


CELL17_TARGET_EVIDENCE_ID = (
    "target-evidence:"
    "81011a6ef78817e01e3ed484b1b4c368"
)


CELL17_QUESTION_DRAFT_ID = (
    "question-draft:"
    "e6ab324c3d0f766c7f858230eded54b1"
)


CELL17_FULL_572_BLOCK_STATUS = (
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


# ============================================================
# 3) Verify accepted Question Draft
# ============================================================

cell17_question_draft = copy.deepcopy(
    cell16_current_session[
        "artifacts"
    ][
        "question_draft"
    ]
)


cell17_question_identity_valid = all([
    cell17_question_draft.get(
        "question_draft_id"
    )
    ==
    CELL17_QUESTION_DRAFT_ID,

    cell17_question_draft.get(
        "pilot_case_id"
    )
    ==
    CELL17_PILOT_ID,

    cell17_question_draft.get(
        "target_evidence_id"
    )
    ==
    CELL17_TARGET_EVIDENCE_ID,

    cell17_question_draft.get(
        "machine_validation_status"
    )
    ==
    "PASS_PENDING_HUMAN_ADJUDICATION",
])


# ============================================================
# 4) Human ACCEPT record
# ============================================================

CELL17_HUMAN_QUESTION_DECISION = (
    "ACCEPT"
)


CELL17_HUMAN_QUESTION_NOTE = (
    "Human Reviewer accepted revision 2 as the current "
    "question wording for this Pilot."
)


cell17_question_accept_review = {
    "review_record_version":
        "human_question_review_record_v1",

    "pilot_case_id":
        CELL17_PILOT_ID,

    "golden_case_id":
        CELL17_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL17_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL17_TARGET_EVIDENCE_ID,

    "review_stage":
        "QUESTION_DRAFT",

    "reviewed_question_draft_id":
        CELL17_QUESTION_DRAFT_ID,

    "reviewed_question_text":
        cell17_question_draft.get(
            "question_text"
        ),

    "human_decision":
        CELL17_HUMAN_QUESTION_DECISION,

    "human_note":
        CELL17_HUMAN_QUESTION_NOTE,

    "evidence_policy":
        (
            "HUMAN_OR_MODEL_INTERPRETATION_IS_NOT_SOURCE_"
            "EVIDENCE_UNLESS_PRESENT_IN_VALIDATED_TARGET"
        ),
}


cell17_question_accept_review_sha256 = (
    cell10_stable_sha(
        cell17_question_accept_review
    )
)


# ============================================================
# 5) Apply real Question ACCEPT
# ============================================================

cell17_answer_pending_session = (
    cell13_apply_event(
        cell16_current_session,
        "QUESTION_ACCEPTED",
        human_note=
            CELL17_HUMAN_QUESTION_NOTE,
    )
)


cell17_question_accept_transition_valid = all([
    cell17_answer_pending_session.get(
        "current_state"
    )
    ==
    "EXPECTED_ANSWER_DRAFT_PENDING",

    len(
        cell17_answer_pending_session.get(
            "human_decisions",
            [],
        )
    )
    ==
    2,

    cell17_answer_pending_session[
        "human_decisions"
    ][
        -1
    ].get(
        "event"
    )
    ==
    "QUESTION_ACCEPTED",

    cell17_answer_pending_session.get(
        "queue_advance_permitted"
    )
    is False,
])


# ============================================================
# 6) Expected Answer Draft
#
# SOURCE-ONLY:
# No human discussion or external knowledge is allowed.
# ============================================================

CELL17_EXPECTED_ANSWER_TEXT = (
    "مشکل این است که ممکن است همان شناخت ناقص را به کل "
    "موضوع تعمیم بدهیم و بخواهیم بر اساس آن تصمیم بگیریم "
    "یا از کل موضوع استفاده کنیم. چنین کاری شاید در ظاهر "
    "مناسب به نظر برسد، اما بخش‌های یک موضوع می‌توانند به "
    "هم وابسته باشند و نمی‌شود یک بخش را کاملاً جدا از بقیه "
    "در نظر گرفت. در مثال متن نیز فیل یک کل به‌هم‌پیوسته است "
    "و شناخت یک قسمت از آن، به‌تنهایی تصویر کاملی از فیل "
    "نمی‌دهد."
)


cell17_expected_answer_text = (
    CELL17_EXPECTED_ANSWER_TEXT.strip()
)


cell17_expected_answer_text_sha256 = (
    hashlib.sha256(
        cell17_expected_answer_text.encode(
            "utf-8"
        )
    ).hexdigest()
)


# ============================================================
# 7) Evidence normalization
# ============================================================

cell17_normalized_evidence = (
    cell15_normalize_text(
        cell14_combined_target_text
    )
)


cell17_normalized_answer = (
    cell15_normalize_text(
        cell17_expected_answer_text
    )
)


# ============================================================
# 8) Source-only grounding checks
# ============================================================

cell17_grounding_checks = {
    "partial_view_or_incomplete_image":
        (
            "شناخت ناقص"
            in
            cell17_normalized_answer
        ),

    "whole_subject":
        (
            "کل موضوع"
            in
            cell17_normalized_answer
        ),

    "decision_or_use":
        (
            "تصمیم"
            in
            cell17_normalized_answer
            or
            "استفاده"
            in
            cell17_normalized_answer
        ),

    "apparently_suitable":
        (
            "در ظاهر مناسب"
            in
            cell17_normalized_answer
        ),

    "connected_whole":
        (
            "کل به هم پیوسته"
            in
            cell17_normalized_answer
            or
            "کل به‌هم پیوسته"
            in
            cell17_expected_answer_text
        ),

    "elephant_example":
        (
            "فیل"
            in
            cell17_normalized_answer
        ),
}


cell17_grounding_checks_valid = all(
    cell17_grounding_checks.values()
)


cell17_evidence_support_checks = {
    "evidence_has_image_concept":
        (
            "تصویر"
            in
            cell17_normalized_evidence
        ),

    "evidence_has_use_concept":
        (
            "به کار بگیرند"
            in
            cell17_normalized_evidence
        ),

    "evidence_has_apparent_suitability":
        (
            "مناسب به نظر برسد"
            in
            cell17_normalized_evidence
        ),

    "evidence_has_connected_whole":
        (
            "کل به هم پیوسته"
            in
            cell17_normalized_evidence
        ),

    "evidence_has_nonseparable_parts":
        (
            "جدای گوش"
            in
            cell17_normalized_evidence
        ),
}


cell17_evidence_support_valid = all(
    cell17_evidence_support_checks.values()
)


# ============================================================
# 9) Guard against unsupported additions
# ============================================================

CELL17_FORBIDDEN_UNSUPPORTED_TERMS = [
    "مولوی",
    "مثنوی",
    "مثنوی معنوی",
    "طویله",
    "تاریکی",
    "نور",
    "سواد مالی",
    "دانش مالی",
    "آگاهی مالی",
    "سرمایه گذاری",
    "سرمایه‌گذاری",
]


cell17_unsupported_term_hits = [
    term

    for term
    in CELL17_FORBIDDEN_UNSUPPORTED_TERMS

    if cell15_normalize_text(
        term
    )
    in
    cell17_normalized_answer
]


cell17_no_unsupported_additions = (
    len(
        cell17_unsupported_term_hits
    )
    ==
    0
)


# ============================================================
# 10) Source opacity / internal metadata
# ============================================================

cell17_internal_id_hits = [
    pattern

    for pattern
    in CELL15_INTERNAL_ID_PATTERNS

    if re.search(
        pattern,
        cell17_expected_answer_text,
        flags=re.IGNORECASE,
    )
]


cell17_internal_metadata_hits = [
    term

    for term
    in CELL15_INTERNAL_METADATA_TERMS

    if term.lower()
    in
    cell17_expected_answer_text.lower()
]


cell17_source_identity_hits = [
    term

    for term
    in CELL15_SOURCE_IDENTITY_TERMS

    if term.lower()
    in
    cell17_expected_answer_text.lower()
]


cell17_no_internal_id_leakage = (
    len(
        cell17_internal_id_hits
    )
    ==
    0
)


cell17_no_internal_metadata_leakage = (
    len(
        cell17_internal_metadata_hits
    )
    ==
    0
)


cell17_source_identity_opacity_valid = (
    len(
        cell17_source_identity_hits
    )
    ==
    0
)


# ============================================================
# 11) Answer quality / alignment
# ============================================================

cell17_answer_nonempty = (
    bool(
        cell17_expected_answer_text
    )
)


cell17_answer_is_plain_language = all([
    len(
        cell17_expected_answer_text
    )
    >
    40,

    "مشکل"
    in
    cell17_normalized_answer,

    "کل"
    in
    cell17_normalized_answer,

    "بخش"
    in
    cell17_normalized_answer,
])


cell17_answer_addresses_question = all([
    "چه مشکلی"
    in
    cell15_normalize_text(
        cell16_question_text
    ),

    "مشکل"
    in
    cell17_normalized_answer,

    "تصمیم"
    in
    cell17_normalized_answer,

    "بخش"
    in
    cell17_normalized_answer,

    "کل"
    in
    cell17_normalized_answer,
])


cell17_answer_machine_validation = {
    "nonempty":
        cell17_answer_nonempty,

    "plain_language":
        cell17_answer_is_plain_language,

    "addresses_question":
        cell17_answer_addresses_question,

    "grounded_in_target":
        cell17_grounding_checks_valid,

    "target_contains_required_support":
        cell17_evidence_support_valid,

    "no_unsupported_additions":
        cell17_no_unsupported_additions,

    "no_internal_id_leakage":
        cell17_no_internal_id_leakage,

    "no_internal_metadata_leakage":
        cell17_no_internal_metadata_leakage,

    "source_identity_opacity":
        cell17_source_identity_opacity_valid,

    "external_knowledge_used":
        False,
}


cell17_answer_machine_validation_pass = all([
    cell17_answer_machine_validation[
        "nonempty"
    ],

    cell17_answer_machine_validation[
        "plain_language"
    ],

    cell17_answer_machine_validation[
        "addresses_question"
    ],

    cell17_answer_machine_validation[
        "grounded_in_target"
    ],

    cell17_answer_machine_validation[
        "target_contains_required_support"
    ],

    cell17_answer_machine_validation[
        "no_unsupported_additions"
    ],

    cell17_answer_machine_validation[
        "no_internal_id_leakage"
    ],

    cell17_answer_machine_validation[
        "no_internal_metadata_leakage"
    ],

    cell17_answer_machine_validation[
        "source_identity_opacity"
    ],
])


# ============================================================
# 12) Stable local Expected Answer Draft ID
#
# This is a continuation-workflow draft identity.
# It does not claim a pre-existing global ID contract.
# ============================================================

cell17_expected_answer_identity_payload = {
    "identity_version":
        "expected_answer_draft_identity_v1",

    "pilot_case_id":
        CELL17_PILOT_ID,

    "question_draft_id":
        CELL17_QUESTION_DRAFT_ID,

    "target_evidence_id":
        CELL17_TARGET_EVIDENCE_ID,

    "expected_answer_text_sha256":
        cell17_expected_answer_text_sha256,
}


cell17_expected_answer_draft_id = (
    "expected-answer-draft:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell17_expected_answer_identity_payload
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


# ============================================================
# 13) Expected Answer Draft artifact
# ============================================================

cell17_expected_answer_record = {
    "record_version":
        "expected_answer_draft_record_v1",

    "expected_answer_draft_id":
        cell17_expected_answer_draft_id,

    "pilot_case_id":
        CELL17_PILOT_ID,

    "golden_case_id":
        CELL17_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL17_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL17_TARGET_EVIDENCE_ID,

    "question_draft_id":
        CELL17_QUESTION_DRAFT_ID,

    "question_text":
        cell16_question_text,

    "expected_answer_text":
        cell17_expected_answer_text,

    "expected_answer_text_sha256":
        cell17_expected_answer_text_sha256,

    "evidence_source_policy":
        "CURRENT_VALIDATED_TARGET_EVIDENCE_ONLY",

    "external_knowledge_used":
        False,

    "human_context_used_as_evidence":
        False,

    "machine_validation_status":
        (
            "PASS_PENDING_HUMAN_ADJUDICATION"

            if cell17_answer_machine_validation_pass

            else "FAIL_REQUIRES_REVISION"
        ),

    "machine_validation":
        copy.deepcopy(
            cell17_answer_machine_validation
        ),

    "human_adjudication_status":
        "PENDING",

    "citation_validation_status":
        "BLOCKED_PENDING_EXPECTED_ANSWER_ACCEPTANCE",

    "full_572_generation":
        CELL17_FULL_572_BLOCK_STATUS,
}


cell17_expected_answer_record_sha256 = (
    cell10_stable_sha(
        cell17_expected_answer_record
    )
)


# ============================================================
# 14) Register Expected Answer Draft
# ============================================================

cell17_current_session = None


if all([
    cell17_question_accept_transition_valid,
    cell17_answer_machine_validation_pass,
]):

    cell17_current_session = (
        cell13_apply_event(
            cell17_answer_pending_session,
            "EXPECTED_ANSWER_REGISTERED",
            artifact=
                cell17_expected_answer_record,
        )
    )


# ============================================================
# 15) Update Human Review lineage in memory
# ============================================================

cell17_human_review_case = copy.deepcopy(
    cell16_human_review_case
)


for model_draft in (
    cell17_human_review_case.get(
        "model_drafts",
        [],
    )
):

    if (
        model_draft.get(
            "question_draft_id"
        )
        ==
        CELL17_QUESTION_DRAFT_ID
    ):

        model_draft[
            "human_decision"
        ] = "ACCEPT"


cell17_human_review_case[
    "human_reviews"
].append(
    copy.deepcopy(
        cell17_question_accept_review
    )
)


cell17_human_review_case[
    "final_question_draft_id"
] = (
    CELL17_QUESTION_DRAFT_ID
)


cell17_human_review_case[
    "final_status"
] = (
    "QUESTION_ACCEPTED_EXPECTED_ANSWER_PENDING_REVIEW"
)


cell17_human_review_case[
    "expected_answer_review"
] = {
    "expected_answer_draft_id":
        cell17_expected_answer_draft_id,

    "expected_answer_text":
        cell17_expected_answer_text,

    "machine_validation_status":
        cell17_expected_answer_record[
            "machine_validation_status"
        ],

    "human_decision":
        "PENDING",

    "source_evidence":
        copy.deepcopy(
            cell14_target_texts
        ),
}


cell17_human_review_case_sha256 = (
    cell10_stable_sha(
        cell17_human_review_case
    )
)


# ============================================================
# 16) Post-registration validation
# ============================================================

cell17_expected_answer_registered = all([
    isinstance(
        cell17_current_session,
        dict,
    ),

    cell17_current_session.get(
        "current_state"
    )
    ==
    "EXPECTED_ANSWER_HUMAN_REVIEW_PENDING",

    cell17_current_session[
        "artifacts"
    ][
        "question_draft"
    ].get(
        "question_draft_id"
    )
    ==
    CELL17_QUESTION_DRAFT_ID,

    isinstance(
        cell17_current_session[
            "artifacts"
        ][
            "expected_answer"
        ],
        dict,
    ),

    cell17_current_session[
        "artifacts"
    ][
        "expected_answer"
    ].get(
        "expected_answer_draft_id"
    )
    ==
    cell17_expected_answer_draft_id,

    cell17_current_session[
        "artifacts"
    ][
        "citation_mapping"
    ]
    is None,

    len(
        cell17_current_session.get(
            "human_decisions",
            [],
        )
    )
    ==
    2,

    cell17_current_session.get(
        "queue_advance_permitted"
    )
    is False,

    cell17_current_session.get(
        "queue_advance_performed"
    )
    is False,
])


# ============================================================
# 17) Preserve base Seed / Queue
# ============================================================

cell17_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell14_seed_before_sha256
)


cell17_queue_still_22 = (
    len(
        cell13_seed[
            "final_runtime"
        ][
            "cell95_remaining_pilot_ids"
        ]
    )
    ==
    22
)


cell17_next_pilot_unchanged = (
    cell13_seed[
        "final_runtime"
    ][
        "cell95_next_pilot_id"
    ]
    ==
    CELL17_PILOT_ID
)


cell17_full_572_blocked = all([
    isinstance(
        cell17_current_session,
        dict,
    ),

    cell17_current_session.get(
        "full_572_generation"
    )
    ==
    CELL17_FULL_572_BLOCK_STATUS,

    cell17_expected_answer_record[
        "full_572_generation"
    ]
    ==
    CELL17_FULL_572_BLOCK_STATUS,
])


# ============================================================
# 18) Final Gate
# ============================================================

cell17_gate = all([
    len(
        cell17_failures
    )
    ==
    0,

    cell17_question_identity_valid,

    cell17_question_accept_transition_valid,

    cell17_answer_machine_validation_pass,

    cell17_expected_answer_registered,

    cell17_seed_unchanged,

    cell17_queue_still_22,

    cell17_next_pilot_unchanged,

    cell17_full_572_blocked,
])


# ============================================================
# 19) Diagnostics
# ============================================================

print(
    "=== QUESTION ACCEPTED + "
    "EXPECTED ANSWER DRAFT ==="
)


print(
    "\n=== SOURCE EVIDENCE ==="
)

print(
    cell14_combined_target_text
)


print(
    "\n=== ACCEPTED QUESTION ==="
)

print(
    cell16_question_text
)

print(
    "Question Draft ID:",
    CELL17_QUESTION_DRAFT_ID,
)

print(
    "Human decision:",
    CELL17_HUMAN_QUESTION_DECISION,
)

print(
    "Question ACCEPT transition valid:",
    cell17_question_accept_transition_valid,
)


print(
    "\n=== EXPECTED ANSWER DRAFT ==="
)

print(
    cell17_expected_answer_text
)

print(
    "\nExpected Answer Draft ID:",
    cell17_expected_answer_draft_id,
)

print(
    "Expected Answer SHA256:",
    cell17_expected_answer_text_sha256,
)


print(
    "\n=== ANSWER GROUNDING ==="
)

for name, status in (
    cell17_grounding_checks.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== SOURCE SUPPORT ==="
)

for name, status in (
    cell17_evidence_support_checks.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== UNSUPPORTED ADDITION CHECK ==="
)

print(
    "Unsupported term hits:",
    cell17_unsupported_term_hits,
)

print(
    "Human context used as evidence:",
    False,
)

print(
    "External knowledge used:",
    False,
)


print(
    "\n=== ANSWER MACHINE VALIDATION ==="
)

for name, status in (
    cell17_answer_machine_validation.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "Machine validation PASS:",
    cell17_answer_machine_validation_pass,
)

print(
    "Answer record SHA256:",
    cell17_expected_answer_record_sha256,
)


print(
    "\n=== REAL WORKFLOW STATE ==="
)

print(
    "Current state:",
    (
        cell17_current_session.get(
            "current_state"
        )

        if isinstance(
            cell17_current_session,
            dict,
        )

        else None
    ),
)

print(
    "Expected Answer registered:",
    cell17_expected_answer_registered,
)

print(
    "Citation:",
    (
        cell17_current_session[
            "artifacts"
        ][
            "citation_mapping"
        ]

        if isinstance(
            cell17_current_session,
            dict,
        )

        else None
    ),
)

print(
    "Human decisions recorded:",
    (
        len(
            cell17_current_session.get(
                "human_decisions",
                [],
            )
        )

        if isinstance(
            cell17_current_session,
            dict,
        )

        else None
    ),
)


print(
    "\n=== HUMAN REVIEW PACKAGE ==="
)

print(
    "Question revisions:",
    len(
        cell17_human_review_case[
            "model_drafts"
        ]
    ),
)

print(
    "Human question reviews:",
    len(
        cell17_human_review_case[
            "human_reviews"
        ]
    ),
)

print(
    "Final Question Draft ID:",
    cell17_human_review_case[
        "final_question_draft_id"
    ],
)

print(
    "Expected Answer human decision:",
    cell17_human_review_case[
        "expected_answer_review"
    ][
        "human_decision"
    ],
)

print(
    "Human Review Case SHA256:",
    cell17_human_review_case_sha256,
)


print(
    "\n=== QUEUE / SAFETY ==="
)

print(
    "Seed unchanged:",
    cell17_seed_unchanged,
)

print(
    "Queue still 22:",
    cell17_queue_still_22,
)

print(
    "Next Pilot unchanged:",
    cell17_next_pilot_unchanged,
)

print(
    "Queue advance permitted:",
    (
        cell17_current_session.get(
            "queue_advance_permitted"
        )

        if isinstance(
            cell17_current_session,
            dict,
        )

        else None
    ),
)

print(
    "Full 572 blocked:",
    cell17_full_572_blocked,
)

print(
    "Project/file mutation performed:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell17_failures
)


print(
    "\nCODE CELL 17 GATE:",
    (
        "PASS"
        if cell17_gate
        else "FAIL"
    ),
)


assert cell17_gate, (
    "Question ACCEPT / Expected Answer Draft validation "
    "failed. Repair this SAME Code Cell 17 before Human "
    "Expected Answer Adjudication."
)

=== QUESTION ACCEPTED + EXPECTED ANSWER DRAFT ===

=== SOURCE EVIDENCE ===
تلقی مردم از فیل، به عنوان ناودان و ستون و تخت و بادبزن، آنجا تبدیل به مشکل می شود که بخواهند فیل را بر اساس همان تصویر خود، به کار بگیرند. استفاده از فیل به عنوان ستون ممکن است مناسب به نظر برسد اما فیل یک کل به‌هم‌ پیوسته است و در واقعیت، پای فیل جدای گوش و خرطوم و عاجش نیست.

=== ACCEPTED QUESTION ===
چه مشکلی پیش می‌آید اگر بر اساس شناختی که فقط از یک بخش داریم، درباره کل موضوع تصمیم بگیریم؟
Question Draft ID: question-draft:e6ab324c3d0f766c7f858230eded54b1
Human decision: ACCEPT
Question ACCEPT transition valid: True

=== EXPECTED ANSWER DRAFT ===
مشکل این است که ممکن است همان شناخت ناقص را به کل موضوع تعمیم بدهیم و بخواهیم بر اساس آن تصمیم بگیریم یا از کل موضوع استفاده کنیم. چنین کاری شاید در ظاهر مناسب به نظر برسد، اما بخش‌های یک موضوع می‌توانند به هم وابسته باشند و نمی‌شود یک بخش را کاملاً جدا از بقیه در نظر گرفت. در مثال متن نیز فیل یک کل به‌هم‌پیوسته است و شناخت یک قسمت از آن، به‌تنهایی تصویر کاملی از 

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 18 — ثبت Human ACCEPT جواب و ساخت Citation Draft

Expected Answer توسط Human Reviewer پذیرفته شد.

در این Cell:

- تصمیم انسانی `ACCEPT` برای Expected Answer ثبت می‌شود؛
- Citation فقط از Target Evidence و Locatorهای از قبل ثبت‌شده‌ی همان Evidence ساخته می‌شود؛
- هیچ URL، Document ID، محدوده‌ی متنی یا منبع جدیدی حدس زده نمی‌شود؛
- Citation Coverage و Locator Consistency به‌صورت ماشینی بررسی می‌شوند؛
- Human Review Lineage به‌روزرسانی می‌شود.

در صورت PASS، Workflow فقط تا:

`CITATION_HUMAN_REVIEW_PENDING`

جلو می‌رود.

Citation هنوز از نظر انسانی پذیرفته نشده است، Queue جلو نمی‌رود و Full 572 Generation همچنان Blocked باقی می‌ماند.

هیچ فایل، Commit، Push، PR یا Artifact خارجی ایجاد یا تغییر داده نمی‌شود.

</div>

In [18]:
# Code Cell 18

cell18_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell17_gate") is not True:
    cell18_failures.append("CELL17_NOT_PASS")


if not isinstance(
    globals().get("cell17_current_session"),
    dict,
):
    cell18_failures.append(
        "CELL17_CURRENT_SESSION_MISSING"
    )


if (
    globals().get(
        "cell17_current_session",
        {},
    ).get("current_state")
    !=
    "EXPECTED_ANSWER_HUMAN_REVIEW_PENDING"
):
    cell18_failures.append(
        "EXPECTED_ANSWER_NOT_AT_HUMAN_REVIEW_GATE"
    )


# ============================================================
# 2) Fixed identities
# ============================================================

CELL18_PILOT_ID = (
    "question-pilot:"
    "3005b48518a7aeb532234332df23c952"
)

CELL18_GOLDEN_CASE_ID = (
    "golden-case:"
    "3b93ce20ef4b617b9f922baddfbd1e0f"
)

CELL18_TASK_DRAFT_ID = (
    "task-draft:"
    "f48d78f0a6a588de906ca8534219a604"
)

CELL18_TARGET_EVIDENCE_ID = (
    "target-evidence:"
    "81011a6ef78817e01e3ed484b1b4c368"
)

CELL18_QUESTION_DRAFT_ID = (
    "question-draft:"
    "e6ab324c3d0f766c7f858230eded54b1"
)

CELL18_EXPECTED_ANSWER_DRAFT_ID = (
    "expected-answer-draft:"
    "b81cdf7a3a1935da65fb1fed8830e212"
)

CELL18_FULL_572_BLOCK_STATUS = (
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


# ============================================================
# 3) Verify Expected Answer artifact
# ============================================================

cell18_expected_answer_record = copy.deepcopy(
    cell17_current_session[
        "artifacts"
    ][
        "expected_answer"
    ]
)


cell18_expected_answer_identity_valid = all([
    cell18_expected_answer_record.get(
        "expected_answer_draft_id"
    )
    ==
    CELL18_EXPECTED_ANSWER_DRAFT_ID,

    cell18_expected_answer_record.get(
        "pilot_case_id"
    )
    ==
    CELL18_PILOT_ID,

    cell18_expected_answer_record.get(
        "question_draft_id"
    )
    ==
    CELL18_QUESTION_DRAFT_ID,

    cell18_expected_answer_record.get(
        "target_evidence_id"
    )
    ==
    CELL18_TARGET_EVIDENCE_ID,

    cell18_expected_answer_record.get(
        "machine_validation_status"
    )
    ==
    "PASS_PENDING_HUMAN_ADJUDICATION",
])


# ============================================================
# 4) Human ACCEPT for Expected Answer
# ============================================================

CELL18_HUMAN_ANSWER_DECISION = "ACCEPT"

CELL18_HUMAN_ANSWER_NOTE = (
    "Human Reviewer accepted the Expected Answer "
    "as grounded in the validated Target Evidence."
)


cell18_answer_accept_review = {
    "review_record_version":
        "human_expected_answer_review_record_v1",

    "pilot_case_id":
        CELL18_PILOT_ID,

    "golden_case_id":
        CELL18_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL18_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL18_TARGET_EVIDENCE_ID,

    "question_draft_id":
        CELL18_QUESTION_DRAFT_ID,

    "expected_answer_draft_id":
        CELL18_EXPECTED_ANSWER_DRAFT_ID,

    "review_stage":
        "EXPECTED_ANSWER",

    "expected_answer_text":
        cell18_expected_answer_record.get(
            "expected_answer_text"
        ),

    "human_decision":
        CELL18_HUMAN_ANSWER_DECISION,

    "human_note":
        CELL18_HUMAN_ANSWER_NOTE,

    "source_policy":
        "VALIDATED_TARGET_EVIDENCE_ONLY",

    "human_context_used_as_evidence":
        False,

    "external_knowledge_used":
        False,
}


cell18_answer_accept_review_sha256 = (
    cell10_stable_sha(
        cell18_answer_accept_review
    )
)


# ============================================================
# 5) Apply real Expected Answer ACCEPT
# ============================================================

cell18_citation_pending_session = (
    cell13_apply_event(
        cell17_current_session,
        "EXPECTED_ANSWER_ACCEPTED",
        human_note=
            CELL18_HUMAN_ANSWER_NOTE,
    )
)


cell18_answer_accept_transition_valid = all([
    cell18_citation_pending_session.get(
        "current_state"
    )
    ==
    "CITATION_MAPPING_PENDING",

    len(
        cell18_citation_pending_session.get(
            "human_decisions",
            [],
        )
    )
    ==
    3,

    cell18_citation_pending_session[
        "human_decisions"
    ][
        -1
    ].get(
        "event"
    )
    ==
    "EXPECTED_ANSWER_ACCEPTED",

    cell18_citation_pending_session.get(
        "queue_advance_permitted"
    )
    is False,

    cell18_citation_pending_session.get(
        "queue_advance_performed"
    )
    is False,
])


# ============================================================
# 6) Extract exact Target Parts
#
# No fabricated locator or source information is allowed.
# ============================================================

cell18_source_target_parts = copy.deepcopy(
    cell14_target_evidence.get(
        "target_parts",
        [],
    )
)


cell18_source_target_parts_valid = (
    isinstance(
        cell18_source_target_parts,
        list,
    )
    and
    len(
        cell18_source_target_parts
    )
    >
    0
)


cell18_citation_entries = []


if cell18_source_target_parts_valid:

    for index, part in enumerate(
        cell18_source_target_parts
    ):

        if not isinstance(part, dict):
            continue


        locator = copy.deepcopy(
            part.get(
                "locator",
                {},
            )
        )


        cell18_citation_entries.append({
            "citation_index":
                index,

            "target_evidence_id":
                CELL18_TARGET_EVIDENCE_ID,

            "document_id":
                part.get(
                    "document_id"
                ),

            "source_id":
                part.get(
                    "source_id"
                ),

            "role":
                part.get(
                    "role"
                ),

            "char_start":
                part.get(
                    "char_start"
                ),

            "char_end":
                part.get(
                    "char_end"
                ),

            "char_count":
                part.get(
                    "char_count"
                ),

            "token_count":
                part.get(
                    "token_count"
                ),

            "locator":
                locator,

            "attribution":
                copy.deepcopy(
                    part.get(
                        "attribution"
                    )
                ),

            "locator_origin":
                "COPIED_FROM_VALIDATED_TARGET_EVIDENCE",
        })


# ============================================================
# 7) Validate exact citation coverage
# ============================================================

cell18_target_text_by_document = {
    row.get("document_id"):
        row

    for row
    in cell14_target_texts

    if isinstance(row, dict)
}


cell18_citation_validation_rows = []


for entry in cell18_citation_entries:

    document_id = entry.get(
        "document_id"
    )


    text_part = (
        cell18_target_text_by_document.get(
            document_id
        )
    )


    locator = (
        entry.get(
            "locator"
        )
        or
        {}
    )


    canonical_url = (
        locator.get(
            "canonical_url"
        )
    )


    url = (
        locator.get(
            "url"
        )
    )


    row_valid = all([
        isinstance(
            document_id,
            str,
        )
        and
        bool(
            document_id
        ),

        isinstance(
            text_part,
            dict,
        ),

        entry.get(
            "char_start"
        )
        ==
        text_part.get(
            "char_start"
        ),

        entry.get(
            "char_end"
        )
        ==
        text_part.get(
            "char_end"
        ),

        entry.get(
            "role"
        )
        ==
        text_part.get(
            "role"
        ),

        isinstance(
            canonical_url,
            str,
        )
        and
        canonical_url.startswith(
            "https://"
        ),

        isinstance(
            url,
            str,
        )
        and
        url.startswith(
            "https://"
        ),

        entry.get(
            "locator_origin"
        )
        ==
        "COPIED_FROM_VALIDATED_TARGET_EVIDENCE",
    ])


    cell18_citation_validation_rows.append({
        "citation_index":
            entry.get(
                "citation_index"
            ),

        "document_id":
            document_id,

        "char_range_match":
            (
                isinstance(
                    text_part,
                    dict,
                )
                and
                entry.get(
                    "char_start"
                )
                ==
                text_part.get(
                    "char_start"
                )
                and
                entry.get(
                    "char_end"
                )
                ==
                text_part.get(
                    "char_end"
                )
            ),

        "role_match":
            (
                isinstance(
                    text_part,
                    dict,
                )
                and
                entry.get(
                    "role"
                )
                ==
                text_part.get(
                    "role"
                )
            ),

        "canonical_url_present":
            isinstance(
                canonical_url,
                str,
            )
            and
            canonical_url.startswith(
                "https://"
            ),

        "url_present":
            isinstance(
                url,
                str,
            )
            and
            url.startswith(
                "https://"
            ),

        "valid":
            row_valid,
    })


cell18_all_citation_entries_valid = all([
    len(
        cell18_citation_entries
    )
    ==
    len(
        cell14_target_texts
    ),

    len(
        cell18_citation_entries
    )
    >
    0,

    all(
        row.get(
            "valid"
        )
        is True

        for row
        in cell18_citation_validation_rows
    ),
])


# ============================================================
# 8) Exact-source / no-fabrication validation
# ============================================================

cell18_source_part_projection = [
    {
        "document_id":
            part.get(
                "document_id"
            ),

        "source_id":
            part.get(
                "source_id"
            ),

        "role":
            part.get(
                "role"
            ),

        "char_start":
            part.get(
                "char_start"
            ),

        "char_end":
            part.get(
                "char_end"
            ),

        "locator":
            copy.deepcopy(
                part.get(
                    "locator",
                    {},
                )
            ),
    }

    for part
    in cell18_source_target_parts

    if isinstance(
        part,
        dict,
    )
]


cell18_citation_projection = [
    {
        "document_id":
            entry.get(
                "document_id"
            ),

        "source_id":
            entry.get(
                "source_id"
            ),

        "role":
            entry.get(
                "role"
            ),

        "char_start":
            entry.get(
                "char_start"
            ),

        "char_end":
            entry.get(
                "char_end"
            ),

        "locator":
            copy.deepcopy(
                entry.get(
                    "locator",
                    {},
                )
            ),
    }

    for entry
    in cell18_citation_entries
]


cell18_no_locator_fabrication = (
    cell18_source_part_projection
    ==
    cell18_citation_projection
)


# ============================================================
# 9) Stable local Citation Mapping ID
# ============================================================

cell18_citation_identity_payload = {
    "identity_version":
        "citation_mapping_identity_v1",

    "pilot_case_id":
        CELL18_PILOT_ID,

    "question_draft_id":
        CELL18_QUESTION_DRAFT_ID,

    "expected_answer_draft_id":
        CELL18_EXPECTED_ANSWER_DRAFT_ID,

    "target_evidence_id":
        CELL18_TARGET_EVIDENCE_ID,

    "citation_projection":
        copy.deepcopy(
            cell18_citation_projection
        ),
}


cell18_citation_mapping_id = (
    "citation-mapping:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell18_citation_identity_payload
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


# ============================================================
# 10) Citation Machine Validation
# ============================================================

cell18_citation_machine_validation = {
    "target_parts_present":
        cell18_source_target_parts_valid,

    "citation_entries_present":
        (
            len(
                cell18_citation_entries
            )
            >
            0
        ),

    "all_target_parts_covered":
        (
            len(
                cell18_citation_entries
            )
            ==
            len(
                cell14_target_texts
            )
        ),

    "all_entries_valid":
        cell18_all_citation_entries_valid,

    "locator_fabrication_absent":
        cell18_no_locator_fabrication,

    "target_evidence_identity_match":
        all(
            entry.get(
                "target_evidence_id"
            )
            ==
            CELL18_TARGET_EVIDENCE_ID

            for entry
            in cell18_citation_entries
        ),

    "external_locator_lookup_used":
        False,
}


cell18_citation_machine_validation_pass = all([
    cell18_citation_machine_validation[
        "target_parts_present"
    ],

    cell18_citation_machine_validation[
        "citation_entries_present"
    ],

    cell18_citation_machine_validation[
        "all_target_parts_covered"
    ],

    cell18_citation_machine_validation[
        "all_entries_valid"
    ],

    cell18_citation_machine_validation[
        "locator_fabrication_absent"
    ],

    cell18_citation_machine_validation[
        "target_evidence_identity_match"
    ],
])


# ============================================================
# 11) Build Citation Mapping Record
# ============================================================

cell18_citation_mapping_record = {
    "record_version":
        "citation_mapping_record_v1",

    "citation_mapping_id":
        cell18_citation_mapping_id,

    "pilot_case_id":
        CELL18_PILOT_ID,

    "golden_case_id":
        CELL18_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL18_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL18_TARGET_EVIDENCE_ID,

    "question_draft_id":
        CELL18_QUESTION_DRAFT_ID,

    "expected_answer_draft_id":
        CELL18_EXPECTED_ANSWER_DRAFT_ID,

    "citation_entries":
        copy.deepcopy(
            cell18_citation_entries
        ),

    "citation_scope":
        "VALIDATED_TARGET_EVIDENCE_ONLY",

    "locator_source":
        "EXISTING_TARGET_EVIDENCE_LOCATORS",

    "external_locator_lookup_used":
        False,

    "machine_validation_status":
        (
            "PASS_PENDING_HUMAN_ADJUDICATION"

            if cell18_citation_machine_validation_pass

            else "FAIL_REQUIRES_REVISION"
        ),

    "machine_validation":
        copy.deepcopy(
            cell18_citation_machine_validation
        ),

    "human_adjudication_status":
        "PENDING",

    "full_572_generation":
        CELL18_FULL_572_BLOCK_STATUS,
}


cell18_citation_mapping_record_sha256 = (
    cell10_stable_sha(
        cell18_citation_mapping_record
    )
)


# ============================================================
# 12) Register Citation Draft
# ============================================================

cell18_current_session = None


if all([
    cell18_expected_answer_identity_valid,
    cell18_answer_accept_transition_valid,
    cell18_citation_machine_validation_pass,
]):

    cell18_current_session = (
        cell13_apply_event(
            cell18_citation_pending_session,
            "CITATION_MAPPING_REGISTERED",
            artifact=
                cell18_citation_mapping_record,
        )
    )


# ============================================================
# 13) Update Human Review package
# ============================================================

cell18_human_review_case = copy.deepcopy(
    cell17_human_review_case
)


cell18_human_review_case[
    "expected_answer_review"
][
    "human_decision"
] = "ACCEPT"


cell18_human_review_case[
    "expected_answer_review"
][
    "human_review"
] = copy.deepcopy(
    cell18_answer_accept_review
)


cell18_human_review_case[
    "citation_review"
] = {
    "citation_mapping_id":
        cell18_citation_mapping_id,

    "citation_entries":
        copy.deepcopy(
            cell18_citation_entries
        ),

    "machine_validation_status":
        cell18_citation_mapping_record[
            "machine_validation_status"
        ],

    "human_decision":
        "PENDING",
}


cell18_human_review_case[
    "final_status"
] = (
    "QUESTION_AND_ANSWER_ACCEPTED_"
    "CITATION_PENDING_REVIEW"
)


cell18_human_review_case_sha256 = (
    cell10_stable_sha(
        cell18_human_review_case
    )
)


# ============================================================
# 14) Post-registration validation
# ============================================================

cell18_citation_registered = all([
    isinstance(
        cell18_current_session,
        dict,
    ),

    cell18_current_session.get(
        "current_state"
    )
    ==
    "CITATION_HUMAN_REVIEW_PENDING",

    isinstance(
        cell18_current_session[
            "artifacts"
        ][
            "citation_mapping"
        ],
        dict,
    ),

    cell18_current_session[
        "artifacts"
    ][
        "citation_mapping"
    ].get(
        "citation_mapping_id"
    )
    ==
    cell18_citation_mapping_id,

    len(
        cell18_current_session.get(
            "human_decisions",
            [],
        )
    )
    ==
    3,

    cell18_current_session.get(
        "queue_advance_permitted"
    )
    is False,

    cell18_current_session.get(
        "queue_advance_performed"
    )
    is False,
])


# ============================================================
# 15) Preserve Seed / Queue
# ============================================================

cell18_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell14_seed_before_sha256
)


cell18_queue_still_22 = (
    len(
        cell13_seed[
            "final_runtime"
        ][
            "cell95_remaining_pilot_ids"
        ]
    )
    ==
    22
)


cell18_next_pilot_unchanged = (
    cell13_seed[
        "final_runtime"
    ][
        "cell95_next_pilot_id"
    ]
    ==
    CELL18_PILOT_ID
)


cell18_full_572_blocked = all([
    isinstance(
        cell18_current_session,
        dict,
    ),

    cell18_current_session.get(
        "full_572_generation"
    )
    ==
    CELL18_FULL_572_BLOCK_STATUS,

    cell18_citation_mapping_record[
        "full_572_generation"
    ]
    ==
    CELL18_FULL_572_BLOCK_STATUS,
])


# ============================================================
# 16) Final Gate
# ============================================================

cell18_gate = all([
    len(
        cell18_failures
    )
    ==
    0,

    cell18_expected_answer_identity_valid,

    cell18_answer_accept_transition_valid,

    cell18_source_target_parts_valid,

    cell18_all_citation_entries_valid,

    cell18_no_locator_fabrication,

    cell18_citation_machine_validation_pass,

    cell18_citation_registered,

    cell18_seed_unchanged,

    cell18_queue_still_22,

    cell18_next_pilot_unchanged,

    cell18_full_572_blocked,
])


# ============================================================
# 17) Diagnostics
# ============================================================

print(
    "=== EXPECTED ANSWER ACCEPTED + "
    "CITATION DRAFT ==="
)


print(
    "\n=== ACCEPTED QUESTION ==="
)

print(
    cell16_question_text
)


print(
    "\n=== ACCEPTED EXPECTED ANSWER ==="
)

print(
    cell18_expected_answer_record[
        "expected_answer_text"
    ]
)

print(
    "Human decision:",
    CELL18_HUMAN_ANSWER_DECISION,
)

print(
    "Answer ACCEPT transition valid:",
    cell18_answer_accept_transition_valid,
)


print(
    "\n=== CITATION DRAFT ==="
)

print(
    "Citation Mapping ID:",
    cell18_citation_mapping_id,
)

print(
    "Citation entry count:",
    len(
        cell18_citation_entries
    ),
)


for entry in cell18_citation_entries:

    print(
        "\n--- CITATION ENTRY",
        entry[
            "citation_index"
        ],
        "---"
    )

    print(
        "Document ID:",
        entry.get(
            "document_id"
        ),
    )

    print(
        "Role:",
        entry.get(
            "role"
        ),
    )

    print(
        "Chars:",
        (
            entry.get(
                "char_start"
            ),
            entry.get(
                "char_end"
            ),
        ),
    )

    print(
        "Canonical URL:",
        (
            entry.get(
                "locator",
                {}
            ).get(
                "canonical_url"
            )
        ),
    )

    print(
        "URL:",
        (
            entry.get(
                "locator",
                {}
            ).get(
                "url"
            )
        ),
    )

    print(
        "Locator origin:",
        entry.get(
            "locator_origin"
        ),
    )


print(
    "\n=== CITATION VALIDATION ==="
)

for row in (
    cell18_citation_validation_rows
):

    print(
        row
    )


print(
    "\n=== CITATION MACHINE GATES ==="
)

for name, status in (
    cell18_citation_machine_validation.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "Machine validation PASS:",
    cell18_citation_machine_validation_pass,
)

print(
    "No locator fabrication:",
    cell18_no_locator_fabrication,
)

print(
    "Citation record SHA256:",
    cell18_citation_mapping_record_sha256,
)


print(
    "\n=== REAL WORKFLOW STATE ==="
)

print(
    "Current state:",
    (
        cell18_current_session.get(
            "current_state"
        )

        if isinstance(
            cell18_current_session,
            dict,
        )

        else None
    ),
)

print(
    "Citation registered:",
    cell18_citation_registered,
)

print(
    "Human decisions recorded:",
    (
        len(
            cell18_current_session.get(
                "human_decisions",
                [],
            )
        )

        if isinstance(
            cell18_current_session,
            dict,
        )

        else None
    ),
)

print(
    "Queue advance permitted:",
    (
        cell18_current_session.get(
            "queue_advance_permitted"
        )

        if isinstance(
            cell18_current_session,
            dict,
        )

        else None
    ),
)


print(
    "\n=== HUMAN REVIEW PACKAGE ==="
)

print(
    "Question decision:",
    "ACCEPT",
)

print(
    "Expected Answer decision:",
    cell18_human_review_case[
        "expected_answer_review"
    ][
        "human_decision"
    ],
)

print(
    "Citation decision:",
    cell18_human_review_case[
        "citation_review"
    ][
        "human_decision"
    ],
)

print(
    "Human Review Case SHA256:",
    cell18_human_review_case_sha256,
)


print(
    "\n=== QUEUE / SAFETY ==="
)

print(
    "Seed unchanged:",
    cell18_seed_unchanged,
)

print(
    "Queue still 22:",
    cell18_queue_still_22,
)

print(
    "Next Pilot unchanged:",
    cell18_next_pilot_unchanged,
)

print(
    "Full 572 blocked:",
    cell18_full_572_blocked,
)

print(
    "Project/file mutation performed:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell18_failures
)


print(
    "\nCODE CELL 18 GATE:",
    (
        "PASS"
        if cell18_gate
        else "FAIL"
    ),
)


assert cell18_gate, (
    "Expected Answer ACCEPT / Citation Draft validation "
    "failed. Repair this SAME Code Cell 18 before final "
    "Human Citation Adjudication and Pilot completion."
)

=== EXPECTED ANSWER ACCEPTED + CITATION DRAFT ===

=== ACCEPTED QUESTION ===
چه مشکلی پیش می‌آید اگر بر اساس شناختی که فقط از یک بخش داریم، درباره کل موضوع تصمیم بگیریم؟

=== ACCEPTED EXPECTED ANSWER ===
مشکل این است که ممکن است همان شناخت ناقص را به کل موضوع تعمیم بدهیم و بخواهیم بر اساس آن تصمیم بگیریم یا از کل موضوع استفاده کنیم. چنین کاری شاید در ظاهر مناسب به نظر برسد، اما بخش‌های یک موضوع می‌توانند به هم وابسته باشند و نمی‌شود یک بخش را کاملاً جدا از بقیه در نظر گرفت. در مثال متن نیز فیل یک کل به‌هم‌پیوسته است و شناخت یک قسمت از آن، به‌تنهایی تصویر کاملی از فیل نمی‌دهد.
Human decision: ACCEPT
Answer ACCEPT transition valid: True

=== CITATION DRAFT ===
Citation Mapping ID: citation-mapping:ef5f045a886b6160eaa44636329de992
Citation entry count: 1

--- CITATION ENTRY 0 ---
Document ID: website:5C92BB0D
Role: LOCAL_SENTENCE_WINDOW
Chars: (1123, 1401)
Canonical URL: https://www.fintelligence.ir/Blog/Detail/5C92BB0D
URL: https://www.fintelligence.ir/Blog/Detail/5C92BB0D/%D9%81%DB%8C%D

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 19 — پذیرش Citation، تکمیل Pilot #2، پیشروی Queue و آماده‌سازی Pilot بعدی

Citation مربوط به Pilot #2 توسط Human Reviewer پذیرفته شد.

در این Cell:

- تصمیم انسانی `ACCEPT` برای Citation ثبت می‌شود؛
- Question، Expected Answer و Citation پذیرفته‌شده در یک Completed Pilot Record جمع می‌شوند؛
- Human Review Lineage این Pilot کامل می‌شود؛
- وضعیت Author Final Review همچنان `NOT_STARTED` باقی می‌ماند تا بعداً دکتر رودی آن را جداگانه بررسی کند؛
- Queue فعال continuation از 22 به 21 Pilot کاهش پیدا می‌کند؛
- Pilot تکمیل‌شده از Queue فعال خارج می‌شود؛
- Pilot بعدی به‌صورت خودکار انتخاب می‌شود؛
- Evidence و Workspace اولیه‌ی Pilot بعدی آماده و نمایش داده می‌شوند.

برای حفظ provenance، Compact Seed اولیه تغییر نمی‌کند. از این Cell به بعد یک `Continuation State` مستقل، authoritative state ادامه‌ی کار خواهد بود.

این مرحله هیچ Question جدیدی برای Pilot بعدی تولید نمی‌کند. ابتدا Evidence اصلی Pilot بعدی نمایش داده می‌شود تا Question Draft بعدی بر اساس همان منبع ساخته شود.

Full 572 Generation همچنان Blocked است.

هیچ فایل، Commit، Push، PR یا Artifact خارجی ایجاد یا تغییر داده نمی‌شود.

</div>

In [19]:
# Code Cell 19

cell19_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell18_gate") is not True:
    cell19_failures.append(
        "CELL18_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell18_current_session"
    ),
    dict,
):
    cell19_failures.append(
        "CELL18_CURRENT_SESSION_MISSING"
    )


if (
    globals().get(
        "cell18_current_session",
        {},
    ).get(
        "current_state"
    )
    !=
    "CITATION_HUMAN_REVIEW_PENDING"
):
    cell19_failures.append(
        "CITATION_NOT_AT_HUMAN_REVIEW_GATE"
    )


# ============================================================
# 2) Fixed current-Pilot identities
# ============================================================

CELL19_COMPLETED_PILOT_ID = (
    "question-pilot:"
    "3005b48518a7aeb532234332df23c952"
)

CELL19_FULL_572_BLOCK_STATUS = (
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


# ============================================================
# 3) Freeze immutable Compact Seed
# ============================================================

cell19_base_seed_sha_before = (
    cell10_stable_sha(
        cell13_seed
    )
)


cell19_base_remaining_ids = copy.deepcopy(
    cell13_seed[
        "final_runtime"
    ][
        "cell95_remaining_pilot_ids"
    ]
)


cell19_base_remaining_queue = copy.deepcopy(
    cell13_seed[
        "final_runtime"
    ][
        "cell95_remaining_pilot_queue"
    ]
)


cell19_base_next_pilot_id = (
    cell13_seed[
        "final_runtime"
    ][
        "cell95_next_pilot_id"
    ]
)


# ============================================================
# 4) Human ACCEPT for Citation
# ============================================================

CELL19_HUMAN_CITATION_DECISION = (
    "ACCEPT"
)


CELL19_HUMAN_CITATION_NOTE = (
    "Human Reviewer accepted the Citation Mapping "
    "because it points to the already validated Target "
    "Evidence locator without fabricated source or range."
)


cell19_citation_accept_review = {
    "review_record_version":
        "human_citation_review_record_v1",

    "pilot_case_id":
        CELL19_COMPLETED_PILOT_ID,

    "golden_case_id":
        cell18_citation_mapping_record.get(
            "golden_case_id"
        ),

    "task_draft_id":
        cell18_citation_mapping_record.get(
            "task_draft_id"
        ),

    "target_evidence_id":
        cell18_citation_mapping_record.get(
            "target_evidence_id"
        ),

    "question_draft_id":
        cell18_citation_mapping_record.get(
            "question_draft_id"
        ),

    "expected_answer_draft_id":
        cell18_citation_mapping_record.get(
            "expected_answer_draft_id"
        ),

    "citation_mapping_id":
        cell18_citation_mapping_record.get(
            "citation_mapping_id"
        ),

    "review_stage":
        "CITATION",

    "human_decision":
        CELL19_HUMAN_CITATION_DECISION,

    "human_note":
        CELL19_HUMAN_CITATION_NOTE,

    "locator_policy":
        "EXISTING_VALIDATED_TARGET_LOCATORS_ONLY",

    "external_locator_lookup_used":
        False,
}


cell19_citation_accept_review_sha256 = (
    cell10_stable_sha(
        cell19_citation_accept_review
    )
)


# ============================================================
# 5) Apply real Citation ACCEPT
# ============================================================

cell19_pre_advance_session = (
    cell13_apply_event(
        cell18_current_session,
        "CITATION_ACCEPTED",
        human_note=
            CELL19_HUMAN_CITATION_NOTE,
    )
)


cell19_citation_accept_transition_valid = all([
    cell19_pre_advance_session.get(
        "current_state"
    )
    ==
    "PILOT_COMPLETE_PENDING_QUEUE_ADVANCE",

    cell19_pre_advance_session.get(
        "queue_advance_permitted"
    )
    is True,

    cell19_pre_advance_session.get(
        "queue_advance_performed"
    )
    is False,

    len(
        cell19_pre_advance_session.get(
            "human_decisions",
            [],
        )
    )
    ==
    4,

    cell19_pre_advance_session[
        "human_decisions"
    ][
        -1
    ].get(
        "event"
    )
    ==
    "CITATION_ACCEPTED",
])


# ============================================================
# 6) Complete Human Review package for Pilot #2
# ============================================================

cell19_completed_human_review_case = (
    copy.deepcopy(
        cell18_human_review_case
    )
)


cell19_completed_human_review_case[
    "citation_review"
][
    "human_decision"
] = "ACCEPT"


cell19_completed_human_review_case[
    "citation_review"
][
    "human_review"
] = copy.deepcopy(
    cell19_citation_accept_review
)


cell19_completed_human_review_case[
    "final_status"
] = (
    "HUMAN_REVIEW_COMPLETE_"
    "AUTHOR_FINAL_REVIEW_PENDING"
)


cell19_completed_human_review_case[
    "author_final_review"
] = {
    "reviewer":
        "AUTHOR",

    "status":
        "NOT_STARTED",

    "decision":
        None,

    "notes":
        None,
}


cell19_completed_human_review_case_sha256 = (
    cell10_stable_sha(
        cell19_completed_human_review_case
    )
)


# ============================================================
# 7) Build Completed Pilot Record
# ============================================================

cell19_completed_pilot_record = {
    "record_version":
        "completed_pilot_review_record_v1",

    "pilot_case_id":
        CELL19_COMPLETED_PILOT_ID,

    "completion_scope":
        "QUESTION_ANSWER_CITATION_HUMAN_REVIEW",

    "author_final_review_status":
        "NOT_STARTED",

    "question":
        copy.deepcopy(
            cell18_current_session[
                "artifacts"
            ][
                "question_draft"
            ]
        ),

    "expected_answer":
        copy.deepcopy(
            cell18_current_session[
                "artifacts"
            ][
                "expected_answer"
            ]
        ),

    "citation_mapping":
        copy.deepcopy(
            cell18_current_session[
                "artifacts"
            ][
                "citation_mapping"
            ]
        ),

    "human_review_case":
        copy.deepcopy(
            cell19_completed_human_review_case
        ),

    "source_evidence":
        copy.deepcopy(
            cell14_target_texts
        ),

    "full_572_generation":
        CELL19_FULL_572_BLOCK_STATUS,
}


cell19_completed_pilot_record_sha256 = (
    cell10_stable_sha(
        cell19_completed_pilot_record
    )
)


# ============================================================
# 8) Advance continuation Queue
#
# The original Compact Seed remains immutable.
# This new Continuation State becomes authoritative forward state.
# ============================================================

cell19_queue_preconditions_valid = all([
    isinstance(
        cell19_base_remaining_ids,
        list,
    ),

    len(
        cell19_base_remaining_ids
    )
    ==
    22,

    cell19_base_remaining_ids[
        0
    ]
    ==
    CELL19_COMPLETED_PILOT_ID,

    isinstance(
        cell19_base_remaining_queue,
        list,
    ),

    len(
        cell19_base_remaining_queue
    )
    ==
    22,

    cell19_base_next_pilot_id
    ==
    CELL19_COMPLETED_PILOT_ID,

    cell19_citation_accept_transition_valid,
])


if cell19_queue_preconditions_valid:

    cell19_remaining_pilot_ids = copy.deepcopy(
        cell19_base_remaining_ids[
            1:
        ]
    )

    cell19_remaining_pilot_queue = copy.deepcopy(
        cell19_base_remaining_queue[
            1:
        ]
    )

else:

    cell19_remaining_pilot_ids = []

    cell19_remaining_pilot_queue = []


cell19_next_pilot_id = (
    cell19_remaining_pilot_ids[
        0
    ]

    if cell19_remaining_pilot_ids

    else None
)


cell19_next_pilot_queue_entry = (
    copy.deepcopy(
        cell19_remaining_pilot_queue[
            0
        ]
    )

    if cell19_remaining_pilot_queue

    else None
)


cell19_remaining_queue_sha256 = (
    cell10_stable_sha(
        cell19_remaining_pilot_queue
    )
)


cell19_remaining_ids_sha256 = (
    cell10_stable_sha(
        cell19_remaining_pilot_ids
    )
)


# ============================================================
# 9) Mark completed session after Queue advance
# ============================================================

cell19_completed_session = copy.deepcopy(
    cell19_pre_advance_session
)


cell19_completed_session[
    "current_state"
] = "PILOT_COMPLETE"


cell19_completed_session[
    "queue_advance_permitted"
] = False


cell19_completed_session[
    "queue_advance_performed"
] = True


cell19_completed_session[
    "completion_status"
] = (
    "HUMAN_REVIEW_COMPLETE_"
    "AUTHOR_FINAL_REVIEW_PENDING"
)


cell19_completed_session[
    "session_sha256"
] = cell10_stable_sha({
    key:
        value

    for key, value
    in cell19_completed_session.items()

    if key
    !=
    "session_sha256"
})


# ============================================================
# 10) Build authoritative Continuation State
# ============================================================

cell19_continuation_state = {
    "state_version":
        "compact_pilot_continuation_state_v1",

    "authoritative_forward_state":
        True,

    "base_seed_sha256":
        cell19_base_seed_sha_before,

    "completed_pilot_id":
        CELL19_COMPLETED_PILOT_ID,

    "completed_pilot_record":
        copy.deepcopy(
            cell19_completed_pilot_record
        ),

    "completed_session":
        copy.deepcopy(
            cell19_completed_session
        ),

    "remaining_pilot_ids":
        copy.deepcopy(
            cell19_remaining_pilot_ids
        ),

    "remaining_pilot_queue":
        copy.deepcopy(
            cell19_remaining_pilot_queue
        ),

    "remaining_pilot_ids_sha256":
        cell19_remaining_ids_sha256,

    "remaining_pilot_queue_sha256":
        cell19_remaining_queue_sha256,

    "remaining_count":
        len(
            cell19_remaining_pilot_ids
        ),

    "next_pilot_id":
        cell19_next_pilot_id,

    "next_pilot_queue_entry":
        copy.deepcopy(
            cell19_next_pilot_queue_entry
        ),

    "full_572_generation":
        CELL19_FULL_572_BLOCK_STATUS,
}


cell19_continuation_state_sha256 = (
    cell10_stable_sha(
        cell19_continuation_state
    )
)


# ============================================================
# 11) Validate Queue advancement
# ============================================================

cell19_queue_advanced_valid = all([
    cell19_queue_preconditions_valid,

    len(
        cell19_remaining_pilot_ids
    )
    ==
    21,

    len(
        cell19_remaining_pilot_queue
    )
    ==
    21,

    CELL19_COMPLETED_PILOT_ID
    not in
    cell19_remaining_pilot_ids,

    cell19_next_pilot_id
    is not None,

    cell19_next_pilot_id
    !=
    CELL19_COMPLETED_PILOT_ID,

    cell19_completed_session[
        "queue_advance_performed"
    ]
    is True,
])


# ============================================================
# 12) Resolve next Pilot packet
# ============================================================

cell19_pilot_by_id = (
    cell13_seed[
        "final_runtime"
    ][
        "question_drafting_pilot_by_id"
    ]
)


cell19_next_pilot_packet = (
    copy.deepcopy(
        cell19_pilot_by_id.get(
            cell19_next_pilot_id
        )
    )

    if isinstance(
        cell19_pilot_by_id,
        dict,
    )

    else None
)


cell19_next_pilot_packet_valid = (
    isinstance(
        cell19_next_pilot_packet,
        dict,
    )
)


# ============================================================
# 13) Resolve next Pilot identity chain
# ============================================================

cell19_next_golden_case_id = None
cell19_next_task_draft_id = None
cell19_next_target_evidence_id = None


if cell19_next_pilot_packet_valid:

    try:

        cell19_next_golden_case_id = (
            cell13_exact_single_id(
                cell19_next_pilot_packet,
                "golden_case",
            )
        )


        cell19_next_task_draft_id = (
            cell13_exact_single_id(
                cell19_next_pilot_packet,
                "task_draft",
            )
        )


        cell19_next_target_evidence_id = (
            cell13_exact_single_id(
                cell19_next_pilot_packet,
                "target_evidence",
            )
        )


    except Exception as exc:

        cell19_failures.append({
            "reason":
                "NEXT_PILOT_ID_CHAIN_RESOLUTION_FAILED",

            "error":
                repr(
                    exc
                ),
        })


cell19_next_identity_chain_valid = all([
    isinstance(
        cell19_next_golden_case_id,
        str,
    ),

    isinstance(
        cell19_next_task_draft_id,
        str,
    ),

    isinstance(
        cell19_next_target_evidence_id,
        str,
    ),
])


# ============================================================
# 14) Resolve next Pilot support
# ============================================================

cell19_next_golden_case = None
cell19_next_task_plan = None
cell19_next_target_evidence = None


if cell19_next_identity_chain_valid:

    try:

        cell19_next_golden_case = copy.deepcopy(
            cell13_active_support[
                "golden_case_records_by_id"
            ][
                cell19_next_golden_case_id
            ]
        )


        cell19_next_task_plan = (
            cell13_find_unique_mapping_value(
                cell13_active_support.get(
                    "task_draft_plans",
                    {},
                ),
                "task_draft",
                cell19_next_task_draft_id,
            )[
                "value"
            ]
        )


        cell19_next_target_evidence = (
            cell13_find_unique_list_record(
                cell13_active_support.get(
                    "target_evidence_records",
                    [],
                ),
                "target_evidence_id",
                cell19_next_target_evidence_id,
            )
        )


    except Exception as exc:

        cell19_failures.append({
            "reason":
                "NEXT_PILOT_SUPPORT_RESOLUTION_FAILED",

            "error":
                repr(
                    exc
                ),
        })


# ============================================================
# 15) Resolve next Pilot classification
# ============================================================

cell19_next_source_type_match = (
    cell14_find_first_scalar(
        cell19_next_pilot_packet,
        [
            "source_type",
            "source_kind",
            "source",
        ],
    )
)


cell19_next_stratum_match = (
    cell14_find_first_scalar(
        cell19_next_pilot_packet,
        [
            "stratum",
            "stratum_name",
            "stratum_id",
        ],
    )
)


cell19_next_task_family_match = (
    cell14_find_first_scalar(
        cell19_next_pilot_packet,
        [
            "task_family",
        ],
    )
)


cell19_next_question_archetype_match = (
    cell14_find_first_scalar(
        cell19_next_pilot_packet,
        [
            "question_archetype",
        ],
    )
)


cell19_next_source_type = (
    cell19_next_source_type_match.get(
        "value"
    )

    if isinstance(
        cell19_next_source_type_match,
        dict,
    )

    else None
)


cell19_next_stratum = (
    cell19_next_stratum_match.get(
        "value"
    )

    if isinstance(
        cell19_next_stratum_match,
        dict,
    )

    else None
)


cell19_next_task_family = (
    cell19_next_task_family_match.get(
        "value"
    )

    if isinstance(
        cell19_next_task_family_match,
        dict,
    )

    else None
)


cell19_next_question_archetype = (
    cell19_next_question_archetype_match.get(
        "value"
    )

    if isinstance(
        cell19_next_question_archetype_match,
        dict,
    )

    else None
)


# ============================================================
# 16) Extract next Pilot Target Text
# ============================================================

cell19_next_target_text_parts_raw = []


if isinstance(
    cell19_next_pilot_packet,
    dict,
):

    raw_parts = (
        cell19_next_pilot_packet.get(
            "target_text_parts"
        )
    )


    if isinstance(
        raw_parts,
        list,
    ):

        cell19_next_target_text_parts_raw = (
            copy.deepcopy(
                raw_parts
            )
        )


cell19_next_target_text_parts = []


for index, part in enumerate(
    cell19_next_target_text_parts_raw
):

    if not isinstance(
        part,
        dict,
    ):
        continue


    text_value = None


    for field_name in (
        "text",
        "target_text",
        "content",
        "evidence_text",
        "text_content",
    ):

        candidate = (
            part.get(
                field_name
            )
        )


        if (
            isinstance(
                candidate,
                str,
            )
            and
            candidate.strip()
        ):

            text_value = (
                candidate.strip()
            )

            break


    if text_value is None:
        continue


    cell19_next_target_text_parts.append({
        "part_index":
            index,

        "document_id":
            part.get(
                "document_id"
            ),

        "role":
            part.get(
                "role"
            ),

        "char_start":
            part.get(
                "char_start"
            ),

        "char_end":
            part.get(
                "char_end"
            ),

        "text":
            text_value,

        "text_sha256":
            hashlib.sha256(
                text_value.encode(
                    "utf-8"
                )
            ).hexdigest(),
    })


cell19_next_combined_target_text = (
    "\n\n".join(
        row[
            "text"
        ]

        for row
        in cell19_next_target_text_parts
    )
)


cell19_next_target_text_available = (
    isinstance(
        cell19_next_combined_target_text,
        str,
    )
    and
    bool(
        cell19_next_combined_target_text.strip()
    )
)


# ============================================================
# 17) Build next Pilot Human-Review Workspace
# ============================================================

cell19_next_pilot_workspace = {
    "workspace_version":
        "compact_human_review_workspace_v1",

    "status":
        "READY_FOR_QUESTION_DRAFT",

    "pilot_case_id":
        cell19_next_pilot_id,

    "golden_case_id":
        cell19_next_golden_case_id,

    "task_draft_id":
        cell19_next_task_draft_id,

    "target_evidence_id":
        cell19_next_target_evidence_id,

    "source_type":
        cell19_next_source_type,

    "stratum":
        cell19_next_stratum,

    "task_family":
        cell19_next_task_family,

    "question_archetype":
        cell19_next_question_archetype,

    "source_evidence":
        copy.deepcopy(
            cell19_next_target_text_parts
        ),

    "combined_target_text":
        cell19_next_combined_target_text,

    "golden_case":
        copy.deepcopy(
            cell19_next_golden_case
        ),

    "task_plan":
        copy.deepcopy(
            cell19_next_task_plan
        ),

    "target_evidence":
        copy.deepcopy(
            cell19_next_target_evidence
        ),

    "question_draft":
        None,

    "human_question_decision":
        None,

    "expected_answer":
        None,

    "citation_mapping":
        None,

    "workflow_state":
        "QUESTION_DRAFT_PENDING",

    "full_572_generation":
        CELL19_FULL_572_BLOCK_STATUS,
}


cell19_next_pilot_workspace_sha256 = (
    cell10_stable_sha(
        cell19_next_pilot_workspace
    )
)


cell19_next_workspace_ready = all([
    cell19_next_pilot_packet_valid,

    cell19_next_identity_chain_valid,

    isinstance(
        cell19_next_golden_case,
        dict,
    ),

    isinstance(
        cell19_next_task_plan,
        dict,
    ),

    isinstance(
        cell19_next_target_evidence,
        dict,
    ),

    cell19_next_target_text_available,

    cell19_next_pilot_workspace[
        "workflow_state"
    ]
    ==
    "QUESTION_DRAFT_PENDING",
])


# ============================================================
# 18) Human Review Registry
#
# Pilot #1 is retained as historical accepted material.
# Pilot #2 has the full compact human-review lineage.
# ============================================================

cell19_historical_pilot_1 = copy.deepcopy(
    cell13_seed.get(
        "completed_pilot",
        {},
    )
)


cell19_human_review_registry = {
    "registry_version":
        "human_review_registry_v1",

    "author_final_review_status":
        "NOT_STARTED",

    "historical_completed_pilot":
        cell19_historical_pilot_1,

    "completed_human_review_cases":
        [
            copy.deepcopy(
                cell19_completed_human_review_case
            )
        ],

    "active_workspace":
        copy.deepcopy(
            cell19_next_pilot_workspace
        ),

    "remaining_pilot_count":
        21,

    "full_572_generation":
        CELL19_FULL_572_BLOCK_STATUS,
}


cell19_human_review_registry_sha256 = (
    cell10_stable_sha(
        cell19_human_review_registry
    )
)


# ============================================================
# 19) Base Seed immutability
# ============================================================

cell19_base_seed_sha_after = (
    cell10_stable_sha(
        cell13_seed
    )
)


cell19_base_seed_unchanged = (
    cell19_base_seed_sha_before
    ==
    cell19_base_seed_sha_after
)


# ============================================================
# 20) Full 572 remains blocked
# ============================================================

cell19_full_572_blocked = all([
    cell19_continuation_state[
        "full_572_generation"
    ]
    ==
    CELL19_FULL_572_BLOCK_STATUS,

    cell19_next_pilot_workspace[
        "full_572_generation"
    ]
    ==
    CELL19_FULL_572_BLOCK_STATUS,

    cell19_human_review_registry[
        "full_572_generation"
    ]
    ==
    CELL19_FULL_572_BLOCK_STATUS,
])


# ============================================================
# 21) Final Gate
# ============================================================

cell19_gate = all([
    len(
        cell19_failures
    )
    ==
    0,

    cell19_citation_accept_transition_valid,

    cell19_queue_advanced_valid,

    cell19_completed_session[
        "current_state"
    ]
    ==
    "PILOT_COMPLETE",

    cell19_completed_session[
        "queue_advance_performed"
    ]
    is True,

    cell19_next_workspace_ready,

    cell19_base_seed_unchanged,

    cell19_full_572_blocked,
])


# ============================================================
# 22) Diagnostics
# ============================================================

print(
    "=== PILOT #2 COMPLETION + "
    "QUEUE ADVANCE + NEXT WORKSPACE ==="
)


print(
    "\n=== CITATION HUMAN REVIEW ==="
)

print(
    "Decision:",
    CELL19_HUMAN_CITATION_DECISION,
)

print(
    "Citation Mapping ID:",
    cell18_citation_mapping_record.get(
        "citation_mapping_id"
    ),
)

print(
    "Citation ACCEPT transition valid:",
    cell19_citation_accept_transition_valid,
)


print(
    "\n=== COMPLETED PILOT ==="
)

print(
    "Pilot:",
    CELL19_COMPLETED_PILOT_ID,
)

print(
    "Question:",
    cell16_question_text,
)

print(
    "Expected Answer:",
    cell17_expected_answer_text,
)

print(
    "Completion status:",
    cell19_completed_session.get(
        "completion_status"
    ),
)

print(
    "Author Final Review:",
    "NOT_STARTED",
)

print(
    "Completed Pilot SHA256:",
    cell19_completed_pilot_record_sha256,
)


print(
    "\n=== QUEUE ADVANCE ==="
)

print(
    "Base immutable queue:",
    len(
        cell19_base_remaining_ids
    ),
)

print(
    "Authoritative continuation queue:",
    len(
        cell19_remaining_pilot_ids
    ),
)

print(
    "Completed Pilot removed:",
    (
        CELL19_COMPLETED_PILOT_ID
        not in
        cell19_remaining_pilot_ids
    ),
)

print(
    "Queue advance performed:",
    cell19_completed_session[
        "queue_advance_performed"
    ],
)

print(
    "Next Pilot:",
    cell19_next_pilot_id,
)

print(
    "Continuation State SHA256:",
    cell19_continuation_state_sha256,
)


print(
    "\n=== NEXT PILOT IDENTITY ==="
)

print(
    "Pilot:",
    cell19_next_pilot_id,
)

print(
    "Golden Case:",
    cell19_next_golden_case_id,
)

print(
    "Task Draft:",
    cell19_next_task_draft_id,
)

print(
    "Target Evidence:",
    cell19_next_target_evidence_id,
)

print(
    "Source type:",
    cell19_next_source_type,
)

print(
    "Stratum:",
    cell19_next_stratum,
)

print(
    "Task family:",
    cell19_next_task_family,
)

print(
    "Question archetype:",
    cell19_next_question_archetype,
)


print(
    "\n=== NEXT PILOT SOURCE EVIDENCE ==="
)

print(
    "Target part count:",
    len(
        cell19_next_target_text_parts
    ),
)


for row in (
    cell19_next_target_text_parts
):

    print(
        "\n--- SOURCE PART",
        row[
            "part_index"
        ],
        "---"
    )

    print(
        "Document:",
        row[
            "document_id"
        ],
    )

    print(
        "Role:",
        row[
            "role"
        ],
    )

    print(
        "Chars:",
        (
            row[
                "char_start"
            ],
            row[
                "char_end"
            ],
        ),
    )

    print(
        "\nTEXT:\n"
        +
        row[
            "text"
        ]
    )


print(
    "\n=== NEXT WORKSPACE ==="
)

print(
    "Ready:",
    cell19_next_workspace_ready,
)

print(
    "Workflow state:",
    cell19_next_pilot_workspace[
        "workflow_state"
    ],
)

print(
    "Question Draft:",
    cell19_next_pilot_workspace[
        "question_draft"
    ],
)

print(
    "Expected Answer:",
    cell19_next_pilot_workspace[
        "expected_answer"
    ],
)

print(
    "Citation:",
    cell19_next_pilot_workspace[
        "citation_mapping"
    ],
)

print(
    "Workspace SHA256:",
    cell19_next_pilot_workspace_sha256,
)


print(
    "\n=== HUMAN REVIEW REGISTRY ==="
)

print(
    "Historical Pilot #1 retained:",
    bool(
        cell19_historical_pilot_1
    ),
)

print(
    "Completed compact Human Review cases:",
    len(
        cell19_human_review_registry[
            "completed_human_review_cases"
        ]
    ),
)

print(
    "Active Pilot:",
    cell19_human_review_registry[
        "active_workspace"
    ][
        "pilot_case_id"
    ],
)

print(
    "Registry SHA256:",
    cell19_human_review_registry_sha256,
)


print(
    "\n=== SAFETY / IMMUTABILITY ==="
)

print(
    "Base Seed unchanged:",
    cell19_base_seed_unchanged,
)

print(
    "Base Seed still has 22:",
    len(
        cell19_base_remaining_ids
    )
    ==
    22,
)

print(
    "Continuation state has 21:",
    len(
        cell19_remaining_pilot_ids
    )
    ==
    21,
)

print(
    "Full 572 blocked:",
    cell19_full_572_blocked,
)

print(
    "Human Review Notebook file written:",
    False,
)

print(
    "Project/file mutation performed:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell19_failures
)


print(
    "\nCODE CELL 19 GATE:",
    (
        "PASS"
        if cell19_gate
        else "FAIL"
    ),
)


assert cell19_gate, (
    "Pilot #2 completion / queue advancement / "
    "next-Pilot workspace preparation failed. "
    "Repair this SAME Code Cell 19 before "
    "authoring the next Pilot question."
)

=== PILOT #2 COMPLETION + QUEUE ADVANCE + NEXT WORKSPACE ===

=== CITATION HUMAN REVIEW ===
Decision: ACCEPT
Citation Mapping ID: citation-mapping:ef5f045a886b6160eaa44636329de992
Citation ACCEPT transition valid: True

=== COMPLETED PILOT ===
Pilot: question-pilot:3005b48518a7aeb532234332df23c952
Question: چه مشکلی پیش می‌آید اگر بر اساس شناختی که فقط از یک بخش داریم، درباره کل موضوع تصمیم بگیریم؟
Expected Answer: مشکل این است که ممکن است همان شناخت ناقص را به کل موضوع تعمیم بدهیم و بخواهیم بر اساس آن تصمیم بگیریم یا از کل موضوع استفاده کنیم. چنین کاری شاید در ظاهر مناسب به نظر برسد، اما بخش‌های یک موضوع می‌توانند به هم وابسته باشند و نمی‌شود یک بخش را کاملاً جدا از بقیه در نظر گرفت. در مثال متن نیز فیل یک کل به‌هم‌پیوسته است و شناخت یک قسمت از آن، به‌تنهایی تصویر کاملی از فیل نمی‌دهد.
Completion status: HUMAN_REVIEW_COMPLETE_AUTHOR_FINAL_REVIEW_PENDING
Author Final Review: NOT_STARTED
Completed Pilot SHA256: 409468d96811e54b1dddfa217cbb5d9df32e04a28630b915c69a7be4bea4439d

=== QUEUE 

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 20 — Question Accepted + Expected Answer Draft برای Pilot بعدی

Evidence اصلی این Pilot درباره وضعیت سواد مالی در اروپا و سنجش‌های انجام‌شده در سال‌های 2014 و 2023 است.

پس از Human Review، سؤال زیر پذیرفته شد:

**«سواد مالی در اروپا چقدر مورد توجه قرار گرفته و چند درصد از اروپایی‌ها سواد مالی مناسبی دارند؟»**

در این Cell به‌صورت فشرده:

1. Question Draft واقعی ساخته و Machine Validate می‌شود؛
2. Human Decision برابر `ACCEPT` ثبت می‌شود؛
3. Expected Answer Draft فقط از Evidence همین Pilot ساخته می‌شود؛
4. Answer برای Human Review متوقف می‌شود.

عبارت «حدود 50 درصد» یک داده‌ی خارجی نیست؛ صرفاً تبدیل مستقیم «نیمی» به درصد است و این derivation به‌صورت صریح در lineage ثبت می‌شود.

هیچ ادعایی درباره جدید بودن مفهوم سواد مالی، وضعیت ایران، ضرورت آموزش آن به همه، یا سایر برداشت‌های انسانی وارد Answer نمی‌شود؛ زیرا این مطالب در Evidence فعلی وجود ندارند.

Citation هنوز ساخته نمی‌شود، Queue همچنان 21 باقی می‌ماند و Full 572 Generation Blocked است.

</div>

In [20]:
# Code Cell 20

cell20_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell19_gate") is not True:
    cell20_failures.append(
        "CELL19_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell19_next_pilot_workspace"
    ),
    dict,
):
    cell20_failures.append(
        "NEXT_PILOT_WORKSPACE_MISSING"
    )


if not callable(
    globals().get(
        "cell12_question_id_function"
    )
):
    cell20_failures.append(
        "QUESTION_ID_BUILDER_MISSING"
    )


# ============================================================
# 2) Current Pilot identity
# ============================================================

CELL20_PILOT_ID = (
    cell19_next_pilot_id
)

CELL20_GOLDEN_CASE_ID = (
    cell19_next_golden_case_id
)

CELL20_TASK_DRAFT_ID = (
    cell19_next_task_draft_id
)

CELL20_TARGET_EVIDENCE_ID = (
    cell19_next_target_evidence_id
)

CELL20_FULL_572_BLOCK_STATUS = (
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


cell20_identity_valid = all([
    isinstance(
        CELL20_PILOT_ID,
        str,
    ),

    isinstance(
        CELL20_GOLDEN_CASE_ID,
        str,
    ),

    isinstance(
        CELL20_TASK_DRAFT_ID,
        str,
    ),

    isinstance(
        CELL20_TARGET_EVIDENCE_ID,
        str,
    ),

    cell19_next_pilot_workspace.get(
        "workflow_state"
    )
    ==
    "QUESTION_DRAFT_PENDING",
])


# ============================================================
# 3) Build current Pilot bundle + fresh session
# ============================================================

cell20_pilot_bundle = {
    "pilot_case_id":
        CELL20_PILOT_ID,

    "golden_case_id":
        CELL20_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL20_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL20_TARGET_EVIDENCE_ID,
}


cell20_initial_session = (
    cell13_build_session(
        cell20_pilot_bundle
    )
)


cell20_initial_session_valid = all([
    cell20_initial_session[
        "current_state"
    ]
    ==
    "QUESTION_DRAFT_PENDING",

    cell20_initial_session[
        "queue_advance_permitted"
    ]
    is False,

    cell20_initial_session[
        "queue_advance_performed"
    ]
    is False,
])


# ============================================================
# 4) Authoritative fields
# ============================================================

cell20_field_sources = [
    (
        "pilot_packet",
        cell19_next_pilot_packet,
    ),
    (
        "task_plan",
        cell19_next_task_plan,
    ),
    (
        "golden_case",
        cell19_next_golden_case,
    ),
    (
        "target_evidence",
        cell19_next_target_evidence,
    ),
]


cell20_task_family_resolution = (
    cell15_resolve_field(
        "task_family",
        cell20_field_sources,
    )
)

cell20_question_archetype_resolution = (
    cell15_resolve_field(
        "question_archetype",
        cell20_field_sources,
    )
)

cell20_required_dimensions_resolution = (
    cell15_resolve_field(
        "required_dimensions",
        cell20_field_sources,
    )
)

cell20_authority_expectation_resolution = (
    cell15_resolve_field(
        "authority_expectation",
        cell20_field_sources,
    )
)

cell20_context_expectation_resolution = (
    cell15_resolve_field(
        "context_expectation",
        cell20_field_sources,
    )
)

cell20_evidence_scope_resolution = (
    cell15_resolve_field(
        "evidence_scope",
        cell20_field_sources,
    )
)

cell20_target_status_resolution = (
    cell15_resolve_field(
        "target_evidence_status",
        cell20_field_sources,
    )
)


cell20_task_family = (
    cell20_task_family_resolution[
        "value"
    ]
)

cell20_question_archetype = (
    cell20_question_archetype_resolution[
        "value"
    ]
)

cell20_required_dimensions = (
    cell20_required_dimensions_resolution[
        "value"
    ]
)

cell20_authority_expectation = (
    cell20_authority_expectation_resolution[
        "value"
    ]
)

cell20_context_expectation = (
    cell20_context_expectation_resolution[
        "value"
    ]
)

cell20_evidence_scope = (
    cell20_evidence_scope_resolution[
        "value"
    ]
)

cell20_target_evidence_status = (
    cell20_target_status_resolution[
        "value"
    ]
)


if isinstance(
    cell20_required_dimensions,
    tuple,
):
    cell20_required_dimensions = list(
        cell20_required_dimensions
    )

elif isinstance(
    cell20_required_dimensions,
    set,
):
    cell20_required_dimensions = sorted(
        cell20_required_dimensions
    )


# ============================================================
# 5) Accepted REAL Question
# ============================================================

CELL20_QUESTION_TEXT = (
    "سواد مالی در اروپا چقدر مورد توجه قرار گرفته "
    "و چند درصد از اروپایی‌ها سواد مالی مناسبی دارند؟"
)


cell20_question_text = (
    CELL20_QUESTION_TEXT.strip()
)


cell20_question_text_sha256 = (
    hashlib.sha256(
        cell20_question_text.encode(
            "utf-8"
        )
    ).hexdigest()
)


cell20_question_draft_id = (
    cell12_question_id_function(
        pilot_case_id=
            CELL20_PILOT_ID,

        question_text_sha256=
            cell20_question_text_sha256,
    )
)


# ============================================================
# 6) Question Machine Validation
# ============================================================

cell20_normalized_question = (
    cell15_normalize_text(
        cell20_question_text
    )
)

cell20_normalized_evidence = (
    cell15_normalize_text(
        cell19_next_combined_target_text
    )
)


cell20_question_internal_id_hits = [
    pattern

    for pattern
    in CELL15_INTERNAL_ID_PATTERNS

    if re.search(
        pattern,
        cell20_question_text,
        flags=re.IGNORECASE,
    )
]


cell20_question_internal_metadata_hits = [
    term

    for term
    in CELL15_INTERNAL_METADATA_TERMS

    if term.lower()
    in
    cell20_question_text.lower()
]


cell20_question_grounding_checks = {
    "financial_literacy":
        (
            "سواد مالی"
            in
            cell20_normalized_question
            and
            "سواد مالی"
            in
            cell20_normalized_evidence
        ),

    "europe":
        (
            "اروپا"
            in
            cell20_normalized_question
            and
            "اروپ"
            in
            cell20_normalized_evidence
        ),

    "appropriate_literacy_share":
        (
            "درصد"
            in
            cell20_normalized_question
            and
            "نیمی"
            in
            cell20_normalized_evidence
        ),

    "attention_or_study":
        (
            "مورد توجه"
            in
            cell20_normalized_question
            and
            "نظرسنجی"
            in
            cell20_normalized_evidence
        ),
}


cell20_question_grounding_valid = all(
    cell20_question_grounding_checks.values()
)


cell20_question_machine_gates = {
    "nonempty":
        bool(
            cell20_question_text
        ),

    "natural_question":
        cell20_question_text.endswith(
            (
                "؟",
                "?",
            )
        ),

    "target_text_available":
        cell19_next_target_text_available,

    "groundable_in_target":
        cell20_question_grounding_valid,

    "no_internal_id_leakage":
        (
            len(
                cell20_question_internal_id_hits
            )
            ==
            0
        ),

    "no_internal_metadata_leakage":
        (
            len(
                cell20_question_internal_metadata_hits
            )
            ==
            0
        ),

    "task_family_present":
        (
            cell20_task_family
            is not None
        ),

    "question_archetype_present":
        (
            cell20_question_archetype
            is not None
        ),

    "authority_expectation_present":
        (
            cell20_authority_expectation
            is not None
        ),

    "context_expectation_present":
        (
            cell20_context_expectation
            is not None
        ),

    "external_knowledge_required":
        False,
}


cell20_question_machine_validation_pass = all([
    cell20_question_machine_gates[
        "nonempty"
    ],

    cell20_question_machine_gates[
        "natural_question"
    ],

    cell20_question_machine_gates[
        "target_text_available"
    ],

    cell20_question_machine_gates[
        "groundable_in_target"
    ],

    cell20_question_machine_gates[
        "no_internal_id_leakage"
    ],

    cell20_question_machine_gates[
        "no_internal_metadata_leakage"
    ],

    cell20_question_machine_gates[
        "task_family_present"
    ],

    cell20_question_machine_gates[
        "question_archetype_present"
    ],

    cell20_question_machine_gates[
        "authority_expectation_present"
    ],

    cell20_question_machine_gates[
        "context_expectation_present"
    ],
])


# ============================================================
# 7) Build Question Draft Record
# ============================================================

cell20_question_draft_record = {
    "question_draft_id":
        cell20_question_draft_id,

    "question_draft_schema_version":
        cell15_question_schema_version,

    "pilot_case_id":
        CELL20_PILOT_ID,

    "golden_case_id":
        CELL20_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL20_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL20_TARGET_EVIDENCE_ID,

    "source_type":
        cell19_next_source_type,

    "stratum":
        cell19_next_stratum,

    "task_family":
        cell20_task_family,

    "question_archetype":
        cell20_question_archetype,

    "required_dimensions":
        copy.deepcopy(
            cell20_required_dimensions
        ),

    "authority_expectation":
        copy.deepcopy(
            cell20_authority_expectation
        ),

    "context_expectation":
        copy.deepcopy(
            cell20_context_expectation
        ),

    "evidence_scope":
        copy.deepcopy(
            cell20_evidence_scope
        ),

    "question_text":
        cell20_question_text,

    "question_text_sha256":
        cell20_question_text_sha256,

    "authoring_mode":
        cell15_authoring_mode,

    "draft_status":
        "DRAFT_CREATED_PENDING_VALIDATION",

    "machine_validation_status":
        (
            "PASS_PENDING_HUMAN_ADJUDICATION"

            if cell20_question_machine_validation_pass

            else "FAIL_REQUIRES_REVISION"
        ),

    "human_adjudication_status":
        "ACCEPT",

    "expected_answer_status":
        "READY_FOR_CONSTRUCTION",

    "citation_target_validation_status":
        "BLOCKED_PENDING_EXPECTED_ANSWER_ACCEPTANCE",

    "machine_validation":
        copy.deepcopy(
            cell20_question_machine_gates
        ),

    "full_572_generation":
        CELL20_FULL_572_BLOCK_STATUS,
}


# ============================================================
# 8) Human ACCEPT record for Question
# ============================================================

CELL20_HUMAN_QUESTION_NOTE = (
    "Human Reviewer accepted this Question after reviewing "
    "multiple alternative phrasings. Human interpretive "
    "discussion is review metadata only and is not evidence."
)


cell20_question_human_review = {
    "review_record_version":
        "human_question_review_record_v1",

    "pilot_case_id":
        CELL20_PILOT_ID,

    "question_draft_id":
        cell20_question_draft_id,

    "question_text":
        cell20_question_text,

    "human_decision":
        "ACCEPT",

    "human_note":
        CELL20_HUMAN_QUESTION_NOTE,

    "human_context_evidence_status":
        "NON_EVIDENCE",

    "source_evidence":
        copy.deepcopy(
            cell19_next_target_text_parts
        ),
}


# ============================================================
# 9) Register Question + Human ACCEPT
# ============================================================

cell20_question_review_session = None
cell20_answer_pending_session = None


if cell20_question_machine_validation_pass:

    cell20_question_review_session = (
        cell13_apply_event(
            cell20_initial_session,
            "QUESTION_DRAFT_REGISTERED",
            artifact=
                cell20_question_draft_record,
        )
    )


    cell20_answer_pending_session = (
        cell13_apply_event(
            cell20_question_review_session,
            "QUESTION_ACCEPTED",
            human_note=
                CELL20_HUMAN_QUESTION_NOTE,
        )
    )


cell20_question_accept_valid = all([
    isinstance(
        cell20_answer_pending_session,
        dict,
    ),

    cell20_answer_pending_session.get(
        "current_state"
    )
    ==
    "EXPECTED_ANSWER_DRAFT_PENDING",

    len(
        cell20_answer_pending_session.get(
            "human_decisions",
            [],
        )
    )
    ==
    1,

    cell20_answer_pending_session[
        "human_decisions"
    ][
        0
    ].get(
        "event"
    )
    ==
    "QUESTION_ACCEPTED",
])


# ============================================================
# 10) Expected Answer Draft
#
# Only the validated source evidence is allowed.
# ============================================================

CELL20_EXPECTED_ANSWER_TEXT = (
    "بر اساس متن، تنها نیمی از اروپایی‌ها، یعنی حدود "
    "50 درصد، سواد مالی مناسبی دارند. سواد مالی در اروپا "
    "مورد بررسی گسترده قرار گرفته است؛ در سال 2014 "
    "نظرسنجی جهانی سواد مالی یکی از ابزارهای مقایسه سطح "
    "سواد مالی کشورهای اروپایی بود و در بهار 2023 نیز "
    "کمیسیون اروپا نظرسنجی جامعی در هر 27 کشور عضو "
    "اتحادیه اروپا انجام داد. این سنجش جدید به دلیل پوشش "
    "کامل کشورهای عضو، اطلاعات قابل‌اتکاتری برای "
    "پژوهشگران و سیاست‌گذاران فراهم کرد."
)


cell20_expected_answer_text = (
    CELL20_EXPECTED_ANSWER_TEXT.strip()
)


cell20_expected_answer_text_sha256 = (
    hashlib.sha256(
        cell20_expected_answer_text.encode(
            "utf-8"
        )
    ).hexdigest()
)


# ============================================================
# 11) Explicit derivation audit
# ============================================================

cell20_derived_claims = [
    {
        "answer_claim":
            "حدود 50 درصد",

        "source_phrase":
            "تنها نیمی از اروپایی‌ها سواد مالی مناسبی دارند",

        "derivation":
            "1/2 = 50%",

        "derivation_type":
            "DIRECT_ARITHMETIC_NORMALIZATION",

        "external_knowledge":
            False,
    }
]


cell20_derived_claims_valid = all([
    "نیمی"
    in
    cell20_normalized_evidence,

    "50 درصد"
    in
    cell15_normalize_text(
        cell20_expected_answer_text
    ),

    cell20_derived_claims[
        0
    ][
        "external_knowledge"
    ]
    is False,
])


# ============================================================
# 12) Answer grounding validation
# ============================================================

cell20_normalized_answer = (
    cell15_normalize_text(
        cell20_expected_answer_text
    )
)


cell20_answer_grounding_checks = {
    "half_of_europeans":
        (
            "نیمی"
            in
            cell20_normalized_evidence
            and
            "50 درصد"
            in
            cell20_normalized_answer
        ),

    "2014_survey":
        (
            "2014"
            in
            cell20_normalized_evidence
            and
            "2014"
            in
            cell20_normalized_answer
        ),

    "2023_commission_survey":
        (
            "2023"
            in
            cell20_normalized_evidence
            and
            "2023"
            in
            cell20_normalized_answer
        ),

    "27_eu_countries":
        (
            "27"
            in
            cell20_normalized_evidence
            and
            "27"
            in
            cell20_normalized_answer
        ),

    "researchers_and_policy_makers":
        (
            "پژوهشگران"
            in
            cell20_normalized_evidence
            and
            "پژوهشگران"
            in
            cell20_normalized_answer
            and
            "سیاست گذاران"
            in
            cell20_normalized_evidence
            and
            "سیاست گذاران"
            in
            cell20_normalized_answer
        ),

    "reliable_findings":
        (
            "قابل اتکایی"
            in
            cell20_normalized_evidence
            and
            "قابل اتکاتری"
            in
            cell20_normalized_answer
        ),
}


cell20_answer_grounding_valid = all(
    cell20_answer_grounding_checks.values()
)


# ============================================================
# 13) Unsupported-claim guard
# ============================================================

CELL20_FORBIDDEN_UNSUPPORTED_TERMS = [
    "ایران",
    "ایرانی",
    "همه باید",
    "همه مردم باید",
    "مفهوم جدید",
    "موضوع جدید",
    "تازه به وجود آمده",
    "به تازگی ایجاد شده",
    "قدیمی نیست",
]


cell20_unsupported_answer_hits = [
    term

    for term
    in CELL20_FORBIDDEN_UNSUPPORTED_TERMS

    if cell15_normalize_text(
        term
    )
    in
    cell20_normalized_answer
]


cell20_answer_no_unsupported_claims = (
    len(
        cell20_unsupported_answer_hits
    )
    ==
    0
)


# ============================================================
# 14) Answer machine validation
# ============================================================

cell20_answer_machine_gates = {
    "nonempty":
        bool(
            cell20_expected_answer_text
        ),

    "addresses_attention":
        (
            "مورد بررسی گسترده"
            in
            cell20_normalized_answer
        ),

    "addresses_percentage":
        (
            "50 درصد"
            in
            cell20_normalized_answer
        ),

    "source_grounded":
        cell20_answer_grounding_valid,

    "derived_percentage_audited":
        cell20_derived_claims_valid,

    "no_unsupported_claims":
        cell20_answer_no_unsupported_claims,

    "human_context_used_as_evidence":
        False,

    "external_knowledge_used":
        False,
}


cell20_answer_machine_validation_pass = all([
    cell20_answer_machine_gates[
        "nonempty"
    ],

    cell20_answer_machine_gates[
        "addresses_attention"
    ],

    cell20_answer_machine_gates[
        "addresses_percentage"
    ],

    cell20_answer_machine_gates[
        "source_grounded"
    ],

    cell20_answer_machine_gates[
        "derived_percentage_audited"
    ],

    cell20_answer_machine_gates[
        "no_unsupported_claims"
    ],
])


# ============================================================
# 15) Stable local Expected Answer Draft ID
# ============================================================

cell20_answer_identity_payload = {
    "identity_version":
        "expected_answer_draft_identity_v1",

    "pilot_case_id":
        CELL20_PILOT_ID,

    "question_draft_id":
        cell20_question_draft_id,

    "target_evidence_id":
        CELL20_TARGET_EVIDENCE_ID,

    "expected_answer_text_sha256":
        cell20_expected_answer_text_sha256,
}


cell20_expected_answer_draft_id = (
    "expected-answer-draft:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell20_answer_identity_payload
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


# ============================================================
# 16) Expected Answer Record
# ============================================================

cell20_expected_answer_record = {
    "record_version":
        "expected_answer_draft_record_v1",

    "expected_answer_draft_id":
        cell20_expected_answer_draft_id,

    "pilot_case_id":
        CELL20_PILOT_ID,

    "golden_case_id":
        CELL20_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL20_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL20_TARGET_EVIDENCE_ID,

    "question_draft_id":
        cell20_question_draft_id,

    "question_text":
        cell20_question_text,

    "expected_answer_text":
        cell20_expected_answer_text,

    "expected_answer_text_sha256":
        cell20_expected_answer_text_sha256,

    "source_evidence_policy":
        "CURRENT_VALIDATED_TARGET_EVIDENCE_ONLY",

    "derived_claims":
        copy.deepcopy(
            cell20_derived_claims
        ),

    "external_knowledge_used":
        False,

    "human_context_used_as_evidence":
        False,

    "machine_validation_status":
        (
            "PASS_PENDING_HUMAN_ADJUDICATION"

            if cell20_answer_machine_validation_pass

            else "FAIL_REQUIRES_REVISION"
        ),

    "machine_validation":
        copy.deepcopy(
            cell20_answer_machine_gates
        ),

    "human_adjudication_status":
        "PENDING",

    "citation_validation_status":
        "BLOCKED_PENDING_EXPECTED_ANSWER_ACCEPTANCE",

    "full_572_generation":
        CELL20_FULL_572_BLOCK_STATUS,
}


# ============================================================
# 17) Register Expected Answer Draft
# ============================================================

cell20_current_session = None


if all([
    cell20_question_accept_valid,
    cell20_answer_machine_validation_pass,
]):

    cell20_current_session = (
        cell13_apply_event(
            cell20_answer_pending_session,
            "EXPECTED_ANSWER_REGISTERED",
            artifact=
                cell20_expected_answer_record,
        )
    )


cell20_expected_answer_registered = all([
    isinstance(
        cell20_current_session,
        dict,
    ),

    cell20_current_session.get(
        "current_state"
    )
    ==
    "EXPECTED_ANSWER_HUMAN_REVIEW_PENDING",

    isinstance(
        cell20_current_session[
            "artifacts"
        ][
            "question_draft"
        ],
        dict,
    ),

    isinstance(
        cell20_current_session[
            "artifacts"
        ][
            "expected_answer"
        ],
        dict,
    ),

    cell20_current_session[
        "artifacts"
    ][
        "citation_mapping"
    ]
    is None,

    len(
        cell20_current_session.get(
            "human_decisions",
            [],
        )
    )
    ==
    1,

    cell20_current_session.get(
        "queue_advance_permitted"
    )
    is False,

    cell20_current_session.get(
        "queue_advance_performed"
    )
    is False,
])


# ============================================================
# 18) Update compact Human Review Registry
# ============================================================

cell20_human_review_registry = copy.deepcopy(
    cell19_human_review_registry
)


cell20_active_human_review_case = {
    "human_review_case_version":
        "human_review_case_v1",

    "pilot_case_id":
        CELL20_PILOT_ID,

    "golden_case_id":
        CELL20_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL20_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL20_TARGET_EVIDENCE_ID,

    "source_evidence":
        copy.deepcopy(
            cell19_next_target_text_parts
        ),

    "model_drafts":
        [
            {
                "revision":
                    1,

                "question_draft_id":
                    cell20_question_draft_id,

                "question_text":
                    cell20_question_text,

                "machine_validation":
                    (
                        "PASS"
                        if
                        cell20_question_machine_validation_pass
                        else
                        "FAIL"
                    ),

                "human_decision":
                    "ACCEPT",
            }
        ],

    "human_reviews":
        [
            copy.deepcopy(
                cell20_question_human_review
            )
        ],

    "final_question_draft_id":
        cell20_question_draft_id,

    "expected_answer_review":
        {
            "expected_answer_draft_id":
                cell20_expected_answer_draft_id,

            "expected_answer_text":
                cell20_expected_answer_text,

            "machine_validation_status":
                cell20_expected_answer_record[
                    "machine_validation_status"
                ],

            "human_decision":
                "PENDING",

            "derived_claims":
                copy.deepcopy(
                    cell20_derived_claims
                ),
        },

    "citation_review":
        None,

    "author_final_review":
        {
            "status":
                "NOT_STARTED",

            "decision":
                None,

            "notes":
                None,
        },

    "final_status":
        "QUESTION_ACCEPTED_EXPECTED_ANSWER_PENDING_REVIEW",
}


cell20_human_review_registry[
    "active_workspace"
] = copy.deepcopy(
    cell20_active_human_review_case
)


cell20_human_review_registry_sha256 = (
    cell10_stable_sha(
        cell20_human_review_registry
    )
)


# ============================================================
# 19) Queue / state immutability
# ============================================================

cell20_continuation_queue_still_21 = (
    len(
        cell19_continuation_state[
            "remaining_pilot_ids"
        ]
    )
    ==
    21
)


cell20_next_pilot_unchanged = (
    cell19_continuation_state[
        "next_pilot_id"
    ]
    ==
    CELL20_PILOT_ID
)


cell20_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell20_full_572_blocked = all([
    isinstance(
        cell20_current_session,
        dict,
    ),

    cell20_current_session.get(
        "full_572_generation"
    )
    ==
    CELL20_FULL_572_BLOCK_STATUS,

    cell20_expected_answer_record[
        "full_572_generation"
    ]
    ==
    CELL20_FULL_572_BLOCK_STATUS,
])


# ============================================================
# 20) Final Gate
# ============================================================

cell20_gate = all([
    len(
        cell20_failures
    )
    ==
    0,

    cell20_identity_valid,

    cell20_initial_session_valid,

    cell20_question_machine_validation_pass,

    cell20_question_accept_valid,

    cell20_answer_machine_validation_pass,

    cell20_expected_answer_registered,

    cell20_continuation_queue_still_21,

    cell20_next_pilot_unchanged,

    cell20_base_seed_unchanged,

    cell20_full_572_blocked,
])


# ============================================================
# 21) Diagnostics
# ============================================================

print(
    "=== COMPACT PILOT WORKFLOW: "
    "QUESTION ACCEPTED + ANSWER DRAFT ==="
)


print(
    "\n=== SOURCE EVIDENCE ==="
)

print(
    cell19_next_combined_target_text
)


print(
    "\n=== ACCEPTED QUESTION ==="
)

print(
    cell20_question_text
)

print(
    "Question Draft ID:",
    cell20_question_draft_id,
)

print(
    "Human decision:",
    "ACCEPT",
)

print(
    "Question Machine PASS:",
    cell20_question_machine_validation_pass,
)

print(
    "Question grounding:",
    cell20_question_grounding_checks,
)


print(
    "\n=== EXPECTED ANSWER DRAFT ==="
)

print(
    cell20_expected_answer_text
)

print(
    "\nExpected Answer Draft ID:",
    cell20_expected_answer_draft_id,
)


print(
    "\n=== DERIVED CLAIM AUDIT ==="
)

for row in cell20_derived_claims:
    print(row)


print(
    "\n=== ANSWER GROUNDING ==="
)

for name, status in (
    cell20_answer_grounding_checks.items()
):
    print(
        f"{name}:",
        status,
    )


print(
    "\n=== ANSWER MACHINE GATES ==="
)

for name, status in (
    cell20_answer_machine_gates.items()
):
    print(
        f"{name}:",
        status,
    )


print(
    "Unsupported answer hits:",
    cell20_unsupported_answer_hits,
)

print(
    "Machine validation PASS:",
    cell20_answer_machine_validation_pass,
)


print(
    "\n=== REAL WORKFLOW STATE ==="
)

print(
    "Current state:",
    (
        cell20_current_session.get(
            "current_state"
        )

        if isinstance(
            cell20_current_session,
            dict,
        )

        else None
    ),
)

print(
    "Expected Answer registered:",
    cell20_expected_answer_registered,
)

print(
    "Human decisions recorded:",
    (
        len(
            cell20_current_session.get(
                "human_decisions",
                [],
            )
        )

        if isinstance(
            cell20_current_session,
            dict,
        )

        else None
    ),
)

print(
    "Citation:",
    (
        cell20_current_session[
            "artifacts"
        ][
            "citation_mapping"
        ]

        if isinstance(
            cell20_current_session,
            dict,
        )

        else None
    ),
)


print(
    "\n=== HUMAN REVIEW REGISTRY ==="
)

print(
    "Active Pilot:",
    cell20_active_human_review_case[
        "pilot_case_id"
    ],
)

print(
    "Question decision:",
    "ACCEPT",
)

print(
    "Expected Answer decision:",
    cell20_active_human_review_case[
        "expected_answer_review"
    ][
        "human_decision"
    ],
)

print(
    "Author Final Review:",
    "NOT_STARTED",
)

print(
    "Registry SHA256:",
    cell20_human_review_registry_sha256,
)


print(
    "\n=== QUEUE / SAFETY ==="
)

print(
    "Continuation queue still 21:",
    cell20_continuation_queue_still_21,
)

print(
    "Next Pilot unchanged:",
    cell20_next_pilot_unchanged,
)

print(
    "Base Seed unchanged:",
    cell20_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell20_full_572_blocked,
)

print(
    "Project/file mutation performed:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell20_failures
)


print(
    "\nCODE CELL 20 GATE:",
    (
        "PASS"
        if cell20_gate
        else "FAIL"
    ),
)


assert cell20_gate, (
    "Compact Question ACCEPT + Expected Answer Draft "
    "workflow failed. Repair this SAME Code Cell 20 "
    "before Human Expected Answer review."
)

=== COMPACT PILOT WORKFLOW: QUESTION ACCEPTED + ANSWER DRAFT ===

=== SOURCE EVIDENCE ===
تنها نیمی از اروپایی‌ها سواد مالی مناسبی دارند؛ در پژوهشی تازه در بهار 2023، کمیسیون اروپا نظرسنجی جامعی دربارۀ سواد مالی در تمام کشورهای عضو اتحادیۀ اروپا انجام داده که علاوه بر مقایسۀ وضعیت دانش مالی کشورهای مختلف، حاوی اطلاعات ارزشمندی برای پژوهشگران و سیاست‌گذاران است.
پیش از انتشار این مجموعه داده، ابزار اصلی برای مقایسۀ سطح سواد مالی کشورهای اروپایی، نظرسنجی جهانی سواد مالی در سال 2014 بود که برخی اعضای اُای‌سی‌دی و کشورهای عضو اتحادیۀ اروپا در آن شرکت نکرده بودند و این مسأله مانعی برای ارزیابی دقیق پژوهشگران از وضع موجود و مسیر مطلوب به حساب می‌آمد. اما سنجش جدید و جامع 2023، شامل 27 کشور عضو اتحادیۀ اروپا است و یافته‌های قابل اتکایی در اختیار ما می‌گذارد.

=== ACCEPTED QUESTION ===
سواد مالی در اروپا چقدر مورد توجه قرار گرفته و چند درصد از اروپایی‌ها سواد مالی مناسبی دارند؟
Question Draft ID: question-draft:992525a168edf83380610af7920e6a3e
Human decision: ACCEPT
Question Machine PASS: True

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 21 — ایجاد Persistent Human Review Registry و Author Review Notebook

از این مرحله Human Reviewها فقط در حافظه‌ی Kernel باقی نمی‌مانند.

دو Artifact محلی ایجاد می‌شود:

1. یک فایل JSON به‌عنوان منبع حقیقت پایدار Human Review؛
2. یک Notebook سبک و خوانا برای مرور نهایی توسط دکتر رودی.

Notebook اصلی همچنان محیط کاری ما باقی می‌ماند. پس از هر تغییر در Pilotهای بعدی، تابع Sync همین Cell، Registry JSON را به‌روزرسانی می‌کند.

Author Review Notebook داده‌ی مستقلی نگه نمی‌دارد و همیشه Registry JSON را می‌خواند؛ بنابراین لازم نیست هنگام ادامه‌ی کار هر دو Notebook هم‌زمان باز باشند.

Registry فعلی شامل:

- Pilot تاریخی #1؛
- Human Review کامل Pilot #2؛
- Pilot فعال فعلی و Question پذیرفته‌شده و Expected Answer در انتظار Human Review؛
- وضعیت مستقل Author Final Review.

در این Cell فقط همین دو فایل محلی ایجاد یا به‌روزرسانی می‌شوند.

هیچ Commit، Push، PR یا Merge انجام نمی‌شود.

</div>

In [21]:
# Code Cell 21 — REPLACEMENT

cell21_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell20_gate") is not True:
    cell21_failures.append(
        "CELL20_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell20_human_review_registry"
    ),
    dict,
):
    cell21_failures.append(
        "HUMAN_REVIEW_REGISTRY_MISSING"
    )


# ============================================================
# 2) Resolve Repository Root safely
#
# Do not depend on a historical variable named `repo_root`.
# ============================================================

def cell21_find_repo_root():
    candidates = []


    # Prefer paths already proven to belong to this repository.
    for global_name in (
        "prior_notebook_path",
        "handoff_path",
    ):

        candidate = globals().get(
            global_name
        )


        if candidate is None:
            continue


        try:
            candidate_path = Path(
                candidate
            ).resolve()

            if candidate_path.is_file():
                candidate_path = (
                    candidate_path.parent
                )

            candidates.append(
                candidate_path
            )

        except Exception:
            pass


    # Current working directory is a fallback.
    try:
        candidates.append(
            Path.cwd().resolve()
        )
    except Exception:
        pass


    searched = set()


    for candidate in candidates:

        for path in (
            [candidate]
            +
            list(
                candidate.parents
            )
        ):

            path_text = str(
                path
            )


            if path_text in searched:
                continue


            searched.add(
                path_text
            )


            # Strongest signal: Git repository root.
            if (
                path
                /
                ".git"
            ).exists():

                return path


            # Secondary structural signal for this project.
            if all([
                (
                    path
                    /
                    "notebooks"
                    /
                    "phase2"
                ).exists(),

                (
                    path
                    /
                    "data"
                    /
                    "phase2"
                ).exists(),
            ]):

                return path


    raise RuntimeError(
        "Could not resolve the RAG_finance repository root."
    )


try:

    cell21_repo_root = (
        cell21_find_repo_root()
    )


except Exception as exc:

    cell21_repo_root = None

    cell21_failures.append({
        "reason":
            "REPO_ROOT_RESOLUTION_FAILED",

        "error":
            repr(
                exc
            ),
    })


# ============================================================
# 3) Persistent paths
# ============================================================

if isinstance(
    cell21_repo_root,
    Path,
):

    cell21_review_dir = (
        cell21_repo_root
        /
        "data"
        /
        "phase2"
        /
        "human_review"
    )


    cell21_registry_path = (
        cell21_review_dir
        /
        "human_review_registry.json"
    )


    cell21_author_notebook_path = (
        cell21_repo_root
        /
        "notebooks"
        /
        "phase2"
        /
        "12c_human_review_for_author.ipynb"
    )


else:

    cell21_review_dir = None
    cell21_registry_path = None
    cell21_author_notebook_path = None


# ============================================================
# 4) JSON-safe canonicalizer
# ============================================================

def cell21_json_safe(
    value,
    seen=None,
):

    if seen is None:
        seen = set()


    if value is None or isinstance(
        value,
        (
            str,
            int,
            float,
            bool,
        ),
    ):
        return value


    if isinstance(
        value,
        Path,
    ):
        return str(
            value
        )


    if isinstance(
        value,
        bytes,
    ):
        return {
            "__type__":
                "bytes",

            "hex":
                value.hex(),
        }


    if callable(
        value
    ):
        return {
            "__type__":
                "callable",

            "name":
                getattr(
                    value,
                    "__name__",
                    type(
                        value
                    ).__name__,
                ),
        }


    if isinstance(
        value,
        dict,
    ):

        object_id = id(
            value
        )


        if object_id in seen:
            return {
                "__type__":
                    "cycle"
            }


        seen.add(
            object_id
        )


        try:

            result = {}


            for key, nested in (
                value.items()
            ):

                if isinstance(
                    key,
                    str,
                ):
                    safe_key = key

                else:

                    safe_key = json.dumps(
                        cell21_json_safe(
                            key,
                            seen,
                        ),
                        ensure_ascii=False,
                        sort_keys=True,
                    )


                result[
                    safe_key
                ] = cell21_json_safe(
                    nested,
                    seen,
                )


            return result


        finally:

            seen.discard(
                object_id
            )


    if isinstance(
        value,
        (
            list,
            tuple,
        ),
    ):

        object_id = id(
            value
        )


        if object_id in seen:
            return [
                "__cycle__"
            ]


        seen.add(
            object_id
        )


        try:

            return [
                cell21_json_safe(
                    nested,
                    seen,
                )

                for nested
                in value
            ]


        finally:

            seen.discard(
                object_id
            )


    if isinstance(
        value,
        (
            set,
            frozenset,
        ),
    ):

        safe_items = [
            cell21_json_safe(
                nested,
                seen,
            )

            for nested
            in value
        ]


        return sorted(
            safe_items,
            key=lambda item:
                json.dumps(
                    item,
                    ensure_ascii=False,
                    sort_keys=True,
                    default=str,
                ),
        )


    return {
        "__type__":
            type(
                value
            ).__name__,

        "repr":
            repr(
                value
            ),
    }


# ============================================================
# 5) Build persistent Registry payload
# ============================================================

cell21_registry_payload = {
    "registry_version":
        "persistent_human_review_registry_v1",

    "project":
        "RAG_finance",

    "phase":
        "phase2_golden_dataset",

    "purpose":
        "HUMAN_AND_AUTHOR_REVIEW",

    "workflow_policy":
        {
            "source_evidence_is_authoritative":
                True,

            "human_comments_are_evidence":
                False,

            "model_interpretation_is_evidence":
                False,

            "author_final_review_required":
                True,

            "full_572_generation":
                (
                    "BLOCKED_PENDING_"
                    "FULL_PILOT_WORKFLOW_VALIDATION"
                ),
        },

    "historical_completed_pilot":
        copy.deepcopy(
            cell20_human_review_registry.get(
                "historical_completed_pilot"
            )
        ),

    "completed_human_review_cases":
        copy.deepcopy(
            cell20_human_review_registry.get(
                "completed_human_review_cases",
                [],
            )
        ),

    "active_workspace":
        copy.deepcopy(
            cell20_human_review_registry.get(
                "active_workspace"
            )
        ),

    "remaining_pilot_count":
        21,

    "continuation_state":
        {
            "next_pilot_id":
                cell19_continuation_state.get(
                    "next_pilot_id"
                ),

            "remaining_pilot_ids":
                copy.deepcopy(
                    cell19_continuation_state.get(
                        "remaining_pilot_ids",
                        [],
                    )
                ),

            "remaining_count":
                cell19_continuation_state.get(
                    "remaining_count"
                ),

            "continuation_state_sha256":
                cell19_continuation_state_sha256,
        },

    "author_review":
        {
            "status":
                "NOT_STARTED",

            "reviewer":
                "AUTHOR",

            "cases_ready_for_review":
                1,

            "notes":
                None,
        },
}


cell21_registry_payload = (
    cell21_json_safe(
        cell21_registry_payload
    )
)


cell21_registry_sha256 = (
    cell10_stable_sha(
        cell21_registry_payload
    )
)


# ============================================================
# 6) Reusable Registry sync function
#
# Future cells can call:
#
# cell21_sync_human_review_registry(updated_registry)
# ============================================================

def cell21_sync_human_review_registry(
    registry_payload,
):

    if not isinstance(
        cell21_registry_path,
        Path,
    ):

        raise RuntimeError(
            "Registry path is unavailable."
        )


    safe_payload = (
        cell21_json_safe(
            registry_payload
        )
    )


    serialized = json.dumps(
        safe_payload,
        ensure_ascii=False,
        sort_keys=True,
        indent=2,
    )


    cell21_review_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    temp_path = (
        cell21_registry_path.with_suffix(
            ".json.tmp"
        )
    )


    temp_path.write_text(
        serialized,
        encoding="utf-8",
    )


    temp_path.replace(
        cell21_registry_path
    )


    loaded = json.loads(
        cell21_registry_path.read_text(
            encoding="utf-8"
        )
    )


    expected_sha = (
        cell10_stable_sha(
            safe_payload
        )
    )


    loaded_sha = (
        cell10_stable_sha(
            loaded
        )
    )


    if loaded_sha != expected_sha:

        raise AssertionError(
            "Persistent Human Review Registry "
            "round-trip fingerprint mismatch."
        )


    return {
        "path":
            str(
                cell21_registry_path
            ),

        "sha256":
            loaded_sha,

        "bytes":
            cell21_registry_path.stat().st_size,
    }


# ============================================================
# 7) Write initial persistent Registry
# ============================================================

cell21_registry_write_result = None


if isinstance(
    cell21_registry_path,
    Path,
):

    try:

        cell21_registry_write_result = (
            cell21_sync_human_review_registry(
                cell21_registry_payload
            )
        )


    except Exception as exc:

        cell21_failures.append({
            "reason":
                "REGISTRY_WRITE_FAILED",

            "error":
                repr(
                    exc
                ),
        })


# ============================================================
# 8) Build lightweight Author Review Notebook
#
# This Notebook is a viewer/report.
# The JSON Registry remains the source of truth.
# ============================================================

cell21_author_nb_cells = []


cell21_author_nb_cells.append({
    "id":
        "intro-review",

    "cell_type":
        "markdown",

    "metadata":
        {},

    "source":
        [
            "# Human / Author Review — Golden Dataset\n",
            "\n",
            "این Notebook برای مرور انسانی و تأیید نهایی مؤلف ساخته شده است.\n",
            "\n",
            "منبع حقیقت این Notebook فایل "
            "`human_review_registry.json` است و داده‌های Review "
            "در خود این Notebook نگهداری نمی‌شوند.\n",
            "\n",
            "برای مشاهده آخرین وضعیت، کافی است Notebook را Run کنید."
        ],
})


cell21_author_loader_code = r'''
from pathlib import Path
import json


def find_registry():
    candidates = []

    cwd = Path.cwd().resolve()

    candidates.extend([
        cwd
        / "data"
        / "phase2"
        / "human_review"
        / "human_review_registry.json",

        cwd
        / ".."
        / ".."
        / "data"
        / "phase2"
        / "human_review"
        / "human_review_registry.json",
    ])

    for parent in [cwd] + list(cwd.parents):
        candidates.append(
            parent
            / "data"
            / "phase2"
            / "human_review"
            / "human_review_registry.json"
        )

    checked = set()

    for candidate in candidates:
        candidate = candidate.resolve()

        if str(candidate) in checked:
            continue

        checked.add(str(candidate))

        if candidate.exists():
            return candidate

    raise FileNotFoundError(
        "human_review_registry.json was not found."
    )


REGISTRY_PATH = find_registry()

review_registry = json.loads(
    REGISTRY_PATH.read_text(
        encoding="utf-8"
    )
)

print("Registry:", REGISTRY_PATH)

print(
    "Completed review cases:",
    len(
        review_registry.get(
            "completed_human_review_cases",
            []
        )
    )
)

print(
    "Active Pilot:",
    (
        review_registry.get(
            "active_workspace"
        )
        or {}
    ).get(
        "pilot_case_id"
    )
)

print(
    "Remaining Pilots:",
    review_registry.get(
        "remaining_pilot_count"
    )
)
'''


cell21_author_nb_cells.append({
    "id":
        "load-registry",

    "cell_type":
        "code",

    "execution_count":
        None,

    "metadata":
        {},

    "outputs":
        [],

    "source":
        cell21_author_loader_code.splitlines(
            keepends=True
        ),
})


cell21_author_renderer_code = r'''
from IPython.display import display, Markdown


def show_text(value):
    if value is None:
        return "—"

    return str(value)


def show_evidence_parts(parts):
    for part in parts or []:

        if not isinstance(part, dict):
            continue

        text = part.get("text")

        if not text:
            continue

        display(
            Markdown(
                "> "
                +
                show_text(text).replace(
                    "\n",
                    "\n> "
                )
            )
        )


completed_cases = review_registry.get(
    "completed_human_review_cases",
    []
)


display(
    Markdown(
        "# موارد تکمیل‌شده برای مرور مؤلف"
    )
)


for index, case in enumerate(
    completed_cases,
    start=1,
):

    display(
        Markdown(
            f"## Case {index}"
        )
    )


    display(
        Markdown(
            "### متن اصلی منبع"
        )
    )


    evidence_parts = (
        case.get(
            "source_evidence"
        )
        or
        case.get(
            "source_evidence_snapshot"
        )
        or
        []
    )


    show_evidence_parts(
        evidence_parts
    )


    drafts = case.get(
        "model_drafts",
        []
    )


    if drafts:

        display(
            Markdown(
                "### تاریخچه سؤال"
            )
        )


        for draft in drafts:

            display(
                Markdown(
                    "**Revision "
                    +
                    show_text(
                        draft.get(
                            "revision"
                        )
                    )
                    +
                    "**  \n"
                    +
                    show_text(
                        draft.get(
                            "question_text"
                        )
                    )
                    +
                    "  \nHuman Decision: **"
                    +
                    show_text(
                        draft.get(
                            "human_decision"
                        )
                    )
                    +
                    "**"
                )
            )


    answer_review = case.get(
        "expected_answer_review"
    )


    if isinstance(
        answer_review,
        dict,
    ):

        display(
            Markdown(
                "### پاسخ پیشنهادی"
            )
        )

        display(
            Markdown(
                show_text(
                    answer_review.get(
                        "expected_answer_text"
                    )
                )
                +
                "  \nHuman Decision: **"
                +
                show_text(
                    answer_review.get(
                        "human_decision"
                    )
                )
                +
                "**"
            )
        )


    citation_review = case.get(
        "citation_review"
    )


    if isinstance(
        citation_review,
        dict,
    ):

        display(
            Markdown(
                "### Citation"
            )
        )

        display(
            Markdown(
                "Human Decision: **"
                +
                show_text(
                    citation_review.get(
                        "human_decision"
                    )
                )
                +
                "**"
            )
        )


    author_review = (
        case.get(
            "author_final_review"
        )
        or
        {}
    )


    display(
        Markdown(
            "### تأیید نهایی مؤلف\n"
            "Status: **"
            +
            show_text(
                author_review.get(
                    "status"
                )
            )
            +
            "**"
        )
    )


active_case = review_registry.get(
    "active_workspace"
)


if isinstance(
    active_case,
    dict,
):

    display(
        Markdown(
            "# Case فعال"
        )
    )


    display(
        Markdown(
            "### متن اصلی منبع"
        )
    )


    show_evidence_parts(
        active_case.get(
            "source_evidence"
        )
        or
        []
    )


    drafts = active_case.get(
        "model_drafts",
        []
    )


    for draft in drafts:

        display(
            Markdown(
                "### سؤال\n"
                +
                show_text(
                    draft.get(
                        "question_text"
                    )
                )
                +
                "\n\nHuman Decision: **"
                +
                show_text(
                    draft.get(
                        "human_decision"
                    )
                )
                +
                "**"
            )
        )


    answer_review = active_case.get(
        "expected_answer_review"
    )


    if isinstance(
        answer_review,
        dict,
    ):

        display(
            Markdown(
                "### پاسخ پیشنهادی\n"
                +
                show_text(
                    answer_review.get(
                        "expected_answer_text"
                    )
                )
                +
                "\n\nHuman Decision: **"
                +
                show_text(
                    answer_review.get(
                        "human_decision"
                    )
                )
                +
                "**"
            )
        )


    display(
        Markdown(
            "### تأیید نهایی مؤلف\n"
            "Status: **"
            +
            show_text(
                (
                    active_case.get(
                        "author_final_review"
                    )
                    or
                    {}
                ).get(
                    "status"
                )
            )
            +
            "**"
        )
    )
'''


cell21_author_nb_cells.append({
    "id":
        "render-review",

    "cell_type":
        "code",

    "execution_count":
        None,

    "metadata":
        {},

    "outputs":
        [],

    "source":
        cell21_author_renderer_code.splitlines(
            keepends=True
        ),
})


cell21_author_notebook = {
    "cells":
        cell21_author_nb_cells,

    "metadata":
        {
            "kernelspec":
                {
                    "display_name":
                        "Python 3",

                    "language":
                        "python",

                    "name":
                        "python3",
                },

            "language_info":
                {
                    "name":
                        "python",

                    "version":
                        "3",
                },
        },

    "nbformat":
        4,

    "nbformat_minor":
        5,
}


# ============================================================
# 9) Write Author Review Notebook
# ============================================================

if isinstance(
    cell21_author_notebook_path,
    Path,
):

    try:

        cell21_author_notebook_path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )


        cell21_author_notebook_path.write_text(
            json.dumps(
                cell21_author_notebook,
                ensure_ascii=False,
                indent=2,
            ),
            encoding="utf-8",
        )


    except Exception as exc:

        cell21_failures.append({
            "reason":
                "AUTHOR_NOTEBOOK_WRITE_FAILED",

            "error":
                repr(
                    exc
                ),
        })


# ============================================================
# 10) Validate written artifacts
# ============================================================

cell21_registry_exists = (
    isinstance(
        cell21_registry_path,
        Path,
    )
    and
    cell21_registry_path.exists()
)


cell21_author_notebook_exists = (
    isinstance(
        cell21_author_notebook_path,
        Path,
    )
    and
    cell21_author_notebook_path.exists()
)


cell21_loaded_registry = None
cell21_loaded_notebook = None


if cell21_registry_exists:

    try:

        cell21_loaded_registry = json.loads(
            cell21_registry_path.read_text(
                encoding="utf-8"
            )
        )

    except Exception as exc:

        cell21_failures.append({
            "reason":
                "REGISTRY_READBACK_FAILED",

            "error":
                repr(
                    exc
                ),
        })


if cell21_author_notebook_exists:

    try:

        cell21_loaded_notebook = json.loads(
            cell21_author_notebook_path.read_text(
                encoding="utf-8"
            )
        )

    except Exception as exc:

        cell21_failures.append({
            "reason":
                "AUTHOR_NOTEBOOK_READBACK_FAILED",

            "error":
                repr(
                    exc
                ),
        })


cell21_registry_roundtrip_valid = (
    isinstance(
        cell21_loaded_registry,
        dict,
    )
    and
    cell10_stable_sha(
        cell21_loaded_registry
    )
    ==
    cell21_registry_sha256
)


cell21_author_notebook_valid = all([
    isinstance(
        cell21_loaded_notebook,
        dict,
    ),

    (
        cell21_loaded_notebook
        or
        {}
    ).get(
        "nbformat"
    )
    ==
    4,

    isinstance(
        (
            cell21_loaded_notebook
            or
            {}
        ).get(
            "cells"
        ),
        list,
    ),

    len(
        (
            cell21_loaded_notebook
            or
            {}
        ).get(
            "cells",
            [],
        )
    )
    ==
    3,

    all(
        isinstance(
            cell.get(
                "id"
            ),
            str,
        )
        and
        bool(
            cell.get(
                "id"
            )
        )

        for cell
        in (
            (
                cell21_loaded_notebook
                or
                {}
            ).get(
                "cells",
                [],
            )
        )
    ),
])


# ============================================================
# 11) Persistence surface validation
# ============================================================

cell21_completed_case_count = (
    len(
        cell21_loaded_registry.get(
            "completed_human_review_cases",
            [],
        )
    )

    if isinstance(
        cell21_loaded_registry,
        dict,
    )

    else 0
)


cell21_active_case = (
    (
        cell21_loaded_registry.get(
            "active_workspace"
        )
        or
        {}
    )

    if isinstance(
        cell21_loaded_registry,
        dict,
    )

    else {}
)


cell21_active_model_drafts = (
    cell21_active_case.get(
        "model_drafts",
        [],
    )
)


cell21_current_state_persisted = all([
    cell21_completed_case_count
    ==
    1,

    isinstance(
        cell21_loaded_registry,
        dict,
    ),

    cell21_loaded_registry.get(
        "historical_completed_pilot"
    )
    is not None,

    cell21_active_case.get(
        "pilot_case_id"
    )
    ==
    cell20_active_human_review_case[
        "pilot_case_id"
    ],

    cell21_active_case.get(
        "final_question_draft_id"
    )
    ==
    cell20_question_draft_id,

    cell21_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "expected_answer_draft_id"
    )
    ==
    cell20_expected_answer_draft_id,

    cell21_loaded_registry.get(
        "remaining_pilot_count"
    )
    ==
    21,

    len(
        cell21_active_model_drafts
    )
    >=
    1,
])


# ============================================================
# 12) Confirm exact expected files only
# ============================================================

cell21_expected_write_paths = {
    str(
        cell21_registry_path
    ),

    str(
        cell21_author_notebook_path
    ),
}


cell21_expected_write_surface_valid = all([
    cell21_registry_exists,
    cell21_author_notebook_exists,
    len(
        cell21_expected_write_paths
    )
    ==
    2,
])


# ============================================================
# 13) Final Gate
# ============================================================

cell21_gate = all([
    len(
        cell21_failures
    )
    ==
    0,

    isinstance(
        cell21_repo_root,
        Path,
    ),

    cell21_registry_exists,

    cell21_author_notebook_exists,

    cell21_registry_roundtrip_valid,

    cell21_author_notebook_valid,

    cell21_current_state_persisted,

    cell21_expected_write_surface_valid,
])


# ============================================================
# 14) Diagnostics
# ============================================================

print(
    "=== PERSISTENT HUMAN REVIEW SYSTEM ==="
)


print(
    "\n=== REPOSITORY ==="
)

print(
    "Repository root:",
    cell21_repo_root,
)

print(
    "Root resolved:",
    isinstance(
        cell21_repo_root,
        Path,
    ),
)


print(
    "\n=== REGISTRY ==="
)

print(
    "Path:",
    cell21_registry_path,
)

print(
    "Exists:",
    cell21_registry_exists,
)

print(
    "Bytes:",
    (
        cell21_registry_path.stat().st_size

        if cell21_registry_exists

        else None
    ),
)

print(
    "SHA256:",
    cell21_registry_sha256,
)

print(
    "Round-trip valid:",
    cell21_registry_roundtrip_valid,
)


print(
    "\n=== AUTHOR REVIEW NOTEBOOK ==="
)

print(
    "Path:",
    cell21_author_notebook_path,
)

print(
    "Exists:",
    cell21_author_notebook_exists,
)

print(
    "Valid notebook:",
    cell21_author_notebook_valid,
)

print(
    "Cells:",
    (
        len(
            cell21_loaded_notebook.get(
                "cells",
                [],
            )
        )

        if isinstance(
            cell21_loaded_notebook,
            dict,
        )

        else None
    ),
)


print(
    "\n=== PERSISTED REVIEW STATE ==="
)

print(
    "Historical Pilot #1 retained:",
    (
        isinstance(
            cell21_loaded_registry,
            dict,
        )
        and
        cell21_loaded_registry.get(
            "historical_completed_pilot"
        )
        is not None
    ),
)

print(
    "Completed Human Review cases:",
    cell21_completed_case_count,
)

print(
    "Active Pilot:",
    cell21_active_case.get(
        "pilot_case_id"
    ),
)

print(
    "Active Question:",
    (
        cell21_active_model_drafts[
            -1
        ].get(
            "question_text"
        )

        if cell21_active_model_drafts

        else None
    ),
)

print(
    "Active Answer decision:",
    cell21_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    ),
)

print(
    "Remaining Pilots:",
    (
        cell21_loaded_registry.get(
            "remaining_pilot_count"
        )

        if isinstance(
            cell21_loaded_registry,
            dict,
        )

        else None
    ),
)

print(
    "Persistence state valid:",
    cell21_current_state_persisted,
)


print(
    "\n=== FUTURE WORKFLOW ==="
)

print(
    "Continue working only in this notebook:",
    True,
)

print(
    "Need Author Notebook open during work:",
    False,
)

print(
    "Future Registry sync function:",
    "cell21_sync_human_review_registry(...)",
)

print(
    "Files intentionally written:",
    2,
)

print(
    "Commit / Push performed:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell21_failures
)


print(
    "\nCODE CELL 21 GATE:",
    (
        "PASS"
        if cell21_gate
        else "FAIL"
    ),
)


assert cell21_gate, (
    "Persistent Human Review Registry / Author Review "
    "Notebook creation failed. Repair this SAME Code Cell 21 "
    "before continuing Pilot review."
)

=== PERSISTENT HUMAN REVIEW SYSTEM ===

=== REPOSITORY ===
Repository root: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance
Root resolved: True

=== REGISTRY ===
Path: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/data/phase2/human_review/human_review_registry.json
Exists: True
Bytes: 77499
SHA256: 9967f7522713c60aab5f1d2843480a5cbefe948fd5fb4dcda398685553e55a9b
Round-trip valid: True

=== AUTHOR REVIEW NOTEBOOK ===
Path: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2/12c_human_review_for_author.ipynb
Exists: True
Valid notebook: True
Cells: 3

=== PERSISTED REVIEW STATE ===
Historical Pilot #1 retained: True
Completed Human Review cases: 1
Active Pilot: question-pilot:083ad832854722e126c05d0b89d00cb1
Active Question: سواد مالی در اروپا چقدر مورد توجه قرار گرفته و چند درصد از اروپایی‌ها سواد مالی مناسبی دارند؟
Active Answer decision: PENDING
Remaining Pilots: 21
Persistence state valid: Tr

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 22 — ثبت Answer ACCEPT، ساخت Citation و Sync پایدار Human Review

Expected Answer این Pilot توسط Human Reviewer پذیرفته شد.

در این Cell:

- تصمیم انسانی `ACCEPT` برای Expected Answer ثبت می‌شود؛
- Citation فقط از Target Evidence و Locatorهای از قبل اعتبارسنجی‌شده ساخته می‌شود؛
- هیچ URL، Document ID یا محدوده‌ی متنی جدیدی حدس زده نمی‌شود؛
- Citation به‌صورت ماشینی اعتبارسنجی می‌شود؛
- Workflow تا `CITATION_HUMAN_REVIEW_PENDING` پیش می‌رود؛
- Human Review Registry به‌روزرسانی و با `human_review_registry.json` روی دیسک Sync می‌شود.

Notebook مخصوص دکتر رودی نیازی به بازنویسی ندارد، زیرا هنگام اجرا آخرین Registry JSON را می‌خواند.

در این مرحله Citation هنوز Human ACCEPT نشده، Pilot کامل نشده و Queue همچنان 21 باقی می‌ماند.

Full 572 Generation نیز همچنان Blocked است.

</div>

In [22]:
# Code Cell 22

cell22_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get(
    "cell21_gate"
) is not True:
    cell22_failures.append(
        "CELL21_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell20_current_session"
    ),
    dict,
):
    cell22_failures.append(
        "CELL20_CURRENT_SESSION_MISSING"
    )


if (
    globals().get(
        "cell20_current_session",
        {},
    ).get(
        "current_state"
    )
    !=
    "EXPECTED_ANSWER_HUMAN_REVIEW_PENDING"
):
    cell22_failures.append(
        "EXPECTED_ANSWER_NOT_AT_HUMAN_REVIEW_GATE"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell22_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Current Pilot identity
# ============================================================

CELL22_PILOT_ID = (
    cell19_next_pilot_id
)

CELL22_GOLDEN_CASE_ID = (
    cell19_next_golden_case_id
)

CELL22_TASK_DRAFT_ID = (
    cell19_next_task_draft_id
)

CELL22_TARGET_EVIDENCE_ID = (
    cell19_next_target_evidence_id
)

CELL22_QUESTION_DRAFT_ID = (
    cell20_question_draft_id
)

CELL22_EXPECTED_ANSWER_DRAFT_ID = (
    cell20_expected_answer_draft_id
)

CELL22_FULL_572_BLOCK_STATUS = (
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


# ============================================================
# 3) Verify Expected Answer artifact
# ============================================================

cell22_expected_answer_record = copy.deepcopy(
    cell20_current_session[
        "artifacts"
    ][
        "expected_answer"
    ]
)


cell22_expected_answer_identity_valid = all([
    cell22_expected_answer_record.get(
        "expected_answer_draft_id"
    )
    ==
    CELL22_EXPECTED_ANSWER_DRAFT_ID,

    cell22_expected_answer_record.get(
        "pilot_case_id"
    )
    ==
    CELL22_PILOT_ID,

    cell22_expected_answer_record.get(
        "question_draft_id"
    )
    ==
    CELL22_QUESTION_DRAFT_ID,

    cell22_expected_answer_record.get(
        "target_evidence_id"
    )
    ==
    CELL22_TARGET_EVIDENCE_ID,

    cell22_expected_answer_record.get(
        "machine_validation_status"
    )
    ==
    "PASS_PENDING_HUMAN_ADJUDICATION",
])


# ============================================================
# 4) Human ACCEPT for Expected Answer
# ============================================================

CELL22_HUMAN_ANSWER_NOTE = (
    "Human Reviewer accepted the Expected Answer "
    "after reviewing it against the validated source evidence."
)


cell22_answer_accept_review = {
    "review_record_version":
        "human_expected_answer_review_record_v1",

    "pilot_case_id":
        CELL22_PILOT_ID,

    "golden_case_id":
        CELL22_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL22_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL22_TARGET_EVIDENCE_ID,

    "question_draft_id":
        CELL22_QUESTION_DRAFT_ID,

    "expected_answer_draft_id":
        CELL22_EXPECTED_ANSWER_DRAFT_ID,

    "review_stage":
        "EXPECTED_ANSWER",

    "expected_answer_text":
        cell22_expected_answer_record.get(
            "expected_answer_text"
        ),

    "human_decision":
        "ACCEPT",

    "human_note":
        CELL22_HUMAN_ANSWER_NOTE,

    "human_context_used_as_evidence":
        False,

    "external_knowledge_used":
        False,
}


cell22_answer_accept_review_sha256 = (
    cell10_stable_sha(
        cell22_answer_accept_review
    )
)


# ============================================================
# 5) Apply real Expected Answer ACCEPT
# ============================================================

cell22_citation_pending_session = (
    cell13_apply_event(
        cell20_current_session,
        "EXPECTED_ANSWER_ACCEPTED",
        human_note=
            CELL22_HUMAN_ANSWER_NOTE,
    )
)


cell22_answer_accept_transition_valid = all([
    cell22_citation_pending_session.get(
        "current_state"
    )
    ==
    "CITATION_MAPPING_PENDING",

    len(
        cell22_citation_pending_session.get(
            "human_decisions",
            [],
        )
    )
    ==
    2,

    cell22_citation_pending_session[
        "human_decisions"
    ][
        -1
    ].get(
        "event"
    )
    ==
    "EXPECTED_ANSWER_ACCEPTED",

    cell22_citation_pending_session.get(
        "queue_advance_permitted"
    )
    is False,

    cell22_citation_pending_session.get(
        "queue_advance_performed"
    )
    is False,
])


# ============================================================
# 6) Exact Target Parts from validated Evidence
# ============================================================

cell22_source_target_parts = copy.deepcopy(
    cell19_next_target_evidence.get(
        "target_parts",
        [],
    )
)


cell22_source_target_parts_valid = all([
    isinstance(
        cell22_source_target_parts,
        list,
    ),

    len(
        cell22_source_target_parts
    )
    >
    0,
])


cell22_citation_entries = []


for index, part in enumerate(
    cell22_source_target_parts
):

    if not isinstance(
        part,
        dict,
    ):
        continue


    cell22_citation_entries.append({
        "citation_index":
            index,

        "target_evidence_id":
            CELL22_TARGET_EVIDENCE_ID,

        "document_id":
            part.get(
                "document_id"
            ),

        "source_id":
            part.get(
                "source_id"
            ),

        "role":
            part.get(
                "role"
            ),

        "char_start":
            part.get(
                "char_start"
            ),

        "char_end":
            part.get(
                "char_end"
            ),

        "char_count":
            part.get(
                "char_count"
            ),

        "token_count":
            part.get(
                "token_count"
            ),

        "locator":
            copy.deepcopy(
                part.get(
                    "locator",
                    {},
                )
            ),

        "attribution":
            copy.deepcopy(
                part.get(
                    "attribution"
                )
            ),

        "locator_origin":
            "COPIED_FROM_VALIDATED_TARGET_EVIDENCE",
    })


# ============================================================
# 7) Validate exact locator / range projection
# ============================================================

cell22_source_projection = [
    {
        "document_id":
            part.get(
                "document_id"
            ),

        "source_id":
            part.get(
                "source_id"
            ),

        "role":
            part.get(
                "role"
            ),

        "char_start":
            part.get(
                "char_start"
            ),

        "char_end":
            part.get(
                "char_end"
            ),

        "locator":
            copy.deepcopy(
                part.get(
                    "locator",
                    {},
                )
            ),
    }

    for part
    in cell22_source_target_parts

    if isinstance(
        part,
        dict,
    )
]


cell22_citation_projection = [
    {
        "document_id":
            entry.get(
                "document_id"
            ),

        "source_id":
            entry.get(
                "source_id"
            ),

        "role":
            entry.get(
                "role"
            ),

        "char_start":
            entry.get(
                "char_start"
            ),

        "char_end":
            entry.get(
                "char_end"
            ),

        "locator":
            copy.deepcopy(
                entry.get(
                    "locator",
                    {},
                )
            ),
    }

    for entry
    in cell22_citation_entries
]


cell22_no_locator_fabrication = (
    cell22_source_projection
    ==
    cell22_citation_projection
)


# ============================================================
# 8) Validate against displayed Target Text Parts
# ============================================================

cell22_displayed_parts_by_document = {
    row.get(
        "document_id"
    ):
        row

    for row
    in cell19_next_target_text_parts

    if isinstance(
        row,
        dict,
    )
}


cell22_citation_validation_rows = []


for entry in cell22_citation_entries:

    document_id = (
        entry.get(
            "document_id"
        )
    )


    displayed_part = (
        cell22_displayed_parts_by_document.get(
            document_id
        )
    )


    locator = (
        entry.get(
            "locator"
        )
        or
        {}
    )


    canonical_url = (
        locator.get(
            "canonical_url"
        )
    )


    url = (
        locator.get(
            "url"
        )
    )


    row_valid = all([
        isinstance(
            document_id,
            str,
        ),

        isinstance(
            displayed_part,
            dict,
        ),

        entry.get(
            "char_start"
        )
        ==
        displayed_part.get(
            "char_start"
        ),

        entry.get(
            "char_end"
        )
        ==
        displayed_part.get(
            "char_end"
        ),

        entry.get(
            "role"
        )
        ==
        displayed_part.get(
            "role"
        ),

        (
            canonical_url
            is None
            or
            (
                isinstance(
                    canonical_url,
                    str,
                )
                and
                canonical_url.startswith(
                    (
                        "http://",
                        "https://",
                    )
                )
            )
        ),

        (
            url
            is None
            or
            (
                isinstance(
                    url,
                    str,
                )
                and
                url.startswith(
                    (
                        "http://",
                        "https://",
                    )
                )
            )
        ),

        entry.get(
            "locator_origin"
        )
        ==
        "COPIED_FROM_VALIDATED_TARGET_EVIDENCE",
    ])


    cell22_citation_validation_rows.append({
        "citation_index":
            entry.get(
                "citation_index"
            ),

        "document_id":
            document_id,

        "range_match":
            (
                isinstance(
                    displayed_part,
                    dict,
                )
                and
                entry.get(
                    "char_start"
                )
                ==
                displayed_part.get(
                    "char_start"
                )
                and
                entry.get(
                    "char_end"
                )
                ==
                displayed_part.get(
                    "char_end"
                )
            ),

        "role_match":
            (
                isinstance(
                    displayed_part,
                    dict,
                )
                and
                entry.get(
                    "role"
                )
                ==
                displayed_part.get(
                    "role"
                )
            ),

        "valid":
            row_valid,
    })


cell22_all_citation_entries_valid = all([
    len(
        cell22_citation_entries
    )
    >
    0,

    len(
        cell22_citation_entries
    )
    ==
    len(
        cell19_next_target_text_parts
    ),

    all(
        row.get(
            "valid"
        )
        is True

        for row
        in cell22_citation_validation_rows
    ),
])


# ============================================================
# 9) Stable local Citation Mapping ID
# ============================================================

cell22_citation_identity_payload = {
    "identity_version":
        "citation_mapping_identity_v1",

    "pilot_case_id":
        CELL22_PILOT_ID,

    "question_draft_id":
        CELL22_QUESTION_DRAFT_ID,

    "expected_answer_draft_id":
        CELL22_EXPECTED_ANSWER_DRAFT_ID,

    "target_evidence_id":
        CELL22_TARGET_EVIDENCE_ID,

    "citation_projection":
        copy.deepcopy(
            cell22_citation_projection
        ),
}


cell22_citation_mapping_id = (
    "citation-mapping:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell22_citation_identity_payload
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


# ============================================================
# 10) Citation Machine Validation
# ============================================================

cell22_citation_machine_validation = {
    "target_parts_present":
        cell22_source_target_parts_valid,

    "citation_entries_present":
        (
            len(
                cell22_citation_entries
            )
            >
            0
        ),

    "all_target_parts_covered":
        (
            len(
                cell22_citation_entries
            )
            ==
            len(
                cell19_next_target_text_parts
            )
        ),

    "all_entries_valid":
        cell22_all_citation_entries_valid,

    "locator_fabrication_absent":
        cell22_no_locator_fabrication,

    "target_evidence_identity_match":
        all(
            entry.get(
                "target_evidence_id"
            )
            ==
            CELL22_TARGET_EVIDENCE_ID

            for entry
            in cell22_citation_entries
        ),

    "external_locator_lookup_used":
        False,
}


cell22_citation_machine_validation_pass = all([
    cell22_citation_machine_validation[
        "target_parts_present"
    ],

    cell22_citation_machine_validation[
        "citation_entries_present"
    ],

    cell22_citation_machine_validation[
        "all_target_parts_covered"
    ],

    cell22_citation_machine_validation[
        "all_entries_valid"
    ],

    cell22_citation_machine_validation[
        "locator_fabrication_absent"
    ],

    cell22_citation_machine_validation[
        "target_evidence_identity_match"
    ],
])


# ============================================================
# 11) Citation Mapping Record
# ============================================================

cell22_citation_mapping_record = {
    "record_version":
        "citation_mapping_record_v1",

    "citation_mapping_id":
        cell22_citation_mapping_id,

    "pilot_case_id":
        CELL22_PILOT_ID,

    "golden_case_id":
        CELL22_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL22_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL22_TARGET_EVIDENCE_ID,

    "question_draft_id":
        CELL22_QUESTION_DRAFT_ID,

    "expected_answer_draft_id":
        CELL22_EXPECTED_ANSWER_DRAFT_ID,

    "citation_entries":
        copy.deepcopy(
            cell22_citation_entries
        ),

    "citation_scope":
        "VALIDATED_TARGET_EVIDENCE_ONLY",

    "locator_source":
        "EXISTING_TARGET_EVIDENCE_LOCATORS",

    "external_locator_lookup_used":
        False,

    "machine_validation_status":
        (
            "PASS_PENDING_HUMAN_ADJUDICATION"

            if cell22_citation_machine_validation_pass

            else "FAIL_REQUIRES_REVISION"
        ),

    "machine_validation":
        copy.deepcopy(
            cell22_citation_machine_validation
        ),

    "human_adjudication_status":
        "PENDING",

    "full_572_generation":
        CELL22_FULL_572_BLOCK_STATUS,
}


cell22_citation_mapping_record_sha256 = (
    cell10_stable_sha(
        cell22_citation_mapping_record
    )
)


# ============================================================
# 12) Register Citation Draft
# ============================================================

cell22_current_session = None


if all([
    cell22_expected_answer_identity_valid,
    cell22_answer_accept_transition_valid,
    cell22_citation_machine_validation_pass,
]):

    cell22_current_session = (
        cell13_apply_event(
            cell22_citation_pending_session,
            "CITATION_MAPPING_REGISTERED",
            artifact=
                cell22_citation_mapping_record,
        )
    )


cell22_citation_registered = all([
    isinstance(
        cell22_current_session,
        dict,
    ),

    cell22_current_session.get(
        "current_state"
    )
    ==
    "CITATION_HUMAN_REVIEW_PENDING",

    isinstance(
        cell22_current_session[
            "artifacts"
        ][
            "citation_mapping"
        ],
        dict,
    ),

    cell22_current_session[
        "artifacts"
    ][
        "citation_mapping"
    ].get(
        "citation_mapping_id"
    )
    ==
    cell22_citation_mapping_id,

    len(
        cell22_current_session.get(
            "human_decisions",
            [],
        )
    )
    ==
    2,

    cell22_current_session.get(
        "queue_advance_permitted"
    )
    is False,

    cell22_current_session.get(
        "queue_advance_performed"
    )
    is False,
])


# ============================================================
# 13) Update in-memory Human Review Registry
# ============================================================

cell22_human_review_registry = copy.deepcopy(
    cell20_human_review_registry
)


cell22_active_case = copy.deepcopy(
    cell20_active_human_review_case
)


cell22_active_case[
    "expected_answer_review"
][
    "human_decision"
] = "ACCEPT"


cell22_active_case[
    "expected_answer_review"
][
    "human_review"
] = copy.deepcopy(
    cell22_answer_accept_review
)


cell22_active_case[
    "citation_review"
] = {
    "citation_mapping_id":
        cell22_citation_mapping_id,

    "citation_entries":
        copy.deepcopy(
            cell22_citation_entries
        ),

    "machine_validation_status":
        cell22_citation_mapping_record[
            "machine_validation_status"
        ],

    "human_decision":
        "PENDING",
}


cell22_active_case[
    "final_status"
] = (
    "QUESTION_AND_ANSWER_ACCEPTED_"
    "CITATION_PENDING_REVIEW"
)


cell22_human_review_registry[
    "active_workspace"
] = copy.deepcopy(
    cell22_active_case
)


cell22_human_review_registry[
    "remaining_pilot_count"
] = 21


# ============================================================
# 14) Build persistent Registry payload
# ============================================================

cell22_persistent_registry_payload = {
    "registry_version":
        "persistent_human_review_registry_v1",

    "project":
        "RAG_finance",

    "phase":
        "phase2_golden_dataset",

    "purpose":
        "HUMAN_AND_AUTHOR_REVIEW",

    "workflow_policy":
        {
            "source_evidence_is_authoritative":
                True,

            "human_comments_are_evidence":
                False,

            "model_interpretation_is_evidence":
                False,

            "author_final_review_required":
                True,

            "full_572_generation":
                CELL22_FULL_572_BLOCK_STATUS,
        },

    "historical_completed_pilot":
        copy.deepcopy(
            cell22_human_review_registry.get(
                "historical_completed_pilot"
            )
        ),

    "completed_human_review_cases":
        copy.deepcopy(
            cell22_human_review_registry.get(
                "completed_human_review_cases",
                [],
            )
        ),

    "active_workspace":
        copy.deepcopy(
            cell22_human_review_registry.get(
                "active_workspace"
            )
        ),

    "remaining_pilot_count":
        21,

    "continuation_state":
        {
            "next_pilot_id":
                cell19_continuation_state.get(
                    "next_pilot_id"
                ),

            "remaining_pilot_ids":
                copy.deepcopy(
                    cell19_continuation_state.get(
                        "remaining_pilot_ids",
                        [],
                    )
                ),

            "remaining_count":
                cell19_continuation_state.get(
                    "remaining_count"
                ),

            "continuation_state_sha256":
                cell19_continuation_state_sha256,
        },

    "author_review":
        copy.deepcopy(
            cell21_loaded_registry.get(
                "author_review",
                {
                    "status":
                        "NOT_STARTED",

                    "reviewer":
                        "AUTHOR",

                    "cases_ready_for_review":
                        1,

                    "notes":
                        None,
                },
            )
        ),
}


# ============================================================
# 15) Sync Registry to disk
# ============================================================

cell22_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell22_persistent_registry_payload
    )
)


cell22_registry_sync_valid = all([
    isinstance(
        cell22_registry_sync_result,
        dict,
    ),

    isinstance(
        cell22_registry_sync_result.get(
            "sha256"
        ),
        str,
    ),

    cell21_registry_path.exists(),
])


# ============================================================
# 16) Read back persisted state
# ============================================================

cell22_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell22_persisted_active_case = (
    cell22_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell22_persistence_state_valid = all([
    cell22_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL22_PILOT_ID,

    cell22_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell22_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "citation_mapping_id"
    )
    ==
    cell22_citation_mapping_id,

    cell22_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "PENDING",

    cell22_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    21,
])


# ============================================================
# 17) Queue / safety
# ============================================================

cell22_queue_still_21 = (
    len(
        cell19_continuation_state[
            "remaining_pilot_ids"
        ]
    )
    ==
    21
)


cell22_next_pilot_unchanged = (
    cell19_continuation_state[
        "next_pilot_id"
    ]
    ==
    CELL22_PILOT_ID
)


cell22_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell22_full_572_blocked = all([
    isinstance(
        cell22_current_session,
        dict,
    ),

    cell22_current_session.get(
        "full_572_generation"
    )
    ==
    CELL22_FULL_572_BLOCK_STATUS,

    cell22_citation_mapping_record[
        "full_572_generation"
    ]
    ==
    CELL22_FULL_572_BLOCK_STATUS,
])


# ============================================================
# 18) Final Gate
# ============================================================

cell22_gate = all([
    len(
        cell22_failures
    )
    ==
    0,

    cell22_expected_answer_identity_valid,

    cell22_answer_accept_transition_valid,

    cell22_source_target_parts_valid,

    cell22_no_locator_fabrication,

    cell22_all_citation_entries_valid,

    cell22_citation_machine_validation_pass,

    cell22_citation_registered,

    cell22_registry_sync_valid,

    cell22_persistence_state_valid,

    cell22_queue_still_21,

    cell22_next_pilot_unchanged,

    cell22_base_seed_unchanged,

    cell22_full_572_blocked,
])


# ============================================================
# 19) Diagnostics
# ============================================================

print(
    "=== ANSWER ACCEPTED + CITATION + "
    "PERSISTENT REGISTRY SYNC ==="
)


print(
    "\n=== ACCEPTED QUESTION ==="
)

print(
    cell20_question_text
)


print(
    "\n=== ACCEPTED ANSWER ==="
)

print(
    cell20_expected_answer_text
)

print(
    "Human Answer Decision:",
    "ACCEPT",
)


print(
    "\n=== CITATION DRAFT ==="
)

print(
    "Citation Mapping ID:",
    cell22_citation_mapping_id,
)

print(
    "Citation entry count:",
    len(
        cell22_citation_entries
    ),
)


for entry in cell22_citation_entries:

    print(
        "\n--- CITATION ENTRY",
        entry.get(
            "citation_index"
        ),
        "---"
    )

    print(
        "Document:",
        entry.get(
            "document_id"
        ),
    )

    print(
        "Role:",
        entry.get(
            "role"
        ),
    )

    print(
        "Chars:",
        (
            entry.get(
                "char_start"
            ),
            entry.get(
                "char_end"
            ),
        ),
    )

    print(
        "Canonical URL:",
        (
            entry.get(
                "locator",
                {}
            ).get(
                "canonical_url"
            )
        ),
    )

    print(
        "URL:",
        (
            entry.get(
                "locator",
                {}
            ).get(
                "url"
            )
        ),
    )


print(
    "\n=== CITATION MACHINE VALIDATION ==="
)

for name, status in (
    cell22_citation_machine_validation.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "Machine validation PASS:",
    cell22_citation_machine_validation_pass,
)

print(
    "No locator fabrication:",
    cell22_no_locator_fabrication,
)


print(
    "\n=== WORKFLOW STATE ==="
)

print(
    "Current state:",
    (
        cell22_current_session.get(
            "current_state"
        )

        if isinstance(
            cell22_current_session,
            dict,
        )

        else None
    ),
)

print(
    "Citation decision:",
    "PENDING",
)

print(
    "Queue advance permitted:",
    (
        cell22_current_session.get(
            "queue_advance_permitted"
        )

        if isinstance(
            cell22_current_session,
            dict,
        )

        else None
    ),
)


print(
    "\n=== PERSISTENCE ==="
)

print(
    "Registry path:",
    cell21_registry_path,
)

print(
    "Registry synced:",
    cell22_registry_sync_valid,
)

print(
    "Persisted Answer decision:",
    cell22_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    ),
)

print(
    "Persisted Citation decision:",
    cell22_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "human_decision"
    ),
)

print(
    "Persistence state valid:",
    cell22_persistence_state_valid,
)

print(
    "Author Notebook rewrite required:",
    False,
)


print(
    "\n=== QUEUE / SAFETY ==="
)

print(
    "Queue still 21:",
    cell22_queue_still_21,
)

print(
    "Next Pilot unchanged:",
    cell22_next_pilot_unchanged,
)

print(
    "Base Seed unchanged:",
    cell22_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell22_full_572_blocked,
)

print(
    "Commit / Push performed:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell22_failures
)


print(
    "\nCODE CELL 22 GATE:",
    (
        "PASS"
        if cell22_gate
        else "FAIL"
    ),
)


assert cell22_gate, (
    "Answer ACCEPT / Citation / persistent Registry sync "
    "failed. Repair this SAME Code Cell 22 before final "
    "Human Citation review."
)

=== ANSWER ACCEPTED + CITATION + PERSISTENT REGISTRY SYNC ===

=== ACCEPTED QUESTION ===
سواد مالی در اروپا چقدر مورد توجه قرار گرفته و چند درصد از اروپایی‌ها سواد مالی مناسبی دارند؟

=== ACCEPTED ANSWER ===
بر اساس متن، تنها نیمی از اروپایی‌ها، یعنی حدود 50 درصد، سواد مالی مناسبی دارند. سواد مالی در اروپا مورد بررسی گسترده قرار گرفته است؛ در سال 2014 نظرسنجی جهانی سواد مالی یکی از ابزارهای مقایسه سطح سواد مالی کشورهای اروپایی بود و در بهار 2023 نیز کمیسیون اروپا نظرسنجی جامعی در هر 27 کشور عضو اتحادیه اروپا انجام داد. این سنجش جدید به دلیل پوشش کامل کشورهای عضو، اطلاعات قابل‌اتکاتری برای پژوهشگران و سیاست‌گذاران فراهم کرد.
Human Answer Decision: ACCEPT

=== CITATION DRAFT ===
Citation Mapping ID: citation-mapping:c0e5afc01b8d73443666ddbbcdd5ad6e
Citation entry count: 1

--- CITATION ENTRY 0 ---
Document: website:596080EF
Role: SOURCE_DOCUMENT
Chars: (0, 671)
Canonical URL: https://www.fintelligence.ir/Blog/Detail/596080EF
URL: https://www.fintelligence.ir/Blog/Detail/596080EF/%D8%B3%D

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 23 — Citation ACCEPT، تکمیل Pilot، Sync Registry و آماده‌سازی Pilot بعدی

Citation این Pilot توسط Human Reviewer پذیرفته شد.

در این مرحله:

- Citation با تصمیم `ACCEPT` ثبت می‌شود؛
- Pilot فعلی از نظر Question + Expected Answer + Citation تکمیل می‌شود؛
- پرونده‌ی کامل Human Review آن به Completed Cases منتقل می‌شود؛
- وضعیت Author Final Review همچنان `NOT_STARTED` باقی می‌ماند؛
- Registry دائمی روی دیسک Sync می‌شود؛
- Queue از 21 به 20 کاهش پیدا می‌کند؛
- Pilot بعدی انتخاب می‌شود؛
- Evidence اصلی Pilot بعدی برای شروع Human Question Review نمایش داده می‌شود.

Notebook مخصوص دکتر رودی بازنویسی نمی‌شود؛ هنگام اجرا آخرین JSON را می‌خواند.

Full 572 Generation همچنان Blocked است.

</div>

In [23]:
# Code Cell 23

cell23_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell22_gate") is not True:
    cell23_failures.append(
        "CELL22_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell22_current_session"
    ),
    dict,
):
    cell23_failures.append(
        "CELL22_CURRENT_SESSION_MISSING"
    )


if (
    globals().get(
        "cell22_current_session",
        {},
    ).get(
        "current_state"
    )
    !=
    "CITATION_HUMAN_REVIEW_PENDING"
):
    cell23_failures.append(
        "CITATION_NOT_AT_HUMAN_REVIEW_GATE"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell23_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Current Pilot
# ============================================================

CELL23_COMPLETED_PILOT_ID = (
    cell19_next_pilot_id
)

CELL23_FULL_572_BLOCK_STATUS = (
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


# ============================================================
# 3) Human Citation ACCEPT
# ============================================================

CELL23_HUMAN_CITATION_NOTE = (
    "Human Reviewer accepted the Citation Mapping "
    "after reviewing the validated source locator."
)


cell23_citation_human_review = {
    "review_record_version":
        "human_citation_review_record_v1",

    "pilot_case_id":
        CELL23_COMPLETED_PILOT_ID,

    "citation_mapping_id":
        cell22_citation_mapping_id,

    "target_evidence_id":
        cell19_next_target_evidence_id,

    "human_decision":
        "ACCEPT",

    "human_note":
        CELL23_HUMAN_CITATION_NOTE,

    "external_locator_lookup_used":
        False,
}


# ============================================================
# 4) Apply Citation ACCEPT
# ============================================================

cell23_pre_advance_session = (
    cell13_apply_event(
        cell22_current_session,
        "CITATION_ACCEPTED",
        human_note=
            CELL23_HUMAN_CITATION_NOTE,
    )
)


cell23_citation_accept_valid = all([
    cell23_pre_advance_session.get(
        "current_state"
    )
    ==
    "PILOT_COMPLETE_PENDING_QUEUE_ADVANCE",

    cell23_pre_advance_session.get(
        "queue_advance_permitted"
    )
    is True,

    cell23_pre_advance_session.get(
        "queue_advance_performed"
    )
    is False,

    len(
        cell23_pre_advance_session.get(
            "human_decisions",
            [],
        )
    )
    ==
    3,

    cell23_pre_advance_session[
        "human_decisions"
    ][
        -1
    ].get(
        "event"
    )
    ==
    "CITATION_ACCEPTED",
])


# ============================================================
# 5) Finalize Human Review Case
# ============================================================

cell23_completed_human_review_case = (
    copy.deepcopy(
        cell22_active_case
    )
)


cell23_completed_human_review_case[
    "citation_review"
][
    "human_decision"
] = "ACCEPT"


cell23_completed_human_review_case[
    "citation_review"
][
    "human_review"
] = copy.deepcopy(
    cell23_citation_human_review
)


cell23_completed_human_review_case[
    "author_final_review"
] = {
    "status":
        "NOT_STARTED",

    "decision":
        None,

    "notes":
        None,
}


cell23_completed_human_review_case[
    "final_status"
] = (
    "HUMAN_REVIEW_COMPLETE_"
    "AUTHOR_FINAL_REVIEW_PENDING"
)


cell23_completed_human_review_case_sha256 = (
    cell10_stable_sha(
        cell23_completed_human_review_case
    )
)


# ============================================================
# 6) Queue advancement
# ============================================================

cell23_parent_remaining_ids = copy.deepcopy(
    cell19_continuation_state[
        "remaining_pilot_ids"
    ]
)


cell23_parent_remaining_queue = copy.deepcopy(
    cell19_continuation_state[
        "remaining_pilot_queue"
    ]
)


cell23_queue_preconditions_valid = all([
    len(
        cell23_parent_remaining_ids
    )
    ==
    21,

    len(
        cell23_parent_remaining_queue
    )
    ==
    21,

    cell23_parent_remaining_ids[
        0
    ]
    ==
    CELL23_COMPLETED_PILOT_ID,

    cell23_citation_accept_valid,
])


if cell23_queue_preconditions_valid:

    cell23_remaining_pilot_ids = copy.deepcopy(
        cell23_parent_remaining_ids[
            1:
        ]
    )

    cell23_remaining_pilot_queue = copy.deepcopy(
        cell23_parent_remaining_queue[
            1:
        ]
    )

else:

    cell23_remaining_pilot_ids = []
    cell23_remaining_pilot_queue = []


cell23_next_pilot_id = (
    cell23_remaining_pilot_ids[
        0
    ]

    if cell23_remaining_pilot_ids

    else None
)


cell23_next_pilot_queue_entry = (
    copy.deepcopy(
        cell23_remaining_pilot_queue[
            0
        ]
    )

    if cell23_remaining_pilot_queue

    else None
)


cell23_queue_advanced_valid = all([
    len(
        cell23_remaining_pilot_ids
    )
    ==
    20,

    len(
        cell23_remaining_pilot_queue
    )
    ==
    20,

    CELL23_COMPLETED_PILOT_ID
    not in
    cell23_remaining_pilot_ids,

    isinstance(
        cell23_next_pilot_id,
        str,
    ),

    cell23_next_pilot_id
    !=
    CELL23_COMPLETED_PILOT_ID,
])


# ============================================================
# 7) Completed workflow session
# ============================================================

cell23_completed_session = copy.deepcopy(
    cell23_pre_advance_session
)


cell23_completed_session[
    "current_state"
] = "PILOT_COMPLETE"


cell23_completed_session[
    "queue_advance_permitted"
] = False


cell23_completed_session[
    "queue_advance_performed"
] = True


cell23_completed_session[
    "completion_status"
] = (
    "HUMAN_REVIEW_COMPLETE_"
    "AUTHOR_FINAL_REVIEW_PENDING"
)


# ============================================================
# 8) New authoritative Continuation State
# ============================================================

cell23_continuation_state = {
    "state_version":
        "compact_pilot_continuation_state_v2",

    "authoritative_forward_state":
        True,

    "parent_state_sha256":
        cell19_continuation_state_sha256,

    "just_completed_pilot_id":
        CELL23_COMPLETED_PILOT_ID,

    "just_completed_case_sha256":
        cell23_completed_human_review_case_sha256,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell23_remaining_pilot_ids
        ),

    "remaining_pilot_queue":
        copy.deepcopy(
            cell23_remaining_pilot_queue
        ),

    "remaining_count":
        20,

    "next_pilot_id":
        cell23_next_pilot_id,

    "next_pilot_queue_entry":
        copy.deepcopy(
            cell23_next_pilot_queue_entry
        ),

    "full_572_generation":
        CELL23_FULL_572_BLOCK_STATUS,
}


cell23_continuation_state_sha256 = (
    cell10_stable_sha(
        cell23_continuation_state
    )
)


# ============================================================
# 9) Resolve next Pilot packet
# ============================================================

cell23_pilot_by_id = (
    cell13_seed[
        "final_runtime"
    ][
        "question_drafting_pilot_by_id"
    ]
)


cell23_next_pilot_packet = copy.deepcopy(
    cell23_pilot_by_id.get(
        cell23_next_pilot_id
    )
)


cell23_next_packet_valid = isinstance(
    cell23_next_pilot_packet,
    dict,
)


# ============================================================
# 10) Resolve next identity chain
# ============================================================

cell23_next_golden_case_id = None
cell23_next_task_draft_id = None
cell23_next_target_evidence_id = None


if cell23_next_packet_valid:

    try:

        cell23_next_golden_case_id = (
            cell13_exact_single_id(
                cell23_next_pilot_packet,
                "golden_case",
            )
        )


        cell23_next_task_draft_id = (
            cell13_exact_single_id(
                cell23_next_pilot_packet,
                "task_draft",
            )
        )


        cell23_next_target_evidence_id = (
            cell13_exact_single_id(
                cell23_next_pilot_packet,
                "target_evidence",
            )
        )


    except Exception as exc:

        cell23_failures.append({
            "reason":
                "NEXT_ID_CHAIN_FAILED",

            "error":
                repr(
                    exc
                ),
        })


cell23_next_identity_valid = all([
    isinstance(
        cell23_next_golden_case_id,
        str,
    ),

    isinstance(
        cell23_next_task_draft_id,
        str,
    ),

    isinstance(
        cell23_next_target_evidence_id,
        str,
    ),
])


# ============================================================
# 11) Resolve next support records
# ============================================================

cell23_next_golden_case = None
cell23_next_task_plan = None
cell23_next_target_evidence = None


if cell23_next_identity_valid:

    try:

        cell23_next_golden_case = copy.deepcopy(
            cell13_active_support[
                "golden_case_records_by_id"
            ][
                cell23_next_golden_case_id
            ]
        )


        cell23_next_task_plan = copy.deepcopy(
            cell13_find_unique_mapping_value(
                cell13_active_support.get(
                    "task_draft_plans",
                    {},
                ),
                "task_draft",
                cell23_next_task_draft_id,
            )[
                "value"
            ]
        )


        cell23_next_target_evidence = copy.deepcopy(
            cell13_find_unique_list_record(
                cell13_active_support.get(
                    "target_evidence_records",
                    [],
                ),
                "target_evidence_id",
                cell23_next_target_evidence_id,
            )
        )


    except Exception as exc:

        cell23_failures.append({
            "reason":
                "NEXT_SUPPORT_RESOLUTION_FAILED",

            "error":
                repr(
                    exc
                ),
        })


# ============================================================
# 12) Classification
# ============================================================

def cell23_scalar(
    mapping,
    names,
):

    match = cell14_find_first_scalar(
        mapping,
        names,
    )


    if isinstance(
        match,
        dict,
    ):
        return match.get(
            "value"
        )


    return None


cell23_next_source_type = cell23_scalar(
    cell23_next_pilot_packet,
    [
        "source_type",
        "source_kind",
        "source",
    ],
)


cell23_next_stratum = cell23_scalar(
    cell23_next_pilot_packet,
    [
        "stratum",
        "stratum_name",
        "stratum_id",
    ],
)


cell23_next_task_family = cell23_scalar(
    cell23_next_pilot_packet,
    [
        "task_family",
    ],
)


cell23_next_question_archetype = cell23_scalar(
    cell23_next_pilot_packet,
    [
        "question_archetype",
    ],
)


# ============================================================
# 13) Extract next Evidence text
# ============================================================

cell23_raw_target_parts = []


if cell23_next_packet_valid:

    candidate_parts = (
        cell23_next_pilot_packet.get(
            "target_text_parts"
        )
    )


    if isinstance(
        candidate_parts,
        list,
    ):
        cell23_raw_target_parts = copy.deepcopy(
            candidate_parts
        )


if (
    not cell23_raw_target_parts
    and
    isinstance(
        cell23_next_target_evidence,
        dict,
    )
):

    candidate_parts = (
        cell23_next_target_evidence.get(
            "target_parts"
        )
    )


    if isinstance(
        candidate_parts,
        list,
    ):
        cell23_raw_target_parts = copy.deepcopy(
            candidate_parts
        )


cell23_next_target_text_parts = []


for index, part in enumerate(
    cell23_raw_target_parts
):

    if not isinstance(
        part,
        dict,
    ):
        continue


    text_value = None


    for field_name in (
        "text",
        "target_text",
        "content",
        "evidence_text",
        "text_content",
    ):

        candidate = part.get(
            field_name
        )


        if (
            isinstance(
                candidate,
                str,
            )
            and
            candidate.strip()
        ):

            text_value = candidate.strip()
            break


    if text_value is None:
        continue


    cell23_next_target_text_parts.append({
        "part_index":
            index,

        "document_id":
            part.get(
                "document_id"
            ),

        "role":
            part.get(
                "role"
            ),

        "char_start":
            part.get(
                "char_start"
            ),

        "char_end":
            part.get(
                "char_end"
            ),

        "text":
            text_value,

        "text_sha256":
            hashlib.sha256(
                text_value.encode(
                    "utf-8"
                )
            ).hexdigest(),
    })


cell23_next_combined_target_text = (
    "\n\n".join(
        row[
            "text"
        ]

        for row
        in cell23_next_target_text_parts
    )
)


cell23_next_target_text_available = bool(
    cell23_next_combined_target_text.strip()
)


# ============================================================
# 14) Build next Pilot workspace
# ============================================================

cell23_next_pilot_workspace = {
    "workspace_version":
        "compact_human_review_workspace_v1",

    "status":
        "READY_FOR_QUESTION_DRAFT",

    "pilot_case_id":
        cell23_next_pilot_id,

    "golden_case_id":
        cell23_next_golden_case_id,

    "task_draft_id":
        cell23_next_task_draft_id,

    "target_evidence_id":
        cell23_next_target_evidence_id,

    "source_type":
        cell23_next_source_type,

    "stratum":
        cell23_next_stratum,

    "task_family":
        cell23_next_task_family,

    "question_archetype":
        cell23_next_question_archetype,

    "source_evidence":
        copy.deepcopy(
            cell23_next_target_text_parts
        ),

    "combined_target_text":
        cell23_next_combined_target_text,

    "golden_case":
        copy.deepcopy(
            cell23_next_golden_case
        ),

    "task_plan":
        copy.deepcopy(
            cell23_next_task_plan
        ),

    "target_evidence":
        copy.deepcopy(
            cell23_next_target_evidence
        ),

    "model_drafts":
        [],

    "human_reviews":
        [],

    "expected_answer_review":
        None,

    "citation_review":
        None,

    "author_final_review":
        {
            "status":
                "NOT_STARTED",

            "decision":
                None,

            "notes":
                None,
        },

    "workflow_state":
        "QUESTION_DRAFT_PENDING",

    "full_572_generation":
        CELL23_FULL_572_BLOCK_STATUS,
}


cell23_next_workspace_ready = all([
    cell23_next_packet_valid,
    cell23_next_identity_valid,
    isinstance(
        cell23_next_golden_case,
        dict,
    ),
    isinstance(
        cell23_next_task_plan,
        dict,
    ),
    isinstance(
        cell23_next_target_evidence,
        dict,
    ),
    cell23_next_target_text_available,
])


# ============================================================
# 15) Update Human Review Registry
# ============================================================

cell23_human_review_registry = copy.deepcopy(
    cell22_human_review_registry
)


cell23_completed_cases = copy.deepcopy(
    cell23_human_review_registry.get(
        "completed_human_review_cases",
        [],
    )
)


cell23_completed_cases.append(
    copy.deepcopy(
        cell23_completed_human_review_case
    )
)


cell23_human_review_registry[
    "completed_human_review_cases"
] = cell23_completed_cases


cell23_human_review_registry[
    "active_workspace"
] = copy.deepcopy(
    cell23_next_pilot_workspace
)


cell23_human_review_registry[
    "remaining_pilot_count"
] = 20


# ============================================================
# 16) Persistent Registry payload
# ============================================================

cell23_persistent_registry_payload = copy.deepcopy(
    cell22_persisted_registry
)


cell23_persistent_registry_payload[
    "completed_human_review_cases"
] = copy.deepcopy(
    cell23_completed_cases
)


cell23_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell23_next_pilot_workspace
)


cell23_persistent_registry_payload[
    "remaining_pilot_count"
] = 20


cell23_persistent_registry_payload[
    "continuation_state"
] = {
    "next_pilot_id":
        cell23_next_pilot_id,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell23_remaining_pilot_ids
        ),

    "remaining_count":
        20,

    "continuation_state_sha256":
        cell23_continuation_state_sha256,
}


cell23_persistent_registry_payload[
    "author_review"
][
    "cases_ready_for_review"
] = len(
    cell23_completed_cases
)


# ============================================================
# 17) Persist
# ============================================================

cell23_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell23_persistent_registry_payload
    )
)


cell23_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell23_persisted_active_case = (
    cell23_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell23_persistence_valid = all([
    len(
        cell23_persisted_registry.get(
            "completed_human_review_cases",
            [],
        )
    )
    ==
    2,

    cell23_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    20,

    cell23_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    cell23_next_pilot_id,

    cell23_persisted_registry.get(
        "continuation_state",
        {},
    ).get(
        "next_pilot_id"
    )
    ==
    cell23_next_pilot_id,
])


# ============================================================
# 18) Safety
# ============================================================

cell23_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell23_full_572_blocked = (
    cell23_continuation_state[
        "full_572_generation"
    ]
    ==
    CELL23_FULL_572_BLOCK_STATUS
)


# ============================================================
# 19) Final Gate
# ============================================================

cell23_gate = all([
    len(
        cell23_failures
    )
    ==
    0,

    cell23_citation_accept_valid,

    cell23_queue_preconditions_valid,

    cell23_queue_advanced_valid,

    cell23_next_workspace_ready,

    cell23_persistence_valid,

    cell23_base_seed_unchanged,

    cell23_full_572_blocked,
])


# ============================================================
# 20) Diagnostics
# ============================================================

print(
    "=== PILOT COMPLETED + "
    "REGISTRY SYNC + NEXT PILOT ==="
)


print(
    "\n=== COMPLETED PILOT ==="
)

print(
    "Pilot:",
    CELL23_COMPLETED_PILOT_ID,
)

print(
    "Question:",
    cell20_question_text,
)

print(
    "Answer:",
    cell20_expected_answer_text,
)

print(
    "Citation:",
    "ACCEPT",
)

print(
    "Human Review status:",
    cell23_completed_human_review_case[
        "final_status"
    ],
)

print(
    "Author Final Review:",
    "NOT_STARTED",
)


print(
    "\n=== QUEUE ==="
)

print(
    "Before:",
    len(
        cell23_parent_remaining_ids
    ),
)

print(
    "After:",
    len(
        cell23_remaining_pilot_ids
    ),
)

print(
    "Completed Pilot removed:",
    (
        CELL23_COMPLETED_PILOT_ID
        not in
        cell23_remaining_pilot_ids
    ),
)

print(
    "Next Pilot:",
    cell23_next_pilot_id,
)


print(
    "\n=== PERSISTENT REGISTRY ==="
)

print(
    "Path:",
    cell21_registry_path,
)

print(
    "Completed Human Review cases:",
    len(
        cell23_persisted_registry.get(
            "completed_human_review_cases",
            [],
        )
    ),
)

print(
    "Remaining Pilots:",
    cell23_persisted_registry.get(
        "remaining_pilot_count"
    ),
)

print(
    "Active Pilot:",
    cell23_persisted_active_case.get(
        "pilot_case_id"
    ),
)

print(
    "Registry sync valid:",
    cell23_persistence_valid,
)


print(
    "\n=== NEXT PILOT IDENTITY ==="
)

print(
    "Pilot:",
    cell23_next_pilot_id,
)

print(
    "Golden Case:",
    cell23_next_golden_case_id,
)

print(
    "Task Draft:",
    cell23_next_task_draft_id,
)

print(
    "Target Evidence:",
    cell23_next_target_evidence_id,
)

print(
    "Source type:",
    cell23_next_source_type,
)

print(
    "Stratum:",
    cell23_next_stratum,
)

print(
    "Task family:",
    cell23_next_task_family,
)

print(
    "Question archetype:",
    cell23_next_question_archetype,
)


print(
    "\n=== NEXT PILOT SOURCE EVIDENCE ==="
)

print(
    cell23_next_combined_target_text
)


print(
    "\n=== SAFETY ==="
)

print(
    "Base Seed unchanged:",
    cell23_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell23_full_572_blocked,
)

print(
    "Author Notebook rewrite:",
    False,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell23_failures
)


print(
    "\nCODE CELL 23 GATE:",
    (
        "PASS"
        if cell23_gate
        else "FAIL"
    ),
)


assert cell23_gate, (
    "Citation ACCEPT / Pilot completion / Queue advance / "
    "Registry sync / next Pilot preparation failed. "
    "Repair this SAME Code Cell 23 before continuing."
)

=== PILOT COMPLETED + REGISTRY SYNC + NEXT PILOT ===

=== COMPLETED PILOT ===
Pilot: question-pilot:083ad832854722e126c05d0b89d00cb1
Question: سواد مالی در اروپا چقدر مورد توجه قرار گرفته و چند درصد از اروپایی‌ها سواد مالی مناسبی دارند؟
Answer: بر اساس متن، تنها نیمی از اروپایی‌ها، یعنی حدود 50 درصد، سواد مالی مناسبی دارند. سواد مالی در اروپا مورد بررسی گسترده قرار گرفته است؛ در سال 2014 نظرسنجی جهانی سواد مالی یکی از ابزارهای مقایسه سطح سواد مالی کشورهای اروپایی بود و در بهار 2023 نیز کمیسیون اروپا نظرسنجی جامعی در هر 27 کشور عضو اتحادیه اروپا انجام داد. این سنجش جدید به دلیل پوشش کامل کشورهای عضو، اطلاعات قابل‌اتکاتری برای پژوهشگران و سیاست‌گذاران فراهم کرد.
Citation: ACCEPT
Human Review status: HUMAN_REVIEW_COMPLETE_AUTHOR_FINAL_REVIEW_PENDING
Author Final Review: NOT_STARTED

=== QUEUE ===
Before: 21
After: 20
Completed Pilot removed: True
Next Pilot: question-pilot:b801c37bf6fc78b00fcc0c79b368ef7d

=== PERSISTENT REGISTRY ===
Path: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 24 — Question ACCEPT + Expected Answer Draft + Persistent Sync

سؤال این Pilot پس از Human Review پذیرفته شد:

**«داشتن هدف برای پس‌انداز چه تأثیری بر انگیزه بچه‌ها برای به دست آوردن پول و رسیدن به خواسته‌هایشان دارد؟»**

در این مرحله:

- Question Draft ثبت و با تصمیم انسانی `ACCEPT` نهایی می‌شود؛
- Expected Answer فقط از Evidence همین Pilot ساخته می‌شود؛
- Answer برای Human Review متوقف می‌شود؛
- Human Review Registry روی دیسک Sync می‌شود؛
- Citation هنوز ساخته نمی‌شود؛
- Queue همچنان 20 باقی می‌ماند؛
- Full 572 Generation همچنان Blocked است.

</div>

In [24]:
# Code Cell 24 — REPLACEMENT

cell24_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell23_gate") is not True:
    cell24_failures.append(
        "CELL23_NOT_PASS"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell24_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Current Pilot identity
# ============================================================

CELL24_PILOT_ID = (
    cell23_next_pilot_id
)

CELL24_GOLDEN_CASE_ID = (
    cell23_next_golden_case_id
)

CELL24_TASK_DRAFT_ID = (
    cell23_next_task_draft_id
)

CELL24_TARGET_EVIDENCE_ID = (
    cell23_next_target_evidence_id
)

CELL24_FULL_572_BLOCK_STATUS = (
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


cell24_identity_valid = all([
    isinstance(
        CELL24_PILOT_ID,
        str,
    ),

    isinstance(
        CELL24_GOLDEN_CASE_ID,
        str,
    ),

    isinstance(
        CELL24_TASK_DRAFT_ID,
        str,
    ),

    isinstance(
        CELL24_TARGET_EVIDENCE_ID,
        str,
    ),

    cell23_next_pilot_workspace.get(
        "workflow_state"
    )
    ==
    "QUESTION_DRAFT_PENDING",
])


# ============================================================
# 3) Start compact workflow session
# ============================================================

cell24_pilot_bundle = {
    "pilot_case_id":
        CELL24_PILOT_ID,

    "golden_case_id":
        CELL24_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL24_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL24_TARGET_EVIDENCE_ID,
}


cell24_initial_session = (
    cell13_build_session(
        cell24_pilot_bundle
    )
)


cell24_initial_session_valid = (
    cell24_initial_session.get(
        "current_state"
    )
    ==
    "QUESTION_DRAFT_PENDING"
)


# ============================================================
# 4) Accepted Question
# ============================================================

CELL24_QUESTION_TEXT = (
    "داشتن هدف برای پس‌انداز چه تأثیری بر انگیزه بچه‌ها "
    "برای به دست آوردن پول و رسیدن به خواسته‌هایشان دارد؟"
)


cell24_question_text = (
    CELL24_QUESTION_TEXT.strip()
)


cell24_question_text_sha256 = (
    hashlib.sha256(
        cell24_question_text.encode(
            "utf-8"
        )
    ).hexdigest()
)


cell24_question_draft_id = (
    cell12_question_id_function(
        pilot_case_id=
            CELL24_PILOT_ID,

        question_text_sha256=
            cell24_question_text_sha256,
    )
)


# ============================================================
# 5) Persian-safe normalization helpers
# ============================================================

def cell24_fa_surface(
    value,
):

    text = cell15_normalize_text(
        value
    )

    replacements = {
        "ۀ": "ه",
        "ة": "ه",
        "ؤ": "و",
        "ئ": "ی",
        "أ": "ا",
        "إ": "ا",
        "ٱ": "ا",
    }

    for old, new in replacements.items():
        text = text.replace(
            old,
            new,
        )

    text = text.replace(
        "\u200c",
        " ",
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    return text


def cell24_has_any(
    text,
    terms,
):

    return any(
        term in text
        for term
        in terms
    )


cell24_normalized_question = (
    cell24_fa_surface(
        cell24_question_text
    )
)


cell24_normalized_evidence = (
    cell24_fa_surface(
        cell23_next_combined_target_text
    )
)


# ============================================================
# 6) Question grounding
# ============================================================

cell24_question_grounding_checks = {
    "saving":
        all([
            cell24_has_any(
                cell24_normalized_question,
                [
                    "پس انداز",
                    "پس‌انداز",
                ],
            ),

            cell24_has_any(
                cell24_normalized_evidence,
                [
                    "پس انداز",
                    "پس‌انداز",
                ],
            ),
        ]),

    "goal":
        (
            "هدف"
            in
            cell24_normalized_question
            and
            "هدف"
            in
            cell24_normalized_evidence
        ),

    "motivation":
        (
            "انگیز"
            in
            cell24_normalized_question
            and
            "انگیز"
            in
            cell24_normalized_evidence
        ),

    "money":
        (
            "پول"
            in
            cell24_normalized_question
            and
            "پول"
            in
            cell24_normalized_evidence
        ),

    "desired_outcome":
        (
            "خواسته"
            in
            cell24_normalized_question
            and
            "خواسته"
            in
            cell24_normalized_evidence
        ),
}


cell24_question_machine_pass = all(
    cell24_question_grounding_checks.values()
)


# ============================================================
# 7) Question Draft Record
# ============================================================

cell24_question_draft_record = {
    "record_version":
        "question_draft_record_v1",

    "question_draft_id":
        cell24_question_draft_id,

    "pilot_case_id":
        CELL24_PILOT_ID,

    "golden_case_id":
        CELL24_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL24_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL24_TARGET_EVIDENCE_ID,

    "source_type":
        cell23_next_source_type,

    "stratum":
        cell23_next_stratum,

    "task_family":
        cell23_next_task_family,

    "question_archetype":
        cell23_next_question_archetype,

    "question_text":
        cell24_question_text,

    "question_text_sha256":
        cell24_question_text_sha256,

    "source_evidence_policy":
        "CURRENT_VALIDATED_TARGET_EVIDENCE_ONLY",

    "machine_validation_status":
        (
            "PASS_PENDING_HUMAN_ADJUDICATION"
            if cell24_question_machine_pass
            else "FAIL_REQUIRES_REVISION"
        ),

    "human_adjudication_status":
        "ACCEPT",

    "external_knowledge_used":
        False,

    "full_572_generation":
        CELL24_FULL_572_BLOCK_STATUS,
}


# ============================================================
# 8) Register Question
# ============================================================

cell24_question_review_session = (
    cell13_apply_event(
        cell24_initial_session,
        "QUESTION_DRAFT_REGISTERED",
        artifact=
            cell24_question_draft_record,
    )
)


CELL24_HUMAN_QUESTION_NOTE = (
    "Human Reviewer accepted the revised Question wording "
    "as a source-grounded and audience-appropriate formulation."
)


cell24_answer_pending_session = (
    cell13_apply_event(
        cell24_question_review_session,
        "QUESTION_ACCEPTED",
        human_note=
            CELL24_HUMAN_QUESTION_NOTE,
    )
)


cell24_question_accept_valid = (
    cell24_answer_pending_session.get(
        "current_state"
    )
    ==
    "EXPECTED_ANSWER_DRAFT_PENDING"
)


# ============================================================
# 9) Expected Answer Draft
# ============================================================

CELL24_EXPECTED_ANSWER_TEXT = (
    "داشتن یک هدف برای پس‌انداز می‌تواند به بچه‌ها انگیزه بدهد "
    "تا برای رسیدن زودتر به خواسته‌شان پول بیشتری به دست بیاورند "
    "و آن را پس‌انداز کنند. در مثال متن نیز کسی که پولش را "
    "پس‌انداز کرده بود، توانست با پس‌اندازش چیزهایی را که دوست داشت "
    "بخرد، در حالی که دیگری همه پولش را خرج کرده بود."
)


cell24_expected_answer_text = (
    CELL24_EXPECTED_ANSWER_TEXT.strip()
)


cell24_expected_answer_text_sha256 = (
    hashlib.sha256(
        cell24_expected_answer_text.encode(
            "utf-8"
        )
    ).hexdigest()
)


# ============================================================
# 10) Expected Answer grounding
# ============================================================

cell24_normalized_answer = (
    cell24_fa_surface(
        cell24_expected_answer_text
    )
)


cell24_answer_grounding_checks = {
    "saving_goal":
        (
            "هدف"
            in
            cell24_normalized_evidence
            and
            "هدف"
            in
            cell24_normalized_answer
        ),

    "motivation":
        (
            "انگیز"
            in
            cell24_normalized_evidence
            and
            "انگیز"
            in
            cell24_normalized_answer
        ),

    "earn_more_money":
        (
            "پول بیشتر"
            in
            cell24_normalized_evidence
            and
            "پول بیشتر"
            in
            cell24_normalized_answer
        ),

    "reach_goal_sooner":
        (
            "زودتر"
            in
            cell24_normalized_evidence
            and
            "زودتر"
            in
            cell24_normalized_answer
        ),

    "buy_desired_things":
        (
            "چیزهایی"
            in
            cell24_normalized_evidence
            and
            "چیزهایی"
            in
            cell24_normalized_answer
        ),

    "spent_all_money":
        all([
            cell24_has_any(
                cell24_normalized_evidence,
                [
                    "همه پول",
                    "همه پول هایش",
                    "همه پولهایش",
                ],
            ),

            cell24_has_any(
                cell24_normalized_answer,
                [
                    "همه پول",
                    "همه پولش",
                ],
            ),
        ]),
}


cell24_answer_machine_pass = all(
    cell24_answer_grounding_checks.values()
)


# ============================================================
# 11) Expected Answer Draft ID
# ============================================================

cell24_answer_identity_payload = {
    "identity_version":
        "expected_answer_draft_identity_v1",

    "pilot_case_id":
        CELL24_PILOT_ID,

    "question_draft_id":
        cell24_question_draft_id,

    "target_evidence_id":
        CELL24_TARGET_EVIDENCE_ID,

    "expected_answer_text_sha256":
        cell24_expected_answer_text_sha256,
}


cell24_expected_answer_draft_id = (
    "expected-answer-draft:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell24_answer_identity_payload
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


# ============================================================
# 12) Expected Answer Record
# ============================================================

cell24_expected_answer_record = {
    "record_version":
        "expected_answer_draft_record_v1",

    "expected_answer_draft_id":
        cell24_expected_answer_draft_id,

    "pilot_case_id":
        CELL24_PILOT_ID,

    "golden_case_id":
        CELL24_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL24_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL24_TARGET_EVIDENCE_ID,

    "question_draft_id":
        cell24_question_draft_id,

    "question_text":
        cell24_question_text,

    "expected_answer_text":
        cell24_expected_answer_text,

    "expected_answer_text_sha256":
        cell24_expected_answer_text_sha256,

    "source_evidence_policy":
        "CURRENT_VALIDATED_TARGET_EVIDENCE_ONLY",

    "machine_validation_status":
        (
            "PASS_PENDING_HUMAN_ADJUDICATION"
            if cell24_answer_machine_pass
            else "FAIL_REQUIRES_REVISION"
        ),

    "human_adjudication_status":
        "PENDING",

    "external_knowledge_used":
        False,

    "human_context_used_as_evidence":
        False,

    "full_572_generation":
        CELL24_FULL_572_BLOCK_STATUS,
}


# ============================================================
# 13) Register Expected Answer
# ============================================================

cell24_current_session = (
    cell13_apply_event(
        cell24_answer_pending_session,
        "EXPECTED_ANSWER_REGISTERED",
        artifact=
            cell24_expected_answer_record,
    )
)


cell24_answer_registered_valid = all([
    cell24_current_session.get(
        "current_state"
    )
    ==
    "EXPECTED_ANSWER_HUMAN_REVIEW_PENDING",

    isinstance(
        cell24_current_session.get(
            "artifacts",
            {}
        ).get(
            "expected_answer"
        ),
        dict,
    ),

    cell24_current_session.get(
        "queue_advance_performed"
    )
    is False,
])


# ============================================================
# 14) Human Review Case
# ============================================================

cell24_active_human_review_case = {
    "human_review_case_version":
        "human_review_case_v1",

    "pilot_case_id":
        CELL24_PILOT_ID,

    "golden_case_id":
        CELL24_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL24_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL24_TARGET_EVIDENCE_ID,

    "source_evidence":
        copy.deepcopy(
            cell23_next_target_text_parts
        ),

    "model_drafts":
        [
            {
                "revision":
                    1,

                "question_draft_id":
                    cell24_question_draft_id,

                "question_text":
                    cell24_question_text,

                "machine_validation":
                    (
                        "PASS"
                        if cell24_question_machine_pass
                        else "FAIL"
                    ),

                "human_decision":
                    "ACCEPT",
            }
        ],

    "human_reviews":
        [
            {
                "review_stage":
                    "QUESTION",

                "human_decision":
                    "ACCEPT",

                "human_note":
                    CELL24_HUMAN_QUESTION_NOTE,

                "human_context_evidence_status":
                    "NON_EVIDENCE",
            }
        ],

    "final_question_draft_id":
        cell24_question_draft_id,

    "expected_answer_review":
        {
            "expected_answer_draft_id":
                cell24_expected_answer_draft_id,

            "expected_answer_text":
                cell24_expected_answer_text,

            "machine_validation_status":
                cell24_expected_answer_record[
                    "machine_validation_status"
                ],

            "human_decision":
                "PENDING",
        },

    "citation_review":
        None,

    "author_final_review":
        {
            "status":
                "NOT_STARTED",

            "decision":
                None,

            "notes":
                None,
        },

    "final_status":
        "QUESTION_ACCEPTED_EXPECTED_ANSWER_PENDING_REVIEW",
}


# ============================================================
# 15) Update persistent Registry
# ============================================================

cell24_persistent_registry_payload = copy.deepcopy(
    cell23_persisted_registry
)


cell24_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell24_active_human_review_case
)


cell24_persistent_registry_payload[
    "remaining_pilot_count"
] = 20


cell24_persistent_registry_payload[
    "continuation_state"
] = {
    "next_pilot_id":
        CELL24_PILOT_ID,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell23_remaining_pilot_ids
        ),

    "remaining_count":
        20,

    "continuation_state_sha256":
        cell23_continuation_state_sha256,
}


cell24_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell24_persistent_registry_payload
    )
)


# ============================================================
# 16) Read back Registry
# ============================================================

cell24_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell24_persisted_active_case = (
    cell24_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell24_persistence_valid = all([
    cell24_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL24_PILOT_ID,

    cell24_persisted_active_case.get(
        "final_question_draft_id"
    )
    ==
    cell24_question_draft_id,

    cell24_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "expected_answer_draft_id"
    )
    ==
    cell24_expected_answer_draft_id,

    cell24_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "PENDING",

    cell24_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "machine_validation_status"
    )
    ==
    "PASS_PENDING_HUMAN_ADJUDICATION",

    cell24_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    20,
])


# ============================================================
# 17) Safety
# ============================================================

cell24_queue_still_20 = (
    len(
        cell23_remaining_pilot_ids
    )
    ==
    20
)


cell24_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell24_full_572_blocked = (
    cell24_expected_answer_record[
        "full_572_generation"
    ]
    ==
    CELL24_FULL_572_BLOCK_STATUS
)


# ============================================================
# 18) Final Gate
# ============================================================

cell24_gate = all([
    len(
        cell24_failures
    )
    ==
    0,

    cell24_identity_valid,

    cell24_initial_session_valid,

    cell24_question_machine_pass,

    cell24_question_accept_valid,

    cell24_answer_machine_pass,

    cell24_answer_registered_valid,

    cell24_persistence_valid,

    cell24_queue_still_20,

    cell24_base_seed_unchanged,

    cell24_full_572_blocked,
])


# ============================================================
# 19) Diagnostics
# ============================================================

print(
    "=== QUESTION ACCEPTED + EXPECTED ANSWER DRAFT "
    "+ REGISTRY SYNC ==="
)


print(
    "\n=== SOURCE EVIDENCE ==="
)

print(
    cell23_next_combined_target_text
)


print(
    "\n=== ACCEPTED QUESTION ==="
)

print(
    cell24_question_text
)

print(
    "Question Draft ID:",
    cell24_question_draft_id,
)

print(
    "Human Decision:",
    "ACCEPT",
)

print(
    "Question Machine PASS:",
    cell24_question_machine_pass,
)


print(
    "\n=== QUESTION GROUNDING ==="
)

for name, status in (
    cell24_question_grounding_checks.items()
):
    print(
        f"{name}:",
        status,
    )


print(
    "\n=== EXPECTED ANSWER DRAFT ==="
)

print(
    cell24_expected_answer_text
)

print(
    "Expected Answer Draft ID:",
    cell24_expected_answer_draft_id,
)

print(
    "Machine PASS:",
    cell24_answer_machine_pass,
)


print(
    "\n=== ANSWER GROUNDING ==="
)

for name, status in (
    cell24_answer_grounding_checks.items()
):
    print(
        f"{name}:",
        status,
    )


print(
    "\n=== WORKFLOW ==="
)

print(
    "Current State:",
    cell24_current_session.get(
        "current_state"
    ),
)

print(
    "Question Decision:",
    "ACCEPT",
)

print(
    "Expected Answer Decision:",
    "PENDING",
)

print(
    "Citation:",
    None,
)


print(
    "\n=== PERSISTENCE ==="
)

print(
    "Registry:",
    cell21_registry_path,
)

print(
    "Persisted Active Pilot:",
    cell24_persisted_active_case.get(
        "pilot_case_id"
    ),
)

print(
    "Persisted Question Machine Status:",
    (
        cell24_persisted_active_case.get(
            "model_drafts",
            [{}],
        )[-1].get(
            "machine_validation"
        )
    ),
)

print(
    "Persisted Answer Machine Status:",
    cell24_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "machine_validation_status"
    ),
)

print(
    "Persisted Answer Decision:",
    cell24_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    ),
)

print(
    "Persistence valid:",
    cell24_persistence_valid,
)


print(
    "\n=== QUEUE / SAFETY ==="
)

print(
    "Queue still 20:",
    cell24_queue_still_20,
)

print(
    "Base Seed unchanged:",
    cell24_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell24_full_572_blocked,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell24_failures
)


print(
    "\nCODE CELL 24 GATE:",
    (
        "PASS"
        if cell24_gate
        else "FAIL"
    ),
)


assert cell24_gate, (
    "Question ACCEPT / Expected Answer Draft / "
    "Registry Sync failed. Repair this SAME Code Cell 24 "
    "before Expected Answer Human Review."
)

=== QUESTION ACCEPTED + EXPECTED ANSWER DRAFT + REGISTRY SYNC ===

=== SOURCE EVIDENCE ===
یکی از آن‌ها پس‌انداز می‌کند و دیگری همۀ پول‌هایش را خرج می‌کند تا روزی که به فروشگاهی بزرگ می‌روند و خواهر موجود کوچولو با پس‌اندازش چیزهایی می‌خرد که دوست دارد.
5.در راه رسیدن به هدف پس‌اندازی، بچه‌ها انگیزۀ به دست آوردن پول بیشتر برای رسیدن زودتر به خواسته‌شان را دارند و ممکن است خودشان بخواهند دست به کار شوند و پول پس‌انداز کنند.

=== ACCEPTED QUESTION ===
داشتن هدف برای پس‌انداز چه تأثیری بر انگیزه بچه‌ها برای به دست آوردن پول و رسیدن به خواسته‌هایشان دارد؟
Question Draft ID: question-draft:d14395bbe4c2f16bfba208eadab41590
Human Decision: ACCEPT
Question Machine PASS: True

=== QUESTION GROUNDING ===
saving: True
goal: True
motivation: True
money: True
desired_outcome: True

=== EXPECTED ANSWER DRAFT ===
داشتن یک هدف برای پس‌انداز می‌تواند به بچه‌ها انگیزه بدهد تا برای رسیدن زودتر به خواسته‌شان پول بیشتری به دست بیاورند و آن را پس‌انداز کنند. در مثال متن نیز کسی که پولش را پس‌انداز کرده بود،

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 25 — Expected Answer ACCEPT + Citation Draft + Persistent Sync

Expected Answer این Pilot توسط Human Reviewer پذیرفته شد.

در این مرحله:

- تصمیم انسانی `ACCEPT` برای Expected Answer ثبت می‌شود؛
- Citation فقط از Target Evidence اعتبارسنجی‌شده ساخته می‌شود؛
- Document ID، Locator و محدوده‌های متنی از منبع موجود کپی می‌شوند و هیچ Locator جدیدی ساخته نمی‌شود؛
- Citation به‌صورت ماشینی اعتبارسنجی می‌شود؛
- Workflow در وضعیت `CITATION_HUMAN_REVIEW_PENDING` متوقف می‌شود؛
- Registry دائمی Human Review روی دیسک Sync می‌شود.

در این مرحله Pilot هنوز کامل نشده و Queue همچنان 20 باقی می‌ماند.

</div>

In [25]:
# Code Cell 25

cell25_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell24_gate") is not True:
    cell25_failures.append(
        "CELL24_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell24_current_session"
    ),
    dict,
):
    cell25_failures.append(
        "CELL24_CURRENT_SESSION_MISSING"
    )


if (
    globals().get(
        "cell24_current_session",
        {},
    ).get(
        "current_state"
    )
    !=
    "EXPECTED_ANSWER_HUMAN_REVIEW_PENDING"
):
    cell25_failures.append(
        "EXPECTED_ANSWER_NOT_AT_HUMAN_REVIEW_GATE"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell25_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Current Pilot identity
# ============================================================

CELL25_PILOT_ID = (
    cell23_next_pilot_id
)

CELL25_GOLDEN_CASE_ID = (
    cell23_next_golden_case_id
)

CELL25_TASK_DRAFT_ID = (
    cell23_next_task_draft_id
)

CELL25_TARGET_EVIDENCE_ID = (
    cell23_next_target_evidence_id
)

CELL25_QUESTION_DRAFT_ID = (
    cell24_question_draft_id
)

CELL25_EXPECTED_ANSWER_DRAFT_ID = (
    cell24_expected_answer_draft_id
)

CELL25_FULL_572_BLOCK_STATUS = (
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


# ============================================================
# 3) Expected Answer identity validation
# ============================================================

cell25_expected_answer_record = copy.deepcopy(
    cell24_current_session[
        "artifacts"
    ][
        "expected_answer"
    ]
)


cell25_expected_answer_identity_valid = all([
    cell25_expected_answer_record.get(
        "expected_answer_draft_id"
    )
    ==
    CELL25_EXPECTED_ANSWER_DRAFT_ID,

    cell25_expected_answer_record.get(
        "pilot_case_id"
    )
    ==
    CELL25_PILOT_ID,

    cell25_expected_answer_record.get(
        "question_draft_id"
    )
    ==
    CELL25_QUESTION_DRAFT_ID,

    cell25_expected_answer_record.get(
        "target_evidence_id"
    )
    ==
    CELL25_TARGET_EVIDENCE_ID,

    cell25_expected_answer_record.get(
        "machine_validation_status"
    )
    ==
    "PASS_PENDING_HUMAN_ADJUDICATION",
])


# ============================================================
# 4) Human ACCEPT for Expected Answer
# ============================================================

CELL25_HUMAN_ANSWER_NOTE = (
    "Human Reviewer accepted the Expected Answer "
    "after reviewing it against the validated source evidence."
)


cell25_answer_human_review = {
    "review_record_version":
        "human_expected_answer_review_record_v1",

    "pilot_case_id":
        CELL25_PILOT_ID,

    "question_draft_id":
        CELL25_QUESTION_DRAFT_ID,

    "expected_answer_draft_id":
        CELL25_EXPECTED_ANSWER_DRAFT_ID,

    "target_evidence_id":
        CELL25_TARGET_EVIDENCE_ID,

    "human_decision":
        "ACCEPT",

    "human_note":
        CELL25_HUMAN_ANSWER_NOTE,

    "human_context_used_as_evidence":
        False,

    "external_knowledge_used":
        False,
}


# ============================================================
# 5) Apply Expected Answer ACCEPT
# ============================================================

cell25_citation_pending_session = (
    cell13_apply_event(
        cell24_current_session,
        "EXPECTED_ANSWER_ACCEPTED",
        human_note=
            CELL25_HUMAN_ANSWER_NOTE,
    )
)


cell25_answer_accept_valid = all([
    cell25_citation_pending_session.get(
        "current_state"
    )
    ==
    "CITATION_MAPPING_PENDING",

    len(
        cell25_citation_pending_session.get(
            "human_decisions",
            [],
        )
    )
    ==
    2,

    cell25_citation_pending_session[
        "human_decisions"
    ][
        -1
    ].get(
        "event"
    )
    ==
    "EXPECTED_ANSWER_ACCEPTED",

    cell25_citation_pending_session.get(
        "queue_advance_performed"
    )
    is False,
])


# ============================================================
# 6) Read validated Target Evidence parts
# ============================================================

cell25_target_parts = copy.deepcopy(
    cell23_next_target_evidence.get(
        "target_parts",
        [],
    )
)


cell25_target_parts_valid = all([
    isinstance(
        cell25_target_parts,
        list,
    ),

    len(
        cell25_target_parts
    )
    >
    0,
])


# ============================================================
# 7) Build Citation entries
# ============================================================

cell25_citation_entries = []


for index, part in enumerate(
    cell25_target_parts
):

    if not isinstance(
        part,
        dict,
    ):
        continue


    cell25_citation_entries.append({
        "citation_index":
            index,

        "target_evidence_id":
            CELL25_TARGET_EVIDENCE_ID,

        "document_id":
            part.get(
                "document_id"
            ),

        "source_id":
            part.get(
                "source_id"
            ),

        "role":
            part.get(
                "role"
            ),

        "char_start":
            part.get(
                "char_start"
            ),

        "char_end":
            part.get(
                "char_end"
            ),

        "char_count":
            part.get(
                "char_count"
            ),

        "token_count":
            part.get(
                "token_count"
            ),

        "locator":
            copy.deepcopy(
                part.get(
                    "locator",
                    {},
                )
            ),

        "attribution":
            copy.deepcopy(
                part.get(
                    "attribution"
                )
            ),

        "locator_origin":
            "COPIED_FROM_VALIDATED_TARGET_EVIDENCE",
    })


# ============================================================
# 8) Exact projection validation
# ============================================================

cell25_source_projection = [
    {
        "document_id":
            part.get(
                "document_id"
            ),

        "source_id":
            part.get(
                "source_id"
            ),

        "role":
            part.get(
                "role"
            ),

        "char_start":
            part.get(
                "char_start"
            ),

        "char_end":
            part.get(
                "char_end"
            ),

        "locator":
            copy.deepcopy(
                part.get(
                    "locator",
                    {},
                )
            ),
    }

    for part
    in cell25_target_parts

    if isinstance(
        part,
        dict,
    )
]


cell25_citation_projection = [
    {
        "document_id":
            entry.get(
                "document_id"
            ),

        "source_id":
            entry.get(
                "source_id"
            ),

        "role":
            entry.get(
                "role"
            ),

        "char_start":
            entry.get(
                "char_start"
            ),

        "char_end":
            entry.get(
                "char_end"
            ),

        "locator":
            copy.deepcopy(
                entry.get(
                    "locator",
                    {},
                )
            ),
    }

    for entry
    in cell25_citation_entries
]


cell25_no_locator_fabrication = (
    cell25_source_projection
    ==
    cell25_citation_projection
)


# ============================================================
# 9) Validate Citation against displayed Evidence
# ============================================================

cell25_displayed_parts = copy.deepcopy(
    cell23_next_target_text_parts
)


cell25_displayed_by_document = {
    row.get(
        "document_id"
    ):
        row

    for row
    in cell25_displayed_parts

    if isinstance(
        row,
        dict,
    )
}


cell25_citation_validation_rows = []


for entry in cell25_citation_entries:

    document_id = entry.get(
        "document_id"
    )


    displayed = (
        cell25_displayed_by_document.get(
            document_id
        )
    )


    locator = (
        entry.get(
            "locator"
        )
        or
        {}
    )


    canonical_url = locator.get(
        "canonical_url"
    )


    url = locator.get(
        "url"
    )


    row_valid = all([
        isinstance(
            document_id,
            str,
        ),

        isinstance(
            displayed,
            dict,
        ),

        entry.get(
            "char_start"
        )
        ==
        displayed.get(
            "char_start"
        ),

        entry.get(
            "char_end"
        )
        ==
        displayed.get(
            "char_end"
        ),

        entry.get(
            "role"
        )
        ==
        displayed.get(
            "role"
        ),

        (
            canonical_url is None
            or
            (
                isinstance(
                    canonical_url,
                    str,
                )
                and
                canonical_url.startswith(
                    (
                        "http://",
                        "https://",
                    )
                )
            )
        ),

        (
            url is None
            or
            (
                isinstance(
                    url,
                    str,
                )
                and
                url.startswith(
                    (
                        "http://",
                        "https://",
                    )
                )
            )
        ),

        entry.get(
            "locator_origin"
        )
        ==
        "COPIED_FROM_VALIDATED_TARGET_EVIDENCE",
    ])


    cell25_citation_validation_rows.append({
        "citation_index":
            entry.get(
                "citation_index"
            ),

        "document_id":
            document_id,

        "range_match":
            (
                isinstance(
                    displayed,
                    dict,
                )
                and
                entry.get(
                    "char_start"
                )
                ==
                displayed.get(
                    "char_start"
                )
                and
                entry.get(
                    "char_end"
                )
                ==
                displayed.get(
                    "char_end"
                )
            ),

        "role_match":
            (
                isinstance(
                    displayed,
                    dict,
                )
                and
                entry.get(
                    "role"
                )
                ==
                displayed.get(
                    "role"
                )
            ),

        "valid":
            row_valid,
    })


cell25_all_entries_valid = all([
    len(
        cell25_citation_entries
    )
    >
    0,

    len(
        cell25_citation_entries
    )
    ==
    len(
        cell25_displayed_parts
    ),

    all(
        row.get(
            "valid"
        )
        is True

        for row
        in cell25_citation_validation_rows
    ),
])


# ============================================================
# 10) Citation Mapping ID
# ============================================================

cell25_citation_identity_payload = {
    "identity_version":
        "citation_mapping_identity_v1",

    "pilot_case_id":
        CELL25_PILOT_ID,

    "question_draft_id":
        CELL25_QUESTION_DRAFT_ID,

    "expected_answer_draft_id":
        CELL25_EXPECTED_ANSWER_DRAFT_ID,

    "target_evidence_id":
        CELL25_TARGET_EVIDENCE_ID,

    "citation_projection":
        copy.deepcopy(
            cell25_citation_projection
        ),
}


cell25_citation_mapping_id = (
    "citation-mapping:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell25_citation_identity_payload
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


# ============================================================
# 11) Citation Machine Validation
# ============================================================

cell25_citation_machine_validation = {
    "target_parts_present":
        cell25_target_parts_valid,

    "citation_entries_present":
        (
            len(
                cell25_citation_entries
            )
            >
            0
        ),

    "all_target_parts_covered":
        (
            len(
                cell25_citation_entries
            )
            ==
            len(
                cell25_displayed_parts
            )
        ),

    "all_entries_valid":
        cell25_all_entries_valid,

    "locator_fabrication_absent":
        cell25_no_locator_fabrication,

    "target_evidence_identity_match":
        all(
            entry.get(
                "target_evidence_id"
            )
            ==
            CELL25_TARGET_EVIDENCE_ID

            for entry
            in cell25_citation_entries
        ),

    "external_locator_lookup_used":
        False,
}


cell25_citation_machine_pass = all([
    cell25_citation_machine_validation[
        "target_parts_present"
    ],

    cell25_citation_machine_validation[
        "citation_entries_present"
    ],

    cell25_citation_machine_validation[
        "all_target_parts_covered"
    ],

    cell25_citation_machine_validation[
        "all_entries_valid"
    ],

    cell25_citation_machine_validation[
        "locator_fabrication_absent"
    ],

    cell25_citation_machine_validation[
        "target_evidence_identity_match"
    ],
])


# ============================================================
# 12) Citation Mapping Record
# ============================================================

cell25_citation_mapping_record = {
    "record_version":
        "citation_mapping_record_v1",

    "citation_mapping_id":
        cell25_citation_mapping_id,

    "pilot_case_id":
        CELL25_PILOT_ID,

    "golden_case_id":
        CELL25_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL25_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL25_TARGET_EVIDENCE_ID,

    "question_draft_id":
        CELL25_QUESTION_DRAFT_ID,

    "expected_answer_draft_id":
        CELL25_EXPECTED_ANSWER_DRAFT_ID,

    "citation_entries":
        copy.deepcopy(
            cell25_citation_entries
        ),

    "citation_scope":
        "VALIDATED_TARGET_EVIDENCE_ONLY",

    "locator_source":
        "EXISTING_TARGET_EVIDENCE_LOCATORS",

    "external_locator_lookup_used":
        False,

    "machine_validation_status":
        (
            "PASS_PENDING_HUMAN_ADJUDICATION"
            if cell25_citation_machine_pass
            else "FAIL_REQUIRES_REVISION"
        ),

    "human_adjudication_status":
        "PENDING",

    "full_572_generation":
        CELL25_FULL_572_BLOCK_STATUS,
}


# ============================================================
# 13) Register Citation
# ============================================================

cell25_current_session = (
    cell13_apply_event(
        cell25_citation_pending_session,
        "CITATION_MAPPING_REGISTERED",
        artifact=
            cell25_citation_mapping_record,
    )
)


cell25_citation_registered_valid = all([
    cell25_current_session.get(
        "current_state"
    )
    ==
    "CITATION_HUMAN_REVIEW_PENDING",

    isinstance(
        cell25_current_session.get(
            "artifacts",
            {}
        ).get(
            "citation_mapping"
        ),
        dict,
    ),

    cell25_current_session[
        "artifacts"
    ][
        "citation_mapping"
    ].get(
        "citation_mapping_id"
    )
    ==
    cell25_citation_mapping_id,

    cell25_current_session.get(
        "queue_advance_performed"
    )
    is False,
])


# ============================================================
# 14) Update Human Review active case
# ============================================================

cell25_active_human_review_case = copy.deepcopy(
    cell24_active_human_review_case
)


cell25_active_human_review_case[
    "expected_answer_review"
][
    "human_decision"
] = "ACCEPT"


cell25_active_human_review_case[
    "expected_answer_review"
][
    "human_review"
] = copy.deepcopy(
    cell25_answer_human_review
)


cell25_active_human_review_case[
    "citation_review"
] = {
    "citation_mapping_id":
        cell25_citation_mapping_id,

    "citation_entries":
        copy.deepcopy(
            cell25_citation_entries
        ),

    "machine_validation_status":
        cell25_citation_mapping_record[
            "machine_validation_status"
        ],

    "human_decision":
        "PENDING",
}


cell25_active_human_review_case[
    "final_status"
] = (
    "QUESTION_AND_ANSWER_ACCEPTED_"
    "CITATION_PENDING_REVIEW"
)


# ============================================================
# 15) Persistent Registry update
# ============================================================

cell25_persistent_registry_payload = copy.deepcopy(
    cell24_persisted_registry
)


cell25_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell25_active_human_review_case
)


cell25_persistent_registry_payload[
    "remaining_pilot_count"
] = 20


cell25_persistent_registry_payload[
    "continuation_state"
] = {
    "next_pilot_id":
        CELL25_PILOT_ID,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell23_remaining_pilot_ids
        ),

    "remaining_count":
        20,

    "continuation_state_sha256":
        cell23_continuation_state_sha256,
}


cell25_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell25_persistent_registry_payload
    )
)


# ============================================================
# 16) Read back Registry
# ============================================================

cell25_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell25_persisted_active_case = (
    cell25_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell25_persistence_valid = all([
    cell25_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL25_PILOT_ID,

    cell25_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell25_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "citation_mapping_id"
    )
    ==
    cell25_citation_mapping_id,

    cell25_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "PENDING",

    cell25_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    20,
])


# ============================================================
# 17) Safety
# ============================================================

cell25_queue_still_20 = (
    len(
        cell23_remaining_pilot_ids
    )
    ==
    20
)


cell25_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell25_full_572_blocked = (
    cell25_citation_mapping_record[
        "full_572_generation"
    ]
    ==
    CELL25_FULL_572_BLOCK_STATUS
)


# ============================================================
# 18) Final Gate
# ============================================================

cell25_gate = all([
    len(
        cell25_failures
    )
    ==
    0,

    cell25_expected_answer_identity_valid,

    cell25_answer_accept_valid,

    cell25_target_parts_valid,

    cell25_no_locator_fabrication,

    cell25_all_entries_valid,

    cell25_citation_machine_pass,

    cell25_citation_registered_valid,

    cell25_persistence_valid,

    cell25_queue_still_20,

    cell25_base_seed_unchanged,

    cell25_full_572_blocked,
])


# ============================================================
# 19) Diagnostics
# ============================================================

print(
    "=== ANSWER ACCEPTED + CITATION DRAFT "
    "+ REGISTRY SYNC ==="
)


print(
    "\n=== ACCEPTED QUESTION ==="
)

print(
    cell24_question_text
)


print(
    "\n=== ACCEPTED ANSWER ==="
)

print(
    cell24_expected_answer_text
)

print(
    "Human Answer Decision:",
    "ACCEPT",
)


print(
    "\n=== CITATION DRAFT ==="
)

print(
    "Citation Mapping ID:",
    cell25_citation_mapping_id,
)

print(
    "Citation entry count:",
    len(
        cell25_citation_entries
    ),
)


for entry in cell25_citation_entries:

    print(
        "\n--- CITATION ENTRY",
        entry.get(
            "citation_index"
        ),
        "---"
    )

    print(
        "Document:",
        entry.get(
            "document_id"
        ),
    )

    print(
        "Role:",
        entry.get(
            "role"
        ),
    )

    print(
        "Chars:",
        (
            entry.get(
                "char_start"
            ),
            entry.get(
                "char_end"
            ),
        ),
    )

    print(
        "Canonical URL:",
        (
            entry.get(
                "locator",
                {}
            ).get(
                "canonical_url"
            )
        ),
    )

    print(
        "URL:",
        (
            entry.get(
                "locator",
                {}
            ).get(
                "url"
            )
        ),
    )


print(
    "\n=== CITATION MACHINE VALIDATION ==="
)

for name, status in (
    cell25_citation_machine_validation.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "Machine PASS:",
    cell25_citation_machine_pass,
)

print(
    "No locator fabrication:",
    cell25_no_locator_fabrication,
)


print(
    "\n=== WORKFLOW ==="
)

print(
    "Current State:",
    cell25_current_session.get(
        "current_state"
    ),
)

print(
    "Question Decision:",
    "ACCEPT",
)

print(
    "Answer Decision:",
    "ACCEPT",
)

print(
    "Citation Decision:",
    "PENDING",
)


print(
    "\n=== PERSISTENCE ==="
)

print(
    "Registry:",
    cell21_registry_path,
)

print(
    "Persisted Answer Decision:",
    cell25_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    ),
)

print(
    "Persisted Citation Decision:",
    cell25_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "human_decision"
    ),
)

print(
    "Persistence valid:",
    cell25_persistence_valid,
)


print(
    "\n=== QUEUE / SAFETY ==="
)

print(
    "Queue still 20:",
    cell25_queue_still_20,
)

print(
    "Base Seed unchanged:",
    cell25_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell25_full_572_blocked,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell25_failures
)


print(
    "\nCODE CELL 25 GATE:",
    (
        "PASS"
        if cell25_gate
        else "FAIL"
    ),
)


assert cell25_gate, (
    "Expected Answer ACCEPT / Citation Draft / "
    "Registry Sync failed. Repair this SAME Code Cell 25 "
    "before Citation Human Review."
)

=== ANSWER ACCEPTED + CITATION DRAFT + REGISTRY SYNC ===

=== ACCEPTED QUESTION ===
داشتن هدف برای پس‌انداز چه تأثیری بر انگیزه بچه‌ها برای به دست آوردن پول و رسیدن به خواسته‌هایشان دارد؟

=== ACCEPTED ANSWER ===
داشتن یک هدف برای پس‌انداز می‌تواند به بچه‌ها انگیزه بدهد تا برای رسیدن زودتر به خواسته‌شان پول بیشتری به دست بیاورند و آن را پس‌انداز کنند. در مثال متن نیز کسی که پولش را پس‌انداز کرده بود، توانست با پس‌اندازش چیزهایی را که دوست داشت بخرد، در حالی که دیگری همه پولش را خرج کرده بود.
Human Answer Decision: ACCEPT

=== CITATION DRAFT ===
Citation Mapping ID: citation-mapping:6cc191d209ce82b4cf39f79f2336cb6a
Citation entry count: 1

--- CITATION ENTRY 0 ---
Document: website:C185157D
Role: LOCAL_SENTENCE_WINDOW
Chars: (1607, 1942)
Canonical URL: https://www.fintelligence.ir/Blog/Detail/C185157D
URL: https://www.fintelligence.ir/Blog/Detail/C185157D/%D9%BE%D8%B3-%D8%A7%D9%86%D8%AF%D8%A7%D8%B2-%D8%AF%D8%B1%D8%B3%DB%8C-%D8%A8%D8%B1%D8%A7%DB%8C-%D8%A2%DB%8C%D9%86%D8%AF%D9%87

=== CIT

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 26 — Citation ACCEPT، تکمیل Pilot، Queue Advance و آماده‌سازی Pilot بعدی

Citation این Pilot توسط Human Reviewer پذیرفته شد.

در این مرحله:

- Citation با تصمیم `ACCEPT` ثبت می‌شود؛
- Question + Expected Answer + Citation این Pilot به‌عنوان Human Review کامل بسته می‌شوند؛
- پرونده در `completed_human_review_cases` ذخیره می‌شود؛
- Author Final Review همچنان `NOT_STARTED` باقی می‌ماند؛
- Registry دائمی روی دیسک Sync می‌شود؛
- Queue از 20 به 19 کاهش پیدا می‌کند؛
- Pilot بعدی انتخاب و Evidence اصلی آن نمایش داده می‌شود.

Human Review Notebook جداگانه نیازی به بازنویسی ندارد و آخرین Registry JSON را هنگام اجرا می‌خواند.

Full 572 Generation همچنان Blocked است.

</div>

In [26]:
# Code Cell 26

cell26_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell25_gate") is not True:
    cell26_failures.append(
        "CELL25_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell25_current_session"
    ),
    dict,
):
    cell26_failures.append(
        "CELL25_CURRENT_SESSION_MISSING"
    )


if (
    globals().get(
        "cell25_current_session",
        {},
    ).get(
        "current_state"
    )
    !=
    "CITATION_HUMAN_REVIEW_PENDING"
):
    cell26_failures.append(
        "CITATION_NOT_AT_HUMAN_REVIEW_GATE"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell26_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Current Pilot
# ============================================================

CELL26_COMPLETED_PILOT_ID = (
    cell23_next_pilot_id
)

CELL26_FULL_572_BLOCK_STATUS = (
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


# ============================================================
# 3) Human Citation ACCEPT
# ============================================================

CELL26_HUMAN_CITATION_NOTE = (
    "Human Reviewer accepted the Citation Mapping "
    "after reviewing the validated source locator."
)


cell26_citation_human_review = {
    "review_record_version":
        "human_citation_review_record_v1",

    "pilot_case_id":
        CELL26_COMPLETED_PILOT_ID,

    "citation_mapping_id":
        cell25_citation_mapping_id,

    "target_evidence_id":
        cell23_next_target_evidence_id,

    "human_decision":
        "ACCEPT",

    "human_note":
        CELL26_HUMAN_CITATION_NOTE,

    "external_locator_lookup_used":
        False,
}


# ============================================================
# 4) Apply Citation ACCEPT
# ============================================================

cell26_pre_advance_session = (
    cell13_apply_event(
        cell25_current_session,
        "CITATION_ACCEPTED",
        human_note=
            CELL26_HUMAN_CITATION_NOTE,
    )
)


cell26_citation_accept_valid = all([
    cell26_pre_advance_session.get(
        "current_state"
    )
    ==
    "PILOT_COMPLETE_PENDING_QUEUE_ADVANCE",

    cell26_pre_advance_session.get(
        "queue_advance_permitted"
    )
    is True,

    cell26_pre_advance_session.get(
        "queue_advance_performed"
    )
    is False,

    len(
        cell26_pre_advance_session.get(
            "human_decisions",
            [],
        )
    )
    ==
    3,

    cell26_pre_advance_session[
        "human_decisions"
    ][
        -1
    ].get(
        "event"
    )
    ==
    "CITATION_ACCEPTED",
])


# ============================================================
# 5) Finalize Human Review Case
# ============================================================

cell26_completed_human_review_case = (
    copy.deepcopy(
        cell25_active_human_review_case
    )
)


cell26_completed_human_review_case[
    "citation_review"
][
    "human_decision"
] = "ACCEPT"


cell26_completed_human_review_case[
    "citation_review"
][
    "human_review"
] = copy.deepcopy(
    cell26_citation_human_review
)


cell26_completed_human_review_case[
    "author_final_review"
] = {
    "status":
        "NOT_STARTED",

    "decision":
        None,

    "notes":
        None,
}


cell26_completed_human_review_case[
    "final_status"
] = (
    "HUMAN_REVIEW_COMPLETE_"
    "AUTHOR_FINAL_REVIEW_PENDING"
)


cell26_completed_human_review_case_sha256 = (
    cell10_stable_sha(
        cell26_completed_human_review_case
    )
)


# ============================================================
# 6) Queue advancement
# ============================================================

cell26_parent_remaining_ids = copy.deepcopy(
    cell23_continuation_state[
        "remaining_pilot_ids"
    ]
)


cell26_parent_remaining_queue = copy.deepcopy(
    cell23_continuation_state[
        "remaining_pilot_queue"
    ]
)


cell26_queue_preconditions_valid = all([
    len(
        cell26_parent_remaining_ids
    )
    ==
    20,

    len(
        cell26_parent_remaining_queue
    )
    ==
    20,

    cell26_parent_remaining_ids[
        0
    ]
    ==
    CELL26_COMPLETED_PILOT_ID,

    cell26_citation_accept_valid,
])


if cell26_queue_preconditions_valid:

    cell26_remaining_pilot_ids = copy.deepcopy(
        cell26_parent_remaining_ids[
            1:
        ]
    )

    cell26_remaining_pilot_queue = copy.deepcopy(
        cell26_parent_remaining_queue[
            1:
        ]
    )

else:

    cell26_remaining_pilot_ids = []
    cell26_remaining_pilot_queue = []


cell26_next_pilot_id = (
    cell26_remaining_pilot_ids[
        0
    ]

    if cell26_remaining_pilot_ids

    else None
)


cell26_next_pilot_queue_entry = (
    copy.deepcopy(
        cell26_remaining_pilot_queue[
            0
        ]
    )

    if cell26_remaining_pilot_queue

    else None
)


cell26_queue_advanced_valid = all([
    len(
        cell26_remaining_pilot_ids
    )
    ==
    19,

    len(
        cell26_remaining_pilot_queue
    )
    ==
    19,

    CELL26_COMPLETED_PILOT_ID
    not in
    cell26_remaining_pilot_ids,

    isinstance(
        cell26_next_pilot_id,
        str,
    ),

    cell26_next_pilot_id
    !=
    CELL26_COMPLETED_PILOT_ID,
])


# ============================================================
# 7) Completed session
# ============================================================

cell26_completed_session = copy.deepcopy(
    cell26_pre_advance_session
)


cell26_completed_session[
    "current_state"
] = "PILOT_COMPLETE"


cell26_completed_session[
    "queue_advance_permitted"
] = False


cell26_completed_session[
    "queue_advance_performed"
] = True


cell26_completed_session[
    "completion_status"
] = (
    "HUMAN_REVIEW_COMPLETE_"
    "AUTHOR_FINAL_REVIEW_PENDING"
)


# ============================================================
# 8) New authoritative Continuation State
# ============================================================

cell26_continuation_state = {
    "state_version":
        "compact_pilot_continuation_state_v3",

    "authoritative_forward_state":
        True,

    "parent_state_sha256":
        cell23_continuation_state_sha256,

    "just_completed_pilot_id":
        CELL26_COMPLETED_PILOT_ID,

    "just_completed_case_sha256":
        cell26_completed_human_review_case_sha256,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell26_remaining_pilot_ids
        ),

    "remaining_pilot_queue":
        copy.deepcopy(
            cell26_remaining_pilot_queue
        ),

    "remaining_count":
        19,

    "next_pilot_id":
        cell26_next_pilot_id,

    "next_pilot_queue_entry":
        copy.deepcopy(
            cell26_next_pilot_queue_entry
        ),

    "full_572_generation":
        CELL26_FULL_572_BLOCK_STATUS,
}


cell26_continuation_state_sha256 = (
    cell10_stable_sha(
        cell26_continuation_state
    )
)


# ============================================================
# 9) Resolve next Pilot packet
# ============================================================

cell26_pilot_by_id = (
    cell13_seed[
        "final_runtime"
    ][
        "question_drafting_pilot_by_id"
    ]
)


cell26_next_pilot_packet = copy.deepcopy(
    cell26_pilot_by_id.get(
        cell26_next_pilot_id
    )
)


cell26_next_packet_valid = isinstance(
    cell26_next_pilot_packet,
    dict,
)


# ============================================================
# 10) Resolve next Pilot identity chain
# ============================================================

cell26_next_golden_case_id = None
cell26_next_task_draft_id = None
cell26_next_target_evidence_id = None


if cell26_next_packet_valid:

    try:

        cell26_next_golden_case_id = (
            cell13_exact_single_id(
                cell26_next_pilot_packet,
                "golden_case",
            )
        )


        cell26_next_task_draft_id = (
            cell13_exact_single_id(
                cell26_next_pilot_packet,
                "task_draft",
            )
        )


        cell26_next_target_evidence_id = (
            cell13_exact_single_id(
                cell26_next_pilot_packet,
                "target_evidence",
            )
        )


    except Exception as exc:

        cell26_failures.append({
            "reason":
                "NEXT_ID_CHAIN_FAILED",

            "error":
                repr(
                    exc
                ),
        })


cell26_next_identity_valid = all([
    isinstance(
        cell26_next_golden_case_id,
        str,
    ),

    isinstance(
        cell26_next_task_draft_id,
        str,
    ),

    isinstance(
        cell26_next_target_evidence_id,
        str,
    ),
])


# ============================================================
# 11) Resolve support records
# ============================================================

cell26_next_golden_case = None
cell26_next_task_plan = None
cell26_next_target_evidence = None


if cell26_next_identity_valid:

    try:

        cell26_next_golden_case = copy.deepcopy(
            cell13_active_support[
                "golden_case_records_by_id"
            ][
                cell26_next_golden_case_id
            ]
        )


        cell26_next_task_plan = copy.deepcopy(
            cell13_find_unique_mapping_value(
                cell13_active_support.get(
                    "task_draft_plans",
                    {},
                ),
                "task_draft",
                cell26_next_task_draft_id,
            )[
                "value"
            ]
        )


        cell26_next_target_evidence = copy.deepcopy(
            cell13_find_unique_list_record(
                cell13_active_support.get(
                    "target_evidence_records",
                    [],
                ),
                "target_evidence_id",
                cell26_next_target_evidence_id,
            )
        )


    except Exception as exc:

        cell26_failures.append({
            "reason":
                "NEXT_SUPPORT_RESOLUTION_FAILED",

            "error":
                repr(
                    exc
                ),
        })


# ============================================================
# 12) Classification helper
# ============================================================

def cell26_scalar(
    mapping,
    names,
):

    match = cell14_find_first_scalar(
        mapping,
        names,
    )

    if isinstance(
        match,
        dict,
    ):
        return match.get(
            "value"
        )

    return None


cell26_next_source_type = cell26_scalar(
    cell26_next_pilot_packet,
    [
        "source_type",
        "source_kind",
        "source",
    ],
)


cell26_next_stratum = cell26_scalar(
    cell26_next_pilot_packet,
    [
        "stratum",
        "stratum_name",
        "stratum_id",
    ],
)


cell26_next_task_family = cell26_scalar(
    cell26_next_pilot_packet,
    [
        "task_family",
    ],
)


cell26_next_question_archetype = cell26_scalar(
    cell26_next_pilot_packet,
    [
        "question_archetype",
    ],
)


# ============================================================
# 13) Extract next Evidence text
# ============================================================

cell26_raw_target_parts = []


if cell26_next_packet_valid:

    packet_parts = (
        cell26_next_pilot_packet.get(
            "target_text_parts"
        )
    )

    if isinstance(
        packet_parts,
        list,
    ):
        cell26_raw_target_parts = copy.deepcopy(
            packet_parts
        )


if (
    not cell26_raw_target_parts
    and
    isinstance(
        cell26_next_target_evidence,
        dict,
    )
):

    evidence_parts = (
        cell26_next_target_evidence.get(
            "target_parts"
        )
    )

    if isinstance(
        evidence_parts,
        list,
    ):
        cell26_raw_target_parts = copy.deepcopy(
            evidence_parts
        )


cell26_next_target_text_parts = []


for index, part in enumerate(
    cell26_raw_target_parts
):

    if not isinstance(
        part,
        dict,
    ):
        continue


    text_value = None


    for field_name in (
        "text",
        "target_text",
        "content",
        "evidence_text",
        "text_content",
    ):

        candidate = part.get(
            field_name
        )

        if (
            isinstance(
                candidate,
                str,
            )
            and
            candidate.strip()
        ):

            text_value = candidate.strip()
            break


    if text_value is None:
        continue


    cell26_next_target_text_parts.append({
        "part_index":
            index,

        "document_id":
            part.get(
                "document_id"
            ),

        "role":
            part.get(
                "role"
            ),

        "char_start":
            part.get(
                "char_start"
            ),

        "char_end":
            part.get(
                "char_end"
            ),

        "text":
            text_value,

        "text_sha256":
            hashlib.sha256(
                text_value.encode(
                    "utf-8"
                )
            ).hexdigest(),
    })


cell26_next_combined_target_text = (
    "\n\n".join(
        row[
            "text"
        ]

        for row
        in cell26_next_target_text_parts
    )
)


cell26_next_target_text_available = bool(
    cell26_next_combined_target_text.strip()
)


# ============================================================
# 14) Build next Pilot workspace
# ============================================================

cell26_next_pilot_workspace = {
    "workspace_version":
        "compact_human_review_workspace_v1",

    "status":
        "READY_FOR_QUESTION_DRAFT",

    "pilot_case_id":
        cell26_next_pilot_id,

    "golden_case_id":
        cell26_next_golden_case_id,

    "task_draft_id":
        cell26_next_task_draft_id,

    "target_evidence_id":
        cell26_next_target_evidence_id,

    "source_type":
        cell26_next_source_type,

    "stratum":
        cell26_next_stratum,

    "task_family":
        cell26_next_task_family,

    "question_archetype":
        cell26_next_question_archetype,

    "source_evidence":
        copy.deepcopy(
            cell26_next_target_text_parts
        ),

    "combined_target_text":
        cell26_next_combined_target_text,

    "golden_case":
        copy.deepcopy(
            cell26_next_golden_case
        ),

    "task_plan":
        copy.deepcopy(
            cell26_next_task_plan
        ),

    "target_evidence":
        copy.deepcopy(
            cell26_next_target_evidence
        ),

    "model_drafts":
        [],

    "human_reviews":
        [],

    "expected_answer_review":
        None,

    "citation_review":
        None,

    "author_final_review":
        {
            "status":
                "NOT_STARTED",

            "decision":
                None,

            "notes":
                None,
        },

    "workflow_state":
        "QUESTION_DRAFT_PENDING",

    "full_572_generation":
        CELL26_FULL_572_BLOCK_STATUS,
}


cell26_next_workspace_ready = all([
    cell26_next_packet_valid,

    cell26_next_identity_valid,

    isinstance(
        cell26_next_golden_case,
        dict,
    ),

    isinstance(
        cell26_next_task_plan,
        dict,
    ),

    isinstance(
        cell26_next_target_evidence,
        dict,
    ),

    cell26_next_target_text_available,
])


# ============================================================
# 15) Update persistent Registry
# ============================================================

cell26_completed_cases = copy.deepcopy(
    cell25_persisted_registry.get(
        "completed_human_review_cases",
        [],
    )
)


cell26_completed_cases.append(
    copy.deepcopy(
        cell26_completed_human_review_case
    )
)


cell26_persistent_registry_payload = copy.deepcopy(
    cell25_persisted_registry
)


cell26_persistent_registry_payload[
    "completed_human_review_cases"
] = copy.deepcopy(
    cell26_completed_cases
)


cell26_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell26_next_pilot_workspace
)


cell26_persistent_registry_payload[
    "remaining_pilot_count"
] = 19


cell26_persistent_registry_payload[
    "continuation_state"
] = {
    "next_pilot_id":
        cell26_next_pilot_id,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell26_remaining_pilot_ids
        ),

    "remaining_count":
        19,

    "continuation_state_sha256":
        cell26_continuation_state_sha256,
}


cell26_author_review = (
    cell26_persistent_registry_payload.get(
        "author_review"
    )
)


if not isinstance(
    cell26_author_review,
    dict,
):

    cell26_author_review = {
        "status":
            "NOT_STARTED",

        "reviewer":
            "AUTHOR",

        "cases_ready_for_review":
            0,

        "notes":
            None,
    }


cell26_author_review[
    "cases_ready_for_review"
] = len(
    cell26_completed_cases
)


cell26_persistent_registry_payload[
    "author_review"
] = cell26_author_review


# ============================================================
# 16) Sync Registry to disk
# ============================================================

cell26_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell26_persistent_registry_payload
    )
)


cell26_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell26_persisted_active_case = (
    cell26_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell26_persistence_valid = all([
    len(
        cell26_persisted_registry.get(
            "completed_human_review_cases",
            [],
        )
    )
    ==
    len(
        cell26_completed_cases
    ),

    cell26_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    19,

    cell26_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    cell26_next_pilot_id,

    cell26_persisted_registry.get(
        "continuation_state",
        {},
    ).get(
        "next_pilot_id"
    )
    ==
    cell26_next_pilot_id,

    cell26_persisted_registry.get(
        "author_review",
        {},
    ).get(
        "cases_ready_for_review"
    )
    ==
    len(
        cell26_completed_cases
    ),
])


# ============================================================
# 17) Safety
# ============================================================

cell26_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell26_full_572_blocked = (
    cell26_continuation_state[
        "full_572_generation"
    ]
    ==
    CELL26_FULL_572_BLOCK_STATUS
)


# ============================================================
# 18) Final Gate
# ============================================================

cell26_gate = all([
    len(
        cell26_failures
    )
    ==
    0,

    cell26_citation_accept_valid,

    cell26_queue_preconditions_valid,

    cell26_queue_advanced_valid,

    cell26_next_workspace_ready,

    cell26_persistence_valid,

    cell26_base_seed_unchanged,

    cell26_full_572_blocked,
])


# ============================================================
# 19) Diagnostics
# ============================================================

print(
    "=== PILOT COMPLETED + REGISTRY SYNC "
    "+ NEXT PILOT ==="
)


print(
    "\n=== COMPLETED PILOT ==="
)

print(
    "Pilot:",
    CELL26_COMPLETED_PILOT_ID,
)

print(
    "Question:",
    cell24_question_text,
)

print(
    "Answer:",
    cell24_expected_answer_text,
)

print(
    "Citation:",
    "ACCEPT",
)

print(
    "Human Review status:",
    cell26_completed_human_review_case[
        "final_status"
    ],
)

print(
    "Author Final Review:",
    "NOT_STARTED",
)


print(
    "\n=== QUEUE ==="
)

print(
    "Before:",
    len(
        cell26_parent_remaining_ids
    ),
)

print(
    "After:",
    len(
        cell26_remaining_pilot_ids
    ),
)

print(
    "Completed Pilot removed:",
    (
        CELL26_COMPLETED_PILOT_ID
        not in
        cell26_remaining_pilot_ids
    ),
)

print(
    "Next Pilot:",
    cell26_next_pilot_id,
)


print(
    "\n=== PERSISTENT REGISTRY ==="
)

print(
    "Path:",
    cell21_registry_path,
)

print(
    "Completed Human Review cases:",
    len(
        cell26_persisted_registry.get(
            "completed_human_review_cases",
            [],
        )
    ),
)

print(
    "Cases ready for Author Review:",
    cell26_persisted_registry.get(
        "author_review",
        {},
    ).get(
        "cases_ready_for_review"
    ),
)

print(
    "Remaining Pilots:",
    cell26_persisted_registry.get(
        "remaining_pilot_count"
    ),
)

print(
    "Active Pilot:",
    cell26_persisted_active_case.get(
        "pilot_case_id"
    ),
)

print(
    "Persistence valid:",
    cell26_persistence_valid,
)


print(
    "\n=== NEXT PILOT IDENTITY ==="
)

print(
    "Pilot:",
    cell26_next_pilot_id,
)

print(
    "Golden Case:",
    cell26_next_golden_case_id,
)

print(
    "Task Draft:",
    cell26_next_task_draft_id,
)

print(
    "Target Evidence:",
    cell26_next_target_evidence_id,
)

print(
    "Source type:",
    cell26_next_source_type,
)

print(
    "Stratum:",
    cell26_next_stratum,
)

print(
    "Task family:",
    cell26_next_task_family,
)

print(
    "Question archetype:",
    cell26_next_question_archetype,
)


print(
    "\n=== NEXT PILOT SOURCE EVIDENCE ==="
)

print(
    cell26_next_combined_target_text
)


print(
    "\n=== SAFETY ==="
)

print(
    "Base Seed unchanged:",
    cell26_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell26_full_572_blocked,
)

print(
    "Author Notebook rewrite:",
    False,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell26_failures
)


print(
    "\nCODE CELL 26 GATE:",
    (
        "PASS"
        if cell26_gate
        else "FAIL"
    ),
)


assert cell26_gate, (
    "Citation ACCEPT / Pilot completion / Queue advance / "
    "Registry sync / next Pilot preparation failed. "
    "Repair this SAME Code Cell 26 before continuing."
)

=== PILOT COMPLETED + REGISTRY SYNC + NEXT PILOT ===

=== COMPLETED PILOT ===
Pilot: question-pilot:b801c37bf6fc78b00fcc0c79b368ef7d
Question: داشتن هدف برای پس‌انداز چه تأثیری بر انگیزه بچه‌ها برای به دست آوردن پول و رسیدن به خواسته‌هایشان دارد؟
Answer: داشتن یک هدف برای پس‌انداز می‌تواند به بچه‌ها انگیزه بدهد تا برای رسیدن زودتر به خواسته‌شان پول بیشتری به دست بیاورند و آن را پس‌انداز کنند. در مثال متن نیز کسی که پولش را پس‌انداز کرده بود، توانست با پس‌اندازش چیزهایی را که دوست داشت بخرد، در حالی که دیگری همه پولش را خرج کرده بود.
Citation: ACCEPT
Human Review status: HUMAN_REVIEW_COMPLETE_AUTHOR_FINAL_REVIEW_PENDING
Author Final Review: NOT_STARTED

=== QUEUE ===
Before: 20
After: 19
Completed Pilot removed: True
Next Pilot: question-pilot:909b695797935246c92c7a66ebe0f773

=== PERSISTENT REGISTRY ===
Path: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/data/phase2/human_review/human_review_registry.json
Completed Human Review cases: 3
Cases ready for Autho

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 27 — Question ACCEPT + Expected Answer Draft + Persistent Sync

سؤال این Pilot پس از Human Review پذیرفته شد:

**«در تمرین ۸، برای بررسی هزینه‌های ماهانه افراد چه مراحلی باید انجام شود؟»**

در این مرحله:

- Question Draft ثبت و با تصمیم انسانی `ACCEPT` ذخیره می‌شود؛
- Expected Answer فقط از دستور صریح تمرین ساخته می‌شود؛
- ترتیب اصلی مراحل تمرین حفظ می‌شود؛
- هیچ توضیح یا توصیه‌ای خارج از متن کتاب اضافه نمی‌شود؛
- Answer برای Human Review متوقف می‌شود؛
- Registry دائمی روی دیسک Sync می‌شود؛
- Citation هنوز ساخته نمی‌شود؛
- Queue همچنان 19 باقی می‌ماند.

</div>

In [27]:
# Code Cell 27

cell27_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell26_gate") is not True:
    cell27_failures.append(
        "CELL26_NOT_PASS"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell27_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Current Pilot identity
# ============================================================

CELL27_PILOT_ID = (
    cell26_next_pilot_id
)

CELL27_GOLDEN_CASE_ID = (
    cell26_next_golden_case_id
)

CELL27_TASK_DRAFT_ID = (
    cell26_next_task_draft_id
)

CELL27_TARGET_EVIDENCE_ID = (
    cell26_next_target_evidence_id
)

CELL27_FULL_572_BLOCK_STATUS = (
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


cell27_identity_valid = all([
    isinstance(
        CELL27_PILOT_ID,
        str,
    ),

    isinstance(
        CELL27_GOLDEN_CASE_ID,
        str,
    ),

    isinstance(
        CELL27_TASK_DRAFT_ID,
        str,
    ),

    isinstance(
        CELL27_TARGET_EVIDENCE_ID,
        str,
    ),

    cell26_next_pilot_workspace.get(
        "workflow_state"
    )
    ==
    "QUESTION_DRAFT_PENDING",
])


# ============================================================
# 3) Start workflow session
# ============================================================

cell27_pilot_bundle = {
    "pilot_case_id":
        CELL27_PILOT_ID,

    "golden_case_id":
        CELL27_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL27_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL27_TARGET_EVIDENCE_ID,
}


cell27_initial_session = (
    cell13_build_session(
        cell27_pilot_bundle
    )
)


cell27_initial_session_valid = (
    cell27_initial_session.get(
        "current_state"
    )
    ==
    "QUESTION_DRAFT_PENDING"
)


# ============================================================
# 4) Persian-safe normalization
# ============================================================

def cell27_fa_surface(
    value,
):

    text = cell15_normalize_text(
        value
    )

    replacements = {
        "ۀ": "ه",
        "ة": "ه",
        "ؤ": "و",
        "ئ": "ی",
        "أ": "ا",
        "إ": "ا",
        "ٱ": "ا",
    }

    for old, new in replacements.items():
        text = text.replace(
            old,
            new,
        )

    text = text.replace(
        "\u200c",
        " ",
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    return text


# ============================================================
# 5) Accepted Question
# ============================================================

CELL27_QUESTION_TEXT = (
    "در تمرین ۸، برای بررسی هزینه‌های ماهانه افراد "
    "چه مراحلی باید انجام شود؟"
)


cell27_question_text = (
    CELL27_QUESTION_TEXT.strip()
)


cell27_question_text_sha256 = (
    hashlib.sha256(
        cell27_question_text.encode(
            "utf-8"
        )
    ).hexdigest()
)


cell27_question_draft_id = (
    cell12_question_id_function(
        pilot_case_id=
            CELL27_PILOT_ID,

        question_text_sha256=
            cell27_question_text_sha256,
    )
)


# ============================================================
# 6) Question grounding
# ============================================================

cell27_normalized_question = (
    cell27_fa_surface(
        cell27_question_text
    )
)


cell27_normalized_evidence = (
    cell27_fa_surface(
        cell26_next_combined_target_text
    )
)


cell27_question_grounding_checks = {
    "exercise":
        (
            "تمرین"
            in
            cell27_normalized_question
            and
            "تمرین"
            in
            cell27_normalized_evidence
        ),

    "monthly_expense":
        (
            "هزینه های ماهانه"
            in
            cell27_normalized_question
            and
            "هزینه ماهانه"
            in
            cell27_normalized_evidence
        ),

    "procedure_requested":
        (
            "چه مراحلی"
            in
            cell27_normalized_question
        ),
}


cell27_question_machine_pass = all(
    cell27_question_grounding_checks.values()
)


# ============================================================
# 7) Question Draft Record
# ============================================================

cell27_question_draft_record = {
    "record_version":
        "question_draft_record_v1",

    "question_draft_id":
        cell27_question_draft_id,

    "pilot_case_id":
        CELL27_PILOT_ID,

    "golden_case_id":
        CELL27_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL27_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL27_TARGET_EVIDENCE_ID,

    "source_type":
        cell26_next_source_type,

    "stratum":
        cell26_next_stratum,

    "task_family":
        cell26_next_task_family,

    "question_archetype":
        cell26_next_question_archetype,

    "question_text":
        cell27_question_text,

    "question_text_sha256":
        cell27_question_text_sha256,

    "source_evidence_policy":
        "CURRENT_VALIDATED_TARGET_EVIDENCE_ONLY",

    "machine_validation_status":
        (
            "PASS_PENDING_HUMAN_ADJUDICATION"
            if cell27_question_machine_pass
            else "FAIL_REQUIRES_REVISION"
        ),

    "human_adjudication_status":
        "ACCEPT",

    "external_knowledge_used":
        False,

    "full_572_generation":
        CELL27_FULL_572_BLOCK_STATUS,
}


# ============================================================
# 8) Register Question + Human ACCEPT
# ============================================================

cell27_question_review_session = (
    cell13_apply_event(
        cell27_initial_session,
        "QUESTION_DRAFT_REGISTERED",
        artifact=
            cell27_question_draft_record,
    )
)


CELL27_HUMAN_QUESTION_NOTE = (
    "Human Reviewer accepted the Question as a direct "
    "structure-aware retrieval question grounded in Exercise 8."
)


cell27_answer_pending_session = (
    cell13_apply_event(
        cell27_question_review_session,
        "QUESTION_ACCEPTED",
        human_note=
            CELL27_HUMAN_QUESTION_NOTE,
    )
)


cell27_question_accept_valid = (
    cell27_answer_pending_session.get(
        "current_state"
    )
    ==
    "EXPECTED_ANSWER_DRAFT_PENDING"
)


# ============================================================
# 9) Expected Answer Draft
# ============================================================

CELL27_EXPECTED_ANSWER_TEXT = (
    "باید از هر مقطع سنی دو نفر انتخاب شوند، "
    "درباره میزان هزینه ماهانه آن‌ها مصاحبه شود، "
    "به آن‌ها اطمینان داده شود که اطلاعاتشان محرمانه خواهد ماند "
    "و در پایان نتیجه مصاحبه‌ها تحلیل شود."
)


cell27_expected_answer_text = (
    CELL27_EXPECTED_ANSWER_TEXT.strip()
)


cell27_expected_answer_text_sha256 = (
    hashlib.sha256(
        cell27_expected_answer_text.encode(
            "utf-8"
        )
    ).hexdigest()
)


# ============================================================
# 10) Answer grounding
# ============================================================

cell27_normalized_answer = (
    cell27_fa_surface(
        cell27_expected_answer_text
    )
)


cell27_answer_grounding_checks = {
    "two_people_each_age_group":
        all([
            "هر مقطع سنی دو نفر"
            in
            cell27_normalized_evidence,

            "هر مقطع سنی دو نفر"
            in
            cell27_normalized_answer,
        ]),

    "monthly_expense_interview":
        all([
            "هزینه ماهانه"
            in
            cell27_normalized_evidence,

            "هزینه ماهانه"
            in
            cell27_normalized_answer,

            "مصاحبه"
            in
            cell27_normalized_evidence,

            "مصاحبه"
            in
            cell27_normalized_answer,
        ]),

    "confidentiality":
        (
            "محرمانه"
            in
            cell27_normalized_evidence
            and
            "محرمانه"
            in
            cell27_normalized_answer
        ),

    "analysis":
        (
            "تحلیل"
            in
            cell27_normalized_evidence
            and
            "تحلیل"
            in
            cell27_normalized_answer
        ),
}


cell27_answer_machine_pass = all(
    cell27_answer_grounding_checks.values()
)


# ============================================================
# 11) Expected Answer Draft ID
# ============================================================

cell27_answer_identity_payload = {
    "identity_version":
        "expected_answer_draft_identity_v1",

    "pilot_case_id":
        CELL27_PILOT_ID,

    "question_draft_id":
        cell27_question_draft_id,

    "target_evidence_id":
        CELL27_TARGET_EVIDENCE_ID,

    "expected_answer_text_sha256":
        cell27_expected_answer_text_sha256,
}


cell27_expected_answer_draft_id = (
    "expected-answer-draft:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell27_answer_identity_payload
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


# ============================================================
# 12) Expected Answer Record
# ============================================================

cell27_expected_answer_record = {
    "record_version":
        "expected_answer_draft_record_v1",

    "expected_answer_draft_id":
        cell27_expected_answer_draft_id,

    "pilot_case_id":
        CELL27_PILOT_ID,

    "golden_case_id":
        CELL27_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL27_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL27_TARGET_EVIDENCE_ID,

    "question_draft_id":
        cell27_question_draft_id,

    "question_text":
        cell27_question_text,

    "expected_answer_text":
        cell27_expected_answer_text,

    "expected_answer_text_sha256":
        cell27_expected_answer_text_sha256,

    "source_evidence_policy":
        "CURRENT_VALIDATED_TARGET_EVIDENCE_ONLY",

    "machine_validation_status":
        (
            "PASS_PENDING_HUMAN_ADJUDICATION"
            if cell27_answer_machine_pass
            else "FAIL_REQUIRES_REVISION"
        ),

    "human_adjudication_status":
        "PENDING",

    "external_knowledge_used":
        False,

    "human_context_used_as_evidence":
        False,

    "full_572_generation":
        CELL27_FULL_572_BLOCK_STATUS,
}


# ============================================================
# 13) Register Expected Answer
# ============================================================

cell27_current_session = (
    cell13_apply_event(
        cell27_answer_pending_session,
        "EXPECTED_ANSWER_REGISTERED",
        artifact=
            cell27_expected_answer_record,
    )
)


cell27_answer_registered_valid = all([
    cell27_current_session.get(
        "current_state"
    )
    ==
    "EXPECTED_ANSWER_HUMAN_REVIEW_PENDING",

    isinstance(
        cell27_current_session.get(
            "artifacts",
            {}
        ).get(
            "expected_answer"
        ),
        dict,
    ),

    cell27_current_session.get(
        "queue_advance_performed"
    )
    is False,
])


# ============================================================
# 14) Human Review Case
# ============================================================

cell27_active_human_review_case = {
    "human_review_case_version":
        "human_review_case_v1",

    "pilot_case_id":
        CELL27_PILOT_ID,

    "golden_case_id":
        CELL27_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL27_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL27_TARGET_EVIDENCE_ID,

    "source_evidence":
        copy.deepcopy(
            cell26_next_target_text_parts
        ),

    "model_drafts":
        [
            {
                "revision":
                    1,

                "question_draft_id":
                    cell27_question_draft_id,

                "question_text":
                    cell27_question_text,

                "machine_validation":
                    (
                        "PASS"
                        if cell27_question_machine_pass
                        else "FAIL"
                    ),

                "human_decision":
                    "ACCEPT",
            }
        ],

    "human_reviews":
        [
            {
                "review_stage":
                    "QUESTION",

                "human_decision":
                    "ACCEPT",

                "human_note":
                    CELL27_HUMAN_QUESTION_NOTE,

                "human_context_evidence_status":
                    "NON_EVIDENCE",
            }
        ],

    "final_question_draft_id":
        cell27_question_draft_id,

    "expected_answer_review":
        {
            "expected_answer_draft_id":
                cell27_expected_answer_draft_id,

            "expected_answer_text":
                cell27_expected_answer_text,

            "machine_validation_status":
                cell27_expected_answer_record[
                    "machine_validation_status"
                ],

            "human_decision":
                "PENDING",
        },

    "citation_review":
        None,

    "author_final_review":
        {
            "status":
                "NOT_STARTED",

            "decision":
                None,

            "notes":
                None,
        },

    "final_status":
        "QUESTION_ACCEPTED_EXPECTED_ANSWER_PENDING_REVIEW",
}


# ============================================================
# 15) Persistent Registry update
# ============================================================

cell27_persistent_registry_payload = copy.deepcopy(
    cell26_persisted_registry
)


cell27_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell27_active_human_review_case
)


cell27_persistent_registry_payload[
    "remaining_pilot_count"
] = 19


cell27_persistent_registry_payload[
    "continuation_state"
] = {
    "next_pilot_id":
        CELL27_PILOT_ID,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell26_remaining_pilot_ids
        ),

    "remaining_count":
        19,

    "continuation_state_sha256":
        cell26_continuation_state_sha256,
}


cell27_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell27_persistent_registry_payload
    )
)


# ============================================================
# 16) Read back Registry
# ============================================================

cell27_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell27_persisted_active_case = (
    cell27_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell27_persistence_valid = all([
    cell27_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL27_PILOT_ID,

    cell27_persisted_active_case.get(
        "final_question_draft_id"
    )
    ==
    cell27_question_draft_id,

    cell27_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "expected_answer_draft_id"
    )
    ==
    cell27_expected_answer_draft_id,

    cell27_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "PENDING",

    cell27_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    19,
])


# ============================================================
# 17) Safety
# ============================================================

cell27_queue_still_19 = (
    len(
        cell26_remaining_pilot_ids
    )
    ==
    19
)


cell27_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell27_full_572_blocked = (
    cell27_expected_answer_record[
        "full_572_generation"
    ]
    ==
    CELL27_FULL_572_BLOCK_STATUS
)


# ============================================================
# 18) Final Gate
# ============================================================

cell27_gate = all([
    len(
        cell27_failures
    )
    ==
    0,

    cell27_identity_valid,

    cell27_initial_session_valid,

    cell27_question_machine_pass,

    cell27_question_accept_valid,

    cell27_answer_machine_pass,

    cell27_answer_registered_valid,

    cell27_persistence_valid,

    cell27_queue_still_19,

    cell27_base_seed_unchanged,

    cell27_full_572_blocked,
])


# ============================================================
# 19) Diagnostics
# ============================================================

print(
    "=== QUESTION ACCEPTED + EXPECTED ANSWER DRAFT "
    "+ REGISTRY SYNC ==="
)


print(
    "\n=== SOURCE EVIDENCE ==="
)

print(
    cell26_next_combined_target_text
)


print(
    "\n=== ACCEPTED QUESTION ==="
)

print(
    cell27_question_text
)

print(
    "Question Draft ID:",
    cell27_question_draft_id,
)

print(
    "Question Machine PASS:",
    cell27_question_machine_pass,
)


print(
    "\n=== EXPECTED ANSWER DRAFT ==="
)

print(
    cell27_expected_answer_text
)

print(
    "Expected Answer Draft ID:",
    cell27_expected_answer_draft_id,
)

print(
    "Machine PASS:",
    cell27_answer_machine_pass,
)


print(
    "\n=== ANSWER GROUNDING ==="
)

for name, status in (
    cell27_answer_grounding_checks.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== WORKFLOW ==="
)

print(
    "Current State:",
    cell27_current_session.get(
        "current_state"
    ),
)

print(
    "Question Decision:",
    "ACCEPT",
)

print(
    "Expected Answer Decision:",
    "PENDING",
)

print(
    "Citation:",
    None,
)


print(
    "\n=== PERSISTENCE ==="
)

print(
    "Registry:",
    cell21_registry_path,
)

print(
    "Persisted Active Pilot:",
    cell27_persisted_active_case.get(
        "pilot_case_id"
    ),
)

print(
    "Persisted Answer Decision:",
    cell27_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    ),
)

print(
    "Persistence valid:",
    cell27_persistence_valid,
)


print(
    "\n=== QUEUE / SAFETY ==="
)

print(
    "Queue still 19:",
    cell27_queue_still_19,
)

print(
    "Base Seed unchanged:",
    cell27_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell27_full_572_blocked,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell27_failures
)


print(
    "\nCODE CELL 27 GATE:",
    (
        "PASS"
        if cell27_gate
        else "FAIL"
    ),
)


assert cell27_gate, (
    "Question ACCEPT / Expected Answer Draft / Registry Sync "
    "failed. Repair this SAME Code Cell 27 before "
    "Expected Answer Human Review."
)

=== QUESTION ACCEPTED + EXPECTED ANSWER DRAFT + REGISTRY SYNC ===

=== SOURCE EVIDENCE ===
تمرین 8

هر مقطع سنی دو نفر را انتخاب کنید و درباره میزان هزینه ماهانه آنها مصاحبه کنید. به آن‌ها اطمینان دهید که این اطلاعات محرمانه خواهد بود. نتیجه را تحلیل کنید.

=== ACCEPTED QUESTION ===
در تمرین ۸، برای بررسی هزینه‌های ماهانه افراد چه مراحلی باید انجام شود؟
Question Draft ID: question-draft:147b34f2d29c31c873ad9591cb9ca3e4
Question Machine PASS: True

=== EXPECTED ANSWER DRAFT ===
باید از هر مقطع سنی دو نفر انتخاب شوند، درباره میزان هزینه ماهانه آن‌ها مصاحبه شود، به آن‌ها اطمینان داده شود که اطلاعاتشان محرمانه خواهد ماند و در پایان نتیجه مصاحبه‌ها تحلیل شود.
Expected Answer Draft ID: expected-answer-draft:898d20ffa27dcb9feda6af40cf1aac2a
Machine PASS: True

=== ANSWER GROUNDING ===
two_people_each_age_group: True
monthly_expense_interview: True
confidentiality: True
analysis: True

=== WORKFLOW ===
Current State: EXPECTED_ANSWER_HUMAN_REVIEW_PENDING
Question Decision: ACCEPT
Expected Answe

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 28 — Expected Answer ACCEPT + Citation Draft + Persistent Sync

Expected Answer این Pilot توسط Human Reviewer پذیرفته شد.

در این مرحله:

- تصمیم انسانی `ACCEPT` برای Expected Answer ثبت می‌شود؛
- Citation فقط از Target Evidence اعتبارسنجی‌شده ساخته می‌شود؛
- Document ID، Role، Locator و محدوده‌های متنی عیناً از Evidence موجود گرفته می‌شوند؛
- هیچ Locator یا منبع جدیدی ساخته یا جست‌وجو نمی‌شود؛
- Citation به‌صورت ماشینی اعتبارسنجی می‌شود؛
- Workflow در وضعیت `CITATION_HUMAN_REVIEW_PENDING` متوقف می‌شود؛
- Human Review Registry روی دیسک Sync می‌شود؛
- Queue همچنان 19 باقی می‌ماند.

</div>

In [28]:
# Code Cell 28

cell28_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell27_gate") is not True:
    cell28_failures.append(
        "CELL27_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell27_current_session"
    ),
    dict,
):
    cell28_failures.append(
        "CELL27_CURRENT_SESSION_MISSING"
    )


if (
    globals().get(
        "cell27_current_session",
        {},
    ).get(
        "current_state"
    )
    !=
    "EXPECTED_ANSWER_HUMAN_REVIEW_PENDING"
):
    cell28_failures.append(
        "EXPECTED_ANSWER_NOT_AT_HUMAN_REVIEW_GATE"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell28_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Current Pilot identity
# ============================================================

CELL28_PILOT_ID = (
    cell26_next_pilot_id
)

CELL28_GOLDEN_CASE_ID = (
    cell26_next_golden_case_id
)

CELL28_TASK_DRAFT_ID = (
    cell26_next_task_draft_id
)

CELL28_TARGET_EVIDENCE_ID = (
    cell26_next_target_evidence_id
)

CELL28_QUESTION_DRAFT_ID = (
    cell27_question_draft_id
)

CELL28_EXPECTED_ANSWER_DRAFT_ID = (
    cell27_expected_answer_draft_id
)

CELL28_FULL_572_BLOCK_STATUS = (
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


# ============================================================
# 3) Validate Expected Answer artifact
# ============================================================

cell28_expected_answer_record = copy.deepcopy(
    cell27_current_session[
        "artifacts"
    ][
        "expected_answer"
    ]
)


cell28_expected_answer_identity_valid = all([
    cell28_expected_answer_record.get(
        "expected_answer_draft_id"
    )
    ==
    CELL28_EXPECTED_ANSWER_DRAFT_ID,

    cell28_expected_answer_record.get(
        "pilot_case_id"
    )
    ==
    CELL28_PILOT_ID,

    cell28_expected_answer_record.get(
        "question_draft_id"
    )
    ==
    CELL28_QUESTION_DRAFT_ID,

    cell28_expected_answer_record.get(
        "target_evidence_id"
    )
    ==
    CELL28_TARGET_EVIDENCE_ID,

    cell28_expected_answer_record.get(
        "machine_validation_status"
    )
    ==
    "PASS_PENDING_HUMAN_ADJUDICATION",
])


# ============================================================
# 4) Human ACCEPT for Expected Answer
# ============================================================

CELL28_HUMAN_ANSWER_NOTE = (
    "Human Reviewer accepted the Expected Answer "
    "after reviewing it against the validated source evidence."
)


cell28_answer_human_review = {
    "review_record_version":
        "human_expected_answer_review_record_v1",

    "pilot_case_id":
        CELL28_PILOT_ID,

    "question_draft_id":
        CELL28_QUESTION_DRAFT_ID,

    "expected_answer_draft_id":
        CELL28_EXPECTED_ANSWER_DRAFT_ID,

    "target_evidence_id":
        CELL28_TARGET_EVIDENCE_ID,

    "human_decision":
        "ACCEPT",

    "human_note":
        CELL28_HUMAN_ANSWER_NOTE,

    "human_context_used_as_evidence":
        False,

    "external_knowledge_used":
        False,
}


# ============================================================
# 5) Apply Expected Answer ACCEPT
# ============================================================

cell28_citation_pending_session = (
    cell13_apply_event(
        cell27_current_session,
        "EXPECTED_ANSWER_ACCEPTED",
        human_note=
            CELL28_HUMAN_ANSWER_NOTE,
    )
)


cell28_answer_accept_valid = all([
    cell28_citation_pending_session.get(
        "current_state"
    )
    ==
    "CITATION_MAPPING_PENDING",

    len(
        cell28_citation_pending_session.get(
            "human_decisions",
            [],
        )
    )
    ==
    2,

    cell28_citation_pending_session[
        "human_decisions"
    ][
        -1
    ].get(
        "event"
    )
    ==
    "EXPECTED_ANSWER_ACCEPTED",

    cell28_citation_pending_session.get(
        "queue_advance_performed"
    )
    is False,
])


# ============================================================
# 6) Read validated Target Evidence parts
# ============================================================

cell28_target_parts = copy.deepcopy(
    cell26_next_target_evidence.get(
        "target_parts",
        [],
    )
)


cell28_target_parts_valid = all([
    isinstance(
        cell28_target_parts,
        list,
    ),

    len(
        cell28_target_parts
    )
    >
    0,
])


# ============================================================
# 7) Build Citation entries
# ============================================================

cell28_citation_entries = []


for index, part in enumerate(
    cell28_target_parts
):

    if not isinstance(
        part,
        dict,
    ):
        continue


    cell28_citation_entries.append({
        "citation_index":
            index,

        "target_evidence_id":
            CELL28_TARGET_EVIDENCE_ID,

        "document_id":
            part.get(
                "document_id"
            ),

        "source_id":
            part.get(
                "source_id"
            ),

        "role":
            part.get(
                "role"
            ),

        "char_start":
            part.get(
                "char_start"
            ),

        "char_end":
            part.get(
                "char_end"
            ),

        "char_count":
            part.get(
                "char_count"
            ),

        "token_count":
            part.get(
                "token_count"
            ),

        "locator":
            copy.deepcopy(
                part.get(
                    "locator",
                    {},
                )
            ),

        "attribution":
            copy.deepcopy(
                part.get(
                    "attribution"
                )
            ),

        "locator_origin":
            "COPIED_FROM_VALIDATED_TARGET_EVIDENCE",
    })


# ============================================================
# 8) Exact projection validation
# ============================================================

cell28_source_projection = [
    {
        "document_id":
            part.get(
                "document_id"
            ),

        "source_id":
            part.get(
                "source_id"
            ),

        "role":
            part.get(
                "role"
            ),

        "char_start":
            part.get(
                "char_start"
            ),

        "char_end":
            part.get(
                "char_end"
            ),

        "locator":
            copy.deepcopy(
                part.get(
                    "locator",
                    {},
                )
            ),
    }

    for part
    in cell28_target_parts

    if isinstance(
        part,
        dict,
    )
]


cell28_citation_projection = [
    {
        "document_id":
            entry.get(
                "document_id"
            ),

        "source_id":
            entry.get(
                "source_id"
            ),

        "role":
            entry.get(
                "role"
            ),

        "char_start":
            entry.get(
                "char_start"
            ),

        "char_end":
            entry.get(
                "char_end"
            ),

        "locator":
            copy.deepcopy(
                entry.get(
                    "locator",
                    {},
                )
            ),
    }

    for entry
    in cell28_citation_entries
]


cell28_no_locator_fabrication = (
    cell28_source_projection
    ==
    cell28_citation_projection
)


# ============================================================
# 9) Validate against displayed Evidence
# ============================================================

cell28_displayed_parts = copy.deepcopy(
    cell26_next_target_text_parts
)


cell28_displayed_by_identity = {
    (
        row.get(
            "document_id"
        ),
        row.get(
            "role"
        ),
        row.get(
            "char_start"
        ),
        row.get(
            "char_end"
        ),
    ):
        row

    for row
    in cell28_displayed_parts

    if isinstance(
        row,
        dict,
    )
}


cell28_citation_validation_rows = []


for entry in cell28_citation_entries:

    identity = (
        entry.get(
            "document_id"
        ),
        entry.get(
            "role"
        ),
        entry.get(
            "char_start"
        ),
        entry.get(
            "char_end"
        ),
    )


    displayed = (
        cell28_displayed_by_identity.get(
            identity
        )
    )


    locator = (
        entry.get(
            "locator"
        )
        or
        {}
    )


    canonical_url = locator.get(
        "canonical_url"
    )


    url = locator.get(
        "url"
    )


    row_valid = all([
        isinstance(
            entry.get(
                "document_id"
            ),
            str,
        ),

        isinstance(
            displayed,
            dict,
        ),

        (
            canonical_url is None
            or
            (
                isinstance(
                    canonical_url,
                    str,
                )
                and
                canonical_url.startswith(
                    (
                        "http://",
                        "https://",
                    )
                )
            )
        ),

        (
            url is None
            or
            (
                isinstance(
                    url,
                    str,
                )
                and
                url.startswith(
                    (
                        "http://",
                        "https://",
                    )
                )
            )
        ),

        entry.get(
            "locator_origin"
        )
        ==
        "COPIED_FROM_VALIDATED_TARGET_EVIDENCE",
    ])


    cell28_citation_validation_rows.append({
        "citation_index":
            entry.get(
                "citation_index"
            ),

        "document_id":
            entry.get(
                "document_id"
            ),

        "role":
            entry.get(
                "role"
            ),

        "char_start":
            entry.get(
                "char_start"
            ),

        "char_end":
            entry.get(
                "char_end"
            ),

        "displayed_match":
            isinstance(
                displayed,
                dict,
            ),

        "valid":
            row_valid,
    })


cell28_all_entries_valid = all([
    len(
        cell28_citation_entries
    )
    >
    0,

    len(
        cell28_citation_entries
    )
    ==
    len(
        cell28_displayed_parts
    ),

    all(
        row.get(
            "valid"
        )
        is True

        for row
        in cell28_citation_validation_rows
    ),
])


# ============================================================
# 10) Citation Mapping ID
# ============================================================

cell28_citation_identity_payload = {
    "identity_version":
        "citation_mapping_identity_v1",

    "pilot_case_id":
        CELL28_PILOT_ID,

    "question_draft_id":
        CELL28_QUESTION_DRAFT_ID,

    "expected_answer_draft_id":
        CELL28_EXPECTED_ANSWER_DRAFT_ID,

    "target_evidence_id":
        CELL28_TARGET_EVIDENCE_ID,

    "citation_projection":
        copy.deepcopy(
            cell28_citation_projection
        ),
}


cell28_citation_mapping_id = (
    "citation-mapping:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell28_citation_identity_payload
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


# ============================================================
# 11) Citation Machine Validation
# ============================================================

cell28_citation_machine_validation = {
    "target_parts_present":
        cell28_target_parts_valid,

    "citation_entries_present":
        (
            len(
                cell28_citation_entries
            )
            >
            0
        ),

    "all_target_parts_covered":
        (
            len(
                cell28_citation_entries
            )
            ==
            len(
                cell28_displayed_parts
            )
        ),

    "all_entries_valid":
        cell28_all_entries_valid,

    "locator_fabrication_absent":
        cell28_no_locator_fabrication,

    "target_evidence_identity_match":
        all(
            entry.get(
                "target_evidence_id"
            )
            ==
            CELL28_TARGET_EVIDENCE_ID

            for entry
            in cell28_citation_entries
        ),

    "external_locator_lookup_used":
        False,
}


cell28_citation_machine_pass = all([
    cell28_citation_machine_validation[
        "target_parts_present"
    ],

    cell28_citation_machine_validation[
        "citation_entries_present"
    ],

    cell28_citation_machine_validation[
        "all_target_parts_covered"
    ],

    cell28_citation_machine_validation[
        "all_entries_valid"
    ],

    cell28_citation_machine_validation[
        "locator_fabrication_absent"
    ],

    cell28_citation_machine_validation[
        "target_evidence_identity_match"
    ],
])


# ============================================================
# 12) Citation Mapping Record
# ============================================================

cell28_citation_mapping_record = {
    "record_version":
        "citation_mapping_record_v1",

    "citation_mapping_id":
        cell28_citation_mapping_id,

    "pilot_case_id":
        CELL28_PILOT_ID,

    "golden_case_id":
        CELL28_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL28_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL28_TARGET_EVIDENCE_ID,

    "question_draft_id":
        CELL28_QUESTION_DRAFT_ID,

    "expected_answer_draft_id":
        CELL28_EXPECTED_ANSWER_DRAFT_ID,

    "citation_entries":
        copy.deepcopy(
            cell28_citation_entries
        ),

    "citation_scope":
        "VALIDATED_TARGET_EVIDENCE_ONLY",

    "locator_source":
        "EXISTING_TARGET_EVIDENCE_LOCATORS",

    "external_locator_lookup_used":
        False,

    "machine_validation_status":
        (
            "PASS_PENDING_HUMAN_ADJUDICATION"
            if cell28_citation_machine_pass
            else "FAIL_REQUIRES_REVISION"
        ),

    "human_adjudication_status":
        "PENDING",

    "full_572_generation":
        CELL28_FULL_572_BLOCK_STATUS,
}


# ============================================================
# 13) Register Citation
# ============================================================

cell28_current_session = (
    cell13_apply_event(
        cell28_citation_pending_session,
        "CITATION_MAPPING_REGISTERED",
        artifact=
            cell28_citation_mapping_record,
    )
)


cell28_citation_registered_valid = all([
    cell28_current_session.get(
        "current_state"
    )
    ==
    "CITATION_HUMAN_REVIEW_PENDING",

    isinstance(
        cell28_current_session.get(
            "artifacts",
            {}
        ).get(
            "citation_mapping"
        ),
        dict,
    ),

    cell28_current_session[
        "artifacts"
    ][
        "citation_mapping"
    ].get(
        "citation_mapping_id"
    )
    ==
    cell28_citation_mapping_id,

    cell28_current_session.get(
        "queue_advance_performed"
    )
    is False,
])


# ============================================================
# 14) Update Human Review Case
# ============================================================

cell28_active_human_review_case = copy.deepcopy(
    cell27_active_human_review_case
)


cell28_active_human_review_case[
    "expected_answer_review"
][
    "human_decision"
] = "ACCEPT"


cell28_active_human_review_case[
    "expected_answer_review"
][
    "human_review"
] = copy.deepcopy(
    cell28_answer_human_review
)


cell28_active_human_review_case[
    "citation_review"
] = {
    "citation_mapping_id":
        cell28_citation_mapping_id,

    "citation_entries":
        copy.deepcopy(
            cell28_citation_entries
        ),

    "machine_validation_status":
        cell28_citation_mapping_record[
            "machine_validation_status"
        ],

    "human_decision":
        "PENDING",
}


cell28_active_human_review_case[
    "final_status"
] = (
    "QUESTION_AND_ANSWER_ACCEPTED_"
    "CITATION_PENDING_REVIEW"
)


# ============================================================
# 15) Persistent Registry update
# ============================================================

cell28_persistent_registry_payload = copy.deepcopy(
    cell27_persisted_registry
)


cell28_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell28_active_human_review_case
)


cell28_persistent_registry_payload[
    "remaining_pilot_count"
] = 19


cell28_persistent_registry_payload[
    "continuation_state"
] = {
    "next_pilot_id":
        CELL28_PILOT_ID,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell26_remaining_pilot_ids
        ),

    "remaining_count":
        19,

    "continuation_state_sha256":
        cell26_continuation_state_sha256,
}


cell28_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell28_persistent_registry_payload
    )
)


# ============================================================
# 16) Read back Registry
# ============================================================

cell28_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell28_persisted_active_case = (
    cell28_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell28_persistence_valid = all([
    cell28_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL28_PILOT_ID,

    cell28_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell28_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "citation_mapping_id"
    )
    ==
    cell28_citation_mapping_id,

    cell28_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "PENDING",

    cell28_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    19,
])


# ============================================================
# 17) Safety
# ============================================================

cell28_queue_still_19 = (
    len(
        cell26_remaining_pilot_ids
    )
    ==
    19
)


cell28_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell28_full_572_blocked = (
    cell28_citation_mapping_record[
        "full_572_generation"
    ]
    ==
    CELL28_FULL_572_BLOCK_STATUS
)


# ============================================================
# 18) Final Gate
# ============================================================

cell28_gate = all([
    len(
        cell28_failures
    )
    ==
    0,

    cell28_expected_answer_identity_valid,

    cell28_answer_accept_valid,

    cell28_target_parts_valid,

    cell28_no_locator_fabrication,

    cell28_all_entries_valid,

    cell28_citation_machine_pass,

    cell28_citation_registered_valid,

    cell28_persistence_valid,

    cell28_queue_still_19,

    cell28_base_seed_unchanged,

    cell28_full_572_blocked,
])


# ============================================================
# 19) Diagnostics
# ============================================================

print(
    "=== ANSWER ACCEPTED + CITATION DRAFT "
    "+ REGISTRY SYNC ==="
)


print(
    "\n=== ACCEPTED QUESTION ==="
)

print(
    cell27_question_text
)


print(
    "\n=== ACCEPTED ANSWER ==="
)

print(
    cell27_expected_answer_text
)

print(
    "Human Answer Decision:",
    "ACCEPT",
)


print(
    "\n=== CITATION DRAFT ==="
)

print(
    "Citation Mapping ID:",
    cell28_citation_mapping_id,
)

print(
    "Citation entry count:",
    len(
        cell28_citation_entries
    ),
)


for entry in cell28_citation_entries:

    print(
        "\n--- CITATION ENTRY",
        entry.get(
            "citation_index"
        ),
        "---"
    )

    print(
        "Document:",
        entry.get(
            "document_id"
        ),
    )

    print(
        "Role:",
        entry.get(
            "role"
        ),
    )

    print(
        "Chars:",
        (
            entry.get(
                "char_start"
            ),
            entry.get(
                "char_end"
            ),
        ),
    )

    print(
        "Canonical URL:",
        (
            entry.get(
                "locator",
                {}
            ).get(
                "canonical_url"
            )
        ),
    )

    print(
        "URL:",
        (
            entry.get(
                "locator",
                {}
            ).get(
                "url"
            )
        ),
    )


print(
    "\n=== CITATION MACHINE VALIDATION ==="
)

for name, status in (
    cell28_citation_machine_validation.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "Machine PASS:",
    cell28_citation_machine_pass,
)

print(
    "No locator fabrication:",
    cell28_no_locator_fabrication,
)


print(
    "\n=== WORKFLOW ==="
)

print(
    "Current State:",
    cell28_current_session.get(
        "current_state"
    ),
)

print(
    "Question Decision:",
    "ACCEPT",
)

print(
    "Answer Decision:",
    "ACCEPT",
)

print(
    "Citation Decision:",
    "PENDING",
)


print(
    "\n=== PERSISTENCE ==="
)

print(
    "Registry:",
    cell21_registry_path,
)

print(
    "Persisted Answer Decision:",
    cell28_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    ),
)

print(
    "Persisted Citation Decision:",
    cell28_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "human_decision"
    ),
)

print(
    "Persistence valid:",
    cell28_persistence_valid,
)


print(
    "\n=== QUEUE / SAFETY ==="
)

print(
    "Queue still 19:",
    cell28_queue_still_19,
)

print(
    "Base Seed unchanged:",
    cell28_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell28_full_572_blocked,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell28_failures
)


print(
    "\nCODE CELL 28 GATE:",
    (
        "PASS"
        if cell28_gate
        else "FAIL"
    ),
)


assert cell28_gate, (
    "Expected Answer ACCEPT / Citation Draft / Registry Sync "
    "failed. Repair this SAME Code Cell 28 before "
    "Citation Human Review."
)

=== ANSWER ACCEPTED + CITATION DRAFT + REGISTRY SYNC ===

=== ACCEPTED QUESTION ===
در تمرین ۸، برای بررسی هزینه‌های ماهانه افراد چه مراحلی باید انجام شود؟

=== ACCEPTED ANSWER ===
باید از هر مقطع سنی دو نفر انتخاب شوند، درباره میزان هزینه ماهانه آن‌ها مصاحبه شود، به آن‌ها اطمینان داده شود که اطلاعاتشان محرمانه خواهد ماند و در پایان نتیجه مصاحبه‌ها تحلیل شود.
Human Answer Decision: ACCEPT

=== CITATION DRAFT ===
Citation Mapping ID: citation-mapping:0fe2439cd888def58854539b9653767c
Citation entry count: 2

--- CITATION ENTRY 0 ---
Document: book:trainer-training-financial-literacy:fa-source:p000329
Role: PROTECTED_REGION_MEMBER
Chars: (0, 7)
Canonical URL: None
URL: None

--- CITATION ENTRY 1 ---
Document: book:trainer-training-financial-literacy:fa-source:p000330
Role: PROTECTED_REGION_MEMBER
Chars: (0, 156)
Canonical URL: None
URL: None

=== CITATION MACHINE VALIDATION ===
target_parts_present: True
citation_entries_present: True
all_target_parts_covered: True
all_entries_valid: True

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 29 — Citation ACCEPT، تکمیل Pilot، Queue Advance و آماده‌سازی Pilot بعدی

Citation این Pilot توسط Human Reviewer پذیرفته شد.

در این مرحله:

- Citation با تصمیم `ACCEPT` ثبت می‌شود؛
- Question + Expected Answer + Citation این Pilot کامل می‌شوند؛
- پرونده Human Review به Completed Cases منتقل می‌شود؛
- Author Final Review همچنان `NOT_STARTED` باقی می‌ماند؛
- Registry دائمی روی دیسک Sync می‌شود؛
- Queue از 19 به 18 کاهش پیدا می‌کند؛
- Pilot بعدی انتخاب و Evidence آن برای مرحله سؤال آماده می‌شود.

Full 572 Generation همچنان Blocked است.

</div>

In [29]:
# Code Cell 29

cell29_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell28_gate") is not True:
    cell29_failures.append(
        "CELL28_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell28_current_session"
    ),
    dict,
):
    cell29_failures.append(
        "CELL28_CURRENT_SESSION_MISSING"
    )


if (
    globals().get(
        "cell28_current_session",
        {},
    ).get(
        "current_state"
    )
    !=
    "CITATION_HUMAN_REVIEW_PENDING"
):
    cell29_failures.append(
        "CITATION_NOT_AT_HUMAN_REVIEW_GATE"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell29_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Current Pilot
# ============================================================

CELL29_COMPLETED_PILOT_ID = (
    cell26_next_pilot_id
)

CELL29_FULL_572_BLOCK_STATUS = (
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


# ============================================================
# 3) Human Citation ACCEPT
# ============================================================

CELL29_HUMAN_CITATION_NOTE = (
    "Human Reviewer accepted the Citation Mapping "
    "after reviewing the validated book source locators."
)


cell29_citation_human_review = {
    "review_record_version":
        "human_citation_review_record_v1",

    "pilot_case_id":
        CELL29_COMPLETED_PILOT_ID,

    "citation_mapping_id":
        cell28_citation_mapping_id,

    "target_evidence_id":
        cell26_next_target_evidence_id,

    "human_decision":
        "ACCEPT",

    "human_note":
        CELL29_HUMAN_CITATION_NOTE,

    "external_locator_lookup_used":
        False,
}


# ============================================================
# 4) Apply Citation ACCEPT
# ============================================================

cell29_pre_advance_session = (
    cell13_apply_event(
        cell28_current_session,
        "CITATION_ACCEPTED",
        human_note=
            CELL29_HUMAN_CITATION_NOTE,
    )
)


cell29_citation_accept_valid = all([
    cell29_pre_advance_session.get(
        "current_state"
    )
    ==
    "PILOT_COMPLETE_PENDING_QUEUE_ADVANCE",

    cell29_pre_advance_session.get(
        "queue_advance_permitted"
    )
    is True,

    cell29_pre_advance_session.get(
        "queue_advance_performed"
    )
    is False,

    len(
        cell29_pre_advance_session.get(
            "human_decisions",
            [],
        )
    )
    ==
    3,

    cell29_pre_advance_session[
        "human_decisions"
    ][
        -1
    ].get(
        "event"
    )
    ==
    "CITATION_ACCEPTED",
])


# ============================================================
# 5) Finalize Human Review Case
# ============================================================

cell29_completed_human_review_case = (
    copy.deepcopy(
        cell28_active_human_review_case
    )
)


cell29_completed_human_review_case[
    "citation_review"
][
    "human_decision"
] = "ACCEPT"


cell29_completed_human_review_case[
    "citation_review"
][
    "human_review"
] = copy.deepcopy(
    cell29_citation_human_review
)


cell29_completed_human_review_case[
    "author_final_review"
] = {
    "status":
        "NOT_STARTED",

    "decision":
        None,

    "notes":
        None,
}


cell29_completed_human_review_case[
    "final_status"
] = (
    "HUMAN_REVIEW_COMPLETE_"
    "AUTHOR_FINAL_REVIEW_PENDING"
)


cell29_completed_human_review_case_sha256 = (
    cell10_stable_sha(
        cell29_completed_human_review_case
    )
)


# ============================================================
# 6) Queue advancement
# ============================================================

cell29_parent_remaining_ids = copy.deepcopy(
    cell26_continuation_state[
        "remaining_pilot_ids"
    ]
)


cell29_parent_remaining_queue = copy.deepcopy(
    cell26_continuation_state[
        "remaining_pilot_queue"
    ]
)


cell29_queue_preconditions_valid = all([
    len(
        cell29_parent_remaining_ids
    )
    ==
    19,

    len(
        cell29_parent_remaining_queue
    )
    ==
    19,

    cell29_parent_remaining_ids[
        0
    ]
    ==
    CELL29_COMPLETED_PILOT_ID,

    cell29_citation_accept_valid,
])


if cell29_queue_preconditions_valid:

    cell29_remaining_pilot_ids = copy.deepcopy(
        cell29_parent_remaining_ids[
            1:
        ]
    )

    cell29_remaining_pilot_queue = copy.deepcopy(
        cell29_parent_remaining_queue[
            1:
        ]
    )

else:

    cell29_remaining_pilot_ids = []
    cell29_remaining_pilot_queue = []


cell29_next_pilot_id = (
    cell29_remaining_pilot_ids[
        0
    ]

    if cell29_remaining_pilot_ids

    else None
)


cell29_next_pilot_queue_entry = (
    copy.deepcopy(
        cell29_remaining_pilot_queue[
            0
        ]
    )

    if cell29_remaining_pilot_queue

    else None
)


cell29_queue_advanced_valid = all([
    len(
        cell29_remaining_pilot_ids
    )
    ==
    18,

    len(
        cell29_remaining_pilot_queue
    )
    ==
    18,

    CELL29_COMPLETED_PILOT_ID
    not in
    cell29_remaining_pilot_ids,

    isinstance(
        cell29_next_pilot_id,
        str,
    ),

    cell29_next_pilot_id
    !=
    CELL29_COMPLETED_PILOT_ID,
])


# ============================================================
# 7) Completed session
# ============================================================

cell29_completed_session = copy.deepcopy(
    cell29_pre_advance_session
)


cell29_completed_session[
    "current_state"
] = "PILOT_COMPLETE"


cell29_completed_session[
    "queue_advance_permitted"
] = False


cell29_completed_session[
    "queue_advance_performed"
] = True


cell29_completed_session[
    "completion_status"
] = (
    "HUMAN_REVIEW_COMPLETE_"
    "AUTHOR_FINAL_REVIEW_PENDING"
)


# ============================================================
# 8) New authoritative Continuation State
# ============================================================

cell29_continuation_state = {
    "state_version":
        "compact_pilot_continuation_state_v4",

    "authoritative_forward_state":
        True,

    "parent_state_sha256":
        cell26_continuation_state_sha256,

    "just_completed_pilot_id":
        CELL29_COMPLETED_PILOT_ID,

    "just_completed_case_sha256":
        cell29_completed_human_review_case_sha256,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell29_remaining_pilot_ids
        ),

    "remaining_pilot_queue":
        copy.deepcopy(
            cell29_remaining_pilot_queue
        ),

    "remaining_count":
        18,

    "next_pilot_id":
        cell29_next_pilot_id,

    "next_pilot_queue_entry":
        copy.deepcopy(
            cell29_next_pilot_queue_entry
        ),

    "full_572_generation":
        CELL29_FULL_572_BLOCK_STATUS,
}


cell29_continuation_state_sha256 = (
    cell10_stable_sha(
        cell29_continuation_state
    )
)


# ============================================================
# 9) Resolve next Pilot packet
# ============================================================

cell29_pilot_by_id = (
    cell13_seed[
        "final_runtime"
    ][
        "question_drafting_pilot_by_id"
    ]
)


cell29_next_pilot_packet = copy.deepcopy(
    cell29_pilot_by_id.get(
        cell29_next_pilot_id
    )
)


cell29_next_packet_valid = isinstance(
    cell29_next_pilot_packet,
    dict,
)


# ============================================================
# 10) Resolve identity chain
# ============================================================

cell29_next_golden_case_id = None
cell29_next_task_draft_id = None
cell29_next_target_evidence_id = None


if cell29_next_packet_valid:

    try:

        cell29_next_golden_case_id = (
            cell13_exact_single_id(
                cell29_next_pilot_packet,
                "golden_case",
            )
        )


        cell29_next_task_draft_id = (
            cell13_exact_single_id(
                cell29_next_pilot_packet,
                "task_draft",
            )
        )


        cell29_next_target_evidence_id = (
            cell13_exact_single_id(
                cell29_next_pilot_packet,
                "target_evidence",
            )
        )


    except Exception as exc:

        cell29_failures.append({
            "reason":
                "NEXT_ID_CHAIN_FAILED",

            "error":
                repr(
                    exc
                ),
        })


cell29_next_identity_valid = all([
    isinstance(
        cell29_next_golden_case_id,
        str,
    ),

    isinstance(
        cell29_next_task_draft_id,
        str,
    ),

    isinstance(
        cell29_next_target_evidence_id,
        str,
    ),
])


# ============================================================
# 11) Resolve support records
# ============================================================

cell29_next_golden_case = None
cell29_next_task_plan = None
cell29_next_target_evidence = None


if cell29_next_identity_valid:

    try:

        cell29_next_golden_case = copy.deepcopy(
            cell13_active_support[
                "golden_case_records_by_id"
            ][
                cell29_next_golden_case_id
            ]
        )


        cell29_next_task_plan = copy.deepcopy(
            cell13_find_unique_mapping_value(
                cell13_active_support.get(
                    "task_draft_plans",
                    {},
                ),
                "task_draft",
                cell29_next_task_draft_id,
            )[
                "value"
            ]
        )


        cell29_next_target_evidence = copy.deepcopy(
            cell13_find_unique_list_record(
                cell13_active_support.get(
                    "target_evidence_records",
                    [],
                ),
                "target_evidence_id",
                cell29_next_target_evidence_id,
            )
        )


    except Exception as exc:

        cell29_failures.append({
            "reason":
                "NEXT_SUPPORT_RESOLUTION_FAILED",

            "error":
                repr(
                    exc
                ),
        })


# ============================================================
# 12) Classification helper
# ============================================================

def cell29_scalar(
    mapping,
    names,
):

    match = cell14_find_first_scalar(
        mapping,
        names,
    )

    if isinstance(
        match,
        dict,
    ):
        return match.get(
            "value"
        )

    return None


cell29_next_source_type = cell29_scalar(
    cell29_next_pilot_packet,
    [
        "source_type",
        "source_kind",
        "source",
    ],
)


cell29_next_stratum = cell29_scalar(
    cell29_next_pilot_packet,
    [
        "stratum",
        "stratum_name",
        "stratum_id",
    ],
)


cell29_next_task_family = cell29_scalar(
    cell29_next_pilot_packet,
    [
        "task_family",
    ],
)


cell29_next_question_archetype = cell29_scalar(
    cell29_next_pilot_packet,
    [
        "question_archetype",
    ],
)


# ============================================================
# 13) Extract next Evidence
# ============================================================

cell29_raw_target_parts = []


if cell29_next_packet_valid:

    packet_parts = (
        cell29_next_pilot_packet.get(
            "target_text_parts"
        )
    )

    if isinstance(
        packet_parts,
        list,
    ):
        cell29_raw_target_parts = copy.deepcopy(
            packet_parts
        )


if (
    not cell29_raw_target_parts
    and
    isinstance(
        cell29_next_target_evidence,
        dict,
    )
):

    evidence_parts = (
        cell29_next_target_evidence.get(
            "target_parts"
        )
    )

    if isinstance(
        evidence_parts,
        list,
    ):
        cell29_raw_target_parts = copy.deepcopy(
            evidence_parts
        )


cell29_next_target_text_parts = []


for index, part in enumerate(
    cell29_raw_target_parts
):

    if not isinstance(
        part,
        dict,
    ):
        continue


    text_value = None


    for field_name in (
        "text",
        "target_text",
        "content",
        "evidence_text",
        "text_content",
    ):

        candidate = part.get(
            field_name
        )

        if (
            isinstance(
                candidate,
                str,
            )
            and
            candidate.strip()
        ):

            text_value = candidate.strip()
            break


    if text_value is None:
        continue


    cell29_next_target_text_parts.append({
        "part_index":
            index,

        "document_id":
            part.get(
                "document_id"
            ),

        "role":
            part.get(
                "role"
            ),

        "char_start":
            part.get(
                "char_start"
            ),

        "char_end":
            part.get(
                "char_end"
            ),

        "text":
            text_value,

        "text_sha256":
            hashlib.sha256(
                text_value.encode(
                    "utf-8"
                )
            ).hexdigest(),
    })


cell29_next_combined_target_text = (
    "\n\n".join(
        row[
            "text"
        ]

        for row
        in cell29_next_target_text_parts
    )
)


cell29_next_target_text_available = bool(
    cell29_next_combined_target_text.strip()
)


# ============================================================
# 14) Build next Pilot workspace
# ============================================================

cell29_next_pilot_workspace = {
    "workspace_version":
        "compact_human_review_workspace_v1",

    "status":
        "READY_FOR_QUESTION_DRAFT",

    "pilot_case_id":
        cell29_next_pilot_id,

    "golden_case_id":
        cell29_next_golden_case_id,

    "task_draft_id":
        cell29_next_task_draft_id,

    "target_evidence_id":
        cell29_next_target_evidence_id,

    "source_type":
        cell29_next_source_type,

    "stratum":
        cell29_next_stratum,

    "task_family":
        cell29_next_task_family,

    "question_archetype":
        cell29_next_question_archetype,

    "source_evidence":
        copy.deepcopy(
            cell29_next_target_text_parts
        ),

    "combined_target_text":
        cell29_next_combined_target_text,

    "golden_case":
        copy.deepcopy(
            cell29_next_golden_case
        ),

    "task_plan":
        copy.deepcopy(
            cell29_next_task_plan
        ),

    "target_evidence":
        copy.deepcopy(
            cell29_next_target_evidence
        ),

    "model_drafts":
        [],

    "human_reviews":
        [],

    "expected_answer_review":
        None,

    "citation_review":
        None,

    "author_final_review":
        {
            "status":
                "NOT_STARTED",

            "decision":
                None,

            "notes":
                None,
        },

    "workflow_state":
        "QUESTION_DRAFT_PENDING",

    "full_572_generation":
        CELL29_FULL_572_BLOCK_STATUS,
}


cell29_next_workspace_ready = all([
    cell29_next_packet_valid,

    cell29_next_identity_valid,

    isinstance(
        cell29_next_golden_case,
        dict,
    ),

    isinstance(
        cell29_next_task_plan,
        dict,
    ),

    isinstance(
        cell29_next_target_evidence,
        dict,
    ),

    cell29_next_target_text_available,
])


# ============================================================
# 15) Update persistent Registry
# ============================================================

cell29_completed_cases = copy.deepcopy(
    cell28_persisted_registry.get(
        "completed_human_review_cases",
        [],
    )
)


cell29_completed_cases.append(
    copy.deepcopy(
        cell29_completed_human_review_case
    )
)


cell29_persistent_registry_payload = copy.deepcopy(
    cell28_persisted_registry
)


cell29_persistent_registry_payload[
    "completed_human_review_cases"
] = copy.deepcopy(
    cell29_completed_cases
)


cell29_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell29_next_pilot_workspace
)


cell29_persistent_registry_payload[
    "remaining_pilot_count"
] = 18


cell29_persistent_registry_payload[
    "continuation_state"
] = {
    "next_pilot_id":
        cell29_next_pilot_id,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell29_remaining_pilot_ids
        ),

    "remaining_count":
        18,

    "continuation_state_sha256":
        cell29_continuation_state_sha256,
}


cell29_author_review = (
    cell29_persistent_registry_payload.get(
        "author_review"
    )
)


if not isinstance(
    cell29_author_review,
    dict,
):

    cell29_author_review = {
        "status":
            "NOT_STARTED",

        "reviewer":
            "AUTHOR",

        "cases_ready_for_review":
            0,

        "notes":
            None,
    }


cell29_author_review[
    "cases_ready_for_review"
] = len(
    cell29_completed_cases
)


cell29_persistent_registry_payload[
    "author_review"
] = cell29_author_review


# ============================================================
# 16) Persist
# ============================================================

cell29_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell29_persistent_registry_payload
    )
)


cell29_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell29_persisted_active_case = (
    cell29_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell29_persistence_valid = all([
    len(
        cell29_persisted_registry.get(
            "completed_human_review_cases",
            [],
        )
    )
    ==
    len(
        cell29_completed_cases
    ),

    cell29_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    18,

    cell29_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    cell29_next_pilot_id,

    cell29_persisted_registry.get(
        "continuation_state",
        {},
    ).get(
        "next_pilot_id"
    )
    ==
    cell29_next_pilot_id,

    cell29_persisted_registry.get(
        "author_review",
        {},
    ).get(
        "cases_ready_for_review"
    )
    ==
    len(
        cell29_completed_cases
    ),
])


# ============================================================
# 17) Safety
# ============================================================

cell29_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell29_full_572_blocked = (
    cell29_continuation_state[
        "full_572_generation"
    ]
    ==
    CELL29_FULL_572_BLOCK_STATUS
)


# ============================================================
# 18) Final Gate
# ============================================================

cell29_gate = all([
    len(
        cell29_failures
    )
    ==
    0,

    cell29_citation_accept_valid,

    cell29_queue_preconditions_valid,

    cell29_queue_advanced_valid,

    cell29_next_workspace_ready,

    cell29_persistence_valid,

    cell29_base_seed_unchanged,

    cell29_full_572_blocked,
])


# ============================================================
# 19) Diagnostics
# ============================================================

print(
    "=== PILOT COMPLETED + REGISTRY SYNC "
    "+ NEXT PILOT ==="
)


print(
    "\n=== COMPLETED PILOT ==="
)

print(
    "Pilot:",
    CELL29_COMPLETED_PILOT_ID,
)

print(
    "Question:",
    cell27_question_text,
)

print(
    "Answer:",
    cell27_expected_answer_text,
)

print(
    "Citation:",
    "ACCEPT",
)

print(
    "Human Review status:",
    cell29_completed_human_review_case[
        "final_status"
    ],
)

print(
    "Author Final Review:",
    "NOT_STARTED",
)


print(
    "\n=== QUEUE ==="
)

print(
    "Before:",
    len(
        cell29_parent_remaining_ids
    ),
)

print(
    "After:",
    len(
        cell29_remaining_pilot_ids
    ),
)

print(
    "Completed Pilot removed:",
    (
        CELL29_COMPLETED_PILOT_ID
        not in
        cell29_remaining_pilot_ids
    ),
)

print(
    "Next Pilot:",
    cell29_next_pilot_id,
)


print(
    "\n=== PERSISTENT REGISTRY ==="
)

print(
    "Path:",
    cell21_registry_path,
)

print(
    "Completed Human Review cases:",
    len(
        cell29_persisted_registry.get(
            "completed_human_review_cases",
            [],
        )
    ),
)

print(
    "Cases ready for Author Review:",
    cell29_persisted_registry.get(
        "author_review",
        {},
    ).get(
        "cases_ready_for_review"
    ),
)

print(
    "Remaining Pilots:",
    cell29_persisted_registry.get(
        "remaining_pilot_count"
    ),
)

print(
    "Active Pilot:",
    cell29_persisted_active_case.get(
        "pilot_case_id"
    ),
)

print(
    "Persistence valid:",
    cell29_persistence_valid,
)


print(
    "\n=== NEXT PILOT IDENTITY ==="
)

print(
    "Pilot:",
    cell29_next_pilot_id,
)

print(
    "Golden Case:",
    cell29_next_golden_case_id,
)

print(
    "Task Draft:",
    cell29_next_task_draft_id,
)

print(
    "Target Evidence:",
    cell29_next_target_evidence_id,
)

print(
    "Source type:",
    cell29_next_source_type,
)

print(
    "Stratum:",
    cell29_next_stratum,
)

print(
    "Task family:",
    cell29_next_task_family,
)

print(
    "Question archetype:",
    cell29_next_question_archetype,
)


print(
    "\n=== NEXT PILOT SOURCE EVIDENCE ==="
)

print(
    cell29_next_combined_target_text
)


print(
    "\n=== SAFETY ==="
)

print(
    "Base Seed unchanged:",
    cell29_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell29_full_572_blocked,
)

print(
    "Author Notebook rewrite:",
    False,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell29_failures
)


print(
    "\nCODE CELL 29 GATE:",
    (
        "PASS"
        if cell29_gate
        else "FAIL"
    ),
)


assert cell29_gate, (
    "Citation ACCEPT / Pilot completion / Queue advance / "
    "Registry sync / next Pilot preparation failed. "
    "Repair this SAME Code Cell 29 before continuing."
)

=== PILOT COMPLETED + REGISTRY SYNC + NEXT PILOT ===

=== COMPLETED PILOT ===
Pilot: question-pilot:909b695797935246c92c7a66ebe0f773
Question: در تمرین ۸، برای بررسی هزینه‌های ماهانه افراد چه مراحلی باید انجام شود؟
Answer: باید از هر مقطع سنی دو نفر انتخاب شوند، درباره میزان هزینه ماهانه آن‌ها مصاحبه شود، به آن‌ها اطمینان داده شود که اطلاعاتشان محرمانه خواهد ماند و در پایان نتیجه مصاحبه‌ها تحلیل شود.
Citation: ACCEPT
Human Review status: HUMAN_REVIEW_COMPLETE_AUTHOR_FINAL_REVIEW_PENDING
Author Final Review: NOT_STARTED

=== QUEUE ===
Before: 19
After: 18
Completed Pilot removed: True
Next Pilot: question-pilot:3023c1c546df2fe2211f79702b6e3dbb

=== PERSISTENT REGISTRY ===
Path: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/data/phase2/human_review/human_review_registry.json
Completed Human Review cases: 4
Cases ready for Author Review: 4
Remaining Pilots: 18
Active Pilot: question-pilot:3023c1c546df2fe2211f79702b6e3dbb
Persistence valid: True

=== NEXT PILOT 

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 30 — Human RETARGET Decision + Replacement Candidate Discovery

Human Review نشان داد Target Evidence فعلی، اگرچه از نظر ساختاری یک Tiny Table Region معتبر است، بیشتر شبیه فهرست عناوین است و سؤال طبیعی و ارزشمند برای Golden Dataset از آن ساخته نمی‌شود.

بنابراین:

- Pilot حذف نمی‌شود؛
- Task Family و Coverage موردنیاز حفظ می‌شود؛
- Target Evidence فعلی با تصمیم انسانی `RETARGET` کنار گذاشته می‌شود؛
- وضعیت Pilot به `BLOCKED_REQUIRES_RETARGET` می‌رود؛
- دلیل Retarget در Human Review Registry ثبت می‌شود؛
- Queue فعلاً 18 باقی می‌ماند؛
- Candidateهای جایگزین از Target Evidenceهای موجود جست‌وجو می‌شوند؛
- هیچ Candidate به‌صورت خودکار پذیرفته یا فعال نمی‌شود.

هدف، پیدا کردن یک Table Region معنادارتر برای همان نوع Retrieval Test است.

</div>

In [30]:
# Code Cell 30

cell30_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell29_gate") is not True:
    cell30_failures.append(
        "CELL29_NOT_PASS"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell30_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Current Pilot / Target
# ============================================================

CELL30_PILOT_ID = (
    cell29_next_pilot_id
)

CELL30_CURRENT_TARGET_ID = (
    cell29_next_target_evidence_id
)

CELL30_TASK_FAMILY = (
    cell29_next_task_family
)

CELL30_QUESTION_ARCHETYPE = (
    cell29_next_question_archetype
)

CELL30_STRATUM = (
    cell29_next_stratum
)

CELL30_SOURCE_TYPE = (
    cell29_next_source_type
)


cell30_current_identity_valid = all([
    isinstance(
        CELL30_PILOT_ID,
        str,
    ),

    isinstance(
        CELL30_CURRENT_TARGET_ID,
        str,
    ),

    CELL30_TASK_FAMILY
    ==
    "TINY_TABLE_EVIDENCE_RETRIEVAL",

    CELL30_QUESTION_ARCHETYPE
    ==
    "STRUCTURE_AWARE_RETRIEVAL",

    CELL30_STRATUM
    ==
    "intrinsically_tiny_table_region",

    CELL30_SOURCE_TYPE
    ==
    "book",
])


# ============================================================
# 3) Human RETARGET decision
# ============================================================

CELL30_RETARGET_DECISION = (
    "RETARGET"
)


CELL30_RETARGET_REASON = (
    "The current Target Evidence is index-like / title-list "
    "content. Although structurally valid as a tiny table "
    "region, it does not support a sufficiently natural, "
    "useful, user-facing Golden QA item. Preserve the Pilot "
    "coverage and replace only the Target Evidence with a "
    "more meaningful table region."
)


cell30_human_retarget_review = {
    "review_record_version":
        "human_retarget_review_record_v1",

    "pilot_case_id":
        CELL30_PILOT_ID,

    "current_target_evidence_id":
        CELL30_CURRENT_TARGET_ID,

    "decision":
        CELL30_RETARGET_DECISION,

    "reason":
        CELL30_RETARGET_REASON,

    "problem_level":
        "TARGET_EVIDENCE",

    "draft_problem":
        False,

    "pilot_rejected":
        False,

    "coverage_preserved":
        True,

    "required_task_family":
        CELL30_TASK_FAMILY,

    "required_question_archetype":
        CELL30_QUESTION_ARCHETYPE,

    "preferred_stratum":
        CELL30_STRATUM,

    "preferred_source_type":
        CELL30_SOURCE_TYPE,

    "human_context_evidence_status":
        "NON_EVIDENCE",
}


cell30_human_retarget_review_sha256 = (
    cell10_stable_sha(
        cell30_human_retarget_review
    )
)


# ============================================================
# 4) Mark active workspace as blocked for Retarget
# ============================================================

cell30_retarget_workspace = copy.deepcopy(
    cell29_next_pilot_workspace
)


cell30_retarget_workspace[
    "status"
] = "RETARGET_REQUIRED"


cell30_retarget_workspace[
    "workflow_state"
] = "BLOCKED_REQUIRES_RETARGET"


cell30_retarget_workspace[
    "retarget_review"
] = copy.deepcopy(
    cell30_human_retarget_review
)


cell30_retarget_workspace[
    "retarget_status"
] = {
    "decision":
        "RETARGET",

    "current_target_rejected_for_golden_qa":
        True,

    "replacement_selected":
        False,

    "replacement_target_evidence_id":
        None,
}


cell30_retarget_workspace[
    "model_drafts"
] = []


cell30_retarget_workspace[
    "expected_answer_review"
] = None


cell30_retarget_workspace[
    "citation_review"
] = None


# ============================================================
# 5) Generic recursive helpers
# ============================================================

def cell30_walk(
    value,
):

    if isinstance(
        value,
        dict,
    ):

        yield value

        for nested in value.values():
            yield from cell30_walk(
                nested
            )


    elif isinstance(
        value,
        (
            list,
            tuple,
        ),
    ):

        for nested in value:
            yield from cell30_walk(
                nested
            )


def cell30_first_scalar(
    value,
    names,
):

    for mapping in cell30_walk(
        value
    ):

        for name in names:

            candidate = mapping.get(
                name
            )

            if isinstance(
                candidate,
                (
                    str,
                    int,
                    float,
                    bool,
                ),
            ):

                return candidate

    return None


def cell30_deep_contains(
    value,
    needle,
):

    if value == needle:
        return True


    if isinstance(
        value,
        dict,
    ):

        return any(
            cell30_deep_contains(
                nested,
                needle,
            )

            for nested
            in value.values()
        )


    if isinstance(
        value,
        (
            list,
            tuple,
            set,
        ),
    ):

        return any(
            cell30_deep_contains(
                nested,
                needle,
            )

            for nested
            in value
        )


    return False


def cell30_extract_text_parts(
    target_record,
):

    parts = None


    if isinstance(
        target_record,
        dict,
    ):

        for field_name in (
            "target_parts",
            "target_text_parts",
            "parts",
        ):

            candidate = target_record.get(
                field_name
            )

            if isinstance(
                candidate,
                list,
            ):

                parts = candidate
                break


    if not isinstance(
        parts,
        list,
    ):

        return []


    rows = []


    for index, part in enumerate(
        parts
    ):

        if not isinstance(
            part,
            dict,
        ):
            continue


        text_value = None


        for field_name in (
            "text",
            "target_text",
            "content",
            "evidence_text",
            "text_content",
        ):

            candidate = part.get(
                field_name
            )

            if (
                isinstance(
                    candidate,
                    str,
                )
                and
                candidate.strip()
            ):

                text_value = candidate.strip()
                break


        if text_value is None:
            continue


        rows.append({
            "part_index":
                index,

            "document_id":
                part.get(
                    "document_id"
                ),

            "role":
                part.get(
                    "role"
                ),

            "char_start":
                part.get(
                    "char_start"
                ),

            "char_end":
                part.get(
                    "char_end"
                ),

            "text":
                text_value,
        })


    return rows


# ============================================================
# 6) Candidate pool
# ============================================================

cell30_target_records = copy.deepcopy(
    cell13_active_support.get(
        "target_evidence_records",
        [],
    )
)


if not isinstance(
    cell30_target_records,
    list,
):

    cell30_target_records = []


cell30_task_plans = (
    cell13_active_support.get(
        "task_draft_plans",
        {}
    )
)


# ============================================================
# 7) Existing active Target IDs
# ============================================================

cell30_active_target_ids = set()


for packet in (
    cell13_seed[
        "final_runtime"
    ][
        "question_drafting_pilot_packets"
    ]
):

    try:

        target_id = (
            cell13_exact_single_id(
                packet,
                "target_evidence",
            )
        )


        if isinstance(
            target_id,
            str,
        ):

            cell30_active_target_ids.add(
                target_id
            )


    except Exception:

        pass


# ============================================================
# 8) Build candidate records
# ============================================================

cell30_candidate_rows = []


for target_record in (
    cell30_target_records
):

    if not isinstance(
        target_record,
        dict,
    ):
        continue


    target_id = (
        target_record.get(
            "target_evidence_id"
        )
    )


    if not isinstance(
        target_id,
        str,
    ):
        continue


    if target_id == CELL30_CURRENT_TARGET_ID:
        continue


    # Avoid targets already used by another active Pilot.
    if target_id in cell30_active_target_ids:
        continue


    linked_task_plans = []


    if isinstance(
        cell30_task_plans,
        dict,
    ):

        task_plan_iterable = (
            cell30_task_plans.values()
        )

    elif isinstance(
        cell30_task_plans,
        list,
    ):

        task_plan_iterable = (
            cell30_task_plans
        )

    else:

        task_plan_iterable = []


    for task_plan in task_plan_iterable:

        if cell30_deep_contains(
            task_plan,
            target_id,
        ):

            linked_task_plans.append(
                task_plan
            )


    task_family = (
        cell30_first_scalar(
            linked_task_plans,
            [
                "task_family",
            ],
        )
        or
        cell30_first_scalar(
            target_record,
            [
                "task_family",
            ],
        )
    )


    question_archetype = (
        cell30_first_scalar(
            linked_task_plans,
            [
                "question_archetype",
            ],
        )
        or
        cell30_first_scalar(
            target_record,
            [
                "question_archetype",
            ],
        )
    )


    stratum = (
        cell30_first_scalar(
            linked_task_plans,
            [
                "stratum",
                "stratum_name",
                "stratum_id",
            ],
        )
        or
        cell30_first_scalar(
            target_record,
            [
                "stratum",
                "stratum_name",
                "stratum_id",
            ],
        )
    )


    source_type = (
        cell30_first_scalar(
            target_record,
            [
                "source_type",
                "source_kind",
            ],
        )
    )


    text_parts = (
        cell30_extract_text_parts(
            target_record
        )
    )


    combined_text = (
        "\n\n".join(
            row[
                "text"
            ]

            for row
            in text_parts
        )
    )


    if not combined_text.strip():
        continue


    # Compatibility score.
    compatibility_score = 0


    if task_family == CELL30_TASK_FAMILY:
        compatibility_score += 4


    if (
        question_archetype
        ==
        CELL30_QUESTION_ARCHETYPE
    ):
        compatibility_score += 3


    if stratum == CELL30_STRATUM:
        compatibility_score += 2


    if source_type == CELL30_SOURCE_TYPE:
        compatibility_score += 1


    normalized_text = (
        cell15_normalize_text(
            combined_text
        )
    )


    # Only a weak heuristic; Human Review remains authoritative.
    index_like_markers = sum([
        "عنوان"
        in
        normalized_text,

        (
            normalized_text.count(
                "\n"
            )
            >=
            4
        ),

        (
            len(
                normalized_text.split()
            )
            <
            25
        ),
    ])


    cell30_candidate_rows.append({
        "target_evidence_id":
            target_id,

        "task_family":
            task_family,

        "question_archetype":
            question_archetype,

        "stratum":
            stratum,

        "source_type":
            source_type,

        "compatibility_score":
            compatibility_score,

        "index_like_marker_count":
            index_like_markers,

        "text_parts":
            copy.deepcopy(
                text_parts
            ),

        "combined_text":
            combined_text,

        "text_sha256":
            hashlib.sha256(
                combined_text.encode(
                    "utf-8"
                )
            ).hexdigest(),
    })


# ============================================================
# 9) Rank shortlist
# ============================================================

cell30_candidate_rows = sorted(
    cell30_candidate_rows,
    key=lambda row: (
        -row[
            "compatibility_score"
        ],
        row[
            "index_like_marker_count"
        ],
        row[
            "target_evidence_id"
        ],
    ),
)


cell30_candidate_shortlist = (
    cell30_candidate_rows[
        :10
    ]
)


cell30_candidate_search_status = (
    "READY_WITH_CANDIDATES"

    if cell30_candidate_shortlist

    else
    "RETARGET_RECORDED_CANDIDATE_POOL_NEEDS_EXPANSION"
)


# ============================================================
# 10) Persist RETARGET state
# ============================================================

cell30_persistent_registry_payload = copy.deepcopy(
    cell29_persisted_registry
)


cell30_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell30_retarget_workspace
)


cell30_persistent_registry_payload[
    "remaining_pilot_count"
] = 18


cell30_persistent_registry_payload[
    "continuation_state"
] = {
    "next_pilot_id":
        CELL30_PILOT_ID,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell29_remaining_pilot_ids
        ),

    "remaining_count":
        18,

    "continuation_state_sha256":
        cell29_continuation_state_sha256,

    "active_pilot_status":
        "BLOCKED_REQUIRES_RETARGET",
}


cell30_persistent_registry_payload[
    "retarget_candidate_discovery"
] = {
    "pilot_case_id":
        CELL30_PILOT_ID,

    "current_target_evidence_id":
        CELL30_CURRENT_TARGET_ID,

    "status":
        cell30_candidate_search_status,

    "candidate_count":
        len(
            cell30_candidate_rows
        ),

    "shortlist":
        copy.deepcopy(
            cell30_candidate_shortlist
        ),
}


cell30_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell30_persistent_registry_payload
    )
)


# ============================================================
# 11) Read back persisted Registry
# ============================================================

cell30_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell30_persisted_active_case = (
    cell30_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell30_persistence_valid = all([
    cell30_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL30_PILOT_ID,

    cell30_persisted_active_case.get(
        "workflow_state"
    )
    ==
    "BLOCKED_REQUIRES_RETARGET",

    cell30_persisted_active_case.get(
        "retarget_status",
        {},
    ).get(
        "decision"
    )
    ==
    "RETARGET",

    cell30_persisted_active_case.get(
        "retarget_status",
        {},
    ).get(
        "replacement_selected"
    )
    is False,

    cell30_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    18,
])


# ============================================================
# 12) Safety
# ============================================================

cell30_queue_still_18 = (
    len(
        cell29_remaining_pilot_ids
    )
    ==
    18
)


cell30_current_pilot_still_first = (
    cell29_remaining_pilot_ids[
        0
    ]
    ==
    CELL30_PILOT_ID
)


cell30_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


# ============================================================
# 13) Final Gate
# ============================================================

cell30_gate = all([
    len(
        cell30_failures
    )
    ==
    0,

    cell30_current_identity_valid,

    cell30_persistence_valid,

    cell30_queue_still_18,

    cell30_current_pilot_still_first,

    cell30_base_seed_unchanged,
])


# ============================================================
# 14) Diagnostics
# ============================================================

print(
    "=== HUMAN RETARGET + "
    "REPLACEMENT CANDIDATE DISCOVERY ==="
)


print(
    "\n=== CURRENT PILOT ==="
)

print(
    "Pilot:",
    CELL30_PILOT_ID,
)

print(
    "Current Target:",
    CELL30_CURRENT_TARGET_ID,
)

print(
    "Task Family:",
    CELL30_TASK_FAMILY,
)

print(
    "Question Archetype:",
    CELL30_QUESTION_ARCHETYPE,
)

print(
    "Stratum:",
    CELL30_STRATUM,
)

print(
    "Source Type:",
    CELL30_SOURCE_TYPE,
)


print(
    "\n=== HUMAN DECISION ==="
)

print(
    "Decision:",
    CELL30_RETARGET_DECISION,
)

print(
    "Pilot rejected:",
    False,
)

print(
    "Current Target rejected for Golden QA:",
    True,
)

print(
    "Coverage preserved:",
    True,
)

print(
    "Workflow state:",
    cell30_retarget_workspace[
        "workflow_state"
    ],
)


print(
    "\n=== RETARGET REASON ==="
)

print(
    CELL30_RETARGET_REASON
)


print(
    "\n=== CANDIDATE DISCOVERY ==="
)

print(
    "Search status:",
    cell30_candidate_search_status,
)

print(
    "Candidate count:",
    len(
        cell30_candidate_rows
    ),
)

print(
    "Shortlist count:",
    len(
        cell30_candidate_shortlist
    ),
)


for rank, row in enumerate(
    cell30_candidate_shortlist,
    start=1,
):

    print(
        "\n=============================="
    )

    print(
        "CANDIDATE",
        rank,
    )

    print(
        "Target:",
        row[
            "target_evidence_id"
        ],
    )

    print(
        "Compatibility score:",
        row[
            "compatibility_score"
        ],
    )

    print(
        "Task Family:",
        row[
            "task_family"
        ],
    )

    print(
        "Question Archetype:",
        row[
            "question_archetype"
        ],
    )

    print(
        "Stratum:",
        row[
            "stratum"
        ],
    )

    print(
        "Source Type:",
        row[
            "source_type"
        ],
    )

    print(
        "Index-like marker count:",
        row[
            "index_like_marker_count"
        ],
    )

    print(
        "\nTEXT:\n"
        +
        row[
            "combined_text"
        ]
    )


print(
    "\n=== PERSISTENCE / QUEUE ==="
)

print(
    "Registry synced:",
    cell30_persistence_valid,
)

print(
    "Queue still 18:",
    cell30_queue_still_18,
)

print(
    "Current Pilot still first:",
    cell30_current_pilot_still_first,
)

print(
    "Replacement selected:",
    False,
)

print(
    "Base Seed unchanged:",
    cell30_base_seed_unchanged,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell30_failures
)


print(
    "\nCODE CELL 30 GATE:",
    (
        "PASS"
        if cell30_gate
        else "FAIL"
    ),
)


assert cell30_gate, (
    "Human RETARGET registration / candidate discovery "
    "failed. Repair this SAME Code Cell 30 before selecting "
    "a replacement Target Evidence."
)

=== HUMAN RETARGET + REPLACEMENT CANDIDATE DISCOVERY ===

=== CURRENT PILOT ===
Pilot: question-pilot:3023c1c546df2fe2211f79702b6e3dbb
Current Target: target-evidence:cccb56b3cc1c8c0f2006c44010235fc4
Task Family: TINY_TABLE_EVIDENCE_RETRIEVAL
Question Archetype: STRUCTURE_AWARE_RETRIEVAL
Stratum: intrinsically_tiny_table_region
Source Type: book

=== HUMAN DECISION ===
Decision: RETARGET
Pilot rejected: False
Current Target rejected for Golden QA: True
Coverage preserved: True
Workflow state: BLOCKED_REQUIRES_RETARGET

=== RETARGET REASON ===
The current Target Evidence is index-like / title-list content. Although structurally valid as a tiny table region, it does not support a sufficiently natural, useful, user-facing Golden QA item. Preserve the Pilot coverage and replace only the Target Evidence with a more meaningful table region.

=== CANDIDATE DISCOVERY ===
Search status: RETARGET_RECORDED_CANDIDATE_POOL_NEEDS_EXPANSION
Candidate count: 0
Shortlist count: 0

=== PERSISTENCE / QUE

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 31 — Expanded Retarget Candidate Search

جست‌وجوی استاندارد Cell 30 در `target_evidence_records` هیچ Candidate آزاد و سازگاری پیدا نکرد.

بنابراین در این مرحله جست‌وجو به کل Support/Runtime موجود گسترش داده می‌شود تا Regionهای کتابی با ساختار جدولی یا شبه‌جدولی پیدا شوند.

دو سطح Candidate بررسی می‌شود:

- **STRICT_TABLE**: ساختار یا Metadata صریحاً نشانه‌ی Table دارد؛
- **TABLE_LIKE**: متن ساختار عددی/ردیفی دارد و می‌تواند Candidate اولیه برای بررسی انسانی باشد.

هیچ Candidate در این Cell فعال نمی‌شود.

Human Reviewer باید ابتدا متن Candidateها را ببیند و فقط در صورت مناسب بودن، Target Replacement در Cell بعدی ساخته شود.

Queue همچنان 18 باقی می‌ماند و Pilot فعلی همچنان `BLOCKED_REQUIRES_RETARGET` است.

</div>

In [31]:
# Code Cell 31

cell31_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell30_gate") is not True:
    cell31_failures.append(
        "CELL30_NOT_PASS"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell31_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Current blocked Pilot
# ============================================================

CELL31_PILOT_ID = (
    cell29_next_pilot_id
)

CELL31_REJECTED_TARGET_ID = (
    cell29_next_target_evidence_id
)


cell31_current_state_valid = all([
    CELL31_PILOT_ID
    ==
    cell30_retarget_workspace.get(
        "pilot_case_id"
    ),

    cell30_retarget_workspace.get(
        "workflow_state"
    )
    ==
    "BLOCKED_REQUIRES_RETARGET",

    cell30_retarget_workspace.get(
        "retarget_status",
        {},
    ).get(
        "replacement_selected"
    )
    is False,
])


# ============================================================
# 3) Generic recursive walk
# ============================================================

def cell31_walk(
    value,
    path="root",
):

    if isinstance(
        value,
        dict,
    ):

        yield path, value

        for key, nested in value.items():

            yield from cell31_walk(
                nested,
                (
                    path
                    +
                    "."
                    +
                    str(
                        key
                    )
                ),
            )


    elif isinstance(
        value,
        (
            list,
            tuple,
        ),
    ):

        for index, nested in enumerate(
            value
        ):

            yield from cell31_walk(
                nested,
                (
                    path
                    +
                    "["
                    +
                    str(
                        index
                    )
                    +
                    "]"
                ),
            )


# ============================================================
# 4) Text / metadata helpers
# ============================================================

CELL31_TEXT_FIELDS = (
    "text",
    "content",
    "target_text",
    "evidence_text",
    "text_content",
    "raw_text",
    "normalized_text",
)


CELL31_DOCUMENT_FIELDS = (
    "document_id",
    "source_document_id",
    "doc_id",
)


def cell31_first_text(
    mapping,
):

    if not isinstance(
        mapping,
        dict,
    ):
        return None


    for field_name in (
        CELL31_TEXT_FIELDS
    ):

        value = mapping.get(
            field_name
        )

        if (
            isinstance(
                value,
                str,
            )
            and
            value.strip()
        ):

            return value.strip()


    return None


def cell31_first_document_id(
    mapping,
):

    if not isinstance(
        mapping,
        dict,
    ):
        return None


    for field_name in (
        CELL31_DOCUMENT_FIELDS
    ):

        value = mapping.get(
            field_name
        )

        if isinstance(
            value,
            str,
        ):

            return value


    return None


def cell31_scalar_surface(
    mapping,
):

    values = []


    if not isinstance(
        mapping,
        dict,
    ):
        return ""


    for key, value in mapping.items():

        values.append(
            str(
                key
            )
        )


        if isinstance(
            value,
            (
                str,
                int,
                float,
                bool,
            ),
        ):

            values.append(
                str(
                    value
                )
            )


    return (
        " ".join(
            values
        )
        .lower()
    )


# ============================================================
# 5) Candidate scoring
# ============================================================

def cell31_score_candidate(
    mapping,
    path,
):

    text = cell31_first_text(
        mapping
    )


    if not text:
        return None


    document_id = (
        cell31_first_document_id(
            mapping
        )
    )


    surface = (
        cell31_scalar_surface(
            mapping
        )
    )


    normalized = (
        cell15_normalize_text(
            text
        )
    )


    word_count = len(
        normalized.split()
    )


    line_count = len([
        line

        for line
        in text.splitlines()

        if line.strip()
    ])


    number_count = len(
        re.findall(
            r"\d+",
            normalized,
        )
    )


    # Explicit structural table signal.
    strict_table_signal = any([
        "table"
        in
        surface,

        "جدول"
        in
        surface,

        "table_region"
        in
        surface,

        "table-region"
        in
        surface,

        "protected_table"
        in
        surface,
    ])


    # Book signal.
    book_signal = any([
        (
            isinstance(
                document_id,
                str,
            )
            and
            document_id.startswith(
                "book:"
            )
        ),

        "book:"
        in
        surface,

        "fa-source:p"
        in
        surface,
    ])


    # Table-like textual structure.
    table_like_signal = any([
        line_count >= 3
        and
        number_count >= 2,

        "\t"
        in
        text,

        "|"
        in
        text,

        (
            line_count >= 4
            and
            word_count >= 20
        ),
    ])


    # Index/title-list penalty.
    index_like_penalty = 0


    if (
        "عنوان"
        in
        normalized
    ):
        index_like_penalty += 2


    if (
        "فهرست"
        in
        normalized
    ):
        index_like_penalty += 3


    if (
        word_count
        <
        20
    ):
        index_like_penalty += 2


    if (
        line_count >= 4
        and
        word_count
        <
        35
    ):
        index_like_penalty += 1


    # Meaningfulness heuristic.
    meaningfulness_score = 0


    if word_count >= 25:
        meaningfulness_score += 2


    if word_count >= 50:
        meaningfulness_score += 1


    if number_count >= 2:
        meaningfulness_score += 1


    if line_count >= 3:
        meaningfulness_score += 1


    if any(
        term
        in
        normalized

        for term
        in (
            "هزینه",
            "درآمد",
            "پس انداز",
            "بودجه",
            "پول",
            "درصد",
            "ریال",
            "تومان",
            "سرمایه",
            "وام",
            "بدهی",
            "خرید",
            "قیمت",
            "سود",
        )
    ):
        meaningfulness_score += 2


    structural_score = 0


    if strict_table_signal:
        structural_score += 6


    if table_like_signal:
        structural_score += 3


    if book_signal:
        structural_score += 2


    total_score = (
        structural_score
        +
        meaningfulness_score
        -
        index_like_penalty
    )


    candidate_class = (
        "STRICT_TABLE"

        if strict_table_signal

        else (
            "TABLE_LIKE"

            if table_like_signal

            else None
        )
    )


    if candidate_class is None:
        return None


    return {
        "path":
            path,

        "document_id":
            document_id,

        "candidate_class":
            candidate_class,

        "strict_table_signal":
            strict_table_signal,

        "table_like_signal":
            table_like_signal,

        "book_signal":
            book_signal,

        "word_count":
            word_count,

        "line_count":
            line_count,

        "number_count":
            number_count,

        "index_like_penalty":
            index_like_penalty,

        "meaningfulness_score":
            meaningfulness_score,

        "structural_score":
            structural_score,

        "total_score":
            total_score,

        "text":
            text,

        "text_sha256":
            hashlib.sha256(
                text.encode(
                    "utf-8"
                )
            ).hexdigest(),

        "source_mapping":
            copy.deepcopy(
                mapping
            ),
    }


# ============================================================
# 6) Expanded search surfaces
# ============================================================

cell31_search_surfaces = {
    "active_support":
        cell13_active_support,

    "final_runtime":
        cell13_seed.get(
            "final_runtime",
            {},
        ),

    "completed_pilot":
        cell13_seed.get(
            "completed_pilot",
            {},
        ),
}


cell31_raw_candidates = []


for surface_name, surface_value in (
    cell31_search_surfaces.items()
):

    for path, mapping in cell31_walk(
        surface_value,
        path=surface_name,
    ):

        candidate = (
            cell31_score_candidate(
                mapping,
                path,
            )
        )


        if candidate is None:
            continue


        candidate[
            "surface"
        ] = surface_name


        cell31_raw_candidates.append(
            candidate
        )


# ============================================================
# 7) Remove duplicates / current rejected Evidence
# ============================================================

cell31_rejected_text_sha = (
    hashlib.sha256(
        cell29_next_combined_target_text.encode(
            "utf-8"
        )
    ).hexdigest()
)


cell31_unique_candidates_by_sha = {}


for candidate in (
    cell31_raw_candidates
):

    text_sha = candidate[
        "text_sha256"
    ]


    # Do not return the rejected text itself.
    if (
        text_sha
        ==
        cell31_rejected_text_sha
    ):
        continue


    existing = (
        cell31_unique_candidates_by_sha.get(
            text_sha
        )
    )


    if (
        existing is None
        or
        candidate[
            "total_score"
        ]
        >
        existing[
            "total_score"
        ]
    ):

        cell31_unique_candidates_by_sha[
            text_sha
        ] = candidate


cell31_unique_candidates = list(
    cell31_unique_candidates_by_sha.values()
)


# ============================================================
# 8) Prefer book-backed Candidates
# ============================================================

cell31_book_candidates = [
    candidate

    for candidate
    in cell31_unique_candidates

    if candidate[
        "book_signal"
    ]
]


cell31_rank_pool = (
    cell31_book_candidates

    if cell31_book_candidates

    else cell31_unique_candidates
)


cell31_ranked_candidates = sorted(
    cell31_rank_pool,
    key=lambda row: (
        0
        if row[
            "candidate_class"
        ]
        ==
        "STRICT_TABLE"
        else 1,

        -row[
            "total_score"
        ],

        row[
            "index_like_penalty"
        ],

        -row[
            "word_count"
        ],

        row[
            "text_sha256"
        ],
    ),
)


# ============================================================
# 9) Human-review shortlist
# ============================================================

cell31_candidate_shortlist = (
    cell31_ranked_candidates[
        :12
    ]
)


if cell31_candidate_shortlist:

    cell31_search_status = (
        "EXPANDED_CANDIDATES_READY_FOR_HUMAN_REVIEW"
    )

else:

    cell31_search_status = (
        "NO_REPLACEMENT_FOUND_IN_COMPACT_RUNTIME_"
        "SOURCE_LEVEL_EXPANSION_REQUIRED"
    )


# ============================================================
# 10) Build safe persistent shortlist
#
# Do not persist full source mappings.
# ============================================================

cell31_persistent_shortlist = []


for rank, candidate in enumerate(
    cell31_candidate_shortlist,
    start=1,
):

    cell31_persistent_shortlist.append({
        "rank":
            rank,

        "candidate_class":
            candidate[
                "candidate_class"
            ],

        "document_id":
            candidate[
                "document_id"
            ],

        "path":
            candidate[
                "path"
            ],

        "total_score":
            candidate[
                "total_score"
            ],

        "word_count":
            candidate[
                "word_count"
            ],

        "line_count":
            candidate[
                "line_count"
            ],

        "number_count":
            candidate[
                "number_count"
            ],

        "index_like_penalty":
            candidate[
                "index_like_penalty"
            ],

        "text":
            candidate[
                "text"
            ],

        "text_sha256":
            candidate[
                "text_sha256"
            ],
    })


# ============================================================
# 11) Update Registry discovery state
# ============================================================

cell31_persistent_registry_payload = copy.deepcopy(
    cell30_persisted_registry
)


cell31_persistent_registry_payload[
    "retarget_candidate_discovery"
] = {
    "pilot_case_id":
        CELL31_PILOT_ID,

    "rejected_target_evidence_id":
        CELL31_REJECTED_TARGET_ID,

    "search_scope":
        (
            "EXPANDED_COMPACT_SUPPORT_AND_RUNTIME"
        ),

    "status":
        cell31_search_status,

    "raw_candidate_count":
        len(
            cell31_raw_candidates
        ),

    "unique_candidate_count":
        len(
            cell31_unique_candidates
        ),

    "book_candidate_count":
        len(
            cell31_book_candidates
        ),

    "shortlist_count":
        len(
            cell31_persistent_shortlist
        ),

    "shortlist":
        copy.deepcopy(
            cell31_persistent_shortlist
        ),

    "replacement_selected":
        False,
}


cell31_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell31_persistent_registry_payload
    )
)


# ============================================================
# 12) Read-back validation
# ============================================================

cell31_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell31_persisted_discovery = (
    cell31_persisted_registry.get(
        "retarget_candidate_discovery"
    )
    or
    {}
)


cell31_persisted_active_case = (
    cell31_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell31_persistence_valid = all([
    cell31_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL31_PILOT_ID,

    cell31_persisted_active_case.get(
        "workflow_state"
    )
    ==
    "BLOCKED_REQUIRES_RETARGET",

    cell31_persisted_discovery.get(
        "replacement_selected"
    )
    is False,

    cell31_persisted_discovery.get(
        "status"
    )
    ==
    cell31_search_status,

    cell31_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    18,
])


# ============================================================
# 13) Queue / safety
# ============================================================

cell31_queue_still_18 = (
    len(
        cell29_remaining_pilot_ids
    )
    ==
    18
)


cell31_current_pilot_still_first = (
    cell29_remaining_pilot_ids[
        0
    ]
    ==
    CELL31_PILOT_ID
)


cell31_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


# ============================================================
# 14) Final Gate
# ============================================================

cell31_gate = all([
    len(
        cell31_failures
    )
    ==
    0,

    cell31_current_state_valid,

    cell31_persistence_valid,

    cell31_queue_still_18,

    cell31_current_pilot_still_first,

    cell31_base_seed_unchanged,
])


# ============================================================
# 15) Diagnostics
# ============================================================

print(
    "=== EXPANDED RETARGET CANDIDATE SEARCH ==="
)


print(
    "\n=== CURRENT BLOCKED PILOT ==="
)

print(
    "Pilot:",
    CELL31_PILOT_ID,
)

print(
    "Rejected Target:",
    CELL31_REJECTED_TARGET_ID,
)

print(
    "Workflow:",
    "BLOCKED_REQUIRES_RETARGET",
)


print(
    "\n=== SEARCH RESULT ==="
)

print(
    "Status:",
    cell31_search_status,
)

print(
    "Raw Candidates:",
    len(
        cell31_raw_candidates
    ),
)

print(
    "Unique Candidates:",
    len(
        cell31_unique_candidates
    ),
)

print(
    "Book Candidates:",
    len(
        cell31_book_candidates
    ),
)

print(
    "Human-review Shortlist:",
    len(
        cell31_candidate_shortlist
    ),
)


for rank, candidate in enumerate(
    cell31_candidate_shortlist,
    start=1,
):

    print(
        "\n========================================"
    )

    print(
        "CANDIDATE",
        rank,
    )

    print(
        "Class:",
        candidate[
            "candidate_class"
        ],
    )

    print(
        "Score:",
        candidate[
            "total_score"
        ],
    )

    print(
        "Document:",
        candidate[
            "document_id"
        ],
    )

    print(
        "Path:",
        candidate[
            "path"
        ],
    )

    print(
        "Words:",
        candidate[
            "word_count"
        ],
    )

    print(
        "Lines:",
        candidate[
            "line_count"
        ],
    )

    print(
        "Numbers:",
        candidate[
            "number_count"
        ],
    )

    print(
        "Index-like penalty:",
        candidate[
            "index_like_penalty"
        ],
    )

    print(
        "\nTEXT:\n"
        +
        candidate[
            "text"
        ]
    )


print(
    "\n=== RETARGET SAFETY ==="
)

print(
    "Replacement selected:",
    False,
)

print(
    "Queue still 18:",
    cell31_queue_still_18,
)

print(
    "Current Pilot still first:",
    cell31_current_pilot_still_first,
)

print(
    "Registry synced:",
    cell31_persistence_valid,
)

print(
    "Base Seed unchanged:",
    cell31_base_seed_unchanged,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell31_failures
)


print(
    "\nCODE CELL 31 GATE:",
    (
        "PASS"
        if cell31_gate
        else "FAIL"
    ),
)


assert cell31_gate, (
    "Expanded Retarget candidate discovery failed. "
    "Repair this SAME Code Cell 31 before selecting "
    "a replacement Target."
)

=== EXPANDED RETARGET CANDIDATE SEARCH ===

=== CURRENT BLOCKED PILOT ===
Pilot: question-pilot:3023c1c546df2fe2211f79702b6e3dbb
Rejected Target: target-evidence:cccb56b3cc1c8c0f2006c44010235fc4
Workflow: BLOCKED_REQUIRES_RETARGET

=== SEARCH RESULT ===
Status: EXPANDED_CANDIDATES_READY_FOR_HUMAN_REVIEW
Raw Candidates: 92
Unique Candidates: 46
Book Candidates: 42
Human-review Shortlist: 12

CANDIDATE 1
Class: STRICT_TABLE
Score: 8
Document: book:adult-financial-literacy-course:fa-source:p000641
Path: final_runtime.question_drafting_pilot_packets[7].target_text_parts[18]
Words: 12
Lines: 1
Numbers: 0
Index-like penalty: 2

TEXT:
حقوق ماهیانهام با توجه به بودجه و امکانات سازمانم، سهم عادلانهای است.

CANDIDATE 2
Class: STRICT_TABLE
Score: 8
Document: book:adult-financial-literacy-course:fa-source:p000627
Path: final_runtime.question_drafting_pilot_packets[7].target_text_parts[4]
Words: 9
Lines: 1
Numbers: 0
Index-like penalty: 2

TEXT:
کار من کاری مفید و سودمند برای جامعه است.

CANDIDATE 

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 32 — Source-Level Retarget Discovery

Candidateهای Cell 31 به‌عنوان Replacement انتخاب نمی‌شوند، زیرا آن‌ها Fragmentهای منفرد از Target Parts مربوط به Pilotهای فعال دیگر هستند و یک Table Region مستقل و آزاد محسوب نمی‌شوند.

در این مرحله Retarget Discovery از Compact Runtime به Source/Corpus واقعی Repository گسترش پیدا می‌کند.

قواعد:

- همان کتابِ Target فعلی در اولویت است؛
- Evidence مصرف‌شده توسط Pilotهای فعال دیگر کنار گذاشته می‌شود؛
- Target ردشده دوباره پیشنهاد نمی‌شود؛
- Candidate باید نشانه‌ای از ساختار Table یا Table-like داشته باشد؛
- معنادار بودن برای سؤال کاربر امتیاز مثبت می‌گیرد؛
- هیچ Candidate به‌طور خودکار انتخاب یا فعال نمی‌شود؛
- این مرحله فقط Read از Source Corpus و Sync وضعیت Discovery در Registry انجام می‌دهد.

Pilot همچنان `BLOCKED_REQUIRES_RETARGET` و Queue همچنان 18 است.

</div>

In [32]:
# Code Cell 32

cell32_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell31_gate") is not True:
    cell32_failures.append(
        "CELL31_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell21_repo_root"
    ),
    Path,
):
    cell32_failures.append(
        "REPO_ROOT_MISSING"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell32_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Current blocked Pilot
# ============================================================

CELL32_PILOT_ID = (
    cell29_next_pilot_id
)

CELL32_REJECTED_TARGET_ID = (
    cell29_next_target_evidence_id
)


cell32_current_state_valid = all([
    CELL32_PILOT_ID
    ==
    cell30_retarget_workspace.get(
        "pilot_case_id"
    ),

    cell30_retarget_workspace.get(
        "workflow_state"
    )
    ==
    "BLOCKED_REQUIRES_RETARGET",
])


# ============================================================
# 3) Record why Cell 31 Candidates are not selectable
# ============================================================

cell32_cell31_disposition = {
    "decision":
        "DO_NOT_SELECT",

    "reason":
        (
            "Cell 31 Candidates are target_text_parts "
            "inside active Pilot packets rather than unused "
            "standalone source-level table regions."
        ),

    "candidate_count":
        len(
            cell31_candidate_shortlist
        ),

    "replacement_selected":
        False,
}


# ============================================================
# 4) Determine current source-book identity
# ============================================================

cell32_current_document_ids = [
    row.get(
        "document_id"
    )

    for row
    in cell29_next_target_text_parts

    if (
        isinstance(
            row,
            dict,
        )
        and
        isinstance(
            row.get(
                "document_id"
            ),
            str,
        )
    )
]


def cell32_book_root(
    document_id,
):

    if not isinstance(
        document_id,
        str,
    ):
        return None


    match = re.match(
        r"^(book:.+?:fa-source):p\d+$",
        document_id,
    )


    if match:

        return match.group(
            1
        )


    if document_id.startswith(
        "book:"
    ):

        return re.sub(
            r":p\d+$",
            "",
            document_id,
        )


    return None


cell32_book_roots = sorted({
    root

    for root
    in (
        cell32_book_root(
            document_id
        )

        for document_id
        in cell32_current_document_ids
    )

    if isinstance(
        root,
        str,
    )
})


cell32_book_identity_valid = (
    len(
        cell32_book_roots
    )
    >=
    1
)


# ============================================================
# 5) Build active-document exclusion set
# ============================================================

cell32_active_document_ids = set()


for packet in (
    cell13_seed[
        "final_runtime"
    ][
        "question_drafting_pilot_packets"
    ]
):

    if not isinstance(
        packet,
        dict,
    ):
        continue


    parts = packet.get(
        "target_text_parts",
        [],
    )


    if not isinstance(
        parts,
        list,
    ):
        continue


    for part in parts:

        if not isinstance(
            part,
            dict,
        ):
            continue


        document_id = part.get(
            "document_id"
        )


        if isinstance(
            document_id,
            str,
        ):

            cell32_active_document_ids.add(
                document_id
            )


cell32_rejected_document_ids = set(
    cell32_current_document_ids
)


# ============================================================
# 6) Source data search roots
# ============================================================

cell32_search_roots = []


for relative_path in (
    "data",
    "artifacts",
    "sources",
):

    candidate = (
        cell21_repo_root
        /
        relative_path
    )


    if candidate.exists():

        cell32_search_roots.append(
            candidate
        )


# Always retain repository root as last-resort discovery scope.
if not cell32_search_roots:

    cell32_search_roots = [
        cell21_repo_root
    ]


# ============================================================
# 7) Find files containing current book identity
# ============================================================

CELL32_ALLOWED_SUFFIXES = {
    ".json",
    ".jsonl",
    ".ndjson",
    ".txt",
    ".md",
    ".csv",
    ".tsv",
}


CELL32_MAX_FILE_BYTES = (
    25
    *
    1024
    *
    1024
)


cell32_matching_files = set()


def cell32_rg_files(
    root,
    pattern,
):

    try:

        completed = subprocess.run(
            [
                "rg",
                "-l",
                "-F",
                pattern,
                str(
                    root
                ),
            ],
            capture_output=True,
            text=True,
            timeout=30,
            check=False,
        )


        if completed.returncode not in (
            0,
            1,
        ):

            return []


        return [
            Path(
                line.strip()
            ).resolve()

            for line
            in completed.stdout.splitlines()

            if line.strip()
        ]


    except Exception:

        return []


for root in cell32_search_roots:

    for book_root in cell32_book_roots:

        for path in cell32_rg_files(
            root,
            book_root,
        ):

            if (
                path.suffix.lower()
                in
                CELL32_ALLOWED_SUFFIXES
            ):

                try:

                    if (
                        path.stat().st_size
                        <=
                        CELL32_MAX_FILE_BYTES
                    ):

                        cell32_matching_files.add(
                            path
                        )

                except Exception:

                    pass


# ============================================================
# 8) Python fallback if ripgrep finds nothing
# ============================================================

if (
    not cell32_matching_files
    and
    cell32_book_roots
):

    for root in cell32_search_roots:

        scanned_count = 0


        for path in root.rglob(
            "*"
        ):

            if scanned_count >= 500:
                break


            if (
                not path.is_file()
                or
                path.suffix.lower()
                not in
                CELL32_ALLOWED_SUFFIXES
            ):
                continue


            scanned_count += 1


            try:

                if (
                    path.stat().st_size
                    >
                    CELL32_MAX_FILE_BYTES
                ):
                    continue


                text = path.read_text(
                    encoding="utf-8",
                    errors="ignore",
                )


                if any(
                    root_id
                    in
                    text

                    for root_id
                    in cell32_book_roots
                ):

                    cell32_matching_files.add(
                        path.resolve()
                    )


            except Exception:

                pass


cell32_matching_files = sorted(
    cell32_matching_files
)


# ============================================================
# 9) Recursive mapping helpers
# ============================================================

def cell32_walk(
    value,
    path="root",
):

    if isinstance(
        value,
        dict,
    ):

        yield path, value


        for key, nested in value.items():

            yield from cell32_walk(
                nested,
                (
                    path
                    +
                    "."
                    +
                    str(
                        key
                    )
                ),
            )


    elif isinstance(
        value,
        (
            list,
            tuple,
        ),
    ):

        for index, nested in enumerate(
            value
        ):

            yield from cell32_walk(
                nested,
                (
                    path
                    +
                    "["
                    +
                    str(
                        index
                    )
                    +
                    "]"
                ),
            )


def cell32_first_text(
    mapping,
):

    for field_name in (
        "text",
        "content",
        "target_text",
        "evidence_text",
        "text_content",
        "raw_text",
        "normalized_text",
    ):

        value = mapping.get(
            field_name
        )


        if (
            isinstance(
                value,
                str,
            )
            and
            value.strip()
        ):

            return value.strip()


    return None


def cell32_first_document_id(
    mapping,
):

    for field_name in (
        "document_id",
        "source_document_id",
        "doc_id",
        "id",
    ):

        value = mapping.get(
            field_name
        )


        if (
            isinstance(
                value,
                str,
            )
            and
            value.startswith(
                "book:"
            )
        ):

            return value


    return None


def cell32_mapping_surface(
    mapping,
):

    items = []


    for key, value in mapping.items():

        items.append(
            str(
                key
            )
        )


        if isinstance(
            value,
            (
                str,
                int,
                float,
                bool,
            ),
        ):

            items.append(
                str(
                    value
                )
            )


    return (
        " ".join(
            items
        )
        .lower()
    )


# ============================================================
# 10) Parse candidate source files
# ============================================================

cell32_source_records = []


def cell32_add_json_object(
    value,
    file_path,
):

    for object_path, mapping in cell32_walk(
        value,
        path="json",
    ):

        document_id = (
            cell32_first_document_id(
                mapping
            )
        )


        text = cell32_first_text(
            mapping
        )


        if (
            not isinstance(
                document_id,
                str,
            )
            or
            not isinstance(
                text,
                str,
            )
        ):
            continue


        book_root = cell32_book_root(
            document_id
        )


        if (
            cell32_book_roots
            and
            book_root
            not in
            cell32_book_roots
        ):
            continue


        cell32_source_records.append({
            "file_path":
                str(
                    file_path
                ),

            "object_path":
                object_path,

            "document_id":
                document_id,

            "text":
                text,

            "metadata_surface":
                cell32_mapping_surface(
                    mapping
                ),

            "source_mapping":
                copy.deepcopy(
                    mapping
                ),
        })


for path in cell32_matching_files:

    suffix = path.suffix.lower()


    try:

        if suffix == ".json":

            value = json.loads(
                path.read_text(
                    encoding="utf-8"
                )
            )


            cell32_add_json_object(
                value,
                path,
            )


        elif suffix in (
            ".jsonl",
            ".ndjson",
        ):

            with path.open(
                "r",
                encoding="utf-8",
            ) as handle:

                for line_number, line in enumerate(
                    handle,
                    start=1,
                ):

                    line = line.strip()


                    if not line:
                        continue


                    try:

                        value = json.loads(
                            line
                        )

                    except Exception:

                        continue


                    before_count = len(
                        cell32_source_records
                    )


                    cell32_add_json_object(
                        value,
                        path,
                    )


                    for row in (
                        cell32_source_records[
                            before_count:
                        ]
                    ):

                        row[
                            "object_path"
                        ] = (
                            "jsonl_line:"
                            +
                            str(
                                line_number
                            )
                            +
                            ":"
                            +
                            row[
                                "object_path"
                            ]
                        )


    except Exception as exc:

        cell32_failures.append({
            "reason":
                "SOURCE_FILE_PARSE_WARNING",

            "file":
                str(
                    path
                ),

            "error":
                repr(
                    exc
                ),
        })


# Parse warnings should not fail discovery.
cell32_hard_failures = [
    failure

    for failure
    in cell32_failures

    if not (
        isinstance(
            failure,
            dict,
        )
        and
        failure.get(
            "reason"
        )
        ==
        "SOURCE_FILE_PARSE_WARNING"
    )
]


# ============================================================
# 11) Score source-level Table Candidates
# ============================================================

cell32_source_candidates = []


for record in cell32_source_records:

    document_id = record[
        "document_id"
    ]


    text = record[
        "text"
    ]


    # Exclude the rejected Evidence.
    if document_id in cell32_rejected_document_ids:
        continue


    # Exclude material already used by any active Pilot.
    if document_id in cell32_active_document_ids:
        continue


    normalized = cell15_normalize_text(
        text
    )


    word_count = len(
        normalized.split()
    )


    line_count = len([
        line

        for line
        in text.splitlines()

        if line.strip()
    ])


    number_count = len(
        re.findall(
            r"\d+",
            normalized,
        )
    )


    surface = record[
        "metadata_surface"
    ]


    strict_table_signal = any([
        "table"
        in
        surface,

        "جدول"
        in
        surface,

        "table_region"
        in
        surface,

        "table-row"
        in
        surface,

        "table_row"
        in
        surface,
    ])


    table_like_signal = any([
        (
            line_count >= 3
            and
            number_count >= 2
        ),

        "\t"
        in
        text,

        "|"
        in
        text,
    ])


    if not (
        strict_table_signal
        or
        table_like_signal
    ):
        continue


    financial_terms = [
        term

        for term
        in (
            "پول",
            "درآمد",
            "هزینه",
            "پس انداز",
            "بودجه",
            "قیمت",
            "خرید",
            "سرمایه",
            "سود",
            "بدهی",
            "وام",
            "درصد",
            "ریال",
            "تومان",
            "حقوق",
        )

        if term
        in
        normalized
    ]


    index_like_penalty = 0


    if "فهرست" in normalized:
        index_like_penalty += 4


    if (
        "عنوان"
        in
        normalized
        and
        word_count < 45
    ):
        index_like_penalty += 2


    if word_count < 15:
        index_like_penalty += 2


    score = 0


    if strict_table_signal:
        score += 5


    if table_like_signal:
        score += 3


    if word_count >= 20:
        score += 2


    if word_count >= 40:
        score += 1


    if line_count >= 3:
        score += 1


    if number_count >= 2:
        score += 1


    score += min(
        len(
            financial_terms
        ),
        3,
    )


    score -= index_like_penalty


    cell32_source_candidates.append({
        "document_id":
            document_id,

        "file_path":
            record[
                "file_path"
            ],

        "object_path":
            record[
                "object_path"
            ],

        "candidate_class":
            (
                "STRICT_TABLE"
                if strict_table_signal
                else "TABLE_LIKE"
            ),

        "score":
            score,

        "word_count":
            word_count,

        "line_count":
            line_count,

        "number_count":
            number_count,

        "financial_terms":
            financial_terms,

        "index_like_penalty":
            index_like_penalty,

        "text":
            text,

        "text_sha256":
            hashlib.sha256(
                text.encode(
                    "utf-8"
                )
            ).hexdigest(),

        "source_mapping":
            record[
                "source_mapping"
            ],
    })


# ============================================================
# 12) Deduplicate and rank
# ============================================================

cell32_candidates_by_sha = {}


for candidate in (
    cell32_source_candidates
):

    text_sha = candidate[
        "text_sha256"
    ]


    existing = (
        cell32_candidates_by_sha.get(
            text_sha
        )
    )


    if (
        existing is None
        or
        candidate[
            "score"
        ]
        >
        existing[
            "score"
        ]
    ):

        cell32_candidates_by_sha[
            text_sha
        ] = candidate


cell32_unique_source_candidates = list(
    cell32_candidates_by_sha.values()
)


cell32_ranked_candidates = sorted(
    cell32_unique_source_candidates,
    key=lambda row: (
        0
        if row[
            "candidate_class"
        ]
        ==
        "STRICT_TABLE"
        else 1,

        -row[
            "score"
        ],

        row[
            "index_like_penalty"
        ],

        -row[
            "word_count"
        ],

        row[
            "document_id"
        ],
    ),
)


cell32_candidate_shortlist = (
    cell32_ranked_candidates[
        :12
    ]
)


cell32_search_status = (
    "SOURCE_LEVEL_CANDIDATES_READY_FOR_HUMAN_REVIEW"

    if cell32_candidate_shortlist

    else
    "NO_UNUSED_SOURCE_LEVEL_TABLE_CANDIDATE_FOUND"
)


# ============================================================
# 13) Safe Registry representation
# ============================================================

cell32_persistent_shortlist = []


for rank, candidate in enumerate(
    cell32_candidate_shortlist,
    start=1,
):

    cell32_persistent_shortlist.append({
        "rank":
            rank,

        "document_id":
            candidate[
                "document_id"
            ],

        "candidate_class":
            candidate[
                "candidate_class"
            ],

        "score":
            candidate[
                "score"
            ],

        "word_count":
            candidate[
                "word_count"
            ],

        "line_count":
            candidate[
                "line_count"
            ],

        "number_count":
            candidate[
                "number_count"
            ],

        "financial_terms":
            copy.deepcopy(
                candidate[
                    "financial_terms"
                ]
            ),

        "index_like_penalty":
            candidate[
                "index_like_penalty"
            ],

        "file_path":
            candidate[
                "file_path"
            ],

        "object_path":
            candidate[
                "object_path"
            ],

        "text":
            candidate[
                "text"
            ],

        "text_sha256":
            candidate[
                "text_sha256"
            ],
    })


# ============================================================
# 14) Persist discovery result
# ============================================================

cell32_persistent_registry_payload = copy.deepcopy(
    cell31_persisted_registry
)


cell32_persistent_registry_payload[
    "retarget_candidate_discovery"
] = {
    "pilot_case_id":
        CELL32_PILOT_ID,

    "rejected_target_evidence_id":
        CELL32_REJECTED_TARGET_ID,

    "cell31_candidates_disposition":
        copy.deepcopy(
            cell32_cell31_disposition
        ),

    "search_scope":
        "SOURCE_CORPUS_SAME_BOOK",

    "book_roots":
        copy.deepcopy(
            cell32_book_roots
        ),

    "matching_source_file_count":
        len(
            cell32_matching_files
        ),

    "source_record_count":
        len(
            cell32_source_records
        ),

    "unused_table_candidate_count":
        len(
            cell32_unique_source_candidates
        ),

    "status":
        cell32_search_status,

    "shortlist":
        copy.deepcopy(
            cell32_persistent_shortlist
        ),

    "replacement_selected":
        False,
}


cell32_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell32_persistent_registry_payload
    )
)


# ============================================================
# 15) Read-back validation
# ============================================================

cell32_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell32_persisted_discovery = (
    cell32_persisted_registry.get(
        "retarget_candidate_discovery"
    )
    or
    {}
)


cell32_persistence_valid = all([
    cell32_persisted_registry.get(
        "active_workspace",
        {},
    ).get(
        "workflow_state"
    )
    ==
    "BLOCKED_REQUIRES_RETARGET",

    cell32_persisted_discovery.get(
        "search_scope"
    )
    ==
    "SOURCE_CORPUS_SAME_BOOK",

    cell32_persisted_discovery.get(
        "status"
    )
    ==
    cell32_search_status,

    cell32_persisted_discovery.get(
        "replacement_selected"
    )
    is False,

    cell32_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    18,
])


# ============================================================
# 16) Safety
# ============================================================

cell32_queue_still_18 = (
    len(
        cell29_remaining_pilot_ids
    )
    ==
    18
)


cell32_current_pilot_still_first = (
    cell29_remaining_pilot_ids[
        0
    ]
    ==
    CELL32_PILOT_ID
)


cell32_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


# ============================================================
# 17) Final Gate
# ============================================================

cell32_gate = all([
    len(
        cell32_hard_failures
    )
    ==
    0,

    cell32_current_state_valid,

    cell32_book_identity_valid,

    cell32_persistence_valid,

    cell32_queue_still_18,

    cell32_current_pilot_still_first,

    cell32_base_seed_unchanged,
])


# ============================================================
# 18) Diagnostics
# ============================================================

print(
    "=== SOURCE-LEVEL RETARGET DISCOVERY ==="
)


print(
    "\n=== CURRENT PILOT ==="
)

print(
    "Pilot:",
    CELL32_PILOT_ID,
)

print(
    "Rejected Target:",
    CELL32_REJECTED_TARGET_ID,
)

print(
    "Book roots:",
    cell32_book_roots,
)


print(
    "\n=== CELL 31 DISPOSITION ==="
)

print(
    "Candidates:",
    cell32_cell31_disposition[
        "candidate_count"
    ],
)

print(
    "Decision:",
    cell32_cell31_disposition[
        "decision"
    ],
)

print(
    "Reason:",
    cell32_cell31_disposition[
        "reason"
    ],
)


print(
    "\n=== SOURCE DISCOVERY ==="
)

print(
    "Matching source files:",
    len(
        cell32_matching_files
    ),
)

for path in cell32_matching_files[
    :20
]:

    print(
        "-",
        path,
    )


print(
    "Source records:",
    len(
        cell32_source_records
    ),
)

print(
    "Unused table candidates:",
    len(
        cell32_unique_source_candidates
    ),
)

print(
    "Status:",
    cell32_search_status,
)


print(
    "\n=== HUMAN-REVIEW SHORTLIST ==="
)


for rank, candidate in enumerate(
    cell32_candidate_shortlist,
    start=1,
):

    print(
        "\n========================================"
    )

    print(
        "CANDIDATE",
        rank,
    )

    print(
        "Class:",
        candidate[
            "candidate_class"
        ],
    )

    print(
        "Score:",
        candidate[
            "score"
        ],
    )

    print(
        "Document:",
        candidate[
            "document_id"
        ],
    )

    print(
        "Financial terms:",
        candidate[
            "financial_terms"
        ],
    )

    print(
        "Words:",
        candidate[
            "word_count"
        ],
    )

    print(
        "Lines:",
        candidate[
            "line_count"
        ],
    )

    print(
        "Numbers:",
        candidate[
            "number_count"
        ],
    )

    print(
        "Index-like penalty:",
        candidate[
            "index_like_penalty"
        ],
    )

    print(
        "\nTEXT:\n"
        +
        candidate[
            "text"
        ]
    )


print(
    "\n=== SAFETY ==="
)

print(
    "Replacement selected:",
    False,
)

print(
    "Queue still 18:",
    cell32_queue_still_18,
)

print(
    "Current Pilot still first:",
    cell32_current_pilot_still_first,
)

print(
    "Registry synced:",
    cell32_persistence_valid,
)

print(
    "Base Seed unchanged:",
    cell32_base_seed_unchanged,
)

print(
    "Source files modified:",
    False,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== WARNINGS ==="
)

print(
    [
        failure

        for failure
        in cell32_failures

        if isinstance(
            failure,
            dict,
        )
    ]
)


print(
    "\n=== HARD FAILURES ==="
)

print(
    cell32_hard_failures
)


print(
    "\nCODE CELL 32 GATE:",
    (
        "PASS"
        if cell32_gate
        else "FAIL"
    ),
)


assert cell32_gate, (
    "Source-level Retarget discovery failed. "
    "Repair this SAME Code Cell 32 before selecting "
    "a replacement Target."
)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


=== SOURCE-LEVEL RETARGET DISCOVERY ===

=== CURRENT PILOT ===
Pilot: question-pilot:3023c1c546df2fe2211f79702b6e3dbb
Rejected Target: target-evidence:cccb56b3cc1c8c0f2006c44010235fc4
Book roots: ['book:trainer-training-financial-literacy:fa-source']

=== CELL 31 DISPOSITION ===
Candidates: 12
Decision: DO_NOT_SELECT
Reason: Cell 31 Candidates are target_text_parts inside active Pilot packets rather than unused standalone source-level table regions.

=== SOURCE DISCOVERY ===
Matching source files: 10
- /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/taxonomy/taxonomy_v1.json
- /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/data/phase2/human_review/human_review_registry.json
- /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/data/phase2/interim/book_ingested_v1.json
- /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/data/phase2/interim/book_ingested_v2.json
- /Users/ma

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 33 — Cross-Book Retarget Discovery

جست‌وجوی Source-Level در همان کتاب هیچ Table Region آزاد و مناسب پیدا نکرد.

بنابراین Retarget به کتاب‌های دیگرِ corpus گسترش داده می‌شود، اما نوع آزمون تغییر نمی‌کند:

- Task Family همچنان `TINY_TABLE_EVIDENCE_RETRIEVAL` است؛
- Question Archetype همچنان `STRUCTURE_AWARE_RETRIEVAL` است؛
- Target ردشده دوباره استفاده نمی‌شود؛
- Documentهای مصرف‌شده توسط Pilotهای دیگر کنار گذاشته می‌شوند؛
- کتاب فعلی از جست‌وجوی این مرحله کنار گذاشته می‌شود؛
- Page Regionهای منفرد و پنجره‌های دوصفحۀ مجاور بررسی می‌شوند؛
- Candidateها فقط برای Human Review رتبه‌بندی می‌شوند؛
- هیچ Target جدیدی در این Cell ساخته یا فعال نمی‌شود.

Queue همچنان 18 باقی می‌ماند و Pilot فعلی تا انتخاب Human Reviewer در وضعیت `BLOCKED_REQUIRES_RETARGET` می‌ماند.

</div>

In [33]:
# Code Cell 33

cell33_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell32_gate") is not True:
    cell33_failures.append(
        "CELL32_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell21_repo_root"
    ),
    Path,
):
    cell33_failures.append(
        "REPO_ROOT_MISSING"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell33_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Current blocked Pilot
# ============================================================

CELL33_PILOT_ID = (
    cell29_next_pilot_id
)

CELL33_REJECTED_TARGET_ID = (
    cell29_next_target_evidence_id
)

CELL33_TASK_FAMILY = (
    "TINY_TABLE_EVIDENCE_RETRIEVAL"
)

CELL33_QUESTION_ARCHETYPE = (
    "STRUCTURE_AWARE_RETRIEVAL"
)


cell33_current_state_valid = all([
    CELL33_PILOT_ID
    ==
    cell30_retarget_workspace.get(
        "pilot_case_id"
    ),

    cell30_retarget_workspace.get(
        "workflow_state"
    )
    ==
    "BLOCKED_REQUIRES_RETARGET",

    cell32_persisted_discovery.get(
        "status"
    )
    ==
    "NO_UNUSED_SOURCE_LEVEL_TABLE_CANDIDATE_FOUND",
])


# ============================================================
# 3) Current book exclusion
# ============================================================

cell33_excluded_book_roots = set(
    cell32_book_roots
)


cell33_active_document_ids = set(
    cell32_active_document_ids
)


cell33_rejected_document_ids = set(
    cell32_rejected_document_ids
)


# ============================================================
# 4) Canonical book-ingestion files
#
# Prefer v2. v1 is fallback/additional coverage.
# No subprocess is used.
# ============================================================

cell33_candidate_files = []


for relative_path in (
    "data/phase2/interim/book_ingested_v2.json",
    "data/phase2/interim/book_ingested_v1.json",
):

    path = (
        cell21_repo_root
        /
        relative_path
    )


    if path.exists():
        cell33_candidate_files.append(
            path
        )


if not cell33_candidate_files:
    cell33_failures.append(
        "CANONICAL_BOOK_INGESTION_FILES_NOT_FOUND"
    )


# ============================================================
# 5) Generic helpers
# ============================================================

def cell33_book_root(
    document_id,
):

    if not isinstance(
        document_id,
        str,
    ):
        return None


    match = re.match(
        r"^(book:.+?:fa-source):p\d+$",
        document_id,
    )


    if match:
        return match.group(
            1
        )


    return None


def cell33_page_number(
    document_id,
):

    if not isinstance(
        document_id,
        str,
    ):
        return None


    match = re.search(
        r":p(\d+)$",
        document_id,
    )


    if not match:
        return None


    return int(
        match.group(
            1
        )
    )


def cell33_walk(
    value,
    path="root",
):

    if isinstance(
        value,
        dict,
    ):

        yield path, value


        for key, nested in value.items():

            yield from cell33_walk(
                nested,
                (
                    path
                    +
                    "."
                    +
                    str(
                        key
                    )
                ),
            )


    elif isinstance(
        value,
        (
            list,
            tuple,
        ),
    ):

        for index, nested in enumerate(
            value
        ):

            yield from cell33_walk(
                nested,
                (
                    path
                    +
                    "["
                    +
                    str(
                        index
                    )
                    +
                    "]"
                ),
            )


def cell33_first_document_id(
    mapping,
):

    for field_name in (
        "document_id",
        "source_document_id",
        "doc_id",
        "id",
    ):

        value = mapping.get(
            field_name
        )


        if (
            isinstance(
                value,
                str,
            )
            and
            re.match(
                r"^book:.+?:fa-source:p\d+$",
                value,
            )
        ):

            return value


    return None


def cell33_first_text(
    mapping,
):

    for field_name in (
        "text",
        "content",
        "raw_text",
        "normalized_text",
        "text_content",
    ):

        value = mapping.get(
            field_name
        )


        if (
            isinstance(
                value,
                str,
            )
            and
            value.strip()
        ):

            return value.strip()


    return None


def cell33_metadata_surface(
    mapping,
):

    values = []


    for key, value in mapping.items():

        values.append(
            str(
                key
            )
        )


        if isinstance(
            value,
            (
                str,
                int,
                float,
                bool,
            ),
        ):

            values.append(
                str(
                    value
                )
            )


    return (
        " ".join(
            values
        )
        .lower()
    )


# ============================================================
# 6) Load all page-level source records
# ============================================================

cell33_page_records_by_document = {}


for file_priority, path in enumerate(
    cell33_candidate_files
):

    try:

        payload = json.loads(
            path.read_text(
                encoding="utf-8"
            )
        )


    except Exception as exc:

        cell33_failures.append({
            "reason":
                "BOOK_INGESTION_PARSE_FAILED",

            "file":
                str(
                    path
                ),

            "error":
                repr(
                    exc
                ),
        })

        continue


    for object_path, mapping in cell33_walk(
        payload,
        path="json",
    ):

        document_id = (
            cell33_first_document_id(
                mapping
            )
        )


        text = cell33_first_text(
            mapping
        )


        if (
            not isinstance(
                document_id,
                str,
            )
            or
            not isinstance(
                text,
                str,
            )
        ):
            continue


        book_root = cell33_book_root(
            document_id
        )


        page_number = cell33_page_number(
            document_id
        )


        if (
            not isinstance(
                book_root,
                str,
            )
            or
            not isinstance(
                page_number,
                int,
            )
        ):
            continue


        # This Cell intentionally expands to OTHER books.
        if book_root in cell33_excluded_book_roots:
            continue


        # Exclude pages already consumed by active Pilots.
        if document_id in cell33_active_document_ids:
            continue


        if document_id in cell33_rejected_document_ids:
            continue


        record = {
            "document_id":
                document_id,

            "book_root":
                book_root,

            "page_number":
                page_number,

            "text":
                text,

            "metadata_surface":
                cell33_metadata_surface(
                    mapping
                ),

            "source_mapping":
                copy.deepcopy(
                    mapping
                ),

            "file_path":
                str(
                    path
                ),

            "file_priority":
                file_priority,

            "object_path":
                object_path,
        }


        existing = (
            cell33_page_records_by_document.get(
                document_id
            )
        )


        # v2 appears first and therefore wins.
        if (
            existing is None
            or
            file_priority
            <
            existing[
                "file_priority"
            ]
        ):

            cell33_page_records_by_document[
                document_id
            ] = record


cell33_page_records = list(
    cell33_page_records_by_document.values()
)


# ============================================================
# 7) Group pages by book
# ============================================================

cell33_pages_by_book = {}


for record in cell33_page_records:

    cell33_pages_by_book.setdefault(
        record[
            "book_root"
        ],
        [],
    ).append(
        record
    )


for book_root in cell33_pages_by_book:

    cell33_pages_by_book[
        book_root
    ] = sorted(
        cell33_pages_by_book[
            book_root
        ],
        key=lambda row: row[
            "page_number"
        ],
    )


# ============================================================
# 8) Build one-page and two-page source Regions
# ============================================================

cell33_regions = []


for book_root, pages in (
    cell33_pages_by_book.items()
):

    # Single-page Regions.
    for page in pages:

        cell33_regions.append({
            "book_root":
                book_root,

            "region_size":
                1,

            "document_ids": [
                page[
                    "document_id"
                ]
            ],

            "page_numbers": [
                page[
                    "page_number"
                ]
            ],

            "text":
                page[
                    "text"
                ],

            "metadata_surface":
                page[
                    "metadata_surface"
                ],

            "source_records": [
                copy.deepcopy(
                    page
                )
            ],
        })


    # Two-page adjacent Regions.
    for index in range(
        len(
            pages
        )
        -
        1
    ):

        first = pages[
            index
        ]

        second = pages[
            index
            +
            1
        ]


        if (
            second[
                "page_number"
            ]
            !=
            first[
                "page_number"
            ]
            +
            1
        ):
            continue


        combined_text = (
            first[
                "text"
            ]
            +
            "\n\n"
            +
            second[
                "text"
            ]
        )


        combined_surface = (
            first[
                "metadata_surface"
            ]
            +
            " "
            +
            second[
                "metadata_surface"
            ]
        )


        cell33_regions.append({
            "book_root":
                book_root,

            "region_size":
                2,

            "document_ids": [
                first[
                    "document_id"
                ],
                second[
                    "document_id"
                ],
            ],

            "page_numbers": [
                first[
                    "page_number"
                ],
                second[
                    "page_number"
                ],
            ],

            "text":
                combined_text,

            "metadata_surface":
                combined_surface,

            "source_records": [
                copy.deepcopy(
                    first
                ),
                copy.deepcopy(
                    second
                ),
            ],
        })


# ============================================================
# 9) Table / meaningfulness scoring
# ============================================================

CELL33_FINANCIAL_TERMS = (
    "پول",
    "درآمد",
    "هزینه",
    "پس انداز",
    "پس‌انداز",
    "بودجه",
    "قیمت",
    "خرید",
    "سرمایه",
    "سود",
    "بدهی",
    "وام",
    "درصد",
    "ریال",
    "تومان",
    "حقوق",
    "دارایی",
    "بازده",
    "ریسک",
    "بیمه",
    "مالیات",
)


cell33_scored_regions = []


for region in cell33_regions:

    text = region[
        "text"
    ]


    normalized = (
        cell15_normalize_text(
            text
        )
    )


    lines = [
        line.strip()

        for line
        in text.splitlines()

        if line.strip()
    ]


    line_count = len(
        lines
    )


    word_count = len(
        normalized.split()
    )


    number_count = len(
        re.findall(
            r"\d+",
            normalized,
        )
    )


    short_line_count = sum(
        1

        for line
        in lines

        if len(
            line
        )
        <=
        90
    )


    short_line_ratio = (
        short_line_count
        /
        line_count

        if line_count

        else 0.0
    )


    surface = region[
        "metadata_surface"
    ]


    explicit_table_signal = any([
        "table"
        in
        surface,

        "table_region"
        in
        surface,

        "table_row"
        in
        surface,

        "table-cell"
        in
        surface,

        "جدول"
        in
        normalized,
    ])


    delimited_signal = any([
        "\t"
        in
        text,

        "|"
        in
        text,
    ])


    row_like_signal = any([
        (
            line_count >= 5
            and
            number_count >= 3
        ),

        (
            line_count >= 7
            and
            short_line_ratio >= 0.60
            and
            number_count >= 2
        ),

        (
            line_count >= 8
            and
            short_line_ratio >= 0.75
        ),
    ])


    if not any([
        explicit_table_signal,
        delimited_signal,
        row_like_signal,
    ]):
        continue


    financial_terms = [
        term

        for term
        in CELL33_FINANCIAL_TERMS

        if term
        in
        normalized
    ]


    index_like_penalty = 0


    if "فهرست" in normalized:
        index_like_penalty += 5


    if (
        "عنوان"
        in
        normalized
        and
        word_count < 60
    ):
        index_like_penalty += 2


    if (
        word_count < 20
        and
        line_count >= 4
    ):
        index_like_penalty += 2


    if (
        number_count >= 5
        and
        word_count < 45
        and
        any(
            marker
            in
            normalized

            for marker
            in (
                "فصل",
                "بخش",
                "صفحه",
            )
        )
    ):
        index_like_penalty += 3


    score = 0


    if explicit_table_signal:
        score += 6


    if delimited_signal:
        score += 5


    if row_like_signal:
        score += 4


    if line_count >= 5:
        score += 1


    if number_count >= 3:
        score += 2


    if number_count >= 6:
        score += 1


    if word_count >= 30:
        score += 1


    if word_count >= 60:
        score += 1


    score += min(
        len(
            financial_terms
        ),
        4,
    )


    # Prefer compact Regions for this Tiny Table task.
    if region[
        "region_size"
    ] == 1:
        score += 1


    score -= index_like_penalty


    candidate_class = (
        "STRICT_TABLE"

        if (
            explicit_table_signal
            or
            delimited_signal
        )

        else
        "TABLE_LIKE"
    )


    identity_payload = {
        "book_root":
            region[
                "book_root"
            ],

        "document_ids":
            copy.deepcopy(
                region[
                    "document_ids"
                ]
            ),

        "text_sha256":
            hashlib.sha256(
                text.encode(
                    "utf-8"
                )
            ).hexdigest(),
    }


    candidate_id = (
        "retarget-candidate:"
        +
        hashlib.sha256(
            cell10_canonical_json(
                identity_payload
            ).encode(
                "utf-8"
            )
        ).hexdigest()[
            :32
        ]
    )


    cell33_scored_regions.append({
        "candidate_id":
            candidate_id,

        "candidate_class":
            candidate_class,

        "score":
            score,

        "book_root":
            region[
                "book_root"
            ],

        "region_size":
            region[
                "region_size"
            ],

        "document_ids":
            copy.deepcopy(
                region[
                    "document_ids"
                ]
            ),

        "page_numbers":
            copy.deepcopy(
                region[
                    "page_numbers"
                ]
            ),

        "word_count":
            word_count,

        "line_count":
            line_count,

        "number_count":
            number_count,

        "short_line_ratio":
            round(
                short_line_ratio,
                4,
            ),

        "explicit_table_signal":
            explicit_table_signal,

        "delimited_signal":
            delimited_signal,

        "row_like_signal":
            row_like_signal,

        "financial_terms":
            copy.deepcopy(
                financial_terms
            ),

        "index_like_penalty":
            index_like_penalty,

        "text":
            text,

        "text_sha256":
            identity_payload[
                "text_sha256"
            ],

        "source_records":
            copy.deepcopy(
                region[
                    "source_records"
                ]
            ),
    })


# ============================================================
# 10) Rank + diversify by book
# ============================================================

cell33_scored_regions = sorted(
    cell33_scored_regions,
    key=lambda row: (
        0
        if row[
            "candidate_class"
        ]
        ==
        "STRICT_TABLE"
        else 1,

        -row[
            "score"
        ],

        row[
            "index_like_penalty"
        ],

        row[
            "region_size"
        ],

        row[
            "candidate_id"
        ],
    ),
)


cell33_candidate_shortlist = []
cell33_book_selection_counts = {}


for candidate in cell33_scored_regions:

    book_root = candidate[
        "book_root"
    ]


    already_from_book = (
        cell33_book_selection_counts.get(
            book_root,
            0,
        )
    )


    # Prevent one book from dominating the shortlist.
    if already_from_book >= 3:
        continue


    cell33_candidate_shortlist.append(
        candidate
    )


    cell33_book_selection_counts[
        book_root
    ] = (
        already_from_book
        +
        1
    )


    if len(
        cell33_candidate_shortlist
    ) >= 12:
        break


cell33_search_status = (
    "CROSS_BOOK_CANDIDATES_READY_FOR_HUMAN_REVIEW"

    if cell33_candidate_shortlist

    else
    "NO_CROSS_BOOK_TABLE_CANDIDATE_FOUND"
)


# ============================================================
# 11) Persistent shortlist
# ============================================================

cell33_persistent_shortlist = []


for rank, candidate in enumerate(
    cell33_candidate_shortlist,
    start=1,
):

    cell33_persistent_shortlist.append({
        "rank":
            rank,

        "candidate_id":
            candidate[
                "candidate_id"
            ],

        "candidate_class":
            candidate[
                "candidate_class"
            ],

        "score":
            candidate[
                "score"
            ],

        "book_root":
            candidate[
                "book_root"
            ],

        "region_size":
            candidate[
                "region_size"
            ],

        "document_ids":
            copy.deepcopy(
                candidate[
                    "document_ids"
                ]
            ),

        "page_numbers":
            copy.deepcopy(
                candidate[
                    "page_numbers"
                ]
            ),

        "word_count":
            candidate[
                "word_count"
            ],

        "line_count":
            candidate[
                "line_count"
            ],

        "number_count":
            candidate[
                "number_count"
            ],

        "financial_terms":
            copy.deepcopy(
                candidate[
                    "financial_terms"
                ]
            ),

        "index_like_penalty":
            candidate[
                "index_like_penalty"
            ],

        "text":
            candidate[
                "text"
            ],

        "text_sha256":
            candidate[
                "text_sha256"
            ],
    })


# ============================================================
# 12) Persist discovery state
# ============================================================

cell33_persistent_registry_payload = copy.deepcopy(
    cell32_persisted_registry
)


cell33_persistent_registry_payload[
    "retarget_candidate_discovery"
] = {
    "pilot_case_id":
        CELL33_PILOT_ID,

    "rejected_target_evidence_id":
        CELL33_REJECTED_TARGET_ID,

    "required_task_family":
        CELL33_TASK_FAMILY,

    "required_question_archetype":
        CELL33_QUESTION_ARCHETYPE,

    "search_scope":
        "CROSS_BOOK_CANONICAL_INGESTION",

    "excluded_original_book_roots":
        sorted(
            cell33_excluded_book_roots
        ),

    "canonical_files":
        [
            str(
                path
            )

            for path
            in cell33_candidate_files
        ],

    "available_other_book_count":
        len(
            cell33_pages_by_book
        ),

    "available_page_count":
        len(
            cell33_page_records
        ),

    "region_count":
        len(
            cell33_regions
        ),

    "table_candidate_count":
        len(
            cell33_scored_regions
        ),

    "status":
        cell33_search_status,

    "shortlist":
        copy.deepcopy(
            cell33_persistent_shortlist
        ),

    "replacement_selected":
        False,
}


cell33_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell33_persistent_registry_payload
    )
)


# ============================================================
# 13) Read-back validation
# ============================================================

cell33_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell33_persisted_discovery = (
    cell33_persisted_registry.get(
        "retarget_candidate_discovery"
    )
    or
    {}
)


cell33_persistence_valid = all([
    cell33_persisted_registry.get(
        "active_workspace",
        {},
    ).get(
        "workflow_state"
    )
    ==
    "BLOCKED_REQUIRES_RETARGET",

    cell33_persisted_discovery.get(
        "search_scope"
    )
    ==
    "CROSS_BOOK_CANONICAL_INGESTION",

    cell33_persisted_discovery.get(
        "status"
    )
    ==
    cell33_search_status,

    cell33_persisted_discovery.get(
        "replacement_selected"
    )
    is False,

    cell33_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    18,
])


# ============================================================
# 14) Safety
# ============================================================

cell33_queue_still_18 = (
    len(
        cell29_remaining_pilot_ids
    )
    ==
    18
)


cell33_current_pilot_still_first = (
    cell29_remaining_pilot_ids[
        0
    ]
    ==
    CELL33_PILOT_ID
)


cell33_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


# ============================================================
# 15) Hard failures
# ============================================================

cell33_hard_failures = [
    failure

    for failure
    in cell33_failures

    if not (
        isinstance(
            failure,
            dict,
        )
        and
        failure.get(
            "reason"
        )
        ==
        "BOOK_INGESTION_PARSE_WARNING"
    )
]


# ============================================================
# 16) Final Gate
# ============================================================

cell33_gate = all([
    len(
        cell33_hard_failures
    )
    ==
    0,

    cell33_current_state_valid,

    len(
        cell33_candidate_files
    )
    >
    0,

    cell33_persistence_valid,

    cell33_queue_still_18,

    cell33_current_pilot_still_first,

    cell33_base_seed_unchanged,
])


# ============================================================
# 17) Diagnostics
# ============================================================

print(
    "=== CROSS-BOOK RETARGET DISCOVERY ==="
)


print(
    "\n=== CURRENT BLOCKED PILOT ==="
)

print(
    "Pilot:",
    CELL33_PILOT_ID,
)

print(
    "Required Task Family:",
    CELL33_TASK_FAMILY,
)

print(
    "Required Question Archetype:",
    CELL33_QUESTION_ARCHETYPE,
)

print(
    "Excluded original book:",
    sorted(
        cell33_excluded_book_roots
    ),
)


print(
    "\n=== CANONICAL SOURCE FILES ==="
)

for path in cell33_candidate_files:

    print(
        "-",
        path,
    )


print(
    "\n=== DISCOVERY COUNTS ==="
)

print(
    "Other books:",
    len(
        cell33_pages_by_book
    ),
)

print(
    "Available unused pages:",
    len(
        cell33_page_records
    ),
)

print(
    "Candidate Regions:",
    len(
        cell33_regions
    ),
)

print(
    "Table / Table-like Regions:",
    len(
        cell33_scored_regions
    ),
)

print(
    "Human-review Shortlist:",
    len(
        cell33_candidate_shortlist
    ),
)

print(
    "Status:",
    cell33_search_status,
)


print(
    "\n=== HUMAN-REVIEW SHORTLIST ==="
)


for rank, candidate in enumerate(
    cell33_candidate_shortlist,
    start=1,
):

    print(
        "\n========================================"
    )

    print(
        "CANDIDATE",
        rank,
    )

    print(
        "Candidate ID:",
        candidate[
            "candidate_id"
        ],
    )

    print(
        "Class:",
        candidate[
            "candidate_class"
        ],
    )

    print(
        "Score:",
        candidate[
            "score"
        ],
    )

    print(
        "Book:",
        candidate[
            "book_root"
        ],
    )

    print(
        "Documents:",
        candidate[
            "document_ids"
        ],
    )

    print(
        "Pages:",
        candidate[
            "page_numbers"
        ],
    )

    print(
        "Region size:",
        candidate[
            "region_size"
        ],
    )

    print(
        "Words:",
        candidate[
            "word_count"
        ],
    )

    print(
        "Lines:",
        candidate[
            "line_count"
        ],
    )

    print(
        "Numbers:",
        candidate[
            "number_count"
        ],
    )

    print(
        "Financial terms:",
        candidate[
            "financial_terms"
        ],
    )

    print(
        "Index-like penalty:",
        candidate[
            "index_like_penalty"
        ],
    )

    print(
        "\nTEXT:\n"
        +
        candidate[
            "text"
        ]
    )


print(
    "\n=== RETARGET SAFETY ==="
)

print(
    "Replacement selected:",
    False,
)

print(
    "Queue still 18:",
    cell33_queue_still_18,
)

print(
    "Current Pilot still first:",
    cell33_current_pilot_still_first,
)

print(
    "Registry synced:",
    cell33_persistence_valid,
)

print(
    "Base Seed unchanged:",
    cell33_base_seed_unchanged,
)

print(
    "Source files modified:",
    False,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell33_failures
)


print(
    "\nCODE CELL 33 GATE:",
    (
        "PASS"
        if cell33_gate
        else "FAIL"
    ),
)


assert cell33_gate, (
    "Cross-book Retarget discovery failed. "
    "Repair this SAME Code Cell 33 before selecting "
    "a replacement Target."
)

=== CROSS-BOOK RETARGET DISCOVERY ===

=== CURRENT BLOCKED PILOT ===
Pilot: question-pilot:3023c1c546df2fe2211f79702b6e3dbb
Required Task Family: TINY_TABLE_EVIDENCE_RETRIEVAL
Required Question Archetype: STRUCTURE_AWARE_RETRIEVAL
Excluded original book: ['book:trainer-training-financial-literacy:fa-source']

=== CANONICAL SOURCE FILES ===
- /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/data/phase2/interim/book_ingested_v2.json
- /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/data/phase2/interim/book_ingested_v1.json

=== DISCOVERY COUNTS ===
Other books: 0
Available unused pages: 0
Candidate Regions: 0
Table / Table-like Regions: 0
Human-review Shortlist: 0
Status: NO_CROSS_BOOK_TABLE_CANDIDATE_FOUND

=== HUMAN-REVIEW SHORTLIST ===

=== RETARGET SAFETY ===
Replacement selected: False
Queue still 18: True
Current Pilot still first: True
Registry synced: True
Base Seed unchanged: True
Source files modified: False
Commit / Push: Fals

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 34 — Book Source Inventory Diagnostic

Cell 33 هیچ کتاب جایگزینی در `book_ingested_v1/v2` پیدا نکرد، اما Compact Runtime قبلاً وجود کتاب‌های دیگری مثل `adult-financial-literacy-course` را نشان داده است.

بنابراین این مرحله بررسی می‌کند:

- چه Book Rootهایی واقعاً در Runtime وجود دارند؛
- چه Book Rootهایی واقعاً داخل فایل‌های `book_ingested_v1/v2` دیده می‌شوند؛
- کدام Book Rootهای Runtime در ingestion فعلی غایب‌اند؛
- فایل واقعی Repository که آن Book Rootهای غایب در آن ذخیره شده‌اند کجاست؛
- آیا مشکل Cell 33 ناشی از Schema/Storage Scope بوده است یا واقعاً Source Corpus مربوطه در دسترس نیست.

این Cell فقط Diagnostic و Read-only نسبت به Source Corpus است.

هیچ Replacement Target انتخاب نمی‌شود و Queue همچنان 18 باقی می‌ماند.

</div>

In [34]:
# Code Cell 34

cell34_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell33_gate") is not True:
    cell34_failures.append(
        "CELL33_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell21_repo_root"
    ),
    Path,
):
    cell34_failures.append(
        "REPO_ROOT_MISSING"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell34_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Current Retarget state
# ============================================================

CELL34_PILOT_ID = (
    cell29_next_pilot_id
)


cell34_current_state_valid = all([
    CELL34_PILOT_ID
    ==
    cell30_retarget_workspace.get(
        "pilot_case_id"
    ),

    cell30_retarget_workspace.get(
        "workflow_state"
    )
    ==
    "BLOCKED_REQUIRES_RETARGET",

    len(
        cell29_remaining_pilot_ids
    )
    ==
    18,
])


# ============================================================
# 3) Book identifier helpers
# ============================================================

CELL34_BOOK_DOC_PATTERN = re.compile(
    r"book:[A-Za-z0-9._-]+:fa-source:p\d+"
)


CELL34_BOOK_ROOT_PATTERN = re.compile(
    r"^(book:[A-Za-z0-9._-]+:fa-source):p\d+$"
)


def cell34_book_root(
    document_id,
):

    if not isinstance(
        document_id,
        str,
    ):
        return None


    match = (
        CELL34_BOOK_ROOT_PATTERN.match(
            document_id
        )
    )


    if not match:
        return None


    return match.group(
        1
    )


def cell34_walk_strings(
    value,
    path="root",
):

    if isinstance(
        value,
        str,
    ):

        yield path, value


    elif isinstance(
        value,
        dict,
    ):

        for key, nested in value.items():

            yield from cell34_walk_strings(
                nested,
                (
                    path
                    +
                    "."
                    +
                    str(
                        key
                    )
                ),
            )


    elif isinstance(
        value,
        (
            list,
            tuple,
        ),
    ):

        for index, nested in enumerate(
            value
        ):

            yield from cell34_walk_strings(
                nested,
                (
                    path
                    +
                    "["
                    +
                    str(
                        index
                    )
                    +
                    "]"
                ),
            )


def cell34_extract_book_documents(
    value,
):

    matches = []


    for path, string_value in (
        cell34_walk_strings(
            value
        )
    ):

        for match in (
            CELL34_BOOK_DOC_PATTERN.findall(
                string_value
            )
        ):

            matches.append({
                "path":
                    path,

                "document_id":
                    match,

                "book_root":
                    cell34_book_root(
                        match
                    ),
            })


    return matches


# ============================================================
# 4) Runtime Book inventory
# ============================================================

cell34_runtime_book_matches = (
    cell34_extract_book_documents(
        cell13_seed.get(
            "final_runtime",
            {},
        )
    )
)


cell34_runtime_book_roots = sorted({
    row[
        "book_root"
    ]

    for row
    in cell34_runtime_book_matches

    if isinstance(
        row.get(
            "book_root"
        ),
        str,
    )
})


cell34_runtime_document_ids = sorted({
    row[
        "document_id"
    ]

    for row
    in cell34_runtime_book_matches
})


# ============================================================
# 5) Ingestion-file Book inventory
# ============================================================

cell34_ingestion_inventory = []


for path in cell33_candidate_files:

    try:

        raw_text = path.read_text(
            encoding="utf-8",
            errors="ignore",
        )


    except Exception as exc:

        cell34_failures.append({
            "reason":
                "INGESTION_FILE_READ_FAILED",

            "file":
                str(
                    path
                ),

            "error":
                repr(
                    exc
                ),
        })

        continue


    document_ids = sorted(
        set(
            CELL34_BOOK_DOC_PATTERN.findall(
                raw_text
            )
        )
    )


    book_roots = sorted({
        root

        for root
        in (
            cell34_book_root(
                document_id
            )

            for document_id
            in document_ids
        )

        if isinstance(
            root,
            str,
        )
    })


    cell34_ingestion_inventory.append({
        "file":
            str(
                path
            ),

        "document_count":
            len(
                document_ids
            ),

        "book_roots":
            book_roots,

        "document_samples":
            document_ids[
                :10
            ],
    })


cell34_ingestion_book_roots = sorted({
    root

    for row
    in cell34_ingestion_inventory

    for root
    in row[
        "book_roots"
    ]
})


# ============================================================
# 6) Compare Runtime vs canonical ingestion
# ============================================================

cell34_runtime_roots_missing_from_ingestion = sorted(
    set(
        cell34_runtime_book_roots
    )
    -
    set(
        cell34_ingestion_book_roots
    )
)


cell34_ingestion_roots_missing_from_runtime = sorted(
    set(
        cell34_ingestion_book_roots
    )
    -
    set(
        cell34_runtime_book_roots
    )
)


cell34_schema_scope_mismatch_detected = (
    len(
        cell34_runtime_roots_missing_from_ingestion
    )
    >
    0
)


# ============================================================
# 7) Locate missing Runtime books elsewhere in Repository
# ============================================================

CELL34_ALLOWED_SUFFIXES = {
    ".json",
    ".jsonl",
    ".ndjson",
    ".txt",
    ".md",
    ".csv",
    ".tsv",
}


CELL34_MAX_FILE_BYTES = (
    30
    *
    1024
    *
    1024
)


cell34_repo_search_roots = []


for relative_path in (
    "data/phase2",
    "artifacts/phase2",
):

    root = (
        cell21_repo_root
        /
        relative_path
    )


    if root.exists():

        cell34_repo_search_roots.append(
            root
        )


cell34_missing_root_locations = {
    book_root: []

    for book_root
    in cell34_runtime_roots_missing_from_ingestion
}


cell34_scanned_file_count = 0


for root in cell34_repo_search_roots:

    for path in root.rglob(
        "*"
    ):

        if not path.is_file():
            continue


        if (
            path.suffix.lower()
            not in
            CELL34_ALLOWED_SUFFIXES
        ):
            continue


        try:

            if (
                path.stat().st_size
                >
                CELL34_MAX_FILE_BYTES
            ):
                continue


        except Exception:

            continue


        cell34_scanned_file_count += 1


        try:

            file_text = path.read_text(
                encoding="utf-8",
                errors="ignore",
            )


        except Exception:

            continue


        for book_root in (
            cell34_runtime_roots_missing_from_ingestion
        ):

            if (
                book_root
                not in
                file_text
            ):
                continue


            location_list = (
                cell34_missing_root_locations[
                    book_root
                ]
            )


            if len(
                location_list
            ) >= 20:
                continue


            matching_documents = sorted(
                set(
                    document_id

                    for document_id
                    in CELL34_BOOK_DOC_PATTERN.findall(
                        file_text
                    )

                    if (
                        cell34_book_root(
                            document_id
                        )
                        ==
                        book_root
                    )
                )
            )


            location_list.append({
                "file":
                    str(
                        path.resolve()
                    ),

                "matching_document_count":
                    len(
                        matching_documents
                    ),

                "document_samples":
                    matching_documents[
                        :8
                    ],
            })


# ============================================================
# 8) Runtime sample paths by Book
# ============================================================

cell34_runtime_samples_by_book = {}


for book_root in cell34_runtime_book_roots:

    rows = [
        row

        for row
        in cell34_runtime_book_matches

        if (
            row[
                "book_root"
            ]
            ==
            book_root
        )
    ]


    cell34_runtime_samples_by_book[
        book_root
    ] = rows[
        :8
    ]


# ============================================================
# 9) Diagnostic interpretation
# ============================================================

cell34_missing_roots_found_elsewhere = {
    book_root:
        (
            len(
                locations
            )
            >
            0
        )

    for book_root, locations
    in cell34_missing_root_locations.items()
}


if (
    cell34_schema_scope_mismatch_detected
    and
    any(
        cell34_missing_roots_found_elsewhere.values()
    )
):

    cell34_diagnostic_status = (
        "CELL33_SOURCE_SCOPE_TOO_NARROW_"
        "ALTERNATE_BOOK_SOURCES_LOCATED"
    )


elif cell34_schema_scope_mismatch_detected:

    cell34_diagnostic_status = (
        "RUNTIME_HAS_ADDITIONAL_BOOKS_"
        "SOURCE_FILES_NOT_YET_LOCATED"
    )


else:

    cell34_diagnostic_status = (
        "RUNTIME_AND_INGESTION_BOOK_SCOPE_MATCH"
    )


# ============================================================
# 10) Persist diagnostic state
# ============================================================

cell34_persistent_registry_payload = copy.deepcopy(
    cell33_persisted_registry
)


cell34_persistent_registry_payload[
    "retarget_candidate_discovery"
] = {
    "pilot_case_id":
        CELL34_PILOT_ID,

    "search_scope":
        "BOOK_SOURCE_INVENTORY_DIAGNOSTIC",

    "diagnostic_status":
        cell34_diagnostic_status,

    "runtime_book_roots":
        copy.deepcopy(
            cell34_runtime_book_roots
        ),

    "canonical_ingestion_book_roots":
        copy.deepcopy(
            cell34_ingestion_book_roots
        ),

    "runtime_roots_missing_from_ingestion":
        copy.deepcopy(
            cell34_runtime_roots_missing_from_ingestion
        ),

    "missing_root_locations":
        copy.deepcopy(
            cell34_missing_root_locations
        ),

    "replacement_selected":
        False,
}


cell34_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell34_persistent_registry_payload
    )
)


# ============================================================
# 11) Read-back validation
# ============================================================

cell34_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell34_persisted_discovery = (
    cell34_persisted_registry.get(
        "retarget_candidate_discovery"
    )
    or
    {}
)


cell34_persistence_valid = all([
    cell34_persisted_registry.get(
        "active_workspace",
        {},
    ).get(
        "workflow_state"
    )
    ==
    "BLOCKED_REQUIRES_RETARGET",

    cell34_persisted_discovery.get(
        "search_scope"
    )
    ==
    "BOOK_SOURCE_INVENTORY_DIAGNOSTIC",

    cell34_persisted_discovery.get(
        "diagnostic_status"
    )
    ==
    cell34_diagnostic_status,

    cell34_persisted_discovery.get(
        "replacement_selected"
    )
    is False,

    cell34_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    18,
])


# ============================================================
# 12) Queue / safety
# ============================================================

cell34_queue_still_18 = (
    len(
        cell29_remaining_pilot_ids
    )
    ==
    18
)


cell34_current_pilot_still_first = (
    cell29_remaining_pilot_ids[
        0
    ]
    ==
    CELL34_PILOT_ID
)


cell34_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


# ============================================================
# 13) Hard failures
# ============================================================

cell34_hard_failures = [
    failure

    for failure
    in cell34_failures

    if not isinstance(
        failure,
        dict,
    )
]


# ============================================================
# 14) Final Gate
# ============================================================

cell34_gate = all([
    len(
        cell34_hard_failures
    )
    ==
    0,

    cell34_current_state_valid,

    len(
        cell34_runtime_book_roots
    )
    >
    0,

    cell34_persistence_valid,

    cell34_queue_still_18,

    cell34_current_pilot_still_first,

    cell34_base_seed_unchanged,
])


# ============================================================
# 15) Diagnostics
# ============================================================

print(
    "=== BOOK SOURCE INVENTORY DIAGNOSTIC ==="
)


print(
    "\n=== RUNTIME BOOKS ==="
)

print(
    "Runtime Book Roots:",
    len(
        cell34_runtime_book_roots
    ),
)


for book_root in (
    cell34_runtime_book_roots
):

    print(
        "\n-",
        book_root,
    )


    for sample in (
        cell34_runtime_samples_by_book[
            book_root
        ][
            :3
        ]
    ):

        print(
            "  Document:",
            sample[
                "document_id"
            ],
        )

        print(
            "  Runtime path:",
            sample[
                "path"
            ],
        )


print(
    "\n=== CANONICAL INGESTION INVENTORY ==="
)


for row in cell34_ingestion_inventory:

    print(
        "\nFile:",
        row[
            "file"
        ],
    )

    print(
        "Document IDs:",
        row[
            "document_count"
        ],
    )

    print(
        "Book Roots:",
        row[
            "book_roots"
        ],
    )


print(
    "\n=== SCOPE COMPARISON ==="
)

print(
    "Runtime roots:",
    cell34_runtime_book_roots,
)

print(
    "Ingestion roots:",
    cell34_ingestion_book_roots,
)

print(
    "Runtime roots missing from ingestion:",
    cell34_runtime_roots_missing_from_ingestion,
)

print(
    "Schema / Source scope mismatch:",
    cell34_schema_scope_mismatch_detected,
)


print(
    "\n=== REPOSITORY LOCATION SEARCH ==="
)

print(
    "Files scanned:",
    cell34_scanned_file_count,
)


for book_root in (
    cell34_runtime_roots_missing_from_ingestion
):

    print(
        "\nBOOK:",
        book_root,
    )


    locations = (
        cell34_missing_root_locations[
            book_root
        ]
    )


    print(
        "Located elsewhere:",
        bool(
            locations
        ),
    )


    for location in locations[
        :10
    ]:

        print(
            "  File:",
            location[
                "file"
            ],
        )

        print(
            "  Matching documents:",
            location[
                "matching_document_count"
            ],
        )

        print(
            "  Samples:",
            location[
                "document_samples"
            ],
        )


print(
    "\n=== DIAGNOSTIC CONCLUSION ==="
)

print(
    "Status:",
    cell34_diagnostic_status,
)


print(
    "\n=== RETARGET SAFETY ==="
)

print(
    "Replacement selected:",
    False,
)

print(
    "Queue still 18:",
    cell34_queue_still_18,
)

print(
    "Current Pilot still first:",
    cell34_current_pilot_still_first,
)

print(
    "Registry synced:",
    cell34_persistence_valid,
)

print(
    "Base Seed unchanged:",
    cell34_base_seed_unchanged,
)

print(
    "Source files modified:",
    False,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES / WARNINGS ==="
)

print(
    cell34_failures
)


print(
    "\nCODE CELL 34 GATE:",
    (
        "PASS"
        if cell34_gate
        else "FAIL"
    ),
)


assert cell34_gate, (
    "Book source inventory diagnostic failed. "
    "Repair this SAME Code Cell 34 before continuing "
    "Retarget discovery."
)

=== BOOK SOURCE INVENTORY DIAGNOSTIC ===

=== RUNTIME BOOKS ===
Runtime Book Roots: 2

- book:adult-financial-literacy-course:fa-source
  Document: book:adult-financial-literacy-course:fa-source:p001307
  Runtime path: root.question_drafting_pilot_packets[6].target_text_parts[0].document_id
  Document: book:adult-financial-literacy-course:fa-source:p001307
  Runtime path: root.question_drafting_pilot_packets[6].target_text_parts[0].locator.paragraph_id
  Document: book:adult-financial-literacy-course:fa-source:p001308
  Runtime path: root.question_drafting_pilot_packets[6].target_text_parts[1].document_id

- book:trainer-training-financial-literacy:fa-source
  Document: book:trainer-training-financial-literacy:fa-source:p000329
  Runtime path: root.question_drafting_pilot_packets[4].target_text_parts[0].document_id
  Document: book:trainer-training-financial-literacy:fa-source:p000329
  Runtime path: root.question_drafting_pilot_packets[4].target_text_parts[0].locator.paragraph_id
  Do

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 35 — Canonical Book Ingestion Schema Probe

Cell 34 ثابت کرد سه Book Root داخل `book_ingested_v1/v2` وجود دارند، اما Parser قبلی نتوانسته Page Recordهای آن‌ها را استخراج کند.

بنابراین در این مرحله ساختار واقعی `book_ingested_v2.json` مستقیماً بررسی می‌شود.

برای چند Document واقعی از هر کتاب:

- محل دقیق `document_id` در JSON پیدا می‌شود؛
- Mapping والد و Ancestorهای نزدیک بررسی می‌شوند؛
- کلیدهای موجود در هر سطح نمایش داده می‌شوند؛
- Text Fieldهای احتمالی در همان ساختار مشخص می‌شوند؛
- هیچ Candidate، Target یا Question جدیدی ساخته نمی‌شود.

هدف این Cell فقط ساخت Parser صحیح بر اساس Schema واقعی داده است.

Queue همچنان 18 و Pilot همچنان `BLOCKED_REQUIRES_RETARGET` باقی می‌ماند.

</div>

In [35]:
# Code Cell 35

cell35_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell34_gate") is not True:
    cell35_failures.append(
        "CELL34_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell21_repo_root"
    ),
    Path,
):
    cell35_failures.append(
        "REPO_ROOT_MISSING"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell35_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Canonical v2 source
# ============================================================

cell35_book_v2_path = (
    cell21_repo_root
    /
    "data/phase2/interim/book_ingested_v2.json"
)


if not cell35_book_v2_path.exists():
    cell35_failures.append(
        "BOOK_INGESTED_V2_NOT_FOUND"
    )


cell35_payload = None


if not cell35_failures:

    try:

        cell35_payload = json.loads(
            cell35_book_v2_path.read_text(
                encoding="utf-8"
            )
        )


    except Exception as exc:

        cell35_failures.append({
            "reason":
                "BOOK_V2_PARSE_FAILED",

            "error":
                repr(
                    exc
                ),
        })


# ============================================================
# 3) Top-level Schema
# ============================================================

cell35_top_level_type = type(
    cell35_payload
).__name__


cell35_top_level_keys = (
    sorted(
        cell35_payload.keys()
    )

    if isinstance(
        cell35_payload,
        dict,
    )

    else []
)


cell35_top_level_length = (
    len(
        cell35_payload
    )

    if isinstance(
        cell35_payload,
        (
            dict,
            list,
        ),
    )

    else None
)


# ============================================================
# 4) Probe Documents
# ============================================================

cell35_probe_document_ids = []


# Known adult source from Runtime.
for candidate in (
    "book:adult-financial-literacy-course:fa-source:p001307",
    "book:adult-financial-literacy-course:fa-source:p000641",
):

    if candidate not in cell35_probe_document_ids:
        cell35_probe_document_ids.append(
            candidate
        )


# Current trainer book source.
for candidate in (
    "book:trainer-training-financial-literacy:fa-source:p000329",
    "book:trainer-training-financial-literacy:fa-source:p000330",
):

    if candidate not in cell35_probe_document_ids:
        cell35_probe_document_ids.append(
            candidate
        )


# Discover one financial-literacy-for-all Document directly
# from the raw canonical file.
cell35_raw_text = (
    cell35_book_v2_path.read_text(
        encoding="utf-8",
        errors="ignore",
    )
    if cell35_book_v2_path.exists()
    else ""
)


cell35_financial_for_all_matches = sorted(
    set(
        re.findall(
            r"book:financial-literacy-for-all:"
            r"fa-source:p\d+",
            cell35_raw_text,
        )
    )
)


if cell35_financial_for_all_matches:

    cell35_probe_document_ids.append(
        cell35_financial_for_all_matches[
            0
        ]
    )


# ============================================================
# 5) Recursive path walker
# ============================================================

def cell35_walk_nodes(
    value,
    path=(),
    ancestors=(),
):

    yield {
        "path":
            path,

        "value":
            value,

        "ancestors":
            ancestors,
    }


    if isinstance(
        value,
        dict,
    ):

        new_ancestors = (
            ancestors
            +
            (
                value,
            )
        )


        for key, nested in value.items():

            yield from cell35_walk_nodes(
                nested,
                path=(
                    path
                    +
                    (
                        key,
                    )
                ),
                ancestors=
                    new_ancestors,
            )


    elif isinstance(
        value,
        list,
    ):

        new_ancestors = (
            ancestors
            +
            (
                value,
            )
        )


        for index, nested in enumerate(
            value
        ):

            yield from cell35_walk_nodes(
                nested,
                path=(
                    path
                    +
                    (
                        index,
                    )
                ),
                ancestors=
                    new_ancestors,
            )


# ============================================================
# 6) Mapping summary helper
# ============================================================

CELL35_TEXT_HINT_FIELDS = (
    "text",
    "content",
    "raw_text",
    "normalized_text",
    "text_content",
    "page_text",
    "paragraph_text",
    "body",
    "value",
)


def cell35_mapping_summary(
    mapping,
):

    if not isinstance(
        mapping,
        dict,
    ):

        return None


    scalar_fields = {}


    text_candidates = {}


    for key, value in mapping.items():

        if isinstance(
            value,
            (
                str,
                int,
                float,
                bool,
            ),
        ):

            preview = value


            if isinstance(
                value,
                str,
            ):

                preview = (
                    value[
                        :300
                    ]
                )


            scalar_fields[
                str(
                    key
                )
            ] = preview


        if (
            isinstance(
                value,
                str,
            )
            and
            value.strip()
            and
            (
                str(
                    key
                ).lower()
                in
                CELL35_TEXT_HINT_FIELDS

                or
                len(
                    value.strip()
                )
                >=
                80
            )
        ):

            text_candidates[
                str(
                    key
                )
            ] = (
                value.strip()[
                    :800
                ]
            )


    return {
        "keys":
            sorted(
                str(
                    key
                )

                for key
                in mapping.keys()
            ),

        "scalar_fields":
            scalar_fields,

        "text_candidates":
            text_candidates,
    }


# ============================================================
# 7) Locate exact Document IDs and inspect ancestors
# ============================================================

cell35_probe_results = {}


if cell35_payload is not None:

    cell35_all_nodes = list(
        cell35_walk_nodes(
            cell35_payload
        )
    )


    for document_id in (
        cell35_probe_document_ids
    ):

        hits = []


        for node in cell35_all_nodes:

            value = node[
                "value"
            ]


            if (
                isinstance(
                    value,
                    str,
                )
                and
                value
                ==
                document_id
            ):

                ancestor_dicts = [
                    ancestor

                    for ancestor
                    in reversed(
                        node[
                            "ancestors"
                        ]
                    )

                    if isinstance(
                        ancestor,
                        dict,
                    )
                ]


                ancestor_summaries = []


                for depth, mapping in enumerate(
                    ancestor_dicts[
                        :5
                    ]
                ):

                    ancestor_summaries.append({
                        "ancestor_depth":
                            depth,

                        "summary":
                            cell35_mapping_summary(
                                mapping
                            ),
                    })


                hits.append({
                    "path":
                        list(
                            node[
                                "path"
                            ]
                        ),

                    "ancestor_summaries":
                        ancestor_summaries,
                })


        cell35_probe_results[
            document_id
        ] = hits


# ============================================================
# 8) Infer recurring schema signatures
# ============================================================

cell35_schema_signatures = {}


for document_id, hits in (
    cell35_probe_results.items()
):

    for hit in hits:

        for ancestor in hit[
            "ancestor_summaries"
        ]:

            summary = ancestor[
                "summary"
            ]


            if not isinstance(
                summary,
                dict,
            ):
                continue


            signature = tuple(
                summary[
                    "keys"
                ]
            )


            signature_key = (
                repr(
                    signature
                )
            )


            if (
                signature_key
                not in
                cell35_schema_signatures
            ):

                cell35_schema_signatures[
                    signature_key
                ] = {
                    "keys":
                        list(
                            signature
                        ),

                    "occurrences":
                        0,

                    "documents":
                        [],
                }


            cell35_schema_signatures[
                signature_key
            ][
                "occurrences"
            ] += 1


            if (
                document_id
                not in
                cell35_schema_signatures[
                    signature_key
                ][
                    "documents"
                ]
            ):

                cell35_schema_signatures[
                    signature_key
                ][
                    "documents"
                ].append(
                    document_id
                )


cell35_ranked_schema_signatures = sorted(
    cell35_schema_signatures.values(),
    key=lambda row: (
        -row[
            "occurrences"
        ],
        len(
            row[
                "keys"
            ]
        ),
        repr(
            row[
                "keys"
            ]
        ),
    ),
)


# ============================================================
# 9) Probe validity
# ============================================================

cell35_documents_with_hits = [
    document_id

    for document_id, hits
    in cell35_probe_results.items()

    if hits
]


cell35_probe_valid = all([
    isinstance(
        cell35_payload,
        (
            dict,
            list,
        ),
    ),

    len(
        cell35_probe_document_ids
    )
    >=
    4,

    len(
        cell35_documents_with_hits
    )
    >=
    3,
])


# ============================================================
# 10) Persist diagnostic summary
# ============================================================

cell35_persistent_registry_payload = copy.deepcopy(
    cell34_persisted_registry
)


cell35_persistent_registry_payload[
    "retarget_candidate_discovery"
] = {
    "pilot_case_id":
        cell29_next_pilot_id,

    "search_scope":
        "CANONICAL_BOOK_INGESTION_SCHEMA_PROBE",

    "source_file":
        str(
            cell35_book_v2_path
        ),

    "top_level_type":
        cell35_top_level_type,

    "top_level_keys":
        copy.deepcopy(
            cell35_top_level_keys
        ),

    "probe_document_ids":
        copy.deepcopy(
            cell35_probe_document_ids
        ),

    "documents_with_hits":
        copy.deepcopy(
            cell35_documents_with_hits
        ),

    "schema_signature_count":
        len(
            cell35_ranked_schema_signatures
        ),

    "replacement_selected":
        False,
}


cell35_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell35_persistent_registry_payload
    )
)


# ============================================================
# 11) Read-back validation
# ============================================================

cell35_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell35_persisted_discovery = (
    cell35_persisted_registry.get(
        "retarget_candidate_discovery"
    )
    or
    {}
)


cell35_persistence_valid = all([
    cell35_persisted_registry.get(
        "active_workspace",
        {},
    ).get(
        "workflow_state"
    )
    ==
    "BLOCKED_REQUIRES_RETARGET",

    cell35_persisted_discovery.get(
        "search_scope"
    )
    ==
    "CANONICAL_BOOK_INGESTION_SCHEMA_PROBE",

    cell35_persisted_discovery.get(
        "replacement_selected"
    )
    is False,

    cell35_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    18,
])


# ============================================================
# 12) Safety
# ============================================================

cell35_queue_still_18 = (
    len(
        cell29_remaining_pilot_ids
    )
    ==
    18
)


cell35_current_pilot_still_first = (
    cell29_remaining_pilot_ids[
        0
    ]
    ==
    cell29_next_pilot_id
)


cell35_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


# ============================================================
# 13) Hard failures
# ============================================================

cell35_hard_failures = [
    failure

    for failure
    in cell35_failures

    if not isinstance(
        failure,
        dict,
    )
]


# ============================================================
# 14) Final Gate
# ============================================================

cell35_gate = all([
    len(
        cell35_hard_failures
    )
    ==
    0,

    cell35_probe_valid,

    cell35_persistence_valid,

    cell35_queue_still_18,

    cell35_current_pilot_still_first,

    cell35_base_seed_unchanged,
])


# ============================================================
# 15) Diagnostics
# ============================================================

print(
    "=== CANONICAL BOOK INGESTION SCHEMA PROBE ==="
)


print(
    "\n=== SOURCE FILE ==="
)

print(
    cell35_book_v2_path
)


print(
    "\n=== TOP LEVEL ==="
)

print(
    "Type:",
    cell35_top_level_type,
)

print(
    "Length:",
    cell35_top_level_length,
)

print(
    "Keys:",
    cell35_top_level_keys,
)


print(
    "\n=== DOCUMENT PROBES ==="
)


for document_id in (
    cell35_probe_document_ids
):

    hits = cell35_probe_results.get(
        document_id,
        [],
    )


    print(
        "\n========================================"
    )

    print(
        "DOCUMENT:",
        document_id,
    )

    print(
        "Hit count:",
        len(
            hits
        ),
    )


    for hit_index, hit in enumerate(
        hits[
            :3
        ],
        start=1,
    ):

        print(
            "\n--- HIT",
            hit_index,
            "---"
        )

        print(
            "Path:",
            hit[
                "path"
            ],
        )


        for ancestor in hit[
            "ancestor_summaries"
        ]:

            print(
                "\nAncestor depth:",
                ancestor[
                    "ancestor_depth"
                ],
            )


            summary = ancestor[
                "summary"
            ]


            print(
                "Keys:",
                summary[
                    "keys"
                ],
            )

            print(
                "Scalar fields:",
                summary[
                    "scalar_fields"
                ],
            )

            print(
                "Text candidates:",
                summary[
                    "text_candidates"
                ],
            )


print(
    "\n=== RECURRING SCHEMA SIGNATURES ==="
)


for index, signature in enumerate(
    cell35_ranked_schema_signatures[
        :12
    ],
    start=1,
):

    print(
        "\nSIGNATURE",
        index,
    )

    print(
        "Occurrences:",
        signature[
            "occurrences"
        ],
    )

    print(
        "Keys:",
        signature[
            "keys"
        ],
    )

    print(
        "Documents:",
        signature[
            "documents"
        ],
    )


print(
    "\n=== PROBE SUMMARY ==="
)

print(
    "Probe Documents:",
    len(
        cell35_probe_document_ids
    ),
)

print(
    "Documents with hits:",
    len(
        cell35_documents_with_hits
    ),
)

print(
    "Schema signatures:",
    len(
        cell35_ranked_schema_signatures
    ),
)

print(
    "Probe valid:",
    cell35_probe_valid,
)


print(
    "\n=== RETARGET SAFETY ==="
)

print(
    "Replacement selected:",
    False,
)

print(
    "Queue still 18:",
    cell35_queue_still_18,
)

print(
    "Current Pilot still first:",
    cell35_current_pilot_still_first,
)

print(
    "Registry synced:",
    cell35_persistence_valid,
)

print(
    "Base Seed unchanged:",
    cell35_base_seed_unchanged,
)

print(
    "Source files modified:",
    False,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES / WARNINGS ==="
)

print(
    cell35_failures
)


print(
    "\nCODE CELL 35 GATE:",
    (
        "PASS"
        if cell35_gate
        else "FAIL"
    ),
)


assert cell35_gate, (
    "Canonical ingestion schema probe failed. "
    "Repair this SAME Code Cell 35 before building "
    "the corrected source-level Retarget parser."
)

=== CANONICAL BOOK INGESTION SCHEMA PROBE ===

=== SOURCE FILE ===
/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/data/phase2/interim/book_ingested_v2.json

=== TOP LEVEL ===
Type: dict
Length: 14
Keys: ['artifact_type', 'artifact_version', 'coverage_state', 'edition_count', 'edition_records', 'package_asset_count', 'package_assets', 'paragraph_count', 'processing_state', 'schema_version', 'source_manifest', 'supersedes', 'supplemental_content', 'supplemental_content_count']

=== DOCUMENT PROBES ===

DOCUMENT: book:adult-financial-literacy-course:fa-source:p001307
Hit count: 1

--- HIT 1 ---
Path: ['edition_records', 0, 'paragraphs', 1306, 'paragraph_id']

Ancestor depth: 0
Keys: ['edition_id', 'heading_level', 'heading_path', 'in_table', 'is_heading', 'paragraph_id', 'paragraph_number', 'style', 'text', 'text_sha256', 'work_id']
Scalar fields: {'work_id': 'book:adult-financial-literacy-course', 'edition_id': 'book:adult-financial-literacy-course:fa-source', 

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 36 — Canonical Contiguous Table Region Discovery

Cell 35 Schema واقعی Book Ingestion را مشخص کرد:

`edition_records[*].paragraphs[*]`

و هر Paragraph دارای فیلدهای مستقیمی مانند:

- `paragraph_id`
- `paragraph_number`
- `text`
- `in_table`
- `heading_path`

است.

در این مرحله:

- تمام Paragraphهای `in_table=True` مستقیماً از `book_ingested_v2.json` خوانده می‌شوند؛
- Paragraphهای جدولیِ متوالی به یک Table Region واقعی تبدیل می‌شوند؛
- Region ردشده فعلی حذف می‌شود؛
- Regionهایی که با Evidence مصرف‌شده Pilotهای دیگر هم‌پوشانی دارند کنار گذاشته می‌شوند؛
- جدول‌های فهرست‌مانند امتیاز منفی می‌گیرند؛
- جدول‌های دارای محتوای مالی، عدد، مقایسه یا اطلاعات قابل‌پرسش امتیاز مثبت می‌گیرند؛
- Candidateها فقط برای Human Review نمایش داده می‌شوند؛
- هیچ Replacement Target به‌طور خودکار ساخته یا فعال نمی‌شود.

Task Family همچنان `TINY_TABLE_EVIDENCE_RETRIEVAL` و Question Archetype همچنان `STRUCTURE_AWARE_RETRIEVAL` باقی می‌مانند.

</div>

In [36]:
# Code Cell 36

cell36_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell35_gate") is not True:
    cell36_failures.append(
        "CELL35_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell35_payload"
    ),
    dict,
):
    cell36_failures.append(
        "BOOK_V2_PAYLOAD_MISSING"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell36_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Current blocked Pilot
# ============================================================

CELL36_PILOT_ID = (
    cell29_next_pilot_id
)

CELL36_REJECTED_TARGET_ID = (
    cell29_next_target_evidence_id
)

CELL36_TASK_FAMILY = (
    "TINY_TABLE_EVIDENCE_RETRIEVAL"
)

CELL36_QUESTION_ARCHETYPE = (
    "STRUCTURE_AWARE_RETRIEVAL"
)


cell36_current_state_valid = all([
    CELL36_PILOT_ID
    ==
    cell30_retarget_workspace.get(
        "pilot_case_id"
    ),

    cell30_retarget_workspace.get(
        "workflow_state"
    )
    ==
    "BLOCKED_REQUIRES_RETARGET",

    len(
        cell29_remaining_pilot_ids
    )
    ==
    18,
])


# ============================================================
# 3) Exclusion sets
# ============================================================

cell36_used_document_ids = set(
    cell32_active_document_ids
)


cell36_rejected_document_ids = set(
    cell32_rejected_document_ids
)


# ============================================================
# 4) Canonical edition records
# ============================================================

cell36_edition_records = (
    cell35_payload.get(
        "edition_records",
        []
    )
)


if not isinstance(
    cell36_edition_records,
    list,
):
    cell36_edition_records = []


cell36_editions_valid = (
    len(
        cell36_edition_records
    )
    >
    0
)


# ============================================================
# 5) Normalize paragraph records
# ============================================================

cell36_paragraphs_by_edition = {}


for edition in cell36_edition_records:

    if not isinstance(
        edition,
        dict,
    ):
        continue


    edition_id = edition.get(
        "edition_id"
    )


    work_id = edition.get(
        "work_id"
    )


    title = edition.get(
        "title"
    )


    author = edition.get(
        "author"
    )


    paragraphs = edition.get(
        "paragraphs",
        [],
    )


    if (
        not isinstance(
            edition_id,
            str,
        )
        or
        not isinstance(
            paragraphs,
            list,
        )
    ):
        continue


    normalized_paragraphs = []


    for paragraph in paragraphs:

        if not isinstance(
            paragraph,
            dict,
        ):
            continue


        paragraph_id = paragraph.get(
            "paragraph_id"
        )


        paragraph_number = paragraph.get(
            "paragraph_number"
        )


        text = paragraph.get(
            "text"
        )


        in_table = paragraph.get(
            "in_table"
        )


        if (
            not isinstance(
                paragraph_id,
                str,
            )
            or
            not isinstance(
                paragraph_number,
                int,
            )
            or
            not isinstance(
                text,
                str,
            )
        ):
            continue


        normalized_paragraphs.append({
            "work_id":
                work_id,

            "edition_id":
                edition_id,

            "title":
                title,

            "author":
                author,

            "paragraph_id":
                paragraph_id,

            "paragraph_number":
                paragraph_number,

            "text":
                text.strip(),

            "text_sha256":
                paragraph.get(
                    "text_sha256"
                ),

            "style":
                paragraph.get(
                    "style"
                ),

            "heading_path":
                copy.deepcopy(
                    paragraph.get(
                        "heading_path"
                    )
                ),

            "heading_level":
                paragraph.get(
                    "heading_level"
                ),

            "is_heading":
                paragraph.get(
                    "is_heading"
                ),

            "in_table":
                (
                    in_table
                    is True
                ),
        })


    cell36_paragraphs_by_edition[
        edition_id
    ] = sorted(
        normalized_paragraphs,
        key=lambda row: row[
            "paragraph_number"
        ],
    )


# ============================================================
# 6) Group contiguous in_table=True Paragraphs
# ============================================================

cell36_table_regions = []


for edition_id, paragraphs in (
    cell36_paragraphs_by_edition.items()
):

    current_region = []


    def cell36_flush_region(
        region_rows,
    ):

        if not region_rows:
            return


        document_ids = [
            row[
                "paragraph_id"
            ]

            for row
            in region_rows
        ]


        combined_text = (
            "\n".join(
                row[
                    "text"
                ]

                for row
                in region_rows

                if row[
                    "text"
                ]
            )
        )


        if not combined_text.strip():
            return


        heading_paths = []


        for row in region_rows:

            heading_path = row.get(
                "heading_path"
            )


            if (
                heading_path is not None
                and
                heading_path
                not in
                heading_paths
            ):

                heading_paths.append(
                    copy.deepcopy(
                        heading_path
                    )
                )


        identity_payload = {
            "edition_id":
                edition_id,

            "start_paragraph_id":
                document_ids[
                    0
                ],

            "end_paragraph_id":
                document_ids[
                    -1
                ],

            "text_sha256":
                hashlib.sha256(
                    combined_text.encode(
                        "utf-8"
                    )
                ).hexdigest(),
        }


        region_id = (
            "table-region:"
            +
            hashlib.sha256(
                cell10_canonical_json(
                    identity_payload
                ).encode(
                    "utf-8"
                )
            ).hexdigest()[
                :32
            ]
        )


        cell36_table_regions.append({
            "region_id":
                region_id,

            "work_id":
                region_rows[
                    0
                ][
                    "work_id"
                ],

            "edition_id":
                edition_id,

            "book_title":
                region_rows[
                    0
                ][
                    "title"
                ],

            "author":
                region_rows[
                    0
                ][
                    "author"
                ],

            "start_paragraph_number":
                region_rows[
                    0
                ][
                    "paragraph_number"
                ],

            "end_paragraph_number":
                region_rows[
                    -1
                ][
                    "paragraph_number"
                ],

            "document_ids":
                document_ids,

            "paragraph_count":
                len(
                    region_rows
                ),

            "heading_paths":
                heading_paths,

            "text":
                combined_text,

            "text_sha256":
                identity_payload[
                    "text_sha256"
                ],

            "paragraphs":
                copy.deepcopy(
                    region_rows
                ),
        })


    for paragraph in paragraphs:

        if paragraph[
            "in_table"
        ] is True:

            if not current_region:

                current_region = [
                    paragraph
                ]


            else:

                previous_number = (
                    current_region[
                        -1
                    ][
                        "paragraph_number"
                    ]
                )


                if (
                    paragraph[
                        "paragraph_number"
                    ]
                    ==
                    previous_number
                    +
                    1
                ):

                    current_region.append(
                        paragraph
                    )


                else:

                    cell36_flush_region(
                        current_region
                    )


                    current_region = [
                        paragraph
                    ]


        else:

            cell36_flush_region(
                current_region
            )


            current_region = []


    cell36_flush_region(
        current_region
    )


# ============================================================
# 7) Exclude rejected / already-used Regions
# ============================================================

cell36_unused_table_regions = []


for region in cell36_table_regions:

    document_ids = set(
        region[
            "document_ids"
        ]
    )


    rejected_overlap = bool(
        document_ids
        &
        cell36_rejected_document_ids
    )


    active_overlap = bool(
        document_ids
        &
        cell36_used_document_ids
    )


    if rejected_overlap:
        continue


    if active_overlap:
        continue


    cell36_unused_table_regions.append(
        region
    )


# ============================================================
# 8) Candidate scoring
# ============================================================

CELL36_FINANCIAL_TERMS = (
    "پول",
    "درآمد",
    "هزینه",
    "پس انداز",
    "پس‌انداز",
    "بودجه",
    "خرید",
    "قیمت",
    "حقوق",
    "سرمایه",
    "سود",
    "بازده",
    "ریسک",
    "بدهی",
    "وام",
    "بیمه",
    "مالیات",
    "درصد",
    "ریال",
    "تومان",
    "دارایی",
    "مصرف",
    "پس‌اندازی",
    "اعتبار",
)


cell36_scored_candidates = []


for region in cell36_unused_table_regions:

    text = region[
        "text"
    ]


    normalized = (
        cell15_normalize_text(
            text
        )
    )


    lines = [
        line.strip()

        for line
        in text.splitlines()

        if line.strip()
    ]


    line_count = len(
        lines
    )


    word_count = len(
        normalized.split()
    )


    number_count = len(
        re.findall(
            r"\d+",
            normalized,
        )
    )


    percent_count = len(
        re.findall(
            r"(?:٪|%)",
            text,
        )
    )


    currency_count = sum([
        normalized.count(
            "تومان"
        ),

        normalized.count(
            "ریال"
        ),
    ])


    financial_terms = [
        term

        for term
        in CELL36_FINANCIAL_TERMS

        if term
        in
        normalized
    ]


    unique_lines = len(
        set(
            lines
        )
    )


    repeated_line_count = (
        line_count
        -
        unique_lines
    )


    short_lines = [
        line

        for line
        in lines

        if len(
            line
        )
        <=
        100
    ]


    short_line_ratio = (
        len(
            short_lines
        )
        /
        line_count

        if line_count

        else 0.0
    )


    # Index / contents penalties.
    index_like_penalty = 0


    if "فهرست" in normalized:
        index_like_penalty += 6


    if (
        "عنوان"
        in
        normalized
        and
        word_count
        <
        80
    ):
        index_like_penalty += 3


    if (
        word_count
        <
        25
    ):
        index_like_penalty += 2


    if (
        line_count >= 5
        and
        number_count >= 3
        and
        len(
            financial_terms
        )
        ==
        0
        and
        word_count
        <
        60
    ):
        index_like_penalty += 2


    # Questionnaire / isolated Likert-item penalty.
    questionnaire_penalty = 0


    questionnaire_markers = (
        "کاملاً موافق",
        "کاملا موافق",
        "موافقم",
        "مخالفم",
        "گزینه",
        "پرسشنامه",
    )


    if any(
        marker
        in
        normalized

        for marker
        in questionnaire_markers
    ):
        questionnaire_penalty += 2


    # Meaningfulness score.
    meaningfulness_score = 0


    meaningfulness_score += min(
        len(
            financial_terms
        ),
        5,
    )


    if number_count >= 2:
        meaningfulness_score += 2


    if number_count >= 5:
        meaningfulness_score += 1


    if percent_count >= 1:
        meaningfulness_score += 2


    if currency_count >= 1:
        meaningfulness_score += 2


    if line_count >= 4:
        meaningfulness_score += 1


    if word_count >= 35:
        meaningfulness_score += 1


    if word_count >= 70:
        meaningfulness_score += 1


    if (
        short_line_ratio
        >=
        0.60
    ):
        meaningfulness_score += 1


    if repeated_line_count > 0:
        meaningfulness_score -= 1


    # Tiny-table suitability.
    size_score = 0


    if (
        2
        <=
        region[
            "paragraph_count"
        ]
        <=
        12
    ):
        size_score += 3


    elif (
        13
        <=
        region[
            "paragraph_count"
        ]
        <=
        25
    ):
        size_score += 1


    elif (
        region[
            "paragraph_count"
        ]
        >
        50
    ):
        size_score -= 3


    total_score = (
        meaningfulness_score
        +
        size_score
        -
        index_like_penalty
        -
        questionnaire_penalty
    )


    cell36_scored_candidates.append({
        **copy.deepcopy(
            region
        ),

        "line_count":
            line_count,

        "word_count":
            word_count,

        "number_count":
            number_count,

        "percent_count":
            percent_count,

        "currency_count":
            currency_count,

        "financial_terms":
            copy.deepcopy(
                financial_terms
            ),

        "short_line_ratio":
            round(
                short_line_ratio,
                4,
            ),

        "index_like_penalty":
            index_like_penalty,

        "questionnaire_penalty":
            questionnaire_penalty,

        "meaningfulness_score":
            meaningfulness_score,

        "size_score":
            size_score,

        "total_score":
            total_score,
    })


# ============================================================
# 9) Rank candidates
# ============================================================

cell36_ranked_candidates = sorted(
    cell36_scored_candidates,
    key=lambda row: (
        -row[
            "total_score"
        ],

        row[
            "index_like_penalty"
        ],

        row[
            "questionnaire_penalty"
        ],

        abs(
            row[
                "paragraph_count"
            ]
            -
            6
        ),

        row[
            "edition_id"
        ],

        row[
            "start_paragraph_number"
        ],
    ),
)


# ============================================================
# 10) Diversified Human-review Shortlist
# ============================================================

cell36_candidate_shortlist = []

cell36_book_counts = {}


for candidate in cell36_ranked_candidates:

    edition_id = candidate[
        "edition_id"
    ]


    book_count = (
        cell36_book_counts.get(
            edition_id,
            0,
        )
    )


    # Maximum 5 candidates from one book.
    if book_count >= 5:
        continue


    cell36_candidate_shortlist.append(
        candidate
    )


    cell36_book_counts[
        edition_id
    ] = (
        book_count
        +
        1
    )


    if len(
        cell36_candidate_shortlist
    ) >= 12:
        break


cell36_search_status = (
    "CANONICAL_TABLE_REGIONS_READY_FOR_HUMAN_REVIEW"

    if cell36_candidate_shortlist

    else
    "NO_UNUSED_CANONICAL_TABLE_REGION_FOUND"
)


# ============================================================
# 11) Persistent safe shortlist
# ============================================================

cell36_persistent_shortlist = []


for rank, candidate in enumerate(
    cell36_candidate_shortlist,
    start=1,
):

    cell36_persistent_shortlist.append({
        "rank":
            rank,

        "region_id":
            candidate[
                "region_id"
            ],

        "book_title":
            candidate[
                "book_title"
            ],

        "edition_id":
            candidate[
                "edition_id"
            ],

        "start_paragraph_number":
            candidate[
                "start_paragraph_number"
            ],

        "end_paragraph_number":
            candidate[
                "end_paragraph_number"
            ],

        "document_ids":
            copy.deepcopy(
                candidate[
                    "document_ids"
                ]
            ),

        "paragraph_count":
            candidate[
                "paragraph_count"
            ],

        "total_score":
            candidate[
                "total_score"
            ],

        "financial_terms":
            copy.deepcopy(
                candidate[
                    "financial_terms"
                ]
            ),

        "number_count":
            candidate[
                "number_count"
            ],

        "percent_count":
            candidate[
                "percent_count"
            ],

        "currency_count":
            candidate[
                "currency_count"
            ],

        "index_like_penalty":
            candidate[
                "index_like_penalty"
            ],

        "questionnaire_penalty":
            candidate[
                "questionnaire_penalty"
            ],

        "heading_paths":
            copy.deepcopy(
                candidate[
                    "heading_paths"
                ]
            ),

        "text":
            candidate[
                "text"
            ],

        "text_sha256":
            candidate[
                "text_sha256"
            ],
    })


# ============================================================
# 12) Persist Discovery
# ============================================================

cell36_persistent_registry_payload = copy.deepcopy(
    cell35_persisted_registry
)


cell36_persistent_registry_payload[
    "retarget_candidate_discovery"
] = {
    "pilot_case_id":
        CELL36_PILOT_ID,

    "rejected_target_evidence_id":
        CELL36_REJECTED_TARGET_ID,

    "required_task_family":
        CELL36_TASK_FAMILY,

    "required_question_archetype":
        CELL36_QUESTION_ARCHETYPE,

    "search_scope":
        "CANONICAL_CONTIGUOUS_IN_TABLE_REGIONS",

    "schema_source":
        "book_ingested_v2.edition_records[*].paragraphs[*]",

    "edition_count":
        len(
            cell36_paragraphs_by_edition
        ),

    "raw_table_region_count":
        len(
            cell36_table_regions
        ),

    "unused_table_region_count":
        len(
            cell36_unused_table_regions
        ),

    "scored_candidate_count":
        len(
            cell36_scored_candidates
        ),

    "status":
        cell36_search_status,

    "shortlist":
        copy.deepcopy(
            cell36_persistent_shortlist
        ),

    "replacement_selected":
        False,
}


cell36_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell36_persistent_registry_payload
    )
)


# ============================================================
# 13) Read-back validation
# ============================================================

cell36_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell36_persisted_discovery = (
    cell36_persisted_registry.get(
        "retarget_candidate_discovery"
    )
    or
    {}
)


cell36_persistence_valid = all([
    cell36_persisted_registry.get(
        "active_workspace",
        {},
    ).get(
        "workflow_state"
    )
    ==
    "BLOCKED_REQUIRES_RETARGET",

    cell36_persisted_discovery.get(
        "search_scope"
    )
    ==
    "CANONICAL_CONTIGUOUS_IN_TABLE_REGIONS",

    cell36_persisted_discovery.get(
        "status"
    )
    ==
    cell36_search_status,

    cell36_persisted_discovery.get(
        "replacement_selected"
    )
    is False,

    cell36_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    18,
])


# ============================================================
# 14) Safety
# ============================================================

cell36_queue_still_18 = (
    len(
        cell29_remaining_pilot_ids
    )
    ==
    18
)


cell36_current_pilot_still_first = (
    cell29_remaining_pilot_ids[
        0
    ]
    ==
    CELL36_PILOT_ID
)


cell36_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


# ============================================================
# 15) Final Gate
# ============================================================

cell36_gate = all([
    len(
        cell36_failures
    )
    ==
    0,

    cell36_current_state_valid,

    cell36_editions_valid,

    cell36_persistence_valid,

    cell36_queue_still_18,

    cell36_current_pilot_still_first,

    cell36_base_seed_unchanged,
])


# ============================================================
# 16) Diagnostics
# ============================================================

print(
    "=== CANONICAL CONTIGUOUS TABLE REGION DISCOVERY ==="
)


print(
    "\n=== CURRENT BLOCKED PILOT ==="
)

print(
    "Pilot:",
    CELL36_PILOT_ID,
)

print(
    "Task Family:",
    CELL36_TASK_FAMILY,
)

print(
    "Question Archetype:",
    CELL36_QUESTION_ARCHETYPE,
)


print(
    "\n=== CANONICAL TABLE INVENTORY ==="
)

print(
    "Editions:",
    len(
        cell36_paragraphs_by_edition
    ),
)

print(
    "Raw contiguous Table Regions:",
    len(
        cell36_table_regions
    ),
)

print(
    "Unused Table Regions:",
    len(
        cell36_unused_table_regions
    ),
)

print(
    "Scored Candidates:",
    len(
        cell36_scored_candidates
    ),
)

print(
    "Human-review Shortlist:",
    len(
        cell36_candidate_shortlist
    ),
)

print(
    "Status:",
    cell36_search_status,
)


print(
    "\n=== HUMAN-REVIEW SHORTLIST ==="
)


for rank, candidate in enumerate(
    cell36_candidate_shortlist,
    start=1,
):

    print(
        "\n========================================"
    )

    print(
        "CANDIDATE",
        rank,
    )

    print(
        "Region ID:",
        candidate[
            "region_id"
        ],
    )

    print(
        "Book:",
        candidate[
            "book_title"
        ],
    )

    print(
        "Edition:",
        candidate[
            "edition_id"
        ],
    )

    print(
        "Paragraphs:",
        (
            candidate[
                "start_paragraph_number"
            ],
            candidate[
                "end_paragraph_number"
            ],
        ),
    )

    print(
        "Paragraph count:",
        candidate[
            "paragraph_count"
        ],
    )

    print(
        "Score:",
        candidate[
            "total_score"
        ],
    )

    print(
        "Financial terms:",
        candidate[
            "financial_terms"
        ],
    )

    print(
        "Numbers:",
        candidate[
            "number_count"
        ],
    )

    print(
        "Percent markers:",
        candidate[
            "percent_count"
        ],
    )

    print(
        "Currency markers:",
        candidate[
            "currency_count"
        ],
    )

    print(
        "Index-like penalty:",
        candidate[
            "index_like_penalty"
        ],
    )

    print(
        "Questionnaire penalty:",
        candidate[
            "questionnaire_penalty"
        ],
    )

    print(
        "Heading paths:",
        candidate[
            "heading_paths"
        ],
    )

    print(
        "\nTEXT:\n"
        +
        candidate[
            "text"
        ]
    )


print(
    "\n=== RETARGET SAFETY ==="
)

print(
    "Replacement selected:",
    False,
)

print(
    "Queue still 18:",
    cell36_queue_still_18,
)

print(
    "Current Pilot still first:",
    cell36_current_pilot_still_first,
)

print(
    "Registry synced:",
    cell36_persistence_valid,
)

print(
    "Base Seed unchanged:",
    cell36_base_seed_unchanged,
)

print(
    "Source files modified:",
    False,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell36_failures
)


print(
    "\nCODE CELL 36 GATE:",
    (
        "PASS"
        if cell36_gate
        else "FAIL"
    ),
)


assert cell36_gate, (
    "Canonical contiguous Table Region discovery failed. "
    "Repair this SAME Code Cell 36 before Human selection "
    "of a replacement Target."
)

=== CANONICAL CONTIGUOUS TABLE REGION DISCOVERY ===

=== CURRENT BLOCKED PILOT ===
Pilot: question-pilot:3023c1c546df2fe2211f79702b6e3dbb
Task Family: TINY_TABLE_EVIDENCE_RETRIEVAL
Question Archetype: STRUCTURE_AWARE_RETRIEVAL

=== CANONICAL TABLE INVENTORY ===
Editions: 3
Raw contiguous Table Regions: 39
Unused Table Regions: 36
Scored Candidates: 36
Human-review Shortlist: 12
Status: CANONICAL_TABLE_REGIONS_READY_FOR_HUMAN_REVIEW

=== HUMAN-REVIEW SHORTLIST ===

CANDIDATE 1
Region ID: table-region:5f3faf051a2c0c76973a498dae8e2a00
Book: سواد مالی برای همه
Edition: book:financial-literacy-for-all:fa-source
Paragraphs: (750, 764)
Paragraph count: 15
Score: 12
Financial terms: ['درآمد', 'سرمایه', 'سود', 'ریسک', 'بدهی', 'وام', 'درصد', 'تومان']
Numbers: 0
Percent markers: 0
Currency markers: 4
Index-like penalty: 0
Questionnaire penalty: 0
Heading paths: [['ترکیب سرمایه\u200cگذاری پیشنهادی']]

TEXT:
فلسفه سرمایه﻿گذاری
می﻿خواهم اصل سرمایه﻿ام در پنج سال آینده دست کم برابر با نرخ تورم رشد کند

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 37 — Replacement Target Selection + Question ACCEPT + Expected Answer Draft

Human Reviewer، Table Region مربوط به «نمره اعتباری من» را به‌عنوان Replacement Target انتخاب کرد.

سؤال پذیرفته‌شده:

**برای محاسبه نمره اعتباری، چه شاخص‌هایی بررسی می‌شوند و نمره نهایی چگونه به دست می‌آید؟**

در این مرحله:

- Table Region انتخاب‌شده از Canonical Book Ingestion ثبت می‌شود؛
- Target Evidence قبلی در Lineage به‌عنوان Retargeted حفظ می‌شود؛
- Replacement Target Evidence جدید ساخته می‌شود؛
- همان Pilot و Coverage حفظ می‌شوند؛
- Question با تصمیم Human `ACCEPT` ثبت می‌شود؛
- Expected Answer Draft فقط از Evidence انتخاب‌شده ساخته می‌شود؛
- Registry دائمی Sync می‌شود؛
- Citation هنوز ساخته نمی‌شود؛
- Queue همچنان 18 باقی می‌ماند.

</div>

In [37]:
# Code Cell 37

cell37_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell36_gate") is not True:
    cell37_failures.append(
        "CELL36_NOT_PASS"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell37_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Human-selected canonical Table Region
# ============================================================

CELL37_SELECTED_REGION_ID = (
    "table-region:68e3e189e456efd98c14b4f380b29703"
)


cell37_selected_candidates = [
    candidate

    for candidate
    in cell36_candidate_shortlist

    if candidate.get(
        "region_id"
    )
    ==
    CELL37_SELECTED_REGION_ID
]


cell37_selected_candidate = (
    copy.deepcopy(
        cell37_selected_candidates[
            0
        ]
    )

    if len(
        cell37_selected_candidates
    )
    ==
    1

    else None
)


if not isinstance(
    cell37_selected_candidate,
    dict,
):
    cell37_failures.append(
        "SELECTED_REGION_NOT_FOUND_UNIQUELY"
    )


# ============================================================
# 3) Current Pilot identity
# ============================================================

CELL37_PILOT_ID = (
    cell29_next_pilot_id
)

CELL37_GOLDEN_CASE_ID = (
    cell29_next_golden_case_id
)

CELL37_TASK_DRAFT_ID = (
    cell29_next_task_draft_id
)

CELL37_OLD_TARGET_EVIDENCE_ID = (
    cell29_next_target_evidence_id
)


CELL37_ACCEPTED_QUESTION = (
    "برای محاسبه نمره اعتباری، چه شاخص‌هایی بررسی می‌شوند "
    "و نمره نهایی چگونه به دست می‌آید؟"
)


CELL37_EXPECTED_ANSWER = (
    "چهار شاخص شامل «درآمد خالص ماهانه و توان بازپرداخت اقساط»، "
    "«پایداری شغل و کسب‌وکار»، «عملکرد مالی» و «رفتار اعتباری» "
    "هستند. هر شاخص از یک تا ده امتیاز می‌گیرد؛ سپس امتیازها "
    "با هم جمع می‌شوند و مجموع امتیاز بر چهار تقسیم می‌شود تا "
    "نمره اعتباری به دست آید."
)


CELL37_FULL_572_BLOCK_STATUS = (
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


# ============================================================
# 4) Validate selected Evidence semantics
# ============================================================

cell37_selected_text = (
    cell37_selected_candidate.get(
        "text",
        ""
    )

    if isinstance(
        cell37_selected_candidate,
        dict,
    )

    else ""
)


def cell37_fa_surface(
    text,
):

    value = cell15_normalize_text(
        text
    )


    value = (
        value
        .replace(
            "ي",
            "ی",
        )
        .replace(
            "ك",
            "ک",
        )
        .replace(
            "\u200c",
            " ",
        )
    )


    value = re.sub(
        r"\s+",
        " ",
        value,
    )


    return value.strip()


cell37_surface = (
    cell37_fa_surface(
        cell37_selected_text
    )
)


cell37_evidence_grounding = {
    "net_monthly_income_and_repayment":
        (
            "درآمد خالص ماهانه"
            in
            cell37_surface
            and
            "توان بازپرداخت اقساط"
            in
            cell37_surface
        ),

    "job_stability":
        (
            "پایداری شغل"
            in
            cell37_surface
        ),

    "financial_performance":
        (
            "عملکرد مالی"
            in
            cell37_surface
        ),

    "credit_behavior":
        (
            "رفتار اعتباری"
            in
            cell37_surface
        ),

    "score_one_to_ten":
        (
            "امتیاز از یک تا ده"
            in
            cell37_surface
        ),

    "divide_by_four":
        (
            "تقسیم بر چهار"
            in
            cell37_surface
        ),

    "credit_score":
        (
            "نمره اعتباری"
            in
            cell37_surface
        ),
}


cell37_evidence_grounding_pass = all(
    cell37_evidence_grounding.values()
)


# ============================================================
# 5) Build canonical Replacement Target Parts
# ============================================================

cell37_replacement_target_parts = []


if isinstance(
    cell37_selected_candidate,
    dict,
):

    for index, paragraph in enumerate(
        cell37_selected_candidate.get(
            "paragraphs",
            [],
        )
    ):

        if not isinstance(
            paragraph,
            dict,
        ):
            continue


        paragraph_id = paragraph.get(
            "paragraph_id"
        )


        paragraph_number = paragraph.get(
            "paragraph_number"
        )


        text = paragraph.get(
            "text",
            ""
        )


        if (
            not isinstance(
                paragraph_id,
                str,
            )
            or
            not isinstance(
                text,
                str,
            )
            or
            not text
        ):
            continue


        target_part_identity = {
            "document_id":
                paragraph_id,

            "role":
                "PROTECTED_REGION_MEMBER",

            "char_start":
                0,

            "char_end":
                len(
                    text
                ),

            "text_sha256":
                hashlib.sha256(
                    text.encode(
                        "utf-8"
                    )
                ).hexdigest(),
        }


        target_part_id = (
            "target-part:"
            +
            hashlib.sha256(
                cell10_canonical_json(
                    target_part_identity
                ).encode(
                    "utf-8"
                )
            ).hexdigest()[
                :32
            ]
        )


        cell37_replacement_target_parts.append({
            "target_part_id":
                target_part_id,

            "document_id":
                paragraph_id,

            "source_id":
                paragraph.get(
                    "edition_id"
                ),

            "role":
                "PROTECTED_REGION_MEMBER",

            "char_start":
                0,

            "char_end":
                len(
                    text
                ),

            "char_count":
                len(
                    text
                ),

            "text":
                text,

            "text_sha256":
                target_part_identity[
                    "text_sha256"
                ],

            "locator": {
                "paragraph_id":
                    paragraph_id,

                "paragraph_number":
                    paragraph_number,
            },

            "locator_origin":
                "CANONICAL_BOOK_INGESTION_V2",

            "attribution": {
                "work_id":
                    paragraph.get(
                        "work_id"
                    ),

                "edition_id":
                    paragraph.get(
                        "edition_id"
                    ),

                "book_title":
                    paragraph.get(
                        "title"
                    ),

                "author":
                    paragraph.get(
                        "author"
                    ),
            },
        })


cell37_target_parts_valid = all([
    len(
        cell37_replacement_target_parts
    )
    ==
    8,

    all(
        part.get(
            "role"
        )
        ==
        "PROTECTED_REGION_MEMBER"

        for part
        in cell37_replacement_target_parts
    ),

    all(
        part.get(
            "char_start"
        )
        ==
        0

        for part
        in cell37_replacement_target_parts
    ),
])


# ============================================================
# 6) Deterministic Replacement Target Evidence ID
# ============================================================

cell37_target_identity_payload = {
    "identity_version":
        "retarget_target_evidence_identity_v1",

    "pilot_case_id":
        CELL37_PILOT_ID,

    "supersedes_target_evidence_id":
        CELL37_OLD_TARGET_EVIDENCE_ID,

    "selected_region_id":
        CELL37_SELECTED_REGION_ID,

    "target_parts": [
        {
            "document_id":
                part[
                    "document_id"
                ],

            "role":
                part[
                    "role"
                ],

            "char_start":
                part[
                    "char_start"
                ],

            "char_end":
                part[
                    "char_end"
                ],

            "text_sha256":
                part[
                    "text_sha256"
                ],
        }

        for part
        in cell37_replacement_target_parts
    ],
}


cell37_replacement_target_evidence_id = (
    "target-evidence:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell37_target_identity_payload
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


# ============================================================
# 7) Replacement Target Evidence Record
# ============================================================

cell37_replacement_target_evidence = copy.deepcopy(
    cell29_next_target_evidence
)


cell37_replacement_target_evidence[
    "target_evidence_id"
] = (
    cell37_replacement_target_evidence_id
)


cell37_replacement_target_evidence[
    "target_parts"
] = copy.deepcopy(
    cell37_replacement_target_parts
)


cell37_replacement_target_evidence[
    "retarget_lineage"
] = {
    "status":
        "HUMAN_SELECTED_REPLACEMENT",

    "supersedes_target_evidence_id":
        CELL37_OLD_TARGET_EVIDENCE_ID,

    "replacement_target_evidence_id":
        cell37_replacement_target_evidence_id,

    "selected_region_id":
        CELL37_SELECTED_REGION_ID,

    "selection_reason":
        (
            "Meaningful compact table supporting a natural "
            "structure-aware financial-literacy question."
        ),

    "source":
        "CANONICAL_BOOK_INGESTION_V2",

    "external_knowledge_used":
        False,
}


cell37_replacement_target_evidence[
    "human_retarget_status"
] = "ACCEPT"


# ============================================================
# 8) Display-ready Target Text Parts
# ============================================================

cell37_replacement_target_text_parts = [
    {
        "part_index":
            index,

        "document_id":
            part[
                "document_id"
            ],

        "role":
            part[
                "role"
            ],

        "char_start":
            part[
                "char_start"
            ],

        "char_end":
            part[
                "char_end"
            ],

        "text":
            part[
                "text"
            ],

        "text_sha256":
            part[
                "text_sha256"
            ],
    }

    for index, part in enumerate(
        cell37_replacement_target_parts
    )
]


cell37_replacement_combined_target_text = (
    "\n".join(
        part[
            "text"
        ]

        for part
        in cell37_replacement_target_text_parts
    )
)


# ============================================================
# 9) Build Retargeted Pilot Packet
# ============================================================

def cell37_replace_scalar(
    value,
    old_value,
    new_value,
):

    if isinstance(
        value,
        dict,
    ):

        return {
            key:
                cell37_replace_scalar(
                    nested,
                    old_value,
                    new_value,
                )

            for key, nested
            in value.items()
        }


    if isinstance(
        value,
        list,
    ):

        return [
            cell37_replace_scalar(
                nested,
                old_value,
                new_value,
            )

            for nested
            in value
        ]


    if value == old_value:
        return new_value


    return value


cell37_retargeted_pilot_packet = (
    cell37_replace_scalar(
        copy.deepcopy(
            cell29_next_pilot_packet
        ),
        CELL37_OLD_TARGET_EVIDENCE_ID,
        cell37_replacement_target_evidence_id,
    )
)


if (
    isinstance(
        cell37_retargeted_pilot_packet,
        dict,
    )
    and
    "target_text_parts"
    in
    cell37_retargeted_pilot_packet
):

    cell37_retargeted_pilot_packet[
        "target_text_parts"
    ] = copy.deepcopy(
        cell37_replacement_target_text_parts
    )


cell37_retargeted_pilot_packet[
    "retarget_lineage"
] = {
    "decision":
        "RETARGET",

    "old_target_evidence_id":
        CELL37_OLD_TARGET_EVIDENCE_ID,

    "new_target_evidence_id":
        cell37_replacement_target_evidence_id,

    "selected_region_id":
        CELL37_SELECTED_REGION_ID,

    "human_selected":
        True,
}


# ============================================================
# 10) Start fresh Engine session for Retargeted Pilot
# ============================================================

cell37_question_pending_session = (
    cell13_build_session(
        cell37_retargeted_pilot_packet
    )
)


cell37_session_start_valid = (
    cell37_question_pending_session.get(
        "current_state"
    )
    ==
    "QUESTION_DRAFT_PENDING"
)


# ============================================================
# 11) Question Draft ID
# ============================================================

cell37_question_identity_payload = {
    "identity_version":
        "retarget_question_draft_identity_v1",

    "pilot_case_id":
        CELL37_PILOT_ID,

    "target_evidence_id":
        cell37_replacement_target_evidence_id,

    "question_text":
        CELL37_ACCEPTED_QUESTION,

    "revision_number":
        1,
}


cell37_question_draft_id = (
    "question-draft:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell37_question_identity_payload
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


# ============================================================
# 12) Question Machine Validation
# ============================================================

cell37_question_machine_validation = {
    "credit_score_present":
        (
            "نمره اعتباری"
            in
            cell37_fa_surface(
                CELL37_ACCEPTED_QUESTION
            )
        ),

    "indicators_requested":
        (
            "شاخص"
            in
            cell37_fa_surface(
                CELL37_ACCEPTED_QUESTION
            )
        ),

    "calculation_requested":
        (
            "چگونه"
            in
            cell37_fa_surface(
                CELL37_ACCEPTED_QUESTION
            )
        ),

    "source_grounded":
        cell37_evidence_grounding_pass,

    "external_knowledge_used":
        False,
}


cell37_question_machine_pass = all([
    cell37_question_machine_validation[
        "credit_score_present"
    ],

    cell37_question_machine_validation[
        "indicators_requested"
    ],

    cell37_question_machine_validation[
        "calculation_requested"
    ],

    cell37_question_machine_validation[
        "source_grounded"
    ],
])


# ============================================================
# 13) Build Question artifact
# ============================================================

cell37_question_record = {
    "record_version":
        "question_draft_record_v1",

    "question_draft_id":
        cell37_question_draft_id,

    "pilot_case_id":
        CELL37_PILOT_ID,

    "golden_case_id":
        CELL37_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL37_TASK_DRAFT_ID,

    "target_evidence_id":
        cell37_replacement_target_evidence_id,

    "question_text":
        CELL37_ACCEPTED_QUESTION,

    "revision_number":
        1,

    "machine_validation_status":
        (
            "PASS_PENDING_HUMAN_ADJUDICATION"
            if cell37_question_machine_pass
            else
            "FAIL_REQUIRES_REVISION"
        ),

    "human_adjudication_status":
        "PENDING",

    "external_knowledge_used":
        False,
}


# ============================================================
# 14) Register Question
# ============================================================

cell37_question_review_session = (
    cell13_apply_event(
        cell37_question_pending_session,
        "QUESTION_DRAFT_REGISTERED",
        artifact=
            cell37_question_record,
    )
)


cell37_question_registered_valid = (
    cell37_question_review_session.get(
        "current_state"
    )
    ==
    "QUESTION_HUMAN_REVIEW_PENDING"
)


# ============================================================
# 15) Human ACCEPT Question
# ============================================================

CELL37_QUESTION_HUMAN_NOTE = (
    "Human Reviewer accepted the replacement-target question "
    "as natural, useful, and directly grounded in the selected "
    "canonical table region."
)


cell37_answer_pending_session = (
    cell13_apply_event(
        cell37_question_review_session,
        "QUESTION_ACCEPTED",
        human_note=
            CELL37_QUESTION_HUMAN_NOTE,
    )
)


cell37_question_accept_valid = (
    cell37_answer_pending_session.get(
        "current_state"
    )
    ==
    "EXPECTED_ANSWER_DRAFT_PENDING"
)


# ============================================================
# 16) Expected Answer Draft ID
# ============================================================

cell37_answer_identity_payload = {
    "identity_version":
        "retarget_expected_answer_identity_v1",

    "pilot_case_id":
        CELL37_PILOT_ID,

    "question_draft_id":
        cell37_question_draft_id,

    "target_evidence_id":
        cell37_replacement_target_evidence_id,

    "answer_text":
        CELL37_EXPECTED_ANSWER,

    "revision_number":
        1,
}


cell37_expected_answer_draft_id = (
    "expected-answer-draft:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell37_answer_identity_payload
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


# ============================================================
# 17) Expected Answer Machine Validation
# ============================================================

cell37_answer_surface = (
    cell37_fa_surface(
        CELL37_EXPECTED_ANSWER
    )
)


cell37_answer_grounding = {
    "net_income_and_repayment":
        (
            "درآمد خالص ماهانه"
            in
            cell37_answer_surface
            and
            "توان بازپرداخت اقساط"
            in
            cell37_answer_surface
        ),

    "job_stability":
        (
            "پایداری شغل"
            in
            cell37_answer_surface
        ),

    "financial_performance":
        (
            "عملکرد مالی"
            in
            cell37_answer_surface
        ),

    "credit_behavior":
        (
            "رفتار اعتباری"
            in
            cell37_answer_surface
        ),

    "one_to_ten":
        (
            "یک تا ده"
            in
            cell37_answer_surface
        ),

    "divide_by_four":
        (
            "بر چهار تقسیم"
            in
            cell37_answer_surface
        ),

    "credit_score":
        (
            "نمره اعتباری"
            in
            cell37_answer_surface
        ),
}


cell37_answer_machine_pass = all(
    cell37_answer_grounding.values()
)


# ============================================================
# 18) Build Expected Answer artifact
# ============================================================

cell37_expected_answer_record = {
    "record_version":
        "expected_answer_draft_record_v1",

    "expected_answer_draft_id":
        cell37_expected_answer_draft_id,

    "pilot_case_id":
        CELL37_PILOT_ID,

    "golden_case_id":
        CELL37_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL37_TASK_DRAFT_ID,

    "question_draft_id":
        cell37_question_draft_id,

    "target_evidence_id":
        cell37_replacement_target_evidence_id,

    "expected_answer_text":
        CELL37_EXPECTED_ANSWER,

    "revision_number":
        1,

    "machine_validation_status":
        (
            "PASS_PENDING_HUMAN_ADJUDICATION"
            if cell37_answer_machine_pass
            else
            "FAIL_REQUIRES_REVISION"
        ),

    "human_adjudication_status":
        "PENDING",

    "external_knowledge_used":
        False,
}


# ============================================================
# 19) Register Expected Answer Draft
# ============================================================

cell37_current_session = (
    cell13_apply_event(
        cell37_answer_pending_session,
        "EXPECTED_ANSWER_REGISTERED",
        artifact=
            cell37_expected_answer_record,
    )
)


cell37_answer_registered_valid = (
    cell37_current_session.get(
        "current_state"
    )
    ==
    "EXPECTED_ANSWER_HUMAN_REVIEW_PENDING"
)


# ============================================================
# 20) Build active Human Review workspace
# ============================================================

cell37_active_human_review_case = copy.deepcopy(
    cell30_retarget_workspace
)


cell37_active_human_review_case[
    "status"
] = "ACTIVE_REPLACEMENT_TARGET"


cell37_active_human_review_case[
    "workflow_state"
] = (
    "EXPECTED_ANSWER_HUMAN_REVIEW_PENDING"
)


cell37_active_human_review_case[
    "target_evidence_id"
] = (
    cell37_replacement_target_evidence_id
)


cell37_active_human_review_case[
    "source_evidence"
] = copy.deepcopy(
    cell37_replacement_target_text_parts
)


cell37_active_human_review_case[
    "combined_target_text"
] = (
    cell37_replacement_combined_target_text
)


cell37_active_human_review_case[
    "replacement_target_evidence"
] = copy.deepcopy(
    cell37_replacement_target_evidence
)


cell37_active_human_review_case[
    "retarget_status"
] = {
    "decision":
        "RETARGET",

    "current_target_rejected_for_golden_qa":
        True,

    "replacement_selected":
        True,

    "old_target_evidence_id":
        CELL37_OLD_TARGET_EVIDENCE_ID,

    "replacement_target_evidence_id":
        cell37_replacement_target_evidence_id,

    "selected_region_id":
        CELL37_SELECTED_REGION_ID,

    "human_selection":
        "ACCEPT",
}


cell37_active_human_review_case[
    "question_review"
] = {
    "question_draft_id":
        cell37_question_draft_id,

    "question_text":
        CELL37_ACCEPTED_QUESTION,

    "machine_validation_status":
        cell37_question_record[
            "machine_validation_status"
        ],

    "human_decision":
        "ACCEPT",

    "human_note":
        CELL37_QUESTION_HUMAN_NOTE,
}


cell37_active_human_review_case[
    "expected_answer_review"
] = {
    "expected_answer_draft_id":
        cell37_expected_answer_draft_id,

    "expected_answer_text":
        CELL37_EXPECTED_ANSWER,

    "machine_validation_status":
        cell37_expected_answer_record[
            "machine_validation_status"
        ],

    "human_decision":
        "PENDING",
}


cell37_active_human_review_case[
    "citation_review"
] = None


cell37_active_human_review_case[
    "author_final_review"
] = {
    "status":
        "NOT_STARTED",

    "decision":
        None,

    "notes":
        None,
}


# ============================================================
# 21) Persist Registry
# ============================================================

cell37_persistent_registry_payload = copy.deepcopy(
    cell36_persisted_registry
)


cell37_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell37_active_human_review_case
)


cell37_persistent_registry_payload[
    "remaining_pilot_count"
] = 18


cell37_persistent_registry_payload[
    "continuation_state"
] = {
    "next_pilot_id":
        CELL37_PILOT_ID,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell29_remaining_pilot_ids
        ),

    "remaining_count":
        18,

    "continuation_state_sha256":
        cell29_continuation_state_sha256,

    "active_pilot_status":
        "EXPECTED_ANSWER_HUMAN_REVIEW_PENDING",

    "active_target_evidence_id":
        cell37_replacement_target_evidence_id,
}


cell37_persistent_registry_payload[
    "retarget_candidate_discovery"
] = {
    "pilot_case_id":
        CELL37_PILOT_ID,

    "status":
        "REPLACEMENT_SELECTED",

    "selected_region_id":
        CELL37_SELECTED_REGION_ID,

    "old_target_evidence_id":
        CELL37_OLD_TARGET_EVIDENCE_ID,

    "replacement_target_evidence_id":
        cell37_replacement_target_evidence_id,

    "replacement_selected":
        True,

    "human_decision":
        "ACCEPT",
}


cell37_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell37_persistent_registry_payload
    )
)


# ============================================================
# 22) Read-back validation
# ============================================================

cell37_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell37_persisted_active_case = (
    cell37_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell37_persistence_valid = all([
    cell37_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL37_PILOT_ID,

    cell37_persisted_active_case.get(
        "target_evidence_id"
    )
    ==
    cell37_replacement_target_evidence_id,

    cell37_persisted_active_case.get(
        "retarget_status",
        {},
    ).get(
        "replacement_selected"
    )
    is True,

    cell37_persisted_active_case.get(
        "question_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell37_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "PENDING",

    cell37_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    18,
])


# ============================================================
# 23) Safety
# ============================================================

cell37_queue_still_18 = (
    len(
        cell29_remaining_pilot_ids
    )
    ==
    18
)


cell37_current_pilot_still_first = (
    cell29_remaining_pilot_ids[
        0
    ]
    ==
    CELL37_PILOT_ID
)


cell37_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell37_full_572_blocked = (
    CELL37_FULL_572_BLOCK_STATUS
    ==
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


# ============================================================
# 24) Final Gate
# ============================================================

cell37_gate = all([
    len(
        cell37_failures
    )
    ==
    0,

    cell37_evidence_grounding_pass,

    cell37_target_parts_valid,

    cell37_session_start_valid,

    cell37_question_machine_pass,

    cell37_question_registered_valid,

    cell37_question_accept_valid,

    cell37_answer_machine_pass,

    cell37_answer_registered_valid,

    cell37_persistence_valid,

    cell37_queue_still_18,

    cell37_current_pilot_still_first,

    cell37_base_seed_unchanged,

    cell37_full_572_blocked,
])


# ============================================================
# 25) Diagnostics
# ============================================================

print(
    "=== RETARGET SELECTED + QUESTION ACCEPTED "
    "+ EXPECTED ANSWER DRAFT ==="
)


print(
    "\n=== REPLACEMENT TARGET ==="
)

print(
    "Pilot:",
    CELL37_PILOT_ID,
)

print(
    "Old Target:",
    CELL37_OLD_TARGET_EVIDENCE_ID,
)

print(
    "Selected Region:",
    CELL37_SELECTED_REGION_ID,
)

print(
    "Replacement Target:",
    cell37_replacement_target_evidence_id,
)

print(
    "Target Parts:",
    len(
        cell37_replacement_target_parts
    ),
)


print(
    "\n=== SOURCE EVIDENCE ==="
)

print(
    cell37_replacement_combined_target_text
)


print(
    "\n=== ACCEPTED QUESTION ==="
)

print(
    CELL37_ACCEPTED_QUESTION
)

print(
    "Question Draft ID:",
    cell37_question_draft_id,
)

print(
    "Question Machine PASS:",
    cell37_question_machine_pass,
)


print(
    "\n=== EXPECTED ANSWER DRAFT ==="
)

print(
    CELL37_EXPECTED_ANSWER
)

print(
    "Expected Answer Draft ID:",
    cell37_expected_answer_draft_id,
)

print(
    "Machine PASS:",
    cell37_answer_machine_pass,
)


print(
    "\n=== ANSWER GROUNDING ==="
)

for name, status in (
    cell37_answer_grounding.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== WORKFLOW ==="
)

print(
    "Current State:",
    cell37_current_session.get(
        "current_state"
    ),
)

print(
    "Retarget:",
    "ACCEPT",
)

print(
    "Question Decision:",
    "ACCEPT",
)

print(
    "Expected Answer Decision:",
    "PENDING",
)

print(
    "Citation:",
    None,
)


print(
    "\n=== PERSISTENCE ==="
)

print(
    "Registry:",
    cell21_registry_path,
)

print(
    "Persisted Replacement Target:",
    cell37_persisted_active_case.get(
        "target_evidence_id"
    ),
)

print(
    "Persisted Question Decision:",
    cell37_persisted_active_case.get(
        "question_review",
        {},
    ).get(
        "human_decision"
    ),
)

print(
    "Persisted Answer Decision:",
    cell37_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    ),
)

print(
    "Persistence valid:",
    cell37_persistence_valid,
)


print(
    "\n=== QUEUE / SAFETY ==="
)

print(
    "Queue still 18:",
    cell37_queue_still_18,
)

print(
    "Base Seed unchanged:",
    cell37_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell37_full_572_blocked,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell37_failures
)


print(
    "\nCODE CELL 37 GATE:",
    (
        "PASS"
        if cell37_gate
        else "FAIL"
    ),
)


assert cell37_gate, (
    "Replacement Target selection / Question ACCEPT / "
    "Expected Answer Draft failed. Repair this SAME "
    "Code Cell 37 before Answer Human Review."
)

=== RETARGET SELECTED + QUESTION ACCEPTED + EXPECTED ANSWER DRAFT ===

=== REPLACEMENT TARGET ===
Pilot: question-pilot:3023c1c546df2fe2211f79702b6e3dbb
Old Target: target-evidence:cccb56b3cc1c8c0f2006c44010235fc4
Selected Region: table-region:68e3e189e456efd98c14b4f380b29703
Replacement Target: target-evidence:843545be6778b09e04b5f9ef650e518f
Target Parts: 8

=== SOURCE EVIDENCE ===
شاخص
امتیاز از یک تا ده
درآمد خالص ماهانه و توان بازپرداخت اقساط
پایداری شغل و کسب و کار
عملکرد مالی
رفتار اعتباری
جمع امتیاز
جمع امتیاز تقسیم بر چهار: نمره اعتباری شما

=== ACCEPTED QUESTION ===
برای محاسبه نمره اعتباری، چه شاخص‌هایی بررسی می‌شوند و نمره نهایی چگونه به دست می‌آید؟
Question Draft ID: question-draft:34e06328275a298f3d54edf460f543a5
Question Machine PASS: True

=== EXPECTED ANSWER DRAFT ===
چهار شاخص شامل «درآمد خالص ماهانه و توان بازپرداخت اقساط»، «پایداری شغل و کسب‌وکار»، «عملکرد مالی» و «رفتار اعتباری» هستند. هر شاخص از یک تا ده امتیاز می‌گیرد؛ سپس امتیازها با هم جمع می‌شوند و مجموع امتیا

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 38 — Expected Answer ACCEPT + Citation Draft + Registry Sync

Expected Answer مربوط به Replacement Target توسط Human Reviewer پذیرفته شد.

در این مرحله:

- Expected Answer با تصمیم `ACCEPT` ثبت می‌شود؛
- Citation فقط از ۸ Target Part معتبرِ Replacement Evidence ساخته می‌شود؛
- `paragraph_id` و `paragraph_number` مستقیماً از Canonical Book Ingestion حفظ می‌شوند؛
- هیچ Locator جدید یا منبع خارجی ساخته نمی‌شود؛
- Citation به Human Review می‌رود؛
- Registry دائمی Sync می‌شود؛
- Queue همچنان 18 باقی می‌ماند؛
- Author Final Review هنوز `NOT_STARTED` است.

</div>

In [38]:
# Code Cell 38

cell38_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell37_gate") is not True:
    cell38_failures.append(
        "CELL37_NOT_PASS"
    )


if (
    globals().get(
        "cell37_current_session",
        {},
    ).get(
        "current_state"
    )
    !=
    "EXPECTED_ANSWER_HUMAN_REVIEW_PENDING"
):
    cell38_failures.append(
        "EXPECTED_ANSWER_NOT_AT_HUMAN_REVIEW_GATE"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell38_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Current identities
# ============================================================

CELL38_PILOT_ID = (
    CELL37_PILOT_ID
)

CELL38_GOLDEN_CASE_ID = (
    CELL37_GOLDEN_CASE_ID
)

CELL38_TASK_DRAFT_ID = (
    CELL37_TASK_DRAFT_ID
)

CELL38_TARGET_EVIDENCE_ID = (
    cell37_replacement_target_evidence_id
)

CELL38_QUESTION_DRAFT_ID = (
    cell37_question_draft_id
)

CELL38_EXPECTED_ANSWER_DRAFT_ID = (
    cell37_expected_answer_draft_id
)

CELL38_FULL_572_BLOCK_STATUS = (
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


# ============================================================
# 3) Human ACCEPT for Expected Answer
# ============================================================

CELL38_HUMAN_ANSWER_NOTE = (
    "Human Reviewer accepted the Expected Answer after "
    "reviewing it against the selected canonical table evidence."
)


cell38_answer_human_review = {
    "review_record_version":
        "human_expected_answer_review_record_v1",

    "pilot_case_id":
        CELL38_PILOT_ID,

    "question_draft_id":
        CELL38_QUESTION_DRAFT_ID,

    "expected_answer_draft_id":
        CELL38_EXPECTED_ANSWER_DRAFT_ID,

    "target_evidence_id":
        CELL38_TARGET_EVIDENCE_ID,

    "human_decision":
        "ACCEPT",

    "human_note":
        CELL38_HUMAN_ANSWER_NOTE,

    "human_context_used_as_evidence":
        False,

    "external_knowledge_used":
        False,
}


# ============================================================
# 4) Apply Expected Answer ACCEPT
# ============================================================

cell38_citation_pending_session = (
    cell13_apply_event(
        cell37_current_session,
        "EXPECTED_ANSWER_ACCEPTED",
        human_note=
            CELL38_HUMAN_ANSWER_NOTE,
    )
)


cell38_answer_accept_valid = all([
    cell38_citation_pending_session.get(
        "current_state"
    )
    ==
    "CITATION_MAPPING_PENDING",

    cell38_citation_pending_session.get(
        "queue_advance_performed"
    )
    is False,
])


# ============================================================
# 5) Build Citation entries from Replacement Target Parts
# ============================================================

cell38_target_parts = copy.deepcopy(
    cell37_replacement_target_parts
)


cell38_citation_entries = []


for index, part in enumerate(
    cell38_target_parts
):

    if not isinstance(
        part,
        dict,
    ):
        continue


    cell38_citation_entries.append({
        "citation_index":
            index,

        "target_evidence_id":
            CELL38_TARGET_EVIDENCE_ID,

        "document_id":
            part.get(
                "document_id"
            ),

        "source_id":
            part.get(
                "source_id"
            ),

        "role":
            part.get(
                "role"
            ),

        "char_start":
            part.get(
                "char_start"
            ),

        "char_end":
            part.get(
                "char_end"
            ),

        "char_count":
            part.get(
                "char_count"
            ),

        "locator":
            copy.deepcopy(
                part.get(
                    "locator",
                    {},
                )
            ),

        "attribution":
            copy.deepcopy(
                part.get(
                    "attribution"
                )
            ),

        "locator_origin":
            "COPIED_FROM_REPLACEMENT_TARGET_EVIDENCE",

        "external_locator_lookup_used":
            False,
    })


# ============================================================
# 6) Exact Citation projection validation
# ============================================================

cell38_source_projection = [
    {
        "document_id":
            part.get(
                "document_id"
            ),

        "source_id":
            part.get(
                "source_id"
            ),

        "role":
            part.get(
                "role"
            ),

        "char_start":
            part.get(
                "char_start"
            ),

        "char_end":
            part.get(
                "char_end"
            ),

        "locator":
            copy.deepcopy(
                part.get(
                    "locator",
                    {},
                )
            ),
    }

    for part
    in cell38_target_parts
]


cell38_citation_projection = [
    {
        "document_id":
            entry.get(
                "document_id"
            ),

        "source_id":
            entry.get(
                "source_id"
            ),

        "role":
            entry.get(
                "role"
            ),

        "char_start":
            entry.get(
                "char_start"
            ),

        "char_end":
            entry.get(
                "char_end"
            ),

        "locator":
            copy.deepcopy(
                entry.get(
                    "locator",
                    {},
                )
            ),
    }

    for entry
    in cell38_citation_entries
]


cell38_no_locator_fabrication = (
    cell38_source_projection
    ==
    cell38_citation_projection
)


# ============================================================
# 7) Citation entry validation
# ============================================================

cell38_citation_validation_rows = []


for entry in cell38_citation_entries:

    locator = (
        entry.get(
            "locator"
        )
        or
        {}
    )


    row_valid = all([
        isinstance(
            entry.get(
                "document_id"
            ),
            str,
        ),

        entry.get(
            "role"
        )
        ==
        "PROTECTED_REGION_MEMBER",

        entry.get(
            "char_start"
        )
        ==
        0,

        isinstance(
            entry.get(
                "char_end"
            ),
            int,
        ),

        entry.get(
            "char_end"
        )
        >
        0,

        locator.get(
            "paragraph_id"
        )
        ==
        entry.get(
            "document_id"
        ),

        isinstance(
            locator.get(
                "paragraph_number"
            ),
            int,
        ),

        entry.get(
            "locator_origin"
        )
        ==
        "COPIED_FROM_REPLACEMENT_TARGET_EVIDENCE",

        entry.get(
            "external_locator_lookup_used"
        )
        is False,
    ])


    cell38_citation_validation_rows.append({
        "citation_index":
            entry.get(
                "citation_index"
            ),

        "document_id":
            entry.get(
                "document_id"
            ),

        "paragraph_number":
            locator.get(
                "paragraph_number"
            ),

        "valid":
            row_valid,
    })


cell38_all_entries_valid = all([
    len(
        cell38_citation_entries
    )
    ==
    8,

    len(
        cell38_citation_validation_rows
    )
    ==
    8,

    all(
        row.get(
            "valid"
        )
        is True

        for row
        in cell38_citation_validation_rows
    ),
])


# ============================================================
# 8) Deterministic Citation Mapping ID
# ============================================================

cell38_citation_identity_payload = {
    "identity_version":
        "retarget_citation_mapping_identity_v1",

    "pilot_case_id":
        CELL38_PILOT_ID,

    "question_draft_id":
        CELL38_QUESTION_DRAFT_ID,

    "expected_answer_draft_id":
        CELL38_EXPECTED_ANSWER_DRAFT_ID,

    "target_evidence_id":
        CELL38_TARGET_EVIDENCE_ID,

    "selected_region_id":
        CELL37_SELECTED_REGION_ID,

    "citation_projection":
        copy.deepcopy(
            cell38_citation_projection
        ),
}


cell38_citation_mapping_id = (
    "citation-mapping:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell38_citation_identity_payload
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


# ============================================================
# 9) Citation Machine Validation
# ============================================================

cell38_citation_machine_validation = {
    "exactly_eight_target_parts":
        (
            len(
                cell38_target_parts
            )
            ==
            8
        ),

    "exactly_eight_citation_entries":
        (
            len(
                cell38_citation_entries
            )
            ==
            8
        ),

    "all_entries_valid":
        cell38_all_entries_valid,

    "locator_fabrication_absent":
        cell38_no_locator_fabrication,

    "all_target_parts_covered":
        (
            len(
                cell38_citation_entries
            )
            ==
            len(
                cell38_target_parts
            )
        ),

    "target_evidence_identity_match":
        all(
            entry.get(
                "target_evidence_id"
            )
            ==
            CELL38_TARGET_EVIDENCE_ID

            for entry
            in cell38_citation_entries
        ),

    "canonical_paragraph_locators_present":
        all(
            isinstance(
                entry.get(
                    "locator",
                    {},
                ).get(
                    "paragraph_number"
                ),
                int,
            )

            for entry
            in cell38_citation_entries
        ),

    "external_locator_lookup_used":
        False,
}


cell38_citation_machine_pass = all([
    cell38_citation_machine_validation[
        "exactly_eight_target_parts"
    ],

    cell38_citation_machine_validation[
        "exactly_eight_citation_entries"
    ],

    cell38_citation_machine_validation[
        "all_entries_valid"
    ],

    cell38_citation_machine_validation[
        "locator_fabrication_absent"
    ],

    cell38_citation_machine_validation[
        "all_target_parts_covered"
    ],

    cell38_citation_machine_validation[
        "target_evidence_identity_match"
    ],

    cell38_citation_machine_validation[
        "canonical_paragraph_locators_present"
    ],
])


# ============================================================
# 10) Citation Mapping Record
# ============================================================

cell38_citation_mapping_record = {
    "record_version":
        "citation_mapping_record_v1",

    "citation_mapping_id":
        cell38_citation_mapping_id,

    "pilot_case_id":
        CELL38_PILOT_ID,

    "golden_case_id":
        CELL38_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL38_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL38_TARGET_EVIDENCE_ID,

    "question_draft_id":
        CELL38_QUESTION_DRAFT_ID,

    "expected_answer_draft_id":
        CELL38_EXPECTED_ANSWER_DRAFT_ID,

    "selected_region_id":
        CELL37_SELECTED_REGION_ID,

    "citation_entries":
        copy.deepcopy(
            cell38_citation_entries
        ),

    "citation_scope":
        "REPLACEMENT_VALIDATED_TARGET_EVIDENCE_ONLY",

    "locator_source":
        "CANONICAL_BOOK_INGESTION_V2",

    "external_locator_lookup_used":
        False,

    "machine_validation_status":
        (
            "PASS_PENDING_HUMAN_ADJUDICATION"
            if cell38_citation_machine_pass
            else
            "FAIL_REQUIRES_REVISION"
        ),

    "human_adjudication_status":
        "PENDING",

    "full_572_generation":
        CELL38_FULL_572_BLOCK_STATUS,
}


# ============================================================
# 11) Register Citation
# ============================================================

cell38_current_session = (
    cell13_apply_event(
        cell38_citation_pending_session,
        "CITATION_MAPPING_REGISTERED",
        artifact=
            cell38_citation_mapping_record,
    )
)


cell38_citation_registered_valid = all([
    cell38_current_session.get(
        "current_state"
    )
    ==
    "CITATION_HUMAN_REVIEW_PENDING",

    isinstance(
        cell38_current_session.get(
            "artifacts",
            {}
        ).get(
            "citation_mapping"
        ),
        dict,
    ),

    cell38_current_session[
        "artifacts"
    ][
        "citation_mapping"
    ].get(
        "citation_mapping_id"
    )
    ==
    cell38_citation_mapping_id,

    cell38_current_session.get(
        "queue_advance_performed"
    )
    is False,
])


# ============================================================
# 12) Update Human Review workspace
# ============================================================

cell38_active_human_review_case = copy.deepcopy(
    cell37_active_human_review_case
)


cell38_active_human_review_case[
    "workflow_state"
] = "CITATION_HUMAN_REVIEW_PENDING"


cell38_active_human_review_case[
    "expected_answer_review"
][
    "human_decision"
] = "ACCEPT"


cell38_active_human_review_case[
    "expected_answer_review"
][
    "human_review"
] = copy.deepcopy(
    cell38_answer_human_review
)


cell38_active_human_review_case[
    "citation_review"
] = {
    "citation_mapping_id":
        cell38_citation_mapping_id,

    "citation_entries":
        copy.deepcopy(
            cell38_citation_entries
        ),

    "machine_validation_status":
        cell38_citation_mapping_record[
            "machine_validation_status"
        ],

    "human_decision":
        "PENDING",
}


cell38_active_human_review_case[
    "final_status"
] = (
    "RETARGET_QUESTION_AND_ANSWER_ACCEPTED_"
    "CITATION_PENDING_REVIEW"
)


# ============================================================
# 13) Persist Registry
# ============================================================

cell38_persistent_registry_payload = copy.deepcopy(
    cell37_persisted_registry
)


cell38_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell38_active_human_review_case
)


cell38_persistent_registry_payload[
    "remaining_pilot_count"
] = 18


cell38_persistent_registry_payload[
    "continuation_state"
] = {
    "next_pilot_id":
        CELL38_PILOT_ID,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell29_remaining_pilot_ids
        ),

    "remaining_count":
        18,

    "continuation_state_sha256":
        cell29_continuation_state_sha256,

    "active_pilot_status":
        "CITATION_HUMAN_REVIEW_PENDING",

    "active_target_evidence_id":
        CELL38_TARGET_EVIDENCE_ID,
}


cell38_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell38_persistent_registry_payload
    )
)


# ============================================================
# 14) Read-back validation
# ============================================================

cell38_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell38_persisted_active_case = (
    cell38_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell38_persistence_valid = all([
    cell38_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL38_PILOT_ID,

    cell38_persisted_active_case.get(
        "target_evidence_id"
    )
    ==
    CELL38_TARGET_EVIDENCE_ID,

    cell38_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell38_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "citation_mapping_id"
    )
    ==
    cell38_citation_mapping_id,

    cell38_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "PENDING",

    cell38_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    18,
])


# ============================================================
# 15) Safety
# ============================================================

cell38_queue_still_18 = (
    len(
        cell29_remaining_pilot_ids
    )
    ==
    18
)


cell38_current_pilot_still_first = (
    cell29_remaining_pilot_ids[
        0
    ]
    ==
    CELL38_PILOT_ID
)


cell38_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell38_full_572_blocked = (
    cell38_citation_mapping_record[
        "full_572_generation"
    ]
    ==
    CELL38_FULL_572_BLOCK_STATUS
)


# ============================================================
# 16) Final Gate
# ============================================================

cell38_gate = all([
    len(
        cell38_failures
    )
    ==
    0,

    cell38_answer_accept_valid,

    cell38_no_locator_fabrication,

    cell38_all_entries_valid,

    cell38_citation_machine_pass,

    cell38_citation_registered_valid,

    cell38_persistence_valid,

    cell38_queue_still_18,

    cell38_current_pilot_still_first,

    cell38_base_seed_unchanged,

    cell38_full_572_blocked,
])


# ============================================================
# 17) Diagnostics
# ============================================================

print(
    "=== ANSWER ACCEPTED + RETARGET CITATION DRAFT "
    "+ REGISTRY SYNC ==="
)


print(
    "\n=== ACCEPTED QUESTION ==="
)

print(
    CELL37_ACCEPTED_QUESTION
)


print(
    "\n=== ACCEPTED ANSWER ==="
)

print(
    CELL37_EXPECTED_ANSWER
)

print(
    "Human Answer Decision:",
    "ACCEPT",
)


print(
    "\n=== CITATION DRAFT ==="
)

print(
    "Citation Mapping ID:",
    cell38_citation_mapping_id,
)

print(
    "Target Evidence:",
    CELL38_TARGET_EVIDENCE_ID,
)

print(
    "Selected Region:",
    CELL37_SELECTED_REGION_ID,
)

print(
    "Citation entry count:",
    len(
        cell38_citation_entries
    ),
)


for entry in cell38_citation_entries:

    print(
        "\n--- CITATION ENTRY",
        entry.get(
            "citation_index"
        ),
        "---"
    )

    print(
        "Document:",
        entry.get(
            "document_id"
        ),
    )

    print(
        "Role:",
        entry.get(
            "role"
        ),
    )

    print(
        "Chars:",
        (
            entry.get(
                "char_start"
            ),
            entry.get(
                "char_end"
            ),
        ),
    )

    print(
        "Paragraph Number:",
        entry.get(
            "locator",
            {},
        ).get(
            "paragraph_number"
        ),
    )


print(
    "\n=== CITATION MACHINE VALIDATION ==="
)

for name, status in (
    cell38_citation_machine_validation.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "Machine PASS:",
    cell38_citation_machine_pass,
)

print(
    "No locator fabrication:",
    cell38_no_locator_fabrication,
)


print(
    "\n=== WORKFLOW ==="
)

print(
    "Current State:",
    cell38_current_session.get(
        "current_state"
    ),
)

print(
    "Retarget:",
    "ACCEPT",
)

print(
    "Question Decision:",
    "ACCEPT",
)

print(
    "Answer Decision:",
    "ACCEPT",
)

print(
    "Citation Decision:",
    "PENDING",
)


print(
    "\n=== PERSISTENCE ==="
)

print(
    "Registry:",
    cell21_registry_path,
)

print(
    "Persisted Answer Decision:",
    cell38_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    ),
)

print(
    "Persisted Citation Decision:",
    cell38_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "human_decision"
    ),
)

print(
    "Persistence valid:",
    cell38_persistence_valid,
)


print(
    "\n=== QUEUE / SAFETY ==="
)

print(
    "Queue still 18:",
    cell38_queue_still_18,
)

print(
    "Base Seed unchanged:",
    cell38_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell38_full_572_blocked,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell38_failures
)


print(
    "\nCODE CELL 38 GATE:",
    (
        "PASS"
        if cell38_gate
        else "FAIL"
    ),
)


assert cell38_gate, (
    "Expected Answer ACCEPT / Retarget Citation Draft / "
    "Registry Sync failed. Repair this SAME Code Cell 38 "
    "before Citation Human Review."
)

=== ANSWER ACCEPTED + RETARGET CITATION DRAFT + REGISTRY SYNC ===

=== ACCEPTED QUESTION ===
برای محاسبه نمره اعتباری، چه شاخص‌هایی بررسی می‌شوند و نمره نهایی چگونه به دست می‌آید؟

=== ACCEPTED ANSWER ===
چهار شاخص شامل «درآمد خالص ماهانه و توان بازپرداخت اقساط»، «پایداری شغل و کسب‌وکار»، «عملکرد مالی» و «رفتار اعتباری» هستند. هر شاخص از یک تا ده امتیاز می‌گیرد؛ سپس امتیازها با هم جمع می‌شوند و مجموع امتیاز بر چهار تقسیم می‌شود تا نمره اعتباری به دست آید.
Human Answer Decision: ACCEPT

=== CITATION DRAFT ===
Citation Mapping ID: citation-mapping:9a6c268221ce7404d11edf5a8863d461
Target Evidence: target-evidence:843545be6778b09e04b5f9ef650e518f
Selected Region: table-region:68e3e189e456efd98c14b4f380b29703
Citation entry count: 8

--- CITATION ENTRY 0 ---
Document: book:adult-financial-literacy-course:fa-source:p000463
Role: PROTECTED_REGION_MEMBER
Chars: (0, 4)
Paragraph Number: 463

--- CITATION ENTRY 1 ---
Document: book:adult-financial-literacy-course:fa-source:p000464
Role: PROTECTE

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 39 — Citation ACCEPT + Retargeted Pilot Completion + Queue Advance

Citation مربوط به Replacement Target توسط Human Reviewer پذیرفته شد.

در این مرحله:

- Citation با تصمیم `ACCEPT` ثبت می‌شود؛
- Pilot رتارگت‌شده به‌طور کامل بسته می‌شود؛
- Lineage مربوط به Target قبلی و Replacement Target حفظ می‌شود؛
- پرونده Human Review به Completed Cases منتقل می‌شود؛
- Author Final Review همچنان `NOT_STARTED` باقی می‌ماند؛
- Queue از 18 به 17 کاهش پیدا می‌کند؛
- Pilot بعدی و Source Evidence آن آماده می‌شود؛
- Registry دائمی Sync می‌شود؛
- Full 572 Generation همچنان Blocked باقی می‌ماند.

</div>

In [39]:
# Code Cell 39

cell39_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell38_gate") is not True:
    cell39_failures.append(
        "CELL38_NOT_PASS"
    )


if (
    globals().get(
        "cell38_current_session",
        {},
    ).get(
        "current_state"
    )
    !=
    "CITATION_HUMAN_REVIEW_PENDING"
):
    cell39_failures.append(
        "CITATION_NOT_AT_HUMAN_REVIEW_GATE"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell39_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Current completed Pilot identity
# ============================================================

CELL39_COMPLETED_PILOT_ID = (
    CELL38_PILOT_ID
)

CELL39_COMPLETED_TARGET_ID = (
    CELL38_TARGET_EVIDENCE_ID
)

CELL39_FULL_572_BLOCK_STATUS = (
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


# ============================================================
# 3) Human Citation ACCEPT
# ============================================================

CELL39_HUMAN_CITATION_NOTE = (
    "Human Reviewer accepted the Citation Mapping for the "
    "retargeted Pilot after reviewing all canonical paragraph "
    "locators against the selected replacement table region."
)


cell39_citation_human_review = {
    "review_record_version":
        "human_citation_review_record_v1",

    "pilot_case_id":
        CELL39_COMPLETED_PILOT_ID,

    "citation_mapping_id":
        cell38_citation_mapping_id,

    "target_evidence_id":
        CELL39_COMPLETED_TARGET_ID,

    "selected_region_id":
        CELL37_SELECTED_REGION_ID,

    "human_decision":
        "ACCEPT",

    "human_note":
        CELL39_HUMAN_CITATION_NOTE,

    "external_locator_lookup_used":
        False,
}


# ============================================================
# 4) Apply Citation ACCEPT
# ============================================================

cell39_pre_advance_session = (
    cell13_apply_event(
        cell38_current_session,
        "CITATION_ACCEPTED",
        human_note=
            CELL39_HUMAN_CITATION_NOTE,
    )
)


cell39_citation_accept_valid = all([
    cell39_pre_advance_session.get(
        "current_state"
    )
    ==
    "PILOT_COMPLETE_PENDING_QUEUE_ADVANCE",

    cell39_pre_advance_session.get(
        "queue_advance_permitted"
    )
    is True,

    cell39_pre_advance_session.get(
        "queue_advance_performed"
    )
    is False,

    len(
        cell39_pre_advance_session.get(
            "human_decisions",
            [],
        )
    )
    ==
    3,

    cell39_pre_advance_session[
        "human_decisions"
    ][
        -1
    ].get(
        "event"
    )
    ==
    "CITATION_ACCEPTED",
])


# ============================================================
# 5) Finalize Human Review Case
# ============================================================

cell39_completed_human_review_case = copy.deepcopy(
    cell38_active_human_review_case
)


cell39_completed_human_review_case[
    "workflow_state"
] = "PILOT_COMPLETE"


cell39_completed_human_review_case[
    "citation_review"
][
    "human_decision"
] = "ACCEPT"


cell39_completed_human_review_case[
    "citation_review"
][
    "human_review"
] = copy.deepcopy(
    cell39_citation_human_review
)


cell39_completed_human_review_case[
    "author_final_review"
] = {
    "status":
        "NOT_STARTED",

    "decision":
        None,

    "notes":
        None,
}


cell39_completed_human_review_case[
    "final_status"
] = (
    "HUMAN_REVIEW_COMPLETE_"
    "AUTHOR_FINAL_REVIEW_PENDING"
)


cell39_completed_human_review_case[
    "retarget_completion"
] = {
    "retarget_used":
        True,

    "old_target_evidence_id":
        CELL37_OLD_TARGET_EVIDENCE_ID,

    "replacement_target_evidence_id":
        CELL39_COMPLETED_TARGET_ID,

    "selected_region_id":
        CELL37_SELECTED_REGION_ID,

    "replacement_human_decision":
        "ACCEPT",
}


cell39_completed_human_review_case_sha256 = (
    cell10_stable_sha(
        cell39_completed_human_review_case
    )
)


# ============================================================
# 6) Queue preconditions
# ============================================================

cell39_parent_remaining_ids = copy.deepcopy(
    cell29_continuation_state[
        "remaining_pilot_ids"
    ]
)


cell39_parent_remaining_queue = copy.deepcopy(
    cell29_continuation_state[
        "remaining_pilot_queue"
    ]
)


cell39_queue_preconditions_valid = all([
    len(
        cell39_parent_remaining_ids
    )
    ==
    18,

    len(
        cell39_parent_remaining_queue
    )
    ==
    18,

    cell39_parent_remaining_ids[
        0
    ]
    ==
    CELL39_COMPLETED_PILOT_ID,

    cell39_citation_accept_valid,
])


# ============================================================
# 7) Advance Queue
# ============================================================

if cell39_queue_preconditions_valid:

    cell39_remaining_pilot_ids = copy.deepcopy(
        cell39_parent_remaining_ids[
            1:
        ]
    )

    cell39_remaining_pilot_queue = copy.deepcopy(
        cell39_parent_remaining_queue[
            1:
        ]
    )

else:

    cell39_remaining_pilot_ids = []
    cell39_remaining_pilot_queue = []


cell39_next_pilot_id = (
    cell39_remaining_pilot_ids[
        0
    ]

    if cell39_remaining_pilot_ids

    else None
)


cell39_next_pilot_queue_entry = (
    copy.deepcopy(
        cell39_remaining_pilot_queue[
            0
        ]
    )

    if cell39_remaining_pilot_queue

    else None
)


cell39_queue_advanced_valid = all([
    len(
        cell39_remaining_pilot_ids
    )
    ==
    17,

    len(
        cell39_remaining_pilot_queue
    )
    ==
    17,

    CELL39_COMPLETED_PILOT_ID
    not in
    cell39_remaining_pilot_ids,

    isinstance(
        cell39_next_pilot_id,
        str,
    ),

    cell39_next_pilot_id
    !=
    CELL39_COMPLETED_PILOT_ID,
])


# ============================================================
# 8) Completed Engine session
# ============================================================

cell39_completed_session = copy.deepcopy(
    cell39_pre_advance_session
)


cell39_completed_session[
    "current_state"
] = "PILOT_COMPLETE"


cell39_completed_session[
    "queue_advance_permitted"
] = False


cell39_completed_session[
    "queue_advance_performed"
] = True


cell39_completed_session[
    "completion_status"
] = (
    "HUMAN_REVIEW_COMPLETE_"
    "AUTHOR_FINAL_REVIEW_PENDING"
)


# ============================================================
# 9) New authoritative Continuation State
# ============================================================

cell39_continuation_state = {
    "state_version":
        "compact_pilot_continuation_state_v5",

    "authoritative_forward_state":
        True,

    "parent_state_sha256":
        cell29_continuation_state_sha256,

    "just_completed_pilot_id":
        CELL39_COMPLETED_PILOT_ID,

    "just_completed_target_evidence_id":
        CELL39_COMPLETED_TARGET_ID,

    "just_completed_case_sha256":
        cell39_completed_human_review_case_sha256,

    "retarget_used":
        True,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell39_remaining_pilot_ids
        ),

    "remaining_pilot_queue":
        copy.deepcopy(
            cell39_remaining_pilot_queue
        ),

    "remaining_count":
        17,

    "next_pilot_id":
        cell39_next_pilot_id,

    "next_pilot_queue_entry":
        copy.deepcopy(
            cell39_next_pilot_queue_entry
        ),

    "full_572_generation":
        CELL39_FULL_572_BLOCK_STATUS,
}


cell39_continuation_state_sha256 = (
    cell10_stable_sha(
        cell39_continuation_state
    )
)


# ============================================================
# 10) Resolve next Pilot packet
# ============================================================

cell39_pilot_by_id = (
    cell13_seed[
        "final_runtime"
    ][
        "question_drafting_pilot_by_id"
    ]
)


cell39_next_pilot_packet = copy.deepcopy(
    cell39_pilot_by_id.get(
        cell39_next_pilot_id
    )
)


cell39_next_packet_valid = isinstance(
    cell39_next_pilot_packet,
    dict,
)


# ============================================================
# 11) Resolve identity chain
# ============================================================

cell39_next_golden_case_id = None
cell39_next_task_draft_id = None
cell39_next_target_evidence_id = None


if cell39_next_packet_valid:

    try:

        cell39_next_golden_case_id = (
            cell13_exact_single_id(
                cell39_next_pilot_packet,
                "golden_case",
            )
        )


        cell39_next_task_draft_id = (
            cell13_exact_single_id(
                cell39_next_pilot_packet,
                "task_draft",
            )
        )


        cell39_next_target_evidence_id = (
            cell13_exact_single_id(
                cell39_next_pilot_packet,
                "target_evidence",
            )
        )


    except Exception as exc:

        cell39_failures.append({
            "reason":
                "NEXT_ID_CHAIN_FAILED",

            "error":
                repr(
                    exc
                ),
        })


cell39_next_identity_valid = all([
    isinstance(
        cell39_next_golden_case_id,
        str,
    ),

    isinstance(
        cell39_next_task_draft_id,
        str,
    ),

    isinstance(
        cell39_next_target_evidence_id,
        str,
    ),
])


# ============================================================
# 12) Resolve next support records
# ============================================================

cell39_next_golden_case = None
cell39_next_task_plan = None
cell39_next_target_evidence = None


if cell39_next_identity_valid:

    try:

        cell39_next_golden_case = copy.deepcopy(
            cell13_active_support[
                "golden_case_records_by_id"
            ][
                cell39_next_golden_case_id
            ]
        )


        cell39_next_task_plan = copy.deepcopy(
            cell13_find_unique_mapping_value(
                cell13_active_support.get(
                    "task_draft_plans",
                    {},
                ),
                "task_draft",
                cell39_next_task_draft_id,
            )[
                "value"
            ]
        )


        cell39_next_target_evidence = copy.deepcopy(
            cell13_find_unique_list_record(
                cell13_active_support.get(
                    "target_evidence_records",
                    [],
                ),
                "target_evidence_id",
                cell39_next_target_evidence_id,
            )
        )


    except Exception as exc:

        cell39_failures.append({
            "reason":
                "NEXT_SUPPORT_RESOLUTION_FAILED",

            "error":
                repr(
                    exc
                ),
        })


# ============================================================
# 13) Classification helper
# ============================================================

def cell39_scalar(
    mapping,
    names,
):

    match = cell14_find_first_scalar(
        mapping,
        names,
    )


    if isinstance(
        match,
        dict,
    ):

        return match.get(
            "value"
        )


    return None


cell39_next_source_type = cell39_scalar(
    cell39_next_pilot_packet,
    [
        "source_type",
        "source_kind",
        "source",
    ],
)


cell39_next_stratum = cell39_scalar(
    cell39_next_pilot_packet,
    [
        "stratum",
        "stratum_name",
        "stratum_id",
    ],
)


cell39_next_task_family = cell39_scalar(
    cell39_next_pilot_packet,
    [
        "task_family",
    ],
)


cell39_next_question_archetype = cell39_scalar(
    cell39_next_pilot_packet,
    [
        "question_archetype",
    ],
)


# ============================================================
# 14) Extract next Source Evidence
# ============================================================

cell39_raw_target_parts = []


if cell39_next_packet_valid:

    packet_parts = (
        cell39_next_pilot_packet.get(
            "target_text_parts"
        )
    )


    if isinstance(
        packet_parts,
        list,
    ):

        cell39_raw_target_parts = copy.deepcopy(
            packet_parts
        )


if (
    not cell39_raw_target_parts
    and
    isinstance(
        cell39_next_target_evidence,
        dict,
    )
):

    evidence_parts = (
        cell39_next_target_evidence.get(
            "target_parts"
        )
    )


    if isinstance(
        evidence_parts,
        list,
    ):

        cell39_raw_target_parts = copy.deepcopy(
            evidence_parts
        )


cell39_next_target_text_parts = []


for index, part in enumerate(
    cell39_raw_target_parts
):

    if not isinstance(
        part,
        dict,
    ):
        continue


    text_value = None


    for field_name in (
        "text",
        "target_text",
        "content",
        "evidence_text",
        "text_content",
    ):

        candidate = part.get(
            field_name
        )


        if (
            isinstance(
                candidate,
                str,
            )
            and
            candidate.strip()
        ):

            text_value = candidate.strip()
            break


    if text_value is None:
        continue


    cell39_next_target_text_parts.append({
        "part_index":
            index,

        "document_id":
            part.get(
                "document_id"
            ),

        "role":
            part.get(
                "role"
            ),

        "char_start":
            part.get(
                "char_start"
            ),

        "char_end":
            part.get(
                "char_end"
            ),

        "text":
            text_value,

        "text_sha256":
            hashlib.sha256(
                text_value.encode(
                    "utf-8"
                )
            ).hexdigest(),
    })


cell39_next_combined_target_text = (
    "\n\n".join(
        row[
            "text"
        ]

        for row
        in cell39_next_target_text_parts
    )
)


cell39_next_target_text_available = bool(
    cell39_next_combined_target_text.strip()
)


# ============================================================
# 15) Build next Pilot Human Review workspace
# ============================================================

cell39_next_pilot_workspace = {
    "workspace_version":
        "compact_human_review_workspace_v1",

    "status":
        "READY_FOR_QUESTION_DRAFT",

    "pilot_case_id":
        cell39_next_pilot_id,

    "golden_case_id":
        cell39_next_golden_case_id,

    "task_draft_id":
        cell39_next_task_draft_id,

    "target_evidence_id":
        cell39_next_target_evidence_id,

    "source_type":
        cell39_next_source_type,

    "stratum":
        cell39_next_stratum,

    "task_family":
        cell39_next_task_family,

    "question_archetype":
        cell39_next_question_archetype,

    "source_evidence":
        copy.deepcopy(
            cell39_next_target_text_parts
        ),

    "combined_target_text":
        cell39_next_combined_target_text,

    "golden_case":
        copy.deepcopy(
            cell39_next_golden_case
        ),

    "task_plan":
        copy.deepcopy(
            cell39_next_task_plan
        ),

    "target_evidence":
        copy.deepcopy(
            cell39_next_target_evidence
        ),

    "model_drafts":
        [],

    "human_reviews":
        [],

    "expected_answer_review":
        None,

    "citation_review":
        None,

    "author_final_review": {
        "status":
            "NOT_STARTED",

        "decision":
            None,

        "notes":
            None,
    },

    "workflow_state":
        "QUESTION_DRAFT_PENDING",

    "full_572_generation":
        CELL39_FULL_572_BLOCK_STATUS,
}


cell39_next_workspace_ready = all([
    cell39_next_packet_valid,

    cell39_next_identity_valid,

    isinstance(
        cell39_next_golden_case,
        dict,
    ),

    isinstance(
        cell39_next_task_plan,
        dict,
    ),

    isinstance(
        cell39_next_target_evidence,
        dict,
    ),

    cell39_next_target_text_available,
])


# ============================================================
# 16) Update Completed Human Review cases
# ============================================================

cell39_completed_cases = copy.deepcopy(
    cell38_persisted_registry.get(
        "completed_human_review_cases",
        [],
    )
)


cell39_completed_cases.append(
    copy.deepcopy(
        cell39_completed_human_review_case
    )
)


# ============================================================
# 17) Build persistent Registry payload
# ============================================================

cell39_persistent_registry_payload = copy.deepcopy(
    cell38_persisted_registry
)


cell39_persistent_registry_payload[
    "completed_human_review_cases"
] = copy.deepcopy(
    cell39_completed_cases
)


cell39_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell39_next_pilot_workspace
)


cell39_persistent_registry_payload[
    "remaining_pilot_count"
] = 17


cell39_persistent_registry_payload[
    "continuation_state"
] = {
    "next_pilot_id":
        cell39_next_pilot_id,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell39_remaining_pilot_ids
        ),

    "remaining_count":
        17,

    "continuation_state_sha256":
        cell39_continuation_state_sha256,
}


cell39_persistent_registry_payload[
    "retarget_candidate_discovery"
] = {
    "pilot_case_id":
        CELL39_COMPLETED_PILOT_ID,

    "status":
        "RETARGET_WORKFLOW_COMPLETE",

    "old_target_evidence_id":
        CELL37_OLD_TARGET_EVIDENCE_ID,

    "replacement_target_evidence_id":
        CELL39_COMPLETED_TARGET_ID,

    "selected_region_id":
        CELL37_SELECTED_REGION_ID,

    "replacement_selected":
        True,

    "human_review_complete":
        True,

    "author_final_review":
        "NOT_STARTED",
}


# ============================================================
# 18) Author Review state
# ============================================================

cell39_author_review = copy.deepcopy(
    cell39_persistent_registry_payload.get(
        "author_review",
        {}
    )
)


if not isinstance(
    cell39_author_review,
    dict,
):

    cell39_author_review = {}


cell39_author_review[
    "status"
] = "NOT_STARTED"


cell39_author_review[
    "reviewer"
] = "AUTHOR"


cell39_author_review[
    "cases_ready_for_review"
] = len(
    cell39_completed_cases
)


cell39_persistent_registry_payload[
    "author_review"
] = copy.deepcopy(
    cell39_author_review
)


# ============================================================
# 19) Persist
# ============================================================

cell39_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell39_persistent_registry_payload
    )
)


# ============================================================
# 20) Read-back validation
# ============================================================

cell39_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell39_persisted_active_case = (
    cell39_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell39_persistence_valid = all([
    len(
        cell39_persisted_registry.get(
            "completed_human_review_cases",
            [],
        )
    )
    ==
    len(
        cell39_completed_cases
    ),

    cell39_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    17,

    cell39_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    cell39_next_pilot_id,

    cell39_persisted_registry.get(
        "continuation_state",
        {},
    ).get(
        "next_pilot_id"
    )
    ==
    cell39_next_pilot_id,

    cell39_persisted_registry.get(
        "author_review",
        {},
    ).get(
        "cases_ready_for_review"
    )
    ==
    len(
        cell39_completed_cases
    ),

    cell39_persisted_registry.get(
        "retarget_candidate_discovery",
        {},
    ).get(
        "status"
    )
    ==
    "RETARGET_WORKFLOW_COMPLETE",
])


# ============================================================
# 21) Safety
# ============================================================

cell39_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell39_full_572_blocked = (
    cell39_continuation_state[
        "full_572_generation"
    ]
    ==
    CELL39_FULL_572_BLOCK_STATUS
)


# ============================================================
# 22) Final Gate
# ============================================================

cell39_gate = all([
    len(
        cell39_failures
    )
    ==
    0,

    cell39_citation_accept_valid,

    cell39_queue_preconditions_valid,

    cell39_queue_advanced_valid,

    cell39_next_workspace_ready,

    cell39_persistence_valid,

    cell39_base_seed_unchanged,

    cell39_full_572_blocked,
])


# ============================================================
# 23) Diagnostics
# ============================================================

print(
    "=== RETARGETED PILOT COMPLETED "
    "+ QUEUE ADVANCE + NEXT PILOT ==="
)


print(
    "\n=== COMPLETED PILOT ==="
)

print(
    "Pilot:",
    CELL39_COMPLETED_PILOT_ID,
)

print(
    "Old Target:",
    CELL37_OLD_TARGET_EVIDENCE_ID,
)

print(
    "Replacement Target:",
    CELL39_COMPLETED_TARGET_ID,
)

print(
    "Selected Region:",
    CELL37_SELECTED_REGION_ID,
)

print(
    "Question:",
    CELL37_ACCEPTED_QUESTION,
)

print(
    "Answer:",
    CELL37_EXPECTED_ANSWER,
)

print(
    "Citation:",
    "ACCEPT",
)

print(
    "Retarget workflow:",
    "COMPLETE",
)

print(
    "Human Review status:",
    cell39_completed_human_review_case[
        "final_status"
    ],
)

print(
    "Author Final Review:",
    "NOT_STARTED",
)


print(
    "\n=== QUEUE ==="
)

print(
    "Before:",
    len(
        cell39_parent_remaining_ids
    ),
)

print(
    "After:",
    len(
        cell39_remaining_pilot_ids
    ),
)

print(
    "Completed Pilot removed:",
    (
        CELL39_COMPLETED_PILOT_ID
        not in
        cell39_remaining_pilot_ids
    ),
)

print(
    "Next Pilot:",
    cell39_next_pilot_id,
)


print(
    "\n=== PERSISTENT REGISTRY ==="
)

print(
    "Path:",
    cell21_registry_path,
)

print(
    "Completed Human Review cases:",
    len(
        cell39_persisted_registry.get(
            "completed_human_review_cases",
            [],
        )
    ),
)

print(
    "Cases ready for Author Review:",
    cell39_persisted_registry.get(
        "author_review",
        {},
    ).get(
        "cases_ready_for_review"
    ),
)

print(
    "Remaining Pilots:",
    cell39_persisted_registry.get(
        "remaining_pilot_count"
    ),
)

print(
    "Active Pilot:",
    cell39_persisted_active_case.get(
        "pilot_case_id"
    ),
)

print(
    "Persistence valid:",
    cell39_persistence_valid,
)


print(
    "\n=== NEXT PILOT IDENTITY ==="
)

print(
    "Pilot:",
    cell39_next_pilot_id,
)

print(
    "Golden Case:",
    cell39_next_golden_case_id,
)

print(
    "Task Draft:",
    cell39_next_task_draft_id,
)

print(
    "Target Evidence:",
    cell39_next_target_evidence_id,
)

print(
    "Source type:",
    cell39_next_source_type,
)

print(
    "Stratum:",
    cell39_next_stratum,
)

print(
    "Task family:",
    cell39_next_task_family,
)

print(
    "Question archetype:",
    cell39_next_question_archetype,
)


print(
    "\n=== NEXT PILOT SOURCE EVIDENCE ==="
)

print(
    cell39_next_combined_target_text
)


print(
    "\n=== SAFETY ==="
)

print(
    "Base Seed unchanged:",
    cell39_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell39_full_572_blocked,
)

print(
    "Author Notebook rewrite:",
    False,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell39_failures
)


print(
    "\nCODE CELL 39 GATE:",
    (
        "PASS"
        if cell39_gate
        else "FAIL"
    ),
)


assert cell39_gate, (
    "Citation ACCEPT / Retargeted Pilot completion / "
    "Queue advance / Registry sync / next Pilot preparation "
    "failed. Repair this SAME Code Cell 39 before continuing."
)

=== RETARGETED PILOT COMPLETED + QUEUE ADVANCE + NEXT PILOT ===

=== COMPLETED PILOT ===
Pilot: question-pilot:3023c1c546df2fe2211f79702b6e3dbb
Old Target: target-evidence:cccb56b3cc1c8c0f2006c44010235fc4
Replacement Target: target-evidence:843545be6778b09e04b5f9ef650e518f
Selected Region: table-region:68e3e189e456efd98c14b4f380b29703
Question: برای محاسبه نمره اعتباری، چه شاخص‌هایی بررسی می‌شوند و نمره نهایی چگونه به دست می‌آید؟
Answer: چهار شاخص شامل «درآمد خالص ماهانه و توان بازپرداخت اقساط»، «پایداری شغل و کسب‌وکار»، «عملکرد مالی» و «رفتار اعتباری» هستند. هر شاخص از یک تا ده امتیاز می‌گیرد؛ سپس امتیازها با هم جمع می‌شوند و مجموع امتیاز بر چهار تقسیم می‌شود تا نمره اعتباری به دست آید.
Citation: ACCEPT
Retarget workflow: COMPLETE
Human Review status: HUMAN_REVIEW_COMPLETE_AUTHOR_FINAL_REVIEW_PENDING
Author Final Review: NOT_STARTED

=== QUEUE ===
Before: 18
After: 17
Completed Pilot removed: True
Next Pilot: question-pilot:672f5241db66879fb47096c5bb0f5568

=== PERSISTENT REGISTRY ===

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 40 — Question ACCEPT + Expected Answer Draft + Registry Sync

Human Reviewer سؤال این Pilot را پذیرفت:

**آیا آموزش سواد مالی مخصوص سن خاصی است، یا می‌توان آن را متناسب با سنین مختلف آموزش داد؟**

در این مرحله:

- Question با تصمیم Human `ACCEPT` ثبت می‌شود؛
- Expected Answer فقط بر اساس Source Evidence موجود ساخته می‌شود؛
- هیچ سن یا جزئیات خارجی به متن اضافه نمی‌شود؛
- Citation هنوز ساخته نمی‌شود؛
- Registry دائمی Sync می‌شود؛
- Queue همچنان 17 باقی می‌ماند؛
- Full 572 Generation همچنان Blocked است.

</div>

In [40]:
# Code Cell 40

cell40_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell39_gate") is not True:
    cell40_failures.append(
        "CELL39_NOT_PASS"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell40_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Current Pilot identity
# ============================================================

CELL40_PILOT_ID = (
    cell39_next_pilot_id
)

CELL40_GOLDEN_CASE_ID = (
    cell39_next_golden_case_id
)

CELL40_TASK_DRAFT_ID = (
    cell39_next_task_draft_id
)

CELL40_TARGET_EVIDENCE_ID = (
    cell39_next_target_evidence_id
)

CELL40_FULL_572_BLOCK_STATUS = (
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


CELL40_ACCEPTED_QUESTION = (
    "آیا آموزش سواد مالی مخصوص سن خاصی است، "
    "یا می‌توان آن را متناسب با سنین مختلف آموزش داد؟"
)


CELL40_EXPECTED_ANSWER = (
    "آموزش سواد مالی مخصوص یک سن خاص نیست. "
    "بر اساس استاندارد سواد مالی، برای مقاطع سنی مختلف "
    "دانش‌ها و توانایی‌های متناسبی در نظر گرفته می‌شود. "
    "هر سطح نیز بر سطح قبلی توسعه پیدا می‌کند؛ بنابراین "
    "دانش و مهارت‌های مالی می‌توانند هم‌زمان با رشد فرد، "
    "مرحله‌به‌مرحله گسترش پیدا کنند."
)


# ============================================================
# 3) Source Evidence
# ============================================================

cell40_source_text = (
    cell39_next_combined_target_text
)


def cell40_fa_surface(
    text,
):

    value = cell15_normalize_text(
        text
    )


    value = (
        value
        .replace(
            "ي",
            "ی",
        )
        .replace(
            "ك",
            "ک",
        )
        .replace(
            "\u200c",
            " ",
        )
    )


    value = re.sub(
        r"\s+",
        " ",
        value,
    )


    return value.strip()


cell40_source_surface = (
    cell40_fa_surface(
        cell40_source_text
    )
)


# ============================================================
# 4) Evidence grounding
# ============================================================

cell40_evidence_grounding = {
    "age_specific_standard":
        (
            "در چه سنی"
            in
            cell40_source_surface
        ),

    "age_group_practical_examples":
        (
            "مقطع سنی"
            in
            cell40_source_surface
        ),

    "knowledge_and_ability":
        (
            "چه دانش هایی"
            in
            cell40_source_surface
            and
            "چه کارهایی"
            in
            cell40_source_surface
        ),

    "progressive_development":
        (
            "سطح قبلی"
            in
            cell40_source_surface
            and
            "طول رشد فرد"
            in
            cell40_source_surface
        ),
}


cell40_evidence_grounding_pass = all(
    cell40_evidence_grounding.values()
)


# ============================================================
# 5) Start Engine session
# ============================================================

cell40_question_pending_session = (
    cell13_build_session(
        cell39_next_pilot_packet
    )
)


cell40_session_start_valid = (
    cell40_question_pending_session.get(
        "current_state"
    )
    ==
    "QUESTION_DRAFT_PENDING"
)


# ============================================================
# 6) Question Draft ID
# ============================================================

cell40_question_identity_payload = {
    "identity_version":
        "question_draft_identity_v1",

    "pilot_case_id":
        CELL40_PILOT_ID,

    "target_evidence_id":
        CELL40_TARGET_EVIDENCE_ID,

    "question_text":
        CELL40_ACCEPTED_QUESTION,

    "revision_number":
        1,
}


cell40_question_draft_id = (
    "question-draft:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell40_question_identity_payload
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


# ============================================================
# 7) Question Machine Validation
# ============================================================

cell40_question_surface = (
    cell40_fa_surface(
        CELL40_ACCEPTED_QUESTION
    )
)


cell40_question_machine_validation = {
    "financial_literacy_present":
        (
            "سواد مالی"
            in
            cell40_question_surface
        ),

    "age_dimension_present":
        (
            "سن"
            in
            cell40_question_surface
        ),

    "different_ages_present":
        (
            "سنین مختلف"
            in
            cell40_question_surface
        ),

    "source_grounded":
        cell40_evidence_grounding_pass,

    "external_knowledge_used":
        False,
}


cell40_question_machine_pass = all([
    cell40_question_machine_validation[
        "financial_literacy_present"
    ],

    cell40_question_machine_validation[
        "age_dimension_present"
    ],

    cell40_question_machine_validation[
        "different_ages_present"
    ],

    cell40_question_machine_validation[
        "source_grounded"
    ],
])


# ============================================================
# 8) Question artifact
# ============================================================

cell40_question_record = {
    "record_version":
        "question_draft_record_v1",

    "question_draft_id":
        cell40_question_draft_id,

    "pilot_case_id":
        CELL40_PILOT_ID,

    "golden_case_id":
        CELL40_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL40_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL40_TARGET_EVIDENCE_ID,

    "question_text":
        CELL40_ACCEPTED_QUESTION,

    "revision_number":
        1,

    "machine_validation_status":
        (
            "PASS_PENDING_HUMAN_ADJUDICATION"
            if cell40_question_machine_pass
            else
            "FAIL_REQUIRES_REVISION"
        ),

    "human_adjudication_status":
        "PENDING",

    "external_knowledge_used":
        False,
}


# ============================================================
# 9) Register + Human ACCEPT Question
# ============================================================

cell40_question_review_session = (
    cell13_apply_event(
        cell40_question_pending_session,
        "QUESTION_DRAFT_REGISTERED",
        artifact=
            cell40_question_record,
    )
)


CELL40_QUESTION_HUMAN_NOTE = (
    "Human Reviewer accepted the question as a natural "
    "user-facing formulation of the age-progressive financial "
    "literacy standard described in the source evidence."
)


cell40_answer_pending_session = (
    cell13_apply_event(
        cell40_question_review_session,
        "QUESTION_ACCEPTED",
        human_note=
            CELL40_QUESTION_HUMAN_NOTE,
    )
)


cell40_question_accept_valid = all([
    cell40_question_review_session.get(
        "current_state"
    )
    ==
    "QUESTION_HUMAN_REVIEW_PENDING",

    cell40_answer_pending_session.get(
        "current_state"
    )
    ==
    "EXPECTED_ANSWER_DRAFT_PENDING",
])


# ============================================================
# 10) Expected Answer ID
# ============================================================

cell40_answer_identity_payload = {
    "identity_version":
        "expected_answer_identity_v1",

    "pilot_case_id":
        CELL40_PILOT_ID,

    "question_draft_id":
        cell40_question_draft_id,

    "target_evidence_id":
        CELL40_TARGET_EVIDENCE_ID,

    "answer_text":
        CELL40_EXPECTED_ANSWER,

    "revision_number":
        1,
}


cell40_expected_answer_draft_id = (
    "expected-answer-draft:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell40_answer_identity_payload
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


# ============================================================
# 11) Expected Answer Machine Validation
# ============================================================

cell40_answer_surface = (
    cell40_fa_surface(
        CELL40_EXPECTED_ANSWER
    )
)


cell40_answer_grounding = {
    "not_single_age_only":
        (
            "مخصوص یک سن خاص نیست"
            in
            cell40_answer_surface
        ),

    "different_age_groups":
        (
            "مقاطع سنی مختلف"
            in
            cell40_answer_surface
        ),

    "knowledge_and_abilities":
        (
            "دانش ها و توانایی"
            in
            cell40_answer_surface
        ),

    "builds_on_previous_level":
        (
            "سطح قبلی"
            in
            cell40_answer_surface
        ),

    "development_with_growth":
        (
            "رشد فرد"
            in
            cell40_answer_surface
        ),
}


cell40_answer_machine_pass = all([
    cell40_evidence_grounding_pass,

    all(
        cell40_answer_grounding.values()
    ),
])


# ============================================================
# 12) Expected Answer artifact
# ============================================================

cell40_expected_answer_record = {
    "record_version":
        "expected_answer_draft_record_v1",

    "expected_answer_draft_id":
        cell40_expected_answer_draft_id,

    "pilot_case_id":
        CELL40_PILOT_ID,

    "golden_case_id":
        CELL40_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL40_TASK_DRAFT_ID,

    "question_draft_id":
        cell40_question_draft_id,

    "target_evidence_id":
        CELL40_TARGET_EVIDENCE_ID,

    "expected_answer_text":
        CELL40_EXPECTED_ANSWER,

    "revision_number":
        1,

    "machine_validation_status":
        (
            "PASS_PENDING_HUMAN_ADJUDICATION"
            if cell40_answer_machine_pass
            else
            "FAIL_REQUIRES_REVISION"
        ),

    "human_adjudication_status":
        "PENDING",

    "external_knowledge_used":
        False,
}


# ============================================================
# 13) Register Expected Answer
# ============================================================

cell40_current_session = (
    cell13_apply_event(
        cell40_answer_pending_session,
        "EXPECTED_ANSWER_REGISTERED",
        artifact=
            cell40_expected_answer_record,
    )
)


cell40_answer_registered_valid = (
    cell40_current_session.get(
        "current_state"
    )
    ==
    "EXPECTED_ANSWER_HUMAN_REVIEW_PENDING"
)


# ============================================================
# 14) Human Review workspace
# ============================================================

cell40_active_human_review_case = copy.deepcopy(
    cell39_next_pilot_workspace
)


cell40_active_human_review_case[
    "status"
] = "ACTIVE_HUMAN_REVIEW"


cell40_active_human_review_case[
    "workflow_state"
] = (
    "EXPECTED_ANSWER_HUMAN_REVIEW_PENDING"
)


cell40_active_human_review_case[
    "question_review"
] = {
    "question_draft_id":
        cell40_question_draft_id,

    "question_text":
        CELL40_ACCEPTED_QUESTION,

    "machine_validation_status":
        cell40_question_record[
            "machine_validation_status"
        ],

    "human_decision":
        "ACCEPT",

    "human_note":
        CELL40_QUESTION_HUMAN_NOTE,
}


cell40_active_human_review_case[
    "expected_answer_review"
] = {
    "expected_answer_draft_id":
        cell40_expected_answer_draft_id,

    "expected_answer_text":
        CELL40_EXPECTED_ANSWER,

    "machine_validation_status":
        cell40_expected_answer_record[
            "machine_validation_status"
        ],

    "human_decision":
        "PENDING",
}


cell40_active_human_review_case[
    "citation_review"
] = None


cell40_active_human_review_case[
    "author_final_review"
] = {
    "status":
        "NOT_STARTED",

    "decision":
        None,

    "notes":
        None,
}


# ============================================================
# 15) Persist Registry
# ============================================================

cell40_persistent_registry_payload = copy.deepcopy(
    cell39_persisted_registry
)


cell40_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell40_active_human_review_case
)


cell40_persistent_registry_payload[
    "remaining_pilot_count"
] = 17


cell40_persistent_registry_payload[
    "continuation_state"
] = {
    "next_pilot_id":
        CELL40_PILOT_ID,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell39_remaining_pilot_ids
        ),

    "remaining_count":
        17,

    "continuation_state_sha256":
        cell39_continuation_state_sha256,

    "active_pilot_status":
        "EXPECTED_ANSWER_HUMAN_REVIEW_PENDING",

    "active_target_evidence_id":
        CELL40_TARGET_EVIDENCE_ID,
}


cell40_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell40_persistent_registry_payload
    )
)


# ============================================================
# 16) Read-back validation
# ============================================================

cell40_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell40_persisted_active_case = (
    cell40_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell40_persistence_valid = all([
    cell40_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL40_PILOT_ID,

    cell40_persisted_active_case.get(
        "question_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell40_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "PENDING",

    cell40_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    17,
])


# ============================================================
# 17) Safety
# ============================================================

cell40_queue_still_17 = (
    len(
        cell39_remaining_pilot_ids
    )
    ==
    17
)


cell40_current_pilot_still_first = (
    cell39_remaining_pilot_ids[
        0
    ]
    ==
    CELL40_PILOT_ID
)


cell40_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell40_full_572_blocked = (
    CELL40_FULL_572_BLOCK_STATUS
    ==
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


# ============================================================
# 18) Final Gate
# ============================================================

cell40_gate = all([
    len(
        cell40_failures
    )
    ==
    0,

    cell40_evidence_grounding_pass,

    cell40_session_start_valid,

    cell40_question_machine_pass,

    cell40_question_accept_valid,

    cell40_answer_machine_pass,

    cell40_answer_registered_valid,

    cell40_persistence_valid,

    cell40_queue_still_17,

    cell40_current_pilot_still_first,

    cell40_base_seed_unchanged,

    cell40_full_572_blocked,
])


# ============================================================
# 19) Diagnostics
# ============================================================

print(
    "=== QUESTION ACCEPTED + EXPECTED ANSWER DRAFT "
    "+ REGISTRY SYNC ==="
)


print(
    "\n=== SOURCE EVIDENCE ==="
)

print(
    cell40_source_text
)


print(
    "\n=== ACCEPTED QUESTION ==="
)

print(
    CELL40_ACCEPTED_QUESTION
)

print(
    "Question Draft ID:",
    cell40_question_draft_id,
)

print(
    "Question Machine PASS:",
    cell40_question_machine_pass,
)


print(
    "\n=== EXPECTED ANSWER DRAFT ==="
)

print(
    CELL40_EXPECTED_ANSWER
)

print(
    "Expected Answer Draft ID:",
    cell40_expected_answer_draft_id,
)

print(
    "Machine PASS:",
    cell40_answer_machine_pass,
)


print(
    "\n=== ANSWER GROUNDING ==="
)

for name, status in (
    cell40_answer_grounding.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== WORKFLOW ==="
)

print(
    "Current State:",
    cell40_current_session.get(
        "current_state"
    ),
)

print(
    "Question Decision:",
    "ACCEPT",
)

print(
    "Expected Answer Decision:",
    "PENDING",
)

print(
    "Citation:",
    None,
)


print(
    "\n=== PERSISTENCE ==="
)

print(
    "Registry:",
    cell21_registry_path,
)

print(
    "Persisted Active Pilot:",
    cell40_persisted_active_case.get(
        "pilot_case_id"
    ),
)

print(
    "Persisted Question Decision:",
    cell40_persisted_active_case.get(
        "question_review",
        {},
    ).get(
        "human_decision"
    ),
)

print(
    "Persisted Answer Decision:",
    cell40_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    ),
)

print(
    "Persistence valid:",
    cell40_persistence_valid,
)


print(
    "\n=== QUEUE / SAFETY ==="
)

print(
    "Queue still 17:",
    cell40_queue_still_17,
)

print(
    "Base Seed unchanged:",
    cell40_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell40_full_572_blocked,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell40_failures
)


print(
    "\nCODE CELL 40 GATE:",
    (
        "PASS"
        if cell40_gate
        else "FAIL"
    ),
)


assert cell40_gate, (
    "Question ACCEPT / Expected Answer Draft / Registry Sync "
    "failed. Repair this SAME Code Cell 40 before "
    "Expected Answer Human Review."
)

=== QUESTION ACCEPTED + EXPECTED ANSWER DRAFT + REGISTRY SYNC ===

=== SOURCE EVIDENCE ===
استانداردهای سواد مالی پاسخی است به سؤال‌هایی درباره چگونگی آموزش سواد مالی؛ اینکه چه محتوایی از مفاهیم و نظریه‌های اقتصادی و مدیریت امور مالی باید برای عموم مردم تبیین شود و مهارت‌های آن را یادبگیرند؛ هر مفهوم یا مهارت در چه سطحی از مخاطب و چه میزان از آن و با چه ویژگی‌هایی ارائه شود.

استاندارد سواد مالی تعیین می‌کند یک فرد در چه سنی، چه دانش‌هایی را باید داشته باشد و بتواند چه کارهایی را انجام دهد. هر حوزه از استاندارد شامل گزارهای علمی  و نمونه عملی  است؛ نمونه عملی، توانایی‌های متناظر با هر گزاره علمی در یک مقطع سنی را فهرست می‌کند؛ هر گزاره علمی و نمونه عملی، توسعه یافته سطح قبلی خود است؛ به بیان دیگر، یک گزاره دانشی و چارچوب راهنمای آن در طول رشد فرد توسعه می‌یابد.

=== ACCEPTED QUESTION ===
آیا آموزش سواد مالی مخصوص سن خاصی است، یا می‌توان آن را متناسب با سنین مختلف آموزش داد؟
Question Draft ID: question-draft:0b46a390bf09333fbbefdea7a168a38b
Question Machine PASS: True

=== EXPECTED ANSW

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 41 — Expected Answer ACCEPT + Citation Draft + Registry Sync

Expected Answer این Pilot توسط Human Reviewer پذیرفته شد.

در این مرحله:

- Expected Answer با تصمیم `ACCEPT` ثبت می‌شود؛
- Citation فقط از Target Evidence اعتبارسنجی‌شده همین Pilot ساخته می‌شود؛
- Document ID، Role، Locator و محدوده متنی مستقیماً از Evidence موجود کپی می‌شوند؛
- هیچ Locator یا منبع خارجی ساخته نمی‌شود؛
- Citation به مرحله Human Review می‌رود؛
- Registry دائمی Sync می‌شود؛
- Queue همچنان 17 باقی می‌ماند؛
- Author Final Review همچنان `NOT_STARTED` است.

</div>

In [41]:
# Code Cell 41

cell41_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell40_gate") is not True:
    cell41_failures.append(
        "CELL40_NOT_PASS"
    )


if (
    globals().get(
        "cell40_current_session",
        {},
    ).get(
        "current_state"
    )
    !=
    "EXPECTED_ANSWER_HUMAN_REVIEW_PENDING"
):
    cell41_failures.append(
        "EXPECTED_ANSWER_NOT_AT_HUMAN_REVIEW_GATE"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell41_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Current identities
# ============================================================

CELL41_PILOT_ID = (
    CELL40_PILOT_ID
)

CELL41_GOLDEN_CASE_ID = (
    CELL40_GOLDEN_CASE_ID
)

CELL41_TASK_DRAFT_ID = (
    CELL40_TASK_DRAFT_ID
)

CELL41_TARGET_EVIDENCE_ID = (
    CELL40_TARGET_EVIDENCE_ID
)

CELL41_QUESTION_DRAFT_ID = (
    cell40_question_draft_id
)

CELL41_EXPECTED_ANSWER_DRAFT_ID = (
    cell40_expected_answer_draft_id
)

CELL41_FULL_572_BLOCK_STATUS = (
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


# ============================================================
# 3) Human ACCEPT for Expected Answer
# ============================================================

CELL41_HUMAN_ANSWER_NOTE = (
    "Human Reviewer accepted the Expected Answer after "
    "reviewing it against the validated source evidence."
)


cell41_answer_human_review = {
    "review_record_version":
        "human_expected_answer_review_record_v1",

    "pilot_case_id":
        CELL41_PILOT_ID,

    "question_draft_id":
        CELL41_QUESTION_DRAFT_ID,

    "expected_answer_draft_id":
        CELL41_EXPECTED_ANSWER_DRAFT_ID,

    "target_evidence_id":
        CELL41_TARGET_EVIDENCE_ID,

    "human_decision":
        "ACCEPT",

    "human_note":
        CELL41_HUMAN_ANSWER_NOTE,

    "human_context_used_as_evidence":
        False,

    "external_knowledge_used":
        False,
}


# ============================================================
# 4) Apply Expected Answer ACCEPT
# ============================================================

cell41_citation_pending_session = (
    cell13_apply_event(
        cell40_current_session,
        "EXPECTED_ANSWER_ACCEPTED",
        human_note=
            CELL41_HUMAN_ANSWER_NOTE,
    )
)


cell41_answer_accept_valid = all([
    cell41_citation_pending_session.get(
        "current_state"
    )
    ==
    "CITATION_MAPPING_PENDING",

    cell41_citation_pending_session.get(
        "queue_advance_performed"
    )
    is False,

    cell41_citation_pending_session.get(
        "human_decisions",
        [],
    )[
        -1
    ].get(
        "event"
    )
    ==
    "EXPECTED_ANSWER_ACCEPTED",
])


# ============================================================
# 5) Read validated Target Evidence parts
# ============================================================

cell41_target_parts = copy.deepcopy(
    cell39_next_target_evidence.get(
        "target_parts",
        [],
    )
)


cell41_target_parts_valid = all([
    isinstance(
        cell41_target_parts,
        list,
    ),

    len(
        cell41_target_parts
    )
    >
    0,
])


# ============================================================
# 6) Build Citation entries
# ============================================================

cell41_citation_entries = []


for index, part in enumerate(
    cell41_target_parts
):

    if not isinstance(
        part,
        dict,
    ):
        continue


    cell41_citation_entries.append({
        "citation_index":
            index,

        "target_evidence_id":
            CELL41_TARGET_EVIDENCE_ID,

        "document_id":
            part.get(
                "document_id"
            ),

        "source_id":
            part.get(
                "source_id"
            ),

        "role":
            part.get(
                "role"
            ),

        "char_start":
            part.get(
                "char_start"
            ),

        "char_end":
            part.get(
                "char_end"
            ),

        "char_count":
            part.get(
                "char_count"
            ),

        "token_count":
            part.get(
                "token_count"
            ),

        "locator":
            copy.deepcopy(
                part.get(
                    "locator",
                    {},
                )
            ),

        "attribution":
            copy.deepcopy(
                part.get(
                    "attribution"
                )
            ),

        "locator_origin":
            "COPIED_FROM_VALIDATED_TARGET_EVIDENCE",

        "external_locator_lookup_used":
            False,
    })


# ============================================================
# 7) Exact projection validation
# ============================================================

cell41_source_projection = [
    {
        "document_id":
            part.get(
                "document_id"
            ),

        "source_id":
            part.get(
                "source_id"
            ),

        "role":
            part.get(
                "role"
            ),

        "char_start":
            part.get(
                "char_start"
            ),

        "char_end":
            part.get(
                "char_end"
            ),

        "locator":
            copy.deepcopy(
                part.get(
                    "locator",
                    {},
                )
            ),
    }

    for part
    in cell41_target_parts

    if isinstance(
        part,
        dict,
    )
]


cell41_citation_projection = [
    {
        "document_id":
            entry.get(
                "document_id"
            ),

        "source_id":
            entry.get(
                "source_id"
            ),

        "role":
            entry.get(
                "role"
            ),

        "char_start":
            entry.get(
                "char_start"
            ),

        "char_end":
            entry.get(
                "char_end"
            ),

        "locator":
            copy.deepcopy(
                entry.get(
                    "locator",
                    {},
                )
            ),
    }

    for entry
    in cell41_citation_entries
]


cell41_no_locator_fabrication = (
    cell41_source_projection
    ==
    cell41_citation_projection
)


# ============================================================
# 8) Validate against displayed Source Evidence
# ============================================================

cell41_displayed_parts = copy.deepcopy(
    cell39_next_target_text_parts
)


cell41_displayed_by_identity = {
    (
        row.get(
            "document_id"
        ),
        row.get(
            "role"
        ),
        row.get(
            "char_start"
        ),
        row.get(
            "char_end"
        ),
    ):
        row

    for row
    in cell41_displayed_parts

    if isinstance(
        row,
        dict,
    )
}


cell41_validation_rows = []


for entry in cell41_citation_entries:

    identity = (
        entry.get(
            "document_id"
        ),
        entry.get(
            "role"
        ),
        entry.get(
            "char_start"
        ),
        entry.get(
            "char_end"
        ),
    )


    displayed = (
        cell41_displayed_by_identity.get(
            identity
        )
    )


    locator = (
        entry.get(
            "locator"
        )
        or
        {}
    )


    canonical_url = locator.get(
        "canonical_url"
    )


    url = locator.get(
        "url"
    )


    row_valid = all([
        isinstance(
            entry.get(
                "document_id"
            ),
            str,
        ),

        isinstance(
            displayed,
            dict,
        ),

        (
            entry.get(
                "char_start"
            )
            is None
            or
            isinstance(
                entry.get(
                    "char_start"
                ),
                int,
            )
        ),

        (
            entry.get(
                "char_end"
            )
            is None
            or
            isinstance(
                entry.get(
                    "char_end"
                ),
                int,
            )
        ),

        (
            canonical_url is None
            or
            (
                isinstance(
                    canonical_url,
                    str,
                )
                and
                canonical_url.startswith(
                    (
                        "http://",
                        "https://",
                    )
                )
            )
        ),

        (
            url is None
            or
            (
                isinstance(
                    url,
                    str,
                )
                and
                url.startswith(
                    (
                        "http://",
                        "https://",
                    )
                )
            )
        ),

        entry.get(
            "locator_origin"
        )
        ==
        "COPIED_FROM_VALIDATED_TARGET_EVIDENCE",

        entry.get(
            "external_locator_lookup_used"
        )
        is False,
    ])


    cell41_validation_rows.append({
        "citation_index":
            entry.get(
                "citation_index"
            ),

        "document_id":
            entry.get(
                "document_id"
            ),

        "role":
            entry.get(
                "role"
            ),

        "char_start":
            entry.get(
                "char_start"
            ),

        "char_end":
            entry.get(
                "char_end"
            ),

        "displayed_match":
            isinstance(
                displayed,
                dict,
            ),

        "valid":
            row_valid,
    })


cell41_all_entries_valid = all([
    len(
        cell41_citation_entries
    )
    >
    0,

    len(
        cell41_citation_entries
    )
    ==
    len(
        cell41_target_parts
    ),

    len(
        cell41_citation_entries
    )
    ==
    len(
        cell41_displayed_parts
    ),

    all(
        row.get(
            "valid"
        )
        is True

        for row
        in cell41_validation_rows
    ),
])


# ============================================================
# 9) Deterministic Citation Mapping ID
# ============================================================

cell41_citation_identity_payload = {
    "identity_version":
        "citation_mapping_identity_v1",

    "pilot_case_id":
        CELL41_PILOT_ID,

    "question_draft_id":
        CELL41_QUESTION_DRAFT_ID,

    "expected_answer_draft_id":
        CELL41_EXPECTED_ANSWER_DRAFT_ID,

    "target_evidence_id":
        CELL41_TARGET_EVIDENCE_ID,

    "citation_projection":
        copy.deepcopy(
            cell41_citation_projection
        ),
}


cell41_citation_mapping_id = (
    "citation-mapping:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell41_citation_identity_payload
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


# ============================================================
# 10) Citation Machine Validation
# ============================================================

cell41_citation_machine_validation = {
    "target_parts_present":
        cell41_target_parts_valid,

    "citation_entries_present":
        (
            len(
                cell41_citation_entries
            )
            >
            0
        ),

    "all_target_parts_covered":
        (
            len(
                cell41_citation_entries
            )
            ==
            len(
                cell41_target_parts
            )
        ),

    "displayed_evidence_covered":
        (
            len(
                cell41_citation_entries
            )
            ==
            len(
                cell41_displayed_parts
            )
        ),

    "all_entries_valid":
        cell41_all_entries_valid,

    "locator_fabrication_absent":
        cell41_no_locator_fabrication,

    "target_evidence_identity_match":
        all(
            entry.get(
                "target_evidence_id"
            )
            ==
            CELL41_TARGET_EVIDENCE_ID

            for entry
            in cell41_citation_entries
        ),

    "external_locator_lookup_used":
        False,
}


cell41_citation_machine_pass = all([
    cell41_citation_machine_validation[
        "target_parts_present"
    ],

    cell41_citation_machine_validation[
        "citation_entries_present"
    ],

    cell41_citation_machine_validation[
        "all_target_parts_covered"
    ],

    cell41_citation_machine_validation[
        "displayed_evidence_covered"
    ],

    cell41_citation_machine_validation[
        "all_entries_valid"
    ],

    cell41_citation_machine_validation[
        "locator_fabrication_absent"
    ],

    cell41_citation_machine_validation[
        "target_evidence_identity_match"
    ],
])


# ============================================================
# 11) Citation Mapping Record
# ============================================================

cell41_citation_mapping_record = {
    "record_version":
        "citation_mapping_record_v1",

    "citation_mapping_id":
        cell41_citation_mapping_id,

    "pilot_case_id":
        CELL41_PILOT_ID,

    "golden_case_id":
        CELL41_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL41_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL41_TARGET_EVIDENCE_ID,

    "question_draft_id":
        CELL41_QUESTION_DRAFT_ID,

    "expected_answer_draft_id":
        CELL41_EXPECTED_ANSWER_DRAFT_ID,

    "citation_entries":
        copy.deepcopy(
            cell41_citation_entries
        ),

    "citation_scope":
        "VALIDATED_TARGET_EVIDENCE_ONLY",

    "locator_source":
        "EXISTING_TARGET_EVIDENCE_LOCATORS",

    "external_locator_lookup_used":
        False,

    "machine_validation_status":
        (
            "PASS_PENDING_HUMAN_ADJUDICATION"
            if cell41_citation_machine_pass
            else
            "FAIL_REQUIRES_REVISION"
        ),

    "human_adjudication_status":
        "PENDING",

    "full_572_generation":
        CELL41_FULL_572_BLOCK_STATUS,
}


# ============================================================
# 12) Register Citation
# ============================================================

cell41_current_session = (
    cell13_apply_event(
        cell41_citation_pending_session,
        "CITATION_MAPPING_REGISTERED",
        artifact=
            cell41_citation_mapping_record,
    )
)


cell41_citation_registered_valid = all([
    cell41_current_session.get(
        "current_state"
    )
    ==
    "CITATION_HUMAN_REVIEW_PENDING",

    isinstance(
        cell41_current_session.get(
            "artifacts",
            {}
        ).get(
            "citation_mapping"
        ),
        dict,
    ),

    cell41_current_session[
        "artifacts"
    ][
        "citation_mapping"
    ].get(
        "citation_mapping_id"
    )
    ==
    cell41_citation_mapping_id,

    cell41_current_session.get(
        "queue_advance_performed"
    )
    is False,
])


# ============================================================
# 13) Update Human Review workspace
# ============================================================

cell41_active_human_review_case = copy.deepcopy(
    cell40_active_human_review_case
)


cell41_active_human_review_case[
    "workflow_state"
] = "CITATION_HUMAN_REVIEW_PENDING"


cell41_active_human_review_case[
    "expected_answer_review"
][
    "human_decision"
] = "ACCEPT"


cell41_active_human_review_case[
    "expected_answer_review"
][
    "human_review"
] = copy.deepcopy(
    cell41_answer_human_review
)


cell41_active_human_review_case[
    "citation_review"
] = {
    "citation_mapping_id":
        cell41_citation_mapping_id,

    "citation_entries":
        copy.deepcopy(
            cell41_citation_entries
        ),

    "machine_validation_status":
        cell41_citation_mapping_record[
            "machine_validation_status"
        ],

    "human_decision":
        "PENDING",
}


cell41_active_human_review_case[
    "final_status"
] = (
    "QUESTION_AND_ANSWER_ACCEPTED_"
    "CITATION_PENDING_REVIEW"
)


# ============================================================
# 14) Persist Registry
# ============================================================

cell41_persistent_registry_payload = copy.deepcopy(
    cell40_persisted_registry
)


cell41_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell41_active_human_review_case
)


cell41_persistent_registry_payload[
    "remaining_pilot_count"
] = 17


cell41_persistent_registry_payload[
    "continuation_state"
] = {
    "next_pilot_id":
        CELL41_PILOT_ID,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell39_remaining_pilot_ids
        ),

    "remaining_count":
        17,

    "continuation_state_sha256":
        cell39_continuation_state_sha256,

    "active_pilot_status":
        "CITATION_HUMAN_REVIEW_PENDING",

    "active_target_evidence_id":
        CELL41_TARGET_EVIDENCE_ID,
}


cell41_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell41_persistent_registry_payload
    )
)


# ============================================================
# 15) Read-back validation
# ============================================================

cell41_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell41_persisted_active_case = (
    cell41_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell41_persistence_valid = all([
    cell41_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL41_PILOT_ID,

    cell41_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell41_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "citation_mapping_id"
    )
    ==
    cell41_citation_mapping_id,

    cell41_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "PENDING",

    cell41_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    17,
])


# ============================================================
# 16) Safety
# ============================================================

cell41_queue_still_17 = (
    len(
        cell39_remaining_pilot_ids
    )
    ==
    17
)


cell41_current_pilot_still_first = (
    cell39_remaining_pilot_ids[
        0
    ]
    ==
    CELL41_PILOT_ID
)


cell41_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell41_full_572_blocked = (
    cell41_citation_mapping_record[
        "full_572_generation"
    ]
    ==
    CELL41_FULL_572_BLOCK_STATUS
)


# ============================================================
# 17) Final Gate
# ============================================================

cell41_gate = all([
    len(
        cell41_failures
    )
    ==
    0,

    cell41_answer_accept_valid,

    cell41_target_parts_valid,

    cell41_no_locator_fabrication,

    cell41_all_entries_valid,

    cell41_citation_machine_pass,

    cell41_citation_registered_valid,

    cell41_persistence_valid,

    cell41_queue_still_17,

    cell41_current_pilot_still_first,

    cell41_base_seed_unchanged,

    cell41_full_572_blocked,
])


# ============================================================
# 18) Diagnostics
# ============================================================

print(
    "=== ANSWER ACCEPTED + CITATION DRAFT "
    "+ REGISTRY SYNC ==="
)


print(
    "\n=== ACCEPTED QUESTION ==="
)

print(
    CELL40_ACCEPTED_QUESTION
)


print(
    "\n=== ACCEPTED ANSWER ==="
)

print(
    CELL40_EXPECTED_ANSWER
)

print(
    "Human Answer Decision:",
    "ACCEPT",
)


print(
    "\n=== CITATION DRAFT ==="
)

print(
    "Citation Mapping ID:",
    cell41_citation_mapping_id,
)

print(
    "Target Evidence:",
    CELL41_TARGET_EVIDENCE_ID,
)

print(
    "Citation entry count:",
    len(
        cell41_citation_entries
    ),
)


for entry in cell41_citation_entries:

    print(
        "\n--- CITATION ENTRY",
        entry.get(
            "citation_index"
        ),
        "---"
    )

    print(
        "Document:",
        entry.get(
            "document_id"
        ),
    )

    print(
        "Role:",
        entry.get(
            "role"
        ),
    )

    print(
        "Chars:",
        (
            entry.get(
                "char_start"
            ),
            entry.get(
                "char_end"
            ),
        ),
    )

    print(
        "Paragraph ID:",
        entry.get(
            "locator",
            {},
        ).get(
            "paragraph_id"
        ),
    )

    print(
        "Paragraph Number:",
        entry.get(
            "locator",
            {},
        ).get(
            "paragraph_number"
        ),
    )

    print(
        "Canonical URL:",
        entry.get(
            "locator",
            {},
        ).get(
            "canonical_url"
        ),
    )


print(
    "\n=== CITATION MACHINE VALIDATION ==="
)

for name, status in (
    cell41_citation_machine_validation.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "Machine PASS:",
    cell41_citation_machine_pass,
)

print(
    "No locator fabrication:",
    cell41_no_locator_fabrication,
)


print(
    "\n=== WORKFLOW ==="
)

print(
    "Current State:",
    cell41_current_session.get(
        "current_state"
    ),
)

print(
    "Question Decision:",
    "ACCEPT",
)

print(
    "Answer Decision:",
    "ACCEPT",
)

print(
    "Citation Decision:",
    "PENDING",
)


print(
    "\n=== PERSISTENCE ==="
)

print(
    "Registry:",
    cell21_registry_path,
)

print(
    "Persisted Answer Decision:",
    cell41_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    ),
)

print(
    "Persisted Citation Decision:",
    cell41_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "human_decision"
    ),
)

print(
    "Persistence valid:",
    cell41_persistence_valid,
)


print(
    "\n=== QUEUE / SAFETY ==="
)

print(
    "Queue still 17:",
    cell41_queue_still_17,
)

print(
    "Base Seed unchanged:",
    cell41_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell41_full_572_blocked,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell41_failures
)


print(
    "\nCODE CELL 41 GATE:",
    (
        "PASS"
        if cell41_gate
        else "FAIL"
    ),
)


assert cell41_gate, (
    "Expected Answer ACCEPT / Citation Draft / Registry Sync "
    "failed. Repair this SAME Code Cell 41 before "
    "Citation Human Review."
)

=== ANSWER ACCEPTED + CITATION DRAFT + REGISTRY SYNC ===

=== ACCEPTED QUESTION ===
آیا آموزش سواد مالی مخصوص سن خاصی است، یا می‌توان آن را متناسب با سنین مختلف آموزش داد؟

=== ACCEPTED ANSWER ===
آموزش سواد مالی مخصوص یک سن خاص نیست. بر اساس استاندارد سواد مالی، برای مقاطع سنی مختلف دانش‌ها و توانایی‌های متناسبی در نظر گرفته می‌شود. هر سطح نیز بر سطح قبلی توسعه پیدا می‌کند؛ بنابراین دانش و مهارت‌های مالی می‌توانند هم‌زمان با رشد فرد، مرحله‌به‌مرحله گسترش پیدا کنند.
Human Answer Decision: ACCEPT

=== CITATION DRAFT ===
Citation Mapping ID: citation-mapping:7943d214923e42914a7f1b3d4c4a5b93
Target Evidence: target-evidence:f757720919bbcc420dfbd88c5cbac373
Citation entry count: 2

--- CITATION ENTRY 0 ---
Document: book:adult-financial-literacy-course:fa-source:p001307
Role: MULTI_RECORD_WINDOW_LEFT
Chars: (0, 286)
Paragraph ID: book:adult-financial-literacy-course:fa-source:p001307
Paragraph Number: 1307
Canonical URL: None

--- CITATION ENTRY 1 ---
Document: book:adult-financial-literac

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 42 — Citation ACCEPT + Pilot Completion + Queue Advance

Citation این Pilot توسط Human Reviewer پذیرفته شد.

در این مرحله:

- Citation با تصمیم `ACCEPT` ثبت می‌شود؛
- Pilot کامل می‌شود؛
- پرونده Human Review به Completed Cases منتقل می‌شود؛
- Author Final Review همچنان `NOT_STARTED` باقی می‌ماند؛
- Queue از 17 به 16 کاهش پیدا می‌کند؛
- Pilot بعدی و Source Evidence آن آماده می‌شود؛
- Registry دائمی Sync می‌شود؛
- Full 572 Generation همچنان Blocked باقی می‌ماند.

</div>

In [42]:
# Code Cell 42

cell42_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell41_gate") is not True:
    cell42_failures.append(
        "CELL41_NOT_PASS"
    )


if (
    globals().get(
        "cell41_current_session",
        {},
    ).get(
        "current_state"
    )
    !=
    "CITATION_HUMAN_REVIEW_PENDING"
):
    cell42_failures.append(
        "CITATION_NOT_AT_HUMAN_REVIEW_GATE"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell42_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Current completed Pilot identity
# ============================================================

CELL42_COMPLETED_PILOT_ID = (
    CELL41_PILOT_ID
)

CELL42_COMPLETED_TARGET_ID = (
    CELL41_TARGET_EVIDENCE_ID
)

CELL42_FULL_572_BLOCK_STATUS = (
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


# ============================================================
# 3) Human Citation ACCEPT
# ============================================================

CELL42_HUMAN_CITATION_NOTE = (
    "Human Reviewer accepted the Citation Mapping after "
    "reviewing the validated source-evidence locators."
)


cell42_citation_human_review = {
    "review_record_version":
        "human_citation_review_record_v1",

    "pilot_case_id":
        CELL42_COMPLETED_PILOT_ID,

    "citation_mapping_id":
        cell41_citation_mapping_id,

    "target_evidence_id":
        CELL42_COMPLETED_TARGET_ID,

    "human_decision":
        "ACCEPT",

    "human_note":
        CELL42_HUMAN_CITATION_NOTE,

    "external_locator_lookup_used":
        False,
}


# ============================================================
# 4) Apply Citation ACCEPT
# ============================================================

cell42_pre_advance_session = (
    cell13_apply_event(
        cell41_current_session,
        "CITATION_ACCEPTED",
        human_note=
            CELL42_HUMAN_CITATION_NOTE,
    )
)


cell42_citation_accept_valid = all([
    cell42_pre_advance_session.get(
        "current_state"
    )
    ==
    "PILOT_COMPLETE_PENDING_QUEUE_ADVANCE",

    cell42_pre_advance_session.get(
        "queue_advance_permitted"
    )
    is True,

    cell42_pre_advance_session.get(
        "queue_advance_performed"
    )
    is False,

    cell42_pre_advance_session.get(
        "human_decisions",
        [],
    )[
        -1
    ].get(
        "event"
    )
    ==
    "CITATION_ACCEPTED",
])


# ============================================================
# 5) Finalize Human Review Case
# ============================================================

cell42_completed_human_review_case = copy.deepcopy(
    cell41_active_human_review_case
)


cell42_completed_human_review_case[
    "workflow_state"
] = "PILOT_COMPLETE"


cell42_completed_human_review_case[
    "citation_review"
][
    "human_decision"
] = "ACCEPT"


cell42_completed_human_review_case[
    "citation_review"
][
    "human_review"
] = copy.deepcopy(
    cell42_citation_human_review
)


cell42_completed_human_review_case[
    "author_final_review"
] = {
    "status":
        "NOT_STARTED",

    "decision":
        None,

    "notes":
        None,
}


cell42_completed_human_review_case[
    "final_status"
] = (
    "HUMAN_REVIEW_COMPLETE_"
    "AUTHOR_FINAL_REVIEW_PENDING"
)


cell42_completed_human_review_case_sha256 = (
    cell10_stable_sha(
        cell42_completed_human_review_case
    )
)


# ============================================================
# 6) Queue preconditions
# ============================================================

cell42_parent_remaining_ids = copy.deepcopy(
    cell39_continuation_state[
        "remaining_pilot_ids"
    ]
)


cell42_parent_remaining_queue = copy.deepcopy(
    cell39_continuation_state[
        "remaining_pilot_queue"
    ]
)


cell42_queue_preconditions_valid = all([
    len(
        cell42_parent_remaining_ids
    )
    ==
    17,

    len(
        cell42_parent_remaining_queue
    )
    ==
    17,

    cell42_parent_remaining_ids[
        0
    ]
    ==
    CELL42_COMPLETED_PILOT_ID,

    cell42_citation_accept_valid,
])


# ============================================================
# 7) Advance Queue
# ============================================================

if cell42_queue_preconditions_valid:

    cell42_remaining_pilot_ids = copy.deepcopy(
        cell42_parent_remaining_ids[
            1:
        ]
    )

    cell42_remaining_pilot_queue = copy.deepcopy(
        cell42_parent_remaining_queue[
            1:
        ]
    )

else:

    cell42_remaining_pilot_ids = []
    cell42_remaining_pilot_queue = []


cell42_next_pilot_id = (
    cell42_remaining_pilot_ids[
        0
    ]

    if cell42_remaining_pilot_ids

    else None
)


cell42_next_pilot_queue_entry = (
    copy.deepcopy(
        cell42_remaining_pilot_queue[
            0
        ]
    )

    if cell42_remaining_pilot_queue

    else None
)


cell42_queue_advanced_valid = all([
    len(
        cell42_remaining_pilot_ids
    )
    ==
    16,

    len(
        cell42_remaining_pilot_queue
    )
    ==
    16,

    CELL42_COMPLETED_PILOT_ID
    not in
    cell42_remaining_pilot_ids,

    isinstance(
        cell42_next_pilot_id,
        str,
    ),

    cell42_next_pilot_id
    !=
    CELL42_COMPLETED_PILOT_ID,
])


# ============================================================
# 8) Completed Engine session
# ============================================================

cell42_completed_session = copy.deepcopy(
    cell42_pre_advance_session
)


cell42_completed_session[
    "current_state"
] = "PILOT_COMPLETE"


cell42_completed_session[
    "queue_advance_permitted"
] = False


cell42_completed_session[
    "queue_advance_performed"
] = True


cell42_completed_session[
    "completion_status"
] = (
    "HUMAN_REVIEW_COMPLETE_"
    "AUTHOR_FINAL_REVIEW_PENDING"
)


# ============================================================
# 9) New authoritative Continuation State
# ============================================================

cell42_continuation_state = {
    "state_version":
        "compact_pilot_continuation_state_v6",

    "authoritative_forward_state":
        True,

    "parent_state_sha256":
        cell39_continuation_state_sha256,

    "just_completed_pilot_id":
        CELL42_COMPLETED_PILOT_ID,

    "just_completed_target_evidence_id":
        CELL42_COMPLETED_TARGET_ID,

    "just_completed_case_sha256":
        cell42_completed_human_review_case_sha256,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell42_remaining_pilot_ids
        ),

    "remaining_pilot_queue":
        copy.deepcopy(
            cell42_remaining_pilot_queue
        ),

    "remaining_count":
        16,

    "next_pilot_id":
        cell42_next_pilot_id,

    "next_pilot_queue_entry":
        copy.deepcopy(
            cell42_next_pilot_queue_entry
        ),

    "full_572_generation":
        CELL42_FULL_572_BLOCK_STATUS,
}


cell42_continuation_state_sha256 = (
    cell10_stable_sha(
        cell42_continuation_state
    )
)


# ============================================================
# 10) Resolve next Pilot packet
# ============================================================

cell42_pilot_by_id = (
    cell13_seed[
        "final_runtime"
    ][
        "question_drafting_pilot_by_id"
    ]
)


cell42_next_pilot_packet = copy.deepcopy(
    cell42_pilot_by_id.get(
        cell42_next_pilot_id
    )
)


cell42_next_packet_valid = isinstance(
    cell42_next_pilot_packet,
    dict,
)


# ============================================================
# 11) Resolve next identity chain
# ============================================================

cell42_next_golden_case_id = None
cell42_next_task_draft_id = None
cell42_next_target_evidence_id = None


if cell42_next_packet_valid:

    try:

        cell42_next_golden_case_id = (
            cell13_exact_single_id(
                cell42_next_pilot_packet,
                "golden_case",
            )
        )


        cell42_next_task_draft_id = (
            cell13_exact_single_id(
                cell42_next_pilot_packet,
                "task_draft",
            )
        )


        cell42_next_target_evidence_id = (
            cell13_exact_single_id(
                cell42_next_pilot_packet,
                "target_evidence",
            )
        )


    except Exception as exc:

        cell42_failures.append({
            "reason":
                "NEXT_ID_CHAIN_FAILED",

            "error":
                repr(
                    exc
                ),
        })


cell42_next_identity_valid = all([
    isinstance(
        cell42_next_golden_case_id,
        str,
    ),

    isinstance(
        cell42_next_task_draft_id,
        str,
    ),

    isinstance(
        cell42_next_target_evidence_id,
        str,
    ),
])


# ============================================================
# 12) Resolve next support records
# ============================================================

cell42_next_golden_case = None
cell42_next_task_plan = None
cell42_next_target_evidence = None


if cell42_next_identity_valid:

    try:

        cell42_next_golden_case = copy.deepcopy(
            cell13_active_support[
                "golden_case_records_by_id"
            ][
                cell42_next_golden_case_id
            ]
        )


        cell42_next_task_plan = copy.deepcopy(
            cell13_find_unique_mapping_value(
                cell13_active_support.get(
                    "task_draft_plans",
                    {},
                ),
                "task_draft",
                cell42_next_task_draft_id,
            )[
                "value"
            ]
        )


        cell42_next_target_evidence = copy.deepcopy(
            cell13_find_unique_list_record(
                cell13_active_support.get(
                    "target_evidence_records",
                    [],
                ),
                "target_evidence_id",
                cell42_next_target_evidence_id,
            )
        )


    except Exception as exc:

        cell42_failures.append({
            "reason":
                "NEXT_SUPPORT_RESOLUTION_FAILED",

            "error":
                repr(
                    exc
                ),
        })


# ============================================================
# 13) Classification helper
# ============================================================

def cell42_scalar(
    mapping,
    names,
):

    match = cell14_find_first_scalar(
        mapping,
        names,
    )


    if isinstance(
        match,
        dict,
    ):

        return match.get(
            "value"
        )


    return None


cell42_next_source_type = cell42_scalar(
    cell42_next_pilot_packet,
    [
        "source_type",
        "source_kind",
        "source",
    ],
)


cell42_next_stratum = cell42_scalar(
    cell42_next_pilot_packet,
    [
        "stratum",
        "stratum_name",
        "stratum_id",
    ],
)


cell42_next_task_family = cell42_scalar(
    cell42_next_pilot_packet,
    [
        "task_family",
    ],
)


cell42_next_question_archetype = cell42_scalar(
    cell42_next_pilot_packet,
    [
        "question_archetype",
    ],
)


# ============================================================
# 14) Extract next Source Evidence
# ============================================================

cell42_raw_target_parts = []


if cell42_next_packet_valid:

    packet_parts = (
        cell42_next_pilot_packet.get(
            "target_text_parts"
        )
    )


    if isinstance(
        packet_parts,
        list,
    ):

        cell42_raw_target_parts = copy.deepcopy(
            packet_parts
        )


if (
    not cell42_raw_target_parts
    and
    isinstance(
        cell42_next_target_evidence,
        dict,
    )
):

    evidence_parts = (
        cell42_next_target_evidence.get(
            "target_parts"
        )
    )


    if isinstance(
        evidence_parts,
        list,
    ):

        cell42_raw_target_parts = copy.deepcopy(
            evidence_parts
        )


cell42_next_target_text_parts = []


for index, part in enumerate(
    cell42_raw_target_parts
):

    if not isinstance(
        part,
        dict,
    ):
        continue


    text_value = None


    for field_name in (
        "text",
        "target_text",
        "content",
        "evidence_text",
        "text_content",
    ):

        candidate = part.get(
            field_name
        )


        if (
            isinstance(
                candidate,
                str,
            )
            and
            candidate.strip()
        ):

            text_value = candidate.strip()
            break


    if text_value is None:
        continue


    cell42_next_target_text_parts.append({
        "part_index":
            index,

        "document_id":
            part.get(
                "document_id"
            ),

        "role":
            part.get(
                "role"
            ),

        "char_start":
            part.get(
                "char_start"
            ),

        "char_end":
            part.get(
                "char_end"
            ),

        "text":
            text_value,

        "text_sha256":
            hashlib.sha256(
                text_value.encode(
                    "utf-8"
                )
            ).hexdigest(),
    })


cell42_next_combined_target_text = (
    "\n\n".join(
        row[
            "text"
        ]

        for row
        in cell42_next_target_text_parts
    )
)


cell42_next_target_text_available = bool(
    cell42_next_combined_target_text.strip()
)


# ============================================================
# 15) Build next Pilot workspace
# ============================================================

cell42_next_pilot_workspace = {
    "workspace_version":
        "compact_human_review_workspace_v1",

    "status":
        "READY_FOR_QUESTION_DRAFT",

    "pilot_case_id":
        cell42_next_pilot_id,

    "golden_case_id":
        cell42_next_golden_case_id,

    "task_draft_id":
        cell42_next_task_draft_id,

    "target_evidence_id":
        cell42_next_target_evidence_id,

    "source_type":
        cell42_next_source_type,

    "stratum":
        cell42_next_stratum,

    "task_family":
        cell42_next_task_family,

    "question_archetype":
        cell42_next_question_archetype,

    "source_evidence":
        copy.deepcopy(
            cell42_next_target_text_parts
        ),

    "combined_target_text":
        cell42_next_combined_target_text,

    "golden_case":
        copy.deepcopy(
            cell42_next_golden_case
        ),

    "task_plan":
        copy.deepcopy(
            cell42_next_task_plan
        ),

    "target_evidence":
        copy.deepcopy(
            cell42_next_target_evidence
        ),

    "model_drafts":
        [],

    "human_reviews":
        [],

    "expected_answer_review":
        None,

    "citation_review":
        None,

    "author_final_review": {
        "status":
            "NOT_STARTED",

        "decision":
            None,

        "notes":
            None,
    },

    "workflow_state":
        "QUESTION_DRAFT_PENDING",

    "full_572_generation":
        CELL42_FULL_572_BLOCK_STATUS,
}


cell42_next_workspace_ready = all([
    cell42_next_packet_valid,

    cell42_next_identity_valid,

    isinstance(
        cell42_next_golden_case,
        dict,
    ),

    isinstance(
        cell42_next_task_plan,
        dict,
    ),

    isinstance(
        cell42_next_target_evidence,
        dict,
    ),

    cell42_next_target_text_available,
])


# ============================================================
# 16) Update Completed Human Review cases
# ============================================================

cell42_completed_cases = copy.deepcopy(
    cell41_persisted_registry.get(
        "completed_human_review_cases",
        [],
    )
)


cell42_completed_cases.append(
    copy.deepcopy(
        cell42_completed_human_review_case
    )
)


# ============================================================
# 17) Build persistent Registry payload
# ============================================================

cell42_persistent_registry_payload = copy.deepcopy(
    cell41_persisted_registry
)


cell42_persistent_registry_payload[
    "completed_human_review_cases"
] = copy.deepcopy(
    cell42_completed_cases
)


cell42_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell42_next_pilot_workspace
)


cell42_persistent_registry_payload[
    "remaining_pilot_count"
] = 16


cell42_persistent_registry_payload[
    "continuation_state"
] = {
    "next_pilot_id":
        cell42_next_pilot_id,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell42_remaining_pilot_ids
        ),

    "remaining_count":
        16,

    "continuation_state_sha256":
        cell42_continuation_state_sha256,
}


# ============================================================
# 18) Author Review state
# ============================================================

cell42_author_review = copy.deepcopy(
    cell42_persistent_registry_payload.get(
        "author_review",
        {},
    )
)


if not isinstance(
    cell42_author_review,
    dict,
):

    cell42_author_review = {}


cell42_author_review[
    "status"
] = "NOT_STARTED"


cell42_author_review[
    "reviewer"
] = "AUTHOR"


cell42_author_review[
    "cases_ready_for_review"
] = len(
    cell42_completed_cases
)


cell42_persistent_registry_payload[
    "author_review"
] = copy.deepcopy(
    cell42_author_review
)


# ============================================================
# 19) Persist
# ============================================================

cell42_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell42_persistent_registry_payload
    )
)


# ============================================================
# 20) Read-back validation
# ============================================================

cell42_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell42_persisted_active_case = (
    cell42_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell42_persistence_valid = all([
    len(
        cell42_persisted_registry.get(
            "completed_human_review_cases",
            [],
        )
    )
    ==
    len(
        cell42_completed_cases
    ),

    cell42_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    16,

    cell42_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    cell42_next_pilot_id,

    cell42_persisted_registry.get(
        "continuation_state",
        {},
    ).get(
        "next_pilot_id"
    )
    ==
    cell42_next_pilot_id,

    cell42_persisted_registry.get(
        "author_review",
        {},
    ).get(
        "cases_ready_for_review"
    )
    ==
    len(
        cell42_completed_cases
    ),
])


# ============================================================
# 21) Safety
# ============================================================

cell42_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell42_full_572_blocked = (
    cell42_continuation_state[
        "full_572_generation"
    ]
    ==
    CELL42_FULL_572_BLOCK_STATUS
)


# ============================================================
# 22) Final Gate
# ============================================================

cell42_gate = all([
    len(
        cell42_failures
    )
    ==
    0,

    cell42_citation_accept_valid,

    cell42_queue_preconditions_valid,

    cell42_queue_advanced_valid,

    cell42_next_workspace_ready,

    cell42_persistence_valid,

    cell42_base_seed_unchanged,

    cell42_full_572_blocked,
])


# ============================================================
# 23) Diagnostics
# ============================================================

print(
    "=== PILOT COMPLETED + QUEUE ADVANCE "
    "+ NEXT PILOT ==="
)


print(
    "\n=== COMPLETED PILOT ==="
)

print(
    "Pilot:",
    CELL42_COMPLETED_PILOT_ID,
)

print(
    "Question:",
    CELL40_ACCEPTED_QUESTION,
)

print(
    "Answer:",
    CELL40_EXPECTED_ANSWER,
)

print(
    "Citation:",
    "ACCEPT",
)

print(
    "Human Review status:",
    cell42_completed_human_review_case[
        "final_status"
    ],
)

print(
    "Author Final Review:",
    "NOT_STARTED",
)


print(
    "\n=== QUEUE ==="
)

print(
    "Before:",
    len(
        cell42_parent_remaining_ids
    ),
)

print(
    "After:",
    len(
        cell42_remaining_pilot_ids
    ),
)

print(
    "Completed Pilot removed:",
    (
        CELL42_COMPLETED_PILOT_ID
        not in
        cell42_remaining_pilot_ids
    ),
)

print(
    "Next Pilot:",
    cell42_next_pilot_id,
)


print(
    "\n=== PERSISTENT REGISTRY ==="
)

print(
    "Path:",
    cell21_registry_path,
)

print(
    "Completed Human Review cases:",
    len(
        cell42_persisted_registry.get(
            "completed_human_review_cases",
            [],
        )
    ),
)

print(
    "Cases ready for Author Review:",
    cell42_persisted_registry.get(
        "author_review",
        {},
    ).get(
        "cases_ready_for_review"
    ),
)

print(
    "Remaining Pilots:",
    cell42_persisted_registry.get(
        "remaining_pilot_count"
    ),
)

print(
    "Active Pilot:",
    cell42_persisted_active_case.get(
        "pilot_case_id"
    ),
)

print(
    "Persistence valid:",
    cell42_persistence_valid,
)


print(
    "\n=== NEXT PILOT IDENTITY ==="
)

print(
    "Pilot:",
    cell42_next_pilot_id,
)

print(
    "Golden Case:",
    cell42_next_golden_case_id,
)

print(
    "Task Draft:",
    cell42_next_task_draft_id,
)

print(
    "Target Evidence:",
    cell42_next_target_evidence_id,
)

print(
    "Source type:",
    cell42_next_source_type,
)

print(
    "Stratum:",
    cell42_next_stratum,
)

print(
    "Task family:",
    cell42_next_task_family,
)

print(
    "Question archetype:",
    cell42_next_question_archetype,
)


print(
    "\n=== NEXT PILOT SOURCE EVIDENCE ==="
)

print(
    cell42_next_combined_target_text
)


print(
    "\n=== SAFETY ==="
)

print(
    "Base Seed unchanged:",
    cell42_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell42_full_572_blocked,
)

print(
    "Author Notebook rewrite:",
    False,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell42_failures
)


print(
    "\nCODE CELL 42 GATE:",
    (
        "PASS"
        if cell42_gate
        else "FAIL"
    ),
)


assert cell42_gate, (
    "Citation ACCEPT / Pilot completion / Queue advance / "
    "Registry sync / next Pilot preparation failed. "
    "Repair this SAME Code Cell 42 before continuing."
)

=== PILOT COMPLETED + QUEUE ADVANCE + NEXT PILOT ===

=== COMPLETED PILOT ===
Pilot: question-pilot:672f5241db66879fb47096c5bb0f5568
Question: آیا آموزش سواد مالی مخصوص سن خاصی است، یا می‌توان آن را متناسب با سنین مختلف آموزش داد؟
Answer: آموزش سواد مالی مخصوص یک سن خاص نیست. بر اساس استاندارد سواد مالی، برای مقاطع سنی مختلف دانش‌ها و توانایی‌های متناسبی در نظر گرفته می‌شود. هر سطح نیز بر سطح قبلی توسعه پیدا می‌کند؛ بنابراین دانش و مهارت‌های مالی می‌توانند هم‌زمان با رشد فرد، مرحله‌به‌مرحله گسترش پیدا کنند.
Citation: ACCEPT
Human Review status: HUMAN_REVIEW_COMPLETE_AUTHOR_FINAL_REVIEW_PENDING
Author Final Review: NOT_STARTED

=== QUEUE ===
Before: 17
After: 16
Completed Pilot removed: True
Next Pilot: question-pilot:7334395fa6456f24ed250e2b354cc184

=== PERSISTENT REGISTRY ===
Path: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/data/phase2/human_review/human_review_registry.json
Completed Human Review cases: 6
Cases ready for Author Review: 6
Remaining Pilo

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 43 — Question ACCEPT + Expected Answer Draft + Registry Sync

Human Reviewer سؤال این Pilot را پذیرفت:

**در این پرسشنامه، برای سنجش رضایت شغلی چه جنبه‌هایی از شغل و محیط کار بررسی می‌شوند؟**

در این مرحله:

- Question با تصمیم Human `ACCEPT` ثبت می‌شود؛
- Expected Answer فقط از ردیف‌های همین جدول ساخته می‌شود؛
- علامت `(R)` تفسیر نمی‌شود، چون معنای آن در Evidence فعلی توضیح داده نشده است؛
- Citation هنوز ساخته نمی‌شود؛
- Registry دائمی Sync می‌شود؛
- Queue همچنان 16 باقی می‌ماند؛
- Full 572 Generation همچنان Blocked است.

</div>

In [43]:
# Code Cell 43

cell43_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell42_gate") is not True:
    cell43_failures.append(
        "CELL42_NOT_PASS"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell43_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Current Pilot identity
# ============================================================

CELL43_PILOT_ID = (
    cell42_next_pilot_id
)

CELL43_GOLDEN_CASE_ID = (
    cell42_next_golden_case_id
)

CELL43_TASK_DRAFT_ID = (
    cell42_next_task_draft_id
)

CELL43_TARGET_EVIDENCE_ID = (
    cell42_next_target_evidence_id
)

CELL43_FULL_572_BLOCK_STATUS = (
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


CELL43_ACCEPTED_QUESTION = (
    "در این پرسشنامه، برای سنجش رضایت شغلی چه جنبه‌هایی "
    "از شغل و محیط کار بررسی می‌شوند؟"
)


CELL43_EXPECTED_ANSWER = (
    "این پرسشنامه چند جنبه اصلی را بررسی می‌کند: "
    "ماهیت و جذابیت خودِ کار، رفتار و عملکرد سرپرست، "
    "حقوق و مزایا، ویژگی‌ها و رفتار همکاران، "
    "فرصت‌های رشد و ترفیع، و شرایط فیزیکی و امکانات "
    "محیط کار مانند فضای کاری، نور، تهویه، تجهیزات، "
    "ساختمان و کیفیت غذا."
)


# ============================================================
# 3) Source Evidence
# ============================================================

cell43_source_text = (
    cell42_next_combined_target_text
)


def cell43_fa_surface(
    text,
):

    value = cell15_normalize_text(
        text
    )

    value = (
        value
        .replace(
            "ي",
            "ی",
        )
        .replace(
            "ك",
            "ک",
        )
        .replace(
            "\u200c",
            " ",
        )
    )

    value = re.sub(
        r"\s+",
        " ",
        value,
    )

    return value.strip()


cell43_source_surface = (
    cell43_fa_surface(
        cell43_source_text
    )
)


# ============================================================
# 4) Evidence grounding
# ============================================================

cell43_evidence_grounding = {
    "work_itself":
        (
            "کار من"
            in
            cell43_source_surface
        ),

    "supervisor":
        (
            "سرپرستم"
            in
            cell43_source_surface
        ),

    "salary_and_benefits":
        (
            "حقوق"
            in
            cell43_source_surface
            and
            "مزایای دریافتی"
            in
            cell43_source_surface
        ),

    "coworkers":
        (
            "همکارانم"
            in
            cell43_source_surface
        ),

    "growth_and_promotion":
        (
            (
                "ترفیع"
                in
                cell43_source_surface
            )
            and
            (
                "رشد"
                in
                cell43_source_surface
            )
        ),

    "physical_work_environment":
        all([
            "محیط کاری"
            in
            cell43_source_surface,

            "نور"
            in
            cell43_source_surface,

            "تهویه"
            in
            cell43_source_surface,

            "تجهیزات اداری"
            in
            cell43_source_surface,
        ]),

    "building_and_food":
        (
            "ساختمان مورد استفاده"
            in
            cell43_source_surface
            and
            "کیفیت غذا"
            in
            cell43_source_surface
        ),
}


cell43_evidence_grounding_pass = all(
    cell43_evidence_grounding.values()
)


# ============================================================
# 5) Start Engine session
# ============================================================

cell43_question_pending_session = (
    cell13_build_session(
        cell42_next_pilot_packet
    )
)


cell43_session_start_valid = (
    cell43_question_pending_session.get(
        "current_state"
    )
    ==
    "QUESTION_DRAFT_PENDING"
)


# ============================================================
# 6) Question Draft ID
# ============================================================

cell43_question_identity_payload = {
    "identity_version":
        "question_draft_identity_v1",

    "pilot_case_id":
        CELL43_PILOT_ID,

    "target_evidence_id":
        CELL43_TARGET_EVIDENCE_ID,

    "question_text":
        CELL43_ACCEPTED_QUESTION,

    "revision_number":
        1,
}


cell43_question_draft_id = (
    "question-draft:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell43_question_identity_payload
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


# ============================================================
# 7) Question Machine Validation
# ============================================================

cell43_question_surface = (
    cell43_fa_surface(
        CELL43_ACCEPTED_QUESTION
    )
)


cell43_question_machine_validation = {
    "job_satisfaction_present":
        (
            "رضایت شغلی"
            in
            cell43_question_surface
        ),

    "job_dimension_present":
        (
            "شغل"
            in
            cell43_question_surface
        ),

    "work_environment_present":
        (
            "محیط کار"
            in
            cell43_question_surface
        ),

    "source_grounded":
        cell43_evidence_grounding_pass,

    "r_marker_interpreted":
        False,

    "external_knowledge_used":
        False,
}


cell43_question_machine_pass = all([
    cell43_question_machine_validation[
        "job_satisfaction_present"
    ],

    cell43_question_machine_validation[
        "job_dimension_present"
    ],

    cell43_question_machine_validation[
        "work_environment_present"
    ],

    cell43_question_machine_validation[
        "source_grounded"
    ],

    cell43_question_machine_validation[
        "r_marker_interpreted"
    ]
    is False,
])


# ============================================================
# 8) Question artifact
# ============================================================

cell43_question_record = {
    "record_version":
        "question_draft_record_v1",

    "question_draft_id":
        cell43_question_draft_id,

    "pilot_case_id":
        CELL43_PILOT_ID,

    "golden_case_id":
        CELL43_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL43_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL43_TARGET_EVIDENCE_ID,

    "question_text":
        CELL43_ACCEPTED_QUESTION,

    "revision_number":
        1,

    "machine_validation_status":
        (
            "PASS_PENDING_HUMAN_ADJUDICATION"
            if cell43_question_machine_pass
            else
            "FAIL_REQUIRES_REVISION"
        ),

    "human_adjudication_status":
        "PENDING",

    "external_knowledge_used":
        False,
}


# ============================================================
# 9) Register + Human ACCEPT Question
# ============================================================

cell43_question_review_session = (
    cell13_apply_event(
        cell43_question_pending_session,
        "QUESTION_DRAFT_REGISTERED",
        artifact=
            cell43_question_record,
    )
)


CELL43_QUESTION_HUMAN_NOTE = (
    "Human Reviewer accepted the question as a natural "
    "summary-retrieval question over the large table run. "
    "The meaning of the (R) marker was not inferred."
)


cell43_answer_pending_session = (
    cell13_apply_event(
        cell43_question_review_session,
        "QUESTION_ACCEPTED",
        human_note=
            CELL43_QUESTION_HUMAN_NOTE,
    )
)


cell43_question_accept_valid = all([
    cell43_question_review_session.get(
        "current_state"
    )
    ==
    "QUESTION_HUMAN_REVIEW_PENDING",

    cell43_answer_pending_session.get(
        "current_state"
    )
    ==
    "EXPECTED_ANSWER_DRAFT_PENDING",
])


# ============================================================
# 10) Expected Answer Draft ID
# ============================================================

cell43_answer_identity_payload = {
    "identity_version":
        "expected_answer_identity_v1",

    "pilot_case_id":
        CELL43_PILOT_ID,

    "question_draft_id":
        cell43_question_draft_id,

    "target_evidence_id":
        CELL43_TARGET_EVIDENCE_ID,

    "answer_text":
        CELL43_EXPECTED_ANSWER,

    "revision_number":
        1,
}


cell43_expected_answer_draft_id = (
    "expected-answer-draft:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell43_answer_identity_payload
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


# ============================================================
# 11) Expected Answer Machine Validation
# ============================================================

cell43_answer_surface = (
    cell43_fa_surface(
        CELL43_EXPECTED_ANSWER
    )
)


cell43_answer_grounding = {
    "work_itself":
        (
            "ماهیت و جذابیت"
            in
            cell43_answer_surface
            and
            "کار"
            in
            cell43_answer_surface
        ),

    "supervisor":
        (
            "سرپرست"
            in
            cell43_answer_surface
        ),

    "salary_and_benefits":
        (
            "حقوق و مزایا"
            in
            cell43_answer_surface
        ),

    "coworkers":
        (
            "همکاران"
            in
            cell43_answer_surface
        ),

    "growth_and_promotion":
        (
            "رشد و ترفیع"
            in
            cell43_answer_surface
        ),

    "workspace":
        (
            "فضای کاری"
            in
            cell43_answer_surface
        ),

    "lighting":
        (
            "نور"
            in
            cell43_answer_surface
        ),

    "ventilation":
        (
            "تهویه"
            in
            cell43_answer_surface
        ),

    "equipment":
        (
            "تجهیزات"
            in
            cell43_answer_surface
        ),

    "building":
        (
            "ساختمان"
            in
            cell43_answer_surface
        ),

    "food_quality":
        (
            "کیفیت غذا"
            in
            cell43_answer_surface
        ),
}


cell43_answer_machine_pass = all([
    cell43_evidence_grounding_pass,

    all(
        cell43_answer_grounding.values()
    ),
])


# ============================================================
# 12) Expected Answer artifact
# ============================================================

cell43_expected_answer_record = {
    "record_version":
        "expected_answer_draft_record_v1",

    "expected_answer_draft_id":
        cell43_expected_answer_draft_id,

    "pilot_case_id":
        CELL43_PILOT_ID,

    "golden_case_id":
        CELL43_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL43_TASK_DRAFT_ID,

    "question_draft_id":
        cell43_question_draft_id,

    "target_evidence_id":
        CELL43_TARGET_EVIDENCE_ID,

    "expected_answer_text":
        CELL43_EXPECTED_ANSWER,

    "revision_number":
        1,

    "machine_validation_status":
        (
            "PASS_PENDING_HUMAN_ADJUDICATION"
            if cell43_answer_machine_pass
            else
            "FAIL_REQUIRES_REVISION"
        ),

    "human_adjudication_status":
        "PENDING",

    "r_marker_interpreted":
        False,

    "external_knowledge_used":
        False,
}


# ============================================================
# 13) Register Expected Answer
# ============================================================

cell43_current_session = (
    cell13_apply_event(
        cell43_answer_pending_session,
        "EXPECTED_ANSWER_REGISTERED",
        artifact=
            cell43_expected_answer_record,
    )
)


cell43_answer_registered_valid = (
    cell43_current_session.get(
        "current_state"
    )
    ==
    "EXPECTED_ANSWER_HUMAN_REVIEW_PENDING"
)


# ============================================================
# 14) Human Review workspace
# ============================================================

cell43_active_human_review_case = copy.deepcopy(
    cell42_next_pilot_workspace
)


cell43_active_human_review_case[
    "status"
] = "ACTIVE_HUMAN_REVIEW"


cell43_active_human_review_case[
    "workflow_state"
] = (
    "EXPECTED_ANSWER_HUMAN_REVIEW_PENDING"
)


cell43_active_human_review_case[
    "question_review"
] = {
    "question_draft_id":
        cell43_question_draft_id,

    "question_text":
        CELL43_ACCEPTED_QUESTION,

    "machine_validation_status":
        cell43_question_record[
            "machine_validation_status"
        ],

    "human_decision":
        "ACCEPT",

    "human_note":
        CELL43_QUESTION_HUMAN_NOTE,
}


cell43_active_human_review_case[
    "expected_answer_review"
] = {
    "expected_answer_draft_id":
        cell43_expected_answer_draft_id,

    "expected_answer_text":
        CELL43_EXPECTED_ANSWER,

    "machine_validation_status":
        cell43_expected_answer_record[
            "machine_validation_status"
        ],

    "human_decision":
        "PENDING",
}


cell43_active_human_review_case[
    "citation_review"
] = None


cell43_active_human_review_case[
    "author_final_review"
] = {
    "status":
        "NOT_STARTED",

    "decision":
        None,

    "notes":
        None,
}


# ============================================================
# 15) Persist Registry
# ============================================================

cell43_persistent_registry_payload = copy.deepcopy(
    cell42_persisted_registry
)


cell43_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell43_active_human_review_case
)


cell43_persistent_registry_payload[
    "remaining_pilot_count"
] = 16


cell43_persistent_registry_payload[
    "continuation_state"
] = {
    "next_pilot_id":
        CELL43_PILOT_ID,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell42_remaining_pilot_ids
        ),

    "remaining_count":
        16,

    "continuation_state_sha256":
        cell42_continuation_state_sha256,

    "active_pilot_status":
        "EXPECTED_ANSWER_HUMAN_REVIEW_PENDING",

    "active_target_evidence_id":
        CELL43_TARGET_EVIDENCE_ID,
}


cell43_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell43_persistent_registry_payload
    )
)


# ============================================================
# 16) Read-back validation
# ============================================================

cell43_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell43_persisted_active_case = (
    cell43_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell43_persistence_valid = all([
    cell43_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL43_PILOT_ID,

    cell43_persisted_active_case.get(
        "question_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell43_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "PENDING",

    cell43_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    16,
])


# ============================================================
# 17) Safety
# ============================================================

cell43_queue_still_16 = (
    len(
        cell42_remaining_pilot_ids
    )
    ==
    16
)


cell43_current_pilot_still_first = (
    cell42_remaining_pilot_ids[
        0
    ]
    ==
    CELL43_PILOT_ID
)


cell43_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell43_full_572_blocked = (
    CELL43_FULL_572_BLOCK_STATUS
    ==
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


# ============================================================
# 18) Final Gate
# ============================================================

cell43_gate = all([
    len(
        cell43_failures
    )
    ==
    0,

    cell43_evidence_grounding_pass,

    cell43_session_start_valid,

    cell43_question_machine_pass,

    cell43_question_accept_valid,

    cell43_answer_machine_pass,

    cell43_answer_registered_valid,

    cell43_persistence_valid,

    cell43_queue_still_16,

    cell43_current_pilot_still_first,

    cell43_base_seed_unchanged,

    cell43_full_572_blocked,
])


# ============================================================
# 19) Diagnostics
# ============================================================

print(
    "=== QUESTION ACCEPTED + EXPECTED ANSWER DRAFT "
    "+ REGISTRY SYNC ==="
)


print(
    "\n=== ACCEPTED QUESTION ==="
)

print(
    CELL43_ACCEPTED_QUESTION
)

print(
    "Question Draft ID:",
    cell43_question_draft_id,
)

print(
    "Question Machine PASS:",
    cell43_question_machine_pass,
)


print(
    "\n=== EXPECTED ANSWER DRAFT ==="
)

print(
    CELL43_EXPECTED_ANSWER
)

print(
    "Expected Answer Draft ID:",
    cell43_expected_answer_draft_id,
)

print(
    "Machine PASS:",
    cell43_answer_machine_pass,
)


print(
    "\n=== ANSWER GROUNDING ==="
)

for name, status in (
    cell43_answer_grounding.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== SOURCE INTERPRETATION SAFETY ==="
)

print(
    "(R) marker interpreted:",
    False,
)

print(
    "External knowledge used:",
    False,
)


print(
    "\n=== WORKFLOW ==="
)

print(
    "Current State:",
    cell43_current_session.get(
        "current_state"
    ),
)

print(
    "Question Decision:",
    "ACCEPT",
)

print(
    "Expected Answer Decision:",
    "PENDING",
)

print(
    "Citation:",
    None,
)


print(
    "\n=== PERSISTENCE ==="
)

print(
    "Registry:",
    cell21_registry_path,
)

print(
    "Persisted Active Pilot:",
    cell43_persisted_active_case.get(
        "pilot_case_id"
    ),
)

print(
    "Persisted Question Decision:",
    cell43_persisted_active_case.get(
        "question_review",
        {},
    ).get(
        "human_decision"
    ),
)

print(
    "Persisted Answer Decision:",
    cell43_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    ),
)

print(
    "Persistence valid:",
    cell43_persistence_valid,
)


print(
    "\n=== QUEUE / SAFETY ==="
)

print(
    "Queue still 16:",
    cell43_queue_still_16,
)

print(
    "Base Seed unchanged:",
    cell43_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell43_full_572_blocked,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell43_failures
)


print(
    "\nCODE CELL 43 GATE:",
    (
        "PASS"
        if cell43_gate
        else "FAIL"
    ),
)


assert cell43_gate, (
    "Question ACCEPT / Expected Answer Draft / Registry Sync "
    "failed. Repair this SAME Code Cell 43 before "
    "Expected Answer Human Review."
)

=== QUESTION ACCEPTED + EXPECTED ANSWER DRAFT + REGISTRY SYNC ===

=== ACCEPTED QUESTION ===
در این پرسشنامه، برای سنجش رضایت شغلی چه جنبه‌هایی از شغل و محیط کار بررسی می‌شوند؟
Question Draft ID: question-draft:b4d74de668ff50278bf9ff2bbac71940
Question Machine PASS: True

=== EXPECTED ANSWER DRAFT ===
این پرسشنامه چند جنبه اصلی را بررسی می‌کند: ماهیت و جذابیت خودِ کار، رفتار و عملکرد سرپرست، حقوق و مزایا، ویژگی‌ها و رفتار همکاران، فرصت‌های رشد و ترفیع، و شرایط فیزیکی و امکانات محیط کار مانند فضای کاری، نور، تهویه، تجهیزات، ساختمان و کیفیت غذا.
Expected Answer Draft ID: expected-answer-draft:b9a89ff26950c852d25848e5a0ae2be5
Machine PASS: True

=== ANSWER GROUNDING ===
work_itself: True
supervisor: True
salary_and_benefits: True
coworkers: True
growth_and_promotion: True
workspace: True
lighting: True
ventilation: True
equipment: True
building: True
food_quality: True

=== SOURCE INTERPRETATION SAFETY ===
(R) marker interpreted: False
External knowledge used: False

=== WORKFLOW ===
Curr

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 44 — Expected Answer ACCEPT + Citation Draft + Registry Sync

Expected Answer این Pilot توسط Human Reviewer پذیرفته شد.

در این مرحله:

- Expected Answer با تصمیم `ACCEPT` ثبت می‌شود؛
- Citation فقط از Target Evidence همین Pilot ساخته می‌شود؛
- همه Target Partهای جدول پوشش داده می‌شوند؛
- Locatorها مستقیماً از Evidence موجود کپی می‌شوند؛
- هیچ Locator یا منبع خارجی ساخته نمی‌شود؛
- Citation وارد Human Review می‌شود؛
- Registry دائمی Sync می‌شود؛
- Queue همچنان 16 باقی می‌ماند؛
- Full 572 Generation همچنان Blocked است.

</div>

In [44]:
# Code Cell 44

cell44_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell43_gate") is not True:
    cell44_failures.append(
        "CELL43_NOT_PASS"
    )


if (
    globals().get(
        "cell43_current_session",
        {},
    ).get(
        "current_state"
    )
    !=
    "EXPECTED_ANSWER_HUMAN_REVIEW_PENDING"
):
    cell44_failures.append(
        "EXPECTED_ANSWER_NOT_AT_HUMAN_REVIEW_GATE"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell44_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Current identities
# ============================================================

CELL44_PILOT_ID = (
    CELL43_PILOT_ID
)

CELL44_GOLDEN_CASE_ID = (
    CELL43_GOLDEN_CASE_ID
)

CELL44_TASK_DRAFT_ID = (
    CELL43_TASK_DRAFT_ID
)

CELL44_TARGET_EVIDENCE_ID = (
    CELL43_TARGET_EVIDENCE_ID
)

CELL44_QUESTION_DRAFT_ID = (
    cell43_question_draft_id
)

CELL44_EXPECTED_ANSWER_DRAFT_ID = (
    cell43_expected_answer_draft_id
)

CELL44_FULL_572_BLOCK_STATUS = (
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


# ============================================================
# 3) Human ACCEPT for Expected Answer
# ============================================================

CELL44_HUMAN_ANSWER_NOTE = (
    "Human Reviewer accepted the Expected Answer after "
    "reviewing the summarized job-satisfaction dimensions "
    "against the large-table source evidence."
)


cell44_answer_human_review = {
    "review_record_version":
        "human_expected_answer_review_record_v1",

    "pilot_case_id":
        CELL44_PILOT_ID,

    "question_draft_id":
        CELL44_QUESTION_DRAFT_ID,

    "expected_answer_draft_id":
        CELL44_EXPECTED_ANSWER_DRAFT_ID,

    "target_evidence_id":
        CELL44_TARGET_EVIDENCE_ID,

    "human_decision":
        "ACCEPT",

    "human_note":
        CELL44_HUMAN_ANSWER_NOTE,

    "r_marker_interpreted":
        False,

    "human_context_used_as_evidence":
        False,

    "external_knowledge_used":
        False,
}


# ============================================================
# 4) Apply Expected Answer ACCEPT
# ============================================================

cell44_citation_pending_session = (
    cell13_apply_event(
        cell43_current_session,
        "EXPECTED_ANSWER_ACCEPTED",
        human_note=
            CELL44_HUMAN_ANSWER_NOTE,
    )
)


cell44_answer_accept_valid = all([
    cell44_citation_pending_session.get(
        "current_state"
    )
    ==
    "CITATION_MAPPING_PENDING",

    cell44_citation_pending_session.get(
        "queue_advance_performed"
    )
    is False,

    cell44_citation_pending_session.get(
        "human_decisions",
        [],
    )[
        -1
    ].get(
        "event"
    )
    ==
    "EXPECTED_ANSWER_ACCEPTED",
])


# ============================================================
# 5) Read Target Evidence parts
# ============================================================

cell44_target_parts = copy.deepcopy(
    cell42_next_target_evidence.get(
        "target_parts",
        [],
    )
)


cell44_target_parts_valid = all([
    isinstance(
        cell44_target_parts,
        list,
    ),

    len(
        cell44_target_parts
    )
    >
    0,
])


# ============================================================
# 6) Build Citation entries
# ============================================================

cell44_citation_entries = []


for index, part in enumerate(
    cell44_target_parts
):

    if not isinstance(
        part,
        dict,
    ):
        continue


    cell44_citation_entries.append({
        "citation_index":
            index,

        "target_evidence_id":
            CELL44_TARGET_EVIDENCE_ID,

        "document_id":
            part.get(
                "document_id"
            ),

        "source_id":
            part.get(
                "source_id"
            ),

        "role":
            part.get(
                "role"
            ),

        "char_start":
            part.get(
                "char_start"
            ),

        "char_end":
            part.get(
                "char_end"
            ),

        "char_count":
            part.get(
                "char_count"
            ),

        "token_count":
            part.get(
                "token_count"
            ),

        "locator":
            copy.deepcopy(
                part.get(
                    "locator",
                    {},
                )
            ),

        "attribution":
            copy.deepcopy(
                part.get(
                    "attribution"
                )
            ),

        "locator_origin":
            "COPIED_FROM_VALIDATED_TARGET_EVIDENCE",

        "external_locator_lookup_used":
            False,
    })


# ============================================================
# 7) Exact projection validation
# ============================================================

cell44_source_projection = [
    {
        "document_id":
            part.get(
                "document_id"
            ),

        "source_id":
            part.get(
                "source_id"
            ),

        "role":
            part.get(
                "role"
            ),

        "char_start":
            part.get(
                "char_start"
            ),

        "char_end":
            part.get(
                "char_end"
            ),

        "locator":
            copy.deepcopy(
                part.get(
                    "locator",
                    {},
                )
            ),
    }

    for part
    in cell44_target_parts

    if isinstance(
        part,
        dict,
    )
]


cell44_citation_projection = [
    {
        "document_id":
            entry.get(
                "document_id"
            ),

        "source_id":
            entry.get(
                "source_id"
            ),

        "role":
            entry.get(
                "role"
            ),

        "char_start":
            entry.get(
                "char_start"
            ),

        "char_end":
            entry.get(
                "char_end"
            ),

        "locator":
            copy.deepcopy(
                entry.get(
                    "locator",
                    {},
                )
            ),
    }

    for entry
    in cell44_citation_entries
]


cell44_no_locator_fabrication = (
    cell44_source_projection
    ==
    cell44_citation_projection
)


# ============================================================
# 8) Displayed Evidence coverage
# ============================================================

cell44_displayed_parts = copy.deepcopy(
    cell42_next_target_text_parts
)


cell44_displayed_by_identity = {
    (
        row.get(
            "document_id"
        ),
        row.get(
            "role"
        ),
        row.get(
            "char_start"
        ),
        row.get(
            "char_end"
        ),
    ):
        row

    for row
    in cell44_displayed_parts

    if isinstance(
        row,
        dict,
    )
}


cell44_validation_rows = []


for entry in cell44_citation_entries:

    identity = (
        entry.get(
            "document_id"
        ),
        entry.get(
            "role"
        ),
        entry.get(
            "char_start"
        ),
        entry.get(
            "char_end"
        ),
    )


    displayed = (
        cell44_displayed_by_identity.get(
            identity
        )
    )


    locator = (
        entry.get(
            "locator"
        )
        or
        {}
    )


    canonical_url = locator.get(
        "canonical_url"
    )


    url = locator.get(
        "url"
    )


    row_valid = all([
        isinstance(
            entry.get(
                "document_id"
            ),
            str,
        ),

        isinstance(
            displayed,
            dict,
        ),

        (
            entry.get(
                "char_start"
            )
            is None
            or
            isinstance(
                entry.get(
                    "char_start"
                ),
                int,
            )
        ),

        (
            entry.get(
                "char_end"
            )
            is None
            or
            isinstance(
                entry.get(
                    "char_end"
                ),
                int,
            )
        ),

        (
            canonical_url is None
            or
            (
                isinstance(
                    canonical_url,
                    str,
                )
                and
                canonical_url.startswith(
                    (
                        "http://",
                        "https://",
                    )
                )
            )
        ),

        (
            url is None
            or
            (
                isinstance(
                    url,
                    str,
                )
                and
                url.startswith(
                    (
                        "http://",
                        "https://",
                    )
                )
            )
        ),

        entry.get(
            "locator_origin"
        )
        ==
        "COPIED_FROM_VALIDATED_TARGET_EVIDENCE",

        entry.get(
            "external_locator_lookup_used"
        )
        is False,
    ])


    cell44_validation_rows.append({
        "citation_index":
            entry.get(
                "citation_index"
            ),

        "document_id":
            entry.get(
                "document_id"
            ),

        "role":
            entry.get(
                "role"
            ),

        "char_start":
            entry.get(
                "char_start"
            ),

        "char_end":
            entry.get(
                "char_end"
            ),

        "displayed_match":
            isinstance(
                displayed,
                dict,
            ),

        "valid":
            row_valid,
    })


cell44_all_entries_valid = all([
    len(
        cell44_citation_entries
    )
    >
    0,

    len(
        cell44_citation_entries
    )
    ==
    len(
        cell44_target_parts
    ),

    len(
        cell44_citation_entries
    )
    ==
    len(
        cell44_displayed_parts
    ),

    all(
        row.get(
            "valid"
        )
        is True

        for row
        in cell44_validation_rows
    ),
])


# ============================================================
# 9) Deterministic Citation Mapping ID
# ============================================================

cell44_citation_identity_payload = {
    "identity_version":
        "citation_mapping_identity_v1",

    "pilot_case_id":
        CELL44_PILOT_ID,

    "question_draft_id":
        CELL44_QUESTION_DRAFT_ID,

    "expected_answer_draft_id":
        CELL44_EXPECTED_ANSWER_DRAFT_ID,

    "target_evidence_id":
        CELL44_TARGET_EVIDENCE_ID,

    "citation_projection":
        copy.deepcopy(
            cell44_citation_projection
        ),
}


cell44_citation_mapping_id = (
    "citation-mapping:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell44_citation_identity_payload
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


# ============================================================
# 10) Citation Machine Validation
# ============================================================

cell44_citation_machine_validation = {
    "target_parts_present":
        cell44_target_parts_valid,

    "citation_entries_present":
        (
            len(
                cell44_citation_entries
            )
            >
            0
        ),

    "all_target_parts_covered":
        (
            len(
                cell44_citation_entries
            )
            ==
            len(
                cell44_target_parts
            )
        ),

    "displayed_evidence_covered":
        (
            len(
                cell44_citation_entries
            )
            ==
            len(
                cell44_displayed_parts
            )
        ),

    "all_entries_valid":
        cell44_all_entries_valid,

    "locator_fabrication_absent":
        cell44_no_locator_fabrication,

    "target_evidence_identity_match":
        all(
            entry.get(
                "target_evidence_id"
            )
            ==
            CELL44_TARGET_EVIDENCE_ID

            for entry
            in cell44_citation_entries
        ),

    "r_marker_interpreted":
        False,

    "external_locator_lookup_used":
        False,
}


cell44_citation_machine_pass = all([
    cell44_citation_machine_validation[
        "target_parts_present"
    ],

    cell44_citation_machine_validation[
        "citation_entries_present"
    ],

    cell44_citation_machine_validation[
        "all_target_parts_covered"
    ],

    cell44_citation_machine_validation[
        "displayed_evidence_covered"
    ],

    cell44_citation_machine_validation[
        "all_entries_valid"
    ],

    cell44_citation_machine_validation[
        "locator_fabrication_absent"
    ],

    cell44_citation_machine_validation[
        "target_evidence_identity_match"
    ],

    cell44_citation_machine_validation[
        "r_marker_interpreted"
    ]
    is False,
])


# ============================================================
# 11) Citation Mapping Record
# ============================================================

cell44_citation_mapping_record = {
    "record_version":
        "citation_mapping_record_v1",

    "citation_mapping_id":
        cell44_citation_mapping_id,

    "pilot_case_id":
        CELL44_PILOT_ID,

    "golden_case_id":
        CELL44_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL44_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL44_TARGET_EVIDENCE_ID,

    "question_draft_id":
        CELL44_QUESTION_DRAFT_ID,

    "expected_answer_draft_id":
        CELL44_EXPECTED_ANSWER_DRAFT_ID,

    "citation_entries":
        copy.deepcopy(
            cell44_citation_entries
        ),

    "citation_scope":
        "VALIDATED_TARGET_EVIDENCE_ONLY",

    "locator_source":
        "EXISTING_TARGET_EVIDENCE_LOCATORS",

    "r_marker_interpreted":
        False,

    "external_locator_lookup_used":
        False,

    "machine_validation_status":
        (
            "PASS_PENDING_HUMAN_ADJUDICATION"
            if cell44_citation_machine_pass
            else
            "FAIL_REQUIRES_REVISION"
        ),

    "human_adjudication_status":
        "PENDING",

    "full_572_generation":
        CELL44_FULL_572_BLOCK_STATUS,
}


# ============================================================
# 12) Register Citation
# ============================================================

cell44_current_session = (
    cell13_apply_event(
        cell44_citation_pending_session,
        "CITATION_MAPPING_REGISTERED",
        artifact=
            cell44_citation_mapping_record,
    )
)


cell44_citation_registered_valid = all([
    cell44_current_session.get(
        "current_state"
    )
    ==
    "CITATION_HUMAN_REVIEW_PENDING",

    isinstance(
        cell44_current_session.get(
            "artifacts",
            {}
        ).get(
            "citation_mapping"
        ),
        dict,
    ),

    cell44_current_session[
        "artifacts"
    ][
        "citation_mapping"
    ].get(
        "citation_mapping_id"
    )
    ==
    cell44_citation_mapping_id,

    cell44_current_session.get(
        "queue_advance_performed"
    )
    is False,
])


# ============================================================
# 13) Update Human Review workspace
# ============================================================

cell44_active_human_review_case = copy.deepcopy(
    cell43_active_human_review_case
)


cell44_active_human_review_case[
    "workflow_state"
] = "CITATION_HUMAN_REVIEW_PENDING"


cell44_active_human_review_case[
    "expected_answer_review"
][
    "human_decision"
] = "ACCEPT"


cell44_active_human_review_case[
    "expected_answer_review"
][
    "human_review"
] = copy.deepcopy(
    cell44_answer_human_review
)


cell44_active_human_review_case[
    "citation_review"
] = {
    "citation_mapping_id":
        cell44_citation_mapping_id,

    "citation_entries":
        copy.deepcopy(
            cell44_citation_entries
        ),

    "machine_validation_status":
        cell44_citation_mapping_record[
            "machine_validation_status"
        ],

    "human_decision":
        "PENDING",
}


cell44_active_human_review_case[
    "final_status"
] = (
    "QUESTION_AND_ANSWER_ACCEPTED_"
    "CITATION_PENDING_REVIEW"
)


# ============================================================
# 14) Persist Registry
# ============================================================

cell44_persistent_registry_payload = copy.deepcopy(
    cell43_persisted_registry
)


cell44_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell44_active_human_review_case
)


cell44_persistent_registry_payload[
    "remaining_pilot_count"
] = 16


cell44_persistent_registry_payload[
    "continuation_state"
] = {
    "next_pilot_id":
        CELL44_PILOT_ID,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell42_remaining_pilot_ids
        ),

    "remaining_count":
        16,

    "continuation_state_sha256":
        cell42_continuation_state_sha256,

    "active_pilot_status":
        "CITATION_HUMAN_REVIEW_PENDING",

    "active_target_evidence_id":
        CELL44_TARGET_EVIDENCE_ID,
}


cell44_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell44_persistent_registry_payload
    )
)


# ============================================================
# 15) Read-back validation
# ============================================================

cell44_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell44_persisted_active_case = (
    cell44_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell44_persistence_valid = all([
    cell44_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL44_PILOT_ID,

    cell44_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell44_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "citation_mapping_id"
    )
    ==
    cell44_citation_mapping_id,

    cell44_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "PENDING",

    cell44_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    16,
])


# ============================================================
# 16) Safety
# ============================================================

cell44_queue_still_16 = (
    len(
        cell42_remaining_pilot_ids
    )
    ==
    16
)


cell44_current_pilot_still_first = (
    cell42_remaining_pilot_ids[
        0
    ]
    ==
    CELL44_PILOT_ID
)


cell44_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell44_full_572_blocked = (
    cell44_citation_mapping_record[
        "full_572_generation"
    ]
    ==
    CELL44_FULL_572_BLOCK_STATUS
)


# ============================================================
# 17) Final Gate
# ============================================================

cell44_gate = all([
    len(
        cell44_failures
    )
    ==
    0,

    cell44_answer_accept_valid,

    cell44_target_parts_valid,

    cell44_no_locator_fabrication,

    cell44_all_entries_valid,

    cell44_citation_machine_pass,

    cell44_citation_registered_valid,

    cell44_persistence_valid,

    cell44_queue_still_16,

    cell44_current_pilot_still_first,

    cell44_base_seed_unchanged,

    cell44_full_572_blocked,
])


# ============================================================
# 18) Diagnostics
# ============================================================

print(
    "=== ANSWER ACCEPTED + CITATION DRAFT "
    "+ REGISTRY SYNC ==="
)


print(
    "\n=== ACCEPTED QUESTION ==="
)

print(
    CELL43_ACCEPTED_QUESTION
)


print(
    "\n=== ACCEPTED ANSWER ==="
)

print(
    CELL43_EXPECTED_ANSWER
)

print(
    "Human Answer Decision:",
    "ACCEPT",
)


print(
    "\n=== CITATION DRAFT ==="
)

print(
    "Citation Mapping ID:",
    cell44_citation_mapping_id,
)

print(
    "Target Evidence:",
    CELL44_TARGET_EVIDENCE_ID,
)

print(
    "Citation entry count:",
    len(
        cell44_citation_entries
    ),
)


for entry in cell44_citation_entries:

    print(
        "\n--- CITATION ENTRY",
        entry.get(
            "citation_index"
        ),
        "---"
    )

    print(
        "Document:",
        entry.get(
            "document_id"
        ),
    )

    print(
        "Role:",
        entry.get(
            "role"
        ),
    )

    print(
        "Chars:",
        (
            entry.get(
                "char_start"
            ),
            entry.get(
                "char_end"
            ),
        ),
    )

    print(
        "Paragraph ID:",
        entry.get(
            "locator",
            {},
        ).get(
            "paragraph_id"
        ),
    )

    print(
        "Paragraph Number:",
        entry.get(
            "locator",
            {},
        ).get(
            "paragraph_number"
        ),
    )

    print(
        "Canonical URL:",
        entry.get(
            "locator",
            {},
        ).get(
            "canonical_url"
        ),
    )


print(
    "\n=== CITATION MACHINE VALIDATION ==="
)

for name, status in (
    cell44_citation_machine_validation.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "Machine PASS:",
    cell44_citation_machine_pass,
)

print(
    "No locator fabrication:",
    cell44_no_locator_fabrication,
)


print(
    "\n=== WORKFLOW ==="
)

print(
    "Current State:",
    cell44_current_session.get(
        "current_state"
    ),
)

print(
    "Question Decision:",
    "ACCEPT",
)

print(
    "Answer Decision:",
    "ACCEPT",
)

print(
    "Citation Decision:",
    "PENDING",
)


print(
    "\n=== PERSISTENCE ==="
)

print(
    "Registry:",
    cell21_registry_path,
)

print(
    "Persisted Answer Decision:",
    cell44_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    ),
)

print(
    "Persisted Citation Decision:",
    cell44_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "human_decision"
    ),
)

print(
    "Persistence valid:",
    cell44_persistence_valid,
)


print(
    "\n=== QUEUE / SAFETY ==="
)

print(
    "Queue still 16:",
    cell44_queue_still_16,
)

print(
    "Base Seed unchanged:",
    cell44_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell44_full_572_blocked,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell44_failures
)


print(
    "\nCODE CELL 44 GATE:",
    (
        "PASS"
        if cell44_gate
        else "FAIL"
    ),
)


assert cell44_gate, (
    "Expected Answer ACCEPT / Citation Draft / Registry Sync "
    "failed. Repair this SAME Code Cell 44 before "
    "Citation Human Review."
)

=== ANSWER ACCEPTED + CITATION DRAFT + REGISTRY SYNC ===

=== ACCEPTED QUESTION ===
در این پرسشنامه، برای سنجش رضایت شغلی چه جنبه‌هایی از شغل و محیط کار بررسی می‌شوند؟

=== ACCEPTED ANSWER ===
این پرسشنامه چند جنبه اصلی را بررسی می‌کند: ماهیت و جذابیت خودِ کار، رفتار و عملکرد سرپرست، حقوق و مزایا، ویژگی‌ها و رفتار همکاران، فرصت‌های رشد و ترفیع، و شرایط فیزیکی و امکانات محیط کار مانند فضای کاری، نور، تهویه، تجهیزات، ساختمان و کیفیت غذا.
Human Answer Decision: ACCEPT

=== CITATION DRAFT ===
Citation Mapping ID: citation-mapping:ad406a53c02f94fd6be6a58caa85b370
Target Evidence: target-evidence:638449df7471aaec952c81950f39548b
Citation entry count: 42

--- CITATION ENTRY 0 ---
Document: book:adult-financial-literacy-course:fa-source:p000623
Role: TABLE_RUN_MEMBER
Chars: (0, 4)
Paragraph ID: book:adult-financial-literacy-course:fa-source:p000623
Paragraph Number: 623
Canonical URL: None

--- CITATION ENTRY 1 ---
Document: book:adult-financial-literacy-course:fa-source:p000624
Role: TABLE_RU

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 45 — Citation ACCEPT + Pilot Completion + Queue Advance

Citation این Pilot توسط Human Reviewer پذیرفته شد.

در این مرحله:

- Citation با تصمیم `ACCEPT` ثبت می‌شود؛
- Pilot کامل می‌شود؛
- پرونده Human Review به Completed Cases منتقل می‌شود؛
- Author Final Review همچنان `NOT_STARTED` باقی می‌ماند؛
- Queue از 16 به 15 کاهش پیدا می‌کند؛
- Pilot بعدی و Source Evidence آن آماده می‌شود؛
- Registry دائمی Sync می‌شود؛
- Full 572 Generation همچنان Blocked باقی می‌ماند.

</div>

In [45]:
# Code Cell 45

cell45_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell44_gate") is not True:
    cell45_failures.append(
        "CELL44_NOT_PASS"
    )


if (
    globals().get(
        "cell44_current_session",
        {},
    ).get(
        "current_state"
    )
    !=
    "CITATION_HUMAN_REVIEW_PENDING"
):
    cell45_failures.append(
        "CITATION_NOT_AT_HUMAN_REVIEW_GATE"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell45_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Current completed Pilot identity
# ============================================================

CELL45_COMPLETED_PILOT_ID = (
    CELL44_PILOT_ID
)

CELL45_COMPLETED_TARGET_ID = (
    CELL44_TARGET_EVIDENCE_ID
)

CELL45_FULL_572_BLOCK_STATUS = (
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


# ============================================================
# 3) Human Citation ACCEPT
# ============================================================

CELL45_HUMAN_CITATION_NOTE = (
    "Human Reviewer accepted the Citation Mapping after "
    "reviewing the complete large-table Target Evidence "
    "coverage and validated source locators."
)


cell45_citation_human_review = {
    "review_record_version":
        "human_citation_review_record_v1",

    "pilot_case_id":
        CELL45_COMPLETED_PILOT_ID,

    "citation_mapping_id":
        cell44_citation_mapping_id,

    "target_evidence_id":
        CELL45_COMPLETED_TARGET_ID,

    "human_decision":
        "ACCEPT",

    "human_note":
        CELL45_HUMAN_CITATION_NOTE,

    "r_marker_interpreted":
        False,

    "external_locator_lookup_used":
        False,
}


# ============================================================
# 4) Apply Citation ACCEPT
# ============================================================

cell45_pre_advance_session = (
    cell13_apply_event(
        cell44_current_session,
        "CITATION_ACCEPTED",
        human_note=
            CELL45_HUMAN_CITATION_NOTE,
    )
)


cell45_citation_accept_valid = all([
    cell45_pre_advance_session.get(
        "current_state"
    )
    ==
    "PILOT_COMPLETE_PENDING_QUEUE_ADVANCE",

    cell45_pre_advance_session.get(
        "queue_advance_permitted"
    )
    is True,

    cell45_pre_advance_session.get(
        "queue_advance_performed"
    )
    is False,

    cell45_pre_advance_session.get(
        "human_decisions",
        [],
    )[
        -1
    ].get(
        "event"
    )
    ==
    "CITATION_ACCEPTED",
])


# ============================================================
# 5) Finalize Human Review Case
# ============================================================

cell45_completed_human_review_case = copy.deepcopy(
    cell44_active_human_review_case
)


cell45_completed_human_review_case[
    "workflow_state"
] = "PILOT_COMPLETE"


cell45_completed_human_review_case[
    "citation_review"
][
    "human_decision"
] = "ACCEPT"


cell45_completed_human_review_case[
    "citation_review"
][
    "human_review"
] = copy.deepcopy(
    cell45_citation_human_review
)


cell45_completed_human_review_case[
    "author_final_review"
] = {
    "status":
        "NOT_STARTED",

    "decision":
        None,

    "notes":
        None,
}


cell45_completed_human_review_case[
    "final_status"
] = (
    "HUMAN_REVIEW_COMPLETE_"
    "AUTHOR_FINAL_REVIEW_PENDING"
)


cell45_completed_human_review_case_sha256 = (
    cell10_stable_sha(
        cell45_completed_human_review_case
    )
)


# ============================================================
# 6) Queue preconditions
# ============================================================

cell45_parent_remaining_ids = copy.deepcopy(
    cell42_continuation_state[
        "remaining_pilot_ids"
    ]
)


cell45_parent_remaining_queue = copy.deepcopy(
    cell42_continuation_state[
        "remaining_pilot_queue"
    ]
)


cell45_queue_preconditions_valid = all([
    len(
        cell45_parent_remaining_ids
    )
    ==
    16,

    len(
        cell45_parent_remaining_queue
    )
    ==
    16,

    cell45_parent_remaining_ids[
        0
    ]
    ==
    CELL45_COMPLETED_PILOT_ID,

    cell45_citation_accept_valid,
])


# ============================================================
# 7) Advance Queue
# ============================================================

if cell45_queue_preconditions_valid:

    cell45_remaining_pilot_ids = copy.deepcopy(
        cell45_parent_remaining_ids[
            1:
        ]
    )

    cell45_remaining_pilot_queue = copy.deepcopy(
        cell45_parent_remaining_queue[
            1:
        ]
    )

else:

    cell45_remaining_pilot_ids = []
    cell45_remaining_pilot_queue = []


cell45_next_pilot_id = (
    cell45_remaining_pilot_ids[
        0
    ]

    if cell45_remaining_pilot_ids

    else None
)


cell45_next_pilot_queue_entry = (
    copy.deepcopy(
        cell45_remaining_pilot_queue[
            0
        ]
    )

    if cell45_remaining_pilot_queue

    else None
)


cell45_queue_advanced_valid = all([
    len(
        cell45_remaining_pilot_ids
    )
    ==
    15,

    len(
        cell45_remaining_pilot_queue
    )
    ==
    15,

    CELL45_COMPLETED_PILOT_ID
    not in
    cell45_remaining_pilot_ids,

    isinstance(
        cell45_next_pilot_id,
        str,
    ),

    cell45_next_pilot_id
    !=
    CELL45_COMPLETED_PILOT_ID,
])


# ============================================================
# 8) Completed Engine session
# ============================================================

cell45_completed_session = copy.deepcopy(
    cell45_pre_advance_session
)


cell45_completed_session[
    "current_state"
] = "PILOT_COMPLETE"


cell45_completed_session[
    "queue_advance_permitted"
] = False


cell45_completed_session[
    "queue_advance_performed"
] = True


cell45_completed_session[
    "completion_status"
] = (
    "HUMAN_REVIEW_COMPLETE_"
    "AUTHOR_FINAL_REVIEW_PENDING"
)


# ============================================================
# 9) New authoritative Continuation State
# ============================================================

cell45_continuation_state = {
    "state_version":
        "compact_pilot_continuation_state_v7",

    "authoritative_forward_state":
        True,

    "parent_state_sha256":
        cell42_continuation_state_sha256,

    "just_completed_pilot_id":
        CELL45_COMPLETED_PILOT_ID,

    "just_completed_target_evidence_id":
        CELL45_COMPLETED_TARGET_ID,

    "just_completed_case_sha256":
        cell45_completed_human_review_case_sha256,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell45_remaining_pilot_ids
        ),

    "remaining_pilot_queue":
        copy.deepcopy(
            cell45_remaining_pilot_queue
        ),

    "remaining_count":
        15,

    "next_pilot_id":
        cell45_next_pilot_id,

    "next_pilot_queue_entry":
        copy.deepcopy(
            cell45_next_pilot_queue_entry
        ),

    "full_572_generation":
        CELL45_FULL_572_BLOCK_STATUS,
}


cell45_continuation_state_sha256 = (
    cell10_stable_sha(
        cell45_continuation_state
    )
)


# ============================================================
# 10) Resolve next Pilot packet
# ============================================================

cell45_pilot_by_id = (
    cell13_seed[
        "final_runtime"
    ][
        "question_drafting_pilot_by_id"
    ]
)


cell45_next_pilot_packet = copy.deepcopy(
    cell45_pilot_by_id.get(
        cell45_next_pilot_id
    )
)


cell45_next_packet_valid = isinstance(
    cell45_next_pilot_packet,
    dict,
)


# ============================================================
# 11) Resolve next identity chain
# ============================================================

cell45_next_golden_case_id = None
cell45_next_task_draft_id = None
cell45_next_target_evidence_id = None


if cell45_next_packet_valid:

    try:

        cell45_next_golden_case_id = (
            cell13_exact_single_id(
                cell45_next_pilot_packet,
                "golden_case",
            )
        )


        cell45_next_task_draft_id = (
            cell13_exact_single_id(
                cell45_next_pilot_packet,
                "task_draft",
            )
        )


        cell45_next_target_evidence_id = (
            cell13_exact_single_id(
                cell45_next_pilot_packet,
                "target_evidence",
            )
        )


    except Exception as exc:

        cell45_failures.append({
            "reason":
                "NEXT_ID_CHAIN_FAILED",

            "error":
                repr(
                    exc
                ),
        })


cell45_next_identity_valid = all([
    isinstance(
        cell45_next_golden_case_id,
        str,
    ),

    isinstance(
        cell45_next_task_draft_id,
        str,
    ),

    isinstance(
        cell45_next_target_evidence_id,
        str,
    ),
])


# ============================================================
# 12) Resolve next support records
# ============================================================

cell45_next_golden_case = None
cell45_next_task_plan = None
cell45_next_target_evidence = None


if cell45_next_identity_valid:

    try:

        cell45_next_golden_case = copy.deepcopy(
            cell13_active_support[
                "golden_case_records_by_id"
            ][
                cell45_next_golden_case_id
            ]
        )


        cell45_next_task_plan = copy.deepcopy(
            cell13_find_unique_mapping_value(
                cell13_active_support.get(
                    "task_draft_plans",
                    {},
                ),
                "task_draft",
                cell45_next_task_draft_id,
            )[
                "value"
            ]
        )


        cell45_next_target_evidence = copy.deepcopy(
            cell13_find_unique_list_record(
                cell13_active_support.get(
                    "target_evidence_records",
                    [],
                ),
                "target_evidence_id",
                cell45_next_target_evidence_id,
            )
        )


    except Exception as exc:

        cell45_failures.append({
            "reason":
                "NEXT_SUPPORT_RESOLUTION_FAILED",

            "error":
                repr(
                    exc
                ),
        })


# ============================================================
# 13) Classification helper
# ============================================================

def cell45_scalar(
    mapping,
    names,
):

    match = cell14_find_first_scalar(
        mapping,
        names,
    )

    if isinstance(
        match,
        dict,
    ):

        return match.get(
            "value"
        )

    return None


cell45_next_source_type = cell45_scalar(
    cell45_next_pilot_packet,
    [
        "source_type",
        "source_kind",
        "source",
    ],
)


cell45_next_stratum = cell45_scalar(
    cell45_next_pilot_packet,
    [
        "stratum",
        "stratum_name",
        "stratum_id",
    ],
)


cell45_next_task_family = cell45_scalar(
    cell45_next_pilot_packet,
    [
        "task_family",
    ],
)


cell45_next_question_archetype = cell45_scalar(
    cell45_next_pilot_packet,
    [
        "question_archetype",
    ],
)


# ============================================================
# 14) Extract next Source Evidence
# ============================================================

cell45_raw_target_parts = []


if cell45_next_packet_valid:

    packet_parts = (
        cell45_next_pilot_packet.get(
            "target_text_parts"
        )
    )

    if isinstance(
        packet_parts,
        list,
    ):

        cell45_raw_target_parts = copy.deepcopy(
            packet_parts
        )


if (
    not cell45_raw_target_parts
    and
    isinstance(
        cell45_next_target_evidence,
        dict,
    )
):

    evidence_parts = (
        cell45_next_target_evidence.get(
            "target_parts"
        )
    )

    if isinstance(
        evidence_parts,
        list,
    ):

        cell45_raw_target_parts = copy.deepcopy(
            evidence_parts
        )


cell45_next_target_text_parts = []


for index, part in enumerate(
    cell45_raw_target_parts
):

    if not isinstance(
        part,
        dict,
    ):
        continue


    text_value = None


    for field_name in (
        "text",
        "target_text",
        "content",
        "evidence_text",
        "text_content",
    ):

        candidate = part.get(
            field_name
        )

        if (
            isinstance(
                candidate,
                str,
            )
            and
            candidate.strip()
        ):

            text_value = candidate.strip()
            break


    if text_value is None:
        continue


    cell45_next_target_text_parts.append({
        "part_index":
            index,

        "document_id":
            part.get(
                "document_id"
            ),

        "role":
            part.get(
                "role"
            ),

        "char_start":
            part.get(
                "char_start"
            ),

        "char_end":
            part.get(
                "char_end"
            ),

        "text":
            text_value,

        "text_sha256":
            hashlib.sha256(
                text_value.encode(
                    "utf-8"
                )
            ).hexdigest(),
    })


cell45_next_combined_target_text = (
    "\n\n".join(
        row[
            "text"
        ]

        for row
        in cell45_next_target_text_parts
    )
)


cell45_next_target_text_available = bool(
    cell45_next_combined_target_text.strip()
)


# ============================================================
# 15) Build next Pilot workspace
# ============================================================

cell45_next_pilot_workspace = {
    "workspace_version":
        "compact_human_review_workspace_v1",

    "status":
        "READY_FOR_QUESTION_DRAFT",

    "pilot_case_id":
        cell45_next_pilot_id,

    "golden_case_id":
        cell45_next_golden_case_id,

    "task_draft_id":
        cell45_next_task_draft_id,

    "target_evidence_id":
        cell45_next_target_evidence_id,

    "source_type":
        cell45_next_source_type,

    "stratum":
        cell45_next_stratum,

    "task_family":
        cell45_next_task_family,

    "question_archetype":
        cell45_next_question_archetype,

    "source_evidence":
        copy.deepcopy(
            cell45_next_target_text_parts
        ),

    "combined_target_text":
        cell45_next_combined_target_text,

    "golden_case":
        copy.deepcopy(
            cell45_next_golden_case
        ),

    "task_plan":
        copy.deepcopy(
            cell45_next_task_plan
        ),

    "target_evidence":
        copy.deepcopy(
            cell45_next_target_evidence
        ),

    "model_drafts":
        [],

    "human_reviews":
        [],

    "expected_answer_review":
        None,

    "citation_review":
        None,

    "author_final_review": {
        "status":
            "NOT_STARTED",

        "decision":
            None,

        "notes":
            None,
    },

    "workflow_state":
        "QUESTION_DRAFT_PENDING",

    "full_572_generation":
        CELL45_FULL_572_BLOCK_STATUS,
}


cell45_next_workspace_ready = all([
    cell45_next_packet_valid,

    cell45_next_identity_valid,

    isinstance(
        cell45_next_golden_case,
        dict,
    ),

    isinstance(
        cell45_next_task_plan,
        dict,
    ),

    isinstance(
        cell45_next_target_evidence,
        dict,
    ),

    cell45_next_target_text_available,
])


# ============================================================
# 16) Update Completed Human Review cases
# ============================================================

cell45_completed_cases = copy.deepcopy(
    cell44_persisted_registry.get(
        "completed_human_review_cases",
        [],
    )
)


cell45_completed_cases.append(
    copy.deepcopy(
        cell45_completed_human_review_case
    )
)


# ============================================================
# 17) Build persistent Registry payload
# ============================================================

cell45_persistent_registry_payload = copy.deepcopy(
    cell44_persisted_registry
)


cell45_persistent_registry_payload[
    "completed_human_review_cases"
] = copy.deepcopy(
    cell45_completed_cases
)


cell45_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell45_next_pilot_workspace
)


cell45_persistent_registry_payload[
    "remaining_pilot_count"
] = 15


cell45_persistent_registry_payload[
    "continuation_state"
] = {
    "next_pilot_id":
        cell45_next_pilot_id,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell45_remaining_pilot_ids
        ),

    "remaining_count":
        15,

    "continuation_state_sha256":
        cell45_continuation_state_sha256,
}


# ============================================================
# 18) Author Review state
# ============================================================

cell45_author_review = copy.deepcopy(
    cell45_persistent_registry_payload.get(
        "author_review",
        {},
    )
)


if not isinstance(
    cell45_author_review,
    dict,
):

    cell45_author_review = {}


cell45_author_review[
    "status"
] = "NOT_STARTED"


cell45_author_review[
    "reviewer"
] = "AUTHOR"


cell45_author_review[
    "cases_ready_for_review"
] = len(
    cell45_completed_cases
)


cell45_persistent_registry_payload[
    "author_review"
] = copy.deepcopy(
    cell45_author_review
)


# ============================================================
# 19) Persist
# ============================================================

cell45_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell45_persistent_registry_payload
    )
)


# ============================================================
# 20) Read-back validation
# ============================================================

cell45_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell45_persisted_active_case = (
    cell45_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell45_persistence_valid = all([
    len(
        cell45_persisted_registry.get(
            "completed_human_review_cases",
            [],
        )
    )
    ==
    len(
        cell45_completed_cases
    ),

    cell45_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    15,

    cell45_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    cell45_next_pilot_id,

    cell45_persisted_registry.get(
        "continuation_state",
        {},
    ).get(
        "next_pilot_id"
    )
    ==
    cell45_next_pilot_id,

    cell45_persisted_registry.get(
        "author_review",
        {},
    ).get(
        "cases_ready_for_review"
    )
    ==
    len(
        cell45_completed_cases
    ),
])


# ============================================================
# 21) Safety
# ============================================================

cell45_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell45_full_572_blocked = (
    cell45_continuation_state[
        "full_572_generation"
    ]
    ==
    CELL45_FULL_572_BLOCK_STATUS
)


# ============================================================
# 22) Final Gate
# ============================================================

cell45_gate = all([
    len(
        cell45_failures
    )
    ==
    0,

    cell45_citation_accept_valid,

    cell45_queue_preconditions_valid,

    cell45_queue_advanced_valid,

    cell45_next_workspace_ready,

    cell45_persistence_valid,

    cell45_base_seed_unchanged,

    cell45_full_572_blocked,
])


# ============================================================
# 23) Diagnostics
# ============================================================

print(
    "=== PILOT COMPLETED + QUEUE ADVANCE "
    "+ NEXT PILOT ==="
)


print(
    "\n=== COMPLETED PILOT ==="
)

print(
    "Pilot:",
    CELL45_COMPLETED_PILOT_ID,
)

print(
    "Question:",
    CELL43_ACCEPTED_QUESTION,
)

print(
    "Answer:",
    CELL43_EXPECTED_ANSWER,
)

print(
    "Citation:",
    "ACCEPT",
)

print(
    "Human Review status:",
    cell45_completed_human_review_case[
        "final_status"
    ],
)

print(
    "Author Final Review:",
    "NOT_STARTED",
)


print(
    "\n=== QUEUE ==="
)

print(
    "Before:",
    len(
        cell45_parent_remaining_ids
    ),
)

print(
    "After:",
    len(
        cell45_remaining_pilot_ids
    ),
)

print(
    "Completed Pilot removed:",
    (
        CELL45_COMPLETED_PILOT_ID
        not in
        cell45_remaining_pilot_ids
    ),
)

print(
    "Next Pilot:",
    cell45_next_pilot_id,
)


print(
    "\n=== PERSISTENT REGISTRY ==="
)

print(
    "Path:",
    cell21_registry_path,
)

print(
    "Completed Human Review cases:",
    len(
        cell45_persisted_registry.get(
            "completed_human_review_cases",
            [],
        )
    ),
)

print(
    "Cases ready for Author Review:",
    cell45_persisted_registry.get(
        "author_review",
        {},
    ).get(
        "cases_ready_for_review"
    ),
)

print(
    "Remaining Pilots:",
    cell45_persisted_registry.get(
        "remaining_pilot_count"
    ),
)

print(
    "Active Pilot:",
    cell45_persisted_active_case.get(
        "pilot_case_id"
    ),
)

print(
    "Persistence valid:",
    cell45_persistence_valid,
)


print(
    "\n=== NEXT PILOT IDENTITY ==="
)

print(
    "Pilot:",
    cell45_next_pilot_id,
)

print(
    "Golden Case:",
    cell45_next_golden_case_id,
)

print(
    "Task Draft:",
    cell45_next_task_draft_id,
)

print(
    "Target Evidence:",
    cell45_next_target_evidence_id,
)

print(
    "Source type:",
    cell45_next_source_type,
)

print(
    "Stratum:",
    cell45_next_stratum,
)

print(
    "Task family:",
    cell45_next_task_family,
)

print(
    "Question archetype:",
    cell45_next_question_archetype,
)


print(
    "\n=== NEXT PILOT SOURCE EVIDENCE ==="
)

print(
    cell45_next_combined_target_text
)


print(
    "\n=== SAFETY ==="
)

print(
    "Base Seed unchanged:",
    cell45_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell45_full_572_blocked,
)

print(
    "Author Notebook rewrite:",
    False,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell45_failures
)


print(
    "\nCODE CELL 45 GATE:",
    (
        "PASS"
        if cell45_gate
        else "FAIL"
    ),
)


assert cell45_gate, (
    "Citation ACCEPT / Pilot completion / Queue advance / "
    "Registry sync / next Pilot preparation failed. "
    "Repair this SAME Code Cell 45 before continuing."
)

=== PILOT COMPLETED + QUEUE ADVANCE + NEXT PILOT ===

=== COMPLETED PILOT ===
Pilot: question-pilot:7334395fa6456f24ed250e2b354cc184
Question: در این پرسشنامه، برای سنجش رضایت شغلی چه جنبه‌هایی از شغل و محیط کار بررسی می‌شوند؟
Answer: این پرسشنامه چند جنبه اصلی را بررسی می‌کند: ماهیت و جذابیت خودِ کار، رفتار و عملکرد سرپرست، حقوق و مزایا، ویژگی‌ها و رفتار همکاران، فرصت‌های رشد و ترفیع، و شرایط فیزیکی و امکانات محیط کار مانند فضای کاری، نور، تهویه، تجهیزات، ساختمان و کیفیت غذا.
Citation: ACCEPT
Human Review status: HUMAN_REVIEW_COMPLETE_AUTHOR_FINAL_REVIEW_PENDING
Author Final Review: NOT_STARTED

=== QUEUE ===
Before: 16
After: 15
Completed Pilot removed: True
Next Pilot: question-pilot:d7378ba1655fe80c37029153e753f80b

=== PERSISTENT REGISTRY ===
Path: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/data/phase2/human_review/human_review_registry.json
Completed Human Review cases: 7
Cases ready for Author Review: 7
Remaining Pilots: 15
Active Pilot: question-p

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 46 — Question ACCEPT + Expected Answer Draft + Registry Sync

Human Reviewer سؤال این Pilot را پذیرفت:

**محصول فقط شامل کالا می‌شود یا می‌توان یک خدمت را هم محصول دانست؟**

در این مرحله:

- Question با تصمیم Human `ACCEPT` ثبت می‌شود؛
- Expected Answer فقط از Source Evidence موجود ساخته می‌شود؛
- تفاوت کاربرد واژه «محصول» در ادبیات مدیریت و کاربرد رایج فارسی حفظ می‌شود؛
- واژه Supplemental یعنی `product` به‌عنوان بخشی از Evidence حفظ می‌شود، اما چیزی فراتر از متن درباره آن استنباط نمی‌شود؛
- Citation هنوز ساخته نمی‌شود؛
- Registry دائمی Sync می‌شود؛
- Queue همچنان 15 باقی می‌ماند؛
- Full 572 Generation همچنان Blocked است.

</div>

In [46]:
# Code Cell 46

cell46_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell45_gate") is not True:
    cell46_failures.append(
        "CELL45_NOT_PASS"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell46_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Current Pilot identity
# ============================================================

CELL46_PILOT_ID = (
    cell45_next_pilot_id
)

CELL46_GOLDEN_CASE_ID = (
    cell45_next_golden_case_id
)

CELL46_TASK_DRAFT_ID = (
    cell45_next_task_draft_id
)

CELL46_TARGET_EVIDENCE_ID = (
    cell45_next_target_evidence_id
)

CELL46_FULL_572_BLOCK_STATUS = (
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


CELL46_ACCEPTED_QUESTION = (
    "محصول فقط شامل کالا می‌شود یا می‌توان "
    "یک خدمت را هم محصول دانست؟"
)


CELL46_EXPECTED_ANSWER = (
    "در ادبیات مدیریت، محصول فقط به کالا محدود نمی‌شود "
    "و خدمات را هم شامل می‌شود؛ یعنی هم یک کالا و هم "
    "یک خدمت می‌توانند محصول محسوب شوند. البته در کاربرد "
    "رایج فارسی، واژه «محصول» بیشتر برای کالاها و چیزهای "
    "ملموس به کار می‌رود."
)


# ============================================================
# 3) Source Evidence
# ============================================================

cell46_source_text = (
    cell45_next_combined_target_text
)


def cell46_fa_surface(
    text,
):

    value = cell15_normalize_text(
        text
    )

    value = (
        value
        .replace(
            "ي",
            "ی",
        )
        .replace(
            "ك",
            "ک",
        )
        .replace(
            "\u200c",
            " ",
        )
    )

    value = re.sub(
        r"\s+",
        " ",
        value,
    )

    return value.strip()


cell46_source_surface = (
    cell46_fa_surface(
        cell46_source_text
    )
)


# ============================================================
# 4) Evidence grounding
# ============================================================

cell46_evidence_grounding = {
    "management_context":
        (
            "در ادبیات مدیریت"
            in
            cell46_source_surface
        ),

    "goods_present":
        (
            "کالا"
            in
            cell46_source_surface
        ),

    "services_present":
        (
            "خدمت"
            in
            cell46_source_surface
            or
            "خدمات"
            in
            cell46_source_surface
        ),

    "both_are_products":
        (
            "هم یک کالا و هم یک خدمت محصول محسوب می شود"
            in
            cell46_source_surface
        ),

    "persian_usage_difference":
        (
            "در ادبیات فارسی"
            in
            cell46_source_surface
        ),

    "tangible_goods_usage":
        (
            "ملموس"
            in
            cell46_source_surface
        ),

    "supplemental_product_term":
        (
            "product"
            in
            cell46_source_text.lower()
        ),
}


cell46_evidence_grounding_pass = all(
    cell46_evidence_grounding.values()
)


# ============================================================
# 5) Start Engine session
# ============================================================

cell46_question_pending_session = (
    cell13_build_session(
        cell45_next_pilot_packet
    )
)


cell46_session_start_valid = (
    cell46_question_pending_session.get(
        "current_state"
    )
    ==
    "QUESTION_DRAFT_PENDING"
)


# ============================================================
# 6) Question Draft ID
# ============================================================

cell46_question_identity_payload = {
    "identity_version":
        "question_draft_identity_v1",

    "pilot_case_id":
        CELL46_PILOT_ID,

    "target_evidence_id":
        CELL46_TARGET_EVIDENCE_ID,

    "question_text":
        CELL46_ACCEPTED_QUESTION,

    "revision_number":
        1,
}


cell46_question_draft_id = (
    "question-draft:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell46_question_identity_payload
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


# ============================================================
# 7) Question Machine Validation
# ============================================================

cell46_question_surface = (
    cell46_fa_surface(
        CELL46_ACCEPTED_QUESTION
    )
)


cell46_question_machine_validation = {
    "product_present":
        (
            "محصول"
            in
            cell46_question_surface
        ),

    "goods_present":
        (
            "کالا"
            in
            cell46_question_surface
        ),

    "service_present":
        (
            "خدمت"
            in
            cell46_question_surface
        ),

    "source_grounded":
        cell46_evidence_grounding_pass,

    "external_knowledge_used":
        False,
}


cell46_question_machine_pass = all([
    cell46_question_machine_validation[
        "product_present"
    ],

    cell46_question_machine_validation[
        "goods_present"
    ],

    cell46_question_machine_validation[
        "service_present"
    ],

    cell46_question_machine_validation[
        "source_grounded"
    ],
])


# ============================================================
# 8) Question artifact
# ============================================================

cell46_question_record = {
    "record_version":
        "question_draft_record_v1",

    "question_draft_id":
        cell46_question_draft_id,

    "pilot_case_id":
        CELL46_PILOT_ID,

    "golden_case_id":
        CELL46_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL46_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL46_TARGET_EVIDENCE_ID,

    "question_text":
        CELL46_ACCEPTED_QUESTION,

    "revision_number":
        1,

    "machine_validation_status":
        (
            "PASS_PENDING_HUMAN_ADJUDICATION"
            if cell46_question_machine_pass
            else
            "FAIL_REQUIRES_REVISION"
        ),

    "human_adjudication_status":
        "PENDING",

    "external_knowledge_used":
        False,
}


# ============================================================
# 9) Register + Human ACCEPT Question
# ============================================================

cell46_question_review_session = (
    cell13_apply_event(
        cell46_question_pending_session,
        "QUESTION_DRAFT_REGISTERED",
        artifact=
            cell46_question_record,
    )
)


CELL46_QUESTION_HUMAN_NOTE = (
    "Human Reviewer accepted the revised question because "
    "it directly asks whether a service can be considered a "
    "product, avoiding the ambiguity of the earlier formulation."
)


cell46_answer_pending_session = (
    cell13_apply_event(
        cell46_question_review_session,
        "QUESTION_ACCEPTED",
        human_note=
            CELL46_QUESTION_HUMAN_NOTE,
    )
)


cell46_question_accept_valid = all([
    cell46_question_review_session.get(
        "current_state"
    )
    ==
    "QUESTION_HUMAN_REVIEW_PENDING",

    cell46_answer_pending_session.get(
        "current_state"
    )
    ==
    "EXPECTED_ANSWER_DRAFT_PENDING",
])


# ============================================================
# 10) Expected Answer Draft ID
# ============================================================

cell46_answer_identity_payload = {
    "identity_version":
        "expected_answer_identity_v1",

    "pilot_case_id":
        CELL46_PILOT_ID,

    "question_draft_id":
        cell46_question_draft_id,

    "target_evidence_id":
        CELL46_TARGET_EVIDENCE_ID,

    "answer_text":
        CELL46_EXPECTED_ANSWER,

    "revision_number":
        1,
}


cell46_expected_answer_draft_id = (
    "expected-answer-draft:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell46_answer_identity_payload
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


# ============================================================
# 11) Expected Answer Machine Validation
# ============================================================

cell46_answer_surface = (
    cell46_fa_surface(
        CELL46_EXPECTED_ANSWER
    )
)


cell46_answer_grounding = {
    "management_definition":
        (
            "در ادبیات مدیریت"
            in
            cell46_answer_surface
        ),

    "not_goods_only":
        (
            "فقط به کالا محدود نمی شود"
            in
            cell46_answer_surface
        ),

    "services_included":
        (
            "خدمات را هم شامل می شود"
            in
            cell46_answer_surface
        ),

    "goods_and_services_are_products":
        (
            "هم یک کالا و هم یک خدمت"
            in
            cell46_answer_surface
            and
            "محصول محسوب شوند"
            in
            cell46_answer_surface
        ),

    "common_persian_usage":
        (
            "کاربرد رایج فارسی"
            in
            cell46_answer_surface
        ),

    "tangible_goods":
        (
            "ملموس"
            in
            cell46_answer_surface
        ),
}


cell46_answer_machine_pass = all([
    cell46_evidence_grounding_pass,

    all(
        cell46_answer_grounding.values()
    ),
])


# ============================================================
# 12) Expected Answer artifact
# ============================================================

cell46_expected_answer_record = {
    "record_version":
        "expected_answer_draft_record_v1",

    "expected_answer_draft_id":
        cell46_expected_answer_draft_id,

    "pilot_case_id":
        CELL46_PILOT_ID,

    "golden_case_id":
        CELL46_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL46_TASK_DRAFT_ID,

    "question_draft_id":
        cell46_question_draft_id,

    "target_evidence_id":
        CELL46_TARGET_EVIDENCE_ID,

    "expected_answer_text":
        CELL46_EXPECTED_ANSWER,

    "revision_number":
        1,

    "machine_validation_status":
        (
            "PASS_PENDING_HUMAN_ADJUDICATION"
            if cell46_answer_machine_pass
            else
            "FAIL_REQUIRES_REVISION"
        ),

    "human_adjudication_status":
        "PENDING",

    "supplemental_term_present_in_evidence":
        True,

    "supplemental_term_interpreted_beyond_source":
        False,

    "external_knowledge_used":
        False,
}


# ============================================================
# 13) Register Expected Answer
# ============================================================

cell46_current_session = (
    cell13_apply_event(
        cell46_answer_pending_session,
        "EXPECTED_ANSWER_REGISTERED",
        artifact=
            cell46_expected_answer_record,
    )
)


cell46_answer_registered_valid = (
    cell46_current_session.get(
        "current_state"
    )
    ==
    "EXPECTED_ANSWER_HUMAN_REVIEW_PENDING"
)


# ============================================================
# 14) Human Review workspace
# ============================================================

cell46_active_human_review_case = copy.deepcopy(
    cell45_next_pilot_workspace
)


cell46_active_human_review_case[
    "status"
] = "ACTIVE_HUMAN_REVIEW"


cell46_active_human_review_case[
    "workflow_state"
] = (
    "EXPECTED_ANSWER_HUMAN_REVIEW_PENDING"
)


cell46_active_human_review_case[
    "question_review"
] = {
    "question_draft_id":
        cell46_question_draft_id,

    "question_text":
        CELL46_ACCEPTED_QUESTION,

    "machine_validation_status":
        cell46_question_record[
            "machine_validation_status"
        ],

    "human_decision":
        "ACCEPT",

    "human_note":
        CELL46_QUESTION_HUMAN_NOTE,
}


cell46_active_human_review_case[
    "expected_answer_review"
] = {
    "expected_answer_draft_id":
        cell46_expected_answer_draft_id,

    "expected_answer_text":
        CELL46_EXPECTED_ANSWER,

    "machine_validation_status":
        cell46_expected_answer_record[
            "machine_validation_status"
        ],

    "human_decision":
        "PENDING",
}


cell46_active_human_review_case[
    "citation_review"
] = None


cell46_active_human_review_case[
    "author_final_review"
] = {
    "status":
        "NOT_STARTED",

    "decision":
        None,

    "notes":
        None,
}


# ============================================================
# 15) Persist Registry
# ============================================================

cell46_persistent_registry_payload = copy.deepcopy(
    cell45_persisted_registry
)


cell46_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell46_active_human_review_case
)


cell46_persistent_registry_payload[
    "remaining_pilot_count"
] = 15


cell46_persistent_registry_payload[
    "continuation_state"
] = {
    "next_pilot_id":
        CELL46_PILOT_ID,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell45_remaining_pilot_ids
        ),

    "remaining_count":
        15,

    "continuation_state_sha256":
        cell45_continuation_state_sha256,

    "active_pilot_status":
        "EXPECTED_ANSWER_HUMAN_REVIEW_PENDING",

    "active_target_evidence_id":
        CELL46_TARGET_EVIDENCE_ID,
}


cell46_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell46_persistent_registry_payload
    )
)


# ============================================================
# 16) Read-back validation
# ============================================================

cell46_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell46_persisted_active_case = (
    cell46_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell46_persistence_valid = all([
    cell46_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL46_PILOT_ID,

    cell46_persisted_active_case.get(
        "question_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell46_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "PENDING",

    cell46_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    15,
])


# ============================================================
# 17) Safety
# ============================================================

cell46_queue_still_15 = (
    len(
        cell45_remaining_pilot_ids
    )
    ==
    15
)


cell46_current_pilot_still_first = (
    cell45_remaining_pilot_ids[
        0
    ]
    ==
    CELL46_PILOT_ID
)


cell46_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell46_full_572_blocked = (
    CELL46_FULL_572_BLOCK_STATUS
    ==
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


# ============================================================
# 18) Final Gate
# ============================================================

cell46_gate = all([
    len(
        cell46_failures
    )
    ==
    0,

    cell46_evidence_grounding_pass,

    cell46_session_start_valid,

    cell46_question_machine_pass,

    cell46_question_accept_valid,

    cell46_answer_machine_pass,

    cell46_answer_registered_valid,

    cell46_persistence_valid,

    cell46_queue_still_15,

    cell46_current_pilot_still_first,

    cell46_base_seed_unchanged,

    cell46_full_572_blocked,
])


# ============================================================
# 19) Diagnostics
# ============================================================

print(
    "=== QUESTION ACCEPTED + EXPECTED ANSWER DRAFT "
    "+ REGISTRY SYNC ==="
)


print(
    "\n=== SOURCE EVIDENCE ==="
)

print(
    cell46_source_text
)


print(
    "\n=== ACCEPTED QUESTION ==="
)

print(
    CELL46_ACCEPTED_QUESTION
)

print(
    "Question Draft ID:",
    cell46_question_draft_id,
)

print(
    "Question Machine PASS:",
    cell46_question_machine_pass,
)


print(
    "\n=== EXPECTED ANSWER DRAFT ==="
)

print(
    CELL46_EXPECTED_ANSWER
)

print(
    "Expected Answer Draft ID:",
    cell46_expected_answer_draft_id,
)

print(
    "Machine PASS:",
    cell46_answer_machine_pass,
)


print(
    "\n=== ANSWER GROUNDING ==="
)

for name, status in (
    cell46_answer_grounding.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== SUPPLEMENTAL EVIDENCE SAFETY ==="
)

print(
    "Supplemental 'product' present:",
    cell46_evidence_grounding[
        "supplemental_product_term"
    ],
)

print(
    "Supplemental meaning inferred beyond source:",
    False,
)

print(
    "External knowledge used:",
    False,
)


print(
    "\n=== WORKFLOW ==="
)

print(
    "Current State:",
    cell46_current_session.get(
        "current_state"
    ),
)

print(
    "Question Decision:",
    "ACCEPT",
)

print(
    "Expected Answer Decision:",
    "PENDING",
)

print(
    "Citation:",
    None,
)


print(
    "\n=== PERSISTENCE ==="
)

print(
    "Registry:",
    cell21_registry_path,
)

print(
    "Persisted Active Pilot:",
    cell46_persisted_active_case.get(
        "pilot_case_id"
    ),
)

print(
    "Persisted Question Decision:",
    cell46_persisted_active_case.get(
        "question_review",
        {},
    ).get(
        "human_decision"
    ),
)

print(
    "Persisted Answer Decision:",
    cell46_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    ),
)

print(
    "Persistence valid:",
    cell46_persistence_valid,
)


print(
    "\n=== QUEUE / SAFETY ==="
)

print(
    "Queue still 15:",
    cell46_queue_still_15,
)

print(
    "Base Seed unchanged:",
    cell46_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell46_full_572_blocked,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell46_failures
)


print(
    "\nCODE CELL 46 GATE:",
    (
        "PASS"
        if cell46_gate
        else "FAIL"
    ),
)


assert cell46_gate, (
    "Question ACCEPT / Expected Answer Draft / Registry Sync "
    "failed. Repair this SAME Code Cell 46 before "
    "Expected Answer Human Review."
)

=== QUESTION ACCEPTED + EXPECTED ANSWER DRAFT + REGISTRY SYNC ===

=== SOURCE EVIDENCE ===
در ادبیات مدیریت، محصول شامل دو بخش کالاها و خدمات است. یعنی هم یک کالا و هم یک خدمت محصول محسوب می‌شود. اما در ادبیات فارسی این خیلی رایج نیست و وقتی از کلمۀ محصول استفاده می‌کنیم منظورمان بیشتر کالا است؛ یعنی چیزی که ملموس است.

product

=== ACCEPTED QUESTION ===
محصول فقط شامل کالا می‌شود یا می‌توان یک خدمت را هم محصول دانست؟
Question Draft ID: question-draft:b14e05d577c2ada5360c04c05f2a81bf
Question Machine PASS: True

=== EXPECTED ANSWER DRAFT ===
در ادبیات مدیریت، محصول فقط به کالا محدود نمی‌شود و خدمات را هم شامل می‌شود؛ یعنی هم یک کالا و هم یک خدمت می‌توانند محصول محسوب شوند. البته در کاربرد رایج فارسی، واژه «محصول» بیشتر برای کالاها و چیزهای ملموس به کار می‌رود.
Expected Answer Draft ID: expected-answer-draft:38a6107c073a85349f918d9aa139ecdf
Machine PASS: True

=== ANSWER GROUNDING ===
management_definition: True
not_goods_only: True
services_included: True
goods_and_services_are_product

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 47 — Expected Answer ACCEPT + Supplemental Citation Draft + Registry Sync

Expected Answer این Pilot توسط Human Reviewer پذیرفته شد.

در این مرحله:

- Expected Answer با تصمیم `ACCEPT` ثبت می‌شود؛
- Citation فقط از Target Evidence همین Pilot ساخته می‌شود؛
- Parent Evidence و Supplemental Evidence موجود در Target، بدون ساخت Locator جدید، حفظ می‌شوند؛
- واژه `product` فقط به‌عنوان Supplemental Evidence ثبت می‌شود و هیچ معنای اضافه‌ای از آن استنباط نمی‌شود؛
- Citation وارد Human Review می‌شود؛
- Registry دائمی Sync می‌شود؛
- Queue همچنان 15 باقی می‌ماند؛
- Full 572 Generation همچنان Blocked است.

</div>

In [47]:
# Code Cell 47

cell47_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell46_gate") is not True:
    cell47_failures.append(
        "CELL46_NOT_PASS"
    )


if (
    globals().get(
        "cell46_current_session",
        {},
    ).get(
        "current_state"
    )
    !=
    "EXPECTED_ANSWER_HUMAN_REVIEW_PENDING"
):
    cell47_failures.append(
        "EXPECTED_ANSWER_NOT_AT_HUMAN_REVIEW_GATE"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell47_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Current identities
# ============================================================

CELL47_PILOT_ID = (
    CELL46_PILOT_ID
)

CELL47_GOLDEN_CASE_ID = (
    CELL46_GOLDEN_CASE_ID
)

CELL47_TASK_DRAFT_ID = (
    CELL46_TASK_DRAFT_ID
)

CELL47_TARGET_EVIDENCE_ID = (
    CELL46_TARGET_EVIDENCE_ID
)

CELL47_QUESTION_DRAFT_ID = (
    cell46_question_draft_id
)

CELL47_EXPECTED_ANSWER_DRAFT_ID = (
    cell46_expected_answer_draft_id
)

CELL47_FULL_572_BLOCK_STATUS = (
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


# ============================================================
# 3) Human ACCEPT for Expected Answer
# ============================================================

CELL47_HUMAN_ANSWER_NOTE = (
    "Human Reviewer accepted the Expected Answer after "
    "reviewing the distinction between goods and services "
    "against the parent and supplemental source evidence."
)


cell47_answer_human_review = {
    "review_record_version":
        "human_expected_answer_review_record_v1",

    "pilot_case_id":
        CELL47_PILOT_ID,

    "question_draft_id":
        CELL47_QUESTION_DRAFT_ID,

    "expected_answer_draft_id":
        CELL47_EXPECTED_ANSWER_DRAFT_ID,

    "target_evidence_id":
        CELL47_TARGET_EVIDENCE_ID,

    "human_decision":
        "ACCEPT",

    "human_note":
        CELL47_HUMAN_ANSWER_NOTE,

    "supplemental_term_interpreted_beyond_source":
        False,

    "human_context_used_as_evidence":
        False,

    "external_knowledge_used":
        False,
}


# ============================================================
# 4) Apply Expected Answer ACCEPT
# ============================================================

cell47_citation_pending_session = (
    cell13_apply_event(
        cell46_current_session,
        "EXPECTED_ANSWER_ACCEPTED",
        human_note=
            CELL47_HUMAN_ANSWER_NOTE,
    )
)


cell47_answer_accept_valid = all([
    cell47_citation_pending_session.get(
        "current_state"
    )
    ==
    "CITATION_MAPPING_PENDING",

    cell47_citation_pending_session.get(
        "queue_advance_performed"
    )
    is False,

    cell47_citation_pending_session.get(
        "human_decisions",
        [],
    )[
        -1
    ].get(
        "event"
    )
    ==
    "EXPECTED_ANSWER_ACCEPTED",
])


# ============================================================
# 5) Read Target Evidence parts
# ============================================================

cell47_target_parts = copy.deepcopy(
    cell45_next_target_evidence.get(
        "target_parts",
        [],
    )
)


cell47_target_parts_valid = all([
    isinstance(
        cell47_target_parts,
        list,
    ),

    len(
        cell47_target_parts
    )
    >
    0,
])


# ============================================================
# 6) Build Citation entries
# ============================================================

cell47_citation_entries = []


for index, part in enumerate(
    cell47_target_parts
):

    if not isinstance(
        part,
        dict,
    ):
        continue


    cell47_citation_entries.append({
        "citation_index":
            index,

        "target_evidence_id":
            CELL47_TARGET_EVIDENCE_ID,

        "document_id":
            part.get(
                "document_id"
            ),

        "source_id":
            part.get(
                "source_id"
            ),

        "role":
            part.get(
                "role"
            ),

        "char_start":
            part.get(
                "char_start"
            ),

        "char_end":
            part.get(
                "char_end"
            ),

        "char_count":
            part.get(
                "char_count"
            ),

        "token_count":
            part.get(
                "token_count"
            ),

        "locator":
            copy.deepcopy(
                part.get(
                    "locator",
                    {},
                )
            ),

        "attribution":
            copy.deepcopy(
                part.get(
                    "attribution"
                )
            ),

        "locator_origin":
            "COPIED_FROM_VALIDATED_TARGET_EVIDENCE",

        "external_locator_lookup_used":
            False,
    })


# ============================================================
# 7) Exact locator projection validation
# ============================================================

cell47_source_projection = [
    {
        "document_id":
            part.get(
                "document_id"
            ),

        "source_id":
            part.get(
                "source_id"
            ),

        "role":
            part.get(
                "role"
            ),

        "char_start":
            part.get(
                "char_start"
            ),

        "char_end":
            part.get(
                "char_end"
            ),

        "locator":
            copy.deepcopy(
                part.get(
                    "locator",
                    {},
                )
            ),
    }

    for part
    in cell47_target_parts

    if isinstance(
        part,
        dict,
    )
]


cell47_citation_projection = [
    {
        "document_id":
            entry.get(
                "document_id"
            ),

        "source_id":
            entry.get(
                "source_id"
            ),

        "role":
            entry.get(
                "role"
            ),

        "char_start":
            entry.get(
                "char_start"
            ),

        "char_end":
            entry.get(
                "char_end"
            ),

        "locator":
            copy.deepcopy(
                entry.get(
                    "locator",
                    {},
                )
            ),
    }

    for entry
    in cell47_citation_entries
]


cell47_no_locator_fabrication = (
    cell47_source_projection
    ==
    cell47_citation_projection
)


# ============================================================
# 8) Displayed Evidence coverage
# ============================================================

cell47_displayed_parts = copy.deepcopy(
    cell45_next_target_text_parts
)


cell47_displayed_by_identity = {
    (
        row.get(
            "document_id"
        ),
        row.get(
            "role"
        ),
        row.get(
            "char_start"
        ),
        row.get(
            "char_end"
        ),
    ):
        row

    for row
    in cell47_displayed_parts

    if isinstance(
        row,
        dict,
    )
}


cell47_validation_rows = []


for entry in cell47_citation_entries:

    identity = (
        entry.get(
            "document_id"
        ),
        entry.get(
            "role"
        ),
        entry.get(
            "char_start"
        ),
        entry.get(
            "char_end"
        ),
    )


    displayed = (
        cell47_displayed_by_identity.get(
            identity
        )
    )


    locator = (
        entry.get(
            "locator"
        )
        or
        {}
    )


    canonical_url = locator.get(
        "canonical_url"
    )


    url = locator.get(
        "url"
    )


    row_valid = all([
        isinstance(
            entry.get(
                "document_id"
            ),
            str,
        ),

        isinstance(
            displayed,
            dict,
        ),

        (
            entry.get(
                "char_start"
            )
            is None
            or
            isinstance(
                entry.get(
                    "char_start"
                ),
                int,
            )
        ),

        (
            entry.get(
                "char_end"
            )
            is None
            or
            isinstance(
                entry.get(
                    "char_end"
                ),
                int,
            )
        ),

        (
            canonical_url is None
            or
            (
                isinstance(
                    canonical_url,
                    str,
                )
                and
                canonical_url.startswith(
                    (
                        "http://",
                        "https://",
                    )
                )
            )
        ),

        (
            url is None
            or
            (
                isinstance(
                    url,
                    str,
                )
                and
                url.startswith(
                    (
                        "http://",
                        "https://",
                    )
                )
            )
        ),

        entry.get(
            "locator_origin"
        )
        ==
        "COPIED_FROM_VALIDATED_TARGET_EVIDENCE",

        entry.get(
            "external_locator_lookup_used"
        )
        is False,
    ])


    cell47_validation_rows.append({
        "citation_index":
            entry.get(
                "citation_index"
            ),

        "document_id":
            entry.get(
                "document_id"
            ),

        "role":
            entry.get(
                "role"
            ),

        "char_start":
            entry.get(
                "char_start"
            ),

        "char_end":
            entry.get(
                "char_end"
            ),

        "displayed_match":
            isinstance(
                displayed,
                dict,
            ),

        "valid":
            row_valid,
    })


cell47_all_entries_valid = all([
    len(
        cell47_citation_entries
    )
    >
    0,

    len(
        cell47_citation_entries
    )
    ==
    len(
        cell47_target_parts
    ),

    len(
        cell47_citation_entries
    )
    ==
    len(
        cell47_displayed_parts
    ),

    all(
        row.get(
            "valid"
        )
        is True

        for row
        in cell47_validation_rows
    ),
])


# ============================================================
# 9) Supplemental evidence audit
# ============================================================

cell47_role_values = [
    str(
        entry.get(
            "role"
        )
        or
        ""
    ).upper()

    for entry
    in cell47_citation_entries
]


cell47_supplemental_role_present = any(
    (
        "SUPPLEMENT"
        in role
        or
        "FOOTNOTE"
        in role
    )

    for role
    in cell47_role_values
)


cell47_parent_role_present = any(
    (
        "PARENT"
        in role
        or
        "PRIMARY"
        in role
        or
        "MAIN"
        in role
    )

    for role
    in cell47_role_values
)


cell47_product_text_present = (
    "product"
    in
    cell45_next_combined_target_text.lower()
)


# This Pilot is valid even if the original role labels do not
# literally contain PARENT/SUPPLEMENTAL. We preserve original roles.
cell47_supplemental_structure_preserved = all([
    cell47_product_text_present,

    len(
        cell47_citation_entries
    )
    ==
    len(
        cell47_target_parts
    ),

    cell47_no_locator_fabrication,
])


# ============================================================
# 10) Deterministic Citation Mapping ID
# ============================================================

cell47_citation_identity_payload = {
    "identity_version":
        "citation_mapping_identity_v1",

    "pilot_case_id":
        CELL47_PILOT_ID,

    "question_draft_id":
        CELL47_QUESTION_DRAFT_ID,

    "expected_answer_draft_id":
        CELL47_EXPECTED_ANSWER_DRAFT_ID,

    "target_evidence_id":
        CELL47_TARGET_EVIDENCE_ID,

    "citation_projection":
        copy.deepcopy(
            cell47_citation_projection
        ),
}


cell47_citation_mapping_id = (
    "citation-mapping:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell47_citation_identity_payload
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


# ============================================================
# 11) Citation Machine Validation
# ============================================================

cell47_citation_machine_validation = {
    "target_parts_present":
        cell47_target_parts_valid,

    "citation_entries_present":
        (
            len(
                cell47_citation_entries
            )
            >
            0
        ),

    "all_target_parts_covered":
        (
            len(
                cell47_citation_entries
            )
            ==
            len(
                cell47_target_parts
            )
        ),

    "displayed_evidence_covered":
        (
            len(
                cell47_citation_entries
            )
            ==
            len(
                cell47_displayed_parts
            )
        ),

    "all_entries_valid":
        cell47_all_entries_valid,

    "locator_fabrication_absent":
        cell47_no_locator_fabrication,

    "target_evidence_identity_match":
        all(
            entry.get(
                "target_evidence_id"
            )
            ==
            CELL47_TARGET_EVIDENCE_ID

            for entry
            in cell47_citation_entries
        ),

    "supplemental_product_term_present":
        cell47_product_text_present,

    "supplemental_structure_preserved":
        cell47_supplemental_structure_preserved,

    "supplemental_meaning_inferred_beyond_source":
        False,

    "external_locator_lookup_used":
        False,
}


cell47_citation_machine_pass = all([
    cell47_citation_machine_validation[
        "target_parts_present"
    ],

    cell47_citation_machine_validation[
        "citation_entries_present"
    ],

    cell47_citation_machine_validation[
        "all_target_parts_covered"
    ],

    cell47_citation_machine_validation[
        "displayed_evidence_covered"
    ],

    cell47_citation_machine_validation[
        "all_entries_valid"
    ],

    cell47_citation_machine_validation[
        "locator_fabrication_absent"
    ],

    cell47_citation_machine_validation[
        "target_evidence_identity_match"
    ],

    cell47_citation_machine_validation[
        "supplemental_product_term_present"
    ],

    cell47_citation_machine_validation[
        "supplemental_structure_preserved"
    ],
])


# ============================================================
# 12) Citation Mapping Record
# ============================================================

cell47_citation_mapping_record = {
    "record_version":
        "citation_mapping_record_v1",

    "citation_mapping_id":
        cell47_citation_mapping_id,

    "pilot_case_id":
        CELL47_PILOT_ID,

    "golden_case_id":
        CELL47_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL47_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL47_TARGET_EVIDENCE_ID,

    "question_draft_id":
        CELL47_QUESTION_DRAFT_ID,

    "expected_answer_draft_id":
        CELL47_EXPECTED_ANSWER_DRAFT_ID,

    "citation_entries":
        copy.deepcopy(
            cell47_citation_entries
        ),

    "citation_scope":
        "VALIDATED_TARGET_EVIDENCE_ONLY",

    "locator_source":
        "EXISTING_TARGET_EVIDENCE_LOCATORS",

    "supplemental_evidence_preserved":
        True,

    "supplemental_term_interpreted_beyond_source":
        False,

    "external_locator_lookup_used":
        False,

    "machine_validation_status":
        (
            "PASS_PENDING_HUMAN_ADJUDICATION"
            if cell47_citation_machine_pass
            else
            "FAIL_REQUIRES_REVISION"
        ),

    "human_adjudication_status":
        "PENDING",

    "full_572_generation":
        CELL47_FULL_572_BLOCK_STATUS,
}


# ============================================================
# 13) Register Citation
# ============================================================

cell47_current_session = (
    cell13_apply_event(
        cell47_citation_pending_session,
        "CITATION_MAPPING_REGISTERED",
        artifact=
            cell47_citation_mapping_record,
    )
)


cell47_citation_registered_valid = all([
    cell47_current_session.get(
        "current_state"
    )
    ==
    "CITATION_HUMAN_REVIEW_PENDING",

    isinstance(
        cell47_current_session.get(
            "artifacts",
            {}
        ).get(
            "citation_mapping"
        ),
        dict,
    ),

    cell47_current_session[
        "artifacts"
    ][
        "citation_mapping"
    ].get(
        "citation_mapping_id"
    )
    ==
    cell47_citation_mapping_id,

    cell47_current_session.get(
        "queue_advance_performed"
    )
    is False,
])


# ============================================================
# 14) Update Human Review workspace
# ============================================================

cell47_active_human_review_case = copy.deepcopy(
    cell46_active_human_review_case
)


cell47_active_human_review_case[
    "workflow_state"
] = "CITATION_HUMAN_REVIEW_PENDING"


cell47_active_human_review_case[
    "expected_answer_review"
][
    "human_decision"
] = "ACCEPT"


cell47_active_human_review_case[
    "expected_answer_review"
][
    "human_review"
] = copy.deepcopy(
    cell47_answer_human_review
)


cell47_active_human_review_case[
    "citation_review"
] = {
    "citation_mapping_id":
        cell47_citation_mapping_id,

    "citation_entries":
        copy.deepcopy(
            cell47_citation_entries
        ),

    "machine_validation_status":
        cell47_citation_mapping_record[
            "machine_validation_status"
        ],

    "human_decision":
        "PENDING",
}


cell47_active_human_review_case[
    "final_status"
] = (
    "QUESTION_AND_ANSWER_ACCEPTED_"
    "CITATION_PENDING_REVIEW"
)


# ============================================================
# 15) Persist Registry
# ============================================================

cell47_persistent_registry_payload = copy.deepcopy(
    cell46_persisted_registry
)


cell47_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell47_active_human_review_case
)


cell47_persistent_registry_payload[
    "remaining_pilot_count"
] = 15


cell47_persistent_registry_payload[
    "continuation_state"
] = {
    "next_pilot_id":
        CELL47_PILOT_ID,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell45_remaining_pilot_ids
        ),

    "remaining_count":
        15,

    "continuation_state_sha256":
        cell45_continuation_state_sha256,

    "active_pilot_status":
        "CITATION_HUMAN_REVIEW_PENDING",

    "active_target_evidence_id":
        CELL47_TARGET_EVIDENCE_ID,
}


cell47_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell47_persistent_registry_payload
    )
)


# ============================================================
# 16) Read-back validation
# ============================================================

cell47_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell47_persisted_active_case = (
    cell47_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell47_persistence_valid = all([
    cell47_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL47_PILOT_ID,

    cell47_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell47_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "citation_mapping_id"
    )
    ==
    cell47_citation_mapping_id,

    cell47_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "PENDING",

    cell47_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    15,
])


# ============================================================
# 17) Safety
# ============================================================

cell47_queue_still_15 = (
    len(
        cell45_remaining_pilot_ids
    )
    ==
    15
)


cell47_current_pilot_still_first = (
    cell45_remaining_pilot_ids[
        0
    ]
    ==
    CELL47_PILOT_ID
)


cell47_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell47_full_572_blocked = (
    cell47_citation_mapping_record[
        "full_572_generation"
    ]
    ==
    CELL47_FULL_572_BLOCK_STATUS
)


# ============================================================
# 18) Final Gate
# ============================================================

cell47_gate = all([
    len(
        cell47_failures
    )
    ==
    0,

    cell47_answer_accept_valid,

    cell47_target_parts_valid,

    cell47_no_locator_fabrication,

    cell47_all_entries_valid,

    cell47_citation_machine_pass,

    cell47_citation_registered_valid,

    cell47_persistence_valid,

    cell47_queue_still_15,

    cell47_current_pilot_still_first,

    cell47_base_seed_unchanged,

    cell47_full_572_blocked,
])


# ============================================================
# 19) Diagnostics
# ============================================================

print(
    "=== ANSWER ACCEPTED + SUPPLEMENTAL CITATION DRAFT "
    "+ REGISTRY SYNC ==="
)


print(
    "\n=== ACCEPTED QUESTION ==="
)

print(
    CELL46_ACCEPTED_QUESTION
)


print(
    "\n=== ACCEPTED ANSWER ==="
)

print(
    CELL46_EXPECTED_ANSWER
)

print(
    "Human Answer Decision:",
    "ACCEPT",
)


print(
    "\n=== CITATION DRAFT ==="
)

print(
    "Citation Mapping ID:",
    cell47_citation_mapping_id,
)

print(
    "Target Evidence:",
    CELL47_TARGET_EVIDENCE_ID,
)

print(
    "Citation entry count:",
    len(
        cell47_citation_entries
    ),
)


for entry in cell47_citation_entries:

    print(
        "\n--- CITATION ENTRY",
        entry.get(
            "citation_index"
        ),
        "---"
    )

    print(
        "Document:",
        entry.get(
            "document_id"
        ),
    )

    print(
        "Role:",
        entry.get(
            "role"
        ),
    )

    print(
        "Chars:",
        (
            entry.get(
                "char_start"
            ),
            entry.get(
                "char_end"
            ),
        ),
    )

    print(
        "Paragraph ID:",
        entry.get(
            "locator",
            {},
        ).get(
            "paragraph_id"
        ),
    )

    print(
        "Paragraph Number:",
        entry.get(
            "locator",
            {},
        ).get(
            "paragraph_number"
        ),
    )

    print(
        "Canonical URL:",
        entry.get(
            "locator",
            {},
        ).get(
            "canonical_url"
        ),
    )


print(
    "\n=== SUPPLEMENTAL STRUCTURE ==="
)

print(
    "Product term present:",
    cell47_product_text_present,
)

print(
    "Original role labels:",
    cell47_role_values,
)

print(
    "Literal supplemental/footnote role present:",
    cell47_supplemental_role_present,
)

print(
    "Literal parent/main role present:",
    cell47_parent_role_present,
)

print(
    "Structure preserved:",
    cell47_supplemental_structure_preserved,
)

print(
    "Meaning inferred beyond source:",
    False,
)


print(
    "\n=== CITATION MACHINE VALIDATION ==="
)

for name, status in (
    cell47_citation_machine_validation.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "Machine PASS:",
    cell47_citation_machine_pass,
)

print(
    "No locator fabrication:",
    cell47_no_locator_fabrication,
)


print(
    "\n=== WORKFLOW ==="
)

print(
    "Current State:",
    cell47_current_session.get(
        "current_state"
    ),
)

print(
    "Question Decision:",
    "ACCEPT",
)

print(
    "Answer Decision:",
    "ACCEPT",
)

print(
    "Citation Decision:",
    "PENDING",
)


print(
    "\n=== PERSISTENCE ==="
)

print(
    "Registry:",
    cell21_registry_path,
)

print(
    "Persisted Answer Decision:",
    cell47_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    ),
)

print(
    "Persisted Citation Decision:",
    cell47_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "human_decision"
    ),
)

print(
    "Persistence valid:",
    cell47_persistence_valid,
)


print(
    "\n=== QUEUE / SAFETY ==="
)

print(
    "Queue still 15:",
    cell47_queue_still_15,
)

print(
    "Base Seed unchanged:",
    cell47_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell47_full_572_blocked,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell47_failures
)


print(
    "\nCODE CELL 47 GATE:",
    (
        "PASS"
        if cell47_gate
        else "FAIL"
    ),
)


assert cell47_gate, (
    "Expected Answer ACCEPT / Supplemental Citation Draft / "
    "Registry Sync failed. Repair this SAME Code Cell 47 "
    "before Citation Human Review."
)

=== ANSWER ACCEPTED + SUPPLEMENTAL CITATION DRAFT + REGISTRY SYNC ===

=== ACCEPTED QUESTION ===
محصول فقط شامل کالا می‌شود یا می‌توان یک خدمت را هم محصول دانست؟

=== ACCEPTED ANSWER ===
در ادبیات مدیریت، محصول فقط به کالا محدود نمی‌شود و خدمات را هم شامل می‌شود؛ یعنی هم یک کالا و هم یک خدمت می‌توانند محصول محسوب شوند. البته در کاربرد رایج فارسی، واژه «محصول» بیشتر برای کالاها و چیزهای ملموس به کار می‌رود.
Human Answer Decision: ACCEPT

=== CITATION DRAFT ===
Citation Mapping ID: citation-mapping:51c2506b425f9c18eca7b9db246cc1ce
Target Evidence: target-evidence:5ba981494764ee9f5afdaf88ec194746
Citation entry count: 2

--- CITATION ENTRY 0 ---
Document: book:trainer-training-financial-literacy:fa-source:p000429
Role: PARENT_PARAGRAPH
Chars: (0, 229)
Paragraph ID: book:trainer-training-financial-literacy:fa-source:p000429
Paragraph Number: 429
Canonical URL: None

--- CITATION ENTRY 1 ---
Document: book:trainer-training-financial-literacy:fa-source:fn0008
Role: SUPPLEMENTAL_RECORD
Chars:

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 48 — Citation ACCEPT + Pilot Completion + Queue Advance

Citation این Pilot توسط Human Reviewer پذیرفته شد.

در این مرحله:

- Citation با تصمیم `ACCEPT` ثبت می‌شود؛
- ساختار Parent + Supplemental در پرونده نهایی حفظ می‌شود؛
- Pilot کامل و به Completed Human Review Cases منتقل می‌شود؛
- Author Final Review همچنان `NOT_STARTED` باقی می‌ماند؛
- Queue از 15 به 14 کاهش پیدا می‌کند؛
- Pilot بعدی و Source Evidence آن آماده می‌شود؛
- Registry دائمی Sync می‌شود؛
- Full 572 Generation همچنان Blocked باقی می‌ماند.

</div>

In [48]:
# Code Cell 48

cell48_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell47_gate") is not True:
    cell48_failures.append(
        "CELL47_NOT_PASS"
    )


if (
    globals().get(
        "cell47_current_session",
        {},
    ).get(
        "current_state"
    )
    !=
    "CITATION_HUMAN_REVIEW_PENDING"
):
    cell48_failures.append(
        "CITATION_NOT_AT_HUMAN_REVIEW_GATE"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell48_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Current completed Pilot identity
# ============================================================

CELL48_COMPLETED_PILOT_ID = (
    CELL47_PILOT_ID
)

CELL48_COMPLETED_TARGET_ID = (
    CELL47_TARGET_EVIDENCE_ID
)

CELL48_FULL_572_BLOCK_STATUS = (
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


# ============================================================
# 3) Human Citation ACCEPT
# ============================================================

CELL48_HUMAN_CITATION_NOTE = (
    "Human Reviewer accepted the Citation Mapping after "
    "reviewing both the parent paragraph and supplemental "
    "record against the validated Target Evidence."
)


cell48_citation_human_review = {
    "review_record_version":
        "human_citation_review_record_v1",

    "pilot_case_id":
        CELL48_COMPLETED_PILOT_ID,

    "citation_mapping_id":
        cell47_citation_mapping_id,

    "target_evidence_id":
        CELL48_COMPLETED_TARGET_ID,

    "human_decision":
        "ACCEPT",

    "human_note":
        CELL48_HUMAN_CITATION_NOTE,

    "supplemental_structure_preserved":
        True,

    "supplemental_term_interpreted_beyond_source":
        False,

    "external_locator_lookup_used":
        False,
}


# ============================================================
# 4) Apply Citation ACCEPT
# ============================================================

cell48_pre_advance_session = (
    cell13_apply_event(
        cell47_current_session,
        "CITATION_ACCEPTED",
        human_note=
            CELL48_HUMAN_CITATION_NOTE,
    )
)


cell48_citation_accept_valid = all([
    cell48_pre_advance_session.get(
        "current_state"
    )
    ==
    "PILOT_COMPLETE_PENDING_QUEUE_ADVANCE",

    cell48_pre_advance_session.get(
        "queue_advance_permitted"
    )
    is True,

    cell48_pre_advance_session.get(
        "queue_advance_performed"
    )
    is False,

    cell48_pre_advance_session.get(
        "human_decisions",
        [],
    )[
        -1
    ].get(
        "event"
    )
    ==
    "CITATION_ACCEPTED",
])


# ============================================================
# 5) Finalize Human Review Case
# ============================================================

cell48_completed_human_review_case = copy.deepcopy(
    cell47_active_human_review_case
)


cell48_completed_human_review_case[
    "workflow_state"
] = "PILOT_COMPLETE"


cell48_completed_human_review_case[
    "citation_review"
][
    "human_decision"
] = "ACCEPT"


cell48_completed_human_review_case[
    "citation_review"
][
    "human_review"
] = copy.deepcopy(
    cell48_citation_human_review
)


cell48_completed_human_review_case[
    "supplemental_completion"
] = {
    "supplemental_evidence_present":
        True,

    "parent_role_present":
        cell47_parent_role_present,

    "supplemental_role_present":
        cell47_supplemental_role_present,

    "supplemental_structure_preserved":
        cell47_supplemental_structure_preserved,

    "supplemental_meaning_inferred_beyond_source":
        False,
}


cell48_completed_human_review_case[
    "author_final_review"
] = {
    "status":
        "NOT_STARTED",

    "decision":
        None,

    "notes":
        None,
}


cell48_completed_human_review_case[
    "final_status"
] = (
    "HUMAN_REVIEW_COMPLETE_"
    "AUTHOR_FINAL_REVIEW_PENDING"
)


cell48_completed_human_review_case_sha256 = (
    cell10_stable_sha(
        cell48_completed_human_review_case
    )
)


# ============================================================
# 6) Queue preconditions
# ============================================================

cell48_parent_remaining_ids = copy.deepcopy(
    cell45_continuation_state[
        "remaining_pilot_ids"
    ]
)


cell48_parent_remaining_queue = copy.deepcopy(
    cell45_continuation_state[
        "remaining_pilot_queue"
    ]
)


cell48_queue_preconditions_valid = all([
    len(
        cell48_parent_remaining_ids
    )
    ==
    15,

    len(
        cell48_parent_remaining_queue
    )
    ==
    15,

    cell48_parent_remaining_ids[
        0
    ]
    ==
    CELL48_COMPLETED_PILOT_ID,

    cell48_citation_accept_valid,
])


# ============================================================
# 7) Advance Queue
# ============================================================

if cell48_queue_preconditions_valid:

    cell48_remaining_pilot_ids = copy.deepcopy(
        cell48_parent_remaining_ids[
            1:
        ]
    )

    cell48_remaining_pilot_queue = copy.deepcopy(
        cell48_parent_remaining_queue[
            1:
        ]
    )

else:

    cell48_remaining_pilot_ids = []
    cell48_remaining_pilot_queue = []


cell48_next_pilot_id = (
    cell48_remaining_pilot_ids[
        0
    ]

    if cell48_remaining_pilot_ids

    else None
)


cell48_next_pilot_queue_entry = (
    copy.deepcopy(
        cell48_remaining_pilot_queue[
            0
        ]
    )

    if cell48_remaining_pilot_queue

    else None
)


cell48_queue_advanced_valid = all([
    len(
        cell48_remaining_pilot_ids
    )
    ==
    14,

    len(
        cell48_remaining_pilot_queue
    )
    ==
    14,

    CELL48_COMPLETED_PILOT_ID
    not in
    cell48_remaining_pilot_ids,

    isinstance(
        cell48_next_pilot_id,
        str,
    ),

    cell48_next_pilot_id
    !=
    CELL48_COMPLETED_PILOT_ID,
])


# ============================================================
# 8) Completed Engine session
# ============================================================

cell48_completed_session = copy.deepcopy(
    cell48_pre_advance_session
)


cell48_completed_session[
    "current_state"
] = "PILOT_COMPLETE"


cell48_completed_session[
    "queue_advance_permitted"
] = False


cell48_completed_session[
    "queue_advance_performed"
] = True


cell48_completed_session[
    "completion_status"
] = (
    "HUMAN_REVIEW_COMPLETE_"
    "AUTHOR_FINAL_REVIEW_PENDING"
)


# ============================================================
# 9) New authoritative Continuation State
# ============================================================

cell48_continuation_state = {
    "state_version":
        "compact_pilot_continuation_state_v8",

    "authoritative_forward_state":
        True,

    "parent_state_sha256":
        cell45_continuation_state_sha256,

    "just_completed_pilot_id":
        CELL48_COMPLETED_PILOT_ID,

    "just_completed_target_evidence_id":
        CELL48_COMPLETED_TARGET_ID,

    "just_completed_case_sha256":
        cell48_completed_human_review_case_sha256,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell48_remaining_pilot_ids
        ),

    "remaining_pilot_queue":
        copy.deepcopy(
            cell48_remaining_pilot_queue
        ),

    "remaining_count":
        14,

    "next_pilot_id":
        cell48_next_pilot_id,

    "next_pilot_queue_entry":
        copy.deepcopy(
            cell48_next_pilot_queue_entry
        ),

    "full_572_generation":
        CELL48_FULL_572_BLOCK_STATUS,
}


cell48_continuation_state_sha256 = (
    cell10_stable_sha(
        cell48_continuation_state
    )
)


# ============================================================
# 10) Resolve next Pilot packet
# ============================================================

cell48_pilot_by_id = (
    cell13_seed[
        "final_runtime"
    ][
        "question_drafting_pilot_by_id"
    ]
)


cell48_next_pilot_packet = copy.deepcopy(
    cell48_pilot_by_id.get(
        cell48_next_pilot_id
    )
)


cell48_next_packet_valid = isinstance(
    cell48_next_pilot_packet,
    dict,
)


# ============================================================
# 11) Resolve next identity chain
# ============================================================

cell48_next_golden_case_id = None
cell48_next_task_draft_id = None
cell48_next_target_evidence_id = None


if cell48_next_packet_valid:

    try:

        cell48_next_golden_case_id = (
            cell13_exact_single_id(
                cell48_next_pilot_packet,
                "golden_case",
            )
        )


        cell48_next_task_draft_id = (
            cell13_exact_single_id(
                cell48_next_pilot_packet,
                "task_draft",
            )
        )


        cell48_next_target_evidence_id = (
            cell13_exact_single_id(
                cell48_next_pilot_packet,
                "target_evidence",
            )
        )


    except Exception as exc:

        cell48_failures.append({
            "reason":
                "NEXT_ID_CHAIN_FAILED",

            "error":
                repr(
                    exc
                ),
        })


cell48_next_identity_valid = all([
    isinstance(
        cell48_next_golden_case_id,
        str,
    ),

    isinstance(
        cell48_next_task_draft_id,
        str,
    ),

    isinstance(
        cell48_next_target_evidence_id,
        str,
    ),
])


# ============================================================
# 12) Resolve next support records
# ============================================================

cell48_next_golden_case = None
cell48_next_task_plan = None
cell48_next_target_evidence = None


if cell48_next_identity_valid:

    try:

        cell48_next_golden_case = copy.deepcopy(
            cell13_active_support[
                "golden_case_records_by_id"
            ][
                cell48_next_golden_case_id
            ]
        )


        cell48_next_task_plan = copy.deepcopy(
            cell13_find_unique_mapping_value(
                cell13_active_support.get(
                    "task_draft_plans",
                    {},
                ),
                "task_draft",
                cell48_next_task_draft_id,
            )[
                "value"
            ]
        )


        cell48_next_target_evidence = copy.deepcopy(
            cell13_find_unique_list_record(
                cell13_active_support.get(
                    "target_evidence_records",
                    [],
                ),
                "target_evidence_id",
                cell48_next_target_evidence_id,
            )
        )


    except Exception as exc:

        cell48_failures.append({
            "reason":
                "NEXT_SUPPORT_RESOLUTION_FAILED",

            "error":
                repr(
                    exc
                ),
        })


# ============================================================
# 13) Classification helper
# ============================================================

def cell48_scalar(
    mapping,
    names,
):

    match = cell14_find_first_scalar(
        mapping,
        names,
    )

    if isinstance(
        match,
        dict,
    ):

        return match.get(
            "value"
        )

    return None


cell48_next_source_type = cell48_scalar(
    cell48_next_pilot_packet,
    [
        "source_type",
        "source_kind",
        "source",
    ],
)


cell48_next_stratum = cell48_scalar(
    cell48_next_pilot_packet,
    [
        "stratum",
        "stratum_name",
        "stratum_id",
    ],
)


cell48_next_task_family = cell48_scalar(
    cell48_next_pilot_packet,
    [
        "task_family",
    ],
)


cell48_next_question_archetype = cell48_scalar(
    cell48_next_pilot_packet,
    [
        "question_archetype",
    ],
)


# ============================================================
# 14) Extract next Source Evidence
# ============================================================

cell48_raw_target_parts = []


if cell48_next_packet_valid:

    packet_parts = (
        cell48_next_pilot_packet.get(
            "target_text_parts"
        )
    )

    if isinstance(
        packet_parts,
        list,
    ):

        cell48_raw_target_parts = copy.deepcopy(
            packet_parts
        )


if (
    not cell48_raw_target_parts
    and
    isinstance(
        cell48_next_target_evidence,
        dict,
    )
):

    evidence_parts = (
        cell48_next_target_evidence.get(
            "target_parts"
        )
    )

    if isinstance(
        evidence_parts,
        list,
    ):

        cell48_raw_target_parts = copy.deepcopy(
            evidence_parts
        )


cell48_next_target_text_parts = []


for index, part in enumerate(
    cell48_raw_target_parts
):

    if not isinstance(
        part,
        dict,
    ):
        continue


    text_value = None


    for field_name in (
        "text",
        "target_text",
        "content",
        "evidence_text",
        "text_content",
    ):

        candidate = part.get(
            field_name
        )

        if (
            isinstance(
                candidate,
                str,
            )
            and
            candidate.strip()
        ):

            text_value = candidate.strip()
            break


    if text_value is None:
        continue


    cell48_next_target_text_parts.append({
        "part_index":
            index,

        "document_id":
            part.get(
                "document_id"
            ),

        "role":
            part.get(
                "role"
            ),

        "char_start":
            part.get(
                "char_start"
            ),

        "char_end":
            part.get(
                "char_end"
            ),

        "text":
            text_value,

        "text_sha256":
            hashlib.sha256(
                text_value.encode(
                    "utf-8"
                )
            ).hexdigest(),
    })


cell48_next_combined_target_text = (
    "\n\n".join(
        row[
            "text"
        ]

        for row
        in cell48_next_target_text_parts
    )
)


cell48_next_target_text_available = bool(
    cell48_next_combined_target_text.strip()
)


# ============================================================
# 15) Build next Pilot workspace
# ============================================================

cell48_next_pilot_workspace = {
    "workspace_version":
        "compact_human_review_workspace_v1",

    "status":
        "READY_FOR_QUESTION_DRAFT",

    "pilot_case_id":
        cell48_next_pilot_id,

    "golden_case_id":
        cell48_next_golden_case_id,

    "task_draft_id":
        cell48_next_task_draft_id,

    "target_evidence_id":
        cell48_next_target_evidence_id,

    "source_type":
        cell48_next_source_type,

    "stratum":
        cell48_next_stratum,

    "task_family":
        cell48_next_task_family,

    "question_archetype":
        cell48_next_question_archetype,

    "source_evidence":
        copy.deepcopy(
            cell48_next_target_text_parts
        ),

    "combined_target_text":
        cell48_next_combined_target_text,

    "golden_case":
        copy.deepcopy(
            cell48_next_golden_case
        ),

    "task_plan":
        copy.deepcopy(
            cell48_next_task_plan
        ),

    "target_evidence":
        copy.deepcopy(
            cell48_next_target_evidence
        ),

    "model_drafts":
        [],

    "human_reviews":
        [],

    "expected_answer_review":
        None,

    "citation_review":
        None,

    "author_final_review": {
        "status":
            "NOT_STARTED",

        "decision":
            None,

        "notes":
            None,
    },

    "workflow_state":
        "QUESTION_DRAFT_PENDING",

    "full_572_generation":
        CELL48_FULL_572_BLOCK_STATUS,
}


cell48_next_workspace_ready = all([
    cell48_next_packet_valid,

    cell48_next_identity_valid,

    isinstance(
        cell48_next_golden_case,
        dict,
    ),

    isinstance(
        cell48_next_task_plan,
        dict,
    ),

    isinstance(
        cell48_next_target_evidence,
        dict,
    ),

    cell48_next_target_text_available,
])


# ============================================================
# 16) Update Completed Human Review cases
# ============================================================

cell48_completed_cases = copy.deepcopy(
    cell47_persisted_registry.get(
        "completed_human_review_cases",
        [],
    )
)


cell48_completed_cases.append(
    copy.deepcopy(
        cell48_completed_human_review_case
    )
)


# ============================================================
# 17) Build persistent Registry payload
# ============================================================

cell48_persistent_registry_payload = copy.deepcopy(
    cell47_persisted_registry
)


cell48_persistent_registry_payload[
    "completed_human_review_cases"
] = copy.deepcopy(
    cell48_completed_cases
)


cell48_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell48_next_pilot_workspace
)


cell48_persistent_registry_payload[
    "remaining_pilot_count"
] = 14


cell48_persistent_registry_payload[
    "continuation_state"
] = {
    "next_pilot_id":
        cell48_next_pilot_id,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell48_remaining_pilot_ids
        ),

    "remaining_count":
        14,

    "continuation_state_sha256":
        cell48_continuation_state_sha256,
}


# ============================================================
# 18) Author Review state
# ============================================================

cell48_author_review = copy.deepcopy(
    cell48_persistent_registry_payload.get(
        "author_review",
        {},
    )
)


if not isinstance(
    cell48_author_review,
    dict,
):

    cell48_author_review = {}


cell48_author_review[
    "status"
] = "NOT_STARTED"


cell48_author_review[
    "reviewer"
] = "AUTHOR"


cell48_author_review[
    "cases_ready_for_review"
] = len(
    cell48_completed_cases
)


cell48_persistent_registry_payload[
    "author_review"
] = copy.deepcopy(
    cell48_author_review
)


# ============================================================
# 19) Persist
# ============================================================

cell48_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell48_persistent_registry_payload
    )
)


# ============================================================
# 20) Read-back validation
# ============================================================

cell48_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell48_persisted_active_case = (
    cell48_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell48_persistence_valid = all([
    len(
        cell48_persisted_registry.get(
            "completed_human_review_cases",
            [],
        )
    )
    ==
    len(
        cell48_completed_cases
    ),

    cell48_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    14,

    cell48_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    cell48_next_pilot_id,

    cell48_persisted_registry.get(
        "continuation_state",
        {},
    ).get(
        "next_pilot_id"
    )
    ==
    cell48_next_pilot_id,

    cell48_persisted_registry.get(
        "author_review",
        {},
    ).get(
        "cases_ready_for_review"
    )
    ==
    len(
        cell48_completed_cases
    ),
])


# ============================================================
# 21) Safety
# ============================================================

cell48_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell48_full_572_blocked = (
    cell48_continuation_state[
        "full_572_generation"
    ]
    ==
    CELL48_FULL_572_BLOCK_STATUS
)


# ============================================================
# 22) Final Gate
# ============================================================

cell48_gate = all([
    len(
        cell48_failures
    )
    ==
    0,

    cell48_citation_accept_valid,

    cell48_queue_preconditions_valid,

    cell48_queue_advanced_valid,

    cell48_next_workspace_ready,

    cell48_persistence_valid,

    cell48_base_seed_unchanged,

    cell48_full_572_blocked,
])


# ============================================================
# 23) Diagnostics
# ============================================================

print(
    "=== PILOT COMPLETED + QUEUE ADVANCE "
    "+ NEXT PILOT ==="
)


print(
    "\n=== COMPLETED PILOT ==="
)

print(
    "Pilot:",
    CELL48_COMPLETED_PILOT_ID,
)

print(
    "Question:",
    CELL46_ACCEPTED_QUESTION,
)

print(
    "Answer:",
    CELL46_EXPECTED_ANSWER,
)

print(
    "Citation:",
    "ACCEPT",
)

print(
    "Supplemental structure preserved:",
    cell47_supplemental_structure_preserved,
)

print(
    "Human Review status:",
    cell48_completed_human_review_case[
        "final_status"
    ],
)

print(
    "Author Final Review:",
    "NOT_STARTED",
)


print(
    "\n=== QUEUE ==="
)

print(
    "Before:",
    len(
        cell48_parent_remaining_ids
    ),
)

print(
    "After:",
    len(
        cell48_remaining_pilot_ids
    ),
)

print(
    "Completed Pilot removed:",
    (
        CELL48_COMPLETED_PILOT_ID
        not in
        cell48_remaining_pilot_ids
    ),
)

print(
    "Next Pilot:",
    cell48_next_pilot_id,
)


print(
    "\n=== PERSISTENT REGISTRY ==="
)

print(
    "Path:",
    cell21_registry_path,
)

print(
    "Completed Human Review cases:",
    len(
        cell48_persisted_registry.get(
            "completed_human_review_cases",
            [],
        )
    ),
)

print(
    "Cases ready for Author Review:",
    cell48_persisted_registry.get(
        "author_review",
        {},
    ).get(
        "cases_ready_for_review"
    ),
)

print(
    "Remaining Pilots:",
    cell48_persisted_registry.get(
        "remaining_pilot_count"
    ),
)

print(
    "Active Pilot:",
    cell48_persisted_active_case.get(
        "pilot_case_id"
    ),
)

print(
    "Persistence valid:",
    cell48_persistence_valid,
)


print(
    "\n=== NEXT PILOT IDENTITY ==="
)

print(
    "Pilot:",
    cell48_next_pilot_id,
)

print(
    "Golden Case:",
    cell48_next_golden_case_id,
)

print(
    "Task Draft:",
    cell48_next_task_draft_id,
)

print(
    "Target Evidence:",
    cell48_next_target_evidence_id,
)

print(
    "Source type:",
    cell48_next_source_type,
)

print(
    "Stratum:",
    cell48_next_stratum,
)

print(
    "Task family:",
    cell48_next_task_family,
)

print(
    "Question archetype:",
    cell48_next_question_archetype,
)


print(
    "\n=== NEXT PILOT SOURCE EVIDENCE ==="
)

print(
    cell48_next_combined_target_text
)


print(
    "\n=== SAFETY ==="
)

print(
    "Base Seed unchanged:",
    cell48_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell48_full_572_blocked,
)

print(
    "Author Notebook rewrite:",
    False,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell48_failures
)


print(
    "\nCODE CELL 48 GATE:",
    (
        "PASS"
        if cell48_gate
        else "FAIL"
    ),
)


assert cell48_gate, (
    "Citation ACCEPT / Pilot completion / Queue advance / "
    "Registry sync / next Pilot preparation failed. "
    "Repair this SAME Code Cell 48 before continuing."
)

=== PILOT COMPLETED + QUEUE ADVANCE + NEXT PILOT ===

=== COMPLETED PILOT ===
Pilot: question-pilot:d7378ba1655fe80c37029153e753f80b
Question: محصول فقط شامل کالا می‌شود یا می‌توان یک خدمت را هم محصول دانست؟
Answer: در ادبیات مدیریت، محصول فقط به کالا محدود نمی‌شود و خدمات را هم شامل می‌شود؛ یعنی هم یک کالا و هم یک خدمت می‌توانند محصول محسوب شوند. البته در کاربرد رایج فارسی، واژه «محصول» بیشتر برای کالاها و چیزهای ملموس به کار می‌رود.
Citation: ACCEPT
Supplemental structure preserved: True
Human Review status: HUMAN_REVIEW_COMPLETE_AUTHOR_FINAL_REVIEW_PENDING
Author Final Review: NOT_STARTED

=== QUEUE ===
Before: 15
After: 14
Completed Pilot removed: True
Next Pilot: question-pilot:c9b2a6ecc853bf563a45af119621c405

=== PERSISTENT REGISTRY ===
Path: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/data/phase2/human_review/human_review_registry.json
Completed Human Review cases: 8
Cases ready for Author Review: 8
Remaining Pilots: 14
Active Pilot: question-pilot

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 49 — RETARGET Registration + Deep Heading Candidate Discovery

Human Reviewer تصمیم گرفت Target فعلی Retarget شود.

دلیل:

Target فعلی فقط یک Heading با متن **«چه گزینه‌هایی پیش‌رو دارم؟»** است و محتوای کافی برای ساخت یک سؤال و جواب طبیعی و ارزشمند ندارد.

در این Retarget:

- خود Pilot حفظ می‌شود؛
- `deep_heading_path` حفظ می‌شود؛
- `DEEP_HIERARCHICAL_LOCATOR_RETRIEVAL` حفظ می‌شود؛
- Queue جلو نمی‌رود و همچنان 14 باقی می‌ماند؛
- Target فعلی با وضعیت `RETARGET` ثبت می‌شود؛
- از Canonical Book Corpus بخش‌هایی با Heading Path عمیق و محتوای واقعی پیدا می‌شوند؛
- Candidateهای کم‌ارزش، Heading-only و Index-like تا حد ممکن حذف می‌شوند؛
- هیچ Commit یا Push انجام نمی‌شود.

</div>

In [49]:
# Code Cell 49

from pathlib import Path

cell49_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell48_gate") is not True:
    cell49_failures.append(
        "CELL48_NOT_PASS"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell49_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Current Pilot identity
# ============================================================

CELL49_PILOT_ID = (
    cell48_next_pilot_id
)

CELL49_GOLDEN_CASE_ID = (
    cell48_next_golden_case_id
)

CELL49_TASK_DRAFT_ID = (
    cell48_next_task_draft_id
)

CELL49_OLD_TARGET_EVIDENCE_ID = (
    cell48_next_target_evidence_id
)

CELL49_RETARGET_REASON = (
    "Current Target Evidence is heading-only and does not "
    "contain sufficient substantive content for a natural, "
    "answerable Golden QA case. Preserve deep hierarchical "
    "locator coverage while replacing the Target Evidence "
    "with a substantive passage from a deep heading path."
)

CELL49_FULL_572_BLOCK_STATUS = (
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


# ============================================================
# 3) Audit current Target
# ============================================================

cell49_old_target_parts = copy.deepcopy(
    cell48_next_target_evidence.get(
        "target_parts",
        [],
    )
)


cell49_old_target_text = (
    cell48_next_combined_target_text.strip()
)


cell49_old_target_heading_like = all([
    len(
        cell49_old_target_text
    )
    <
    120,

    len(
        cell49_old_target_parts
    )
    <=
    2,

    (
        cell49_old_target_text.endswith(
            "؟"
        )
        or
        cell49_old_target_text.endswith(
            "?"
        )
    ),
])


cell49_old_target_audit = {
    "pilot_case_id":
        CELL49_PILOT_ID,

    "old_target_evidence_id":
        CELL49_OLD_TARGET_EVIDENCE_ID,

    "source_text":
        cell49_old_target_text,

    "target_part_count":
        len(
            cell49_old_target_parts
        ),

    "heading_like":
        cell49_old_target_heading_like,

    "human_decision":
        "RETARGET",

    "human_reason":
        CELL49_RETARGET_REASON,
}


# ============================================================
# 4) Build blocked Retarget workspace
# ============================================================

cell49_retarget_workspace = copy.deepcopy(
    cell48_next_pilot_workspace
)


cell49_retarget_workspace[
    "status"
] = "ACTIVE_RETARGET"


cell49_retarget_workspace[
    "workflow_state"
] = "BLOCKED_REQUIRES_RETARGET"


cell49_retarget_workspace[
    "retarget_review"
] = {
    "decision":
        "RETARGET",

    "old_target_evidence_id":
        CELL49_OLD_TARGET_EVIDENCE_ID,

    "reason":
        CELL49_RETARGET_REASON,

    "preserve_pilot_case_id":
        True,

    "preserve_task_family":
        True,

    "preserve_question_archetype":
        True,

    "preserve_stratum":
        True,

    "replacement_target_evidence_id":
        None,

    "replacement_status":
        "SEARCH_PENDING",
}


# ============================================================
# 5) Resolve canonical ingestion
# ============================================================

cell49_repo_root = (
    cell21_registry_path.parents[
        3
    ]
)


cell49_canonical_path = (
    cell49_repo_root
    /
    "data"
    /
    "phase2"
    /
    "interim"
    /
    "book_ingested_v2.json"
)


cell49_canonical_exists = (
    cell49_canonical_path.exists()
)


if not cell49_canonical_exists:

    cell49_failures.append({
        "reason":
            "CANONICAL_BOOK_INGESTION_NOT_FOUND",

        "path":
            str(
                cell49_canonical_path
            ),
    })


cell49_canonical_payload = {}


if cell49_canonical_exists:

    try:

        cell49_canonical_payload = json.loads(
            cell49_canonical_path.read_text(
                encoding="utf-8"
            )
        )

    except Exception as exc:

        cell49_failures.append({
            "reason":
                "CANONICAL_BOOK_INGESTION_READ_FAILED",

            "error":
                repr(
                    exc
                ),
        })


cell49_edition_records = (
    cell49_canonical_payload.get(
        "edition_records",
        []
    )

    if isinstance(
        cell49_canonical_payload,
        dict,
    )

    else []
)


# ============================================================
# 6) Collect already-used book document IDs
# ============================================================

cell49_used_document_ids = set()


def cell49_collect_document_ids(
    value,
):

    if isinstance(
        value,
        dict,
    ):

        for key, item in value.items():

            if (
                key
                ==
                "document_id"
                and
                isinstance(
                    item,
                    str,
                )
            ):

                cell49_used_document_ids.add(
                    item
                )


            cell49_collect_document_ids(
                item
            )


    elif isinstance(
        value,
        list,
    ):

        for item in value:

            cell49_collect_document_ids(
                item
            )


cell49_collect_document_ids(
    cell48_persisted_registry.get(
        "completed_human_review_cases",
        [],
    )
)


cell49_collect_document_ids(
    cell48_persisted_registry.get(
        "historical_completed_pilot",
        {},
    )
)


cell49_collect_document_ids(
    cell49_old_target_parts
)


# ============================================================
# 7) Determine current source edition
# ============================================================

cell49_current_edition_id = None


for part in cell49_old_target_parts:

    if not isinstance(
        part,
        dict,
    ):
        continue


    document_id = part.get(
        "document_id"
    )


    if not isinstance(
        document_id,
        str,
    ):
        continue


    match = re.match(
        r"^(.*):p\d+$",
        document_id,
    )


    if match:

        cell49_current_edition_id = (
            match.group(
                1
            )
        )

        break


# ============================================================
# 8) Normalize heading paths
# ============================================================

def cell49_heading_path(
    paragraph,
):

    raw = paragraph.get(
        "heading_path"
    )


    if isinstance(
        raw,
        list,
    ):

        return tuple(
            str(
                item
            ).strip()

            for item
            in raw

            if str(
                item
            ).strip()
        )


    if (
        isinstance(
            raw,
            str,
        )
        and
        raw.strip()
    ):

        return (
            raw.strip(),
        )


    return tuple()


def cell49_paragraph_number(
    paragraph,
):

    value = paragraph.get(
        "paragraph_number"
    )


    if isinstance(
        value,
        int,
    ):

        return value


    if isinstance(
        value,
        str,
    ):

        digits = re.sub(
            r"\D+",
            "",
            value,
        )


        if digits:

            return int(
                digits
            )


    return None


# ============================================================
# 9) Inventory heading depths
# ============================================================

cell49_all_paragraph_rows = []


for edition in cell49_edition_records:

    if not isinstance(
        edition,
        dict,
    ):
        continue


    edition_id = edition.get(
        "edition_id"
    )


    paragraphs = edition.get(
        "paragraphs",
        []
    )


    if not isinstance(
        paragraphs,
        list,
    ):
        continue


    for paragraph in paragraphs:

        if not isinstance(
            paragraph,
            dict,
        ):
            continue


        path = cell49_heading_path(
            paragraph
        )


        cell49_all_paragraph_rows.append({
            "edition_id":
                edition_id,

            "paragraph":
                paragraph,

            "heading_path":
                path,

            "depth":
                len(
                    path
                ),
        })


cell49_max_heading_depth = max(
    [
        row[
            "depth"
        ]

        for row
        in cell49_all_paragraph_rows
    ]
    or
    [
        0
    ]
)


cell49_depth_threshold = (
    3
    if cell49_max_heading_depth >= 3
    else
    cell49_max_heading_depth
)


# ============================================================
# 10) Group substantive paragraphs by deep heading path
# ============================================================

cell49_grouped = {}


for row in cell49_all_paragraph_rows:

    edition_id = row[
        "edition_id"
    ]

    paragraph = row[
        "paragraph"
    ]

    heading_path = row[
        "heading_path"
    ]

    depth = row[
        "depth"
    ]


    if depth < cell49_depth_threshold:
        continue


    if paragraph.get(
        "is_heading"
    ) is True:
        continue


    text = paragraph.get(
        "text"
    )


    if not (
        isinstance(
            text,
            str,
        )
        and
        text.strip()
    ):
        continue


    text = text.strip()


    if len(
        text
    ) < 35:
        continue


    paragraph_id = paragraph.get(
        "paragraph_id"
    )


    if (
        isinstance(
            paragraph_id,
            str,
        )
        and
        paragraph_id
        in
        cell49_used_document_ids
    ):
        continue


    key = (
        edition_id,
        heading_path,
    )


    cell49_grouped.setdefault(
        key,
        []
    ).append(
        paragraph
    )


# ============================================================
# 11) Build contiguous candidate windows
# ============================================================

cell49_candidate_rows = []


for (
    edition_id,
    heading_path,
), paragraphs in cell49_grouped.items():

    paragraphs = sorted(
        paragraphs,
        key=lambda item: (
            cell49_paragraph_number(
                item
            )
            if
            cell49_paragraph_number(
                item
            )
            is not None
            else
            10**12
        ),
    )


    contiguous_runs = []
    current_run = []
    previous_number = None


    for paragraph in paragraphs:

        number = cell49_paragraph_number(
            paragraph
        )


        if not current_run:

            current_run = [
                paragraph
            ]


        elif (
            previous_number is not None
            and
            number is not None
            and
            number
            ==
            previous_number + 1
        ):

            current_run.append(
                paragraph
            )


        else:

            contiguous_runs.append(
                current_run
            )

            current_run = [
                paragraph
            ]


        previous_number = number


    if current_run:

        contiguous_runs.append(
            current_run
        )


    for run in contiguous_runs:

        start = 0


        while start < len(
            run
        ):

            window = run[
                start:
                start + 4
            ]


            texts = [
                paragraph.get(
                    "text",
                    "",
                ).strip()

                for paragraph
                in window

                if isinstance(
                    paragraph.get(
                        "text"
                    ),
                    str,
                )
            ]


            combined_text = (
                "\n\n".join(
                    texts
                )
            ).strip()


            paragraph_ids = [
                paragraph.get(
                    "paragraph_id"
                )

                for paragraph
                in window
            ]


            paragraph_numbers = [
                cell49_paragraph_number(
                    paragraph
                )

                for paragraph
                in window
            ]


            average_length = (
                sum(
                    len(
                        text
                    )

                    for text
                    in texts
                )
                /
                len(
                    texts
                )

                if texts

                else 0
            )


            persian_words = set(
                re.findall(
                    r"[\u0600-\u06FF]{2,}",
                    combined_text,
                )
            )


            heading_only_like = all([
                len(
                    combined_text
                )
                <
                120,

                (
                    combined_text.endswith(
                        "؟"
                    )
                    or
                    combined_text.endswith(
                        "?"
                    )
                ),
            ])


            index_like = all([
                len(
                    texts
                )
                >=
                3,

                average_length
                <
                45,
            ])


            substantive = all([
                len(
                    combined_text
                )
                >=
                120,

                len(
                    persian_words
                )
                >=
                12,

                not heading_only_like,

                not index_like,
            ])


            if substantive:

                identity_payload = {
                    "edition_id":
                        edition_id,

                    "heading_path":
                        list(
                            heading_path
                        ),

                    "paragraph_ids":
                        paragraph_ids,
                }


                candidate_id = (
                    "deep-retarget-candidate:"
                    +
                    hashlib.sha256(
                        cell10_canonical_json(
                            identity_payload
                        ).encode(
                            "utf-8"
                        )
                    ).hexdigest()[
                        :32
                    ]
                )


                same_edition = (
                    edition_id
                    ==
                    cell49_current_edition_id
                )


                score = (
                    (
                        50
                        if same_edition
                        else 0
                    )
                    +
                    (
                        len(
                            heading_path
                        )
                        *
                        10
                    )
                    +
                    min(
                        len(
                            combined_text
                        ),
                        800,
                    )
                    /
                    80
                    +
                    min(
                        len(
                            window
                        ),
                        4,
                    )
                    *
                    2
                )


                cell49_candidate_rows.append({
                    "candidate_id":
                        candidate_id,

                    "edition_id":
                        edition_id,

                    "same_edition_as_old_target":
                        same_edition,

                    "heading_depth":
                        len(
                            heading_path
                        ),

                    "heading_path":
                        list(
                            heading_path
                        ),

                    "paragraph_ids":
                        paragraph_ids,

                    "paragraph_numbers":
                        paragraph_numbers,

                    "paragraph_count":
                        len(
                            window
                        ),

                    "character_count":
                        len(
                            combined_text
                        ),

                    "combined_text":
                        combined_text,

                    "score":
                        round(
                            score,
                            4,
                        ),
                })


            start += max(
                1,
                len(
                    window
                ),
            )


# ============================================================
# 12) Deduplicate + rank
# ============================================================

cell49_candidate_by_id = {}


for candidate in cell49_candidate_rows:

    candidate_id = candidate[
        "candidate_id"
    ]


    previous = cell49_candidate_by_id.get(
        candidate_id
    )


    if (
        previous is None
        or
        candidate[
            "score"
        ]
        >
        previous[
            "score"
        ]
    ):

        cell49_candidate_by_id[
            candidate_id
        ] = candidate


cell49_ranked_candidates = sorted(
    cell49_candidate_by_id.values(),
    key=lambda item: (
        -int(
            item[
                "same_edition_as_old_target"
            ]
        ),
        -item[
            "heading_depth"
        ],
        -item[
            "score"
        ],
        item[
            "candidate_id"
        ],
    ),
)


cell49_shortlist = copy.deepcopy(
    cell49_ranked_candidates[
        :8
    ]
)


cell49_candidate_status = (
    "CANDIDATES_READY_FOR_HUMAN_SELECTION"

    if cell49_shortlist

    else
    "NO_DEEP_SUBSTANTIVE_CANDIDATE_FOUND"
)


# ============================================================
# 13) Attach Candidate discovery to Retarget workspace
# ============================================================

cell49_retarget_workspace[
    "retarget_review"
][
    "replacement_status"
] = cell49_candidate_status


cell49_retarget_workspace[
    "retarget_candidate_discovery"
] = {
    "discovery_version":
        "deep_heading_retarget_discovery_v1",

    "canonical_source":
        str(
            cell49_canonical_path
        ),

    "source_type":
        "book",

    "required_stratum":
        "deep_heading_path",

    "required_task_family":
        cell48_next_task_family,

    "required_question_archetype":
        cell48_next_question_archetype,

    "current_edition_id":
        cell49_current_edition_id,

    "maximum_heading_depth":
        cell49_max_heading_depth,

    "minimum_heading_depth_used":
        cell49_depth_threshold,

    "used_document_id_count":
        len(
            cell49_used_document_ids
        ),

    "raw_candidate_count":
        len(
            cell49_candidate_rows
        ),

    "unique_candidate_count":
        len(
            cell49_ranked_candidates
        ),

    "shortlist_count":
        len(
            cell49_shortlist
        ),

    "status":
        cell49_candidate_status,

    "shortlist":
        copy.deepcopy(
            cell49_shortlist
        ),

    "external_lookup_used":
        False,
}


# ============================================================
# 14) Persist Retarget state
# ============================================================

cell49_persistent_registry_payload = copy.deepcopy(
    cell48_persisted_registry
)


cell49_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell49_retarget_workspace
)


cell49_persistent_registry_payload[
    "remaining_pilot_count"
] = 14


cell49_persistent_registry_payload[
    "continuation_state"
] = {
    "next_pilot_id":
        CELL49_PILOT_ID,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell48_remaining_pilot_ids
        ),

    "remaining_count":
        14,

    "continuation_state_sha256":
        cell48_continuation_state_sha256,

    "active_pilot_status":
        "BLOCKED_REQUIRES_RETARGET",

    "active_target_evidence_id":
        CELL49_OLD_TARGET_EVIDENCE_ID,
}


cell49_persistent_registry_payload[
    "retarget_candidate_discovery"
] = copy.deepcopy(
    cell49_retarget_workspace[
        "retarget_candidate_discovery"
    ]
)


cell49_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell49_persistent_registry_payload
    )
)


# ============================================================
# 15) Read-back validation
# ============================================================

cell49_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell49_persisted_active_case = (
    cell49_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell49_persistence_valid = all([
    cell49_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL49_PILOT_ID,

    cell49_persisted_active_case.get(
        "workflow_state"
    )
    ==
    "BLOCKED_REQUIRES_RETARGET",

    cell49_persisted_active_case.get(
        "retarget_review",
        {},
    ).get(
        "decision"
    )
    ==
    "RETARGET",

    cell49_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    14,
])


# ============================================================
# 16) Queue / safety
# ============================================================

cell49_queue_still_14 = all([
    len(
        cell48_remaining_pilot_ids
    )
    ==
    14,

    cell48_remaining_pilot_ids[
        0
    ]
    ==
    CELL49_PILOT_ID,
])


cell49_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell49_full_572_blocked = (
    CELL49_FULL_572_BLOCK_STATUS
    ==
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


# ============================================================
# 17) Final Gate
# ============================================================

cell49_gate = all([
    len(
        cell49_failures
    )
    ==
    0,

    cell49_old_target_heading_like,

    cell49_canonical_exists,

    isinstance(
        cell49_edition_records,
        list,
    ),

    len(
        cell49_edition_records
    )
    >
    0,

    cell49_persistence_valid,

    cell49_queue_still_14,

    cell49_base_seed_unchanged,

    cell49_full_572_blocked,
])


# ============================================================
# 18) Diagnostics
# ============================================================

print(
    "=== RETARGET REGISTERED + DEEP HEADING "
    "CANDIDATE DISCOVERY ==="
)


print(
    "\n=== CURRENT PILOT ==="
)

print(
    "Pilot:",
    CELL49_PILOT_ID,
)

print(
    "Golden Case:",
    CELL49_GOLDEN_CASE_ID,
)

print(
    "Task Draft:",
    CELL49_TASK_DRAFT_ID,
)

print(
    "Old Target:",
    CELL49_OLD_TARGET_EVIDENCE_ID,
)

print(
    "Stratum:",
    cell48_next_stratum,
)

print(
    "Task family:",
    cell48_next_task_family,
)

print(
    "Question archetype:",
    cell48_next_question_archetype,
)


print(
    "\n=== OLD TARGET AUDIT ==="
)

print(
    "Evidence:",
    repr(
        cell49_old_target_text
    ),
)

print(
    "Target part count:",
    len(
        cell49_old_target_parts
    ),
)

print(
    "Heading-like:",
    cell49_old_target_heading_like,
)

print(
    "Human Decision:",
    "RETARGET",
)

print(
    "Reason:",
    CELL49_RETARGET_REASON,
)


print(
    "\n=== CANONICAL SEARCH ==="
)

print(
    "Canonical source:",
    cell49_canonical_path,
)

print(
    "Edition count:",
    len(
        cell49_edition_records
    ),
)

print(
    "Current edition:",
    cell49_current_edition_id,
)

print(
    "Maximum heading depth:",
    cell49_max_heading_depth,
)

print(
    "Minimum depth used:",
    cell49_depth_threshold,
)

print(
    "Used document IDs excluded:",
    len(
        cell49_used_document_ids
    ),
)

print(
    "Raw candidates:",
    len(
        cell49_candidate_rows
    ),
)

print(
    "Unique candidates:",
    len(
        cell49_ranked_candidates
    ),
)

print(
    "Shortlist:",
    len(
        cell49_shortlist
    ),
)

print(
    "Status:",
    cell49_candidate_status,
)


print(
    "\n=== RETARGET CANDIDATE SHORTLIST ==="
)


for index, candidate in enumerate(
    cell49_shortlist,
    start=1,
):

    print(
        "\n"
        +
        "="
        *
        72
    )

    print(
        "CANDIDATE",
        index,
    )

    print(
        "Candidate ID:",
        candidate[
            "candidate_id"
        ],
    )

    print(
        "Edition:",
        candidate[
            "edition_id"
        ],
    )

    print(
        "Same edition as old target:",
        candidate[
            "same_edition_as_old_target"
        ],
    )

    print(
        "Heading depth:",
        candidate[
            "heading_depth"
        ],
    )

    print(
        "Heading path:"
    )

    print(
        " > ".join(
            candidate[
                "heading_path"
            ]
        )
    )

    print(
        "Paragraphs:",
        candidate[
            "paragraph_numbers"
        ],
    )

    print(
        "Characters:",
        candidate[
            "character_count"
        ],
    )

    print(
        "\nEvidence preview:"
    )


    preview = candidate[
        "combined_text"
    ]


    if len(
        preview
    ) > 700:

        preview = (
            preview[
                :700
            ]
            +
            " ..."
        )


    print(
        preview
    )


print(
    "\n=== WORKFLOW / PERSISTENCE ==="
)

print(
    "Workflow:",
    cell49_persisted_active_case.get(
        "workflow_state"
    ),
)

print(
    "Retarget Decision:",
    cell49_persisted_active_case.get(
        "retarget_review",
        {},
    ).get(
        "decision"
    ),
)

print(
    "Registry:",
    cell21_registry_path,
)

print(
    "Persistence valid:",
    cell49_persistence_valid,
)


print(
    "\n=== QUEUE / SAFETY ==="
)

print(
    "Queue still 14:",
    cell49_queue_still_14,
)

print(
    "Base Seed unchanged:",
    cell49_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell49_full_572_blocked,
)

print(
    "External lookup:",
    False,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell49_failures
)


print(
    "\nCODE CELL 49 GATE:",
    (
        "PASS"
        if cell49_gate
        else "FAIL"
    ),
)


assert cell49_gate, (
    "Retarget registration / deep-heading candidate discovery "
    "failed. Repair this SAME Code Cell 49 before continuing."
)

=== RETARGET REGISTERED + DEEP HEADING CANDIDATE DISCOVERY ===

=== CURRENT PILOT ===
Pilot: question-pilot:c9b2a6ecc853bf563a45af119621c405
Golden Case: golden-case:4be137bf669d25e1fcf7be0c84cc2cc8
Task Draft: task-draft:88fa7d52ac01c1b546ac59d70ef5d03c
Old Target: target-evidence:135c9f93d0c5695ce41421903c264783
Stratum: deep_heading_path
Task family: DEEP_HIERARCHICAL_LOCATOR_RETRIEVAL
Question archetype: STRUCTURE_AWARE_RETRIEVAL

=== OLD TARGET AUDIT ===
Evidence: 'چه گزینه\u200cهایی پیش\u200cرو دارم؟'
Target part count: 1
Heading-like: True
Human Decision: RETARGET
Reason: Current Target Evidence is heading-only and does not contain sufficient substantive content for a natural, answerable Golden QA case. Preserve deep hierarchical locator coverage while replacing the Target Evidence with a substantive passage from a deep heading path.

=== CANONICAL SEARCH ===
Canonical source: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/data/phase2/interim/book_inge

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 50 — General Borrowing Evidence Search for Retarget

Human Reviewer فرم سؤال زیر را مناسب دانست:

**در چه شرایطی قرض گرفتن کار درستی است؟**

اما این سؤال هنوز به‌عنوان Question نهایی ثبت نمی‌شود، زیرا Candidate قبلی صریحاً درباره **قرض کودکان** بود و تعمیم آن به حکم کلی قرض گرفتن بدون Evidence کافی مجاز نیست.

در این مرحله:

- سؤال پیشنهادی Human به‌عنوان `PREFERRED_QUESTION` و نه Evidence ثبت می‌شود؛
- Corpus کتاب‌های Canonical برای مطالب مربوط به قرض، وام، بدهی و بازپرداخت بررسی می‌شود؛
- اولویت با Evidence دارای `deep_heading_path` است؛
- متن‌هایی که حکم عمومی‌تری درباره قرض ارائه می‌کنند بالاتر رتبه می‌گیرند؛
- متن‌های صرفاً کودک‌محور مشخص می‌شوند؛
- Queue همچنان 14 باقی می‌ماند؛
- هنوز Replacement Target انتخاب نمی‌شود.

</div>

In [50]:
# Code Cell 50

cell50_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell49_gate") is not True:
    cell50_failures.append(
        "CELL49_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell49_canonical_payload"
    ),
    dict,
):
    cell50_failures.append(
        "CANONICAL_PAYLOAD_MISSING"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell50_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Human-preferred question
# ============================================================

CELL50_PREFERRED_QUESTION = (
    "در چه شرایطی قرض گرفتن کار درستی است؟"
)


CELL50_QUESTION_STATUS = (
    "PREFERRED_PENDING_EVIDENCE_VALIDATION"
)


CELL50_SEARCH_PURPOSE = (
    "Find source evidence that supports a general question "
    "about when borrowing can be appropriate, without "
    "generalizing child-specific evidence beyond the source."
)


# ============================================================
# 3) Search vocabulary
# ============================================================

cell50_primary_terms = [
    "قرض",
    "وام",
]


cell50_secondary_terms = [
    "بدهی",
    "بازپرداخت",
    "پس‌انداز معکوس",
    "پس انداز معکوس",
    "تسهیلات",
    "اقساط",
]


cell50_child_terms = [
    "کودک",
    "کودکان",
    "فرزند",
    "نوجوان",
    "نوجوانان",
]


cell50_general_terms = [
    "افراد",
    "فرد",
    "خانوار",
    "خانواده",
    "شخص",
    "مصرف کننده",
    "مصرف‌کننده",
]


# ============================================================
# 4) Normalization helper
# ============================================================

def cell50_norm(
    text,
):

    if not isinstance(
        text,
        str,
    ):
        return ""


    value = (
        text
        .replace(
            "ي",
            "ی",
        )
        .replace(
            "ك",
            "ک",
        )
        .replace(
            "\u200c",
            " ",
        )
    )


    value = re.sub(
        r"\s+",
        " ",
        value,
    )


    return value.strip()


# ============================================================
# 5) Flatten canonical paragraphs
# ============================================================

cell50_paragraph_rows = []


for edition in cell49_edition_records:

    if not isinstance(
        edition,
        dict,
    ):
        continue


    edition_id = edition.get(
        "edition_id"
    )


    paragraphs = edition.get(
        "paragraphs",
        []
    )


    if not isinstance(
        paragraphs,
        list,
    ):
        continue


    for paragraph in paragraphs:

        if not isinstance(
            paragraph,
            dict,
        ):
            continue


        text = cell50_norm(
            paragraph.get(
                "text"
            )
        )


        if not text:
            continue


        heading_path = (
            cell49_heading_path(
                paragraph
            )
        )


        paragraph_number = (
            cell49_paragraph_number(
                paragraph
            )
        )


        cell50_paragraph_rows.append({
            "edition_id":
                edition_id,

            "paragraph_id":
                paragraph.get(
                    "paragraph_id"
                ),

            "paragraph_number":
                paragraph_number,

            "heading_path":
                heading_path,

            "heading_depth":
                len(
                    heading_path
                ),

            "is_heading":
                paragraph.get(
                    "is_heading"
                )
                is True,

            "in_table":
                paragraph.get(
                    "in_table"
                )
                is True,

            "text":
                text,

            "raw_paragraph":
                paragraph,
        })


# ============================================================
# 6) Find keyword hits
# ============================================================

cell50_keyword_hits = []


for row in cell50_paragraph_rows:

    searchable = cell50_norm(
        " ".join(
            list(
                row[
                    "heading_path"
                ]
            )
            +
            [
                row[
                    "text"
                ]
            ]
        )
    )


    primary_hits = [
        term

        for term
        in cell50_primary_terms

        if term
        in searchable
    ]


    secondary_hits = [
        term

        for term
        in cell50_secondary_terms

        if term
        in searchable
    ]


    if not (
        primary_hits
        or
        secondary_hits
    ):
        continue


    child_hits = [
        term

        for term
        in cell50_child_terms

        if term
        in searchable
    ]


    general_hits = [
        term

        for term
        in cell50_general_terms

        if term
        in searchable
    ]


    cell50_keyword_hits.append({
        **row,

        "primary_hits":
            primary_hits,

        "secondary_hits":
            secondary_hits,

        "child_hits":
            child_hits,

        "general_hits":
            general_hits,
    })


# ============================================================
# 7) Build local context windows
# ============================================================

cell50_by_edition = {}


for row in cell50_paragraph_rows:

    cell50_by_edition.setdefault(
        row[
            "edition_id"
        ],
        [],
    ).append(
        row
    )


for edition_id in cell50_by_edition:

    cell50_by_edition[
        edition_id
    ] = sorted(
        cell50_by_edition[
            edition_id
        ],
        key=lambda row: (
            row[
                "paragraph_number"
            ]
            if
            row[
                "paragraph_number"
            ]
            is not None
            else
            10**12
        ),
    )


cell50_context_candidates = []


for hit in cell50_keyword_hits:

    edition_rows = cell50_by_edition.get(
        hit[
            "edition_id"
        ],
        [],
    )


    hit_number = hit[
        "paragraph_number"
    ]


    if hit_number is None:
        continue


    context_rows = [
        row

        for row
        in edition_rows

        if (
            row[
                "paragraph_number"
            ]
            is not None
            and
            hit_number - 2
            <=
            row[
                "paragraph_number"
            ]
            <=
            hit_number + 2
        )
    ]


    context_rows = [
        row

        for row
        in context_rows

        if not row[
            "is_heading"
        ]
    ]


    context_texts = [
        row[
            "text"
        ]

        for row
        in context_rows

        if len(
            row[
                "text"
            ]
        )
        >=
        20
    ]


    combined_text = "\n\n".join(
        context_texts
    ).strip()


    if len(
        combined_text
    ) < 80:
        continue


    normalized_context = (
        cell50_norm(
            combined_text
        )
    )


    child_hits = [
        term

        for term
        in cell50_child_terms

        if term
        in normalized_context
    ]


    general_hits = [
        term

        for term
        in cell50_general_terms

        if term
        in normalized_context
    ]


    primary_hits = [
        term

        for term
        in cell50_primary_terms

        if term
        in normalized_context
    ]


    secondary_hits = [
        term

        for term
        in cell50_secondary_terms

        if term
        in normalized_context
    ]


    heading_depth = hit[
        "heading_depth"
    ]


    deep_heading = (
        heading_depth
        >=
        3
    )


    child_only_signal = all([
        bool(
            child_hits
        ),

        not bool(
            general_hits
        ),
    ])


    generic_language_signal = any([
        "قرض گرفتن"
        in
        normalized_context,

        "قرض راه حل"
        in
        normalized_context,

        "قرض راه‌حل"
        in
        combined_text,

        "وام گرفتن"
        in
        normalized_context,

        "بدهی"
        in
        normalized_context,
    ])


    score = (
        (
            30
            if deep_heading
            else 0
        )
        +
        len(
            primary_hits
        )
        *
        12
        +
        len(
            secondary_hits
        )
        *
        4
        +
        (
            12
            if generic_language_signal
            else 0
        )
        +
        (
            8
            if general_hits
            else 0
        )
        -
        (
            18
            if child_only_signal
            else 0
        )
        +
        min(
            len(
                combined_text
            ),
            1000,
        )
        /
        100
    )


    identity_payload = {
        "edition_id":
            hit[
                "edition_id"
            ],

        "heading_path":
            list(
                hit[
                    "heading_path"
                ]
            ),

        "paragraph_ids": [
            row[
                "paragraph_id"
            ]

            for row
            in context_rows
        ],

        "hit_paragraph_id":
            hit[
                "paragraph_id"
            ],
    }


    candidate_id = (
        "borrowing-evidence-candidate:"
        +
        hashlib.sha256(
            cell10_canonical_json(
                identity_payload
            ).encode(
                "utf-8"
            )
        ).hexdigest()[
            :32
        ]
    )


    cell50_context_candidates.append({
        "candidate_id":
            candidate_id,

        "edition_id":
            hit[
                "edition_id"
            ],

        "heading_path":
            list(
                hit[
                    "heading_path"
                ]
            ),

        "heading_depth":
            heading_depth,

        "deep_heading":
            deep_heading,

        "hit_paragraph_id":
            hit[
                "paragraph_id"
            ],

        "hit_paragraph_number":
            hit_number,

        "context_paragraph_ids": [
            row[
                "paragraph_id"
            ]

            for row
            in context_rows
        ],

        "context_paragraph_numbers": [
            row[
                "paragraph_number"
            ]

            for row
            in context_rows
        ],

        "primary_hits":
            primary_hits,

        "secondary_hits":
            secondary_hits,

        "child_hits":
            child_hits,

        "general_hits":
            general_hits,

        "child_only_signal":
            child_only_signal,

        "generic_language_signal":
            generic_language_signal,

        "combined_text":
            combined_text,

        "score":
            round(
                score,
                4,
            ),
    })


# ============================================================
# 8) Deduplicate candidates
# ============================================================

cell50_candidate_by_id = {}


for candidate in cell50_context_candidates:

    candidate_id = candidate[
        "candidate_id"
    ]


    previous = (
        cell50_candidate_by_id.get(
            candidate_id
        )
    )


    if (
        previous is None
        or
        candidate[
            "score"
        ]
        >
        previous[
            "score"
        ]
    ):

        cell50_candidate_by_id[
            candidate_id
        ] = candidate


cell50_ranked_candidates = sorted(
    cell50_candidate_by_id.values(),
    key=lambda item: (
        -int(
            item[
                "deep_heading"
            ]
        ),
        int(
            item[
                "child_only_signal"
            ]
        ),
        -item[
            "score"
        ],
        item[
            "candidate_id"
        ],
    ),
)


cell50_shortlist = copy.deepcopy(
    cell50_ranked_candidates[
        :10
    ]
)


# ============================================================
# 9) Classify search result
# ============================================================

cell50_general_deep_candidates = [
    candidate

    for candidate
    in cell50_ranked_candidates

    if (
        candidate[
            "deep_heading"
        ]
        and
        not candidate[
            "child_only_signal"
        ]
    )
]


cell50_child_specific_deep_candidates = [
    candidate

    for candidate
    in cell50_ranked_candidates

    if (
        candidate[
            "deep_heading"
        ]
        and
        candidate[
            "child_only_signal"
        ]
    )
]


if cell50_general_deep_candidates:

    cell50_search_status = (
        "GENERAL_DEEP_BORROWING_EVIDENCE_FOUND"
    )


elif cell50_child_specific_deep_candidates:

    cell50_search_status = (
        "ONLY_CHILD_SPECIFIC_DEEP_BORROWING_EVIDENCE_FOUND"
    )


elif cell50_ranked_candidates:

    cell50_search_status = (
        "BORROWING_EVIDENCE_FOUND_BUT_NOT_DEEP"
    )


else:

    cell50_search_status = (
        "NO_BORROWING_EVIDENCE_FOUND"
    )


# ============================================================
# 10) Update Retarget workspace
# ============================================================

cell50_retarget_workspace = copy.deepcopy(
    cell49_persisted_active_case
)


cell50_retarget_workspace[
    "preferred_question"
] = {
    "question_text":
        CELL50_PREFERRED_QUESTION,

    "status":
        CELL50_QUESTION_STATUS,

    "human_preferred":
        True,

    "is_source_evidence":
        False,

    "reason":
        (
            "Question wording accepted by Human Reviewer, "
            "but final acceptance requires source evidence "
            "that supports a general borrowing formulation."
        ),
}


cell50_retarget_workspace[
    "borrowing_evidence_search"
] = {
    "search_version":
        "general_borrowing_deep_heading_search_v1",

    "purpose":
        CELL50_SEARCH_PURPOSE,

    "primary_terms":
        copy.deepcopy(
            cell50_primary_terms
        ),

    "secondary_terms":
        copy.deepcopy(
            cell50_secondary_terms
        ),

    "keyword_hit_count":
        len(
            cell50_keyword_hits
        ),

    "candidate_count":
        len(
            cell50_ranked_candidates
        ),

    "general_deep_candidate_count":
        len(
            cell50_general_deep_candidates
        ),

    "child_specific_deep_candidate_count":
        len(
            cell50_child_specific_deep_candidates
        ),

    "status":
        cell50_search_status,

    "shortlist":
        copy.deepcopy(
            cell50_shortlist
        ),

    "external_lookup_used":
        False,
}


# ============================================================
# 11) Persist diagnostic state
# ============================================================

cell50_persistent_registry_payload = copy.deepcopy(
    cell49_persisted_registry
)


cell50_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell50_retarget_workspace
)


cell50_persistent_registry_payload[
    "remaining_pilot_count"
] = 14


cell50_persistent_registry_payload[
    "continuation_state"
] = {
    "next_pilot_id":
        CELL49_PILOT_ID,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell48_remaining_pilot_ids
        ),

    "remaining_count":
        14,

    "continuation_state_sha256":
        cell48_continuation_state_sha256,

    "active_pilot_status":
        "BLOCKED_REQUIRES_RETARGET",

    "active_target_evidence_id":
        CELL49_OLD_TARGET_EVIDENCE_ID,

    "preferred_question_status":
        CELL50_QUESTION_STATUS,
}


cell50_persistent_registry_payload[
    "retarget_candidate_discovery"
] = {
    "pilot_case_id":
        CELL49_PILOT_ID,

    "old_target_evidence_id":
        CELL49_OLD_TARGET_EVIDENCE_ID,

    "preferred_question":
        CELL50_PREFERRED_QUESTION,

    "preferred_question_status":
        CELL50_QUESTION_STATUS,

    "search_status":
        cell50_search_status,

    "general_deep_candidate_count":
        len(
            cell50_general_deep_candidates
        ),

    "child_specific_deep_candidate_count":
        len(
            cell50_child_specific_deep_candidates
        ),

    "shortlist":
        copy.deepcopy(
            cell50_shortlist
        ),

    "external_lookup_used":
        False,
}


cell50_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell50_persistent_registry_payload
    )
)


# ============================================================
# 12) Read-back validation
# ============================================================

cell50_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell50_persisted_active_case = (
    cell50_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell50_persistence_valid = all([
    cell50_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL49_PILOT_ID,

    cell50_persisted_active_case.get(
        "workflow_state"
    )
    ==
    "BLOCKED_REQUIRES_RETARGET",

    cell50_persisted_active_case.get(
        "preferred_question",
        {},
    ).get(
        "question_text"
    )
    ==
    CELL50_PREFERRED_QUESTION,

    cell50_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    14,
])


# ============================================================
# 13) Safety
# ============================================================

cell50_queue_still_14 = all([
    len(
        cell48_remaining_pilot_ids
    )
    ==
    14,

    cell48_remaining_pilot_ids[
        0
    ]
    ==
    CELL49_PILOT_ID,
])


cell50_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell50_full_572_blocked = (
    CELL49_FULL_572_BLOCK_STATUS
    ==
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


# ============================================================
# 14) Final Gate
# ============================================================

cell50_gate = all([
    len(
        cell50_failures
    )
    ==
    0,

    isinstance(
        cell50_keyword_hits,
        list,
    ),

    isinstance(
        cell50_ranked_candidates,
        list,
    ),

    cell50_persistence_valid,

    cell50_queue_still_14,

    cell50_base_seed_unchanged,

    cell50_full_572_blocked,
])


# ============================================================
# 15) Diagnostics
# ============================================================

print(
    "=== GENERAL BORROWING EVIDENCE SEARCH ==="
)


print(
    "\n=== HUMAN-PREFERRED QUESTION ==="
)

print(
    CELL50_PREFERRED_QUESTION
)

print(
    "Question Status:",
    CELL50_QUESTION_STATUS,
)

print(
    "Final Question ACCEPT performed:",
    False,
)


print(
    "\n=== SEARCH SUMMARY ==="
)

print(
    "Keyword hits:",
    len(
        cell50_keyword_hits
    ),
)

print(
    "Candidate count:",
    len(
        cell50_ranked_candidates
    ),
)

print(
    "General deep candidates:",
    len(
        cell50_general_deep_candidates
    ),
)

print(
    "Child-specific deep candidates:",
    len(
        cell50_child_specific_deep_candidates
    ),
)

print(
    "Search Status:",
    cell50_search_status,
)


print(
    "\n=== BORROWING EVIDENCE SHORTLIST ==="
)


for index, candidate in enumerate(
    cell50_shortlist,
    start=1,
):

    print(
        "\n"
        +
        "="
        *
        72
    )

    print(
        "CANDIDATE",
        index,
    )

    print(
        "Candidate ID:",
        candidate[
            "candidate_id"
        ],
    )

    print(
        "Edition:",
        candidate[
            "edition_id"
        ],
    )

    print(
        "Heading depth:",
        candidate[
            "heading_depth"
        ],
    )

    print(
        "Deep heading:",
        candidate[
            "deep_heading"
        ],
    )

    print(
        "Heading path:",
        " > ".join(
            candidate[
                "heading_path"
            ]
        ),
    )

    print(
        "Paragraphs:",
        candidate[
            "context_paragraph_numbers"
        ],
    )

    print(
        "Primary hits:",
        candidate[
            "primary_hits"
        ],
    )

    print(
        "Secondary hits:",
        candidate[
            "secondary_hits"
        ],
    )

    print(
        "Child-specific terms:",
        candidate[
            "child_hits"
        ],
    )

    print(
        "General terms:",
        candidate[
            "general_hits"
        ],
    )

    print(
        "Child-only signal:",
        candidate[
            "child_only_signal"
        ],
    )

    print(
        "Score:",
        candidate[
            "score"
        ],
    )

    print(
        "\nEvidence:"
    )

    print(
        candidate[
            "combined_text"
        ]
    )


print(
    "\n=== WORKFLOW / PERSISTENCE ==="
)

print(
    "Workflow:",
    cell50_persisted_active_case.get(
        "workflow_state"
    ),
)

print(
    "Old Target remains active:",
    CELL49_OLD_TARGET_EVIDENCE_ID,
)

print(
    "Replacement Target selected:",
    False,
)

print(
    "Registry:",
    cell21_registry_path,
)

print(
    "Persistence valid:",
    cell50_persistence_valid,
)


print(
    "\n=== QUEUE / SAFETY ==="
)

print(
    "Queue still 14:",
    cell50_queue_still_14,
)

print(
    "Base Seed unchanged:",
    cell50_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell50_full_572_blocked,
)

print(
    "External lookup:",
    False,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell50_failures
)


print(
    "\nCODE CELL 50 GATE:",
    (
        "PASS"
        if cell50_gate
        else "FAIL"
    ),
)


assert cell50_gate, (
    "General borrowing evidence search failed. "
    "Repair this SAME Code Cell 50 before continuing."
)

=== GENERAL BORROWING EVIDENCE SEARCH ===

=== HUMAN-PREFERRED QUESTION ===
در چه شرایطی قرض گرفتن کار درستی است؟
Question Status: PREFERRED_PENDING_EVIDENCE_VALIDATION
Final Question ACCEPT performed: False

=== SEARCH SUMMARY ===
Keyword hits: 235
Candidate count: 209
General deep candidates: 2
Child-specific deep candidates: 0
Search Status: GENERAL_DEEP_BORROWING_EVIDENCE_FOUND

=== BORROWING EVIDENCE SHORTLIST ===

CANDIDATE 1
Candidate ID: borrowing-evidence-candidate:d2233dedf6d941f0e7b5699fc3863bb9
Edition: book:trainer-training-financial-literacy:fa-source
Heading depth: 3
Deep heading: True
Heading path: پول‌توجیبی > افزایش پول‌توجیبی > قرض گرفتن
Paragraphs: [368, 370, 372]
Primary hits: ['قرض']
Secondary hits: ['بازپرداخت', 'پس انداز معکوس']
Child-specific terms: ['کودک', 'کودکان', 'فرزند']
General terms: ['خانواده', 'شخص']
Child-only signal: False
Score: 80.0

Evidence:
از درخواست برای افزایش پول توجیبی بدون هدف مشخص به عنوان فرصتی برای تمرین کسب درآمد و البته تمرینی برای م

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 51 — Borrowing Section Hierarchy Audit

هدف این Cell فقط بررسی ساختار سلسله‌مراتبی بخش عمومی «قرض و بدهی» است.

بررسی می‌کنیم:

- Paragraphهای مربوط به Evidence عمومی قرض دقیقاً زیر چه Headingهایی قرار دارند؛
- `heading_level` واقعی Headingهای اطراف چیست؛
- آیا Heading والد دیگری قبل از `زمانی برای اندیشه > 2. قرض و بدهی` وجود دارد؛
- آیا امکان بازسازی معتبر یک `deep_heading_path` سه‌لایه وجود دارد یا خیر؛
- هیچ Heading یا Locator جدیدی ساخته نمی‌شود؛
- هنوز Replacement Target انتخاب نمی‌شود؛
- Queue همچنان 14 باقی می‌ماند.

</div>

In [51]:
# Code Cell 51

cell51_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell50_gate") is not True:
    cell51_failures.append(
        "CELL50_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell49_edition_records"
    ),
    list,
):
    cell51_failures.append(
        "EDITION_RECORDS_MISSING"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell51_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Target borrowing section
# ============================================================

CELL51_TARGET_EDITION_ID = (
    "book:financial-literacy-for-all:fa-source"
)

CELL51_EXPECTED_HEADING_PATH = [
    "زمانی برای اندیشه",
    "2. قرض و بدهی",
]

CELL51_CORE_PARAGRAPH_NUMBERS = [
    70,
    71,
    72,
]

CELL51_INSPECTION_START = 50
CELL51_INSPECTION_END = 90


# ============================================================
# 3) Resolve edition
# ============================================================

cell51_target_edition = None


for edition in cell49_edition_records:

    if not isinstance(
        edition,
        dict,
    ):
        continue

    if (
        edition.get(
            "edition_id"
        )
        ==
        CELL51_TARGET_EDITION_ID
    ):

        cell51_target_edition = edition
        break


if not isinstance(
    cell51_target_edition,
    dict,
):

    cell51_failures.append(
        "TARGET_EDITION_NOT_FOUND"
    )


cell51_paragraphs = (
    cell51_target_edition.get(
        "paragraphs",
        []
    )

    if isinstance(
        cell51_target_edition,
        dict,
    )

    else []
)


# ============================================================
# 4) Normalize paragraph rows
# ============================================================

cell51_rows = []


for paragraph in cell51_paragraphs:

    if not isinstance(
        paragraph,
        dict,
    ):
        continue


    paragraph_number = (
        cell49_paragraph_number(
            paragraph
        )
    )


    heading_path = (
        cell49_heading_path(
            paragraph
        )
    )


    text = paragraph.get(
        "text"
    )


    if not isinstance(
        text,
        str,
    ):

        text = ""


    cell51_rows.append({
        "paragraph_id":
            paragraph.get(
                "paragraph_id"
            ),

        "paragraph_number":
            paragraph_number,

        "is_heading":
            paragraph.get(
                "is_heading"
            )
            is True,

        "heading_level":
            paragraph.get(
                "heading_level"
            ),

        "heading_path":
            list(
                heading_path
            ),

        "heading_depth":
            len(
                heading_path
            ),

        "text":
            text.strip(),

        "in_table":
            paragraph.get(
                "in_table"
            )
            is True,
    })


cell51_rows = sorted(
    cell51_rows,
    key=lambda row: (
        row[
            "paragraph_number"
        ]
        if
        row[
            "paragraph_number"
        ]
        is not None
        else
        10**12
    ),
)


# ============================================================
# 5) Core borrowing paragraphs
# ============================================================

cell51_core_rows = [
    row

    for row
    in cell51_rows

    if row[
        "paragraph_number"
    ]
    in
    CELL51_CORE_PARAGRAPH_NUMBERS
]


cell51_core_paths = [
    row[
        "heading_path"
    ]

    for row
    in cell51_core_rows
]


cell51_core_path_exact = all([
    len(
        cell51_core_rows
    )
    ==
    len(
        CELL51_CORE_PARAGRAPH_NUMBERS
    ),

    all(
        path
        ==
        CELL51_EXPECTED_HEADING_PATH

        for path
        in cell51_core_paths
    ),
])


# ============================================================
# 6) Local structural window
# ============================================================

cell51_local_rows = [
    row

    for row
    in cell51_rows

    if (
        row[
            "paragraph_number"
        ]
        is not None
        and
        CELL51_INSPECTION_START
        <=
        row[
            "paragraph_number"
        ]
        <=
        CELL51_INSPECTION_END
    )
]


cell51_local_heading_rows = [
    row

    for row
    in cell51_local_rows

    if row[
        "is_heading"
    ]
]


# ============================================================
# 7) Find heading rows that define the target path
# ============================================================

cell51_target_heading_matches = []


for row in cell51_rows:

    if not row[
        "is_heading"
    ]:
        continue


    if row[
        "text"
    ] in CELL51_EXPECTED_HEADING_PATH:

        cell51_target_heading_matches.append(
            copy.deepcopy(
                row
            )
        )


# ============================================================
# 8) Find nearest preceding headings before paragraph 70
# ============================================================

cell51_preceding_heading_rows = [
    row

    for row
    in cell51_rows

    if (
        row[
            "is_heading"
        ]
        and
        row[
            "paragraph_number"
        ]
        is not None
        and
        row[
            "paragraph_number"
        ]
        <
        min(
            CELL51_CORE_PARAGRAPH_NUMBERS
        )
    )
]


cell51_nearest_preceding_headings = (
    cell51_preceding_heading_rows[
        -12:
    ]
)


# ============================================================
# 9) Inspect explicit hierarchy above target section
# ============================================================

cell51_target_heading_level_rows = [
    row

    for row
    in cell51_target_heading_matches

    if row[
        "heading_level"
    ]
    is not None
]


cell51_target_levels = [
    row[
        "heading_level"
    ]

    for row
    in cell51_target_heading_level_rows
]


cell51_possible_parent_candidates = []


for row in cell51_nearest_preceding_headings:

    if not row[
        "text"
    ]:
        continue


    if row[
        "text"
    ] in CELL51_EXPECTED_HEADING_PATH:
        continue


    cell51_possible_parent_candidates.append(
        copy.deepcopy(
            row
        )
    )


# ============================================================
# 10) Check whether any core paragraph already has depth >= 3
# ============================================================

cell51_existing_deep_path_present = any(
    row[
        "heading_depth"
    ]
    >=
    3

    for row
    in cell51_core_rows
)


# ============================================================
# 11) Look for deeper child headings inside borrowing section
# ============================================================

cell51_borrowing_section_rows = [
    row

    for row
    in cell51_rows

    if (
        row[
            "heading_path"
        ][
            :2
        ]
        ==
        CELL51_EXPECTED_HEADING_PATH

        if len(
            row[
                "heading_path"
            ]
        )
        >=
        2

        else False
    )
]


cell51_deeper_child_rows = [
    row

    for row
    in cell51_borrowing_section_rows

    if row[
        "heading_depth"
    ]
    >=
    3
]


cell51_deeper_child_headings = [
    row

    for row
    in cell51_deeper_child_rows

    if row[
        "is_heading"
    ]
]


# ============================================================
# 12) Conservative hierarchy conclusion
# ============================================================

cell51_verified_three_level_path = all([
    cell51_existing_deep_path_present,

    len(
        cell51_deeper_child_rows
    )
    >
    0,
])


if cell51_verified_three_level_path:

    cell51_hierarchy_status = (
        "VERIFIED_EXISTING_THREE_LEVEL_PATH"
    )


elif (
    cell51_core_path_exact
    and
    len(
        cell51_deeper_child_rows
    )
    ==
    0
):

    cell51_hierarchy_status = (
        "VERIFIED_TWO_LEVEL_PATH_ONLY"
    )


else:

    cell51_hierarchy_status = (
        "HIERARCHY_REQUIRES_MANUAL_INSPECTION"
    )


cell51_safe_to_promote_to_deep_heading = (
    cell51_hierarchy_status
    ==
    "VERIFIED_EXISTING_THREE_LEVEL_PATH"
)


# ============================================================
# 13) Update persistent Retarget diagnostic
# ============================================================

cell51_retarget_workspace = copy.deepcopy(
    cell50_persisted_active_case
)


cell51_retarget_workspace[
    "borrowing_hierarchy_audit"
] = {
    "audit_version":
        "borrowing_hierarchy_audit_v1",

    "edition_id":
        CELL51_TARGET_EDITION_ID,

    "preferred_question":
        CELL50_PREFERRED_QUESTION,

    "expected_heading_path":
        copy.deepcopy(
            CELL51_EXPECTED_HEADING_PATH
        ),

    "core_paragraph_numbers":
        copy.deepcopy(
            CELL51_CORE_PARAGRAPH_NUMBERS
        ),

    "core_heading_paths":
        copy.deepcopy(
            cell51_core_paths
        ),

    "core_path_exact":
        cell51_core_path_exact,

    "existing_deep_path_present":
        cell51_existing_deep_path_present,

    "deeper_child_row_count":
        len(
            cell51_deeper_child_rows
        ),

    "deeper_child_heading_count":
        len(
            cell51_deeper_child_headings
        ),

    "hierarchy_status":
        cell51_hierarchy_status,

    "safe_to_promote_to_deep_heading":
        cell51_safe_to_promote_to_deep_heading,

    "locator_fabrication_performed":
        False,

    "replacement_target_selected":
        False,
}


cell51_persistent_registry_payload = copy.deepcopy(
    cell50_persisted_registry
)


cell51_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell51_retarget_workspace
)


cell51_persistent_registry_payload[
    "remaining_pilot_count"
] = 14


cell51_persistent_registry_payload[
    "continuation_state"
] = {
    "next_pilot_id":
        CELL49_PILOT_ID,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell48_remaining_pilot_ids
        ),

    "remaining_count":
        14,

    "continuation_state_sha256":
        cell48_continuation_state_sha256,

    "active_pilot_status":
        "BLOCKED_REQUIRES_RETARGET",

    "active_target_evidence_id":
        CELL49_OLD_TARGET_EVIDENCE_ID,

    "preferred_question_status":
        CELL50_QUESTION_STATUS,

    "borrowing_hierarchy_status":
        cell51_hierarchy_status,
}


cell51_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell51_persistent_registry_payload
    )
)


# ============================================================
# 14) Read-back validation
# ============================================================

cell51_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell51_persisted_active_case = (
    cell51_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell51_persistence_valid = all([
    cell51_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL49_PILOT_ID,

    cell51_persisted_active_case.get(
        "workflow_state"
    )
    ==
    "BLOCKED_REQUIRES_RETARGET",

    cell51_persisted_active_case.get(
        "borrowing_hierarchy_audit",
        {},
    ).get(
        "hierarchy_status"
    )
    ==
    cell51_hierarchy_status,

    cell51_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    14,
])


# ============================================================
# 15) Queue / safety
# ============================================================

cell51_queue_still_14 = all([
    len(
        cell48_remaining_pilot_ids
    )
    ==
    14,

    cell48_remaining_pilot_ids[
        0
    ]
    ==
    CELL49_PILOT_ID,
])


cell51_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell51_full_572_blocked = (
    CELL49_FULL_572_BLOCK_STATUS
    ==
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


# ============================================================
# 16) Final Gate
# ============================================================

cell51_gate = all([
    len(
        cell51_failures
    )
    ==
    0,

    isinstance(
        cell51_target_edition,
        dict,
    ),

    len(
        cell51_core_rows
    )
    ==
    len(
        CELL51_CORE_PARAGRAPH_NUMBERS
    ),

    cell51_persistence_valid,

    cell51_queue_still_14,

    cell51_base_seed_unchanged,

    cell51_full_572_blocked,
])


# ============================================================
# 17) Diagnostics
# ============================================================

print(
    "=== BORROWING SECTION HIERARCHY AUDIT ==="
)


print(
    "\n=== PREFERRED QUESTION ==="
)

print(
    CELL50_PREFERRED_QUESTION
)


print(
    "\n=== CORE BORROWING PARAGRAPHS ==="
)


for row in cell51_core_rows:

    print(
        "\nParagraph:",
        row[
            "paragraph_number"
        ],
    )

    print(
        "Paragraph ID:",
        row[
            "paragraph_id"
        ],
    )

    print(
        "Heading depth:",
        row[
            "heading_depth"
        ],
    )

    print(
        "Heading path:",
        " > ".join(
            row[
                "heading_path"
            ]
        ),
    )

    print(
        "Text:",
        row[
            "text"
        ],
    )


print(
    "\n=== TARGET HEADING ROWS ==="
)


for row in cell51_target_heading_matches:

    print(
        "\nParagraph:",
        row[
            "paragraph_number"
        ],
    )

    print(
        "Heading text:",
        row[
            "text"
        ],
    )

    print(
        "Heading level:",
        row[
            "heading_level"
        ],
    )

    print(
        "Heading path:",
        " > ".join(
            row[
                "heading_path"
            ]
        ),
    )


print(
    "\n=== NEAREST PRECEDING HEADINGS ==="
)


for row in cell51_nearest_preceding_headings:

    print(
        (
            row[
                "paragraph_number"
            ],
            row[
                "heading_level"
            ],
            row[
                "text"
            ],
            " > ".join(
                row[
                    "heading_path"
                ]
            ),
        )
    )


print(
    "\n=== POSSIBLE HIGHER PARENT HEADINGS ==="
)


if cell51_possible_parent_candidates:

    for row in cell51_possible_parent_candidates:

        print(
            (
                row[
                    "paragraph_number"
                ],
                row[
                    "heading_level"
                ],
                row[
                    "text"
                ],
                " > ".join(
                    row[
                        "heading_path"
                    ]
                ),
            )
        )

else:

    print(
        "None"
    )


print(
    "\n=== DEEPER CHILD STRUCTURE INSIDE BORROWING SECTION ==="
)

print(
    "Deeper child rows:",
    len(
        cell51_deeper_child_rows
    ),
)

print(
    "Deeper child headings:",
    len(
        cell51_deeper_child_headings
    ),
)


for row in cell51_deeper_child_headings:

    print(
        (
            row[
                "paragraph_number"
            ],
            row[
                "heading_level"
            ],
            row[
                "text"
            ],
            " > ".join(
                row[
                    "heading_path"
                ]
            ),
        )
    )


print(
    "\n=== HIERARCHY CONCLUSION ==="
)

print(
    "Core path exact:",
    cell51_core_path_exact,
)

print(
    "Existing depth >= 3 on core evidence:",
    cell51_existing_deep_path_present,
)

print(
    "Hierarchy Status:",
    cell51_hierarchy_status,
)

print(
    "Safe to promote borrowing evidence "
    "to deep_heading_path:",
    cell51_safe_to_promote_to_deep_heading,
)

print(
    "Locator fabrication performed:",
    False,
)

print(
    "Replacement Target selected:",
    False,
)


print(
    "\n=== WORKFLOW / PERSISTENCE ==="
)

print(
    "Workflow:",
    cell51_persisted_active_case.get(
        "workflow_state"
    ),
)

print(
    "Queue still 14:",
    cell51_queue_still_14,
)

print(
    "Persistence valid:",
    cell51_persistence_valid,
)

print(
    "Base Seed unchanged:",
    cell51_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell51_full_572_blocked,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell51_failures
)


print(
    "\nCODE CELL 51 GATE:",
    (
        "PASS"
        if cell51_gate
        else "FAIL"
    ),
)


assert cell51_gate, (
    "Borrowing hierarchy audit failed. "
    "Repair this SAME Code Cell 51 before continuing."
)

=== BORROWING SECTION HIERARCHY AUDIT ===

=== PREFERRED QUESTION ===
در چه شرایطی قرض گرفتن کار درستی است؟

=== CORE BORROWING PARAGRAPHS ===

Paragraph: 70
Paragraph ID: book:financial-literacy-for-all:fa-source:p000070
Heading depth: 2
Heading path: زمانی برای اندیشه > 2. قرض و بدهی
Text: به علاوه، آنچه در نیکویی قرض آمده است برای دهنده آن است نه گیرنده. با این وصف، اگر آن هزینه واقعاً ضروری و فوری است و صندوق اضطراری یا پس﻿اندازمان هم کفاف نمی‌دهد، باید از کسی قرض یا از نهادی وام یا تسهیلات بگیریم. این بدترین کاری است که در مدیریت مالی انجام می‌شود.

Paragraph: 71
Paragraph ID: book:financial-literacy-for-all:fa-source:p000071
Heading depth: 2
Heading path: زمانی برای اندیشه > 2. قرض و بدهی
Text: اما پیش از هر کار، باید ظرفیت بدهی‌مان را تعیین کنیم؛ یعنی ببینیم ماهانه چقدر درآمد و چقدر هزینه داریم؟ بر این اساس تا چه میزان و در چه زمان‌هایی می‌توانیم بازپرداخت داشته باشیم؟ و چقدر طول می‌کشد تمام آن را پس دهیم؟ همچنین بررسی کنیم چه کسانی می‌توانند به ما قرض دهند و در صورت دریافت وام 

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 52 — Replacement Target Selection + Question ACCEPT + Expected Answer Draft

Human Reviewer، Candidate شماره 2 را برای Retarget انتخاب کرد:

**Heading Path:**  
`پول‌توجیبی > افزایش پول‌توجیبی > کسب درآمد`

و سؤال زیر را پذیرفت:

**نوجوان برای کسب درآمد از طریق یک کسب‌وکار باید با چه چیزهایی آشنا باشد و والدین چه نقشی دارند؟**

در این مرحله:

- Target قبلی Heading-only کنار گذاشته می‌شود ولی Lineage آن حفظ می‌شود؛
- Candidate منتخب به Replacement Target Evidence تبدیل می‌شود؛
- عمق واقعی Heading Path برابر 3 کنترل می‌شود؛
- سؤال با Human Decision برابر `ACCEPT` ثبت می‌شود؛
- Expected Answer فقط از Replacement Evidence ساخته می‌شود؛
- سؤال عمومی «در چه شرایطی قرض گرفتن کار درستی است؟» برای این Pilot استفاده نمی‌شود، چون Evidence عمومی آن فقط Heading Depth برابر 2 داشت؛
- Citation هنوز ساخته نمی‌شود؛
- Queue همچنان 14 باقی می‌ماند؛
- Registry دائمی Sync می‌شود.

</div>

In [52]:
# Code Cell 52

cell52_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell51_gate") is not True:
    cell52_failures.append(
        "CELL51_NOT_PASS"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell52_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Human-selected Candidate
# ============================================================

CELL52_SELECTED_CANDIDATE_ID = (
    "deep-retarget-candidate:"
    "df5877b705ec5932e085a6158cd4c48a"
)


CELL52_ACCEPTED_QUESTION = (
    "نوجوان برای کسب درآمد از طریق یک کسب‌وکار "
    "باید با چه چیزهایی آشنا باشد و والدین چه نقشی دارند؟"
)


CELL52_EXPECTED_ANSWER = (
    "نوجوان برای کسب درآمد از طریق یک کسب‌وکار باید با "
    "منابع مالی، انسانی، طبیعی و مواد اولیه آشنا باشد، "
    "فرایند تولید را بداند، تا حدی مشتری خود را بشناسد "
    "و فعالیت بازاریابی انجام دهد. در این فرایند، والدین "
    "نقش هیأت‌مدیره و فرزند نقش مدیرعامل را دارند تا از "
    "آسیب‌های احتمالی مالی و غیرمالی جلوگیری شود."
)


CELL52_FULL_572_BLOCK_STATUS = (
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


# ============================================================
# 3) Resolve selected Candidate
# ============================================================

cell52_selected_candidates = [
    candidate

    for candidate
    in cell49_ranked_candidates

    if (
        candidate.get(
            "candidate_id"
        )
        ==
        CELL52_SELECTED_CANDIDATE_ID
    )
]


cell52_selected_candidate = (
    copy.deepcopy(
        cell52_selected_candidates[
            0
        ]
    )

    if len(
        cell52_selected_candidates
    )
    ==
    1

    else None
)


cell52_candidate_unique = (
    len(
        cell52_selected_candidates
    )
    ==
    1
)


if not cell52_candidate_unique:

    cell52_failures.append(
        "SELECTED_CANDIDATE_NOT_UNIQUE"
    )


# ============================================================
# 4) Validate required Retarget coverage
# ============================================================

cell52_candidate_heading_path = (
    cell52_selected_candidate.get(
        "heading_path",
        [],
    )

    if isinstance(
        cell52_selected_candidate,
        dict,
    )

    else []
)


cell52_candidate_paragraph_numbers = (
    cell52_selected_candidate.get(
        "paragraph_numbers",
        [],
    )

    if isinstance(
        cell52_selected_candidate,
        dict,
    )

    else []
)


cell52_candidate_edition_id = (
    cell52_selected_candidate.get(
        "edition_id"
    )

    if isinstance(
        cell52_selected_candidate,
        dict,
    )

    else None
)


cell52_required_heading_path = [
    "پول‌توجیبی",
    "افزایش پول‌توجیبی",
    "کسب درآمد",
]


cell52_candidate_structure_valid = all([
    cell52_candidate_unique,

    cell52_candidate_heading_path
    ==
    cell52_required_heading_path,

    len(
        cell52_candidate_heading_path
    )
    ==
    3,

    cell52_selected_candidate.get(
        "heading_depth"
    )
    ==
    3,

    cell52_candidate_edition_id
    ==
    "book:trainer-training-financial-literacy:fa-source",

    cell52_candidate_paragraph_numbers
    ==
    [
        374
    ],
])


# ============================================================
# 5) Resolve canonical source paragraph
# ============================================================

cell52_canonical_paragraphs = []


for edition in cell49_edition_records:

    if not isinstance(
        edition,
        dict,
    ):
        continue


    if (
        edition.get(
            "edition_id"
        )
        !=
        cell52_candidate_edition_id
    ):
        continue


    for paragraph in edition.get(
        "paragraphs",
        []
    ):

        if not isinstance(
            paragraph,
            dict,
        ):
            continue


        paragraph_number = (
            cell49_paragraph_number(
                paragraph
            )
        )


        if paragraph_number in (
            cell52_candidate_paragraph_numbers
        ):

            cell52_canonical_paragraphs.append(
                copy.deepcopy(
                    paragraph
                )
            )


cell52_canonical_paragraphs = sorted(
    cell52_canonical_paragraphs,
    key=lambda paragraph: (
        cell49_paragraph_number(
            paragraph
        )
        if
        cell49_paragraph_number(
            paragraph
        )
        is not None
        else
        10**12
    ),
)


cell52_canonical_paragraphs_valid = all([
    len(
        cell52_canonical_paragraphs
    )
    ==
    1,

    cell49_paragraph_number(
        cell52_canonical_paragraphs[
            0
        ]
    )
    ==
    374,

    list(
        cell49_heading_path(
            cell52_canonical_paragraphs[
                0
            ]
        )
    )
    ==
    cell52_required_heading_path,

    cell52_canonical_paragraphs[
        0
    ].get(
        "is_heading"
    )
    is not True,
])


# ============================================================
# 6) Build Replacement Target Parts
# ============================================================

cell52_replacement_target_parts = []


for paragraph in cell52_canonical_paragraphs:

    text = paragraph.get(
        "text"
    )


    if not (
        isinstance(
            text,
            str,
        )
        and
        text.strip()
    ):
        continue


    text = text.strip()


    paragraph_id = paragraph.get(
        "paragraph_id"
    )


    paragraph_number = (
        cell49_paragraph_number(
            paragraph
        )
    )


    heading_path = list(
        cell49_heading_path(
            paragraph
        )
    )


    cell52_replacement_target_parts.append({
        "document_id":
            paragraph_id,

        "source_id":
            cell52_candidate_edition_id,

        "role":
            "DEEP_HEADING_PATH_MEMBER",

        "char_start":
            0,

        "char_end":
            len(
                text
            ),

        "char_count":
            len(
                text
            ),

        "text":
            text,

        "text_sha256":
            hashlib.sha256(
                text.encode(
                    "utf-8"
                )
            ).hexdigest(),

        "locator": {
            "paragraph_id":
                paragraph_id,

            "paragraph_number":
                paragraph_number,

            "heading_path":
                copy.deepcopy(
                    heading_path
                ),

            "heading_depth":
                len(
                    heading_path
                ),

            "edition_id":
                cell52_candidate_edition_id,
        },

        "origin":
            "CANONICAL_BOOK_INGESTION_V2",
    })


cell52_replacement_combined_text = (
    "\n\n".join(
        part[
            "text"
        ]

        for part
        in cell52_replacement_target_parts
    )
)


cell52_replacement_parts_valid = all([
    len(
        cell52_replacement_target_parts
    )
    ==
    1,

    bool(
        cell52_replacement_combined_text.strip()
    ),

    all(
        part.get(
            "locator",
            {},
        ).get(
            "heading_depth"
        )
        ==
        3

        for part
        in cell52_replacement_target_parts
    ),

    all(
        part.get(
            "locator",
            {},
        ).get(
            "heading_path"
        )
        ==
        cell52_required_heading_path

        for part
        in cell52_replacement_target_parts
    ),
])


# ============================================================
# 7) Deterministic Replacement Target ID
# ============================================================

cell52_replacement_target_identity = {
    "identity_version":
        "retarget_target_evidence_identity_v1",

    "pilot_case_id":
        CELL49_PILOT_ID,

    "old_target_evidence_id":
        CELL49_OLD_TARGET_EVIDENCE_ID,

    "selected_candidate_id":
        CELL52_SELECTED_CANDIDATE_ID,

    "edition_id":
        cell52_candidate_edition_id,

    "heading_path":
        copy.deepcopy(
            cell52_required_heading_path
        ),

    "target_part_projection": [
        {
            "document_id":
                part.get(
                    "document_id"
                ),

            "char_start":
                part.get(
                    "char_start"
                ),

            "char_end":
                part.get(
                    "char_end"
                ),

            "text_sha256":
                part.get(
                    "text_sha256"
                ),
        }

        for part
        in cell52_replacement_target_parts
    ],
}


cell52_replacement_target_evidence_id = (
    "target-evidence:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell52_replacement_target_identity
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


# ============================================================
# 8) Replacement Target record
# ============================================================

cell52_replacement_target_evidence = {
    "record_version":
        "replacement_target_evidence_v1",

    "target_evidence_id":
        cell52_replacement_target_evidence_id,

    "pilot_case_id":
        CELL49_PILOT_ID,

    "golden_case_id":
        CELL49_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL49_TASK_DRAFT_ID,

    "source_type":
        "book",

    "stratum":
        "deep_heading_path",

    "task_family":
        "DEEP_HIERARCHICAL_LOCATOR_RETRIEVAL",

    "question_archetype":
        "STRUCTURE_AWARE_RETRIEVAL",

    "selected_candidate_id":
        CELL52_SELECTED_CANDIDATE_ID,

    "heading_path":
        copy.deepcopy(
            cell52_required_heading_path
        ),

    "heading_depth":
        3,

    "target_parts":
        copy.deepcopy(
            cell52_replacement_target_parts
        ),

    "combined_target_text":
        cell52_replacement_combined_text,

    "retarget_lineage": {
        "old_target_evidence_id":
            CELL49_OLD_TARGET_EVIDENCE_ID,

        "old_target_human_decision":
            "RETARGET",

        "replacement_target_evidence_id":
            cell52_replacement_target_evidence_id,

        "replacement_human_decision":
            "ACCEPT",
    },

    "external_lookup_used":
        False,
}


# ============================================================
# 9) Build replacement Pilot packet for Engine
# ============================================================

def cell52_replace_target_reference(
    value,
):

    if isinstance(
        value,
        dict,
    ):

        output = {}


        for key, item in value.items():

            if key == "target_text_parts":

                output[
                    key
                ] = copy.deepcopy(
                    cell52_replacement_target_parts
                )


            elif (
                key
                ==
                "target_evidence_id"
            ):

                output[
                    key
                ] = (
                    cell52_replacement_target_evidence_id
                )


            else:

                output[
                    key
                ] = (
                    cell52_replace_target_reference(
                        item
                    )
                )


        return output


    if isinstance(
        value,
        list,
    ):

        return [
            cell52_replace_target_reference(
                item
            )

            for item
            in value
        ]


    if (
        isinstance(
            value,
            str,
        )
        and
        value
        ==
        CELL49_OLD_TARGET_EVIDENCE_ID
    ):

        return (
            cell52_replacement_target_evidence_id
        )


    return copy.deepcopy(
        value
    )


cell52_retargeted_pilot_packet = (
    cell52_replace_target_reference(
        cell48_next_pilot_packet
    )
)


cell52_packet_target_id = (
    cell13_exact_single_id(
        cell52_retargeted_pilot_packet,
        "target_evidence",
    )
)


cell52_packet_valid = (
    cell52_packet_target_id
    ==
    cell52_replacement_target_evidence_id
)


# ============================================================
# 10) Restart Engine for same Pilot with replacement Target
# ============================================================

cell52_question_pending_session = (
    cell13_build_session(
        cell52_retargeted_pilot_packet
    )
)


cell52_session_start_valid = all([
    cell52_question_pending_session.get(
        "current_state"
    )
    ==
    "QUESTION_DRAFT_PENDING",

    cell52_packet_valid,
])


# ============================================================
# 11) Evidence grounding for accepted Question / Answer
# ============================================================

def cell52_norm(
    text,
):

    value = (
        str(
            text
        )
        .replace(
            "ي",
            "ی",
        )
        .replace(
            "ك",
            "ک",
        )
        .replace(
            "\u200c",
            " ",
        )
    )


    value = re.sub(
        r"\s+",
        " ",
        value,
    )


    return value.strip()


cell52_source_surface = cell52_norm(
    cell52_replacement_combined_text
)


cell52_evidence_grounding = {
    "adolescents":
        (
            "نوجوانان"
            in
            cell52_source_surface
        ),

    "financial_resources":
        (
            "منابع مالی"
            in
            cell52_source_surface
        ),

    "human_resources":
        (
            "انسانی"
            in
            cell52_source_surface
        ),

    "natural_resources":
        (
            "طبیعی"
            in
            cell52_source_surface
        ),

    "raw_materials":
        (
            "مواد اولیه"
            in
            cell52_source_surface
        ),

    "production_process":
        (
            "فرایند تولید"
            in
            cell52_source_surface
        ),

    "customer":
        (
            "مشتری"
            in
            cell52_source_surface
        ),

    "marketing":
        (
            "بازاریابی"
            in
            cell52_source_surface
        ),

    "parents_board_role":
        (
            "والدین نقش هیأت مدیره"
            in
            cell52_source_surface
        ),

    "child_ceo_role":
        (
            "فرزند نقش مدیرعامل"
            in
            cell52_source_surface
        ),

    "risk_prevention":
        (
            "آسیب های احتمالی مالی و غیرمالی"
            in
            cell52_source_surface
        ),
}


cell52_evidence_grounding_pass = all(
    cell52_evidence_grounding.values()
)


# ============================================================
# 12) Question Draft ID
# ============================================================

cell52_question_identity_payload = {
    "identity_version":
        "question_draft_identity_v1",

    "pilot_case_id":
        CELL49_PILOT_ID,

    "target_evidence_id":
        cell52_replacement_target_evidence_id,

    "question_text":
        CELL52_ACCEPTED_QUESTION,

    "revision_number":
        1,
}


cell52_question_draft_id = (
    "question-draft:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell52_question_identity_payload
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


cell52_question_machine_pass = all([
    "نوجوان"
    in
    CELL52_ACCEPTED_QUESTION,

    "کسب درآمد"
    in
    CELL52_ACCEPTED_QUESTION,

    "کسب‌وکار"
    in
    CELL52_ACCEPTED_QUESTION,

    "والدین"
    in
    CELL52_ACCEPTED_QUESTION,

    cell52_evidence_grounding_pass,
])


cell52_question_record = {
    "record_version":
        "question_draft_record_v1",

    "question_draft_id":
        cell52_question_draft_id,

    "pilot_case_id":
        CELL49_PILOT_ID,

    "golden_case_id":
        CELL49_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL49_TASK_DRAFT_ID,

    "target_evidence_id":
        cell52_replacement_target_evidence_id,

    "question_text":
        CELL52_ACCEPTED_QUESTION,

    "revision_number":
        1,

    "machine_validation_status":
        (
            "PASS_PENDING_HUMAN_ADJUDICATION"
            if cell52_question_machine_pass
            else
            "FAIL_REQUIRES_REVISION"
        ),

    "human_adjudication_status":
        "PENDING",

    "external_knowledge_used":
        False,
}


# ============================================================
# 13) Register + Human ACCEPT Question
# ============================================================

cell52_question_review_session = (
    cell13_apply_event(
        cell52_question_pending_session,
        "QUESTION_DRAFT_REGISTERED",
        artifact=
            cell52_question_record,
    )
)


CELL52_QUESTION_HUMAN_NOTE = (
    "Human Reviewer accepted this question after the "
    "heading-only original Target was retargeted to the "
    "three-level 'کسب درآمد' source passage."
)


cell52_answer_pending_session = (
    cell13_apply_event(
        cell52_question_review_session,
        "QUESTION_ACCEPTED",
        human_note=
            CELL52_QUESTION_HUMAN_NOTE,
    )
)


cell52_question_accept_valid = all([
    cell52_question_review_session.get(
        "current_state"
    )
    ==
    "QUESTION_HUMAN_REVIEW_PENDING",

    cell52_answer_pending_session.get(
        "current_state"
    )
    ==
    "EXPECTED_ANSWER_DRAFT_PENDING",
])


# ============================================================
# 14) Expected Answer Draft ID
# ============================================================

cell52_answer_identity_payload = {
    "identity_version":
        "expected_answer_identity_v1",

    "pilot_case_id":
        CELL49_PILOT_ID,

    "question_draft_id":
        cell52_question_draft_id,

    "target_evidence_id":
        cell52_replacement_target_evidence_id,

    "answer_text":
        CELL52_EXPECTED_ANSWER,

    "revision_number":
        1,
}


cell52_expected_answer_draft_id = (
    "expected-answer-draft:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell52_answer_identity_payload
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


cell52_answer_machine_pass = all([
    cell52_evidence_grounding_pass,

    "منابع مالی"
    in
    CELL52_EXPECTED_ANSWER,

    "مواد اولیه"
    in
    CELL52_EXPECTED_ANSWER,

    "فرایند تولید"
    in
    CELL52_EXPECTED_ANSWER,

    "مشتری"
    in
    CELL52_EXPECTED_ANSWER,

    "بازاریابی"
    in
    CELL52_EXPECTED_ANSWER,

    "هیأت‌مدیره"
    in
    CELL52_EXPECTED_ANSWER,

    "مدیرعامل"
    in
    CELL52_EXPECTED_ANSWER,

    "مالی و غیرمالی"
    in
    CELL52_EXPECTED_ANSWER,
])


cell52_expected_answer_record = {
    "record_version":
        "expected_answer_draft_record_v1",

    "expected_answer_draft_id":
        cell52_expected_answer_draft_id,

    "pilot_case_id":
        CELL49_PILOT_ID,

    "golden_case_id":
        CELL49_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL49_TASK_DRAFT_ID,

    "question_draft_id":
        cell52_question_draft_id,

    "target_evidence_id":
        cell52_replacement_target_evidence_id,

    "expected_answer_text":
        CELL52_EXPECTED_ANSWER,

    "revision_number":
        1,

    "machine_validation_status":
        (
            "PASS_PENDING_HUMAN_ADJUDICATION"
            if cell52_answer_machine_pass
            else
            "FAIL_REQUIRES_REVISION"
        ),

    "human_adjudication_status":
        "PENDING",

    "external_knowledge_used":
        False,
}


# ============================================================
# 15) Register Expected Answer
# ============================================================

cell52_current_session = (
    cell13_apply_event(
        cell52_answer_pending_session,
        "EXPECTED_ANSWER_REGISTERED",
        artifact=
            cell52_expected_answer_record,
    )
)


cell52_answer_registered_valid = (
    cell52_current_session.get(
        "current_state"
    )
    ==
    "EXPECTED_ANSWER_HUMAN_REVIEW_PENDING"
)


# ============================================================
# 16) Build active Human Review case
# ============================================================

cell52_active_human_review_case = copy.deepcopy(
    cell51_persisted_active_case
)


cell52_active_human_review_case[
    "status"
] = "ACTIVE_HUMAN_REVIEW"


cell52_active_human_review_case[
    "target_evidence_id"
] = (
    cell52_replacement_target_evidence_id
)


cell52_active_human_review_case[
    "target_evidence"
] = copy.deepcopy(
    cell52_replacement_target_evidence
)


cell52_active_human_review_case[
    "source_evidence"
] = copy.deepcopy(
    cell52_replacement_target_parts
)


cell52_active_human_review_case[
    "combined_target_text"
] = (
    cell52_replacement_combined_text
)


cell52_active_human_review_case[
    "workflow_state"
] = (
    "EXPECTED_ANSWER_HUMAN_REVIEW_PENDING"
)


cell52_active_human_review_case[
    "retarget_review"
][
    "replacement_target_evidence_id"
] = cell52_replacement_target_evidence_id


cell52_active_human_review_case[
    "retarget_review"
][
    "replacement_status"
] = "ACCEPT"


cell52_active_human_review_case[
    "retarget_review"
][
    "selected_candidate_id"
] = CELL52_SELECTED_CANDIDATE_ID


cell52_active_human_review_case[
    "retarget_review"
][
    "selected_heading_path"
] = copy.deepcopy(
    cell52_required_heading_path
)


cell52_active_human_review_case[
    "retarget_review"
][
    "selected_heading_depth"
] = 3


# Preserve the useful borrowing question, but do not use it
# as evidence or as the active Question for this Pilot.
cell52_active_human_review_case[
    "deferred_question_candidate"
] = {
    "question_text":
        CELL50_PREFERRED_QUESTION,

    "status":
        "DEFERRED_NOT_USED_FOR_THIS_PILOT",

    "reason":
        (
            "General borrowing evidence was verified at "
            "heading depth 2, while this Pilot requires "
            "deep_heading_path coverage."
        ),

    "is_source_evidence":
        False,
}


cell52_active_human_review_case[
    "question_review"
] = {
    "question_draft_id":
        cell52_question_draft_id,

    "question_text":
        CELL52_ACCEPTED_QUESTION,

    "machine_validation_status":
        cell52_question_record[
            "machine_validation_status"
        ],

    "human_decision":
        "ACCEPT",

    "human_note":
        CELL52_QUESTION_HUMAN_NOTE,
}


cell52_active_human_review_case[
    "expected_answer_review"
] = {
    "expected_answer_draft_id":
        cell52_expected_answer_draft_id,

    "expected_answer_text":
        CELL52_EXPECTED_ANSWER,

    "machine_validation_status":
        cell52_expected_answer_record[
            "machine_validation_status"
        ],

    "human_decision":
        "PENDING",
}


cell52_active_human_review_case[
    "citation_review"
] = None


cell52_active_human_review_case[
    "author_final_review"
] = {
    "status":
        "NOT_STARTED",

    "decision":
        None,

    "notes":
        None,
}


# ============================================================
# 17) Persist Registry
# ============================================================

cell52_persistent_registry_payload = copy.deepcopy(
    cell51_persisted_registry
)


cell52_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell52_active_human_review_case
)


cell52_persistent_registry_payload[
    "remaining_pilot_count"
] = 14


cell52_persistent_registry_payload[
    "continuation_state"
] = {
    "next_pilot_id":
        CELL49_PILOT_ID,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell48_remaining_pilot_ids
        ),

    "remaining_count":
        14,

    "continuation_state_sha256":
        cell48_continuation_state_sha256,

    "active_pilot_status":
        "EXPECTED_ANSWER_HUMAN_REVIEW_PENDING",

    "active_target_evidence_id":
        cell52_replacement_target_evidence_id,

    "retarget_status":
        "REPLACEMENT_ACCEPTED",
}


cell52_persistent_registry_payload[
    "retarget_candidate_discovery"
] = {
    "pilot_case_id":
        CELL49_PILOT_ID,

    "status":
        "REPLACEMENT_TARGET_SELECTED",

    "old_target_evidence_id":
        CELL49_OLD_TARGET_EVIDENCE_ID,

    "selected_candidate_id":
        CELL52_SELECTED_CANDIDATE_ID,

    "replacement_target_evidence_id":
        cell52_replacement_target_evidence_id,

    "heading_path":
        copy.deepcopy(
            cell52_required_heading_path
        ),

    "heading_depth":
        3,

    "human_decision":
        "ACCEPT",

    "external_lookup_used":
        False,
}


cell52_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell52_persistent_registry_payload
    )
)


# ============================================================
# 18) Read-back validation
# ============================================================

cell52_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell52_persisted_active_case = (
    cell52_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell52_persistence_valid = all([
    cell52_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL49_PILOT_ID,

    cell52_persisted_active_case.get(
        "target_evidence_id"
    )
    ==
    cell52_replacement_target_evidence_id,

    cell52_persisted_active_case.get(
        "retarget_review",
        {},
    ).get(
        "replacement_status"
    )
    ==
    "ACCEPT",

    cell52_persisted_active_case.get(
        "question_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell52_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "PENDING",

    cell52_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    14,
])


# ============================================================
# 19) Queue / safety
# ============================================================

cell52_queue_still_14 = all([
    len(
        cell48_remaining_pilot_ids
    )
    ==
    14,

    cell48_remaining_pilot_ids[
        0
    ]
    ==
    CELL49_PILOT_ID,
])


cell52_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell52_full_572_blocked = (
    CELL52_FULL_572_BLOCK_STATUS
    ==
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


# ============================================================
# 20) Final Gate
# ============================================================

cell52_gate = all([
    len(
        cell52_failures
    )
    ==
    0,

    cell52_candidate_structure_valid,

    cell52_canonical_paragraphs_valid,

    cell52_replacement_parts_valid,

    cell52_packet_valid,

    cell52_session_start_valid,

    cell52_evidence_grounding_pass,

    cell52_question_machine_pass,

    cell52_question_accept_valid,

    cell52_answer_machine_pass,

    cell52_answer_registered_valid,

    cell52_persistence_valid,

    cell52_queue_still_14,

    cell52_base_seed_unchanged,

    cell52_full_572_blocked,
])


# ============================================================
# 21) Diagnostics
# ============================================================

print(
    "=== REPLACEMENT TARGET SELECTED + QUESTION ACCEPTED "
    "+ EXPECTED ANSWER DRAFT ==="
)


print(
    "\n=== RETARGET LINEAGE ==="
)

print(
    "Pilot:",
    CELL49_PILOT_ID,
)

print(
    "Old Target:",
    CELL49_OLD_TARGET_EVIDENCE_ID,
)

print(
    "Selected Candidate:",
    CELL52_SELECTED_CANDIDATE_ID,
)

print(
    "Replacement Target:",
    cell52_replacement_target_evidence_id,
)

print(
    "Heading Path:",
    " > ".join(
        cell52_required_heading_path
    ),
)

print(
    "Heading Depth:",
    3,
)

print(
    "Retarget Human Decision:",
    "ACCEPT",
)


print(
    "\n=== REPLACEMENT SOURCE EVIDENCE ==="
)

print(
    cell52_replacement_combined_text
)


print(
    "\n=== ACCEPTED QUESTION ==="
)

print(
    CELL52_ACCEPTED_QUESTION
)

print(
    "Question Draft ID:",
    cell52_question_draft_id,
)

print(
    "Question Machine PASS:",
    cell52_question_machine_pass,
)


print(
    "\n=== EXPECTED ANSWER DRAFT ==="
)

print(
    CELL52_EXPECTED_ANSWER
)

print(
    "Expected Answer Draft ID:",
    cell52_expected_answer_draft_id,
)

print(
    "Machine PASS:",
    cell52_answer_machine_pass,
)


print(
    "\n=== EVIDENCE GROUNDING ==="
)

for name, status in (
    cell52_evidence_grounding.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== DEFERRED BORROWING QUESTION ==="
)

print(
    CELL50_PREFERRED_QUESTION
)

print(
    "Used for this Pilot:",
    False,
)

print(
    "Reason:",
    (
        "General evidence depth = 2; "
        "this Pilot requires depth = 3."
    ),
)


print(
    "\n=== WORKFLOW ==="
)

print(
    "Current State:",
    cell52_current_session.get(
        "current_state"
    ),
)

print(
    "Retarget:",
    "ACCEPT",
)

print(
    "Question Decision:",
    "ACCEPT",
)

print(
    "Expected Answer Decision:",
    "PENDING",
)

print(
    "Citation:",
    None,
)


print(
    "\n=== PERSISTENCE ==="
)

print(
    "Registry:",
    cell21_registry_path,
)

print(
    "Persisted Replacement Target:",
    cell52_persisted_active_case.get(
        "target_evidence_id"
    ),
)

print(
    "Persisted Question Decision:",
    cell52_persisted_active_case.get(
        "question_review",
        {},
    ).get(
        "human_decision"
    ),
)

print(
    "Persisted Answer Decision:",
    cell52_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    ),
)

print(
    "Persistence valid:",
    cell52_persistence_valid,
)


print(
    "\n=== QUEUE / SAFETY ==="
)

print(
    "Queue still 14:",
    cell52_queue_still_14,
)

print(
    "Base Seed unchanged:",
    cell52_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell52_full_572_blocked,
)

print(
    "External lookup:",
    False,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell52_failures
)


print(
    "\nCODE CELL 52 GATE:",
    (
        "PASS"
        if cell52_gate
        else "FAIL"
    ),
)


assert cell52_gate, (
    "Replacement Target selection / Question ACCEPT / "
    "Expected Answer Draft / Registry Sync failed. "
    "Repair this SAME Code Cell 52 before continuing."
)

=== REPLACEMENT TARGET SELECTED + QUESTION ACCEPTED + EXPECTED ANSWER DRAFT ===

=== RETARGET LINEAGE ===
Pilot: question-pilot:c9b2a6ecc853bf563a45af119621c405
Old Target: target-evidence:135c9f93d0c5695ce41421903c264783
Selected Candidate: deep-retarget-candidate:df5877b705ec5932e085a6158cd4c48a
Replacement Target: target-evidence:3fa4a04b2c148b45d3cf6f21e943dcb8
Heading Path: پول‌توجیبی > افزایش پول‌توجیبی > کسب درآمد
Heading Depth: 3
Retarget Human Decision: ACCEPT

=== REPLACEMENT SOURCE EVIDENCE ===
نوجوانان می‌توانند بر اساس آگاهی از ساختار مالی اداره یک کسب و کار  به کسب درآمد بپردازند. آنها باید از منابع مالی، انسانی، طبیعی و مواد اولیه آگاه باشند، فرایند تولید را بدانند و کم و بیش بتوانند مشتری خود را بشناسند و فعالیت بازاریابی انجام دهند. دراینجا نیز والدین نقش هیأت مدیره و فرزند نقش مدیرعامل را خواهد داشت. تا از آسیب‌های احتمالی مالی و غیرمالی جلوگیری شود.

=== ACCEPTED QUESTION ===
نوجوان برای کسب درآمد از طریق یک کسب‌وکار باید با چه چیزهایی آشنا باشد و والدین چه نقشی دارن

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 53 — Expected Answer ACCEPT + Retarget Citation Draft + Registry Sync

Expected Answer این Pilot توسط Human Reviewer پذیرفته شد.

در این مرحله:

- Expected Answer با تصمیم `ACCEPT` ثبت می‌شود؛
- Citation فقط از Replacement Target Evidence منتخب ساخته می‌شود؛
- Heading Path سه‌لایه و Locator پاراگراف Canonical حفظ می‌شود؛
- Target قدیمی Heading-only در Citation استفاده نمی‌شود؛
- Retarget Lineage حفظ می‌شود؛
- Citation وارد Human Review می‌شود؛
- Queue همچنان 14 باقی می‌ماند؛
- Registry دائمی Sync می‌شود؛
- Full 572 Generation همچنان Blocked است.

</div>

In [53]:
# Code Cell 53

cell53_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell52_gate") is not True:
    cell53_failures.append(
        "CELL52_NOT_PASS"
    )


if (
    globals().get(
        "cell52_current_session",
        {},
    ).get(
        "current_state"
    )
    !=
    "EXPECTED_ANSWER_HUMAN_REVIEW_PENDING"
):
    cell53_failures.append(
        "EXPECTED_ANSWER_NOT_AT_HUMAN_REVIEW_GATE"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell53_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Current identities
# ============================================================

CELL53_PILOT_ID = (
    CELL49_PILOT_ID
)

CELL53_GOLDEN_CASE_ID = (
    CELL49_GOLDEN_CASE_ID
)

CELL53_TASK_DRAFT_ID = (
    CELL49_TASK_DRAFT_ID
)

CELL53_OLD_TARGET_EVIDENCE_ID = (
    CELL49_OLD_TARGET_EVIDENCE_ID
)

CELL53_TARGET_EVIDENCE_ID = (
    cell52_replacement_target_evidence_id
)

CELL53_QUESTION_DRAFT_ID = (
    cell52_question_draft_id
)

CELL53_EXPECTED_ANSWER_DRAFT_ID = (
    cell52_expected_answer_draft_id
)

CELL53_FULL_572_BLOCK_STATUS = (
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


# ============================================================
# 3) Human ACCEPT for Expected Answer
# ============================================================

CELL53_HUMAN_ANSWER_NOTE = (
    "Human Reviewer accepted the Expected Answer against the "
    "selected three-level replacement evidence for adolescent "
    "business-income learning."
)


cell53_answer_human_review = {
    "review_record_version":
        "human_expected_answer_review_record_v1",

    "pilot_case_id":
        CELL53_PILOT_ID,

    "question_draft_id":
        CELL53_QUESTION_DRAFT_ID,

    "expected_answer_draft_id":
        CELL53_EXPECTED_ANSWER_DRAFT_ID,

    "target_evidence_id":
        CELL53_TARGET_EVIDENCE_ID,

    "human_decision":
        "ACCEPT",

    "human_note":
        CELL53_HUMAN_ANSWER_NOTE,

    "human_context_used_as_evidence":
        False,

    "external_knowledge_used":
        False,
}


# ============================================================
# 4) Apply Expected Answer ACCEPT
# ============================================================

cell53_citation_pending_session = (
    cell13_apply_event(
        cell52_current_session,
        "EXPECTED_ANSWER_ACCEPTED",
        human_note=
            CELL53_HUMAN_ANSWER_NOTE,
    )
)


cell53_answer_accept_valid = all([
    cell53_citation_pending_session.get(
        "current_state"
    )
    ==
    "CITATION_MAPPING_PENDING",

    cell53_citation_pending_session.get(
        "queue_advance_performed"
    )
    is False,

    cell53_citation_pending_session.get(
        "human_decisions",
        [],
    )[
        -1
    ].get(
        "event"
    )
    ==
    "EXPECTED_ANSWER_ACCEPTED",
])


# ============================================================
# 5) Read Replacement Target Parts
# ============================================================

cell53_target_parts = copy.deepcopy(
    cell52_replacement_target_parts
)


cell53_target_parts_valid = all([
    isinstance(
        cell53_target_parts,
        list,
    ),

    len(
        cell53_target_parts
    )
    ==
    1,

    cell53_target_parts[
        0
    ].get(
        "document_id"
    )
    ==
    "book:trainer-training-financial-literacy:fa-source:p000374",

    cell53_target_parts[
        0
    ].get(
        "locator",
        {},
    ).get(
        "heading_depth"
    )
    ==
    3,

    cell53_target_parts[
        0
    ].get(
        "locator",
        {},
    ).get(
        "heading_path"
    )
    ==
    cell52_required_heading_path,
])


# ============================================================
# 6) Build Citation entries
# ============================================================

cell53_citation_entries = []


for index, part in enumerate(
    cell53_target_parts
):

    if not isinstance(
        part,
        dict,
    ):
        continue


    cell53_citation_entries.append({
        "citation_index":
            index,

        "target_evidence_id":
            CELL53_TARGET_EVIDENCE_ID,

        "document_id":
            part.get(
                "document_id"
            ),

        "source_id":
            part.get(
                "source_id"
            ),

        "role":
            part.get(
                "role"
            ),

        "char_start":
            part.get(
                "char_start"
            ),

        "char_end":
            part.get(
                "char_end"
            ),

        "char_count":
            part.get(
                "char_count"
            ),

        "locator":
            copy.deepcopy(
                part.get(
                    "locator",
                    {},
                )
            ),

        "locator_origin":
            "COPIED_FROM_REPLACEMENT_TARGET_EVIDENCE",

        "external_locator_lookup_used":
            False,
    })


# ============================================================
# 7) Exact projection validation
# ============================================================

cell53_source_projection = [
    {
        "document_id":
            part.get(
                "document_id"
            ),

        "source_id":
            part.get(
                "source_id"
            ),

        "role":
            part.get(
                "role"
            ),

        "char_start":
            part.get(
                "char_start"
            ),

        "char_end":
            part.get(
                "char_end"
            ),

        "locator":
            copy.deepcopy(
                part.get(
                    "locator",
                    {},
                )
            ),
    }

    for part
    in cell53_target_parts
]


cell53_citation_projection = [
    {
        "document_id":
            entry.get(
                "document_id"
            ),

        "source_id":
            entry.get(
                "source_id"
            ),

        "role":
            entry.get(
                "role"
            ),

        "char_start":
            entry.get(
                "char_start"
            ),

        "char_end":
            entry.get(
                "char_end"
            ),

        "locator":
            copy.deepcopy(
                entry.get(
                    "locator",
                    {},
                )
            ),
    }

    for entry
    in cell53_citation_entries
]


cell53_no_locator_fabrication = (
    cell53_source_projection
    ==
    cell53_citation_projection
)


# ============================================================
# 8) Citation entry validation
# ============================================================

cell53_validation_rows = []


for entry in cell53_citation_entries:

    locator = (
        entry.get(
            "locator"
        )
        or
        {}
    )


    row_valid = all([
        entry.get(
            "target_evidence_id"
        )
        ==
        CELL53_TARGET_EVIDENCE_ID,

        entry.get(
            "document_id"
        )
        ==
        "book:trainer-training-financial-literacy:fa-source:p000374",

        entry.get(
            "role"
        )
        ==
        "DEEP_HEADING_PATH_MEMBER",

        entry.get(
            "char_start"
        )
        ==
        0,

        isinstance(
            entry.get(
                "char_end"
            ),
            int,
        ),

        entry.get(
            "char_end"
        )
        >
        0,

        locator.get(
            "paragraph_id"
        )
        ==
        "book:trainer-training-financial-literacy:fa-source:p000374",

        locator.get(
            "paragraph_number"
        )
        ==
        374,

        locator.get(
            "heading_depth"
        )
        ==
        3,

        locator.get(
            "heading_path"
        )
        ==
        cell52_required_heading_path,

        locator.get(
            "edition_id"
        )
        ==
        "book:trainer-training-financial-literacy:fa-source",

        entry.get(
            "locator_origin"
        )
        ==
        "COPIED_FROM_REPLACEMENT_TARGET_EVIDENCE",

        entry.get(
            "external_locator_lookup_used"
        )
        is False,
    ])


    cell53_validation_rows.append({
        "citation_index":
            entry.get(
                "citation_index"
            ),

        "document_id":
            entry.get(
                "document_id"
            ),

        "paragraph_number":
            locator.get(
                "paragraph_number"
            ),

        "heading_depth":
            locator.get(
                "heading_depth"
            ),

        "valid":
            row_valid,
    })


cell53_all_entries_valid = all([
    len(
        cell53_citation_entries
    )
    ==
    1,

    all(
        row.get(
            "valid"
        )
        is True

        for row
        in cell53_validation_rows
    ),
])


# ============================================================
# 9) Deterministic Citation Mapping ID
# ============================================================

cell53_citation_identity_payload = {
    "identity_version":
        "citation_mapping_identity_v1",

    "pilot_case_id":
        CELL53_PILOT_ID,

    "question_draft_id":
        CELL53_QUESTION_DRAFT_ID,

    "expected_answer_draft_id":
        CELL53_EXPECTED_ANSWER_DRAFT_ID,

    "target_evidence_id":
        CELL53_TARGET_EVIDENCE_ID,

    "citation_projection":
        copy.deepcopy(
            cell53_citation_projection
        ),
}


cell53_citation_mapping_id = (
    "citation-mapping:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell53_citation_identity_payload
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


# ============================================================
# 10) Citation Machine Validation
# ============================================================

cell53_citation_machine_validation = {
    "exactly_one_replacement_target_part":
        (
            len(
                cell53_target_parts
            )
            ==
            1
        ),

    "exactly_one_citation_entry":
        (
            len(
                cell53_citation_entries
            )
            ==
            1
        ),

    "all_entries_valid":
        cell53_all_entries_valid,

    "locator_fabrication_absent":
        cell53_no_locator_fabrication,

    "replacement_target_identity_match":
        all(
            entry.get(
                "target_evidence_id"
            )
            ==
            CELL53_TARGET_EVIDENCE_ID

            for entry
            in cell53_citation_entries
        ),

    "old_target_not_cited":
        all(
            entry.get(
                "target_evidence_id"
            )
            !=
            CELL53_OLD_TARGET_EVIDENCE_ID

            for entry
            in cell53_citation_entries
        ),

    "heading_depth_three":
        all(
            entry.get(
                "locator",
                {},
            ).get(
                "heading_depth"
            )
            ==
            3

            for entry
            in cell53_citation_entries
        ),

    "heading_path_exact":
        all(
            entry.get(
                "locator",
                {},
            ).get(
                "heading_path"
            )
            ==
            cell52_required_heading_path

            for entry
            in cell53_citation_entries
        ),

    "external_locator_lookup_used":
        False,
}


cell53_citation_machine_pass = all([
    cell53_citation_machine_validation[
        "exactly_one_replacement_target_part"
    ],

    cell53_citation_machine_validation[
        "exactly_one_citation_entry"
    ],

    cell53_citation_machine_validation[
        "all_entries_valid"
    ],

    cell53_citation_machine_validation[
        "locator_fabrication_absent"
    ],

    cell53_citation_machine_validation[
        "replacement_target_identity_match"
    ],

    cell53_citation_machine_validation[
        "old_target_not_cited"
    ],

    cell53_citation_machine_validation[
        "heading_depth_three"
    ],

    cell53_citation_machine_validation[
        "heading_path_exact"
    ],
])


# ============================================================
# 11) Citation Mapping Record
# ============================================================

cell53_citation_mapping_record = {
    "record_version":
        "citation_mapping_record_v1",

    "citation_mapping_id":
        cell53_citation_mapping_id,

    "pilot_case_id":
        CELL53_PILOT_ID,

    "golden_case_id":
        CELL53_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL53_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL53_TARGET_EVIDENCE_ID,

    "old_target_evidence_id":
        CELL53_OLD_TARGET_EVIDENCE_ID,

    "question_draft_id":
        CELL53_QUESTION_DRAFT_ID,

    "expected_answer_draft_id":
        CELL53_EXPECTED_ANSWER_DRAFT_ID,

    "citation_entries":
        copy.deepcopy(
            cell53_citation_entries
        ),

    "citation_scope":
        "REPLACEMENT_TARGET_EVIDENCE_ONLY",

    "retarget_used":
        True,

    "locator_source":
        "EXISTING_REPLACEMENT_TARGET_LOCATORS",

    "external_locator_lookup_used":
        False,

    "machine_validation_status":
        (
            "PASS_PENDING_HUMAN_ADJUDICATION"
            if cell53_citation_machine_pass
            else
            "FAIL_REQUIRES_REVISION"
        ),

    "human_adjudication_status":
        "PENDING",

    "full_572_generation":
        CELL53_FULL_572_BLOCK_STATUS,
}


# ============================================================
# 12) Register Citation
# ============================================================

cell53_current_session = (
    cell13_apply_event(
        cell53_citation_pending_session,
        "CITATION_MAPPING_REGISTERED",
        artifact=
            cell53_citation_mapping_record,
    )
)


cell53_citation_registered_valid = all([
    cell53_current_session.get(
        "current_state"
    )
    ==
    "CITATION_HUMAN_REVIEW_PENDING",

    cell53_current_session.get(
        "queue_advance_performed"
    )
    is False,

    cell53_current_session.get(
        "artifacts",
        {},
    ).get(
        "citation_mapping",
        {},
    ).get(
        "citation_mapping_id"
    )
    ==
    cell53_citation_mapping_id,
])


# ============================================================
# 13) Update Human Review workspace
# ============================================================

cell53_active_human_review_case = copy.deepcopy(
    cell52_active_human_review_case
)


cell53_active_human_review_case[
    "workflow_state"
] = "CITATION_HUMAN_REVIEW_PENDING"


cell53_active_human_review_case[
    "expected_answer_review"
][
    "human_decision"
] = "ACCEPT"


cell53_active_human_review_case[
    "expected_answer_review"
][
    "human_review"
] = copy.deepcopy(
    cell53_answer_human_review
)


cell53_active_human_review_case[
    "citation_review"
] = {
    "citation_mapping_id":
        cell53_citation_mapping_id,

    "citation_entries":
        copy.deepcopy(
            cell53_citation_entries
        ),

    "machine_validation_status":
        cell53_citation_mapping_record[
            "machine_validation_status"
        ],

    "human_decision":
        "PENDING",
}


cell53_active_human_review_case[
    "final_status"
] = (
    "RETARGET_ACCEPTED_QUESTION_AND_ANSWER_ACCEPTED_"
    "CITATION_PENDING_REVIEW"
)


# ============================================================
# 14) Persist Registry
# ============================================================

cell53_persistent_registry_payload = copy.deepcopy(
    cell52_persisted_registry
)


cell53_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell53_active_human_review_case
)


cell53_persistent_registry_payload[
    "remaining_pilot_count"
] = 14


cell53_persistent_registry_payload[
    "continuation_state"
] = {
    "next_pilot_id":
        CELL53_PILOT_ID,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell48_remaining_pilot_ids
        ),

    "remaining_count":
        14,

    "continuation_state_sha256":
        cell48_continuation_state_sha256,

    "active_pilot_status":
        "CITATION_HUMAN_REVIEW_PENDING",

    "active_target_evidence_id":
        CELL53_TARGET_EVIDENCE_ID,

    "retarget_status":
        "REPLACEMENT_ACCEPTED",
}


cell53_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell53_persistent_registry_payload
    )
)


# ============================================================
# 15) Read-back validation
# ============================================================

cell53_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell53_persisted_active_case = (
    cell53_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell53_persistence_valid = all([
    cell53_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL53_PILOT_ID,

    cell53_persisted_active_case.get(
        "target_evidence_id"
    )
    ==
    CELL53_TARGET_EVIDENCE_ID,

    cell53_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell53_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "citation_mapping_id"
    )
    ==
    cell53_citation_mapping_id,

    cell53_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "PENDING",

    cell53_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    14,
])


# ============================================================
# 16) Queue / safety
# ============================================================

cell53_queue_still_14 = all([
    len(
        cell48_remaining_pilot_ids
    )
    ==
    14,

    cell48_remaining_pilot_ids[
        0
    ]
    ==
    CELL53_PILOT_ID,
])


cell53_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell53_full_572_blocked = (
    cell53_citation_mapping_record[
        "full_572_generation"
    ]
    ==
    CELL53_FULL_572_BLOCK_STATUS
)


# ============================================================
# 17) Final Gate
# ============================================================

cell53_gate = all([
    len(
        cell53_failures
    )
    ==
    0,

    cell53_answer_accept_valid,

    cell53_target_parts_valid,

    cell53_no_locator_fabrication,

    cell53_all_entries_valid,

    cell53_citation_machine_pass,

    cell53_citation_registered_valid,

    cell53_persistence_valid,

    cell53_queue_still_14,

    cell53_base_seed_unchanged,

    cell53_full_572_blocked,
])


# ============================================================
# 18) Diagnostics
# ============================================================

print(
    "=== ANSWER ACCEPTED + RETARGET CITATION DRAFT "
    "+ REGISTRY SYNC ==="
)


print(
    "\n=== ACCEPTED QUESTION ==="
)

print(
    CELL52_ACCEPTED_QUESTION
)


print(
    "\n=== ACCEPTED ANSWER ==="
)

print(
    CELL52_EXPECTED_ANSWER
)

print(
    "Human Answer Decision:",
    "ACCEPT",
)


print(
    "\n=== RETARGET LINEAGE ==="
)

print(
    "Old Target:",
    CELL53_OLD_TARGET_EVIDENCE_ID,
)

print(
    "Replacement Target:",
    CELL53_TARGET_EVIDENCE_ID,
)

print(
    "Heading Path:",
    " > ".join(
        cell52_required_heading_path
    ),
)

print(
    "Heading Depth:",
    3,
)


print(
    "\n=== CITATION DRAFT ==="
)

print(
    "Citation Mapping ID:",
    cell53_citation_mapping_id,
)

print(
    "Citation entry count:",
    len(
        cell53_citation_entries
    ),
)


for entry in cell53_citation_entries:

    print(
        "\n--- CITATION ENTRY",
        entry.get(
            "citation_index"
        ),
        "---"
    )

    print(
        "Document:",
        entry.get(
            "document_id"
        ),
    )

    print(
        "Role:",
        entry.get(
            "role"
        ),
    )

    print(
        "Chars:",
        (
            entry.get(
                "char_start"
            ),
            entry.get(
                "char_end"
            ),
        ),
    )

    print(
        "Paragraph ID:",
        entry.get(
            "locator",
            {},
        ).get(
            "paragraph_id"
        ),
    )

    print(
        "Paragraph Number:",
        entry.get(
            "locator",
            {},
        ).get(
            "paragraph_number"
        ),
    )

    print(
        "Heading Depth:",
        entry.get(
            "locator",
            {},
        ).get(
            "heading_depth"
        ),
    )

    print(
        "Heading Path:",
        " > ".join(
            entry.get(
                "locator",
                {},
            ).get(
                "heading_path",
                [],
            )
        ),
    )


print(
    "\n=== CITATION MACHINE VALIDATION ==="
)

for name, status in (
    cell53_citation_machine_validation.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "Machine PASS:",
    cell53_citation_machine_pass,
)

print(
    "No locator fabrication:",
    cell53_no_locator_fabrication,
)


print(
    "\n=== WORKFLOW ==="
)

print(
    "Current State:",
    cell53_current_session.get(
        "current_state"
    ),
)

print(
    "Retarget:",
    "ACCEPT",
)

print(
    "Question Decision:",
    "ACCEPT",
)

print(
    "Answer Decision:",
    "ACCEPT",
)

print(
    "Citation Decision:",
    "PENDING",
)


print(
    "\n=== PERSISTENCE ==="
)

print(
    "Registry:",
    cell21_registry_path,
)

print(
    "Persisted Answer Decision:",
    cell53_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    ),
)

print(
    "Persisted Citation Decision:",
    cell53_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "human_decision"
    ),
)

print(
    "Persistence valid:",
    cell53_persistence_valid,
)


print(
    "\n=== QUEUE / SAFETY ==="
)

print(
    "Queue still 14:",
    cell53_queue_still_14,
)

print(
    "Base Seed unchanged:",
    cell53_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell53_full_572_blocked,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell53_failures
)


print(
    "\nCODE CELL 53 GATE:",
    (
        "PASS"
        if cell53_gate
        else "FAIL"
    ),
)


assert cell53_gate, (
    "Expected Answer ACCEPT / Retarget Citation Draft / "
    "Registry Sync failed. Repair this SAME Code Cell 53 "
    "before Citation Human Review."
)

=== ANSWER ACCEPTED + RETARGET CITATION DRAFT + REGISTRY SYNC ===

=== ACCEPTED QUESTION ===
نوجوان برای کسب درآمد از طریق یک کسب‌وکار باید با چه چیزهایی آشنا باشد و والدین چه نقشی دارند؟

=== ACCEPTED ANSWER ===
نوجوان برای کسب درآمد از طریق یک کسب‌وکار باید با منابع مالی، انسانی، طبیعی و مواد اولیه آشنا باشد، فرایند تولید را بداند، تا حدی مشتری خود را بشناسد و فعالیت بازاریابی انجام دهد. در این فرایند، والدین نقش هیأت‌مدیره و فرزند نقش مدیرعامل را دارند تا از آسیب‌های احتمالی مالی و غیرمالی جلوگیری شود.
Human Answer Decision: ACCEPT

=== RETARGET LINEAGE ===
Old Target: target-evidence:135c9f93d0c5695ce41421903c264783
Replacement Target: target-evidence:3fa4a04b2c148b45d3cf6f21e943dcb8
Heading Path: پول‌توجیبی > افزایش پول‌توجیبی > کسب درآمد
Heading Depth: 3

=== CITATION DRAFT ===
Citation Mapping ID: citation-mapping:f88a5089a9fcdbe1ffb4d278f05ec3f5
Citation entry count: 1

--- CITATION ENTRY 0 ---
Document: book:trainer-training-financial-literacy:fa-source:p000374
Role: DEEP_HEAD

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 54 — Citation ACCEPT + Pilot #10 Completion + Queue Advance

Citation Mapping این Pilot توسط Human Reviewer پذیرفته شد.

در این مرحله:

- Citation با تصمیم `ACCEPT` ثبت می‌شود؛
- Pilot Retargetشده به‌طور کامل بسته می‌شود؛
- Lineage میان Target قدیمی و Replacement Target حفظ می‌شود؛
- Case کامل به `completed_human_review_cases` منتقل می‌شود؛
- Case برای Author Final Review با وضعیت `NOT_STARTED` آماده می‌شود؛
- Queue فقط یک مرحله، از 14 به 13، جلو می‌رود؛
- Pilot بعدی از Compact Seed بازیابی می‌شود؛
- Source Evidence بعدی برای Human Review نمایش داده می‌شود؛
- Full 572 Generation همچنان Blocked باقی می‌ماند.

</div>

In [54]:
# Code Cell 54 — COMPLETE REPLACEMENT

cell54_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell53_gate") is not True:
    cell54_failures.append(
        "CELL53_NOT_PASS"
    )


if (
    globals().get(
        "cell53_current_session",
        {},
    ).get(
        "current_state"
    )
    !=
    "CITATION_HUMAN_REVIEW_PENDING"
):
    cell54_failures.append(
        "CITATION_NOT_AT_HUMAN_REVIEW_GATE"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell54_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Human Citation ACCEPT
# ============================================================

CELL54_CITATION_HUMAN_NOTE = (
    "Human Reviewer accepted the Citation Mapping for the "
    "three-level replacement Target Evidence."
)


cell54_complete_pending_session = (
    cell13_apply_event(
        cell53_current_session,
        "CITATION_ACCEPTED",
        human_note=
            CELL54_CITATION_HUMAN_NOTE,
    )
)


cell54_human_decisions = (
    cell54_complete_pending_session.get(
        "human_decisions",
        [],
    )
)


cell54_citation_accept_valid = all([
    cell54_complete_pending_session.get(
        "current_state"
    )
    ==
    "PILOT_COMPLETE_PENDING_QUEUE_ADVANCE",

    cell54_complete_pending_session.get(
        "queue_advance_performed"
    )
    is False,

    bool(
        cell54_human_decisions
    ),

    (
        cell54_human_decisions[
            -1
        ].get(
            "event"
        )
        ==
        "CITATION_ACCEPTED"
        if cell54_human_decisions
        else False
    ),
])


# ============================================================
# 3) Final accepted Citation record
# ============================================================

cell54_final_citation_mapping_record = copy.deepcopy(
    cell53_citation_mapping_record
)


cell54_final_citation_mapping_record[
    "human_adjudication_status"
] = "ACCEPT"


cell54_final_citation_mapping_record[
    "human_review"
] = {
    "human_decision":
        "ACCEPT",

    "human_note":
        CELL54_CITATION_HUMAN_NOTE,

    "human_context_used_as_evidence":
        False,

    "external_knowledge_used":
        False,
}


# ============================================================
# 4) Complete Human Review case
# ============================================================

cell54_completed_case = copy.deepcopy(
    cell53_active_human_review_case
)


cell54_completed_case[
    "workflow_state"
] = "PILOT_COMPLETE"


cell54_completed_case[
    "status"
] = "COMPLETED_HUMAN_REVIEW"


cell54_completed_case[
    "final_status"
] = "PILOT_COMPLETE"


cell54_completed_case[
    "citation_review"
][
    "human_decision"
] = "ACCEPT"


cell54_completed_case[
    "citation_review"
][
    "human_note"
] = CELL54_CITATION_HUMAN_NOTE


cell54_completed_case[
    "citation_review"
][
    "citation_mapping_record"
] = copy.deepcopy(
    cell54_final_citation_mapping_record
)


cell54_completed_case[
    "completion_summary"
] = {
    "retarget_used":
        True,

    "old_target_evidence_id":
        CELL53_OLD_TARGET_EVIDENCE_ID,

    "replacement_target_evidence_id":
        CELL53_TARGET_EVIDENCE_ID,

    "selected_candidate_id":
        CELL52_SELECTED_CANDIDATE_ID,

    "question_decision":
        "ACCEPT",

    "expected_answer_decision":
        "ACCEPT",

    "citation_decision":
        "ACCEPT",

    "author_final_review":
        "NOT_STARTED",
}


cell54_completed_case[
    "author_final_review"
] = {
    "status":
        "NOT_STARTED",

    "decision":
        None,

    "notes":
        None,
}


# ============================================================
# 5) Queue advance: 14 -> 13
# ============================================================

cell54_previous_remaining_pilot_ids = copy.deepcopy(
    cell48_remaining_pilot_ids
)


cell54_remaining_pilot_ids = copy.deepcopy(
    cell54_previous_remaining_pilot_ids[
        1:
    ]
)


CELL54_COMPLETED_PILOT_ID = (
    cell54_previous_remaining_pilot_ids[
        0
    ]
)


cell54_queue_advance_valid = all([
    len(
        cell54_previous_remaining_pilot_ids
    )
    ==
    14,

    CELL54_COMPLETED_PILOT_ID
    ==
    CELL49_PILOT_ID,

    len(
        cell54_remaining_pilot_ids
    )
    ==
    13,

    CELL54_COMPLETED_PILOT_ID
    not in
    cell54_remaining_pilot_ids,
])


CELL54_NEXT_PILOT_ID = (
    cell54_remaining_pilot_ids[
        0
    ]
)


# ============================================================
# 6) Find next Pilot packet in immutable Compact Seed
# ============================================================

def cell54_find_pilot_packet_candidates(
    value,
    pilot_id,
    path=(),
):

    results = []


    if isinstance(
        value,
        dict,
    ):

        direct_values = [
            item
            for item
            in value.values()
            if isinstance(
                item,
                str,
            )
        ]


        if pilot_id in direct_values:

            score = 0


            if (
                value.get(
                    "pilot_case_id"
                )
                ==
                pilot_id
            ):
                score += 120


            if (
                value.get(
                    "question_pilot_id"
                )
                ==
                pilot_id
            ):
                score += 110


            if (
                value.get(
                    "pilot_id"
                )
                ==
                pilot_id
            ):
                score += 100


            if (
                "target_text_parts"
                in
                value
            ):
                score += 60


            if (
                "target_evidence_id"
                in
                value
            ):
                score += 30


            if (
                "golden_case_id"
                in
                value
            ):
                score += 20


            if (
                "task_draft_id"
                in
                value
            ):
                score += 20


            results.append({
                "path":
                    path,

                "score":
                    score,

                "packet":
                    value,
            })


        for key, item in value.items():

            results.extend(
                cell54_find_pilot_packet_candidates(
                    item,
                    pilot_id,
                    path
                    +
                    (
                        str(
                            key
                        ),
                    ),
                )
            )


    elif isinstance(
        value,
        list,
    ):

        for index, item in enumerate(
            value
        ):

            results.extend(
                cell54_find_pilot_packet_candidates(
                    item,
                    pilot_id,
                    path
                    +
                    (
                        str(
                            index
                        ),
                    ),
                )
            )


    return results


cell54_raw_next_packet_candidates = (
    cell54_find_pilot_packet_candidates(
        cell13_seed,
        CELL54_NEXT_PILOT_ID,
    )
)


cell54_packet_candidates_by_sha = {}


for candidate in cell54_raw_next_packet_candidates:

    packet_sha = cell10_stable_sha(
        candidate[
            "packet"
        ]
    )


    previous = (
        cell54_packet_candidates_by_sha.get(
            packet_sha
        )
    )


    if (
        previous is None
        or
        candidate[
            "score"
        ]
        >
        previous[
            "score"
        ]
    ):

        cell54_packet_candidates_by_sha[
            packet_sha
        ] = candidate


cell54_ranked_next_packet_candidates = sorted(
    cell54_packet_candidates_by_sha.values(),
    key=lambda item: (
        -item[
            "score"
        ],
        item[
            "path"
        ],
    ),
)


cell54_next_pilot_packet = (
    copy.deepcopy(
        cell54_ranked_next_packet_candidates[
            0
        ][
            "packet"
        ]
    )
    if cell54_ranked_next_packet_candidates
    else None
)


if not isinstance(
    cell54_next_pilot_packet,
    dict,
):

    cell54_failures.append(
        "NEXT_PILOT_PACKET_NOT_FOUND"
    )


# ============================================================
# 7) Extract deterministic identities from next Pilot packet
# ============================================================

def cell54_collect_prefixed_ids(
    value,
    prefix,
):

    found = []


    if isinstance(
        value,
        dict,
    ):

        for item in value.values():

            found.extend(
                cell54_collect_prefixed_ids(
                    item,
                    prefix,
                )
            )


    elif isinstance(
        value,
        list,
    ):

        for item in value:

            found.extend(
                cell54_collect_prefixed_ids(
                    item,
                    prefix,
                )
            )


    elif (
        isinstance(
            value,
            str,
        )
        and
        value.startswith(
            prefix
        )
    ):

        found.append(
            value
        )


    return found


cell54_next_pilot_ids_found = sorted(
    set(
        cell54_collect_prefixed_ids(
            cell54_next_pilot_packet,
            "question-pilot:",
        )
    )
)


cell54_next_golden_case_ids = sorted(
    set(
        cell54_collect_prefixed_ids(
            cell54_next_pilot_packet,
            "golden-case:",
        )
    )
)


cell54_next_task_draft_ids = sorted(
    set(
        cell54_collect_prefixed_ids(
            cell54_next_pilot_packet,
            "task-draft:",
        )
    )
)


cell54_next_target_evidence_ids = sorted(
    set(
        cell54_collect_prefixed_ids(
            cell54_next_pilot_packet,
            "target-evidence:",
        )
    )
)


cell54_next_identity_valid = all([
    cell54_next_pilot_ids_found
    ==
    [
        CELL54_NEXT_PILOT_ID
    ],

    len(
        cell54_next_golden_case_ids
    )
    ==
    1,

    len(
        cell54_next_task_draft_ids
    )
    ==
    1,

    len(
        cell54_next_target_evidence_ids
    )
    ==
    1,
])


CELL54_NEXT_GOLDEN_CASE_ID = (
    cell54_next_golden_case_ids[
        0
    ]
    if
    len(
        cell54_next_golden_case_ids
    )
    ==
    1
    else
    None
)


CELL54_NEXT_TASK_DRAFT_ID = (
    cell54_next_task_draft_ids[
        0
    ]
    if
    len(
        cell54_next_task_draft_ids
    )
    ==
    1
    else
    None
)


CELL54_NEXT_TARGET_EVIDENCE_ID = (
    cell54_next_target_evidence_ids[
        0
    ]
    if
    len(
        cell54_next_target_evidence_ids
    )
    ==
    1
    else
    None
)


# ============================================================
# 8) Extract next Target Evidence parts
# ============================================================

def cell54_find_key_values(
    value,
    target_key,
):

    found = []


    if isinstance(
        value,
        dict,
    ):

        for key, item in value.items():

            if key == target_key:

                found.append(
                    copy.deepcopy(
                        item
                    )
                )


            found.extend(
                cell54_find_key_values(
                    item,
                    target_key,
                )
            )


    elif isinstance(
        value,
        list,
    ):

        for item in value:

            found.extend(
                cell54_find_key_values(
                    item,
                    target_key,
                )
            )


    return found


cell54_target_part_candidates = (
    cell54_find_key_values(
        cell54_next_pilot_packet,
        "target_text_parts",
    )
)


cell54_target_parts_by_sha = {}


for parts in cell54_target_part_candidates:

    if not isinstance(
        parts,
        list,
    ):
        continue


    if len(
        parts
    ) == 0:
        continue


    parts_sha = cell10_stable_sha(
        parts
    )


    cell54_target_parts_by_sha[
        parts_sha
    ] = parts


cell54_unique_target_part_sets = list(
    cell54_target_parts_by_sha.values()
)


cell54_next_target_text_parts = (
    copy.deepcopy(
        cell54_unique_target_part_sets[
            0
        ]
    )
    if
    len(
        cell54_unique_target_part_sets
    )
    ==
    1
    else
    None
)


if not isinstance(
    cell54_next_target_text_parts,
    list,
):

    cell54_failures.append(
        "NEXT_TARGET_TEXT_PARTS_NOT_UNIQUE"
    )


# ============================================================
# 9) Build readable next Evidence
# ============================================================

cell54_next_evidence_texts = []


for part in (
    cell54_next_target_text_parts
    or
    []
):

    if isinstance(
        part,
        str,
    ):

        text = part.strip()


    elif isinstance(
        part,
        dict,
    ):

        text = (
            part.get(
                "text"
            )
            or
            part.get(
                "target_text"
            )
            or
            part.get(
                "content"
            )
            or
            ""
        )


        if isinstance(
            text,
            str,
        ):

            text = text.strip()


        else:

            text = ""


    else:

        text = ""


    if text:

        cell54_next_evidence_texts.append(
            text
        )


cell54_next_combined_target_text = (
    "\n\n".join(
        cell54_next_evidence_texts
    )
)


cell54_next_evidence_available = bool(
    cell54_next_combined_target_text.strip()
)


# ============================================================
# 10) Initialize next Pilot engine session
# ============================================================

cell54_next_pilot_session = (
    cell13_build_session(
        cell54_next_pilot_packet
    )
)


cell54_next_session_valid = (
    cell54_next_pilot_session.get(
        "current_state"
    )
    ==
    "QUESTION_DRAFT_PENDING"
)


# ============================================================
# 11) Update completed Human Review cases
# ============================================================

cell54_existing_completed_cases = copy.deepcopy(
    cell53_persisted_registry.get(
        "completed_human_review_cases",
        [],
    )
)


if not isinstance(
    cell54_existing_completed_cases,
    list,
):

    cell54_failures.append(
        "COMPLETED_HUMAN_REVIEW_CASES_NOT_LIST"
    )

    cell54_existing_completed_cases = []


cell54_existing_completed_cases = [
    case
    for case
    in cell54_existing_completed_cases
    if not (
        isinstance(
            case,
            dict,
        )
        and
        case.get(
            "pilot_case_id"
        )
        ==
        CELL54_COMPLETED_PILOT_ID
    )
]


cell54_completed_human_review_cases = (
    cell54_existing_completed_cases
    +
    [
        copy.deepcopy(
            cell54_completed_case
        )
    ]
)


cell54_completed_case_count_valid = (
    len(
        cell54_completed_human_review_cases
    )
    ==
    9
)


# ============================================================
# 12) Update Author Review queue
# ============================================================

cell54_author_review = copy.deepcopy(
    cell53_persisted_registry.get(
        "author_review",
        {},
    )
)


if not isinstance(
    cell54_author_review,
    dict,
):

    cell54_author_review = {}


# All completed Human Review cases are Author Review-ready.
# Rebuild this list from the authoritative completed case list
# instead of relying on a possibly different historical key shape.
cell54_author_ready_cases = copy.deepcopy(
    cell54_completed_human_review_cases
)


cell54_author_review[
    "cases_ready_for_review"
] = copy.deepcopy(
    cell54_author_ready_cases
)


cell54_author_review[
    "case_count"
] = len(
    cell54_author_ready_cases
)


cell54_author_review[
    "final_review_status"
] = "NOT_STARTED"


cell54_author_ready_count_valid = (
    len(
        cell54_author_ready_cases
    )
    ==
    9
)


cell54_author_ready_identity_valid = (
    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell54_author_ready_cases
        if isinstance(
            case,
            dict,
        )
    }
    ==
    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell54_completed_human_review_cases
        if isinstance(
            case,
            dict,
        )
    }
)


# ============================================================
# 13) Build next active workspace
# ============================================================

cell54_next_active_workspace = {
    "pilot_case_id":
        CELL54_NEXT_PILOT_ID,

    "golden_case_id":
        CELL54_NEXT_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL54_NEXT_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL54_NEXT_TARGET_EVIDENCE_ID,

    "status":
        "ACTIVE_HUMAN_REVIEW",

    "workflow_state":
        "QUESTION_DRAFT_PENDING",

    "source_evidence":
        copy.deepcopy(
            cell54_next_target_text_parts
        ),

    "combined_target_text":
        cell54_next_combined_target_text,

    "question_review":
        None,

    "expected_answer_review":
        None,

    "citation_review":
        None,

    "author_final_review": {
        "status":
            "NOT_STARTED",

        "decision":
            None,

        "notes":
            None,
    },
}


# ============================================================
# 14) New authoritative continuation state
# ============================================================

cell54_continuation_state_core = {
    "next_pilot_id":
        CELL54_NEXT_PILOT_ID,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell54_remaining_pilot_ids
        ),

    "remaining_count":
        13,

    "active_pilot_status":
        "QUESTION_DRAFT_PENDING",

    "active_target_evidence_id":
        CELL54_NEXT_TARGET_EVIDENCE_ID,

    "last_completed_pilot_id":
        CELL54_COMPLETED_PILOT_ID,

    "last_completion_type":
        "RETARGETED_PILOT_COMPLETE",
}


cell54_continuation_state_sha256 = (
    cell10_stable_sha(
        cell54_continuation_state_core
    )
)


cell54_continuation_state = {
    **copy.deepcopy(
        cell54_continuation_state_core
    ),

    "continuation_state_sha256":
        cell54_continuation_state_sha256,
}


# ============================================================
# 15) Persist Registry
# ============================================================

cell54_persistent_registry_payload = copy.deepcopy(
    cell53_persisted_registry
)


cell54_persistent_registry_payload[
    "completed_human_review_cases"
] = copy.deepcopy(
    cell54_completed_human_review_cases
)


cell54_persistent_registry_payload[
    "author_review"
] = copy.deepcopy(
    cell54_author_review
)


cell54_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell54_next_active_workspace
)


cell54_persistent_registry_payload[
    "remaining_pilot_count"
] = 13


cell54_persistent_registry_payload[
    "continuation_state"
] = copy.deepcopy(
    cell54_continuation_state
)


cell54_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell54_persistent_registry_payload
    )
)


# ============================================================
# 16) Read-back validation
# ============================================================

cell54_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell54_persisted_active_case = (
    cell54_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell54_persisted_completed_cases = (
    cell54_persisted_registry.get(
        "completed_human_review_cases",
        [],
    )
)


cell54_persisted_author_ready_cases = (
    cell54_persisted_registry.get(
        "author_review",
        {},
    ).get(
        "cases_ready_for_review",
        [],
    )
)


cell54_persistence_valid = all([
    isinstance(
        cell54_persisted_completed_cases,
        list,
    ),

    len(
        cell54_persisted_completed_cases
    )
    ==
    9,

    isinstance(
        cell54_persisted_author_ready_cases,
        list,
    ),

    len(
        cell54_persisted_author_ready_cases
    )
    ==
    9,

    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell54_persisted_completed_cases
        if isinstance(
            case,
            dict,
        )
    }
    ==
    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell54_persisted_author_ready_cases
        if isinstance(
            case,
            dict,
        )
    },

    cell54_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    13,

    cell54_persisted_registry.get(
        "continuation_state",
        {},
    ).get(
        "continuation_state_sha256"
    )
    ==
    cell54_continuation_state_sha256,

    cell54_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL54_NEXT_PILOT_ID,

    cell54_persisted_active_case.get(
        "workflow_state"
    )
    ==
    "QUESTION_DRAFT_PENDING",
])


# ============================================================
# 17) Safety
# ============================================================

cell54_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell54_full_572_blocked = (
    CELL53_FULL_572_BLOCK_STATUS
    ==
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


cell54_no_commit_push = True


# ============================================================
# 18) Final Gate
# ============================================================

cell54_gate = all([
    len(
        cell54_failures
    )
    ==
    0,

    cell54_citation_accept_valid,

    cell54_queue_advance_valid,

    cell54_next_identity_valid,

    cell54_next_evidence_available,

    cell54_next_session_valid,

    cell54_completed_case_count_valid,

    cell54_author_ready_count_valid,

    cell54_author_ready_identity_valid,

    cell54_persistence_valid,

    cell54_base_seed_unchanged,

    cell54_full_572_blocked,

    cell54_no_commit_push,
])


# ============================================================
# 19) Diagnostics
# ============================================================

print(
    "=== PILOT #10 COMPLETE + QUEUE ADVANCE + NEXT PILOT ==="
)


print(
    "\n=== COMPLETED PILOT ==="
)

print(
    "Pilot:",
    CELL54_COMPLETED_PILOT_ID,
)

print(
    "Retarget:",
    "ACCEPT",
)

print(
    "Question:",
    "ACCEPT",
)

print(
    "Expected Answer:",
    "ACCEPT",
)

print(
    "Citation:",
    "ACCEPT",
)

print(
    "Final Engine State:",
    cell54_complete_pending_session.get(
        "current_state"
    ),
)


print(
    "\n=== COMPLETED RETARGET LINEAGE ==="
)

print(
    "Old Target:",
    CELL53_OLD_TARGET_EVIDENCE_ID,
)

print(
    "Replacement Target:",
    CELL53_TARGET_EVIDENCE_ID,
)

print(
    "Citation Mapping:",
    cell53_citation_mapping_id,
)


print(
    "\n=== QUEUE ADVANCE ==="
)

print(
    "Before:",
    len(
        cell54_previous_remaining_pilot_ids
    ),
)

print(
    "After:",
    len(
        cell54_remaining_pilot_ids
    ),
)

print(
    "Queue advanced exactly once:",
    cell54_queue_advance_valid,
)


print(
    "\n=== HUMAN REVIEW REGISTRY ==="
)

print(
    "Completed Human Review cases:",
    len(
        cell54_completed_human_review_cases
    ),
)

print(
    "Cases ready for Author Review:",
    len(
        cell54_author_ready_cases
    ),
)

print(
    "Author ready identities match completed cases:",
    cell54_author_ready_identity_valid,
)

print(
    "Author Final Review:",
    "NOT_STARTED",
)

print(
    "Registry:",
    cell21_registry_path,
)

print(
    "Persistence valid:",
    cell54_persistence_valid,
)


print(
    "\n=== NEXT PILOT ==="
)

print(
    "Pilot:",
    CELL54_NEXT_PILOT_ID,
)

print(
    "Golden Case:",
    CELL54_NEXT_GOLDEN_CASE_ID,
)

print(
    "Task Draft:",
    CELL54_NEXT_TASK_DRAFT_ID,
)

print(
    "Target Evidence:",
    CELL54_NEXT_TARGET_EVIDENCE_ID,
)

print(
    "Workflow:",
    cell54_next_pilot_session.get(
        "current_state"
    ),
)


print(
    "\n=== NEXT SOURCE EVIDENCE ==="
)

print(
    cell54_next_combined_target_text
)


print(
    "\n=== CONTINUATION STATE ==="
)

print(
    "Remaining Pilots:",
    13,
)

print(
    "Continuation SHA256:",
    cell54_continuation_state_sha256,
)


print(
    "\n=== SAFETY ==="
)

print(
    "Base Seed unchanged:",
    cell54_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell54_full_572_blocked,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell54_failures
)


print(
    "\nCODE CELL 54 GATE:",
    (
        "PASS"
        if cell54_gate
        else "FAIL"
    ),
)


assert cell54_gate, (
    "Citation ACCEPT / Pilot completion / Queue advance / "
    "Next Pilot preparation failed. Repair this SAME "
    "Code Cell 54 before continuing."
)

=== PILOT #10 COMPLETE + QUEUE ADVANCE + NEXT PILOT ===

=== COMPLETED PILOT ===
Pilot: question-pilot:c9b2a6ecc853bf563a45af119621c405
Retarget: ACCEPT
Question: ACCEPT
Expected Answer: ACCEPT
Citation: ACCEPT
Final Engine State: PILOT_COMPLETE_PENDING_QUEUE_ADVANCE

=== COMPLETED RETARGET LINEAGE ===
Old Target: target-evidence:135c9f93d0c5695ce41421903c264783
Replacement Target: target-evidence:3fa4a04b2c148b45d3cf6f21e943dcb8
Citation Mapping: citation-mapping:f88a5089a9fcdbe1ffb4d278f05ec3f5

=== QUEUE ADVANCE ===
Before: 14
After: 13
Queue advanced exactly once: True

=== HUMAN REVIEW REGISTRY ===
Completed Human Review cases: 9
Cases ready for Author Review: 9
Author ready identities match completed cases: True
Author Final Review: NOT_STARTED
Registry: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/data/phase2/human_review/human_review_registry.json
Persistence valid: True

=== NEXT PILOT ===
Pilot: question-pilot:be503b9f0364162c19faa8401f9d7b0f
Gold

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 55 — Transcript Typo Correction + Pilot #11 RETARGET + Source Context Discovery

Human Reviewer دو تصمیم ثبت کرد:

### 1. اصلاح Transcript

عبارت:

**«گونی‌های ماسته»**

یک خطای Transcript است و شکل صحیح آن:

**«گونی‌های ماسه»**

است.

برای حفظ Lineage:

- متن خام Source حذف یا بازنویسی نمی‌شود؛
- شکل خام و شکل اصلاح‌شده هر دو نگهداری می‌شوند؛
- اصلاح با منشأ `HUMAN_CERTIFIED_TRANSCRIPT_CORRECTION` ثبت می‌شود؛
- از این مرحله به بعد، در لایه Normalized Review شکل «گونی‌های ماسه» معتبر است.

### 2. Retarget

این Target برای Golden QA مناسب نیست، زیرا جمله فعلی یک مثال جزئی درباره پناه گرفتن در زمان موشک‌باران است و به‌تنهایی مفهوم اصلی آموزشی را توضیح نمی‌دهد.

بنابراین:

- Human Decision برابر `RETARGET` ثبت می‌شود؛
- Queue جلو نمی‌رود و همچنان 13 باقی می‌ماند؛
- سؤال یا Expected Answer از این جمله ساخته نمی‌شود؛
- متن اطراف همین Source به‌صورت Read-only جست‌وجو می‌شود تا مشخص شود این مثال در خدمت چه مفهوم اصلی قرار گرفته است؛
- برداشت احتمالی «تاب‌آوری مالی» تا زمان تأیید Source فقط Human Context است و Evidence محسوب نمی‌شود.

</div>

In [55]:
# Code Cell 55

from pathlib import Path


cell55_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell54_gate") is not True:
    cell55_failures.append(
        "CELL54_NOT_PASS"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell55_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Current Pilot identities
# ============================================================

CELL55_PILOT_ID = (
    CELL54_NEXT_PILOT_ID
)


CELL55_GOLDEN_CASE_ID = (
    CELL54_NEXT_GOLDEN_CASE_ID
)


CELL55_TASK_DRAFT_ID = (
    CELL54_NEXT_TASK_DRAFT_ID
)


CELL55_OLD_TARGET_EVIDENCE_ID = (
    CELL54_NEXT_TARGET_EVIDENCE_ID
)


CELL55_RAW_TARGET_TEXT = (
    cell54_next_combined_target_text
)


# ============================================================
# 3) Human-certified Transcript correction
# ============================================================

CELL55_RAW_TOKEN = (
    "ماسته"
)


CELL55_CORRECTED_TOKEN = (
    "ماسه"
)


CELL55_CORRECTION_TYPE = (
    "HUMAN_CERTIFIED_TRANSCRIPT_CORRECTION"
)


cell55_typo_present = (
    CELL55_RAW_TOKEN
    in
    CELL55_RAW_TARGET_TEXT
)


if not cell55_typo_present:

    cell55_failures.append(
        "EXPECTED_TRANSCRIPT_TYPO_NOT_FOUND"
    )


cell55_corrected_target_text = (
    CELL55_RAW_TARGET_TEXT.replace(
        CELL55_RAW_TOKEN,
        CELL55_CORRECTED_TOKEN,
    )
)


cell55_correction_applied = all([
    CELL55_RAW_TOKEN
    not in
    cell55_corrected_target_text,

    CELL55_CORRECTED_TOKEN
    in
    cell55_corrected_target_text,

    cell55_corrected_target_text
    !=
    CELL55_RAW_TARGET_TEXT,
])


cell55_transcript_correction = {
    "correction_type":
        CELL55_CORRECTION_TYPE,

    "human_decision":
        "ACCEPT",

    "raw_token":
        CELL55_RAW_TOKEN,

    "corrected_token":
        CELL55_CORRECTED_TOKEN,

    "raw_text":
        CELL55_RAW_TARGET_TEXT,

    "normalized_corrected_text":
        cell55_corrected_target_text,

    "reason":
        (
            "Human Reviewer certified that 'ماسته' is a "
            "transcription typo and that the intended word "
            "is 'ماسه'."
        ),

    "canonical_raw_source_mutated":
        False,

    "normalized_review_text_corrected":
        True,

    "is_external_knowledge":
        False,
}


# ============================================================
# 4) Human RETARGET decision
# ============================================================

CELL55_RETARGET_REASON = (
    "The current evidence is a narrow illustrative statement "
    "about using sandbags and shelter during missile attacks. "
    "By itself it does not expose the broader educational "
    "concept required for a useful Golden QA pair. The Pilot "
    "should preserve its coverage while Target Evidence is "
    "replaced by a substantive source passage."
)


cell55_retarget_review = {
    "human_decision":
        "RETARGET",

    "old_target_evidence_id":
        CELL55_OLD_TARGET_EVIDENCE_ID,

    "reason":
        CELL55_RETARGET_REASON,

    "replacement_target_evidence_id":
        None,

    "replacement_status":
        "PENDING_SOURCE_CONTEXT_DISCOVERY",

    "question_generation_allowed":
        False,

    "expected_answer_generation_allowed":
        False,

    "citation_generation_allowed":
        False,
}


# ============================================================
# 5) Extract current Target metadata
# ============================================================

cell55_current_target_parts = copy.deepcopy(
    cell54_next_target_text_parts
)


cell55_current_document_ids = []


cell55_current_source_ids = []


cell55_current_locators = []


for part in (
    cell55_current_target_parts
    or
    []
):

    if not isinstance(
        part,
        dict,
    ):
        continue


    document_id = part.get(
        "document_id"
    )


    source_id = (
        part.get(
            "source_id"
        )
        or
        part.get(
            "edition_id"
        )
    )


    locator = part.get(
        "locator"
    )


    if isinstance(
        document_id,
        str,
    ):

        cell55_current_document_ids.append(
            document_id
        )


    if isinstance(
        source_id,
        str,
    ):

        cell55_current_source_ids.append(
            source_id
        )


    if isinstance(
        locator,
        dict,
    ):

        cell55_current_locators.append(
            copy.deepcopy(
                locator
            )
        )


cell55_current_document_ids = sorted(
    set(
        cell55_current_document_ids
    )
)


cell55_current_source_ids = sorted(
    set(
        cell55_current_source_ids
    )
)


# ============================================================
# 6) Read-only repository context search
# ============================================================

cell55_repo_root = (
    cell21_registry_path.parents[
        3
    ]
)


cell55_search_roots = [
    cell55_repo_root
    /
    "data"
    /
    "phase2"
    /
    "interim",

    cell55_repo_root
    /
    "data"
    /
    "phase2",
]


cell55_search_terms = [
    "گونی های ماسته",
    "گونی‌های ماسته",
    "گونی های ماسه",
    "گونی‌های ماسه",
    "موشک بارون",
    "موشک‌بارون",
    "موشک باران",
    "موشک‌باران",
]


cell55_source_identity_terms = (
    cell55_current_document_ids
    +
    cell55_current_source_ids
)


cell55_allowed_suffixes = {
    ".json",
    ".jsonl",
    ".txt",
    ".md",
    ".csv",
}


cell55_candidate_files = []


for search_root in cell55_search_roots:

    if not search_root.exists():
        continue


    for path in search_root.rglob(
        "*"
    ):

        if not path.is_file():
            continue


        if path == cell21_registry_path:
            continue


        if path.suffix.lower() not in (
            cell55_allowed_suffixes
        ):
            continue


        try:

            file_size = path.stat().st_size

        except OSError:

            continue


        if file_size > (
            40
            *
            1024
            *
            1024
        ):
            continue


        cell55_candidate_files.append(
            path
        )


cell55_candidate_files = sorted(
    set(
        cell55_candidate_files
    )
)


# ============================================================
# 7) Find source/context matches
# ============================================================

cell55_context_matches = []


for path in cell55_candidate_files:

    try:

        content = path.read_text(
            encoding="utf-8"
        )

    except Exception:

        continue


    matched_terms = [
        term

        for term
        in cell55_search_terms

        if term in content
    ]


    matched_identity_terms = [
        term

        for term
        in cell55_source_identity_terms

        if term in content
    ]


    if not (
        matched_terms
        or
        matched_identity_terms
    ):
        continue


    positions = []


    for term in (
        matched_terms
        +
        matched_identity_terms
    ):

        start = 0


        while True:

            position = content.find(
                term,
                start,
            )


            if position < 0:
                break


            positions.append(
                (
                    position,
                    term,
                )
            )


            start = (
                position
                +
                max(
                    1,
                    len(
                        term
                    ),
                )
            )


    positions = sorted(
        positions,
        key=lambda item: item[
            0
        ],
    )


    for position, term in positions[
        :8
    ]:

        snippet_start = max(
            0,
            position
            -
            1800,
        )


        snippet_end = min(
            len(
                content
            ),
            position
            +
            len(
                term
            )
            +
            1800,
        )


        snippet = content[
            snippet_start:
            snippet_end
        ]


        cell55_context_matches.append({
            "path":
                str(
                    path.relative_to(
                        cell55_repo_root
                    )
                ),

            "matched_term":
                term,

            "match_position":
                position,

            "snippet":
                snippet.strip(),

            "contains_raw_typo":
                (
                    CELL55_RAW_TOKEN
                    in
                    snippet
                ),

            "contains_corrected_word":
                (
                    CELL55_CORRECTED_TOKEN
                    in
                    snippet
                ),

            "matched_by_source_identity":
                (
                    term
                    in
                    cell55_source_identity_terms
                ),
        })


# ============================================================
# 8) Rank and deduplicate context matches
# ============================================================

cell55_context_match_by_key = {}


for match in cell55_context_matches:

    key = (
        match[
            "path"
        ],
        match[
            "match_position"
        ],
    )


    score = 0


    if match[
        "contains_raw_typo"
    ]:
        score += 50


    if match[
        "contains_corrected_word"
    ]:
        score += 20


    if match[
        "matched_by_source_identity"
    ]:
        score += 30


    match_copy = copy.deepcopy(
        match
    )


    match_copy[
        "score"
    ] = score


    previous = (
        cell55_context_match_by_key.get(
            key
        )
    )


    if (
        previous is None
        or
        score
        >
        previous[
            "score"
        ]
    ):

        cell55_context_match_by_key[
            key
        ] = match_copy


cell55_ranked_context_matches = sorted(
    cell55_context_match_by_key.values(),
    key=lambda item: (
        -item[
            "score"
        ],
        item[
            "path"
        ],
        item[
            "match_position"
        ],
    ),
)


cell55_context_shortlist = copy.deepcopy(
    cell55_ranked_context_matches[
        :10
    ]
)


if cell55_context_shortlist:

    CELL55_CONTEXT_SEARCH_STATUS = (
        "SOURCE_CONTEXT_CANDIDATES_FOUND"
    )


else:

    CELL55_CONTEXT_SEARCH_STATUS = (
        "SOURCE_CONTEXT_NOT_FOUND_IN_PHASE2_FILES"
    )


# ============================================================
# 9) Do not promote Human interpretation to Evidence
# ============================================================

cell55_human_context_note = {
    "possible_parent_concept":
        "تاب‌آوری مالی / آمادگی در برابر شوک‌ها",

    "status":
        "HUMAN_HYPOTHESIS_ONLY",

    "is_source_evidence":
        False,

    "may_be_used_in_golden_answer":
        False,

    "resolution":
        "PENDING_SOURCE_CONTEXT_REVIEW",
}


# ============================================================
# 10) Update active Human Review workspace
# ============================================================

cell55_active_workspace = copy.deepcopy(
    cell54_persisted_active_case
)


cell55_active_workspace[
    "workflow_state"
] = "BLOCKED_REQUIRES_RETARGET"


cell55_active_workspace[
    "status"
] = "ACTIVE_RETARGET_REVIEW"


cell55_active_workspace[
    "raw_source_evidence"
] = copy.deepcopy(
    cell55_current_target_parts
)


cell55_active_workspace[
    "raw_combined_target_text"
] = (
    CELL55_RAW_TARGET_TEXT
)


cell55_active_workspace[
    "normalized_combined_target_text"
] = (
    cell55_corrected_target_text
)


cell55_active_workspace[
    "transcript_correction"
] = copy.deepcopy(
    cell55_transcript_correction
)


cell55_active_workspace[
    "retarget_review"
] = copy.deepcopy(
    cell55_retarget_review
)


cell55_active_workspace[
    "human_context_note"
] = copy.deepcopy(
    cell55_human_context_note
)


cell55_active_workspace[
    "source_context_discovery"
] = {
    "search_status":
        CELL55_CONTEXT_SEARCH_STATUS,

    "candidate_count":
        len(
            cell55_ranked_context_matches
        ),

    "shortlist":
        copy.deepcopy(
            cell55_context_shortlist
        ),

    "read_only_search":
        True,

    "external_lookup_used":
        False,
}


# ============================================================
# 11) Preserve authoritative Queue state
# ============================================================

cell55_continuation_state = copy.deepcopy(
    cell54_continuation_state
)


cell55_continuation_state[
    "active_pilot_status"
] = "BLOCKED_REQUIRES_RETARGET"


cell55_continuation_state[
    "active_target_evidence_id"
] = (
    CELL55_OLD_TARGET_EVIDENCE_ID
)


cell55_continuation_state[
    "retarget_reason"
] = (
    "LOW_VALUE_ISOLATED_ILLUSTRATIVE_EVIDENCE"
)


cell55_continuation_core = {
    key:
        value

    for key, value
    in cell55_continuation_state.items()

    if key
    !=
    "continuation_state_sha256"
}


cell55_continuation_state_sha256 = (
    cell10_stable_sha(
        cell55_continuation_core
    )
)


cell55_continuation_state[
    "continuation_state_sha256"
] = (
    cell55_continuation_state_sha256
)


# ============================================================
# 12) Persist Registry
# ============================================================

cell55_persistent_registry_payload = copy.deepcopy(
    cell54_persisted_registry
)


cell55_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell55_active_workspace
)


cell55_persistent_registry_payload[
    "remaining_pilot_count"
] = 13


cell55_persistent_registry_payload[
    "continuation_state"
] = copy.deepcopy(
    cell55_continuation_state
)


cell55_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell55_persistent_registry_payload
    )
)


# ============================================================
# 13) Read-back validation
# ============================================================

cell55_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell55_persisted_active_case = (
    cell55_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell55_persistence_valid = all([
    cell55_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL55_PILOT_ID,

    cell55_persisted_active_case.get(
        "workflow_state"
    )
    ==
    "BLOCKED_REQUIRES_RETARGET",

    cell55_persisted_active_case.get(
        "transcript_correction",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell55_persisted_active_case.get(
        "transcript_correction",
        {},
    ).get(
        "corrected_token"
    )
    ==
    "ماسه",

    cell55_persisted_active_case.get(
        "retarget_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "RETARGET",

    cell55_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    13,
])


# ============================================================
# 14) Queue / safety
# ============================================================

cell55_queue_still_13 = all([
    len(
        cell54_remaining_pilot_ids
    )
    ==
    13,

    cell54_remaining_pilot_ids[
        0
    ]
    ==
    CELL55_PILOT_ID,
])


cell55_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell55_full_572_blocked = (
    cell54_full_572_blocked
    is True
)


# ============================================================
# 15) Final Gate
# ============================================================

cell55_gate = all([
    len(
        cell55_failures
    )
    ==
    0,

    cell55_typo_present,

    cell55_correction_applied,

    cell55_persistence_valid,

    cell55_queue_still_13,

    cell55_base_seed_unchanged,

    cell55_full_572_blocked,
])


# ============================================================
# 16) Diagnostics
# ============================================================

print(
    "=== TRANSCRIPT CORRECTION + PILOT #11 RETARGET "
    "+ SOURCE CONTEXT DISCOVERY ==="
)


print(
    "\n=== CURRENT PILOT ==="
)

print(
    "Pilot:",
    CELL55_PILOT_ID,
)

print(
    "Golden Case:",
    CELL55_GOLDEN_CASE_ID,
)

print(
    "Task Draft:",
    CELL55_TASK_DRAFT_ID,
)

print(
    "Old Target:",
    CELL55_OLD_TARGET_EVIDENCE_ID,
)


print(
    "\n=== RAW TRANSCRIPT ==="
)

print(
    CELL55_RAW_TARGET_TEXT
)


print(
    "\n=== HUMAN-CORRECTED TRANSCRIPT ==="
)

print(
    cell55_corrected_target_text
)

print(
    "Correction:",
    f"{CELL55_RAW_TOKEN} -> {CELL55_CORRECTED_TOKEN}",
)

print(
    "Correction Type:",
    CELL55_CORRECTION_TYPE,
)

print(
    "Canonical raw source mutated:",
    False,
)

print(
    "Normalized review text corrected:",
    True,
)


print(
    "\n=== RETARGET DECISION ==="
)

print(
    "Human Decision:",
    "RETARGET",
)

print(
    "Reason:",
    CELL55_RETARGET_REASON,
)

print(
    "Question generated:",
    False,
)

print(
    "Answer generated:",
    False,
)


print(
    "\n=== CURRENT TARGET METADATA ==="
)

print(
    "Document IDs:",
    cell55_current_document_ids,
)

print(
    "Source IDs:",
    cell55_current_source_ids,
)

print(
    "Locators:",
    cell55_current_locators,
)


print(
    "\n=== SOURCE CONTEXT SEARCH ==="
)

print(
    "Search Status:",
    CELL55_CONTEXT_SEARCH_STATUS,
)

print(
    "Files scanned:",
    len(
        cell55_candidate_files
    ),
)

print(
    "Matches found:",
    len(
        cell55_ranked_context_matches
    ),
)


for index, match in enumerate(
    cell55_context_shortlist,
    start=1,
):

    print(
        "\n"
        +
        "="
        *
        72
    )

    print(
        "CONTEXT CANDIDATE",
        index,
    )

    print(
        "Path:",
        match[
            "path"
        ],
    )

    print(
        "Matched Term:",
        match[
            "matched_term"
        ],
    )

    print(
        "Score:",
        match[
            "score"
        ],
    )

    print(
        "\nSnippet:"
    )

    print(
        match[
            "snippet"
        ]
    )


print(
    "\n=== HUMAN CONTEXT BOUNDARY ==="
)

print(
    "Possible concept:",
    cell55_human_context_note[
        "possible_parent_concept"
    ],
)

print(
    "Status:",
    cell55_human_context_note[
        "status"
    ],
)

print(
    "Used as Evidence:",
    False,
)


print(
    "\n=== WORKFLOW / PERSISTENCE ==="
)

print(
    "Workflow:",
    cell55_persisted_active_case.get(
        "workflow_state"
    ),
)

print(
    "Transcript correction persisted:",
    (
        cell55_persisted_active_case.get(
            "transcript_correction",
            {},
        ).get(
            "corrected_token"
        )
        ==
        "ماسه"
    ),
)

print(
    "Retarget persisted:",
    (
        cell55_persisted_active_case.get(
            "retarget_review",
            {},
        ).get(
            "human_decision"
        )
        ==
        "RETARGET"
    ),
)

print(
    "Persistence valid:",
    cell55_persistence_valid,
)


print(
    "\n=== QUEUE / SAFETY ==="
)

print(
    "Queue still 13:",
    cell55_queue_still_13,
)

print(
    "Base Seed unchanged:",
    cell55_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell55_full_572_blocked,
)

print(
    "External lookup:",
    False,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell55_failures
)


print(
    "\nCODE CELL 55 GATE:",
    (
        "PASS"
        if cell55_gate
        else "FAIL"
    ),
)


assert cell55_gate, (
    "Transcript correction / RETARGET / source context "
    "discovery failed. Repair this SAME Code Cell 55 "
    "before continuing."
)

=== TRANSCRIPT CORRECTION + PILOT #11 RETARGET + SOURCE CONTEXT DISCOVERY ===

=== CURRENT PILOT ===
Pilot: question-pilot:be503b9f0364162c19faa8401f9d7b0f
Golden Case: golden-case:c4fa6c10526f5d53df793ae0ef6ec0b0
Task Draft: task-draft:31d8142e21a1211f06fa49eda7295256
Old Target: target-evidence:6348f61441b9a5816156190f47b087c0

=== RAW TRANSCRIPT ===
یا گونی های ماسته بچینید و موقعی موشک بارون مثلا در زیر زمین قرار بگیرید که اطرافش گونی های ماسته چیده شده

=== HUMAN-CORRECTED TRANSCRIPT ===
یا گونی های ماسه بچینید و موقعی موشک بارون مثلا در زیر زمین قرار بگیرید که اطرافش گونی های ماسه چیده شده
Correction: ماسته -> ماسه
Correction Type: HUMAN_CERTIFIED_TRANSCRIPT_CORRECTION
Canonical raw source mutated: False
Normalized review text corrected: True

=== RETARGET DECISION ===
Human Decision: RETARGET
Reason: The current evidence is a narrow illustrative statement about using sandbags and shelter during missile attacks. By itself it does not expose the broader educational concept require

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 56 — Pilot #11 Coverage Audit + Economic-War Context Reconstruction

Context منبع نشان داد مثال «گونی‌های ماسه» هدف اصلی بحث نیست؛ بلکه مثالی برای مقایسه آموزش‌های کاربردی در جنگ نظامی با ضرورت آموزش سواد مالی در «جنگ اقتصادی» است.

در این مرحله:

- متادیتای واقعی Pilot #11 از Pilot Packet خوانده می‌شود؛
- Transcript اصلی Episode 28 مستقیماً از `transcripts_extracted_v1.jsonl` بازیابی می‌شود؛
- متن خام `ماسته` برای Lineage حفظ می‌شود؛
- در لایه Human-normalized Review، اصلاح قطعی `ماسته → ماسه` اعمال می‌شود؛
- Utteranceهای اطراف مثال بازسازی می‌شوند؛
- چند Candidate منسجم برای Replacement Target ساخته می‌شود؛
- هیچ Candidate هنوز Human ACCEPT نمی‌شود؛
- سؤال پیشنهادی فقط برای Review نمایش داده می‌شود و ثبت نهایی نمی‌شود؛
- Queue همچنان 13 باقی می‌ماند.

</div>

In [56]:
# Code Cell 56

cell56_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell55_gate") is not True:
    cell56_failures.append(
        "CELL55_NOT_PASS"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell56_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Current Pilot
# ============================================================

CELL56_PILOT_ID = CELL55_PILOT_ID
CELL56_GOLDEN_CASE_ID = CELL55_GOLDEN_CASE_ID
CELL56_TASK_DRAFT_ID = CELL55_TASK_DRAFT_ID
CELL56_OLD_TARGET_EVIDENCE_ID = CELL55_OLD_TARGET_EVIDENCE_ID

CELL56_PROGRAM_ID = (
    "program:dar-jostojo-ye-khoshbakhti"
)

CELL56_EPISODE_NUMBER = 28

CELL56_TARGET_UTTERANCE_INDEX = 43


# ============================================================
# 3) Read actual Pilot metadata
# ============================================================

def cell56_collect_key_values(
    value,
    target_key,
):

    found = []


    if isinstance(
        value,
        dict,
    ):

        for key, item in value.items():

            if key == target_key:
                found.append(
                    copy.deepcopy(
                        item
                    )
                )


            found.extend(
                cell56_collect_key_values(
                    item,
                    target_key,
                )
            )


    elif isinstance(
        value,
        list,
    ):

        for item in value:

            found.extend(
                cell56_collect_key_values(
                    item,
                    target_key,
                )
            )


    return found


def cell56_unique_scalar_values(
    key,
):

    values = (
        cell56_collect_key_values(
            cell54_next_pilot_packet,
            key,
        )
    )


    return sorted(
        set(
            value
            for value
            in values
            if isinstance(
                value,
                (
                    str,
                    int,
                    float,
                    bool,
                ),
            )
        ),
        key=lambda value: str(
            value
        ),
    )


cell56_pilot_metadata = {
    "source_type":
        cell56_unique_scalar_values(
            "source_type"
        ),

    "stratum":
        cell56_unique_scalar_values(
            "stratum"
        ),

    "task_family":
        cell56_unique_scalar_values(
            "task_family"
        ),

    "question_archetype":
        cell56_unique_scalar_values(
            "question_archetype"
        ),

    "archetype":
        cell56_unique_scalar_values(
            "archetype"
        ),
}


# ============================================================
# 4) Load canonical extracted Transcript
# ============================================================

cell56_transcript_path = (
    cell55_repo_root
    /
    "data"
    /
    "phase2"
    /
    "interim"
    /
    "transcripts_extracted_v1.jsonl"
)


if not cell56_transcript_path.exists():

    cell56_failures.append(
        "TRANSCRIPT_CANONICAL_FILE_MISSING"
    )


cell56_episode_records = []


if cell56_transcript_path.exists():

    with cell56_transcript_path.open(
        "r",
        encoding="utf-8",
    ) as handle:

        for line_number, line in enumerate(
            handle,
            start=1,
        ):

            line = line.strip()


            if not line:
                continue


            try:

                record = json.loads(
                    line
                )

            except Exception:

                continue


            if (
                record.get(
                    "program_id"
                )
                !=
                CELL56_PROGRAM_ID
            ):
                continue


            source_identity = (
                record.get(
                    "source_identity"
                )
                or
                {}
            )


            if (
                source_identity.get(
                    "episode_number"
                )
                !=
                CELL56_EPISODE_NUMBER
            ):
                continue


            utterance_index = record.get(
                "utterance_index"
            )


            if not isinstance(
                utterance_index,
                int,
            ):
                continue


            content = (
                record.get(
                    "content"
                )
                or
                {}
            )


            source_text = (
                content.get(
                    "source_text"
                )
                or
                content.get(
                    "text"
                )
                or
                ""
            )


            if not isinstance(
                source_text,
                str,
            ):
                source_text = ""


            cell56_episode_records.append({
                "line_number":
                    line_number,

                "utterance_index":
                    utterance_index,

                "utterance_id":
                    record.get(
                        "utterance_id"
                    ),

                "source_text":
                    source_text.strip(),

                "source_locator":
                    copy.deepcopy(
                        record.get(
                            "source_locator"
                        )
                        or
                        {}
                    ),

                "speaker":
                    copy.deepcopy(
                        record.get(
                            "speaker"
                        )
                        or
                        {}
                    ),

                "raw_record":
                    record,
            })


cell56_episode_records = sorted(
    cell56_episode_records,
    key=lambda item: item[
        "utterance_index"
    ],
)


if not cell56_episode_records:

    cell56_failures.append(
        "EPISODE_28_RECORDS_NOT_FOUND"
    )


# ============================================================
# 5) Human-normalized Review text
# ============================================================

def cell56_review_text(
    record,
):

    text = record[
        "source_text"
    ]


    if (
        record[
            "utterance_index"
        ]
        ==
        CELL56_TARGET_UTTERANCE_INDEX
    ):

        text = text.replace(
            "ماسته",
            "ماسه",
        )


    return text


cell56_review_records = []


for record in cell56_episode_records:

    review_record = {
        key:
            copy.deepcopy(
                value
            )
        for key, value
        in record.items()
        if key
        !=
        "raw_record"
    }


    review_record[
        "review_text"
    ] = cell56_review_text(
        record
    )


    review_record[
        "human_transcript_correction_applied"
    ] = (
        record[
            "utterance_index"
        ]
        ==
        CELL56_TARGET_UTTERANCE_INDEX
        and
        "ماسته"
        in
        record[
            "source_text"
        ]
    )


    cell56_review_records.append(
        review_record
    )


# ============================================================
# 6) Persian search normalization
# ============================================================

def cell56_norm(
    text,
):

    if not isinstance(
        text,
        str,
    ):
        return ""


    text = (
        text
        .replace(
            "ي",
            "ی",
        )
        .replace(
            "ك",
            "ک",
        )
        .replace(
            "\u200c",
            " ",
        )
    )


    text = re.sub(
        r"\s+",
        " ",
        text,
    )


    return text.strip()


def cell56_find_first_index(
    phrases,
):

    for record in cell56_review_records:

        normalized = cell56_norm(
            record[
                "review_text"
            ]
        )


        if any(
            cell56_norm(
                phrase
            )
            in
            normalized

            for phrase
            in phrases
        ):

            return record[
                "utterance_index"
            ]


    return None


# ============================================================
# 7) Find semantic anchors in actual Source
# ============================================================

cell56_anchor_indices = {
    "sandbag_example":
        cell56_find_first_index([
            "گونی های ماسه",
            "گونی‌های ماسه",
        ]),

    "less_harm_wartime":
        cell56_find_first_index([
            "کمتر زخمی بشیم",
            "کمتر آسیب ببینیم",
        ]),

    "economic_war_training_question":
        cell56_find_first_index([
            "تو جنگ اقتصادی چه آموزش",
            "در جنگ اقتصادی چه آموزش",
        ]),

    "financial_literacy_necessity":
        cell56_find_first_index([
            "آموزش سواد مالی وقتی از بالا",
            "آموزش سواد مالی",
        ]),

    "better_financial_decisions":
        cell56_find_first_index([
            "بهتر تصمیم گیری مالی کنیم",
            "بهتر تصمیم‌گیری مالی کنیم",
        ]),

    "protect_family":
        cell56_find_first_index([
            "از خودمون و خانوادمون مواظبت",
            "از خودمان و خانواده",
        ]),

    "preserve_money_value":
        cell56_find_first_index([
            "ارزش پولمون رو حفظ",
            "ارزش پولمان را حفظ",
        ]),

    "less_economic_harm":
        cell56_find_first_index([
            "تا کمتر صدمه ببینیم",
            "تا کمتر آسیب ببینیم",
        ]),
}


# ============================================================
# 8) Local source window
# ============================================================

cell56_existing_anchor_values = [
    value
    for value
    in cell56_anchor_indices.values()
    if isinstance(
        value,
        int,
    )
]


if not cell56_existing_anchor_values:

    cell56_failures.append(
        "SEMANTIC_ANCHORS_NOT_FOUND"
    )


cell56_local_start = (
    min(
        cell56_existing_anchor_values
    )
    -
    3
    if cell56_existing_anchor_values
    else 38
)


cell56_local_end = (
    max(
        cell56_existing_anchor_values
    )
    +
    3
    if cell56_existing_anchor_values
    else 70
)


cell56_local_records = [
    record
    for record
    in cell56_review_records
    if (
        cell56_local_start
        <=
        record[
            "utterance_index"
        ]
        <=
        cell56_local_end
    )
]


# ============================================================
# 9) Candidate window builder
# ============================================================

def cell56_build_candidate(
    label,
    start_index,
    end_index,
    purpose,
):

    records = [
        record
        for record
        in cell56_review_records
        if (
            start_index
            <=
            record[
                "utterance_index"
            ]
            <=
            end_index
        )
    ]


    if not records:
        return None


    combined_text = "\n".join(
        record[
            "review_text"
        ]
        for record
        in records
        if record[
            "review_text"
        ]
    ).strip()


    raw_projection = [
        {
            "utterance_id":
                record[
                    "utterance_id"
                ],

            "utterance_index":
                record[
                    "utterance_index"
                ],

            "source_text":
                record[
                    "source_text"
                ],

            "review_text":
                record[
                    "review_text"
                ],

            "source_locator":
                copy.deepcopy(
                    record[
                        "source_locator"
                    ]
                ),

            "human_transcript_correction_applied":
                record[
                    "human_transcript_correction_applied"
                ],
        }
        for record
        in records
    ]


    identity_payload = {
        "pilot_case_id":
            CELL56_PILOT_ID,

        "label":
            label,

        "utterance_ids": [
            record[
                "utterance_id"
            ]
            for record
            in records
        ],

        "review_text":
            combined_text,
    }


    candidate_id = (
        "transcript-retarget-candidate:"
        +
        hashlib.sha256(
            cell10_canonical_json(
                identity_payload
            ).encode(
                "utf-8"
            )
        ).hexdigest()[
            :32
        ]
    )


    return {
        "candidate_id":
            candidate_id,

        "label":
            label,

        "purpose":
            purpose,

        "start_index":
            start_index,

        "end_index":
            end_index,

        "utterance_count":
            len(
                records
            ),

        "records":
            raw_projection,

        "combined_review_text":
            combined_text,

        "contains_sandbag_example":
            any(
                "گونی های ماسه"
                in
                cell56_norm(
                    record[
                        "review_text"
                    ]
                )
                for record
                in records
            ),

        "contains_financial_literacy":
            any(
                "سواد مالی"
                in
                record[
                    "review_text"
                ]
                for record
                in records
            ),

        "contains_better_decision":
            any(
                "تصمیم"
                in
                record[
                    "review_text"
                ]
                and
                "مالی"
                in
                record[
                    "review_text"
                ]
                for record
                in records
            ),

        "contains_less_harm":
            any(
                (
                    "کمتر صدمه"
                    in
                    record[
                        "review_text"
                    ]
                )
                or
                (
                    "کمتر آسیب"
                    in
                    record[
                        "review_text"
                    ]
                )
                for record
                in records
            ),
    }


# ============================================================
# 10) Create replacement candidates from actual anchors
# ============================================================

cell56_candidates = []


sandbag_index = (
    cell56_anchor_indices.get(
        "sandbag_example"
    )
)


economic_question_index = (
    cell56_anchor_indices.get(
        "economic_war_training_question"
    )
)


financial_literacy_index = (
    cell56_anchor_indices.get(
        "financial_literacy_necessity"
    )
)


better_decision_index = (
    cell56_anchor_indices.get(
        "better_financial_decisions"
    )
)


less_harm_index = (
    cell56_anchor_indices.get(
        "less_economic_harm"
    )
)


# Candidate 1: analogy + transition
if (
    isinstance(
        sandbag_index,
        int,
    )
    and
    isinstance(
        economic_question_index,
        int,
    )
):

    candidate = cell56_build_candidate(
        label=
            "WAR_ANALOGY_TO_ECONOMIC_WAR",

        start_index=
            max(
                0,
                sandbag_index
                -
                2,
            ),

        end_index=
            economic_question_index
            +
            2,

        purpose=
            (
                "Preserve the military-war analogy and its "
                "explicit transition to economic-war training."
            ),
    )


    if candidate:
        cell56_candidates.append(
            candidate
        )


# Candidate 2: financial literacy necessity
if (
    isinstance(
        financial_literacy_index,
        int,
    )
    and
    isinstance(
        less_harm_index,
        int,
    )
):

    candidate = cell56_build_candidate(
        label=
            "FINANCIAL_LITERACY_FOR_ECONOMIC_WAR",

        start_index=
            max(
                0,
                financial_literacy_index
                -
                3,
            ),

        end_index=
            less_harm_index
            +
            1,

        purpose=
            (
                "Capture the explicit argument that financial "
                "literacy is needed to reduce harm in an "
                "economic war."
            ),
    )


    if candidate:
        cell56_candidates.append(
            candidate
        )


# Candidate 3: focused actionable rationale
if (
    isinstance(
        better_decision_index,
        int,
    )
    and
    isinstance(
        less_harm_index,
        int,
    )
):

    candidate = cell56_build_candidate(
        label=
            "BETTER_FINANCIAL_DECISIONS_AND_LESS_HARM",

        start_index=
            max(
                0,
                better_decision_index
                -
                2,
            ),

        end_index=
            less_harm_index
            +
            1,

        purpose=
            (
                "Focus on better financial decisions, "
                "protecting the family, preserving money "
                "value, and reducing economic harm."
            ),
    )


    if candidate:
        cell56_candidates.append(
            candidate
        )


# ============================================================
# 11) Deduplicate candidates
# ============================================================

cell56_candidate_map = {}


for candidate in cell56_candidates:

    cell56_candidate_map[
        candidate[
            "candidate_id"
        ]
    ] = candidate


cell56_candidates = list(
    cell56_candidate_map.values()
)


cell56_candidates = sorted(
    cell56_candidates,
    key=lambda candidate: (
        -int(
            candidate[
                "contains_financial_literacy"
            ]
        ),
        -int(
            candidate[
                "contains_better_decision"
            ]
        ),
        -int(
            candidate[
                "contains_less_harm"
            ]
        ),
        candidate[
            "utterance_count"
        ],
        candidate[
            "candidate_id"
        ],
    ),
)


# ============================================================
# 12) Proposed QA — NOT Human accepted yet
# ============================================================

CELL56_PROPOSED_QUESTION = (
    "چرا آموزش سواد مالی در شرایط جنگ اقتصادی ضروری است؟"
)


CELL56_PROPOSED_ANSWER = (
    "چون در شرایط جنگ اقتصادی، مردم می‌توانند مستقیماً "
    "از پیامدهای اقتصادی آسیب ببینند. آموزش سواد مالی "
    "می‌تواند به آن‌ها کمک کند تصمیم‌های مالی بهتری "
    "بگیرند، از خود و خانواده‌شان بهتر مراقبت کنند، "
    "ارزش پولشان را بهتر حفظ کنند و در شرایط اقتصادی "
    "نامطلوب کمتر صدمه ببینند."
)


# ============================================================
# 13) Source support check for proposed answer
# ============================================================

cell56_all_candidate_text = cell56_norm(
    "\n".join(
        candidate[
            "combined_review_text"
        ]
        for candidate
        in cell56_candidates
    )
)


cell56_proposed_answer_grounding = {
    "economic_war":
        (
            "جنگ اقتصادی"
            in
            cell56_all_candidate_text
        ),

    "financial_literacy":
        (
            "سواد مالی"
            in
            cell56_all_candidate_text
        ),

    "better_financial_decisions":
        (
            "بهتر تصمیم گیری مالی"
            in
            cell56_all_candidate_text
            or
            "بهتر تصمیم گیری مالی کنیم"
            in
            cell56_all_candidate_text
        ),

    "protect_self_family":
        (
            "از خودمون و خانوادمون مواظبت"
            in
            cell56_all_candidate_text
        ),

    "preserve_money_value":
        (
            "ارزش پولمون رو حفظ"
            in
            cell56_all_candidate_text
        ),

    "less_harm":
        (
            "کمتر صدمه ببینیم"
            in
            cell56_all_candidate_text
            or
            "کمتر آسیب ببینیم"
            in
            cell56_all_candidate_text
        ),
}


cell56_proposed_answer_grounded = all(
    cell56_proposed_answer_grounding.values()
)


# ============================================================
# 14) Update Retarget workspace
# ============================================================

cell56_active_workspace = copy.deepcopy(
    cell55_persisted_active_case
)


cell56_active_workspace[
    "workflow_state"
] = "BLOCKED_REQUIRES_RETARGET"


cell56_active_workspace[
    "pilot_coverage_audit"
] = {
    "metadata":
        copy.deepcopy(
            cell56_pilot_metadata
        ),

    "source":
        "CURRENT_PILOT_PACKET",

    "coverage_preservation_required":
        True,
}


cell56_active_workspace[
    "source_context_reconstruction"
] = {
    "program_id":
        CELL56_PROGRAM_ID,

    "episode_number":
        CELL56_EPISODE_NUMBER,

    "anchor_indices":
        copy.deepcopy(
            cell56_anchor_indices
        ),

    "local_window_start":
        cell56_local_start,

    "local_window_end":
        cell56_local_end,

    "human_transcript_correction":
        copy.deepcopy(
            cell55_transcript_correction
        ),

    "external_lookup_used":
        False,
}


cell56_active_workspace[
    "retarget_review"
][
    "replacement_status"
] = "CANDIDATES_PENDING_HUMAN_SELECTION"


cell56_active_workspace[
    "retarget_candidates"
] = copy.deepcopy(
    cell56_candidates
)


cell56_active_workspace[
    "proposed_question"
] = {
    "question_text":
        CELL56_PROPOSED_QUESTION,

    "status":
        "PENDING_HUMAN_REVIEW",

    "human_decision":
        "PENDING",

    "is_source_evidence":
        False,
}


cell56_active_workspace[
    "proposed_answer"
] = {
    "answer_text":
        CELL56_PROPOSED_ANSWER,

    "status":
        "PREVIEW_ONLY",

    "grounding_check":
        copy.deepcopy(
            cell56_proposed_answer_grounding
        ),

    "grounded":
        cell56_proposed_answer_grounded,

    "human_decision":
        "NOT_REQUESTED_YET",
}


# ============================================================
# 15) Persist Registry
# ============================================================

cell56_persistent_registry_payload = copy.deepcopy(
    cell55_persisted_registry
)


cell56_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell56_active_workspace
)


cell56_persistent_registry_payload[
    "remaining_pilot_count"
] = 13


cell56_persistent_registry_payload[
    "continuation_state"
] = copy.deepcopy(
    cell55_continuation_state
)


cell56_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell56_persistent_registry_payload
    )
)


# ============================================================
# 16) Read-back validation
# ============================================================

cell56_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell56_persisted_active_case = (
    cell56_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell56_persistence_valid = all([
    cell56_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL56_PILOT_ID,

    cell56_persisted_active_case.get(
        "workflow_state"
    )
    ==
    "BLOCKED_REQUIRES_RETARGET",

    cell56_persisted_active_case.get(
        "retarget_review",
        {},
    ).get(
        "replacement_status"
    )
    ==
    "CANDIDATES_PENDING_HUMAN_SELECTION",

    cell56_persisted_active_case.get(
        "proposed_question",
        {},
    ).get(
        "human_decision"
    )
    ==
    "PENDING",

    cell56_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    13,
])


# ============================================================
# 17) Safety
# ============================================================

cell56_queue_still_13 = all([
    len(
        cell54_remaining_pilot_ids
    )
    ==
    13,

    cell54_remaining_pilot_ids[
        0
    ]
    ==
    CELL56_PILOT_ID,
])


cell56_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell56_full_572_blocked = (
    cell55_full_572_blocked
    is True
)


# ============================================================
# 18) Final Gate
# ============================================================

cell56_gate = all([
    len(
        cell56_failures
    )
    ==
    0,

    len(
        cell56_episode_records
    )
    >
    0,

    len(
        cell56_candidates
    )
    >
    0,

    cell56_proposed_answer_grounded,

    cell56_persistence_valid,

    cell56_queue_still_13,

    cell56_base_seed_unchanged,

    cell56_full_572_blocked,
])


# ============================================================
# 19) Diagnostics
# ============================================================

print(
    "=== PILOT #11 COVERAGE AUDIT + "
    "ECONOMIC-WAR CONTEXT RECONSTRUCTION ==="
)


print(
    "\n=== PILOT METADATA ==="
)

print(
    "Pilot:",
    CELL56_PILOT_ID,
)

print(
    "Golden Case:",
    CELL56_GOLDEN_CASE_ID,
)

print(
    "Task Draft:",
    CELL56_TASK_DRAFT_ID,
)

print(
    "Old Target:",
    CELL56_OLD_TARGET_EVIDENCE_ID,
)


for key, values in (
    cell56_pilot_metadata.items()
):

    print(
        f"{key}:",
        values,
    )


print(
    "\n=== HUMAN TRANSCRIPT CORRECTION ==="
)

print(
    "Raw:",
    CELL55_RAW_TARGET_TEXT,
)

print(
    "Corrected:",
    cell55_corrected_target_text,
)

print(
    "Correction:",
    "ماسته -> ماسه",
)

print(
    "Raw canonical source mutated:",
    False,
)


print(
    "\n=== SEMANTIC ANCHORS ==="
)

for key, value in (
    cell56_anchor_indices.items()
):

    print(
        f"{key}:",
        value,
    )


print(
    "\n=== LOCAL SOURCE CONTEXT ==="
)


for record in cell56_local_records:

    marker = (
        "  <-- ORIGINAL TARGET"
        if
        record[
            "utterance_index"
        ]
        ==
        CELL56_TARGET_UTTERANCE_INDEX
        else
        ""
    )


    print(
        f"[{record['utterance_index']:05d}] "
        f"{record['review_text']}"
        f"{marker}"
    )


print(
    "\n=== RETARGET CANDIDATES ==="
)


for index, candidate in enumerate(
    cell56_candidates,
    start=1,
):

    print(
        "\n"
        +
        "="
        *
        72
    )

    print(
        "CANDIDATE",
        index,
    )

    print(
        "Candidate ID:",
        candidate[
            "candidate_id"
        ],
    )

    print(
        "Label:",
        candidate[
            "label"
        ],
    )

    print(
        "Purpose:",
        candidate[
            "purpose"
        ],
    )

    print(
        "Utterance Range:",
        (
            candidate[
                "start_index"
            ],
            candidate[
                "end_index"
            ],
        ),
    )

    print(
        "Utterance Count:",
        candidate[
            "utterance_count"
        ],
    )

    print(
        "Contains Financial Literacy:",
        candidate[
            "contains_financial_literacy"
        ],
    )

    print(
        "Contains Better Decision:",
        candidate[
            "contains_better_decision"
        ],
    )

    print(
        "Contains Less Harm:",
        candidate[
            "contains_less_harm"
        ],
    )

    print(
        "\nEvidence:"
    )

    print(
        candidate[
            "combined_review_text"
        ]
    )


print(
    "\n=== PROPOSED GOLDEN QUESTION ==="
)

print(
    CELL56_PROPOSED_QUESTION
)

print(
    "Human Decision:",
    "PENDING",
)


print(
    "\n=== PROPOSED ANSWER PREVIEW ==="
)

print(
    CELL56_PROPOSED_ANSWER
)


print(
    "\n=== ANSWER GROUNDING ==="
)

for key, status in (
    cell56_proposed_answer_grounding.items()
):

    print(
        f"{key}:",
        status,
    )


print(
    "Grounded:",
    cell56_proposed_answer_grounded,
)


print(
    "\n=== WORKFLOW / PERSISTENCE ==="
)

print(
    "Workflow:",
    cell56_persisted_active_case.get(
        "workflow_state"
    ),
)

print(
    "Retarget Candidate Status:",
    cell56_persisted_active_case.get(
        "retarget_review",
        {},
    ).get(
        "replacement_status"
    ),
)

print(
    "Persistence valid:",
    cell56_persistence_valid,
)


print(
    "\n=== QUEUE / SAFETY ==="
)

print(
    "Queue still 13:",
    cell56_queue_still_13,
)

print(
    "Base Seed unchanged:",
    cell56_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell56_full_572_blocked,
)

print(
    "External lookup:",
    False,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell56_failures
)


print(
    "\nCODE CELL 56 GATE:",
    (
        "PASS"
        if cell56_gate
        else "FAIL"
    ),
)


assert cell56_gate, (
    "Pilot #11 coverage audit / source context reconstruction "
    "failed. Repair this SAME Code Cell 56 before selecting "
    "Replacement Target Evidence."
)

=== PILOT #11 COVERAGE AUDIT + ECONOMIC-WAR CONTEXT RECONSTRUCTION ===

=== PILOT METADATA ===
Pilot: question-pilot:be503b9f0364162c19faa8401f9d7b0f
Golden Case: golden-case:c4fa6c10526f5d53df793ae0ef6ec0b0
Task Draft: task-draft:31d8142e21a1211f06fa49eda7295256
Old Target: target-evidence:6348f61441b9a5816156190f47b087c0
source_type: ['transcript']
stratum: ['single_speaker_primary']
task_family: ['PRIMARY_TRANSCRIPT_RETRIEVAL']
question_archetype: ['AUTHORITY_CONSTRAINED_RETRIEVAL']
archetype: []

=== HUMAN TRANSCRIPT CORRECTION ===
Raw: یا گونی های ماسته بچینید و موقعی موشک بارون مثلا در زیر زمین قرار بگیرید که اطرافش گونی های ماسته چیده شده
Corrected: یا گونی های ماسه بچینید و موقعی موشک بارون مثلا در زیر زمین قرار بگیرید که اطرافش گونی های ماسه چیده شده
Correction: ماسته -> ماسه
Raw canonical source mutated: False

=== SEMANTIC ANCHORS ===
sandbag_example: 43
less_harm_wartime: 44
economic_war_training_question: 46
financial_literacy_necessity: 56
better_financial_decisions: 68
p

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 57 — Pilot #11 Replacement Target + Question/Answer ACCEPT + Citation Draft

Human Reviewer، Candidate زیر را برای Retarget پذیرفت:

**`BETTER_FINANCIAL_DECISIONS_AND_LESS_HARM`**

از Episode 28 برنامه «در جستجوی خوشبختی»، Utteranceهای 66 تا 79.

سؤال پذیرفته‌شده:

**چرا آموزش سواد مالی در شرایط جنگ اقتصادی ضروری است؟**

جواب پذیرفته‌شده:

**چون آموزش سواد مالی کمک می‌کند افراد تصمیم‌های مالی بهتری بگیرند، از خود و خانواده‌شان بهتر مراقبت کنند، ارزش پولشان را بهتر حفظ کنند و در شرایط اقتصادی نامطلوب کمتر آسیب ببینند.**

در این مرحله:

- Target اولیه «گونی‌های ماسه» Retarget می‌شود؛
- اصلاح Transcript قبلی و Raw Source Lineage حفظ می‌شود؛
- Replacement Target فقط از Transcript واقعی Episode 28 ساخته می‌شود؛
- Attribution همه Utteranceها به Dr. Komeil Roudi کنترل می‌شود؛
- Question و Expected Answer هر دو با Human Decision برابر `ACCEPT` ثبت می‌شوند؛
- Citation Mapping از همان Replacement Evidence ساخته می‌شود؛
- Citation وارد Human Review می‌شود؛
- Queue همچنان 13 باقی می‌ماند.

</div>

In [57]:
# Code Cell 57

cell57_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell56_gate") is not True:
    cell57_failures.append(
        "CELL56_NOT_PASS"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell57_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Human-approved Candidate / QA
# ============================================================

CELL57_SELECTED_CANDIDATE_ID = (
    "transcript-retarget-candidate:"
    "4ccc40bcfcb7f89f0a597f6d76a23b0d"
)


CELL57_ACCEPTED_QUESTION = (
    "چرا آموزش سواد مالی در شرایط جنگ اقتصادی ضروری است؟"
)


CELL57_ACCEPTED_ANSWER = (
    "چون آموزش سواد مالی کمک می‌کند افراد تصمیم‌های مالی "
    "بهتری بگیرند، از خود و خانواده‌شان بهتر مراقبت کنند، "
    "ارزش پولشان را بهتر حفظ کنند و در شرایط اقتصادی "
    "نامطلوب کمتر آسیب ببینند."
)


CELL57_QUESTION_HUMAN_NOTE = (
    "Human Reviewer accepted the question after reviewing "
    "the broader economic-war context."
)


CELL57_ANSWER_HUMAN_NOTE = (
    "Human Reviewer explicitly accepted the answer together "
    "with the question."
)


# ============================================================
# 3) Resolve selected Candidate
# ============================================================

cell57_selected_matches = [
    candidate
    for candidate
    in cell56_candidates
    if (
        candidate.get(
            "candidate_id"
        )
        ==
        CELL57_SELECTED_CANDIDATE_ID
    )
]


cell57_selected_candidate = (
    copy.deepcopy(
        cell57_selected_matches[
            0
        ]
    )
    if
    len(
        cell57_selected_matches
    )
    ==
    1
    else
    None
)


cell57_candidate_unique = (
    len(
        cell57_selected_matches
    )
    ==
    1
)


if not cell57_candidate_unique:
    cell57_failures.append(
        "SELECTED_CANDIDATE_NOT_UNIQUE"
    )


cell57_candidate_valid = all([
    cell57_candidate_unique,

    cell57_selected_candidate.get(
        "label"
    )
    ==
    "BETTER_FINANCIAL_DECISIONS_AND_LESS_HARM",

    cell57_selected_candidate.get(
        "start_index"
    )
    ==
    66,

    cell57_selected_candidate.get(
        "end_index"
    )
    ==
    79,

    cell57_selected_candidate.get(
        "utterance_count"
    )
    ==
    14,

    cell57_selected_candidate.get(
        "contains_financial_literacy"
    )
    is True,

    cell57_selected_candidate.get(
        "contains_better_decision"
    )
    is True,

    cell57_selected_candidate.get(
        "contains_less_harm"
    )
    is True,
])


# ============================================================
# 4) Resolve canonical Transcript records 66..79
# ============================================================

cell57_canonical_records = [
    copy.deepcopy(
        record
    )
    for record
    in cell56_episode_records
    if (
        66
        <=
        record[
            "utterance_index"
        ]
        <=
        79
    )
]


cell57_canonical_records = sorted(
    cell57_canonical_records,
    key=lambda record:
        record[
            "utterance_index"
        ],
)


cell57_expected_indices = list(
    range(
        66,
        80,
    )
)


cell57_canonical_indices = [
    record[
        "utterance_index"
    ]
    for record
    in cell57_canonical_records
]


cell57_canonical_range_valid = all([
    cell57_canonical_indices
    ==
    cell57_expected_indices,

    len(
        cell57_canonical_records
    )
    ==
    14,
])


# ============================================================
# 5) Authority validation
# ============================================================

cell57_authority_rows = []


for record in cell57_canonical_records:

    raw_record = (
        record.get(
            "raw_record"
        )
        or
        {}
    )


    speaker = (
        raw_record.get(
            "speaker"
        )
        or
        {}
    )


    effective = (
        speaker.get(
            "effective"
        )
        or
        {}
    )


    authority_valid = all([
        effective.get(
            "canonical"
        )
        ==
        "Dr. Komeil Roudi",

        effective.get(
            "primary_dr_roudi_corpus"
        )
        is True,
    ])


    cell57_authority_rows.append({
        "utterance_index":
            record[
                "utterance_index"
            ],

        "utterance_id":
            record[
                "utterance_id"
            ],

        "speaker":
            effective.get(
                "canonical"
            ),

        "primary_dr_roudi_corpus":
            effective.get(
                "primary_dr_roudi_corpus"
            ),

        "valid":
            authority_valid,
    })


cell57_authority_valid = all([
    len(
        cell57_authority_rows
    )
    ==
    14,

    all(
        row[
            "valid"
        ]
        is True
        for row
        in cell57_authority_rows
    ),
])


# ============================================================
# 6) Build Replacement Target Parts
# ============================================================

cell57_replacement_target_parts = []


for record in cell57_canonical_records:

    raw_record = (
        record.get(
            "raw_record"
        )
        or
        {}
    )


    source_identity = copy.deepcopy(
        raw_record.get(
            "source_identity"
        )
        or
        {}
    )


    source_locator = copy.deepcopy(
        raw_record.get(
            "source_locator"
        )
        or
        {}
    )


    transcript_unit_id = (
        raw_record.get(
            "transcript_unit_id"
        )
    )


    source_text = (
        record[
            "source_text"
        ]
    )


    utterance_id = (
        record[
            "utterance_id"
        ]
    )


    cell57_replacement_target_parts.append({
        "document_id":
            utterance_id,

        "source_id":
            transcript_unit_id,

        "source_type":
            "transcript",

        "role":
            "PRIMARY_TRANSCRIPT_MEMBER",

        "char_start":
            0,

        "char_end":
            len(
                source_text
            ),

        "char_count":
            len(
                source_text
            ),

        "text":
            source_text,

        "text_sha256":
            hashlib.sha256(
                source_text.encode(
                    "utf-8"
                )
            ).hexdigest(),

        "utterance_index":
            record[
                "utterance_index"
            ],

        "source_locator":
            source_locator,

        "source_identity":
            source_identity,

        "speaker": {
            "canonical":
                "Dr. Komeil Roudi",

            "primary_dr_roudi_corpus":
                True,
        },

        "origin":
            "CANONICAL_TRANSCRIPT_EXTRACTION_V1",
    })


cell57_combined_target_text = (
    "\n".join(
        part[
            "text"
        ]
        for part
        in cell57_replacement_target_parts
    )
)


cell57_parts_valid = all([
    len(
        cell57_replacement_target_parts
    )
    ==
    14,

    [
        part[
            "utterance_index"
        ]
        for part
        in cell57_replacement_target_parts
    ]
    ==
    cell57_expected_indices,

    all(
        part.get(
            "document_id"
        )
        ==
        (
            "program:dar-jostojo-ye-khoshbakhti:"
            f"episode-028:utterance:{part['utterance_index']:05d}"
        )
        for part
        in cell57_replacement_target_parts
    ),

    all(
        part.get(
            "source_type"
        )
        ==
        "transcript"
        for part
        in cell57_replacement_target_parts
    ),

    all(
        part.get(
            "role"
        )
        ==
        "PRIMARY_TRANSCRIPT_MEMBER"
        for part
        in cell57_replacement_target_parts
    ),
])


# ============================================================
# 7) Replacement Target deterministic identity
# ============================================================

cell57_target_identity_payload = {
    "identity_version":
        "transcript_retarget_target_evidence_identity_v1",

    "pilot_case_id":
        CELL56_PILOT_ID,

    "old_target_evidence_id":
        CELL56_OLD_TARGET_EVIDENCE_ID,

    "selected_candidate_id":
        CELL57_SELECTED_CANDIDATE_ID,

    "program_id":
        CELL56_PROGRAM_ID,

    "episode_number":
        CELL56_EPISODE_NUMBER,

    "utterance_projection": [
        {
            "document_id":
                part[
                    "document_id"
                ],

            "utterance_index":
                part[
                    "utterance_index"
                ],

            "char_start":
                part[
                    "char_start"
                ],

            "char_end":
                part[
                    "char_end"
                ],

            "text_sha256":
                part[
                    "text_sha256"
                ],
        }
        for part
        in cell57_replacement_target_parts
    ],
}


cell57_replacement_target_evidence_id = (
    "target-evidence:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell57_target_identity_payload
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


# ============================================================
# 8) Replacement Target record
# ============================================================

cell57_replacement_target_evidence = {
    "record_version":
        "replacement_target_evidence_v1",

    "target_evidence_id":
        cell57_replacement_target_evidence_id,

    "pilot_case_id":
        CELL56_PILOT_ID,

    "golden_case_id":
        CELL56_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL56_TASK_DRAFT_ID,

    "source_type":
        "transcript",

    "stratum":
        "single_speaker_primary",

    "task_family":
        "PRIMARY_TRANSCRIPT_RETRIEVAL",

    "question_archetype":
        "AUTHORITY_CONSTRAINED_RETRIEVAL",

    "selected_candidate_id":
        CELL57_SELECTED_CANDIDATE_ID,

    "program_id":
        CELL56_PROGRAM_ID,

    "episode_number":
        CELL56_EPISODE_NUMBER,

    "speaker_constraint":
        "Dr. Komeil Roudi",

    "primary_corpus_required":
        True,

    "target_parts":
        copy.deepcopy(
            cell57_replacement_target_parts
        ),

    "combined_target_text":
        cell57_combined_target_text,

    "retarget_lineage": {
        "old_target_evidence_id":
            CELL56_OLD_TARGET_EVIDENCE_ID,

        "old_target_human_decision":
            "RETARGET",

        "replacement_target_evidence_id":
            cell57_replacement_target_evidence_id,

        "replacement_human_decision":
            "ACCEPT",
    },

    "transcript_correction_lineage":
        copy.deepcopy(
            cell55_transcript_correction
        ),

    "external_lookup_used":
        False,
}


# ============================================================
# 9) Rebuild Pilot packet with Replacement Target
# ============================================================

def cell57_replace_target_reference(
    value,
):

    if isinstance(
        value,
        dict,
    ):

        output = {}


        for key, item in value.items():

            if key == "target_text_parts":

                output[
                    key
                ] = copy.deepcopy(
                    cell57_replacement_target_parts
                )


            elif key == "target_evidence_id":

                output[
                    key
                ] = (
                    cell57_replacement_target_evidence_id
                )


            else:

                output[
                    key
                ] = (
                    cell57_replace_target_reference(
                        item
                    )
                )


        return output


    if isinstance(
        value,
        list,
    ):

        return [
            cell57_replace_target_reference(
                item
            )
            for item
            in value
        ]


    if (
        isinstance(
            value,
            str,
        )
        and
        value
        ==
        CELL56_OLD_TARGET_EVIDENCE_ID
    ):

        return (
            cell57_replacement_target_evidence_id
        )


    return copy.deepcopy(
        value
    )


cell57_retargeted_pilot_packet = (
    cell57_replace_target_reference(
        cell54_next_pilot_packet
    )
)


cell57_packet_target_id = (
    cell13_exact_single_id(
        cell57_retargeted_pilot_packet,
        "target_evidence",
    )
)


cell57_packet_valid = (
    cell57_packet_target_id
    ==
    cell57_replacement_target_evidence_id
)


# ============================================================
# 10) Restart Engine
# ============================================================

cell57_question_pending_session = (
    cell13_build_session(
        cell57_retargeted_pilot_packet
    )
)


cell57_session_start_valid = (
    cell57_question_pending_session.get(
        "current_state"
    )
    ==
    "QUESTION_DRAFT_PENDING"
)


# ============================================================
# 11) Grounding validation
# ============================================================

cell57_source_surface = cell56_norm(
    cell57_combined_target_text
)


cell57_grounding = {
    "economic_war":
        (
            "جنگ اقتصادی"
            in
            cell57_source_surface
        ),

    "financial_literacy":
        (
            "سواد مالی"
            in
            cell57_source_surface
        ),

    "better_financial_decisions":
        (
            "بهتر تصمیم گیری مالی"
            in
            cell57_source_surface
        ),

    "protect_self_family":
        (
            "از خودمون و خانوادمون مواظبت"
            in
            cell57_source_surface
        ),

    "preserve_money_value":
        (
            "ارزش پولمون رو حفظ"
            in
            cell57_source_surface
        ),

    "less_harm":
        (
            "کمتر صدمه ببینیم"
            in
            cell57_source_surface
            or
            "کمتر آسیب ببینیم"
            in
            cell57_source_surface
        ),
}


cell57_grounding_pass = all(
    cell57_grounding.values()
)


# ============================================================
# 12) Question Draft
# ============================================================

cell57_question_identity = {
    "identity_version":
        "question_draft_identity_v1",

    "pilot_case_id":
        CELL56_PILOT_ID,

    "target_evidence_id":
        cell57_replacement_target_evidence_id,

    "question_text":
        CELL57_ACCEPTED_QUESTION,

    "revision_number":
        1,
}


cell57_question_draft_id = (
    "question-draft:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell57_question_identity
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


cell57_question_record = {
    "record_version":
        "question_draft_record_v1",

    "question_draft_id":
        cell57_question_draft_id,

    "pilot_case_id":
        CELL56_PILOT_ID,

    "golden_case_id":
        CELL56_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL56_TASK_DRAFT_ID,

    "target_evidence_id":
        cell57_replacement_target_evidence_id,

    "question_text":
        CELL57_ACCEPTED_QUESTION,

    "revision_number":
        1,

    "machine_validation_status":
        (
            "PASS_PENDING_HUMAN_ADJUDICATION"
            if cell57_grounding_pass
            else
            "FAIL_REQUIRES_REVISION"
        ),

    "human_adjudication_status":
        "PENDING",

    "external_knowledge_used":
        False,
}


cell57_question_review_session = (
    cell13_apply_event(
        cell57_question_pending_session,
        "QUESTION_DRAFT_REGISTERED",
        artifact=
            cell57_question_record,
    )
)


cell57_answer_pending_session = (
    cell13_apply_event(
        cell57_question_review_session,
        "QUESTION_ACCEPTED",
        human_note=
            CELL57_QUESTION_HUMAN_NOTE,
    )
)


cell57_question_accept_valid = all([
    cell57_question_review_session.get(
        "current_state"
    )
    ==
    "QUESTION_HUMAN_REVIEW_PENDING",

    cell57_answer_pending_session.get(
        "current_state"
    )
    ==
    "EXPECTED_ANSWER_DRAFT_PENDING",
])


# ============================================================
# 13) Expected Answer Draft
# ============================================================

cell57_answer_identity = {
    "identity_version":
        "expected_answer_identity_v1",

    "pilot_case_id":
        CELL56_PILOT_ID,

    "question_draft_id":
        cell57_question_draft_id,

    "target_evidence_id":
        cell57_replacement_target_evidence_id,

    "answer_text":
        CELL57_ACCEPTED_ANSWER,

    "revision_number":
        1,
}


cell57_expected_answer_draft_id = (
    "expected-answer-draft:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell57_answer_identity
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


cell57_expected_answer_record = {
    "record_version":
        "expected_answer_draft_record_v1",

    "expected_answer_draft_id":
        cell57_expected_answer_draft_id,

    "pilot_case_id":
        CELL56_PILOT_ID,

    "golden_case_id":
        CELL56_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL56_TASK_DRAFT_ID,

    "question_draft_id":
        cell57_question_draft_id,

    "target_evidence_id":
        cell57_replacement_target_evidence_id,

    "expected_answer_text":
        CELL57_ACCEPTED_ANSWER,

    "revision_number":
        1,

    "machine_validation_status":
        (
            "PASS_PENDING_HUMAN_ADJUDICATION"
            if cell57_grounding_pass
            else
            "FAIL_REQUIRES_REVISION"
        ),

    "human_adjudication_status":
        "PENDING",

    "external_knowledge_used":
        False,
}


cell57_answer_review_session = (
    cell13_apply_event(
        cell57_answer_pending_session,
        "EXPECTED_ANSWER_REGISTERED",
        artifact=
            cell57_expected_answer_record,
    )
)


cell57_citation_pending_session = (
    cell13_apply_event(
        cell57_answer_review_session,
        "EXPECTED_ANSWER_ACCEPTED",
        human_note=
            CELL57_ANSWER_HUMAN_NOTE,
    )
)


cell57_answer_accept_valid = all([
    cell57_answer_review_session.get(
        "current_state"
    )
    ==
    "EXPECTED_ANSWER_HUMAN_REVIEW_PENDING",

    cell57_citation_pending_session.get(
        "current_state"
    )
    ==
    "CITATION_MAPPING_PENDING",
])


# ============================================================
# 14) Build Citation entries
# ============================================================

cell57_citation_entries = []


for index, part in enumerate(
    cell57_replacement_target_parts
):

    cell57_citation_entries.append({
        "citation_index":
            index,

        "target_evidence_id":
            cell57_replacement_target_evidence_id,

        "document_id":
            part[
                "document_id"
            ],

        "source_id":
            part[
                "source_id"
            ],

        "role":
            part[
                "role"
            ],

        "char_start":
            part[
                "char_start"
            ],

        "char_end":
            part[
                "char_end"
            ],

        "utterance_index":
            part[
                "utterance_index"
            ],

        "source_locator":
            copy.deepcopy(
                part[
                    "source_locator"
                ]
            ),

        "speaker":
            copy.deepcopy(
                part[
                    "speaker"
                ]
            ),

        "locator_origin":
            "COPIED_FROM_REPLACEMENT_TARGET_EVIDENCE",

        "external_locator_lookup_used":
            False,
    })


cell57_citation_projection = [
    {
        "document_id":
            entry[
                "document_id"
            ],

        "source_id":
            entry[
                "source_id"
            ],

        "role":
            entry[
                "role"
            ],

        "char_start":
            entry[
                "char_start"
            ],

        "char_end":
            entry[
                "char_end"
            ],

        "utterance_index":
            entry[
                "utterance_index"
            ],

        "source_locator":
            copy.deepcopy(
                entry[
                    "source_locator"
                ]
            ),
    }
    for entry
    in cell57_citation_entries
]


cell57_target_projection = [
    {
        "document_id":
            part[
                "document_id"
            ],

        "source_id":
            part[
                "source_id"
            ],

        "role":
            part[
                "role"
            ],

        "char_start":
            part[
                "char_start"
            ],

        "char_end":
            part[
                "char_end"
            ],

        "utterance_index":
            part[
                "utterance_index"
            ],

        "source_locator":
            copy.deepcopy(
                part[
                    "source_locator"
                ]
            ),
    }
    for part
    in cell57_replacement_target_parts
]


cell57_no_locator_fabrication = (
    cell57_citation_projection
    ==
    cell57_target_projection
)


# ============================================================
# 15) Citation Mapping ID
# ============================================================

cell57_citation_identity = {
    "identity_version":
        "citation_mapping_identity_v1",

    "pilot_case_id":
        CELL56_PILOT_ID,

    "question_draft_id":
        cell57_question_draft_id,

    "expected_answer_draft_id":
        cell57_expected_answer_draft_id,

    "target_evidence_id":
        cell57_replacement_target_evidence_id,

    "citation_projection":
        copy.deepcopy(
            cell57_citation_projection
        ),
}


cell57_citation_mapping_id = (
    "citation-mapping:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell57_citation_identity
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


cell57_citation_machine_pass = all([
    len(
        cell57_citation_entries
    )
    ==
    14,

    [
        entry[
            "utterance_index"
        ]
        for entry
        in cell57_citation_entries
    ]
    ==
    cell57_expected_indices,

    cell57_no_locator_fabrication,

    all(
        entry[
            "target_evidence_id"
        ]
        ==
        cell57_replacement_target_evidence_id
        for entry
        in cell57_citation_entries
    ),

    all(
        entry[
            "speaker"
        ].get(
            "canonical"
        )
        ==
        "Dr. Komeil Roudi"
        for entry
        in cell57_citation_entries
    ),

    all(
        entry[
            "speaker"
        ].get(
            "primary_dr_roudi_corpus"
        )
        is True
        for entry
        in cell57_citation_entries
    ),

    all(
        entry[
            "external_locator_lookup_used"
        ]
        is False
        for entry
        in cell57_citation_entries
    ),
])


cell57_citation_mapping_record = {
    "record_version":
        "citation_mapping_record_v1",

    "citation_mapping_id":
        cell57_citation_mapping_id,

    "pilot_case_id":
        CELL56_PILOT_ID,

    "golden_case_id":
        CELL56_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL56_TASK_DRAFT_ID,

    "target_evidence_id":
        cell57_replacement_target_evidence_id,

    "old_target_evidence_id":
        CELL56_OLD_TARGET_EVIDENCE_ID,

    "question_draft_id":
        cell57_question_draft_id,

    "expected_answer_draft_id":
        cell57_expected_answer_draft_id,

    "citation_entries":
        copy.deepcopy(
            cell57_citation_entries
        ),

    "citation_scope":
        "REPLACEMENT_TARGET_EVIDENCE_ONLY",

    "authority_constraint":
        "Dr. Komeil Roudi",

    "retarget_used":
        True,

    "locator_source":
        "EXISTING_TRANSCRIPT_SOURCE_LOCATORS",

    "external_locator_lookup_used":
        False,

    "machine_validation_status":
        (
            "PASS_PENDING_HUMAN_ADJUDICATION"
            if cell57_citation_machine_pass
            else
            "FAIL_REQUIRES_REVISION"
        ),

    "human_adjudication_status":
        "PENDING",
}


cell57_current_session = (
    cell13_apply_event(
        cell57_citation_pending_session,
        "CITATION_MAPPING_REGISTERED",
        artifact=
            cell57_citation_mapping_record,
    )
)


cell57_citation_registered_valid = (
    cell57_current_session.get(
        "current_state"
    )
    ==
    "CITATION_HUMAN_REVIEW_PENDING"
)


# ============================================================
# 16) Update active Human Review workspace
# ============================================================

cell57_active_workspace = copy.deepcopy(
    cell56_persisted_active_case
)


cell57_active_workspace[
    "status"
] = "ACTIVE_HUMAN_REVIEW"


cell57_active_workspace[
    "workflow_state"
] = "CITATION_HUMAN_REVIEW_PENDING"


cell57_active_workspace[
    "target_evidence_id"
] = (
    cell57_replacement_target_evidence_id
)


cell57_active_workspace[
    "target_evidence"
] = copy.deepcopy(
    cell57_replacement_target_evidence
)


cell57_active_workspace[
    "source_evidence"
] = copy.deepcopy(
    cell57_replacement_target_parts
)


cell57_active_workspace[
    "combined_target_text"
] = (
    cell57_combined_target_text
)


cell57_active_workspace[
    "retarget_review"
][
    "selected_candidate_id"
] = (
    CELL57_SELECTED_CANDIDATE_ID
)


cell57_active_workspace[
    "retarget_review"
][
    "replacement_target_evidence_id"
] = (
    cell57_replacement_target_evidence_id
)


cell57_active_workspace[
    "retarget_review"
][
    "replacement_status"
] = "ACCEPT"


cell57_active_workspace[
    "question_review"
] = {
    "question_draft_id":
        cell57_question_draft_id,

    "question_text":
        CELL57_ACCEPTED_QUESTION,

    "machine_validation_status":
        cell57_question_record[
            "machine_validation_status"
        ],

    "human_decision":
        "ACCEPT",

    "human_note":
        CELL57_QUESTION_HUMAN_NOTE,
}


cell57_active_workspace[
    "expected_answer_review"
] = {
    "expected_answer_draft_id":
        cell57_expected_answer_draft_id,

    "expected_answer_text":
        CELL57_ACCEPTED_ANSWER,

    "machine_validation_status":
        cell57_expected_answer_record[
            "machine_validation_status"
        ],

    "human_decision":
        "ACCEPT",

    "human_note":
        CELL57_ANSWER_HUMAN_NOTE,
}


cell57_active_workspace[
    "citation_review"
] = {
    "citation_mapping_id":
        cell57_citation_mapping_id,

    "citation_entries":
        copy.deepcopy(
            cell57_citation_entries
        ),

    "machine_validation_status":
        cell57_citation_mapping_record[
            "machine_validation_status"
        ],

    "human_decision":
        "PENDING",
}


cell57_active_workspace[
    "author_final_review"
] = {
    "status":
        "NOT_STARTED",

    "decision":
        None,

    "notes":
        None,
}


# ============================================================
# 17) Updated continuation state
# ============================================================

cell57_continuation_core = {
    "next_pilot_id":
        CELL56_PILOT_ID,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell54_remaining_pilot_ids
        ),

    "remaining_count":
        13,

    "active_pilot_status":
        "CITATION_HUMAN_REVIEW_PENDING",

    "active_target_evidence_id":
        cell57_replacement_target_evidence_id,

    "retarget_status":
        "REPLACEMENT_ACCEPTED",

    "last_completed_pilot_id":
        cell54_continuation_state.get(
            "last_completed_pilot_id"
        ),

    "last_completion_type":
        cell54_continuation_state.get(
            "last_completion_type"
        ),
}


cell57_continuation_state_sha256 = (
    cell10_stable_sha(
        cell57_continuation_core
    )
)


cell57_continuation_state = {
    **copy.deepcopy(
        cell57_continuation_core
    ),

    "continuation_state_sha256":
        cell57_continuation_state_sha256,
}


# ============================================================
# 18) Persist Registry
# ============================================================

cell57_persistent_registry_payload = copy.deepcopy(
    cell56_persisted_registry
)


cell57_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell57_active_workspace
)


cell57_persistent_registry_payload[
    "remaining_pilot_count"
] = 13


cell57_persistent_registry_payload[
    "continuation_state"
] = copy.deepcopy(
    cell57_continuation_state
)


cell57_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell57_persistent_registry_payload
    )
)


# ============================================================
# 19) Read-back validation
# ============================================================

cell57_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell57_persisted_active_case = (
    cell57_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell57_persistence_valid = all([
    cell57_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL56_PILOT_ID,

    cell57_persisted_active_case.get(
        "target_evidence_id"
    )
    ==
    cell57_replacement_target_evidence_id,

    cell57_persisted_active_case.get(
        "workflow_state"
    )
    ==
    "CITATION_HUMAN_REVIEW_PENDING",

    cell57_persisted_active_case.get(
        "retarget_review",
        {},
    ).get(
        "replacement_status"
    )
    ==
    "ACCEPT",

    cell57_persisted_active_case.get(
        "question_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell57_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell57_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "PENDING",

    cell57_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    13,

    cell57_persisted_registry.get(
        "continuation_state",
        {},
    ).get(
        "continuation_state_sha256"
    )
    ==
    cell57_continuation_state_sha256,
])


# ============================================================
# 20) Safety
# ============================================================

cell57_queue_still_13 = all([
    len(
        cell54_remaining_pilot_ids
    )
    ==
    13,

    cell54_remaining_pilot_ids[
        0
    ]
    ==
    CELL56_PILOT_ID,
])


cell57_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell57_full_572_blocked = (
    cell56_full_572_blocked
    is True
)


# ============================================================
# 21) Final Gate
# ============================================================

cell57_gate = all([
    len(
        cell57_failures
    )
    ==
    0,

    cell57_candidate_valid,

    cell57_canonical_range_valid,

    cell57_authority_valid,

    cell57_parts_valid,

    cell57_packet_valid,

    cell57_session_start_valid,

    cell57_grounding_pass,

    cell57_question_accept_valid,

    cell57_answer_accept_valid,

    cell57_no_locator_fabrication,

    cell57_citation_machine_pass,

    cell57_citation_registered_valid,

    cell57_persistence_valid,

    cell57_queue_still_13,

    cell57_base_seed_unchanged,

    cell57_full_572_blocked,
])


# ============================================================
# 22) Diagnostics
# ============================================================

print(
    "=== PILOT #11 REPLACEMENT TARGET + "
    "QUESTION/ANSWER ACCEPT + CITATION DRAFT ==="
)


print(
    "\n=== RETARGET LINEAGE ==="
)

print(
    "Pilot:",
    CELL56_PILOT_ID,
)

print(
    "Old Target:",
    CELL56_OLD_TARGET_EVIDENCE_ID,
)

print(
    "Selected Candidate:",
    CELL57_SELECTED_CANDIDATE_ID,
)

print(
    "Replacement Target:",
    cell57_replacement_target_evidence_id,
)

print(
    "Utterance Range:",
    (
        66,
        79,
    ),
)

print(
    "Source Type:",
    "transcript",
)

print(
    "Stratum:",
    "single_speaker_primary",
)

print(
    "Task Family:",
    "PRIMARY_TRANSCRIPT_RETRIEVAL",
)

print(
    "Question Archetype:",
    "AUTHORITY_CONSTRAINED_RETRIEVAL",
)


print(
    "\n=== AUTHORITY VALIDATION ==="
)

print(
    "All 14 utterances attributed to Dr. Komeil Roudi:",
    cell57_authority_valid,
)


for row in cell57_authority_rows:

    print(
        (
            row[
                "utterance_index"
            ],
            row[
                "speaker"
            ],
            row[
                "primary_dr_roudi_corpus"
            ],
            row[
                "valid"
            ],
        )
    )


print(
    "\n=== REPLACEMENT SOURCE EVIDENCE ==="
)

print(
    cell57_combined_target_text
)


print(
    "\n=== ACCEPTED QUESTION ==="
)

print(
    CELL57_ACCEPTED_QUESTION
)

print(
    "Question Draft ID:",
    cell57_question_draft_id,
)

print(
    "Human Decision:",
    "ACCEPT",
)


print(
    "\n=== ACCEPTED ANSWER ==="
)

print(
    CELL57_ACCEPTED_ANSWER
)

print(
    "Expected Answer Draft ID:",
    cell57_expected_answer_draft_id,
)

print(
    "Human Decision:",
    "ACCEPT",
)


print(
    "\n=== GROUNDING ==="
)

for name, status in (
    cell57_grounding.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "Grounding PASS:",
    cell57_grounding_pass,
)


print(
    "\n=== CITATION DRAFT ==="
)

print(
    "Citation Mapping ID:",
    cell57_citation_mapping_id,
)

print(
    "Citation entry count:",
    len(
        cell57_citation_entries
    ),
)

print(
    "Utterance indices:",
    [
        entry[
            "utterance_index"
        ]
        for entry
        in cell57_citation_entries
    ],
)

print(
    "No locator fabrication:",
    cell57_no_locator_fabrication,
)

print(
    "Citation Machine PASS:",
    cell57_citation_machine_pass,
)


print(
    "\n=== TRANSCRIPT CORRECTION LINEAGE ==="
)

print(
    "Original typo:",
    "ماسته",
)

print(
    "Human correction:",
    "ماسه",
)

print(
    "Correction used inside Replacement Target:",
    False,
)

print(
    "Reason:",
    (
        "Replacement Target uses utterances 66..79; "
        "the corrected utterance was 43."
    ),
)

print(
    "Raw canonical source mutated:",
    False,
)


print(
    "\n=== WORKFLOW ==="
)

print(
    "Current State:",
    cell57_current_session.get(
        "current_state"
    ),
)

print(
    "Retarget:",
    "ACCEPT",
)

print(
    "Question:",
    "ACCEPT",
)

print(
    "Expected Answer:",
    "ACCEPT",
)

print(
    "Citation:",
    "PENDING",
)


print(
    "\n=== PERSISTENCE ==="
)

print(
    "Registry:",
    cell21_registry_path,
)

print(
    "Persistence valid:",
    cell57_persistence_valid,
)


print(
    "\n=== QUEUE / SAFETY ==="
)

print(
    "Queue still 13:",
    cell57_queue_still_13,
)

print(
    "Base Seed unchanged:",
    cell57_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell57_full_572_blocked,
)

print(
    "External lookup:",
    False,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell57_failures
)


print(
    "\nCODE CELL 57 GATE:",
    (
        "PASS"
        if cell57_gate
        else "FAIL"
    ),
)


assert cell57_gate, (
    "Pilot #11 Replacement Target / Question+Answer ACCEPT / "
    "Citation Draft failed. Repair this SAME Code Cell 57 "
    "before Citation Human Review."
)

=== PILOT #11 REPLACEMENT TARGET + QUESTION/ANSWER ACCEPT + CITATION DRAFT ===

=== RETARGET LINEAGE ===
Pilot: question-pilot:be503b9f0364162c19faa8401f9d7b0f
Old Target: target-evidence:6348f61441b9a5816156190f47b087c0
Selected Candidate: transcript-retarget-candidate:4ccc40bcfcb7f89f0a597f6d76a23b0d
Replacement Target: target-evidence:d1abb2900ff054e944a3e5ab1b58d724
Utterance Range: (66, 79)
Source Type: transcript
Stratum: single_speaker_primary
Task Family: PRIMARY_TRANSCRIPT_RETRIEVAL
Question Archetype: AUTHORITY_CONSTRAINED_RETRIEVAL

=== AUTHORITY VALIDATION ===
All 14 utterances attributed to Dr. Komeil Roudi: True
(66, 'Dr. Komeil Roudi', True, True)
(67, 'Dr. Komeil Roudi', True, True)
(68, 'Dr. Komeil Roudi', True, True)
(69, 'Dr. Komeil Roudi', True, True)
(70, 'Dr. Komeil Roudi', True, True)
(71, 'Dr. Komeil Roudi', True, True)
(72, 'Dr. Komeil Roudi', True, True)
(73, 'Dr. Komeil Roudi', True, True)
(74, 'Dr. Komeil Roudi', True, True)
(75, 'Dr. Komeil Roudi', True, Tr

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 58 — Citation ACCEPT + Pilot #11 Completion + Queue Advance

Citation Mapping این Pilot توسط Human Reviewer پذیرفته شد.

در این مرحله:

- Citation با تصمیم `ACCEPT` ثبت می‌شود؛
- Pilot #11 Retargetشده به‌طور کامل بسته می‌شود؛
- Lineage میان Target اولیه و Replacement Target حفظ می‌شود؛
- اصلاح Transcript قبلی `ماسته → ماسه` نیز در Lineage باقی می‌ماند؛
- Case کامل به `completed_human_review_cases` منتقل می‌شود؛
- Case برای Author Final Review با وضعیت `NOT_STARTED` آماده می‌شود؛
- Queue دقیقاً یک مرحله از 13 به 12 جلو می‌رود؛
- Pilot بعدی از Compact Seed بازیابی می‌شود؛
- Source Evidence بعدی برای Human Review نمایش داده می‌شود؛
- Full 572 Generation همچنان Blocked باقی می‌ماند.

</div>

In [58]:
# Code Cell 58

cell58_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell57_gate") is not True:
    cell58_failures.append(
        "CELL57_NOT_PASS"
    )


if (
    globals().get(
        "cell57_current_session",
        {},
    ).get(
        "current_state"
    )
    !=
    "CITATION_HUMAN_REVIEW_PENDING"
):
    cell58_failures.append(
        "CITATION_NOT_AT_HUMAN_REVIEW_GATE"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell58_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Human Citation ACCEPT
# ============================================================

CELL58_CITATION_HUMAN_NOTE = (
    "Human Reviewer accepted the Citation Mapping for the "
    "retargeted primary Transcript evidence."
)


cell58_complete_pending_session = (
    cell13_apply_event(
        cell57_current_session,
        "CITATION_ACCEPTED",
        human_note=
            CELL58_CITATION_HUMAN_NOTE,
    )
)


cell58_human_decisions = (
    cell58_complete_pending_session.get(
        "human_decisions",
        [],
    )
)


cell58_citation_accept_valid = all([
    cell58_complete_pending_session.get(
        "current_state"
    )
    ==
    "PILOT_COMPLETE_PENDING_QUEUE_ADVANCE",

    cell58_complete_pending_session.get(
        "queue_advance_performed"
    )
    is False,

    bool(
        cell58_human_decisions
    ),

    (
        cell58_human_decisions[
            -1
        ].get(
            "event"
        )
        ==
        "CITATION_ACCEPTED"
        if cell58_human_decisions
        else False
    ),
])


# ============================================================
# 3) Final Citation record
# ============================================================

cell58_final_citation_mapping_record = copy.deepcopy(
    cell57_citation_mapping_record
)


cell58_final_citation_mapping_record[
    "human_adjudication_status"
] = "ACCEPT"


cell58_final_citation_mapping_record[
    "human_review"
] = {
    "human_decision":
        "ACCEPT",

    "human_note":
        CELL58_CITATION_HUMAN_NOTE,

    "human_context_used_as_evidence":
        False,

    "external_knowledge_used":
        False,
}


# ============================================================
# 4) Complete Human Review case
# ============================================================

cell58_completed_case = copy.deepcopy(
    cell57_active_workspace
)


cell58_completed_case[
    "workflow_state"
] = "PILOT_COMPLETE"


cell58_completed_case[
    "status"
] = "COMPLETED_HUMAN_REVIEW"


cell58_completed_case[
    "final_status"
] = "PILOT_COMPLETE"


cell58_completed_case[
    "citation_review"
][
    "human_decision"
] = "ACCEPT"


cell58_completed_case[
    "citation_review"
][
    "human_note"
] = CELL58_CITATION_HUMAN_NOTE


cell58_completed_case[
    "citation_review"
][
    "citation_mapping_record"
] = copy.deepcopy(
    cell58_final_citation_mapping_record
)


cell58_completed_case[
    "completion_summary"
] = {
    "retarget_used":
        True,

    "old_target_evidence_id":
        CELL56_OLD_TARGET_EVIDENCE_ID,

    "replacement_target_evidence_id":
        cell57_replacement_target_evidence_id,

    "selected_candidate_id":
        CELL57_SELECTED_CANDIDATE_ID,

    "question_decision":
        "ACCEPT",

    "expected_answer_decision":
        "ACCEPT",

    "citation_decision":
        "ACCEPT",

    "transcript_correction_lineage_preserved":
        True,

    "author_final_review":
        "NOT_STARTED",
}


cell58_completed_case[
    "author_final_review"
] = {
    "status":
        "NOT_STARTED",

    "decision":
        None,

    "notes":
        None,
}


# ============================================================
# 5) Queue advance: 13 -> 12
# ============================================================

cell58_previous_remaining_pilot_ids = copy.deepcopy(
    cell54_remaining_pilot_ids
)


cell58_remaining_pilot_ids = copy.deepcopy(
    cell58_previous_remaining_pilot_ids[
        1:
    ]
)


CELL58_COMPLETED_PILOT_ID = (
    cell58_previous_remaining_pilot_ids[
        0
    ]
)


cell58_queue_advance_valid = all([
    len(
        cell58_previous_remaining_pilot_ids
    )
    ==
    13,

    CELL58_COMPLETED_PILOT_ID
    ==
    CELL56_PILOT_ID,

    len(
        cell58_remaining_pilot_ids
    )
    ==
    12,

    CELL58_COMPLETED_PILOT_ID
    not in
    cell58_remaining_pilot_ids,
])


CELL58_NEXT_PILOT_ID = (
    cell58_remaining_pilot_ids[
        0
    ]
)


# ============================================================
# 6) Find next Pilot packet in immutable Compact Seed
# ============================================================

def cell58_find_pilot_packet_candidates(
    value,
    pilot_id,
    path=(),
):

    results = []


    if isinstance(
        value,
        dict,
    ):

        direct_values = [
            item
            for item
            in value.values()
            if isinstance(
                item,
                str,
            )
        ]


        if pilot_id in direct_values:

            score = 0


            if (
                value.get(
                    "pilot_case_id"
                )
                ==
                pilot_id
            ):
                score += 120


            if (
                value.get(
                    "question_pilot_id"
                )
                ==
                pilot_id
            ):
                score += 110


            if (
                value.get(
                    "pilot_id"
                )
                ==
                pilot_id
            ):
                score += 100


            if (
                "target_text_parts"
                in
                value
            ):
                score += 60


            if (
                "target_evidence_id"
                in
                value
            ):
                score += 30


            if (
                "golden_case_id"
                in
                value
            ):
                score += 20


            if (
                "task_draft_id"
                in
                value
            ):
                score += 20


            results.append({
                "path":
                    path,

                "score":
                    score,

                "packet":
                    value,
            })


        for key, item in value.items():

            results.extend(
                cell58_find_pilot_packet_candidates(
                    item,
                    pilot_id,
                    path
                    +
                    (
                        str(
                            key
                        ),
                    ),
                )
            )


    elif isinstance(
        value,
        list,
    ):

        for index, item in enumerate(
            value
        ):

            results.extend(
                cell58_find_pilot_packet_candidates(
                    item,
                    pilot_id,
                    path
                    +
                    (
                        str(
                            index
                        ),
                    ),
                )
            )


    return results


cell58_raw_next_packet_candidates = (
    cell58_find_pilot_packet_candidates(
        cell13_seed,
        CELL58_NEXT_PILOT_ID,
    )
)


cell58_packet_candidates_by_sha = {}


for candidate in cell58_raw_next_packet_candidates:

    packet_sha = cell10_stable_sha(
        candidate[
            "packet"
        ]
    )


    previous = (
        cell58_packet_candidates_by_sha.get(
            packet_sha
        )
    )


    if (
        previous is None
        or
        candidate[
            "score"
        ]
        >
        previous[
            "score"
        ]
    ):

        cell58_packet_candidates_by_sha[
            packet_sha
        ] = candidate


cell58_ranked_next_packet_candidates = sorted(
    cell58_packet_candidates_by_sha.values(),
    key=lambda item: (
        -item[
            "score"
        ],
        item[
            "path"
        ],
    ),
)


cell58_next_pilot_packet = (
    copy.deepcopy(
        cell58_ranked_next_packet_candidates[
            0
        ][
            "packet"
        ]
    )
    if cell58_ranked_next_packet_candidates
    else None
)


if not isinstance(
    cell58_next_pilot_packet,
    dict,
):

    cell58_failures.append(
        "NEXT_PILOT_PACKET_NOT_FOUND"
    )


# ============================================================
# 7) Extract next Pilot identities
# ============================================================

def cell58_collect_prefixed_ids(
    value,
    prefix,
):

    found = []


    if isinstance(
        value,
        dict,
    ):

        for item in value.values():

            found.extend(
                cell58_collect_prefixed_ids(
                    item,
                    prefix,
                )
            )


    elif isinstance(
        value,
        list,
    ):

        for item in value:

            found.extend(
                cell58_collect_prefixed_ids(
                    item,
                    prefix,
                )
            )


    elif (
        isinstance(
            value,
            str,
        )
        and
        value.startswith(
            prefix
        )
    ):

        found.append(
            value
        )


    return found


cell58_next_pilot_ids_found = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell58_next_pilot_packet,
            "question-pilot:",
        )
    )
)


cell58_next_golden_case_ids = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell58_next_pilot_packet,
            "golden-case:",
        )
    )
)


cell58_next_task_draft_ids = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell58_next_pilot_packet,
            "task-draft:",
        )
    )
)


cell58_next_target_evidence_ids = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell58_next_pilot_packet,
            "target-evidence:",
        )
    )
)


cell58_next_identity_valid = all([
    cell58_next_pilot_ids_found
    ==
    [
        CELL58_NEXT_PILOT_ID
    ],

    len(
        cell58_next_golden_case_ids
    )
    ==
    1,

    len(
        cell58_next_task_draft_ids
    )
    ==
    1,

    len(
        cell58_next_target_evidence_ids
    )
    ==
    1,
])


CELL58_NEXT_GOLDEN_CASE_ID = (
    cell58_next_golden_case_ids[
        0
    ]
    if
    len(
        cell58_next_golden_case_ids
    )
    ==
    1
    else
    None
)


CELL58_NEXT_TASK_DRAFT_ID = (
    cell58_next_task_draft_ids[
        0
    ]
    if
    len(
        cell58_next_task_draft_ids
    )
    ==
    1
    else
    None
)


CELL58_NEXT_TARGET_EVIDENCE_ID = (
    cell58_next_target_evidence_ids[
        0
    ]
    if
    len(
        cell58_next_target_evidence_ids
    )
    ==
    1
    else
    None
)


# ============================================================
# 8) Extract next Target Evidence parts
# ============================================================

def cell58_find_key_values(
    value,
    target_key,
):

    found = []


    if isinstance(
        value,
        dict,
    ):

        for key, item in value.items():

            if key == target_key:

                found.append(
                    copy.deepcopy(
                        item
                    )
                )


            found.extend(
                cell58_find_key_values(
                    item,
                    target_key,
                )
            )


    elif isinstance(
        value,
        list,
    ):

        for item in value:

            found.extend(
                cell58_find_key_values(
                    item,
                    target_key,
                )
            )


    return found


cell58_target_part_candidates = (
    cell58_find_key_values(
        cell58_next_pilot_packet,
        "target_text_parts",
    )
)


cell58_target_parts_by_sha = {}


for parts in cell58_target_part_candidates:

    if not isinstance(
        parts,
        list,
    ):
        continue


    if len(
        parts
    ) == 0:
        continue


    parts_sha = cell10_stable_sha(
        parts
    )


    cell58_target_parts_by_sha[
        parts_sha
    ] = parts


cell58_unique_target_part_sets = list(
    cell58_target_parts_by_sha.values()
)


cell58_next_target_text_parts = (
    copy.deepcopy(
        cell58_unique_target_part_sets[
            0
        ]
    )
    if
    len(
        cell58_unique_target_part_sets
    )
    ==
    1
    else
    None
)


if not isinstance(
    cell58_next_target_text_parts,
    list,
):

    cell58_failures.append(
        "NEXT_TARGET_TEXT_PARTS_NOT_UNIQUE"
    )


# ============================================================
# 9) Build readable next Evidence
# ============================================================

cell58_next_evidence_texts = []


for part in (
    cell58_next_target_text_parts
    or
    []
):

    if isinstance(
        part,
        str,
    ):

        text = part.strip()


    elif isinstance(
        part,
        dict,
    ):

        text = (
            part.get(
                "text"
            )
            or
            part.get(
                "target_text"
            )
            or
            part.get(
                "content"
            )
            or
            ""
        )


        if isinstance(
            text,
            str,
        ):

            text = text.strip()


        else:

            text = ""


    else:

        text = ""


    if text:

        cell58_next_evidence_texts.append(
            text
        )


cell58_next_combined_target_text = (
    "\n\n".join(
        cell58_next_evidence_texts
    )
)


cell58_next_evidence_available = bool(
    cell58_next_combined_target_text.strip()
)


# ============================================================
# 10) Next Pilot engine session
# ============================================================

cell58_next_pilot_session = (
    cell13_build_session(
        cell58_next_pilot_packet
    )
)


cell58_next_session_valid = (
    cell58_next_pilot_session.get(
        "current_state"
    )
    ==
    "QUESTION_DRAFT_PENDING"
)


# ============================================================
# 11) Update completed Human Review cases
# ============================================================

cell58_existing_completed_cases = copy.deepcopy(
    cell57_persisted_registry.get(
        "completed_human_review_cases",
        [],
    )
)


if not isinstance(
    cell58_existing_completed_cases,
    list,
):

    cell58_failures.append(
        "COMPLETED_HUMAN_REVIEW_CASES_NOT_LIST"
    )

    cell58_existing_completed_cases = []


cell58_existing_completed_cases = [
    case
    for case
    in cell58_existing_completed_cases
    if not (
        isinstance(
            case,
            dict,
        )
        and
        case.get(
            "pilot_case_id"
        )
        ==
        CELL58_COMPLETED_PILOT_ID
    )
]


cell58_completed_human_review_cases = (
    cell58_existing_completed_cases
    +
    [
        copy.deepcopy(
            cell58_completed_case
        )
    ]
)


cell58_completed_case_count_valid = (
    len(
        cell58_completed_human_review_cases
    )
    ==
    10
)


# ============================================================
# 12) Rebuild Author Review queue from authoritative completed cases
# ============================================================

cell58_author_review = copy.deepcopy(
    cell57_persisted_registry.get(
        "author_review",
        {},
    )
)


if not isinstance(
    cell58_author_review,
    dict,
):

    cell58_author_review = {}


cell58_author_ready_cases = copy.deepcopy(
    cell58_completed_human_review_cases
)


cell58_author_review[
    "cases_ready_for_review"
] = copy.deepcopy(
    cell58_author_ready_cases
)


cell58_author_review[
    "case_count"
] = len(
    cell58_author_ready_cases
)


cell58_author_review[
    "final_review_status"
] = "NOT_STARTED"


cell58_author_ready_count_valid = (
    len(
        cell58_author_ready_cases
    )
    ==
    10
)


cell58_author_ready_identity_valid = (
    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell58_author_ready_cases
        if isinstance(
            case,
            dict,
        )
    }
    ==
    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell58_completed_human_review_cases
        if isinstance(
            case,
            dict,
        )
    }
)


# ============================================================
# 13) Build next active workspace
# ============================================================

cell58_next_active_workspace = {
    "pilot_case_id":
        CELL58_NEXT_PILOT_ID,

    "golden_case_id":
        CELL58_NEXT_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL58_NEXT_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL58_NEXT_TARGET_EVIDENCE_ID,

    "status":
        "ACTIVE_HUMAN_REVIEW",

    "workflow_state":
        "QUESTION_DRAFT_PENDING",

    "source_evidence":
        copy.deepcopy(
            cell58_next_target_text_parts
        ),

    "combined_target_text":
        cell58_next_combined_target_text,

    "question_review":
        None,

    "expected_answer_review":
        None,

    "citation_review":
        None,

    "author_final_review": {
        "status":
            "NOT_STARTED",

        "decision":
            None,

        "notes":
            None,
    },
}


# ============================================================
# 14) New authoritative continuation state
# ============================================================

cell58_continuation_state_core = {
    "next_pilot_id":
        CELL58_NEXT_PILOT_ID,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell58_remaining_pilot_ids
        ),

    "remaining_count":
        12,

    "active_pilot_status":
        "QUESTION_DRAFT_PENDING",

    "active_target_evidence_id":
        CELL58_NEXT_TARGET_EVIDENCE_ID,

    "last_completed_pilot_id":
        CELL58_COMPLETED_PILOT_ID,

    "last_completion_type":
        "RETARGETED_PILOT_COMPLETE",
}


cell58_continuation_state_sha256 = (
    cell10_stable_sha(
        cell58_continuation_state_core
    )
)


cell58_continuation_state = {
    **copy.deepcopy(
        cell58_continuation_state_core
    ),

    "continuation_state_sha256":
        cell58_continuation_state_sha256,
}


# ============================================================
# 15) Persist Registry
# ============================================================

cell58_persistent_registry_payload = copy.deepcopy(
    cell57_persisted_registry
)


cell58_persistent_registry_payload[
    "completed_human_review_cases"
] = copy.deepcopy(
    cell58_completed_human_review_cases
)


cell58_persistent_registry_payload[
    "author_review"
] = copy.deepcopy(
    cell58_author_review
)


cell58_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell58_next_active_workspace
)


cell58_persistent_registry_payload[
    "remaining_pilot_count"
] = 12


cell58_persistent_registry_payload[
    "continuation_state"
] = copy.deepcopy(
    cell58_continuation_state
)


cell58_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell58_persistent_registry_payload
    )
)


# ============================================================
# 16) Read-back validation
# ============================================================

cell58_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell58_persisted_active_case = (
    cell58_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell58_persisted_completed_cases = (
    cell58_persisted_registry.get(
        "completed_human_review_cases",
        [],
    )
)


cell58_persisted_author_ready_cases = (
    cell58_persisted_registry.get(
        "author_review",
        {},
    ).get(
        "cases_ready_for_review",
        [],
    )
)


cell58_persistence_valid = all([
    isinstance(
        cell58_persisted_completed_cases,
        list,
    ),

    len(
        cell58_persisted_completed_cases
    )
    ==
    10,

    isinstance(
        cell58_persisted_author_ready_cases,
        list,
    ),

    len(
        cell58_persisted_author_ready_cases
    )
    ==
    10,

    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell58_persisted_completed_cases
        if isinstance(
            case,
            dict,
        )
    }
    ==
    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell58_persisted_author_ready_cases
        if isinstance(
            case,
            dict,
        )
    },

    cell58_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    12,

    cell58_persisted_registry.get(
        "continuation_state",
        {},
    ).get(
        "continuation_state_sha256"
    )
    ==
    cell58_continuation_state_sha256,

    cell58_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL58_NEXT_PILOT_ID,

    cell58_persisted_active_case.get(
        "workflow_state"
    )
    ==
    "QUESTION_DRAFT_PENDING",
])


# ============================================================
# 17) Safety
# ============================================================

cell58_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell58_full_572_blocked = (
    cell57_full_572_blocked
    is True
)


cell58_no_commit_push = True


# ============================================================
# 18) Final Gate
# ============================================================

cell58_gate = all([
    len(
        cell58_failures
    )
    ==
    0,

    cell58_citation_accept_valid,

    cell58_queue_advance_valid,

    cell58_next_identity_valid,

    cell58_next_evidence_available,

    cell58_next_session_valid,

    cell58_completed_case_count_valid,

    cell58_author_ready_count_valid,

    cell58_author_ready_identity_valid,

    cell58_persistence_valid,

    cell58_base_seed_unchanged,

    cell58_full_572_blocked,

    cell58_no_commit_push,
])


# ============================================================
# 19) Diagnostics
# ============================================================

print(
    "=== PILOT #11 COMPLETE + QUEUE ADVANCE + NEXT PILOT ==="
)


print(
    "\n=== COMPLETED PILOT ==="
)

print(
    "Pilot:",
    CELL58_COMPLETED_PILOT_ID,
)

print(
    "Retarget:",
    "ACCEPT",
)

print(
    "Question:",
    "ACCEPT",
)

print(
    "Expected Answer:",
    "ACCEPT",
)

print(
    "Citation:",
    "ACCEPT",
)

print(
    "Final Engine State:",
    cell58_complete_pending_session.get(
        "current_state"
    ),
)


print(
    "\n=== COMPLETED RETARGET LINEAGE ==="
)

print(
    "Old Target:",
    CELL56_OLD_TARGET_EVIDENCE_ID,
)

print(
    "Replacement Target:",
    cell57_replacement_target_evidence_id,
)

print(
    "Selected Candidate:",
    CELL57_SELECTED_CANDIDATE_ID,
)

print(
    "Citation Mapping:",
    cell57_citation_mapping_id,
)

print(
    "Transcript correction lineage preserved:",
    True,
)


print(
    "\n=== QUEUE ADVANCE ==="
)

print(
    "Before:",
    len(
        cell58_previous_remaining_pilot_ids
    ),
)

print(
    "After:",
    len(
        cell58_remaining_pilot_ids
    ),
)

print(
    "Queue advanced exactly once:",
    cell58_queue_advance_valid,
)


print(
    "\n=== HUMAN REVIEW REGISTRY ==="
)

print(
    "Completed Human Review cases:",
    len(
        cell58_completed_human_review_cases
    ),
)

print(
    "Cases ready for Author Review:",
    len(
        cell58_author_ready_cases
    ),
)

print(
    "Author ready identities match completed cases:",
    cell58_author_ready_identity_valid,
)

print(
    "Author Final Review:",
    "NOT_STARTED",
)

print(
    "Registry:",
    cell21_registry_path,
)

print(
    "Persistence valid:",
    cell58_persistence_valid,
)


print(
    "\n=== NEXT PILOT ==="
)

print(
    "Pilot:",
    CELL58_NEXT_PILOT_ID,
)

print(
    "Golden Case:",
    CELL58_NEXT_GOLDEN_CASE_ID,
)

print(
    "Task Draft:",
    CELL58_NEXT_TASK_DRAFT_ID,
)

print(
    "Target Evidence:",
    CELL58_NEXT_TARGET_EVIDENCE_ID,
)

print(
    "Workflow:",
    cell58_next_pilot_session.get(
        "current_state"
    ),
)


print(
    "\n=== NEXT SOURCE EVIDENCE ==="
)

print(
    cell58_next_combined_target_text
)


print(
    "\n=== CONTINUATION STATE ==="
)

print(
    "Remaining Pilots:",
    12,
)

print(
    "Continuation SHA256:",
    cell58_continuation_state_sha256,
)


print(
    "\n=== SAFETY ==="
)

print(
    "Base Seed unchanged:",
    cell58_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell58_full_572_blocked,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell58_failures
)


print(
    "\nCODE CELL 58 GATE:",
    (
        "PASS"
        if cell58_gate
        else "FAIL"
    ),
)


assert cell58_gate, (
    "Citation ACCEPT / Pilot #11 completion / Queue advance / "
    "Next Pilot preparation failed. Repair this SAME "
    "Code Cell 58 before continuing."
)

=== PILOT #11 COMPLETE + QUEUE ADVANCE + NEXT PILOT ===

=== COMPLETED PILOT ===
Pilot: question-pilot:be503b9f0364162c19faa8401f9d7b0f
Retarget: ACCEPT
Question: ACCEPT
Expected Answer: ACCEPT
Citation: ACCEPT
Final Engine State: PILOT_COMPLETE_PENDING_QUEUE_ADVANCE

=== COMPLETED RETARGET LINEAGE ===
Old Target: target-evidence:6348f61441b9a5816156190f47b087c0
Replacement Target: target-evidence:d1abb2900ff054e944a3e5ab1b58d724
Selected Candidate: transcript-retarget-candidate:4ccc40bcfcb7f89f0a597f6d76a23b0d
Citation Mapping: citation-mapping:bb6a8445149ec75d7b6f9a4130dd8484
Transcript correction lineage preserved: True

=== QUEUE ADVANCE ===
Before: 13
After: 12
Queue advanced exactly once: True

=== HUMAN REVIEW REGISTRY ===
Completed Human Review cases: 10
Cases ready for Author Review: 10
Author ready identities match completed cases: True
Author Final Review: NOT_STARTED
Registry: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/data/phase2/human_review

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 59 — Question/Answer ACCEPT + Citation Draft for Pilot #12

Human Reviewer سؤال و جواب این Pilot را پذیرفت.

سؤال پذیرفته‌شده:

**برای آموزش سواد مالی به کودک، چه مفاهیمی از حوزه خرج و پس‌انداز را می‌توان متناسب با سن او آموزش داد؟**

جواب پذیرفته‌شده:

**می‌توان درباره نحوه استفاده از پول‌توجیبی، مفهوم پول، تفاوت تعداد اسکناس‌ها با ارزش اسمی آن‌ها، اینکه پول از کجا می‌آید و پشت آن کار و فعالیت وجود دارد، و همچنین مفهوم بانک و دلیل نگهداری پول در بانک به کودک آموزش داد. این مفاهیم باید متناسب با سن کودک ساده‌سازی و ارائه شوند.**

در این مرحله:

- Question با Human Decision برابر `ACCEPT` ثبت می‌شود؛
- Expected Answer نیز با Human Decision برابر `ACCEPT` ثبت می‌شود؛
- Citation فقط از Target Evidence فعلی ساخته می‌شود؛
- هیچ Retarget انجام نمی‌شود؛
- Citation وارد Human Review می‌شود؛
- Queue همچنان 12 باقی می‌ماند؛
- Full 572 Generation همچنان Blocked است.

</div>

In [59]:
# Code Cell 59

cell59_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell58_gate") is not True:
    cell59_failures.append(
        "CELL58_NOT_PASS"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell59_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Accepted Human Review content
# ============================================================

CELL59_PILOT_ID = (
    CELL58_NEXT_PILOT_ID
)


CELL59_GOLDEN_CASE_ID = (
    CELL58_NEXT_GOLDEN_CASE_ID
)


CELL59_TASK_DRAFT_ID = (
    CELL58_NEXT_TASK_DRAFT_ID
)


CELL59_TARGET_EVIDENCE_ID = (
    CELL58_NEXT_TARGET_EVIDENCE_ID
)


CELL59_ACCEPTED_QUESTION = (
    "برای آموزش سواد مالی به کودک، چه مفاهیمی از حوزه "
    "خرج و پس‌انداز را می‌توان متناسب با سن او آموزش داد؟"
)


CELL59_ACCEPTED_ANSWER = (
    "می‌توان درباره نحوه استفاده از پول‌توجیبی، مفهوم پول، "
    "تفاوت تعداد اسکناس‌ها با ارزش اسمی آن‌ها، اینکه پول از "
    "کجا می‌آید و پشت آن کار و فعالیت وجود دارد، و همچنین "
    "مفهوم بانک و دلیل نگهداری پول در بانک به کودک آموزش داد. "
    "این مفاهیم باید متناسب با سن کودک ساده‌سازی و ارائه شوند."
)


CELL59_QUESTION_HUMAN_NOTE = (
    "Human Reviewer accepted the proposed question."
)


CELL59_ANSWER_HUMAN_NOTE = (
    "Human Reviewer accepted the proposed answer."
)


# ============================================================
# 3) Current Target Evidence
# ============================================================

cell59_target_parts = copy.deepcopy(
    cell58_next_target_text_parts
)


cell59_combined_target_text = (
    cell58_next_combined_target_text
)


cell59_target_parts_valid = all([
    isinstance(
        cell59_target_parts,
        list,
    ),

    len(
        cell59_target_parts
    )
    >
    0,

    isinstance(
        cell59_combined_target_text,
        str,
    ),

    bool(
        cell59_combined_target_text.strip()
    ),
])


# ============================================================
# 4) Grounding validation
# ============================================================

def cell59_norm(
    text,
):

    if not isinstance(
        text,
        str,
    ):
        return ""


    value = (
        text
        .replace(
            "ي",
            "ی",
        )
        .replace(
            "ك",
            "ک",
        )
        .replace(
            "\u200c",
            " ",
        )
    )


    value = re.sub(
        r"\s+",
        " ",
        value,
    )


    return value.strip()


cell59_source_surface = cell59_norm(
    cell59_combined_target_text
)


cell59_grounding = {
    "child":
        (
            "کودک"
            in
            cell59_source_surface
        ),

    "age_appropriate":
        (
            "متناسب با سن"
            in
            cell59_source_surface
        ),

    "spending_saving":
        (
            "خرج و پس انداز"
            in
            cell59_source_surface
        ),

    "allowance":
        (
            "پول توجیبی"
            in
            cell59_source_surface
        ),

    "money_concept":
        (
            "پول چیست"
            in
            cell59_source_surface
        ),

    "nominal_value":
        (
            "ارزش اسمی"
            in
            cell59_source_surface
        ),

    "money_origin":
        (
            "پول از کجا می آید"
            in
            cell59_source_surface
        ),

    "work_behind_money":
        (
            "کار و فعالیت"
            in
            cell59_source_surface
        ),

    "bank":
        (
            "بانک چیست"
            in
            cell59_source_surface
        ),

    "bank_storage":
        (
            "پولشان را در آن نگه می دارند"
            in
            cell59_source_surface
        ),
}


cell59_grounding_pass = all(
    cell59_grounding.values()
)


# ============================================================
# 5) Start Engine session
# ============================================================

cell59_question_pending_session = (
    cell13_build_session(
        cell58_next_pilot_packet
    )
)


cell59_session_start_valid = (
    cell59_question_pending_session.get(
        "current_state"
    )
    ==
    "QUESTION_DRAFT_PENDING"
)


# ============================================================
# 6) Question Draft
# ============================================================

cell59_question_identity = {
    "identity_version":
        "question_draft_identity_v1",

    "pilot_case_id":
        CELL59_PILOT_ID,

    "target_evidence_id":
        CELL59_TARGET_EVIDENCE_ID,

    "question_text":
        CELL59_ACCEPTED_QUESTION,

    "revision_number":
        1,
}


cell59_question_draft_id = (
    "question-draft:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell59_question_identity
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


cell59_question_record = {
    "record_version":
        "question_draft_record_v1",

    "question_draft_id":
        cell59_question_draft_id,

    "pilot_case_id":
        CELL59_PILOT_ID,

    "golden_case_id":
        CELL59_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL59_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL59_TARGET_EVIDENCE_ID,

    "question_text":
        CELL59_ACCEPTED_QUESTION,

    "revision_number":
        1,

    "machine_validation_status":
        (
            "PASS_PENDING_HUMAN_ADJUDICATION"
            if cell59_grounding_pass
            else
            "FAIL_REQUIRES_REVISION"
        ),

    "human_adjudication_status":
        "PENDING",

    "external_knowledge_used":
        False,
}


cell59_question_review_session = (
    cell13_apply_event(
        cell59_question_pending_session,
        "QUESTION_DRAFT_REGISTERED",
        artifact=
            cell59_question_record,
    )
)


cell59_answer_pending_session = (
    cell13_apply_event(
        cell59_question_review_session,
        "QUESTION_ACCEPTED",
        human_note=
            CELL59_QUESTION_HUMAN_NOTE,
    )
)


cell59_question_accept_valid = all([
    cell59_question_review_session.get(
        "current_state"
    )
    ==
    "QUESTION_HUMAN_REVIEW_PENDING",

    cell59_answer_pending_session.get(
        "current_state"
    )
    ==
    "EXPECTED_ANSWER_DRAFT_PENDING",
])


# ============================================================
# 7) Expected Answer
# ============================================================

cell59_answer_identity = {
    "identity_version":
        "expected_answer_identity_v1",

    "pilot_case_id":
        CELL59_PILOT_ID,

    "question_draft_id":
        cell59_question_draft_id,

    "target_evidence_id":
        CELL59_TARGET_EVIDENCE_ID,

    "answer_text":
        CELL59_ACCEPTED_ANSWER,

    "revision_number":
        1,
}


cell59_expected_answer_draft_id = (
    "expected-answer-draft:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell59_answer_identity
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


cell59_expected_answer_record = {
    "record_version":
        "expected_answer_draft_record_v1",

    "expected_answer_draft_id":
        cell59_expected_answer_draft_id,

    "pilot_case_id":
        CELL59_PILOT_ID,

    "golden_case_id":
        CELL59_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL59_TASK_DRAFT_ID,

    "question_draft_id":
        cell59_question_draft_id,

    "target_evidence_id":
        CELL59_TARGET_EVIDENCE_ID,

    "expected_answer_text":
        CELL59_ACCEPTED_ANSWER,

    "revision_number":
        1,

    "machine_validation_status":
        (
            "PASS_PENDING_HUMAN_ADJUDICATION"
            if cell59_grounding_pass
            else
            "FAIL_REQUIRES_REVISION"
        ),

    "human_adjudication_status":
        "PENDING",

    "external_knowledge_used":
        False,
}


cell59_answer_review_session = (
    cell13_apply_event(
        cell59_answer_pending_session,
        "EXPECTED_ANSWER_REGISTERED",
        artifact=
            cell59_expected_answer_record,
    )
)


cell59_citation_pending_session = (
    cell13_apply_event(
        cell59_answer_review_session,
        "EXPECTED_ANSWER_ACCEPTED",
        human_note=
            CELL59_ANSWER_HUMAN_NOTE,
    )
)


cell59_answer_accept_valid = all([
    cell59_answer_review_session.get(
        "current_state"
    )
    ==
    "EXPECTED_ANSWER_HUMAN_REVIEW_PENDING",

    cell59_citation_pending_session.get(
        "current_state"
    )
    ==
    "CITATION_MAPPING_PENDING",
])


# ============================================================
# 8) Build Citation entries directly from Target parts
# ============================================================

cell59_citation_entries = []


for index, part in enumerate(
    cell59_target_parts
):

    if isinstance(
        part,
        dict,
    ):

        citation_entry = copy.deepcopy(
            part
        )


        citation_entry[
            "citation_index"
        ] = index


        citation_entry[
            "target_evidence_id"
        ] = CELL59_TARGET_EVIDENCE_ID


        citation_entry[
            "locator_origin"
        ] = "COPIED_FROM_TARGET_EVIDENCE"


        citation_entry[
            "external_locator_lookup_used"
        ] = False


    else:

        citation_entry = {
            "citation_index":
                index,

            "target_evidence_id":
                CELL59_TARGET_EVIDENCE_ID,

            "text":
                part,

            "locator_origin":
                "COPIED_FROM_TARGET_EVIDENCE",

            "external_locator_lookup_used":
                False,
        }


    cell59_citation_entries.append(
        citation_entry
    )


# ============================================================
# 9) No-fabrication validation
# ============================================================

def cell59_strip_added_citation_fields(
    entry,
):

    output = copy.deepcopy(
        entry
    )


    output.pop(
        "citation_index",
        None,
    )


    output.pop(
        "target_evidence_id",
        None,
    )


    output.pop(
        "locator_origin",
        None,
    )


    output.pop(
        "external_locator_lookup_used",
        None,
    )


    return output


cell59_citation_back_projection = [
    cell59_strip_added_citation_fields(
        entry
    )
    for entry
    in cell59_citation_entries
]


cell59_no_locator_fabrication = (
    cell59_citation_back_projection
    ==
    cell59_target_parts
)


# ============================================================
# 10) Citation ID
# ============================================================

cell59_citation_identity = {
    "identity_version":
        "citation_mapping_identity_v1",

    "pilot_case_id":
        CELL59_PILOT_ID,

    "question_draft_id":
        cell59_question_draft_id,

    "expected_answer_draft_id":
        cell59_expected_answer_draft_id,

    "target_evidence_id":
        CELL59_TARGET_EVIDENCE_ID,

    "citation_entries":
        copy.deepcopy(
            cell59_citation_entries
        ),
}


cell59_citation_mapping_id = (
    "citation-mapping:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell59_citation_identity
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


cell59_citation_machine_pass = all([
    len(
        cell59_citation_entries
    )
    ==
    len(
        cell59_target_parts
    ),

    len(
        cell59_citation_entries
    )
    >
    0,

    cell59_no_locator_fabrication,

    all(
        entry.get(
            "target_evidence_id"
        )
        ==
        CELL59_TARGET_EVIDENCE_ID
        for entry
        in cell59_citation_entries
    ),

    all(
        entry.get(
            "external_locator_lookup_used"
        )
        is False
        for entry
        in cell59_citation_entries
    ),
])


cell59_citation_mapping_record = {
    "record_version":
        "citation_mapping_record_v1",

    "citation_mapping_id":
        cell59_citation_mapping_id,

    "pilot_case_id":
        CELL59_PILOT_ID,

    "golden_case_id":
        CELL59_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL59_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL59_TARGET_EVIDENCE_ID,

    "question_draft_id":
        cell59_question_draft_id,

    "expected_answer_draft_id":
        cell59_expected_answer_draft_id,

    "citation_entries":
        copy.deepcopy(
            cell59_citation_entries
        ),

    "citation_scope":
        "CURRENT_TARGET_EVIDENCE_ONLY",

    "retarget_used":
        False,

    "external_locator_lookup_used":
        False,

    "machine_validation_status":
        (
            "PASS_PENDING_HUMAN_ADJUDICATION"
            if cell59_citation_machine_pass
            else
            "FAIL_REQUIRES_REVISION"
        ),

    "human_adjudication_status":
        "PENDING",
}


cell59_current_session = (
    cell13_apply_event(
        cell59_citation_pending_session,
        "CITATION_MAPPING_REGISTERED",
        artifact=
            cell59_citation_mapping_record,
    )
)


cell59_citation_registered_valid = (
    cell59_current_session.get(
        "current_state"
    )
    ==
    "CITATION_HUMAN_REVIEW_PENDING"
)


# ============================================================
# 11) Update active Human Review workspace
# ============================================================

cell59_active_workspace = copy.deepcopy(
    cell58_persisted_active_case
)


cell59_active_workspace[
    "workflow_state"
] = "CITATION_HUMAN_REVIEW_PENDING"


cell59_active_workspace[
    "status"
] = "ACTIVE_HUMAN_REVIEW"


cell59_active_workspace[
    "question_review"
] = {
    "question_draft_id":
        cell59_question_draft_id,

    "question_text":
        CELL59_ACCEPTED_QUESTION,

    "machine_validation_status":
        cell59_question_record[
            "machine_validation_status"
        ],

    "human_decision":
        "ACCEPT",

    "human_note":
        CELL59_QUESTION_HUMAN_NOTE,
}


cell59_active_workspace[
    "expected_answer_review"
] = {
    "expected_answer_draft_id":
        cell59_expected_answer_draft_id,

    "expected_answer_text":
        CELL59_ACCEPTED_ANSWER,

    "machine_validation_status":
        cell59_expected_answer_record[
            "machine_validation_status"
        ],

    "human_decision":
        "ACCEPT",

    "human_note":
        CELL59_ANSWER_HUMAN_NOTE,
}


cell59_active_workspace[
    "citation_review"
] = {
    "citation_mapping_id":
        cell59_citation_mapping_id,

    "citation_entries":
        copy.deepcopy(
            cell59_citation_entries
        ),

    "machine_validation_status":
        cell59_citation_mapping_record[
            "machine_validation_status"
        ],

    "human_decision":
        "PENDING",
}


# ============================================================
# 12) Updated continuation state
# ============================================================

cell59_continuation_core = {
    "next_pilot_id":
        CELL59_PILOT_ID,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell58_remaining_pilot_ids
        ),

    "remaining_count":
        12,

    "active_pilot_status":
        "CITATION_HUMAN_REVIEW_PENDING",

    "active_target_evidence_id":
        CELL59_TARGET_EVIDENCE_ID,

    "last_completed_pilot_id":
        cell58_continuation_state.get(
            "last_completed_pilot_id"
        ),

    "last_completion_type":
        cell58_continuation_state.get(
            "last_completion_type"
        ),
}


cell59_continuation_state_sha256 = (
    cell10_stable_sha(
        cell59_continuation_core
    )
)


cell59_continuation_state = {
    **copy.deepcopy(
        cell59_continuation_core
    ),

    "continuation_state_sha256":
        cell59_continuation_state_sha256,
}


# ============================================================
# 13) Persist Registry
# ============================================================

cell59_persistent_registry_payload = copy.deepcopy(
    cell58_persisted_registry
)


cell59_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell59_active_workspace
)


cell59_persistent_registry_payload[
    "remaining_pilot_count"
] = 12


cell59_persistent_registry_payload[
    "continuation_state"
] = copy.deepcopy(
    cell59_continuation_state
)


cell59_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell59_persistent_registry_payload
    )
)


# ============================================================
# 14) Read-back validation
# ============================================================

cell59_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell59_persisted_active_case = (
    cell59_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell59_persistence_valid = all([
    cell59_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL59_PILOT_ID,

    cell59_persisted_active_case.get(
        "workflow_state"
    )
    ==
    "CITATION_HUMAN_REVIEW_PENDING",

    cell59_persisted_active_case.get(
        "question_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell59_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell59_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "PENDING",

    cell59_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    12,

    cell59_persisted_registry.get(
        "continuation_state",
        {},
    ).get(
        "continuation_state_sha256"
    )
    ==
    cell59_continuation_state_sha256,
])


# ============================================================
# 15) Safety
# ============================================================

cell59_queue_still_12 = all([
    len(
        cell58_remaining_pilot_ids
    )
    ==
    12,

    cell58_remaining_pilot_ids[
        0
    ]
    ==
    CELL59_PILOT_ID,
])


cell59_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell59_full_572_blocked = (
    cell58_full_572_blocked
    is True
)


# ============================================================
# 16) Final Gate
# ============================================================

cell59_gate = all([
    len(
        cell59_failures
    )
    ==
    0,

    cell59_target_parts_valid,

    cell59_grounding_pass,

    cell59_session_start_valid,

    cell59_question_accept_valid,

    cell59_answer_accept_valid,

    cell59_no_locator_fabrication,

    cell59_citation_machine_pass,

    cell59_citation_registered_valid,

    cell59_persistence_valid,

    cell59_queue_still_12,

    cell59_base_seed_unchanged,

    cell59_full_572_blocked,
])


# ============================================================
# 17) Diagnostics
# ============================================================

print(
    "=== PILOT #12 QUESTION/ANSWER ACCEPT + CITATION DRAFT ==="
)


print(
    "\n=== PILOT ==="
)

print(
    "Pilot:",
    CELL59_PILOT_ID,
)

print(
    "Golden Case:",
    CELL59_GOLDEN_CASE_ID,
)

print(
    "Task Draft:",
    CELL59_TASK_DRAFT_ID,
)

print(
    "Target Evidence:",
    CELL59_TARGET_EVIDENCE_ID,
)

print(
    "Retarget:",
    False,
)


print(
    "\n=== SOURCE EVIDENCE ==="
)

print(
    cell59_combined_target_text
)


print(
    "\n=== ACCEPTED QUESTION ==="
)

print(
    CELL59_ACCEPTED_QUESTION
)

print(
    "Question Draft ID:",
    cell59_question_draft_id,
)

print(
    "Human Decision:",
    "ACCEPT",
)


print(
    "\n=== ACCEPTED ANSWER ==="
)

print(
    CELL59_ACCEPTED_ANSWER
)

print(
    "Expected Answer Draft ID:",
    cell59_expected_answer_draft_id,
)

print(
    "Human Decision:",
    "ACCEPT",
)


print(
    "\n=== GROUNDING ==="
)

for name, status in (
    cell59_grounding.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "Grounding PASS:",
    cell59_grounding_pass,
)


print(
    "\n=== CITATION DRAFT ==="
)

print(
    "Citation Mapping ID:",
    cell59_citation_mapping_id,
)

print(
    "Citation entry count:",
    len(
        cell59_citation_entries
    ),
)

print(
    "No locator fabrication:",
    cell59_no_locator_fabrication,
)

print(
    "Citation Machine PASS:",
    cell59_citation_machine_pass,
)


for index, entry in enumerate(
    cell59_citation_entries
):

    print(
        "\n--- CITATION ENTRY",
        index,
        "---"
    )

    print(
        cell10_canonical_json(
            entry
        )
    )


print(
    "\n=== WORKFLOW ==="
)

print(
    "Current State:",
    cell59_current_session.get(
        "current_state"
    ),
)

print(
    "Question:",
    "ACCEPT",
)

print(
    "Expected Answer:",
    "ACCEPT",
)

print(
    "Citation:",
    "PENDING",
)


print(
    "\n=== PERSISTENCE ==="
)

print(
    "Registry:",
    cell21_registry_path,
)

print(
    "Persistence valid:",
    cell59_persistence_valid,
)


print(
    "\n=== QUEUE / SAFETY ==="
)

print(
    "Queue still 12:",
    cell59_queue_still_12,
)

print(
    "Base Seed unchanged:",
    cell59_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell59_full_572_blocked,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell59_failures
)


print(
    "\nCODE CELL 59 GATE:",
    (
        "PASS"
        if cell59_gate
        else "FAIL"
    ),
)


assert cell59_gate, (
    "Pilot #12 Question/Answer ACCEPT / Citation Draft failed. "
    "Repair this SAME Code Cell 59 before Citation Human Review."
)

=== PILOT #12 QUESTION/ANSWER ACCEPT + CITATION DRAFT ===

=== PILOT ===
Pilot: question-pilot:9d1a1208092facc8407fed78c07f5a90
Golden Case: golden-case:d3a7c9b05287cb9386b9c8b67a4ecbca
Task Draft: task-draft:8b810824e6ecae5dba3cf2bf62682ad0
Target Evidence: target-evidence:c991cbc1bbc326922a46ec51bb9efb6c
Retarget: False

=== SOURCE EVIDENCE ===
وقتی مخاطب کودک است، از همان سرفصل‌ها مفاهیم ساده و متناسب با سن انتخاب می‌شود. مثلاً در حوزه خرج و پس‌انداز، «پول‌توجیبی» موضوع مهمی است: کودک چطور آن را مصرف کند؟ پول چیست؟ ارزش پول به تعداد اسکناس‌هاست یا به ارزش اسمی آنها؟ پول از کجا می‌آید؟ پول روی درخت رشد نمی‌کند؛ پشت آن کار و فعالیت وجود دارد. بانک چیست و چرا مردم پولشان را در آن نگه می‌دارند؟ اینها مفاهیمی‌اند که می‌توان از سن مناسب به کودک آموزش داد.

=== ACCEPTED QUESTION ===
برای آموزش سواد مالی به کودک، چه مفاهیمی از حوزه خرج و پس‌انداز را می‌توان متناسب با سن او آموزش داد؟
Question Draft ID: question-draft:ad1e5248b6b0eba92b9ee3ed2c2d4201
Human Decision: ACCEPT

=== ACCEPTED ANSW

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 60 — Citation ACCEPT + Pilot #12 Completion + Queue Advance

Citation Mapping این Pilot توسط Human Reviewer پذیرفته شد.

در این مرحله:

- Citation با تصمیم `ACCEPT` ثبت می‌شود؛
- Pilot #12 به‌طور کامل بسته می‌شود؛
- Question، Expected Answer و Citation نهایی در Human Review Registry حفظ می‌شوند؛
- Case کامل به `completed_human_review_cases` منتقل می‌شود؛
- Case برای Author Final Review با وضعیت `NOT_STARTED` آماده می‌شود؛
- Queue دقیقاً یک مرحله از 12 به 11 جلو می‌رود؛
- Pilot بعدی از Compact Seed بازیابی می‌شود؛
- Source Evidence Pilot بعدی برای Human Review نمایش داده می‌شود؛
- Full 572 Generation همچنان Blocked باقی می‌ماند.

</div>

In [60]:
# Code Cell 60

cell60_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell59_gate") is not True:
    cell60_failures.append(
        "CELL59_NOT_PASS"
    )


if (
    globals().get(
        "cell59_current_session",
        {},
    ).get(
        "current_state"
    )
    !=
    "CITATION_HUMAN_REVIEW_PENDING"
):
    cell60_failures.append(
        "CITATION_NOT_AT_HUMAN_REVIEW_GATE"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell60_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Human Citation ACCEPT
# ============================================================

CELL60_CITATION_HUMAN_NOTE = (
    "Human Reviewer accepted the Citation Mapping for the "
    "current Target Evidence."
)


cell60_complete_pending_session = (
    cell13_apply_event(
        cell59_current_session,
        "CITATION_ACCEPTED",
        human_note=
            CELL60_CITATION_HUMAN_NOTE,
    )
)


cell60_human_decisions = (
    cell60_complete_pending_session.get(
        "human_decisions",
        [],
    )
)


cell60_citation_accept_valid = all([
    cell60_complete_pending_session.get(
        "current_state"
    )
    ==
    "PILOT_COMPLETE_PENDING_QUEUE_ADVANCE",

    cell60_complete_pending_session.get(
        "queue_advance_performed"
    )
    is False,

    bool(
        cell60_human_decisions
    ),

    (
        cell60_human_decisions[
            -1
        ].get(
            "event"
        )
        ==
        "CITATION_ACCEPTED"
        if cell60_human_decisions
        else False
    ),
])


# ============================================================
# 3) Final Citation record
# ============================================================

cell60_final_citation_mapping_record = copy.deepcopy(
    cell59_citation_mapping_record
)


cell60_final_citation_mapping_record[
    "human_adjudication_status"
] = "ACCEPT"


cell60_final_citation_mapping_record[
    "human_review"
] = {
    "human_decision":
        "ACCEPT",

    "human_note":
        CELL60_CITATION_HUMAN_NOTE,

    "human_context_used_as_evidence":
        False,

    "external_knowledge_used":
        False,
}


# ============================================================
# 4) Complete Human Review case
# ============================================================

cell60_completed_case = copy.deepcopy(
    cell59_active_workspace
)


cell60_completed_case[
    "workflow_state"
] = "PILOT_COMPLETE"


cell60_completed_case[
    "status"
] = "COMPLETED_HUMAN_REVIEW"


cell60_completed_case[
    "final_status"
] = "PILOT_COMPLETE"


cell60_completed_case[
    "citation_review"
][
    "human_decision"
] = "ACCEPT"


cell60_completed_case[
    "citation_review"
][
    "human_note"
] = CELL60_CITATION_HUMAN_NOTE


cell60_completed_case[
    "citation_review"
][
    "citation_mapping_record"
] = copy.deepcopy(
    cell60_final_citation_mapping_record
)


cell60_completed_case[
    "completion_summary"
] = {
    "retarget_used":
        False,

    "target_evidence_id":
        CELL59_TARGET_EVIDENCE_ID,

    "question_decision":
        "ACCEPT",

    "expected_answer_decision":
        "ACCEPT",

    "citation_decision":
        "ACCEPT",

    "author_final_review":
        "NOT_STARTED",
}


cell60_completed_case[
    "author_final_review"
] = {
    "status":
        "NOT_STARTED",

    "decision":
        None,

    "notes":
        None,
}


# ============================================================
# 5) Queue advance: 12 -> 11
# ============================================================

cell60_previous_remaining_pilot_ids = copy.deepcopy(
    cell58_remaining_pilot_ids
)


cell60_remaining_pilot_ids = copy.deepcopy(
    cell60_previous_remaining_pilot_ids[
        1:
    ]
)


CELL60_COMPLETED_PILOT_ID = (
    cell60_previous_remaining_pilot_ids[
        0
    ]
)


cell60_queue_advance_valid = all([
    len(
        cell60_previous_remaining_pilot_ids
    )
    ==
    12,

    CELL60_COMPLETED_PILOT_ID
    ==
    CELL59_PILOT_ID,

    len(
        cell60_remaining_pilot_ids
    )
    ==
    11,

    CELL60_COMPLETED_PILOT_ID
    not in
    cell60_remaining_pilot_ids,
])


CELL60_NEXT_PILOT_ID = (
    cell60_remaining_pilot_ids[
        0
    ]
)


# ============================================================
# 6) Find next Pilot packet
# ============================================================

cell60_raw_next_packet_candidates = (
    cell58_find_pilot_packet_candidates(
        cell13_seed,
        CELL60_NEXT_PILOT_ID,
    )
)


cell60_packet_candidates_by_sha = {}


for candidate in cell60_raw_next_packet_candidates:

    packet_sha = cell10_stable_sha(
        candidate[
            "packet"
        ]
    )


    previous = (
        cell60_packet_candidates_by_sha.get(
            packet_sha
        )
    )


    if (
        previous is None
        or
        candidate[
            "score"
        ]
        >
        previous[
            "score"
        ]
    ):

        cell60_packet_candidates_by_sha[
            packet_sha
        ] = candidate


cell60_ranked_next_packet_candidates = sorted(
    cell60_packet_candidates_by_sha.values(),
    key=lambda item: (
        -item[
            "score"
        ],
        item[
            "path"
        ],
    ),
)


cell60_next_pilot_packet = (
    copy.deepcopy(
        cell60_ranked_next_packet_candidates[
            0
        ][
            "packet"
        ]
    )
    if cell60_ranked_next_packet_candidates
    else None
)


if not isinstance(
    cell60_next_pilot_packet,
    dict,
):

    cell60_failures.append(
        "NEXT_PILOT_PACKET_NOT_FOUND"
    )


# ============================================================
# 7) Extract next Pilot identities
# ============================================================

cell60_next_pilot_ids_found = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell60_next_pilot_packet,
            "question-pilot:",
        )
    )
)


cell60_next_golden_case_ids = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell60_next_pilot_packet,
            "golden-case:",
        )
    )
)


cell60_next_task_draft_ids = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell60_next_pilot_packet,
            "task-draft:",
        )
    )
)


cell60_next_target_evidence_ids = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell60_next_pilot_packet,
            "target-evidence:",
        )
    )
)


cell60_next_identity_valid = all([
    cell60_next_pilot_ids_found
    ==
    [
        CELL60_NEXT_PILOT_ID
    ],

    len(
        cell60_next_golden_case_ids
    )
    ==
    1,

    len(
        cell60_next_task_draft_ids
    )
    ==
    1,

    len(
        cell60_next_target_evidence_ids
    )
    ==
    1,
])


CELL60_NEXT_GOLDEN_CASE_ID = (
    cell60_next_golden_case_ids[
        0
    ]
    if
    len(
        cell60_next_golden_case_ids
    )
    ==
    1
    else
    None
)


CELL60_NEXT_TASK_DRAFT_ID = (
    cell60_next_task_draft_ids[
        0
    ]
    if
    len(
        cell60_next_task_draft_ids
    )
    ==
    1
    else
    None
)


CELL60_NEXT_TARGET_EVIDENCE_ID = (
    cell60_next_target_evidence_ids[
        0
    ]
    if
    len(
        cell60_next_target_evidence_ids
    )
    ==
    1
    else
    None
)


# ============================================================
# 8) Extract next Target Evidence
# ============================================================

cell60_target_part_candidates = (
    cell58_find_key_values(
        cell60_next_pilot_packet,
        "target_text_parts",
    )
)


cell60_target_parts_by_sha = {}


for parts in cell60_target_part_candidates:

    if not isinstance(
        parts,
        list,
    ):
        continue


    if len(
        parts
    ) == 0:
        continue


    parts_sha = cell10_stable_sha(
        parts
    )


    cell60_target_parts_by_sha[
        parts_sha
    ] = parts


cell60_unique_target_part_sets = list(
    cell60_target_parts_by_sha.values()
)


cell60_next_target_text_parts = (
    copy.deepcopy(
        cell60_unique_target_part_sets[
            0
        ]
    )
    if
    len(
        cell60_unique_target_part_sets
    )
    ==
    1
    else
    None
)


if not isinstance(
    cell60_next_target_text_parts,
    list,
):

    cell60_failures.append(
        "NEXT_TARGET_TEXT_PARTS_NOT_UNIQUE"
    )


# ============================================================
# 9) Build readable next Evidence
# ============================================================

cell60_next_evidence_texts = []


for part in (
    cell60_next_target_text_parts
    or
    []
):

    if isinstance(
        part,
        str,
    ):

        text = part.strip()


    elif isinstance(
        part,
        dict,
    ):

        text = (
            part.get(
                "text"
            )
            or
            part.get(
                "target_text"
            )
            or
            part.get(
                "content"
            )
            or
            ""
        )


        if isinstance(
            text,
            str,
        ):

            text = text.strip()


        else:

            text = ""


    else:

        text = ""


    if text:

        cell60_next_evidence_texts.append(
            text
        )


cell60_next_combined_target_text = (
    "\n\n".join(
        cell60_next_evidence_texts
    )
)


cell60_next_evidence_available = bool(
    cell60_next_combined_target_text.strip()
)


# ============================================================
# 10) Next Pilot engine session
# ============================================================

cell60_next_pilot_session = (
    cell13_build_session(
        cell60_next_pilot_packet
    )
)


cell60_next_session_valid = (
    cell60_next_pilot_session.get(
        "current_state"
    )
    ==
    "QUESTION_DRAFT_PENDING"
)


# ============================================================
# 11) Update completed Human Review cases
# ============================================================

cell60_existing_completed_cases = copy.deepcopy(
    cell59_persisted_registry.get(
        "completed_human_review_cases",
        [],
    )
)


if not isinstance(
    cell60_existing_completed_cases,
    list,
):

    cell60_failures.append(
        "COMPLETED_HUMAN_REVIEW_CASES_NOT_LIST"
    )

    cell60_existing_completed_cases = []


cell60_existing_completed_cases = [
    case
    for case
    in cell60_existing_completed_cases
    if not (
        isinstance(
            case,
            dict,
        )
        and
        case.get(
            "pilot_case_id"
        )
        ==
        CELL60_COMPLETED_PILOT_ID
    )
]


cell60_completed_human_review_cases = (
    cell60_existing_completed_cases
    +
    [
        copy.deepcopy(
            cell60_completed_case
        )
    ]
)


cell60_completed_case_count_valid = (
    len(
        cell60_completed_human_review_cases
    )
    ==
    11
)


# ============================================================
# 12) Rebuild Author Review queue
# ============================================================

cell60_author_review = copy.deepcopy(
    cell59_persisted_registry.get(
        "author_review",
        {},
    )
)


if not isinstance(
    cell60_author_review,
    dict,
):

    cell60_author_review = {}


cell60_author_ready_cases = copy.deepcopy(
    cell60_completed_human_review_cases
)


cell60_author_review[
    "cases_ready_for_review"
] = copy.deepcopy(
    cell60_author_ready_cases
)


cell60_author_review[
    "case_count"
] = len(
    cell60_author_ready_cases
)


cell60_author_review[
    "final_review_status"
] = "NOT_STARTED"


cell60_author_ready_count_valid = (
    len(
        cell60_author_ready_cases
    )
    ==
    11
)


cell60_author_ready_identity_valid = (
    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell60_author_ready_cases
        if isinstance(
            case,
            dict,
        )
    }
    ==
    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell60_completed_human_review_cases
        if isinstance(
            case,
            dict,
        )
    }
)


# ============================================================
# 13) Build next active workspace
# ============================================================

cell60_next_active_workspace = {
    "pilot_case_id":
        CELL60_NEXT_PILOT_ID,

    "golden_case_id":
        CELL60_NEXT_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL60_NEXT_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL60_NEXT_TARGET_EVIDENCE_ID,

    "status":
        "ACTIVE_HUMAN_REVIEW",

    "workflow_state":
        "QUESTION_DRAFT_PENDING",

    "source_evidence":
        copy.deepcopy(
            cell60_next_target_text_parts
        ),

    "combined_target_text":
        cell60_next_combined_target_text,

    "question_review":
        None,

    "expected_answer_review":
        None,

    "citation_review":
        None,

    "author_final_review": {
        "status":
            "NOT_STARTED",

        "decision":
            None,

        "notes":
            None,
    },
}


# ============================================================
# 14) New authoritative continuation state
# ============================================================

cell60_continuation_state_core = {
    "next_pilot_id":
        CELL60_NEXT_PILOT_ID,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell60_remaining_pilot_ids
        ),

    "remaining_count":
        11,

    "active_pilot_status":
        "QUESTION_DRAFT_PENDING",

    "active_target_evidence_id":
        CELL60_NEXT_TARGET_EVIDENCE_ID,

    "last_completed_pilot_id":
        CELL60_COMPLETED_PILOT_ID,

    "last_completion_type":
        "STANDARD_PILOT_COMPLETE",
}


cell60_continuation_state_sha256 = (
    cell10_stable_sha(
        cell60_continuation_state_core
    )
)


cell60_continuation_state = {
    **copy.deepcopy(
        cell60_continuation_state_core
    ),

    "continuation_state_sha256":
        cell60_continuation_state_sha256,
}


# ============================================================
# 15) Persist Registry
# ============================================================

cell60_persistent_registry_payload = copy.deepcopy(
    cell59_persisted_registry
)


cell60_persistent_registry_payload[
    "completed_human_review_cases"
] = copy.deepcopy(
    cell60_completed_human_review_cases
)


cell60_persistent_registry_payload[
    "author_review"
] = copy.deepcopy(
    cell60_author_review
)


cell60_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell60_next_active_workspace
)


cell60_persistent_registry_payload[
    "remaining_pilot_count"
] = 11


cell60_persistent_registry_payload[
    "continuation_state"
] = copy.deepcopy(
    cell60_continuation_state
)


cell60_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell60_persistent_registry_payload
    )
)


# ============================================================
# 16) Read-back validation
# ============================================================

cell60_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell60_persisted_active_case = (
    cell60_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell60_persisted_completed_cases = (
    cell60_persisted_registry.get(
        "completed_human_review_cases",
        [],
    )
)


cell60_persisted_author_ready_cases = (
    cell60_persisted_registry.get(
        "author_review",
        {},
    ).get(
        "cases_ready_for_review",
        [],
    )
)


cell60_persistence_valid = all([
    isinstance(
        cell60_persisted_completed_cases,
        list,
    ),

    len(
        cell60_persisted_completed_cases
    )
    ==
    11,

    isinstance(
        cell60_persisted_author_ready_cases,
        list,
    ),

    len(
        cell60_persisted_author_ready_cases
    )
    ==
    11,

    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell60_persisted_completed_cases
        if isinstance(
            case,
            dict,
        )
    }
    ==
    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell60_persisted_author_ready_cases
        if isinstance(
            case,
            dict,
        )
    },

    cell60_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    11,

    cell60_persisted_registry.get(
        "continuation_state",
        {},
    ).get(
        "continuation_state_sha256"
    )
    ==
    cell60_continuation_state_sha256,

    cell60_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL60_NEXT_PILOT_ID,

    cell60_persisted_active_case.get(
        "workflow_state"
    )
    ==
    "QUESTION_DRAFT_PENDING",
])


# ============================================================
# 17) Safety
# ============================================================

cell60_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell60_full_572_blocked = (
    cell59_full_572_blocked
    is True
)


cell60_no_commit_push = True


# ============================================================
# 18) Final Gate
# ============================================================

cell60_gate = all([
    len(
        cell60_failures
    )
    ==
    0,

    cell60_citation_accept_valid,

    cell60_queue_advance_valid,

    cell60_next_identity_valid,

    cell60_next_evidence_available,

    cell60_next_session_valid,

    cell60_completed_case_count_valid,

    cell60_author_ready_count_valid,

    cell60_author_ready_identity_valid,

    cell60_persistence_valid,

    cell60_base_seed_unchanged,

    cell60_full_572_blocked,

    cell60_no_commit_push,
])


# ============================================================
# 19) Diagnostics
# ============================================================

print(
    "=== PILOT #12 COMPLETE + QUEUE ADVANCE + NEXT PILOT ==="
)


print(
    "\n=== COMPLETED PILOT ==="
)

print(
    "Pilot:",
    CELL60_COMPLETED_PILOT_ID,
)

print(
    "Retarget:",
    False,
)

print(
    "Question:",
    "ACCEPT",
)

print(
    "Expected Answer:",
    "ACCEPT",
)

print(
    "Citation:",
    "ACCEPT",
)

print(
    "Final Engine State:",
    cell60_complete_pending_session.get(
        "current_state"
    ),
)


print(
    "\n=== COMPLETED ARTIFACTS ==="
)

print(
    "Target Evidence:",
    CELL59_TARGET_EVIDENCE_ID,
)

print(
    "Question Draft:",
    cell59_question_draft_id,
)

print(
    "Expected Answer Draft:",
    cell59_expected_answer_draft_id,
)

print(
    "Citation Mapping:",
    cell59_citation_mapping_id,
)


print(
    "\n=== QUEUE ADVANCE ==="
)

print(
    "Before:",
    len(
        cell60_previous_remaining_pilot_ids
    ),
)

print(
    "After:",
    len(
        cell60_remaining_pilot_ids
    ),
)

print(
    "Queue advanced exactly once:",
    cell60_queue_advance_valid,
)


print(
    "\n=== HUMAN REVIEW REGISTRY ==="
)

print(
    "Completed Human Review cases:",
    len(
        cell60_completed_human_review_cases
    ),
)

print(
    "Cases ready for Author Review:",
    len(
        cell60_author_ready_cases
    ),
)

print(
    "Author ready identities match completed cases:",
    cell60_author_ready_identity_valid,
)

print(
    "Author Final Review:",
    "NOT_STARTED",
)

print(
    "Registry:",
    cell21_registry_path,
)

print(
    "Persistence valid:",
    cell60_persistence_valid,
)


print(
    "\n=== NEXT PILOT ==="
)

print(
    "Pilot:",
    CELL60_NEXT_PILOT_ID,
)

print(
    "Golden Case:",
    CELL60_NEXT_GOLDEN_CASE_ID,
)

print(
    "Task Draft:",
    CELL60_NEXT_TASK_DRAFT_ID,
)

print(
    "Target Evidence:",
    CELL60_NEXT_TARGET_EVIDENCE_ID,
)

print(
    "Workflow:",
    cell60_next_pilot_session.get(
        "current_state"
    ),
)


print(
    "\n=== NEXT SOURCE EVIDENCE ==="
)

print(
    cell60_next_combined_target_text
)


print(
    "\n=== CONTINUATION STATE ==="
)

print(
    "Remaining Pilots:",
    11,
)

print(
    "Continuation SHA256:",
    cell60_continuation_state_sha256,
)


print(
    "\n=== SAFETY ==="
)

print(
    "Base Seed unchanged:",
    cell60_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell60_full_572_blocked,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell60_failures
)


print(
    "\nCODE CELL 60 GATE:",
    (
        "PASS"
        if cell60_gate
        else "FAIL"
    ),
)


assert cell60_gate, (
    "Citation ACCEPT / Pilot #12 completion / Queue advance / "
    "Next Pilot preparation failed. Repair this SAME "
    "Code Cell 60 before continuing."
)

=== PILOT #12 COMPLETE + QUEUE ADVANCE + NEXT PILOT ===

=== COMPLETED PILOT ===
Pilot: question-pilot:9d1a1208092facc8407fed78c07f5a90
Retarget: False
Question: ACCEPT
Expected Answer: ACCEPT
Citation: ACCEPT
Final Engine State: PILOT_COMPLETE_PENDING_QUEUE_ADVANCE

=== COMPLETED ARTIFACTS ===
Target Evidence: target-evidence:c991cbc1bbc326922a46ec51bb9efb6c
Question Draft: question-draft:ad1e5248b6b0eba92b9ee3ed2c2d4201
Expected Answer Draft: expected-answer-draft:46549a9491457aed12fbcd661fda9692
Citation Mapping: citation-mapping:10f1c1a0f57c9a7c6a18bf0953b218f0

=== QUEUE ADVANCE ===
Before: 12
After: 11
Queue advanced exactly once: True

=== HUMAN REVIEW REGISTRY ===
Completed Human Review cases: 11
Cases ready for Author Review: 11
Author ready identities match completed cases: True
Author Final Review: NOT_STARTED
Registry: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/data/phase2/human_review/human_review_registry.json
Persistence valid: True

=== NE

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 61 — Question/Answer ACCEPT + Citation Draft for Pilot #13

Human Reviewer سؤال و جواب این Pilot را پذیرفت.

سؤال پذیرفته‌شده:

**چه زمانی یک پس‌انداز را می‌توان سرمایه‌گذاری دانست؟**

جواب پذیرفته‌شده:

**وقتی ارزش دارایی فقط حفظ شود، هنوز پس‌انداز است. اما اگر ارزش آن به‌طور واقعی و معنادار بیشتر شود و ارزش افزوده ایجاد کند، می‌توان آن را سرمایه‌گذاری دانست.**

در این مرحله:

- Question با تصمیم `ACCEPT` ثبت می‌شود؛
- Expected Answer با تصمیم `ACCEPT` ثبت می‌شود؛
- Citation فقط از Target Evidence فعلی ساخته می‌شود؛
- هیچ Retarget انجام نمی‌شود؛
- Citation وارد Human Review می‌شود؛
- Queue همچنان 11 باقی می‌ماند؛
- Full 572 Generation همچنان Blocked است.

</div>

In [61]:
# Code Cell 61

cell61_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell60_gate") is not True:
    cell61_failures.append(
        "CELL60_NOT_PASS"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell61_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Accepted Human Review content
# ============================================================

CELL61_PILOT_ID = (
    CELL60_NEXT_PILOT_ID
)


CELL61_GOLDEN_CASE_ID = (
    CELL60_NEXT_GOLDEN_CASE_ID
)


CELL61_TASK_DRAFT_ID = (
    CELL60_NEXT_TASK_DRAFT_ID
)


CELL61_TARGET_EVIDENCE_ID = (
    CELL60_NEXT_TARGET_EVIDENCE_ID
)


CELL61_ACCEPTED_QUESTION = (
    "چه زمانی یک پس‌انداز را می‌توان سرمایه‌گذاری دانست؟"
)


CELL61_ACCEPTED_ANSWER = (
    "وقتی ارزش دارایی فقط حفظ شود، هنوز پس‌انداز است. "
    "اما اگر ارزش آن به‌طور واقعی و معنادار بیشتر شود "
    "و ارزش افزوده ایجاد کند، می‌توان آن را سرمایه‌گذاری دانست."
)


CELL61_QUESTION_HUMAN_NOTE = (
    "Human Reviewer accepted the proposed question."
)


CELL61_ANSWER_HUMAN_NOTE = (
    "Human Reviewer accepted the proposed answer."
)


# ============================================================
# 3) Current Target Evidence
# ============================================================

cell61_target_parts = copy.deepcopy(
    cell60_next_target_text_parts
)


cell61_combined_target_text = (
    cell60_next_combined_target_text
)


cell61_target_parts_valid = all([
    isinstance(
        cell61_target_parts,
        list,
    ),

    len(
        cell61_target_parts
    )
    >
    0,

    isinstance(
        cell61_combined_target_text,
        str,
    ),

    bool(
        cell61_combined_target_text.strip()
    ),
])


# ============================================================
# 4) Grounding validation
# ============================================================

def cell61_norm(
    text,
):

    if not isinstance(
        text,
        str,
    ):
        return ""


    value = (
        text
        .replace(
            "ي",
            "ی",
        )
        .replace(
            "ك",
            "ک",
        )
        .replace(
            "\u200c",
            " ",
        )
    )


    value = re.sub(
        r"\s+",
        " ",
        value,
    )


    return value.strip()


cell61_source_surface = cell61_norm(
    cell61_combined_target_text
)


cell61_grounding = {
    "saving_vs_investment":
        (
            "سرمایه گذاری"
            in
            cell61_source_surface
            and
            "پس انداز"
            in
            cell61_source_surface
        ),

    "value_added":
        (
            "ارزش افزوده"
            in
            cell61_source_surface
        ),

    "value_preservation":
        (
            "ارزشش تغییری نمی کنه"
            in
            cell61_source_surface
            or
            "ارزش ثابت"
            in
            cell61_source_surface
        ),

    "inflation_equivalence":
        (
            "معادل تورم"
            in
            cell61_source_surface
        ),

    "still_saving":
        (
            "این هم پس اندازه"
            in
            cell61_source_surface
            or
            "این هم پس انداز"
            in
            cell61_source_surface
        ),

    "meaningful_real_increase":
        (
            "به طور معناداری بیشتر شد"
            in
            cell61_source_surface
        ),
}


cell61_grounding_pass = all(
    cell61_grounding.values()
)


# ============================================================
# 5) Start Engine session
# ============================================================

cell61_question_pending_session = (
    cell13_build_session(
        cell60_next_pilot_packet
    )
)


cell61_session_start_valid = (
    cell61_question_pending_session.get(
        "current_state"
    )
    ==
    "QUESTION_DRAFT_PENDING"
)


# ============================================================
# 6) Question Draft
# ============================================================

cell61_question_identity = {
    "identity_version":
        "question_draft_identity_v1",

    "pilot_case_id":
        CELL61_PILOT_ID,

    "target_evidence_id":
        CELL61_TARGET_EVIDENCE_ID,

    "question_text":
        CELL61_ACCEPTED_QUESTION,

    "revision_number":
        1,
}


cell61_question_draft_id = (
    "question-draft:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell61_question_identity
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


cell61_question_record = {
    "record_version":
        "question_draft_record_v1",

    "question_draft_id":
        cell61_question_draft_id,

    "pilot_case_id":
        CELL61_PILOT_ID,

    "golden_case_id":
        CELL61_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL61_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL61_TARGET_EVIDENCE_ID,

    "question_text":
        CELL61_ACCEPTED_QUESTION,

    "revision_number":
        1,

    "machine_validation_status":
        (
            "PASS_PENDING_HUMAN_ADJUDICATION"
            if cell61_grounding_pass
            else
            "FAIL_REQUIRES_REVISION"
        ),

    "human_adjudication_status":
        "PENDING",

    "external_knowledge_used":
        False,
}


cell61_question_review_session = (
    cell13_apply_event(
        cell61_question_pending_session,
        "QUESTION_DRAFT_REGISTERED",
        artifact=
            cell61_question_record,
    )
)


cell61_answer_pending_session = (
    cell13_apply_event(
        cell61_question_review_session,
        "QUESTION_ACCEPTED",
        human_note=
            CELL61_QUESTION_HUMAN_NOTE,
    )
)


cell61_question_accept_valid = all([
    cell61_question_review_session.get(
        "current_state"
    )
    ==
    "QUESTION_HUMAN_REVIEW_PENDING",

    cell61_answer_pending_session.get(
        "current_state"
    )
    ==
    "EXPECTED_ANSWER_DRAFT_PENDING",
])


# ============================================================
# 7) Expected Answer
# ============================================================

cell61_answer_identity = {
    "identity_version":
        "expected_answer_identity_v1",

    "pilot_case_id":
        CELL61_PILOT_ID,

    "question_draft_id":
        cell61_question_draft_id,

    "target_evidence_id":
        CELL61_TARGET_EVIDENCE_ID,

    "answer_text":
        CELL61_ACCEPTED_ANSWER,

    "revision_number":
        1,
}


cell61_expected_answer_draft_id = (
    "expected-answer-draft:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell61_answer_identity
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


cell61_expected_answer_record = {
    "record_version":
        "expected_answer_draft_record_v1",

    "expected_answer_draft_id":
        cell61_expected_answer_draft_id,

    "pilot_case_id":
        CELL61_PILOT_ID,

    "golden_case_id":
        CELL61_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL61_TASK_DRAFT_ID,

    "question_draft_id":
        cell61_question_draft_id,

    "target_evidence_id":
        CELL61_TARGET_EVIDENCE_ID,

    "expected_answer_text":
        CELL61_ACCEPTED_ANSWER,

    "revision_number":
        1,

    "machine_validation_status":
        (
            "PASS_PENDING_HUMAN_ADJUDICATION"
            if cell61_grounding_pass
            else
            "FAIL_REQUIRES_REVISION"
        ),

    "human_adjudication_status":
        "PENDING",

    "external_knowledge_used":
        False,
}


cell61_answer_review_session = (
    cell13_apply_event(
        cell61_answer_pending_session,
        "EXPECTED_ANSWER_REGISTERED",
        artifact=
            cell61_expected_answer_record,
    )
)


cell61_citation_pending_session = (
    cell13_apply_event(
        cell61_answer_review_session,
        "EXPECTED_ANSWER_ACCEPTED",
        human_note=
            CELL61_ANSWER_HUMAN_NOTE,
    )
)


cell61_answer_accept_valid = all([
    cell61_answer_review_session.get(
        "current_state"
    )
    ==
    "EXPECTED_ANSWER_HUMAN_REVIEW_PENDING",

    cell61_citation_pending_session.get(
        "current_state"
    )
    ==
    "CITATION_MAPPING_PENDING",
])


# ============================================================
# 8) Build Citation entries from Target Evidence
# ============================================================

cell61_citation_entries = []


for index, part in enumerate(
    cell61_target_parts
):

    if isinstance(
        part,
        dict,
    ):

        citation_entry = copy.deepcopy(
            part
        )


        citation_entry[
            "citation_index"
        ] = index


        citation_entry[
            "target_evidence_id"
        ] = CELL61_TARGET_EVIDENCE_ID


        citation_entry[
            "locator_origin"
        ] = "COPIED_FROM_TARGET_EVIDENCE"


        citation_entry[
            "external_locator_lookup_used"
        ] = False


    else:

        citation_entry = {
            "citation_index":
                index,

            "target_evidence_id":
                CELL61_TARGET_EVIDENCE_ID,

            "text":
                part,

            "locator_origin":
                "COPIED_FROM_TARGET_EVIDENCE",

            "external_locator_lookup_used":
                False,
        }


    cell61_citation_entries.append(
        citation_entry
    )


# ============================================================
# 9) No-fabrication validation
# ============================================================

def cell61_strip_added_citation_fields(
    entry,
):

    output = copy.deepcopy(
        entry
    )


    output.pop(
        "citation_index",
        None,
    )


    output.pop(
        "target_evidence_id",
        None,
    )


    output.pop(
        "locator_origin",
        None,
    )


    output.pop(
        "external_locator_lookup_used",
        None,
    )


    return output


cell61_citation_back_projection = [
    cell61_strip_added_citation_fields(
        entry
    )
    for entry
    in cell61_citation_entries
]


cell61_no_locator_fabrication = (
    cell61_citation_back_projection
    ==
    cell61_target_parts
)


# ============================================================
# 10) Citation Mapping ID
# ============================================================

cell61_citation_identity = {
    "identity_version":
        "citation_mapping_identity_v1",

    "pilot_case_id":
        CELL61_PILOT_ID,

    "question_draft_id":
        cell61_question_draft_id,

    "expected_answer_draft_id":
        cell61_expected_answer_draft_id,

    "target_evidence_id":
        CELL61_TARGET_EVIDENCE_ID,

    "citation_entries":
        copy.deepcopy(
            cell61_citation_entries
        ),
}


cell61_citation_mapping_id = (
    "citation-mapping:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell61_citation_identity
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


cell61_citation_machine_pass = all([
    len(
        cell61_citation_entries
    )
    ==
    len(
        cell61_target_parts
    ),

    len(
        cell61_citation_entries
    )
    >
    0,

    cell61_no_locator_fabrication,

    all(
        entry.get(
            "target_evidence_id"
        )
        ==
        CELL61_TARGET_EVIDENCE_ID
        for entry
        in cell61_citation_entries
    ),

    all(
        entry.get(
            "external_locator_lookup_used"
        )
        is False
        for entry
        in cell61_citation_entries
    ),
])


cell61_citation_mapping_record = {
    "record_version":
        "citation_mapping_record_v1",

    "citation_mapping_id":
        cell61_citation_mapping_id,

    "pilot_case_id":
        CELL61_PILOT_ID,

    "golden_case_id":
        CELL61_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL61_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL61_TARGET_EVIDENCE_ID,

    "question_draft_id":
        cell61_question_draft_id,

    "expected_answer_draft_id":
        cell61_expected_answer_draft_id,

    "citation_entries":
        copy.deepcopy(
            cell61_citation_entries
        ),

    "citation_scope":
        "CURRENT_TARGET_EVIDENCE_ONLY",

    "retarget_used":
        False,

    "external_locator_lookup_used":
        False,

    "machine_validation_status":
        (
            "PASS_PENDING_HUMAN_ADJUDICATION"
            if cell61_citation_machine_pass
            else
            "FAIL_REQUIRES_REVISION"
        ),

    "human_adjudication_status":
        "PENDING",
}


cell61_current_session = (
    cell13_apply_event(
        cell61_citation_pending_session,
        "CITATION_MAPPING_REGISTERED",
        artifact=
            cell61_citation_mapping_record,
    )
)


cell61_citation_registered_valid = (
    cell61_current_session.get(
        "current_state"
    )
    ==
    "CITATION_HUMAN_REVIEW_PENDING"
)


# ============================================================
# 11) Update Human Review workspace
# ============================================================

cell61_active_workspace = copy.deepcopy(
    cell60_persisted_active_case
)


cell61_active_workspace[
    "workflow_state"
] = "CITATION_HUMAN_REVIEW_PENDING"


cell61_active_workspace[
    "status"
] = "ACTIVE_HUMAN_REVIEW"


cell61_active_workspace[
    "question_review"
] = {
    "question_draft_id":
        cell61_question_draft_id,

    "question_text":
        CELL61_ACCEPTED_QUESTION,

    "machine_validation_status":
        cell61_question_record[
            "machine_validation_status"
        ],

    "human_decision":
        "ACCEPT",

    "human_note":
        CELL61_QUESTION_HUMAN_NOTE,
}


cell61_active_workspace[
    "expected_answer_review"
] = {
    "expected_answer_draft_id":
        cell61_expected_answer_draft_id,

    "expected_answer_text":
        CELL61_ACCEPTED_ANSWER,

    "machine_validation_status":
        cell61_expected_answer_record[
            "machine_validation_status"
        ],

    "human_decision":
        "ACCEPT",

    "human_note":
        CELL61_ANSWER_HUMAN_NOTE,
}


cell61_active_workspace[
    "citation_review"
] = {
    "citation_mapping_id":
        cell61_citation_mapping_id,

    "citation_entries":
        copy.deepcopy(
            cell61_citation_entries
        ),

    "machine_validation_status":
        cell61_citation_mapping_record[
            "machine_validation_status"
        ],

    "human_decision":
        "PENDING",
}


# ============================================================
# 12) Updated continuation state
# ============================================================

cell61_continuation_core = {
    "next_pilot_id":
        CELL61_PILOT_ID,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell60_remaining_pilot_ids
        ),

    "remaining_count":
        11,

    "active_pilot_status":
        "CITATION_HUMAN_REVIEW_PENDING",

    "active_target_evidence_id":
        CELL61_TARGET_EVIDENCE_ID,

    "last_completed_pilot_id":
        cell60_continuation_state.get(
            "last_completed_pilot_id"
        ),

    "last_completion_type":
        cell60_continuation_state.get(
            "last_completion_type"
        ),
}


cell61_continuation_state_sha256 = (
    cell10_stable_sha(
        cell61_continuation_core
    )
)


cell61_continuation_state = {
    **copy.deepcopy(
        cell61_continuation_core
    ),

    "continuation_state_sha256":
        cell61_continuation_state_sha256,
}


# ============================================================
# 13) Persist Registry
# ============================================================

cell61_persistent_registry_payload = copy.deepcopy(
    cell60_persisted_registry
)


cell61_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell61_active_workspace
)


cell61_persistent_registry_payload[
    "remaining_pilot_count"
] = 11


cell61_persistent_registry_payload[
    "continuation_state"
] = copy.deepcopy(
    cell61_continuation_state
)


cell61_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell61_persistent_registry_payload
    )
)


# ============================================================
# 14) Read-back validation
# ============================================================

cell61_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell61_persisted_active_case = (
    cell61_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell61_persistence_valid = all([
    cell61_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL61_PILOT_ID,

    cell61_persisted_active_case.get(
        "workflow_state"
    )
    ==
    "CITATION_HUMAN_REVIEW_PENDING",

    cell61_persisted_active_case.get(
        "question_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell61_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell61_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "PENDING",

    cell61_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    11,

    cell61_persisted_registry.get(
        "continuation_state",
        {},
    ).get(
        "continuation_state_sha256"
    )
    ==
    cell61_continuation_state_sha256,
])


# ============================================================
# 15) Safety
# ============================================================

cell61_queue_still_11 = all([
    len(
        cell60_remaining_pilot_ids
    )
    ==
    11,

    cell60_remaining_pilot_ids[
        0
    ]
    ==
    CELL61_PILOT_ID,
])


cell61_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell61_full_572_blocked = (
    cell60_full_572_blocked
    is True
)


# ============================================================
# 16) Final Gate
# ============================================================

cell61_gate = all([
    len(
        cell61_failures
    )
    ==
    0,

    cell61_target_parts_valid,

    cell61_grounding_pass,

    cell61_session_start_valid,

    cell61_question_accept_valid,

    cell61_answer_accept_valid,

    cell61_no_locator_fabrication,

    cell61_citation_machine_pass,

    cell61_citation_registered_valid,

    cell61_persistence_valid,

    cell61_queue_still_11,

    cell61_base_seed_unchanged,

    cell61_full_572_blocked,
])


# ============================================================
# 17) Diagnostics
# ============================================================

print(
    "=== PILOT #13 QUESTION/ANSWER ACCEPT + CITATION DRAFT ==="
)


print(
    "\n=== PILOT ==="
)

print(
    "Pilot:",
    CELL61_PILOT_ID,
)

print(
    "Golden Case:",
    CELL61_GOLDEN_CASE_ID,
)

print(
    "Task Draft:",
    CELL61_TASK_DRAFT_ID,
)

print(
    "Target Evidence:",
    CELL61_TARGET_EVIDENCE_ID,
)

print(
    "Retarget:",
    False,
)


print(
    "\n=== ACCEPTED QUESTION ==="
)

print(
    CELL61_ACCEPTED_QUESTION
)

print(
    "Question Draft ID:",
    cell61_question_draft_id,
)

print(
    "Human Decision:",
    "ACCEPT",
)


print(
    "\n=== ACCEPTED ANSWER ==="
)

print(
    CELL61_ACCEPTED_ANSWER
)

print(
    "Expected Answer Draft ID:",
    cell61_expected_answer_draft_id,
)

print(
    "Human Decision:",
    "ACCEPT",
)


print(
    "\n=== GROUNDING ==="
)

for name, status in (
    cell61_grounding.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "Grounding PASS:",
    cell61_grounding_pass,
)


print(
    "\n=== CITATION DRAFT ==="
)

print(
    "Citation Mapping ID:",
    cell61_citation_mapping_id,
)

print(
    "Citation entry count:",
    len(
        cell61_citation_entries
    ),
)

print(
    "No locator fabrication:",
    cell61_no_locator_fabrication,
)

print(
    "Citation Machine PASS:",
    cell61_citation_machine_pass,
)


for index, entry in enumerate(
    cell61_citation_entries
):

    print(
        "\n--- CITATION ENTRY",
        index,
        "---"
    )

    print(
        cell10_canonical_json(
            entry
        )
    )


print(
    "\n=== WORKFLOW ==="
)

print(
    "Current State:",
    cell61_current_session.get(
        "current_state"
    ),
)

print(
    "Question:",
    "ACCEPT",
)

print(
    "Expected Answer:",
    "ACCEPT",
)

print(
    "Citation:",
    "PENDING",
)


print(
    "\n=== PERSISTENCE ==="
)

print(
    "Persistence valid:",
    cell61_persistence_valid,
)


print(
    "\n=== QUEUE / SAFETY ==="
)

print(
    "Queue still 11:",
    cell61_queue_still_11,
)

print(
    "Base Seed unchanged:",
    cell61_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell61_full_572_blocked,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell61_failures
)


print(
    "\nCODE CELL 61 GATE:",
    (
        "PASS"
        if cell61_gate
        else "FAIL"
    ),
)


assert cell61_gate, (
    "Pilot #13 Question/Answer ACCEPT / Citation Draft failed. "
    "Repair this SAME Code Cell 61 before Citation Human Review."
)

=== PILOT #13 QUESTION/ANSWER ACCEPT + CITATION DRAFT ===

=== PILOT ===
Pilot: question-pilot:950359cf41d2817c535d486f8cd49aab
Golden Case: golden-case:6d518619cd942854c250fa239fd3ef99
Task Draft: task-draft:4980533d64e4f92524186b23a7f07a0a
Target Evidence: target-evidence:ddefa7060e7bb45892cc5d44fc8148e3
Retarget: False

=== ACCEPTED QUESTION ===
چه زمانی یک پس‌انداز را می‌توان سرمایه‌گذاری دانست؟
Question Draft ID: question-draft:3bd98dbda83ea348caa3b4225ed155a1
Human Decision: ACCEPT

=== ACCEPTED ANSWER ===
وقتی ارزش دارایی فقط حفظ شود، هنوز پس‌انداز است. اما اگر ارزش آن به‌طور واقعی و معنادار بیشتر شود و ارزش افزوده ایجاد کند، می‌توان آن را سرمایه‌گذاری دانست.
Expected Answer Draft ID: expected-answer-draft:4bdb70be6127aad38fedb81431ca01ac
Human Decision: ACCEPT

=== GROUNDING ===
saving_vs_investment: True
value_added: True
value_preservation: True
inflation_equivalence: True
still_saving: True
meaningful_real_increase: True
Grounding PASS: True

=== CITATION DRAFT ===
Citation 

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 62 — Pilot #13 Attribution / Multi-Speaker Diagnostic

این Cell فقط بررسی می‌کند آیا Pilot #13 عمداً برای پوشش دو گوینده و Attribution Contrast طراحی شده است یا خیر.

هیچ Human Decision جدیدی ثبت نمی‌شود، Citation تغییر نمی‌کند، Queue جلو نمی‌رود و Registry نوشته نمی‌شود.

</div>

In [62]:
# Code Cell 62

cell62_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell61_gate") is not True:
    cell62_failures.append(
        "CELL61_NOT_PASS"
    )


# ============================================================
# 2) Current Pilot
# ============================================================

CELL62_PILOT_ID = CELL61_PILOT_ID


# ============================================================
# 3) Recursive metadata inspection
# ============================================================

def cell62_collect_matching_fields(
    value,
    path=(),
):

    matches = []


    keywords = (
        "speaker",
        "attribution",
        "contrast",
        "authority",
        "multi",
        "task_family",
        "question_archetype",
        "archetype",
        "stratum",
        "role",
    )


    if isinstance(
        value,
        dict,
    ):

        for key, item in value.items():

            key_text = str(
                key
            ).lower()


            if any(
                keyword
                in
                key_text
                for keyword
                in keywords
            ):

                matches.append({
                    "path":
                        path
                        +
                        (
                            str(
                                key
                            ),
                        ),

                    "value":
                        copy.deepcopy(
                            item
                        ),
                })


            matches.extend(
                cell62_collect_matching_fields(
                    item,
                    path
                    +
                    (
                        str(
                            key
                        ),
                    ),
                )
            )


    elif isinstance(
        value,
        list,
    ):

        for index, item in enumerate(
            value
        ):

            matches.extend(
                cell62_collect_matching_fields(
                    item,
                    path
                    +
                    (
                        str(
                            index
                        ),
                    ),
                )
            )


    return matches


cell62_metadata_matches = (
    cell62_collect_matching_fields(
        cell60_next_pilot_packet
    )
)


# ============================================================
# 4) Target-part speaker / role summary
# ============================================================

cell62_target_part_summary = []


for index, part in enumerate(
    cell61_target_parts
):

    if not isinstance(
        part,
        dict,
    ):
        continue


    speaker_state = (
        part.get(
            "speaker_state"
        )
        or
        {}
    )


    attribution = (
        part.get(
            "attribution"
        )
        or
        {}
    )


    cell62_target_part_summary.append({
        "index":
            index,

        "document_id":
            part.get(
                "document_id"
            ),

        "role":
            part.get(
                "role"
            ),

        "speaker_state":
            copy.deepcopy(
                speaker_state
            ),

        "attribution":
            copy.deepcopy(
                attribution
            ),
    })


# ============================================================
# 5) Detect explicit contrast signals
# ============================================================

cell62_roles = [
    item.get(
        "role"
    )
    for item
    in cell62_target_part_summary
]


cell62_has_contrast_roles = all([
    "ATTRIBUTION_CONTRAST_LEFT"
    in
    cell62_roles,

    "ATTRIBUTION_CONTRAST_RIGHT"
    in
    cell62_roles,
])


cell62_effective_speakers = [
    (
        item.get(
            "speaker_state"
        )
        or
        {}
    ).get(
        "effective_canonical"
    )
    or
    (
        item.get(
            "speaker_state"
        )
        or
        {}
    ).get(
        "effective_label"
    )
    for item
    in cell62_target_part_summary
]


cell62_distinct_speakers = sorted(
    set(
        speaker
        for speaker
        in cell62_effective_speakers
        if speaker
    )
)


# ============================================================
# 6) Search packet text for explicit contrast intent
# ============================================================

cell62_packet_serialized = (
    json.dumps(
        cell60_next_pilot_packet,
        ensure_ascii=False,
        sort_keys=True,
        default=str,
    )
    .lower()
)


cell62_contrast_terms = {
    "attribution_contrast":
        (
            "attribution_contrast"
            in
            cell62_packet_serialized
        ),

    "multi_speaker":
        (
            "multi_speaker"
            in
            cell62_packet_serialized
        ),

    "authority_constrained":
        (
            "authority_constrained"
            in
            cell62_packet_serialized
        ),

    "contrast":
        (
            "contrast"
            in
            cell62_packet_serialized
        ),
}


# ============================================================
# 7) Gate
# ============================================================

cell62_gate = all([
    len(
        cell62_failures
    )
    ==
    0,

    len(
        cell62_target_part_summary
    )
    >
    0,
])


# ============================================================
# 8) Diagnostics
# ============================================================

print(
    "=== PILOT #13 ATTRIBUTION / MULTI-SPEAKER DIAGNOSTIC ==="
)


print(
    "\n=== PILOT ==="
)

print(
    "Pilot:",
    CELL62_PILOT_ID,
)


print(
    "\n=== TARGET PARTS ==="
)


for item in cell62_target_part_summary:

    print(
        "\nPart:",
        item[
            "index"
        ],
    )

    print(
        "Document:",
        item[
            "document_id"
        ],
    )

    print(
        "Role:",
        item[
            "role"
        ],
    )

    print(
        "Speaker State:",
        item[
            "speaker_state"
        ],
    )

    print(
        "Attribution:",
        item[
            "attribution"
        ],
    )


print(
    "\n=== CONTRAST SIGNALS ==="
)

print(
    "Roles:",
    cell62_roles,
)

print(
    "Explicit left/right contrast roles:",
    cell62_has_contrast_roles,
)

print(
    "Distinct speakers:",
    cell62_distinct_speakers,
)


for name, status in (
    cell62_contrast_terms.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== RELEVANT PILOT METADATA ==="
)


for match in cell62_metadata_matches:

    print(
        " > ".join(
            match[
                "path"
            ]
        ),
        "=",
        match[
            "value"
        ],
    )


print(
    "\n=== SAFETY ==="
)

print(
    "Registry write:",
    False,
)

print(
    "Queue advance:",
    False,
)

print(
    "Citation changed:",
    False,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell62_failures
)


print(
    "\nCODE CELL 62 GATE:",
    (
        "PASS"
        if cell62_gate
        else "FAIL"
    ),
)


assert cell62_gate, (
    "Pilot #13 attribution diagnostic failed. "
    "Repair this SAME Code Cell 62 before changing Citation."
)

=== PILOT #13 ATTRIBUTION / MULTI-SPEAKER DIAGNOSTIC ===

=== PILOT ===
Pilot: question-pilot:950359cf41d2817c535d486f8cd49aab

=== TARGET PARTS ===

Part: 0
Document: program:uni-tehran:episode-010:utterance:00003
Role: ATTRIBUTION_CONTRAST_LEFT
Speaker State: {'base_status': 'EXPLICIT_SOURCE_MARKER', 'base_raw': 'استاد', 'effective_label': 'استاد', 'effective_canonical': 'Dr. Komeil Roudi', 'effective_primary': True}
Attribution: {'attribution_status': None, 'author': None, 'author_id': None, 'authorship_assertion_source': None, 'byline_raw': None, 'content_authority_type': None, 'credited_author_name': None, 'editorial_credit_raw': None, 'editorial_supervisor_name': None, 'knowledge_owner': None, 'reference_expert_name': None, 'speaker': 'Dr. Komeil Roudi'}

Part: 1
Document: program:uni-tehran:episode-010:utterance:00004
Role: ATTRIBUTION_CONTRAST_RIGHT
Speaker State: {'base_status': 'EXPLICIT_SOURCE_MARKER', 'base_raw': 'مجری', 'effective_label': 'مجری', 'effective_canonical': Non

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 63 — Pilot #13 Citation Attribution Revision

در این مرحله سؤال و جواب پذیرفته‌شده بدون تغییر باقی می‌مانند.

Citation فقط از نظر نقش گوینده اصلاح می‌شود:

- بخش مربوط به **Dr. Komeil Roudi** به‌عنوان Evidence اصلی پاسخ ثبت می‌شود.
- بخش مربوط به **مجری** فقط به‌عنوان Context / Attribution Contrast حفظ می‌شود و Evidence محتوایی پاسخ محسوب نمی‌شود.
- Target Evidence حذف یا Retarget نمی‌شود.
- ساختار Multi-Speaker این Pilot حفظ می‌شود.
- Citation جدید وارد Human Review می‌شود.
- Queue همچنان 11 باقی می‌ماند.

</div>

In [63]:
# Code Cell 63

cell63_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell62_gate") is not True:
    cell63_failures.append(
        "CELL62_NOT_PASS"
    )


if (
    globals().get(
        "cell61_citation_pending_session",
        {},
    ).get(
        "current_state"
    )
    !=
    "CITATION_MAPPING_PENDING"
):
    cell63_failures.append(
        "CITATION_PENDING_SESSION_NOT_AVAILABLE"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell63_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Preserve accepted Question / Answer exactly
# ============================================================

CELL63_ACCEPTED_QUESTION = (
    CELL61_ACCEPTED_QUESTION
)


CELL63_ACCEPTED_ANSWER = (
    CELL61_ACCEPTED_ANSWER
)


cell63_question_unchanged = (
    CELL63_ACCEPTED_QUESTION
    ==
    "چه زمانی یک پس‌انداز را می‌توان سرمایه‌گذاری دانست؟"
)


cell63_answer_unchanged = (
    CELL63_ACCEPTED_ANSWER
    ==
    cell61_expected_answer_record.get(
        "expected_answer_text"
    )
)


# ============================================================
# 3) Validate two-speaker Attribution structure
# ============================================================

cell63_source_entries = copy.deepcopy(
    cell61_citation_entries
)


cell63_two_entry_structure = (
    len(
        cell63_source_entries
    )
    ==
    2
)


cell63_entry0 = (
    cell63_source_entries[
        0
    ]
    if
    cell63_two_entry_structure
    else
    {}
)


cell63_entry1 = (
    cell63_source_entries[
        1
    ]
    if
    cell63_two_entry_structure
    else
    {}
)


cell63_entry0_speaker_state = (
    cell63_entry0.get(
        "speaker_state"
    )
    or
    {}
)


cell63_entry1_speaker_state = (
    cell63_entry1.get(
        "speaker_state"
    )
    or
    {}
)


cell63_attribution_structure_valid = all([
    cell63_two_entry_structure,

    cell63_entry0.get(
        "role"
    )
    ==
    "ATTRIBUTION_CONTRAST_LEFT",

    cell63_entry1.get(
        "role"
    )
    ==
    "ATTRIBUTION_CONTRAST_RIGHT",

    cell63_entry0_speaker_state.get(
        "effective_canonical"
    )
    ==
    "Dr. Komeil Roudi",

    cell63_entry0_speaker_state.get(
        "effective_primary"
    )
    is True,

    cell63_entry1_speaker_state.get(
        "effective_label"
    )
    ==
    "مجری",
])


# ============================================================
# 4) Build revised Citation entries
# ============================================================

cell63_revised_citation_entries = []


for index, entry in enumerate(
    cell63_source_entries
):

    revised_entry = copy.deepcopy(
        entry
    )


    revised_entry[
        "source_role"
    ] = entry.get(
        "role"
    )


    if index == 0:

        revised_entry[
            "answer_support_role"
        ] = "PRIMARY_ANSWER_EVIDENCE"

        revised_entry[
            "answer_support"
        ] = True

        revised_entry[
            "authority_role"
        ] = "PRIMARY_CORPUS_AUTHORITY"

        revised_entry[
            "authority_basis"
        ] = "DR_KOMEIL_ROUDI_EFFECTIVE_PRIMARY"

        revised_entry[
            "use_in_expected_answer_grounding"
        ] = True


    elif index == 1:

        revised_entry[
            "answer_support_role"
        ] = "CONTRAST_CONTEXT_ONLY"

        revised_entry[
            "answer_support"
        ] = False

        revised_entry[
            "authority_role"
        ] = "NON_PRIMARY_SPEAKER_CONTEXT"

        revised_entry[
            "authority_basis"
        ] = "HOST_UTTERANCE_WITHOUT_PRIMARY_AUTHORITY"

        revised_entry[
            "use_in_expected_answer_grounding"
        ] = False


    cell63_revised_citation_entries.append(
        revised_entry
    )


# ============================================================
# 5) Verify answer-support boundary
# ============================================================

cell63_primary_support_entries = [
    entry
    for entry
    in cell63_revised_citation_entries
    if entry.get(
        "answer_support"
    )
    is True
]


cell63_context_only_entries = [
    entry
    for entry
    in cell63_revised_citation_entries
    if entry.get(
        "answer_support"
    )
    is False
]


cell63_answer_support_boundary_valid = all([
    len(
        cell63_primary_support_entries
    )
    ==
    1,

    len(
        cell63_context_only_entries
    )
    ==
    1,

    cell63_primary_support_entries[
        0
    ].get(
        "document_id"
    )
    ==
    "program:uni-tehran:episode-010:utterance:00003",

    cell63_context_only_entries[
        0
    ].get(
        "document_id"
    )
    ==
    "program:uni-tehran:episode-010:utterance:00004",

    cell63_primary_support_entries[
        0
    ].get(
        "authority_role"
    )
    ==
    "PRIMARY_CORPUS_AUTHORITY",

    cell63_context_only_entries[
        0
    ].get(
        "authority_role"
    )
    ==
    "NON_PRIMARY_SPEAKER_CONTEXT",
])


# ============================================================
# 6) No locator / source fabrication
# ============================================================

def cell63_strip_revision_fields(
    entry,
):

    output = copy.deepcopy(
        entry
    )


    for key in [
        "source_role",
        "answer_support_role",
        "answer_support",
        "authority_role",
        "authority_basis",
        "use_in_expected_answer_grounding",
    ]:

        output.pop(
            key,
            None,
        )


    return output


cell63_back_projection = [
    cell63_strip_revision_fields(
        entry
    )
    for entry
    in cell63_revised_citation_entries
]


cell63_no_source_or_locator_fabrication = (
    cell63_back_projection
    ==
    cell61_citation_entries
)


# ============================================================
# 7) Revised Citation Mapping identity
# ============================================================

cell63_citation_identity = {
    "identity_version":
        "citation_mapping_identity_v2_attribution_boundary",

    "pilot_case_id":
        CELL61_PILOT_ID,

    "question_draft_id":
        cell61_question_draft_id,

    "expected_answer_draft_id":
        cell61_expected_answer_draft_id,

    "target_evidence_id":
        CELL61_TARGET_EVIDENCE_ID,

    "supersedes_citation_mapping_id":
        cell61_citation_mapping_id,

    "citation_entries":
        copy.deepcopy(
            cell63_revised_citation_entries
        ),
}


cell63_citation_mapping_id = (
    "citation-mapping:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell63_citation_identity
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


cell63_citation_mapping_record = {
    "record_version":
        "citation_mapping_record_v2_attribution_boundary",

    "citation_mapping_id":
        cell63_citation_mapping_id,

    "pilot_case_id":
        CELL61_PILOT_ID,

    "golden_case_id":
        CELL61_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL61_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL61_TARGET_EVIDENCE_ID,

    "question_draft_id":
        cell61_question_draft_id,

    "expected_answer_draft_id":
        cell61_expected_answer_draft_id,

    "citation_entries":
        copy.deepcopy(
            cell63_revised_citation_entries
        ),

    "citation_scope":
        "MULTI_SPEAKER_ATTRIBUTION_BOUNDARY",

    "primary_answer_evidence_count":
        1,

    "contrast_context_count":
        1,

    "primary_authority":
        "Dr. Komeil Roudi",

    "question_authority_prefix_added":
        False,

    "retarget_used":
        False,

    "supersedes_citation_mapping_id":
        cell61_citation_mapping_id,

    "revision_reason":
        (
            "Preserve the multi-speaker attribution contrast while "
            "restricting answer grounding to the primary Dr. Komeil "
            "Roudi utterance. Host speech remains context only."
        ),

    "external_locator_lookup_used":
        False,

    "machine_validation_status":
        "PASS_PENDING_HUMAN_ADJUDICATION",

    "human_adjudication_status":
        "PENDING",
}


# ============================================================
# 8) Register revised Citation from known citation-pending state
# ============================================================

cell63_current_session = (
    cell13_apply_event(
        cell61_citation_pending_session,
        "CITATION_MAPPING_REGISTERED",
        artifact=
            cell63_citation_mapping_record,
    )
)


cell63_session_valid = (
    cell63_current_session.get(
        "current_state"
    )
    ==
    "CITATION_HUMAN_REVIEW_PENDING"
)


# ============================================================
# 9) Update active Human Review workspace
# ============================================================

cell63_active_workspace = copy.deepcopy(
    cell61_persisted_active_case
)


cell63_active_workspace[
    "workflow_state"
] = "CITATION_HUMAN_REVIEW_PENDING"


cell63_active_workspace[
    "question_review"
][
    "question_text"
] = CELL63_ACCEPTED_QUESTION


cell63_active_workspace[
    "question_review"
][
    "human_decision"
] = "ACCEPT"


cell63_active_workspace[
    "expected_answer_review"
][
    "expected_answer_text"
] = CELL63_ACCEPTED_ANSWER


cell63_active_workspace[
    "expected_answer_review"
][
    "human_decision"
] = "ACCEPT"


cell63_active_workspace[
    "citation_review"
] = {
    "citation_mapping_id":
        cell63_citation_mapping_id,

    "supersedes_citation_mapping_id":
        cell61_citation_mapping_id,

    "citation_entries":
        copy.deepcopy(
            cell63_revised_citation_entries
        ),

    "machine_validation_status":
        "PASS_PENDING_HUMAN_ADJUDICATION",

    "human_decision":
        "PENDING",

    "review_focus":
        "ATTRIBUTION_BOUNDARY",

    "primary_answer_evidence":
        "Dr. Komeil Roudi",

    "contrast_context":
        "مجری",
}


# ============================================================
# 10) Continuation state
# ============================================================

cell63_continuation_core = {
    "next_pilot_id":
        CELL61_PILOT_ID,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell60_remaining_pilot_ids
        ),

    "remaining_count":
        11,

    "active_pilot_status":
        "CITATION_HUMAN_REVIEW_PENDING",

    "active_target_evidence_id":
        CELL61_TARGET_EVIDENCE_ID,

    "active_citation_mapping_id":
        cell63_citation_mapping_id,

    "last_completed_pilot_id":
        cell60_continuation_state.get(
            "last_completed_pilot_id"
        ),

    "last_completion_type":
        cell60_continuation_state.get(
            "last_completion_type"
        ),
}


cell63_continuation_state_sha256 = (
    cell10_stable_sha(
        cell63_continuation_core
    )
)


cell63_continuation_state = {
    **copy.deepcopy(
        cell63_continuation_core
    ),

    "continuation_state_sha256":
        cell63_continuation_state_sha256,
}


# ============================================================
# 11) Persist Registry
# ============================================================

cell63_persistent_registry_payload = copy.deepcopy(
    cell61_persisted_registry
)


cell63_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell63_active_workspace
)


cell63_persistent_registry_payload[
    "remaining_pilot_count"
] = 11


cell63_persistent_registry_payload[
    "continuation_state"
] = copy.deepcopy(
    cell63_continuation_state
)


cell63_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell63_persistent_registry_payload
    )
)


# ============================================================
# 12) Read-back validation
# ============================================================

cell63_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell63_persisted_active_case = (
    cell63_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell63_persistence_valid = all([
    cell63_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL61_PILOT_ID,

    cell63_persisted_active_case.get(
        "workflow_state"
    )
    ==
    "CITATION_HUMAN_REVIEW_PENDING",

    cell63_persisted_active_case.get(
        "question_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell63_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell63_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "citation_mapping_id"
    )
    ==
    cell63_citation_mapping_id,

    cell63_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "PENDING",

    cell63_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    11,

    cell63_persisted_registry.get(
        "continuation_state",
        {},
    ).get(
        "continuation_state_sha256"
    )
    ==
    cell63_continuation_state_sha256,
])


# ============================================================
# 13) Safety
# ============================================================

cell63_queue_still_11 = all([
    len(
        cell60_remaining_pilot_ids
    )
    ==
    11,

    cell60_remaining_pilot_ids[
        0
    ]
    ==
    CELL61_PILOT_ID,
])


cell63_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell63_full_572_blocked = (
    cell61_full_572_blocked
    is True
)


# ============================================================
# 14) Final Gate
# ============================================================

cell63_gate = all([
    len(
        cell63_failures
    )
    ==
    0,

    cell63_question_unchanged,

    cell63_answer_unchanged,

    cell63_attribution_structure_valid,

    cell63_answer_support_boundary_valid,

    cell63_no_source_or_locator_fabrication,

    cell63_session_valid,

    cell63_persistence_valid,

    cell63_queue_still_11,

    cell63_base_seed_unchanged,

    cell63_full_572_blocked,
])


# ============================================================
# 15) Diagnostics
# ============================================================

print(
    "=== PILOT #13 CITATION ATTRIBUTION REVISION ==="
)


print(
    "\n=== QUESTION / ANSWER ==="
)

print(
    "Question unchanged:",
    cell63_question_unchanged,
)

print(
    CELL63_ACCEPTED_QUESTION,
)

print(
    "\nAnswer unchanged:",
    cell63_answer_unchanged,
)

print(
    CELL63_ACCEPTED_ANSWER,
)


print(
    "\n=== ATTRIBUTION BOUNDARY ==="
)

print(
    "Multi-speaker structure preserved:",
    cell63_attribution_structure_valid,
)

print(
    "Primary answer Evidence:",
    cell63_primary_support_entries[
        0
    ].get(
        "document_id"
    )
    if cell63_primary_support_entries
    else None,
)

print(
    "Primary authority:",
    "Dr. Komeil Roudi",
)

print(
    "Contrast Context:",
    cell63_context_only_entries[
        0
    ].get(
        "document_id"
    )
    if cell63_context_only_entries
    else None,
)

print(
    "Contrast speaker:",
    "مجری",
)

print(
    "Host used as Answer Evidence:",
    False,
)


print(
    "\n=== CITATION REVISION ==="
)

print(
    "Old Citation Mapping:",
    cell61_citation_mapping_id,
)

print(
    "New Citation Mapping:",
    cell63_citation_mapping_id,
)

print(
    "Primary Answer Evidence count:",
    len(
        cell63_primary_support_entries
    ),
)

print(
    "Context-only count:",
    len(
        cell63_context_only_entries
    ),
)

print(
    "No source / locator fabrication:",
    cell63_no_source_or_locator_fabrication,
)


for index, entry in enumerate(
    cell63_revised_citation_entries
):

    print(
        "\nEntry",
        index,
    )

    print(
        "Document:",
        entry.get(
            "document_id"
        ),
    )

    print(
        "Source Role:",
        entry.get(
            "source_role"
        ),
    )

    print(
        "Answer Support Role:",
        entry.get(
            "answer_support_role"
        ),
    )

    print(
        "Answer Support:",
        entry.get(
            "answer_support"
        ),
    )

    print(
        "Authority Role:",
        entry.get(
            "authority_role"
        ),
    )


print(
    "\n=== WORKFLOW ==="
)

print(
    "Current State:",
    cell63_current_session.get(
        "current_state"
    ),
)

print(
    "Question:",
    "ACCEPT",
)

print(
    "Expected Answer:",
    "ACCEPT",
)

print(
    "Citation:",
    "PENDING",
)


print(
    "\n=== QUEUE / SAFETY ==="
)

print(
    "Queue still 11:",
    cell63_queue_still_11,
)

print(
    "Persistence valid:",
    cell63_persistence_valid,
)

print(
    "Base Seed unchanged:",
    cell63_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell63_full_572_blocked,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell63_failures
)


print(
    "\nCODE CELL 63 GATE:",
    (
        "PASS"
        if cell63_gate
        else "FAIL"
    ),
)


assert cell63_gate, (
    "Pilot #13 Citation attribution revision failed. "
    "Repair this SAME Code Cell 63 before Citation ACCEPT."
)

=== PILOT #13 CITATION ATTRIBUTION REVISION ===

=== QUESTION / ANSWER ===
Question unchanged: True
چه زمانی یک پس‌انداز را می‌توان سرمایه‌گذاری دانست؟

Answer unchanged: True
وقتی ارزش دارایی فقط حفظ شود، هنوز پس‌انداز است. اما اگر ارزش آن به‌طور واقعی و معنادار بیشتر شود و ارزش افزوده ایجاد کند، می‌توان آن را سرمایه‌گذاری دانست.

=== ATTRIBUTION BOUNDARY ===
Multi-speaker structure preserved: True
Primary answer Evidence: program:uni-tehran:episode-010:utterance:00003
Primary authority: Dr. Komeil Roudi
Contrast Context: program:uni-tehran:episode-010:utterance:00004
Contrast speaker: مجری
Host used as Answer Evidence: False

=== CITATION REVISION ===
Old Citation Mapping: citation-mapping:26d9c8bb2fe13bdf0641d938cc5644db
New Citation Mapping: citation-mapping:f10bbf2a07ccc8956a14b7b47108b70b
Primary Answer Evidence count: 1
Context-only count: 1
No source / locator fabrication: True

Entry 0
Document: program:uni-tehran:episode-010:utterance:00003
Source Role: ATTRIBUTION_CONTRAST_L

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 64 — Citation ACCEPT + Pilot #13 Completion + Queue Advance

Citation اصلاح‌شده این Pilot توسط Human Reviewer پذیرفته شد.

در این مرحله:

- Citation نهایی با تصمیم `ACCEPT` ثبت می‌شود؛
- مرز Attribution میان دکتر رودی و مجری حفظ می‌شود؛
- فقط صحبت دکتر رودی Evidence اصلی جواب محسوب می‌شود؛
- صحبت مجری فقط Context باقی می‌ماند؛
- Pilot #13 کامل می‌شود؛
- Case به Human Review Registry و Author Review اضافه می‌شود؛
- Queue دقیقاً از 11 به 10 جلو می‌رود؛
- Pilot بعدی برای Human Review آماده می‌شود؛
- Full 572 Generation همچنان Blocked باقی می‌ماند.

</div>

In [64]:
# Code Cell 64

cell64_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell63_gate") is not True:
    cell64_failures.append(
        "CELL63_NOT_PASS"
    )


if (
    globals().get(
        "cell63_current_session",
        {},
    ).get(
        "current_state"
    )
    !=
    "CITATION_HUMAN_REVIEW_PENDING"
):
    cell64_failures.append(
        "CITATION_NOT_AT_HUMAN_REVIEW_GATE"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell64_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Human Citation ACCEPT
# ============================================================

CELL64_CITATION_HUMAN_NOTE = (
    "Human Reviewer accepted the revised Citation Mapping. "
    "Dr. Komeil Roudi remains the primary answer authority, "
    "while the host utterance remains contrast context only."
)


cell64_complete_pending_session = (
    cell13_apply_event(
        cell63_current_session,
        "CITATION_ACCEPTED",
        human_note=
            CELL64_CITATION_HUMAN_NOTE,
    )
)


cell64_human_decisions = (
    cell64_complete_pending_session.get(
        "human_decisions",
        [],
    )
)


cell64_citation_accept_valid = all([
    cell64_complete_pending_session.get(
        "current_state"
    )
    ==
    "PILOT_COMPLETE_PENDING_QUEUE_ADVANCE",

    cell64_complete_pending_session.get(
        "queue_advance_performed"
    )
    is False,

    bool(
        cell64_human_decisions
    ),

    (
        cell64_human_decisions[
            -1
        ].get(
            "event"
        )
        ==
        "CITATION_ACCEPTED"
        if cell64_human_decisions
        else False
    ),
])


# ============================================================
# 3) Final revised Citation record
# ============================================================

cell64_final_citation_mapping_record = copy.deepcopy(
    cell63_citation_mapping_record
)


cell64_final_citation_mapping_record[
    "human_adjudication_status"
] = "ACCEPT"


cell64_final_citation_mapping_record[
    "human_review"
] = {
    "human_decision":
        "ACCEPT",

    "human_note":
        CELL64_CITATION_HUMAN_NOTE,

    "primary_answer_authority":
        "Dr. Komeil Roudi",

    "host_used_as_answer_evidence":
        False,

    "human_context_used_as_evidence":
        False,

    "external_knowledge_used":
        False,
}


# ============================================================
# 4) Complete Human Review case
# ============================================================

cell64_completed_case = copy.deepcopy(
    cell63_active_workspace
)


cell64_completed_case[
    "workflow_state"
] = "PILOT_COMPLETE"


cell64_completed_case[
    "status"
] = "COMPLETED_HUMAN_REVIEW"


cell64_completed_case[
    "final_status"
] = "PILOT_COMPLETE"


cell64_completed_case[
    "citation_review"
][
    "human_decision"
] = "ACCEPT"


cell64_completed_case[
    "citation_review"
][
    "human_note"
] = CELL64_CITATION_HUMAN_NOTE


cell64_completed_case[
    "citation_review"
][
    "citation_mapping_record"
] = copy.deepcopy(
    cell64_final_citation_mapping_record
)


cell64_completed_case[
    "completion_summary"
] = {
    "retarget_used":
        False,

    "target_evidence_id":
        CELL61_TARGET_EVIDENCE_ID,

    "question_decision":
        "ACCEPT",

    "expected_answer_decision":
        "ACCEPT",

    "citation_decision":
        "ACCEPT",

    "multi_speaker_attribution":
        True,

    "primary_answer_authority":
        "Dr. Komeil Roudi",

    "contrast_context_speaker":
        "مجری",

    "host_used_as_answer_evidence":
        False,

    "author_final_review":
        "NOT_STARTED",
}


cell64_completed_case[
    "author_final_review"
] = {
    "status":
        "NOT_STARTED",

    "decision":
        None,

    "notes":
        None,
}


# ============================================================
# 5) Queue advance: 11 -> 10
# ============================================================

cell64_previous_remaining_pilot_ids = copy.deepcopy(
    cell60_remaining_pilot_ids
)


cell64_remaining_pilot_ids = copy.deepcopy(
    cell64_previous_remaining_pilot_ids[
        1:
    ]
)


CELL64_COMPLETED_PILOT_ID = (
    cell64_previous_remaining_pilot_ids[
        0
    ]
)


cell64_queue_advance_valid = all([
    len(
        cell64_previous_remaining_pilot_ids
    )
    ==
    11,

    CELL64_COMPLETED_PILOT_ID
    ==
    CELL61_PILOT_ID,

    len(
        cell64_remaining_pilot_ids
    )
    ==
    10,

    CELL64_COMPLETED_PILOT_ID
    not in
    cell64_remaining_pilot_ids,
])


CELL64_NEXT_PILOT_ID = (
    cell64_remaining_pilot_ids[
        0
    ]
)


# ============================================================
# 6) Find next Pilot packet
# ============================================================

cell64_raw_next_packet_candidates = (
    cell58_find_pilot_packet_candidates(
        cell13_seed,
        CELL64_NEXT_PILOT_ID,
    )
)


cell64_packet_candidates_by_sha = {}


for candidate in cell64_raw_next_packet_candidates:

    packet_sha = cell10_stable_sha(
        candidate[
            "packet"
        ]
    )


    previous = (
        cell64_packet_candidates_by_sha.get(
            packet_sha
        )
    )


    if (
        previous is None
        or
        candidate[
            "score"
        ]
        >
        previous[
            "score"
        ]
    ):

        cell64_packet_candidates_by_sha[
            packet_sha
        ] = candidate


cell64_ranked_next_packet_candidates = sorted(
    cell64_packet_candidates_by_sha.values(),
    key=lambda item: (
        -item[
            "score"
        ],
        item[
            "path"
        ],
    ),
)


cell64_next_pilot_packet = (
    copy.deepcopy(
        cell64_ranked_next_packet_candidates[
            0
        ][
            "packet"
        ]
    )
    if cell64_ranked_next_packet_candidates
    else None
)


if not isinstance(
    cell64_next_pilot_packet,
    dict,
):

    cell64_failures.append(
        "NEXT_PILOT_PACKET_NOT_FOUND"
    )


# ============================================================
# 7) Extract next Pilot identities
# ============================================================

cell64_next_pilot_ids_found = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell64_next_pilot_packet,
            "question-pilot:",
        )
    )
)


cell64_next_golden_case_ids = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell64_next_pilot_packet,
            "golden-case:",
        )
    )
)


cell64_next_task_draft_ids = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell64_next_pilot_packet,
            "task-draft:",
        )
    )
)


cell64_next_target_evidence_ids = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell64_next_pilot_packet,
            "target-evidence:",
        )
    )
)


cell64_next_identity_valid = all([
    cell64_next_pilot_ids_found
    ==
    [
        CELL64_NEXT_PILOT_ID
    ],

    len(
        cell64_next_golden_case_ids
    )
    ==
    1,

    len(
        cell64_next_task_draft_ids
    )
    ==
    1,

    len(
        cell64_next_target_evidence_ids
    )
    ==
    1,
])


CELL64_NEXT_GOLDEN_CASE_ID = (
    cell64_next_golden_case_ids[
        0
    ]
    if
    len(
        cell64_next_golden_case_ids
    )
    ==
    1
    else
    None
)


CELL64_NEXT_TASK_DRAFT_ID = (
    cell64_next_task_draft_ids[
        0
    ]
    if
    len(
        cell64_next_task_draft_ids
    )
    ==
    1
    else
    None
)


CELL64_NEXT_TARGET_EVIDENCE_ID = (
    cell64_next_target_evidence_ids[
        0
    ]
    if
    len(
        cell64_next_target_evidence_ids
    )
    ==
    1
    else
    None
)


# ============================================================
# 8) Extract next Target Evidence
# ============================================================

cell64_target_part_candidates = (
    cell58_find_key_values(
        cell64_next_pilot_packet,
        "target_text_parts",
    )
)


cell64_target_parts_by_sha = {}


for parts in cell64_target_part_candidates:

    if not isinstance(
        parts,
        list,
    ):
        continue


    if len(
        parts
    ) == 0:
        continue


    parts_sha = cell10_stable_sha(
        parts
    )


    cell64_target_parts_by_sha[
        parts_sha
    ] = parts


cell64_unique_target_part_sets = list(
    cell64_target_parts_by_sha.values()
)


cell64_next_target_text_parts = (
    copy.deepcopy(
        cell64_unique_target_part_sets[
            0
        ]
    )
    if
    len(
        cell64_unique_target_part_sets
    )
    ==
    1
    else
    None
)


if not isinstance(
    cell64_next_target_text_parts,
    list,
):

    cell64_failures.append(
        "NEXT_TARGET_TEXT_PARTS_NOT_UNIQUE"
    )


# ============================================================
# 9) Build readable next Evidence
# ============================================================

cell64_next_evidence_texts = []


for part in (
    cell64_next_target_text_parts
    or
    []
):

    if isinstance(
        part,
        str,
    ):

        text = part.strip()


    elif isinstance(
        part,
        dict,
    ):

        text = (
            part.get(
                "text"
            )
            or
            part.get(
                "target_text"
            )
            or
            part.get(
                "content"
            )
            or
            ""
        )


        if isinstance(
            text,
            str,
        ):

            text = text.strip()


        else:

            text = ""


    else:

        text = ""


    if text:

        cell64_next_evidence_texts.append(
            text
        )


cell64_next_combined_target_text = (
    "\n\n".join(
        cell64_next_evidence_texts
    )
)


cell64_next_evidence_available = bool(
    cell64_next_combined_target_text.strip()
)


# ============================================================
# 10) Next Pilot engine session
# ============================================================

cell64_next_pilot_session = (
    cell13_build_session(
        cell64_next_pilot_packet
    )
)


cell64_next_session_valid = (
    cell64_next_pilot_session.get(
        "current_state"
    )
    ==
    "QUESTION_DRAFT_PENDING"
)


# ============================================================
# 11) Update completed Human Review cases
# ============================================================

cell64_existing_completed_cases = copy.deepcopy(
    cell63_persisted_registry.get(
        "completed_human_review_cases",
        [],
    )
)


if not isinstance(
    cell64_existing_completed_cases,
    list,
):

    cell64_failures.append(
        "COMPLETED_HUMAN_REVIEW_CASES_NOT_LIST"
    )

    cell64_existing_completed_cases = []


cell64_existing_completed_cases = [
    case
    for case
    in cell64_existing_completed_cases
    if not (
        isinstance(
            case,
            dict,
        )
        and
        case.get(
            "pilot_case_id"
        )
        ==
        CELL64_COMPLETED_PILOT_ID
    )
]


cell64_completed_human_review_cases = (
    cell64_existing_completed_cases
    +
    [
        copy.deepcopy(
            cell64_completed_case
        )
    ]
)


cell64_completed_case_count_valid = (
    len(
        cell64_completed_human_review_cases
    )
    ==
    12
)


# ============================================================
# 12) Rebuild Author Review queue
# ============================================================

cell64_author_review = copy.deepcopy(
    cell63_persisted_registry.get(
        "author_review",
        {},
    )
)


if not isinstance(
    cell64_author_review,
    dict,
):

    cell64_author_review = {}


cell64_author_ready_cases = copy.deepcopy(
    cell64_completed_human_review_cases
)


cell64_author_review[
    "cases_ready_for_review"
] = copy.deepcopy(
    cell64_author_ready_cases
)


cell64_author_review[
    "case_count"
] = len(
    cell64_author_ready_cases
)


cell64_author_review[
    "final_review_status"
] = "NOT_STARTED"


cell64_author_ready_count_valid = (
    len(
        cell64_author_ready_cases
    )
    ==
    12
)


cell64_author_ready_identity_valid = (
    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell64_author_ready_cases
        if isinstance(
            case,
            dict,
        )
    }
    ==
    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell64_completed_human_review_cases
        if isinstance(
            case,
            dict,
        )
    }
)


# ============================================================
# 13) Build next active workspace
# ============================================================

cell64_next_active_workspace = {
    "pilot_case_id":
        CELL64_NEXT_PILOT_ID,

    "golden_case_id":
        CELL64_NEXT_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL64_NEXT_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL64_NEXT_TARGET_EVIDENCE_ID,

    "status":
        "ACTIVE_HUMAN_REVIEW",

    "workflow_state":
        "QUESTION_DRAFT_PENDING",

    "source_evidence":
        copy.deepcopy(
            cell64_next_target_text_parts
        ),

    "combined_target_text":
        cell64_next_combined_target_text,

    "question_review":
        None,

    "expected_answer_review":
        None,

    "citation_review":
        None,

    "author_final_review": {
        "status":
            "NOT_STARTED",

        "decision":
            None,

        "notes":
            None,
    },
}


# ============================================================
# 14) New authoritative continuation state
# ============================================================

cell64_continuation_state_core = {
    "next_pilot_id":
        CELL64_NEXT_PILOT_ID,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell64_remaining_pilot_ids
        ),

    "remaining_count":
        10,

    "active_pilot_status":
        "QUESTION_DRAFT_PENDING",

    "active_target_evidence_id":
        CELL64_NEXT_TARGET_EVIDENCE_ID,

    "last_completed_pilot_id":
        CELL64_COMPLETED_PILOT_ID,

    "last_completion_type":
        "MULTI_SPEAKER_ATTRIBUTION_PILOT_COMPLETE",
}


cell64_continuation_state_sha256 = (
    cell10_stable_sha(
        cell64_continuation_state_core
    )
)


cell64_continuation_state = {
    **copy.deepcopy(
        cell64_continuation_state_core
    ),

    "continuation_state_sha256":
        cell64_continuation_state_sha256,
}


# ============================================================
# 15) Persist Registry
# ============================================================

cell64_persistent_registry_payload = copy.deepcopy(
    cell63_persisted_registry
)


cell64_persistent_registry_payload[
    "completed_human_review_cases"
] = copy.deepcopy(
    cell64_completed_human_review_cases
)


cell64_persistent_registry_payload[
    "author_review"
] = copy.deepcopy(
    cell64_author_review
)


cell64_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell64_next_active_workspace
)


cell64_persistent_registry_payload[
    "remaining_pilot_count"
] = 10


cell64_persistent_registry_payload[
    "continuation_state"
] = copy.deepcopy(
    cell64_continuation_state
)


cell64_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell64_persistent_registry_payload
    )
)


# ============================================================
# 16) Read-back validation
# ============================================================

cell64_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell64_persisted_active_case = (
    cell64_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell64_persisted_completed_cases = (
    cell64_persisted_registry.get(
        "completed_human_review_cases",
        [],
    )
)


cell64_persisted_author_ready_cases = (
    cell64_persisted_registry.get(
        "author_review",
        {},
    ).get(
        "cases_ready_for_review",
        [],
    )
)


cell64_persistence_valid = all([
    isinstance(
        cell64_persisted_completed_cases,
        list,
    ),

    len(
        cell64_persisted_completed_cases
    )
    ==
    12,

    isinstance(
        cell64_persisted_author_ready_cases,
        list,
    ),

    len(
        cell64_persisted_author_ready_cases
    )
    ==
    12,

    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell64_persisted_completed_cases
        if isinstance(
            case,
            dict,
        )
    }
    ==
    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell64_persisted_author_ready_cases
        if isinstance(
            case,
            dict,
        )
    },

    cell64_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    10,

    cell64_persisted_registry.get(
        "continuation_state",
        {},
    ).get(
        "continuation_state_sha256"
    )
    ==
    cell64_continuation_state_sha256,

    cell64_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL64_NEXT_PILOT_ID,

    cell64_persisted_active_case.get(
        "workflow_state"
    )
    ==
    "QUESTION_DRAFT_PENDING",
])


# ============================================================
# 17) Safety
# ============================================================

cell64_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell64_full_572_blocked = (
    cell63_full_572_blocked
    is True
)


cell64_no_commit_push = True


# ============================================================
# 18) Final Gate
# ============================================================

cell64_gate = all([
    len(
        cell64_failures
    )
    ==
    0,

    cell64_citation_accept_valid,

    cell64_queue_advance_valid,

    cell64_next_identity_valid,

    cell64_next_evidence_available,

    cell64_next_session_valid,

    cell64_completed_case_count_valid,

    cell64_author_ready_count_valid,

    cell64_author_ready_identity_valid,

    cell64_persistence_valid,

    cell64_base_seed_unchanged,

    cell64_full_572_blocked,

    cell64_no_commit_push,
])


# ============================================================
# 19) Diagnostics
# ============================================================

print(
    "=== PILOT #13 COMPLETE + QUEUE ADVANCE + NEXT PILOT ==="
)


print(
    "\n=== COMPLETED PILOT ==="
)

print(
    "Pilot:",
    CELL64_COMPLETED_PILOT_ID,
)

print(
    "Question:",
    "ACCEPT",
)

print(
    "Expected Answer:",
    "ACCEPT",
)

print(
    "Citation:",
    "ACCEPT",
)

print(
    "Final Engine State:",
    cell64_complete_pending_session.get(
        "current_state"
    ),
)


print(
    "\n=== ATTRIBUTION RESULT ==="
)

print(
    "Primary Answer Authority:",
    "Dr. Komeil Roudi",
)

print(
    "Host kept as Context:",
    True,
)

print(
    "Host used as Answer Evidence:",
    False,
)

print(
    "Final Citation Mapping:",
    cell63_citation_mapping_id,
)


print(
    "\n=== QUEUE ADVANCE ==="
)

print(
    "Before:",
    len(
        cell64_previous_remaining_pilot_ids
    ),
)

print(
    "After:",
    len(
        cell64_remaining_pilot_ids
    ),
)

print(
    "Queue advanced exactly once:",
    cell64_queue_advance_valid,
)


print(
    "\n=== HUMAN REVIEW REGISTRY ==="
)

print(
    "Completed Human Review cases:",
    len(
        cell64_completed_human_review_cases
    ),
)

print(
    "Cases ready for Author Review:",
    len(
        cell64_author_ready_cases
    ),
)

print(
    "Author ready identities match completed cases:",
    cell64_author_ready_identity_valid,
)

print(
    "Author Final Review:",
    "NOT_STARTED",
)

print(
    "Persistence valid:",
    cell64_persistence_valid,
)


print(
    "\n=== NEXT PILOT ==="
)

print(
    "Pilot:",
    CELL64_NEXT_PILOT_ID,
)

print(
    "Golden Case:",
    CELL64_NEXT_GOLDEN_CASE_ID,
)

print(
    "Task Draft:",
    CELL64_NEXT_TASK_DRAFT_ID,
)

print(
    "Target Evidence:",
    CELL64_NEXT_TARGET_EVIDENCE_ID,
)

print(
    "Workflow:",
    cell64_next_pilot_session.get(
        "current_state"
    ),
)


print(
    "\n=== NEXT SOURCE EVIDENCE ==="
)

print(
    cell64_next_combined_target_text
)


print(
    "\n=== CONTINUATION STATE ==="
)

print(
    "Remaining Pilots:",
    10,
)

print(
    "Continuation SHA256:",
    cell64_continuation_state_sha256,
)


print(
    "\n=== SAFETY ==="
)

print(
    "Base Seed unchanged:",
    cell64_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell64_full_572_blocked,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell64_failures
)


print(
    "\nCODE CELL 64 GATE:",
    (
        "PASS"
        if cell64_gate
        else "FAIL"
    ),
)


assert cell64_gate, (
    "Citation ACCEPT / Pilot #13 completion / Queue advance / "
    "Next Pilot preparation failed. Repair this SAME "
    "Code Cell 64 before continuing."
)

=== PILOT #13 COMPLETE + QUEUE ADVANCE + NEXT PILOT ===

=== COMPLETED PILOT ===
Pilot: question-pilot:950359cf41d2817c535d486f8cd49aab
Question: ACCEPT
Expected Answer: ACCEPT
Citation: ACCEPT
Final Engine State: PILOT_COMPLETE_PENDING_QUEUE_ADVANCE

=== ATTRIBUTION RESULT ===
Primary Answer Authority: Dr. Komeil Roudi
Host kept as Context: True
Host used as Answer Evidence: False
Final Citation Mapping: citation-mapping:f10bbf2a07ccc8956a14b7b47108b70b

=== QUEUE ADVANCE ===
Before: 11
After: 10
Queue advanced exactly once: True

=== HUMAN REVIEW REGISTRY ===
Completed Human Review cases: 12
Cases ready for Author Review: 12
Author ready identities match completed cases: True
Author Final Review: NOT_STARTED
Persistence valid: True

=== NEXT PILOT ===
Pilot: question-pilot:bae57351b7aa1fa7545adb709d1c61fc
Golden Case: golden-case:58a5dead0d22e45e28a5ee25bb36bfeb
Task Draft: task-draft:1ab3ed072bc8c508ea9abc5c8d79f472
Target Evidence: target-evidence:73bbf40cdd65405711a21f389d93d5f9
W

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 65 — Question/Answer ACCEPT + Citation Draft for Pilot #14

Human Reviewer سؤال و جواب این Pilot را پذیرفت.

سؤال پذیرفته‌شده:

**قبل از سرمایه‌گذاری چه چیزهایی باید بررسی شود و افراد غیرمتخصص بهتر است چگونه وارد بازار شوند؟**

جواب پذیرفته‌شده:

**قبل از سرمایه‌گذاری باید نوع ابزار، سطح ریسک و میزان زیانی را که می‌توان تحمل کرد شناخت. برای عموم مردم، ورود غیرمستقیم از طریق صندوق‌های سرمایه‌گذاری یا استفاده از خدمات حرفه‌ای می‌تواند مناسب‌تر از معامله مستقیم و بدون تخصص باشد.**

در این مرحله:

- Question با تصمیم `ACCEPT` ثبت می‌شود؛
- Expected Answer با تصمیم `ACCEPT` ثبت می‌شود؛
- Citation فقط از Target Evidence فعلی ساخته می‌شود؛
- Citation وارد Human Review می‌شود؛
- Queue همچنان 10 باقی می‌ماند؛
- Full 572 Generation همچنان Blocked است.

</div>

In [65]:
# Code Cell 65

cell65_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell64_gate") is not True:
    cell65_failures.append(
        "CELL64_NOT_PASS"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell65_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Accepted Human Review content
# ============================================================

CELL65_PILOT_ID = (
    CELL64_NEXT_PILOT_ID
)


CELL65_GOLDEN_CASE_ID = (
    CELL64_NEXT_GOLDEN_CASE_ID
)


CELL65_TASK_DRAFT_ID = (
    CELL64_NEXT_TASK_DRAFT_ID
)


CELL65_TARGET_EVIDENCE_ID = (
    CELL64_NEXT_TARGET_EVIDENCE_ID
)


CELL65_ACCEPTED_QUESTION = (
    "قبل از سرمایه‌گذاری چه چیزهایی باید بررسی شود و "
    "افراد غیرمتخصص بهتر است چگونه وارد بازار شوند؟"
)


CELL65_ACCEPTED_ANSWER = (
    "قبل از سرمایه‌گذاری باید نوع ابزار، سطح ریسک و میزان "
    "زیانی را که می‌توان تحمل کرد شناخت. برای عموم مردم، "
    "ورود غیرمستقیم از طریق صندوق‌های سرمایه‌گذاری یا "
    "استفاده از خدمات حرفه‌ای می‌تواند مناسب‌تر از معامله "
    "مستقیم و بدون تخصص باشد."
)


CELL65_QUESTION_HUMAN_NOTE = (
    "Human Reviewer accepted the proposed question."
)


CELL65_ANSWER_HUMAN_NOTE = (
    "Human Reviewer accepted the proposed answer."
)


# ============================================================
# 3) Current Target Evidence
# ============================================================

cell65_target_parts = copy.deepcopy(
    cell64_next_target_text_parts
)


cell65_combined_target_text = (
    cell64_next_combined_target_text
)


cell65_target_parts_valid = all([
    isinstance(
        cell65_target_parts,
        list,
    ),

    len(
        cell65_target_parts
    )
    >
    0,

    isinstance(
        cell65_combined_target_text,
        str,
    ),

    bool(
        cell65_combined_target_text.strip()
    ),
])


# ============================================================
# 4) Grounding validation
# ============================================================

def cell65_norm(
    text,
):

    if not isinstance(
        text,
        str,
    ):
        return ""


    value = (
        text
        .replace(
            "ي",
            "ی",
        )
        .replace(
            "ك",
            "ک",
        )
        .replace(
            "\u200c",
            " ",
        )
    )


    value = re.sub(
        r"\s+",
        " ",
        value,
    )


    return value.strip()


cell65_source_surface = cell65_norm(
    cell65_combined_target_text
)


cell65_grounding = {
    "instrument_type":
        (
            "نوع ابزار"
            in
            cell65_source_surface
        ),

    "risk_level":
        (
            "سطح ریسک"
            in
            cell65_source_surface
        ),

    "loss_tolerance":
        (
            "حد زیان قابل تحمل"
            in
            cell65_source_surface
        ),

    "general_public":
        (
            "عموم مردم"
            in
            cell65_source_surface
        ),

    "indirect_entry":
        (
            "ورود غیرمستقیم"
            in
            cell65_source_surface
        ),

    "investment_funds":
        (
            "صندوق های سرمایه گذاری"
            in
            cell65_source_surface
        ),

    "professional_services":
        (
            "خدمات حرفه ای"
            in
            cell65_source_surface
        ),

    "direct_trading_without_expertise":
        (
            "معامله مستقیم و بدون تخصص"
            in
            cell65_source_surface
        ),
}


cell65_grounding_pass = all(
    cell65_grounding.values()
)


# ============================================================
# 5) Start Engine session
# ============================================================

cell65_question_pending_session = (
    cell13_build_session(
        cell64_next_pilot_packet
    )
)


cell65_session_start_valid = (
    cell65_question_pending_session.get(
        "current_state"
    )
    ==
    "QUESTION_DRAFT_PENDING"
)


# ============================================================
# 6) Question Draft
# ============================================================

cell65_question_identity = {
    "identity_version":
        "question_draft_identity_v1",

    "pilot_case_id":
        CELL65_PILOT_ID,

    "target_evidence_id":
        CELL65_TARGET_EVIDENCE_ID,

    "question_text":
        CELL65_ACCEPTED_QUESTION,

    "revision_number":
        1,
}


cell65_question_draft_id = (
    "question-draft:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell65_question_identity
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


cell65_question_record = {
    "record_version":
        "question_draft_record_v1",

    "question_draft_id":
        cell65_question_draft_id,

    "pilot_case_id":
        CELL65_PILOT_ID,

    "golden_case_id":
        CELL65_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL65_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL65_TARGET_EVIDENCE_ID,

    "question_text":
        CELL65_ACCEPTED_QUESTION,

    "revision_number":
        1,

    "machine_validation_status":
        (
            "PASS_PENDING_HUMAN_ADJUDICATION"
            if cell65_grounding_pass
            else
            "FAIL_REQUIRES_REVISION"
        ),

    "human_adjudication_status":
        "PENDING",

    "external_knowledge_used":
        False,
}


cell65_question_review_session = (
    cell13_apply_event(
        cell65_question_pending_session,
        "QUESTION_DRAFT_REGISTERED",
        artifact=
            cell65_question_record,
    )
)


cell65_answer_pending_session = (
    cell13_apply_event(
        cell65_question_review_session,
        "QUESTION_ACCEPTED",
        human_note=
            CELL65_QUESTION_HUMAN_NOTE,
    )
)


cell65_question_accept_valid = all([
    cell65_question_review_session.get(
        "current_state"
    )
    ==
    "QUESTION_HUMAN_REVIEW_PENDING",

    cell65_answer_pending_session.get(
        "current_state"
    )
    ==
    "EXPECTED_ANSWER_DRAFT_PENDING",
])


# ============================================================
# 7) Expected Answer
# ============================================================

cell65_answer_identity = {
    "identity_version":
        "expected_answer_identity_v1",

    "pilot_case_id":
        CELL65_PILOT_ID,

    "question_draft_id":
        cell65_question_draft_id,

    "target_evidence_id":
        CELL65_TARGET_EVIDENCE_ID,

    "answer_text":
        CELL65_ACCEPTED_ANSWER,

    "revision_number":
        1,
}


cell65_expected_answer_draft_id = (
    "expected-answer-draft:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell65_answer_identity
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


cell65_expected_answer_record = {
    "record_version":
        "expected_answer_draft_record_v1",

    "expected_answer_draft_id":
        cell65_expected_answer_draft_id,

    "pilot_case_id":
        CELL65_PILOT_ID,

    "golden_case_id":
        CELL65_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL65_TASK_DRAFT_ID,

    "question_draft_id":
        cell65_question_draft_id,

    "target_evidence_id":
        CELL65_TARGET_EVIDENCE_ID,

    "expected_answer_text":
        CELL65_ACCEPTED_ANSWER,

    "revision_number":
        1,

    "machine_validation_status":
        (
            "PASS_PENDING_HUMAN_ADJUDICATION"
            if cell65_grounding_pass
            else
            "FAIL_REQUIRES_REVISION"
        ),

    "human_adjudication_status":
        "PENDING",

    "external_knowledge_used":
        False,
}


cell65_answer_review_session = (
    cell13_apply_event(
        cell65_answer_pending_session,
        "EXPECTED_ANSWER_REGISTERED",
        artifact=
            cell65_expected_answer_record,
    )
)


cell65_citation_pending_session = (
    cell13_apply_event(
        cell65_answer_review_session,
        "EXPECTED_ANSWER_ACCEPTED",
        human_note=
            CELL65_ANSWER_HUMAN_NOTE,
    )
)


cell65_answer_accept_valid = all([
    cell65_answer_review_session.get(
        "current_state"
    )
    ==
    "EXPECTED_ANSWER_HUMAN_REVIEW_PENDING",

    cell65_citation_pending_session.get(
        "current_state"
    )
    ==
    "CITATION_MAPPING_PENDING",
])


# ============================================================
# 8) Build Citation entries
# ============================================================

cell65_citation_entries = []


for index, part in enumerate(
    cell65_target_parts
):

    if isinstance(
        part,
        dict,
    ):

        citation_entry = copy.deepcopy(
            part
        )


        citation_entry[
            "citation_index"
        ] = index


        citation_entry[
            "target_evidence_id"
        ] = CELL65_TARGET_EVIDENCE_ID


        citation_entry[
            "locator_origin"
        ] = "COPIED_FROM_TARGET_EVIDENCE"


        citation_entry[
            "external_locator_lookup_used"
        ] = False


    else:

        citation_entry = {
            "citation_index":
                index,

            "target_evidence_id":
                CELL65_TARGET_EVIDENCE_ID,

            "text":
                part,

            "locator_origin":
                "COPIED_FROM_TARGET_EVIDENCE",

            "external_locator_lookup_used":
                False,
        }


    cell65_citation_entries.append(
        citation_entry
    )


# ============================================================
# 9) No-fabrication validation
# ============================================================

def cell65_strip_added_citation_fields(
    entry,
):

    output = copy.deepcopy(
        entry
    )


    output.pop(
        "citation_index",
        None,
    )


    output.pop(
        "target_evidence_id",
        None,
    )


    output.pop(
        "locator_origin",
        None,
    )


    output.pop(
        "external_locator_lookup_used",
        None,
    )


    return output


cell65_citation_back_projection = [
    cell65_strip_added_citation_fields(
        entry
    )
    for entry
    in cell65_citation_entries
]


cell65_no_locator_fabrication = (
    cell65_citation_back_projection
    ==
    cell65_target_parts
)


# ============================================================
# 10) Citation Mapping ID
# ============================================================

cell65_citation_identity = {
    "identity_version":
        "citation_mapping_identity_v1",

    "pilot_case_id":
        CELL65_PILOT_ID,

    "question_draft_id":
        cell65_question_draft_id,

    "expected_answer_draft_id":
        cell65_expected_answer_draft_id,

    "target_evidence_id":
        CELL65_TARGET_EVIDENCE_ID,

    "citation_entries":
        copy.deepcopy(
            cell65_citation_entries
        ),
}


cell65_citation_mapping_id = (
    "citation-mapping:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell65_citation_identity
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


cell65_citation_machine_pass = all([
    len(
        cell65_citation_entries
    )
    ==
    len(
        cell65_target_parts
    ),

    len(
        cell65_citation_entries
    )
    >
    0,

    cell65_no_locator_fabrication,

    all(
        entry.get(
            "target_evidence_id"
        )
        ==
        CELL65_TARGET_EVIDENCE_ID
        for entry
        in cell65_citation_entries
    ),

    all(
        entry.get(
            "external_locator_lookup_used"
        )
        is False
        for entry
        in cell65_citation_entries
    ),
])


cell65_citation_mapping_record = {
    "record_version":
        "citation_mapping_record_v1",

    "citation_mapping_id":
        cell65_citation_mapping_id,

    "pilot_case_id":
        CELL65_PILOT_ID,

    "golden_case_id":
        CELL65_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL65_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL65_TARGET_EVIDENCE_ID,

    "question_draft_id":
        cell65_question_draft_id,

    "expected_answer_draft_id":
        cell65_expected_answer_draft_id,

    "citation_entries":
        copy.deepcopy(
            cell65_citation_entries
        ),

    "citation_scope":
        "CURRENT_TARGET_EVIDENCE_ONLY",

    "retarget_used":
        False,

    "external_locator_lookup_used":
        False,

    "machine_validation_status":
        (
            "PASS_PENDING_HUMAN_ADJUDICATION"
            if cell65_citation_machine_pass
            else
            "FAIL_REQUIRES_REVISION"
        ),

    "human_adjudication_status":
        "PENDING",
}


cell65_current_session = (
    cell13_apply_event(
        cell65_citation_pending_session,
        "CITATION_MAPPING_REGISTERED",
        artifact=
            cell65_citation_mapping_record,
    )
)


cell65_citation_registered_valid = (
    cell65_current_session.get(
        "current_state"
    )
    ==
    "CITATION_HUMAN_REVIEW_PENDING"
)


# ============================================================
# 11) Update active Human Review workspace
# ============================================================

cell65_active_workspace = copy.deepcopy(
    cell64_persisted_active_case
)


cell65_active_workspace[
    "workflow_state"
] = "CITATION_HUMAN_REVIEW_PENDING"


cell65_active_workspace[
    "status"
] = "ACTIVE_HUMAN_REVIEW"


cell65_active_workspace[
    "question_review"
] = {
    "question_draft_id":
        cell65_question_draft_id,

    "question_text":
        CELL65_ACCEPTED_QUESTION,

    "machine_validation_status":
        cell65_question_record[
            "machine_validation_status"
        ],

    "human_decision":
        "ACCEPT",

    "human_note":
        CELL65_QUESTION_HUMAN_NOTE,
}


cell65_active_workspace[
    "expected_answer_review"
] = {
    "expected_answer_draft_id":
        cell65_expected_answer_draft_id,

    "expected_answer_text":
        CELL65_ACCEPTED_ANSWER,

    "machine_validation_status":
        cell65_expected_answer_record[
            "machine_validation_status"
        ],

    "human_decision":
        "ACCEPT",

    "human_note":
        CELL65_ANSWER_HUMAN_NOTE,
}


cell65_active_workspace[
    "citation_review"
] = {
    "citation_mapping_id":
        cell65_citation_mapping_id,

    "citation_entries":
        copy.deepcopy(
            cell65_citation_entries
        ),

    "machine_validation_status":
        cell65_citation_mapping_record[
            "machine_validation_status"
        ],

    "human_decision":
        "PENDING",
}


# ============================================================
# 12) Continuation state
# ============================================================

cell65_continuation_core = {
    "next_pilot_id":
        CELL65_PILOT_ID,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell64_remaining_pilot_ids
        ),

    "remaining_count":
        10,

    "active_pilot_status":
        "CITATION_HUMAN_REVIEW_PENDING",

    "active_target_evidence_id":
        CELL65_TARGET_EVIDENCE_ID,

    "last_completed_pilot_id":
        cell64_continuation_state.get(
            "last_completed_pilot_id"
        ),

    "last_completion_type":
        cell64_continuation_state.get(
            "last_completion_type"
        ),
}


cell65_continuation_state_sha256 = (
    cell10_stable_sha(
        cell65_continuation_core
    )
)


cell65_continuation_state = {
    **copy.deepcopy(
        cell65_continuation_core
    ),

    "continuation_state_sha256":
        cell65_continuation_state_sha256,
}


# ============================================================
# 13) Persist Registry
# ============================================================

cell65_persistent_registry_payload = copy.deepcopy(
    cell64_persisted_registry
)


cell65_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell65_active_workspace
)


cell65_persistent_registry_payload[
    "remaining_pilot_count"
] = 10


cell65_persistent_registry_payload[
    "continuation_state"
] = copy.deepcopy(
    cell65_continuation_state
)


cell65_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell65_persistent_registry_payload
    )
)


# ============================================================
# 14) Read-back validation
# ============================================================

cell65_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell65_persisted_active_case = (
    cell65_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell65_persistence_valid = all([
    cell65_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL65_PILOT_ID,

    cell65_persisted_active_case.get(
        "workflow_state"
    )
    ==
    "CITATION_HUMAN_REVIEW_PENDING",

    cell65_persisted_active_case.get(
        "question_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell65_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell65_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "PENDING",

    cell65_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    10,

    cell65_persisted_registry.get(
        "continuation_state",
        {},
    ).get(
        "continuation_state_sha256"
    )
    ==
    cell65_continuation_state_sha256,
])


# ============================================================
# 15) Safety
# ============================================================

cell65_queue_still_10 = all([
    len(
        cell64_remaining_pilot_ids
    )
    ==
    10,

    cell64_remaining_pilot_ids[
        0
    ]
    ==
    CELL65_PILOT_ID,
])


cell65_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell65_full_572_blocked = (
    cell64_full_572_blocked
    is True
)


# ============================================================
# 16) Final Gate
# ============================================================

cell65_gate = all([
    len(
        cell65_failures
    )
    ==
    0,

    cell65_target_parts_valid,

    cell65_grounding_pass,

    cell65_session_start_valid,

    cell65_question_accept_valid,

    cell65_answer_accept_valid,

    cell65_no_locator_fabrication,

    cell65_citation_machine_pass,

    cell65_citation_registered_valid,

    cell65_persistence_valid,

    cell65_queue_still_10,

    cell65_base_seed_unchanged,

    cell65_full_572_blocked,
])


# ============================================================
# 17) Diagnostics
# ============================================================

print(
    "=== PILOT #14 QUESTION/ANSWER ACCEPT + CITATION DRAFT ==="
)


print(
    "\n=== PILOT ==="
)

print(
    "Pilot:",
    CELL65_PILOT_ID,
)

print(
    "Golden Case:",
    CELL65_GOLDEN_CASE_ID,
)

print(
    "Task Draft:",
    CELL65_TASK_DRAFT_ID,
)

print(
    "Target Evidence:",
    CELL65_TARGET_EVIDENCE_ID,
)


print(
    "\n=== ACCEPTED QUESTION ==="
)

print(
    CELL65_ACCEPTED_QUESTION
)

print(
    "Question Draft ID:",
    cell65_question_draft_id,
)

print(
    "Human Decision:",
    "ACCEPT",
)


print(
    "\n=== ACCEPTED ANSWER ==="
)

print(
    CELL65_ACCEPTED_ANSWER
)

print(
    "Expected Answer Draft ID:",
    cell65_expected_answer_draft_id,
)

print(
    "Human Decision:",
    "ACCEPT",
)


print(
    "\n=== GROUNDING ==="
)

for name, status in (
    cell65_grounding.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "Grounding PASS:",
    cell65_grounding_pass,
)


print(
    "\n=== CITATION DRAFT ==="
)

print(
    "Citation Mapping ID:",
    cell65_citation_mapping_id,
)

print(
    "Citation entry count:",
    len(
        cell65_citation_entries
    ),
)

print(
    "No locator fabrication:",
    cell65_no_locator_fabrication,
)

print(
    "Citation Machine PASS:",
    cell65_citation_machine_pass,
)


for index, entry in enumerate(
    cell65_citation_entries
):

    print(
        "\n--- CITATION ENTRY",
        index,
        "---"
    )

    print(
        cell10_canonical_json(
            entry
        )
    )


print(
    "\n=== WORKFLOW ==="
)

print(
    "Current State:",
    cell65_current_session.get(
        "current_state"
    ),
)

print(
    "Question:",
    "ACCEPT",
)

print(
    "Expected Answer:",
    "ACCEPT",
)

print(
    "Citation:",
    "PENDING",
)


print(
    "\n=== QUEUE / SAFETY ==="
)

print(
    "Queue still 10:",
    cell65_queue_still_10,
)

print(
    "Persistence valid:",
    cell65_persistence_valid,
)

print(
    "Base Seed unchanged:",
    cell65_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell65_full_572_blocked,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell65_failures
)


print(
    "\nCODE CELL 65 GATE:",
    (
        "PASS"
        if cell65_gate
        else "FAIL"
    ),
)


assert cell65_gate, (
    "Pilot #14 Question/Answer ACCEPT / Citation Draft failed. "
    "Repair this SAME Code Cell 65 before Citation Human Review."
)

=== PILOT #14 QUESTION/ANSWER ACCEPT + CITATION DRAFT ===

=== PILOT ===
Pilot: question-pilot:bae57351b7aa1fa7545adb709d1c61fc
Golden Case: golden-case:58a5dead0d22e45e28a5ee25bb36bfeb
Task Draft: task-draft:1ab3ed072bc8c508ea9abc5c8d79f472
Target Evidence: target-evidence:73bbf40cdd65405711a21f389d93d5f9

=== ACCEPTED QUESTION ===
قبل از سرمایه‌گذاری چه چیزهایی باید بررسی شود و افراد غیرمتخصص بهتر است چگونه وارد بازار شوند؟
Question Draft ID: question-draft:592dc738711c4a4c13d8c778c7d3ec3c
Human Decision: ACCEPT

=== ACCEPTED ANSWER ===
قبل از سرمایه‌گذاری باید نوع ابزار، سطح ریسک و میزان زیانی را که می‌توان تحمل کرد شناخت. برای عموم مردم، ورود غیرمستقیم از طریق صندوق‌های سرمایه‌گذاری یا استفاده از خدمات حرفه‌ای می‌تواند مناسب‌تر از معامله مستقیم و بدون تخصص باشد.
Expected Answer Draft ID: expected-answer-draft:34ec65edfdfa3d09e26313f0241d0b4f
Human Decision: ACCEPT

=== GROUNDING ===
instrument_type: True
risk_level: True
loss_tolerance: True
general_public: True
indirect_entry: Tru

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 66 — Pilot #14 Speaker Continuity Diagnostic

در Citation فعلی، بخش اصلی پاسخ در Utterance `00005` قرار دارد، اما Speaker آن در داده فعلی `UNRESOLVED` است.

این Cell فقط بررسی می‌کند:

- Utteranceهای اطراف `00004` و `00005` در Episode 36 چه هستند؛
- Speaker هر Utterance چگونه در Source ثبت شده است؛
- آیا Source خام نشانه‌ای برای انتساب Utterance `00005` دارد یا خیر؛
- Metadata خود Pilot چه محدودیت Attributionای دارد.

هیچ Citation، Human Decision، Registry یا Queue تغییر نمی‌کند.

</div>

In [66]:
# Code Cell 66

cell66_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell65_gate") is not True:
    cell66_failures.append(
        "CELL65_NOT_PASS"
    )


# ============================================================
# 2) Current source identity
# ============================================================

CELL66_PROGRAM_ID = "program:chehelston"
CELL66_EPISODE_NUMBER = 36

cell66_target_indices = [
    4,
    5,
]


# ============================================================
# 3) Locate canonical extracted Transcript
# ============================================================

cell66_transcript_path = (
    cell21_registry_path.parents[1]
    /
    "interim"
    /
    "transcripts_extracted_v1.jsonl"
)


if not cell66_transcript_path.exists():
    cell66_failures.append(
        "TRANSCRIPTS_EXTRACTED_V1_NOT_FOUND"
    )


cell66_episode_records = []


if cell66_transcript_path.exists():

    with cell66_transcript_path.open(
        "r",
        encoding="utf-8",
    ) as handle:

        for line_number, line in enumerate(
            handle,
            start=1,
        ):

            line = line.strip()

            if not line:
                continue


            try:
                record = json.loads(
                    line
                )
            except Exception:
                continue


            if (
                record.get(
                    "program_id"
                )
                !=
                CELL66_PROGRAM_ID
            ):
                continue


            source_identity = (
                record.get(
                    "source_identity"
                )
                or
                {}
            )


            if (
                source_identity.get(
                    "episode_number"
                )
                !=
                CELL66_EPISODE_NUMBER
            ):
                continue


            utterance_index = record.get(
                "utterance_index"
            )


            if not isinstance(
                utterance_index,
                int,
            ):
                continue


            if not (
                1
                <=
                utterance_index
                <=
                9
            ):
                continue


            content = (
                record.get(
                    "content"
                )
                or
                {}
            )


            speaker = (
                record.get(
                    "speaker"
                )
                or
                {}
            )


            base = (
                speaker.get(
                    "base"
                )
                or
                {}
            )


            effective = (
                speaker.get(
                    "effective"
                )
                or
                {}
            )


            locator = (
                record.get(
                    "source_locator"
                )
                or
                {}
            )


            cell66_episode_records.append({
                "jsonl_line_number":
                    line_number,

                "utterance_index":
                    utterance_index,

                "utterance_id":
                    record.get(
                        "utterance_id"
                    ),

                "source_text":
                    (
                        content.get(
                            "source_text"
                        )
                        or
                        content.get(
                            "text"
                        )
                        or
                        ""
                    ),

                "base_status":
                    base.get(
                        "status"
                    ),

                "base_raw":
                    base.get(
                        "raw"
                    ),

                "base_canonical":
                    base.get(
                        "canonical"
                    ),

                "effective_label":
                    effective.get(
                        "label"
                    ),

                "effective_canonical":
                    effective.get(
                        "canonical"
                    ),

                "effective_basis":
                    effective.get(
                        "basis"
                    ),

                "effective_primary":
                    effective.get(
                        "primary_dr_roudi_corpus"
                    ),

                "source_line_start":
                    locator.get(
                        "source_line_start"
                    ),

                "source_line_end":
                    locator.get(
                        "source_line_end"
                    ),

                "relative_path":
                    source_identity.get(
                        "relative_path"
                    ),

                "physical_file_id":
                    source_identity.get(
                        "physical_file_id"
                    ),
            })


cell66_episode_records = sorted(
    cell66_episode_records,
    key=lambda item:
        item[
            "utterance_index"
        ],
)


if not cell66_episode_records:
    cell66_failures.append(
        "EPISODE_CONTEXT_NOT_FOUND"
    )


# ============================================================
# 4) Resolve raw source file
# ============================================================

cell66_relative_paths = sorted(
    set(
        item.get(
            "relative_path"
        )
        for item
        in cell66_episode_records
        if item.get(
            "relative_path"
        )
    )
)


cell66_raw_source_path = None


if len(
    cell66_relative_paths
) == 1:

    cell66_repo_root = (
        cell21_registry_path.parents[3]
    )


    candidate_path = (
        cell66_repo_root
        /
        cell66_relative_paths[
            0
        ]
    )


    if candidate_path.exists():
        cell66_raw_source_path = (
            candidate_path
        )


# ============================================================
# 5) Read raw source lines around target
# ============================================================

cell66_raw_line_window = []


if cell66_raw_source_path is not None:

    raw_lines = (
        cell66_raw_source_path
        .read_text(
            encoding="utf-8"
        )
        .splitlines()
    )


    target_source_lines = [
        value
        for record
        in cell66_episode_records
        if (
            record[
                "utterance_index"
            ]
            in
            cell66_target_indices
        )
        for value
        in [
            record.get(
                "source_line_start"
            ),
            record.get(
                "source_line_end"
            ),
        ]
        if isinstance(
            value,
            int,
        )
    ]


    if target_source_lines:

        window_start = max(
            1,
            min(
                target_source_lines
            )
            -
            6,
        )


        window_end = min(
            len(
                raw_lines
            ),
            max(
                target_source_lines
            )
            +
            6,
        )


        for source_line_number in range(
            window_start,
            window_end
            +
            1,
        ):

            cell66_raw_line_window.append({
                "line_number":
                    source_line_number,

                "text":
                    raw_lines[
                        source_line_number
                        -
                        1
                    ],
            })


# ============================================================
# 6) Pilot metadata inspection
# ============================================================

def cell66_collect_fields(
    value,
    path=(),
):

    matches = []


    keywords = (
        "speaker",
        "attribution",
        "authority",
        "multi",
        "contrast",
        "stratum",
        "task_family",
        "question_archetype",
        "guardrail",
    )


    if isinstance(
        value,
        dict,
    ):

        for key, item in value.items():

            key_text = str(
                key
            ).lower()


            if any(
                keyword
                in
                key_text
                for keyword
                in keywords
            ):

                matches.append({
                    "path":
                        path
                        +
                        (
                            str(
                                key
                            ),
                        ),

                    "value":
                        copy.deepcopy(
                            item
                        ),
                })


            matches.extend(
                cell66_collect_fields(
                    item,
                    path
                    +
                    (
                        str(
                            key
                        ),
                    ),
                )
            )


    elif isinstance(
        value,
        list,
    ):

        for index, item in enumerate(
            value
        ):

            matches.extend(
                cell66_collect_fields(
                    item,
                    path
                    +
                    (
                        str(
                            index
                        ),
                    ),
                )
            )


    return matches


cell66_pilot_metadata = (
    cell66_collect_fields(
        cell64_next_pilot_packet
    )
)


# ============================================================
# 7) Direct target speaker comparison
# ============================================================

cell66_target_record_map = {
    record[
        "utterance_index"
    ]:
        record
    for record
    in cell66_episode_records
}


cell66_utterance_4 = (
    cell66_target_record_map.get(
        4
    )
)


cell66_utterance_5 = (
    cell66_target_record_map.get(
        5
    )
)


cell66_utterance_4_primary = (
    isinstance(
        cell66_utterance_4,
        dict,
    )
    and
    cell66_utterance_4.get(
        "effective_canonical"
    )
    ==
    "Dr. Komeil Roudi"
    and
    cell66_utterance_4.get(
        "effective_primary"
    )
    is True
)


cell66_utterance_5_resolved = (
    isinstance(
        cell66_utterance_5,
        dict,
    )
    and
    cell66_utterance_5.get(
        "effective_canonical"
    )
    is not None
)


# ============================================================
# 8) Diagnostic gate
# ============================================================

cell66_gate = all([
    len(
        cell66_failures
    )
    ==
    0,

    isinstance(
        cell66_utterance_4,
        dict,
    ),

    isinstance(
        cell66_utterance_5,
        dict,
    ),

    cell66_utterance_4_primary,
])


# ============================================================
# 9) Diagnostics
# ============================================================

print(
    "=== PILOT #14 SPEAKER CONTINUITY DIAGNOSTIC ==="
)


print(
    "\n=== TARGET SPEAKER STATUS ==="
)

print(
    "Utterance 00004 primary Dr. Komeil Roudi:",
    cell66_utterance_4_primary,
)

print(
    "Utterance 00005 currently resolved:",
    cell66_utterance_5_resolved,
)


print(
    "\n=== EPISODE 36 LOCAL UTTERANCES ==="
)


for record in cell66_episode_records:

    marker = ""

    if record[
        "utterance_index"
    ] == 4:
        marker = "  <-- TARGET LEFT"

    elif record[
        "utterance_index"
    ] == 5:
        marker = "  <-- TARGET RIGHT"


    print(
        "\n"
        f"[{record['utterance_index']:05d}]"
        f"{marker}"
    )

    print(
        "Source lines:",
        (
            record.get(
                "source_line_start"
            ),
            record.get(
                "source_line_end"
            ),
        ),
    )

    print(
        "Base raw:",
        record.get(
            "base_raw"
        ),
    )

    print(
        "Base status:",
        record.get(
            "base_status"
        ),
    )

    print(
        "Effective label:",
        record.get(
            "effective_label"
        ),
    )

    print(
        "Effective canonical:",
        record.get(
            "effective_canonical"
        ),
    )

    print(
        "Effective basis:",
        record.get(
            "effective_basis"
        ),
    )

    print(
        "Primary:",
        record.get(
            "effective_primary"
        ),
    )

    print(
        "Text:"
    )

    print(
        record.get(
            "source_text"
        )
    )


print(
    "\n=== RAW SOURCE FILE ==="
)

print(
    "Relative path:",
    (
        cell66_relative_paths[
            0
        ]
        if len(
            cell66_relative_paths
        ) == 1
        else
        cell66_relative_paths
    ),
)

print(
    "Raw source accessible:",
    cell66_raw_source_path
    is not None,
)


if cell66_raw_line_window:

    print(
        "\n=== RAW SOURCE LINES AROUND TARGET ==="
    )


    for row in cell66_raw_line_window:

        print(
            f"{row['line_number']:04d}: "
            f"{row['text']}"
        )


print(
    "\n=== RELEVANT PILOT METADATA ==="
)


for match in cell66_pilot_metadata:

    print(
        " > ".join(
            match[
                "path"
            ]
        ),
        "=",
        match[
            "value"
        ],
    )


print(
    "\n=== SAFETY ==="
)

print(
    "Citation changed:",
    False,
)

print(
    "Registry write:",
    False,
)

print(
    "Queue advance:",
    False,
)

print(
    "Human Decision added:",
    False,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell66_failures
)


print(
    "\nCODE CELL 66 GATE:",
    (
        "PASS"
        if cell66_gate
        else "FAIL"
    ),
)


assert cell66_gate, (
    "Pilot #14 speaker continuity diagnostic failed. "
    "Repair this SAME Code Cell 66 before Citation review."
)

=== PILOT #14 SPEAKER CONTINUITY DIAGNOSTIC ===

=== TARGET SPEAKER STATUS ===
Utterance 00004 primary Dr. Komeil Roudi: True
Utterance 00005 currently resolved: False

=== EPISODE 36 LOCAL UTTERANCES ===

[00001]
Source lines: (21, 21)
Base raw: None
Base status: UNRESOLVED
Effective label: None
Effective canonical: None
Effective basis: BASE_PARSER_ATTRIBUTION
Primary: None
Text:
به نام خداوند دانای توانای مهربان. سلام و عرض ادب به شما شنوندگان عزیز رادیو گفت‌وگو. من هادی نجف هستم و امروز با موضوع «سواد مالی زنان در جهان» گفت‌وگو می‌کنم با دکتر کمیل رودی، مدیر آکادمی هوش مالی، و سرکار خانم دکتر کوثر اخوان، استاد دانشگاه و پژوهشگر سواد مالی.

[00002]
Source lines: (23, 23)
Base raw: None
Base status: UNRESOLVED
Effective label: None
Effective canonical: None
Effective basis: BASE_PARSER_ATTRIBUTION
Primary: None
Text:
از شما هم می‌خواهم به شماره پیامک 30001035 بگویید توانمندسازی مالی زنان را تا چه اندازه مهم می‌دانید. شماره تماس 162 و وب‌سایت رادیو گفت‌وگو هم مسیرهای ارتباطی برنامه‌ان

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 67 — Human Speaker Continuity Certification + Pilot #14 Retarget Discovery

Human Reviewer تأیید کرد:

اگر یک گوینده با برچسب صریح مشخص شده باشد و پاراگراف‌های بعدی بدون معرفی گوینده جدید ادامه پیدا کنند، آن بخش‌ها ادامه صحبت همان گوینده هستند تا زمانی که یک برچسب صریح برای گوینده جدید ظاهر شود.

برای Target فعلی:

- Utterance `00004` با برچسب صریح `دکتر رودی` شروع شده است.
- Utterance `00005` بدون تغییر گوینده ادامه پیدا کرده است.
- در Utterance `00006` برچسب صریح `مجری` ظاهر می‌شود.

بنابراین Utterance `00005` با Human-certified attribution متعلق به Dr. Komeil Roudi است.

از آنجا که هر دو بخش Target فعلی در حقیقت متعلق به یک گوینده‌اند، این Target دیگر پوشش واقعی `MULTI_SPEAKER_ATTRIBUTION_CHEHELSTON` را تأمین نمی‌کند.

در این مرحله:

- Raw Source تغییر نمی‌کند؛
- Human Attribution Overlay ثبت می‌شود؛
- سؤال و جواب پذیرفته‌شده قبلی برای Lineage حفظ می‌شوند؛
- Pilot وارد `BLOCKED_REQUIRES_RETARGET` می‌شود؛
- Candidateهای واقعاً چندگوینده‌ای از Corpus چهلستون کشف می‌شوند؛
- Queue همچنان 10 باقی می‌ماند.

</div>

In [67]:
# Code Cell 67

cell67_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell66_gate") is not True:
    cell67_failures.append(
        "CELL66_NOT_PASS"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell67_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Current Pilot
# ============================================================

CELL67_PILOT_ID = CELL65_PILOT_ID
CELL67_GOLDEN_CASE_ID = CELL65_GOLDEN_CASE_ID
CELL67_TASK_DRAFT_ID = CELL65_TASK_DRAFT_ID
CELL67_OLD_TARGET_EVIDENCE_ID = CELL65_TARGET_EVIDENCE_ID


# ============================================================
# 3) Human-certified speaker continuity
# ============================================================

cell67_human_speaker_certification = {
    "document_id":
        "program:chehelston:episode-036:utterance:00005",

    "human_decision":
        "SPEAKER_CONTINUITY_CERTIFIED",

    "certified_speaker":
        "Dr. Komeil Roudi",

    "certified_primary":
        True,

    "certification_type":
        "HUMAN_CERTIFIED_SPEAKER_CONTINUITY",

    "preceding_explicit_speaker_document_id":
        "program:chehelston:episode-036:utterance:00004",

    "preceding_explicit_speaker_label":
        "دکتر رودی",

    "next_explicit_speaker_document_id":
        "program:chehelston:episode-036:utterance:00006",

    "next_explicit_speaker_label":
        "مجری",

    "reason":
        (
            "The unlabeled utterance continues the explicitly labeled "
            "Dr. Komeil Roudi turn. No new speaker label appears until "
            "the explicit host marker in utterance 00006."
        ),

    "canonical_raw_source_mutated":
        False,

    "parser_output_mutated":
        False,

    "human_overlay_only":
        True,
}


cell67_continuity_policy = {
    "policy_type":
        "HUMAN_REVIEW_ATTRIBUTION_POLICY",

    "policy_name":
        "EXPLICIT_SPEAKER_CONTINUITY_UNTIL_NEXT_EXPLICIT_LABEL",

    "rule":
        (
            "When an explicitly labeled speaker begins a turn, "
            "subsequent unlabeled continuation paragraphs belong to "
            "that speaker until another explicit speaker label appears."
        ),

    "application_scope":
        "HUMAN_ADJUDICATION_OVERLAY",

    "automatic_unknown_speaker_inference_enabled":
        False,

    "canonical_parser_guardrail_changed":
        False,
}


# ============================================================
# 4) Validate source boundary used for certification
# ============================================================

cell67_record_map = {
    record[
        "utterance_index"
    ]:
        record
    for record
    in cell66_episode_records
}


cell67_record_4 = cell67_record_map.get(
    4
)

cell67_record_5 = cell67_record_map.get(
    5
)

cell67_record_6 = cell67_record_map.get(
    6
)


cell67_certification_boundary_valid = all([
    isinstance(
        cell67_record_4,
        dict,
    ),

    isinstance(
        cell67_record_5,
        dict,
    ),

    isinstance(
        cell67_record_6,
        dict,
    ),

    cell67_record_4.get(
        "base_raw"
    )
    ==
    "دکتر رودی",

    cell67_record_4.get(
        "effective_canonical"
    )
    ==
    "Dr. Komeil Roudi",

    cell67_record_4.get(
        "effective_primary"
    )
    is True,

    cell67_record_5.get(
        "base_status"
    )
    ==
    "UNRESOLVED",

    cell67_record_6.get(
        "base_raw"
    )
    ==
    "مجری",
])


if not cell67_certification_boundary_valid:
    cell67_failures.append(
        "HUMAN_CERTIFICATION_SOURCE_BOUNDARY_INVALID"
    )


# ============================================================
# 5) Correct current Target interpretation without mutating raw
# ============================================================

cell67_corrected_target_parts = copy.deepcopy(
    cell65_target_parts
)


cell67_overlay_applied = False


for part in cell67_corrected_target_parts:

    if not isinstance(
        part,
        dict,
    ):
        continue


    if (
        part.get(
            "document_id"
        )
        ==
        "program:chehelston:episode-036:utterance:00005"
    ):

        part[
            "human_attribution_overlay"
        ] = {
            "status":
                "HUMAN_CERTIFIED",

            "effective_canonical":
                "Dr. Komeil Roudi",

            "effective_label":
                "دکتر رودی",

            "effective_primary":
                True,

            "basis":
                "HUMAN_CERTIFIED_SPEAKER_CONTINUITY",

            "raw_speaker_state_preserved":
                True,
        }


        cell67_overlay_applied = True


# ============================================================
# 6) Determine whether old Target still satisfies Pilot coverage
# ============================================================

cell67_corrected_effective_speakers = []


for part in cell67_corrected_target_parts:

    if not isinstance(
        part,
        dict,
    ):
        continue


    overlay = (
        part.get(
            "human_attribution_overlay"
        )
        or
        {}
    )


    speaker_state = (
        part.get(
            "speaker_state"
        )
        or
        {}
    )


    effective_speaker = (
        overlay.get(
            "effective_canonical"
        )
        or
        speaker_state.get(
            "effective_canonical"
        )
        or
        overlay.get(
            "effective_label"
        )
        or
        speaker_state.get(
            "effective_label"
        )
    )


    if effective_speaker:
        cell67_corrected_effective_speakers.append(
            effective_speaker
        )


cell67_corrected_distinct_speakers = sorted(
    set(
        cell67_corrected_effective_speakers
    )
)


cell67_old_target_is_true_multi_speaker = (
    len(
        cell67_corrected_distinct_speakers
    )
    >=
    2
)


cell67_retarget_required = all([
    cell67_overlay_applied,

    cell67_corrected_distinct_speakers
    ==
    [
        "Dr. Komeil Roudi"
    ],

    cell67_old_target_is_true_multi_speaker
    is False,
])


# ============================================================
# 7) Load all Chehelston transcript records
# ============================================================

cell67_chehelston_records = []


with cell66_transcript_path.open(
    "r",
    encoding="utf-8",
) as handle:

    for line_number, line in enumerate(
        handle,
        start=1,
    ):

        line = line.strip()


        if not line:
            continue


        try:

            record = json.loads(
                line
            )

        except Exception:

            continue


        if (
            record.get(
                "program_id"
            )
            !=
            "program:chehelston"
        ):
            continue


        source_identity = (
            record.get(
                "source_identity"
            )
            or
            {}
        )


        episode_number = source_identity.get(
            "episode_number"
        )


        utterance_index = record.get(
            "utterance_index"
        )


        if not isinstance(
            episode_number,
            int,
        ):
            continue


        if not isinstance(
            utterance_index,
            int,
        ):
            continue


        speaker = (
            record.get(
                "speaker"
            )
            or
            {}
        )


        base = (
            speaker.get(
                "base"
            )
            or
            {}
        )


        effective = (
            speaker.get(
                "effective"
            )
            or
            {}
        )


        content = (
            record.get(
                "content"
            )
            or
            {}
        )


        source_text = (
            content.get(
                "source_text"
            )
            or
            content.get(
                "text"
            )
            or
            ""
        )


        if not isinstance(
            source_text,
            str,
        ):
            source_text = ""


        cell67_chehelston_records.append({
            "episode_number":
                episode_number,

            "utterance_index":
                utterance_index,

            "utterance_id":
                record.get(
                    "utterance_id"
                ),

            "source_text":
                source_text.strip(),

            "base_status":
                base.get(
                    "status"
                ),

            "base_raw":
                base.get(
                    "raw"
                ),

            "effective_label":
                effective.get(
                    "label"
                ),

            "effective_canonical":
                effective.get(
                    "canonical"
                ),

            "effective_primary":
                effective.get(
                    "primary_dr_roudi_corpus"
                ),

            "source_locator":
                copy.deepcopy(
                    record.get(
                        "source_locator"
                    )
                    or
                    {}
                ),

            "source_identity":
                copy.deepcopy(
                    source_identity
                ),

            "jsonl_line_number":
                line_number,
        })


cell67_chehelston_records = sorted(
    cell67_chehelston_records,
    key=lambda item: (
        item[
            "episode_number"
        ],
        item[
            "utterance_index"
        ],
    ),
)


# ============================================================
# 8) Group records by Episode
# ============================================================

cell67_episode_map = {}


for record in cell67_chehelston_records:

    cell67_episode_map.setdefault(
        record[
            "episode_number"
        ],
        [],
    ).append(
        record
    )


# ============================================================
# 9) Build speaker turns using Human-certified continuity rule
# ============================================================

cell67_all_turns = []


for episode_number, records in sorted(
    cell67_episode_map.items()
):

    current_turn = None


    for record in records:

        explicit_marker = (
            record.get(
                "base_status"
            )
            ==
            "EXPLICIT_SOURCE_MARKER"
        )


        if explicit_marker:

            if current_turn is not None:
                cell67_all_turns.append(
                    current_turn
                )


            speaker_name = (
                record.get(
                    "effective_canonical"
                )
                or
                record.get(
                    "effective_label"
                )
                or
                record.get(
                    "base_raw"
                )
            )


            current_turn = {
                "episode_number":
                    episode_number,

                "speaker":
                    speaker_name,

                "speaker_explicit_label":
                    record.get(
                        "base_raw"
                    ),

                "speaker_is_primary_dr_roudi":
                    (
                        record.get(
                            "effective_canonical"
                        )
                        ==
                        "Dr. Komeil Roudi"
                        and
                        record.get(
                            "effective_primary"
                        )
                        is True
                    ),

                "start_index":
                    record[
                        "utterance_index"
                    ],

                "end_index":
                    record[
                        "utterance_index"
                    ],

                "records": [
                    copy.deepcopy(
                        record
                    )
                ],
            }


        elif current_turn is not None:

            current_turn[
                "records"
            ].append(
                copy.deepcopy(
                    record
                )
            )


            current_turn[
                "end_index"
            ] = record[
                "utterance_index"
            ]


    if current_turn is not None:
        cell67_all_turns.append(
            current_turn
        )


# ============================================================
# 10) Complete turn text
# ============================================================

for turn in cell67_all_turns:

    turn[
        "combined_text"
    ] = "\n".join(
        record[
            "source_text"
        ]
        for record
        in turn[
            "records"
        ]
        if record[
            "source_text"
        ]
    ).strip()


    turn[
        "char_count"
    ] = len(
        turn[
            "combined_text"
        ]
    )


# ============================================================
# 11) Discover true adjacent multi-speaker boundaries
# ============================================================

cell67_candidate_pairs = []


turns_by_episode = {}


for turn in cell67_all_turns:

    turns_by_episode.setdefault(
        turn[
            "episode_number"
        ],
        [],
    ).append(
        turn
    )


for episode_number, turns in sorted(
    turns_by_episode.items()
):

    turns = sorted(
        turns,
        key=lambda item:
            item[
                "start_index"
            ],
    )


    for index in range(
        len(
            turns
        )
        -
        1
    ):

        left = turns[
            index
        ]

        right = turns[
            index
            +
            1
        ]


        if not left.get(
            "speaker"
        ):
            continue


        if not right.get(
            "speaker"
        ):
            continue


        if (
            left[
                "speaker"
            ]
            ==
            right[
                "speaker"
            ]
        ):
            continue


        one_side_is_primary_dr = (
            left.get(
                "speaker_is_primary_dr_roudi"
            )
            is True
            or
            right.get(
                "speaker_is_primary_dr_roudi"
            )
            is True
        )


        if not one_side_is_primary_dr:
            continue


        if (
            left[
                "char_count"
            ]
            <
            20
            or
            right[
                "char_count"
            ]
            <
            20
        ):
            continue


        direction = (
            "OTHER_TO_DR"
            if
            (
                left.get(
                    "speaker_is_primary_dr_roudi"
                )
                is not True
                and
                right.get(
                    "speaker_is_primary_dr_roudi"
                )
                is True
            )
            else
            "DR_TO_OTHER"
        )


        other_turn = (
            left
            if direction
            ==
            "OTHER_TO_DR"
            else
            right
        )


        dr_turn = (
            right
            if direction
            ==
            "OTHER_TO_DR"
            else
            left
        )


        question_signal = (
            "؟"
            in
            other_turn[
                "combined_text"
            ]
        )


        preferred_direction = (
            direction
            ==
            "OTHER_TO_DR"
        )


        size_score = min(
            dr_turn[
                "char_count"
            ],
            1200,
        ) / 1200


        score = (
            100
            +
            (
                40
                if preferred_direction
                else 0
            )
            +
            (
                35
                if question_signal
                else 0
            )
            +
            int(
                25
                *
                size_score
            )
        )


        identity_payload = {
            "episode_number":
                episode_number,

            "left_speaker":
                left[
                    "speaker"
                ],

            "left_start":
                left[
                    "start_index"
                ],

            "left_end":
                left[
                    "end_index"
                ],

            "right_speaker":
                right[
                    "speaker"
                ],

            "right_start":
                right[
                    "start_index"
                ],

            "right_end":
                right[
                    "end_index"
                ],
        }


        candidate_id = (
            "multi-speaker-retarget-candidate:"
            +
            hashlib.sha256(
                cell10_canonical_json(
                    identity_payload
                ).encode(
                    "utf-8"
                )
            ).hexdigest()[
                :32
            ]
        )


        cell67_candidate_pairs.append({
            "candidate_id":
                candidate_id,

            "score":
                score,

            "episode_number":
                episode_number,

            "direction":
                direction,

            "question_signal":
                question_signal,

            "left":
                copy.deepcopy(
                    left
                ),

            "right":
                copy.deepcopy(
                    right
                ),

            "dr_turn":
                copy.deepcopy(
                    dr_turn
                ),

            "other_turn":
                copy.deepcopy(
                    other_turn
                ),
        })


# ============================================================
# 12) Rank / deduplicate candidates
# ============================================================

cell67_candidate_map = {}


for candidate in cell67_candidate_pairs:

    previous = cell67_candidate_map.get(
        candidate[
            "candidate_id"
        ]
    )


    if (
        previous is None
        or
        candidate[
            "score"
        ]
        >
        previous[
            "score"
        ]
    ):

        cell67_candidate_map[
            candidate[
                "candidate_id"
            ]
        ] = candidate


cell67_ranked_candidates = sorted(
    cell67_candidate_map.values(),
    key=lambda item: (
        -item[
            "score"
        ],
        item[
            "episode_number"
        ],
        item[
            "left"
        ][
            "start_index"
        ],
    ),
)


cell67_top_candidates = copy.deepcopy(
    cell67_ranked_candidates[
        :8
    ]
)


if not cell67_top_candidates:
    cell67_failures.append(
        "NO_TRUE_MULTI_SPEAKER_CANDIDATES_FOUND"
    )


# ============================================================
# 13) Preserve accepted QA as superseded by Retarget
# ============================================================

cell67_superseded_qa = {
    "question_draft_id":
        cell65_question_draft_id,

    "question_text":
        CELL65_ACCEPTED_QUESTION,

    "question_human_decision":
        "ACCEPT",

    "expected_answer_draft_id":
        cell65_expected_answer_draft_id,

    "expected_answer_text":
        CELL65_ACCEPTED_ANSWER,

    "expected_answer_human_decision":
        "ACCEPT",

    "status":
        "SUPERSEDED_BY_TARGET_RETARGET",

    "reason":
        (
            "The QA remains semantically useful, but the corrected "
            "Target no longer satisfies the Pilot's required true "
            "multi-speaker attribution coverage."
        ),

    "may_be_reused_in_future_compatible_pilot":
        True,
}


# ============================================================
# 14) Update active Human Review workspace
# ============================================================

cell67_active_workspace = copy.deepcopy(
    cell65_persisted_active_case
)


cell67_active_workspace[
    "workflow_state"
] = "BLOCKED_REQUIRES_RETARGET"


cell67_active_workspace[
    "status"
] = "ACTIVE_RETARGET_REVIEW"


cell67_active_workspace[
    "human_attribution_correction"
] = copy.deepcopy(
    cell67_human_speaker_certification
)


cell67_active_workspace[
    "human_attribution_policy"
] = copy.deepcopy(
    cell67_continuity_policy
)


cell67_active_workspace[
    "corrected_target_interpretation"
] = {
    "old_target_evidence_id":
        CELL67_OLD_TARGET_EVIDENCE_ID,

    "effective_speakers_after_human_overlay":
        copy.deepcopy(
            cell67_corrected_distinct_speakers
        ),

    "true_multi_speaker_after_correction":
        cell67_old_target_is_true_multi_speaker,

    "coverage_valid_after_correction":
        False,

    "raw_target_preserved":
        True,
}


cell67_active_workspace[
    "retarget_review"
] = {
    "human_decision":
        "RETARGET",

    "reason":
        (
            "Human-certified speaker continuity shows that both old "
            "Target parts are spoken by Dr. Komeil Roudi. Therefore "
            "the old Target does not satisfy the true multi-speaker "
            "attribution coverage required by this Pilot."
        ),

    "old_target_evidence_id":
        CELL67_OLD_TARGET_EVIDENCE_ID,

    "replacement_status":
        "CANDIDATES_PENDING_HUMAN_SELECTION",

    "candidate_count":
        len(
            cell67_top_candidates
        ),

    "candidates":
        copy.deepcopy(
            cell67_top_candidates
        ),

    "superseded_accepted_qa":
        copy.deepcopy(
            cell67_superseded_qa
        ),
}


# ============================================================
# 15) Continuation state
# ============================================================

cell67_continuation_core = {
    "next_pilot_id":
        CELL67_PILOT_ID,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell64_remaining_pilot_ids
        ),

    "remaining_count":
        10,

    "active_pilot_status":
        "BLOCKED_REQUIRES_RETARGET",

    "active_target_evidence_id":
        CELL67_OLD_TARGET_EVIDENCE_ID,

    "retarget_status":
        "CANDIDATES_PENDING_HUMAN_SELECTION",

    "last_completed_pilot_id":
        cell64_continuation_state.get(
            "last_completed_pilot_id"
        ),

    "last_completion_type":
        cell64_continuation_state.get(
            "last_completion_type"
        ),
}


cell67_continuation_state_sha256 = (
    cell10_stable_sha(
        cell67_continuation_core
    )
)


cell67_continuation_state = {
    **copy.deepcopy(
        cell67_continuation_core
    ),

    "continuation_state_sha256":
        cell67_continuation_state_sha256,
}


# ============================================================
# 16) Persist Registry
# ============================================================

cell67_persistent_registry_payload = copy.deepcopy(
    cell65_persisted_registry
)


cell67_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell67_active_workspace
)


cell67_persistent_registry_payload[
    "remaining_pilot_count"
] = 10


cell67_persistent_registry_payload[
    "continuation_state"
] = copy.deepcopy(
    cell67_continuation_state
)


cell67_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell67_persistent_registry_payload
    )
)


# ============================================================
# 17) Read-back validation
# ============================================================

cell67_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell67_persisted_active_case = (
    cell67_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell67_persistence_valid = all([
    cell67_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL67_PILOT_ID,

    cell67_persisted_active_case.get(
        "workflow_state"
    )
    ==
    "BLOCKED_REQUIRES_RETARGET",

    cell67_persisted_active_case.get(
        "human_attribution_correction",
        {},
    ).get(
        "certified_speaker"
    )
    ==
    "Dr. Komeil Roudi",

    cell67_persisted_active_case.get(
        "retarget_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "RETARGET",

    cell67_persisted_active_case.get(
        "retarget_review",
        {},
    ).get(
        "replacement_status"
    )
    ==
    "CANDIDATES_PENDING_HUMAN_SELECTION",

    cell67_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    10,

    cell67_persisted_registry.get(
        "continuation_state",
        {},
    ).get(
        "continuation_state_sha256"
    )
    ==
    cell67_continuation_state_sha256,
])


# ============================================================
# 18) Safety
# ============================================================

cell67_queue_still_10 = all([
    len(
        cell64_remaining_pilot_ids
    )
    ==
    10,

    cell64_remaining_pilot_ids[
        0
    ]
    ==
    CELL67_PILOT_ID,
])


cell67_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell67_full_572_blocked = (
    cell65_full_572_blocked
    is True
)


# ============================================================
# 19) Final Gate
# ============================================================

cell67_gate = all([
    len(
        cell67_failures
    )
    ==
    0,

    cell67_certification_boundary_valid,

    cell67_overlay_applied,

    cell67_retarget_required,

    len(
        cell67_top_candidates
    )
    >
    0,

    cell67_persistence_valid,

    cell67_queue_still_10,

    cell67_base_seed_unchanged,

    cell67_full_572_blocked,
])


# ============================================================
# 20) Diagnostics
# ============================================================

print(
    "=== PILOT #14 HUMAN SPEAKER CERTIFICATION + RETARGET DISCOVERY ==="
)


print(
    "\n=== HUMAN SPEAKER CERTIFICATION ==="
)

print(
    "Document:",
    cell67_human_speaker_certification[
        "document_id"
    ],
)

print(
    "Certified Speaker:",
    "Dr. Komeil Roudi",
)

print(
    "Certification Type:",
    "HUMAN_CERTIFIED_SPEAKER_CONTINUITY",
)

print(
    "Previous explicit speaker:",
    "دکتر رودی",
)

print(
    "Next explicit speaker:",
    "مجری",
)

print(
    "Raw canonical source mutated:",
    False,
)

print(
    "Parser output mutated:",
    False,
)


print(
    "\n=== CONTINUITY POLICY ==="
)

print(
    cell67_continuity_policy[
        "rule"
    ]
)

print(
    "Automatic parser inference enabled:",
    False,
)


print(
    "\n=== OLD TARGET AFTER CORRECTION ==="
)

print(
    "Old Target:",
    CELL67_OLD_TARGET_EVIDENCE_ID,
)

print(
    "Effective speakers:",
    cell67_corrected_distinct_speakers,
)

print(
    "True multi-speaker:",
    cell67_old_target_is_true_multi_speaker,
)

print(
    "Retarget required:",
    cell67_retarget_required,
)


print(
    "\n=== SUPERSEDED BUT PRESERVED QA ==="
)

print(
    "Question:",
    CELL65_ACCEPTED_QUESTION,
)

print(
    "Answer:",
    CELL65_ACCEPTED_ANSWER,
)

print(
    "Reusable later:",
    True,
)


print(
    "\n=== TRUE MULTI-SPEAKER RETARGET CANDIDATES ==="
)


for index, candidate in enumerate(
    cell67_top_candidates,
    start=1,
):

    print(
        "\n"
        +
        "="
        *
        72
    )

    print(
        "CANDIDATE",
        index,
    )

    print(
        "Candidate ID:",
        candidate[
            "candidate_id"
        ],
    )

    print(
        "Score:",
        candidate[
            "score"
        ],
    )

    print(
        "Episode:",
        candidate[
            "episode_number"
        ],
    )

    print(
        "Direction:",
        candidate[
            "direction"
        ],
    )

    print(
        "Other speaker question signal:",
        candidate[
            "question_signal"
        ],
    )


    print(
        "\nLEFT TURN"
    )

    print(
        "Speaker:",
        candidate[
            "left"
        ][
            "speaker"
        ],
    )

    print(
        "Utterances:",
        (
            candidate[
                "left"
            ][
                "start_index"
            ],
            candidate[
                "left"
            ][
                "end_index"
            ],
        ),
    )

    print(
        candidate[
            "left"
        ][
            "combined_text"
        ]
    )


    print(
        "\nRIGHT TURN"
    )

    print(
        "Speaker:",
        candidate[
            "right"
        ][
            "speaker"
        ],
    )

    print(
        "Utterances:",
        (
            candidate[
                "right"
            ][
                "start_index"
            ],
            candidate[
                "right"
            ][
                "end_index"
            ],
        ),
    )

    print(
        candidate[
            "right"
        ][
            "combined_text"
        ]
    )


print(
    "\n=== WORKFLOW / PERSISTENCE ==="
)

print(
    "Workflow:",
    cell67_persisted_active_case.get(
        "workflow_state"
    ),
)

print(
    "Retarget:",
    "PENDING HUMAN CANDIDATE SELECTION",
)

print(
    "Persistence valid:",
    cell67_persistence_valid,
)


print(
    "\n=== QUEUE / SAFETY ==="
)

print(
    "Queue still 10:",
    cell67_queue_still_10,
)

print(
    "Base Seed unchanged:",
    cell67_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell67_full_572_blocked,
)

print(
    "External lookup:",
    False,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell67_failures
)


print(
    "\nCODE CELL 67 GATE:",
    (
        "PASS"
        if cell67_gate
        else "FAIL"
    ),
)


assert cell67_gate, (
    "Pilot #14 Human speaker certification / Retarget discovery "
    "failed. Repair this SAME Code Cell 67 before selecting "
    "a replacement multi-speaker Target."
)

=== PILOT #14 HUMAN SPEAKER CERTIFICATION + RETARGET DISCOVERY ===

=== HUMAN SPEAKER CERTIFICATION ===
Document: program:chehelston:episode-036:utterance:00005
Certified Speaker: Dr. Komeil Roudi
Certification Type: HUMAN_CERTIFIED_SPEAKER_CONTINUITY
Previous explicit speaker: دکتر رودی
Next explicit speaker: مجری
Raw canonical source mutated: False
Parser output mutated: False

=== CONTINUITY POLICY ===
When an explicitly labeled speaker begins a turn, subsequent unlabeled continuation paragraphs belong to that speaker until another explicit speaker label appears.
Automatic parser inference enabled: False

=== OLD TARGET AFTER CORRECTION ===
Old Target: target-evidence:73bbf40cdd65405711a21f389d93d5f9
Effective speakers: ['Dr. Komeil Roudi']
True multi-speaker: False
Retarget required: True

=== SUPERSEDED BUT PRESERVED QA ===
Question: قبل از سرمایه‌گذاری چه چیزهایی باید بررسی شود و افراد غیرمتخصص بهتر است چگونه وارد بازار شوند؟
Answer: قبل از سرمایه‌گذاری باید نوع ابزار، سطح ریسک و

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 68 — Pilot #14 Replacement Target + Question/Answer ACCEPT + Citation Draft

Human Reviewer، Candidate 6 را برای Retarget پذیرفت.

این Candidate یک ساختار واقعی دوگوینده دارد:

- مجری سؤال را مطرح می‌کند؛
- Dr. Komeil Roudi پاسخ اصلی را ارائه می‌دهد.

سؤال پذیرفته‌شده:

**در یک کسب‌وکار خانگی، چرا باید پول کسب‌وکار از پول خانواده جدا شود و این تفکیک چگونه انجام می‌شود؟**

جواب پذیرفته‌شده:

**چون مخلوط‌شدن پول کسب‌وکار و خانواده باعث می‌شود سود و زیان واقعی کسب‌وکار مشخص نباشد و حتی امنیت مالی خانواده هم به خطر بیفتد. باید هزینه‌های تولید جداگانه ثبت شوند، حساب و پول کسب‌وکار از حساب خانه جدا باشد، برای صاحب کسب‌وکار حقوق مشخصی در نظر گرفته شود و در پایان بررسی شود که پس از همه هزینه‌ها واقعاً سودی باقی مانده است یا نه.**

در این مرحله:

- Target قبلی با Candidate 6 جایگزین می‌شود؛
- ساختار واقعی Multi-Speaker حفظ می‌شود؛
- سؤال مجری فقط Context است؛
- پاسخ دکتر رودی Evidence اصلی جواب است؛
- Question و Expected Answer هر دو با تصمیم `ACCEPT` ثبت می‌شوند؛
- Citation وارد Human Review می‌شود؛
- Queue همچنان 10 باقی می‌ماند؛
- Full 572 Generation همچنان Blocked است.

</div>

In [68]:
# Code Cell 68 — COMPLETE CORRECTED REPLACEMENT

cell68_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell67_gate") is not True:
    cell68_failures.append(
        "CELL67_NOT_PASS"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell68_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


if not callable(
    globals().get(
        "cell58_collect_prefixed_ids"
    )
):
    cell68_failures.append(
        "PREFIX_ID_HELPER_MISSING"
    )


# ============================================================
# 2) Human-selected Candidate + accepted QA
# ============================================================

CELL68_SELECTED_CANDIDATE_ID = (
    "multi-speaker-retarget-candidate:"
    "6046f784fb3ef3aeade5d3d272d40d6f"
)


CELL68_ACCEPTED_QUESTION = (
    "در یک کسب‌وکار خانگی، چرا باید پول کسب‌وکار از پول خانواده "
    "جدا شود و این تفکیک چگونه انجام می‌شود؟"
)


CELL68_ACCEPTED_ANSWER = (
    "چون مخلوط‌شدن پول کسب‌وکار و خانواده باعث می‌شود سود و زیان "
    "واقعی کسب‌وکار مشخص نباشد و حتی امنیت مالی خانواده هم به خطر "
    "بیفتد. باید هزینه‌های تولید جداگانه ثبت شوند، حساب و پول "
    "کسب‌وکار از حساب خانه جدا باشد، برای صاحب کسب‌وکار حقوق مشخصی "
    "در نظر گرفته شود و در پایان بررسی شود که پس از همه هزینه‌ها "
    "واقعاً سودی باقی مانده است یا نه."
)


CELL68_QUESTION_HUMAN_NOTE = (
    "Human Reviewer accepted the retargeted multi-speaker question."
)


CELL68_ANSWER_HUMAN_NOTE = (
    "Human Reviewer accepted the retargeted answer."
)


CELL68_CONTINUITY_BASIS = (
    "HUMAN_CERTIFIED_SPEAKER_CONTINUITY"
)


# ============================================================
# 3) Resolve selected Candidate
# ============================================================

cell68_selected_matches = [
    candidate
    for candidate
    in cell67_top_candidates
    if (
        candidate.get(
            "candidate_id"
        )
        ==
        CELL68_SELECTED_CANDIDATE_ID
    )
]


cell68_candidate_unique = (
    len(
        cell68_selected_matches
    )
    ==
    1
)


cell68_selected_candidate = (
    copy.deepcopy(
        cell68_selected_matches[
            0
        ]
    )
    if cell68_candidate_unique
    else {}
)


cell68_candidate_valid = all([
    cell68_candidate_unique,

    cell68_selected_candidate.get(
        "episode_number"
    )
    ==
    14,

    cell68_selected_candidate.get(
        "direction"
    )
    ==
    "OTHER_TO_DR",

    cell68_selected_candidate.get(
        "question_signal"
    )
    is True,

    cell68_selected_candidate.get(
        "left",
        {},
    ).get(
        "speaker"
    )
    ==
    "مجری",

    cell68_selected_candidate.get(
        "right",
        {},
    ).get(
        "speaker"
    )
    ==
    "Dr. Komeil Roudi",

    cell68_selected_candidate.get(
        "right",
        {},
    ).get(
        "speaker_is_primary_dr_roudi"
    )
    is True,
])


if not cell68_candidate_valid:
    cell68_failures.append(
        "SELECTED_CANDIDATE_INVALID"
    )


# ============================================================
# 4) Extract source records
# ============================================================

cell68_host_records = copy.deepcopy(
    cell68_selected_candidate.get(
        "left",
        {},
    ).get(
        "records",
        [],
    )
)


cell68_dr_records = copy.deepcopy(
    cell68_selected_candidate.get(
        "right",
        {},
    ).get(
        "records",
        [],
    )
)


cell68_host_indices = [
    record.get(
        "utterance_index"
    )
    for record
    in cell68_host_records
]


cell68_dr_indices = [
    record.get(
        "utterance_index"
    )
    for record
    in cell68_dr_records
]


# ============================================================
# 5) Validate true speaker-turn structure
# ============================================================

cell68_first_dr_record = (
    cell68_dr_records[
        0
    ]
    if cell68_dr_records
    else {}
)


cell68_first_dr_explicit = all([
    cell68_first_dr_record.get(
        "utterance_index"
    )
    ==
    38,

    cell68_first_dr_record.get(
        "base_status"
    )
    ==
    "EXPLICIT_SOURCE_MARKER",

    cell68_first_dr_record.get(
        "base_raw"
    )
    ==
    "دکتر رودی",

    cell68_first_dr_record.get(
        "effective_canonical"
    )
    ==
    "Dr. Komeil Roudi",

    cell68_first_dr_record.get(
        "effective_primary"
    )
    is True,
])


cell68_continuation_records = (
    cell68_dr_records[
        1:
    ]
)


cell68_no_new_explicit_speaker_inside_dr_turn = all(
    (
        record.get(
            "base_status"
        )
        !=
        "EXPLICIT_SOURCE_MARKER"
    )
    or
    (
        record.get(
            "base_raw"
        )
        ==
        "دکتر رودی"
    )
    for record
    in cell68_continuation_records
)


cell68_source_records_valid = all([
    cell68_host_indices
    ==
    [
        37
    ],

    cell68_dr_indices
    ==
    list(
        range(
            38,
            48,
        )
    ),

    cell68_selected_candidate.get(
        "left",
        {},
    ).get(
        "speaker"
    )
    ==
    "مجری",

    cell68_selected_candidate.get(
        "right",
        {},
    ).get(
        "speaker"
    )
    ==
    "Dr. Komeil Roudi",

    cell68_selected_candidate.get(
        "right",
        {},
    ).get(
        "speaker_is_primary_dr_roudi"
    )
    is True,

    cell68_first_dr_explicit,

    cell68_no_new_explicit_speaker_inside_dr_turn,
])


if not cell68_source_records_valid:
    cell68_failures.append(
        "SOURCE_RECORDS_INVALID"
    )


# ============================================================
# 6) Human-certified continuity policy for this Target
# ============================================================

cell68_human_continuity_policy = {
    "policy_name":
        "EXPLICIT_SPEAKER_CONTINUITY_UNTIL_NEXT_EXPLICIT_LABEL",

    "policy_source":
        "HUMAN_REVIEW_CELL67",

    "rule":
        (
            "An explicitly labeled speaker remains the speaker of "
            "subsequent unlabeled continuation paragraphs until a "
            "new explicit speaker label appears."
        ),

    "primary_speaker":
        "Dr. Komeil Roudi",

    "explicit_start_utterance":
        38,

    "continuation_utterances":
        list(
            range(
                39,
                48,
            )
        ),

    "canonical_raw_source_mutated":
        False,

    "parser_output_mutated":
        False,

    "human_overlay_only":
        True,

    "automatic_unknown_speaker_inference_enabled":
        False,
}


# ============================================================
# 7) Build exact Replacement Target parts
# ============================================================

cell68_replacement_target_parts = []


for record in (
    cell68_host_records
    +
    cell68_dr_records
):

    utterance_index = record.get(
        "utterance_index"
    )


    is_host = (
        utterance_index
        ==
        37
    )


    is_dr = (
        38
        <=
        utterance_index
        <=
        47
    )


    source_text = (
        record.get(
            "source_text"
        )
        or
        ""
    )


    raw_speaker_state = {
        "base_status":
            record.get(
                "base_status"
            ),

        "base_raw":
            record.get(
                "base_raw"
            ),

        "effective_label":
            record.get(
                "effective_label"
            ),

        "effective_canonical":
            record.get(
                "effective_canonical"
            ),

        "effective_primary":
            record.get(
                "effective_primary"
            ),
    }


    if is_host:

        human_attribution_overlay = {
            "status":
                "SOURCE_EXPLICIT",

            "effective_label":
                "مجری",

            "effective_canonical":
                None,

            "effective_primary":
                False,

            "basis":
                "EXPLICIT_SOURCE_MARKER",

            "raw_speaker_state_preserved":
                True,
        }


    elif is_dr and utterance_index == 38:

        human_attribution_overlay = {
            "status":
                "SOURCE_EXPLICIT",

            "effective_label":
                "دکتر رودی",

            "effective_canonical":
                "Dr. Komeil Roudi",

            "effective_primary":
                True,

            "basis":
                "EXPLICIT_SOURCE_MARKER",

            "raw_speaker_state_preserved":
                True,
        }


    elif is_dr:

        human_attribution_overlay = {
            "status":
                "HUMAN_CERTIFIED",

            "effective_label":
                "دکتر رودی",

            "effective_canonical":
                "Dr. Komeil Roudi",

            "effective_primary":
                True,

            "basis":
                CELL68_CONTINUITY_BASIS,

            "continuation_from_utterance":
                38,

            "raw_speaker_state_preserved":
                True,
        }


    else:

        human_attribution_overlay = {
            "status":
                "UNRESOLVED",

            "effective_label":
                None,

            "effective_canonical":
                None,

            "effective_primary":
                None,

            "basis":
                None,

            "raw_speaker_state_preserved":
                True,
        }


    cell68_replacement_target_parts.append({
        "document_id":
            record.get(
                "utterance_id"
            ),

        "source_type":
            "transcript",

        "role":
            (
                "ATTRIBUTION_CONTRAST_LEFT"
                if is_host
                else
                "ATTRIBUTION_CONTRAST_RIGHT"
            ),

        "text":
            source_text,

        "text_sha256":
            hashlib.sha256(
                source_text.encode(
                    "utf-8"
                )
            ).hexdigest(),

        "char_start":
            0,

        "char_end":
            len(
                source_text
            ),

        "locator":
            copy.deepcopy(
                record.get(
                    "source_locator"
                )
                or
                {}
            ),

        "speaker_state":
            raw_speaker_state,

        "human_attribution_overlay":
            human_attribution_overlay,

        "attribution": {
            "speaker":
                (
                    "Dr. Komeil Roudi"
                    if is_dr
                    else None
                ),

            "attribution_basis":
                (
                    human_attribution_overlay.get(
                        "basis"
                    )
                ),
        },

        "utterance_index":
            utterance_index,
    })


cell68_combined_target_text = (
    "\n".join(
        part.get(
            "text",
            "",
        )
        for part
        in cell68_replacement_target_parts
    )
)


# ============================================================
# 8) Effective speaker helper
# ============================================================

def cell68_effective_speaker(
    part,
):

    overlay = (
        part.get(
            "human_attribution_overlay"
        )
        or
        {}
    )


    raw_state = (
        part.get(
            "speaker_state"
        )
        or
        {}
    )


    canonical = (
        overlay.get(
            "effective_canonical"
        )
        or
        raw_state.get(
            "effective_canonical"
        )
    )


    label = (
        overlay.get(
            "effective_label"
        )
        or
        raw_state.get(
            "effective_label"
        )
        or
        raw_state.get(
            "base_raw"
        )
    )


    overlay_primary = overlay.get(
        "effective_primary"
    )


    raw_primary = raw_state.get(
        "effective_primary"
    )


    if overlay_primary is not None:

        primary = overlay_primary

    else:

        primary = raw_primary


    return {
        "canonical":
            canonical,

        "label":
            label,

        "primary":
            primary,

        "basis":
            (
                overlay.get(
                    "basis"
                )
                or
                "BASE_PARSER_ATTRIBUTION"
            ),
    }


cell68_effective_speaker_states = [
    cell68_effective_speaker(
        part
    )
    for part
    in cell68_replacement_target_parts
]


# ============================================================
# 9) Validate Replacement Target speaker boundary
# ============================================================

cell68_host_effective = (
    cell68_effective_speaker_states[
        0
    ]
    if cell68_effective_speaker_states
    else {}
)


cell68_dr_effective_states = (
    cell68_effective_speaker_states[
        1:
    ]
)


cell68_target_parts_valid = all([
    len(
        cell68_replacement_target_parts
    )
    ==
    11,

    [
        part.get(
            "utterance_index"
        )
        for part
        in cell68_replacement_target_parts
    ]
    ==
    list(
        range(
            37,
            48,
        )
    ),

    cell68_replacement_target_parts[
        0
    ].get(
        "role"
    )
    ==
    "ATTRIBUTION_CONTRAST_LEFT",

    all(
        part.get(
            "role"
        )
        ==
        "ATTRIBUTION_CONTRAST_RIGHT"
        for part
        in cell68_replacement_target_parts[
            1:
        ]
    ),

    cell68_host_effective.get(
        "label"
    )
    ==
    "مجری",

    all(
        state.get(
            "canonical"
        )
        ==
        "Dr. Komeil Roudi"
        for state
        in cell68_dr_effective_states
    ),

    all(
        state.get(
            "primary"
        )
        is True
        for state
        in cell68_dr_effective_states
    ),
])


if not cell68_target_parts_valid:
    cell68_failures.append(
        "REPLACEMENT_TARGET_PARTS_INVALID"
    )


# ============================================================
# 10) Deterministic Replacement Target ID
# ============================================================

cell68_target_identity_payload = {
    "identity_version":
        "multi_speaker_retarget_target_v2_human_continuity",

    "pilot_case_id":
        CELL67_PILOT_ID,

    "old_target_evidence_id":
        CELL67_OLD_TARGET_EVIDENCE_ID,

    "selected_candidate_id":
        CELL68_SELECTED_CANDIDATE_ID,

    "utterance_projection": [
        {
            "document_id":
                part.get(
                    "document_id"
                ),

            "role":
                part.get(
                    "role"
                ),

            "text_sha256":
                part.get(
                    "text_sha256"
                ),

            "human_attribution_overlay":
                copy.deepcopy(
                    part.get(
                        "human_attribution_overlay"
                    )
                ),
        }
        for part
        in cell68_replacement_target_parts
    ],
}


cell68_replacement_target_evidence_id = (
    "target-evidence:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell68_target_identity_payload
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


cell68_replacement_target_evidence = {
    "record_version":
        "replacement_target_evidence_v2_human_continuity",

    "target_evidence_id":
        cell68_replacement_target_evidence_id,

    "pilot_case_id":
        CELL67_PILOT_ID,

    "golden_case_id":
        CELL67_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL67_TASK_DRAFT_ID,

    "source_type":
        "transcript",

    "stratum":
        "multi_speaker_chehelston",

    "task_family":
        "MULTI_SPEAKER_ATTRIBUTION_CHEHELSTON",

    "question_archetype":
        "ATTRIBUTION_BOUNDARY",

    "authority_expectation":
        "UTTERANCE_LEVEL_AUTHORITY",

    "selected_candidate_id":
        CELL68_SELECTED_CANDIDATE_ID,

    "target_parts":
        copy.deepcopy(
            cell68_replacement_target_parts
        ),

    "combined_target_text":
        cell68_combined_target_text,

    "human_attribution_policy":
        copy.deepcopy(
            cell68_human_continuity_policy
        ),

    "retarget_lineage": {
        "old_target_evidence_id":
            CELL67_OLD_TARGET_EVIDENCE_ID,

        "old_target_human_decision":
            "RETARGET",

        "replacement_target_evidence_id":
            cell68_replacement_target_evidence_id,

        "replacement_human_decision":
            "ACCEPT",
    },

    "canonical_raw_source_mutated":
        False,

    "parser_output_mutated":
        False,

    "external_lookup_used":
        False,
}


# ============================================================
# 11) Rebuild Pilot packet with Replacement Target
# ============================================================

def cell68_replace_target_reference(
    value,
):

    if isinstance(
        value,
        dict,
    ):

        output = {}


        for key, item in value.items():

            if key == "target_text_parts":

                output[
                    key
                ] = copy.deepcopy(
                    cell68_replacement_target_parts
                )


            elif key == "target_evidence_id":

                output[
                    key
                ] = (
                    cell68_replacement_target_evidence_id
                )


            else:

                output[
                    key
                ] = cell68_replace_target_reference(
                    item
                )


        return output


    if isinstance(
        value,
        list,
    ):

        return [
            cell68_replace_target_reference(
                item
            )
            for item
            in value
        ]


    if (
        isinstance(
            value,
            str,
        )
        and
        value
        ==
        CELL67_OLD_TARGET_EVIDENCE_ID
    ):

        return (
            cell68_replacement_target_evidence_id
        )


    return copy.deepcopy(
        value
    )


cell68_retargeted_pilot_packet = (
    cell68_replace_target_reference(
        cell64_next_pilot_packet
    )
)


# ============================================================
# 12) Packet Target ID validation
# ============================================================

cell68_packet_target_ids = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell68_retargeted_pilot_packet,
            "target-evidence:",
        )
    )
)


cell68_old_target_still_present = (
    CELL67_OLD_TARGET_EVIDENCE_ID
    in
    cell68_packet_target_ids
)


cell68_packet_valid = all([
    cell68_packet_target_ids
    ==
    [
        cell68_replacement_target_evidence_id
    ],

    cell68_old_target_still_present
    is False,
])


if not cell68_packet_valid:
    cell68_failures.append(
        "RETARGETED_PACKET_ID_VALIDATION_FAILED"
    )


# ============================================================
# 13) Grounding validation
# ============================================================

def cell68_norm(
    text,
):

    if not isinstance(
        text,
        str,
    ):
        return ""


    value = (
        text
        .replace(
            "ي",
            "ی",
        )
        .replace(
            "ك",
            "ک",
        )
        .replace(
            "\u200c",
            "",
        )
    )


    value = re.sub(
        r"\s+",
        " ",
        value,
    )


    return value.strip()


cell68_source_surface = cell68_norm(
    cell68_combined_target_text
)


cell68_grounding = {
    "mixed_money_problem":
        (
            "پول کسبوکار و پول خانواده"
            in
            cell68_source_surface
        ),

    "profit_visibility":
        (
            "سودآور است یا نه"
            in
            cell68_source_surface
            or
            "سود و زیان واقعی"
            in
            cell68_source_surface
        ),

    "production_costs":
        (
            "هزینه تولید"
            in
            cell68_source_surface
        ),

    "financial_separation":
        (
            "تفکیک مالی"
            in
            cell68_source_surface
        ),

    "separate_accounts":
        (
            "حساب و پول کسبوکار را از حساب خانه جدا کنید"
            in
            cell68_source_surface
        ),

    "owner_salary":
        (
            "حقوق مشخص"
            in
            cell68_source_surface
        ),

    "year_end_profit":
        (
            "پایان سال مالی"
            in
            cell68_source_surface
        ),

    "family_security":
        (
            "امنیت مالی خانواده"
            in
            cell68_source_surface
        ),
}


cell68_grounding_pass = all(
    cell68_grounding.values()
)


if not cell68_grounding_pass:
    cell68_failures.append(
        "ANSWER_GROUNDING_VALIDATION_FAILED"
    )


# ============================================================
# 14) Restart Engine
# ============================================================

cell68_question_pending_session = (
    cell13_build_session(
        cell68_retargeted_pilot_packet
    )
)


cell68_session_start_valid = (
    cell68_question_pending_session.get(
        "current_state"
    )
    ==
    "QUESTION_DRAFT_PENDING"
)


if not cell68_session_start_valid:
    cell68_failures.append(
        "ENGINE_SESSION_START_INVALID"
    )


# ============================================================
# 15) Question Draft + Human ACCEPT
# ============================================================

cell68_question_identity = {
    "identity_version":
        "question_draft_identity_v1",

    "pilot_case_id":
        CELL67_PILOT_ID,

    "target_evidence_id":
        cell68_replacement_target_evidence_id,

    "question_text":
        CELL68_ACCEPTED_QUESTION,

    "revision_number":
        1,
}


cell68_question_draft_id = (
    "question-draft:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell68_question_identity
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


cell68_question_record = {
    "record_version":
        "question_draft_record_v1",

    "question_draft_id":
        cell68_question_draft_id,

    "pilot_case_id":
        CELL67_PILOT_ID,

    "golden_case_id":
        CELL67_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL67_TASK_DRAFT_ID,

    "target_evidence_id":
        cell68_replacement_target_evidence_id,

    "question_text":
        CELL68_ACCEPTED_QUESTION,

    "revision_number":
        1,

    "machine_validation_status":
        "PASS_PENDING_HUMAN_ADJUDICATION",

    "human_adjudication_status":
        "PENDING",

    "external_knowledge_used":
        False,
}


cell68_question_review_session = (
    cell13_apply_event(
        cell68_question_pending_session,
        "QUESTION_DRAFT_REGISTERED",
        artifact=
            cell68_question_record,
    )
)


cell68_answer_pending_session = (
    cell13_apply_event(
        cell68_question_review_session,
        "QUESTION_ACCEPTED",
        human_note=
            CELL68_QUESTION_HUMAN_NOTE,
    )
)


cell68_question_accept_valid = all([
    cell68_question_review_session.get(
        "current_state"
    )
    ==
    "QUESTION_HUMAN_REVIEW_PENDING",

    cell68_answer_pending_session.get(
        "current_state"
    )
    ==
    "EXPECTED_ANSWER_DRAFT_PENDING",
])


if not cell68_question_accept_valid:
    cell68_failures.append(
        "QUESTION_ACCEPT_TRANSITION_INVALID"
    )


# ============================================================
# 16) Expected Answer + Human ACCEPT
# ============================================================

cell68_answer_identity = {
    "identity_version":
        "expected_answer_identity_v1",

    "pilot_case_id":
        CELL67_PILOT_ID,

    "question_draft_id":
        cell68_question_draft_id,

    "target_evidence_id":
        cell68_replacement_target_evidence_id,

    "answer_text":
        CELL68_ACCEPTED_ANSWER,

    "revision_number":
        1,
}


cell68_expected_answer_draft_id = (
    "expected-answer-draft:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell68_answer_identity
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


cell68_expected_answer_record = {
    "record_version":
        "expected_answer_draft_record_v1",

    "expected_answer_draft_id":
        cell68_expected_answer_draft_id,

    "pilot_case_id":
        CELL67_PILOT_ID,

    "golden_case_id":
        CELL67_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL67_TASK_DRAFT_ID,

    "question_draft_id":
        cell68_question_draft_id,

    "target_evidence_id":
        cell68_replacement_target_evidence_id,

    "expected_answer_text":
        CELL68_ACCEPTED_ANSWER,

    "revision_number":
        1,

    "machine_validation_status":
        "PASS_PENDING_HUMAN_ADJUDICATION",

    "human_adjudication_status":
        "PENDING",

    "external_knowledge_used":
        False,
}


cell68_answer_review_session = (
    cell13_apply_event(
        cell68_answer_pending_session,
        "EXPECTED_ANSWER_REGISTERED",
        artifact=
            cell68_expected_answer_record,
    )
)


cell68_citation_pending_session = (
    cell13_apply_event(
        cell68_answer_review_session,
        "EXPECTED_ANSWER_ACCEPTED",
        human_note=
            CELL68_ANSWER_HUMAN_NOTE,
    )
)


cell68_answer_accept_valid = all([
    cell68_answer_review_session.get(
        "current_state"
    )
    ==
    "EXPECTED_ANSWER_HUMAN_REVIEW_PENDING",

    cell68_citation_pending_session.get(
        "current_state"
    )
    ==
    "CITATION_MAPPING_PENDING",
])


if not cell68_answer_accept_valid:
    cell68_failures.append(
        "ANSWER_ACCEPT_TRANSITION_INVALID"
    )


# ============================================================
# 17) Build Citation entries
# ============================================================

cell68_citation_entries = []


for index, part in enumerate(
    cell68_replacement_target_parts
):

    effective_state = cell68_effective_speaker(
        part
    )


    is_host = (
        part.get(
            "utterance_index"
        )
        ==
        37
    )


    entry = copy.deepcopy(
        part
    )


    entry[
        "citation_index"
    ] = index


    entry[
        "target_evidence_id"
    ] = (
        cell68_replacement_target_evidence_id
    )


    entry[
        "locator_origin"
    ] = (
        "COPIED_FROM_REPLACEMENT_TARGET_EVIDENCE"
    )


    entry[
        "external_locator_lookup_used"
    ] = False


    entry[
        "answer_support"
    ] = not is_host


    entry[
        "answer_support_role"
    ] = (
        "CONTEXT_QUESTION_ONLY"
        if is_host
        else
        "PRIMARY_ANSWER_EVIDENCE"
    )


    entry[
        "authority_role"
    ] = (
        "NON_PRIMARY_SPEAKER_CONTEXT"
        if is_host
        else
        "PRIMARY_CORPUS_AUTHORITY"
    )


    entry[
        "use_in_expected_answer_grounding"
    ] = not is_host


    entry[
        "resolved_authority"
    ] = {
        "effective_label":
            effective_state.get(
                "label"
            ),

        "effective_canonical":
            effective_state.get(
                "canonical"
            ),

        "effective_primary":
            effective_state.get(
                "primary"
            ),

        "basis":
            effective_state.get(
                "basis"
            ),
    }


    cell68_citation_entries.append(
        entry
    )


cell68_primary_entries = [
    entry
    for entry
    in cell68_citation_entries
    if entry.get(
        "answer_support"
    )
    is True
]


cell68_context_entries = [
    entry
    for entry
    in cell68_citation_entries
    if entry.get(
        "answer_support"
    )
    is False
]


# ============================================================
# 18) Attribution boundary validation
# ============================================================

cell68_citation_boundary_valid = all([
    len(
        cell68_context_entries
    )
    ==
    1,

    len(
        cell68_primary_entries
    )
    ==
    10,

    cell68_context_entries[
        0
    ].get(
        "utterance_index"
    )
    ==
    37,

    cell68_context_entries[
        0
    ].get(
        "answer_support_role"
    )
    ==
    "CONTEXT_QUESTION_ONLY",

    cell68_context_entries[
        0
    ].get(
        "use_in_expected_answer_grounding"
    )
    is False,

    all(
        entry.get(
            "resolved_authority",
            {},
        ).get(
            "effective_canonical"
        )
        ==
        "Dr. Komeil Roudi"
        for entry
        in cell68_primary_entries
    ),

    all(
        entry.get(
            "resolved_authority",
            {},
        ).get(
            "effective_primary"
        )
        is True
        for entry
        in cell68_primary_entries
    ),

    all(
        entry.get(
            "authority_role"
        )
        ==
        "PRIMARY_CORPUS_AUTHORITY"
        for entry
        in cell68_primary_entries
    ),

    all(
        entry.get(
            "use_in_expected_answer_grounding"
        )
        is True
        for entry
        in cell68_primary_entries
    ),
])


if not cell68_citation_boundary_valid:
    cell68_failures.append(
        "CITATION_ATTRIBUTION_BOUNDARY_INVALID"
    )


# ============================================================
# 19) Citation no-fabrication validation
# ============================================================

def cell68_strip_citation_fields(
    entry,
):

    output = copy.deepcopy(
        entry
    )


    for key in [
        "citation_index",
        "target_evidence_id",
        "locator_origin",
        "external_locator_lookup_used",
        "answer_support",
        "answer_support_role",
        "authority_role",
        "use_in_expected_answer_grounding",
        "resolved_authority",
    ]:

        output.pop(
            key,
            None,
        )


    return output


cell68_citation_back_projection = [
    cell68_strip_citation_fields(
        entry
    )
    for entry
    in cell68_citation_entries
]


cell68_no_locator_fabrication = (
    cell68_citation_back_projection
    ==
    cell68_replacement_target_parts
)


if not cell68_no_locator_fabrication:
    cell68_failures.append(
        "CITATION_LOCATOR_FABRICATION_DETECTED"
    )


# ============================================================
# 20) Citation Mapping
# ============================================================

cell68_citation_identity = {
    "identity_version":
        "citation_mapping_identity_v3_human_continuity",

    "pilot_case_id":
        CELL67_PILOT_ID,

    "question_draft_id":
        cell68_question_draft_id,

    "expected_answer_draft_id":
        cell68_expected_answer_draft_id,

    "target_evidence_id":
        cell68_replacement_target_evidence_id,

    "citation_entries":
        copy.deepcopy(
            cell68_citation_entries
        ),
}


cell68_citation_mapping_id = (
    "citation-mapping:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell68_citation_identity
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


cell68_citation_mapping_record = {
    "record_version":
        "citation_mapping_record_v3_human_continuity",

    "citation_mapping_id":
        cell68_citation_mapping_id,

    "pilot_case_id":
        CELL67_PILOT_ID,

    "golden_case_id":
        CELL67_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL67_TASK_DRAFT_ID,

    "target_evidence_id":
        cell68_replacement_target_evidence_id,

    "question_draft_id":
        cell68_question_draft_id,

    "expected_answer_draft_id":
        cell68_expected_answer_draft_id,

    "citation_entries":
        copy.deepcopy(
            cell68_citation_entries
        ),

    "citation_scope":
        "MULTI_SPEAKER_ATTRIBUTION_BOUNDARY",

    "primary_answer_evidence_count":
        len(
            cell68_primary_entries
        ),

    "context_only_count":
        len(
            cell68_context_entries
        ),

    "primary_authority":
        "Dr. Komeil Roudi",

    "host_used_as_answer_evidence":
        False,

    "human_speaker_continuity_used":
        True,

    "human_speaker_continuity_policy":
        copy.deepcopy(
            cell68_human_continuity_policy
        ),

    "retarget_used":
        True,

    "old_target_evidence_id":
        CELL67_OLD_TARGET_EVIDENCE_ID,

    "external_locator_lookup_used":
        False,

    "canonical_raw_source_mutated":
        False,

    "parser_output_mutated":
        False,

    "machine_validation_status":
        "PASS_PENDING_HUMAN_ADJUDICATION",

    "human_adjudication_status":
        "PENDING",
}


cell68_current_session = (
    cell13_apply_event(
        cell68_citation_pending_session,
        "CITATION_MAPPING_REGISTERED",
        artifact=
            cell68_citation_mapping_record,
    )
)


cell68_citation_registered_valid = (
    cell68_current_session.get(
        "current_state"
    )
    ==
    "CITATION_HUMAN_REVIEW_PENDING"
)


if not cell68_citation_registered_valid:
    cell68_failures.append(
        "CITATION_REGISTRATION_TRANSITION_INVALID"
    )


# ============================================================
# 21) Update active Human Review workspace
# ============================================================

cell68_active_workspace = copy.deepcopy(
    cell67_persisted_active_case
)


cell68_active_workspace[
    "status"
] = "ACTIVE_HUMAN_REVIEW"


cell68_active_workspace[
    "workflow_state"
] = "CITATION_HUMAN_REVIEW_PENDING"


cell68_active_workspace[
    "target_evidence_id"
] = (
    cell68_replacement_target_evidence_id
)


cell68_active_workspace[
    "target_evidence"
] = copy.deepcopy(
    cell68_replacement_target_evidence
)


cell68_active_workspace[
    "source_evidence"
] = copy.deepcopy(
    cell68_replacement_target_parts
)


cell68_active_workspace[
    "combined_target_text"
] = (
    cell68_combined_target_text
)


cell68_active_workspace[
    "human_attribution_policy"
] = copy.deepcopy(
    cell68_human_continuity_policy
)


cell68_active_workspace.setdefault(
    "retarget_review",
    {},
)


cell68_active_workspace[
    "retarget_review"
][
    "selected_candidate_id"
] = CELL68_SELECTED_CANDIDATE_ID


cell68_active_workspace[
    "retarget_review"
][
    "replacement_target_evidence_id"
] = cell68_replacement_target_evidence_id


cell68_active_workspace[
    "retarget_review"
][
    "replacement_status"
] = "ACCEPT"


cell68_active_workspace[
    "question_review"
] = {
    "question_draft_id":
        cell68_question_draft_id,

    "question_text":
        CELL68_ACCEPTED_QUESTION,

    "human_decision":
        "ACCEPT",

    "human_note":
        CELL68_QUESTION_HUMAN_NOTE,
}


cell68_active_workspace[
    "expected_answer_review"
] = {
    "expected_answer_draft_id":
        cell68_expected_answer_draft_id,

    "expected_answer_text":
        CELL68_ACCEPTED_ANSWER,

    "human_decision":
        "ACCEPT",

    "human_note":
        CELL68_ANSWER_HUMAN_NOTE,
}


cell68_active_workspace[
    "citation_review"
] = {
    "citation_mapping_id":
        cell68_citation_mapping_id,

    "citation_mapping_record":
        copy.deepcopy(
            cell68_citation_mapping_record
        ),

    "citation_entries":
        copy.deepcopy(
            cell68_citation_entries
        ),

    "machine_validation_status":
        "PASS_PENDING_HUMAN_ADJUDICATION",

    "human_decision":
        "PENDING",

    "primary_answer_authority":
        "Dr. Komeil Roudi",

    "host_used_as_answer_evidence":
        False,

    "human_speaker_continuity_used":
        True,
}


# ============================================================
# 22) Continuation state
# ============================================================

cell68_continuation_core = {
    "next_pilot_id":
        CELL67_PILOT_ID,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell64_remaining_pilot_ids
        ),

    "remaining_count":
        10,

    "active_pilot_status":
        "CITATION_HUMAN_REVIEW_PENDING",

    "active_target_evidence_id":
        cell68_replacement_target_evidence_id,

    "active_citation_mapping_id":
        cell68_citation_mapping_id,

    "retarget_status":
        "REPLACEMENT_ACCEPTED",

    "human_speaker_continuity":
        True,

    "last_completed_pilot_id":
        cell64_continuation_state.get(
            "last_completed_pilot_id"
        ),

    "last_completion_type":
        cell64_continuation_state.get(
            "last_completion_type"
        ),
}


cell68_continuation_state_sha256 = (
    cell10_stable_sha(
        cell68_continuation_core
    )
)


cell68_continuation_state = {
    **copy.deepcopy(
        cell68_continuation_core
    ),

    "continuation_state_sha256":
        cell68_continuation_state_sha256,
}


# ============================================================
# 23) Persist Registry
# ============================================================

cell68_persistent_registry_payload = copy.deepcopy(
    cell67_persisted_registry
)


cell68_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell68_active_workspace
)


cell68_persistent_registry_payload[
    "remaining_pilot_count"
] = 10


cell68_persistent_registry_payload[
    "continuation_state"
] = copy.deepcopy(
    cell68_continuation_state
)


cell68_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell68_persistent_registry_payload
    )
)


# ============================================================
# 24) Read-back validation
# ============================================================

cell68_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell68_persisted_active_case = (
    cell68_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell68_persistence_valid = all([
    cell68_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL67_PILOT_ID,

    cell68_persisted_active_case.get(
        "target_evidence_id"
    )
    ==
    cell68_replacement_target_evidence_id,

    cell68_persisted_active_case.get(
        "workflow_state"
    )
    ==
    "CITATION_HUMAN_REVIEW_PENDING",

    cell68_persisted_active_case.get(
        "retarget_review",
        {},
    ).get(
        "replacement_status"
    )
    ==
    "ACCEPT",

    cell68_persisted_active_case.get(
        "question_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell68_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell68_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "PENDING",

    cell68_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "human_speaker_continuity_used"
    )
    is True,

    cell68_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    10,

    cell68_persisted_registry.get(
        "continuation_state",
        {},
    ).get(
        "continuation_state_sha256"
    )
    ==
    cell68_continuation_state_sha256,
])


if not cell68_persistence_valid:
    cell68_failures.append(
        "REGISTRY_READBACK_INVALID"
    )


# ============================================================
# 25) Safety
# ============================================================

cell68_queue_still_10 = all([
    len(
        cell64_remaining_pilot_ids
    )
    ==
    10,

    cell64_remaining_pilot_ids[
        0
    ]
    ==
    CELL67_PILOT_ID,
])


cell68_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell68_full_572_blocked = (
    cell67_full_572_blocked
    is True
)


# ============================================================
# 26) Internal Gate report
# ============================================================

cell68_gate_checks = {
    "no_failures":
        len(
            cell68_failures
        )
        ==
        0,

    "candidate_valid":
        cell68_candidate_valid,

    "source_records_valid":
        cell68_source_records_valid,

    "target_parts_valid":
        cell68_target_parts_valid,

    "packet_valid":
        cell68_packet_valid,

    "grounding_pass":
        cell68_grounding_pass,

    "session_start_valid":
        cell68_session_start_valid,

    "question_accept_valid":
        cell68_question_accept_valid,

    "answer_accept_valid":
        cell68_answer_accept_valid,

    "citation_boundary_valid":
        cell68_citation_boundary_valid,

    "no_locator_fabrication":
        cell68_no_locator_fabrication,

    "citation_registered_valid":
        cell68_citation_registered_valid,

    "persistence_valid":
        cell68_persistence_valid,

    "queue_still_10":
        cell68_queue_still_10,

    "base_seed_unchanged":
        cell68_base_seed_unchanged,

    "full_572_blocked":
        cell68_full_572_blocked,
}


cell68_gate = all(
    cell68_gate_checks.values()
)


# ============================================================
# 27) Diagnostics
# ============================================================

print(
    "=== PILOT #14 RETARGET ACCEPT + QA ACCEPT + CITATION DRAFT ==="
)


print(
    "\n=== RETARGET ==="
)

print(
    "Old Target:",
    CELL67_OLD_TARGET_EVIDENCE_ID,
)

print(
    "Selected Candidate:",
    CELL68_SELECTED_CANDIDATE_ID,
)

print(
    "Replacement Target:",
    cell68_replacement_target_evidence_id,
)

print(
    "Episode:",
    14,
)

print(
    "True Multi-Speaker:",
    True,
)


print(
    "\n=== SPEAKER CONTINUITY ==="
)

print(
    "Host:",
    "Utterance 37",
)

print(
    "Explicit Dr. Roudi start:",
    "Utterance 38",
)

print(
    "Human-certified Dr. continuation:",
    "Utterances 39..47",
)

print(
    "Raw Parser mutated:",
    False,
)

print(
    "Raw Source mutated:",
    False,
)

print(
    "Human overlay used:",
    True,
)


print(
    "\n=== EFFECTIVE PRIMARY EVIDENCE ==="
)

for part in cell68_replacement_target_parts[
    1:
]:

    state = cell68_effective_speaker(
        part
    )


    print(
        f"Utterance {part.get('utterance_index')}:",
        state.get(
            "canonical"
        ),
        "| Primary:",
        state.get(
            "primary"
        ),
        "| Basis:",
        state.get(
            "basis"
        ),
    )


print(
    "\n=== PACKET VALIDATION ==="
)

print(
    "Target IDs found:",
    cell68_packet_target_ids,
)

print(
    "Old Target still present:",
    cell68_old_target_still_present,
)

print(
    "Packet valid:",
    cell68_packet_valid,
)


print(
    "\n=== ACCEPTED QUESTION ==="
)

print(
    CELL68_ACCEPTED_QUESTION
)

print(
    "Question Draft ID:",
    cell68_question_draft_id,
)

print(
    "Human Decision:",
    "ACCEPT",
)


print(
    "\n=== ACCEPTED ANSWER ==="
)

print(
    CELL68_ACCEPTED_ANSWER
)

print(
    "Expected Answer Draft ID:",
    cell68_expected_answer_draft_id,
)

print(
    "Human Decision:",
    "ACCEPT",
)


print(
    "\n=== GROUNDING ==="
)

for name, status in cell68_grounding.items():

    print(
        f"{name}:",
        status,
    )


print(
    "Grounding PASS:",
    cell68_grounding_pass,
)


print(
    "\n=== CITATION DRAFT ==="
)

print(
    "Citation Mapping ID:",
    cell68_citation_mapping_id,
)

print(
    "Citation entries:",
    len(
        cell68_citation_entries
    ),
)

print(
    "Context-only entries:",
    len(
        cell68_context_entries
    ),
)

print(
    "Primary Answer Evidence entries:",
    len(
        cell68_primary_entries
    ),
)

print(
    "Citation boundary valid:",
    cell68_citation_boundary_valid,
)

print(
    "No locator fabrication:",
    cell68_no_locator_fabrication,
)

print(
    "Primary Authority:",
    "Dr. Komeil Roudi",
)

print(
    "Host used as Answer Evidence:",
    False,
)


print(
    "\n=== WORKFLOW ==="
)

print(
    "Current State:",
    cell68_current_session.get(
        "current_state"
    ),
)

print(
    "Retarget:",
    "ACCEPT",
)

print(
    "Question:",
    "ACCEPT",
)

print(
    "Expected Answer:",
    "ACCEPT",
)

print(
    "Citation:",
    "PENDING",
)


print(
    "\n=== ALL INTERNAL GATES ==="
)

for name, status in cell68_gate_checks.items():

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== QUEUE / SAFETY ==="
)

print(
    "Queue still 10:",
    cell68_queue_still_10,
)

print(
    "Persistence valid:",
    cell68_persistence_valid,
)

print(
    "Base Seed unchanged:",
    cell68_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell68_full_572_blocked,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell68_failures
)


print(
    "\nCODE CELL 68 GATE:",
    (
        "PASS"
        if cell68_gate
        else "FAIL"
    ),
)


assert cell68_gate, (
    "Pilot #14 Replacement Target / Human-certified speaker "
    "continuity / Question+Answer ACCEPT / Citation Draft failed. "
    "Inspect ALL INTERNAL GATES in this SAME Code Cell 68."
)

=== PILOT #14 RETARGET ACCEPT + QA ACCEPT + CITATION DRAFT ===

=== RETARGET ===
Old Target: target-evidence:73bbf40cdd65405711a21f389d93d5f9
Selected Candidate: multi-speaker-retarget-candidate:6046f784fb3ef3aeade5d3d272d40d6f
Replacement Target: target-evidence:69ef57919773eec6ad2cf9b4b66a15fe
Episode: 14
True Multi-Speaker: True

=== SPEAKER CONTINUITY ===
Host: Utterance 37
Explicit Dr. Roudi start: Utterance 38
Human-certified Dr. continuation: Utterances 39..47
Raw Parser mutated: False
Raw Source mutated: False
Human overlay used: True

=== EFFECTIVE PRIMARY EVIDENCE ===
Utterance 38: Dr. Komeil Roudi | Primary: True | Basis: EXPLICIT_SOURCE_MARKER
Utterance 39: Dr. Komeil Roudi | Primary: True | Basis: HUMAN_CERTIFIED_SPEAKER_CONTINUITY
Utterance 40: Dr. Komeil Roudi | Primary: True | Basis: HUMAN_CERTIFIED_SPEAKER_CONTINUITY
Utterance 41: Dr. Komeil Roudi | Primary: True | Basis: HUMAN_CERTIFIED_SPEAKER_CONTINUITY
Utterance 42: Dr. Komeil Roudi | Primary: True | Basis: HUMAN_C

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 69 — Citation ACCEPT + Pilot #14 Completion + Queue Advance

Citation نهایی Pilot #14 توسط Human Reviewer پذیرفته شد.

در این مرحله:

- Citation با تصمیم `ACCEPT` ثبت می‌شود؛
- Human-certified speaker continuity و Retarget lineage حفظ می‌شوند؛
- Pilot #14 کامل می‌شود؛
- Case به Human Review Registry و Author Review اضافه می‌شود؛
- Queue دقیقاً از 10 به 9 جلو می‌رود؛
- Pilot بعدی برای Human Review آماده می‌شود؛
- Full 572 Generation همچنان Blocked باقی می‌ماند.

</div>

In [69]:
# Code Cell 69

cell69_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell68_gate") is not True:
    cell69_failures.append(
        "CELL68_NOT_PASS"
    )


if (
    globals().get(
        "cell68_current_session",
        {},
    ).get(
        "current_state"
    )
    !=
    "CITATION_HUMAN_REVIEW_PENDING"
):
    cell69_failures.append(
        "CITATION_NOT_AT_HUMAN_REVIEW_GATE"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell69_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Human Citation ACCEPT
# ============================================================

CELL69_CITATION_HUMAN_NOTE = (
    "Human Reviewer accepted the final retargeted Citation Mapping. "
    "The host remains context-only. Dr. Komeil Roudi remains the "
    "primary answer authority, including human-certified speaker "
    "continuity for the unlabeled continuation utterances."
)


cell69_complete_pending_session = (
    cell13_apply_event(
        cell68_current_session,
        "CITATION_ACCEPTED",
        human_note=
            CELL69_CITATION_HUMAN_NOTE,
    )
)


cell69_citation_accept_valid = (
    cell69_complete_pending_session.get(
        "current_state"
    )
    ==
    "PILOT_COMPLETE_PENDING_QUEUE_ADVANCE"
)


if not cell69_citation_accept_valid:
    cell69_failures.append(
        "CITATION_ACCEPT_TRANSITION_INVALID"
    )


# ============================================================
# 3) Final Citation record
# ============================================================

cell69_final_citation_mapping_record = copy.deepcopy(
    cell68_citation_mapping_record
)


cell69_final_citation_mapping_record[
    "human_adjudication_status"
] = "ACCEPT"


cell69_final_citation_mapping_record[
    "human_review"
] = {
    "human_decision":
        "ACCEPT",

    "human_note":
        CELL69_CITATION_HUMAN_NOTE,

    "primary_answer_authority":
        "Dr. Komeil Roudi",

    "host_used_as_answer_evidence":
        False,

    "human_speaker_continuity_used":
        True,

    "canonical_raw_source_mutated":
        False,

    "parser_output_mutated":
        False,

    "external_knowledge_used":
        False,
}


# ============================================================
# 4) Complete Human Review case
# ============================================================

cell69_completed_case = copy.deepcopy(
    cell68_active_workspace
)


cell69_completed_case[
    "workflow_state"
] = "PILOT_COMPLETE"


cell69_completed_case[
    "status"
] = "COMPLETED_HUMAN_REVIEW"


cell69_completed_case[
    "final_status"
] = "PILOT_COMPLETE"


cell69_completed_case.setdefault(
    "citation_review",
    {},
)


cell69_completed_case[
    "citation_review"
][
    "human_decision"
] = "ACCEPT"


cell69_completed_case[
    "citation_review"
][
    "human_note"
] = CELL69_CITATION_HUMAN_NOTE


cell69_completed_case[
    "citation_review"
][
    "citation_mapping_record"
] = copy.deepcopy(
    cell69_final_citation_mapping_record
)


cell69_completed_case[
    "completion_summary"
] = {
    "retarget_used":
        True,

    "old_target_evidence_id":
        CELL67_OLD_TARGET_EVIDENCE_ID,

    "final_target_evidence_id":
        cell68_replacement_target_evidence_id,

    "selected_retarget_candidate_id":
        CELL68_SELECTED_CANDIDATE_ID,

    "question_decision":
        "ACCEPT",

    "expected_answer_decision":
        "ACCEPT",

    "citation_decision":
        "ACCEPT",

    "multi_speaker_attribution":
        True,

    "primary_answer_authority":
        "Dr. Komeil Roudi",

    "host_used_as_answer_evidence":
        False,

    "human_speaker_continuity_used":
        True,

    "raw_source_mutated":
        False,

    "parser_output_mutated":
        False,

    "author_final_review":
        "NOT_STARTED",
}


cell69_completed_case[
    "author_final_review"
] = {
    "status":
        "NOT_STARTED",

    "decision":
        None,

    "notes":
        None,
}


# ============================================================
# 5) Queue advance: 10 -> 9
# ============================================================

cell69_previous_remaining_pilot_ids = copy.deepcopy(
    cell64_remaining_pilot_ids
)


CELL69_COMPLETED_PILOT_ID = (
    cell69_previous_remaining_pilot_ids[
        0
    ]
)


cell69_remaining_pilot_ids = copy.deepcopy(
    cell69_previous_remaining_pilot_ids[
        1:
    ]
)


cell69_queue_advance_valid = all([
    len(
        cell69_previous_remaining_pilot_ids
    )
    ==
    10,

    CELL69_COMPLETED_PILOT_ID
    ==
    CELL67_PILOT_ID,

    len(
        cell69_remaining_pilot_ids
    )
    ==
    9,

    CELL69_COMPLETED_PILOT_ID
    not in
    cell69_remaining_pilot_ids,
])


if not cell69_queue_advance_valid:
    cell69_failures.append(
        "QUEUE_ADVANCE_INVALID"
    )


CELL69_NEXT_PILOT_ID = (
    cell69_remaining_pilot_ids[
        0
    ]
)


# ============================================================
# 6) Find next Pilot packet
# ============================================================

cell69_raw_next_packet_candidates = (
    cell58_find_pilot_packet_candidates(
        cell13_seed,
        CELL69_NEXT_PILOT_ID,
    )
)


cell69_packet_candidates_by_sha = {}


for candidate in cell69_raw_next_packet_candidates:

    packet_sha = cell10_stable_sha(
        candidate[
            "packet"
        ]
    )


    previous = (
        cell69_packet_candidates_by_sha.get(
            packet_sha
        )
    )


    if (
        previous is None
        or
        candidate[
            "score"
        ]
        >
        previous[
            "score"
        ]
    ):

        cell69_packet_candidates_by_sha[
            packet_sha
        ] = candidate


cell69_ranked_next_packet_candidates = sorted(
    cell69_packet_candidates_by_sha.values(),
    key=lambda item: (
        -item[
            "score"
        ],
        item[
            "path"
        ],
    ),
)


cell69_next_pilot_packet = (
    copy.deepcopy(
        cell69_ranked_next_packet_candidates[
            0
        ][
            "packet"
        ]
    )
    if cell69_ranked_next_packet_candidates
    else None
)


if not isinstance(
    cell69_next_pilot_packet,
    dict,
):

    cell69_failures.append(
        "NEXT_PILOT_PACKET_NOT_FOUND"
    )


# ============================================================
# 7) Extract next Pilot identities
# ============================================================

cell69_next_pilot_ids_found = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell69_next_pilot_packet,
            "question-pilot:",
        )
    )
)


cell69_next_golden_case_ids = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell69_next_pilot_packet,
            "golden-case:",
        )
    )
)


cell69_next_task_draft_ids = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell69_next_pilot_packet,
            "task-draft:",
        )
    )
)


cell69_next_target_evidence_ids = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell69_next_pilot_packet,
            "target-evidence:",
        )
    )
)


cell69_next_identity_valid = all([
    cell69_next_pilot_ids_found
    ==
    [
        CELL69_NEXT_PILOT_ID
    ],

    len(
        cell69_next_golden_case_ids
    )
    ==
    1,

    len(
        cell69_next_task_draft_ids
    )
    ==
    1,

    len(
        cell69_next_target_evidence_ids
    )
    ==
    1,
])


if not cell69_next_identity_valid:
    cell69_failures.append(
        "NEXT_PILOT_IDENTITY_INVALID"
    )


CELL69_NEXT_GOLDEN_CASE_ID = (
    cell69_next_golden_case_ids[
        0
    ]
    if
    len(
        cell69_next_golden_case_ids
    )
    ==
    1
    else
    None
)


CELL69_NEXT_TASK_DRAFT_ID = (
    cell69_next_task_draft_ids[
        0
    ]
    if
    len(
        cell69_next_task_draft_ids
    )
    ==
    1
    else
    None
)


CELL69_NEXT_TARGET_EVIDENCE_ID = (
    cell69_next_target_evidence_ids[
        0
    ]
    if
    len(
        cell69_next_target_evidence_ids
    )
    ==
    1
    else
    None
)


# ============================================================
# 8) Extract next Target Evidence
# ============================================================

cell69_target_part_candidates = (
    cell58_find_key_values(
        cell69_next_pilot_packet,
        "target_text_parts",
    )
)


cell69_target_parts_by_sha = {}


for parts in cell69_target_part_candidates:

    if not isinstance(
        parts,
        list,
    ):
        continue


    if len(
        parts
    ) == 0:
        continue


    parts_sha = cell10_stable_sha(
        parts
    )


    cell69_target_parts_by_sha[
        parts_sha
    ] = parts


cell69_unique_target_part_sets = list(
    cell69_target_parts_by_sha.values()
)


cell69_next_target_text_parts = (
    copy.deepcopy(
        cell69_unique_target_part_sets[
            0
        ]
    )
    if
    len(
        cell69_unique_target_part_sets
    )
    ==
    1
    else
    None
)


if not isinstance(
    cell69_next_target_text_parts,
    list,
):

    cell69_failures.append(
        "NEXT_TARGET_TEXT_PARTS_NOT_UNIQUE"
    )


# ============================================================
# 9) Readable next Evidence
# ============================================================

cell69_next_evidence_texts = []


for part in (
    cell69_next_target_text_parts
    or
    []
):

    if isinstance(
        part,
        str,
    ):

        text = part.strip()


    elif isinstance(
        part,
        dict,
    ):

        text = (
            part.get(
                "text"
            )
            or
            part.get(
                "target_text"
            )
            or
            part.get(
                "content"
            )
            or
            ""
        )


        if isinstance(
            text,
            str,
        ):

            text = text.strip()


        else:

            text = ""


    else:

        text = ""


    if text:

        cell69_next_evidence_texts.append(
            text
        )


cell69_next_combined_target_text = (
    "\n\n".join(
        cell69_next_evidence_texts
    )
)


cell69_next_evidence_available = bool(
    cell69_next_combined_target_text.strip()
)


if not cell69_next_evidence_available:
    cell69_failures.append(
        "NEXT_EVIDENCE_EMPTY"
    )


# ============================================================
# 10) Extract next Pilot metadata
# ============================================================

def cell69_unique_scalar_for_key(
    payload,
    key,
):

    values = (
        cell58_find_key_values(
            payload,
            key,
        )
    )


    normalized = []


    for value in values:

        if isinstance(
            value,
            (
                str,
                int,
                float,
                bool,
            ),
        ):

            if value not in normalized:
                normalized.append(
                    value
                )


    if len(
        normalized
    ) == 1:

        return normalized[
            0
        ]


    return None


CELL69_NEXT_STRATUM = (
    cell69_unique_scalar_for_key(
        cell69_next_pilot_packet,
        "stratum",
    )
)


CELL69_NEXT_TASK_FAMILY = (
    cell69_unique_scalar_for_key(
        cell69_next_pilot_packet,
        "task_family",
    )
)


CELL69_NEXT_ARCHETYPE = (
    cell69_unique_scalar_for_key(
        cell69_next_pilot_packet,
        "question_archetype",
    )
)


CELL69_NEXT_AUTHORITY_EXPECTATION = (
    cell69_unique_scalar_for_key(
        cell69_next_pilot_packet,
        "authority_expectation",
    )
)


# ============================================================
# 11) Next Pilot Engine session
# ============================================================

cell69_next_pilot_session = (
    cell13_build_session(
        cell69_next_pilot_packet
    )
)


cell69_next_session_valid = (
    cell69_next_pilot_session.get(
        "current_state"
    )
    ==
    "QUESTION_DRAFT_PENDING"
)


if not cell69_next_session_valid:
    cell69_failures.append(
        "NEXT_SESSION_INVALID"
    )


# ============================================================
# 12) Update completed Human Review cases
# ============================================================

cell69_existing_completed_cases = copy.deepcopy(
    cell68_persisted_registry.get(
        "completed_human_review_cases",
        [],
    )
)


if not isinstance(
    cell69_existing_completed_cases,
    list,
):

    cell69_existing_completed_cases = []

    cell69_failures.append(
        "COMPLETED_CASES_NOT_LIST"
    )


cell69_existing_completed_cases = [
    case
    for case
    in cell69_existing_completed_cases
    if not (
        isinstance(
            case,
            dict,
        )
        and
        case.get(
            "pilot_case_id"
        )
        ==
        CELL69_COMPLETED_PILOT_ID
    )
]


cell69_completed_human_review_cases = (
    cell69_existing_completed_cases
    +
    [
        copy.deepcopy(
            cell69_completed_case
        )
    ]
)


cell69_completed_case_count_valid = (
    len(
        cell69_completed_human_review_cases
    )
    ==
    13
)


if not cell69_completed_case_count_valid:
    cell69_failures.append(
        "COMPLETED_CASE_COUNT_INVALID"
    )


# ============================================================
# 13) Rebuild Author Review queue
# ============================================================

cell69_author_review = copy.deepcopy(
    cell68_persisted_registry.get(
        "author_review",
        {},
    )
)


if not isinstance(
    cell69_author_review,
    dict,
):

    cell69_author_review = {}


cell69_author_ready_cases = copy.deepcopy(
    cell69_completed_human_review_cases
)


cell69_author_review[
    "cases_ready_for_review"
] = copy.deepcopy(
    cell69_author_ready_cases
)


cell69_author_review[
    "case_count"
] = len(
    cell69_author_ready_cases
)


cell69_author_review[
    "final_review_status"
] = "NOT_STARTED"


cell69_author_ready_count_valid = (
    len(
        cell69_author_ready_cases
    )
    ==
    13
)


cell69_author_ready_identity_valid = (
    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell69_author_ready_cases
        if isinstance(
            case,
            dict,
        )
    }
    ==
    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell69_completed_human_review_cases
        if isinstance(
            case,
            dict,
        )
    }
)


if not cell69_author_ready_count_valid:
    cell69_failures.append(
        "AUTHOR_READY_COUNT_INVALID"
    )


if not cell69_author_ready_identity_valid:
    cell69_failures.append(
        "AUTHOR_READY_IDENTITY_INVALID"
    )


# ============================================================
# 14) Build next active workspace
# ============================================================

cell69_next_active_workspace = {
    "pilot_case_id":
        CELL69_NEXT_PILOT_ID,

    "golden_case_id":
        CELL69_NEXT_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL69_NEXT_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL69_NEXT_TARGET_EVIDENCE_ID,

    "status":
        "ACTIVE_HUMAN_REVIEW",

    "workflow_state":
        "QUESTION_DRAFT_PENDING",

    "stratum":
        CELL69_NEXT_STRATUM,

    "task_family":
        CELL69_NEXT_TASK_FAMILY,

    "question_archetype":
        CELL69_NEXT_ARCHETYPE,

    "authority_expectation":
        CELL69_NEXT_AUTHORITY_EXPECTATION,

    "source_evidence":
        copy.deepcopy(
            cell69_next_target_text_parts
        ),

    "combined_target_text":
        cell69_next_combined_target_text,

    "question_review":
        None,

    "expected_answer_review":
        None,

    "citation_review":
        None,

    "author_final_review": {
        "status":
            "NOT_STARTED",

        "decision":
            None,

        "notes":
            None,
    },
}


# ============================================================
# 15) New authoritative continuation state
# ============================================================

cell69_continuation_state_core = {
    "next_pilot_id":
        CELL69_NEXT_PILOT_ID,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell69_remaining_pilot_ids
        ),

    "remaining_count":
        9,

    "active_pilot_status":
        "QUESTION_DRAFT_PENDING",

    "active_target_evidence_id":
        CELL69_NEXT_TARGET_EVIDENCE_ID,

    "last_completed_pilot_id":
        CELL69_COMPLETED_PILOT_ID,

    "last_completion_type":
        "RETARGETED_MULTI_SPEAKER_PILOT_COMPLETE",
}


cell69_continuation_state_sha256 = (
    cell10_stable_sha(
        cell69_continuation_state_core
    )
)


cell69_continuation_state = {
    **copy.deepcopy(
        cell69_continuation_state_core
    ),

    "continuation_state_sha256":
        cell69_continuation_state_sha256,
}


# ============================================================
# 16) Persist Registry
# ============================================================

cell69_persistent_registry_payload = copy.deepcopy(
    cell68_persisted_registry
)


cell69_persistent_registry_payload[
    "completed_human_review_cases"
] = copy.deepcopy(
    cell69_completed_human_review_cases
)


cell69_persistent_registry_payload[
    "author_review"
] = copy.deepcopy(
    cell69_author_review
)


cell69_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell69_next_active_workspace
)


cell69_persistent_registry_payload[
    "remaining_pilot_count"
] = 9


cell69_persistent_registry_payload[
    "continuation_state"
] = copy.deepcopy(
    cell69_continuation_state
)


cell69_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell69_persistent_registry_payload
    )
)


# ============================================================
# 17) Read-back validation
# ============================================================

cell69_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell69_persisted_active_case = (
    cell69_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell69_persisted_completed_cases = (
    cell69_persisted_registry.get(
        "completed_human_review_cases",
        [],
    )
)


cell69_persisted_author_ready_cases = (
    cell69_persisted_registry.get(
        "author_review",
        {},
    ).get(
        "cases_ready_for_review",
        [],
    )
)


cell69_persistence_valid = all([
    isinstance(
        cell69_persisted_completed_cases,
        list,
    ),

    len(
        cell69_persisted_completed_cases
    )
    ==
    13,

    isinstance(
        cell69_persisted_author_ready_cases,
        list,
    ),

    len(
        cell69_persisted_author_ready_cases
    )
    ==
    13,

    cell69_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    9,

    cell69_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL69_NEXT_PILOT_ID,

    cell69_persisted_active_case.get(
        "workflow_state"
    )
    ==
    "QUESTION_DRAFT_PENDING",

    cell69_persisted_registry.get(
        "continuation_state",
        {},
    ).get(
        "continuation_state_sha256"
    )
    ==
    cell69_continuation_state_sha256,

    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell69_persisted_completed_cases
        if isinstance(
            case,
            dict,
        )
    }
    ==
    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell69_persisted_author_ready_cases
        if isinstance(
            case,
            dict,
        )
    },
])


if not cell69_persistence_valid:
    cell69_failures.append(
        "REGISTRY_READBACK_INVALID"
    )


# ============================================================
# 18) Safety
# ============================================================

cell69_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell69_full_572_blocked = (
    cell68_full_572_blocked
    is True
)


cell69_no_commit_push = True


# ============================================================
# 19) Internal Gate report
# ============================================================

cell69_gate_checks = {
    "no_failures":
        len(
            cell69_failures
        )
        ==
        0,

    "citation_accept_valid":
        cell69_citation_accept_valid,

    "queue_advance_valid":
        cell69_queue_advance_valid,

    "next_identity_valid":
        cell69_next_identity_valid,

    "next_evidence_available":
        cell69_next_evidence_available,

    "next_session_valid":
        cell69_next_session_valid,

    "completed_case_count_valid":
        cell69_completed_case_count_valid,

    "author_ready_count_valid":
        cell69_author_ready_count_valid,

    "author_ready_identity_valid":
        cell69_author_ready_identity_valid,

    "persistence_valid":
        cell69_persistence_valid,

    "base_seed_unchanged":
        cell69_base_seed_unchanged,

    "full_572_blocked":
        cell69_full_572_blocked,

    "no_commit_push":
        cell69_no_commit_push,
}


cell69_gate = all(
    cell69_gate_checks.values()
)


# ============================================================
# 20) Diagnostics
# ============================================================

print(
    "=== PILOT #14 COMPLETE + QUEUE ADVANCE + NEXT PILOT ==="
)


print(
    "\n=== COMPLETED PILOT ==="
)

print(
    "Pilot:",
    CELL69_COMPLETED_PILOT_ID,
)

print(
    "Final Target:",
    cell68_replacement_target_evidence_id,
)

print(
    "Question:",
    "ACCEPT",
)

print(
    "Expected Answer:",
    "ACCEPT",
)

print(
    "Citation:",
    "ACCEPT",
)

print(
    "Final Citation Mapping:",
    cell68_citation_mapping_id,
)

print(
    "Final Engine State:",
    cell69_complete_pending_session.get(
        "current_state"
    ),
)


print(
    "\n=== ATTRIBUTION / RETARGET ==="
)

print(
    "Retarget used:",
    True,
)

print(
    "Primary Answer Authority:",
    "Dr. Komeil Roudi",
)

print(
    "Host used as Answer Evidence:",
    False,
)

print(
    "Human Speaker Continuity used:",
    True,
)

print(
    "Raw Source mutated:",
    False,
)

print(
    "Parser Output mutated:",
    False,
)


print(
    "\n=== QUEUE ADVANCE ==="
)

print(
    "Before:",
    len(
        cell69_previous_remaining_pilot_ids
    ),
)

print(
    "After:",
    len(
        cell69_remaining_pilot_ids
    ),
)

print(
    "Queue advanced exactly once:",
    cell69_queue_advance_valid,
)


print(
    "\n=== HUMAN REVIEW REGISTRY ==="
)

print(
    "Completed Human Review cases:",
    len(
        cell69_completed_human_review_cases
    ),
)

print(
    "Cases ready for Author Review:",
    len(
        cell69_author_ready_cases
    ),
)

print(
    "Author Final Review:",
    "NOT_STARTED",
)

print(
    "Persistence valid:",
    cell69_persistence_valid,
)


print(
    "\n=== NEXT PILOT ==="
)

print(
    "Pilot:",
    CELL69_NEXT_PILOT_ID,
)

print(
    "Golden Case:",
    CELL69_NEXT_GOLDEN_CASE_ID,
)

print(
    "Task Draft:",
    CELL69_NEXT_TASK_DRAFT_ID,
)

print(
    "Target Evidence:",
    CELL69_NEXT_TARGET_EVIDENCE_ID,
)

print(
    "Stratum:",
    CELL69_NEXT_STRATUM,
)

print(
    "Task Family:",
    CELL69_NEXT_TASK_FAMILY,
)

print(
    "Question Archetype:",
    CELL69_NEXT_ARCHETYPE,
)

print(
    "Authority Expectation:",
    CELL69_NEXT_AUTHORITY_EXPECTATION,
)

print(
    "Workflow:",
    cell69_next_pilot_session.get(
        "current_state"
    ),
)


print(
    "\n=== NEXT SOURCE EVIDENCE ==="
)

print(
    cell69_next_combined_target_text
)


print(
    "\n=== CONTINUATION STATE ==="
)

print(
    "Remaining Pilots:",
    9,
)

print(
    "Continuation SHA256:",
    cell69_continuation_state_sha256,
)


print(
    "\n=== ALL INTERNAL GATES ==="
)

for name, status in cell69_gate_checks.items():

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== SAFETY ==="
)

print(
    "Base Seed unchanged:",
    cell69_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell69_full_572_blocked,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell69_failures
)


print(
    "\nCODE CELL 69 GATE:",
    (
        "PASS"
        if cell69_gate
        else "FAIL"
    ),
)


assert cell69_gate, (
    "Pilot #14 Citation ACCEPT / completion / queue advance / "
    "next Pilot preparation failed. Repair this SAME Code Cell 69."
)

=== PILOT #14 COMPLETE + QUEUE ADVANCE + NEXT PILOT ===

=== COMPLETED PILOT ===
Pilot: question-pilot:bae57351b7aa1fa7545adb709d1c61fc
Final Target: target-evidence:69ef57919773eec6ad2cf9b4b66a15fe
Question: ACCEPT
Expected Answer: ACCEPT
Citation: ACCEPT
Final Citation Mapping: citation-mapping:d97e653f245955491c8565c5a1319d4c
Final Engine State: PILOT_COMPLETE_PENDING_QUEUE_ADVANCE

=== ATTRIBUTION / RETARGET ===
Retarget used: True
Primary Answer Authority: Dr. Komeil Roudi
Host used as Answer Evidence: False
Human Speaker Continuity used: True
Raw Source mutated: False
Parser Output mutated: False

=== QUEUE ADVANCE ===
Before: 10
After: 9
Queue advanced exactly once: True

=== HUMAN REVIEW REGISTRY ===
Completed Human Review cases: 13
Cases ready for Author Review: 13
Author Final Review: NOT_STARTED
Persistence valid: True

=== NEXT PILOT ===
Pilot: question-pilot:ad83a223fe893fb19b5a612dca2de6dc
Golden Case: golden-case:6cff8a216025f79fc610d6f29ac7741f
Task Draft: task-draft:a3

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 70 — Pilot #15 Multi-Speaker Attribution Diagnostic

Pilot فعلی از نوع:

`MULTI_SPEAKER_ATTRIBUTION_ASRE_SHIRIN`

است و Authority Rule آن می‌گوید:

`DO_NOT_INFER_PRIMARY_FROM_LABEL`

بنابراین پیش از ساخت سؤال باید مشخص شود:

- هر Target Part متعلق به چه Speakerای است؛
- کدام بخش‌ها Primary Authority هستند؛
- آیا Speaker continuity لازم است یا نه؛
- آیا بخش‌های پایانی تجربه شخصی گوینده دیگری هستند یا ادامه صحبت دکتر رودی.

این Cell فقط Diagnostic است و:

- Registry را تغییر نمی‌دهد؛
- Queue را جلو نمی‌برد؛
- Human Decision جدید ثبت نمی‌کند؛
- Citation یا Target را تغییر نمی‌دهد.

</div>

In [70]:
# Code Cell 70

cell70_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell69_gate") is not True:
    cell70_failures.append(
        "CELL69_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell69_next_target_text_parts"
    ),
    list,
):
    cell70_failures.append(
        "NEXT_TARGET_PARTS_MISSING"
    )


# ============================================================
# 2) Pilot identity
# ============================================================

CELL70_PILOT_ID = (
    CELL69_NEXT_PILOT_ID
)


CELL70_TARGET_EVIDENCE_ID = (
    CELL69_NEXT_TARGET_EVIDENCE_ID
)


CELL70_TASK_FAMILY = (
    CELL69_NEXT_TASK_FAMILY
)


CELL70_AUTHORITY_EXPECTATION = (
    CELL69_NEXT_AUTHORITY_EXPECTATION
)


# ============================================================
# 3) Inspect each Target Part
# ============================================================

cell70_part_diagnostics = []


for index, part in enumerate(
    cell69_next_target_text_parts
):

    if isinstance(
        part,
        dict,
    ):

        speaker_state = (
            part.get(
                "speaker_state"
            )
            or
            {}
        )


        attribution = (
            part.get(
                "attribution"
            )
            or
            {}
        )


        locator = (
            part.get(
                "locator"
            )
            or
            {}
        )


        text = (
            part.get(
                "text"
            )
            or
            part.get(
                "target_text"
            )
            or
            part.get(
                "content"
            )
            or
            ""
        )


        cell70_part_diagnostics.append({
            "index":
                index,

            "document_id":
                part.get(
                    "document_id"
                ),

            "role":
                part.get(
                    "role"
                ),

            "speaker_base_status":
                speaker_state.get(
                    "base_status"
                ),

            "speaker_base_raw":
                speaker_state.get(
                    "base_raw"
                ),

            "speaker_effective_label":
                speaker_state.get(
                    "effective_label"
                ),

            "speaker_effective_canonical":
                speaker_state.get(
                    "effective_canonical"
                ),

            "speaker_effective_primary":
                speaker_state.get(
                    "effective_primary"
                ),

            "attribution_speaker":
                attribution.get(
                    "speaker"
                ),

            "source_line_start":
                locator.get(
                    "source_line_start"
                ),

            "source_line_end":
                locator.get(
                    "source_line_end"
                ),

            "text":
                text,
        })


    else:

        cell70_part_diagnostics.append({
            "index":
                index,

            "document_id":
                None,

            "role":
                None,

            "speaker_base_status":
                None,

            "speaker_base_raw":
                None,

            "speaker_effective_label":
                None,

            "speaker_effective_canonical":
                None,

            "speaker_effective_primary":
                None,

            "attribution_speaker":
                None,

            "source_line_start":
                None,

            "source_line_end":
                None,

            "text":
                str(
                    part
                ),
        })


# ============================================================
# 4) Effective speaker inventory
# ============================================================

cell70_explicit_speakers = []


cell70_primary_parts = []


cell70_unresolved_parts = []


for row in cell70_part_diagnostics:

    speaker = (
        row.get(
            "speaker_effective_canonical"
        )
        or
        row.get(
            "speaker_effective_label"
        )
        or
        row.get(
            "speaker_base_raw"
        )
        or
        row.get(
            "attribution_speaker"
        )
    )


    if speaker:

        cell70_explicit_speakers.append(
            speaker
        )


    if (
        row.get(
            "speaker_effective_primary"
        )
        is True
    ):

        cell70_primary_parts.append(
            row
        )


    if (
        row.get(
            "speaker_base_status"
        )
        ==
        "UNRESOLVED"
        or
        speaker is None
    ):

        cell70_unresolved_parts.append(
            row
        )


cell70_distinct_speakers = sorted(
    set(
        cell70_explicit_speakers
    )
)


# ============================================================
# 5) Structural flags
# ============================================================

cell70_has_primary_dr = any(
    row.get(
        "speaker_effective_canonical"
    )
    ==
    "Dr. Komeil Roudi"
    and
    row.get(
        "speaker_effective_primary"
    )
    is True
    for row
    in cell70_part_diagnostics
)


cell70_has_unresolved = (
    len(
        cell70_unresolved_parts
    )
    >
    0
)


cell70_is_multi_speaker_task = (
    CELL70_TASK_FAMILY
    ==
    "MULTI_SPEAKER_ATTRIBUTION_ASRE_SHIRIN"
)


cell70_authority_rule_valid = (
    CELL70_AUTHORITY_EXPECTATION
    ==
    "DO_NOT_INFER_PRIMARY_FROM_LABEL"
)


# ============================================================
# 6) Safety
# ============================================================

cell70_registry_write = False

cell70_queue_advance = False

cell70_human_decision_added = False

cell70_target_changed = False

cell70_citation_changed = False


# ============================================================
# 7) Gate
# ============================================================

cell70_gate = all([
    len(
        cell70_failures
    )
    ==
    0,

    cell70_is_multi_speaker_task,

    cell70_authority_rule_valid,

    len(
        cell70_part_diagnostics
    )
    >
    0,
])


# ============================================================
# 8) Diagnostics
# ============================================================

print(
    "=== PILOT #15 MULTI-SPEAKER ATTRIBUTION DIAGNOSTIC ==="
)


print(
    "\n=== PILOT ==="
)

print(
    "Pilot:",
    CELL70_PILOT_ID,
)

print(
    "Target Evidence:",
    CELL70_TARGET_EVIDENCE_ID,
)

print(
    "Task Family:",
    CELL70_TASK_FAMILY,
)

print(
    "Authority Expectation:",
    CELL70_AUTHORITY_EXPECTATION,
)


print(
    "\n=== SPEAKER INVENTORY ==="
)

print(
    "Distinct detected speakers:",
    cell70_distinct_speakers,
)

print(
    "Primary Dr. Roudi detected:",
    cell70_has_primary_dr,
)

print(
    "Unresolved parts exist:",
    cell70_has_unresolved,
)

print(
    "Unresolved part count:",
    len(
        cell70_unresolved_parts
    ),
)


print(
    "\n=== TARGET PARTS ==="
)


for row in cell70_part_diagnostics:

    print(
        "\n"
        +
        "="
        *
        72
    )

    print(
        "PART",
        row[
            "index"
        ],
    )

    print(
        "Document:",
        row.get(
            "document_id"
        ),
    )

    print(
        "Role:",
        row.get(
            "role"
        ),
    )

    print(
        "Base status:",
        row.get(
            "speaker_base_status"
        ),
    )

    print(
        "Base raw:",
        row.get(
            "speaker_base_raw"
        ),
    )

    print(
        "Effective label:",
        row.get(
            "speaker_effective_label"
        ),
    )

    print(
        "Effective canonical:",
        row.get(
            "speaker_effective_canonical"
        ),
    )

    print(
        "Effective primary:",
        row.get(
            "speaker_effective_primary"
        ),
    )

    print(
        "Attribution speaker:",
        row.get(
            "attribution_speaker"
        ),
    )

    print(
        "Source lines:",
        (
            row.get(
                "source_line_start"
            ),
            row.get(
                "source_line_end"
            ),
        ),
    )

    print(
        "Text:"
    )

    print(
        row.get(
            "text"
        )
    )


print(
    "\n=== SAFETY ==="
)

print(
    "Registry write:",
    cell70_registry_write,
)

print(
    "Queue advance:",
    cell70_queue_advance,
)

print(
    "Human Decision added:",
    cell70_human_decision_added,
)

print(
    "Target changed:",
    cell70_target_changed,
)

print(
    "Citation changed:",
    cell70_citation_changed,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell70_failures
)


print(
    "\nCODE CELL 70 GATE:",
    (
        "PASS"
        if cell70_gate
        else "FAIL"
    ),
)


assert cell70_gate, (
    "Pilot #15 multi-speaker attribution diagnostic failed. "
    "Repair this SAME Code Cell 70 before drafting the question."
)

=== PILOT #15 MULTI-SPEAKER ATTRIBUTION DIAGNOSTIC ===

=== PILOT ===
Pilot: question-pilot:ad83a223fe893fb19b5a612dca2de6dc
Target Evidence: target-evidence:79b2a51718be9d9b30700948a2efb767
Task Family: MULTI_SPEAKER_ATTRIBUTION_ASRE_SHIRIN
Authority Expectation: DO_NOT_INFER_PRIMARY_FROM_LABEL

=== SPEAKER INVENTORY ===
Distinct detected speakers: ['دکتر کمیل رودی', 'یکی از شرکت\u200cکنندگان']
Primary Dr. Roudi detected: False
Unresolved parts exist: False
Unresolved part count: 0

=== TARGET PARTS ===

PART 0
Document: program:asre-shirin:episode-018:utterance:00004
Role: ATTRIBUTION_CONTRAST_LEFT
Base status: EXPLICIT_SOURCE_MARKER
Base raw: دکتر کمیل رودی
Effective label: دکتر کمیل رودی
Effective canonical: None
Effective primary: None
Attribution speaker: None
Source lines: (24, 38)
Text:
ما سواد مالی رو از خرج شروع کردیم؛ گفتیم خرج رو باید بشناسیم و مدیریت کنیم، پس‌انداز و بودجه‌بندی داشته باشیم، بدهی رو مدیریت کنیم، سرمایه‌گذاری کنیم، بیمه داشته باشیم و ریسک رو مدیریت کنیم و در

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 71 — Human Authority Certification + Question/Answer ACCEPT + Citation Draft

Human Reviewer تأیید کرد که برچسب صریح:

`دکتر کمیل رودی`

در این Source به:

`Dr. Komeil Roudi`

و Primary Authority این Corpus اشاره دارد.

همچنین سؤال و جواب زیر پذیرفته شدند.

سؤال:

**چرا تجربه‌های کاری کوتاه و کار داوطلبانه می‌توانند به انتخاب شغل مناسب کمک کنند؟**

جواب:

**چون فرد در این تجربه‌ها علاوه بر یادگیری خودِ کار، می‌تواند رزومه و ارتباطاتش را توسعه دهد و بفهمد با چه نوع مدیر، همکار و محیط کاری عملکرد بهتری دارد. داشتن چند تجربه کوتاه یا کار داوطلبانه کمک می‌کند فرد پیش از انتخاب جدی یک مسیر شغلی، شناخت واقعی‌تری از علایق و شرایط مناسب خودش پیدا کند.**

در Citation:

- Part 0 دکتر رودی، Evidence اصلی جواب است؛
- Part 1 شرکت‌کننده فقط Multi-Speaker Context است؛
- Raw Source و Parser اصلی تغییر نمی‌کنند؛
- Queue همچنان 9 باقی می‌ماند؛
- Citation برای Human Review آماده می‌شود.

</div>

In [71]:
# Code Cell 71

cell71_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell70_gate") is not True:
    cell71_failures.append(
        "CELL70_NOT_PASS"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell71_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Pilot identities
# ============================================================

CELL71_PILOT_ID = (
    CELL69_NEXT_PILOT_ID
)

CELL71_GOLDEN_CASE_ID = (
    CELL69_NEXT_GOLDEN_CASE_ID
)

CELL71_TASK_DRAFT_ID = (
    CELL69_NEXT_TASK_DRAFT_ID
)

CELL71_TARGET_EVIDENCE_ID = (
    CELL69_NEXT_TARGET_EVIDENCE_ID
)


# ============================================================
# 3) Accepted Question / Answer
# ============================================================

CELL71_ACCEPTED_QUESTION = (
    "چرا تجربه‌های کاری کوتاه و کار داوطلبانه می‌توانند "
    "به انتخاب شغل مناسب کمک کنند؟"
)


CELL71_ACCEPTED_ANSWER = (
    "چون فرد در این تجربه‌ها علاوه بر یادگیری خودِ کار، "
    "می‌تواند رزومه و ارتباطاتش را توسعه دهد و بفهمد با چه "
    "نوع مدیر، همکار و محیط کاری عملکرد بهتری دارد. داشتن چند "
    "تجربه کوتاه یا کار داوطلبانه کمک می‌کند فرد پیش از انتخاب "
    "جدی یک مسیر شغلی، شناخت واقعی‌تری از علایق و شرایط مناسب "
    "خودش پیدا کند."
)


CELL71_QUESTION_HUMAN_NOTE = (
    "Human Reviewer accepted the question."
)


CELL71_ANSWER_HUMAN_NOTE = (
    "Human Reviewer accepted the expected answer."
)


# ============================================================
# 4) Human authority certification
# ============================================================

cell71_human_authority_certification = {
    "source_label":
        "دکتر کمیل رودی",

    "certified_canonical":
        "Dr. Komeil Roudi",

    "certified_primary":
        True,

    "certification_type":
        "HUMAN_CERTIFIED_AUTHORITY_IDENTITY",

    "basis":
        (
            "Human Reviewer confirmed that the explicit source label "
            "'دکتر کمیل رودی' refers to Dr. Komeil Roudi, the primary "
            "authority of this corpus."
        ),

    "canonical_raw_source_mutated":
        False,

    "parser_output_mutated":
        False,

    "human_overlay_only":
        True,
}


# ============================================================
# 5) Target Parts
# ============================================================

cell71_target_parts = copy.deepcopy(
    cell69_next_target_text_parts
)


cell71_target_parts_valid = all([
    isinstance(
        cell71_target_parts,
        list,
    ),

    len(
        cell71_target_parts
    )
    ==
    2,

    isinstance(
        cell71_target_parts[
            0
        ],
        dict,
    ),

    isinstance(
        cell71_target_parts[
            1
        ],
        dict,
    ),
])


if not cell71_target_parts_valid:
    cell71_failures.append(
        "TARGET_PARTS_INVALID"
    )


# ============================================================
# 6) Validate source labels before overlay
# ============================================================

cell71_part0 = copy.deepcopy(
    cell71_target_parts[
        0
    ]
)


cell71_part1 = copy.deepcopy(
    cell71_target_parts[
        1
    ]
)


cell71_part0_state = (
    cell71_part0.get(
        "speaker_state"
    )
    or
    {}
)


cell71_part1_state = (
    cell71_part1.get(
        "speaker_state"
    )
    or
    {}
)


cell71_source_boundary_valid = all([
    cell71_part0.get(
        "role"
    )
    ==
    "ATTRIBUTION_CONTRAST_LEFT",

    cell71_part0_state.get(
        "base_status"
    )
    ==
    "EXPLICIT_SOURCE_MARKER",

    cell71_part0_state.get(
        "base_raw"
    )
    ==
    "دکتر کمیل رودی",

    cell71_part1.get(
        "role"
    )
    ==
    "ATTRIBUTION_CONTRAST_RIGHT",

    cell71_part1_state.get(
        "base_status"
    )
    ==
    "EXPLICIT_SOURCE_MARKER",

    cell71_part1_state.get(
        "base_raw"
    )
    ==
    "یکی از شرکت‌کنندگان",
])


if not cell71_source_boundary_valid:
    cell71_failures.append(
        "SOURCE_ATTRIBUTION_BOUNDARY_INVALID"
    )


# ============================================================
# 7) Human attribution overlays
# ============================================================

cell71_review_target_parts = copy.deepcopy(
    cell71_target_parts
)


cell71_review_target_parts[
    0
][
    "human_attribution_overlay"
] = {
    "status":
        "HUMAN_CERTIFIED",

    "effective_label":
        "دکتر کمیل رودی",

    "effective_canonical":
        "Dr. Komeil Roudi",

    "effective_primary":
        True,

    "basis":
        "HUMAN_CERTIFIED_AUTHORITY_IDENTITY",

    "raw_speaker_state_preserved":
        True,
}


cell71_review_target_parts[
    1
][
    "human_attribution_overlay"
] = {
    "status":
        "SOURCE_EXPLICIT_NON_PRIMARY_CONTEXT",

    "effective_label":
        "یکی از شرکت‌کنندگان",

    "effective_canonical":
        None,

    "effective_primary":
        False,

    "basis":
        "EXPLICIT_SOURCE_MARKER",

    "raw_speaker_state_preserved":
        True,
}


# ============================================================
# 8) Grounding validation
# ============================================================

def cell71_norm(
    text,
):

    if not isinstance(
        text,
        str,
    ):
        return ""


    value = (
        text
        .replace(
            "ي",
            "ی",
        )
        .replace(
            "ك",
            "ک",
        )
        .replace(
            "\u200c",
            "",
        )
    )


    value = re.sub(
        r"\s+",
        " ",
        value,
    )


    return value.strip()


cell71_primary_text = cell71_norm(
    cell71_review_target_parts[
        0
    ].get(
        "text",
        "",
    )
)


cell71_grounding = {
    "work_learning":
        (
            "کار رو یاد بگیرم"
            in
            cell71_primary_text
        ),

    "resume":
        (
            "رزومهم رو پر کنم"
            in
            cell71_primary_text
        ),

    "connections":
        (
            "ارتباطاتم رو توسعه بدم"
            in
            cell71_primary_text
        ),

    "manager_fit":
        (
            "با چه مدیری بهترم"
            in
            cell71_primary_text
        ),

    "coworker_fit":
        (
            "با چه همکارهایی راحتترم"
            in
            cell71_primary_text
        ),

    "environment_fit":
        (
            "تو چه محیطی عملکرد بهتری دارم"
            in
            cell71_primary_text
        ),

    "volunteer_work":
        (
            "کار داوطلبانه"
            in
            cell71_primary_text
        ),

    "short_experiences":
        (
            "چند تجربه کوتاه"
            in
            cell71_primary_text
        ),

    "job_fit":
        (
            "چه شغلی براش مناسبه"
            in
            cell71_primary_text
        ),
}


cell71_grounding_pass = all(
    cell71_grounding.values()
)


if not cell71_grounding_pass:
    cell71_failures.append(
        "ANSWER_GROUNDING_FAILED"
    )


# ============================================================
# 9) Engine session
# ============================================================

cell71_question_pending_session = (
    cell13_build_session(
        cell69_next_pilot_packet
    )
)


cell71_session_start_valid = (
    cell71_question_pending_session.get(
        "current_state"
    )
    ==
    "QUESTION_DRAFT_PENDING"
)


if not cell71_session_start_valid:
    cell71_failures.append(
        "ENGINE_SESSION_START_INVALID"
    )


# ============================================================
# 10) Question Draft + ACCEPT
# ============================================================

cell71_question_identity = {
    "identity_version":
        "question_draft_identity_v1",

    "pilot_case_id":
        CELL71_PILOT_ID,

    "target_evidence_id":
        CELL71_TARGET_EVIDENCE_ID,

    "question_text":
        CELL71_ACCEPTED_QUESTION,

    "revision_number":
        1,
}


cell71_question_draft_id = (
    "question-draft:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell71_question_identity
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


cell71_question_record = {
    "record_version":
        "question_draft_record_v1",

    "question_draft_id":
        cell71_question_draft_id,

    "pilot_case_id":
        CELL71_PILOT_ID,

    "golden_case_id":
        CELL71_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL71_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL71_TARGET_EVIDENCE_ID,

    "question_text":
        CELL71_ACCEPTED_QUESTION,

    "revision_number":
        1,

    "machine_validation_status":
        "PASS_PENDING_HUMAN_ADJUDICATION",

    "human_adjudication_status":
        "PENDING",

    "external_knowledge_used":
        False,
}


cell71_question_review_session = (
    cell13_apply_event(
        cell71_question_pending_session,
        "QUESTION_DRAFT_REGISTERED",
        artifact=
            cell71_question_record,
    )
)


cell71_answer_pending_session = (
    cell13_apply_event(
        cell71_question_review_session,
        "QUESTION_ACCEPTED",
        human_note=
            CELL71_QUESTION_HUMAN_NOTE,
    )
)


cell71_question_accept_valid = all([
    cell71_question_review_session.get(
        "current_state"
    )
    ==
    "QUESTION_HUMAN_REVIEW_PENDING",

    cell71_answer_pending_session.get(
        "current_state"
    )
    ==
    "EXPECTED_ANSWER_DRAFT_PENDING",
])


if not cell71_question_accept_valid:
    cell71_failures.append(
        "QUESTION_ACCEPT_TRANSITION_INVALID"
    )


# ============================================================
# 11) Expected Answer + ACCEPT
# ============================================================

cell71_answer_identity = {
    "identity_version":
        "expected_answer_identity_v1",

    "pilot_case_id":
        CELL71_PILOT_ID,

    "question_draft_id":
        cell71_question_draft_id,

    "target_evidence_id":
        CELL71_TARGET_EVIDENCE_ID,

    "answer_text":
        CELL71_ACCEPTED_ANSWER,

    "revision_number":
        1,
}


cell71_expected_answer_draft_id = (
    "expected-answer-draft:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell71_answer_identity
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


cell71_expected_answer_record = {
    "record_version":
        "expected_answer_draft_record_v1",

    "expected_answer_draft_id":
        cell71_expected_answer_draft_id,

    "pilot_case_id":
        CELL71_PILOT_ID,

    "golden_case_id":
        CELL71_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL71_TASK_DRAFT_ID,

    "question_draft_id":
        cell71_question_draft_id,

    "target_evidence_id":
        CELL71_TARGET_EVIDENCE_ID,

    "expected_answer_text":
        CELL71_ACCEPTED_ANSWER,

    "revision_number":
        1,

    "machine_validation_status":
        "PASS_PENDING_HUMAN_ADJUDICATION",

    "human_adjudication_status":
        "PENDING",

    "external_knowledge_used":
        False,
}


cell71_answer_review_session = (
    cell13_apply_event(
        cell71_answer_pending_session,
        "EXPECTED_ANSWER_REGISTERED",
        artifact=
            cell71_expected_answer_record,
    )
)


cell71_citation_pending_session = (
    cell13_apply_event(
        cell71_answer_review_session,
        "EXPECTED_ANSWER_ACCEPTED",
        human_note=
            CELL71_ANSWER_HUMAN_NOTE,
    )
)


cell71_answer_accept_valid = all([
    cell71_answer_review_session.get(
        "current_state"
    )
    ==
    "EXPECTED_ANSWER_HUMAN_REVIEW_PENDING",

    cell71_citation_pending_session.get(
        "current_state"
    )
    ==
    "CITATION_MAPPING_PENDING",
])


if not cell71_answer_accept_valid:
    cell71_failures.append(
        "ANSWER_ACCEPT_TRANSITION_INVALID"
    )


# ============================================================
# 12) Citation entries
# ============================================================

cell71_citation_entries = []


for index, part in enumerate(
    cell71_review_target_parts
):

    is_primary = (
        index
        ==
        0
    )


    entry = copy.deepcopy(
        part
    )


    entry[
        "citation_index"
    ] = index


    entry[
        "target_evidence_id"
    ] = CELL71_TARGET_EVIDENCE_ID


    entry[
        "locator_origin"
    ] = "COPIED_FROM_TARGET_EVIDENCE"


    entry[
        "external_locator_lookup_used"
    ] = False


    entry[
        "answer_support"
    ] = is_primary


    entry[
        "answer_support_role"
    ] = (
        "PRIMARY_ANSWER_EVIDENCE"
        if is_primary
        else
        "MULTI_SPEAKER_CONTEXT_ONLY"
    )


    entry[
        "authority_role"
    ] = (
        "PRIMARY_CORPUS_AUTHORITY"
        if is_primary
        else
        "NON_PRIMARY_SPEAKER_CONTEXT"
    )


    entry[
        "use_in_expected_answer_grounding"
    ] = is_primary


    cell71_citation_entries.append(
        entry
    )


cell71_primary_entries = [
    entry
    for entry
    in cell71_citation_entries
    if entry.get(
        "answer_support"
    )
    is True
]


cell71_context_entries = [
    entry
    for entry
    in cell71_citation_entries
    if entry.get(
        "answer_support"
    )
    is False
]


cell71_citation_boundary_valid = all([
    len(
        cell71_primary_entries
    )
    ==
    1,

    len(
        cell71_context_entries
    )
    ==
    1,

    cell71_primary_entries[
        0
    ].get(
        "human_attribution_overlay",
        {},
    ).get(
        "effective_canonical"
    )
    ==
    "Dr. Komeil Roudi",

    cell71_primary_entries[
        0
    ].get(
        "human_attribution_overlay",
        {},
    ).get(
        "effective_primary"
    )
    is True,

    cell71_primary_entries[
        0
    ].get(
        "authority_role"
    )
    ==
    "PRIMARY_CORPUS_AUTHORITY",

    cell71_context_entries[
        0
    ].get(
        "authority_role"
    )
    ==
    "NON_PRIMARY_SPEAKER_CONTEXT",

    cell71_context_entries[
        0
    ].get(
        "use_in_expected_answer_grounding"
    )
    is False,
])


if not cell71_citation_boundary_valid:
    cell71_failures.append(
        "CITATION_ATTRIBUTION_BOUNDARY_INVALID"
    )


# ============================================================
# 13) No locator fabrication
# ============================================================

def cell71_strip_review_fields(
    entry,
):

    output = copy.deepcopy(
        entry
    )


    for key in [
        "citation_index",
        "target_evidence_id",
        "locator_origin",
        "external_locator_lookup_used",
        "answer_support",
        "answer_support_role",
        "authority_role",
        "use_in_expected_answer_grounding",
        "human_attribution_overlay",
    ]:

        output.pop(
            key,
            None,
        )


    return output


cell71_citation_back_projection = [
    cell71_strip_review_fields(
        entry
    )
    for entry
    in cell71_citation_entries
]


cell71_no_locator_fabrication = (
    cell71_citation_back_projection
    ==
    cell71_target_parts
)


if not cell71_no_locator_fabrication:
    cell71_failures.append(
        "CITATION_LOCATOR_FABRICATION_DETECTED"
    )


# ============================================================
# 14) Citation Mapping
# ============================================================

cell71_citation_identity = {
    "identity_version":
        "citation_mapping_identity_v2_human_authority",

    "pilot_case_id":
        CELL71_PILOT_ID,

    "question_draft_id":
        cell71_question_draft_id,

    "expected_answer_draft_id":
        cell71_expected_answer_draft_id,

    "target_evidence_id":
        CELL71_TARGET_EVIDENCE_ID,

    "citation_entries":
        copy.deepcopy(
            cell71_citation_entries
        ),
}


cell71_citation_mapping_id = (
    "citation-mapping:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell71_citation_identity
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


cell71_citation_mapping_record = {
    "record_version":
        "citation_mapping_record_v2_human_authority",

    "citation_mapping_id":
        cell71_citation_mapping_id,

    "pilot_case_id":
        CELL71_PILOT_ID,

    "golden_case_id":
        CELL71_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL71_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL71_TARGET_EVIDENCE_ID,

    "question_draft_id":
        cell71_question_draft_id,

    "expected_answer_draft_id":
        cell71_expected_answer_draft_id,

    "citation_entries":
        copy.deepcopy(
            cell71_citation_entries
        ),

    "citation_scope":
        "MULTI_SPEAKER_ATTRIBUTION_BOUNDARY",

    "primary_answer_evidence_count":
        1,

    "context_only_count":
        1,

    "primary_authority":
        "Dr. Komeil Roudi",

    "participant_used_as_answer_evidence":
        False,

    "human_authority_certification_used":
        True,

    "human_authority_certification":
        copy.deepcopy(
            cell71_human_authority_certification
        ),

    "external_locator_lookup_used":
        False,

    "canonical_raw_source_mutated":
        False,

    "parser_output_mutated":
        False,

    "machine_validation_status":
        "PASS_PENDING_HUMAN_ADJUDICATION",

    "human_adjudication_status":
        "PENDING",
}


cell71_current_session = (
    cell13_apply_event(
        cell71_citation_pending_session,
        "CITATION_MAPPING_REGISTERED",
        artifact=
            cell71_citation_mapping_record,
    )
)


cell71_citation_registered_valid = (
    cell71_current_session.get(
        "current_state"
    )
    ==
    "CITATION_HUMAN_REVIEW_PENDING"
)


if not cell71_citation_registered_valid:
    cell71_failures.append(
        "CITATION_REGISTRATION_TRANSITION_INVALID"
    )


# ============================================================
# 15) Active Human Review workspace
# ============================================================

cell71_active_workspace = copy.deepcopy(
    cell69_persisted_active_case
)


cell71_active_workspace[
    "status"
] = "ACTIVE_HUMAN_REVIEW"


cell71_active_workspace[
    "workflow_state"
] = "CITATION_HUMAN_REVIEW_PENDING"


cell71_active_workspace[
    "human_authority_certification"
] = copy.deepcopy(
    cell71_human_authority_certification
)


cell71_active_workspace[
    "source_evidence"
] = copy.deepcopy(
    cell71_review_target_parts
)


cell71_active_workspace[
    "question_review"
] = {
    "question_draft_id":
        cell71_question_draft_id,

    "question_text":
        CELL71_ACCEPTED_QUESTION,

    "human_decision":
        "ACCEPT",

    "human_note":
        CELL71_QUESTION_HUMAN_NOTE,
}


cell71_active_workspace[
    "expected_answer_review"
] = {
    "expected_answer_draft_id":
        cell71_expected_answer_draft_id,

    "expected_answer_text":
        CELL71_ACCEPTED_ANSWER,

    "human_decision":
        "ACCEPT",

    "human_note":
        CELL71_ANSWER_HUMAN_NOTE,
}


cell71_active_workspace[
    "citation_review"
] = {
    "citation_mapping_id":
        cell71_citation_mapping_id,

    "citation_mapping_record":
        copy.deepcopy(
            cell71_citation_mapping_record
        ),

    "citation_entries":
        copy.deepcopy(
            cell71_citation_entries
        ),

    "machine_validation_status":
        "PASS_PENDING_HUMAN_ADJUDICATION",

    "human_decision":
        "PENDING",

    "primary_answer_authority":
        "Dr. Komeil Roudi",

    "participant_used_as_answer_evidence":
        False,

    "human_authority_certification_used":
        True,
}


# ============================================================
# 16) Continuation state
# ============================================================

cell71_continuation_core = {
    "next_pilot_id":
        CELL71_PILOT_ID,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell69_remaining_pilot_ids
        ),

    "remaining_count":
        9,

    "active_pilot_status":
        "CITATION_HUMAN_REVIEW_PENDING",

    "active_target_evidence_id":
        CELL71_TARGET_EVIDENCE_ID,

    "active_citation_mapping_id":
        cell71_citation_mapping_id,

    "human_authority_certification":
        True,

    "last_completed_pilot_id":
        cell69_continuation_state.get(
            "last_completed_pilot_id"
        ),

    "last_completion_type":
        cell69_continuation_state.get(
            "last_completion_type"
        ),
}


cell71_continuation_state_sha256 = (
    cell10_stable_sha(
        cell71_continuation_core
    )
)


cell71_continuation_state = {
    **copy.deepcopy(
        cell71_continuation_core
    ),

    "continuation_state_sha256":
        cell71_continuation_state_sha256,
}


# ============================================================
# 17) Persist Registry
# ============================================================

cell71_persistent_registry_payload = copy.deepcopy(
    cell69_persisted_registry
)


cell71_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell71_active_workspace
)


cell71_persistent_registry_payload[
    "remaining_pilot_count"
] = 9


cell71_persistent_registry_payload[
    "continuation_state"
] = copy.deepcopy(
    cell71_continuation_state
)


cell71_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell71_persistent_registry_payload
    )
)


# ============================================================
# 18) Read-back validation
# ============================================================

cell71_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell71_persisted_active_case = (
    cell71_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell71_persistence_valid = all([
    cell71_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL71_PILOT_ID,

    cell71_persisted_active_case.get(
        "workflow_state"
    )
    ==
    "CITATION_HUMAN_REVIEW_PENDING",

    cell71_persisted_active_case.get(
        "human_authority_certification",
        {},
    ).get(
        "certified_canonical"
    )
    ==
    "Dr. Komeil Roudi",

    cell71_persisted_active_case.get(
        "question_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell71_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell71_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "PENDING",

    cell71_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    9,

    cell71_persisted_registry.get(
        "continuation_state",
        {},
    ).get(
        "continuation_state_sha256"
    )
    ==
    cell71_continuation_state_sha256,
])


if not cell71_persistence_valid:
    cell71_failures.append(
        "REGISTRY_READBACK_INVALID"
    )


# ============================================================
# 19) Safety
# ============================================================

cell71_queue_still_9 = all([
    len(
        cell69_remaining_pilot_ids
    )
    ==
    9,

    cell69_remaining_pilot_ids[
        0
    ]
    ==
    CELL71_PILOT_ID,
])


cell71_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell71_full_572_blocked = (
    cell69_full_572_blocked
    is True
)


# ============================================================
# 20) Internal Gates
# ============================================================

cell71_gate_checks = {
    "no_failures":
        len(
            cell71_failures
        )
        ==
        0,

    "target_parts_valid":
        cell71_target_parts_valid,

    "source_boundary_valid":
        cell71_source_boundary_valid,

    "grounding_pass":
        cell71_grounding_pass,

    "session_start_valid":
        cell71_session_start_valid,

    "question_accept_valid":
        cell71_question_accept_valid,

    "answer_accept_valid":
        cell71_answer_accept_valid,

    "citation_boundary_valid":
        cell71_citation_boundary_valid,

    "no_locator_fabrication":
        cell71_no_locator_fabrication,

    "citation_registered_valid":
        cell71_citation_registered_valid,

    "persistence_valid":
        cell71_persistence_valid,

    "queue_still_9":
        cell71_queue_still_9,

    "base_seed_unchanged":
        cell71_base_seed_unchanged,

    "full_572_blocked":
        cell71_full_572_blocked,
}


cell71_gate = all(
    cell71_gate_checks.values()
)


# ============================================================
# 21) Diagnostics
# ============================================================

print(
    "=== PILOT #15 AUTHORITY CERTIFICATION + QA ACCEPT + CITATION DRAFT ==="
)


print(
    "\n=== HUMAN AUTHORITY CERTIFICATION ==="
)

print(
    "Source Label:",
    "دکتر کمیل رودی",
)

print(
    "Certified Canonical:",
    "Dr. Komeil Roudi",
)

print(
    "Certified Primary:",
    True,
)

print(
    "Raw Source mutated:",
    False,
)

print(
    "Parser mutated:",
    False,
)


print(
    "\n=== ACCEPTED QUESTION ==="
)

print(
    CELL71_ACCEPTED_QUESTION
)

print(
    "Question Draft ID:",
    cell71_question_draft_id,
)

print(
    "Human Decision:",
    "ACCEPT",
)


print(
    "\n=== ACCEPTED ANSWER ==="
)

print(
    CELL71_ACCEPTED_ANSWER
)

print(
    "Expected Answer Draft ID:",
    cell71_expected_answer_draft_id,
)

print(
    "Human Decision:",
    "ACCEPT",
)


print(
    "\n=== GROUNDING ==="
)

for name, status in cell71_grounding.items():

    print(
        f"{name}:",
        status,
    )


print(
    "Grounding PASS:",
    cell71_grounding_pass,
)


print(
    "\n=== CITATION DRAFT ==="
)

print(
    "Citation Mapping ID:",
    cell71_citation_mapping_id,
)

print(
    "Citation entries:",
    len(
        cell71_citation_entries
    ),
)

print(
    "Primary Answer Evidence entries:",
    len(
        cell71_primary_entries
    ),
)

print(
    "Context-only entries:",
    len(
        cell71_context_entries
    ),
)

print(
    "Primary Authority:",
    "Dr. Komeil Roudi",
)

print(
    "Participant used as Answer Evidence:",
    False,
)

print(
    "Citation boundary valid:",
    cell71_citation_boundary_valid,
)

print(
    "No locator fabrication:",
    cell71_no_locator_fabrication,
)


print(
    "\n=== WORKFLOW ==="
)

print(
    "Current State:",
    cell71_current_session.get(
        "current_state"
    ),
)

print(
    "Question:",
    "ACCEPT",
)

print(
    "Expected Answer:",
    "ACCEPT",
)

print(
    "Citation:",
    "PENDING",
)


print(
    "\n=== ALL INTERNAL GATES ==="
)

for name, status in cell71_gate_checks.items():

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== QUEUE / SAFETY ==="
)

print(
    "Queue still 9:",
    cell71_queue_still_9,
)

print(
    "Persistence valid:",
    cell71_persistence_valid,
)

print(
    "Base Seed unchanged:",
    cell71_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell71_full_572_blocked,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell71_failures
)


print(
    "\nCODE CELL 71 GATE:",
    (
        "PASS"
        if cell71_gate
        else "FAIL"
    ),
)


assert cell71_gate, (
    "Pilot #15 Human authority certification / Question+Answer "
    "ACCEPT / Citation Draft failed. Repair this SAME Code Cell 71."
)

=== PILOT #15 AUTHORITY CERTIFICATION + QA ACCEPT + CITATION DRAFT ===

=== HUMAN AUTHORITY CERTIFICATION ===
Source Label: دکتر کمیل رودی
Certified Canonical: Dr. Komeil Roudi
Certified Primary: True
Raw Source mutated: False
Parser mutated: False

=== ACCEPTED QUESTION ===
چرا تجربه‌های کاری کوتاه و کار داوطلبانه می‌توانند به انتخاب شغل مناسب کمک کنند؟
Question Draft ID: question-draft:22a2384adf58d31f85e9d4c0f5a75743
Human Decision: ACCEPT

=== ACCEPTED ANSWER ===
چون فرد در این تجربه‌ها علاوه بر یادگیری خودِ کار، می‌تواند رزومه و ارتباطاتش را توسعه دهد و بفهمد با چه نوع مدیر، همکار و محیط کاری عملکرد بهتری دارد. داشتن چند تجربه کوتاه یا کار داوطلبانه کمک می‌کند فرد پیش از انتخاب جدی یک مسیر شغلی، شناخت واقعی‌تری از علایق و شرایط مناسب خودش پیدا کند.
Expected Answer Draft ID: expected-answer-draft:005076560e6e6c9e9373b7336f9f56d4
Human Decision: ACCEPT

=== GROUNDING ===
work_learning: True
resume: True
connections: True
manager_fit: True
coworker_fit: True
environment_fit: True
vol

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 72 — Citation ACCEPT + Pilot #15 Completion + Queue Advance

Citation نهایی Pilot #15 توسط Human Reviewer پذیرفته شد.

در این مرحله:

- Citation با تصمیم `ACCEPT` ثبت می‌شود؛
- Human Authority Certification مربوط به `دکتر کمیل رودی → Dr. Komeil Roudi` حفظ می‌شود؛
- Part شرکت‌کننده فقط Context باقی می‌ماند؛
- Pilot #15 کامل می‌شود؛
- Case به Human Review Registry و Author Review اضافه می‌شود؛
- تعداد Caseهای کامل از 13 به 14 می‌رسد؛
- Queue دقیقاً از 9 به 8 جلو می‌رود؛
- Pilot بعدی برای Human Review آماده می‌شود؛
- Full 572 Generation همچنان Blocked باقی می‌ماند.

</div>

In [72]:
# Code Cell 72

cell72_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell71_gate") is not True:
    cell72_failures.append(
        "CELL71_NOT_PASS"
    )


if (
    globals().get(
        "cell71_current_session",
        {},
    ).get(
        "current_state"
    )
    !=
    "CITATION_HUMAN_REVIEW_PENDING"
):
    cell72_failures.append(
        "CITATION_NOT_AT_HUMAN_REVIEW_GATE"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell72_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Human Citation ACCEPT
# ============================================================

CELL72_CITATION_HUMAN_NOTE = (
    "Human Reviewer accepted the final Citation Mapping. "
    "The source label 'دکتر کمیل رودی' remains human-certified "
    "as Dr. Komeil Roudi / Primary Corpus Authority. "
    "The participant utterance remains context-only and is not "
    "used as Expected Answer evidence."
)


cell72_complete_pending_session = (
    cell13_apply_event(
        cell71_current_session,
        "CITATION_ACCEPTED",
        human_note=
            CELL72_CITATION_HUMAN_NOTE,
    )
)


cell72_citation_accept_valid = (
    cell72_complete_pending_session.get(
        "current_state"
    )
    ==
    "PILOT_COMPLETE_PENDING_QUEUE_ADVANCE"
)


if not cell72_citation_accept_valid:
    cell72_failures.append(
        "CITATION_ACCEPT_TRANSITION_INVALID"
    )


# ============================================================
# 3) Final Citation Mapping
# ============================================================

cell72_final_citation_mapping_record = copy.deepcopy(
    cell71_citation_mapping_record
)


cell72_final_citation_mapping_record[
    "human_adjudication_status"
] = "ACCEPT"


cell72_final_citation_mapping_record[
    "human_review"
] = {
    "human_decision":
        "ACCEPT",

    "human_note":
        CELL72_CITATION_HUMAN_NOTE,

    "primary_answer_authority":
        "Dr. Komeil Roudi",

    "participant_used_as_answer_evidence":
        False,

    "human_authority_certification_used":
        True,

    "canonical_raw_source_mutated":
        False,

    "parser_output_mutated":
        False,

    "external_knowledge_used":
        False,
}


# ============================================================
# 4) Complete Human Review case
# ============================================================

cell72_completed_case = copy.deepcopy(
    cell71_active_workspace
)


cell72_completed_case[
    "workflow_state"
] = "PILOT_COMPLETE"


cell72_completed_case[
    "status"
] = "COMPLETED_HUMAN_REVIEW"


cell72_completed_case[
    "final_status"
] = "PILOT_COMPLETE"


cell72_completed_case.setdefault(
    "citation_review",
    {},
)


cell72_completed_case[
    "citation_review"
][
    "human_decision"
] = "ACCEPT"


cell72_completed_case[
    "citation_review"
][
    "human_note"
] = CELL72_CITATION_HUMAN_NOTE


cell72_completed_case[
    "citation_review"
][
    "citation_mapping_record"
] = copy.deepcopy(
    cell72_final_citation_mapping_record
)


cell72_completed_case[
    "completion_summary"
] = {
    "retarget_used":
        False,

    "target_evidence_id":
        CELL71_TARGET_EVIDENCE_ID,

    "question_decision":
        "ACCEPT",

    "expected_answer_decision":
        "ACCEPT",

    "citation_decision":
        "ACCEPT",

    "multi_speaker_attribution":
        True,

    "primary_answer_authority":
        "Dr. Komeil Roudi",

    "participant_used_as_answer_evidence":
        False,

    "human_authority_certification_used":
        True,

    "source_label":
        "دکتر کمیل رودی",

    "certified_canonical":
        "Dr. Komeil Roudi",

    "raw_source_mutated":
        False,

    "parser_output_mutated":
        False,

    "author_final_review":
        "NOT_STARTED",
}


cell72_completed_case[
    "author_final_review"
] = {
    "status":
        "NOT_STARTED",

    "decision":
        None,

    "notes":
        None,
}


# ============================================================
# 5) Queue advance: 9 -> 8
# ============================================================

cell72_previous_remaining_pilot_ids = copy.deepcopy(
    cell69_remaining_pilot_ids
)


CELL72_COMPLETED_PILOT_ID = (
    cell72_previous_remaining_pilot_ids[
        0
    ]
)


cell72_remaining_pilot_ids = copy.deepcopy(
    cell72_previous_remaining_pilot_ids[
        1:
    ]
)


cell72_queue_advance_valid = all([
    len(
        cell72_previous_remaining_pilot_ids
    )
    ==
    9,

    CELL72_COMPLETED_PILOT_ID
    ==
    CELL71_PILOT_ID,

    len(
        cell72_remaining_pilot_ids
    )
    ==
    8,

    CELL72_COMPLETED_PILOT_ID
    not in
    cell72_remaining_pilot_ids,
])


if not cell72_queue_advance_valid:
    cell72_failures.append(
        "QUEUE_ADVANCE_INVALID"
    )


CELL72_NEXT_PILOT_ID = (
    cell72_remaining_pilot_ids[
        0
    ]
)


# ============================================================
# 6) Find next Pilot packet
# ============================================================

cell72_raw_next_packet_candidates = (
    cell58_find_pilot_packet_candidates(
        cell13_seed,
        CELL72_NEXT_PILOT_ID,
    )
)


cell72_packet_candidates_by_sha = {}


for candidate in cell72_raw_next_packet_candidates:

    packet_sha = cell10_stable_sha(
        candidate[
            "packet"
        ]
    )


    previous = (
        cell72_packet_candidates_by_sha.get(
            packet_sha
        )
    )


    if (
        previous is None
        or
        candidate[
            "score"
        ]
        >
        previous[
            "score"
        ]
    ):

        cell72_packet_candidates_by_sha[
            packet_sha
        ] = candidate


cell72_ranked_next_packet_candidates = sorted(
    cell72_packet_candidates_by_sha.values(),
    key=lambda item: (
        -item[
            "score"
        ],
        item[
            "path"
        ],
    ),
)


cell72_next_pilot_packet = (
    copy.deepcopy(
        cell72_ranked_next_packet_candidates[
            0
        ][
            "packet"
        ]
    )
    if cell72_ranked_next_packet_candidates
    else None
)


if not isinstance(
    cell72_next_pilot_packet,
    dict,
):

    cell72_failures.append(
        "NEXT_PILOT_PACKET_NOT_FOUND"
    )


# ============================================================
# 7) Extract next Pilot identities
# ============================================================

cell72_next_pilot_ids_found = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell72_next_pilot_packet,
            "question-pilot:",
        )
    )
)


cell72_next_golden_case_ids = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell72_next_pilot_packet,
            "golden-case:",
        )
    )
)


cell72_next_task_draft_ids = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell72_next_pilot_packet,
            "task-draft:",
        )
    )
)


cell72_next_target_evidence_ids = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell72_next_pilot_packet,
            "target-evidence:",
        )
    )
)


cell72_next_identity_valid = all([
    cell72_next_pilot_ids_found
    ==
    [
        CELL72_NEXT_PILOT_ID
    ],

    len(
        cell72_next_golden_case_ids
    )
    ==
    1,

    len(
        cell72_next_task_draft_ids
    )
    ==
    1,

    len(
        cell72_next_target_evidence_ids
    )
    ==
    1,
])


if not cell72_next_identity_valid:
    cell72_failures.append(
        "NEXT_PILOT_IDENTITY_INVALID"
    )


CELL72_NEXT_GOLDEN_CASE_ID = (
    cell72_next_golden_case_ids[
        0
    ]
    if
    len(
        cell72_next_golden_case_ids
    )
    ==
    1
    else
    None
)


CELL72_NEXT_TASK_DRAFT_ID = (
    cell72_next_task_draft_ids[
        0
    ]
    if
    len(
        cell72_next_task_draft_ids
    )
    ==
    1
    else
    None
)


CELL72_NEXT_TARGET_EVIDENCE_ID = (
    cell72_next_target_evidence_ids[
        0
    ]
    if
    len(
        cell72_next_target_evidence_ids
    )
    ==
    1
    else
    None
)


# ============================================================
# 8) Extract next Target Evidence
# ============================================================

cell72_target_part_candidates = (
    cell58_find_key_values(
        cell72_next_pilot_packet,
        "target_text_parts",
    )
)


cell72_target_parts_by_sha = {}


for parts in cell72_target_part_candidates:

    if not isinstance(
        parts,
        list,
    ):
        continue


    if len(
        parts
    ) == 0:
        continue


    parts_sha = cell10_stable_sha(
        parts
    )


    cell72_target_parts_by_sha[
        parts_sha
    ] = parts


cell72_unique_target_part_sets = list(
    cell72_target_parts_by_sha.values()
)


cell72_next_target_text_parts = (
    copy.deepcopy(
        cell72_unique_target_part_sets[
            0
        ]
    )
    if
    len(
        cell72_unique_target_part_sets
    )
    ==
    1
    else
    None
)


if not isinstance(
    cell72_next_target_text_parts,
    list,
):

    cell72_failures.append(
        "NEXT_TARGET_TEXT_PARTS_NOT_UNIQUE"
    )


# ============================================================
# 9) Build readable next Evidence
# ============================================================

cell72_next_evidence_texts = []


for part in (
    cell72_next_target_text_parts
    or
    []
):

    if isinstance(
        part,
        str,
    ):

        text = part.strip()


    elif isinstance(
        part,
        dict,
    ):

        text = (
            part.get(
                "text"
            )
            or
            part.get(
                "target_text"
            )
            or
            part.get(
                "content"
            )
            or
            ""
        )


        if isinstance(
            text,
            str,
        ):

            text = text.strip()


        else:

            text = ""


    else:

        text = ""


    if text:

        cell72_next_evidence_texts.append(
            text
        )


cell72_next_combined_target_text = (
    "\n\n".join(
        cell72_next_evidence_texts
    )
)


cell72_next_evidence_available = bool(
    cell72_next_combined_target_text.strip()
)


if not cell72_next_evidence_available:
    cell72_failures.append(
        "NEXT_EVIDENCE_EMPTY"
    )


# ============================================================
# 10) Extract next Pilot metadata
# ============================================================

def cell72_unique_scalar_for_key(
    payload,
    key,
):

    values = (
        cell58_find_key_values(
            payload,
            key,
        )
    )


    normalized = []


    for value in values:

        if isinstance(
            value,
            (
                str,
                int,
                float,
                bool,
            ),
        ):

            if value not in normalized:
                normalized.append(
                    value
                )


    if len(
        normalized
    ) == 1:

        return normalized[
            0
        ]


    return None


CELL72_NEXT_STRATUM = (
    cell72_unique_scalar_for_key(
        cell72_next_pilot_packet,
        "stratum",
    )
)


CELL72_NEXT_TASK_FAMILY = (
    cell72_unique_scalar_for_key(
        cell72_next_pilot_packet,
        "task_family",
    )
)


CELL72_NEXT_ARCHETYPE = (
    cell72_unique_scalar_for_key(
        cell72_next_pilot_packet,
        "question_archetype",
    )
)


CELL72_NEXT_AUTHORITY_EXPECTATION = (
    cell72_unique_scalar_for_key(
        cell72_next_pilot_packet,
        "authority_expectation",
    )
)


# ============================================================
# 11) Next Pilot Engine session
# ============================================================

cell72_next_pilot_session = (
    cell13_build_session(
        cell72_next_pilot_packet
    )
)


cell72_next_session_valid = (
    cell72_next_pilot_session.get(
        "current_state"
    )
    ==
    "QUESTION_DRAFT_PENDING"
)


if not cell72_next_session_valid:
    cell72_failures.append(
        "NEXT_SESSION_INVALID"
    )


# ============================================================
# 12) Update completed Human Review cases
# ============================================================

cell72_existing_completed_cases = copy.deepcopy(
    cell71_persisted_registry.get(
        "completed_human_review_cases",
        [],
    )
)


if not isinstance(
    cell72_existing_completed_cases,
    list,
):

    cell72_existing_completed_cases = []

    cell72_failures.append(
        "COMPLETED_CASES_NOT_LIST"
    )


cell72_existing_completed_cases = [
    case
    for case
    in cell72_existing_completed_cases
    if not (
        isinstance(
            case,
            dict,
        )
        and
        case.get(
            "pilot_case_id"
        )
        ==
        CELL72_COMPLETED_PILOT_ID
    )
]


cell72_completed_human_review_cases = (
    cell72_existing_completed_cases
    +
    [
        copy.deepcopy(
            cell72_completed_case
        )
    ]
)


cell72_completed_case_count_valid = (
    len(
        cell72_completed_human_review_cases
    )
    ==
    14
)


if not cell72_completed_case_count_valid:
    cell72_failures.append(
        "COMPLETED_CASE_COUNT_INVALID"
    )


# ============================================================
# 13) Rebuild Author Review queue
# ============================================================

cell72_author_review = copy.deepcopy(
    cell71_persisted_registry.get(
        "author_review",
        {},
    )
)


if not isinstance(
    cell72_author_review,
    dict,
):

    cell72_author_review = {}


cell72_author_ready_cases = copy.deepcopy(
    cell72_completed_human_review_cases
)


cell72_author_review[
    "cases_ready_for_review"
] = copy.deepcopy(
    cell72_author_ready_cases
)


cell72_author_review[
    "case_count"
] = len(
    cell72_author_ready_cases
)


cell72_author_review[
    "final_review_status"
] = "NOT_STARTED"


cell72_author_ready_count_valid = (
    len(
        cell72_author_ready_cases
    )
    ==
    14
)


cell72_author_ready_identity_valid = (
    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell72_author_ready_cases
        if isinstance(
            case,
            dict,
        )
    }
    ==
    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell72_completed_human_review_cases
        if isinstance(
            case,
            dict,
        )
    }
)


if not cell72_author_ready_count_valid:
    cell72_failures.append(
        "AUTHOR_READY_COUNT_INVALID"
    )


if not cell72_author_ready_identity_valid:
    cell72_failures.append(
        "AUTHOR_READY_IDENTITY_INVALID"
    )


# ============================================================
# 14) Build next active workspace
# ============================================================

cell72_next_active_workspace = {
    "pilot_case_id":
        CELL72_NEXT_PILOT_ID,

    "golden_case_id":
        CELL72_NEXT_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL72_NEXT_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL72_NEXT_TARGET_EVIDENCE_ID,

    "status":
        "ACTIVE_HUMAN_REVIEW",

    "workflow_state":
        "QUESTION_DRAFT_PENDING",

    "stratum":
        CELL72_NEXT_STRATUM,

    "task_family":
        CELL72_NEXT_TASK_FAMILY,

    "question_archetype":
        CELL72_NEXT_ARCHETYPE,

    "authority_expectation":
        CELL72_NEXT_AUTHORITY_EXPECTATION,

    "source_evidence":
        copy.deepcopy(
            cell72_next_target_text_parts
        ),

    "combined_target_text":
        cell72_next_combined_target_text,

    "question_review":
        None,

    "expected_answer_review":
        None,

    "citation_review":
        None,

    "author_final_review": {
        "status":
            "NOT_STARTED",

        "decision":
            None,

        "notes":
            None,
    },
}


# ============================================================
# 15) New authoritative continuation state
# ============================================================

cell72_continuation_state_core = {
    "next_pilot_id":
        CELL72_NEXT_PILOT_ID,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell72_remaining_pilot_ids
        ),

    "remaining_count":
        8,

    "active_pilot_status":
        "QUESTION_DRAFT_PENDING",

    "active_target_evidence_id":
        CELL72_NEXT_TARGET_EVIDENCE_ID,

    "last_completed_pilot_id":
        CELL72_COMPLETED_PILOT_ID,

    "last_completion_type":
        "MULTI_SPEAKER_HUMAN_AUTHORITY_PILOT_COMPLETE",
}


cell72_continuation_state_sha256 = (
    cell10_stable_sha(
        cell72_continuation_state_core
    )
)


cell72_continuation_state = {
    **copy.deepcopy(
        cell72_continuation_state_core
    ),

    "continuation_state_sha256":
        cell72_continuation_state_sha256,
}


# ============================================================
# 16) Persist Registry
# ============================================================

cell72_persistent_registry_payload = copy.deepcopy(
    cell71_persisted_registry
)


cell72_persistent_registry_payload[
    "completed_human_review_cases"
] = copy.deepcopy(
    cell72_completed_human_review_cases
)


cell72_persistent_registry_payload[
    "author_review"
] = copy.deepcopy(
    cell72_author_review
)


cell72_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell72_next_active_workspace
)


cell72_persistent_registry_payload[
    "remaining_pilot_count"
] = 8


cell72_persistent_registry_payload[
    "continuation_state"
] = copy.deepcopy(
    cell72_continuation_state
)


cell72_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell72_persistent_registry_payload
    )
)


# ============================================================
# 17) Read-back validation
# ============================================================

cell72_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell72_persisted_active_case = (
    cell72_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell72_persisted_completed_cases = (
    cell72_persisted_registry.get(
        "completed_human_review_cases",
        [],
    )
)


cell72_persisted_author_ready_cases = (
    cell72_persisted_registry.get(
        "author_review",
        {},
    ).get(
        "cases_ready_for_review",
        [],
    )
)


cell72_persistence_valid = all([
    isinstance(
        cell72_persisted_completed_cases,
        list,
    ),

    len(
        cell72_persisted_completed_cases
    )
    ==
    14,

    isinstance(
        cell72_persisted_author_ready_cases,
        list,
    ),

    len(
        cell72_persisted_author_ready_cases
    )
    ==
    14,

    cell72_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    8,

    cell72_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL72_NEXT_PILOT_ID,

    cell72_persisted_active_case.get(
        "workflow_state"
    )
    ==
    "QUESTION_DRAFT_PENDING",

    cell72_persisted_registry.get(
        "continuation_state",
        {},
    ).get(
        "continuation_state_sha256"
    )
    ==
    cell72_continuation_state_sha256,

    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell72_persisted_completed_cases
        if isinstance(
            case,
            dict,
        )
    }
    ==
    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell72_persisted_author_ready_cases
        if isinstance(
            case,
            dict,
        )
    },
])


if not cell72_persistence_valid:
    cell72_failures.append(
        "REGISTRY_READBACK_INVALID"
    )


# ============================================================
# 18) Safety
# ============================================================

cell72_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell72_full_572_blocked = (
    cell71_full_572_blocked
    is True
)


cell72_no_commit_push = True


# ============================================================
# 19) Internal Gates
# ============================================================

cell72_gate_checks = {
    "no_failures":
        len(
            cell72_failures
        )
        ==
        0,

    "citation_accept_valid":
        cell72_citation_accept_valid,

    "queue_advance_valid":
        cell72_queue_advance_valid,

    "next_identity_valid":
        cell72_next_identity_valid,

    "next_evidence_available":
        cell72_next_evidence_available,

    "next_session_valid":
        cell72_next_session_valid,

    "completed_case_count_valid":
        cell72_completed_case_count_valid,

    "author_ready_count_valid":
        cell72_author_ready_count_valid,

    "author_ready_identity_valid":
        cell72_author_ready_identity_valid,

    "persistence_valid":
        cell72_persistence_valid,

    "base_seed_unchanged":
        cell72_base_seed_unchanged,

    "full_572_blocked":
        cell72_full_572_blocked,

    "no_commit_push":
        cell72_no_commit_push,
}


cell72_gate = all(
    cell72_gate_checks.values()
)


# ============================================================
# 20) Diagnostics
# ============================================================

print(
    "=== PILOT #15 COMPLETE + QUEUE ADVANCE + NEXT PILOT ==="
)


print(
    "\n=== COMPLETED PILOT ==="
)

print(
    "Pilot:",
    CELL72_COMPLETED_PILOT_ID,
)

print(
    "Target:",
    CELL71_TARGET_EVIDENCE_ID,
)

print(
    "Question:",
    "ACCEPT",
)

print(
    "Expected Answer:",
    "ACCEPT",
)

print(
    "Citation:",
    "ACCEPT",
)

print(
    "Final Citation Mapping:",
    cell71_citation_mapping_id,
)

print(
    "Final Engine State:",
    cell72_complete_pending_session.get(
        "current_state"
    ),
)


print(
    "\n=== AUTHORITY RESULT ==="
)

print(
    "Source Label:",
    "دکتر کمیل رودی",
)

print(
    "Certified Canonical:",
    "Dr. Komeil Roudi",
)

print(
    "Primary Authority:",
    True,
)

print(
    "Participant used as Answer Evidence:",
    False,
)

print(
    "Raw Source mutated:",
    False,
)

print(
    "Parser mutated:",
    False,
)


print(
    "\n=== QUEUE ADVANCE ==="
)

print(
    "Before:",
    len(
        cell72_previous_remaining_pilot_ids
    ),
)

print(
    "After:",
    len(
        cell72_remaining_pilot_ids
    ),
)

print(
    "Queue advanced exactly once:",
    cell72_queue_advance_valid,
)


print(
    "\n=== HUMAN REVIEW REGISTRY ==="
)

print(
    "Completed Human Review cases:",
    len(
        cell72_completed_human_review_cases
    ),
)

print(
    "Cases ready for Author Review:",
    len(
        cell72_author_ready_cases
    ),
)

print(
    "Author Final Review:",
    "NOT_STARTED",
)

print(
    "Persistence valid:",
    cell72_persistence_valid,
)


print(
    "\n=== NEXT PILOT ==="
)

print(
    "Pilot:",
    CELL72_NEXT_PILOT_ID,
)

print(
    "Golden Case:",
    CELL72_NEXT_GOLDEN_CASE_ID,
)

print(
    "Task Draft:",
    CELL72_NEXT_TASK_DRAFT_ID,
)

print(
    "Target Evidence:",
    CELL72_NEXT_TARGET_EVIDENCE_ID,
)

print(
    "Stratum:",
    CELL72_NEXT_STRATUM,
)

print(
    "Task Family:",
    CELL72_NEXT_TASK_FAMILY,
)

print(
    "Question Archetype:",
    CELL72_NEXT_ARCHETYPE,
)

print(
    "Authority Expectation:",
    CELL72_NEXT_AUTHORITY_EXPECTATION,
)

print(
    "Workflow:",
    cell72_next_pilot_session.get(
        "current_state"
    ),
)


print(
    "\n=== NEXT SOURCE EVIDENCE ==="
)

print(
    cell72_next_combined_target_text
)


print(
    "\n=== CONTINUATION STATE ==="
)

print(
    "Remaining Pilots:",
    8,
)

print(
    "Continuation SHA256:",
    cell72_continuation_state_sha256,
)


print(
    "\n=== ALL INTERNAL GATES ==="
)

for name, status in cell72_gate_checks.items():

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== SAFETY ==="
)

print(
    "Base Seed unchanged:",
    cell72_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell72_full_572_blocked,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell72_failures
)


print(
    "\nCODE CELL 72 GATE:",
    (
        "PASS"
        if cell72_gate
        else "FAIL"
    ),
)


assert cell72_gate, (
    "Pilot #15 Citation ACCEPT / completion / queue advance / "
    "next Pilot preparation failed. Repair this SAME Code Cell 72."
)

=== PILOT #15 COMPLETE + QUEUE ADVANCE + NEXT PILOT ===

=== COMPLETED PILOT ===
Pilot: question-pilot:ad83a223fe893fb19b5a612dca2de6dc
Target: target-evidence:79b2a51718be9d9b30700948a2efb767
Question: ACCEPT
Expected Answer: ACCEPT
Citation: ACCEPT
Final Citation Mapping: citation-mapping:40042ae14778bde4ef8e2277fe32d9dc
Final Engine State: PILOT_COMPLETE_PENDING_QUEUE_ADVANCE

=== AUTHORITY RESULT ===
Source Label: دکتر کمیل رودی
Certified Canonical: Dr. Komeil Roudi
Primary Authority: True
Participant used as Answer Evidence: False
Raw Source mutated: False
Parser mutated: False

=== QUEUE ADVANCE ===
Before: 9
After: 8
Queue advanced exactly once: True

=== HUMAN REVIEW REGISTRY ===
Completed Human Review cases: 14
Cases ready for Author Review: 14
Author Final Review: NOT_STARTED
Persistence valid: True

=== NEXT PILOT ===
Pilot: question-pilot:d9882d0f4166b051bca7f45505cfc753
Golden Case: golden-case:a5e67d7d6ad32b9bdbe77f4a5d526cc5
Task Draft: task-draft:b015a09a58084bd6d29cb20

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 73 — Pilot #16 Multi-Speaker Attribution Diagnostic

Pilot فعلی از نوع:

`MULTI_SPEAKER_ATTRIBUTION_KHANE_TO`

است و Authority آن باید در سطح Utterance بررسی شود.

پیش از ساخت Question بررسی می‌کنیم:

- هر Target Part متعلق به چه Speakerای است؛
- کدام بخش واقعاً Dr. Komeil Roudi است؛
- آیا بخش پایانی متعلق به مجری یا گوینده دیگری است؛
- آیا Speaker continuity لازم است یا نه؛
- کدام بخش می‌تواند Evidence اصلی Expected Answer باشد.

این Cell فقط Diagnostic است و هیچ داده‌ای را تغییر نمی‌دهد.

</div>

In [73]:
# Code Cell 73

cell73_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell72_gate") is not True:
    cell73_failures.append(
        "CELL72_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell72_next_target_text_parts"
    ),
    list,
):
    cell73_failures.append(
        "NEXT_TARGET_PARTS_MISSING"
    )


# ============================================================
# 2) Pilot identity
# ============================================================

CELL73_PILOT_ID = (
    CELL72_NEXT_PILOT_ID
)


CELL73_TARGET_EVIDENCE_ID = (
    CELL72_NEXT_TARGET_EVIDENCE_ID
)


CELL73_TASK_FAMILY = (
    CELL72_NEXT_TASK_FAMILY
)


CELL73_AUTHORITY_EXPECTATION = (
    CELL72_NEXT_AUTHORITY_EXPECTATION
)


# ============================================================
# 3) Inspect Target Parts
# ============================================================

cell73_part_diagnostics = []


for index, part in enumerate(
    cell72_next_target_text_parts
):

    if isinstance(
        part,
        dict,
    ):

        speaker_state = (
            part.get(
                "speaker_state"
            )
            or
            {}
        )


        attribution = (
            part.get(
                "attribution"
            )
            or
            {}
        )


        locator = (
            part.get(
                "locator"
            )
            or
            {}
        )


        text = (
            part.get(
                "text"
            )
            or
            part.get(
                "target_text"
            )
            or
            part.get(
                "content"
            )
            or
            ""
        )


        cell73_part_diagnostics.append({
            "index":
                index,

            "document_id":
                part.get(
                    "document_id"
                ),

            "role":
                part.get(
                    "role"
                ),

            "speaker_base_status":
                speaker_state.get(
                    "base_status"
                ),

            "speaker_base_raw":
                speaker_state.get(
                    "base_raw"
                ),

            "speaker_effective_label":
                speaker_state.get(
                    "effective_label"
                ),

            "speaker_effective_canonical":
                speaker_state.get(
                    "effective_canonical"
                ),

            "speaker_effective_primary":
                speaker_state.get(
                    "effective_primary"
                ),

            "attribution_speaker":
                attribution.get(
                    "speaker"
                ),

            "source_line_start":
                locator.get(
                    "source_line_start"
                ),

            "source_line_end":
                locator.get(
                    "source_line_end"
                ),

            "text":
                text,
        })


    else:

        cell73_part_diagnostics.append({
            "index":
                index,

            "document_id":
                None,

            "role":
                None,

            "speaker_base_status":
                None,

            "speaker_base_raw":
                None,

            "speaker_effective_label":
                None,

            "speaker_effective_canonical":
                None,

            "speaker_effective_primary":
                None,

            "attribution_speaker":
                None,

            "source_line_start":
                None,

            "source_line_end":
                None,

            "text":
                str(
                    part
                ),
        })


# ============================================================
# 4) Speaker inventory
# ============================================================

cell73_detected_speakers = []

cell73_primary_parts = []

cell73_unresolved_parts = []


for row in cell73_part_diagnostics:

    speaker = (
        row.get(
            "speaker_effective_canonical"
        )
        or
        row.get(
            "speaker_effective_label"
        )
        or
        row.get(
            "speaker_base_raw"
        )
        or
        row.get(
            "attribution_speaker"
        )
    )


    if speaker:

        cell73_detected_speakers.append(
            speaker
        )


    if (
        row.get(
            "speaker_effective_primary"
        )
        is True
    ):

        cell73_primary_parts.append(
            row
        )


    if (
        row.get(
            "speaker_base_status"
        )
        ==
        "UNRESOLVED"
        or
        speaker is None
    ):

        cell73_unresolved_parts.append(
            row
        )


cell73_distinct_speakers = sorted(
    set(
        cell73_detected_speakers
    )
)


# ============================================================
# 5) Metadata inspection
# ============================================================

def cell73_collect_relevant_metadata(
    value,
    path=(),
):

    output = []


    keywords = (
        "speaker",
        "attribution",
        "authority",
        "guardrail",
        "multi",
        "contrast",
        "stratum",
        "task_family",
        "question_archetype",
    )


    if isinstance(
        value,
        dict,
    ):

        for key, item in value.items():

            key_text = str(
                key
            ).lower()


            if any(
                keyword
                in
                key_text
                for keyword
                in keywords
            ):

                output.append({
                    "path":
                        path
                        +
                        (
                            str(
                                key
                            ),
                        ),

                    "value":
                        copy.deepcopy(
                            item
                        ),
                })


            output.extend(
                cell73_collect_relevant_metadata(
                    item,
                    path
                    +
                    (
                        str(
                            key
                        ),
                    ),
                )
            )


    elif isinstance(
        value,
        list,
    ):

        for index, item in enumerate(
            value
        ):

            output.extend(
                cell73_collect_relevant_metadata(
                    item,
                    path
                    +
                    (
                        str(
                            index
                        ),
                    ),
                )
            )


    return output


cell73_relevant_metadata = (
    cell73_collect_relevant_metadata(
        cell72_next_pilot_packet
    )
)


# ============================================================
# 6) Structural validation
# ============================================================

cell73_is_expected_task = (
    CELL73_TASK_FAMILY
    ==
    "MULTI_SPEAKER_ATTRIBUTION_KHANE_TO"
)


cell73_authority_rule_valid = (
    CELL73_AUTHORITY_EXPECTATION
    ==
    "UTTERANCE_LEVEL_AUTHORITY"
)


cell73_diagnostic_structure_valid = all([
    len(
        cell73_part_diagnostics
    )
    >
    0,

    cell73_is_expected_task,

    cell73_authority_rule_valid,
])


if not cell73_diagnostic_structure_valid:
    cell73_failures.append(
        "DIAGNOSTIC_STRUCTURE_INVALID"
    )


# ============================================================
# 7) Safety
# ============================================================

cell73_registry_write = False

cell73_queue_advance = False

cell73_human_decision_added = False

cell73_target_changed = False

cell73_citation_changed = False

cell73_commit_push = False


# ============================================================
# 8) Final Gate
# ============================================================

cell73_gate = all([
    len(
        cell73_failures
    )
    ==
    0,

    cell73_diagnostic_structure_valid,
])


# ============================================================
# 9) Diagnostics
# ============================================================

print(
    "=== PILOT #16 MULTI-SPEAKER ATTRIBUTION DIAGNOSTIC ==="
)


print(
    "\n=== PILOT ==="
)

print(
    "Pilot:",
    CELL73_PILOT_ID,
)

print(
    "Target Evidence:",
    CELL73_TARGET_EVIDENCE_ID,
)

print(
    "Task Family:",
    CELL73_TASK_FAMILY,
)

print(
    "Authority Expectation:",
    CELL73_AUTHORITY_EXPECTATION,
)


print(
    "\n=== SPEAKER INVENTORY ==="
)

print(
    "Distinct detected speakers:",
    cell73_distinct_speakers,
)

print(
    "Primary-marked part count:",
    len(
        cell73_primary_parts
    ),
)

print(
    "Unresolved part count:",
    len(
        cell73_unresolved_parts
    ),
)


print(
    "\n=== TARGET PARTS ==="
)


for row in cell73_part_diagnostics:

    print(
        "\n"
        +
        "="
        *
        72
    )

    print(
        "PART",
        row[
            "index"
        ],
    )

    print(
        "Document:",
        row.get(
            "document_id"
        ),
    )

    print(
        "Role:",
        row.get(
            "role"
        ),
    )

    print(
        "Base status:",
        row.get(
            "speaker_base_status"
        ),
    )

    print(
        "Base raw:",
        row.get(
            "speaker_base_raw"
        ),
    )

    print(
        "Effective label:",
        row.get(
            "speaker_effective_label"
        ),
    )

    print(
        "Effective canonical:",
        row.get(
            "speaker_effective_canonical"
        ),
    )

    print(
        "Effective primary:",
        row.get(
            "speaker_effective_primary"
        ),
    )

    print(
        "Attribution speaker:",
        row.get(
            "attribution_speaker"
        ),
    )

    print(
        "Source lines:",
        (
            row.get(
                "source_line_start"
            ),
            row.get(
                "source_line_end"
            ),
        ),
    )

    print(
        "Text:"
    )

    print(
        row.get(
            "text"
        )
    )


print(
    "\n=== RELEVANT PILOT METADATA ==="
)


for match in cell73_relevant_metadata:

    print(
        " > ".join(
            match[
                "path"
            ]
        ),
        "=",
        match[
            "value"
        ],
    )


print(
    "\n=== SAFETY ==="
)

print(
    "Registry write:",
    cell73_registry_write,
)

print(
    "Queue advance:",
    cell73_queue_advance,
)

print(
    "Human Decision added:",
    cell73_human_decision_added,
)

print(
    "Target changed:",
    cell73_target_changed,
)

print(
    "Citation changed:",
    cell73_citation_changed,
)

print(
    "Commit / Push:",
    cell73_commit_push,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell73_failures
)


print(
    "\nCODE CELL 73 GATE:",
    (
        "PASS"
        if cell73_gate
        else "FAIL"
    ),
)


assert cell73_gate, (
    "Pilot #16 multi-speaker attribution diagnostic failed. "
    "Repair this SAME Code Cell 73 before drafting the question."
)

=== PILOT #16 MULTI-SPEAKER ATTRIBUTION DIAGNOSTIC ===

=== PILOT ===
Pilot: question-pilot:d9882d0f4166b051bca7f45505cfc753
Target Evidence: target-evidence:955f0cbf8cc5fd2a85bd1ec7e468bf8a
Task Family: MULTI_SPEAKER_ATTRIBUTION_KHANE_TO
Authority Expectation: UTTERANCE_LEVEL_AUTHORITY

=== SPEAKER INVENTORY ===
Distinct detected speakers: ['دکتر کمیل رودی', 'فرهاد جم']
Primary-marked part count: 0
Unresolved part count: 0

=== TARGET PARTS ===

PART 0
Document: program:khane-to:episode-001:utterance:00036
Role: ATTRIBUTION_CONTRAST_LEFT
Base status: EXPLICIT_SOURCE_MARKER
Base raw: دکتر کمیل رودی
Effective label: دکتر کمیل رودی
Effective canonical: None
Effective primary: None
Attribution speaker: None
Source lines: (149, 151)
Text:
نگران بازپرداختش هستی. خب اینجا موضوع اعتبار مطرح می‌شه: من چقدر معتبرم؟ چه میزان می‌تونم بدهی ایجاد بکنم؟ در طول چه مدتی می‌تونم پرداخت بکنم؟ با چه اقساطی؟ همهٔ این‌ها تو این حوزهٔ دوم که مدیریت بدهی و اعتباره مطرح می‌شه. خب من می‌خوام خرج‌هام رو پوشش بد

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 74 — Question/Answer ACCEPT + Citation Draft

برای Pilot #16، Human Reviewer سؤال و جواب زیر را پذیرفت.

سؤال:

**چرا نباید درآمد فرد فقط به شغلش وابسته باشد و سرمایه‌گذاری چه کمکی می‌کند؟**

جواب:

**چون درآمدی که فقط به شغل وابسته باشد ممکن است در دوره‌هایی مثل بازنشستگی، با کاهش توان کار فرد کمتر شود، در حالی که هزینه‌ها می‌توانند بیشتر شوند. برای همین بخشی از پول باید سرمایه‌گذاری شود تا علاوه بر کمک به حفظ ارزش پول، امکان رشد آن و ایجاد بازده هم وجود داشته باشد. البته سرمایه‌گذاری با ریسک همراه است و باید مدیریت ریسک و حفاظت از دارایی‌ها نیز مورد توجه قرار گیرد.**

در Attribution:

- Part 0 با برچسب صریح `دکتر کمیل رودی`، با استفاده از Human Authority Certification قبلی، Primary Answer Evidence است؛
- Part 1 با برچسب صریح `فرهاد جم` فقط Multi-Speaker Context است؛
- Raw Source و Parser تغییر نمی‌کنند؛
- Citation برای Human Review آماده می‌شود؛
- Queue همچنان 8 باقی می‌ماند.

</div>

In [74]:
# Code Cell 74

cell74_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell73_gate") is not True:
    cell74_failures.append(
        "CELL73_NOT_PASS"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell74_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


if not isinstance(
    globals().get(
        "cell71_human_authority_certification"
    ),
    dict,
):
    cell74_failures.append(
        "PRIOR_HUMAN_AUTHORITY_CERTIFICATION_MISSING"
    )


# ============================================================
# 2) Pilot identities
# ============================================================

CELL74_PILOT_ID = (
    CELL72_NEXT_PILOT_ID
)

CELL74_GOLDEN_CASE_ID = (
    CELL72_NEXT_GOLDEN_CASE_ID
)

CELL74_TASK_DRAFT_ID = (
    CELL72_NEXT_TASK_DRAFT_ID
)

CELL74_TARGET_EVIDENCE_ID = (
    CELL72_NEXT_TARGET_EVIDENCE_ID
)


# ============================================================
# 3) Accepted Question / Answer
# ============================================================

CELL74_ACCEPTED_QUESTION = (
    "چرا نباید درآمد فرد فقط به شغلش وابسته باشد و "
    "سرمایه‌گذاری چه کمکی می‌کند؟"
)


CELL74_ACCEPTED_ANSWER = (
    "چون درآمدی که فقط به شغل وابسته باشد ممکن است در دوره‌هایی "
    "مثل بازنشستگی، با کاهش توان کار فرد کمتر شود، در حالی که "
    "هزینه‌ها می‌توانند بیشتر شوند. برای همین بخشی از پول باید "
    "سرمایه‌گذاری شود تا علاوه بر کمک به حفظ ارزش پول، امکان رشد "
    "آن و ایجاد بازده هم وجود داشته باشد. البته سرمایه‌گذاری با "
    "ریسک همراه است و باید مدیریت ریسک و حفاظت از دارایی‌ها نیز "
    "مورد توجه قرار گیرد."
)


CELL74_QUESTION_HUMAN_NOTE = (
    "Human Reviewer accepted the question."
)


CELL74_ANSWER_HUMAN_NOTE = (
    "Human Reviewer accepted the expected answer."
)


# ============================================================
# 4) Reuse certified authority identity
# ============================================================

cell74_human_authority_certification = copy.deepcopy(
    cell71_human_authority_certification
)


cell74_human_authority_certification[
    "reuse_context"
] = {
    "pilot_case_id":
        CELL74_PILOT_ID,

    "source_label":
        "دکتر کمیل رودی",

    "reuse_reason":
        (
            "The same explicitly labeled corpus speaker identity "
            "was previously human-certified."
        ),
}


cell74_authority_certification_valid = all([
    cell74_human_authority_certification.get(
        "source_label"
    )
    ==
    "دکتر کمیل رودی",

    cell74_human_authority_certification.get(
        "certified_canonical"
    )
    ==
    "Dr. Komeil Roudi",

    cell74_human_authority_certification.get(
        "certified_primary"
    )
    is True,

    cell74_human_authority_certification.get(
        "human_overlay_only"
    )
    is True,

    cell74_human_authority_certification.get(
        "canonical_raw_source_mutated"
    )
    is False,

    cell74_human_authority_certification.get(
        "parser_output_mutated"
    )
    is False,
])


if not cell74_authority_certification_valid:
    cell74_failures.append(
        "HUMAN_AUTHORITY_CERTIFICATION_INVALID"
    )


# ============================================================
# 5) Target Parts
# ============================================================

cell74_target_parts = copy.deepcopy(
    cell72_next_target_text_parts
)


cell74_target_parts_valid = all([
    isinstance(
        cell74_target_parts,
        list,
    ),

    len(
        cell74_target_parts
    )
    ==
    2,

    isinstance(
        cell74_target_parts[
            0
        ],
        dict,
    ),

    isinstance(
        cell74_target_parts[
            1
        ],
        dict,
    ),
])


if not cell74_target_parts_valid:
    cell74_failures.append(
        "TARGET_PARTS_INVALID"
    )


cell74_part0 = copy.deepcopy(
    cell74_target_parts[
        0
    ]
)


cell74_part1 = copy.deepcopy(
    cell74_target_parts[
        1
    ]
)


cell74_part0_state = (
    cell74_part0.get(
        "speaker_state"
    )
    or
    {}
)


cell74_part1_state = (
    cell74_part1.get(
        "speaker_state"
    )
    or
    {}
)


# ============================================================
# 6) Validate explicit source boundary
# ============================================================

cell74_source_boundary_valid = all([
    cell74_part0.get(
        "role"
    )
    ==
    "ATTRIBUTION_CONTRAST_LEFT",

    cell74_part0_state.get(
        "base_status"
    )
    ==
    "EXPLICIT_SOURCE_MARKER",

    cell74_part0_state.get(
        "base_raw"
    )
    ==
    "دکتر کمیل رودی",

    cell74_part1.get(
        "role"
    )
    ==
    "ATTRIBUTION_CONTRAST_RIGHT",

    cell74_part1_state.get(
        "base_status"
    )
    ==
    "EXPLICIT_SOURCE_MARKER",

    cell74_part1_state.get(
        "base_raw"
    )
    ==
    "فرهاد جم",
])


if not cell74_source_boundary_valid:
    cell74_failures.append(
        "SOURCE_ATTRIBUTION_BOUNDARY_INVALID"
    )


# ============================================================
# 7) Human attribution overlays
# ============================================================

cell74_review_target_parts = copy.deepcopy(
    cell74_target_parts
)


cell74_review_target_parts[
    0
][
    "human_attribution_overlay"
] = {
    "status":
        "HUMAN_CERTIFIED_REUSED",

    "effective_label":
        "دکتر کمیل رودی",

    "effective_canonical":
        "Dr. Komeil Roudi",

    "effective_primary":
        True,

    "basis":
        "HUMAN_CERTIFIED_AUTHORITY_IDENTITY",

    "certification_reused":
        True,

    "raw_speaker_state_preserved":
        True,
}


cell74_review_target_parts[
    1
][
    "human_attribution_overlay"
] = {
    "status":
        "SOURCE_EXPLICIT_NON_PRIMARY_CONTEXT",

    "effective_label":
        "فرهاد جم",

    "effective_canonical":
        None,

    "effective_primary":
        False,

    "basis":
        "EXPLICIT_SOURCE_MARKER",

    "raw_speaker_state_preserved":
        True,
}


# ============================================================
# 8) Grounding validation
# ============================================================

def cell74_norm(
    text,
):

    if not isinstance(
        text,
        str,
    ):
        return ""


    value = (
        text
        .replace(
            "ي",
            "ی",
        )
        .replace(
            "ك",
            "ک",
        )
        .replace(
            "\u200c",
            "",
        )
        .replace(
            "ۀ",
            "ه",
        )
    )


    value = re.sub(
        r"\s+",
        " ",
        value,
    )


    return value.strip()


cell74_primary_text = cell74_norm(
    cell74_review_target_parts[
        0
    ].get(
        "text",
        "",
    )
)


cell74_grounding = {
    "income_job_dependency":
        (
            "درآمدها وابسته به شغلمه"
            in
            cell74_primary_text
        ),

    "retirement":
        (
            "در زمان بازنشستگی"
            in
            cell74_primary_text
        ),

    "lower_ability":
        (
            "توانایی من کمتر میشه"
            in
            cell74_primary_text
        ),

    "higher_costs":
        (
            "هزینههای من هم بیشتر میشه"
            in
            cell74_primary_text
        ),

    "invest_part_of_money":
        (
            "بخشی از پول رو باید به کار بندازی"
            in
            cell74_primary_text
        ),

    "investment":
        (
            "به کار انداختن پول یعنی سرمایهگذاری"
            in
            cell74_primary_text
        ),

    "preserve_value":
        (
            "ارزش پول حفظ بشه"
            in
            cell74_primary_text
        ),

    "growth":
        (
            "منجر میشه رشد بکنه"
            in
            cell74_primary_text
        ),

    "return":
        (
            "به شما بازده برسونه"
            in
            cell74_primary_text
        ),

    "risk":
        (
            "سرمایهگذاری خطر داره"
            in
            cell74_primary_text
            and
            "ریسک"
            in
            cell74_primary_text
        ),

    "risk_management":
        (
            "مدیریت ریسک و بیمه"
            in
            cell74_primary_text
        ),

    "asset_protection":
        (
            "از داراییهام و سرمایهام مواظبت بکنم"
            in
            cell74_primary_text
        ),
}


cell74_grounding_pass = all(
    cell74_grounding.values()
)


if not cell74_grounding_pass:
    cell74_failures.append(
        "ANSWER_GROUNDING_FAILED"
    )


# ============================================================
# 9) Engine session
# ============================================================

cell74_question_pending_session = (
    cell13_build_session(
        cell72_next_pilot_packet
    )
)


cell74_session_start_valid = (
    cell74_question_pending_session.get(
        "current_state"
    )
    ==
    "QUESTION_DRAFT_PENDING"
)


if not cell74_session_start_valid:
    cell74_failures.append(
        "ENGINE_SESSION_START_INVALID"
    )


# ============================================================
# 10) Question Draft + Human ACCEPT
# ============================================================

cell74_question_identity = {
    "identity_version":
        "question_draft_identity_v1",

    "pilot_case_id":
        CELL74_PILOT_ID,

    "target_evidence_id":
        CELL74_TARGET_EVIDENCE_ID,

    "question_text":
        CELL74_ACCEPTED_QUESTION,

    "revision_number":
        1,
}


cell74_question_draft_id = (
    "question-draft:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell74_question_identity
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


cell74_question_record = {
    "record_version":
        "question_draft_record_v1",

    "question_draft_id":
        cell74_question_draft_id,

    "pilot_case_id":
        CELL74_PILOT_ID,

    "golden_case_id":
        CELL74_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL74_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL74_TARGET_EVIDENCE_ID,

    "question_text":
        CELL74_ACCEPTED_QUESTION,

    "revision_number":
        1,

    "machine_validation_status":
        "PASS_PENDING_HUMAN_ADJUDICATION",

    "human_adjudication_status":
        "PENDING",

    "external_knowledge_used":
        False,
}


cell74_question_review_session = (
    cell13_apply_event(
        cell74_question_pending_session,
        "QUESTION_DRAFT_REGISTERED",
        artifact=
            cell74_question_record,
    )
)


cell74_answer_pending_session = (
    cell13_apply_event(
        cell74_question_review_session,
        "QUESTION_ACCEPTED",
        human_note=
            CELL74_QUESTION_HUMAN_NOTE,
    )
)


cell74_question_accept_valid = all([
    cell74_question_review_session.get(
        "current_state"
    )
    ==
    "QUESTION_HUMAN_REVIEW_PENDING",

    cell74_answer_pending_session.get(
        "current_state"
    )
    ==
    "EXPECTED_ANSWER_DRAFT_PENDING",
])


if not cell74_question_accept_valid:
    cell74_failures.append(
        "QUESTION_ACCEPT_TRANSITION_INVALID"
    )


# ============================================================
# 11) Expected Answer + Human ACCEPT
# ============================================================

cell74_answer_identity = {
    "identity_version":
        "expected_answer_identity_v1",

    "pilot_case_id":
        CELL74_PILOT_ID,

    "question_draft_id":
        cell74_question_draft_id,

    "target_evidence_id":
        CELL74_TARGET_EVIDENCE_ID,

    "answer_text":
        CELL74_ACCEPTED_ANSWER,

    "revision_number":
        1,
}


cell74_expected_answer_draft_id = (
    "expected-answer-draft:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell74_answer_identity
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


cell74_expected_answer_record = {
    "record_version":
        "expected_answer_draft_record_v1",

    "expected_answer_draft_id":
        cell74_expected_answer_draft_id,

    "pilot_case_id":
        CELL74_PILOT_ID,

    "golden_case_id":
        CELL74_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL74_TASK_DRAFT_ID,

    "question_draft_id":
        cell74_question_draft_id,

    "target_evidence_id":
        CELL74_TARGET_EVIDENCE_ID,

    "expected_answer_text":
        CELL74_ACCEPTED_ANSWER,

    "revision_number":
        1,

    "machine_validation_status":
        "PASS_PENDING_HUMAN_ADJUDICATION",

    "human_adjudication_status":
        "PENDING",

    "external_knowledge_used":
        False,
}


cell74_answer_review_session = (
    cell13_apply_event(
        cell74_answer_pending_session,
        "EXPECTED_ANSWER_REGISTERED",
        artifact=
            cell74_expected_answer_record,
    )
)


cell74_citation_pending_session = (
    cell13_apply_event(
        cell74_answer_review_session,
        "EXPECTED_ANSWER_ACCEPTED",
        human_note=
            CELL74_ANSWER_HUMAN_NOTE,
    )
)


cell74_answer_accept_valid = all([
    cell74_answer_review_session.get(
        "current_state"
    )
    ==
    "EXPECTED_ANSWER_HUMAN_REVIEW_PENDING",

    cell74_citation_pending_session.get(
        "current_state"
    )
    ==
    "CITATION_MAPPING_PENDING",
])


if not cell74_answer_accept_valid:
    cell74_failures.append(
        "ANSWER_ACCEPT_TRANSITION_INVALID"
    )


# ============================================================
# 12) Citation entries
# ============================================================

cell74_citation_entries = []


for index, part in enumerate(
    cell74_review_target_parts
):

    is_primary = (
        index
        ==
        0
    )


    entry = copy.deepcopy(
        part
    )


    entry[
        "citation_index"
    ] = index


    entry[
        "target_evidence_id"
    ] = CELL74_TARGET_EVIDENCE_ID


    entry[
        "locator_origin"
    ] = "COPIED_FROM_TARGET_EVIDENCE"


    entry[
        "external_locator_lookup_used"
    ] = False


    entry[
        "answer_support"
    ] = is_primary


    entry[
        "answer_support_role"
    ] = (
        "PRIMARY_ANSWER_EVIDENCE"
        if is_primary
        else
        "MULTI_SPEAKER_CONTEXT_ONLY"
    )


    entry[
        "authority_role"
    ] = (
        "PRIMARY_CORPUS_AUTHORITY"
        if is_primary
        else
        "NON_PRIMARY_SPEAKER_CONTEXT"
    )


    entry[
        "use_in_expected_answer_grounding"
    ] = is_primary


    cell74_citation_entries.append(
        entry
    )


cell74_primary_entries = [
    entry
    for entry
    in cell74_citation_entries
    if entry.get(
        "answer_support"
    )
    is True
]


cell74_context_entries = [
    entry
    for entry
    in cell74_citation_entries
    if entry.get(
        "answer_support"
    )
    is False
]


cell74_citation_boundary_valid = all([
    len(
        cell74_primary_entries
    )
    ==
    1,

    len(
        cell74_context_entries
    )
    ==
    1,

    cell74_primary_entries[
        0
    ].get(
        "human_attribution_overlay",
        {},
    ).get(
        "effective_canonical"
    )
    ==
    "Dr. Komeil Roudi",

    cell74_primary_entries[
        0
    ].get(
        "human_attribution_overlay",
        {},
    ).get(
        "effective_primary"
    )
    is True,

    cell74_primary_entries[
        0
    ].get(
        "authority_role"
    )
    ==
    "PRIMARY_CORPUS_AUTHORITY",

    cell74_context_entries[
        0
    ].get(
        "human_attribution_overlay",
        {},
    ).get(
        "effective_label"
    )
    ==
    "فرهاد جم",

    cell74_context_entries[
        0
    ].get(
        "authority_role"
    )
    ==
    "NON_PRIMARY_SPEAKER_CONTEXT",

    cell74_context_entries[
        0
    ].get(
        "use_in_expected_answer_grounding"
    )
    is False,
])


if not cell74_citation_boundary_valid:
    cell74_failures.append(
        "CITATION_ATTRIBUTION_BOUNDARY_INVALID"
    )


# ============================================================
# 13) No locator fabrication
# ============================================================

def cell74_strip_review_fields(
    entry,
):

    output = copy.deepcopy(
        entry
    )


    for key in [
        "citation_index",
        "target_evidence_id",
        "locator_origin",
        "external_locator_lookup_used",
        "answer_support",
        "answer_support_role",
        "authority_role",
        "use_in_expected_answer_grounding",
        "human_attribution_overlay",
    ]:

        output.pop(
            key,
            None,
        )


    return output


cell74_citation_back_projection = [
    cell74_strip_review_fields(
        entry
    )
    for entry
    in cell74_citation_entries
]


cell74_no_locator_fabrication = (
    cell74_citation_back_projection
    ==
    cell74_target_parts
)


if not cell74_no_locator_fabrication:
    cell74_failures.append(
        "CITATION_LOCATOR_FABRICATION_DETECTED"
    )


# ============================================================
# 14) Citation Mapping
# ============================================================

cell74_citation_identity = {
    "identity_version":
        "citation_mapping_identity_v2_reused_human_authority",

    "pilot_case_id":
        CELL74_PILOT_ID,

    "question_draft_id":
        cell74_question_draft_id,

    "expected_answer_draft_id":
        cell74_expected_answer_draft_id,

    "target_evidence_id":
        CELL74_TARGET_EVIDENCE_ID,

    "citation_entries":
        copy.deepcopy(
            cell74_citation_entries
        ),
}


cell74_citation_mapping_id = (
    "citation-mapping:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell74_citation_identity
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


cell74_citation_mapping_record = {
    "record_version":
        "citation_mapping_record_v2_reused_human_authority",

    "citation_mapping_id":
        cell74_citation_mapping_id,

    "pilot_case_id":
        CELL74_PILOT_ID,

    "golden_case_id":
        CELL74_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL74_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL74_TARGET_EVIDENCE_ID,

    "question_draft_id":
        cell74_question_draft_id,

    "expected_answer_draft_id":
        cell74_expected_answer_draft_id,

    "citation_entries":
        copy.deepcopy(
            cell74_citation_entries
        ),

    "citation_scope":
        "MULTI_SPEAKER_ATTRIBUTION_BOUNDARY",

    "primary_answer_evidence_count":
        1,

    "context_only_count":
        1,

    "primary_authority":
        "Dr. Komeil Roudi",

    "secondary_speaker":
        "فرهاد جم",

    "secondary_speaker_used_as_answer_evidence":
        False,

    "human_authority_certification_used":
        True,

    "human_authority_certification_reused":
        True,

    "human_authority_certification":
        copy.deepcopy(
            cell74_human_authority_certification
        ),

    "external_locator_lookup_used":
        False,

    "canonical_raw_source_mutated":
        False,

    "parser_output_mutated":
        False,

    "machine_validation_status":
        "PASS_PENDING_HUMAN_ADJUDICATION",

    "human_adjudication_status":
        "PENDING",
}


cell74_current_session = (
    cell13_apply_event(
        cell74_citation_pending_session,
        "CITATION_MAPPING_REGISTERED",
        artifact=
            cell74_citation_mapping_record,
    )
)


cell74_citation_registered_valid = (
    cell74_current_session.get(
        "current_state"
    )
    ==
    "CITATION_HUMAN_REVIEW_PENDING"
)


if not cell74_citation_registered_valid:
    cell74_failures.append(
        "CITATION_REGISTRATION_TRANSITION_INVALID"
    )


# ============================================================
# 15) Active Human Review workspace
# ============================================================

cell74_active_workspace = copy.deepcopy(
    cell72_persisted_active_case
)


cell74_active_workspace[
    "status"
] = "ACTIVE_HUMAN_REVIEW"


cell74_active_workspace[
    "workflow_state"
] = "CITATION_HUMAN_REVIEW_PENDING"


cell74_active_workspace[
    "human_authority_certification"
] = copy.deepcopy(
    cell74_human_authority_certification
)


cell74_active_workspace[
    "source_evidence"
] = copy.deepcopy(
    cell74_review_target_parts
)


cell74_active_workspace[
    "question_review"
] = {
    "question_draft_id":
        cell74_question_draft_id,

    "question_text":
        CELL74_ACCEPTED_QUESTION,

    "human_decision":
        "ACCEPT",

    "human_note":
        CELL74_QUESTION_HUMAN_NOTE,
}


cell74_active_workspace[
    "expected_answer_review"
] = {
    "expected_answer_draft_id":
        cell74_expected_answer_draft_id,

    "expected_answer_text":
        CELL74_ACCEPTED_ANSWER,

    "human_decision":
        "ACCEPT",

    "human_note":
        CELL74_ANSWER_HUMAN_NOTE,
}


cell74_active_workspace[
    "citation_review"
] = {
    "citation_mapping_id":
        cell74_citation_mapping_id,

    "citation_mapping_record":
        copy.deepcopy(
            cell74_citation_mapping_record
        ),

    "citation_entries":
        copy.deepcopy(
            cell74_citation_entries
        ),

    "machine_validation_status":
        "PASS_PENDING_HUMAN_ADJUDICATION",

    "human_decision":
        "PENDING",

    "primary_answer_authority":
        "Dr. Komeil Roudi",

    "secondary_speaker":
        "فرهاد جم",

    "secondary_speaker_used_as_answer_evidence":
        False,

    "human_authority_certification_reused":
        True,
}


# ============================================================
# 16) Continuation state
# ============================================================

cell74_continuation_core = {
    "next_pilot_id":
        CELL74_PILOT_ID,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell72_remaining_pilot_ids
        ),

    "remaining_count":
        8,

    "active_pilot_status":
        "CITATION_HUMAN_REVIEW_PENDING",

    "active_target_evidence_id":
        CELL74_TARGET_EVIDENCE_ID,

    "active_citation_mapping_id":
        cell74_citation_mapping_id,

    "human_authority_certification_reused":
        True,

    "last_completed_pilot_id":
        cell72_continuation_state.get(
            "last_completed_pilot_id"
        ),

    "last_completion_type":
        cell72_continuation_state.get(
            "last_completion_type"
        ),
}


cell74_continuation_state_sha256 = (
    cell10_stable_sha(
        cell74_continuation_core
    )
)


cell74_continuation_state = {
    **copy.deepcopy(
        cell74_continuation_core
    ),

    "continuation_state_sha256":
        cell74_continuation_state_sha256,
}


# ============================================================
# 17) Persist Registry
# ============================================================

cell74_persistent_registry_payload = copy.deepcopy(
    cell72_persisted_registry
)


cell74_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell74_active_workspace
)


cell74_persistent_registry_payload[
    "remaining_pilot_count"
] = 8


cell74_persistent_registry_payload[
    "continuation_state"
] = copy.deepcopy(
    cell74_continuation_state
)


cell74_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell74_persistent_registry_payload
    )
)


# ============================================================
# 18) Read-back validation
# ============================================================

cell74_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell74_persisted_active_case = (
    cell74_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell74_persistence_valid = all([
    cell74_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL74_PILOT_ID,

    cell74_persisted_active_case.get(
        "workflow_state"
    )
    ==
    "CITATION_HUMAN_REVIEW_PENDING",

    cell74_persisted_active_case.get(
        "human_authority_certification",
        {},
    ).get(
        "certified_canonical"
    )
    ==
    "Dr. Komeil Roudi",

    cell74_persisted_active_case.get(
        "question_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell74_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell74_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "PENDING",

    cell74_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "secondary_speaker_used_as_answer_evidence"
    )
    is False,

    cell74_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    8,

    cell74_persisted_registry.get(
        "continuation_state",
        {},
    ).get(
        "continuation_state_sha256"
    )
    ==
    cell74_continuation_state_sha256,
])


if not cell74_persistence_valid:
    cell74_failures.append(
        "REGISTRY_READBACK_INVALID"
    )


# ============================================================
# 19) Safety
# ============================================================

cell74_queue_still_8 = all([
    len(
        cell72_remaining_pilot_ids
    )
    ==
    8,

    cell72_remaining_pilot_ids[
        0
    ]
    ==
    CELL74_PILOT_ID,
])


cell74_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell74_full_572_blocked = (
    cell72_full_572_blocked
    is True
)


# ============================================================
# 20) Internal Gates
# ============================================================

cell74_gate_checks = {
    "no_failures":
        len(
            cell74_failures
        )
        ==
        0,

    "authority_certification_valid":
        cell74_authority_certification_valid,

    "target_parts_valid":
        cell74_target_parts_valid,

    "source_boundary_valid":
        cell74_source_boundary_valid,

    "grounding_pass":
        cell74_grounding_pass,

    "session_start_valid":
        cell74_session_start_valid,

    "question_accept_valid":
        cell74_question_accept_valid,

    "answer_accept_valid":
        cell74_answer_accept_valid,

    "citation_boundary_valid":
        cell74_citation_boundary_valid,

    "no_locator_fabrication":
        cell74_no_locator_fabrication,

    "citation_registered_valid":
        cell74_citation_registered_valid,

    "persistence_valid":
        cell74_persistence_valid,

    "queue_still_8":
        cell74_queue_still_8,

    "base_seed_unchanged":
        cell74_base_seed_unchanged,

    "full_572_blocked":
        cell74_full_572_blocked,
}


cell74_gate = all(
    cell74_gate_checks.values()
)


# ============================================================
# 21) Diagnostics
# ============================================================

print(
    "=== PILOT #16 QA ACCEPT + CITATION DRAFT ==="
)


print(
    "\n=== AUTHORITY ==="
)

print(
    "Source Label:",
    "دکتر کمیل رودی",
)

print(
    "Certified Canonical:",
    "Dr. Komeil Roudi",
)

print(
    "Certification reused:",
    True,
)

print(
    "Secondary Speaker:",
    "فرهاد جم",
)

print(
    "Secondary used as Answer Evidence:",
    False,
)

print(
    "Raw Source mutated:",
    False,
)

print(
    "Parser mutated:",
    False,
)


print(
    "\n=== ACCEPTED QUESTION ==="
)

print(
    CELL74_ACCEPTED_QUESTION
)

print(
    "Question Draft ID:",
    cell74_question_draft_id,
)

print(
    "Human Decision:",
    "ACCEPT",
)


print(
    "\n=== ACCEPTED ANSWER ==="
)

print(
    CELL74_ACCEPTED_ANSWER
)

print(
    "Expected Answer Draft ID:",
    cell74_expected_answer_draft_id,
)

print(
    "Human Decision:",
    "ACCEPT",
)


print(
    "\n=== GROUNDING ==="
)

for name, status in cell74_grounding.items():

    print(
        f"{name}:",
        status,
    )


print(
    "Grounding PASS:",
    cell74_grounding_pass,
)


print(
    "\n=== CITATION DRAFT ==="
)

print(
    "Citation Mapping ID:",
    cell74_citation_mapping_id,
)

print(
    "Citation entries:",
    len(
        cell74_citation_entries
    ),
)

print(
    "Primary Answer Evidence entries:",
    len(
        cell74_primary_entries
    ),
)

print(
    "Context-only entries:",
    len(
        cell74_context_entries
    ),
)

print(
    "Primary Authority:",
    "Dr. Komeil Roudi",
)

print(
    "Secondary Speaker:",
    "فرهاد جم",
)

print(
    "Secondary used as Answer Evidence:",
    False,
)

print(
    "Citation boundary valid:",
    cell74_citation_boundary_valid,
)

print(
    "No locator fabrication:",
    cell74_no_locator_fabrication,
)


print(
    "\n=== WORKFLOW ==="
)

print(
    "Current State:",
    cell74_current_session.get(
        "current_state"
    ),
)

print(
    "Question:",
    "ACCEPT",
)

print(
    "Expected Answer:",
    "ACCEPT",
)

print(
    "Citation:",
    "PENDING",
)


print(
    "\n=== ALL INTERNAL GATES ==="
)

for name, status in cell74_gate_checks.items():

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== QUEUE / SAFETY ==="
)

print(
    "Queue still 8:",
    cell74_queue_still_8,
)

print(
    "Persistence valid:",
    cell74_persistence_valid,
)

print(
    "Base Seed unchanged:",
    cell74_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell74_full_572_blocked,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell74_failures
)


print(
    "\nCODE CELL 74 GATE:",
    (
        "PASS"
        if cell74_gate
        else "FAIL"
    ),
)


assert cell74_gate, (
    "Pilot #16 Question+Answer ACCEPT / Citation Draft failed. "
    "Repair this SAME Code Cell 74 before Citation Human Review."
)

=== PILOT #16 QA ACCEPT + CITATION DRAFT ===

=== AUTHORITY ===
Source Label: دکتر کمیل رودی
Certified Canonical: Dr. Komeil Roudi
Certification reused: True
Secondary Speaker: فرهاد جم
Secondary used as Answer Evidence: False
Raw Source mutated: False
Parser mutated: False

=== ACCEPTED QUESTION ===
چرا نباید درآمد فرد فقط به شغلش وابسته باشد و سرمایه‌گذاری چه کمکی می‌کند؟
Question Draft ID: question-draft:be73186ebb5477ddd277cf02e1205ee6
Human Decision: ACCEPT

=== ACCEPTED ANSWER ===
چون درآمدی که فقط به شغل وابسته باشد ممکن است در دوره‌هایی مثل بازنشستگی، با کاهش توان کار فرد کمتر شود، در حالی که هزینه‌ها می‌توانند بیشتر شوند. برای همین بخشی از پول باید سرمایه‌گذاری شود تا علاوه بر کمک به حفظ ارزش پول، امکان رشد آن و ایجاد بازده هم وجود داشته باشد. البته سرمایه‌گذاری با ریسک همراه است و باید مدیریت ریسک و حفاظت از دارایی‌ها نیز مورد توجه قرار گیرد.
Expected Answer Draft ID: expected-answer-draft:672475eee7f0c321393a7c751a33877d
Human Decision: ACCEPT

=== GROUNDING ===
income_job_d

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 75 — Citation ACCEPT + Pilot #16 Completion + Queue Advance

Citation نهایی Pilot #16 توسط Human Reviewer پذیرفته شد.

در این مرحله:

- Citation با تصمیم `ACCEPT` ثبت می‌شود؛
- Human Authority Certification مربوط به `دکتر کمیل رودی → Dr. Komeil Roudi` حفظ می‌شود؛
- `فرهاد جم` فقط Multi-Speaker Context باقی می‌ماند؛
- Pilot #16 کامل می‌شود؛
- تعداد Caseهای کامل Human Review از 14 به 15 می‌رسد؛
- تعداد Caseهای آماده Author Review از 14 به 15 می‌رسد؛
- Queue دقیقاً از 8 به 7 جلو می‌رود؛
- Pilot بعدی برای Human Review آماده می‌شود؛
- Author Final Review همچنان `NOT_STARTED` است؛
- Full 572 Generation همچنان Blocked باقی می‌ماند.

</div>

In [75]:
# Code Cell 75

cell75_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell74_gate") is not True:
    cell75_failures.append(
        "CELL74_NOT_PASS"
    )


if (
    globals().get(
        "cell74_current_session",
        {},
    ).get(
        "current_state"
    )
    !=
    "CITATION_HUMAN_REVIEW_PENDING"
):
    cell75_failures.append(
        "CITATION_NOT_AT_HUMAN_REVIEW_GATE"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell75_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Human Citation ACCEPT
# ============================================================

CELL75_CITATION_HUMAN_NOTE = (
    "Human Reviewer accepted the final Citation Mapping. "
    "Dr. Komeil Roudi remains the primary answer authority via "
    "the previously human-certified source identity. "
    "Farhad Jam remains multi-speaker context only and is not "
    "used as Expected Answer evidence."
)


cell75_complete_pending_session = (
    cell13_apply_event(
        cell74_current_session,
        "CITATION_ACCEPTED",
        human_note=
            CELL75_CITATION_HUMAN_NOTE,
    )
)


cell75_citation_accept_valid = (
    cell75_complete_pending_session.get(
        "current_state"
    )
    ==
    "PILOT_COMPLETE_PENDING_QUEUE_ADVANCE"
)


if not cell75_citation_accept_valid:
    cell75_failures.append(
        "CITATION_ACCEPT_TRANSITION_INVALID"
    )


# ============================================================
# 3) Final Citation Mapping
# ============================================================

cell75_final_citation_mapping_record = copy.deepcopy(
    cell74_citation_mapping_record
)


cell75_final_citation_mapping_record[
    "human_adjudication_status"
] = "ACCEPT"


cell75_final_citation_mapping_record[
    "human_review"
] = {
    "human_decision":
        "ACCEPT",

    "human_note":
        CELL75_CITATION_HUMAN_NOTE,

    "primary_answer_authority":
        "Dr. Komeil Roudi",

    "secondary_speaker":
        "فرهاد جم",

    "secondary_speaker_used_as_answer_evidence":
        False,

    "human_authority_certification_used":
        True,

    "human_authority_certification_reused":
        True,

    "canonical_raw_source_mutated":
        False,

    "parser_output_mutated":
        False,

    "external_knowledge_used":
        False,
}


# ============================================================
# 4) Complete Human Review case
# ============================================================

cell75_completed_case = copy.deepcopy(
    cell74_active_workspace
)


cell75_completed_case[
    "workflow_state"
] = "PILOT_COMPLETE"


cell75_completed_case[
    "status"
] = "COMPLETED_HUMAN_REVIEW"


cell75_completed_case[
    "final_status"
] = "PILOT_COMPLETE"


cell75_completed_case.setdefault(
    "citation_review",
    {},
)


cell75_completed_case[
    "citation_review"
][
    "human_decision"
] = "ACCEPT"


cell75_completed_case[
    "citation_review"
][
    "human_note"
] = CELL75_CITATION_HUMAN_NOTE


cell75_completed_case[
    "citation_review"
][
    "citation_mapping_record"
] = copy.deepcopy(
    cell75_final_citation_mapping_record
)


cell75_completed_case[
    "completion_summary"
] = {
    "retarget_used":
        False,

    "target_evidence_id":
        CELL74_TARGET_EVIDENCE_ID,

    "question_decision":
        "ACCEPT",

    "expected_answer_decision":
        "ACCEPT",

    "citation_decision":
        "ACCEPT",

    "multi_speaker_attribution":
        True,

    "primary_answer_authority":
        "Dr. Komeil Roudi",

    "secondary_speaker":
        "فرهاد جم",

    "secondary_speaker_used_as_answer_evidence":
        False,

    "human_authority_certification_used":
        True,

    "human_authority_certification_reused":
        True,

    "raw_source_mutated":
        False,

    "parser_output_mutated":
        False,

    "author_final_review":
        "NOT_STARTED",
}


cell75_completed_case[
    "author_final_review"
] = {
    "status":
        "NOT_STARTED",

    "decision":
        None,

    "notes":
        None,
}


# ============================================================
# 5) Queue advance: 8 -> 7
# ============================================================

cell75_previous_remaining_pilot_ids = copy.deepcopy(
    cell72_remaining_pilot_ids
)


CELL75_COMPLETED_PILOT_ID = (
    cell75_previous_remaining_pilot_ids[
        0
    ]
)


cell75_remaining_pilot_ids = copy.deepcopy(
    cell75_previous_remaining_pilot_ids[
        1:
    ]
)


cell75_queue_advance_valid = all([
    len(
        cell75_previous_remaining_pilot_ids
    )
    ==
    8,

    CELL75_COMPLETED_PILOT_ID
    ==
    CELL74_PILOT_ID,

    len(
        cell75_remaining_pilot_ids
    )
    ==
    7,

    CELL75_COMPLETED_PILOT_ID
    not in
    cell75_remaining_pilot_ids,
])


if not cell75_queue_advance_valid:
    cell75_failures.append(
        "QUEUE_ADVANCE_INVALID"
    )


CELL75_NEXT_PILOT_ID = (
    cell75_remaining_pilot_ids[
        0
    ]
)


# ============================================================
# 6) Find next Pilot packet
# ============================================================

cell75_raw_next_packet_candidates = (
    cell58_find_pilot_packet_candidates(
        cell13_seed,
        CELL75_NEXT_PILOT_ID,
    )
)


cell75_packet_candidates_by_sha = {}


for candidate in cell75_raw_next_packet_candidates:

    packet_sha = cell10_stable_sha(
        candidate[
            "packet"
        ]
    )


    previous = (
        cell75_packet_candidates_by_sha.get(
            packet_sha
        )
    )


    if (
        previous is None
        or
        candidate[
            "score"
        ]
        >
        previous[
            "score"
        ]
    ):

        cell75_packet_candidates_by_sha[
            packet_sha
        ] = candidate


cell75_ranked_next_packet_candidates = sorted(
    cell75_packet_candidates_by_sha.values(),
    key=lambda item: (
        -item[
            "score"
        ],
        item[
            "path"
        ],
    ),
)


cell75_next_pilot_packet = (
    copy.deepcopy(
        cell75_ranked_next_packet_candidates[
            0
        ][
            "packet"
        ]
    )
    if cell75_ranked_next_packet_candidates
    else None
)


if not isinstance(
    cell75_next_pilot_packet,
    dict,
):

    cell75_failures.append(
        "NEXT_PILOT_PACKET_NOT_FOUND"
    )


# ============================================================
# 7) Extract next Pilot identities
# ============================================================

cell75_next_pilot_ids_found = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell75_next_pilot_packet,
            "question-pilot:",
        )
    )
)


cell75_next_golden_case_ids = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell75_next_pilot_packet,
            "golden-case:",
        )
    )
)


cell75_next_task_draft_ids = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell75_next_pilot_packet,
            "task-draft:",
        )
    )
)


cell75_next_target_evidence_ids = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell75_next_pilot_packet,
            "target-evidence:",
        )
    )
)


cell75_next_identity_valid = all([
    cell75_next_pilot_ids_found
    ==
    [
        CELL75_NEXT_PILOT_ID
    ],

    len(
        cell75_next_golden_case_ids
    )
    ==
    1,

    len(
        cell75_next_task_draft_ids
    )
    ==
    1,

    len(
        cell75_next_target_evidence_ids
    )
    ==
    1,
])


if not cell75_next_identity_valid:
    cell75_failures.append(
        "NEXT_PILOT_IDENTITY_INVALID"
    )


CELL75_NEXT_GOLDEN_CASE_ID = (
    cell75_next_golden_case_ids[
        0
    ]
    if
    len(
        cell75_next_golden_case_ids
    )
    ==
    1
    else
    None
)


CELL75_NEXT_TASK_DRAFT_ID = (
    cell75_next_task_draft_ids[
        0
    ]
    if
    len(
        cell75_next_task_draft_ids
    )
    ==
    1
    else
    None
)


CELL75_NEXT_TARGET_EVIDENCE_ID = (
    cell75_next_target_evidence_ids[
        0
    ]
    if
    len(
        cell75_next_target_evidence_ids
    )
    ==
    1
    else
    None
)


# ============================================================
# 8) Extract next Target Evidence
# ============================================================

cell75_target_part_candidates = (
    cell58_find_key_values(
        cell75_next_pilot_packet,
        "target_text_parts",
    )
)


cell75_target_parts_by_sha = {}


for parts in cell75_target_part_candidates:

    if not isinstance(
        parts,
        list,
    ):
        continue


    if len(
        parts
    ) == 0:
        continue


    parts_sha = cell10_stable_sha(
        parts
    )


    cell75_target_parts_by_sha[
        parts_sha
    ] = parts


cell75_unique_target_part_sets = list(
    cell75_target_parts_by_sha.values()
)


cell75_next_target_text_parts = (
    copy.deepcopy(
        cell75_unique_target_part_sets[
            0
        ]
    )
    if
    len(
        cell75_unique_target_part_sets
    )
    ==
    1
    else
    None
)


if not isinstance(
    cell75_next_target_text_parts,
    list,
):

    cell75_failures.append(
        "NEXT_TARGET_TEXT_PARTS_NOT_UNIQUE"
    )


# ============================================================
# 9) Build readable next Evidence
# ============================================================

cell75_next_evidence_texts = []


for part in (
    cell75_next_target_text_parts
    or
    []
):

    if isinstance(
        part,
        str,
    ):

        text = part.strip()


    elif isinstance(
        part,
        dict,
    ):

        text = (
            part.get(
                "text"
            )
            or
            part.get(
                "target_text"
            )
            or
            part.get(
                "content"
            )
            or
            ""
        )


        if isinstance(
            text,
            str,
        ):

            text = text.strip()


        else:

            text = ""


    else:

        text = ""


    if text:

        cell75_next_evidence_texts.append(
            text
        )


cell75_next_combined_target_text = (
    "\n\n".join(
        cell75_next_evidence_texts
    )
)


cell75_next_evidence_available = bool(
    cell75_next_combined_target_text.strip()
)


if not cell75_next_evidence_available:
    cell75_failures.append(
        "NEXT_EVIDENCE_EMPTY"
    )


# ============================================================
# 10) Extract next Pilot metadata
# ============================================================

def cell75_unique_scalar_for_key(
    payload,
    key,
):

    values = (
        cell58_find_key_values(
            payload,
            key,
        )
    )


    normalized = []


    for value in values:

        if isinstance(
            value,
            (
                str,
                int,
                float,
                bool,
            ),
        ):

            if value not in normalized:
                normalized.append(
                    value
                )


    if len(
        normalized
    ) == 1:

        return normalized[
            0
        ]


    return None


CELL75_NEXT_STRATUM = (
    cell75_unique_scalar_for_key(
        cell75_next_pilot_packet,
        "stratum",
    )
)


CELL75_NEXT_TASK_FAMILY = (
    cell75_unique_scalar_for_key(
        cell75_next_pilot_packet,
        "task_family",
    )
)


CELL75_NEXT_ARCHETYPE = (
    cell75_unique_scalar_for_key(
        cell75_next_pilot_packet,
        "question_archetype",
    )
)


CELL75_NEXT_AUTHORITY_EXPECTATION = (
    cell75_unique_scalar_for_key(
        cell75_next_pilot_packet,
        "authority_expectation",
    )
)


# ============================================================
# 11) Next Pilot Engine session
# ============================================================

cell75_next_pilot_session = (
    cell13_build_session(
        cell75_next_pilot_packet
    )
)


cell75_next_session_valid = (
    cell75_next_pilot_session.get(
        "current_state"
    )
    ==
    "QUESTION_DRAFT_PENDING"
)


if not cell75_next_session_valid:
    cell75_failures.append(
        "NEXT_SESSION_INVALID"
    )


# ============================================================
# 12) Update completed Human Review cases
# ============================================================

cell75_existing_completed_cases = copy.deepcopy(
    cell74_persisted_registry.get(
        "completed_human_review_cases",
        [],
    )
)


if not isinstance(
    cell75_existing_completed_cases,
    list,
):

    cell75_existing_completed_cases = []

    cell75_failures.append(
        "COMPLETED_CASES_NOT_LIST"
    )


cell75_existing_completed_cases = [
    case
    for case
    in cell75_existing_completed_cases
    if not (
        isinstance(
            case,
            dict,
        )
        and
        case.get(
            "pilot_case_id"
        )
        ==
        CELL75_COMPLETED_PILOT_ID
    )
]


cell75_completed_human_review_cases = (
    cell75_existing_completed_cases
    +
    [
        copy.deepcopy(
            cell75_completed_case
        )
    ]
)


cell75_completed_case_count_valid = (
    len(
        cell75_completed_human_review_cases
    )
    ==
    15
)


if not cell75_completed_case_count_valid:
    cell75_failures.append(
        "COMPLETED_CASE_COUNT_INVALID"
    )


# ============================================================
# 13) Rebuild Author Review queue
# ============================================================

cell75_author_review = copy.deepcopy(
    cell74_persisted_registry.get(
        "author_review",
        {},
    )
)


if not isinstance(
    cell75_author_review,
    dict,
):

    cell75_author_review = {}


cell75_author_ready_cases = copy.deepcopy(
    cell75_completed_human_review_cases
)


cell75_author_review[
    "cases_ready_for_review"
] = copy.deepcopy(
    cell75_author_ready_cases
)


cell75_author_review[
    "case_count"
] = len(
    cell75_author_ready_cases
)


cell75_author_review[
    "final_review_status"
] = "NOT_STARTED"


cell75_author_ready_count_valid = (
    len(
        cell75_author_ready_cases
    )
    ==
    15
)


cell75_author_ready_identity_valid = (
    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell75_author_ready_cases
        if isinstance(
            case,
            dict,
        )
    }
    ==
    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell75_completed_human_review_cases
        if isinstance(
            case,
            dict,
        )
    }
)


if not cell75_author_ready_count_valid:
    cell75_failures.append(
        "AUTHOR_READY_COUNT_INVALID"
    )


if not cell75_author_ready_identity_valid:
    cell75_failures.append(
        "AUTHOR_READY_IDENTITY_INVALID"
    )


# ============================================================
# 14) Build next active workspace
# ============================================================

cell75_next_active_workspace = {
    "pilot_case_id":
        CELL75_NEXT_PILOT_ID,

    "golden_case_id":
        CELL75_NEXT_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL75_NEXT_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL75_NEXT_TARGET_EVIDENCE_ID,

    "status":
        "ACTIVE_HUMAN_REVIEW",

    "workflow_state":
        "QUESTION_DRAFT_PENDING",

    "stratum":
        CELL75_NEXT_STRATUM,

    "task_family":
        CELL75_NEXT_TASK_FAMILY,

    "question_archetype":
        CELL75_NEXT_ARCHETYPE,

    "authority_expectation":
        CELL75_NEXT_AUTHORITY_EXPECTATION,

    "source_evidence":
        copy.deepcopy(
            cell75_next_target_text_parts
        ),

    "combined_target_text":
        cell75_next_combined_target_text,

    "question_review":
        None,

    "expected_answer_review":
        None,

    "citation_review":
        None,

    "author_final_review": {
        "status":
            "NOT_STARTED",

        "decision":
            None,

        "notes":
            None,
    },
}


# ============================================================
# 15) Continuation state
# ============================================================

cell75_continuation_state_core = {
    "next_pilot_id":
        CELL75_NEXT_PILOT_ID,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell75_remaining_pilot_ids
        ),

    "remaining_count":
        7,

    "active_pilot_status":
        "QUESTION_DRAFT_PENDING",

    "active_target_evidence_id":
        CELL75_NEXT_TARGET_EVIDENCE_ID,

    "last_completed_pilot_id":
        CELL75_COMPLETED_PILOT_ID,

    "last_completion_type":
        "MULTI_SPEAKER_HUMAN_AUTHORITY_PILOT_COMPLETE",
}


cell75_continuation_state_sha256 = (
    cell10_stable_sha(
        cell75_continuation_state_core
    )
)


cell75_continuation_state = {
    **copy.deepcopy(
        cell75_continuation_state_core
    ),

    "continuation_state_sha256":
        cell75_continuation_state_sha256,
}


# ============================================================
# 16) Persist Registry
# ============================================================

cell75_persistent_registry_payload = copy.deepcopy(
    cell74_persisted_registry
)


cell75_persistent_registry_payload[
    "completed_human_review_cases"
] = copy.deepcopy(
    cell75_completed_human_review_cases
)


cell75_persistent_registry_payload[
    "author_review"
] = copy.deepcopy(
    cell75_author_review
)


cell75_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell75_next_active_workspace
)


cell75_persistent_registry_payload[
    "remaining_pilot_count"
] = 7


cell75_persistent_registry_payload[
    "continuation_state"
] = copy.deepcopy(
    cell75_continuation_state
)


cell75_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell75_persistent_registry_payload
    )
)


# ============================================================
# 17) Read-back validation
# ============================================================

cell75_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell75_persisted_active_case = (
    cell75_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell75_persisted_completed_cases = (
    cell75_persisted_registry.get(
        "completed_human_review_cases",
        [],
    )
)


cell75_persisted_author_ready_cases = (
    cell75_persisted_registry.get(
        "author_review",
        {},
    ).get(
        "cases_ready_for_review",
        [],
    )
)


cell75_persistence_valid = all([
    isinstance(
        cell75_persisted_completed_cases,
        list,
    ),

    len(
        cell75_persisted_completed_cases
    )
    ==
    15,

    isinstance(
        cell75_persisted_author_ready_cases,
        list,
    ),

    len(
        cell75_persisted_author_ready_cases
    )
    ==
    15,

    cell75_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    7,

    cell75_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL75_NEXT_PILOT_ID,

    cell75_persisted_active_case.get(
        "workflow_state"
    )
    ==
    "QUESTION_DRAFT_PENDING",

    cell75_persisted_registry.get(
        "continuation_state",
        {},
    ).get(
        "continuation_state_sha256"
    )
    ==
    cell75_continuation_state_sha256,

    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell75_persisted_completed_cases
        if isinstance(
            case,
            dict,
        )
    }
    ==
    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell75_persisted_author_ready_cases
        if isinstance(
            case,
            dict,
        )
    },
])


if not cell75_persistence_valid:
    cell75_failures.append(
        "REGISTRY_READBACK_INVALID"
    )


# ============================================================
# 18) Safety
# ============================================================

cell75_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell75_full_572_blocked = (
    cell74_full_572_blocked
    is True
)


cell75_no_commit_push = True


# ============================================================
# 19) Internal Gates
# ============================================================

cell75_gate_checks = {
    "no_failures":
        len(
            cell75_failures
        )
        ==
        0,

    "citation_accept_valid":
        cell75_citation_accept_valid,

    "queue_advance_valid":
        cell75_queue_advance_valid,

    "next_identity_valid":
        cell75_next_identity_valid,

    "next_evidence_available":
        cell75_next_evidence_available,

    "next_session_valid":
        cell75_next_session_valid,

    "completed_case_count_valid":
        cell75_completed_case_count_valid,

    "author_ready_count_valid":
        cell75_author_ready_count_valid,

    "author_ready_identity_valid":
        cell75_author_ready_identity_valid,

    "persistence_valid":
        cell75_persistence_valid,

    "base_seed_unchanged":
        cell75_base_seed_unchanged,

    "full_572_blocked":
        cell75_full_572_blocked,

    "no_commit_push":
        cell75_no_commit_push,
}


cell75_gate = all(
    cell75_gate_checks.values()
)


# ============================================================
# 20) Diagnostics
# ============================================================

print(
    "=== PILOT #16 COMPLETE + QUEUE ADVANCE + NEXT PILOT ==="
)


print(
    "\n=== COMPLETED PILOT ==="
)

print(
    "Pilot:",
    CELL75_COMPLETED_PILOT_ID,
)

print(
    "Target:",
    CELL74_TARGET_EVIDENCE_ID,
)

print(
    "Question:",
    "ACCEPT",
)

print(
    "Expected Answer:",
    "ACCEPT",
)

print(
    "Citation:",
    "ACCEPT",
)

print(
    "Final Citation Mapping:",
    cell74_citation_mapping_id,
)

print(
    "Final Engine State:",
    cell75_complete_pending_session.get(
        "current_state"
    ),
)


print(
    "\n=== AUTHORITY RESULT ==="
)

print(
    "Primary Authority:",
    "Dr. Komeil Roudi",
)

print(
    "Secondary Speaker:",
    "فرهاد جم",
)

print(
    "Secondary used as Answer Evidence:",
    False,
)

print(
    "Human Authority Certification reused:",
    True,
)

print(
    "Raw Source mutated:",
    False,
)

print(
    "Parser mutated:",
    False,
)


print(
    "\n=== QUEUE ADVANCE ==="
)

print(
    "Before:",
    len(
        cell75_previous_remaining_pilot_ids
    ),
)

print(
    "After:",
    len(
        cell75_remaining_pilot_ids
    ),
)

print(
    "Queue advanced exactly once:",
    cell75_queue_advance_valid,
)


print(
    "\n=== HUMAN REVIEW REGISTRY ==="
)

print(
    "Completed Human Review cases:",
    len(
        cell75_completed_human_review_cases
    ),
)

print(
    "Cases ready for Author Review:",
    len(
        cell75_author_ready_cases
    ),
)

print(
    "Author Final Review:",
    "NOT_STARTED",
)

print(
    "Persistence valid:",
    cell75_persistence_valid,
)


print(
    "\n=== NEXT PILOT ==="
)

print(
    "Pilot:",
    CELL75_NEXT_PILOT_ID,
)

print(
    "Golden Case:",
    CELL75_NEXT_GOLDEN_CASE_ID,
)

print(
    "Task Draft:",
    CELL75_NEXT_TASK_DRAFT_ID,
)

print(
    "Target Evidence:",
    CELL75_NEXT_TARGET_EVIDENCE_ID,
)

print(
    "Stratum:",
    CELL75_NEXT_STRATUM,
)

print(
    "Task Family:",
    CELL75_NEXT_TASK_FAMILY,
)

print(
    "Question Archetype:",
    CELL75_NEXT_ARCHETYPE,
)

print(
    "Authority Expectation:",
    CELL75_NEXT_AUTHORITY_EXPECTATION,
)

print(
    "Workflow:",
    cell75_next_pilot_session.get(
        "current_state"
    ),
)


print(
    "\n=== NEXT SOURCE EVIDENCE ==="
)

print(
    cell75_next_combined_target_text
)


print(
    "\n=== CONTINUATION STATE ==="
)

print(
    "Remaining Pilots:",
    7,
)

print(
    "Continuation SHA256:",
    cell75_continuation_state_sha256,
)


print(
    "\n=== ALL INTERNAL GATES ==="
)

for name, status in cell75_gate_checks.items():

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== SAFETY ==="
)

print(
    "Base Seed unchanged:",
    cell75_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell75_full_572_blocked,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell75_failures
)


print(
    "\nCODE CELL 75 GATE:",
    (
        "PASS"
        if cell75_gate
        else "FAIL"
    ),
)


assert cell75_gate, (
    "Pilot #16 Citation ACCEPT / completion / queue advance / "
    "next Pilot preparation failed. Repair this SAME Code Cell 75."
)

=== PILOT #16 COMPLETE + QUEUE ADVANCE + NEXT PILOT ===

=== COMPLETED PILOT ===
Pilot: question-pilot:d9882d0f4166b051bca7f45505cfc753
Target: target-evidence:955f0cbf8cc5fd2a85bd1ec7e468bf8a
Question: ACCEPT
Expected Answer: ACCEPT
Citation: ACCEPT
Final Citation Mapping: citation-mapping:17f4d9072999bade1d0efaa9a3d84418
Final Engine State: PILOT_COMPLETE_PENDING_QUEUE_ADVANCE

=== AUTHORITY RESULT ===
Primary Authority: Dr. Komeil Roudi
Secondary Speaker: فرهاد جم
Secondary used as Answer Evidence: False
Human Authority Certification reused: True
Raw Source mutated: False
Parser mutated: False

=== QUEUE ADVANCE ===
Before: 8
After: 7
Queue advanced exactly once: True

=== HUMAN REVIEW REGISTRY ===
Completed Human Review cases: 15
Cases ready for Author Review: 15
Author Final Review: NOT_STARTED
Persistence valid: True

=== NEXT PILOT ===
Pilot: question-pilot:dc845f69631a7000938b161802bc3030
Golden Case: golden-case:b6fe9a260ae3e29f43e1d67e3da3c980
Task Draft: task-draft:820c84eb2

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 76 — Pilot #17 Multi-Speaker Attribution Diagnostic

Pilot فعلی از نوع:

`MULTI_SPEAKER_ATTRIBUTION_MIZAN`

است و Authority باید در سطح Utterance بررسی شود.

پیش از ساخت سؤال بررسی می‌کنیم:

- Part اول متعلق به چه Speakerای است؛
- پاسخ تفصیلی متعلق به چه Speakerای است؛
- آیا پاسخ از Dr. Komeil Roudi است یا خیر؛
- آیا Human Authority Certification قبلی قابل reuse است؛
- آیا Target فعلی واقعاً Multi-Speaker معتبر است؛
- کدام Part باید Primary Answer Evidence و کدام Part فقط Context باشد.

این Cell فقط Diagnostic است و هیچ داده‌ای را تغییر نمی‌دهد.

</div>

In [76]:
# Code Cell 76

cell76_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell75_gate") is not True:
    cell76_failures.append(
        "CELL75_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell75_next_target_text_parts"
    ),
    list,
):
    cell76_failures.append(
        "NEXT_TARGET_PARTS_MISSING"
    )


# ============================================================
# 2) Pilot identity
# ============================================================

CELL76_PILOT_ID = (
    CELL75_NEXT_PILOT_ID
)


CELL76_TARGET_EVIDENCE_ID = (
    CELL75_NEXT_TARGET_EVIDENCE_ID
)


CELL76_TASK_FAMILY = (
    CELL75_NEXT_TASK_FAMILY
)


CELL76_AUTHORITY_EXPECTATION = (
    CELL75_NEXT_AUTHORITY_EXPECTATION
)


# ============================================================
# 3) Inspect Target Parts
# ============================================================

cell76_part_diagnostics = []


for index, part in enumerate(
    cell75_next_target_text_parts
):

    if isinstance(
        part,
        dict,
    ):

        speaker_state = (
            part.get(
                "speaker_state"
            )
            or
            {}
        )


        attribution = (
            part.get(
                "attribution"
            )
            or
            {}
        )


        locator = (
            part.get(
                "locator"
            )
            or
            {}
        )


        text = (
            part.get(
                "text"
            )
            or
            part.get(
                "target_text"
            )
            or
            part.get(
                "content"
            )
            or
            ""
        )


        cell76_part_diagnostics.append({
            "index":
                index,

            "document_id":
                part.get(
                    "document_id"
                ),

            "role":
                part.get(
                    "role"
                ),

            "speaker_base_status":
                speaker_state.get(
                    "base_status"
                ),

            "speaker_base_raw":
                speaker_state.get(
                    "base_raw"
                ),

            "speaker_effective_label":
                speaker_state.get(
                    "effective_label"
                ),

            "speaker_effective_canonical":
                speaker_state.get(
                    "effective_canonical"
                ),

            "speaker_effective_primary":
                speaker_state.get(
                    "effective_primary"
                ),

            "attribution_speaker":
                attribution.get(
                    "speaker"
                ),

            "source_line_start":
                locator.get(
                    "source_line_start"
                ),

            "source_line_end":
                locator.get(
                    "source_line_end"
                ),

            "text":
                text,
        })


    else:

        cell76_part_diagnostics.append({
            "index":
                index,

            "document_id":
                None,

            "role":
                None,

            "speaker_base_status":
                None,

            "speaker_base_raw":
                None,

            "speaker_effective_label":
                None,

            "speaker_effective_canonical":
                None,

            "speaker_effective_primary":
                None,

            "attribution_speaker":
                None,

            "source_line_start":
                None,

            "source_line_end":
                None,

            "text":
                str(
                    part
                ),
        })


# ============================================================
# 4) Speaker inventory
# ============================================================

cell76_detected_speakers = []

cell76_primary_parts = []

cell76_unresolved_parts = []


for row in cell76_part_diagnostics:

    speaker = (
        row.get(
            "speaker_effective_canonical"
        )
        or
        row.get(
            "speaker_effective_label"
        )
        or
        row.get(
            "speaker_base_raw"
        )
        or
        row.get(
            "attribution_speaker"
        )
    )


    if speaker:

        cell76_detected_speakers.append(
            speaker
        )


    if (
        row.get(
            "speaker_effective_primary"
        )
        is True
    ):

        cell76_primary_parts.append(
            row
        )


    if (
        row.get(
            "speaker_base_status"
        )
        ==
        "UNRESOLVED"
        or
        speaker is None
    ):

        cell76_unresolved_parts.append(
            row
        )


cell76_distinct_speakers = sorted(
    set(
        cell76_detected_speakers
    )
)


# ============================================================
# 5) Inspect relevant metadata
# ============================================================

def cell76_collect_relevant_metadata(
    value,
    path=(),
):

    output = []


    keywords = (
        "speaker",
        "attribution",
        "authority",
        "guardrail",
        "multi",
        "contrast",
        "stratum",
        "task_family",
        "question_archetype",
    )


    if isinstance(
        value,
        dict,
    ):

        for key, item in value.items():

            key_text = str(
                key
            ).lower()


            if any(
                keyword
                in
                key_text
                for keyword
                in keywords
            ):

                output.append({
                    "path":
                        path
                        +
                        (
                            str(
                                key
                            ),
                        ),

                    "value":
                        copy.deepcopy(
                            item
                        ),
                })


            output.extend(
                cell76_collect_relevant_metadata(
                    item,
                    path
                    +
                    (
                        str(
                            key
                        ),
                    ),
                )
            )


    elif isinstance(
        value,
        list,
    ):

        for index, item in enumerate(
            value
        ):

            output.extend(
                cell76_collect_relevant_metadata(
                    item,
                    path
                    +
                    (
                        str(
                            index
                        ),
                    ),
                )
            )


    return output


cell76_relevant_metadata = (
    cell76_collect_relevant_metadata(
        cell75_next_pilot_packet
    )
)


# ============================================================
# 6) Check prior Human Authority Certification
# ============================================================

cell76_prior_authority_cert_available = (
    isinstance(
        globals().get(
            "cell71_human_authority_certification"
        ),
        dict,
    )
)


cell76_prior_authority_cert = (
    copy.deepcopy(
        cell71_human_authority_certification
    )
    if cell76_prior_authority_cert_available
    else {}
)


cell76_prior_cert_valid = all([
    cell76_prior_authority_cert_available,

    cell76_prior_authority_cert.get(
        "source_label"
    )
    ==
    "دکتر کمیل رودی",

    cell76_prior_authority_cert.get(
        "certified_canonical"
    )
    ==
    "Dr. Komeil Roudi",

    cell76_prior_authority_cert.get(
        "certified_primary"
    )
    is True,
])


# ============================================================
# 7) Structural validation
# ============================================================

cell76_is_expected_task = (
    CELL76_TASK_FAMILY
    ==
    "MULTI_SPEAKER_ATTRIBUTION_MIZAN"
)


cell76_authority_rule_valid = (
    CELL76_AUTHORITY_EXPECTATION
    ==
    "UTTERANCE_LEVEL_AUTHORITY"
)


cell76_diagnostic_structure_valid = all([
    len(
        cell76_part_diagnostics
    )
    >
    0,

    cell76_is_expected_task,

    cell76_authority_rule_valid,
])


if not cell76_diagnostic_structure_valid:
    cell76_failures.append(
        "DIAGNOSTIC_STRUCTURE_INVALID"
    )


# ============================================================
# 8) Safety
# ============================================================

cell76_registry_write = False

cell76_queue_advance = False

cell76_human_decision_added = False

cell76_target_changed = False

cell76_citation_changed = False

cell76_commit_push = False


# ============================================================
# 9) Gate
# ============================================================

cell76_gate = all([
    len(
        cell76_failures
    )
    ==
    0,

    cell76_diagnostic_structure_valid,
])


# ============================================================
# 10) Diagnostics
# ============================================================

print(
    "=== PILOT #17 MULTI-SPEAKER ATTRIBUTION DIAGNOSTIC ==="
)


print(
    "\n=== PILOT ==="
)

print(
    "Pilot:",
    CELL76_PILOT_ID,
)

print(
    "Target Evidence:",
    CELL76_TARGET_EVIDENCE_ID,
)

print(
    "Task Family:",
    CELL76_TASK_FAMILY,
)

print(
    "Authority Expectation:",
    CELL76_AUTHORITY_EXPECTATION,
)


print(
    "\n=== SPEAKER INVENTORY ==="
)

print(
    "Distinct detected speakers:",
    cell76_distinct_speakers,
)

print(
    "Primary-marked part count:",
    len(
        cell76_primary_parts
    ),
)

print(
    "Unresolved part count:",
    len(
        cell76_unresolved_parts
    ),
)

print(
    "Prior Dr. Roudi Human Authority Certification available:",
    cell76_prior_cert_valid,
)


print(
    "\n=== TARGET PARTS ==="
)


for row in cell76_part_diagnostics:

    print(
        "\n"
        +
        "="
        *
        72
    )

    print(
        "PART",
        row[
            "index"
        ],
    )

    print(
        "Document:",
        row.get(
            "document_id"
        ),
    )

    print(
        "Role:",
        row.get(
            "role"
        ),
    )

    print(
        "Base status:",
        row.get(
            "speaker_base_status"
        ),
    )

    print(
        "Base raw:",
        row.get(
            "speaker_base_raw"
        ),
    )

    print(
        "Effective label:",
        row.get(
            "speaker_effective_label"
        ),
    )

    print(
        "Effective canonical:",
        row.get(
            "speaker_effective_canonical"
        ),
    )

    print(
        "Effective primary:",
        row.get(
            "speaker_effective_primary"
        ),
    )

    print(
        "Attribution speaker:",
        row.get(
            "attribution_speaker"
        ),
    )

    print(
        "Source lines:",
        (
            row.get(
                "source_line_start"
            ),
            row.get(
                "source_line_end"
            ),
        ),
    )

    print(
        "Text:"
    )

    print(
        row.get(
            "text"
        )
    )


print(
    "\n=== RELEVANT PILOT METADATA ==="
)


for match in cell76_relevant_metadata:

    print(
        " > ".join(
            match[
                "path"
            ]
        ),
        "=",
        match[
            "value"
        ],
    )


print(
    "\n=== SAFETY ==="
)

print(
    "Registry write:",
    cell76_registry_write,
)

print(
    "Queue advance:",
    cell76_queue_advance,
)

print(
    "Human Decision added:",
    cell76_human_decision_added,
)

print(
    "Target changed:",
    cell76_target_changed,
)

print(
    "Citation changed:",
    cell76_citation_changed,
)

print(
    "Commit / Push:",
    cell76_commit_push,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell76_failures
)


print(
    "\nCODE CELL 76 GATE:",
    (
        "PASS"
        if cell76_gate
        else "FAIL"
    ),
)


assert cell76_gate, (
    "Pilot #17 multi-speaker attribution diagnostic failed. "
    "Repair this SAME Code Cell 76 before drafting the question."
)

=== PILOT #17 MULTI-SPEAKER ATTRIBUTION DIAGNOSTIC ===

=== PILOT ===
Pilot: question-pilot:dc845f69631a7000938b161802bc3030
Target Evidence: target-evidence:43487bce46faaf036499ac2921fd1280
Task Family: MULTI_SPEAKER_ATTRIBUTION_MIZAN
Authority Expectation: UTTERANCE_LEVEL_AUTHORITY

=== SPEAKER INVENTORY ===
Distinct detected speakers: ['یکی از حاضران']
Primary-marked part count: 0
Unresolved part count: 1
Prior Dr. Roudi Human Authority Certification available: True

=== TARGET PARTS ===

PART 0
Document: program:mizan:unit-001:utterance:00034
Role: ATTRIBUTION_CONTRAST_LEFT
Base status: NARRATIVE_CUE
Base raw: یکی از حاضران
Effective label: یکی از حاضران
Effective canonical: None
Effective primary: None
Attribution speaker: None
Source lines: (85, 85)
Text:
یکی از حاضران اشاره می‌کند که در متون دینی «الفقر فخری» هم نقل شده است.

PART 1
Document: program:mizan:unit-001:utterance:00035
Role: ATTRIBUTION_CONTRAST_RIGHT
Base status: UNRESOLVED
Base raw: None
Effective label: None
Effec

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 77 — Pilot #17 Raw Source Speaker Context Diagnostic

Part دوم Target فعلی Speaker مشخصی ندارد و با وضعیت:

`UNRESOLVED`

ثبت شده است.

با توجه به Guardrail:

`unknown_speaker_inference = FORBIDDEN`

نمی‌توان فقط از محتوای پاسخ یا عبارت «بله» نتیجه گرفت که گوینده Dr. Komeil Roudi است.

در این Cell متن خام Source در اطراف Utteranceهای 34 و 35 بررسی می‌شود تا مشخص شود:

- آخرین Speaker label صریح پیش از پاسخ چه کسی است؛
- آیا بین آن Speaker و پاسخ، Speaker جدیدی معرفی شده است یا نه؛
- آیا قاعده Human-certified speaker continuity قابل اعمال است یا خیر.

این Cell فقط Diagnostic است و هیچ Registry، Queue، Target، Citation یا Human Decision را تغییر نمی‌دهد.

</div>

In [77]:
# Code Cell 77

from pathlib import Path


cell77_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell76_gate") is not True:
    cell77_failures.append(
        "CELL76_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell75_next_target_text_parts"
    ),
    list,
):
    cell77_failures.append(
        "TARGET_PARTS_MISSING"
    )


# ============================================================
# 2) Resolve repository root
# ============================================================

cell77_repo_root = None


for candidate in [
    cell21_registry_path.parent,
    *cell21_registry_path.parents,
]:

    if (
        (
            candidate
            /
            "data"
        ).exists()
        and
        (
            candidate
            /
            "notebooks"
        ).exists()
    ):

        cell77_repo_root = candidate
        break


cell77_repo_root_valid = (
    isinstance(
        cell77_repo_root,
        Path,
    )
    and
    cell77_repo_root.exists()
)


if not cell77_repo_root_valid:
    cell77_failures.append(
        "REPO_ROOT_NOT_FOUND"
    )


# ============================================================
# 3) Target snippets
# ============================================================

cell77_part0_text = (
    cell75_next_target_text_parts[
        0
    ].get(
        "text",
        "",
    )
)


cell77_part1_text = (
    cell75_next_target_text_parts[
        1
    ].get(
        "text",
        "",
    )
)


CELL77_SEARCH_SNIPPET = (
    "بله؛ باید این دو مفهوم را از هم تفکیک کنیم."
)


# ============================================================
# 4) Normalization helper
# ============================================================

def cell77_norm(
    text,
):

    if not isinstance(
        text,
        str,
    ):
        return ""


    value = (
        text
        .replace(
            "ي",
            "ی",
        )
        .replace(
            "ك",
            "ک",
        )
        .replace(
            "\u200c",
            "",
        )
        .replace(
            "\ufeff",
            "",
        )
    )


    value = re.sub(
        r"\s+",
        " ",
        value,
    )


    return value.strip()


cell77_search_norm = cell77_norm(
    CELL77_SEARCH_SNIPPET
)


# ============================================================
# 5) Candidate raw-source roots
# ============================================================

cell77_search_roots = []


for relative_path in [
    Path(
        "data/transcripts/final"
    ),
    Path(
        "data/transcripts"
    ),
    Path(
        "data"
    ),
]:

    candidate = (
        cell77_repo_root
        /
        relative_path
        if cell77_repo_root_valid
        else None
    )


    if (
        candidate is not None
        and
        candidate.exists()
        and
        candidate not in cell77_search_roots
    ):

        cell77_search_roots.append(
            candidate
        )


if not cell77_search_roots:
    cell77_failures.append(
        "NO_SEARCH_ROOTS"
    )


# ============================================================
# 6) Search raw text files
# ============================================================

cell77_matches = []

cell77_seen_paths = set()


for search_root in cell77_search_roots:

    for suffix in [
        "*.md",
        "*.txt",
    ]:

        for path in search_root.rglob(
            suffix
        ):

            resolved = str(
                path.resolve()
            )


            if resolved in cell77_seen_paths:
                continue


            cell77_seen_paths.add(
                resolved
            )


            try:

                raw_text = path.read_text(
                    encoding="utf-8"
                )


            except Exception:

                continue


            raw_norm = cell77_norm(
                raw_text
            )


            if (
                cell77_search_norm
                not in
                raw_norm
            ):

                continue


            lines = raw_text.splitlines()


            matching_line_numbers = []


            for line_index, line in enumerate(
                lines,
                start=1,
            ):

                if (
                    cell77_search_norm
                    in
                    cell77_norm(
                        line
                    )
                ):

                    matching_line_numbers.append(
                        line_index
                    )


            cell77_matches.append({
                "path":
                    path,

                "line_numbers":
                    matching_line_numbers,

                "lines":
                    lines,
            })


# ============================================================
# 7) Unique source validation
# ============================================================

cell77_unique_source_paths = sorted(
    {
        str(
            match[
                "path"
            ].resolve()
        )
        for match
        in cell77_matches
    }
)


cell77_unique_source_valid = (
    len(
        cell77_unique_source_paths
    )
    ==
    1
)


if not cell77_unique_source_valid:
    cell77_failures.append(
        "RAW_SOURCE_NOT_UNIQUE"
    )


cell77_source_match = (
    cell77_matches[
        0
    ]
    if cell77_unique_source_valid
    else None
)


# ============================================================
# 8) Determine context window
# ============================================================

cell77_context_rows = []


if cell77_source_match is not None:

    cell77_lines = (
        cell77_source_match[
            "lines"
        ]
    )


    match_lines = (
        cell77_source_match[
            "line_numbers"
        ]
    )


    if match_lines:

        cell77_anchor_line = (
            match_lines[
                0
            ]
        )


    else:

        cell77_anchor_line = 87


    cell77_context_start = max(
        1,
        cell77_anchor_line
        -
        12,
    )


    cell77_context_end = min(
        len(
            cell77_lines
        ),
        cell77_anchor_line
        +
        12,
    )


    for line_number in range(
        cell77_context_start,
        cell77_context_end
        +
        1,
    ):

        text = (
            cell77_lines[
                line_number
                -
                1
            ]
        )


        cell77_context_rows.append({
            "line_number":
                line_number,

            "text":
                text,
        })


# ============================================================
# 9) Speaker-label candidates in context
# ============================================================

cell77_label_candidates = []


speaker_label_pattern = re.compile(
    r"^\s*([^:：]{1,80})\s*[:：]\s*(.*)$"
)


for row in cell77_context_rows:

    match = speaker_label_pattern.match(
        row[
            "text"
        ]
    )


    if not match:
        continue


    possible_label = (
        match.group(
            1
        ).strip()
    )


    possible_content = (
        match.group(
            2
        ).strip()
    )


    cell77_label_candidates.append({
        "line_number":
            row[
                "line_number"
            ],

        "label":
            possible_label,

        "content":
            possible_content,
    })


# ============================================================
# 10) Locate Target statements in context
# ============================================================

cell77_part0_context_lines = []

cell77_part1_context_lines = []


part0_norm = cell77_norm(
    cell77_part0_text
)


part1_norm = cell77_norm(
    cell77_part1_text
)


for row in cell77_context_rows:

    line_norm = cell77_norm(
        row[
            "text"
        ]
    )


    if (
        line_norm
        and
        (
            line_norm in part0_norm
            or
            part0_norm in line_norm
        )
    ):

        cell77_part0_context_lines.append(
            row[
                "line_number"
            ]
        )


    if (
        line_norm
        and
        (
            line_norm in part1_norm
            or
            part1_norm in line_norm
        )
    ):

        cell77_part1_context_lines.append(
            row[
                "line_number"
            ]
        )


# ============================================================
# 11) Safety
# ============================================================

cell77_registry_write = False

cell77_queue_advance = False

cell77_human_decision_added = False

cell77_target_changed = False

cell77_citation_changed = False

cell77_raw_source_mutated = False

cell77_parser_mutated = False

cell77_commit_push = False


# ============================================================
# 12) Gate
# ============================================================

cell77_gate_checks = {
    "no_failures":
        len(
            cell77_failures
        )
        ==
        0,

    "repo_root_valid":
        cell77_repo_root_valid,

    "unique_raw_source":
        cell77_unique_source_valid,

    "context_available":
        len(
            cell77_context_rows
        )
        >
        0,
}


cell77_gate = all(
    cell77_gate_checks.values()
)


# ============================================================
# 13) Diagnostics
# ============================================================

print(
    "=== PILOT #17 RAW SOURCE SPEAKER CONTEXT DIAGNOSTIC ==="
)


print(
    "\n=== SOURCE RESOLUTION ==="
)

print(
    "Repository Root:",
    cell77_repo_root,
)

print(
    "Matched Source Count:",
    len(
        cell77_unique_source_paths
    ),
)

print(
    "Matched Sources:",
)

for path in cell77_unique_source_paths:

    print(
        "-",
        path,
    )


if cell77_source_match is not None:

    print(
        "\nResolved Source:",
        cell77_source_match[
            "path"
        ],
    )

    print(
        "Matched Line Numbers:",
        cell77_source_match[
            "line_numbers"
        ],
    )


print(
    "\n=== RAW SOURCE CONTEXT ==="
)


for row in cell77_context_rows:

    marker = (
        ">>>"
        if (
            row[
                "line_number"
            ]
            in
            (
                cell77_source_match[
                    "line_numbers"
                ]
                if
                cell77_source_match
                is not None
                else
                []
            )
        )
        else
        "   "
    )


    print(
        f"{marker} L{row['line_number']:04d}: "
        f"{row['text']}"
    )


print(
    "\n=== SPEAKER-LABEL CANDIDATES ==="
)


if cell77_label_candidates:

    for item in cell77_label_candidates:

        print(
            f"L{item['line_number']}:",
            repr(
                item[
                    "label"
                ]
            ),
            "->",
            item[
                "content"
            ][
                :160
            ],
        )


else:

    print(
        "No colon-style speaker labels detected "
        "inside the context window."
    )


print(
    "\n=== TARGET LOCATION CHECK ==="
)

print(
    "Part 0 context lines:",
    cell77_part0_context_lines,
)

print(
    "Part 1 context lines:",
    cell77_part1_context_lines,
)


print(
    "\n=== IMPORTANT RULE ==="
)

print(
    "No speaker inference has been made."
)

print(
    "If the raw source shows an explicit Dr. Roudi label "
    "before Part 1 with no intervening speaker change, "
    "Human-certified speaker continuity may be applied."
)


print(
    "\n=== ALL INTERNAL GATES ==="
)

for name, status in cell77_gate_checks.items():

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== SAFETY ==="
)

print(
    "Registry write:",
    cell77_registry_write,
)

print(
    "Queue advance:",
    cell77_queue_advance,
)

print(
    "Human Decision added:",
    cell77_human_decision_added,
)

print(
    "Target changed:",
    cell77_target_changed,
)

print(
    "Citation changed:",
    cell77_citation_changed,
)

print(
    "Raw Source mutated:",
    cell77_raw_source_mutated,
)

print(
    "Parser mutated:",
    cell77_parser_mutated,
)

print(
    "Commit / Push:",
    cell77_commit_push,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell77_failures
)


print(
    "\nCODE CELL 77 GATE:",
    (
        "PASS"
        if cell77_gate
        else
        "FAIL"
    ),
)


assert cell77_gate, (
    "Pilot #17 raw-source speaker context diagnostic failed. "
    "Repair this SAME Code Cell 77 before any speaker adjudication."
)

=== PILOT #17 RAW SOURCE SPEAKER CONTEXT DIAGNOSTIC ===

=== SOURCE RESOLUTION ===
Repository Root: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance
Matched Source Count: 1
Matched Sources:
- /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/data/transcripts/final/chehelstion_finup_ketabbaz_bankmellat/میزان.md

Resolved Source: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/data/transcripts/final/chehelstion_finup_ketabbaz_bankmellat/میزان.md
Matched Line Numbers: [87]

=== RAW SOURCE CONTEXT ===
    L0075: 
    L0076: «فریدونِ فرخ فرشته نبود / ز مشک و ز عنبر سرشته نبود
    L0077: به داد و دهش یافت آن نیکویی / تو داد و دهش کن، فریدون تویی»
    L0078: 
    L0079: دهندگی ظرفیت عاطفی هم ایجاد می‌کند. ثبات مالی به ما تمرکز فکری و امکان نگاه آینده‌نگر می‌دهد؛ فرصت می‌دهد خلاقانه، سیستمی و کل‌نگر فکر کنیم. بخشی از ظرفیت فکرکردن ما در عالم واقعی با امنیت و فراغ بال مالی تقویت می‌شود.
    L0080: 
    L0081: در روایات ه

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 78 — Pilot #17 Upstream Speaker Lineage Diagnostic

در Raw Source، پاسخ Target بدون Speaker label صریح آمده است.

بنابراین برای جلوگیری از `unsupported_attribution` باید بررسی شود:

- نزدیک‌ترین Speaker marker معتبر قبل از Utterance 35 چیست؛
- Utteranceهای اطراف آن در `transcripts_extracted_v1.jsonl` چه Speaker stateای دارند؛
- آیا یک Speaker turn معتبر از Dr. Komeil Roudi پیش از این بخش شروع شده است؛
- آیا بین آن Speaker و Target، Speaker جدیدی معرفی شده یا نه.

این Cell فقط Diagnostic است و هیچ Human Decision، Registry، Queue، Target یا Citation را تغییر نمی‌دهد.

</div>

In [78]:
# Code Cell 78

cell78_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell77_gate") is not True:
    cell78_failures.append(
        "CELL77_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell77_source_match"
    ),
    dict,
):
    cell78_failures.append(
        "RAW_SOURCE_MATCH_MISSING"
    )


# ============================================================
# 2) Raw source
# ============================================================

cell78_source_path = (
    cell77_source_match.get(
        "path"
    )
    if isinstance(
        globals().get(
            "cell77_source_match"
        ),
        dict,
    )
    else None
)


cell78_source_lines = (
    cell77_source_match.get(
        "lines",
        [],
    )
    if isinstance(
        globals().get(
            "cell77_source_match"
        ),
        dict,
    )
    else []
)


cell78_source_valid = all([
    cell78_source_path is not None,
    len(
        cell78_source_lines
    )
    >
    0,
])


if not cell78_source_valid:
    cell78_failures.append(
        "RAW_SOURCE_INVALID"
    )


# ============================================================
# 3) Wider upstream raw context
# ============================================================

CELL78_TARGET_LINE = 87

CELL78_RAW_START = max(
    1,
    CELL78_TARGET_LINE - 45,
)

CELL78_RAW_END = min(
    len(
        cell78_source_lines
    ),
    CELL78_TARGET_LINE + 15,
)


cell78_wide_context = []


for line_number in range(
    CELL78_RAW_START,
    CELL78_RAW_END + 1,
):

    cell78_wide_context.append({
        "line_number":
            line_number,

        "text":
            cell78_source_lines[
                line_number - 1
            ],
    })


# ============================================================
# 4) Speaker-related raw cues
# ============================================================

CELL78_SPEAKER_TERMS = [
    "دکتر رودی",
    "دکتر کمیل رودی",
    "کمیل رودی",
    "کمیل",
    "رودی",
    "یکی از حاضران",
    "حاضر",
    "مجری",
    "میزبان",
    "شرکت‌کننده",
    "شرکت کننده",
]


cell78_raw_speaker_cues = []


for row in cell78_wide_context:

    normalized = cell77_norm(
        row[
            "text"
        ]
    )


    matched_terms = [
        term
        for term
        in CELL78_SPEAKER_TERMS
        if cell77_norm(
            term
        )
        in normalized
    ]


    if matched_terms:

        cell78_raw_speaker_cues.append({
            "line_number":
                row[
                    "line_number"
                ],

            "matched_terms":
                matched_terms,

            "text":
                row[
                    "text"
                ],
        })


# ============================================================
# 5) Nearest prior speaker-related raw cue
# ============================================================

cell78_prior_raw_cues = [
    item
    for item
    in cell78_raw_speaker_cues
    if item[
        "line_number"
    ]
    <
    CELL78_TARGET_LINE
]


cell78_nearest_prior_raw_cue = (
    max(
        cell78_prior_raw_cues,
        key=lambda item: item[
            "line_number"
        ],
    )
    if cell78_prior_raw_cues
    else None
)


# ============================================================
# 6) Extracted transcript registry path
# ============================================================

cell78_extracted_path = (
    cell21_registry_path.parents[
        1
    ]
    /
    "interim"
    /
    "transcripts_extracted_v1.jsonl"
)


cell78_extracted_path_valid = (
    cell78_extracted_path.exists()
)


if not cell78_extracted_path_valid:
    cell78_failures.append(
        "EXTRACTED_TRANSCRIPT_JSONL_MISSING"
    )


# ============================================================
# 7) Load Mizan extracted records
# ============================================================

cell78_mizan_records = []


if cell78_extracted_path_valid:

    with cell78_extracted_path.open(
        "r",
        encoding="utf-8",
    ) as handle:

        for raw_line in handle:

            raw_line = raw_line.strip()


            if not raw_line:
                continue


            try:

                record = json.loads(
                    raw_line
                )


            except Exception:

                continue


            record_id = str(
                record.get(
                    "utterance_id",
                    "",
                )
            )


            if record_id.startswith(
                "program:mizan:unit-001:utterance:"
            ):

                cell78_mizan_records.append(
                    record
                )


# ============================================================
# 8) Resolve utterance index helper
# ============================================================

def cell78_utterance_index(
    record,
):

    direct = record.get(
        "utterance_index"
    )


    if isinstance(
        direct,
        int,
    ):
        return direct


    utterance_id = str(
        record.get(
            "utterance_id",
            "",
        )
    )


    match = re.search(
        r"utterance:(\d+)$",
        utterance_id,
    )


    if match:

        return int(
            match.group(
                1
            )
        )


    return None


# ============================================================
# 9) Extract neighborhood around target
# ============================================================

cell78_neighbor_records = []


for record in cell78_mizan_records:

    index = cell78_utterance_index(
        record
    )


    if (
        isinstance(
            index,
            int,
        )
        and
        24
        <=
        index
        <=
        42
    ):

        speaker_state = (
            record.get(
                "speaker_state"
            )
            or
            {}
        )


        cell78_neighbor_records.append({
            "utterance_index":
                index,

            "utterance_id":
                record.get(
                    "utterance_id"
                ),

            "source_text":
                (
                    record.get(
                        "source_text"
                    )
                    or
                    record.get(
                        "text"
                    )
                    or
                    ""
                ),

            "base_status":
                speaker_state.get(
                    "base_status"
                ),

            "base_raw":
                speaker_state.get(
                    "base_raw"
                ),

            "effective_label":
                speaker_state.get(
                    "effective_label"
                ),

            "effective_canonical":
                speaker_state.get(
                    "effective_canonical"
                ),

            "effective_primary":
                speaker_state.get(
                    "effective_primary"
                ),

            "source_locator":
                copy.deepcopy(
                    record.get(
                        "source_locator"
                    )
                    or
                    {}
                ),
        })


cell78_neighbor_records = sorted(
    cell78_neighbor_records,
    key=lambda item: item[
        "utterance_index"
    ],
)


# ============================================================
# 10) Find nearest prior explicit Dr marker in extracted data
# ============================================================

cell78_prior_explicit_dr_candidates = []


for row in cell78_neighbor_records:

    if row[
        "utterance_index"
    ] >= 35:
        continue


    raw_label = (
        row.get(
            "base_raw"
        )
        or
        row.get(
            "effective_label"
        )
        or
        ""
    )


    normalized_label = cell77_norm(
        raw_label
    )


    if (
        "دکتر"
        in normalized_label
        and
        (
            "رودی"
            in normalized_label
            or
            "کمیل"
            in normalized_label
        )
    ):

        cell78_prior_explicit_dr_candidates.append(
            row
        )


cell78_nearest_prior_explicit_dr = (
    max(
        cell78_prior_explicit_dr_candidates,
        key=lambda item: item[
            "utterance_index"
        ],
    )
    if cell78_prior_explicit_dr_candidates
    else None
)


# ============================================================
# 11) Find intervening speaker markers
# ============================================================

cell78_intervening_records = []


if cell78_nearest_prior_explicit_dr is not None:

    start_index = (
        cell78_nearest_prior_explicit_dr[
            "utterance_index"
        ]
    )


    for row in cell78_neighbor_records:

        if not (
            start_index
            <
            row[
                "utterance_index"
            ]
            <
            35
        ):
            continue


        if row.get(
            "base_status"
        ) in [
            "EXPLICIT_SOURCE_MARKER",
            "NARRATIVE_CUE",
        ]:

            cell78_intervening_records.append(
                row
            )


# ============================================================
# 12) Structural indicators only
# ============================================================

cell78_has_prior_explicit_dr = (
    cell78_nearest_prior_explicit_dr
    is not None
)


cell78_has_intervening_speaker_marker = (
    len(
        cell78_intervening_records
    )
    >
    0
)


cell78_continuity_candidate = all([
    cell78_has_prior_explicit_dr,
    not cell78_has_intervening_speaker_marker,
])


# ============================================================
# 13) Safety
# ============================================================

cell78_registry_write = False

cell78_queue_advance = False

cell78_human_decision_added = False

cell78_target_changed = False

cell78_citation_changed = False

cell78_speaker_certified = False

cell78_raw_source_mutated = False

cell78_parser_mutated = False

cell78_commit_push = False


# ============================================================
# 14) Gate
# ============================================================

cell78_gate_checks = {
    "no_failures":
        len(
            cell78_failures
        )
        ==
        0,

    "raw_source_valid":
        cell78_source_valid,

    "extracted_path_valid":
        cell78_extracted_path_valid,

    "mizan_records_found":
        len(
            cell78_mizan_records
        )
        >
        0,

    "neighbor_records_found":
        len(
            cell78_neighbor_records
        )
        >
        0,
}


cell78_gate = all(
    cell78_gate_checks.values()
)


# ============================================================
# 15) Diagnostics
# ============================================================

print(
    "=== PILOT #17 UPSTREAM SPEAKER LINEAGE DIAGNOSTIC ==="
)


print(
    "\n=== RAW SOURCE ==="
)

print(
    "Source:",
    cell78_source_path,
)

print(
    "Context lines:",
    (
        CELL78_RAW_START,
        CELL78_RAW_END,
    ),
)


print(
    "\n=== RAW SPEAKER-RELATED CUES ==="
)


if cell78_raw_speaker_cues:

    for item in cell78_raw_speaker_cues:

        print(
            f"L{item['line_number']}:",
            item[
                "matched_terms"
            ],
            "->",
            item[
                "text"
            ],
        )


else:

    print(
        "No speaker-related raw cues found."
    )


print(
    "\nNearest prior raw cue:"
)

print(
    cell78_nearest_prior_raw_cue
)


print(
    "\n=== EXTRACTED UTTERANCE NEIGHBORHOOD ==="
)


for row in cell78_neighbor_records:

    print(
        "\n"
        +
        "-"
        *
        72
    )

    print(
        "Utterance:",
        row[
            "utterance_index"
        ],
    )

    print(
        "ID:",
        row.get(
            "utterance_id"
        ),
    )

    print(
        "Base status:",
        row.get(
            "base_status"
        ),
    )

    print(
        "Base raw:",
        row.get(
            "base_raw"
        ),
    )

    print(
        "Effective label:",
        row.get(
            "effective_label"
        ),
    )

    print(
        "Effective canonical:",
        row.get(
            "effective_canonical"
        ),
    )

    print(
        "Effective primary:",
        row.get(
            "effective_primary"
        ),
    )

    print(
        "Text:",
        row.get(
            "source_text"
        ),
    )


print(
    "\n=== PRIOR DR. ROUDI MARKER ==="
)

print(
    "Found:",
    cell78_has_prior_explicit_dr,
)

print(
    "Nearest prior explicit Dr. marker:"
)

print(
    cell78_nearest_prior_explicit_dr
)


print(
    "\n=== INTERVENING SPEAKER MARKERS ==="
)

print(
    "Count:",
    len(
        cell78_intervening_records
    ),
)


for row in cell78_intervening_records:

    print(
        row
    )


print(
    "\n=== CONTINUITY CANDIDATE ==="
)

print(
    "Prior explicit Dr. marker exists:",
    cell78_has_prior_explicit_dr,
)

print(
    "Intervening speaker marker exists:",
    cell78_has_intervening_speaker_marker,
)

print(
    "Human continuity candidate:",
    cell78_continuity_candidate,
)


print(
    "\nIMPORTANT:"
)

print(
    "No speaker certification has been made in this Cell."
)

print(
    "This Cell only determines whether sufficient source lineage "
    "exists for a later Human-certified attribution decision."
)


print(
    "\n=== ALL INTERNAL GATES ==="
)

for name, status in cell78_gate_checks.items():

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== SAFETY ==="
)

print(
    "Registry write:",
    cell78_registry_write,
)

print(
    "Queue advance:",
    cell78_queue_advance,
)

print(
    "Human Decision added:",
    cell78_human_decision_added,
)

print(
    "Speaker certified:",
    cell78_speaker_certified,
)

print(
    "Target changed:",
    cell78_target_changed,
)

print(
    "Citation changed:",
    cell78_citation_changed,
)

print(
    "Raw Source mutated:",
    cell78_raw_source_mutated,
)

print(
    "Parser mutated:",
    cell78_parser_mutated,
)

print(
    "Commit / Push:",
    cell78_commit_push,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell78_failures
)


print(
    "\nCODE CELL 78 GATE:",
    (
        "PASS"
        if cell78_gate
        else "FAIL"
    ),
)


assert cell78_gate, (
    "Pilot #17 upstream speaker lineage diagnostic failed. "
    "Repair this SAME Code Cell 78 before attribution."
)

=== PILOT #17 UPSTREAM SPEAKER LINEAGE DIAGNOSTIC ===

=== RAW SOURCE ===
Source: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/data/transcripts/final/chehelstion_finup_ketabbaz_bankmellat/میزان.md
Context lines: (42, 102)

=== RAW SPEAKER-RELATED CUES ===
L85: ['یکی از حاضران', 'حاضر'] -> یکی از حاضران اشاره می‌کند که در متون دینی «الفقر فخری» هم نقل شده است.
L89: ['یکی از حاضران', 'حاضر'] -> یکی از حاضران می‌گوید: پس باید شعارها و ذهنیت‌ها عوض شوند.
L97: ['یکی از حاضران', 'حاضر'] -> یکی از حاضران می‌گوید: یعنی پول لازم است ولی کافی نیست؟

Nearest prior raw cue:
{'line_number': 85, 'matched_terms': ['یکی از حاضران', 'حاضر'], 'text': 'یکی از حاضران اشاره می\u200cکند که در متون دینی «الفقر فخری» هم نقل شده است.'}

=== EXTRACTED UTTERANCE NEIGHBORHOOD ===

------------------------------------------------------------------------
Utterance: 24
ID: program:mizan:unit-001:utterance:00024
Base status: None
Base raw: None
Effective label: None
Effective canonical: N

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 79 — Pilot #17 Mizan Speaker-Structure / Retarget Candidate Discovery

بررسی Cell 78 نشان داد Target فعلی برای Attribution امن نیست:

- Part اول مربوط به «یکی از حاضران» است؛
- Part دوم Speaker مشخصی ندارد؛
- هیچ Speaker marker معتبر برای Dr. Komeil Roudi پیش از پاسخ پیدا نشد؛
- Guardrailهای `unknown_speaker_inference` و `unsupported_attribution` هرگونه حدس را ممنوع می‌کنند.

بنابراین Target فعلی Candidate مناسبی برای Golden Dataset نیست و باید Retarget شود.

در این Cell، کل Source «میزان» بررسی می‌شود تا مشخص شود:

- آیا در جای دیگری برچسب صریح برای دکتر رودی وجود دارد؛
- چه Speaker cueهای صریح یا روایی در Source وجود دارند؛
- چه رفت‌وبرگشت‌های Multi-Speaker بالقوه‌ای وجود دارد؛
- آیا Candidate معتبر برای جایگزینی Target فعلی در همین Source پیدا می‌شود.

این Cell فقط Discovery است و هنوز هیچ Human RETARGET Decision، Registry write یا Queue advance انجام نمی‌دهد.

</div>

In [79]:
# Code Cell 79

cell79_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell78_gate") is not True:
    cell79_failures.append(
        "CELL78_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell78_source_lines"
    ),
    list,
):
    cell79_failures.append(
        "MIZAN_RAW_SOURCE_LINES_MISSING"
    )


# ============================================================
# 2) Pilot identity
# ============================================================

CELL79_PILOT_ID = (
    CELL76_PILOT_ID
)


CELL79_OLD_TARGET_EVIDENCE_ID = (
    CELL76_TARGET_EVIDENCE_ID
)


CELL79_TASK_FAMILY = (
    CELL76_TASK_FAMILY
)


CELL79_STRATUM = (
    CELL75_NEXT_STRATUM
)


# ============================================================
# 3) Source
# ============================================================

cell79_source_path = (
    cell78_source_path
)


cell79_source_lines = copy.deepcopy(
    cell78_source_lines
)


cell79_source_valid = all([
    cell79_source_path is not None,

    len(
        cell79_source_lines
    )
    >
    0,
])


if not cell79_source_valid:
    cell79_failures.append(
        "MIZAN_SOURCE_INVALID"
    )


# ============================================================
# 4) Normalization
# ============================================================

def cell79_norm(
    text,
):

    if not isinstance(
        text,
        str,
    ):
        return ""


    value = (
        text
        .replace(
            "ي",
            "ی",
        )
        .replace(
            "ك",
            "ک",
        )
        .replace(
            "\u200c",
            "",
        )
        .replace(
            "\ufeff",
            "",
        )
    )


    value = re.sub(
        r"\s+",
        " ",
        value,
    )


    return value.strip()


# ============================================================
# 5) Detect explicit colon-style speaker labels
# ============================================================

cell79_colon_labels = []


cell79_colon_pattern = re.compile(
    r"^\s*([^:：]{1,100})\s*[:：]\s*(.+)$"
)


for line_number, line in enumerate(
    cell79_source_lines,
    start=1,
):

    match = cell79_colon_pattern.match(
        line
    )


    if not match:
        continue


    label = match.group(
        1
    ).strip()


    content = match.group(
        2
    ).strip()


    label_norm = cell79_norm(
        label
    )


    likely_speaker = any(
        token
        in
        label_norm
        for token
        in [
            "دکتر",
            "رودی",
            "کمیل",
            "مجری",
            "میزبان",
            "حاضر",
            "شرکتکننده",
            "شرکت کننده",
            "استاد",
        ]
    )


    if likely_speaker:

        cell79_colon_labels.append({
            "line_number":
                line_number,

            "label":
                label,

            "content":
                content,
        })


# ============================================================
# 6) Detect Dr. Roudi mentions anywhere
# ============================================================

cell79_dr_mentions = []


for line_number, line in enumerate(
    cell79_source_lines,
    start=1,
):

    norm = cell79_norm(
        line
    )


    matched_terms = []


    for term in [
        "دکتر کمیل رودی",
        "دکتر رودی",
        "کمیل رودی",
        "دکتر کمیل",
        "رودی",
    ]:

        if cell79_norm(
            term
        ) in norm:

            matched_terms.append(
                term
            )


    if matched_terms:

        cell79_dr_mentions.append({
            "line_number":
                line_number,

            "matched_terms":
                matched_terms,

            "text":
                line,
        })


# ============================================================
# 7) Detect participant / audience narrative cues
# ============================================================

cell79_participant_cues = []


CELL79_PARTICIPANT_PATTERNS = [
    "یکی از حاضران",
    "یکی از شرکتکنندگان",
    "یکی از شرکت کنندگان",
    "یکی از دوستان",
    "یکی از حضار",
]


for line_number, line in enumerate(
    cell79_source_lines,
    start=1,
):

    norm = cell79_norm(
        line
    )


    matched = [
        pattern
        for pattern
        in CELL79_PARTICIPANT_PATTERNS
        if cell79_norm(
            pattern
        )
        in norm
    ]


    if matched:

        cell79_participant_cues.append({
            "line_number":
                line_number,

            "matched_patterns":
                matched,

            "text":
                line,
        })


# ============================================================
# 8) Detect dialogue-like audience cue -> response windows
# ============================================================

cell79_dialogue_candidates = []


for cue in cell79_participant_cues:

    cue_line = cue[
        "line_number"
    ]


    # Find next non-empty line after participant cue.
    next_nonempty_line = None
    next_nonempty_text = None


    for probe in range(
        cue_line + 1,
        min(
            len(
                cell79_source_lines
            ),
            cue_line + 8,
        )
        + 1,
    ):

        text = (
            cell79_source_lines[
                probe - 1
            ]
        )


        if text.strip():

            next_nonempty_line = probe
            next_nonempty_text = text
            break


    if next_nonempty_line is None:
        continue


    # Find the next participant/speaker cue after the response start.
    following_boundary_line = None


    for probe in range(
        next_nonempty_line + 1,
        min(
            len(
                cell79_source_lines
            ),
            next_nonempty_line + 20,
        )
        + 1,
    ):

        text = (
            cell79_source_lines[
                probe - 1
            ]
        )


        norm = cell79_norm(
            text
        )


        participant_boundary = any(
            cell79_norm(
                pattern
            )
            in
            norm
            for pattern
            in CELL79_PARTICIPANT_PATTERNS
        )


        colon_boundary = bool(
            cell79_colon_pattern.match(
                text
            )
        )


        if (
            participant_boundary
            or
            colon_boundary
        ):

            following_boundary_line = probe
            break


    response_end = (
        (
            following_boundary_line
            -
            1
        )
        if following_boundary_line
        is not None
        else
        min(
            len(
                cell79_source_lines
            ),
            next_nonempty_line + 8,
        )
    )


    response_lines = []


    for probe in range(
        next_nonempty_line,
        response_end + 1,
    ):

        text = (
            cell79_source_lines[
                probe - 1
            ]
        )


        if text.strip():

            response_lines.append(
                text.strip()
            )


    response_text = (
        "\n".join(
            response_lines
        )
    )


    cell79_dialogue_candidates.append({
        "cue_line":
            cue_line,

        "cue_text":
            cue[
                "text"
            ],

        "response_start":
            next_nonempty_line,

        "response_end":
            response_end,

        "response_text":
            response_text,

        "next_boundary_line":
            following_boundary_line,
    })


# ============================================================
# 9) Search for explicit Dr marker near each dialogue candidate
# ============================================================

for candidate in cell79_dialogue_candidates:

    cue_line = candidate[
        "cue_line"
    ]


    search_start = max(
        1,
        cue_line - 25,
    )


    search_end = min(
        len(
            cell79_source_lines
        ),
        candidate[
            "response_end"
        ],
    )


    explicit_dr_markers = []


    for probe in range(
        search_start,
        search_end + 1,
    ):

        text = (
            cell79_source_lines[
                probe - 1
            ]
        )


        norm = cell79_norm(
            text
        )


        if (
            (
                "دکتر"
                in
                norm
            )
            and
            (
                "رودی"
                in
                norm
                or
                "کمیل"
                in
                norm
            )
        ):

            explicit_dr_markers.append({
                "line_number":
                    probe,

                "text":
                    text,
            })


    candidate[
        "nearby_dr_markers"
    ] = explicit_dr_markers


    candidate[
        "has_nearby_dr_marker"
    ] = (
        len(
            explicit_dr_markers
        )
        >
        0
    )


# ============================================================
# 10) Build ranked retarget discovery list
# ============================================================

cell79_ranked_candidates = []


for index, candidate in enumerate(
    cell79_dialogue_candidates,
    start=1,
):

    response_norm = cell79_norm(
        candidate[
            "response_text"
        ]
    )


    cue_norm = cell79_norm(
        candidate[
            "cue_text"
        ]
    )


    score = 0


    if candidate[
        "has_nearby_dr_marker"
    ]:

        score += 10


    if len(
        response_norm
    ) >= 120:

        score += 3


    if len(
        response_norm
    ) >= 250:

        score += 2


    if (
        "؟"
        in
        cue_norm
        or
        "میگوید"
        in
        cue_norm
        or
        "اشاره میکند"
        in
        cue_norm
    ):

        score += 2


    if (
        "بله"
        in
        response_norm
        or
        "دقیقاً"
        in
        response_norm
        or
        "درست"
        in
        response_norm
    ):

        score += 1


    identity_payload = {
        "pilot_case_id":
            CELL79_PILOT_ID,

        "source":
            str(
                cell79_source_path
            ),

        "cue_line":
            candidate[
                "cue_line"
            ],

        "response_start":
            candidate[
                "response_start"
            ],

        "response_end":
            candidate[
                "response_end"
            ],

        "cue_text":
            candidate[
                "cue_text"
            ],

        "response_text":
            candidate[
                "response_text"
            ],
    }


    candidate_id = (
        "mizan-retarget-candidate:"
        +
        hashlib.sha256(
            cell10_canonical_json(
                identity_payload
            ).encode(
                "utf-8"
            )
        ).hexdigest()[
            :32
        ]
    )


    cell79_ranked_candidates.append({
        "candidate_id":
            candidate_id,

        "score":
            score,

        **copy.deepcopy(
            candidate
        ),
    })


cell79_ranked_candidates = sorted(
    cell79_ranked_candidates,
    key=lambda item: (
        -item[
            "score"
        ],
        item[
            "cue_line"
        ],
    ),
)


# ============================================================
# 11) Summary flags
# ============================================================

cell79_any_explicit_dr_anywhere = (
    len(
        cell79_dr_mentions
    )
    >
    0
)


cell79_any_candidate_with_dr_marker = any(
    candidate[
        "has_nearby_dr_marker"
    ]
    for candidate
    in cell79_ranked_candidates
)


cell79_current_target_retarget_required = all([
    globals().get(
        "cell78_has_prior_explicit_dr"
    )
    is False,

    globals().get(
        "cell78_continuity_candidate"
    )
    is False,

    len(
        cell76_unresolved_parts
    )
    >
    0,
])


# ============================================================
# 12) Safety
# ============================================================

cell79_registry_write = False

cell79_queue_advance = False

cell79_human_retarget_recorded = False

cell79_target_changed = False

cell79_citation_changed = False

cell79_raw_source_mutated = False

cell79_parser_mutated = False

cell79_commit_push = False


# ============================================================
# 13) Gate
# ============================================================

cell79_gate_checks = {
    "no_failures":
        len(
            cell79_failures
        )
        ==
        0,

    "source_valid":
        cell79_source_valid,

    "current_target_retarget_required":
        cell79_current_target_retarget_required,

    "participant_cues_found":
        len(
            cell79_participant_cues
        )
        >
        0,

    "dialogue_candidates_found":
        len(
            cell79_ranked_candidates
        )
        >
        0,
}


cell79_gate = all(
    cell79_gate_checks.values()
)


# ============================================================
# 14) Diagnostics
# ============================================================

print(
    "=== PILOT #17 MIZAN RETARGET CANDIDATE DISCOVERY ==="
)


print(
    "\n=== CURRENT TARGET ==="
)

print(
    "Pilot:",
    CELL79_PILOT_ID,
)

print(
    "Old Target:",
    CELL79_OLD_TARGET_EVIDENCE_ID,
)

print(
    "Task Family:",
    CELL79_TASK_FAMILY,
)

print(
    "Retarget required:",
    cell79_current_target_retarget_required,
)


print(
    "\n=== WHOLE-SOURCE SPEAKER STRUCTURE ==="
)

print(
    "Explicit/colon-style speaker labels:",
    len(
        cell79_colon_labels
    ),
)

print(
    "Dr. Roudi mentions anywhere:",
    len(
        cell79_dr_mentions
    ),
)

print(
    "Participant narrative cues:",
    len(
        cell79_participant_cues
    ),
)

print(
    "Dialogue-like candidates:",
    len(
        cell79_ranked_candidates
    ),
)

print(
    "Any candidate with nearby Dr. marker:",
    cell79_any_candidate_with_dr_marker,
)


print(
    "\n=== DR. ROUDI MENTIONS ==="
)


if cell79_dr_mentions:

    for item in cell79_dr_mentions:

        print(
            f"L{item['line_number']}:",
            item[
                "matched_terms"
            ],
            "->",
            item[
                "text"
            ],
        )


else:

    print(
        "No Dr. Roudi marker/mention found anywhere in the raw source."
    )


print(
    "\n=== EXPLICIT SPEAKER LABELS ==="
)


if cell79_colon_labels:

    for item in cell79_colon_labels:

        print(
            f"L{item['line_number']}:",
            repr(
                item[
                    "label"
                ]
            ),
            "->",
            item[
                "content"
            ][
                :220
            ],
        )


else:

    print(
        "No explicit colon-style speaker labels found."
    )


print(
    "\n=== TOP RETARGET CANDIDATES ==="
)


for rank, candidate in enumerate(
    cell79_ranked_candidates[
        :12
    ],
    start=1,
):

    print(
        "\n"
        +
        "="
        *
        72
    )

    print(
        "RANK:",
        rank,
    )

    print(
        "Candidate ID:",
        candidate[
            "candidate_id"
        ],
    )

    print(
        "Score:",
        candidate[
            "score"
        ],
    )

    print(
        "Audience Cue Line:",
        candidate[
            "cue_line"
        ],
    )

    print(
        "Audience Cue:"
    )

    print(
        candidate[
            "cue_text"
        ]
    )

    print(
        "Response Lines:",
        (
            candidate[
                "response_start"
            ],
            candidate[
                "response_end"
            ],
        ),
    )

    print(
        "Nearby Dr. marker:",
        candidate[
            "has_nearby_dr_marker"
        ],
    )


    if candidate[
        "nearby_dr_markers"
    ]:

        print(
            "Nearby Dr. markers:"
        )

        for marker in candidate[
            "nearby_dr_markers"
        ]:

            print(
                marker
            )


    print(
        "Response:"
    )

    print(
        candidate[
            "response_text"
        ]
    )


print(
    "\n=== ALL INTERNAL GATES ==="
)

for name, status in cell79_gate_checks.items():

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== SAFETY ==="
)

print(
    "Registry write:",
    cell79_registry_write,
)

print(
    "Queue advance:",
    cell79_queue_advance,
)

print(
    "Human RETARGET recorded:",
    cell79_human_retarget_recorded,
)

print(
    "Target changed:",
    cell79_target_changed,
)

print(
    "Citation changed:",
    cell79_citation_changed,
)

print(
    "Raw Source mutated:",
    cell79_raw_source_mutated,
)

print(
    "Parser mutated:",
    cell79_parser_mutated,
)

print(
    "Commit / Push:",
    cell79_commit_push,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell79_failures
)


print(
    "\nCODE CELL 79 GATE:",
    (
        "PASS"
        if cell79_gate
        else "FAIL"
    ),
)


assert cell79_gate, (
    "Pilot #17 Mizan retarget candidate discovery failed. "
    "Repair this SAME Code Cell 79 before selecting replacement evidence."
)

=== PILOT #17 MIZAN RETARGET CANDIDATE DISCOVERY ===

=== CURRENT TARGET ===
Pilot: question-pilot:dc845f69631a7000938b161802bc3030
Old Target: target-evidence:43487bce46faaf036499ac2921fd1280
Task Family: MULTI_SPEAKER_ATTRIBUTION_MIZAN
Retarget required: True

=== WHOLE-SOURCE SPEAKER STRUCTURE ===
Explicit/colon-style speaker labels: 8
Dr. Roudi mentions anywhere: 6
Participant narrative cues: 13
Dialogue-like candidates: 13
Any candidate with nearby Dr. marker: True

=== DR. ROUDI MENTIONS ===
L22: ['رودی'] -> یک معرفی کوتاه از خودم بکنم. اسمم کمیل و نام خانوادگی‌ام رودی است. دکترای مدیریت از دانشگاه تهران دارم با گرایش رفتار سازمانی. رفتار سازمانی یعنی کاربرد بخشی از دانش روان‌شناسی در سازمان‌ها و مدیریت. حوزه موردعلاقه من تصمیم‌گیری است و به تصمیم‌گیری فردی علاقه داشتم؛ در نتیجه وارد حوزه توسعه فردی شدم. تقریباً می‌توانم بگویم حدود بیست سال است که در این حوزه کار می‌کنم. سال 1396 از رساله دکتری‌ام دفاع کردم و از آن زمان تا امروز هم در مسیر یادگیری و علم‌آموزی کم نگذاشته‌ام.
L191:

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 80 — Pilot #17 RETARGET + Authority Policy + Question/Answer ACCEPT + Citation Draft

Human Reviewer تأیید کرد که در Golden Dataset:

- `Dr. Komeil Roudi` منبع اصلی و Primary Authority است؛
- حرف مجری، حاضر، شرکت‌کننده یا هر Speaker غیر از دکتر به‌تنهایی Evidence معتبر برای Expected Answer نیست؛
- سخن Speaker غیرPrimary فقط می‌تواند Question / Context / Contrast باشد؛
- فقط زمانی که دکتر رودی آن موضوع را تأیید، رد، اصلاح یا توضیح دهد، پاسخ خود دکتر می‌تواند Evidence اصلی شود.

Target قبلی Pilot #17 به دلیل Speaker نامشخص پاسخ، `RETARGET` می‌شود.

Replacement Candidate:

`mizan-retarget-candidate:f1b5d1c3c5f5197395156b27c9ce2a50`

در این Replacement:

- عبارت «یکی از حاضران می‌گوید: قناعت.» فقط Context است؛
- توضیح بعدی دکتر رودی درباره قناعت و خرج‌کردن، Primary Answer Evidence است.

سؤال پذیرفته‌شده:

**قناعت در مدیریت خرج یعنی چه و چرا بعضی هزینه‌ها ممکن است با نیاز واقعی فرد ارتباطی نداشته باشند؟**

جواب پذیرفته‌شده:

**قناعت در مدیریت خرج یعنی برای چیزی هزینه کنیم که واقعاً می‌خواهیم و با ارزش‌ها و نیازهای ما هماهنگ است، و برای چیزی که نمی‌خواهیم خرج نکنیم. بعضی هزینه‌ها ممکن است برای اثبات قدرت، شخصیت یا جایگاه انجام شوند؛ در این حالت پول به‌جای یک نیاز واقعی، صرف جبران یا پاسخ‌دادن به یک مسئله روانی می‌شود.**

Citation بعد از این Cell برای Human Review آماده خواهد بود و Queue همچنان 7 باقی می‌ماند.

</div>

In [80]:
# Code Cell 80

cell80_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell79_gate") is not True:
    cell80_failures.append(
        "CELL79_NOT_PASS"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell80_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


if not isinstance(
    globals().get(
        "cell75_next_pilot_packet"
    ),
    dict,
):
    cell80_failures.append(
        "PILOT_PACKET_MISSING"
    )


# ============================================================
# 2) Human decisions
# ============================================================

CELL80_SELECTED_CANDIDATE_ID = (
    "mizan-retarget-candidate:"
    "f1b5d1c3c5f5197395156b27c9ce2a50"
)


CELL80_ACCEPTED_QUESTION = (
    "قناعت در مدیریت خرج یعنی چه و چرا بعضی هزینه‌ها ممکن است "
    "با نیاز واقعی فرد ارتباطی نداشته باشند؟"
)


CELL80_ACCEPTED_ANSWER = (
    "قناعت در مدیریت خرج یعنی برای چیزی هزینه کنیم که واقعاً "
    "می‌خواهیم و با ارزش‌ها و نیازهای ما هماهنگ است، و برای چیزی "
    "که نمی‌خواهیم خرج نکنیم. بعضی هزینه‌ها ممکن است برای اثبات "
    "قدرت، شخصیت یا جایگاه انجام شوند؛ در این حالت پول به‌جای یک "
    "نیاز واقعی، صرف جبران یا پاسخ‌دادن به یک مسئله روانی می‌شود."
)


CELL80_RETARGET_REASON = (
    "The original target contains a participant narrative cue followed "
    "by an unresolved response speaker. No valid upstream Dr. Roudi "
    "speaker marker or continuity lineage was found, while unsupported "
    "speaker inference is forbidden."
)


CELL80_QUESTION_HUMAN_NOTE = (
    "Human Reviewer accepted the retargeted question."
)


CELL80_ANSWER_HUMAN_NOTE = (
    "Human Reviewer accepted the retargeted expected answer."
)


# ============================================================
# 3) Pilot identities
# ============================================================

CELL80_PILOT_ID = (
    CELL79_PILOT_ID
)


CELL80_GOLDEN_CASE_ID = (
    CELL75_NEXT_GOLDEN_CASE_ID
)


CELL80_TASK_DRAFT_ID = (
    CELL75_NEXT_TASK_DRAFT_ID
)


CELL80_OLD_TARGET_EVIDENCE_ID = (
    CELL79_OLD_TARGET_EVIDENCE_ID
)


# ============================================================
# 4) Resolve selected Candidate
# ============================================================

cell80_selected_matches = [
    candidate
    for candidate
    in cell79_ranked_candidates
    if (
        candidate.get(
            "candidate_id"
        )
        ==
        CELL80_SELECTED_CANDIDATE_ID
    )
]


cell80_candidate_unique = (
    len(
        cell80_selected_matches
    )
    ==
    1
)


cell80_selected_candidate = (
    copy.deepcopy(
        cell80_selected_matches[
            0
        ]
    )
    if cell80_candidate_unique
    else {}
)


cell80_candidate_valid = all([
    cell80_candidate_unique,

    cell80_selected_candidate.get(
        "cue_line"
    )
    ==
    225,

    cell80_selected_candidate.get(
        "response_start"
    )
    ==
    227,

    cell80_selected_candidate.get(
        "response_end"
    )
    ==
    244,

    "قناعت"
    in
    cell80_selected_candidate.get(
        "cue_text",
        "",
    ),

    len(
        cell80_selected_candidate.get(
            "response_text",
            "",
        )
    )
    >
    100,
])


if not cell80_candidate_valid:
    cell80_failures.append(
        "SELECTED_RETARGET_CANDIDATE_INVALID"
    )


# ============================================================
# 5) Global Human Authority Policy
# ============================================================

cell80_authority_policy = {
    "policy_version":
        "human_authority_policy_v1",

    "primary_authority":
        "Dr. Komeil Roudi",

    "primary_rule":
        (
            "Expected Answers must be grounded in Dr. Komeil Roudi's "
            "validated source speech."
        ),

    "non_primary_speakers":
        [
            "host",
            "presenter",
            "participant",
            "audience_member",
            "guest",
            "other_non_primary_speaker",
        ],

    "non_primary_rule":
        (
            "Speech from hosts, presenters, participants, audience "
            "members, or other non-primary speakers is not independent "
            "answer authority."
        ),

    "allowed_non_primary_roles":
        [
            "QUESTION_CONTEXT",
            "CONTRAST_CONTEXT",
            "DISCUSSION_TRIGGER",
            "STRUCTURAL_CONTEXT",
        ],

    "promotion_rule":
        (
            "A non-primary statement may motivate a question or provide "
            "context, but the Golden answer may only use the corresponding "
            "claim when Dr. Komeil Roudi confirms, rejects, corrects, "
            "qualifies, or explains it in validated source evidence."
        ),

    "secondary_to_primary_promotion":
        "FORBIDDEN",

    "external_validation_substitution":
        "FORBIDDEN",

    "human_certified":
        True,
}


# ============================================================
# 6) Source-level Human Attribution Certification
# ============================================================

cell80_self_intro_line = (
    cell79_source_lines[
        21
    ]
    if
    len(
        cell79_source_lines
    )
    >=
    22
    else
    ""
)


cell80_self_intro_norm = cell79_norm(
    cell80_self_intro_line
)


cell80_self_intro_valid = all([
    "اسمم کمیل"
    in
    cell80_self_intro_norm,

    "نام خانوادگیام رودی است"
    in
    cell80_self_intro_norm,
])


if not cell80_self_intro_valid:
    cell80_failures.append(
        "SOURCE_SELF_IDENTIFICATION_NOT_VERIFIED"
    )


cell80_source_level_attribution_certification = {
    "certification_type":
        "HUMAN_CERTIFIED_SOURCE_LEVEL_PRIMARY_NARRATOR",

    "source":
        str(
            cell79_source_path
        ),

    "self_identification_line":
        22,

    "self_identification_text":
        cell80_self_intro_line,

    "certified_source_identity":
        "Dr. Komeil Roudi",

    "certified_primary":
        True,

    "speaker_rule":
        (
            "For this source, unlabeled main-speaker passages may be "
            "treated as Dr. Komeil Roudi when Human Review confirms the "
            "source-level narrator identity and no explicit intervening "
            "non-primary speaker cue owns that passage."
        ),

    "explicit_non_primary_speaker_overrides":
        True,

    "automatic_unknown_speaker_inference":
        False,

    "human_overlay_only":
        True,

    "canonical_raw_source_mutated":
        False,

    "parser_output_mutated":
        False,
}


# ============================================================
# 7) Replacement Evidence texts
# ============================================================

cell80_context_text = (
    cell80_selected_candidate.get(
        "cue_text",
        "",
    ).strip()
)


cell80_primary_text = (
    cell80_selected_candidate.get(
        "response_text",
        "",
    ).strip()
)


cell80_source_relative = (
    str(
        cell79_source_path.relative_to(
            cell77_repo_root
        )
    )
    if (
        cell79_source_path is not None
        and
        cell77_repo_root is not None
    )
    else
    str(
        cell79_source_path
    )
)


# ============================================================
# 8) Replacement Target Parts
# ============================================================

cell80_replacement_target_parts = [
    {
        "source_type":
            "transcript",

        "source_path":
            cell80_source_relative,

        "role":
            "ATTRIBUTION_CONTRAST_LEFT",

        "text":
            cell80_context_text,

        "text_sha256":
            hashlib.sha256(
                cell80_context_text.encode(
                    "utf-8"
                )
            ).hexdigest(),

        "char_start":
            0,

        "char_end":
            len(
                cell80_context_text
            ),

        "locator": {
            "source_path":
                cell80_source_relative,

            "source_line_start":
                225,

            "source_line_end":
                225,

            "locator_origin":
                "RAW_SOURCE_VERIFIED_CELL79",
        },

        "speaker_state": {
            "base_status":
                "NARRATIVE_CUE",

            "base_raw":
                "یکی از حاضران",

            "effective_label":
                "یکی از حاضران",

            "effective_canonical":
                None,

            "effective_primary":
                None,
        },

        "human_attribution_overlay": {
            "status":
                "NON_PRIMARY_CONTEXT",

            "effective_label":
                "یکی از حاضران",

            "effective_canonical":
                None,

            "effective_primary":
                False,

            "basis":
                "EXPLICIT_NARRATIVE_CUE",

            "raw_source_preserved":
                True,
        },

        "utterance_role":
            "DISCUSSION_TRIGGER",
    },

    {
        "source_type":
            "transcript",

        "source_path":
            cell80_source_relative,

        "role":
            "ATTRIBUTION_CONTRAST_RIGHT",

        "text":
            cell80_primary_text,

        "text_sha256":
            hashlib.sha256(
                cell80_primary_text.encode(
                    "utf-8"
                )
            ).hexdigest(),

        "char_start":
            0,

        "char_end":
            len(
                cell80_primary_text
            ),

        "locator": {
            "source_path":
                cell80_source_relative,

            "source_line_start":
                227,

            "source_line_end":
                244,

            "locator_origin":
                "RAW_SOURCE_VERIFIED_CELL79",
        },

        "speaker_state": {
            "base_status":
                "UNRESOLVED_IN_PARSER",

            "base_raw":
                None,

            "effective_label":
                None,

            "effective_canonical":
                None,

            "effective_primary":
                None,
        },

        "human_attribution_overlay": {
            "status":
                "HUMAN_CERTIFIED",

            "effective_label":
                "دکتر کمیل رودی",

            "effective_canonical":
                "Dr. Komeil Roudi",

            "effective_primary":
                True,

            "basis":
                "HUMAN_CERTIFIED_SOURCE_LEVEL_PRIMARY_NARRATOR",

            "source_self_identification_line":
                22,

            "raw_source_preserved":
                True,

            "parser_output_preserved":
                True,
        },

        "utterance_role":
            "PRIMARY_ANSWER_EVIDENCE",
    },
]


cell80_replacement_parts_valid = all([
    len(
        cell80_replacement_target_parts
    )
    ==
    2,

    cell80_replacement_target_parts[
        0
    ][
        "human_attribution_overlay"
    ][
        "effective_primary"
    ]
    is False,

    cell80_replacement_target_parts[
        1
    ][
        "human_attribution_overlay"
    ][
        "effective_canonical"
    ]
    ==
    "Dr. Komeil Roudi",

    cell80_replacement_target_parts[
        1
    ][
        "human_attribution_overlay"
    ][
        "effective_primary"
    ]
    is True,
])


if not cell80_replacement_parts_valid:
    cell80_failures.append(
        "REPLACEMENT_TARGET_PARTS_INVALID"
    )


# ============================================================
# 9) Deterministic Replacement Target ID
# ============================================================

cell80_target_identity = {
    "identity_version":
        "mizan_retarget_target_v1",

    "pilot_case_id":
        CELL80_PILOT_ID,

    "old_target_evidence_id":
        CELL80_OLD_TARGET_EVIDENCE_ID,

    "selected_candidate_id":
        CELL80_SELECTED_CANDIDATE_ID,

    "source":
        cell80_source_relative,

    "parts": [
        {
            "role":
                part[
                    "role"
                ],

            "text_sha256":
                part[
                    "text_sha256"
                ],

            "locator":
                copy.deepcopy(
                    part[
                        "locator"
                    ]
                ),

            "authority":
                copy.deepcopy(
                    part[
                        "human_attribution_overlay"
                    ]
                ),
        }
        for part
        in cell80_replacement_target_parts
    ],
}


CELL80_REPLACEMENT_TARGET_EVIDENCE_ID = (
    "target-evidence:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell80_target_identity
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


cell80_replacement_target_record = {
    "record_version":
        "replacement_target_evidence_v1",

    "target_evidence_id":
        CELL80_REPLACEMENT_TARGET_EVIDENCE_ID,

    "pilot_case_id":
        CELL80_PILOT_ID,

    "golden_case_id":
        CELL80_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL80_TASK_DRAFT_ID,

    "source_type":
        "transcript",

    "stratum":
        "multi_speaker_mizan",

    "task_family":
        "MULTI_SPEAKER_ATTRIBUTION_MIZAN",

    "question_archetype":
        "ATTRIBUTION_BOUNDARY",

    "authority_expectation":
        "UTTERANCE_LEVEL_AUTHORITY",

    "selected_candidate_id":
        CELL80_SELECTED_CANDIDATE_ID,

    "target_parts":
        copy.deepcopy(
            cell80_replacement_target_parts
        ),

    "combined_target_text":
        (
            cell80_context_text
            +
            "\n\n"
            +
            cell80_primary_text
        ),

    "source_level_attribution_certification":
        copy.deepcopy(
            cell80_source_level_attribution_certification
        ),

    "retarget_lineage": {
        "old_target_evidence_id":
            CELL80_OLD_TARGET_EVIDENCE_ID,

        "old_target_human_decision":
            "RETARGET",

        "retarget_reason":
            CELL80_RETARGET_REASON,

        "replacement_target_evidence_id":
            CELL80_REPLACEMENT_TARGET_EVIDENCE_ID,

        "replacement_target_human_decision":
            "ACCEPT",
    },

    "raw_source_mutated":
        False,

    "parser_output_mutated":
        False,

    "external_knowledge_used":
        False,
}


# ============================================================
# 10) Rebuild Pilot Packet
# ============================================================

def cell80_replace_target(
    value,
):

    if isinstance(
        value,
        dict,
    ):

        output = {}


        for key, item in value.items():

            if key == "target_text_parts":

                output[
                    key
                ] = copy.deepcopy(
                    cell80_replacement_target_parts
                )


            elif key == "target_evidence_id":

                output[
                    key
                ] = (
                    CELL80_REPLACEMENT_TARGET_EVIDENCE_ID
                )


            else:

                output[
                    key
                ] = cell80_replace_target(
                    item
                )


        return output


    if isinstance(
        value,
        list,
    ):

        return [
            cell80_replace_target(
                item
            )
            for item
            in value
        ]


    if (
        isinstance(
            value,
            str,
        )
        and
        value
        ==
        CELL80_OLD_TARGET_EVIDENCE_ID
    ):

        return (
            CELL80_REPLACEMENT_TARGET_EVIDENCE_ID
        )


    return copy.deepcopy(
        value
    )


cell80_retargeted_pilot_packet = (
    cell80_replace_target(
        cell75_next_pilot_packet
    )
)


cell80_packet_target_ids = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell80_retargeted_pilot_packet,
            "target-evidence:",
        )
    )
)


cell80_packet_valid = all([
    cell80_packet_target_ids
    ==
    [
        CELL80_REPLACEMENT_TARGET_EVIDENCE_ID
    ],

    CELL80_OLD_TARGET_EVIDENCE_ID
    not in
    cell80_packet_target_ids,
])


if not cell80_packet_valid:
    cell80_failures.append(
        "RETARGETED_PACKET_INVALID"
    )


# ============================================================
# 11) Grounding Validation
# ============================================================

def cell80_norm(
    text,
):

    if not isinstance(
        text,
        str,
    ):
        return ""


    value = (
        text
        .replace(
            "ي",
            "ی",
        )
        .replace(
            "ك",
            "ک",
        )
        .replace(
            "\u200c",
            "",
        )
        .replace(
            "\ufeff",
            "",
        )
    )


    value = re.sub(
        r"\s+",
        " ",
        value,
    )


    return value.strip()


cell80_primary_surface = cell80_norm(
    cell80_primary_text
)


cell80_grounding = {
    "definition":
        (
            "قناعت از این زاویه یعنی مدیریت خرج"
            in
            cell80_primary_surface
        ),

    "real_want":
        (
            "برای چیزی که واقعاً میخواهی"
            in
            cell80_primary_surface
        ),

    "values_and_needs":
        (
            "چهارچوب ارزشها و نیازهایت"
            in
            cell80_primary_surface
        ),

    "do_not_want":
        (
            "برای چیزی که نمیخواهی خرج نکن"
            in
            cell80_primary_surface
        ),

    "unneeded_spending":
        (
            "چیزی که نیاز ندارد هزینه میکند"
            in
            cell80_primary_surface
        ),

    "power_or_identity":
        (
            "اثبات توانایی و اقتدار"
            in
            cell80_primary_surface
            or
            "اثبات قدرت"
            in
            cell80_primary_surface
        ),

    "psychological_issue":
        (
            "مسئله روانی"
            in
            cell80_primary_surface
        ),
}


cell80_grounding_pass = all(
    cell80_grounding.values()
)


if not cell80_grounding_pass:
    cell80_failures.append(
        "ANSWER_GROUNDING_FAILED"
    )


# ============================================================
# 12) Restart Engine
# ============================================================

cell80_question_pending_session = (
    cell13_build_session(
        cell80_retargeted_pilot_packet
    )
)


cell80_session_start_valid = (
    cell80_question_pending_session.get(
        "current_state"
    )
    ==
    "QUESTION_DRAFT_PENDING"
)


if not cell80_session_start_valid:
    cell80_failures.append(
        "ENGINE_SESSION_START_INVALID"
    )


# ============================================================
# 13) Question Draft + ACCEPT
# ============================================================

cell80_question_identity = {
    "identity_version":
        "question_draft_identity_v1",

    "pilot_case_id":
        CELL80_PILOT_ID,

    "target_evidence_id":
        CELL80_REPLACEMENT_TARGET_EVIDENCE_ID,

    "question_text":
        CELL80_ACCEPTED_QUESTION,

    "revision_number":
        1,
}


cell80_question_draft_id = (
    "question-draft:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell80_question_identity
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


cell80_question_record = {
    "record_version":
        "question_draft_record_v1",

    "question_draft_id":
        cell80_question_draft_id,

    "pilot_case_id":
        CELL80_PILOT_ID,

    "golden_case_id":
        CELL80_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL80_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL80_REPLACEMENT_TARGET_EVIDENCE_ID,

    "question_text":
        CELL80_ACCEPTED_QUESTION,

    "revision_number":
        1,

    "machine_validation_status":
        "PASS_PENDING_HUMAN_ADJUDICATION",

    "human_adjudication_status":
        "PENDING",

    "external_knowledge_used":
        False,
}


cell80_question_review_session = (
    cell13_apply_event(
        cell80_question_pending_session,
        "QUESTION_DRAFT_REGISTERED",
        artifact=
            cell80_question_record,
    )
)


cell80_answer_pending_session = (
    cell13_apply_event(
        cell80_question_review_session,
        "QUESTION_ACCEPTED",
        human_note=
            CELL80_QUESTION_HUMAN_NOTE,
    )
)


cell80_question_accept_valid = (
    cell80_answer_pending_session.get(
        "current_state"
    )
    ==
    "EXPECTED_ANSWER_DRAFT_PENDING"
)


if not cell80_question_accept_valid:
    cell80_failures.append(
        "QUESTION_ACCEPT_TRANSITION_INVALID"
    )


# ============================================================
# 14) Expected Answer + ACCEPT
# ============================================================

cell80_answer_identity = {
    "identity_version":
        "expected_answer_identity_v1",

    "pilot_case_id":
        CELL80_PILOT_ID,

    "question_draft_id":
        cell80_question_draft_id,

    "target_evidence_id":
        CELL80_REPLACEMENT_TARGET_EVIDENCE_ID,

    "answer_text":
        CELL80_ACCEPTED_ANSWER,

    "revision_number":
        1,
}


cell80_expected_answer_draft_id = (
    "expected-answer-draft:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell80_answer_identity
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


cell80_expected_answer_record = {
    "record_version":
        "expected_answer_draft_record_v1",

    "expected_answer_draft_id":
        cell80_expected_answer_draft_id,

    "pilot_case_id":
        CELL80_PILOT_ID,

    "golden_case_id":
        CELL80_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL80_TASK_DRAFT_ID,

    "question_draft_id":
        cell80_question_draft_id,

    "target_evidence_id":
        CELL80_REPLACEMENT_TARGET_EVIDENCE_ID,

    "expected_answer_text":
        CELL80_ACCEPTED_ANSWER,

    "revision_number":
        1,

    "machine_validation_status":
        "PASS_PENDING_HUMAN_ADJUDICATION",

    "human_adjudication_status":
        "PENDING",

    "external_knowledge_used":
        False,
}


cell80_answer_review_session = (
    cell13_apply_event(
        cell80_answer_pending_session,
        "EXPECTED_ANSWER_REGISTERED",
        artifact=
            cell80_expected_answer_record,
    )
)


cell80_citation_pending_session = (
    cell13_apply_event(
        cell80_answer_review_session,
        "EXPECTED_ANSWER_ACCEPTED",
        human_note=
            CELL80_ANSWER_HUMAN_NOTE,
    )
)


cell80_answer_accept_valid = (
    cell80_citation_pending_session.get(
        "current_state"
    )
    ==
    "CITATION_MAPPING_PENDING"
)


if not cell80_answer_accept_valid:
    cell80_failures.append(
        "ANSWER_ACCEPT_TRANSITION_INVALID"
    )


# ============================================================
# 15) Citation Entries
# ============================================================

cell80_citation_entries = []


for index, part in enumerate(
    cell80_replacement_target_parts
):

    is_primary = (
        index
        ==
        1
    )


    entry = copy.deepcopy(
        part
    )


    entry[
        "citation_index"
    ] = index


    entry[
        "target_evidence_id"
    ] = (
        CELL80_REPLACEMENT_TARGET_EVIDENCE_ID
    )


    entry[
        "answer_support"
    ] = is_primary


    entry[
        "answer_support_role"
    ] = (
        "PRIMARY_ANSWER_EVIDENCE"
        if is_primary
        else
        "DISCUSSION_TRIGGER_CONTEXT_ONLY"
    )


    entry[
        "authority_role"
    ] = (
        "PRIMARY_CORPUS_AUTHORITY"
        if is_primary
        else
        "NON_PRIMARY_SPEAKER_CONTEXT"
    )


    entry[
        "use_in_expected_answer_grounding"
    ] = is_primary


    entry[
        "external_locator_lookup_used"
    ] = False


    cell80_citation_entries.append(
        entry
    )


cell80_primary_entries = [
    entry
    for entry
    in cell80_citation_entries
    if entry.get(
        "answer_support"
    )
    is True
]


cell80_context_entries = [
    entry
    for entry
    in cell80_citation_entries
    if entry.get(
        "answer_support"
    )
    is False
]


cell80_citation_boundary_valid = all([
    len(
        cell80_primary_entries
    )
    ==
    1,

    len(
        cell80_context_entries
    )
    ==
    1,

    cell80_primary_entries[
        0
    ].get(
        "human_attribution_overlay",
        {},
    ).get(
        "effective_canonical"
    )
    ==
    "Dr. Komeil Roudi",

    cell80_primary_entries[
        0
    ].get(
        "human_attribution_overlay",
        {},
    ).get(
        "effective_primary"
    )
    is True,

    cell80_context_entries[
        0
    ].get(
        "human_attribution_overlay",
        {},
    ).get(
        "effective_primary"
    )
    is False,

    cell80_context_entries[
        0
    ].get(
        "use_in_expected_answer_grounding"
    )
    is False,
])


if not cell80_citation_boundary_valid:
    cell80_failures.append(
        "CITATION_AUTHORITY_BOUNDARY_INVALID"
    )


# ============================================================
# 16) No Locator Fabrication
# ============================================================

def cell80_strip_citation_fields(
    entry,
):

    output = copy.deepcopy(
        entry
    )


    for key in [
        "citation_index",
        "target_evidence_id",
        "answer_support",
        "answer_support_role",
        "authority_role",
        "use_in_expected_answer_grounding",
        "external_locator_lookup_used",
    ]:

        output.pop(
            key,
            None,
        )


    return output


cell80_citation_back_projection = [
    cell80_strip_citation_fields(
        entry
    )
    for entry
    in cell80_citation_entries
]


cell80_no_locator_fabrication = (
    cell80_citation_back_projection
    ==
    cell80_replacement_target_parts
)


if not cell80_no_locator_fabrication:
    cell80_failures.append(
        "CITATION_LOCATOR_FABRICATION_DETECTED"
    )


# ============================================================
# 17) Citation Mapping
# ============================================================

cell80_citation_identity = {
    "identity_version":
        "citation_mapping_identity_v1_mizan_retarget",

    "pilot_case_id":
        CELL80_PILOT_ID,

    "question_draft_id":
        cell80_question_draft_id,

    "expected_answer_draft_id":
        cell80_expected_answer_draft_id,

    "target_evidence_id":
        CELL80_REPLACEMENT_TARGET_EVIDENCE_ID,

    "citation_entries":
        copy.deepcopy(
            cell80_citation_entries
        ),
}


cell80_citation_mapping_id = (
    "citation-mapping:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell80_citation_identity
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


cell80_citation_mapping_record = {
    "record_version":
        "citation_mapping_record_v1_mizan_retarget",

    "citation_mapping_id":
        cell80_citation_mapping_id,

    "pilot_case_id":
        CELL80_PILOT_ID,

    "golden_case_id":
        CELL80_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL80_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL80_REPLACEMENT_TARGET_EVIDENCE_ID,

    "question_draft_id":
        cell80_question_draft_id,

    "expected_answer_draft_id":
        cell80_expected_answer_draft_id,

    "citation_entries":
        copy.deepcopy(
            cell80_citation_entries
        ),

    "primary_answer_authority":
        "Dr. Komeil Roudi",

    "non_primary_context":
        "یکی از حاضران",

    "non_primary_used_as_answer_evidence":
        False,

    "authority_policy":
        copy.deepcopy(
            cell80_authority_policy
        ),

    "source_level_attribution_certification":
        copy.deepcopy(
            cell80_source_level_attribution_certification
        ),

    "retarget_used":
        True,

    "old_target_evidence_id":
        CELL80_OLD_TARGET_EVIDENCE_ID,

    "external_locator_lookup_used":
        False,

    "raw_source_mutated":
        False,

    "parser_output_mutated":
        False,

    "machine_validation_status":
        "PASS_PENDING_HUMAN_ADJUDICATION",

    "human_adjudication_status":
        "PENDING",
}


cell80_current_session = (
    cell13_apply_event(
        cell80_citation_pending_session,
        "CITATION_MAPPING_REGISTERED",
        artifact=
            cell80_citation_mapping_record,
    )
)


cell80_citation_registered_valid = (
    cell80_current_session.get(
        "current_state"
    )
    ==
    "CITATION_HUMAN_REVIEW_PENDING"
)


if not cell80_citation_registered_valid:
    cell80_failures.append(
        "CITATION_REGISTRATION_INVALID"
    )


# ============================================================
# 18) Active Workspace
# ============================================================

cell80_active_workspace = copy.deepcopy(
    cell75_persisted_active_case
)


cell80_active_workspace[
    "status"
] = "ACTIVE_HUMAN_REVIEW"


cell80_active_workspace[
    "workflow_state"
] = "CITATION_HUMAN_REVIEW_PENDING"


cell80_active_workspace[
    "target_evidence_id"
] = (
    CELL80_REPLACEMENT_TARGET_EVIDENCE_ID
)


cell80_active_workspace[
    "target_evidence"
] = copy.deepcopy(
    cell80_replacement_target_record
)


cell80_active_workspace[
    "source_evidence"
] = copy.deepcopy(
    cell80_replacement_target_parts
)


cell80_active_workspace[
    "combined_target_text"
] = (
    cell80_replacement_target_record[
        "combined_target_text"
    ]
)


cell80_active_workspace[
    "authority_policy"
] = copy.deepcopy(
    cell80_authority_policy
)


cell80_active_workspace[
    "source_level_attribution_certification"
] = copy.deepcopy(
    cell80_source_level_attribution_certification
)


cell80_active_workspace[
    "retarget_review"
] = {
    "old_target_evidence_id":
        CELL80_OLD_TARGET_EVIDENCE_ID,

    "old_target_human_decision":
        "RETARGET",

    "retarget_reason":
        CELL80_RETARGET_REASON,

    "selected_candidate_id":
        CELL80_SELECTED_CANDIDATE_ID,

    "replacement_target_evidence_id":
        CELL80_REPLACEMENT_TARGET_EVIDENCE_ID,

    "replacement_target_human_decision":
        "ACCEPT",
}


cell80_active_workspace[
    "question_review"
] = {
    "question_draft_id":
        cell80_question_draft_id,

    "question_text":
        CELL80_ACCEPTED_QUESTION,

    "human_decision":
        "ACCEPT",

    "human_note":
        CELL80_QUESTION_HUMAN_NOTE,
}


cell80_active_workspace[
    "expected_answer_review"
] = {
    "expected_answer_draft_id":
        cell80_expected_answer_draft_id,

    "expected_answer_text":
        CELL80_ACCEPTED_ANSWER,

    "human_decision":
        "ACCEPT",

    "human_note":
        CELL80_ANSWER_HUMAN_NOTE,
}


cell80_active_workspace[
    "citation_review"
] = {
    "citation_mapping_id":
        cell80_citation_mapping_id,

    "citation_mapping_record":
        copy.deepcopy(
            cell80_citation_mapping_record
        ),

    "citation_entries":
        copy.deepcopy(
            cell80_citation_entries
        ),

    "human_decision":
        "PENDING",

    "machine_validation_status":
        "PASS_PENDING_HUMAN_ADJUDICATION",

    "primary_answer_authority":
        "Dr. Komeil Roudi",

    "non_primary_used_as_answer_evidence":
        False,
}


# ============================================================
# 19) Continuation State
# ============================================================

cell80_continuation_core = {
    "next_pilot_id":
        CELL80_PILOT_ID,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell75_remaining_pilot_ids
        ),

    "remaining_count":
        7,

    "active_pilot_status":
        "CITATION_HUMAN_REVIEW_PENDING",

    "active_target_evidence_id":
        CELL80_REPLACEMENT_TARGET_EVIDENCE_ID,

    "active_citation_mapping_id":
        cell80_citation_mapping_id,

    "retarget_status":
        "REPLACEMENT_ACCEPTED",

    "authority_policy_active":
        True,

    "last_completed_pilot_id":
        cell75_continuation_state.get(
            "last_completed_pilot_id"
        ),

    "last_completion_type":
        cell75_continuation_state.get(
            "last_completion_type"
        ),
}


cell80_continuation_state_sha256 = (
    cell10_stable_sha(
        cell80_continuation_core
    )
)


cell80_continuation_state = {
    **copy.deepcopy(
        cell80_continuation_core
    ),

    "continuation_state_sha256":
        cell80_continuation_state_sha256,
}


# ============================================================
# 20) Persist Global Authority Policy + Active Case
# ============================================================

cell80_persistent_registry_payload = copy.deepcopy(
    cell75_persisted_registry
)


cell80_workflow_policy = copy.deepcopy(
    cell80_persistent_registry_payload.get(
        "workflow_policy",
        {},
    )
)


if not isinstance(
    cell80_workflow_policy,
    dict,
):

    cell80_workflow_policy = {}


cell80_workflow_policy[
    "authority_evidence_policy"
] = copy.deepcopy(
    cell80_authority_policy
)


cell80_persistent_registry_payload[
    "workflow_policy"
] = cell80_workflow_policy


cell80_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell80_active_workspace
)


cell80_persistent_registry_payload[
    "remaining_pilot_count"
] = 7


cell80_persistent_registry_payload[
    "continuation_state"
] = copy.deepcopy(
    cell80_continuation_state
)


cell80_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell80_persistent_registry_payload
    )
)


# ============================================================
# 21) Read-back Validation
# ============================================================

cell80_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell80_persisted_active_case = (
    cell80_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell80_persisted_authority_policy = (
    cell80_persisted_registry.get(
        "workflow_policy",
        {},
    ).get(
        "authority_evidence_policy",
        {},
    )
)


cell80_persistence_valid = all([
    cell80_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL80_PILOT_ID,

    cell80_persisted_active_case.get(
        "target_evidence_id"
    )
    ==
    CELL80_REPLACEMENT_TARGET_EVIDENCE_ID,

    cell80_persisted_active_case.get(
        "workflow_state"
    )
    ==
    "CITATION_HUMAN_REVIEW_PENDING",

    cell80_persisted_active_case.get(
        "retarget_review",
        {},
    ).get(
        "old_target_human_decision"
    )
    ==
    "RETARGET",

    cell80_persisted_active_case.get(
        "question_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell80_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell80_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "PENDING",

    cell80_persisted_authority_policy.get(
        "primary_authority"
    )
    ==
    "Dr. Komeil Roudi",

    cell80_persisted_authority_policy.get(
        "secondary_to_primary_promotion"
    )
    ==
    "FORBIDDEN",

    cell80_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    7,

    cell80_persisted_registry.get(
        "continuation_state",
        {},
    ).get(
        "continuation_state_sha256"
    )
    ==
    cell80_continuation_state_sha256,
])


if not cell80_persistence_valid:
    cell80_failures.append(
        "REGISTRY_READBACK_INVALID"
    )


# ============================================================
# 22) Safety
# ============================================================

cell80_queue_still_7 = all([
    len(
        cell75_remaining_pilot_ids
    )
    ==
    7,

    cell75_remaining_pilot_ids[
        0
    ]
    ==
    CELL80_PILOT_ID,
])


cell80_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell80_full_572_blocked = (
    cell75_full_572_blocked
    is True
)


cell80_no_commit_push = True


# ============================================================
# 23) Internal Gates
# ============================================================

cell80_gate_checks = {
    "no_failures":
        len(
            cell80_failures
        )
        ==
        0,

    "candidate_valid":
        cell80_candidate_valid,

    "self_identification_valid":
        cell80_self_intro_valid,

    "replacement_parts_valid":
        cell80_replacement_parts_valid,

    "packet_valid":
        cell80_packet_valid,

    "grounding_pass":
        cell80_grounding_pass,

    "session_start_valid":
        cell80_session_start_valid,

    "question_accept_valid":
        cell80_question_accept_valid,

    "answer_accept_valid":
        cell80_answer_accept_valid,

    "citation_boundary_valid":
        cell80_citation_boundary_valid,

    "no_locator_fabrication":
        cell80_no_locator_fabrication,

    "citation_registered_valid":
        cell80_citation_registered_valid,

    "persistence_valid":
        cell80_persistence_valid,

    "queue_still_7":
        cell80_queue_still_7,

    "base_seed_unchanged":
        cell80_base_seed_unchanged,

    "full_572_blocked":
        cell80_full_572_blocked,

    "no_commit_push":
        cell80_no_commit_push,
}


cell80_gate = all(
    cell80_gate_checks.values()
)


# ============================================================
# 24) Diagnostics
# ============================================================

print(
    "=== PILOT #17 RETARGET + AUTHORITY POLICY + QA ACCEPT + CITATION DRAFT ==="
)


print(
    "\n=== GLOBAL AUTHORITY POLICY ==="
)

print(
    "Primary Authority:",
    "Dr. Komeil Roudi",
)

print(
    "Host / Participant independent answer authority:",
    False,
)

print(
    "Non-primary speakers may provide Context / Question:",
    True,
)

print(
    "Secondary-to-primary promotion:",
    "FORBIDDEN",
)

print(
    "Policy persisted:",
    (
        cell80_persisted_authority_policy.get(
            "primary_authority"
        )
        ==
        "Dr. Komeil Roudi"
    ),
)


print(
    "\n=== RETARGET ==="
)

print(
    "Old Target:",
    CELL80_OLD_TARGET_EVIDENCE_ID,
)

print(
    "Old Target Decision:",
    "RETARGET",
)

print(
    "Selected Candidate:",
    CELL80_SELECTED_CANDIDATE_ID,
)

print(
    "Replacement Target:",
    CELL80_REPLACEMENT_TARGET_EVIDENCE_ID,
)

print(
    "Replacement Decision:",
    "ACCEPT",
)


print(
    "\n=== ATTRIBUTION ==="
)

print(
    "Context Speaker:",
    "یکی از حاضران",
)

print(
    "Context used as Answer Evidence:",
    False,
)

print(
    "Primary Speaker:",
    "Dr. Komeil Roudi",
)

print(
    "Source-level Human Certification:",
    True,
)

print(
    "Self-identification line:",
    22,
)

print(
    "Raw Source mutated:",
    False,
)

print(
    "Parser mutated:",
    False,
)


print(
    "\n=== ACCEPTED QUESTION ==="
)

print(
    CELL80_ACCEPTED_QUESTION
)

print(
    "Question Draft ID:",
    cell80_question_draft_id,
)

print(
    "Human Decision:",
    "ACCEPT",
)


print(
    "\n=== ACCEPTED ANSWER ==="
)

print(
    CELL80_ACCEPTED_ANSWER
)

print(
    "Expected Answer Draft ID:",
    cell80_expected_answer_draft_id,
)

print(
    "Human Decision:",
    "ACCEPT",
)


print(
    "\n=== GROUNDING ==="
)

for name, status in cell80_grounding.items():

    print(
        f"{name}:",
        status,
    )


print(
    "Grounding PASS:",
    cell80_grounding_pass,
)


print(
    "\n=== CITATION DRAFT ==="
)

print(
    "Citation Mapping ID:",
    cell80_citation_mapping_id,
)

print(
    "Citation entries:",
    len(
        cell80_citation_entries
    ),
)

print(
    "Primary Answer Evidence entries:",
    len(
        cell80_primary_entries
    ),
)

print(
    "Context-only entries:",
    len(
        cell80_context_entries
    ),
)

print(
    "Context used as Answer Evidence:",
    False,
)

print(
    "Citation boundary valid:",
    cell80_citation_boundary_valid,
)

print(
    "No locator fabrication:",
    cell80_no_locator_fabrication,
)


print(
    "\n=== WORKFLOW ==="
)

print(
    "Current State:",
    cell80_current_session.get(
        "current_state"
    ),
)

print(
    "Retarget:",
    "ACCEPT",
)

print(
    "Question:",
    "ACCEPT",
)

print(
    "Expected Answer:",
    "ACCEPT",
)

print(
    "Citation:",
    "PENDING",
)


print(
    "\n=== ALL INTERNAL GATES ==="
)

for name, status in cell80_gate_checks.items():

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== QUEUE / SAFETY ==="
)

print(
    "Queue still 7:",
    cell80_queue_still_7,
)

print(
    "Persistence valid:",
    cell80_persistence_valid,
)

print(
    "Base Seed unchanged:",
    cell80_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell80_full_572_blocked,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell80_failures
)


print(
    "\nCODE CELL 80 GATE:",
    (
        "PASS"
        if cell80_gate
        else "FAIL"
    ),
)


assert cell80_gate, (
    "Pilot #17 RETARGET / Authority Policy / Question+Answer ACCEPT / "
    "Citation Draft failed. Repair this SAME Code Cell 80."
)

=== PILOT #17 RETARGET + AUTHORITY POLICY + QA ACCEPT + CITATION DRAFT ===

=== GLOBAL AUTHORITY POLICY ===
Primary Authority: Dr. Komeil Roudi
Host / Participant independent answer authority: False
Non-primary speakers may provide Context / Question: True
Secondary-to-primary promotion: FORBIDDEN
Policy persisted: True

=== RETARGET ===
Old Target: target-evidence:43487bce46faaf036499ac2921fd1280
Old Target Decision: RETARGET
Selected Candidate: mizan-retarget-candidate:f1b5d1c3c5f5197395156b27c9ce2a50
Replacement Target: target-evidence:380e4a9119baf7cc3a0f02c81cc462a9
Replacement Decision: ACCEPT

=== ATTRIBUTION ===
Context Speaker: یکی از حاضران
Context used as Answer Evidence: False
Primary Speaker: Dr. Komeil Roudi
Source-level Human Certification: True
Self-identification line: 22
Raw Source mutated: False
Parser mutated: False

=== ACCEPTED QUESTION ===
قناعت در مدیریت خرج یعنی چه و چرا بعضی هزینه‌ها ممکن است با نیاز واقعی فرد ارتباطی نداشته باشند؟
Question Draft ID: question-

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 81 — Citation ACCEPT + Pilot #17 Completion + Queue Advance

Citation نهایی Pilot #17 توسط Human Reviewer پذیرفته شد.

در این مرحله:

- Citation با تصمیم `ACCEPT` ثبت می‌شود؛
- Retarget lineage مربوط به Target قبلی حفظ می‌شود؛
- Authority Policy سراسری حفظ می‌شود؛
- «یکی از حاضران» فقط Context باقی می‌ماند؛
- فقط Dr. Komeil Roudi Primary Answer Authority است؛
- Pilot #17 کامل می‌شود؛
- تعداد Caseهای کامل Human Review از 15 به 16 می‌رسد؛
- تعداد Caseهای آماده Author Review از 15 به 16 می‌رسد؛
- Queue دقیقاً از 7 به 6 جلو می‌رود؛
- Pilot بعدی برای Human Review آماده می‌شود؛
- Author Final Review همچنان `NOT_STARTED` است؛
- Full 572 Generation همچنان Blocked باقی می‌ماند.

</div>

In [81]:
# Code Cell 81

cell81_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell80_gate") is not True:
    cell81_failures.append(
        "CELL80_NOT_PASS"
    )


if (
    globals().get(
        "cell80_current_session",
        {},
    ).get(
        "current_state"
    )
    !=
    "CITATION_HUMAN_REVIEW_PENDING"
):
    cell81_failures.append(
        "CITATION_NOT_AT_HUMAN_REVIEW_GATE"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell81_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Human Citation ACCEPT
# ============================================================

CELL81_CITATION_HUMAN_NOTE = (
    "Human Reviewer accepted the final Citation Mapping. "
    "The audience-member statement remains context-only. "
    "Dr. Komeil Roudi remains the sole primary answer authority. "
    "The retarget lineage and global authority evidence policy "
    "must remain preserved."
)


cell81_complete_pending_session = (
    cell13_apply_event(
        cell80_current_session,
        "CITATION_ACCEPTED",
        human_note=
            CELL81_CITATION_HUMAN_NOTE,
    )
)


cell81_citation_accept_valid = (
    cell81_complete_pending_session.get(
        "current_state"
    )
    ==
    "PILOT_COMPLETE_PENDING_QUEUE_ADVANCE"
)


if not cell81_citation_accept_valid:
    cell81_failures.append(
        "CITATION_ACCEPT_TRANSITION_INVALID"
    )


# ============================================================
# 3) Final Citation Mapping
# ============================================================

cell81_final_citation_mapping_record = copy.deepcopy(
    cell80_citation_mapping_record
)


cell81_final_citation_mapping_record[
    "human_adjudication_status"
] = "ACCEPT"


cell81_final_citation_mapping_record[
    "human_review"
] = {
    "human_decision":
        "ACCEPT",

    "human_note":
        CELL81_CITATION_HUMAN_NOTE,

    "primary_answer_authority":
        "Dr. Komeil Roudi",

    "non_primary_context":
        "یکی از حاضران",

    "non_primary_used_as_answer_evidence":
        False,

    "retarget_used":
        True,

    "authority_policy_active":
        True,

    "raw_source_mutated":
        False,

    "parser_output_mutated":
        False,

    "external_knowledge_used":
        False,
}


# ============================================================
# 4) Complete Human Review case
# ============================================================

cell81_completed_case = copy.deepcopy(
    cell80_active_workspace
)


cell81_completed_case[
    "workflow_state"
] = "PILOT_COMPLETE"


cell81_completed_case[
    "status"
] = "COMPLETED_HUMAN_REVIEW"


cell81_completed_case[
    "final_status"
] = "PILOT_COMPLETE"


cell81_completed_case.setdefault(
    "citation_review",
    {},
)


cell81_completed_case[
    "citation_review"
][
    "human_decision"
] = "ACCEPT"


cell81_completed_case[
    "citation_review"
][
    "human_note"
] = CELL81_CITATION_HUMAN_NOTE


cell81_completed_case[
    "citation_review"
][
    "citation_mapping_record"
] = copy.deepcopy(
    cell81_final_citation_mapping_record
)


cell81_completed_case[
    "completion_summary"
] = {
    "retarget_used":
        True,

    "old_target_evidence_id":
        CELL80_OLD_TARGET_EVIDENCE_ID,

    "final_target_evidence_id":
        CELL80_REPLACEMENT_TARGET_EVIDENCE_ID,

    "selected_retarget_candidate_id":
        CELL80_SELECTED_CANDIDATE_ID,

    "question_decision":
        "ACCEPT",

    "expected_answer_decision":
        "ACCEPT",

    "citation_decision":
        "ACCEPT",

    "multi_speaker_attribution":
        True,

    "primary_answer_authority":
        "Dr. Komeil Roudi",

    "non_primary_context":
        "یکی از حاضران",

    "non_primary_used_as_answer_evidence":
        False,

    "authority_policy_active":
        True,

    "source_level_attribution_certification":
        True,

    "raw_source_mutated":
        False,

    "parser_output_mutated":
        False,

    "author_final_review":
        "NOT_STARTED",
}


cell81_completed_case[
    "author_final_review"
] = {
    "status":
        "NOT_STARTED",

    "decision":
        None,

    "notes":
        None,
}


# ============================================================
# 5) Queue advance: 7 -> 6
# ============================================================

cell81_previous_remaining_pilot_ids = copy.deepcopy(
    cell75_remaining_pilot_ids
)


CELL81_COMPLETED_PILOT_ID = (
    cell81_previous_remaining_pilot_ids[
        0
    ]
)


cell81_remaining_pilot_ids = copy.deepcopy(
    cell81_previous_remaining_pilot_ids[
        1:
    ]
)


cell81_queue_advance_valid = all([
    len(
        cell81_previous_remaining_pilot_ids
    )
    ==
    7,

    CELL81_COMPLETED_PILOT_ID
    ==
    CELL80_PILOT_ID,

    len(
        cell81_remaining_pilot_ids
    )
    ==
    6,

    CELL81_COMPLETED_PILOT_ID
    not in
    cell81_remaining_pilot_ids,
])


if not cell81_queue_advance_valid:
    cell81_failures.append(
        "QUEUE_ADVANCE_INVALID"
    )


CELL81_NEXT_PILOT_ID = (
    cell81_remaining_pilot_ids[
        0
    ]
)


# ============================================================
# 6) Find next Pilot packet
# ============================================================

cell81_raw_next_packet_candidates = (
    cell58_find_pilot_packet_candidates(
        cell13_seed,
        CELL81_NEXT_PILOT_ID,
    )
)


cell81_packet_candidates_by_sha = {}


for candidate in cell81_raw_next_packet_candidates:

    packet_sha = cell10_stable_sha(
        candidate[
            "packet"
        ]
    )


    previous = (
        cell81_packet_candidates_by_sha.get(
            packet_sha
        )
    )


    if (
        previous is None
        or
        candidate[
            "score"
        ]
        >
        previous[
            "score"
        ]
    ):

        cell81_packet_candidates_by_sha[
            packet_sha
        ] = candidate


cell81_ranked_next_packet_candidates = sorted(
    cell81_packet_candidates_by_sha.values(),
    key=lambda item: (
        -item[
            "score"
        ],
        item[
            "path"
        ],
    ),
)


cell81_next_pilot_packet = (
    copy.deepcopy(
        cell81_ranked_next_packet_candidates[
            0
        ][
            "packet"
        ]
    )
    if cell81_ranked_next_packet_candidates
    else None
)


if not isinstance(
    cell81_next_pilot_packet,
    dict,
):

    cell81_failures.append(
        "NEXT_PILOT_PACKET_NOT_FOUND"
    )


# ============================================================
# 7) Extract next Pilot identities
# ============================================================

cell81_next_pilot_ids_found = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell81_next_pilot_packet,
            "question-pilot:",
        )
    )
)


cell81_next_golden_case_ids = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell81_next_pilot_packet,
            "golden-case:",
        )
    )
)


cell81_next_task_draft_ids = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell81_next_pilot_packet,
            "task-draft:",
        )
    )
)


cell81_next_target_evidence_ids = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell81_next_pilot_packet,
            "target-evidence:",
        )
    )
)


cell81_next_identity_valid = all([
    cell81_next_pilot_ids_found
    ==
    [
        CELL81_NEXT_PILOT_ID
    ],

    len(
        cell81_next_golden_case_ids
    )
    ==
    1,

    len(
        cell81_next_task_draft_ids
    )
    ==
    1,

    len(
        cell81_next_target_evidence_ids
    )
    ==
    1,
])


if not cell81_next_identity_valid:
    cell81_failures.append(
        "NEXT_PILOT_IDENTITY_INVALID"
    )


CELL81_NEXT_GOLDEN_CASE_ID = (
    cell81_next_golden_case_ids[
        0
    ]
    if
    len(
        cell81_next_golden_case_ids
    )
    ==
    1
    else
    None
)


CELL81_NEXT_TASK_DRAFT_ID = (
    cell81_next_task_draft_ids[
        0
    ]
    if
    len(
        cell81_next_task_draft_ids
    )
    ==
    1
    else
    None
)


CELL81_NEXT_TARGET_EVIDENCE_ID = (
    cell81_next_target_evidence_ids[
        0
    ]
    if
    len(
        cell81_next_target_evidence_ids
    )
    ==
    1
    else
    None
)


# ============================================================
# 8) Extract next Target Evidence
# ============================================================

cell81_target_part_candidates = (
    cell58_find_key_values(
        cell81_next_pilot_packet,
        "target_text_parts",
    )
)


cell81_target_parts_by_sha = {}


for parts in cell81_target_part_candidates:

    if not isinstance(
        parts,
        list,
    ):
        continue


    if len(
        parts
    ) == 0:
        continue


    parts_sha = cell10_stable_sha(
        parts
    )


    cell81_target_parts_by_sha[
        parts_sha
    ] = parts


cell81_unique_target_part_sets = list(
    cell81_target_parts_by_sha.values()
)


cell81_next_target_text_parts = (
    copy.deepcopy(
        cell81_unique_target_part_sets[
            0
        ]
    )
    if
    len(
        cell81_unique_target_part_sets
    )
    ==
    1
    else
    None
)


if not isinstance(
    cell81_next_target_text_parts,
    list,
):

    cell81_failures.append(
        "NEXT_TARGET_TEXT_PARTS_NOT_UNIQUE"
    )


# ============================================================
# 9) Build readable next Evidence
# ============================================================

cell81_next_evidence_texts = []


for part in (
    cell81_next_target_text_parts
    or
    []
):

    if isinstance(
        part,
        str,
    ):

        text = part.strip()


    elif isinstance(
        part,
        dict,
    ):

        text = (
            part.get(
                "text"
            )
            or
            part.get(
                "target_text"
            )
            or
            part.get(
                "content"
            )
            or
            ""
        )


        if isinstance(
            text,
            str,
        ):

            text = text.strip()


        else:

            text = ""


    else:

        text = ""


    if text:

        cell81_next_evidence_texts.append(
            text
        )


cell81_next_combined_target_text = (
    "\n\n".join(
        cell81_next_evidence_texts
    )
)


cell81_next_evidence_available = bool(
    cell81_next_combined_target_text.strip()
)


if not cell81_next_evidence_available:
    cell81_failures.append(
        "NEXT_EVIDENCE_EMPTY"
    )


# ============================================================
# 10) Extract next Pilot metadata
# ============================================================

def cell81_unique_scalar_for_key(
    payload,
    key,
):

    values = (
        cell58_find_key_values(
            payload,
            key,
        )
    )


    normalized = []


    for value in values:

        if isinstance(
            value,
            (
                str,
                int,
                float,
                bool,
            ),
        ):

            if value not in normalized:
                normalized.append(
                    value
                )


    if len(
        normalized
    ) == 1:

        return normalized[
            0
        ]


    return None


CELL81_NEXT_STRATUM = (
    cell81_unique_scalar_for_key(
        cell81_next_pilot_packet,
        "stratum",
    )
)


CELL81_NEXT_TASK_FAMILY = (
    cell81_unique_scalar_for_key(
        cell81_next_pilot_packet,
        "task_family",
    )
)


CELL81_NEXT_ARCHETYPE = (
    cell81_unique_scalar_for_key(
        cell81_next_pilot_packet,
        "question_archetype",
    )
)


CELL81_NEXT_AUTHORITY_EXPECTATION = (
    cell81_unique_scalar_for_key(
        cell81_next_pilot_packet,
        "authority_expectation",
    )
)


# ============================================================
# 11) Next Pilot Engine session
# ============================================================

cell81_next_pilot_session = (
    cell13_build_session(
        cell81_next_pilot_packet
    )
)


cell81_next_session_valid = (
    cell81_next_pilot_session.get(
        "current_state"
    )
    ==
    "QUESTION_DRAFT_PENDING"
)


if not cell81_next_session_valid:
    cell81_failures.append(
        "NEXT_SESSION_INVALID"
    )


# ============================================================
# 12) Update completed Human Review cases
# ============================================================

cell81_existing_completed_cases = copy.deepcopy(
    cell80_persisted_registry.get(
        "completed_human_review_cases",
        [],
    )
)


if not isinstance(
    cell81_existing_completed_cases,
    list,
):

    cell81_existing_completed_cases = []

    cell81_failures.append(
        "COMPLETED_CASES_NOT_LIST"
    )


cell81_existing_completed_cases = [
    case
    for case
    in cell81_existing_completed_cases
    if not (
        isinstance(
            case,
            dict,
        )
        and
        case.get(
            "pilot_case_id"
        )
        ==
        CELL81_COMPLETED_PILOT_ID
    )
]


cell81_completed_human_review_cases = (
    cell81_existing_completed_cases
    +
    [
        copy.deepcopy(
            cell81_completed_case
        )
    ]
)


cell81_completed_case_count_valid = (
    len(
        cell81_completed_human_review_cases
    )
    ==
    16
)


if not cell81_completed_case_count_valid:
    cell81_failures.append(
        "COMPLETED_CASE_COUNT_INVALID"
    )


# ============================================================
# 13) Rebuild Author Review queue
# ============================================================

cell81_author_review = copy.deepcopy(
    cell80_persisted_registry.get(
        "author_review",
        {},
    )
)


if not isinstance(
    cell81_author_review,
    dict,
):

    cell81_author_review = {}


cell81_author_ready_cases = copy.deepcopy(
    cell81_completed_human_review_cases
)


cell81_author_review[
    "cases_ready_for_review"
] = copy.deepcopy(
    cell81_author_ready_cases
)


cell81_author_review[
    "case_count"
] = len(
    cell81_author_ready_cases
)


cell81_author_review[
    "final_review_status"
] = "NOT_STARTED"


cell81_author_ready_count_valid = (
    len(
        cell81_author_ready_cases
    )
    ==
    16
)


cell81_author_ready_identity_valid = (
    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell81_author_ready_cases
        if isinstance(
            case,
            dict,
        )
    }
    ==
    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell81_completed_human_review_cases
        if isinstance(
            case,
            dict,
        )
    }
)


if not cell81_author_ready_count_valid:
    cell81_failures.append(
        "AUTHOR_READY_COUNT_INVALID"
    )


if not cell81_author_ready_identity_valid:
    cell81_failures.append(
        "AUTHOR_READY_IDENTITY_INVALID"
    )


# ============================================================
# 14) Preserve Global Authority Policy
# ============================================================

cell81_authority_policy = copy.deepcopy(
    cell80_persisted_registry.get(
        "workflow_policy",
        {},
    ).get(
        "authority_evidence_policy",
        {},
    )
)


cell81_authority_policy_valid = all([
    cell81_authority_policy.get(
        "primary_authority"
    )
    ==
    "Dr. Komeil Roudi",

    cell81_authority_policy.get(
        "secondary_to_primary_promotion"
    )
    ==
    "FORBIDDEN",

    cell81_authority_policy.get(
        "human_certified"
    )
    is True,
])


if not cell81_authority_policy_valid:
    cell81_failures.append(
        "GLOBAL_AUTHORITY_POLICY_INVALID"
    )


# ============================================================
# 15) Build next active workspace
# ============================================================

cell81_next_active_workspace = {
    "pilot_case_id":
        CELL81_NEXT_PILOT_ID,

    "golden_case_id":
        CELL81_NEXT_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL81_NEXT_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL81_NEXT_TARGET_EVIDENCE_ID,

    "status":
        "ACTIVE_HUMAN_REVIEW",

    "workflow_state":
        "QUESTION_DRAFT_PENDING",

    "stratum":
        CELL81_NEXT_STRATUM,

    "task_family":
        CELL81_NEXT_TASK_FAMILY,

    "question_archetype":
        CELL81_NEXT_ARCHETYPE,

    "authority_expectation":
        CELL81_NEXT_AUTHORITY_EXPECTATION,

    "authority_policy":
        copy.deepcopy(
            cell81_authority_policy
        ),

    "source_evidence":
        copy.deepcopy(
            cell81_next_target_text_parts
        ),

    "combined_target_text":
        cell81_next_combined_target_text,

    "question_review":
        None,

    "expected_answer_review":
        None,

    "citation_review":
        None,

    "author_final_review": {
        "status":
            "NOT_STARTED",

        "decision":
            None,

        "notes":
            None,
    },
}


# ============================================================
# 16) Continuation state
# ============================================================

cell81_continuation_state_core = {
    "next_pilot_id":
        CELL81_NEXT_PILOT_ID,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell81_remaining_pilot_ids
        ),

    "remaining_count":
        6,

    "active_pilot_status":
        "QUESTION_DRAFT_PENDING",

    "active_target_evidence_id":
        CELL81_NEXT_TARGET_EVIDENCE_ID,

    "last_completed_pilot_id":
        CELL81_COMPLETED_PILOT_ID,

    "last_completion_type":
        "RETARGETED_MULTI_SPEAKER_AUTHORITY_POLICY_PILOT_COMPLETE",

    "authority_policy_active":
        True,
}


cell81_continuation_state_sha256 = (
    cell10_stable_sha(
        cell81_continuation_state_core
    )
)


cell81_continuation_state = {
    **copy.deepcopy(
        cell81_continuation_state_core
    ),

    "continuation_state_sha256":
        cell81_continuation_state_sha256,
}


# ============================================================
# 17) Persist Registry
# ============================================================

cell81_persistent_registry_payload = copy.deepcopy(
    cell80_persisted_registry
)


cell81_persistent_registry_payload[
    "completed_human_review_cases"
] = copy.deepcopy(
    cell81_completed_human_review_cases
)


cell81_persistent_registry_payload[
    "author_review"
] = copy.deepcopy(
    cell81_author_review
)


cell81_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell81_next_active_workspace
)


cell81_persistent_registry_payload[
    "remaining_pilot_count"
] = 6


cell81_persistent_registry_payload[
    "continuation_state"
] = copy.deepcopy(
    cell81_continuation_state
)


cell81_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell81_persistent_registry_payload
    )
)


# ============================================================
# 18) Read-back validation
# ============================================================

cell81_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell81_persisted_active_case = (
    cell81_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell81_persisted_completed_cases = (
    cell81_persisted_registry.get(
        "completed_human_review_cases",
        [],
    )
)


cell81_persisted_author_ready_cases = (
    cell81_persisted_registry.get(
        "author_review",
        {},
    ).get(
        "cases_ready_for_review",
        [],
    )
)


cell81_persisted_authority_policy = (
    cell81_persisted_registry.get(
        "workflow_policy",
        {},
    ).get(
        "authority_evidence_policy",
        {},
    )
)


cell81_persistence_valid = all([
    isinstance(
        cell81_persisted_completed_cases,
        list,
    ),

    len(
        cell81_persisted_completed_cases
    )
    ==
    16,

    isinstance(
        cell81_persisted_author_ready_cases,
        list,
    ),

    len(
        cell81_persisted_author_ready_cases
    )
    ==
    16,

    cell81_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    6,

    cell81_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL81_NEXT_PILOT_ID,

    cell81_persisted_active_case.get(
        "workflow_state"
    )
    ==
    "QUESTION_DRAFT_PENDING",

    cell81_persisted_registry.get(
        "continuation_state",
        {},
    ).get(
        "continuation_state_sha256"
    )
    ==
    cell81_continuation_state_sha256,

    cell81_persisted_authority_policy.get(
        "primary_authority"
    )
    ==
    "Dr. Komeil Roudi",

    cell81_persisted_authority_policy.get(
        "secondary_to_primary_promotion"
    )
    ==
    "FORBIDDEN",

    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell81_persisted_completed_cases
        if isinstance(
            case,
            dict,
        )
    }
    ==
    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell81_persisted_author_ready_cases
        if isinstance(
            case,
            dict,
        )
    },
])


if not cell81_persistence_valid:
    cell81_failures.append(
        "REGISTRY_READBACK_INVALID"
    )


# ============================================================
# 19) Safety
# ============================================================

cell81_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell81_full_572_blocked = (
    cell80_full_572_blocked
    is True
)


cell81_no_commit_push = True


# ============================================================
# 20) Internal Gates
# ============================================================

cell81_gate_checks = {
    "no_failures":
        len(
            cell81_failures
        )
        ==
        0,

    "citation_accept_valid":
        cell81_citation_accept_valid,

    "queue_advance_valid":
        cell81_queue_advance_valid,

    "next_identity_valid":
        cell81_next_identity_valid,

    "next_evidence_available":
        cell81_next_evidence_available,

    "next_session_valid":
        cell81_next_session_valid,

    "completed_case_count_valid":
        cell81_completed_case_count_valid,

    "author_ready_count_valid":
        cell81_author_ready_count_valid,

    "author_ready_identity_valid":
        cell81_author_ready_identity_valid,

    "authority_policy_valid":
        cell81_authority_policy_valid,

    "persistence_valid":
        cell81_persistence_valid,

    "base_seed_unchanged":
        cell81_base_seed_unchanged,

    "full_572_blocked":
        cell81_full_572_blocked,

    "no_commit_push":
        cell81_no_commit_push,
}


cell81_gate = all(
    cell81_gate_checks.values()
)


# ============================================================
# 21) Diagnostics
# ============================================================

print(
    "=== PILOT #17 COMPLETE + QUEUE ADVANCE + NEXT PILOT ==="
)


print(
    "\n=== COMPLETED PILOT ==="
)

print(
    "Pilot:",
    CELL81_COMPLETED_PILOT_ID,
)

print(
    "Old Target:",
    CELL80_OLD_TARGET_EVIDENCE_ID,
)

print(
    "Final Target:",
    CELL80_REPLACEMENT_TARGET_EVIDENCE_ID,
)

print(
    "Retarget:",
    "ACCEPT",
)

print(
    "Question:",
    "ACCEPT",
)

print(
    "Expected Answer:",
    "ACCEPT",
)

print(
    "Citation:",
    "ACCEPT",
)

print(
    "Final Citation Mapping:",
    cell80_citation_mapping_id,
)

print(
    "Final Engine State:",
    cell81_complete_pending_session.get(
        "current_state"
    ),
)


print(
    "\n=== AUTHORITY RESULT ==="
)

print(
    "Primary Authority:",
    "Dr. Komeil Roudi",
)

print(
    "Non-primary Context:",
    "یکی از حاضران",
)

print(
    "Non-primary used as Answer Evidence:",
    False,
)

print(
    "Secondary-to-primary promotion:",
    "FORBIDDEN",
)

print(
    "Global Authority Policy preserved:",
    cell81_authority_policy_valid,
)

print(
    "Raw Source mutated:",
    False,
)

print(
    "Parser mutated:",
    False,
)


print(
    "\n=== QUEUE ADVANCE ==="
)

print(
    "Before:",
    len(
        cell81_previous_remaining_pilot_ids
    ),
)

print(
    "After:",
    len(
        cell81_remaining_pilot_ids
    ),
)

print(
    "Queue advanced exactly once:",
    cell81_queue_advance_valid,
)


print(
    "\n=== HUMAN REVIEW REGISTRY ==="
)

print(
    "Completed Human Review cases:",
    len(
        cell81_completed_human_review_cases
    ),
)

print(
    "Cases ready for Author Review:",
    len(
        cell81_author_ready_cases
    ),
)

print(
    "Author Final Review:",
    "NOT_STARTED",
)

print(
    "Persistence valid:",
    cell81_persistence_valid,
)


print(
    "\n=== NEXT PILOT ==="
)

print(
    "Pilot:",
    CELL81_NEXT_PILOT_ID,
)

print(
    "Golden Case:",
    CELL81_NEXT_GOLDEN_CASE_ID,
)

print(
    "Task Draft:",
    CELL81_NEXT_TASK_DRAFT_ID,
)

print(
    "Target Evidence:",
    CELL81_NEXT_TARGET_EVIDENCE_ID,
)

print(
    "Stratum:",
    CELL81_NEXT_STRATUM,
)

print(
    "Task Family:",
    CELL81_NEXT_TASK_FAMILY,
)

print(
    "Question Archetype:",
    CELL81_NEXT_ARCHETYPE,
)

print(
    "Authority Expectation:",
    CELL81_NEXT_AUTHORITY_EXPECTATION,
)

print(
    "Workflow:",
    cell81_next_pilot_session.get(
        "current_state"
    ),
)


print(
    "\n=== NEXT SOURCE EVIDENCE ==="
)

print(
    cell81_next_combined_target_text
)


print(
    "\n=== CONTINUATION STATE ==="
)

print(
    "Remaining Pilots:",
    6,
)

print(
    "Continuation SHA256:",
    cell81_continuation_state_sha256,
)


print(
    "\n=== ALL INTERNAL GATES ==="
)

for name, status in cell81_gate_checks.items():

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== SAFETY ==="
)

print(
    "Base Seed unchanged:",
    cell81_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell81_full_572_blocked,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell81_failures
)


print(
    "\nCODE CELL 81 GATE:",
    (
        "PASS"
        if cell81_gate
        else "FAIL"
    ),
)


assert cell81_gate, (
    "Pilot #17 Citation ACCEPT / completion / queue advance / "
    "next Pilot preparation failed. Repair this SAME Code Cell 81."
)

=== PILOT #17 COMPLETE + QUEUE ADVANCE + NEXT PILOT ===

=== COMPLETED PILOT ===
Pilot: question-pilot:dc845f69631a7000938b161802bc3030
Old Target: target-evidence:43487bce46faaf036499ac2921fd1280
Final Target: target-evidence:380e4a9119baf7cc3a0f02c81cc462a9
Retarget: ACCEPT
Question: ACCEPT
Expected Answer: ACCEPT
Citation: ACCEPT
Final Citation Mapping: citation-mapping:15de40622182d2e2891d683a10876001
Final Engine State: PILOT_COMPLETE_PENDING_QUEUE_ADVANCE

=== AUTHORITY RESULT ===
Primary Authority: Dr. Komeil Roudi
Non-primary Context: یکی از حاضران
Non-primary used as Answer Evidence: False
Secondary-to-primary promotion: FORBIDDEN
Global Authority Policy preserved: True
Raw Source mutated: False
Parser mutated: False

=== QUEUE ADVANCE ===
Before: 7
After: 6
Queue advanced exactly once: True

=== HUMAN REVIEW REGISTRY ===
Completed Human Review cases: 16
Cases ready for Author Review: 16
Author Final Review: NOT_STARTED
Persistence valid: True

=== NEXT PILOT ===
Pilot: questi

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 82 — Pilot #18 Primary / Non-Primary Boundary Diagnostic

Pilot فعلی باید شامل مرز واقعی بین:

- Primary Authority
- و یک Speaker غیرPrimary صریح

باشد.

قبل از ساخت سؤال بررسی می‌کنیم:

- هر Target Part متعلق به چه Speakerای است؛
- کدام Part Primary است؛
- کدام Part Non-Primary است؛
- آیا هر دو Speaker به‌صورت معتبر و مستقل مشخص شده‌اند؛
- آیا هیچ Speaker unresolved وجود دارد؛
- آیا Target فعلی واقعاً شرط `PRIMARY_AND_EXPLICIT_NONPRIMARY_DISTINCT` را برآورده می‌کند.

این Cell فقط Diagnostic است و هیچ داده‌ای را تغییر نمی‌دهد.

</div>

In [82]:
# Code Cell 82

cell82_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell81_gate") is not True:
    cell82_failures.append(
        "CELL81_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell81_next_target_text_parts"
    ),
    list,
):
    cell82_failures.append(
        "NEXT_TARGET_PARTS_MISSING"
    )


# ============================================================
# 2) Pilot identity
# ============================================================

CELL82_PILOT_ID = (
    CELL81_NEXT_PILOT_ID
)


CELL82_TARGET_EVIDENCE_ID = (
    CELL81_NEXT_TARGET_EVIDENCE_ID
)


CELL82_TASK_FAMILY = (
    CELL81_NEXT_TASK_FAMILY
)


CELL82_AUTHORITY_EXPECTATION = (
    CELL81_NEXT_AUTHORITY_EXPECTATION
)


# ============================================================
# 3) Inspect Target Parts
# ============================================================

cell82_part_diagnostics = []


for index, part in enumerate(
    cell81_next_target_text_parts
):

    if isinstance(
        part,
        dict,
    ):

        speaker_state = (
            part.get(
                "speaker_state"
            )
            or
            {}
        )


        attribution = (
            part.get(
                "attribution"
            )
            or
            {}
        )


        locator = (
            part.get(
                "locator"
            )
            or
            {}
        )


        text = (
            part.get(
                "text"
            )
            or
            part.get(
                "target_text"
            )
            or
            part.get(
                "content"
            )
            or
            ""
        )


        cell82_part_diagnostics.append({
            "index":
                index,

            "document_id":
                part.get(
                    "document_id"
                ),

            "role":
                part.get(
                    "role"
                ),

            "speaker_base_status":
                speaker_state.get(
                    "base_status"
                ),

            "speaker_base_raw":
                speaker_state.get(
                    "base_raw"
                ),

            "speaker_effective_label":
                speaker_state.get(
                    "effective_label"
                ),

            "speaker_effective_canonical":
                speaker_state.get(
                    "effective_canonical"
                ),

            "speaker_effective_primary":
                speaker_state.get(
                    "effective_primary"
                ),

            "attribution_speaker":
                attribution.get(
                    "speaker"
                ),

            "content_authority_type":
                attribution.get(
                    "content_authority_type"
                ),

            "source_line_start":
                locator.get(
                    "source_line_start"
                ),

            "source_line_end":
                locator.get(
                    "source_line_end"
                ),

            "text":
                text,
        })


    else:

        cell82_part_diagnostics.append({
            "index":
                index,

            "document_id":
                None,

            "role":
                None,

            "speaker_base_status":
                None,

            "speaker_base_raw":
                None,

            "speaker_effective_label":
                None,

            "speaker_effective_canonical":
                None,

            "speaker_effective_primary":
                None,

            "attribution_speaker":
                None,

            "content_authority_type":
                None,

            "source_line_start":
                None,

            "source_line_end":
                None,

            "text":
                str(
                    part
                ),
        })


# ============================================================
# 4) Resolve effective speaker per Part
# ============================================================

cell82_resolved_parts = []


for row in cell82_part_diagnostics:

    effective_speaker = (
        row.get(
            "speaker_effective_canonical"
        )
        or
        row.get(
            "speaker_effective_label"
        )
        or
        row.get(
            "speaker_base_raw"
        )
        or
        row.get(
            "attribution_speaker"
        )
    )


    cell82_resolved_parts.append({
        **copy.deepcopy(
            row
        ),

        "resolved_speaker":
            effective_speaker,

        "is_primary":
            (
                row.get(
                    "speaker_effective_primary"
                )
                is True
            ),

        "is_explicit_nonprimary":
            (
                effective_speaker
                is not None
                and
                row.get(
                    "speaker_effective_primary"
                )
                is False
            ),

        "is_unresolved":
            (
                effective_speaker
                is None
                or
                row.get(
                    "speaker_base_status"
                )
                ==
                "UNRESOLVED"
            ),
    })


# ============================================================
# 5) Speaker inventories
# ============================================================

cell82_primary_parts = [
    row
    for row
    in cell82_resolved_parts
    if row[
        "is_primary"
    ]
]


cell82_explicit_nonprimary_parts = [
    row
    for row
    in cell82_resolved_parts
    if row[
        "is_explicit_nonprimary"
    ]
]


cell82_unresolved_parts = [
    row
    for row
    in cell82_resolved_parts
    if row[
        "is_unresolved"
    ]
]


cell82_detected_speakers = sorted(
    {
        row[
            "resolved_speaker"
        ]
        for row
        in cell82_resolved_parts
        if row[
            "resolved_speaker"
        ]
        is not None
    }
)


# ============================================================
# 6) Validate required structure
# ============================================================

cell82_expected_task_family = (
    CELL82_TASK_FAMILY
    ==
    "PRIMARY_NONPRIMARY_BOUNDARY_RETRIEVAL"
)


cell82_expected_authority_rule = (
    CELL82_AUTHORITY_EXPECTATION
    ==
    "PRIMARY_AND_EXPLICIT_NONPRIMARY_DISTINCT"
)


cell82_exact_two_parts = (
    len(
        cell82_resolved_parts
    )
    ==
    2
)


cell82_boundary_already_valid = all([
    cell82_exact_two_parts,

    len(
        cell82_primary_parts
    )
    ==
    1,

    len(
        cell82_explicit_nonprimary_parts
    )
    ==
    1,

    len(
        cell82_unresolved_parts
    )
    ==
    0,

    len(
        cell82_detected_speakers
    )
    ==
    2,
])


# ============================================================
# 7) Inspect relevant metadata
# ============================================================

def cell82_collect_relevant_metadata(
    value,
    path=(),
):

    output = []


    keywords = (
        "speaker",
        "authority",
        "attribution",
        "primary",
        "nonprimary",
        "boundary",
        "guardrail",
        "task_family",
        "question_archetype",
        "stratum",
    )


    if isinstance(
        value,
        dict,
    ):

        for key, item in value.items():

            key_text = str(
                key
            ).lower()


            if any(
                token
                in
                key_text
                for token
                in keywords
            ):

                output.append({
                    "path":
                        path
                        +
                        (
                            str(
                                key
                            ),
                        ),

                    "value":
                        copy.deepcopy(
                            item
                        ),
                })


            output.extend(
                cell82_collect_relevant_metadata(
                    item,
                    path
                    +
                    (
                        str(
                            key
                        ),
                    ),
                )
            )


    elif isinstance(
        value,
        list,
    ):

        for index, item in enumerate(
            value
        ):

            output.extend(
                cell82_collect_relevant_metadata(
                    item,
                    path
                    +
                    (
                        str(
                            index
                        ),
                    ),
                )
            )


    return output


cell82_relevant_metadata = (
    cell82_collect_relevant_metadata(
        cell81_next_pilot_packet
    )
)


# ============================================================
# 8) Global Authority Policy check
# ============================================================

cell82_authority_policy = copy.deepcopy(
    cell81_persisted_registry.get(
        "workflow_policy",
        {},
    ).get(
        "authority_evidence_policy",
        {},
    )
)


cell82_global_authority_policy_valid = all([
    cell82_authority_policy.get(
        "primary_authority"
    )
    ==
    "Dr. Komeil Roudi",

    cell82_authority_policy.get(
        "secondary_to_primary_promotion"
    )
    ==
    "FORBIDDEN",
])


if not cell82_global_authority_policy_valid:
    cell82_failures.append(
        "GLOBAL_AUTHORITY_POLICY_INVALID"
    )


# ============================================================
# 9) Diagnostic structural gate
# ============================================================

cell82_diagnostic_structure_valid = all([
    cell82_expected_task_family,
    cell82_expected_authority_rule,
    len(
        cell82_resolved_parts
    )
    >
    0,
])


if not cell82_diagnostic_structure_valid:
    cell82_failures.append(
        "DIAGNOSTIC_STRUCTURE_INVALID"
    )


# ============================================================
# 10) Safety
# ============================================================

cell82_registry_write = False

cell82_queue_advance = False

cell82_human_decision_added = False

cell82_target_changed = False

cell82_citation_changed = False

cell82_raw_source_mutated = False

cell82_parser_mutated = False

cell82_commit_push = False


# ============================================================
# 11) Gate
# ============================================================

cell82_gate_checks = {
    "no_failures":
        len(
            cell82_failures
        )
        ==
        0,

    "diagnostic_structure_valid":
        cell82_diagnostic_structure_valid,

    "global_authority_policy_valid":
        cell82_global_authority_policy_valid,

    "target_parts_present":
        len(
            cell82_resolved_parts
        )
        >
        0,
}


cell82_gate = all(
    cell82_gate_checks.values()
)


# ============================================================
# 12) Diagnostics
# ============================================================

print(
    "=== PILOT #18 PRIMARY / NON-PRIMARY BOUNDARY DIAGNOSTIC ==="
)


print(
    "\n=== PILOT ==="
)

print(
    "Pilot:",
    CELL82_PILOT_ID,
)

print(
    "Target Evidence:",
    CELL82_TARGET_EVIDENCE_ID,
)

print(
    "Task Family:",
    CELL82_TASK_FAMILY,
)

print(
    "Authority Expectation:",
    CELL82_AUTHORITY_EXPECTATION,
)


print(
    "\n=== TARGET PARTS ==="
)


for row in cell82_resolved_parts:

    print(
        "\n"
        +
        "="
        *
        72
    )

    print(
        "PART:",
        row[
            "index"
        ],
    )

    print(
        "Document:",
        row.get(
            "document_id"
        ),
    )

    print(
        "Role:",
        row.get(
            "role"
        ),
    )

    print(
        "Base status:",
        row.get(
            "speaker_base_status"
        ),
    )

    print(
        "Base raw:",
        row.get(
            "speaker_base_raw"
        ),
    )

    print(
        "Effective label:",
        row.get(
            "speaker_effective_label"
        ),
    )

    print(
        "Effective canonical:",
        row.get(
            "speaker_effective_canonical"
        ),
    )

    print(
        "Effective primary:",
        row.get(
            "speaker_effective_primary"
        ),
    )

    print(
        "Attribution speaker:",
        row.get(
            "attribution_speaker"
        ),
    )

    print(
        "Content authority type:",
        row.get(
            "content_authority_type"
        ),
    )

    print(
        "Resolved speaker:",
        row.get(
            "resolved_speaker"
        ),
    )

    print(
        "Is Primary:",
        row.get(
            "is_primary"
        ),
    )

    print(
        "Is Explicit Non-Primary:",
        row.get(
            "is_explicit_nonprimary"
        ),
    )

    print(
        "Is Unresolved:",
        row.get(
            "is_unresolved"
        ),
    )

    print(
        "Source lines:",
        (
            row.get(
                "source_line_start"
            ),
            row.get(
                "source_line_end"
            ),
        ),
    )

    print(
        "Text:"
    )

    print(
        row.get(
            "text"
        )
    )


print(
    "\n=== BOUNDARY SUMMARY ==="
)

print(
    "Part count:",
    len(
        cell82_resolved_parts
    ),
)

print(
    "Detected speakers:",
    cell82_detected_speakers,
)

print(
    "Primary parts:",
    len(
        cell82_primary_parts
    ),
)

print(
    "Explicit Non-Primary parts:",
    len(
        cell82_explicit_nonprimary_parts
    ),
)

print(
    "Unresolved parts:",
    len(
        cell82_unresolved_parts
    ),
)

print(
    "Boundary already valid:",
    cell82_boundary_already_valid,
)


print(
    "\n=== GLOBAL AUTHORITY POLICY ==="
)

print(
    "Primary Authority:",
    cell82_authority_policy.get(
        "primary_authority"
    ),
)

print(
    "Secondary-to-primary promotion:",
    cell82_authority_policy.get(
        "secondary_to_primary_promotion"
    ),
)

print(
    "Policy valid:",
    cell82_global_authority_policy_valid,
)


print(
    "\n=== RELEVANT PILOT METADATA ==="
)


for match in cell82_relevant_metadata:

    print(
        " > ".join(
            match[
                "path"
            ]
        ),
        "=",
        match[
            "value"
        ],
    )


print(
    "\n=== SAFETY ==="
)

print(
    "Registry write:",
    cell82_registry_write,
)

print(
    "Queue advance:",
    cell82_queue_advance,
)

print(
    "Human Decision added:",
    cell82_human_decision_added,
)

print(
    "Target changed:",
    cell82_target_changed,
)

print(
    "Citation changed:",
    cell82_citation_changed,
)

print(
    "Raw Source mutated:",
    cell82_raw_source_mutated,
)

print(
    "Parser mutated:",
    cell82_parser_mutated,
)

print(
    "Commit / Push:",
    cell82_commit_push,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell82_failures
)


print(
    "\nCODE CELL 82 GATE:",
    (
        "PASS"
        if cell82_gate
        else "FAIL"
    ),
)


assert cell82_gate, (
    "Pilot #18 primary/non-primary boundary diagnostic failed. "
    "Repair this SAME Code Cell 82 before drafting any question."
)

=== PILOT #18 PRIMARY / NON-PRIMARY BOUNDARY DIAGNOSTIC ===

=== PILOT ===
Pilot: question-pilot:980720fef1985610dd288cab04b7041c
Target Evidence: target-evidence:3a0d4cc342f0c61728501ee45568ed35
Task Family: PRIMARY_NONPRIMARY_BOUNDARY_RETRIEVAL
Authority Expectation: PRIMARY_AND_EXPLICIT_NONPRIMARY_DISTINCT

=== TARGET PARTS ===

PART: 0
Document: program:uni-tehran:episode-014:utterance:00002
Role: ADJACENT_PAIR_LEFT
Base status: EXPLICIT_SOURCE_MARKER
Base raw: استاد
Effective label: استاد
Effective canonical: Dr. Komeil Roudi
Effective primary: True
Attribution speaker: Dr. Komeil Roudi
Content authority type: None
Resolved speaker: Dr. Komeil Roudi
Is Primary: True
Is Explicit Non-Primary: False
Is Unresolved: False
Source lines: (7, 9)
Text:
به نظرم می‌رسه که...

PART: 1
Document: program:uni-tehran:episode-014:utterance:00003
Role: ADJACENT_PAIR_RIGHT
Base status: EXPLICIT_SOURCE_MARKER
Base raw: مجری
Effective label: مجری
Effective canonical: None
Effective primary: None
Attri

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 83 — Pilot #18 Post-Host Response Diagnostic

Target فعلی یک مرز واقعی بین Dr. Komeil Roudi و مجری دارد، اما بخش محتوایی اصلی درباره بودجه‌بندی توسط مجری بیان شده است.

طبق Authority Policy:

- حرف مجری به‌تنهایی Expected Answer Evidence نیست؛
- فقط اگر دکتر رودی آن را تأیید، رد، اصلاح یا توضیح دهد، پاسخ خود دکتر می‌تواند مبنای Golden Answer باشد.

در این Cell، Raw Source و Utteranceهای بعد از Target بررسی می‌شوند تا مشخص شود آیا دکتر به توضیح مجری درباره بودجه‌بندی واکنش معتبر دارد یا خیر.

این Cell فقط Diagnostic است و هیچ داده‌ای را تغییر نمی‌دهد.

</div>

In [83]:
# Code Cell 83 — COMPLETE REPLACEMENT

from pathlib import Path


cell83_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell82_gate") is not True:
    cell83_failures.append(
        "CELL82_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell81_next_target_text_parts"
    ),
    list,
):
    cell83_failures.append(
        "TARGET_PARTS_MISSING"
    )


# ============================================================
# 2) Target identities
# ============================================================

CELL83_PILOT_ID = (
    CELL82_PILOT_ID
)


CELL83_TARGET_EVIDENCE_ID = (
    CELL82_TARGET_EVIDENCE_ID
)


CELL83_PRIMARY_DOCUMENT_ID = (
    cell82_resolved_parts[
        0
    ].get(
        "document_id"
    )
)


CELL83_HOST_DOCUMENT_ID = (
    cell82_resolved_parts[
        1
    ].get(
        "document_id"
    )
)


# ============================================================
# 3) Resolve repository root
# ============================================================

cell83_repo_root = None


for candidate in [
    cell21_registry_path.parent,
    *cell21_registry_path.parents,
]:

    if (
        (
            candidate
            /
            "data"
        ).exists()
        and
        (
            candidate
            /
            "notebooks"
        ).exists()
    ):

        cell83_repo_root = candidate
        break


cell83_repo_root_valid = (
    isinstance(
        cell83_repo_root,
        Path,
    )
    and
    cell83_repo_root.exists()
)


if not cell83_repo_root_valid:
    cell83_failures.append(
        "REPO_ROOT_NOT_FOUND"
    )


# ============================================================
# 4) Normalization
# ============================================================

def cell83_norm(
    text,
):

    if not isinstance(
        text,
        str,
    ):
        return ""


    value = (
        text
        .replace(
            "ي",
            "ی",
        )
        .replace(
            "ك",
            "ک",
        )
        .replace(
            "\u200c",
            "",
        )
        .replace(
            "\ufeff",
            "",
        )
    )


    value = re.sub(
        r"\s+",
        " ",
        value,
    )


    return value.strip()


# ============================================================
# 5) Search authoritative raw source
# ============================================================

CELL83_SEARCH_SNIPPET = (
    "در بودجه‌بندی، یعنی اگه بخوای توضیحش بدیم"
)


cell83_search_norm = cell83_norm(
    CELL83_SEARCH_SNIPPET
)


cell83_matches = []


if cell83_repo_root_valid:

    search_root = (
        cell83_repo_root
        /
        "data"
        /
        "transcripts"
        /
        "final"
    )


    if search_root.exists():

        for path in search_root.rglob(
            "*.md"
        ):

            try:

                raw_text = path.read_text(
                    encoding="utf-8"
                )


            except Exception:

                continue


            lines = raw_text.splitlines()


            matching_lines = []


            for line_number, line in enumerate(
                lines,
                start=1,
            ):

                if (
                    cell83_search_norm
                    in
                    cell83_norm(
                        line
                    )
                ):

                    matching_lines.append(
                        line_number
                    )


            if matching_lines:

                cell83_matches.append({
                    "path":
                        path,

                    "lines":
                        lines,

                    "matching_lines":
                        matching_lines,
                })


cell83_unique_paths = sorted(
    {
        str(
            item[
                "path"
            ].resolve()
        )
        for item
        in cell83_matches
    }
)


cell83_unique_source_valid = (
    len(
        cell83_unique_paths
    )
    ==
    1
)


if not cell83_unique_source_valid:
    cell83_failures.append(
        "RAW_SOURCE_NOT_UNIQUE"
    )


cell83_source_match = (
    cell83_matches[
        0
    ]
    if cell83_unique_source_valid
    else None
)


# ============================================================
# 6) Build raw context
# ============================================================

cell83_context_rows = []


if cell83_source_match is not None:

    cell83_lines = (
        cell83_source_match[
            "lines"
        ]
    )


    cell83_anchor_line = (
        cell83_source_match[
            "matching_lines"
        ][
            0
        ]
    )


    cell83_context_start = max(
        1,
        cell83_anchor_line - 8,
    )


    cell83_context_end = min(
        len(
            cell83_lines
        ),
        cell83_anchor_line + 24,
    )


    for line_number in range(
        cell83_context_start,
        cell83_context_end + 1,
    ):

        cell83_context_rows.append({
            "line_number":
                line_number,

            "text":
                cell83_lines[
                    line_number - 1
                ],
        })


# ============================================================
# 7) Detect Markdown speaker labels
# ============================================================

cell83_markdown_speaker_labels = []


cell83_markdown_label_pattern = re.compile(
    r"^\s*\*\*(.+?)\*\*\s*$"
)


for row in cell83_context_rows:

    match = cell83_markdown_label_pattern.match(
        row[
            "text"
        ]
    )


    if not match:
        continue


    label = match.group(
        1
    ).strip()


    cell83_markdown_speaker_labels.append({
        "line_number":
            row[
                "line_number"
            ],

        "label":
            label,
    })


# ============================================================
# 8) Authority mapping from already validated Pilot metadata
# ============================================================

cell83_primary_source_label = (
    cell82_resolved_parts[
        0
    ].get(
        "speaker_effective_label"
    )
)


cell83_primary_canonical = (
    cell82_resolved_parts[
        0
    ].get(
        "speaker_effective_canonical"
    )
)


cell83_primary_flag = (
    cell82_resolved_parts[
        0
    ].get(
        "speaker_effective_primary"
    )
)


cell83_host_source_label = (
    cell82_resolved_parts[
        1
    ].get(
        "speaker_effective_label"
    )
)


cell83_authority_mapping_valid = all([
    cell83_primary_source_label
    ==
    "استاد",

    cell83_primary_canonical
    ==
    "Dr. Komeil Roudi",

    cell83_primary_flag
    is True,

    cell83_host_source_label
    ==
    "مجری",
])


if not cell83_authority_mapping_valid:
    cell83_failures.append(
        "AUTHORITY_MAPPING_INVALID"
    )


# ============================================================
# 9) Locate host Target turn in raw source
# ============================================================

cell83_host_label_rows = [
    row
    for row
    in cell83_markdown_speaker_labels
    if (
        row[
            "label"
        ]
        ==
        "مجری"
        and
        row[
            "line_number"
        ]
        <
        cell83_anchor_line
    )
]


cell83_host_label_row = (
    max(
        cell83_host_label_rows,
        key=lambda row: row[
            "line_number"
        ],
    )
    if cell83_host_label_rows
    else None
)


cell83_host_label_found = (
    cell83_host_label_row
    is not None
)


if not cell83_host_label_found:
    cell83_failures.append(
        "HOST_LABEL_NOT_FOUND"
    )


# ============================================================
# 10) Find first explicit speaker after host statement
# ============================================================

cell83_next_speaker_after_host = None


if (
    cell83_host_label_found
    and
    cell83_source_match
    is not None
):

    for row in cell83_markdown_speaker_labels:

        if (
            row[
                "line_number"
            ]
            >
            cell83_anchor_line
        ):

            cell83_next_speaker_after_host = row
            break


cell83_primary_response_label_valid = all([
    isinstance(
        cell83_next_speaker_after_host,
        dict,
    ),

    (
        cell83_next_speaker_after_host
        or
        {}
    ).get(
        "label"
    )
    ==
    "استاد",
])


if not cell83_primary_response_label_valid:
    cell83_failures.append(
        "PRIMARY_RESPONSE_LABEL_NOT_FOUND"
    )


# ============================================================
# 11) Extract first Dr. Roudi response block
# ============================================================

cell83_primary_response_start = None
cell83_primary_response_end = None
cell83_primary_response_lines = []
cell83_primary_response_text = ""


if (
    cell83_primary_response_label_valid
    and
    cell83_source_match
    is not None
):

    speaker_label_line = (
        cell83_next_speaker_after_host[
            "line_number"
        ]
    )


    next_boundary_line = None


    for row in cell83_markdown_speaker_labels:

        if (
            row[
                "line_number"
            ]
            >
            speaker_label_line
        ):

            next_boundary_line = (
                row[
                    "line_number"
                ]
            )

            break


    candidate_end = (
        next_boundary_line - 1
        if next_boundary_line
        is not None
        else
        min(
            len(
                cell83_lines
            ),
            speaker_label_line + 20,
        )
    )


    nonempty_rows = []


    for line_number in range(
        speaker_label_line + 1,
        candidate_end + 1,
    ):

        text = (
            cell83_lines[
                line_number - 1
            ]
        )


        if text.strip():

            nonempty_rows.append({
                "line_number":
                    line_number,

                "text":
                    text.strip(),
            })


    if nonempty_rows:

        cell83_primary_response_start = (
            nonempty_rows[
                0
            ][
                "line_number"
            ]
        )


        cell83_primary_response_end = (
            nonempty_rows[
                -1
            ][
                "line_number"
            ]
        )


        cell83_primary_response_lines = (
            nonempty_rows
        )


        cell83_primary_response_text = (
            "\n".join(
                row[
                    "text"
                ]
                for row
                in nonempty_rows
            )
        )


cell83_primary_response_exists = bool(
    cell83_primary_response_text.strip()
)


if not cell83_primary_response_exists:
    cell83_failures.append(
        "PRIMARY_RESPONSE_EMPTY"
    )


# ============================================================
# 12) Explicit confirmation check
# ============================================================

cell83_primary_response_norm = cell83_norm(
    cell83_primary_response_text
)


cell83_explicit_confirmation = (
    "دقیقاً"
    in
    cell83_primary_response_norm
)


# ============================================================
# 13) Substantive budget grounding in Dr. response
# ============================================================

cell83_budget_terms_in_primary_response = {
    "budget":
        (
            "بودجهبندی"
            in
            cell83_primary_response_norm
            or
            "بودجه"
            in
            cell83_primary_response_norm
        ),

    "three_month_history":
        (
            "سه ماه اول"
            in
            cell83_primary_response_norm
        ),

    "record_costs":
        (
            "هزینهها رو مینویسی"
            in
            cell83_primary_response_norm
            or
            "هزینه ها رو مینویسی"
            in
            cell83_primary_response_norm
        ),

    "categorize_costs":
        (
            "دستهبندی از هزینهها"
            in
            cell83_primary_response_norm
            or
            "دستهبندی"
            in
            cell83_primary_response_norm
        ),

    "month_four":
        (
            "اول ماه چهارم"
            in
            cell83_primary_response_norm
        ),

    "planned_amount":
        (
            "چه مقدار هزینه میخواهم"
            in
            cell83_primary_response_norm
        ),

    "planned_category":
        (
            "کدوم یکی از این دستهها"
            in
            cell83_primary_response_norm
            or
            "کدام یکی از این دستهها"
            in
            cell83_primary_response_norm
        ),
}


cell83_primary_response_substantive_match = all([
    cell83_budget_terms_in_primary_response[
        "budget"
    ],

    cell83_budget_terms_in_primary_response[
        "three_month_history"
    ],

    cell83_budget_terms_in_primary_response[
        "month_four"
    ],

    cell83_budget_terms_in_primary_response[
        "planned_amount"
    ],

    cell83_budget_terms_in_primary_response[
        "planned_category"
    ],
])


# ============================================================
# 14) Optional extracted transcript lookup
#     Not required for the gate because raw source is authoritative.
# ============================================================

cell83_extracted_candidate_paths = [
    (
        cell83_repo_root
        /
        "data"
        /
        "phase2"
        /
        "interim"
        /
        "transcripts_extracted_v1.jsonl"
    )
    if cell83_repo_root_valid
    else None,

    (
        cell21_registry_path.parents[
            1
        ]
        /
        "interim"
        /
        "transcripts_extracted_v1.jsonl"
    )
    if len(
        cell21_registry_path.parents
    )
    >
    1
    else None,

    (
        cell83_repo_root
        /
        "data"
        /
        "interim"
        /
        "transcripts_extracted_v1.jsonl"
    )
    if cell83_repo_root_valid
    else None,
]


cell83_extracted_path = next(
    (
        path
        for path
        in cell83_extracted_candidate_paths
        if (
            isinstance(
                path,
                Path,
            )
            and
            path.exists()
        )
    ),
    None,
)


cell83_extracted_path_available = (
    cell83_extracted_path
    is not None
)


# ============================================================
# 15) Authority Policy application
# ============================================================

cell83_host_is_nonprimary_context = all([
    cell83_host_source_label
    ==
    "مجری",

    cell82_authority_policy.get(
        "secondary_to_primary_promotion"
    )
    ==
    "FORBIDDEN",
])


cell83_dr_is_primary_answer_authority = all([
    cell83_primary_canonical
    ==
    "Dr. Komeil Roudi",

    cell83_primary_flag
    is True,
])


cell83_host_statement_independently_usable = False


# ============================================================
# 16) Salvage assessment
# ============================================================

cell83_target_can_be_salvaged = all([
    cell83_authority_mapping_valid,

    cell83_host_label_found,

    cell83_primary_response_label_valid,

    cell83_primary_response_exists,

    cell83_explicit_confirmation,

    cell83_primary_response_substantive_match,

    cell83_host_is_nonprimary_context,

    cell83_dr_is_primary_answer_authority,
])


cell83_retarget_likely_required = (
    not cell83_target_can_be_salvaged
)


# ============================================================
# 17) Safe evidence boundary proposal
# ============================================================

cell83_safe_boundary = {
    "host_context": {
        "speaker":
            "مجری",

        "role":
            "QUESTION_OR_CONTEXT_ONLY",

        "answer_authority":
            False,

        "source_line_start":
            cell83_anchor_line,

        "source_line_end":
            cell83_anchor_line,
    },

    "primary_response": {
        "speaker":
            "Dr. Komeil Roudi",

        "source_label":
            "استاد",

        "role":
            "PRIMARY_ANSWER_EVIDENCE",

        "answer_authority":
            True,

        "source_line_start":
            cell83_primary_response_start,

        "source_line_end":
            cell83_primary_response_end,

        "explicit_confirmation":
            cell83_explicit_confirmation,
    },
}


# ============================================================
# 18) Safety
# ============================================================

cell83_registry_write = False

cell83_queue_advance = False

cell83_human_decision_added = False

cell83_target_changed = False

cell83_citation_changed = False

cell83_raw_source_mutated = False

cell83_parser_mutated = False

cell83_commit_push = False


# ============================================================
# 19) Gate
# ============================================================

cell83_gate_checks = {
    "no_failures":
        len(
            cell83_failures
        )
        ==
        0,

    "repo_root_valid":
        cell83_repo_root_valid,

    "unique_raw_source":
        cell83_unique_source_valid,

    "context_available":
        len(
            cell83_context_rows
        )
        >
        0,

    "authority_mapping_valid":
        cell83_authority_mapping_valid,

    "host_label_found":
        cell83_host_label_found,

    "primary_response_label_valid":
        cell83_primary_response_label_valid,

    "primary_response_exists":
        cell83_primary_response_exists,

    "explicit_confirmation":
        cell83_explicit_confirmation,

    "primary_response_substantive_match":
        cell83_primary_response_substantive_match,

    "authority_boundary_valid":
        all([
            cell83_host_is_nonprimary_context,
            cell83_dr_is_primary_answer_authority,
            not cell83_host_statement_independently_usable,
        ]),
}


cell83_gate = all(
    cell83_gate_checks.values()
)


# ============================================================
# 20) Diagnostics
# ============================================================

print(
    "=== PILOT #18 POST-HOST RESPONSE DIAGNOSTIC ==="
)


print(
    "\n=== RAW SOURCE ==="
)

print(
    "Source:",
    (
        cell83_source_match[
            "path"
        ]
        if cell83_source_match
        is not None
        else None
    ),
)

print(
    "Matched lines:",
    (
        cell83_source_match[
            "matching_lines"
        ]
        if cell83_source_match
        is not None
        else []
    ),
)


print(
    "\n=== RAW SOURCE CONTEXT ==="
)


for row in cell83_context_rows:

    print(
        f"L{row['line_number']:04d}: "
        f"{row['text']}"
    )


print(
    "\n=== MARKDOWN SPEAKER LABELS ==="
)


for item in cell83_markdown_speaker_labels:

    print(
        f"L{item['line_number']}:",
        item[
            "label"
        ],
    )


print(
    "\n=== AUTHORITY MAPPING ==="
)

print(
    "Primary source label:",
    cell83_primary_source_label,
)

print(
    "Primary canonical:",
    cell83_primary_canonical,
)

print(
    "Primary flag:",
    cell83_primary_flag,
)

print(
    "Host source label:",
    cell83_host_source_label,
)

print(
    "Mapping valid:",
    cell83_authority_mapping_valid,
)


print(
    "\n=== HOST TARGET TURN ==="
)

print(
    "Host label line:",
    (
        cell83_host_label_row[
            "line_number"
        ]
        if cell83_host_label_row
        is not None
        else None
    ),
)

print(
    "Host substantive line:",
    cell83_anchor_line,
)

print(
    "Host independent Answer Authority:",
    False,
)


print(
    "\n=== FIRST PRIMARY RESPONSE AFTER HOST ==="
)

print(
    "Speaker label:",
    (
        cell83_next_speaker_after_host[
            "label"
        ]
        if cell83_next_speaker_after_host
        is not None
        else None
    ),
)

print(
    "Canonical speaker:",
    "Dr. Komeil Roudi",
)

print(
    "Response lines:",
    (
        cell83_primary_response_start,
        cell83_primary_response_end,
    ),
)

print(
    "Response:"
)

print(
    cell83_primary_response_text
)


print(
    "\n=== PRIMARY RESPONSE VALIDATION ==="
)

print(
    "Explicit confirmation ('دقیقاً'):",
    cell83_explicit_confirmation,
)


for name, status in (
    cell83_budget_terms_in_primary_response.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "Substantive budget match:",
    cell83_primary_response_substantive_match,
)


print(
    "\n=== SAFE AUTHORITY BOUNDARY ==="
)

print(
    "Host:",
    "CONTEXT ONLY",
)

print(
    "Host used as independent Answer Evidence:",
    False,
)

print(
    "Dr. Komeil Roudi:",
    "PRIMARY ANSWER EVIDENCE",
)

print(
    "Secondary-to-primary promotion:",
    "FORBIDDEN",
)


print(
    "\n=== TARGET ASSESSMENT ==="
)

print(
    "Target can be salvaged:",
    cell83_target_can_be_salvaged,
)

print(
    "Retarget likely required:",
    cell83_retarget_likely_required,
)

print(
    "IMPORTANT: No Human Question/Answer/Citation decision "
    "has been recorded in this Cell."
)


print(
    "\n=== OPTIONAL EXTRACTED TRANSCRIPT ==="
)

print(
    "Available:",
    cell83_extracted_path_available,
)

print(
    "Resolved path:",
    cell83_extracted_path,
)

print(
    "Required for this diagnostic:",
    False,
)


print(
    "\n=== ALL INTERNAL GATES ==="
)

for name, status in cell83_gate_checks.items():

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== SAFETY ==="
)

print(
    "Registry write:",
    cell83_registry_write,
)

print(
    "Queue advance:",
    cell83_queue_advance,
)

print(
    "Human Decision added:",
    cell83_human_decision_added,
)

print(
    "Target changed:",
    cell83_target_changed,
)

print(
    "Citation changed:",
    cell83_citation_changed,
)

print(
    "Raw Source mutated:",
    cell83_raw_source_mutated,
)

print(
    "Parser mutated:",
    cell83_parser_mutated,
)

print(
    "Commit / Push:",
    cell83_commit_push,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell83_failures
)


print(
    "\nCODE CELL 83 GATE:",
    (
        "PASS"
        if cell83_gate
        else "FAIL"
    ),
)


assert cell83_gate, (
    "Pilot #18 post-host response diagnostic failed. "
    "Repair this SAME Code Cell 83 before Question/Answer drafting."
)

=== PILOT #18 POST-HOST RESPONSE DIAGNOSTIC ===

=== RAW SOURCE ===
Source: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/data/transcripts/final/uni-tehran/episode-14.md
Matched lines: [13]

=== RAW SOURCE CONTEXT ===
L0005: توی سرفصل‌های سواد مالی از خرج و پس‌انداز شروع کردیم، اگه خاطرتون باشه، و اولین قدم رو گفتیم نوشتنه و در مورد اهمیت نوشتنِ در واقع تک‌تک تراکنش‌های مالی که در طول روز و در طول زندگیتون دارید صحبت کردیم. بعد رسیدیم به الگوهای پنج‌گانه و حالا وقت بودجه‌بندیه.
L0006: 
L0007: **استاد**
L0008: 
L0009: به نظرم می‌رسه که...
L0010: 
L0011: **مجری**
L0012: 
L0013: در بودجه‌بندی، یعنی اگه بخوای توضیحش بدیم که بودجه‌بندی کجا به کار ما میاد، شاید بهتر باشه بگیم که اون نوشتنه بود که گفتیم یه جوری نگاهی به عقب داره. یعنی داری شما گذشته‌ت رو می‌بینی که چه رفتاری داری انجام می‌دی. وقتی می‌رسی به بودجه‌بندی، انگاری می‌خوای به آینده‌ت نگاه بکنی؛ یعنی یه برنامه‌ریزی برای آینده داشته باشی. توی نوشتن می‌گفتی دیروز و امروز چه کردم، چه کرده‌ام، ولی حالا می‌خوا

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 84 — Pilot #18 Question/Answer ACCEPT + Primary Evidence Extension + Citation Draft

Human Reviewer سؤال و جواب Pilot #18 را پذیرفت.

Target اصلی Retarget نمی‌شود، زیرا Raw Source نشان داد:

- بخش مجری فقط Context است؛
- دکتر رودی بلافاصله با «دقیقاً، دقیقاً» موضوع را تأیید می‌کند؛
- سپس خودش تعریف عملی بودجه‌بندی را توضیح می‌دهد.

برای رعایت Authority Policy، پاسخ مجری به‌تنهایی Evidence جواب محسوب نمی‌شود.  
یک `Human-certified Supporting Evidence Extension` از پاسخ صریح دکتر رودی در خطوط 17 تا 19 به Citation اضافه می‌شود.

سؤال پذیرفته‌شده:

**بودجه‌بندی از چه زمانی شروع می‌شود و چه تفاوتی با ثبت هزینه‌های ماه‌های قبل دارد؟**

جواب پذیرفته‌شده:

**در سه ماه اول، هزینه‌ها ثبت و در پایان هر ماه دسته‌بندی می‌شوند تا الگوی خرج‌کردن مشخص شود. از ابتدای ماه چهارم، پیش از انجام هزینه‌ها تعیین می‌کنیم که در هر دسته چه مقدار می‌خواهیم خرج کنیم؛ این برنامه‌ریزی از قبل همان بودجه‌بندی است.**

بعد از این Cell، Citation در وضعیت Human Review قرار می‌گیرد و Queue همچنان 6 باقی می‌ماند.

</div>

In [84]:
# Code Cell 84

cell84_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell83_gate") is not True:
    cell84_failures.append(
        "CELL83_NOT_PASS"
    )


if globals().get("cell83_target_can_be_salvaged") is not True:
    cell84_failures.append(
        "TARGET_NOT_SALVAGEABLE"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell84_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Human-accepted Question / Answer
# ============================================================

CELL84_ACCEPTED_QUESTION = (
    "بودجه‌بندی از چه زمانی شروع می‌شود و چه تفاوتی با ثبت "
    "هزینه‌های ماه‌های قبل دارد؟"
)


CELL84_ACCEPTED_ANSWER = (
    "در سه ماه اول، هزینه‌ها ثبت و در پایان هر ماه دسته‌بندی "
    "می‌شوند تا الگوی خرج‌کردن مشخص شود. از ابتدای ماه چهارم، "
    "پیش از انجام هزینه‌ها تعیین می‌کنیم که در هر دسته چه مقدار "
    "می‌خواهیم خرج کنیم؛ این برنامه‌ریزی از قبل همان بودجه‌بندی است."
)


CELL84_PILOT_ID = (
    CELL82_PILOT_ID
)


CELL84_GOLDEN_CASE_ID = (
    CELL81_NEXT_GOLDEN_CASE_ID
)


CELL84_TASK_DRAFT_ID = (
    CELL81_NEXT_TASK_DRAFT_ID
)


CELL84_TARGET_EVIDENCE_ID = (
    CELL82_TARGET_EVIDENCE_ID
)


# ============================================================
# 3) Original Target remains unchanged
# ============================================================

cell84_original_target_parts = copy.deepcopy(
    cell81_next_target_text_parts
)


cell84_original_target_valid = all([
    isinstance(
        cell84_original_target_parts,
        list,
    ),

    len(
        cell84_original_target_parts
    )
    ==
    2,

    cell82_resolved_parts[
        0
    ].get(
        "speaker_effective_canonical"
    )
    ==
    "Dr. Komeil Roudi",

    cell82_resolved_parts[
        0
    ].get(
        "speaker_effective_primary"
    )
    is True,

    cell82_resolved_parts[
        1
    ].get(
        "speaker_effective_label"
    )
    ==
    "مجری",
])


if not cell84_original_target_valid:
    cell84_failures.append(
        "ORIGINAL_TARGET_BOUNDARY_INVALID"
    )


# ============================================================
# 4) Verified Dr. Roudi supporting evidence extension
# ============================================================

cell84_source_path = (
    cell83_source_match[
        "path"
    ]
)


cell84_source_relative = str(
    cell84_source_path.relative_to(
        cell83_repo_root
    )
)


cell84_primary_support_text = (
    cell83_primary_response_text.strip()
)


cell84_primary_support_start = (
    cell83_primary_response_start
)


cell84_primary_support_end = (
    cell83_primary_response_end
)


cell84_raw_support_lines = []


for line_number in range(
    cell84_primary_support_start,
    cell84_primary_support_end + 1,
):

    raw_line = (
        cell83_source_match[
            "lines"
        ][
            line_number - 1
        ]
    )


    if raw_line.strip():

        cell84_raw_support_lines.append(
            raw_line.strip()
        )


cell84_raw_support_text = "\n".join(
    cell84_raw_support_lines
)


cell84_support_text_exact = (
    cell84_raw_support_text
    ==
    cell84_primary_support_text
)


if not cell84_support_text_exact:
    cell84_failures.append(
        "PRIMARY_SUPPORT_TEXT_NOT_EXACT"
    )


cell84_supporting_evidence = {
    "evidence_type":
        "HUMAN_CERTIFIED_SUPPORTING_EVIDENCE_EXTENSION",

    "source_type":
        "transcript",

    "source_path":
        cell84_source_relative,

    "document_context":
        "program:uni-tehran:episode-014",

    "speaker_source_label":
        "استاد",

    "speaker_canonical":
        "Dr. Komeil Roudi",

    "speaker_primary":
        True,

    "role":
        "PRIMARY_ANSWER_EVIDENCE",

    "text":
        cell84_primary_support_text,

    "text_sha256":
        hashlib.sha256(
            cell84_primary_support_text.encode(
                "utf-8"
            )
        ).hexdigest(),

    "locator": {
        "source_path":
            cell84_source_relative,

        "source_line_start":
            cell84_primary_support_start,

        "source_line_end":
            cell84_primary_support_end,

        "locator_origin":
            "RAW_SOURCE_VERIFIED_CELL83",
    },

    "authority_basis": {
        "explicit_source_label":
            "استاد",

        "validated_canonical":
            "Dr. Komeil Roudi",

        "validated_primary":
            True,

        "explicit_confirmation":
            "دقیقاً، دقیقاً.",

        "host_statement_used_as_independent_answer_evidence":
            False,

        "secondary_to_primary_promotion":
            "FORBIDDEN",
    },

    "raw_source_mutated":
        False,

    "parser_output_mutated":
        False,
}


cell84_supporting_evidence_valid = all([
    cell84_support_text_exact,

    cell84_supporting_evidence[
        "speaker_canonical"
    ]
    ==
    "Dr. Komeil Roudi",

    cell84_supporting_evidence[
        "speaker_primary"
    ]
    is True,

    cell83_explicit_confirmation
    is True,

    cell83_primary_response_substantive_match
    is True,
])


if not cell84_supporting_evidence_valid:
    cell84_failures.append(
        "SUPPORTING_EVIDENCE_INVALID"
    )


# ============================================================
# 5) Grounding
# ============================================================

def cell84_norm(
    text,
):

    if not isinstance(
        text,
        str,
    ):
        return ""


    value = (
        text
        .replace(
            "ي",
            "ی",
        )
        .replace(
            "ك",
            "ک",
        )
        .replace(
            "\u200c",
            "",
        )
        .replace(
            "\ufeff",
            "",
        )
    )


    value = re.sub(
        r"\s+",
        " ",
        value,
    )


    return value.strip()


cell84_primary_surface = cell84_norm(
    cell84_primary_support_text
)


cell84_grounding = {
    "three_months":
        (
            "سه ماه اول"
            in
            cell84_primary_surface
        ),

    "record_expenses":
        (
            "هزینهها رو مینویسی"
            in
            cell84_primary_surface
        ),

    "categorize_expenses":
        (
            "دستهبندی از هزینهها"
            in
            cell84_primary_surface
        ),

    "month_four":
        (
            "اول ماه چهارم"
            in
            cell84_primary_surface
        ),

    "planned_expense":
        (
            "چه مقدار هزینه میخواهم"
            in
            cell84_primary_surface
        ),

    "category":
        (
            "کدوم یکی از این دستهها"
            in
            cell84_primary_surface
        ),

    "budget_definition":
        (
            "اینو بهش میگیم بودجهبندی"
            in
            cell84_primary_surface
        ),
}


cell84_grounding_pass = all(
    cell84_grounding.values()
)


if not cell84_grounding_pass:
    cell84_failures.append(
        "ANSWER_GROUNDING_FAILED"
    )


# ============================================================
# 6) Start Engine from original Pilot packet
# ============================================================

cell84_question_pending_session = (
    cell13_build_session(
        cell81_next_pilot_packet
    )
)


cell84_session_start_valid = (
    cell84_question_pending_session.get(
        "current_state"
    )
    ==
    "QUESTION_DRAFT_PENDING"
)


if not cell84_session_start_valid:
    cell84_failures.append(
        "ENGINE_SESSION_START_INVALID"
    )


# ============================================================
# 7) Question Draft + Human ACCEPT
# ============================================================

cell84_question_identity = {
    "identity_version":
        "question_draft_identity_v1",

    "pilot_case_id":
        CELL84_PILOT_ID,

    "target_evidence_id":
        CELL84_TARGET_EVIDENCE_ID,

    "question_text":
        CELL84_ACCEPTED_QUESTION,

    "revision_number":
        1,
}


cell84_question_draft_id = (
    "question-draft:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell84_question_identity
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


cell84_question_record = {
    "record_version":
        "question_draft_record_v1",

    "question_draft_id":
        cell84_question_draft_id,

    "pilot_case_id":
        CELL84_PILOT_ID,

    "golden_case_id":
        CELL84_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL84_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL84_TARGET_EVIDENCE_ID,

    "question_text":
        CELL84_ACCEPTED_QUESTION,

    "revision_number":
        1,

    "human_adjudication_status":
        "PENDING",

    "external_knowledge_used":
        False,
}


cell84_question_review_session = (
    cell13_apply_event(
        cell84_question_pending_session,
        "QUESTION_DRAFT_REGISTERED",
        artifact=
            cell84_question_record,
    )
)


cell84_answer_pending_session = (
    cell13_apply_event(
        cell84_question_review_session,
        "QUESTION_ACCEPTED",
        human_note=
            "Human Reviewer accepted the question.",
    )
)


cell84_question_accept_valid = (
    cell84_answer_pending_session.get(
        "current_state"
    )
    ==
    "EXPECTED_ANSWER_DRAFT_PENDING"
)


if not cell84_question_accept_valid:
    cell84_failures.append(
        "QUESTION_ACCEPT_TRANSITION_INVALID"
    )


# ============================================================
# 8) Expected Answer + Human ACCEPT
# ============================================================

cell84_answer_identity = {
    "identity_version":
        "expected_answer_identity_v1",

    "pilot_case_id":
        CELL84_PILOT_ID,

    "question_draft_id":
        cell84_question_draft_id,

    "target_evidence_id":
        CELL84_TARGET_EVIDENCE_ID,

    "answer_text":
        CELL84_ACCEPTED_ANSWER,

    "revision_number":
        1,
}


cell84_expected_answer_draft_id = (
    "expected-answer-draft:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell84_answer_identity
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


cell84_expected_answer_record = {
    "record_version":
        "expected_answer_draft_record_v1",

    "expected_answer_draft_id":
        cell84_expected_answer_draft_id,

    "pilot_case_id":
        CELL84_PILOT_ID,

    "golden_case_id":
        CELL84_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL84_TASK_DRAFT_ID,

    "question_draft_id":
        cell84_question_draft_id,

    "target_evidence_id":
        CELL84_TARGET_EVIDENCE_ID,

    "expected_answer_text":
        CELL84_ACCEPTED_ANSWER,

    "revision_number":
        1,

    "human_adjudication_status":
        "PENDING",

    "external_knowledge_used":
        False,
}


cell84_answer_review_session = (
    cell13_apply_event(
        cell84_answer_pending_session,
        "EXPECTED_ANSWER_REGISTERED",
        artifact=
            cell84_expected_answer_record,
    )
)


cell84_citation_pending_session = (
    cell13_apply_event(
        cell84_answer_review_session,
        "EXPECTED_ANSWER_ACCEPTED",
        human_note=
            "Human Reviewer accepted the expected answer.",
    )
)


cell84_answer_accept_valid = (
    cell84_citation_pending_session.get(
        "current_state"
    )
    ==
    "CITATION_MAPPING_PENDING"
)


if not cell84_answer_accept_valid:
    cell84_failures.append(
        "ANSWER_ACCEPT_TRANSITION_INVALID"
    )


# ============================================================
# 9) Citation entries
# ============================================================

cell84_citation_entries = []


# Original Dr fragment: Primary speaker, but not answer-supporting.
cell84_part0_entry = copy.deepcopy(
    cell84_original_target_parts[
        0
    ]
)

cell84_part0_entry.update({
    "citation_index":
        0,

    "authority_role":
        "PRIMARY_CONTEXT_ONLY",

    "answer_support":
        False,

    "use_in_expected_answer_grounding":
        False,

    "external_locator_lookup_used":
        False,
})

cell84_citation_entries.append(
    cell84_part0_entry
)


# Host statement: context only.
cell84_part1_entry = copy.deepcopy(
    cell84_original_target_parts[
        1
    ]
)

cell84_part1_entry.update({
    "citation_index":
        1,

    "authority_role":
        "NON_PRIMARY_HOST_CONTEXT",

    "answer_support":
        False,

    "use_in_expected_answer_grounding":
        False,

    "human_authority_overlay": {
        "speaker":
            "مجری",

        "effective_primary":
            False,

        "role":
            "CONTEXT_ONLY",

        "secondary_to_primary_promotion":
            "FORBIDDEN",
    },

    "external_locator_lookup_used":
        False,
})

cell84_citation_entries.append(
    cell84_part1_entry
)


# Dr. Roudi's direct response: sole answer evidence.
cell84_support_entry = copy.deepcopy(
    cell84_supporting_evidence
)

cell84_support_entry.update({
    "citation_index":
        2,

    "authority_role":
        "PRIMARY_ANSWER_EVIDENCE",

    "answer_support":
        True,

    "use_in_expected_answer_grounding":
        True,

    "external_locator_lookup_used":
        False,
})

cell84_citation_entries.append(
    cell84_support_entry
)


cell84_primary_answer_entries = [
    entry
    for entry
    in cell84_citation_entries
    if entry.get(
        "answer_support"
    )
    is True
]


cell84_context_only_entries = [
    entry
    for entry
    in cell84_citation_entries
    if entry.get(
        "answer_support"
    )
    is False
]


cell84_citation_boundary_valid = all([
    len(
        cell84_citation_entries
    )
    ==
    3,

    len(
        cell84_primary_answer_entries
    )
    ==
    1,

    len(
        cell84_context_only_entries
    )
    ==
    2,

    cell84_primary_answer_entries[
        0
    ].get(
        "speaker_canonical"
    )
    ==
    "Dr. Komeil Roudi",

    cell84_primary_answer_entries[
        0
    ].get(
        "speaker_primary"
    )
    is True,

    cell84_part1_entry.get(
        "answer_support"
    )
    is False,

    cell84_part1_entry.get(
        "human_authority_overlay",
        {},
    ).get(
        "effective_primary"
    )
    is False,
])


if not cell84_citation_boundary_valid:
    cell84_failures.append(
        "CITATION_AUTHORITY_BOUNDARY_INVALID"
    )


# ============================================================
# 10) Locator integrity
# ============================================================

cell84_support_locator_valid = all([
    cell84_support_entry[
        "locator"
    ][
        "source_line_start"
    ]
    ==
    cell83_primary_response_start,

    cell84_support_entry[
        "locator"
    ][
        "source_line_end"
    ]
    ==
    cell83_primary_response_end,

    cell84_support_text_exact,
])


cell84_no_locator_fabrication = all([
    cell84_support_locator_valid,

    cell84_part0_entry.get(
        "locator"
    )
    ==
    cell84_original_target_parts[
        0
    ].get(
        "locator"
    ),

    cell84_part1_entry.get(
        "locator"
    )
    ==
    cell84_original_target_parts[
        1
    ].get(
        "locator"
    ),
])


if not cell84_no_locator_fabrication:
    cell84_failures.append(
        "LOCATOR_INTEGRITY_FAILED"
    )


# ============================================================
# 11) Citation Mapping
# ============================================================

cell84_citation_identity = {
    "identity_version":
        "citation_mapping_identity_v1_primary_extension",

    "pilot_case_id":
        CELL84_PILOT_ID,

    "question_draft_id":
        cell84_question_draft_id,

    "expected_answer_draft_id":
        cell84_expected_answer_draft_id,

    "target_evidence_id":
        CELL84_TARGET_EVIDENCE_ID,

    "citation_entries":
        copy.deepcopy(
            cell84_citation_entries
        ),
}


cell84_citation_mapping_id = (
    "citation-mapping:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell84_citation_identity
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


cell84_citation_mapping_record = {
    "record_version":
        "citation_mapping_record_v1_primary_extension",

    "citation_mapping_id":
        cell84_citation_mapping_id,

    "pilot_case_id":
        CELL84_PILOT_ID,

    "golden_case_id":
        CELL84_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL84_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL84_TARGET_EVIDENCE_ID,

    "question_draft_id":
        cell84_question_draft_id,

    "expected_answer_draft_id":
        cell84_expected_answer_draft_id,

    "citation_entries":
        copy.deepcopy(
            cell84_citation_entries
        ),

    "supporting_evidence_extension":
        copy.deepcopy(
            cell84_supporting_evidence
        ),

    "primary_answer_authority":
        "Dr. Komeil Roudi",

    "host_context":
        "مجری",

    "host_used_as_independent_answer_evidence":
        False,

    "secondary_to_primary_promotion":
        "FORBIDDEN",

    "retarget_used":
        False,

    "target_salvaged":
        True,

    "raw_source_mutated":
        False,

    "parser_output_mutated":
        False,

    "external_knowledge_used":
        False,

    "human_adjudication_status":
        "PENDING",
}


cell84_current_session = (
    cell13_apply_event(
        cell84_citation_pending_session,
        "CITATION_MAPPING_REGISTERED",
        artifact=
            cell84_citation_mapping_record,
    )
)


cell84_citation_registered_valid = (
    cell84_current_session.get(
        "current_state"
    )
    ==
    "CITATION_HUMAN_REVIEW_PENDING"
)


if not cell84_citation_registered_valid:
    cell84_failures.append(
        "CITATION_REGISTRATION_INVALID"
    )


# ============================================================
# 12) Active workspace
# ============================================================

cell84_active_workspace = copy.deepcopy(
    cell81_persisted_active_case
)


cell84_active_workspace[
    "workflow_state"
] = "CITATION_HUMAN_REVIEW_PENDING"


cell84_active_workspace[
    "target_salvage"
] = {
    "retarget_used":
        False,

    "target_salvaged":
        True,

    "original_target_evidence_id":
        CELL84_TARGET_EVIDENCE_ID,

    "supporting_evidence_extension":
        copy.deepcopy(
            cell84_supporting_evidence
        ),

    "reason":
        (
            "The non-primary host statement is immediately confirmed "
            "and substantively explained by Dr. Komeil Roudi."
        ),
}


cell84_active_workspace[
    "question_review"
] = {
    "question_draft_id":
        cell84_question_draft_id,

    "question_text":
        CELL84_ACCEPTED_QUESTION,

    "human_decision":
        "ACCEPT",
}


cell84_active_workspace[
    "expected_answer_review"
] = {
    "expected_answer_draft_id":
        cell84_expected_answer_draft_id,

    "expected_answer_text":
        CELL84_ACCEPTED_ANSWER,

    "human_decision":
        "ACCEPT",
}


cell84_active_workspace[
    "citation_review"
] = {
    "citation_mapping_id":
        cell84_citation_mapping_id,

    "citation_mapping_record":
        copy.deepcopy(
            cell84_citation_mapping_record
        ),

    "human_decision":
        "PENDING",

    "primary_answer_authority":
        "Dr. Komeil Roudi",

    "host_used_as_independent_answer_evidence":
        False,
}


# ============================================================
# 13) Continuation state
# ============================================================

cell84_continuation_core = {
    "next_pilot_id":
        CELL84_PILOT_ID,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell81_remaining_pilot_ids
        ),

    "remaining_count":
        6,

    "active_pilot_status":
        "CITATION_HUMAN_REVIEW_PENDING",

    "active_target_evidence_id":
        CELL84_TARGET_EVIDENCE_ID,

    "active_citation_mapping_id":
        cell84_citation_mapping_id,

    "target_salvaged":
        True,

    "retarget_used":
        False,

    "authority_policy_active":
        True,

    "last_completed_pilot_id":
        cell81_continuation_state.get(
            "last_completed_pilot_id"
        ),
}


cell84_continuation_state_sha256 = (
    cell10_stable_sha(
        cell84_continuation_core
    )
)


cell84_continuation_state = {
    **copy.deepcopy(
        cell84_continuation_core
    ),

    "continuation_state_sha256":
        cell84_continuation_state_sha256,
}


# ============================================================
# 14) Persist
# ============================================================

cell84_persistent_registry_payload = copy.deepcopy(
    cell81_persisted_registry
)


cell84_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell84_active_workspace
)


cell84_persistent_registry_payload[
    "remaining_pilot_count"
] = 6


cell84_persistent_registry_payload[
    "continuation_state"
] = copy.deepcopy(
    cell84_continuation_state
)


cell84_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell84_persistent_registry_payload
    )
)


cell84_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell84_persisted_active_case = (
    cell84_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell84_persistence_valid = all([
    cell84_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL84_PILOT_ID,

    cell84_persisted_active_case.get(
        "workflow_state"
    )
    ==
    "CITATION_HUMAN_REVIEW_PENDING",

    cell84_persisted_active_case.get(
        "question_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell84_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell84_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "PENDING",

    cell84_persisted_active_case.get(
        "target_salvage",
        {},
    ).get(
        "target_salvaged"
    )
    is True,

    cell84_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    6,

    cell84_persisted_registry.get(
        "continuation_state",
        {},
    ).get(
        "continuation_state_sha256"
    )
    ==
    cell84_continuation_state_sha256,
])


if not cell84_persistence_valid:
    cell84_failures.append(
        "REGISTRY_READBACK_INVALID"
    )


# ============================================================
# 15) Safety
# ============================================================

cell84_queue_still_6 = all([
    len(
        cell81_remaining_pilot_ids
    )
    ==
    6,

    cell81_remaining_pilot_ids[
        0
    ]
    ==
    CELL84_PILOT_ID,
])


cell84_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell84_full_572_blocked = (
    cell81_full_572_blocked
    is True
)


# ============================================================
# 16) Gate
# ============================================================

cell84_gate_checks = {
    "no_failures":
        len(
            cell84_failures
        )
        ==
        0,

    "original_target_valid":
        cell84_original_target_valid,

    "supporting_evidence_valid":
        cell84_supporting_evidence_valid,

    "grounding_pass":
        cell84_grounding_pass,

    "session_start_valid":
        cell84_session_start_valid,

    "question_accept_valid":
        cell84_question_accept_valid,

    "answer_accept_valid":
        cell84_answer_accept_valid,

    "citation_boundary_valid":
        cell84_citation_boundary_valid,

    "no_locator_fabrication":
        cell84_no_locator_fabrication,

    "citation_registered_valid":
        cell84_citation_registered_valid,

    "persistence_valid":
        cell84_persistence_valid,

    "queue_still_6":
        cell84_queue_still_6,

    "base_seed_unchanged":
        cell84_base_seed_unchanged,

    "full_572_blocked":
        cell84_full_572_blocked,
}


cell84_gate = all(
    cell84_gate_checks.values()
)


# ============================================================
# 17) Diagnostics
# ============================================================

print(
    "=== PILOT #18 QA ACCEPT + PRIMARY EVIDENCE EXTENSION + CITATION DRAFT ==="
)


print(
    "\n=== TARGET SALVAGE ==="
)

print(
    "Original Target:",
    CELL84_TARGET_EVIDENCE_ID,
)

print(
    "Retarget used:",
    False,
)

print(
    "Target salvaged:",
    True,
)

print(
    "Primary support lines:",
    (
        cell84_primary_support_start,
        cell84_primary_support_end,
    ),
)


print(
    "\n=== AUTHORITY ==="
)

print(
    "Host:",
    "مجری",
)

print(
    "Host independent Answer Evidence:",
    False,
)

print(
    "Primary Answer Authority:",
    "Dr. Komeil Roudi",
)

print(
    "Dr. explicit confirmation:",
    cell83_explicit_confirmation,
)

print(
    "Secondary-to-primary promotion:",
    "FORBIDDEN",
)

print(
    "Raw Source mutated:",
    False,
)

print(
    "Parser mutated:",
    False,
)


print(
    "\n=== ACCEPTED QUESTION ==="
)

print(
    CELL84_ACCEPTED_QUESTION
)

print(
    "Question Draft ID:",
    cell84_question_draft_id,
)

print(
    "Human Decision:",
    "ACCEPT",
)


print(
    "\n=== ACCEPTED ANSWER ==="
)

print(
    CELL84_ACCEPTED_ANSWER
)

print(
    "Expected Answer Draft ID:",
    cell84_expected_answer_draft_id,
)

print(
    "Human Decision:",
    "ACCEPT",
)


print(
    "\n=== GROUNDING ==="
)

for name, status in cell84_grounding.items():

    print(
        f"{name}:",
        status,
    )


print(
    "Grounding PASS:",
    cell84_grounding_pass,
)


print(
    "\n=== CITATION DRAFT ==="
)

print(
    "Citation Mapping ID:",
    cell84_citation_mapping_id,
)

print(
    "Citation entries:",
    len(
        cell84_citation_entries
    ),
)

print(
    "Primary Answer Evidence entries:",
    len(
        cell84_primary_answer_entries
    ),
)

print(
    "Context-only entries:",
    len(
        cell84_context_only_entries
    ),
)

print(
    "Host used as Answer Evidence:",
    False,
)

print(
    "Citation boundary valid:",
    cell84_citation_boundary_valid,
)

print(
    "No locator fabrication:",
    cell84_no_locator_fabrication,
)


print(
    "\n=== WORKFLOW ==="
)

print(
    "Current State:",
    cell84_current_session.get(
        "current_state"
    ),
)

print(
    "Question:",
    "ACCEPT",
)

print(
    "Expected Answer:",
    "ACCEPT",
)

print(
    "Citation:",
    "PENDING",
)


print(
    "\n=== ALL INTERNAL GATES ==="
)

for name, status in cell84_gate_checks.items():

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== QUEUE / SAFETY ==="
)

print(
    "Queue still 6:",
    cell84_queue_still_6,
)

print(
    "Persistence valid:",
    cell84_persistence_valid,
)

print(
    "Base Seed unchanged:",
    cell84_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell84_full_572_blocked,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell84_failures
)


print(
    "\nCODE CELL 84 GATE:",
    (
        "PASS"
        if cell84_gate
        else "FAIL"
    ),
)


assert cell84_gate, (
    "Pilot #18 Question+Answer ACCEPT / primary evidence extension / "
    "Citation Draft failed. Repair this SAME Code Cell 84."
)

=== PILOT #18 QA ACCEPT + PRIMARY EVIDENCE EXTENSION + CITATION DRAFT ===

=== TARGET SALVAGE ===
Original Target: target-evidence:3a0d4cc342f0c61728501ee45568ed35
Retarget used: False
Target salvaged: True
Primary support lines: (17, 19)

=== AUTHORITY ===
Host: مجری
Host independent Answer Evidence: False
Primary Answer Authority: Dr. Komeil Roudi
Dr. explicit confirmation: True
Secondary-to-primary promotion: FORBIDDEN
Raw Source mutated: False
Parser mutated: False

=== ACCEPTED QUESTION ===
بودجه‌بندی از چه زمانی شروع می‌شود و چه تفاوتی با ثبت هزینه‌های ماه‌های قبل دارد؟
Question Draft ID: question-draft:bfbe6d85ba57c72a74d93ad3c85efe5d
Human Decision: ACCEPT

=== ACCEPTED ANSWER ===
در سه ماه اول، هزینه‌ها ثبت و در پایان هر ماه دسته‌بندی می‌شوند تا الگوی خرج‌کردن مشخص شود. از ابتدای ماه چهارم، پیش از انجام هزینه‌ها تعیین می‌کنیم که در هر دسته چه مقدار می‌خواهیم خرج کنیم؛ این برنامه‌ریزی از قبل همان بودجه‌بندی است.
Expected Answer Draft ID: expected-answer-draft:bf9ce630820c560d85

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 85 — Citation ACCEPT + Pilot #18 Completion + Queue Advance

Citation نهایی Pilot #18 توسط Human Reviewer پذیرفته شد.

در این مرحله:

- Citation با تصمیم `ACCEPT` ثبت می‌شود؛
- Target اصلی بدون Retarget حفظ می‌شود؛
- `Human-certified Supporting Evidence Extension` مربوط به پاسخ Dr. Komeil Roudi حفظ می‌شود؛
- مجری فقط Context باقی می‌ماند و Answer Authority نیست؛
- Pilot #18 کامل می‌شود؛
- تعداد Caseهای کامل Human Review از 16 به 17 می‌رسد؛
- تعداد Caseهای آماده Author Review از 16 به 17 می‌رسد؛
- Queue دقیقاً از 6 به 5 جلو می‌رود؛
- Pilot بعدی برای Human Review آماده می‌شود؛
- Author Final Review همچنان `NOT_STARTED` است؛
- Full 572 Generation همچنان Blocked باقی می‌ماند.

</div>

In [85]:
# Code Cell 85

cell85_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell84_gate") is not True:
    cell85_failures.append(
        "CELL84_NOT_PASS"
    )


if (
    globals().get(
        "cell84_current_session",
        {},
    ).get(
        "current_state"
    )
    !=
    "CITATION_HUMAN_REVIEW_PENDING"
):
    cell85_failures.append(
        "CITATION_NOT_AT_HUMAN_REVIEW_GATE"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell85_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Human Citation ACCEPT
# ============================================================

CELL85_CITATION_HUMAN_NOTE = (
    "Human Reviewer accepted the final Citation Mapping. "
    "The host remains context-only and is not independent answer authority. "
    "Dr. Komeil Roudi's verified response remains the sole primary "
    "answer evidence through the supporting evidence extension."
)


cell85_complete_pending_session = (
    cell13_apply_event(
        cell84_current_session,
        "CITATION_ACCEPTED",
        human_note=
            CELL85_CITATION_HUMAN_NOTE,
    )
)


cell85_citation_accept_valid = (
    cell85_complete_pending_session.get(
        "current_state"
    )
    ==
    "PILOT_COMPLETE_PENDING_QUEUE_ADVANCE"
)


if not cell85_citation_accept_valid:
    cell85_failures.append(
        "CITATION_ACCEPT_TRANSITION_INVALID"
    )


# ============================================================
# 3) Final Citation Mapping
# ============================================================

cell85_final_citation_mapping_record = copy.deepcopy(
    cell84_citation_mapping_record
)


cell85_final_citation_mapping_record[
    "human_adjudication_status"
] = "ACCEPT"


cell85_final_citation_mapping_record[
    "human_review"
] = {
    "human_decision":
        "ACCEPT",

    "human_note":
        CELL85_CITATION_HUMAN_NOTE,

    "primary_answer_authority":
        "Dr. Komeil Roudi",

    "host_context":
        "مجری",

    "host_used_as_independent_answer_evidence":
        False,

    "target_salvaged":
        True,

    "retarget_used":
        False,

    "supporting_evidence_extension_used":
        True,

    "raw_source_mutated":
        False,

    "parser_output_mutated":
        False,

    "external_knowledge_used":
        False,
}


# ============================================================
# 4) Complete Human Review case
# ============================================================

cell85_completed_case = copy.deepcopy(
    cell84_active_workspace
)


cell85_completed_case[
    "workflow_state"
] = "PILOT_COMPLETE"


cell85_completed_case[
    "status"
] = "COMPLETED_HUMAN_REVIEW"


cell85_completed_case[
    "final_status"
] = "PILOT_COMPLETE"


cell85_completed_case.setdefault(
    "citation_review",
    {},
)


cell85_completed_case[
    "citation_review"
][
    "human_decision"
] = "ACCEPT"


cell85_completed_case[
    "citation_review"
][
    "human_note"
] = CELL85_CITATION_HUMAN_NOTE


cell85_completed_case[
    "citation_review"
][
    "citation_mapping_record"
] = copy.deepcopy(
    cell85_final_citation_mapping_record
)


cell85_completed_case[
    "completion_summary"
] = {
    "retarget_used":
        False,

    "target_salvaged":
        True,

    "target_evidence_id":
        CELL84_TARGET_EVIDENCE_ID,

    "supporting_evidence_extension_used":
        True,

    "supporting_evidence_source_lines":
        (
            cell84_primary_support_start,
            cell84_primary_support_end,
        ),

    "question_decision":
        "ACCEPT",

    "expected_answer_decision":
        "ACCEPT",

    "citation_decision":
        "ACCEPT",

    "primary_answer_authority":
        "Dr. Komeil Roudi",

    "host_context":
        "مجری",

    "host_used_as_independent_answer_evidence":
        False,

    "secondary_to_primary_promotion":
        "FORBIDDEN",

    "raw_source_mutated":
        False,

    "parser_output_mutated":
        False,

    "author_final_review":
        "NOT_STARTED",
}


cell85_completed_case[
    "author_final_review"
] = {
    "status":
        "NOT_STARTED",

    "decision":
        None,

    "notes":
        None,
}


# ============================================================
# 5) Queue advance: 6 -> 5
# ============================================================

cell85_previous_remaining_pilot_ids = copy.deepcopy(
    cell81_remaining_pilot_ids
)


CELL85_COMPLETED_PILOT_ID = (
    cell85_previous_remaining_pilot_ids[
        0
    ]
)


cell85_remaining_pilot_ids = copy.deepcopy(
    cell85_previous_remaining_pilot_ids[
        1:
    ]
)


cell85_queue_advance_valid = all([
    len(
        cell85_previous_remaining_pilot_ids
    )
    ==
    6,

    CELL85_COMPLETED_PILOT_ID
    ==
    CELL84_PILOT_ID,

    len(
        cell85_remaining_pilot_ids
    )
    ==
    5,

    CELL85_COMPLETED_PILOT_ID
    not in
    cell85_remaining_pilot_ids,
])


if not cell85_queue_advance_valid:
    cell85_failures.append(
        "QUEUE_ADVANCE_INVALID"
    )


CELL85_NEXT_PILOT_ID = (
    cell85_remaining_pilot_ids[
        0
    ]
)


# ============================================================
# 6) Find next Pilot packet
# ============================================================

cell85_raw_next_packet_candidates = (
    cell58_find_pilot_packet_candidates(
        cell13_seed,
        CELL85_NEXT_PILOT_ID,
    )
)


cell85_packet_candidates_by_sha = {}


for candidate in cell85_raw_next_packet_candidates:

    packet_sha = cell10_stable_sha(
        candidate[
            "packet"
        ]
    )


    previous = (
        cell85_packet_candidates_by_sha.get(
            packet_sha
        )
    )


    if (
        previous is None
        or
        candidate[
            "score"
        ]
        >
        previous[
            "score"
        ]
    ):

        cell85_packet_candidates_by_sha[
            packet_sha
        ] = candidate


cell85_ranked_next_packet_candidates = sorted(
    cell85_packet_candidates_by_sha.values(),
    key=lambda item: (
        -item[
            "score"
        ],
        item[
            "path"
        ],
    ),
)


cell85_next_pilot_packet = (
    copy.deepcopy(
        cell85_ranked_next_packet_candidates[
            0
        ][
            "packet"
        ]
    )
    if cell85_ranked_next_packet_candidates
    else None
)


if not isinstance(
    cell85_next_pilot_packet,
    dict,
):

    cell85_failures.append(
        "NEXT_PILOT_PACKET_NOT_FOUND"
    )


# ============================================================
# 7) Extract next Pilot identities
# ============================================================

cell85_next_pilot_ids_found = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell85_next_pilot_packet,
            "question-pilot:",
        )
    )
)


cell85_next_golden_case_ids = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell85_next_pilot_packet,
            "golden-case:",
        )
    )
)


cell85_next_task_draft_ids = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell85_next_pilot_packet,
            "task-draft:",
        )
    )
)


cell85_next_target_evidence_ids = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell85_next_pilot_packet,
            "target-evidence:",
        )
    )
)


cell85_next_identity_valid = all([
    cell85_next_pilot_ids_found
    ==
    [
        CELL85_NEXT_PILOT_ID
    ],

    len(
        cell85_next_golden_case_ids
    )
    ==
    1,

    len(
        cell85_next_task_draft_ids
    )
    ==
    1,

    len(
        cell85_next_target_evidence_ids
    )
    ==
    1,
])


if not cell85_next_identity_valid:
    cell85_failures.append(
        "NEXT_PILOT_IDENTITY_INVALID"
    )


CELL85_NEXT_GOLDEN_CASE_ID = (
    cell85_next_golden_case_ids[
        0
    ]
    if
    len(
        cell85_next_golden_case_ids
    )
    ==
    1
    else
    None
)


CELL85_NEXT_TASK_DRAFT_ID = (
    cell85_next_task_draft_ids[
        0
    ]
    if
    len(
        cell85_next_task_draft_ids
    )
    ==
    1
    else
    None
)


CELL85_NEXT_TARGET_EVIDENCE_ID = (
    cell85_next_target_evidence_ids[
        0
    ]
    if
    len(
        cell85_next_target_evidence_ids
    )
    ==
    1
    else
    None
)


# ============================================================
# 8) Extract next Target Evidence
# ============================================================

cell85_target_part_candidates = (
    cell58_find_key_values(
        cell85_next_pilot_packet,
        "target_text_parts",
    )
)


cell85_target_parts_by_sha = {}


for parts in cell85_target_part_candidates:

    if not isinstance(
        parts,
        list,
    ):
        continue


    if len(
        parts
    ) == 0:
        continue


    parts_sha = cell10_stable_sha(
        parts
    )


    cell85_target_parts_by_sha[
        parts_sha
    ] = parts


cell85_unique_target_part_sets = list(
    cell85_target_parts_by_sha.values()
)


cell85_next_target_text_parts = (
    copy.deepcopy(
        cell85_unique_target_part_sets[
            0
        ]
    )
    if
    len(
        cell85_unique_target_part_sets
    )
    ==
    1
    else
    None
)


if not isinstance(
    cell85_next_target_text_parts,
    list,
):

    cell85_failures.append(
        "NEXT_TARGET_TEXT_PARTS_NOT_UNIQUE"
    )


# ============================================================
# 9) Build readable next Evidence
# ============================================================

cell85_next_evidence_texts = []


for part in (
    cell85_next_target_text_parts
    or
    []
):

    if isinstance(
        part,
        str,
    ):

        text = part.strip()


    elif isinstance(
        part,
        dict,
    ):

        text = (
            part.get(
                "text"
            )
            or
            part.get(
                "target_text"
            )
            or
            part.get(
                "content"
            )
            or
            ""
        )


        if isinstance(
            text,
            str,
        ):

            text = text.strip()


        else:

            text = ""


    else:

        text = ""


    if text:

        cell85_next_evidence_texts.append(
            text
        )


cell85_next_combined_target_text = (
    "\n\n".join(
        cell85_next_evidence_texts
    )
)


cell85_next_evidence_available = bool(
    cell85_next_combined_target_text.strip()
)


if not cell85_next_evidence_available:
    cell85_failures.append(
        "NEXT_EVIDENCE_EMPTY"
    )


# ============================================================
# 10) Extract next Pilot metadata
# ============================================================

def cell85_unique_scalar_for_key(
    payload,
    key,
):

    values = (
        cell58_find_key_values(
            payload,
            key,
        )
    )


    normalized = []


    for value in values:

        if isinstance(
            value,
            (
                str,
                int,
                float,
                bool,
            ),
        ):

            if value not in normalized:
                normalized.append(
                    value
                )


    if len(
        normalized
    ) == 1:

        return normalized[
            0
        ]


    return None


CELL85_NEXT_STRATUM = (
    cell85_unique_scalar_for_key(
        cell85_next_pilot_packet,
        "stratum",
    )
)


CELL85_NEXT_TASK_FAMILY = (
    cell85_unique_scalar_for_key(
        cell85_next_pilot_packet,
        "task_family",
    )
)


CELL85_NEXT_ARCHETYPE = (
    cell85_unique_scalar_for_key(
        cell85_next_pilot_packet,
        "question_archetype",
    )
)


CELL85_NEXT_AUTHORITY_EXPECTATION = (
    cell85_unique_scalar_for_key(
        cell85_next_pilot_packet,
        "authority_expectation",
    )
)


# ============================================================
# 11) Next Pilot Engine session
# ============================================================

cell85_next_pilot_session = (
    cell13_build_session(
        cell85_next_pilot_packet
    )
)


cell85_next_session_valid = (
    cell85_next_pilot_session.get(
        "current_state"
    )
    ==
    "QUESTION_DRAFT_PENDING"
)


if not cell85_next_session_valid:
    cell85_failures.append(
        "NEXT_SESSION_INVALID"
    )


# ============================================================
# 12) Update completed Human Review cases
# ============================================================

cell85_existing_completed_cases = copy.deepcopy(
    cell84_persisted_registry.get(
        "completed_human_review_cases",
        [],
    )
)


if not isinstance(
    cell85_existing_completed_cases,
    list,
):

    cell85_existing_completed_cases = []

    cell85_failures.append(
        "COMPLETED_CASES_NOT_LIST"
    )


cell85_existing_completed_cases = [
    case
    for case
    in cell85_existing_completed_cases
    if not (
        isinstance(
            case,
            dict,
        )
        and
        case.get(
            "pilot_case_id"
        )
        ==
        CELL85_COMPLETED_PILOT_ID
    )
]


cell85_completed_human_review_cases = (
    cell85_existing_completed_cases
    +
    [
        copy.deepcopy(
            cell85_completed_case
        )
    ]
)


cell85_completed_case_count_valid = (
    len(
        cell85_completed_human_review_cases
    )
    ==
    17
)


if not cell85_completed_case_count_valid:
    cell85_failures.append(
        "COMPLETED_CASE_COUNT_INVALID"
    )


# ============================================================
# 13) Rebuild Author Review queue
# ============================================================

cell85_author_review = copy.deepcopy(
    cell84_persisted_registry.get(
        "author_review",
        {},
    )
)


if not isinstance(
    cell85_author_review,
    dict,
):

    cell85_author_review = {}


cell85_author_ready_cases = copy.deepcopy(
    cell85_completed_human_review_cases
)


cell85_author_review[
    "cases_ready_for_review"
] = copy.deepcopy(
    cell85_author_ready_cases
)


cell85_author_review[
    "case_count"
] = len(
    cell85_author_ready_cases
)


cell85_author_review[
    "final_review_status"
] = "NOT_STARTED"


cell85_author_ready_count_valid = (
    len(
        cell85_author_ready_cases
    )
    ==
    17
)


cell85_author_ready_identity_valid = (
    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell85_author_ready_cases
        if isinstance(
            case,
            dict,
        )
    }
    ==
    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell85_completed_human_review_cases
        if isinstance(
            case,
            dict,
        )
    }
)


if not cell85_author_ready_count_valid:
    cell85_failures.append(
        "AUTHOR_READY_COUNT_INVALID"
    )


if not cell85_author_ready_identity_valid:
    cell85_failures.append(
        "AUTHOR_READY_IDENTITY_INVALID"
    )


# ============================================================
# 14) Preserve Global Authority Policy
# ============================================================

cell85_authority_policy = copy.deepcopy(
    cell84_persisted_registry.get(
        "workflow_policy",
        {},
    ).get(
        "authority_evidence_policy",
        {},
    )
)


cell85_authority_policy_valid = all([
    cell85_authority_policy.get(
        "primary_authority"
    )
    ==
    "Dr. Komeil Roudi",

    cell85_authority_policy.get(
        "secondary_to_primary_promotion"
    )
    ==
    "FORBIDDEN",
])


if not cell85_authority_policy_valid:
    cell85_failures.append(
        "GLOBAL_AUTHORITY_POLICY_INVALID"
    )


# ============================================================
# 15) Build next active workspace
# ============================================================

cell85_next_active_workspace = {
    "pilot_case_id":
        CELL85_NEXT_PILOT_ID,

    "golden_case_id":
        CELL85_NEXT_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL85_NEXT_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL85_NEXT_TARGET_EVIDENCE_ID,

    "status":
        "ACTIVE_HUMAN_REVIEW",

    "workflow_state":
        "QUESTION_DRAFT_PENDING",

    "stratum":
        CELL85_NEXT_STRATUM,

    "task_family":
        CELL85_NEXT_TASK_FAMILY,

    "question_archetype":
        CELL85_NEXT_ARCHETYPE,

    "authority_expectation":
        CELL85_NEXT_AUTHORITY_EXPECTATION,

    "authority_policy":
        copy.deepcopy(
            cell85_authority_policy
        ),

    "source_evidence":
        copy.deepcopy(
            cell85_next_target_text_parts
        ),

    "combined_target_text":
        cell85_next_combined_target_text,

    "question_review":
        None,

    "expected_answer_review":
        None,

    "citation_review":
        None,

    "author_final_review": {
        "status":
            "NOT_STARTED",

        "decision":
            None,

        "notes":
            None,
    },
}


# ============================================================
# 16) Continuation state
# ============================================================

cell85_continuation_state_core = {
    "next_pilot_id":
        CELL85_NEXT_PILOT_ID,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell85_remaining_pilot_ids
        ),

    "remaining_count":
        5,

    "active_pilot_status":
        "QUESTION_DRAFT_PENDING",

    "active_target_evidence_id":
        CELL85_NEXT_TARGET_EVIDENCE_ID,

    "last_completed_pilot_id":
        CELL85_COMPLETED_PILOT_ID,

    "last_completion_type":
        "SALVAGED_PRIMARY_NONPRIMARY_BOUNDARY_PILOT_COMPLETE",

    "authority_policy_active":
        True,
}


cell85_continuation_state_sha256 = (
    cell10_stable_sha(
        cell85_continuation_state_core
    )
)


cell85_continuation_state = {
    **copy.deepcopy(
        cell85_continuation_state_core
    ),

    "continuation_state_sha256":
        cell85_continuation_state_sha256,
}


# ============================================================
# 17) Persist Registry
# ============================================================

cell85_persistent_registry_payload = copy.deepcopy(
    cell84_persisted_registry
)


cell85_persistent_registry_payload[
    "completed_human_review_cases"
] = copy.deepcopy(
    cell85_completed_human_review_cases
)


cell85_persistent_registry_payload[
    "author_review"
] = copy.deepcopy(
    cell85_author_review
)


cell85_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell85_next_active_workspace
)


cell85_persistent_registry_payload[
    "remaining_pilot_count"
] = 5


cell85_persistent_registry_payload[
    "continuation_state"
] = copy.deepcopy(
    cell85_continuation_state
)


cell85_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell85_persistent_registry_payload
    )
)


# ============================================================
# 18) Read-back validation
# ============================================================

cell85_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell85_persisted_active_case = (
    cell85_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell85_persisted_completed_cases = (
    cell85_persisted_registry.get(
        "completed_human_review_cases",
        [],
    )
)


cell85_persisted_author_ready_cases = (
    cell85_persisted_registry.get(
        "author_review",
        {},
    ).get(
        "cases_ready_for_review",
        [],
    )
)


cell85_persisted_authority_policy = (
    cell85_persisted_registry.get(
        "workflow_policy",
        {},
    ).get(
        "authority_evidence_policy",
        {},
    )
)


cell85_persistence_valid = all([
    isinstance(
        cell85_persisted_completed_cases,
        list,
    ),

    len(
        cell85_persisted_completed_cases
    )
    ==
    17,

    isinstance(
        cell85_persisted_author_ready_cases,
        list,
    ),

    len(
        cell85_persisted_author_ready_cases
    )
    ==
    17,

    cell85_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    5,

    cell85_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL85_NEXT_PILOT_ID,

    cell85_persisted_active_case.get(
        "workflow_state"
    )
    ==
    "QUESTION_DRAFT_PENDING",

    cell85_persisted_registry.get(
        "continuation_state",
        {},
    ).get(
        "continuation_state_sha256"
    )
    ==
    cell85_continuation_state_sha256,

    cell85_persisted_authority_policy.get(
        "primary_authority"
    )
    ==
    "Dr. Komeil Roudi",

    cell85_persisted_authority_policy.get(
        "secondary_to_primary_promotion"
    )
    ==
    "FORBIDDEN",

    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell85_persisted_completed_cases
        if isinstance(
            case,
            dict,
        )
    }
    ==
    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell85_persisted_author_ready_cases
        if isinstance(
            case,
            dict,
        )
    },
])


if not cell85_persistence_valid:
    cell85_failures.append(
        "REGISTRY_READBACK_INVALID"
    )


# ============================================================
# 19) Safety
# ============================================================

cell85_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell85_full_572_blocked = (
    cell84_full_572_blocked
    is True
)


cell85_no_commit_push = True


# ============================================================
# 20) Internal Gates
# ============================================================

cell85_gate_checks = {
    "no_failures":
        len(
            cell85_failures
        )
        ==
        0,

    "citation_accept_valid":
        cell85_citation_accept_valid,

    "queue_advance_valid":
        cell85_queue_advance_valid,

    "next_identity_valid":
        cell85_next_identity_valid,

    "next_evidence_available":
        cell85_next_evidence_available,

    "next_session_valid":
        cell85_next_session_valid,

    "completed_case_count_valid":
        cell85_completed_case_count_valid,

    "author_ready_count_valid":
        cell85_author_ready_count_valid,

    "author_ready_identity_valid":
        cell85_author_ready_identity_valid,

    "authority_policy_valid":
        cell85_authority_policy_valid,

    "persistence_valid":
        cell85_persistence_valid,

    "base_seed_unchanged":
        cell85_base_seed_unchanged,

    "full_572_blocked":
        cell85_full_572_blocked,

    "no_commit_push":
        cell85_no_commit_push,
}


cell85_gate = all(
    cell85_gate_checks.values()
)


# ============================================================
# 21) Diagnostics
# ============================================================

print(
    "=== PILOT #18 COMPLETE + QUEUE ADVANCE + NEXT PILOT ==="
)


print(
    "\n=== COMPLETED PILOT ==="
)

print(
    "Pilot:",
    CELL85_COMPLETED_PILOT_ID,
)

print(
    "Target:",
    CELL84_TARGET_EVIDENCE_ID,
)

print(
    "Retarget used:",
    False,
)

print(
    "Target salvaged:",
    True,
)

print(
    "Question:",
    "ACCEPT",
)

print(
    "Expected Answer:",
    "ACCEPT",
)

print(
    "Citation:",
    "ACCEPT",
)

print(
    "Final Citation Mapping:",
    cell84_citation_mapping_id,
)

print(
    "Final Engine State:",
    cell85_complete_pending_session.get(
        "current_state"
    ),
)


print(
    "\n=== AUTHORITY RESULT ==="
)

print(
    "Primary Authority:",
    "Dr. Komeil Roudi",
)

print(
    "Host Context:",
    "مجری",
)

print(
    "Host used as independent Answer Evidence:",
    False,
)

print(
    "Supporting Evidence Extension:",
    True,
)

print(
    "Supporting lines:",
    (
        cell84_primary_support_start,
        cell84_primary_support_end,
    ),
)

print(
    "Secondary-to-primary promotion:",
    "FORBIDDEN",
)

print(
    "Raw Source mutated:",
    False,
)

print(
    "Parser mutated:",
    False,
)


print(
    "\n=== QUEUE ADVANCE ==="
)

print(
    "Before:",
    len(
        cell85_previous_remaining_pilot_ids
    ),
)

print(
    "After:",
    len(
        cell85_remaining_pilot_ids
    ),
)

print(
    "Queue advanced exactly once:",
    cell85_queue_advance_valid,
)


print(
    "\n=== HUMAN REVIEW REGISTRY ==="
)

print(
    "Completed Human Review cases:",
    len(
        cell85_completed_human_review_cases
    ),
)

print(
    "Cases ready for Author Review:",
    len(
        cell85_author_ready_cases
    ),
)

print(
    "Author Final Review:",
    "NOT_STARTED",
)

print(
    "Persistence valid:",
    cell85_persistence_valid,
)


print(
    "\n=== NEXT PILOT ==="
)

print(
    "Pilot:",
    CELL85_NEXT_PILOT_ID,
)

print(
    "Golden Case:",
    CELL85_NEXT_GOLDEN_CASE_ID,
)

print(
    "Task Draft:",
    CELL85_NEXT_TASK_DRAFT_ID,
)

print(
    "Target Evidence:",
    CELL85_NEXT_TARGET_EVIDENCE_ID,
)

print(
    "Stratum:",
    CELL85_NEXT_STRATUM,
)

print(
    "Task Family:",
    CELL85_NEXT_TASK_FAMILY,
)

print(
    "Question Archetype:",
    CELL85_NEXT_ARCHETYPE,
)

print(
    "Authority Expectation:",
    CELL85_NEXT_AUTHORITY_EXPECTATION,
)

print(
    "Workflow:",
    cell85_next_pilot_session.get(
        "current_state"
    ),
)


print(
    "\n=== NEXT SOURCE EVIDENCE ==="
)

print(
    cell85_next_combined_target_text
)


print(
    "\n=== CONTINUATION STATE ==="
)

print(
    "Remaining Pilots:",
    5,
)

print(
    "Continuation SHA256:",
    cell85_continuation_state_sha256,
)


print(
    "\n=== ALL INTERNAL GATES ==="
)

for name, status in cell85_gate_checks.items():

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== SAFETY ==="
)

print(
    "Base Seed unchanged:",
    cell85_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell85_full_572_blocked,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell85_failures
)


print(
    "\nCODE CELL 85 GATE:",
    (
        "PASS"
        if cell85_gate
        else "FAIL"
    ),
)


assert cell85_gate, (
    "Pilot #18 Citation ACCEPT / completion / queue advance / "
    "next Pilot preparation failed. Repair this SAME Code Cell 85."
)

=== PILOT #18 COMPLETE + QUEUE ADVANCE + NEXT PILOT ===

=== COMPLETED PILOT ===
Pilot: question-pilot:980720fef1985610dd288cab04b7041c
Target: target-evidence:3a0d4cc342f0c61728501ee45568ed35
Retarget used: False
Target salvaged: True
Question: ACCEPT
Expected Answer: ACCEPT
Citation: ACCEPT
Final Citation Mapping: citation-mapping:45fd63fc35196e63409271889644ffe9
Final Engine State: PILOT_COMPLETE_PENDING_QUEUE_ADVANCE

=== AUTHORITY RESULT ===
Primary Authority: Dr. Komeil Roudi
Host Context: مجری
Host used as independent Answer Evidence: False
Supporting Evidence Extension: True
Supporting lines: (17, 19)
Secondary-to-primary promotion: FORBIDDEN
Raw Source mutated: False
Parser mutated: False

=== QUEUE ADVANCE ===
Before: 6
After: 5
Queue advanced exactly once: True

=== HUMAN REVIEW REGISTRY ===
Completed Human Review cases: 17
Cases ready for Author Review: 17
Author Final Review: NOT_STARTED
Persistence valid: True

=== NEXT PILOT ===
Pilot: question-pilot:6ba9960127f4b21ec2dc

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 86 — Pilot #19 Mixed Authority Diagnostic

Pilot فعلی از نوع:

`MIXED_AUTHORITY_CONTEXT_RETRIEVAL`

است و Authority Rule آن:

`NO_FILE_LEVEL_AUTHORITY_FLATTENING`

می‌باشد.

بنابراین پیش از ساخت سؤال باید بررسی شود:

- Target چند Part دارد؛
- هر Part متعلق به چه Speakerای است؛
- آیا Dr. Komeil Roudi در Target حضور دارد؛
- آیا Speaker غیرPrimary هم وجود دارد؛
- کدام بخش می‌تواند Answer Evidence باشد؛
- آیا بخشی unresolved است؛
- آیا Target فعلی بدون Retarget قابل استفاده است.

این Cell فقط Diagnostic است و هیچ داده‌ای را تغییر نمی‌دهد.

</div>

In [86]:
# Code Cell 86

cell86_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell85_gate") is not True:
    cell86_failures.append(
        "CELL85_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell85_next_target_text_parts"
    ),
    list,
):
    cell86_failures.append(
        "NEXT_TARGET_PARTS_MISSING"
    )


# ============================================================
# 2) Pilot identity
# ============================================================

CELL86_PILOT_ID = (
    CELL85_NEXT_PILOT_ID
)


CELL86_TARGET_EVIDENCE_ID = (
    CELL85_NEXT_TARGET_EVIDENCE_ID
)


CELL86_TASK_FAMILY = (
    CELL85_NEXT_TASK_FAMILY
)


CELL86_AUTHORITY_EXPECTATION = (
    CELL85_NEXT_AUTHORITY_EXPECTATION
)


# ============================================================
# 3) Inspect Target Parts
# ============================================================

cell86_part_diagnostics = []


for index, part in enumerate(
    cell85_next_target_text_parts
):

    if isinstance(
        part,
        dict,
    ):

        speaker_state = (
            part.get(
                "speaker_state"
            )
            or
            {}
        )


        attribution = (
            part.get(
                "attribution"
            )
            or
            {}
        )


        locator = (
            part.get(
                "locator"
            )
            or
            {}
        )


        text = (
            part.get(
                "text"
            )
            or
            part.get(
                "target_text"
            )
            or
            part.get(
                "content"
            )
            or
            ""
        )


        cell86_part_diagnostics.append({
            "index":
                index,

            "document_id":
                part.get(
                    "document_id"
                ),

            "role":
                part.get(
                    "role"
                ),

            "speaker_base_status":
                speaker_state.get(
                    "base_status"
                ),

            "speaker_base_raw":
                speaker_state.get(
                    "base_raw"
                ),

            "speaker_effective_label":
                speaker_state.get(
                    "effective_label"
                ),

            "speaker_effective_canonical":
                speaker_state.get(
                    "effective_canonical"
                ),

            "speaker_effective_primary":
                speaker_state.get(
                    "effective_primary"
                ),

            "attribution_speaker":
                attribution.get(
                    "speaker"
                ),

            "content_authority_type":
                attribution.get(
                    "content_authority_type"
                ),

            "source_line_start":
                locator.get(
                    "source_line_start"
                ),

            "source_line_end":
                locator.get(
                    "source_line_end"
                ),

            "text":
                text,
        })


    else:

        cell86_part_diagnostics.append({
            "index":
                index,

            "document_id":
                None,

            "role":
                None,

            "speaker_base_status":
                None,

            "speaker_base_raw":
                None,

            "speaker_effective_label":
                None,

            "speaker_effective_canonical":
                None,

            "speaker_effective_primary":
                None,

            "attribution_speaker":
                None,

            "content_authority_type":
                None,

            "source_line_start":
                None,

            "source_line_end":
                None,

            "text":
                str(
                    part
                ),
        })


# ============================================================
# 4) Resolve speaker / authority
# ============================================================

cell86_resolved_parts = []


for row in cell86_part_diagnostics:

    resolved_speaker = (
        row.get(
            "speaker_effective_canonical"
        )
        or
        row.get(
            "speaker_effective_label"
        )
        or
        row.get(
            "speaker_base_raw"
        )
        or
        row.get(
            "attribution_speaker"
        )
    )


    is_primary = (
        row.get(
            "speaker_effective_primary"
        )
        is True
        or
        resolved_speaker
        ==
        "Dr. Komeil Roudi"
    )


    is_explicit_nonprimary = (
        resolved_speaker
        is not None
        and
        not is_primary
    )


    is_unresolved = (
        resolved_speaker
        is None
        or
        row.get(
            "speaker_base_status"
        )
        ==
        "UNRESOLVED"
    )


    cell86_resolved_parts.append({
        **copy.deepcopy(
            row
        ),

        "resolved_speaker":
            resolved_speaker,

        "is_primary":
            is_primary,

        "is_explicit_nonprimary":
            is_explicit_nonprimary,

        "is_unresolved":
            is_unresolved,
    })


# ============================================================
# 5) Inventories
# ============================================================

cell86_primary_parts = [
    row
    for row
    in cell86_resolved_parts
    if row[
        "is_primary"
    ]
]


cell86_nonprimary_parts = [
    row
    for row
    in cell86_resolved_parts
    if row[
        "is_explicit_nonprimary"
    ]
]


cell86_unresolved_parts = [
    row
    for row
    in cell86_resolved_parts
    if row[
        "is_unresolved"
    ]
]


cell86_detected_speakers = sorted(
    {
        row[
            "resolved_speaker"
        ]
        for row
        in cell86_resolved_parts
        if row[
            "resolved_speaker"
        ]
        is not None
    }
)


# ============================================================
# 6) Global Authority Policy
# ============================================================

cell86_authority_policy = copy.deepcopy(
    cell85_persisted_registry.get(
        "workflow_policy",
        {},
    ).get(
        "authority_evidence_policy",
        {},
    )
)


cell86_authority_policy_valid = all([
    cell86_authority_policy.get(
        "primary_authority"
    )
    ==
    "Dr. Komeil Roudi",

    cell86_authority_policy.get(
        "secondary_to_primary_promotion"
    )
    ==
    "FORBIDDEN",
])


if not cell86_authority_policy_valid:
    cell86_failures.append(
        "GLOBAL_AUTHORITY_POLICY_INVALID"
    )


# ============================================================
# 7) Task structure
# ============================================================

cell86_expected_task = (
    CELL86_TASK_FAMILY
    ==
    "MIXED_AUTHORITY_CONTEXT_RETRIEVAL"
)


cell86_expected_authority_rule = (
    CELL86_AUTHORITY_EXPECTATION
    ==
    "NO_FILE_LEVEL_AUTHORITY_FLATTENING"
)


cell86_structure_valid = all([
    cell86_expected_task,
    cell86_expected_authority_rule,
    len(
        cell86_resolved_parts
    )
    >
    0,
])


if not cell86_structure_valid:
    cell86_failures.append(
        "DIAGNOSTIC_STRUCTURE_INVALID"
    )


# ============================================================
# 8) Preliminary usability assessment
# ============================================================

cell86_has_primary = (
    len(
        cell86_primary_parts
    )
    >
    0
)


cell86_has_nonprimary = (
    len(
        cell86_nonprimary_parts
    )
    >
    0
)


cell86_has_unresolved = (
    len(
        cell86_unresolved_parts
    )
    >
    0
)


cell86_mixed_authority_present = all([
    cell86_has_primary,
    cell86_has_nonprimary,
])


cell86_needs_deeper_context = (
    cell86_has_unresolved
    or
    not cell86_has_primary
)


cell86_target_preliminarily_usable = all([
    cell86_has_primary,
    not cell86_has_unresolved,
])


# ============================================================
# 9) Relevant metadata
# ============================================================

def cell86_collect_relevant_metadata(
    value,
    path=(),
):

    output = []


    keywords = (
        "speaker",
        "authority",
        "attribution",
        "primary",
        "mixed",
        "guardrail",
        "task_family",
        "question_archetype",
        "stratum",
    )


    if isinstance(
        value,
        dict,
    ):

        for key, item in value.items():

            key_text = str(
                key
            ).lower()


            if any(
                token
                in
                key_text
                for token
                in keywords
            ):

                output.append({
                    "path":
                        path
                        +
                        (
                            str(
                                key
                            ),
                        ),

                    "value":
                        copy.deepcopy(
                            item
                        ),
                })


            output.extend(
                cell86_collect_relevant_metadata(
                    item,
                    path
                    +
                    (
                        str(
                            key
                        ),
                    ),
                )
            )


    elif isinstance(
        value,
        list,
    ):

        for index, item in enumerate(
            value
        ):

            output.extend(
                cell86_collect_relevant_metadata(
                    item,
                    path
                    +
                    (
                        str(
                            index
                        ),
                    ),
                )
            )


    return output


cell86_relevant_metadata = (
    cell86_collect_relevant_metadata(
        cell85_next_pilot_packet
    )
)


# ============================================================
# 10) Safety
# ============================================================

cell86_registry_write = False

cell86_queue_advance = False

cell86_human_decision_added = False

cell86_target_changed = False

cell86_citation_changed = False

cell86_raw_source_mutated = False

cell86_parser_mutated = False

cell86_commit_push = False


# ============================================================
# 11) Gate
# ============================================================

cell86_gate_checks = {
    "no_failures":
        len(
            cell86_failures
        )
        ==
        0,

    "structure_valid":
        cell86_structure_valid,

    "authority_policy_valid":
        cell86_authority_policy_valid,

    "target_parts_present":
        len(
            cell86_resolved_parts
        )
        >
        0,
}


cell86_gate = all(
    cell86_gate_checks.values()
)


# ============================================================
# 12) Diagnostics
# ============================================================

print(
    "=== PILOT #19 MIXED AUTHORITY DIAGNOSTIC ==="
)


print(
    "\n=== PILOT ==="
)

print(
    "Pilot:",
    CELL86_PILOT_ID,
)

print(
    "Target Evidence:",
    CELL86_TARGET_EVIDENCE_ID,
)

print(
    "Task Family:",
    CELL86_TASK_FAMILY,
)

print(
    "Authority Expectation:",
    CELL86_AUTHORITY_EXPECTATION,
)


print(
    "\n=== TARGET PARTS ==="
)


for row in cell86_resolved_parts:

    print(
        "\n"
        +
        "="
        *
        72
    )

    print(
        "PART:",
        row[
            "index"
        ],
    )

    print(
        "Document:",
        row.get(
            "document_id"
        ),
    )

    print(
        "Role:",
        row.get(
            "role"
        ),
    )

    print(
        "Base status:",
        row.get(
            "speaker_base_status"
        ),
    )

    print(
        "Base raw:",
        row.get(
            "speaker_base_raw"
        ),
    )

    print(
        "Effective label:",
        row.get(
            "speaker_effective_label"
        ),
    )

    print(
        "Effective canonical:",
        row.get(
            "speaker_effective_canonical"
        ),
    )

    print(
        "Effective primary:",
        row.get(
            "speaker_effective_primary"
        ),
    )

    print(
        "Attribution speaker:",
        row.get(
            "attribution_speaker"
        ),
    )

    print(
        "Resolved speaker:",
        row.get(
            "resolved_speaker"
        ),
    )

    print(
        "Is Primary:",
        row.get(
            "is_primary"
        ),
    )

    print(
        "Is Explicit Non-Primary:",
        row.get(
            "is_explicit_nonprimary"
        ),
    )

    print(
        "Is Unresolved:",
        row.get(
            "is_unresolved"
        ),
    )

    print(
        "Source lines:",
        (
            row.get(
                "source_line_start"
            ),
            row.get(
                "source_line_end"
            ),
        ),
    )

    print(
        "Text:"
    )

    print(
        row.get(
            "text"
        )
    )


print(
    "\n=== AUTHORITY SUMMARY ==="
)

print(
    "Detected speakers:",
    cell86_detected_speakers,
)

print(
    "Primary parts:",
    len(
        cell86_primary_parts
    ),
)

print(
    "Non-Primary parts:",
    len(
        cell86_nonprimary_parts
    ),
)

print(
    "Unresolved parts:",
    len(
        cell86_unresolved_parts
    ),
)

print(
    "Mixed Authority present:",
    cell86_mixed_authority_present,
)

print(
    "Needs deeper context:",
    cell86_needs_deeper_context,
)

print(
    "Target preliminarily usable:",
    cell86_target_preliminarily_usable,
)


print(
    "\n=== GLOBAL AUTHORITY POLICY ==="
)

print(
    "Primary Authority:",
    cell86_authority_policy.get(
        "primary_authority"
    ),
)

print(
    "Secondary-to-primary promotion:",
    cell86_authority_policy.get(
        "secondary_to_primary_promotion"
    ),
)

print(
    "Policy valid:",
    cell86_authority_policy_valid,
)


print(
    "\n=== RELEVANT PILOT METADATA ==="
)


for match in cell86_relevant_metadata:

    print(
        " > ".join(
            match[
                "path"
            ]
        ),
        "=",
        match[
            "value"
        ],
    )


print(
    "\n=== SAFETY ==="
)

print(
    "Registry write:",
    cell86_registry_write,
)

print(
    "Queue advance:",
    cell86_queue_advance,
)

print(
    "Human Decision added:",
    cell86_human_decision_added,
)

print(
    "Target changed:",
    cell86_target_changed,
)

print(
    "Citation changed:",
    cell86_citation_changed,
)

print(
    "Raw Source mutated:",
    cell86_raw_source_mutated,
)

print(
    "Parser mutated:",
    cell86_parser_mutated,
)

print(
    "Commit / Push:",
    cell86_commit_push,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell86_failures
)


print(
    "\nCODE CELL 86 GATE:",
    (
        "PASS"
        if cell86_gate
        else "FAIL"
    ),
)


assert cell86_gate, (
    "Pilot #19 mixed-authority diagnostic failed. "
    "Repair this SAME Code Cell 86 before drafting the question."
)

=== PILOT #19 MIXED AUTHORITY DIAGNOSTIC ===

=== PILOT ===
Pilot: question-pilot:6ba9960127f4b21ec2dc1ab2547ae3b5
Target Evidence: target-evidence:aabbc6aed0c573b34019e6212cd05cad
Task Family: MIXED_AUTHORITY_CONTEXT_RETRIEVAL
Authority Expectation: NO_FILE_LEVEL_AUTHORITY_FLATTENING

=== TARGET PARTS ===

PART: 0
Document: program:uni-tehran:episode-011:utterance:00035
Role: AUTHORITY_CONTRAST_LEFT
Base status: EXPLICIT_SOURCE_MARKER
Base raw: استاد
Effective label: استاد
Effective canonical: Dr. Komeil Roudi
Effective primary: True
Attribution speaker: Dr. Komeil Roudi
Resolved speaker: Dr. Komeil Roudi
Is Primary: True
Is Explicit Non-Primary: False
Is Unresolved: False
Source lines: (191, 193)
Text:
ولی یه چیزی که اینجا هست اینه که ممکنه اون درآمدی که من دارم واقعاً نقد نباشه. یعنی مثلاً فرض کن من دارم یه جایی مشغول هستم، به من می‌گن که در سال ۲۰۰ میلیون تومن بهت حقوق می‌دیم، ولی ۲۹ اسفند حقوقت رو می‌دیم. این درآمد هست یا نه؟ قطعاً هست. ولی وقتی که من آخر سال قراره این حقوق رو بگی

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 87 — Pilot #19 Question/Answer ACCEPT + Citation Draft

Human Reviewer سؤال و جواب Pilot #19 را پذیرفت.

در این Pilot:

- Part 0 متعلق به Dr. Komeil Roudi و Primary Authority است؛
- Part 1 متعلق به مجری و Non-Primary Context است؛
- حرف مجری به‌تنهایی Answer Evidence محسوب نمی‌شود؛
- Expected Answer فقط از توضیح Dr. Komeil Roudi درباره تفاوت درآمد، زمان دریافت وجه و بحران نقدینگی ساخته می‌شود؛
- Retarget انجام نمی‌شود.

سؤال پذیرفته‌شده:

**چرا ممکن است فرد با وجود درآمد مناسب و حتی وضعیت مالی رو‌به‌رشد، باز هم دچار بحران نقدینگی شود؟**

جواب پذیرفته‌شده:

**چون زمان دریافت درآمدها ممکن است با زمان پرداخت هزینه‌ها هماهنگ نباشد. مثلاً فرد ممکن است درآمد خوبی داشته باشد، اما پول آن را دیرتر دریافت کند، در حالی که هزینه‌هایش زودتر سررسید می‌شوند. به همین دلیل فقط بررسی درآمد و هزینه کافی نیست و باید جریان ورود و خروج وجه نقد هم بررسی شود تا کمبود نقدینگی پیش از وقوع مشخص شود.**

بعد از این Cell، Citation در وضعیت Human Review قرار می‌گیرد و Queue همچنان 5 باقی می‌ماند.

</div>

In [87]:
# Code Cell 87

cell87_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell86_gate") is not True:
    cell87_failures.append(
        "CELL86_NOT_PASS"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell87_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Human-accepted Question / Answer
# ============================================================

CELL87_ACCEPTED_QUESTION = (
    "چرا ممکن است فرد با وجود درآمد مناسب و حتی وضعیت مالی "
    "رو‌به‌رشد، باز هم دچار بحران نقدینگی شود؟"
)


CELL87_ACCEPTED_ANSWER = (
    "چون زمان دریافت درآمدها ممکن است با زمان پرداخت هزینه‌ها "
    "هماهنگ نباشد. مثلاً فرد ممکن است درآمد خوبی داشته باشد، "
    "اما پول آن را دیرتر دریافت کند، در حالی که هزینه‌هایش "
    "زودتر سررسید می‌شوند. به همین دلیل فقط بررسی درآمد و هزینه "
    "کافی نیست و باید جریان ورود و خروج وجه نقد هم بررسی شود "
    "تا کمبود نقدینگی پیش از وقوع مشخص شود."
)


CELL87_PILOT_ID = (
    CELL86_PILOT_ID
)


CELL87_GOLDEN_CASE_ID = (
    CELL85_NEXT_GOLDEN_CASE_ID
)


CELL87_TASK_DRAFT_ID = (
    CELL85_NEXT_TASK_DRAFT_ID
)


CELL87_TARGET_EVIDENCE_ID = (
    CELL86_TARGET_EVIDENCE_ID
)


# ============================================================
# 3) Target validation
# ============================================================

cell87_target_parts = copy.deepcopy(
    cell85_next_target_text_parts
)


cell87_target_valid = all([
    isinstance(
        cell87_target_parts,
        list,
    ),

    len(
        cell87_target_parts
    )
    ==
    2,

    len(
        cell86_primary_parts
    )
    ==
    1,

    len(
        cell86_nonprimary_parts
    )
    ==
    1,

    len(
        cell86_unresolved_parts
    )
    ==
    0,

    cell86_primary_parts[
        0
    ].get(
        "resolved_speaker"
    )
    ==
    "Dr. Komeil Roudi",

    cell86_nonprimary_parts[
        0
    ].get(
        "resolved_speaker"
    )
    ==
    "مجری",
])


if not cell87_target_valid:
    cell87_failures.append(
        "TARGET_AUTHORITY_STRUCTURE_INVALID"
    )


# ============================================================
# 4) Primary Answer Evidence
# ============================================================

cell87_primary_part = copy.deepcopy(
    cell87_target_parts[
        0
    ]
)


cell87_context_part = copy.deepcopy(
    cell87_target_parts[
        1
    ]
)


cell87_primary_text = (
    cell87_primary_part.get(
        "text"
    )
    or
    cell87_primary_part.get(
        "target_text"
    )
    or
    cell87_primary_part.get(
        "content"
    )
    or
    ""
)


cell87_context_text = (
    cell87_context_part.get(
        "text"
    )
    or
    cell87_context_part.get(
        "target_text"
    )
    or
    cell87_context_part.get(
        "content"
    )
    or
    ""
)


# ============================================================
# 5) Grounding
# ============================================================

def cell87_norm(
    text,
):

    if not isinstance(
        text,
        str,
    ):
        return ""


    value = (
        text
        .replace(
            "ي",
            "ی",
        )
        .replace(
            "ك",
            "ک",
        )
        .replace(
            "\u200c",
            "",
        )
        .replace(
            "\ufeff",
            "",
        )
    )


    value = re.sub(
        r"\s+",
        " ",
        value,
    )


    return value.strip()


cell87_primary_surface = cell87_norm(
    cell87_primary_text
)


cell87_grounding = {
    "income_not_cash":
        (
            "درآمدی که من دارم واقعاً نقد نباشه"
            in
            cell87_primary_surface
        ),

    "income_exists":
        (
            "این درآمد هست یا نه؟ قطعاً هست"
            in
            cell87_primary_surface
        ),

    "late_income":
        (
            "۲۹ اسفند حقوقت رو میدیم"
            in
            cell87_primary_surface
        ),

    "expenses_cannot_wait":
        (
            "هزینههام رو که نمیتونم بندازم آخر سال"
            in
            cell87_primary_surface
        ),

    "positive_profit_loss_possible":
        (
            "صورت‌حساب سود و زیانم"
            in
            cell87_primary_text
            or
            "صورت حساب سود و زیانم"
            in
            cell87_primary_surface
        ),

    "liquidity_crisis":
        (
            "بحران دیگه اینجا ممکنه داشته باشه و اون نقدینگیه"
            in
            cell87_primary_surface
        ),

    "cash_flow_statement":
        (
            "صورت جریان وجوه نقد"
            in
            cell87_primary_surface
        ),

    "cash_inflows_outflows":
        (
            "ورودیها و خروجیهای نقدی"
            in
            cell87_primary_surface
        ),

    "avoid_liquidity_crisis":
        (
            "به بحران نقدینگی نخورم"
            in
            cell87_primary_surface
        ),
}


cell87_grounding_pass = all(
    cell87_grounding.values()
)


if not cell87_grounding_pass:
    cell87_failures.append(
        "ANSWER_GROUNDING_FAILED"
    )


# ============================================================
# 6) Start Engine
# ============================================================

cell87_question_pending_session = (
    cell13_build_session(
        cell85_next_pilot_packet
    )
)


cell87_session_start_valid = (
    cell87_question_pending_session.get(
        "current_state"
    )
    ==
    "QUESTION_DRAFT_PENDING"
)


if not cell87_session_start_valid:
    cell87_failures.append(
        "ENGINE_SESSION_START_INVALID"
    )


# ============================================================
# 7) Question Draft + Human ACCEPT
# ============================================================

cell87_question_identity = {
    "identity_version":
        "question_draft_identity_v1",

    "pilot_case_id":
        CELL87_PILOT_ID,

    "target_evidence_id":
        CELL87_TARGET_EVIDENCE_ID,

    "question_text":
        CELL87_ACCEPTED_QUESTION,

    "revision_number":
        1,
}


cell87_question_draft_id = (
    "question-draft:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell87_question_identity
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


cell87_question_record = {
    "record_version":
        "question_draft_record_v1",

    "question_draft_id":
        cell87_question_draft_id,

    "pilot_case_id":
        CELL87_PILOT_ID,

    "golden_case_id":
        CELL87_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL87_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL87_TARGET_EVIDENCE_ID,

    "question_text":
        CELL87_ACCEPTED_QUESTION,

    "revision_number":
        1,

    "human_adjudication_status":
        "PENDING",

    "external_knowledge_used":
        False,
}


cell87_question_review_session = (
    cell13_apply_event(
        cell87_question_pending_session,
        "QUESTION_DRAFT_REGISTERED",
        artifact=
            cell87_question_record,
    )
)


cell87_answer_pending_session = (
    cell13_apply_event(
        cell87_question_review_session,
        "QUESTION_ACCEPTED",
        human_note=
            "Human Reviewer accepted the question.",
    )
)


cell87_question_accept_valid = (
    cell87_answer_pending_session.get(
        "current_state"
    )
    ==
    "EXPECTED_ANSWER_DRAFT_PENDING"
)


if not cell87_question_accept_valid:
    cell87_failures.append(
        "QUESTION_ACCEPT_TRANSITION_INVALID"
    )


# ============================================================
# 8) Expected Answer + Human ACCEPT
# ============================================================

cell87_answer_identity = {
    "identity_version":
        "expected_answer_identity_v1",

    "pilot_case_id":
        CELL87_PILOT_ID,

    "question_draft_id":
        cell87_question_draft_id,

    "target_evidence_id":
        CELL87_TARGET_EVIDENCE_ID,

    "answer_text":
        CELL87_ACCEPTED_ANSWER,

    "revision_number":
        1,
}


cell87_expected_answer_draft_id = (
    "expected-answer-draft:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell87_answer_identity
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


cell87_expected_answer_record = {
    "record_version":
        "expected_answer_draft_record_v1",

    "expected_answer_draft_id":
        cell87_expected_answer_draft_id,

    "pilot_case_id":
        CELL87_PILOT_ID,

    "golden_case_id":
        CELL87_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL87_TASK_DRAFT_ID,

    "question_draft_id":
        cell87_question_draft_id,

    "target_evidence_id":
        CELL87_TARGET_EVIDENCE_ID,

    "expected_answer_text":
        CELL87_ACCEPTED_ANSWER,

    "revision_number":
        1,

    "human_adjudication_status":
        "PENDING",

    "external_knowledge_used":
        False,
}


cell87_answer_review_session = (
    cell13_apply_event(
        cell87_answer_pending_session,
        "EXPECTED_ANSWER_REGISTERED",
        artifact=
            cell87_expected_answer_record,
    )
)


cell87_citation_pending_session = (
    cell13_apply_event(
        cell87_answer_review_session,
        "EXPECTED_ANSWER_ACCEPTED",
        human_note=
            "Human Reviewer accepted the expected answer.",
    )
)


cell87_answer_accept_valid = (
    cell87_citation_pending_session.get(
        "current_state"
    )
    ==
    "CITATION_MAPPING_PENDING"
)


if not cell87_answer_accept_valid:
    cell87_failures.append(
        "ANSWER_ACCEPT_TRANSITION_INVALID"
    )


# ============================================================
# 9) Citation Entries
# ============================================================

cell87_primary_entry = copy.deepcopy(
    cell87_primary_part
)


cell87_primary_entry.update({
    "citation_index":
        0,

    "authority_role":
        "PRIMARY_ANSWER_EVIDENCE",

    "answer_support":
        True,

    "use_in_expected_answer_grounding":
        True,

    "human_authority_overlay": {
        "speaker":
            "Dr. Komeil Roudi",

        "effective_primary":
            True,

        "role":
            "PRIMARY_ANSWER_EVIDENCE",
    },

    "external_locator_lookup_used":
        False,
})


cell87_context_entry = copy.deepcopy(
    cell87_context_part
)


cell87_context_entry.update({
    "citation_index":
        1,

    "authority_role":
        "NON_PRIMARY_HOST_CONTEXT",

    "answer_support":
        False,

    "use_in_expected_answer_grounding":
        False,

    "human_authority_overlay": {
        "speaker":
            "مجری",

        "effective_primary":
            False,

        "role":
            "CONTEXT_ONLY",

        "secondary_to_primary_promotion":
            "FORBIDDEN",
    },

    "external_locator_lookup_used":
        False,
})


cell87_citation_entries = [
    cell87_primary_entry,
    cell87_context_entry,
]


cell87_primary_answer_entries = [
    entry
    for entry
    in cell87_citation_entries
    if entry.get(
        "answer_support"
    )
    is True
]


cell87_context_only_entries = [
    entry
    for entry
    in cell87_citation_entries
    if entry.get(
        "answer_support"
    )
    is False
]


cell87_citation_boundary_valid = all([
    len(
        cell87_citation_entries
    )
    ==
    2,

    len(
        cell87_primary_answer_entries
    )
    ==
    1,

    len(
        cell87_context_only_entries
    )
    ==
    1,

    cell87_primary_entry.get(
        "human_authority_overlay",
        {},
    ).get(
        "speaker"
    )
    ==
    "Dr. Komeil Roudi",

    cell87_primary_entry.get(
        "human_authority_overlay",
        {},
    ).get(
        "effective_primary"
    )
    is True,

    cell87_context_entry.get(
        "human_authority_overlay",
        {},
    ).get(
        "speaker"
    )
    ==
    "مجری",

    cell87_context_entry.get(
        "answer_support"
    )
    is False,

    cell87_context_entry.get(
        "use_in_expected_answer_grounding"
    )
    is False,
])


if not cell87_citation_boundary_valid:
    cell87_failures.append(
        "CITATION_AUTHORITY_BOUNDARY_INVALID"
    )


# ============================================================
# 10) Locator integrity
# ============================================================

cell87_no_locator_fabrication = all([
    cell87_primary_entry.get(
        "locator"
    )
    ==
    cell87_primary_part.get(
        "locator"
    ),

    cell87_context_entry.get(
        "locator"
    )
    ==
    cell87_context_part.get(
        "locator"
    ),
])


if not cell87_no_locator_fabrication:
    cell87_failures.append(
        "LOCATOR_INTEGRITY_FAILED"
    )


# ============================================================
# 11) Citation Mapping
# ============================================================

cell87_citation_identity = {
    "identity_version":
        "citation_mapping_identity_v1_mixed_authority",

    "pilot_case_id":
        CELL87_PILOT_ID,

    "question_draft_id":
        cell87_question_draft_id,

    "expected_answer_draft_id":
        cell87_expected_answer_draft_id,

    "target_evidence_id":
        CELL87_TARGET_EVIDENCE_ID,

    "citation_entries":
        copy.deepcopy(
            cell87_citation_entries
        ),
}


cell87_citation_mapping_id = (
    "citation-mapping:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell87_citation_identity
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


cell87_citation_mapping_record = {
    "record_version":
        "citation_mapping_record_v1_mixed_authority",

    "citation_mapping_id":
        cell87_citation_mapping_id,

    "pilot_case_id":
        CELL87_PILOT_ID,

    "golden_case_id":
        CELL87_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL87_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL87_TARGET_EVIDENCE_ID,

    "question_draft_id":
        cell87_question_draft_id,

    "expected_answer_draft_id":
        cell87_expected_answer_draft_id,

    "citation_entries":
        copy.deepcopy(
            cell87_citation_entries
        ),

    "primary_answer_authority":
        "Dr. Komeil Roudi",

    "non_primary_context":
        "مجری",

    "non_primary_used_as_answer_evidence":
        False,

    "secondary_to_primary_promotion":
        "FORBIDDEN",

    "file_level_authority_flattening":
        False,

    "retarget_used":
        False,

    "raw_source_mutated":
        False,

    "parser_output_mutated":
        False,

    "external_knowledge_used":
        False,

    "human_adjudication_status":
        "PENDING",
}


cell87_current_session = (
    cell13_apply_event(
        cell87_citation_pending_session,
        "CITATION_MAPPING_REGISTERED",
        artifact=
            cell87_citation_mapping_record,
    )
)


cell87_citation_registered_valid = (
    cell87_current_session.get(
        "current_state"
    )
    ==
    "CITATION_HUMAN_REVIEW_PENDING"
)


if not cell87_citation_registered_valid:
    cell87_failures.append(
        "CITATION_REGISTRATION_INVALID"
    )


# ============================================================
# 12) Active Workspace
# ============================================================

cell87_active_workspace = copy.deepcopy(
    cell85_persisted_active_case
)


cell87_active_workspace[
    "workflow_state"
] = "CITATION_HUMAN_REVIEW_PENDING"


cell87_active_workspace[
    "question_review"
] = {
    "question_draft_id":
        cell87_question_draft_id,

    "question_text":
        CELL87_ACCEPTED_QUESTION,

    "human_decision":
        "ACCEPT",
}


cell87_active_workspace[
    "expected_answer_review"
] = {
    "expected_answer_draft_id":
        cell87_expected_answer_draft_id,

    "expected_answer_text":
        CELL87_ACCEPTED_ANSWER,

    "human_decision":
        "ACCEPT",
}


cell87_active_workspace[
    "citation_review"
] = {
    "citation_mapping_id":
        cell87_citation_mapping_id,

    "citation_mapping_record":
        copy.deepcopy(
            cell87_citation_mapping_record
        ),

    "human_decision":
        "PENDING",

    "primary_answer_authority":
        "Dr. Komeil Roudi",

    "non_primary_context":
        "مجری",

    "non_primary_used_as_answer_evidence":
        False,
}


# ============================================================
# 13) Continuation state
# ============================================================

cell87_continuation_core = {
    "next_pilot_id":
        CELL87_PILOT_ID,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell85_remaining_pilot_ids
        ),

    "remaining_count":
        5,

    "active_pilot_status":
        "CITATION_HUMAN_REVIEW_PENDING",

    "active_target_evidence_id":
        CELL87_TARGET_EVIDENCE_ID,

    "active_citation_mapping_id":
        cell87_citation_mapping_id,

    "retarget_used":
        False,

    "authority_policy_active":
        True,

    "last_completed_pilot_id":
        cell85_continuation_state.get(
            "last_completed_pilot_id"
        ),
}


cell87_continuation_state_sha256 = (
    cell10_stable_sha(
        cell87_continuation_core
    )
)


cell87_continuation_state = {
    **copy.deepcopy(
        cell87_continuation_core
    ),

    "continuation_state_sha256":
        cell87_continuation_state_sha256,
}


# ============================================================
# 14) Persist
# ============================================================

cell87_persistent_registry_payload = copy.deepcopy(
    cell85_persisted_registry
)


cell87_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell87_active_workspace
)


cell87_persistent_registry_payload[
    "remaining_pilot_count"
] = 5


cell87_persistent_registry_payload[
    "continuation_state"
] = copy.deepcopy(
    cell87_continuation_state
)


cell87_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell87_persistent_registry_payload
    )
)


# ============================================================
# 15) Read-back validation
# ============================================================

cell87_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell87_persisted_active_case = (
    cell87_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell87_persistence_valid = all([
    cell87_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL87_PILOT_ID,

    cell87_persisted_active_case.get(
        "workflow_state"
    )
    ==
    "CITATION_HUMAN_REVIEW_PENDING",

    cell87_persisted_active_case.get(
        "question_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell87_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell87_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "PENDING",

    cell87_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "non_primary_used_as_answer_evidence"
    )
    is False,

    cell87_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    5,

    cell87_persisted_registry.get(
        "continuation_state",
        {},
    ).get(
        "continuation_state_sha256"
    )
    ==
    cell87_continuation_state_sha256,
])


if not cell87_persistence_valid:
    cell87_failures.append(
        "REGISTRY_READBACK_INVALID"
    )


# ============================================================
# 16) Safety
# ============================================================

cell87_queue_still_5 = all([
    len(
        cell85_remaining_pilot_ids
    )
    ==
    5,

    cell85_remaining_pilot_ids[
        0
    ]
    ==
    CELL87_PILOT_ID,
])


cell87_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell87_full_572_blocked = (
    cell85_full_572_blocked
    is True
)


cell87_no_commit_push = True


# ============================================================
# 17) Gate
# ============================================================

cell87_gate_checks = {
    "no_failures":
        len(
            cell87_failures
        )
        ==
        0,

    "target_valid":
        cell87_target_valid,

    "grounding_pass":
        cell87_grounding_pass,

    "session_start_valid":
        cell87_session_start_valid,

    "question_accept_valid":
        cell87_question_accept_valid,

    "answer_accept_valid":
        cell87_answer_accept_valid,

    "citation_boundary_valid":
        cell87_citation_boundary_valid,

    "no_locator_fabrication":
        cell87_no_locator_fabrication,

    "citation_registered_valid":
        cell87_citation_registered_valid,

    "persistence_valid":
        cell87_persistence_valid,

    "queue_still_5":
        cell87_queue_still_5,

    "base_seed_unchanged":
        cell87_base_seed_unchanged,

    "full_572_blocked":
        cell87_full_572_blocked,

    "no_commit_push":
        cell87_no_commit_push,
}


cell87_gate = all(
    cell87_gate_checks.values()
)


# ============================================================
# 18) Diagnostics
# ============================================================

print(
    "=== PILOT #19 QA ACCEPT + CITATION DRAFT ==="
)


print(
    "\n=== AUTHORITY ==="
)

print(
    "Primary Answer Authority:",
    "Dr. Komeil Roudi",
)

print(
    "Non-Primary Context:",
    "مجری",
)

print(
    "Non-Primary used as Answer Evidence:",
    False,
)

print(
    "File-level Authority Flattening:",
    False,
)

print(
    "Secondary-to-primary promotion:",
    "FORBIDDEN",
)

print(
    "Retarget used:",
    False,
)


print(
    "\n=== ACCEPTED QUESTION ==="
)

print(
    CELL87_ACCEPTED_QUESTION
)

print(
    "Question Draft ID:",
    cell87_question_draft_id,
)

print(
    "Human Decision:",
    "ACCEPT",
)


print(
    "\n=== ACCEPTED ANSWER ==="
)

print(
    CELL87_ACCEPTED_ANSWER
)

print(
    "Expected Answer Draft ID:",
    cell87_expected_answer_draft_id,
)

print(
    "Human Decision:",
    "ACCEPT",
)


print(
    "\n=== GROUNDING ==="
)

for name, status in cell87_grounding.items():

    print(
        f"{name}:",
        status,
    )


print(
    "Grounding PASS:",
    cell87_grounding_pass,
)


print(
    "\n=== CITATION DRAFT ==="
)

print(
    "Citation Mapping ID:",
    cell87_citation_mapping_id,
)

print(
    "Citation entries:",
    len(
        cell87_citation_entries
    ),
)

print(
    "Primary Answer Evidence entries:",
    len(
        cell87_primary_answer_entries
    ),
)

print(
    "Context-only entries:",
    len(
        cell87_context_only_entries
    ),
)

print(
    "Non-Primary used as Answer Evidence:",
    False,
)

print(
    "Citation boundary valid:",
    cell87_citation_boundary_valid,
)

print(
    "No locator fabrication:",
    cell87_no_locator_fabrication,
)


print(
    "\n=== WORKFLOW ==="
)

print(
    "Current State:",
    cell87_current_session.get(
        "current_state"
    ),
)

print(
    "Question:",
    "ACCEPT",
)

print(
    "Expected Answer:",
    "ACCEPT",
)

print(
    "Citation:",
    "PENDING",
)


print(
    "\n=== ALL INTERNAL GATES ==="
)

for name, status in cell87_gate_checks.items():

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== QUEUE / SAFETY ==="
)

print(
    "Queue still 5:",
    cell87_queue_still_5,
)

print(
    "Persistence valid:",
    cell87_persistence_valid,
)

print(
    "Base Seed unchanged:",
    cell87_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell87_full_572_blocked,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell87_failures
)


print(
    "\nCODE CELL 87 GATE:",
    (
        "PASS"
        if cell87_gate
        else "FAIL"
    ),
)


assert cell87_gate, (
    "Pilot #19 Question+Answer ACCEPT / Citation Draft failed. "
    "Repair this SAME Code Cell 87."
)

=== PILOT #19 QA ACCEPT + CITATION DRAFT ===

=== AUTHORITY ===
Primary Answer Authority: Dr. Komeil Roudi
Non-Primary Context: مجری
Non-Primary used as Answer Evidence: False
File-level Authority Flattening: False
Secondary-to-primary promotion: FORBIDDEN
Retarget used: False

=== ACCEPTED QUESTION ===
چرا ممکن است فرد با وجود درآمد مناسب و حتی وضعیت مالی رو‌به‌رشد، باز هم دچار بحران نقدینگی شود؟
Question Draft ID: question-draft:5692a98d684ddaafd1e0d0006e2fe960
Human Decision: ACCEPT

=== ACCEPTED ANSWER ===
چون زمان دریافت درآمدها ممکن است با زمان پرداخت هزینه‌ها هماهنگ نباشد. مثلاً فرد ممکن است درآمد خوبی داشته باشد، اما پول آن را دیرتر دریافت کند، در حالی که هزینه‌هایش زودتر سررسید می‌شوند. به همین دلیل فقط بررسی درآمد و هزینه کافی نیست و باید جریان ورود و خروج وجه نقد هم بررسی شود تا کمبود نقدینگی پیش از وقوع مشخص شود.
Expected Answer Draft ID: expected-answer-draft:ad1779dad0df73a6832e7eeb34627afa
Human Decision: ACCEPT

=== GROUNDING ===
income_not_cash: True
income_exists: Tru

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 88 — Citation ACCEPT + Pilot #19 Completion + Queue Advance

Citation نهایی Pilot #19 توسط Human Reviewer پذیرفته شد.

در این مرحله:

- Citation با تصمیم `ACCEPT` ثبت می‌شود؛
- Dr. Komeil Roudi تنها Primary Answer Authority باقی می‌ماند؛
- مجری فقط Context است و مستقل از دکتر Answer Evidence محسوب نمی‌شود؛
- Pilot #19 کامل می‌شود؛
- تعداد Caseهای کامل Human Review از 17 به 18 می‌رسد؛
- تعداد Caseهای آماده Author Review از 17 به 18 می‌رسد؛
- Queue دقیقاً از 5 به 4 جلو می‌رود؛
- Pilot بعدی برای Human Review آماده می‌شود؛
- Author Final Review همچنان `NOT_STARTED` است؛
- Full 572 Generation همچنان Blocked باقی می‌ماند.

</div>

In [88]:
# Code Cell 88

cell88_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell87_gate") is not True:
    cell88_failures.append(
        "CELL87_NOT_PASS"
    )


if (
    globals().get(
        "cell87_current_session",
        {},
    ).get(
        "current_state"
    )
    !=
    "CITATION_HUMAN_REVIEW_PENDING"
):
    cell88_failures.append(
        "CITATION_NOT_AT_HUMAN_REVIEW_GATE"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell88_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Human Citation ACCEPT
# ============================================================

CELL88_CITATION_HUMAN_NOTE = (
    "Human Reviewer accepted the final Citation Mapping. "
    "Dr. Komeil Roudi remains the sole primary answer authority. "
    "The host remains context-only and is not used as independent "
    "Expected Answer evidence."
)


cell88_complete_pending_session = (
    cell13_apply_event(
        cell87_current_session,
        "CITATION_ACCEPTED",
        human_note=
            CELL88_CITATION_HUMAN_NOTE,
    )
)


cell88_citation_accept_valid = (
    cell88_complete_pending_session.get(
        "current_state"
    )
    ==
    "PILOT_COMPLETE_PENDING_QUEUE_ADVANCE"
)


if not cell88_citation_accept_valid:
    cell88_failures.append(
        "CITATION_ACCEPT_TRANSITION_INVALID"
    )


# ============================================================
# 3) Final Citation Mapping
# ============================================================

cell88_final_citation_mapping_record = copy.deepcopy(
    cell87_citation_mapping_record
)


cell88_final_citation_mapping_record[
    "human_adjudication_status"
] = "ACCEPT"


cell88_final_citation_mapping_record[
    "human_review"
] = {
    "human_decision":
        "ACCEPT",

    "human_note":
        CELL88_CITATION_HUMAN_NOTE,

    "primary_answer_authority":
        "Dr. Komeil Roudi",

    "non_primary_context":
        "مجری",

    "non_primary_used_as_answer_evidence":
        False,

    "file_level_authority_flattening":
        False,

    "secondary_to_primary_promotion":
        "FORBIDDEN",

    "retarget_used":
        False,

    "raw_source_mutated":
        False,

    "parser_output_mutated":
        False,

    "external_knowledge_used":
        False,
}


# ============================================================
# 4) Complete Human Review case
# ============================================================

cell88_completed_case = copy.deepcopy(
    cell87_active_workspace
)


cell88_completed_case[
    "workflow_state"
] = "PILOT_COMPLETE"


cell88_completed_case[
    "status"
] = "COMPLETED_HUMAN_REVIEW"


cell88_completed_case[
    "final_status"
] = "PILOT_COMPLETE"


cell88_completed_case.setdefault(
    "citation_review",
    {},
)


cell88_completed_case[
    "citation_review"
][
    "human_decision"
] = "ACCEPT"


cell88_completed_case[
    "citation_review"
][
    "human_note"
] = CELL88_CITATION_HUMAN_NOTE


cell88_completed_case[
    "citation_review"
][
    "citation_mapping_record"
] = copy.deepcopy(
    cell88_final_citation_mapping_record
)


cell88_completed_case[
    "completion_summary"
] = {
    "retarget_used":
        False,

    "target_evidence_id":
        CELL87_TARGET_EVIDENCE_ID,

    "question_decision":
        "ACCEPT",

    "expected_answer_decision":
        "ACCEPT",

    "citation_decision":
        "ACCEPT",

    "task_family":
        "MIXED_AUTHORITY_CONTEXT_RETRIEVAL",

    "primary_answer_authority":
        "Dr. Komeil Roudi",

    "non_primary_context":
        "مجری",

    "non_primary_used_as_answer_evidence":
        False,

    "file_level_authority_flattening":
        False,

    "secondary_to_primary_promotion":
        "FORBIDDEN",

    "raw_source_mutated":
        False,

    "parser_output_mutated":
        False,

    "author_final_review":
        "NOT_STARTED",
}


cell88_completed_case[
    "author_final_review"
] = {
    "status":
        "NOT_STARTED",

    "decision":
        None,

    "notes":
        None,
}


# ============================================================
# 5) Queue advance: 5 -> 4
# ============================================================

cell88_previous_remaining_pilot_ids = copy.deepcopy(
    cell85_remaining_pilot_ids
)


CELL88_COMPLETED_PILOT_ID = (
    cell88_previous_remaining_pilot_ids[
        0
    ]
)


cell88_remaining_pilot_ids = copy.deepcopy(
    cell88_previous_remaining_pilot_ids[
        1:
    ]
)


cell88_queue_advance_valid = all([
    len(
        cell88_previous_remaining_pilot_ids
    )
    ==
    5,

    CELL88_COMPLETED_PILOT_ID
    ==
    CELL87_PILOT_ID,

    len(
        cell88_remaining_pilot_ids
    )
    ==
    4,

    CELL88_COMPLETED_PILOT_ID
    not in
    cell88_remaining_pilot_ids,
])


if not cell88_queue_advance_valid:
    cell88_failures.append(
        "QUEUE_ADVANCE_INVALID"
    )


CELL88_NEXT_PILOT_ID = (
    cell88_remaining_pilot_ids[
        0
    ]
)


# ============================================================
# 6) Find next Pilot packet
# ============================================================

cell88_raw_next_packet_candidates = (
    cell58_find_pilot_packet_candidates(
        cell13_seed,
        CELL88_NEXT_PILOT_ID,
    )
)


cell88_packet_candidates_by_sha = {}


for candidate in cell88_raw_next_packet_candidates:

    packet_sha = cell10_stable_sha(
        candidate[
            "packet"
        ]
    )


    previous = (
        cell88_packet_candidates_by_sha.get(
            packet_sha
        )
    )


    if (
        previous is None
        or
        candidate[
            "score"
        ]
        >
        previous[
            "score"
        ]
    ):

        cell88_packet_candidates_by_sha[
            packet_sha
        ] = candidate


cell88_ranked_next_packet_candidates = sorted(
    cell88_packet_candidates_by_sha.values(),
    key=lambda item: (
        -item[
            "score"
        ],
        item[
            "path"
        ],
    ),
)


cell88_next_pilot_packet = (
    copy.deepcopy(
        cell88_ranked_next_packet_candidates[
            0
        ][
            "packet"
        ]
    )
    if cell88_ranked_next_packet_candidates
    else None
)


if not isinstance(
    cell88_next_pilot_packet,
    dict,
):

    cell88_failures.append(
        "NEXT_PILOT_PACKET_NOT_FOUND"
    )


# ============================================================
# 7) Extract next Pilot identities
# ============================================================

cell88_next_pilot_ids_found = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell88_next_pilot_packet,
            "question-pilot:",
        )
    )
)


cell88_next_golden_case_ids = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell88_next_pilot_packet,
            "golden-case:",
        )
    )
)


cell88_next_task_draft_ids = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell88_next_pilot_packet,
            "task-draft:",
        )
    )
)


cell88_next_target_evidence_ids = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell88_next_pilot_packet,
            "target-evidence:",
        )
    )
)


cell88_next_identity_valid = all([
    cell88_next_pilot_ids_found
    ==
    [
        CELL88_NEXT_PILOT_ID
    ],

    len(
        cell88_next_golden_case_ids
    )
    ==
    1,

    len(
        cell88_next_task_draft_ids
    )
    ==
    1,

    len(
        cell88_next_target_evidence_ids
    )
    ==
    1,
])


if not cell88_next_identity_valid:
    cell88_failures.append(
        "NEXT_PILOT_IDENTITY_INVALID"
    )


CELL88_NEXT_GOLDEN_CASE_ID = (
    cell88_next_golden_case_ids[
        0
    ]
    if
    len(
        cell88_next_golden_case_ids
    )
    ==
    1
    else
    None
)


CELL88_NEXT_TASK_DRAFT_ID = (
    cell88_next_task_draft_ids[
        0
    ]
    if
    len(
        cell88_next_task_draft_ids
    )
    ==
    1
    else
    None
)


CELL88_NEXT_TARGET_EVIDENCE_ID = (
    cell88_next_target_evidence_ids[
        0
    ]
    if
    len(
        cell88_next_target_evidence_ids
    )
    ==
    1
    else
    None
)


# ============================================================
# 8) Extract next Target Evidence
# ============================================================

cell88_target_part_candidates = (
    cell58_find_key_values(
        cell88_next_pilot_packet,
        "target_text_parts",
    )
)


cell88_target_parts_by_sha = {}


for parts in cell88_target_part_candidates:

    if not isinstance(
        parts,
        list,
    ):
        continue


    if len(
        parts
    ) == 0:
        continue


    parts_sha = cell10_stable_sha(
        parts
    )


    cell88_target_parts_by_sha[
        parts_sha
    ] = parts


cell88_unique_target_part_sets = list(
    cell88_target_parts_by_sha.values()
)


cell88_next_target_text_parts = (
    copy.deepcopy(
        cell88_unique_target_part_sets[
            0
        ]
    )
    if
    len(
        cell88_unique_target_part_sets
    )
    ==
    1
    else
    None
)


if not isinstance(
    cell88_next_target_text_parts,
    list,
):

    cell88_failures.append(
        "NEXT_TARGET_TEXT_PARTS_NOT_UNIQUE"
    )


# ============================================================
# 9) Build readable next Evidence
# ============================================================

cell88_next_evidence_texts = []


for part in (
    cell88_next_target_text_parts
    or
    []
):

    if isinstance(
        part,
        str,
    ):

        text = part.strip()


    elif isinstance(
        part,
        dict,
    ):

        text = (
            part.get(
                "text"
            )
            or
            part.get(
                "target_text"
            )
            or
            part.get(
                "content"
            )
            or
            ""
        )


        if isinstance(
            text,
            str,
        ):

            text = text.strip()


        else:

            text = ""


    else:

        text = ""


    if text:

        cell88_next_evidence_texts.append(
            text
        )


cell88_next_combined_target_text = (
    "\n\n".join(
        cell88_next_evidence_texts
    )
)


cell88_next_evidence_available = bool(
    cell88_next_combined_target_text.strip()
)


if not cell88_next_evidence_available:
    cell88_failures.append(
        "NEXT_EVIDENCE_EMPTY"
    )


# ============================================================
# 10) Extract next Pilot metadata
# ============================================================

def cell88_unique_scalar_for_key(
    payload,
    key,
):

    values = (
        cell58_find_key_values(
            payload,
            key,
        )
    )


    normalized = []


    for value in values:

        if isinstance(
            value,
            (
                str,
                int,
                float,
                bool,
            ),
        ):

            if value not in normalized:
                normalized.append(
                    value
                )


    if len(
        normalized
    ) == 1:

        return normalized[
            0
        ]


    return None


CELL88_NEXT_STRATUM = (
    cell88_unique_scalar_for_key(
        cell88_next_pilot_packet,
        "stratum",
    )
)


CELL88_NEXT_TASK_FAMILY = (
    cell88_unique_scalar_for_key(
        cell88_next_pilot_packet,
        "task_family",
    )
)


CELL88_NEXT_ARCHETYPE = (
    cell88_unique_scalar_for_key(
        cell88_next_pilot_packet,
        "question_archetype",
    )
)


CELL88_NEXT_AUTHORITY_EXPECTATION = (
    cell88_unique_scalar_for_key(
        cell88_next_pilot_packet,
        "authority_expectation",
    )
)


# ============================================================
# 11) Next Pilot Engine session
# ============================================================

cell88_next_pilot_session = (
    cell13_build_session(
        cell88_next_pilot_packet
    )
)


cell88_next_session_valid = (
    cell88_next_pilot_session.get(
        "current_state"
    )
    ==
    "QUESTION_DRAFT_PENDING"
)


if not cell88_next_session_valid:
    cell88_failures.append(
        "NEXT_SESSION_INVALID"
    )


# ============================================================
# 12) Update completed Human Review cases
# ============================================================

cell88_existing_completed_cases = copy.deepcopy(
    cell87_persisted_registry.get(
        "completed_human_review_cases",
        [],
    )
)


if not isinstance(
    cell88_existing_completed_cases,
    list,
):

    cell88_existing_completed_cases = []

    cell88_failures.append(
        "COMPLETED_CASES_NOT_LIST"
    )


cell88_existing_completed_cases = [
    case
    for case
    in cell88_existing_completed_cases
    if not (
        isinstance(
            case,
            dict,
        )
        and
        case.get(
            "pilot_case_id"
        )
        ==
        CELL88_COMPLETED_PILOT_ID
    )
]


cell88_completed_human_review_cases = (
    cell88_existing_completed_cases
    +
    [
        copy.deepcopy(
            cell88_completed_case
        )
    ]
)


cell88_completed_case_count_valid = (
    len(
        cell88_completed_human_review_cases
    )
    ==
    18
)


if not cell88_completed_case_count_valid:
    cell88_failures.append(
        "COMPLETED_CASE_COUNT_INVALID"
    )


# ============================================================
# 13) Rebuild Author Review queue
# ============================================================

cell88_author_review = copy.deepcopy(
    cell87_persisted_registry.get(
        "author_review",
        {},
    )
)


if not isinstance(
    cell88_author_review,
    dict,
):

    cell88_author_review = {}


cell88_author_ready_cases = copy.deepcopy(
    cell88_completed_human_review_cases
)


cell88_author_review[
    "cases_ready_for_review"
] = copy.deepcopy(
    cell88_author_ready_cases
)


cell88_author_review[
    "case_count"
] = len(
    cell88_author_ready_cases
)


cell88_author_review[
    "final_review_status"
] = "NOT_STARTED"


cell88_author_ready_count_valid = (
    len(
        cell88_author_ready_cases
    )
    ==
    18
)


cell88_author_ready_identity_valid = (
    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell88_author_ready_cases
        if isinstance(
            case,
            dict,
        )
    }
    ==
    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell88_completed_human_review_cases
        if isinstance(
            case,
            dict,
        )
    }
)


if not cell88_author_ready_count_valid:
    cell88_failures.append(
        "AUTHOR_READY_COUNT_INVALID"
    )


if not cell88_author_ready_identity_valid:
    cell88_failures.append(
        "AUTHOR_READY_IDENTITY_INVALID"
    )


# ============================================================
# 14) Preserve Global Authority Policy
# ============================================================

cell88_authority_policy = copy.deepcopy(
    cell87_persisted_registry.get(
        "workflow_policy",
        {},
    ).get(
        "authority_evidence_policy",
        {},
    )
)


cell88_authority_policy_valid = all([
    cell88_authority_policy.get(
        "primary_authority"
    )
    ==
    "Dr. Komeil Roudi",

    cell88_authority_policy.get(
        "secondary_to_primary_promotion"
    )
    ==
    "FORBIDDEN",
])


if not cell88_authority_policy_valid:
    cell88_failures.append(
        "GLOBAL_AUTHORITY_POLICY_INVALID"
    )


# ============================================================
# 15) Build next active workspace
# ============================================================

cell88_next_active_workspace = {
    "pilot_case_id":
        CELL88_NEXT_PILOT_ID,

    "golden_case_id":
        CELL88_NEXT_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL88_NEXT_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL88_NEXT_TARGET_EVIDENCE_ID,

    "status":
        "ACTIVE_HUMAN_REVIEW",

    "workflow_state":
        "QUESTION_DRAFT_PENDING",

    "stratum":
        CELL88_NEXT_STRATUM,

    "task_family":
        CELL88_NEXT_TASK_FAMILY,

    "question_archetype":
        CELL88_NEXT_ARCHETYPE,

    "authority_expectation":
        CELL88_NEXT_AUTHORITY_EXPECTATION,

    "authority_policy":
        copy.deepcopy(
            cell88_authority_policy
        ),

    "source_evidence":
        copy.deepcopy(
            cell88_next_target_text_parts
        ),

    "combined_target_text":
        cell88_next_combined_target_text,

    "question_review":
        None,

    "expected_answer_review":
        None,

    "citation_review":
        None,

    "author_final_review": {
        "status":
            "NOT_STARTED",

        "decision":
            None,

        "notes":
            None,
    },
}


# ============================================================
# 16) Continuation State
# ============================================================

cell88_continuation_state_core = {
    "next_pilot_id":
        CELL88_NEXT_PILOT_ID,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell88_remaining_pilot_ids
        ),

    "remaining_count":
        4,

    "active_pilot_status":
        "QUESTION_DRAFT_PENDING",

    "active_target_evidence_id":
        CELL88_NEXT_TARGET_EVIDENCE_ID,

    "last_completed_pilot_id":
        CELL88_COMPLETED_PILOT_ID,

    "last_completion_type":
        "MIXED_AUTHORITY_CONTEXT_PILOT_COMPLETE",

    "authority_policy_active":
        True,
}


cell88_continuation_state_sha256 = (
    cell10_stable_sha(
        cell88_continuation_state_core
    )
)


cell88_continuation_state = {
    **copy.deepcopy(
        cell88_continuation_state_core
    ),

    "continuation_state_sha256":
        cell88_continuation_state_sha256,
}


# ============================================================
# 17) Persist Registry
# ============================================================

cell88_persistent_registry_payload = copy.deepcopy(
    cell87_persisted_registry
)


cell88_persistent_registry_payload[
    "completed_human_review_cases"
] = copy.deepcopy(
    cell88_completed_human_review_cases
)


cell88_persistent_registry_payload[
    "author_review"
] = copy.deepcopy(
    cell88_author_review
)


cell88_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell88_next_active_workspace
)


cell88_persistent_registry_payload[
    "remaining_pilot_count"
] = 4


cell88_persistent_registry_payload[
    "continuation_state"
] = copy.deepcopy(
    cell88_continuation_state
)


cell88_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell88_persistent_registry_payload
    )
)


# ============================================================
# 18) Read-back validation
# ============================================================

cell88_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell88_persisted_active_case = (
    cell88_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell88_persisted_completed_cases = (
    cell88_persisted_registry.get(
        "completed_human_review_cases",
        [],
    )
)


cell88_persisted_author_ready_cases = (
    cell88_persisted_registry.get(
        "author_review",
        {},
    ).get(
        "cases_ready_for_review",
        [],
    )
)


cell88_persisted_authority_policy = (
    cell88_persisted_registry.get(
        "workflow_policy",
        {},
    ).get(
        "authority_evidence_policy",
        {},
    )
)


cell88_persistence_valid = all([
    isinstance(
        cell88_persisted_completed_cases,
        list,
    ),

    len(
        cell88_persisted_completed_cases
    )
    ==
    18,

    isinstance(
        cell88_persisted_author_ready_cases,
        list,
    ),

    len(
        cell88_persisted_author_ready_cases
    )
    ==
    18,

    cell88_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    4,

    cell88_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL88_NEXT_PILOT_ID,

    cell88_persisted_active_case.get(
        "workflow_state"
    )
    ==
    "QUESTION_DRAFT_PENDING",

    cell88_persisted_registry.get(
        "continuation_state",
        {},
    ).get(
        "continuation_state_sha256"
    )
    ==
    cell88_continuation_state_sha256,

    cell88_persisted_authority_policy.get(
        "primary_authority"
    )
    ==
    "Dr. Komeil Roudi",

    cell88_persisted_authority_policy.get(
        "secondary_to_primary_promotion"
    )
    ==
    "FORBIDDEN",

    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell88_persisted_completed_cases
        if isinstance(
            case,
            dict,
        )
    }
    ==
    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell88_persisted_author_ready_cases
        if isinstance(
            case,
            dict,
        )
    },
])


if not cell88_persistence_valid:
    cell88_failures.append(
        "REGISTRY_READBACK_INVALID"
    )


# ============================================================
# 19) Safety
# ============================================================

cell88_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell88_full_572_blocked = (
    cell87_full_572_blocked
    is True
)


cell88_no_commit_push = True


# ============================================================
# 20) Internal Gates
# ============================================================

cell88_gate_checks = {
    "no_failures":
        len(
            cell88_failures
        )
        ==
        0,

    "citation_accept_valid":
        cell88_citation_accept_valid,

    "queue_advance_valid":
        cell88_queue_advance_valid,

    "next_identity_valid":
        cell88_next_identity_valid,

    "next_evidence_available":
        cell88_next_evidence_available,

    "next_session_valid":
        cell88_next_session_valid,

    "completed_case_count_valid":
        cell88_completed_case_count_valid,

    "author_ready_count_valid":
        cell88_author_ready_count_valid,

    "author_ready_identity_valid":
        cell88_author_ready_identity_valid,

    "authority_policy_valid":
        cell88_authority_policy_valid,

    "persistence_valid":
        cell88_persistence_valid,

    "base_seed_unchanged":
        cell88_base_seed_unchanged,

    "full_572_blocked":
        cell88_full_572_blocked,

    "no_commit_push":
        cell88_no_commit_push,
}


cell88_gate = all(
    cell88_gate_checks.values()
)


# ============================================================
# 21) Diagnostics
# ============================================================

print(
    "=== PILOT #19 COMPLETE + QUEUE ADVANCE + NEXT PILOT ==="
)


print(
    "\n=== COMPLETED PILOT ==="
)

print(
    "Pilot:",
    CELL88_COMPLETED_PILOT_ID,
)

print(
    "Target:",
    CELL87_TARGET_EVIDENCE_ID,
)

print(
    "Retarget used:",
    False,
)

print(
    "Question:",
    "ACCEPT",
)

print(
    "Expected Answer:",
    "ACCEPT",
)

print(
    "Citation:",
    "ACCEPT",
)

print(
    "Final Citation Mapping:",
    cell87_citation_mapping_id,
)

print(
    "Final Engine State:",
    cell88_complete_pending_session.get(
        "current_state"
    ),
)


print(
    "\n=== AUTHORITY RESULT ==="
)

print(
    "Primary Authority:",
    "Dr. Komeil Roudi",
)

print(
    "Non-Primary Context:",
    "مجری",
)

print(
    "Non-Primary used as Answer Evidence:",
    False,
)

print(
    "File-level Authority Flattening:",
    False,
)

print(
    "Secondary-to-primary promotion:",
    "FORBIDDEN",
)

print(
    "Raw Source mutated:",
    False,
)

print(
    "Parser mutated:",
    False,
)


print(
    "\n=== QUEUE ADVANCE ==="
)

print(
    "Before:",
    len(
        cell88_previous_remaining_pilot_ids
    ),
)

print(
    "After:",
    len(
        cell88_remaining_pilot_ids
    ),
)

print(
    "Queue advanced exactly once:",
    cell88_queue_advance_valid,
)


print(
    "\n=== HUMAN REVIEW REGISTRY ==="
)

print(
    "Completed Human Review cases:",
    len(
        cell88_completed_human_review_cases
    ),
)

print(
    "Cases ready for Author Review:",
    len(
        cell88_author_ready_cases
    ),
)

print(
    "Author Final Review:",
    "NOT_STARTED",
)

print(
    "Persistence valid:",
    cell88_persistence_valid,
)


print(
    "\n=== NEXT PILOT ==="
)

print(
    "Pilot:",
    CELL88_NEXT_PILOT_ID,
)

print(
    "Golden Case:",
    CELL88_NEXT_GOLDEN_CASE_ID,
)

print(
    "Task Draft:",
    CELL88_NEXT_TASK_DRAFT_ID,
)

print(
    "Target Evidence:",
    CELL88_NEXT_TARGET_EVIDENCE_ID,
)

print(
    "Stratum:",
    CELL88_NEXT_STRATUM,
)

print(
    "Task Family:",
    CELL88_NEXT_TASK_FAMILY,
)

print(
    "Question Archetype:",
    CELL88_NEXT_ARCHETYPE,
)

print(
    "Authority Expectation:",
    CELL88_NEXT_AUTHORITY_EXPECTATION,
)

print(
    "Workflow:",
    cell88_next_pilot_session.get(
        "current_state"
    ),
)


print(
    "\n=== NEXT SOURCE EVIDENCE ==="
)

print(
    cell88_next_combined_target_text
)


print(
    "\n=== CONTINUATION STATE ==="
)

print(
    "Remaining Pilots:",
    4,
)

print(
    "Continuation SHA256:",
    cell88_continuation_state_sha256,
)


print(
    "\n=== ALL INTERNAL GATES ==="
)

for name, status in cell88_gate_checks.items():

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== SAFETY ==="
)

print(
    "Base Seed unchanged:",
    cell88_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell88_full_572_blocked,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell88_failures
)


print(
    "\nCODE CELL 88 GATE:",
    (
        "PASS"
        if cell88_gate
        else "FAIL"
    ),
)


assert cell88_gate, (
    "Pilot #19 Citation ACCEPT / completion / queue advance / "
    "next Pilot preparation failed. Repair this SAME Code Cell 88."
)

=== PILOT #19 COMPLETE + QUEUE ADVANCE + NEXT PILOT ===

=== COMPLETED PILOT ===
Pilot: question-pilot:6ba9960127f4b21ec2dc1ab2547ae3b5
Target: target-evidence:aabbc6aed0c573b34019e6212cd05cad
Retarget used: False
Question: ACCEPT
Expected Answer: ACCEPT
Citation: ACCEPT
Final Citation Mapping: citation-mapping:4815ef85e71551e6f2d9fe34dd0386b1
Final Engine State: PILOT_COMPLETE_PENDING_QUEUE_ADVANCE

=== AUTHORITY RESULT ===
Primary Authority: Dr. Komeil Roudi
Non-Primary Context: مجری
Non-Primary used as Answer Evidence: False
File-level Authority Flattening: False
Secondary-to-primary promotion: FORBIDDEN
Raw Source mutated: False
Parser mutated: False

=== QUEUE ADVANCE ===
Before: 5
After: 4
Queue advanced exactly once: True

=== HUMAN REVIEW REGISTRY ===
Completed Human Review cases: 18
Cases ready for Author Review: 18
Author Final Review: NOT_STARTED
Persistence valid: True

=== NEXT PILOT ===
Pilot: question-pilot:fe7229122fe65089a77386df155a61d4
Golden Case: golden-case:800158

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 89 — Pilot #20 Long Utterance Authority / Embedded Quote Diagnostic

Pilot فعلی از نوع:

`LONG_UTTERANCE_RETRIEVAL`

است و باید Source Attribution آن بدون تغییر حفظ شود.

متن شامل دو لایه است:

- توضیح اصلی درباره قرض، ناتوانی مالی و اعتیاد به قرض؛
- یک خاطره و نقل‌قول از فردی دیگر که داخل صحبت گوینده اصلی آمده است.

پیش از ساخت سؤال بررسی می‌کنیم:

- Utterance اصلی متعلق به چه Speakerای است؛
- آیا Speaker اصلی Dr. Komeil Roudi است؛
- آیا کل Target یک Utterance واحد است؛
- آیا نقل‌قول داخلی صرفاً Illustration / Context است؛
- آیا جواب را می‌توان فقط از ادعاها و نتیجه‌گیری خود دکتر ساخت؛
- آیا Target بدون Retarget قابل استفاده است.

این Cell فقط Diagnostic است و هیچ داده‌ای را تغییر نمی‌دهد.

</div>

In [89]:
# Code Cell 89

cell89_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell88_gate") is not True:
    cell89_failures.append(
        "CELL88_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell88_next_target_text_parts"
    ),
    list,
):
    cell89_failures.append(
        "NEXT_TARGET_PARTS_MISSING"
    )


# ============================================================
# 2) Pilot identity
# ============================================================

CELL89_PILOT_ID = (
    CELL88_NEXT_PILOT_ID
)


CELL89_TARGET_EVIDENCE_ID = (
    CELL88_NEXT_TARGET_EVIDENCE_ID
)


CELL89_TASK_FAMILY = (
    CELL88_NEXT_TASK_FAMILY
)


CELL89_AUTHORITY_EXPECTATION = (
    CELL88_NEXT_AUTHORITY_EXPECTATION
)


# ============================================================
# 3) Inspect Target Parts
# ============================================================

cell89_part_diagnostics = []


for index, part in enumerate(
    cell88_next_target_text_parts
):

    if isinstance(
        part,
        dict,
    ):

        speaker_state = (
            part.get(
                "speaker_state"
            )
            or
            {}
        )


        attribution = (
            part.get(
                "attribution"
            )
            or
            {}
        )


        locator = (
            part.get(
                "locator"
            )
            or
            {}
        )


        text = (
            part.get(
                "text"
            )
            or
            part.get(
                "target_text"
            )
            or
            part.get(
                "content"
            )
            or
            ""
        )


        cell89_part_diagnostics.append({
            "index":
                index,

            "document_id":
                part.get(
                    "document_id"
                ),

            "role":
                part.get(
                    "role"
                ),

            "speaker_base_status":
                speaker_state.get(
                    "base_status"
                ),

            "speaker_base_raw":
                speaker_state.get(
                    "base_raw"
                ),

            "speaker_effective_label":
                speaker_state.get(
                    "effective_label"
                ),

            "speaker_effective_canonical":
                speaker_state.get(
                    "effective_canonical"
                ),

            "speaker_effective_primary":
                speaker_state.get(
                    "effective_primary"
                ),

            "attribution_speaker":
                attribution.get(
                    "speaker"
                ),

            "source_line_start":
                locator.get(
                    "source_line_start"
                ),

            "source_line_end":
                locator.get(
                    "source_line_end"
                ),

            "text":
                text,
        })


    else:

        cell89_part_diagnostics.append({
            "index":
                index,

            "document_id":
                None,

            "role":
                None,

            "speaker_base_status":
                None,

            "speaker_base_raw":
                None,

            "speaker_effective_label":
                None,

            "speaker_effective_canonical":
                None,

            "speaker_effective_primary":
                None,

            "attribution_speaker":
                None,

            "source_line_start":
                None,

            "source_line_end":
                None,

            "text":
                str(
                    part
                ),
        })


# ============================================================
# 4) Resolve authority
# ============================================================

cell89_resolved_parts = []


for row in cell89_part_diagnostics:

    resolved_speaker = (
        row.get(
            "speaker_effective_canonical"
        )
        or
        row.get(
            "speaker_effective_label"
        )
        or
        row.get(
            "speaker_base_raw"
        )
        or
        row.get(
            "attribution_speaker"
        )
    )


    is_primary = (
        row.get(
            "speaker_effective_primary"
        )
        is True
        or
        resolved_speaker
        ==
        "Dr. Komeil Roudi"
    )


    is_unresolved = (
        resolved_speaker
        is None
        or
        row.get(
            "speaker_base_status"
        )
        ==
        "UNRESOLVED"
    )


    cell89_resolved_parts.append({
        **copy.deepcopy(
            row
        ),

        "resolved_speaker":
            resolved_speaker,

        "is_primary":
            is_primary,

        "is_unresolved":
            is_unresolved,
    })


# ============================================================
# 5) Structural validation
# ============================================================

cell89_exact_one_part = (
    len(
        cell89_resolved_parts
    )
    ==
    1
)


cell89_primary_parts = [
    row
    for row
    in cell89_resolved_parts
    if row[
        "is_primary"
    ]
]


cell89_unresolved_parts = [
    row
    for row
    in cell89_resolved_parts
    if row[
        "is_unresolved"
    ]
]


cell89_primary_valid = all([
    cell89_exact_one_part,

    len(
        cell89_primary_parts
    )
    ==
    1,

    len(
        cell89_unresolved_parts
    )
    ==
    0,

    cell89_primary_parts[
        0
    ].get(
        "resolved_speaker"
    )
    ==
    "Dr. Komeil Roudi",
])


# ============================================================
# 6) Long-Utterance requirements
# ============================================================

cell89_expected_task_family = (
    CELL89_TASK_FAMILY
    ==
    "LONG_UTTERANCE_RETRIEVAL"
)


cell89_expected_authority_rule = (
    CELL89_AUTHORITY_EXPECTATION
    ==
    "PRESERVE_SOURCE_ATTRIBUTION"
)


cell89_text = (
    cell89_resolved_parts[
        0
    ].get(
        "text",
        "",
    )
    if cell89_resolved_parts
    else
    ""
)


cell89_long_enough = (
    len(
        cell89_text
    )
    >
    500
)


# ============================================================
# 7) Embedded quote / anecdote detection
# ============================================================

cell89_embedded_quote_signals = {
    "old_man_story":
        (
            "یکی از این پیرمردهای قدیمی"
            in
            cell89_text
        ),

    "quoted_speech":
        (
            "گفت:"
            in
            cell89_text
            or
            "گفتش که"
            in
            cell89_text
        ),

    "speaker_distances_quote":
        (
            "حالا این البته اون حرف اون پیرمرد بود"
            in
            cell89_text
        ),
}


cell89_embedded_quote_present = all(
    cell89_embedded_quote_signals.values()
)


# ============================================================
# 8) Dr. Roudi substantive claims outside quoted anecdote
# ============================================================

def cell89_norm(
    text,
):

    if not isinstance(
        text,
        str,
    ):
        return ""


    value = (
        text
        .replace(
            "ي",
            "ی",
        )
        .replace(
            "ك",
            "ک",
        )
        .replace(
            "\u200c",
            "",
        )
        .replace(
            "\ufeff",
            "",
        )
    )


    value = re.sub(
        r"\s+",
        " ",
        value,
    )


    return value.strip()


cell89_surface = cell89_norm(
    cell89_text
)


cell89_primary_claims = {
    "borrowing_as_financial_inability":
        (
            "قرض اگر از ادبیات مالی بخوایم بهش نگاه بکنیم"
            in
            cell89_surface
            and
            "یه جور ناتوانی مالیه"
            in
            cell89_surface
        ),

    "expense_management":
        (
            "نتونستیم هزینههامون رو درست مدیریت بکنیم"
            in
            cell89_surface
        ),

    "future_prediction":
        (
            "آینده رو درست پیشبینی بکنیم"
            in
            cell89_surface
        ),

    "debt_risk":
        (
            "لبه پرتگاه"
            in
            cell89_surface
        ),

    "loan_dependency":
        (
            "زندگیمون رو وابسته به قرض بکنیم"
            in
            cell89_surface
        ),

    "loan_addiction":
        (
            "اعتیاد به قرض گرفتن"
            in
            cell89_surface
        ),

    "serious_harm":
        (
            "آسیبهای جدی داره"
            in
            cell89_surface
        ),

    "not_sustainable_foundation":
        (
            "نمیشه آدم روش"
            in
            cell89_surface
            and
            "حساب کرد"
            in
            cell89_surface
        ),
}


cell89_primary_claims_sufficient = all(
    cell89_primary_claims.values()
)


# ============================================================
# 9) Authority Policy
# ============================================================

cell89_authority_policy = copy.deepcopy(
    cell88_persisted_registry.get(
        "workflow_policy",
        {},
    ).get(
        "authority_evidence_policy",
        {},
    )
)


cell89_authority_policy_valid = all([
    cell89_authority_policy.get(
        "primary_authority"
    )
    ==
    "Dr. Komeil Roudi",

    cell89_authority_policy.get(
        "secondary_to_primary_promotion"
    )
    ==
    "FORBIDDEN",
])


if not cell89_authority_policy_valid:
    cell89_failures.append(
        "GLOBAL_AUTHORITY_POLICY_INVALID"
    )


# ============================================================
# 10) Embedded quote authority rule
# ============================================================

cell89_embedded_quote_independent_authority = False


cell89_embedded_quote_role = (
    "ILLUSTRATIVE_CONTEXT_ONLY"
)


cell89_answer_can_exclude_quote = all([
    cell89_primary_valid,
    cell89_primary_claims_sufficient,
    not cell89_embedded_quote_independent_authority,
])


# ============================================================
# 11) Preliminary usability
# ============================================================

cell89_target_preliminarily_usable = all([
    cell89_expected_task_family,
    cell89_expected_authority_rule,
    cell89_primary_valid,
    cell89_long_enough,
    cell89_primary_claims_sufficient,
    cell89_answer_can_exclude_quote,
])


cell89_retarget_likely_required = (
    not cell89_target_preliminarily_usable
)


# ============================================================
# 12) Safety
# ============================================================

cell89_registry_write = False

cell89_queue_advance = False

cell89_human_decision_added = False

cell89_target_changed = False

cell89_citation_changed = False

cell89_raw_source_mutated = False

cell89_parser_mutated = False

cell89_commit_push = False


# ============================================================
# 13) Gate
# ============================================================

cell89_gate_checks = {
    "no_failures":
        len(
            cell89_failures
        )
        ==
        0,

    "expected_task_family":
        cell89_expected_task_family,

    "expected_authority_rule":
        cell89_expected_authority_rule,

    "exact_one_part":
        cell89_exact_one_part,

    "primary_valid":
        cell89_primary_valid,

    "long_enough":
        cell89_long_enough,

    "primary_claims_sufficient":
        cell89_primary_claims_sufficient,

    "authority_policy_valid":
        cell89_authority_policy_valid,

    "answer_can_exclude_quote":
        cell89_answer_can_exclude_quote,
}


cell89_gate = all(
    cell89_gate_checks.values()
)


# ============================================================
# 14) Diagnostics
# ============================================================

print(
    "=== PILOT #20 LONG UTTERANCE AUTHORITY DIAGNOSTIC ==="
)


print(
    "\n=== PILOT ==="
)

print(
    "Pilot:",
    CELL89_PILOT_ID,
)

print(
    "Target Evidence:",
    CELL89_TARGET_EVIDENCE_ID,
)

print(
    "Task Family:",
    CELL89_TASK_FAMILY,
)

print(
    "Authority Expectation:",
    CELL89_AUTHORITY_EXPECTATION,
)


print(
    "\n=== TARGET STRUCTURE ==="
)

print(
    "Part count:",
    len(
        cell89_resolved_parts
    ),
)

print(
    "Primary parts:",
    len(
        cell89_primary_parts
    ),
)

print(
    "Unresolved parts:",
    len(
        cell89_unresolved_parts
    ),
)

print(
    "Long utterance chars:",
    len(
        cell89_text
    ),
)


for row in cell89_resolved_parts:

    print(
        "\n"
        +
        "="
        *
        72
    )

    print(
        "Document:",
        row.get(
            "document_id"
        ),
    )

    print(
        "Base status:",
        row.get(
            "speaker_base_status"
        ),
    )

    print(
        "Base raw:",
        row.get(
            "speaker_base_raw"
        ),
    )

    print(
        "Effective canonical:",
        row.get(
            "speaker_effective_canonical"
        ),
    )

    print(
        "Effective primary:",
        row.get(
            "speaker_effective_primary"
        ),
    )

    print(
        "Resolved speaker:",
        row.get(
            "resolved_speaker"
        ),
    )

    print(
        "Source lines:",
        (
            row.get(
                "source_line_start"
            ),
            row.get(
                "source_line_end"
            ),
        ),
    )


print(
    "\n=== EMBEDDED QUOTE / ANECDOTE ==="
)

for name, status in (
    cell89_embedded_quote_signals.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "Embedded quote present:",
    cell89_embedded_quote_present,
)

print(
    "Embedded quote independent authority:",
    False,
)

print(
    "Embedded quote role:",
    cell89_embedded_quote_role,
)


print(
    "\n=== DR. ROUDI PRIMARY CLAIMS ==="
)

for name, status in (
    cell89_primary_claims.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "Primary claims sufficient:",
    cell89_primary_claims_sufficient,
)


print(
    "\n=== TARGET ASSESSMENT ==="
)

print(
    "Target preliminarily usable:",
    cell89_target_preliminarily_usable,
)

print(
    "Answer can exclude embedded quote:",
    cell89_answer_can_exclude_quote,
)

print(
    "Retarget likely required:",
    cell89_retarget_likely_required,
)

print(
    "IMPORTANT: No Human Question/Answer/Citation decision "
    "has been recorded."
)


print(
    "\n=== ALL INTERNAL GATES ==="
)

for name, status in cell89_gate_checks.items():

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== SAFETY ==="
)

print(
    "Registry write:",
    cell89_registry_write,
)

print(
    "Queue advance:",
    cell89_queue_advance,
)

print(
    "Human Decision added:",
    cell89_human_decision_added,
)

print(
    "Target changed:",
    cell89_target_changed,
)

print(
    "Citation changed:",
    cell89_citation_changed,
)

print(
    "Raw Source mutated:",
    cell89_raw_source_mutated,
)

print(
    "Parser mutated:",
    cell89_parser_mutated,
)

print(
    "Commit / Push:",
    cell89_commit_push,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell89_failures
)


print(
    "\nCODE CELL 89 GATE:",
    (
        "PASS"
        if cell89_gate
        else
        "FAIL"
    ),
)


assert cell89_gate, (
    "Pilot #20 long-utterance authority diagnostic failed. "
    "Repair this SAME Code Cell 89 before drafting the question."
)

=== PILOT #20 LONG UTTERANCE AUTHORITY DIAGNOSTIC ===

=== PILOT ===
Pilot: question-pilot:fe7229122fe65089a77386df155a61d4
Target Evidence: target-evidence:d5f3951a062d8f436caf998e8d71339d
Task Family: LONG_UTTERANCE_RETRIEVAL
Authority Expectation: PRESERVE_SOURCE_ATTRIBUTION

=== TARGET STRUCTURE ===
Part count: 1
Primary parts: 1
Unresolved parts: 0
Long utterance chars: 1804

Document: program:uni-tehran:episode-019:utterance:00004
Base status: EXPLICIT_SOURCE_MARKER
Base raw: استاد
Effective canonical: Dr. Komeil Roudi
Effective primary: True
Resolved speaker: Dr. Komeil Roudi
Source lines: (15, 23)

=== EMBEDDED QUOTE / ANECDOTE ===
old_man_story: True
quoted_speech: True
speaker_distances_quote: True
Embedded quote present: True
Embedded quote independent authority: False
Embedded quote role: ILLUSTRATIVE_CONTEXT_ONLY

=== DR. ROUDI PRIMARY CLAIMS ===
borrowing_as_financial_inability: True
expense_management: True
future_prediction: True
debt_risk: True
loan_dependency: True
lo

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 90 — Pilot #20 Question/Answer ACCEPT + Citation Draft

Human Reviewer سؤال و جواب Pilot #20 را پذیرفت.

در این Pilot:

- کل Target متعلق به Dr. Komeil Roudi و Primary Authority است؛
- خاطره و نقل‌قول پیرمرد داخل صحبت دکتر، فقط `ILLUSTRATIVE_CONTEXT_ONLY` است؛
- هیچ ادعایی از پیرمرد به‌عنوان Evidence مستقل وارد Expected Answer نمی‌شود؛
- Expected Answer فقط از تحلیل خود دکتر درباره قرض، وابستگی به قرض، ناتوانی مالی و آسیب‌های آن ساخته می‌شود؛
- Retarget انجام نمی‌شود.

سؤال پذیرفته‌شده:

**چرا وابسته‌شدن به قرض می‌تواند نشانه ناتوانی مالی باشد و چه خطری برای زندگی مالی ایجاد می‌کند؟**

جواب پذیرفته‌شده:

**چون قرض معمولاً زمانی وارد زندگی می‌شود که فرد نتوانسته هزینه‌ها را درست مدیریت کند یا آینده مالی خود را به‌خوبی پیش‌بینی کند. اگر فرد برای حفظ تعادل دخل‌وخرج به قرض وابسته شود، ممکن است این رفتار به یک عادت یا حتی نوعی اعتیاد به قرض گرفتن تبدیل شود. در این حالت قرض دیگر یک راه‌حل پایدار نیست و می‌تواند فرد را به بدهی‌های بیشتر و آسیب‌های جدی مالی بکشاند.**

بعد از این Cell، Citation در وضعیت Human Review قرار می‌گیرد و Queue همچنان 4 باقی می‌ماند.

</div>

In [90]:
# Code Cell 90

cell90_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell89_gate") is not True:
    cell90_failures.append(
        "CELL89_NOT_PASS"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell90_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Human-accepted Question / Answer
# ============================================================

CELL90_ACCEPTED_QUESTION = (
    "چرا وابسته‌شدن به قرض می‌تواند نشانه ناتوانی مالی باشد "
    "و چه خطری برای زندگی مالی ایجاد می‌کند؟"
)


CELL90_ACCEPTED_ANSWER = (
    "چون قرض معمولاً زمانی وارد زندگی می‌شود که فرد نتوانسته "
    "هزینه‌ها را درست مدیریت کند یا آینده مالی خود را به‌خوبی "
    "پیش‌بینی کند. اگر فرد برای حفظ تعادل دخل‌وخرج به قرض وابسته "
    "شود، ممکن است این رفتار به یک عادت یا حتی نوعی اعتیاد به قرض "
    "گرفتن تبدیل شود. در این حالت قرض دیگر یک راه‌حل پایدار نیست "
    "و می‌تواند فرد را به بدهی‌های بیشتر و آسیب‌های جدی مالی بکشاند."
)


CELL90_PILOT_ID = (
    CELL89_PILOT_ID
)


CELL90_GOLDEN_CASE_ID = (
    CELL88_NEXT_GOLDEN_CASE_ID
)


CELL90_TASK_DRAFT_ID = (
    CELL88_NEXT_TASK_DRAFT_ID
)


CELL90_TARGET_EVIDENCE_ID = (
    CELL89_TARGET_EVIDENCE_ID
)


# ============================================================
# 3) Target validation
# ============================================================

cell90_target_parts = copy.deepcopy(
    cell88_next_target_text_parts
)


cell90_target_valid = all([
    isinstance(
        cell90_target_parts,
        list,
    ),

    len(
        cell90_target_parts
    )
    ==
    1,

    cell89_primary_valid,

    cell89_long_enough,

    cell89_primary_claims_sufficient,

    cell89_answer_can_exclude_quote,

    not cell89_retarget_likely_required,
])


if not cell90_target_valid:
    cell90_failures.append(
        "TARGET_AUTHORITY_STRUCTURE_INVALID"
    )


cell90_primary_part = copy.deepcopy(
    cell90_target_parts[
        0
    ]
)


cell90_primary_text = (
    cell90_primary_part.get(
        "text"
    )
    or
    cell90_primary_part.get(
        "target_text"
    )
    or
    cell90_primary_part.get(
        "content"
    )
    or
    ""
)


# ============================================================
# 4) Embedded quote boundary
# ============================================================

cell90_embedded_quote_policy = {
    "embedded_quote_present":
        cell89_embedded_quote_present,

    "embedded_quote_source":
        "elderly man in narrated anecdote",

    "embedded_quote_role":
        "ILLUSTRATIVE_CONTEXT_ONLY",

    "embedded_quote_independent_authority":
        False,

    "embedded_quote_used_as_expected_answer_evidence":
        False,

    "primary_narrator":
        "Dr. Komeil Roudi",

    "primary_narrator_analysis_used_as_answer_evidence":
        True,

    "source_attribution_preserved":
        True,
}


cell90_embedded_quote_boundary_valid = all([
    cell90_embedded_quote_policy[
        "embedded_quote_present"
    ]
    is True,

    cell90_embedded_quote_policy[
        "embedded_quote_independent_authority"
    ]
    is False,

    cell90_embedded_quote_policy[
        "embedded_quote_used_as_expected_answer_evidence"
    ]
    is False,

    cell90_embedded_quote_policy[
        "primary_narrator"
    ]
    ==
    "Dr. Komeil Roudi",
])


if not cell90_embedded_quote_boundary_valid:
    cell90_failures.append(
        "EMBEDDED_QUOTE_BOUNDARY_INVALID"
    )


# ============================================================
# 5) Grounding
# ============================================================

cell90_grounding = copy.deepcopy(
    cell89_primary_claims
)


cell90_grounding_pass = all([
    cell89_primary_claims_sufficient,

    cell90_grounding.get(
        "borrowing_as_financial_inability"
    )
    is True,

    cell90_grounding.get(
        "expense_management"
    )
    is True,

    cell90_grounding.get(
        "future_prediction"
    )
    is True,

    cell90_grounding.get(
        "loan_dependency"
    )
    is True,

    cell90_grounding.get(
        "loan_addiction"
    )
    is True,

    cell90_grounding.get(
        "serious_harm"
    )
    is True,

    cell90_grounding.get(
        "not_sustainable_foundation"
    )
    is True,
])


if not cell90_grounding_pass:
    cell90_failures.append(
        "ANSWER_GROUNDING_FAILED"
    )


# ============================================================
# 6) Start Engine
# ============================================================

cell90_question_pending_session = (
    cell13_build_session(
        cell88_next_pilot_packet
    )
)


cell90_session_start_valid = (
    cell90_question_pending_session.get(
        "current_state"
    )
    ==
    "QUESTION_DRAFT_PENDING"
)


if not cell90_session_start_valid:
    cell90_failures.append(
        "ENGINE_SESSION_START_INVALID"
    )


# ============================================================
# 7) Question Draft + Human ACCEPT
# ============================================================

cell90_question_identity = {
    "identity_version":
        "question_draft_identity_v1",

    "pilot_case_id":
        CELL90_PILOT_ID,

    "target_evidence_id":
        CELL90_TARGET_EVIDENCE_ID,

    "question_text":
        CELL90_ACCEPTED_QUESTION,

    "revision_number":
        1,
}


cell90_question_draft_id = (
    "question-draft:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell90_question_identity
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


cell90_question_record = {
    "record_version":
        "question_draft_record_v1",

    "question_draft_id":
        cell90_question_draft_id,

    "pilot_case_id":
        CELL90_PILOT_ID,

    "golden_case_id":
        CELL90_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL90_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL90_TARGET_EVIDENCE_ID,

    "question_text":
        CELL90_ACCEPTED_QUESTION,

    "revision_number":
        1,

    "human_adjudication_status":
        "PENDING",

    "external_knowledge_used":
        False,
}


cell90_question_review_session = (
    cell13_apply_event(
        cell90_question_pending_session,
        "QUESTION_DRAFT_REGISTERED",
        artifact=
            cell90_question_record,
    )
)


cell90_answer_pending_session = (
    cell13_apply_event(
        cell90_question_review_session,
        "QUESTION_ACCEPTED",
        human_note=
            "Human Reviewer accepted the question.",
    )
)


cell90_question_accept_valid = (
    cell90_answer_pending_session.get(
        "current_state"
    )
    ==
    "EXPECTED_ANSWER_DRAFT_PENDING"
)


if not cell90_question_accept_valid:
    cell90_failures.append(
        "QUESTION_ACCEPT_TRANSITION_INVALID"
    )


# ============================================================
# 8) Expected Answer + Human ACCEPT
# ============================================================

cell90_answer_identity = {
    "identity_version":
        "expected_answer_identity_v1",

    "pilot_case_id":
        CELL90_PILOT_ID,

    "question_draft_id":
        cell90_question_draft_id,

    "target_evidence_id":
        CELL90_TARGET_EVIDENCE_ID,

    "answer_text":
        CELL90_ACCEPTED_ANSWER,

    "revision_number":
        1,
}


cell90_expected_answer_draft_id = (
    "expected-answer-draft:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell90_answer_identity
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


cell90_expected_answer_record = {
    "record_version":
        "expected_answer_draft_record_v1",

    "expected_answer_draft_id":
        cell90_expected_answer_draft_id,

    "pilot_case_id":
        CELL90_PILOT_ID,

    "golden_case_id":
        CELL90_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL90_TASK_DRAFT_ID,

    "question_draft_id":
        cell90_question_draft_id,

    "target_evidence_id":
        CELL90_TARGET_EVIDENCE_ID,

    "expected_answer_text":
        CELL90_ACCEPTED_ANSWER,

    "revision_number":
        1,

    "human_adjudication_status":
        "PENDING",

    "external_knowledge_used":
        False,

    "embedded_quote_used_as_evidence":
        False,
}


cell90_answer_review_session = (
    cell13_apply_event(
        cell90_answer_pending_session,
        "EXPECTED_ANSWER_REGISTERED",
        artifact=
            cell90_expected_answer_record,
    )
)


cell90_citation_pending_session = (
    cell13_apply_event(
        cell90_answer_review_session,
        "EXPECTED_ANSWER_ACCEPTED",
        human_note=
            "Human Reviewer accepted the expected answer.",
    )
)


cell90_answer_accept_valid = (
    cell90_citation_pending_session.get(
        "current_state"
    )
    ==
    "CITATION_MAPPING_PENDING"
)


if not cell90_answer_accept_valid:
    cell90_failures.append(
        "ANSWER_ACCEPT_TRANSITION_INVALID"
    )


# ============================================================
# 9) Citation Entry
# ============================================================

cell90_primary_entry = copy.deepcopy(
    cell90_primary_part
)


cell90_primary_entry.update({
    "citation_index":
        0,

    "authority_role":
        "PRIMARY_ANSWER_EVIDENCE",

    "answer_support":
        True,

    "use_in_expected_answer_grounding":
        True,

    "speaker_canonical":
        "Dr. Komeil Roudi",

    "speaker_primary":
        True,

    "embedded_quote_policy":
        copy.deepcopy(
            cell90_embedded_quote_policy
        ),

    "embedded_quote_independent_answer_support":
        False,

    "external_locator_lookup_used":
        False,
})


cell90_citation_entries = [
    cell90_primary_entry
]


cell90_primary_answer_entries = [
    entry
    for entry
    in cell90_citation_entries
    if entry.get(
        "answer_support"
    )
    is True
]


cell90_citation_boundary_valid = all([
    len(
        cell90_citation_entries
    )
    ==
    1,

    len(
        cell90_primary_answer_entries
    )
    ==
    1,

    cell90_primary_entry.get(
        "speaker_canonical"
    )
    ==
    "Dr. Komeil Roudi",

    cell90_primary_entry.get(
        "speaker_primary"
    )
    is True,

    cell90_primary_entry.get(
        "embedded_quote_independent_answer_support"
    )
    is False,

    cell90_primary_entry.get(
        "embedded_quote_policy",
        {},
    ).get(
        "embedded_quote_role"
    )
    ==
    "ILLUSTRATIVE_CONTEXT_ONLY",
])


if not cell90_citation_boundary_valid:
    cell90_failures.append(
        "CITATION_AUTHORITY_BOUNDARY_INVALID"
    )


# ============================================================
# 10) Locator integrity
# ============================================================

cell90_no_locator_fabrication = (
    cell90_primary_entry.get(
        "locator"
    )
    ==
    cell90_primary_part.get(
        "locator"
    )
)


if not cell90_no_locator_fabrication:
    cell90_failures.append(
        "LOCATOR_INTEGRITY_FAILED"
    )


# ============================================================
# 11) Citation Mapping
# ============================================================

cell90_citation_identity = {
    "identity_version":
        "citation_mapping_identity_v1_long_utterance",

    "pilot_case_id":
        CELL90_PILOT_ID,

    "question_draft_id":
        cell90_question_draft_id,

    "expected_answer_draft_id":
        cell90_expected_answer_draft_id,

    "target_evidence_id":
        CELL90_TARGET_EVIDENCE_ID,

    "citation_entries":
        copy.deepcopy(
            cell90_citation_entries
        ),

    "embedded_quote_policy":
        copy.deepcopy(
            cell90_embedded_quote_policy
        ),
}


cell90_citation_mapping_id = (
    "citation-mapping:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell90_citation_identity
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


cell90_citation_mapping_record = {
    "record_version":
        "citation_mapping_record_v1_long_utterance",

    "citation_mapping_id":
        cell90_citation_mapping_id,

    "pilot_case_id":
        CELL90_PILOT_ID,

    "golden_case_id":
        CELL90_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL90_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL90_TARGET_EVIDENCE_ID,

    "question_draft_id":
        cell90_question_draft_id,

    "expected_answer_draft_id":
        cell90_expected_answer_draft_id,

    "citation_entries":
        copy.deepcopy(
            cell90_citation_entries
        ),

    "primary_answer_authority":
        "Dr. Komeil Roudi",

    "source_attribution_preserved":
        True,

    "embedded_quote_present":
        True,

    "embedded_quote_role":
        "ILLUSTRATIVE_CONTEXT_ONLY",

    "embedded_quote_used_as_answer_evidence":
        False,

    "retarget_used":
        False,

    "raw_source_mutated":
        False,

    "parser_output_mutated":
        False,

    "external_knowledge_used":
        False,

    "human_adjudication_status":
        "PENDING",
}


cell90_current_session = (
    cell13_apply_event(
        cell90_citation_pending_session,
        "CITATION_MAPPING_REGISTERED",
        artifact=
            cell90_citation_mapping_record,
    )
)


cell90_citation_registered_valid = (
    cell90_current_session.get(
        "current_state"
    )
    ==
    "CITATION_HUMAN_REVIEW_PENDING"
)


if not cell90_citation_registered_valid:
    cell90_failures.append(
        "CITATION_REGISTRATION_INVALID"
    )


# ============================================================
# 12) Active Workspace
# ============================================================

cell90_active_workspace = copy.deepcopy(
    cell88_persisted_active_case
)


cell90_active_workspace[
    "workflow_state"
] = "CITATION_HUMAN_REVIEW_PENDING"


cell90_active_workspace[
    "question_review"
] = {
    "question_draft_id":
        cell90_question_draft_id,

    "question_text":
        CELL90_ACCEPTED_QUESTION,

    "human_decision":
        "ACCEPT",
}


cell90_active_workspace[
    "expected_answer_review"
] = {
    "expected_answer_draft_id":
        cell90_expected_answer_draft_id,

    "expected_answer_text":
        CELL90_ACCEPTED_ANSWER,

    "human_decision":
        "ACCEPT",

    "embedded_quote_used_as_evidence":
        False,
}


cell90_active_workspace[
    "citation_review"
] = {
    "citation_mapping_id":
        cell90_citation_mapping_id,

    "citation_mapping_record":
        copy.deepcopy(
            cell90_citation_mapping_record
        ),

    "human_decision":
        "PENDING",

    "primary_answer_authority":
        "Dr. Komeil Roudi",

    "embedded_quote_role":
        "ILLUSTRATIVE_CONTEXT_ONLY",

    "embedded_quote_used_as_answer_evidence":
        False,
}


# ============================================================
# 13) Continuation state
# ============================================================

cell90_continuation_core = {
    "next_pilot_id":
        CELL90_PILOT_ID,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell88_remaining_pilot_ids
        ),

    "remaining_count":
        4,

    "active_pilot_status":
        "CITATION_HUMAN_REVIEW_PENDING",

    "active_target_evidence_id":
        CELL90_TARGET_EVIDENCE_ID,

    "active_citation_mapping_id":
        cell90_citation_mapping_id,

    "retarget_used":
        False,

    "embedded_quote_excluded_as_independent_authority":
        True,

    "authority_policy_active":
        True,

    "last_completed_pilot_id":
        cell88_continuation_state.get(
            "last_completed_pilot_id"
        ),
}


cell90_continuation_state_sha256 = (
    cell10_stable_sha(
        cell90_continuation_core
    )
)


cell90_continuation_state = {
    **copy.deepcopy(
        cell90_continuation_core
    ),

    "continuation_state_sha256":
        cell90_continuation_state_sha256,
}


# ============================================================
# 14) Persist
# ============================================================

cell90_persistent_registry_payload = copy.deepcopy(
    cell88_persisted_registry
)


cell90_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell90_active_workspace
)


cell90_persistent_registry_payload[
    "remaining_pilot_count"
] = 4


cell90_persistent_registry_payload[
    "continuation_state"
] = copy.deepcopy(
    cell90_continuation_state
)


cell90_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell90_persistent_registry_payload
    )
)


# ============================================================
# 15) Read-back Validation
# ============================================================

cell90_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell90_persisted_active_case = (
    cell90_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell90_persistence_valid = all([
    cell90_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL90_PILOT_ID,

    cell90_persisted_active_case.get(
        "workflow_state"
    )
    ==
    "CITATION_HUMAN_REVIEW_PENDING",

    cell90_persisted_active_case.get(
        "question_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell90_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell90_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "embedded_quote_used_as_evidence"
    )
    is False,

    cell90_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "PENDING",

    cell90_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "embedded_quote_used_as_answer_evidence"
    )
    is False,

    cell90_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    4,

    cell90_persisted_registry.get(
        "continuation_state",
        {},
    ).get(
        "continuation_state_sha256"
    )
    ==
    cell90_continuation_state_sha256,
])


if not cell90_persistence_valid:
    cell90_failures.append(
        "REGISTRY_READBACK_INVALID"
    )


# ============================================================
# 16) Safety
# ============================================================

cell90_queue_still_4 = all([
    len(
        cell88_remaining_pilot_ids
    )
    ==
    4,

    cell88_remaining_pilot_ids[
        0
    ]
    ==
    CELL90_PILOT_ID,
])


cell90_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell90_full_572_blocked = (
    cell88_full_572_blocked
    is True
)


cell90_no_commit_push = True


# ============================================================
# 17) Gate
# ============================================================

cell90_gate_checks = {
    "no_failures":
        len(
            cell90_failures
        )
        ==
        0,

    "target_valid":
        cell90_target_valid,

    "embedded_quote_boundary_valid":
        cell90_embedded_quote_boundary_valid,

    "grounding_pass":
        cell90_grounding_pass,

    "session_start_valid":
        cell90_session_start_valid,

    "question_accept_valid":
        cell90_question_accept_valid,

    "answer_accept_valid":
        cell90_answer_accept_valid,

    "citation_boundary_valid":
        cell90_citation_boundary_valid,

    "no_locator_fabrication":
        cell90_no_locator_fabrication,

    "citation_registered_valid":
        cell90_citation_registered_valid,

    "persistence_valid":
        cell90_persistence_valid,

    "queue_still_4":
        cell90_queue_still_4,

    "base_seed_unchanged":
        cell90_base_seed_unchanged,

    "full_572_blocked":
        cell90_full_572_blocked,

    "no_commit_push":
        cell90_no_commit_push,
}


cell90_gate = all(
    cell90_gate_checks.values()
)


# ============================================================
# 18) Diagnostics
# ============================================================

print(
    "=== PILOT #20 QA ACCEPT + CITATION DRAFT ==="
)


print(
    "\n=== AUTHORITY ==="
)

print(
    "Primary Answer Authority:",
    "Dr. Komeil Roudi",
)

print(
    "Source Attribution preserved:",
    True,
)

print(
    "Embedded quote present:",
    True,
)

print(
    "Embedded quote role:",
    "ILLUSTRATIVE_CONTEXT_ONLY",
)

print(
    "Embedded quote independent authority:",
    False,
)

print(
    "Embedded quote used as Answer Evidence:",
    False,
)

print(
    "Retarget used:",
    False,
)


print(
    "\n=== ACCEPTED QUESTION ==="
)

print(
    CELL90_ACCEPTED_QUESTION
)

print(
    "Question Draft ID:",
    cell90_question_draft_id,
)

print(
    "Human Decision:",
    "ACCEPT",
)


print(
    "\n=== ACCEPTED ANSWER ==="
)

print(
    CELL90_ACCEPTED_ANSWER
)

print(
    "Expected Answer Draft ID:",
    cell90_expected_answer_draft_id,
)

print(
    "Human Decision:",
    "ACCEPT",
)


print(
    "\n=== GROUNDING ==="
)

for name, status in cell90_grounding.items():

    print(
        f"{name}:",
        status,
    )


print(
    "Grounding PASS:",
    cell90_grounding_pass,
)


print(
    "\n=== CITATION DRAFT ==="
)

print(
    "Citation Mapping ID:",
    cell90_citation_mapping_id,
)

print(
    "Citation entries:",
    len(
        cell90_citation_entries
    ),
)

print(
    "Primary Answer Evidence entries:",
    len(
        cell90_primary_answer_entries
    ),
)

print(
    "Embedded quote independent Answer Support:",
    False,
)

print(
    "Citation boundary valid:",
    cell90_citation_boundary_valid,
)

print(
    "No locator fabrication:",
    cell90_no_locator_fabrication,
)


print(
    "\n=== WORKFLOW ==="
)

print(
    "Current State:",
    cell90_current_session.get(
        "current_state"
    ),
)

print(
    "Question:",
    "ACCEPT",
)

print(
    "Expected Answer:",
    "ACCEPT",
)

print(
    "Citation:",
    "PENDING",
)


print(
    "\n=== ALL INTERNAL GATES ==="
)

for name, status in cell90_gate_checks.items():

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== QUEUE / SAFETY ==="
)

print(
    "Queue still 4:",
    cell90_queue_still_4,
)

print(
    "Persistence valid:",
    cell90_persistence_valid,
)

print(
    "Base Seed unchanged:",
    cell90_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell90_full_572_blocked,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell90_failures
)


print(
    "\nCODE CELL 90 GATE:",
    (
        "PASS"
        if cell90_gate
        else "FAIL"
    ),
)


assert cell90_gate, (
    "Pilot #20 Question+Answer ACCEPT / Citation Draft failed. "
    "Repair this SAME Code Cell 90."
)

=== PILOT #20 QA ACCEPT + CITATION DRAFT ===

=== AUTHORITY ===
Primary Answer Authority: Dr. Komeil Roudi
Source Attribution preserved: True
Embedded quote present: True
Embedded quote role: ILLUSTRATIVE_CONTEXT_ONLY
Embedded quote independent authority: False
Embedded quote used as Answer Evidence: False
Retarget used: False

=== ACCEPTED QUESTION ===
چرا وابسته‌شدن به قرض می‌تواند نشانه ناتوانی مالی باشد و چه خطری برای زندگی مالی ایجاد می‌کند؟
Question Draft ID: question-draft:d4c2a822c94c58e69829ea29dcc87ee7
Human Decision: ACCEPT

=== ACCEPTED ANSWER ===
چون قرض معمولاً زمانی وارد زندگی می‌شود که فرد نتوانسته هزینه‌ها را درست مدیریت کند یا آینده مالی خود را به‌خوبی پیش‌بینی کند. اگر فرد برای حفظ تعادل دخل‌وخرج به قرض وابسته شود، ممکن است این رفتار به یک عادت یا حتی نوعی اعتیاد به قرض گرفتن تبدیل شود. در این حالت قرض دیگر یک راه‌حل پایدار نیست و می‌تواند فرد را به بدهی‌های بیشتر و آسیب‌های جدی مالی بکشاند.
Expected Answer Draft ID: expected-answer-draft:e95acbcc8b8e35bb7e91313c361a

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 91 — Citation ACCEPT + Pilot #20 Completion + Queue Advance

Citation نهایی Pilot #20 توسط Human Reviewer پذیرفته شد.

در این مرحله:

- Citation با تصمیم `ACCEPT` ثبت می‌شود؛
- Dr. Komeil Roudi تنها Primary Answer Authority باقی می‌ماند؛
- نقل‌قول پیرمرد فقط `ILLUSTRATIVE_CONTEXT_ONLY` باقی می‌ماند؛
- نقل‌قول داخلی هیچ Answer Authority مستقلی ندارد؛
- Pilot #20 کامل می‌شود؛
- تعداد Caseهای کامل Human Review از 18 به 19 می‌رسد؛
- تعداد Caseهای آماده Author Review از 18 به 19 می‌رسد؛
- Queue دقیقاً از 4 به 3 جلو می‌رود؛
- Pilot بعدی برای Human Review آماده می‌شود؛
- Author Final Review همچنان `NOT_STARTED` است؛
- Full 572 Generation همچنان Blocked باقی می‌ماند.

</div>

In [91]:
# Code Cell 91

cell91_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell90_gate") is not True:
    cell91_failures.append(
        "CELL90_NOT_PASS"
    )


if (
    globals().get(
        "cell90_current_session",
        {},
    ).get(
        "current_state"
    )
    !=
    "CITATION_HUMAN_REVIEW_PENDING"
):
    cell91_failures.append(
        "CITATION_NOT_AT_HUMAN_REVIEW_GATE"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell91_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Human Citation ACCEPT
# ============================================================

CELL91_CITATION_HUMAN_NOTE = (
    "Human Reviewer accepted the final Citation Mapping. "
    "Dr. Komeil Roudi remains the sole primary answer authority. "
    "The embedded elderly-man quotation remains illustrative context only "
    "and is not independent Expected Answer evidence."
)


cell91_complete_pending_session = (
    cell13_apply_event(
        cell90_current_session,
        "CITATION_ACCEPTED",
        human_note=
            CELL91_CITATION_HUMAN_NOTE,
    )
)


cell91_citation_accept_valid = (
    cell91_complete_pending_session.get(
        "current_state"
    )
    ==
    "PILOT_COMPLETE_PENDING_QUEUE_ADVANCE"
)


if not cell91_citation_accept_valid:
    cell91_failures.append(
        "CITATION_ACCEPT_TRANSITION_INVALID"
    )


# ============================================================
# 3) Final Citation Mapping
# ============================================================

cell91_final_citation_mapping_record = copy.deepcopy(
    cell90_citation_mapping_record
)


cell91_final_citation_mapping_record[
    "human_adjudication_status"
] = "ACCEPT"


cell91_final_citation_mapping_record[
    "human_review"
] = {
    "human_decision":
        "ACCEPT",

    "human_note":
        CELL91_CITATION_HUMAN_NOTE,

    "primary_answer_authority":
        "Dr. Komeil Roudi",

    "source_attribution_preserved":
        True,

    "embedded_quote_present":
        True,

    "embedded_quote_role":
        "ILLUSTRATIVE_CONTEXT_ONLY",

    "embedded_quote_independent_authority":
        False,

    "embedded_quote_used_as_answer_evidence":
        False,

    "retarget_used":
        False,

    "raw_source_mutated":
        False,

    "parser_output_mutated":
        False,

    "external_knowledge_used":
        False,
}


# ============================================================
# 4) Complete Human Review case
# ============================================================

cell91_completed_case = copy.deepcopy(
    cell90_active_workspace
)


cell91_completed_case[
    "workflow_state"
] = "PILOT_COMPLETE"


cell91_completed_case[
    "status"
] = "COMPLETED_HUMAN_REVIEW"


cell91_completed_case[
    "final_status"
] = "PILOT_COMPLETE"


cell91_completed_case.setdefault(
    "citation_review",
    {},
)


cell91_completed_case[
    "citation_review"
][
    "human_decision"
] = "ACCEPT"


cell91_completed_case[
    "citation_review"
][
    "human_note"
] = CELL91_CITATION_HUMAN_NOTE


cell91_completed_case[
    "citation_review"
][
    "citation_mapping_record"
] = copy.deepcopy(
    cell91_final_citation_mapping_record
)


cell91_completed_case[
    "completion_summary"
] = {
    "retarget_used":
        False,

    "target_evidence_id":
        CELL90_TARGET_EVIDENCE_ID,

    "question_decision":
        "ACCEPT",

    "expected_answer_decision":
        "ACCEPT",

    "citation_decision":
        "ACCEPT",

    "task_family":
        "LONG_UTTERANCE_RETRIEVAL",

    "primary_answer_authority":
        "Dr. Komeil Roudi",

    "source_attribution_preserved":
        True,

    "embedded_quote_present":
        True,

    "embedded_quote_role":
        "ILLUSTRATIVE_CONTEXT_ONLY",

    "embedded_quote_used_as_answer_evidence":
        False,

    "raw_source_mutated":
        False,

    "parser_output_mutated":
        False,

    "author_final_review":
        "NOT_STARTED",
}


cell91_completed_case[
    "author_final_review"
] = {
    "status":
        "NOT_STARTED",

    "decision":
        None,

    "notes":
        None,
}


# ============================================================
# 5) Queue advance: 4 -> 3
# ============================================================

cell91_previous_remaining_pilot_ids = copy.deepcopy(
    cell88_remaining_pilot_ids
)


CELL91_COMPLETED_PILOT_ID = (
    cell91_previous_remaining_pilot_ids[
        0
    ]
)


cell91_remaining_pilot_ids = copy.deepcopy(
    cell91_previous_remaining_pilot_ids[
        1:
    ]
)


cell91_queue_advance_valid = all([
    len(
        cell91_previous_remaining_pilot_ids
    )
    ==
    4,

    CELL91_COMPLETED_PILOT_ID
    ==
    CELL90_PILOT_ID,

    len(
        cell91_remaining_pilot_ids
    )
    ==
    3,

    CELL91_COMPLETED_PILOT_ID
    not in
    cell91_remaining_pilot_ids,
])


if not cell91_queue_advance_valid:
    cell91_failures.append(
        "QUEUE_ADVANCE_INVALID"
    )


CELL91_NEXT_PILOT_ID = (
    cell91_remaining_pilot_ids[
        0
    ]
)


# ============================================================
# 6) Find next Pilot packet
# ============================================================

cell91_raw_next_packet_candidates = (
    cell58_find_pilot_packet_candidates(
        cell13_seed,
        CELL91_NEXT_PILOT_ID,
    )
)


cell91_packet_candidates_by_sha = {}


for candidate in cell91_raw_next_packet_candidates:

    packet_sha = cell10_stable_sha(
        candidate[
            "packet"
        ]
    )


    previous = (
        cell91_packet_candidates_by_sha.get(
            packet_sha
        )
    )


    if (
        previous is None
        or
        candidate[
            "score"
        ]
        >
        previous[
            "score"
        ]
    ):

        cell91_packet_candidates_by_sha[
            packet_sha
        ] = candidate


cell91_ranked_next_packet_candidates = sorted(
    cell91_packet_candidates_by_sha.values(),
    key=lambda item: (
        -item[
            "score"
        ],
        item[
            "path"
        ],
    ),
)


cell91_next_pilot_packet = (
    copy.deepcopy(
        cell91_ranked_next_packet_candidates[
            0
        ][
            "packet"
        ]
    )
    if cell91_ranked_next_packet_candidates
    else None
)


if not isinstance(
    cell91_next_pilot_packet,
    dict,
):

    cell91_failures.append(
        "NEXT_PILOT_PACKET_NOT_FOUND"
    )


# ============================================================
# 7) Extract next Pilot identities
# ============================================================

cell91_next_pilot_ids_found = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell91_next_pilot_packet,
            "question-pilot:",
        )
    )
)


cell91_next_golden_case_ids = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell91_next_pilot_packet,
            "golden-case:",
        )
    )
)


cell91_next_task_draft_ids = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell91_next_pilot_packet,
            "task-draft:",
        )
    )
)


cell91_next_target_evidence_ids = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell91_next_pilot_packet,
            "target-evidence:",
        )
    )
)


cell91_next_identity_valid = all([
    cell91_next_pilot_ids_found
    ==
    [
        CELL91_NEXT_PILOT_ID
    ],

    len(
        cell91_next_golden_case_ids
    )
    ==
    1,

    len(
        cell91_next_task_draft_ids
    )
    ==
    1,

    len(
        cell91_next_target_evidence_ids
    )
    ==
    1,
])


if not cell91_next_identity_valid:
    cell91_failures.append(
        "NEXT_PILOT_IDENTITY_INVALID"
    )


CELL91_NEXT_GOLDEN_CASE_ID = (
    cell91_next_golden_case_ids[
        0
    ]
    if
    len(
        cell91_next_golden_case_ids
    )
    ==
    1
    else
    None
)


CELL91_NEXT_TASK_DRAFT_ID = (
    cell91_next_task_draft_ids[
        0
    ]
    if
    len(
        cell91_next_task_draft_ids
    )
    ==
    1
    else
    None
)


CELL91_NEXT_TARGET_EVIDENCE_ID = (
    cell91_next_target_evidence_ids[
        0
    ]
    if
    len(
        cell91_next_target_evidence_ids
    )
    ==
    1
    else
    None
)


# ============================================================
# 8) Extract next Target Evidence
# ============================================================

cell91_target_part_candidates = (
    cell58_find_key_values(
        cell91_next_pilot_packet,
        "target_text_parts",
    )
)


cell91_target_parts_by_sha = {}


for parts in cell91_target_part_candidates:

    if not isinstance(
        parts,
        list,
    ):
        continue


    if len(
        parts
    ) == 0:
        continue


    parts_sha = cell10_stable_sha(
        parts
    )


    cell91_target_parts_by_sha[
        parts_sha
    ] = parts


cell91_unique_target_part_sets = list(
    cell91_target_parts_by_sha.values()
)


cell91_next_target_text_parts = (
    copy.deepcopy(
        cell91_unique_target_part_sets[
            0
        ]
    )
    if
    len(
        cell91_unique_target_part_sets
    )
    ==
    1
    else
    None
)


if not isinstance(
    cell91_next_target_text_parts,
    list,
):

    cell91_failures.append(
        "NEXT_TARGET_TEXT_PARTS_NOT_UNIQUE"
    )


# ============================================================
# 9) Build readable next Evidence
# ============================================================

cell91_next_evidence_texts = []


for part in (
    cell91_next_target_text_parts
    or
    []
):

    if isinstance(
        part,
        str,
    ):

        text = part.strip()


    elif isinstance(
        part,
        dict,
    ):

        text = (
            part.get(
                "text"
            )
            or
            part.get(
                "target_text"
            )
            or
            part.get(
                "content"
            )
            or
            ""
        )


        if isinstance(
            text,
            str,
        ):

            text = text.strip()


        else:

            text = ""


    else:

        text = ""


    if text:

        cell91_next_evidence_texts.append(
            text
        )


cell91_next_combined_target_text = (
    "\n\n".join(
        cell91_next_evidence_texts
    )
)


cell91_next_evidence_available = bool(
    cell91_next_combined_target_text.strip()
)


if not cell91_next_evidence_available:
    cell91_failures.append(
        "NEXT_EVIDENCE_EMPTY"
    )


# ============================================================
# 10) Extract next Pilot metadata
# ============================================================

def cell91_unique_scalar_for_key(
    payload,
    key,
):

    values = (
        cell58_find_key_values(
            payload,
            key,
        )
    )


    normalized = []


    for value in values:

        if isinstance(
            value,
            (
                str,
                int,
                float,
                bool,
            ),
        ):

            if value not in normalized:
                normalized.append(
                    value
                )


    if len(
        normalized
    ) == 1:

        return normalized[
            0
        ]


    return None


CELL91_NEXT_STRATUM = (
    cell91_unique_scalar_for_key(
        cell91_next_pilot_packet,
        "stratum",
    )
)


CELL91_NEXT_TASK_FAMILY = (
    cell91_unique_scalar_for_key(
        cell91_next_pilot_packet,
        "task_family",
    )
)


CELL91_NEXT_ARCHETYPE = (
    cell91_unique_scalar_for_key(
        cell91_next_pilot_packet,
        "question_archetype",
    )
)


CELL91_NEXT_AUTHORITY_EXPECTATION = (
    cell91_unique_scalar_for_key(
        cell91_next_pilot_packet,
        "authority_expectation",
    )
)


# ============================================================
# 11) Next Pilot Engine session
# ============================================================

cell91_next_pilot_session = (
    cell13_build_session(
        cell91_next_pilot_packet
    )
)


cell91_next_session_valid = (
    cell91_next_pilot_session.get(
        "current_state"
    )
    ==
    "QUESTION_DRAFT_PENDING"
)


if not cell91_next_session_valid:
    cell91_failures.append(
        "NEXT_SESSION_INVALID"
    )


# ============================================================
# 12) Update completed Human Review cases
# ============================================================

cell91_existing_completed_cases = copy.deepcopy(
    cell90_persisted_registry.get(
        "completed_human_review_cases",
        [],
    )
)


if not isinstance(
    cell91_existing_completed_cases,
    list,
):

    cell91_existing_completed_cases = []

    cell91_failures.append(
        "COMPLETED_CASES_NOT_LIST"
    )


cell91_existing_completed_cases = [
    case
    for case
    in cell91_existing_completed_cases
    if not (
        isinstance(
            case,
            dict,
        )
        and
        case.get(
            "pilot_case_id"
        )
        ==
        CELL91_COMPLETED_PILOT_ID
    )
]


cell91_completed_human_review_cases = (
    cell91_existing_completed_cases
    +
    [
        copy.deepcopy(
            cell91_completed_case
        )
    ]
)


cell91_completed_case_count_valid = (
    len(
        cell91_completed_human_review_cases
    )
    ==
    19
)


if not cell91_completed_case_count_valid:
    cell91_failures.append(
        "COMPLETED_CASE_COUNT_INVALID"
    )


# ============================================================
# 13) Rebuild Author Review queue
# ============================================================

cell91_author_review = copy.deepcopy(
    cell90_persisted_registry.get(
        "author_review",
        {},
    )
)


if not isinstance(
    cell91_author_review,
    dict,
):

    cell91_author_review = {}


cell91_author_ready_cases = copy.deepcopy(
    cell91_completed_human_review_cases
)


cell91_author_review[
    "cases_ready_for_review"
] = copy.deepcopy(
    cell91_author_ready_cases
)


cell91_author_review[
    "case_count"
] = len(
    cell91_author_ready_cases
)


cell91_author_review[
    "final_review_status"
] = "NOT_STARTED"


cell91_author_ready_count_valid = (
    len(
        cell91_author_ready_cases
    )
    ==
    19
)


cell91_author_ready_identity_valid = (
    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell91_author_ready_cases
        if isinstance(
            case,
            dict,
        )
    }
    ==
    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell91_completed_human_review_cases
        if isinstance(
            case,
            dict,
        )
    }
)


if not cell91_author_ready_count_valid:
    cell91_failures.append(
        "AUTHOR_READY_COUNT_INVALID"
    )


if not cell91_author_ready_identity_valid:
    cell91_failures.append(
        "AUTHOR_READY_IDENTITY_INVALID"
    )


# ============================================================
# 14) Preserve Global Authority Policy
# ============================================================

cell91_authority_policy = copy.deepcopy(
    cell90_persisted_registry.get(
        "workflow_policy",
        {},
    ).get(
        "authority_evidence_policy",
        {},
    )
)


cell91_authority_policy_valid = all([
    cell91_authority_policy.get(
        "primary_authority"
    )
    ==
    "Dr. Komeil Roudi",

    cell91_authority_policy.get(
        "secondary_to_primary_promotion"
    )
    ==
    "FORBIDDEN",
])


if not cell91_authority_policy_valid:
    cell91_failures.append(
        "GLOBAL_AUTHORITY_POLICY_INVALID"
    )


# ============================================================
# 15) Build next active workspace
# ============================================================

cell91_next_active_workspace = {
    "pilot_case_id":
        CELL91_NEXT_PILOT_ID,

    "golden_case_id":
        CELL91_NEXT_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL91_NEXT_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL91_NEXT_TARGET_EVIDENCE_ID,

    "status":
        "ACTIVE_HUMAN_REVIEW",

    "workflow_state":
        "QUESTION_DRAFT_PENDING",

    "stratum":
        CELL91_NEXT_STRATUM,

    "task_family":
        CELL91_NEXT_TASK_FAMILY,

    "question_archetype":
        CELL91_NEXT_ARCHETYPE,

    "authority_expectation":
        CELL91_NEXT_AUTHORITY_EXPECTATION,

    "authority_policy":
        copy.deepcopy(
            cell91_authority_policy
        ),

    "source_evidence":
        copy.deepcopy(
            cell91_next_target_text_parts
        ),

    "combined_target_text":
        cell91_next_combined_target_text,

    "question_review":
        None,

    "expected_answer_review":
        None,

    "citation_review":
        None,

    "author_final_review": {
        "status":
            "NOT_STARTED",

        "decision":
            None,

        "notes":
            None,
    },
}


# ============================================================
# 16) Continuation State
# ============================================================

cell91_continuation_state_core = {
    "next_pilot_id":
        CELL91_NEXT_PILOT_ID,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell91_remaining_pilot_ids
        ),

    "remaining_count":
        3,

    "active_pilot_status":
        "QUESTION_DRAFT_PENDING",

    "active_target_evidence_id":
        CELL91_NEXT_TARGET_EVIDENCE_ID,

    "last_completed_pilot_id":
        CELL91_COMPLETED_PILOT_ID,

    "last_completion_type":
        "LONG_UTTERANCE_PRIMARY_AUTHORITY_PILOT_COMPLETE",

    "authority_policy_active":
        True,
}


cell91_continuation_state_sha256 = (
    cell10_stable_sha(
        cell91_continuation_state_core
    )
)


cell91_continuation_state = {
    **copy.deepcopy(
        cell91_continuation_state_core
    ),

    "continuation_state_sha256":
        cell91_continuation_state_sha256,
}


# ============================================================
# 17) Persist Registry
# ============================================================

cell91_persistent_registry_payload = copy.deepcopy(
    cell90_persisted_registry
)


cell91_persistent_registry_payload[
    "completed_human_review_cases"
] = copy.deepcopy(
    cell91_completed_human_review_cases
)


cell91_persistent_registry_payload[
    "author_review"
] = copy.deepcopy(
    cell91_author_review
)


cell91_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell91_next_active_workspace
)


cell91_persistent_registry_payload[
    "remaining_pilot_count"
] = 3


cell91_persistent_registry_payload[
    "continuation_state"
] = copy.deepcopy(
    cell91_continuation_state
)


cell91_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell91_persistent_registry_payload
    )
)


# ============================================================
# 18) Read-back validation
# ============================================================

cell91_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell91_persisted_active_case = (
    cell91_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell91_persisted_completed_cases = (
    cell91_persisted_registry.get(
        "completed_human_review_cases",
        [],
    )
)


cell91_persisted_author_ready_cases = (
    cell91_persisted_registry.get(
        "author_review",
        {},
    ).get(
        "cases_ready_for_review",
        [],
    )
)


cell91_persisted_authority_policy = (
    cell91_persisted_registry.get(
        "workflow_policy",
        {},
    ).get(
        "authority_evidence_policy",
        {},
    )
)


cell91_persistence_valid = all([
    isinstance(
        cell91_persisted_completed_cases,
        list,
    ),

    len(
        cell91_persisted_completed_cases
    )
    ==
    19,

    isinstance(
        cell91_persisted_author_ready_cases,
        list,
    ),

    len(
        cell91_persisted_author_ready_cases
    )
    ==
    19,

    cell91_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    3,

    cell91_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL91_NEXT_PILOT_ID,

    cell91_persisted_active_case.get(
        "workflow_state"
    )
    ==
    "QUESTION_DRAFT_PENDING",

    cell91_persisted_registry.get(
        "continuation_state",
        {},
    ).get(
        "continuation_state_sha256"
    )
    ==
    cell91_continuation_state_sha256,

    cell91_persisted_authority_policy.get(
        "primary_authority"
    )
    ==
    "Dr. Komeil Roudi",

    cell91_persisted_authority_policy.get(
        "secondary_to_primary_promotion"
    )
    ==
    "FORBIDDEN",

    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell91_persisted_completed_cases
        if isinstance(
            case,
            dict,
        )
    }
    ==
    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell91_persisted_author_ready_cases
        if isinstance(
            case,
            dict,
        )
    },
])


if not cell91_persistence_valid:
    cell91_failures.append(
        "REGISTRY_READBACK_INVALID"
    )


# ============================================================
# 19) Safety
# ============================================================

cell91_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell91_full_572_blocked = (
    cell90_full_572_blocked
    is True
)


cell91_no_commit_push = True


# ============================================================
# 20) Internal Gates
# ============================================================

cell91_gate_checks = {
    "no_failures":
        len(
            cell91_failures
        )
        ==
        0,

    "citation_accept_valid":
        cell91_citation_accept_valid,

    "queue_advance_valid":
        cell91_queue_advance_valid,

    "next_identity_valid":
        cell91_next_identity_valid,

    "next_evidence_available":
        cell91_next_evidence_available,

    "next_session_valid":
        cell91_next_session_valid,

    "completed_case_count_valid":
        cell91_completed_case_count_valid,

    "author_ready_count_valid":
        cell91_author_ready_count_valid,

    "author_ready_identity_valid":
        cell91_author_ready_identity_valid,

    "authority_policy_valid":
        cell91_authority_policy_valid,

    "persistence_valid":
        cell91_persistence_valid,

    "base_seed_unchanged":
        cell91_base_seed_unchanged,

    "full_572_blocked":
        cell91_full_572_blocked,

    "no_commit_push":
        cell91_no_commit_push,
}


cell91_gate = all(
    cell91_gate_checks.values()
)


# ============================================================
# 21) Diagnostics
# ============================================================

print(
    "=== PILOT #20 COMPLETE + QUEUE ADVANCE + NEXT PILOT ==="
)


print(
    "\n=== COMPLETED PILOT ==="
)

print(
    "Pilot:",
    CELL91_COMPLETED_PILOT_ID,
)

print(
    "Target:",
    CELL90_TARGET_EVIDENCE_ID,
)

print(
    "Retarget used:",
    False,
)

print(
    "Question:",
    "ACCEPT",
)

print(
    "Expected Answer:",
    "ACCEPT",
)

print(
    "Citation:",
    "ACCEPT",
)

print(
    "Final Citation Mapping:",
    cell90_citation_mapping_id,
)

print(
    "Final Engine State:",
    cell91_complete_pending_session.get(
        "current_state"
    ),
)


print(
    "\n=== AUTHORITY RESULT ==="
)

print(
    "Primary Authority:",
    "Dr. Komeil Roudi",
)

print(
    "Embedded quote present:",
    True,
)

print(
    "Embedded quote role:",
    "ILLUSTRATIVE_CONTEXT_ONLY",
)

print(
    "Embedded quote used as Answer Evidence:",
    False,
)

print(
    "Source Attribution preserved:",
    True,
)

print(
    "Raw Source mutated:",
    False,
)

print(
    "Parser mutated:",
    False,
)


print(
    "\n=== QUEUE ADVANCE ==="
)

print(
    "Before:",
    len(
        cell91_previous_remaining_pilot_ids
    ),
)

print(
    "After:",
    len(
        cell91_remaining_pilot_ids
    ),
)

print(
    "Queue advanced exactly once:",
    cell91_queue_advance_valid,
)


print(
    "\n=== HUMAN REVIEW REGISTRY ==="
)

print(
    "Completed Human Review cases:",
    len(
        cell91_completed_human_review_cases
    ),
)

print(
    "Cases ready for Author Review:",
    len(
        cell91_author_ready_cases
    ),
)

print(
    "Author Final Review:",
    "NOT_STARTED",
)

print(
    "Persistence valid:",
    cell91_persistence_valid,
)


print(
    "\n=== NEXT PILOT ==="
)

print(
    "Pilot:",
    CELL91_NEXT_PILOT_ID,
)

print(
    "Golden Case:",
    CELL91_NEXT_GOLDEN_CASE_ID,
)

print(
    "Task Draft:",
    CELL91_NEXT_TASK_DRAFT_ID,
)

print(
    "Target Evidence:",
    CELL91_NEXT_TARGET_EVIDENCE_ID,
)

print(
    "Stratum:",
    CELL91_NEXT_STRATUM,
)

print(
    "Task Family:",
    CELL91_NEXT_TASK_FAMILY,
)

print(
    "Question Archetype:",
    CELL91_NEXT_ARCHETYPE,
)

print(
    "Authority Expectation:",
    CELL91_NEXT_AUTHORITY_EXPECTATION,
)

print(
    "Workflow:",
    cell91_next_pilot_session.get(
        "current_state"
    ),
)


print(
    "\n=== NEXT SOURCE EVIDENCE ==="
)

print(
    cell91_next_combined_target_text
)


print(
    "\n=== CONTINUATION STATE ==="
)

print(
    "Remaining Pilots:",
    3,
)

print(
    "Continuation SHA256:",
    cell91_continuation_state_sha256,
)


print(
    "\n=== ALL INTERNAL GATES ==="
)

for name, status in cell91_gate_checks.items():

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== SAFETY ==="
)

print(
    "Base Seed unchanged:",
    cell91_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell91_full_572_blocked,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell91_failures
)


print(
    "\nCODE CELL 91 GATE:",
    (
        "PASS"
        if cell91_gate
        else "FAIL"
    ),
)


assert cell91_gate, (
    "Pilot #20 Citation ACCEPT / completion / queue advance / "
    "next Pilot preparation failed. Repair this SAME Code Cell 91."
)

=== PILOT #20 COMPLETE + QUEUE ADVANCE + NEXT PILOT ===

=== COMPLETED PILOT ===
Pilot: question-pilot:fe7229122fe65089a77386df155a61d4
Target: target-evidence:d5f3951a062d8f436caf998e8d71339d
Retarget used: False
Question: ACCEPT
Expected Answer: ACCEPT
Citation: ACCEPT
Final Citation Mapping: citation-mapping:8a576b77652493574bd417da19f9d690
Final Engine State: PILOT_COMPLETE_PENDING_QUEUE_ADVANCE

=== AUTHORITY RESULT ===
Primary Authority: Dr. Komeil Roudi
Embedded quote present: True
Embedded quote role: ILLUSTRATIVE_CONTEXT_ONLY
Embedded quote used as Answer Evidence: False
Source Attribution preserved: True
Raw Source mutated: False
Parser mutated: False

=== QUEUE ADVANCE ===
Before: 4
After: 3
Queue advanced exactly once: True

=== HUMAN REVIEW REGISTRY ===
Completed Human Review cases: 19
Cases ready for Author Review: 19
Author Final Review: NOT_STARTED
Persistence valid: True

=== NEXT PILOT ===
Pilot: question-pilot:dd01033c15c375ae7124e8c7f86467a3
Golden Case: golden-case

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 92 — Pilot #21 Very Long Utterance Authority / Evidence Diagnostic

Pilot فعلی از نوع:

`VERY_LONG_UTTERANCE_RETRIEVAL`

است و Authority Rule آن:

`PRESERVE_SOURCE_ATTRIBUTION`

می‌باشد.

متن درباره دو ابزار ارزیابی سرمایه‌گذاری، یعنی NPV و IRR، توضیح مفصل می‌دهد.

پیش از ساخت سؤال بررسی می‌کنیم:

- Target چند Part دارد؛
- Speaker اصلی چه کسی است؛
- آیا کل متن متعلق به Dr. Komeil Roudi است؛
- آیا Speaker دیگری داخل Target وارد شده است؛
- آیا NPV و IRR هر دو مستقیماً در Source توضیح داده شده‌اند؛
- آیا Target بدون Retarget برای ساخت سؤال مناسب است.

این Cell فقط Diagnostic است و هیچ داده‌ای را تغییر نمی‌دهد.

</div>

In [92]:
# Code Cell 92

cell92_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell91_gate") is not True:
    cell92_failures.append(
        "CELL91_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell91_next_target_text_parts"
    ),
    list,
):
    cell92_failures.append(
        "NEXT_TARGET_PARTS_MISSING"
    )


# ============================================================
# 2) Pilot identity
# ============================================================

CELL92_PILOT_ID = (
    CELL91_NEXT_PILOT_ID
)


CELL92_TARGET_EVIDENCE_ID = (
    CELL91_NEXT_TARGET_EVIDENCE_ID
)


CELL92_TASK_FAMILY = (
    CELL91_NEXT_TASK_FAMILY
)


CELL92_AUTHORITY_EXPECTATION = (
    CELL91_NEXT_AUTHORITY_EXPECTATION
)


# ============================================================
# 3) Inspect Target Parts
# ============================================================

cell92_part_diagnostics = []


for index, part in enumerate(
    cell91_next_target_text_parts
):

    if isinstance(
        part,
        dict,
    ):

        speaker_state = (
            part.get(
                "speaker_state"
            )
            or
            {}
        )


        attribution = (
            part.get(
                "attribution"
            )
            or
            {}
        )


        locator = (
            part.get(
                "locator"
            )
            or
            {}
        )


        text = (
            part.get(
                "text"
            )
            or
            part.get(
                "target_text"
            )
            or
            part.get(
                "content"
            )
            or
            ""
        )


        cell92_part_diagnostics.append({
            "index":
                index,

            "document_id":
                part.get(
                    "document_id"
                ),

            "role":
                part.get(
                    "role"
                ),

            "speaker_base_status":
                speaker_state.get(
                    "base_status"
                ),

            "speaker_base_raw":
                speaker_state.get(
                    "base_raw"
                ),

            "speaker_effective_label":
                speaker_state.get(
                    "effective_label"
                ),

            "speaker_effective_canonical":
                speaker_state.get(
                    "effective_canonical"
                ),

            "speaker_effective_primary":
                speaker_state.get(
                    "effective_primary"
                ),

            "attribution_speaker":
                attribution.get(
                    "speaker"
                ),

            "source_line_start":
                locator.get(
                    "source_line_start"
                ),

            "source_line_end":
                locator.get(
                    "source_line_end"
                ),

            "text":
                text,
        })


    else:

        cell92_part_diagnostics.append({
            "index":
                index,

            "document_id":
                None,

            "role":
                None,

            "speaker_base_status":
                None,

            "speaker_base_raw":
                None,

            "speaker_effective_label":
                None,

            "speaker_effective_canonical":
                None,

            "speaker_effective_primary":
                None,

            "attribution_speaker":
                None,

            "source_line_start":
                None,

            "source_line_end":
                None,

            "text":
                str(
                    part
                ),
        })


# ============================================================
# 4) Resolve authority
# ============================================================

cell92_resolved_parts = []


for row in cell92_part_diagnostics:

    resolved_speaker = (
        row.get(
            "speaker_effective_canonical"
        )
        or
        row.get(
            "speaker_effective_label"
        )
        or
        row.get(
            "speaker_base_raw"
        )
        or
        row.get(
            "attribution_speaker"
        )
    )


    is_primary = (
        row.get(
            "speaker_effective_primary"
        )
        is True
        or
        resolved_speaker
        ==
        "Dr. Komeil Roudi"
    )


    is_unresolved = (
        resolved_speaker
        is None
        or
        row.get(
            "speaker_base_status"
        )
        ==
        "UNRESOLVED"
    )


    cell92_resolved_parts.append({
        **copy.deepcopy(
            row
        ),

        "resolved_speaker":
            resolved_speaker,

        "is_primary":
            is_primary,

        "is_unresolved":
            is_unresolved,
    })


# ============================================================
# 5) Structural validation
# ============================================================

cell92_exact_one_part = (
    len(
        cell92_resolved_parts
    )
    ==
    1
)


cell92_primary_parts = [
    row
    for row
    in cell92_resolved_parts
    if row[
        "is_primary"
    ]
]


cell92_unresolved_parts = [
    row
    for row
    in cell92_resolved_parts
    if row[
        "is_unresolved"
    ]
]


cell92_primary_valid = all([
    cell92_exact_one_part,

    len(
        cell92_primary_parts
    )
    ==
    1,

    len(
        cell92_unresolved_parts
    )
    ==
    0,

    cell92_primary_parts[
        0
    ].get(
        "resolved_speaker"
    )
    ==
    "Dr. Komeil Roudi",
])


# ============================================================
# 6) Very-long Utterance requirements
# ============================================================

cell92_expected_task_family = (
    CELL92_TASK_FAMILY
    ==
    "VERY_LONG_UTTERANCE_RETRIEVAL"
)


cell92_expected_authority_rule = (
    CELL92_AUTHORITY_EXPECTATION
    ==
    "PRESERVE_SOURCE_ATTRIBUTION"
)


cell92_text = (
    cell92_resolved_parts[
        0
    ].get(
        "text",
        "",
    )
    if cell92_resolved_parts
    else
    ""
)


cell92_long_enough = (
    len(
        cell92_text
    )
    >
    800
)


# ============================================================
# 7) Normalize text
# ============================================================

def cell92_norm(
    text,
):

    if not isinstance(
        text,
        str,
    ):
        return ""


    value = (
        text
        .replace(
            "ي",
            "ی",
        )
        .replace(
            "ك",
            "ک",
        )
        .replace(
            "\u200c",
            "",
        )
        .replace(
            "\ufeff",
            "",
        )
    )


    value = re.sub(
        r"\s+",
        " ",
        value,
    )


    return value.strip()


cell92_surface = cell92_norm(
    cell92_text
)


# ============================================================
# 8) NPV grounding
# ============================================================

cell92_npv_claims = {
    "npv_named":
        (
            "Net Present Value"
            in
            cell92_text
            or
            "NPV"
            in
            cell92_text
        ),

    "positive_npv_good":
        (
            "NPV مثبت بود"
            in
            cell92_text
            and
            "سرمایهگذاری خوبیه"
            in
            cell92_surface
        ),

    "higher_npv_better":
        (
            "Net Present Value بیشتری داره"
            in
            cell92_text
            and
            "سرمایهگذاری بهتری است"
            in
            cell92_surface
        ),

    "zero_npv_meets_expectation":
        (
            "اگر صفر باشه"
            in
            cell92_surface
            and
            "کف انتظارات شما رو تأمین میکنه"
            in
            cell92_surface
        ),

    "negative_npv_fails_expectation":
        (
            "اگر منفی باشه"
            in
            cell92_surface
            and
            "انتظارات شما رو تأمین نمیکنه"
            in
            cell92_surface
        ),
}


cell92_npv_claims_sufficient = all(
    cell92_npv_claims.values()
)


# ============================================================
# 9) IRR grounding
# ============================================================

cell92_irr_claims = {
    "irr_named":
        (
            "IRR"
            in
            cell92_text
            and
            "Internal Rate of Return"
            in
            cell92_text
        ),

    "irr_definition":
        (
            "نرخ بهره‌ای است"
            in
            cell92_text
            or
            "نرخ بهرهای است"
            in
            cell92_surface
        )
        and
        (
            "NPV رو بر اساس اون حساب بکنیم"
            in
            cell92_surface
        )
        and
        (
            "صفر"
            in
            cell92_surface
        ),

    "higher_irr_better":
        (
            "هرچی IRR یک پروژه بیشتر باشه"
            in
            cell92_surface
            and
            "پروژه بهتریه"
            in
            cell92_surface
        ),

    "irr_example":
        (
            "۹.۷۶۰۰"
            in
            cell92_text
            or
            "۹.۷۶۰"
            in
            cell92_text
        ),

    "benchmark_comparison":
        (
            "کمتر از نرخ بهره بانکه"
            in
            cell92_surface
            and
            "پروژه خوبی نیست"
            in
            cell92_surface
        ),
}


cell92_irr_claims_sufficient = all(
    cell92_irr_claims.values()
)


# ============================================================
# 10) Speaker-transition signal check inside Target
# ============================================================

cell92_internal_speaker_marker_signals = []


for marker in [
    "**مجری**",
    "**استاد**",
    "مجری:",
    "یکی از حاضران",
    "یکی از شرکت‌کنندگان",
    "یکی از شرکت کنندگان",
]:

    if marker in cell92_text:

        cell92_internal_speaker_marker_signals.append(
            marker
        )


cell92_internal_speaker_transition_detected = (
    len(
        cell92_internal_speaker_marker_signals
    )
    >
    0
)


# ============================================================
# 11) Authority Policy
# ============================================================

cell92_authority_policy = copy.deepcopy(
    cell91_persisted_registry.get(
        "workflow_policy",
        {},
    ).get(
        "authority_evidence_policy",
        {},
    )
)


cell92_authority_policy_valid = all([
    cell92_authority_policy.get(
        "primary_authority"
    )
    ==
    "Dr. Komeil Roudi",

    cell92_authority_policy.get(
        "secondary_to_primary_promotion"
    )
    ==
    "FORBIDDEN",
])


if not cell92_authority_policy_valid:
    cell92_failures.append(
        "GLOBAL_AUTHORITY_POLICY_INVALID"
    )


# ============================================================
# 12) Preliminary usability
# ============================================================

cell92_target_preliminarily_usable = all([
    cell92_expected_task_family,
    cell92_expected_authority_rule,
    cell92_primary_valid,
    cell92_long_enough,
    cell92_npv_claims_sufficient,
    cell92_irr_claims_sufficient,
    not cell92_internal_speaker_transition_detected,
])


cell92_retarget_likely_required = (
    not cell92_target_preliminarily_usable
)


# ============================================================
# 13) Safety
# ============================================================

cell92_registry_write = False

cell92_queue_advance = False

cell92_human_decision_added = False

cell92_target_changed = False

cell92_citation_changed = False

cell92_raw_source_mutated = False

cell92_parser_mutated = False

cell92_commit_push = False


# ============================================================
# 14) Gate
# ============================================================

cell92_gate_checks = {
    "no_failures":
        len(
            cell92_failures
        )
        ==
        0,

    "expected_task_family":
        cell92_expected_task_family,

    "expected_authority_rule":
        cell92_expected_authority_rule,

    "exact_one_part":
        cell92_exact_one_part,

    "primary_valid":
        cell92_primary_valid,

    "long_enough":
        cell92_long_enough,

    "npv_claims_sufficient":
        cell92_npv_claims_sufficient,

    "irr_claims_sufficient":
        cell92_irr_claims_sufficient,

    "no_internal_speaker_transition":
        not cell92_internal_speaker_transition_detected,

    "authority_policy_valid":
        cell92_authority_policy_valid,
}


cell92_gate = all(
    cell92_gate_checks.values()
)


# ============================================================
# 15) Diagnostics
# ============================================================

print(
    "=== PILOT #21 VERY LONG UTTERANCE DIAGNOSTIC ==="
)


print(
    "\n=== PILOT ==="
)

print(
    "Pilot:",
    CELL92_PILOT_ID,
)

print(
    "Target Evidence:",
    CELL92_TARGET_EVIDENCE_ID,
)

print(
    "Task Family:",
    CELL92_TASK_FAMILY,
)

print(
    "Authority Expectation:",
    CELL92_AUTHORITY_EXPECTATION,
)


print(
    "\n=== TARGET STRUCTURE ==="
)

print(
    "Part count:",
    len(
        cell92_resolved_parts
    ),
)

print(
    "Primary parts:",
    len(
        cell92_primary_parts
    ),
)

print(
    "Unresolved parts:",
    len(
        cell92_unresolved_parts
    ),
)

print(
    "Utterance chars:",
    len(
        cell92_text
    ),
)


for row in cell92_resolved_parts:

    print(
        "\n"
        +
        "="
        *
        72
    )

    print(
        "Document:",
        row.get(
            "document_id"
        ),
    )

    print(
        "Base status:",
        row.get(
            "speaker_base_status"
        ),
    )

    print(
        "Base raw:",
        row.get(
            "speaker_base_raw"
        ),
    )

    print(
        "Effective canonical:",
        row.get(
            "speaker_effective_canonical"
        ),
    )

    print(
        "Effective primary:",
        row.get(
            "speaker_effective_primary"
        ),
    )

    print(
        "Resolved speaker:",
        row.get(
            "resolved_speaker"
        ),
    )

    print(
        "Source lines:",
        (
            row.get(
                "source_line_start"
            ),
            row.get(
                "source_line_end"
            ),
        ),
    )


print(
    "\n=== NPV CLAIMS ==="
)

for name, status in (
    cell92_npv_claims.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "NPV claims sufficient:",
    cell92_npv_claims_sufficient,
)


print(
    "\n=== IRR CLAIMS ==="
)

for name, status in (
    cell92_irr_claims.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "IRR claims sufficient:",
    cell92_irr_claims_sufficient,
)


print(
    "\n=== INTERNAL SPEAKER CHECK ==="
)

print(
    "Internal speaker markers:",
    cell92_internal_speaker_marker_signals,
)

print(
    "Internal speaker transition detected:",
    cell92_internal_speaker_transition_detected,
)


print(
    "\n=== TARGET ASSESSMENT ==="
)

print(
    "Target preliminarily usable:",
    cell92_target_preliminarily_usable,
)

print(
    "Retarget likely required:",
    cell92_retarget_likely_required,
)

print(
    "IMPORTANT: No Human Question/Answer/Citation decision "
    "has been recorded."
)


print(
    "\n=== ALL INTERNAL GATES ==="
)

for name, status in cell92_gate_checks.items():

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== SAFETY ==="
)

print(
    "Registry write:",
    cell92_registry_write,
)

print(
    "Queue advance:",
    cell92_queue_advance,
)

print(
    "Human Decision added:",
    cell92_human_decision_added,
)

print(
    "Target changed:",
    cell92_target_changed,
)

print(
    "Citation changed:",
    cell92_citation_changed,
)

print(
    "Raw Source mutated:",
    cell92_raw_source_mutated,
)

print(
    "Parser mutated:",
    cell92_parser_mutated,
)

print(
    "Commit / Push:",
    cell92_commit_push,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell92_failures
)


print(
    "\nCODE CELL 92 GATE:",
    (
        "PASS"
        if cell92_gate
        else "FAIL"
    ),
)


assert cell92_gate, (
    "Pilot #21 very-long-utterance diagnostic failed. "
    "Repair this SAME Code Cell 92 before drafting the question."
)

=== PILOT #21 VERY LONG UTTERANCE DIAGNOSTIC ===

=== PILOT ===
Pilot: question-pilot:dd01033c15c375ae7124e8c7f86467a3
Target Evidence: target-evidence:438f0de8cf57f8d20e77420ec63ef3fc
Task Family: VERY_LONG_UTTERANCE_RETRIEVAL
Authority Expectation: PRESERVE_SOURCE_ATTRIBUTION

=== TARGET STRUCTURE ===
Part count: 1
Primary parts: 1
Unresolved parts: 0
Utterance chars: 3262

Document: program:uni-tehran:episode-029:utterance:00013
Base status: EXPLICIT_SOURCE_MARKER
Base raw: استاد
Effective canonical: Dr. Komeil Roudi
Effective primary: True
Resolved speaker: Dr. Komeil Roudi
Source lines: (113, 125)

=== NPV CLAIMS ===
npv_named: True
positive_npv_good: True
higher_npv_better: True
zero_npv_meets_expectation: True
negative_npv_fails_expectation: True
NPV claims sufficient: True

=== IRR CLAIMS ===
irr_named: True
irr_definition: True
higher_irr_better: True
irr_example: True
benchmark_comparison: True
IRR claims sufficient: True

=== INTERNAL SPEAKER CHECK ===
Internal speaker marke

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 93 — Pilot #21 Question/Answer ACCEPT + Citation Draft

Human Reviewer نسخه نهایی سؤال و جواب Pilot #21 را پذیرفت.

سؤال پذیرفته‌شده:

**NPV و IRR در ارزیابی یک سرمایه‌گذاری چه هستند و چه تفاوتی با یکدیگر دارند؟**

جواب پذیرفته‌شده:

**NPV یا ارزش فعلی خالص نشان می‌دهد یک سرمایه‌گذاری با درنظرگرفتن نرخ بازده مورد انتظار، چه مقدار ارزش ایجاد می‌کند. NPV مثبت یعنی سرمایه‌گذاری از انتظارات تعیین‌شده عبور کرده، NPV صفر یعنی دقیقاً حداقل انتظار را تأمین کرده و NPV منفی یعنی آن انتظارات را برآورده نکرده است. IRR یا نرخ بازده داخلی، نرخی است که اگر NPV را با آن محاسبه کنیم، NPV برابر صفر می‌شود؛ بنابراین IRR نرخ بازده ضمنی خود سرمایه‌گذاری را نشان می‌دهد.**

تمام Answer Evidence از همان Utterance صریح Dr. Komeil Roudi در Source lines 113–125 گرفته می‌شود.

بعد از این Cell، Citation در وضعیت Human Review قرار می‌گیرد و Queue همچنان 3 باقی می‌ماند.

</div>

In [93]:
# Code Cell 93

cell93_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell92_gate") is not True:
    cell93_failures.append(
        "CELL92_NOT_PASS"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell93_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Human-accepted Question / Answer
# ============================================================

CELL93_ACCEPTED_QUESTION = (
    "NPV و IRR در ارزیابی یک سرمایه‌گذاری چه هستند "
    "و چه تفاوتی با یکدیگر دارند؟"
)


CELL93_ACCEPTED_ANSWER = (
    "NPV یا ارزش فعلی خالص نشان می‌دهد یک سرمایه‌گذاری با "
    "درنظرگرفتن نرخ بازده مورد انتظار، چه مقدار ارزش ایجاد می‌کند. "
    "NPV مثبت یعنی سرمایه‌گذاری از انتظارات تعیین‌شده عبور کرده، "
    "NPV صفر یعنی دقیقاً حداقل انتظار را تأمین کرده و NPV منفی "
    "یعنی آن انتظارات را برآورده نکرده است. IRR یا نرخ بازده داخلی، "
    "نرخی است که اگر NPV را با آن محاسبه کنیم، NPV برابر صفر می‌شود؛ "
    "بنابراین IRR نرخ بازده ضمنی خود سرمایه‌گذاری را نشان می‌دهد."
)


CELL93_PILOT_ID = (
    CELL92_PILOT_ID
)


CELL93_GOLDEN_CASE_ID = (
    CELL91_NEXT_GOLDEN_CASE_ID
)


CELL93_TASK_DRAFT_ID = (
    CELL91_NEXT_TASK_DRAFT_ID
)


CELL93_TARGET_EVIDENCE_ID = (
    CELL92_TARGET_EVIDENCE_ID
)


# ============================================================
# 3) Target validation
# ============================================================

cell93_target_parts = copy.deepcopy(
    cell91_next_target_text_parts
)


cell93_target_valid = all([
    isinstance(
        cell93_target_parts,
        list,
    ),

    len(
        cell93_target_parts
    )
    ==
    1,

    cell92_primary_valid,

    cell92_long_enough,

    cell92_npv_claims_sufficient,

    cell92_irr_claims_sufficient,

    not cell92_internal_speaker_transition_detected,

    not cell92_retarget_likely_required,
])


if not cell93_target_valid:
    cell93_failures.append(
        "TARGET_AUTHORITY_STRUCTURE_INVALID"
    )


cell93_primary_part = copy.deepcopy(
    cell93_target_parts[
        0
    ]
)


cell93_primary_text = (
    cell93_primary_part.get(
        "text"
    )
    or
    cell93_primary_part.get(
        "target_text"
    )
    or
    cell93_primary_part.get(
        "content"
    )
    or
    ""
)


# ============================================================
# 4) Grounding
# ============================================================

cell93_grounding = {
    "npv_present":
        (
            "NPV"
            in
            cell93_primary_text
            and
            "Net Present Value"
            in
            cell93_primary_text
        ),

    "npv_positive":
        (
            cell92_npv_claims.get(
                "positive_npv_good"
            )
            is True
        ),

    "npv_zero":
        (
            cell92_npv_claims.get(
                "zero_npv_meets_expectation"
            )
            is True
        ),

    "npv_negative":
        (
            cell92_npv_claims.get(
                "negative_npv_fails_expectation"
            )
            is True
        ),

    "irr_present":
        (
            cell92_irr_claims.get(
                "irr_named"
            )
            is True
        ),

    "irr_zeroes_npv":
        (
            cell92_irr_claims.get(
                "irr_definition"
            )
            is True
        ),

    "irr_is_comparable_return_rate":
        (
            cell92_irr_claims.get(
                "benchmark_comparison"
            )
            is True
        ),
}


cell93_grounding_pass = all(
    cell93_grounding.values()
)


if not cell93_grounding_pass:
    cell93_failures.append(
        "ANSWER_GROUNDING_FAILED"
    )


# ============================================================
# 5) Start Engine
# ============================================================

cell93_question_pending_session = (
    cell13_build_session(
        cell91_next_pilot_packet
    )
)


cell93_session_start_valid = (
    cell93_question_pending_session.get(
        "current_state"
    )
    ==
    "QUESTION_DRAFT_PENDING"
)


if not cell93_session_start_valid:
    cell93_failures.append(
        "ENGINE_SESSION_START_INVALID"
    )


# ============================================================
# 6) Question Draft + Human ACCEPT
# ============================================================

cell93_question_identity = {
    "identity_version":
        "question_draft_identity_v1",

    "pilot_case_id":
        CELL93_PILOT_ID,

    "target_evidence_id":
        CELL93_TARGET_EVIDENCE_ID,

    "question_text":
        CELL93_ACCEPTED_QUESTION,

    "revision_number":
        1,
}


cell93_question_draft_id = (
    "question-draft:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell93_question_identity
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


cell93_question_record = {
    "record_version":
        "question_draft_record_v1",

    "question_draft_id":
        cell93_question_draft_id,

    "pilot_case_id":
        CELL93_PILOT_ID,

    "golden_case_id":
        CELL93_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL93_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL93_TARGET_EVIDENCE_ID,

    "question_text":
        CELL93_ACCEPTED_QUESTION,

    "revision_number":
        1,

    "human_adjudication_status":
        "PENDING",

    "external_knowledge_used":
        False,
}


cell93_question_review_session = (
    cell13_apply_event(
        cell93_question_pending_session,
        "QUESTION_DRAFT_REGISTERED",
        artifact=
            cell93_question_record,
    )
)


cell93_answer_pending_session = (
    cell13_apply_event(
        cell93_question_review_session,
        "QUESTION_ACCEPTED",
        human_note=
            "Human Reviewer accepted the question.",
    )
)


cell93_question_accept_valid = (
    cell93_answer_pending_session.get(
        "current_state"
    )
    ==
    "EXPECTED_ANSWER_DRAFT_PENDING"
)


if not cell93_question_accept_valid:
    cell93_failures.append(
        "QUESTION_ACCEPT_TRANSITION_INVALID"
    )


# ============================================================
# 7) Expected Answer + Human ACCEPT
# ============================================================

cell93_answer_identity = {
    "identity_version":
        "expected_answer_identity_v1",

    "pilot_case_id":
        CELL93_PILOT_ID,

    "question_draft_id":
        cell93_question_draft_id,

    "target_evidence_id":
        CELL93_TARGET_EVIDENCE_ID,

    "answer_text":
        CELL93_ACCEPTED_ANSWER,

    "revision_number":
        1,
}


cell93_expected_answer_draft_id = (
    "expected-answer-draft:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell93_answer_identity
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


cell93_expected_answer_record = {
    "record_version":
        "expected_answer_draft_record_v1",

    "expected_answer_draft_id":
        cell93_expected_answer_draft_id,

    "pilot_case_id":
        CELL93_PILOT_ID,

    "golden_case_id":
        CELL93_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL93_TASK_DRAFT_ID,

    "question_draft_id":
        cell93_question_draft_id,

    "target_evidence_id":
        CELL93_TARGET_EVIDENCE_ID,

    "expected_answer_text":
        CELL93_ACCEPTED_ANSWER,

    "revision_number":
        1,

    "human_adjudication_status":
        "PENDING",

    "external_knowledge_used":
        False,
}


cell93_answer_review_session = (
    cell13_apply_event(
        cell93_answer_pending_session,
        "EXPECTED_ANSWER_REGISTERED",
        artifact=
            cell93_expected_answer_record,
    )
)


cell93_citation_pending_session = (
    cell13_apply_event(
        cell93_answer_review_session,
        "EXPECTED_ANSWER_ACCEPTED",
        human_note=
            "Human Reviewer accepted the expected answer.",
    )
)


cell93_answer_accept_valid = (
    cell93_citation_pending_session.get(
        "current_state"
    )
    ==
    "CITATION_MAPPING_PENDING"
)


if not cell93_answer_accept_valid:
    cell93_failures.append(
        "ANSWER_ACCEPT_TRANSITION_INVALID"
    )


# ============================================================
# 8) Citation Entry
# ============================================================

cell93_primary_entry = copy.deepcopy(
    cell93_primary_part
)


cell93_primary_entry.update({
    "citation_index":
        0,

    "authority_role":
        "PRIMARY_ANSWER_EVIDENCE",

    "answer_support":
        True,

    "use_in_expected_answer_grounding":
        True,

    "speaker_canonical":
        "Dr. Komeil Roudi",

    "speaker_primary":
        True,

    "source_attribution_preserved":
        True,

    "external_locator_lookup_used":
        False,
})


cell93_citation_entries = [
    cell93_primary_entry
]


cell93_citation_boundary_valid = all([
    len(
        cell93_citation_entries
    )
    ==
    1,

    cell93_primary_entry.get(
        "speaker_canonical"
    )
    ==
    "Dr. Komeil Roudi",

    cell93_primary_entry.get(
        "speaker_primary"
    )
    is True,

    cell93_primary_entry.get(
        "answer_support"
    )
    is True,

    cell93_primary_entry.get(
        "use_in_expected_answer_grounding"
    )
    is True,

    cell93_primary_entry.get(
        "source_attribution_preserved"
    )
    is True,
])


if not cell93_citation_boundary_valid:
    cell93_failures.append(
        "CITATION_AUTHORITY_BOUNDARY_INVALID"
    )


# ============================================================
# 9) Locator integrity
# ============================================================

cell93_no_locator_fabrication = (
    cell93_primary_entry.get(
        "locator"
    )
    ==
    cell93_primary_part.get(
        "locator"
    )
)


if not cell93_no_locator_fabrication:
    cell93_failures.append(
        "LOCATOR_INTEGRITY_FAILED"
    )


# ============================================================
# 10) Citation Mapping
# ============================================================

cell93_citation_identity = {
    "identity_version":
        "citation_mapping_identity_v1_very_long_utterance",

    "pilot_case_id":
        CELL93_PILOT_ID,

    "question_draft_id":
        cell93_question_draft_id,

    "expected_answer_draft_id":
        cell93_expected_answer_draft_id,

    "target_evidence_id":
        CELL93_TARGET_EVIDENCE_ID,

    "citation_entries":
        copy.deepcopy(
            cell93_citation_entries
        ),
}


cell93_citation_mapping_id = (
    "citation-mapping:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell93_citation_identity
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


cell93_citation_mapping_record = {
    "record_version":
        "citation_mapping_record_v1_very_long_utterance",

    "citation_mapping_id":
        cell93_citation_mapping_id,

    "pilot_case_id":
        CELL93_PILOT_ID,

    "golden_case_id":
        CELL93_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL93_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL93_TARGET_EVIDENCE_ID,

    "question_draft_id":
        cell93_question_draft_id,

    "expected_answer_draft_id":
        cell93_expected_answer_draft_id,

    "citation_entries":
        copy.deepcopy(
            cell93_citation_entries
        ),

    "primary_answer_authority":
        "Dr. Komeil Roudi",

    "source_attribution_preserved":
        True,

    "internal_speaker_transition":
        False,

    "retarget_used":
        False,

    "raw_source_mutated":
        False,

    "parser_output_mutated":
        False,

    "external_knowledge_used":
        False,

    "human_adjudication_status":
        "PENDING",
}


cell93_current_session = (
    cell13_apply_event(
        cell93_citation_pending_session,
        "CITATION_MAPPING_REGISTERED",
        artifact=
            cell93_citation_mapping_record,
    )
)


cell93_citation_registered_valid = (
    cell93_current_session.get(
        "current_state"
    )
    ==
    "CITATION_HUMAN_REVIEW_PENDING"
)


if not cell93_citation_registered_valid:
    cell93_failures.append(
        "CITATION_REGISTRATION_INVALID"
    )


# ============================================================
# 11) Active Workspace
# ============================================================

cell93_active_workspace = copy.deepcopy(
    cell91_persisted_active_case
)


cell93_active_workspace[
    "workflow_state"
] = "CITATION_HUMAN_REVIEW_PENDING"


cell93_active_workspace[
    "question_review"
] = {
    "question_draft_id":
        cell93_question_draft_id,

    "question_text":
        CELL93_ACCEPTED_QUESTION,

    "human_decision":
        "ACCEPT",
}


cell93_active_workspace[
    "expected_answer_review"
] = {
    "expected_answer_draft_id":
        cell93_expected_answer_draft_id,

    "expected_answer_text":
        CELL93_ACCEPTED_ANSWER,

    "human_decision":
        "ACCEPT",
}


cell93_active_workspace[
    "citation_review"
] = {
    "citation_mapping_id":
        cell93_citation_mapping_id,

    "citation_mapping_record":
        copy.deepcopy(
            cell93_citation_mapping_record
        ),

    "human_decision":
        "PENDING",

    "primary_answer_authority":
        "Dr. Komeil Roudi",
}


# ============================================================
# 12) Continuation state
# ============================================================

cell93_continuation_core = {
    "next_pilot_id":
        CELL93_PILOT_ID,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell91_remaining_pilot_ids
        ),

    "remaining_count":
        3,

    "active_pilot_status":
        "CITATION_HUMAN_REVIEW_PENDING",

    "active_target_evidence_id":
        CELL93_TARGET_EVIDENCE_ID,

    "active_citation_mapping_id":
        cell93_citation_mapping_id,

    "retarget_used":
        False,

    "authority_policy_active":
        True,

    "last_completed_pilot_id":
        cell91_continuation_state.get(
            "last_completed_pilot_id"
        ),
}


cell93_continuation_state_sha256 = (
    cell10_stable_sha(
        cell93_continuation_core
    )
)


cell93_continuation_state = {
    **copy.deepcopy(
        cell93_continuation_core
    ),

    "continuation_state_sha256":
        cell93_continuation_state_sha256,
}


# ============================================================
# 13) Persist
# ============================================================

cell93_persistent_registry_payload = copy.deepcopy(
    cell91_persisted_registry
)


cell93_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell93_active_workspace
)


cell93_persistent_registry_payload[
    "remaining_pilot_count"
] = 3


cell93_persistent_registry_payload[
    "continuation_state"
] = copy.deepcopy(
    cell93_continuation_state
)


cell93_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell93_persistent_registry_payload
    )
)


# ============================================================
# 14) Read-back validation
# ============================================================

cell93_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell93_persisted_active_case = (
    cell93_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell93_persistence_valid = all([
    cell93_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL93_PILOT_ID,

    cell93_persisted_active_case.get(
        "workflow_state"
    )
    ==
    "CITATION_HUMAN_REVIEW_PENDING",

    cell93_persisted_active_case.get(
        "question_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell93_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell93_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "PENDING",

    cell93_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "primary_answer_authority"
    )
    ==
    "Dr. Komeil Roudi",

    cell93_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    3,

    cell93_persisted_registry.get(
        "continuation_state",
        {},
    ).get(
        "continuation_state_sha256"
    )
    ==
    cell93_continuation_state_sha256,
])


if not cell93_persistence_valid:
    cell93_failures.append(
        "REGISTRY_READBACK_INVALID"
    )


# ============================================================
# 15) Safety
# ============================================================

cell93_queue_still_3 = all([
    len(
        cell91_remaining_pilot_ids
    )
    ==
    3,

    cell91_remaining_pilot_ids[
        0
    ]
    ==
    CELL93_PILOT_ID,
])


cell93_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell93_full_572_blocked = (
    cell91_full_572_blocked
    is True
)


cell93_no_commit_push = True


# ============================================================
# 16) Gate
# ============================================================

cell93_gate_checks = {
    "no_failures":
        len(
            cell93_failures
        )
        ==
        0,

    "target_valid":
        cell93_target_valid,

    "grounding_pass":
        cell93_grounding_pass,

    "session_start_valid":
        cell93_session_start_valid,

    "question_accept_valid":
        cell93_question_accept_valid,

    "answer_accept_valid":
        cell93_answer_accept_valid,

    "citation_boundary_valid":
        cell93_citation_boundary_valid,

    "no_locator_fabrication":
        cell93_no_locator_fabrication,

    "citation_registered_valid":
        cell93_citation_registered_valid,

    "persistence_valid":
        cell93_persistence_valid,

    "queue_still_3":
        cell93_queue_still_3,

    "base_seed_unchanged":
        cell93_base_seed_unchanged,

    "full_572_blocked":
        cell93_full_572_blocked,

    "no_commit_push":
        cell93_no_commit_push,
}


cell93_gate = all(
    cell93_gate_checks.values()
)


# ============================================================
# 17) Diagnostics
# ============================================================

print(
    "=== PILOT #21 QA ACCEPT + CITATION DRAFT ==="
)


print(
    "\n=== AUTHORITY ==="
)

print(
    "Primary Answer Authority:",
    "Dr. Komeil Roudi",
)

print(
    "Source Attribution preserved:",
    True,
)

print(
    "Internal speaker transition:",
    False,
)

print(
    "Retarget used:",
    False,
)


print(
    "\n=== ACCEPTED QUESTION ==="
)

print(
    CELL93_ACCEPTED_QUESTION
)

print(
    "Question Draft ID:",
    cell93_question_draft_id,
)

print(
    "Human Decision:",
    "ACCEPT",
)


print(
    "\n=== ACCEPTED ANSWER ==="
)

print(
    CELL93_ACCEPTED_ANSWER
)

print(
    "Expected Answer Draft ID:",
    cell93_expected_answer_draft_id,
)

print(
    "Human Decision:",
    "ACCEPT",
)


print(
    "\n=== GROUNDING ==="
)

for name, status in cell93_grounding.items():

    print(
        f"{name}:",
        status,
    )


print(
    "Grounding PASS:",
    cell93_grounding_pass,
)


print(
    "\n=== CITATION DRAFT ==="
)

print(
    "Citation Mapping ID:",
    cell93_citation_mapping_id,
)

print(
    "Citation entries:",
    len(
        cell93_citation_entries
    ),
)

print(
    "Primary Answer Evidence entries:",
    1,
)

print(
    "Citation boundary valid:",
    cell93_citation_boundary_valid,
)

print(
    "No locator fabrication:",
    cell93_no_locator_fabrication,
)


print(
    "\n=== WORKFLOW ==="
)

print(
    "Current State:",
    cell93_current_session.get(
        "current_state"
    ),
)

print(
    "Question:",
    "ACCEPT",
)

print(
    "Expected Answer:",
    "ACCEPT",
)

print(
    "Citation:",
    "PENDING",
)


print(
    "\n=== ALL INTERNAL GATES ==="
)

for name, status in cell93_gate_checks.items():

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== QUEUE / SAFETY ==="
)

print(
    "Queue still 3:",
    cell93_queue_still_3,
)

print(
    "Persistence valid:",
    cell93_persistence_valid,
)

print(
    "Base Seed unchanged:",
    cell93_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell93_full_572_blocked,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell93_failures
)


print(
    "\nCODE CELL 93 GATE:",
    (
        "PASS"
        if cell93_gate
        else "FAIL"
    ),
)


assert cell93_gate, (
    "Pilot #21 Question+Answer ACCEPT / Citation Draft failed. "
    "Repair this SAME Code Cell 93."
)

=== PILOT #21 QA ACCEPT + CITATION DRAFT ===

=== AUTHORITY ===
Primary Answer Authority: Dr. Komeil Roudi
Source Attribution preserved: True
Internal speaker transition: False
Retarget used: False

=== ACCEPTED QUESTION ===
NPV و IRR در ارزیابی یک سرمایه‌گذاری چه هستند و چه تفاوتی با یکدیگر دارند؟
Question Draft ID: question-draft:bb4bcaafdccde4774c656c5788303e33
Human Decision: ACCEPT

=== ACCEPTED ANSWER ===
NPV یا ارزش فعلی خالص نشان می‌دهد یک سرمایه‌گذاری با درنظرگرفتن نرخ بازده مورد انتظار، چه مقدار ارزش ایجاد می‌کند. NPV مثبت یعنی سرمایه‌گذاری از انتظارات تعیین‌شده عبور کرده، NPV صفر یعنی دقیقاً حداقل انتظار را تأمین کرده و NPV منفی یعنی آن انتظارات را برآورده نکرده است. IRR یا نرخ بازده داخلی، نرخی است که اگر NPV را با آن محاسبه کنیم، NPV برابر صفر می‌شود؛ بنابراین IRR نرخ بازده ضمنی خود سرمایه‌گذاری را نشان می‌دهد.
Expected Answer Draft ID: expected-answer-draft:eec82ef5b5e56f793a2e67776ad1b75b
Human Decision: ACCEPT

=== GROUNDING ===
npv_present: True
npv_positive: True
npv_

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 94 — Citation ACCEPT + Pilot #21 Completion + Queue Advance

Citation نهایی Pilot #21 توسط Human Reviewer پذیرفته شد.

در این مرحله:

- Citation با تصمیم `ACCEPT` ثبت می‌شود؛
- Dr. Komeil Roudi تنها Primary Answer Authority باقی می‌ماند؛
- Pilot #21 کامل می‌شود؛
- تعداد Caseهای کامل Human Review از 19 به 20 می‌رسد؛
- تعداد Caseهای آماده Author Review از 19 به 20 می‌رسد؛
- Queue دقیقاً از 3 به 2 جلو می‌رود؛
- Pilot بعدی برای Human Review آماده می‌شود؛
- Author Final Review همچنان `NOT_STARTED` است؛
- Full 572 Generation همچنان Blocked باقی می‌ماند.

</div>

In [94]:
# Code Cell 94

cell94_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell93_gate") is not True:
    cell94_failures.append(
        "CELL93_NOT_PASS"
    )


if (
    globals().get(
        "cell93_current_session",
        {},
    ).get(
        "current_state"
    )
    !=
    "CITATION_HUMAN_REVIEW_PENDING"
):
    cell94_failures.append(
        "CITATION_NOT_AT_HUMAN_REVIEW_GATE"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell94_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Human Citation ACCEPT
# ============================================================

CELL94_CITATION_HUMAN_NOTE = (
    "Human Reviewer accepted the final Citation Mapping. "
    "Dr. Komeil Roudi remains the sole primary answer authority. "
    "Source attribution is preserved and no external knowledge "
    "is used in the Expected Answer."
)


cell94_complete_pending_session = (
    cell13_apply_event(
        cell93_current_session,
        "CITATION_ACCEPTED",
        human_note=
            CELL94_CITATION_HUMAN_NOTE,
    )
)


cell94_citation_accept_valid = (
    cell94_complete_pending_session.get(
        "current_state"
    )
    ==
    "PILOT_COMPLETE_PENDING_QUEUE_ADVANCE"
)


if not cell94_citation_accept_valid:
    cell94_failures.append(
        "CITATION_ACCEPT_TRANSITION_INVALID"
    )


# ============================================================
# 3) Final Citation Mapping
# ============================================================

cell94_final_citation_mapping_record = copy.deepcopy(
    cell93_citation_mapping_record
)


cell94_final_citation_mapping_record[
    "human_adjudication_status"
] = "ACCEPT"


cell94_final_citation_mapping_record[
    "human_review"
] = {
    "human_decision":
        "ACCEPT",

    "human_note":
        CELL94_CITATION_HUMAN_NOTE,

    "primary_answer_authority":
        "Dr. Komeil Roudi",

    "source_attribution_preserved":
        True,

    "internal_speaker_transition":
        False,

    "retarget_used":
        False,

    "raw_source_mutated":
        False,

    "parser_output_mutated":
        False,

    "external_knowledge_used":
        False,
}


# ============================================================
# 4) Complete Human Review case
# ============================================================

cell94_completed_case = copy.deepcopy(
    cell93_active_workspace
)


cell94_completed_case[
    "workflow_state"
] = "PILOT_COMPLETE"


cell94_completed_case[
    "status"
] = "COMPLETED_HUMAN_REVIEW"


cell94_completed_case[
    "final_status"
] = "PILOT_COMPLETE"


cell94_completed_case.setdefault(
    "citation_review",
    {},
)


cell94_completed_case[
    "citation_review"
][
    "human_decision"
] = "ACCEPT"


cell94_completed_case[
    "citation_review"
][
    "human_note"
] = CELL94_CITATION_HUMAN_NOTE


cell94_completed_case[
    "citation_review"
][
    "citation_mapping_record"
] = copy.deepcopy(
    cell94_final_citation_mapping_record
)


cell94_completed_case[
    "completion_summary"
] = {
    "retarget_used":
        False,

    "target_evidence_id":
        CELL93_TARGET_EVIDENCE_ID,

    "question_decision":
        "ACCEPT",

    "expected_answer_decision":
        "ACCEPT",

    "citation_decision":
        "ACCEPT",

    "task_family":
        "VERY_LONG_UTTERANCE_RETRIEVAL",

    "primary_answer_authority":
        "Dr. Komeil Roudi",

    "source_attribution_preserved":
        True,

    "internal_speaker_transition":
        False,

    "raw_source_mutated":
        False,

    "parser_output_mutated":
        False,

    "author_final_review":
        "NOT_STARTED",
}


cell94_completed_case[
    "author_final_review"
] = {
    "status":
        "NOT_STARTED",

    "decision":
        None,

    "notes":
        None,
}


# ============================================================
# 5) Queue advance: 3 -> 2
# ============================================================

cell94_previous_remaining_pilot_ids = copy.deepcopy(
    cell91_remaining_pilot_ids
)


CELL94_COMPLETED_PILOT_ID = (
    cell94_previous_remaining_pilot_ids[
        0
    ]
)


cell94_remaining_pilot_ids = copy.deepcopy(
    cell94_previous_remaining_pilot_ids[
        1:
    ]
)


cell94_queue_advance_valid = all([
    len(
        cell94_previous_remaining_pilot_ids
    )
    ==
    3,

    CELL94_COMPLETED_PILOT_ID
    ==
    CELL93_PILOT_ID,

    len(
        cell94_remaining_pilot_ids
    )
    ==
    2,

    CELL94_COMPLETED_PILOT_ID
    not in
    cell94_remaining_pilot_ids,
])


if not cell94_queue_advance_valid:
    cell94_failures.append(
        "QUEUE_ADVANCE_INVALID"
    )


CELL94_NEXT_PILOT_ID = (
    cell94_remaining_pilot_ids[
        0
    ]
)


# ============================================================
# 6) Find next Pilot packet
# ============================================================

cell94_raw_next_packet_candidates = (
    cell58_find_pilot_packet_candidates(
        cell13_seed,
        CELL94_NEXT_PILOT_ID,
    )
)


cell94_packet_candidates_by_sha = {}


for candidate in cell94_raw_next_packet_candidates:

    packet_sha = cell10_stable_sha(
        candidate[
            "packet"
        ]
    )


    previous = (
        cell94_packet_candidates_by_sha.get(
            packet_sha
        )
    )


    if (
        previous is None
        or
        candidate[
            "score"
        ]
        >
        previous[
            "score"
        ]
    ):

        cell94_packet_candidates_by_sha[
            packet_sha
        ] = candidate


cell94_ranked_next_packet_candidates = sorted(
    cell94_packet_candidates_by_sha.values(),
    key=lambda item: (
        -item[
            "score"
        ],
        item[
            "path"
        ],
    ),
)


cell94_next_pilot_packet = (
    copy.deepcopy(
        cell94_ranked_next_packet_candidates[
            0
        ][
            "packet"
        ]
    )
    if cell94_ranked_next_packet_candidates
    else None
)


if not isinstance(
    cell94_next_pilot_packet,
    dict,
):

    cell94_failures.append(
        "NEXT_PILOT_PACKET_NOT_FOUND"
    )


# ============================================================
# 7) Extract next Pilot identities
# ============================================================

cell94_next_pilot_ids_found = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell94_next_pilot_packet,
            "question-pilot:",
        )
    )
)


cell94_next_golden_case_ids = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell94_next_pilot_packet,
            "golden-case:",
        )
    )
)


cell94_next_task_draft_ids = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell94_next_pilot_packet,
            "task-draft:",
        )
    )
)


cell94_next_target_evidence_ids = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell94_next_pilot_packet,
            "target-evidence:",
        )
    )
)


cell94_next_identity_valid = all([
    cell94_next_pilot_ids_found
    ==
    [
        CELL94_NEXT_PILOT_ID
    ],

    len(
        cell94_next_golden_case_ids
    )
    ==
    1,

    len(
        cell94_next_task_draft_ids
    )
    ==
    1,

    len(
        cell94_next_target_evidence_ids
    )
    ==
    1,
])


if not cell94_next_identity_valid:
    cell94_failures.append(
        "NEXT_PILOT_IDENTITY_INVALID"
    )


CELL94_NEXT_GOLDEN_CASE_ID = (
    cell94_next_golden_case_ids[
        0
    ]
    if
    len(
        cell94_next_golden_case_ids
    )
    ==
    1
    else
    None
)


CELL94_NEXT_TASK_DRAFT_ID = (
    cell94_next_task_draft_ids[
        0
    ]
    if
    len(
        cell94_next_task_draft_ids
    )
    ==
    1
    else
    None
)


CELL94_NEXT_TARGET_EVIDENCE_ID = (
    cell94_next_target_evidence_ids[
        0
    ]
    if
    len(
        cell94_next_target_evidence_ids
    )
    ==
    1
    else
    None
)


# ============================================================
# 8) Extract next Target Evidence
# ============================================================

cell94_target_part_candidates = (
    cell58_find_key_values(
        cell94_next_pilot_packet,
        "target_text_parts",
    )
)


cell94_target_parts_by_sha = {}


for parts in cell94_target_part_candidates:

    if not isinstance(
        parts,
        list,
    ):
        continue


    if len(
        parts
    ) == 0:
        continue


    parts_sha = cell10_stable_sha(
        parts
    )


    cell94_target_parts_by_sha[
        parts_sha
    ] = parts


cell94_unique_target_part_sets = list(
    cell94_target_parts_by_sha.values()
)


cell94_next_target_text_parts = (
    copy.deepcopy(
        cell94_unique_target_part_sets[
            0
        ]
    )
    if
    len(
        cell94_unique_target_part_sets
    )
    ==
    1
    else
    None
)


if not isinstance(
    cell94_next_target_text_parts,
    list,
):

    cell94_failures.append(
        "NEXT_TARGET_TEXT_PARTS_NOT_UNIQUE"
    )


# ============================================================
# 9) Build readable next Evidence
# ============================================================

cell94_next_evidence_texts = []


for part in (
    cell94_next_target_text_parts
    or
    []
):

    if isinstance(
        part,
        str,
    ):

        text = part.strip()


    elif isinstance(
        part,
        dict,
    ):

        text = (
            part.get(
                "text"
            )
            or
            part.get(
                "target_text"
            )
            or
            part.get(
                "content"
            )
            or
            ""
        )


        if isinstance(
            text,
            str,
        ):

            text = text.strip()


        else:

            text = ""


    else:

        text = ""


    if text:

        cell94_next_evidence_texts.append(
            text
        )


cell94_next_combined_target_text = (
    "\n\n".join(
        cell94_next_evidence_texts
    )
)


cell94_next_evidence_available = bool(
    cell94_next_combined_target_text.strip()
)


if not cell94_next_evidence_available:
    cell94_failures.append(
        "NEXT_EVIDENCE_EMPTY"
    )


# ============================================================
# 10) Extract next Pilot metadata
# ============================================================

def cell94_unique_scalar_for_key(
    payload,
    key,
):

    values = (
        cell58_find_key_values(
            payload,
            key,
        )
    )


    normalized = []


    for value in values:

        if isinstance(
            value,
            (
                str,
                int,
                float,
                bool,
            ),
        ):

            if value not in normalized:
                normalized.append(
                    value
                )


    if len(
        normalized
    ) == 1:

        return normalized[
            0
        ]


    return None


CELL94_NEXT_STRATUM = (
    cell94_unique_scalar_for_key(
        cell94_next_pilot_packet,
        "stratum",
    )
)


CELL94_NEXT_TASK_FAMILY = (
    cell94_unique_scalar_for_key(
        cell94_next_pilot_packet,
        "task_family",
    )
)


CELL94_NEXT_ARCHETYPE = (
    cell94_unique_scalar_for_key(
        cell94_next_pilot_packet,
        "question_archetype",
    )
)


CELL94_NEXT_AUTHORITY_EXPECTATION = (
    cell94_unique_scalar_for_key(
        cell94_next_pilot_packet,
        "authority_expectation",
    )
)


# ============================================================
# 11) Next Pilot Engine session
# ============================================================

cell94_next_pilot_session = (
    cell13_build_session(
        cell94_next_pilot_packet
    )
)


cell94_next_session_valid = (
    cell94_next_pilot_session.get(
        "current_state"
    )
    ==
    "QUESTION_DRAFT_PENDING"
)


if not cell94_next_session_valid:
    cell94_failures.append(
        "NEXT_SESSION_INVALID"
    )


# ============================================================
# 12) Update completed Human Review cases
# ============================================================

cell94_existing_completed_cases = copy.deepcopy(
    cell93_persisted_registry.get(
        "completed_human_review_cases",
        [],
    )
)


if not isinstance(
    cell94_existing_completed_cases,
    list,
):

    cell94_existing_completed_cases = []

    cell94_failures.append(
        "COMPLETED_CASES_NOT_LIST"
    )


cell94_existing_completed_cases = [
    case
    for case
    in cell94_existing_completed_cases
    if not (
        isinstance(
            case,
            dict,
        )
        and
        case.get(
            "pilot_case_id"
        )
        ==
        CELL94_COMPLETED_PILOT_ID
    )
]


cell94_completed_human_review_cases = (
    cell94_existing_completed_cases
    +
    [
        copy.deepcopy(
            cell94_completed_case
        )
    ]
)


cell94_completed_case_count_valid = (
    len(
        cell94_completed_human_review_cases
    )
    ==
    20
)


if not cell94_completed_case_count_valid:
    cell94_failures.append(
        "COMPLETED_CASE_COUNT_INVALID"
    )


# ============================================================
# 13) Rebuild Author Review queue
# ============================================================

cell94_author_review = copy.deepcopy(
    cell93_persisted_registry.get(
        "author_review",
        {},
    )
)


if not isinstance(
    cell94_author_review,
    dict,
):

    cell94_author_review = {}


cell94_author_ready_cases = copy.deepcopy(
    cell94_completed_human_review_cases
)


cell94_author_review[
    "cases_ready_for_review"
] = copy.deepcopy(
    cell94_author_ready_cases
)


cell94_author_review[
    "case_count"
] = len(
    cell94_author_ready_cases
)


cell94_author_review[
    "final_review_status"
] = "NOT_STARTED"


cell94_author_ready_count_valid = (
    len(
        cell94_author_ready_cases
    )
    ==
    20
)


cell94_author_ready_identity_valid = (
    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell94_author_ready_cases
        if isinstance(
            case,
            dict,
        )
    }
    ==
    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell94_completed_human_review_cases
        if isinstance(
            case,
            dict,
        )
    }
)


if not cell94_author_ready_count_valid:
    cell94_failures.append(
        "AUTHOR_READY_COUNT_INVALID"
    )


if not cell94_author_ready_identity_valid:
    cell94_failures.append(
        "AUTHOR_READY_IDENTITY_INVALID"
    )


# ============================================================
# 14) Preserve Global Authority Policy
# ============================================================

cell94_authority_policy = copy.deepcopy(
    cell93_persisted_registry.get(
        "workflow_policy",
        {},
    ).get(
        "authority_evidence_policy",
        {},
    )
)


cell94_authority_policy_valid = all([
    cell94_authority_policy.get(
        "primary_authority"
    )
    ==
    "Dr. Komeil Roudi",

    cell94_authority_policy.get(
        "secondary_to_primary_promotion"
    )
    ==
    "FORBIDDEN",
])


if not cell94_authority_policy_valid:
    cell94_failures.append(
        "GLOBAL_AUTHORITY_POLICY_INVALID"
    )


# ============================================================
# 15) Build next active workspace
# ============================================================

cell94_next_active_workspace = {
    "pilot_case_id":
        CELL94_NEXT_PILOT_ID,

    "golden_case_id":
        CELL94_NEXT_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL94_NEXT_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL94_NEXT_TARGET_EVIDENCE_ID,

    "status":
        "ACTIVE_HUMAN_REVIEW",

    "workflow_state":
        "QUESTION_DRAFT_PENDING",

    "stratum":
        CELL94_NEXT_STRATUM,

    "task_family":
        CELL94_NEXT_TASK_FAMILY,

    "question_archetype":
        CELL94_NEXT_ARCHETYPE,

    "authority_expectation":
        CELL94_NEXT_AUTHORITY_EXPECTATION,

    "authority_policy":
        copy.deepcopy(
            cell94_authority_policy
        ),

    "source_evidence":
        copy.deepcopy(
            cell94_next_target_text_parts
        ),

    "combined_target_text":
        cell94_next_combined_target_text,

    "question_review":
        None,

    "expected_answer_review":
        None,

    "citation_review":
        None,

    "author_final_review": {
        "status":
            "NOT_STARTED",

        "decision":
            None,

        "notes":
            None,
    },
}


# ============================================================
# 16) Continuation State
# ============================================================

cell94_continuation_state_core = {
    "next_pilot_id":
        CELL94_NEXT_PILOT_ID,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell94_remaining_pilot_ids
        ),

    "remaining_count":
        2,

    "active_pilot_status":
        "QUESTION_DRAFT_PENDING",

    "active_target_evidence_id":
        CELL94_NEXT_TARGET_EVIDENCE_ID,

    "last_completed_pilot_id":
        CELL94_COMPLETED_PILOT_ID,

    "last_completion_type":
        "VERY_LONG_UTTERANCE_PRIMARY_AUTHORITY_PILOT_COMPLETE",

    "authority_policy_active":
        True,
}


cell94_continuation_state_sha256 = (
    cell10_stable_sha(
        cell94_continuation_state_core
    )
)


cell94_continuation_state = {
    **copy.deepcopy(
        cell94_continuation_state_core
    ),

    "continuation_state_sha256":
        cell94_continuation_state_sha256,
}


# ============================================================
# 17) Persist Registry
# ============================================================

cell94_persistent_registry_payload = copy.deepcopy(
    cell93_persisted_registry
)


cell94_persistent_registry_payload[
    "completed_human_review_cases"
] = copy.deepcopy(
    cell94_completed_human_review_cases
)


cell94_persistent_registry_payload[
    "author_review"
] = copy.deepcopy(
    cell94_author_review
)


cell94_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell94_next_active_workspace
)


cell94_persistent_registry_payload[
    "remaining_pilot_count"
] = 2


cell94_persistent_registry_payload[
    "continuation_state"
] = copy.deepcopy(
    cell94_continuation_state
)


cell94_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell94_persistent_registry_payload
    )
)


# ============================================================
# 18) Read-back validation
# ============================================================

cell94_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell94_persisted_active_case = (
    cell94_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell94_persisted_completed_cases = (
    cell94_persisted_registry.get(
        "completed_human_review_cases",
        [],
    )
)


cell94_persisted_author_ready_cases = (
    cell94_persisted_registry.get(
        "author_review",
        {},
    ).get(
        "cases_ready_for_review",
        [],
    )
)


cell94_persisted_authority_policy = (
    cell94_persisted_registry.get(
        "workflow_policy",
        {},
    ).get(
        "authority_evidence_policy",
        {},
    )
)


cell94_persistence_valid = all([
    isinstance(
        cell94_persisted_completed_cases,
        list,
    ),

    len(
        cell94_persisted_completed_cases
    )
    ==
    20,

    isinstance(
        cell94_persisted_author_ready_cases,
        list,
    ),

    len(
        cell94_persisted_author_ready_cases
    )
    ==
    20,

    cell94_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    2,

    cell94_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL94_NEXT_PILOT_ID,

    cell94_persisted_active_case.get(
        "workflow_state"
    )
    ==
    "QUESTION_DRAFT_PENDING",

    cell94_persisted_registry.get(
        "continuation_state",
        {},
    ).get(
        "continuation_state_sha256"
    )
    ==
    cell94_continuation_state_sha256,

    cell94_persisted_authority_policy.get(
        "primary_authority"
    )
    ==
    "Dr. Komeil Roudi",

    cell94_persisted_authority_policy.get(
        "secondary_to_primary_promotion"
    )
    ==
    "FORBIDDEN",

    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell94_persisted_completed_cases
        if isinstance(
            case,
            dict,
        )
    }
    ==
    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell94_persisted_author_ready_cases
        if isinstance(
            case,
            dict,
        )
    },
])


if not cell94_persistence_valid:
    cell94_failures.append(
        "REGISTRY_READBACK_INVALID"
    )


# ============================================================
# 19) Safety
# ============================================================

cell94_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell94_full_572_blocked = (
    cell93_full_572_blocked
    is True
)


cell94_no_commit_push = True


# ============================================================
# 20) Internal Gates
# ============================================================

cell94_gate_checks = {
    "no_failures":
        len(
            cell94_failures
        )
        ==
        0,

    "citation_accept_valid":
        cell94_citation_accept_valid,

    "queue_advance_valid":
        cell94_queue_advance_valid,

    "next_identity_valid":
        cell94_next_identity_valid,

    "next_evidence_available":
        cell94_next_evidence_available,

    "next_session_valid":
        cell94_next_session_valid,

    "completed_case_count_valid":
        cell94_completed_case_count_valid,

    "author_ready_count_valid":
        cell94_author_ready_count_valid,

    "author_ready_identity_valid":
        cell94_author_ready_identity_valid,

    "authority_policy_valid":
        cell94_authority_policy_valid,

    "persistence_valid":
        cell94_persistence_valid,

    "base_seed_unchanged":
        cell94_base_seed_unchanged,

    "full_572_blocked":
        cell94_full_572_blocked,

    "no_commit_push":
        cell94_no_commit_push,
}


cell94_gate = all(
    cell94_gate_checks.values()
)


# ============================================================
# 21) Diagnostics
# ============================================================

print(
    "=== PILOT #21 COMPLETE + QUEUE ADVANCE + NEXT PILOT ==="
)


print(
    "\n=== COMPLETED PILOT ==="
)

print(
    "Pilot:",
    CELL94_COMPLETED_PILOT_ID,
)

print(
    "Target:",
    CELL93_TARGET_EVIDENCE_ID,
)

print(
    "Retarget used:",
    False,
)

print(
    "Question:",
    "ACCEPT",
)

print(
    "Expected Answer:",
    "ACCEPT",
)

print(
    "Citation:",
    "ACCEPT",
)

print(
    "Final Citation Mapping:",
    cell93_citation_mapping_id,
)

print(
    "Final Engine State:",
    cell94_complete_pending_session.get(
        "current_state"
    ),
)


print(
    "\n=== AUTHORITY RESULT ==="
)

print(
    "Primary Authority:",
    "Dr. Komeil Roudi",
)

print(
    "Source Attribution preserved:",
    True,
)

print(
    "Internal speaker transition:",
    False,
)

print(
    "Raw Source mutated:",
    False,
)

print(
    "Parser mutated:",
    False,
)


print(
    "\n=== QUEUE ADVANCE ==="
)

print(
    "Before:",
    len(
        cell94_previous_remaining_pilot_ids
    ),
)

print(
    "After:",
    len(
        cell94_remaining_pilot_ids
    ),
)

print(
    "Queue advanced exactly once:",
    cell94_queue_advance_valid,
)


print(
    "\n=== HUMAN REVIEW REGISTRY ==="
)

print(
    "Completed Human Review cases:",
    len(
        cell94_completed_human_review_cases
    ),
)

print(
    "Cases ready for Author Review:",
    len(
        cell94_author_ready_cases
    ),
)

print(
    "Author Final Review:",
    "NOT_STARTED",
)

print(
    "Persistence valid:",
    cell94_persistence_valid,
)


print(
    "\n=== NEXT PILOT ==="
)

print(
    "Pilot:",
    CELL94_NEXT_PILOT_ID,
)

print(
    "Golden Case:",
    CELL94_NEXT_GOLDEN_CASE_ID,
)

print(
    "Task Draft:",
    CELL94_NEXT_TASK_DRAFT_ID,
)

print(
    "Target Evidence:",
    CELL94_NEXT_TARGET_EVIDENCE_ID,
)

print(
    "Stratum:",
    CELL94_NEXT_STRATUM,
)

print(
    "Task Family:",
    CELL94_NEXT_TASK_FAMILY,
)

print(
    "Question Archetype:",
    CELL94_NEXT_ARCHETYPE,
)

print(
    "Authority Expectation:",
    CELL94_NEXT_AUTHORITY_EXPECTATION,
)

print(
    "Workflow:",
    cell94_next_pilot_session.get(
        "current_state"
    ),
)


print(
    "\n=== NEXT SOURCE EVIDENCE ==="
)

print(
    cell94_next_combined_target_text
)


print(
    "\n=== CONTINUATION STATE ==="
)

print(
    "Remaining Pilots:",
    2,
)

print(
    "Continuation SHA256:",
    cell94_continuation_state_sha256,
)


print(
    "\n=== ALL INTERNAL GATES ==="
)

for name, status in cell94_gate_checks.items():

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== SAFETY ==="
)

print(
    "Base Seed unchanged:",
    cell94_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell94_full_572_blocked,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell94_failures
)


print(
    "\nCODE CELL 94 GATE:",
    (
        "PASS"
        if cell94_gate
        else "FAIL"
    ),
)


assert cell94_gate, (
    "Pilot #21 Citation ACCEPT / completion / queue advance / "
    "next Pilot preparation failed. Repair this SAME Code Cell 94."
)

=== PILOT #21 COMPLETE + QUEUE ADVANCE + NEXT PILOT ===

=== COMPLETED PILOT ===
Pilot: question-pilot:dd01033c15c375ae7124e8c7f86467a3
Target: target-evidence:438f0de8cf57f8d20e77420ec63ef3fc
Retarget used: False
Question: ACCEPT
Expected Answer: ACCEPT
Citation: ACCEPT
Final Citation Mapping: citation-mapping:3c7756c07617e67b3bea0616f9b4ea99
Final Engine State: PILOT_COMPLETE_PENDING_QUEUE_ADVANCE

=== AUTHORITY RESULT ===
Primary Authority: Dr. Komeil Roudi
Source Attribution preserved: True
Internal speaker transition: False
Raw Source mutated: False
Parser mutated: False

=== QUEUE ADVANCE ===
Before: 3
After: 2
Queue advanced exactly once: True

=== HUMAN REVIEW REGISTRY ===
Completed Human Review cases: 20
Cases ready for Author Review: 20
Author Final Review: NOT_STARTED
Persistence valid: True

=== NEXT PILOT ===
Pilot: question-pilot:733355977d8932b0778e0d85715c446d
Golden Case: golden-case:2f17f9a429de262f679cf19489990878
Task Draft: task-draft:e8db740e645eb6f68a0f154f7a09e2

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 95 — Pilot #22 Token-Fallback Fragment / Authority Diagnostic

Pilot فعلی از نوع:

`TOKEN_FALLBACK_FRAGMENT_RETRIEVAL`

و Question Archetype آن:

`FRAGMENT_RECONSTRUCTION`

است.

Target از وسط یک جمله شروع شده و در ادامه چند مفهوم مالی را پوشش می‌دهد:

- حد ضرر و حد منفعت؛
- مدیریت ریسک سرمایه‌گذاری؛
- تفاوت زیان و خسارت؛
- بیمه و اصطلاحات اصلی آن.

پیش از ساخت سؤال بررسی می‌کنیم:

- Fragment چند Part دارد؛
- Speaker اصلی چه کسی است؛
- آیا Source Attribution به Dr. Komeil Roudi می‌رسد؛
- Fragment از نظر ابتدا/انتها ناقص یا بریده است یا نه؛
- کدام بخش‌های مفهومی داخل Fragment به‌اندازه کافی کامل هستند؛
- آیا از خود Fragment می‌توان یک سؤال طبیعی و مستقل ساخت؛
- یا باید برای حفظ کیفیت Golden Dataset، Target را Retarget یا با Source Context تکمیل کرد.

این Cell فقط Diagnostic است و هیچ داده‌ای را تغییر نمی‌دهد.

</div>

In [95]:
# Code Cell 95 — REPLACEMENT

cell95_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell94_gate") is not True:
    cell95_failures.append(
        "CELL94_NOT_PASS"
    )


cell95_source_parts = globals().get(
    "cell94_next_target_text_parts"
)


if not isinstance(
    cell95_source_parts,
    list,
):
    cell95_failures.append(
        "NEXT_TARGET_PARTS_MISSING"
    )
    cell95_source_parts = []


# ============================================================
# 2) Pilot identity
# ============================================================

CELL95_PILOT_ID = (
    CELL94_NEXT_PILOT_ID
)


CELL95_TARGET_EVIDENCE_ID = (
    CELL94_NEXT_TARGET_EVIDENCE_ID
)


CELL95_TASK_FAMILY = (
    CELL94_NEXT_TASK_FAMILY
)


CELL95_ARCHETYPE = (
    CELL94_NEXT_ARCHETYPE
)


CELL95_AUTHORITY_EXPECTATION = (
    CELL94_NEXT_AUTHORITY_EXPECTATION
)


# ============================================================
# 3) Inspect Target Parts
# ============================================================

cell95_part_diagnostics = []


for index, part in enumerate(
    cell95_source_parts
):

    if isinstance(
        part,
        dict,
    ):

        speaker_state = (
            part.get(
                "speaker_state"
            )
            or
            {}
        )


        attribution = (
            part.get(
                "attribution"
            )
            or
            {}
        )


        locator = (
            part.get(
                "locator"
            )
            or
            {}
        )


        text = (
            part.get(
                "text"
            )
            or
            part.get(
                "target_text"
            )
            or
            part.get(
                "content"
            )
            or
            ""
        )


        cell95_part_diagnostics.append({
            "index":
                index,

            "document_id":
                part.get(
                    "document_id"
                ),

            "role":
                part.get(
                    "role"
                ),

            "speaker_base_status":
                speaker_state.get(
                    "base_status"
                ),

            "speaker_base_raw":
                speaker_state.get(
                    "base_raw"
                ),

            "speaker_effective_label":
                speaker_state.get(
                    "effective_label"
                ),

            "speaker_effective_canonical":
                speaker_state.get(
                    "effective_canonical"
                ),

            "speaker_effective_primary":
                speaker_state.get(
                    "effective_primary"
                ),

            "attribution_speaker":
                attribution.get(
                    "speaker"
                ),

            "attribution_status":
                attribution.get(
                    "attribution_status"
                ),

            "content_authority_type":
                attribution.get(
                    "content_authority_type"
                ),

            "source_line_start":
                locator.get(
                    "source_line_start"
                ),

            "source_line_end":
                locator.get(
                    "source_line_end"
                ),

            "char_start":
                locator.get(
                    "char_start"
                ),

            "char_end":
                locator.get(
                    "char_end"
                ),

            "text":
                text,

            "raw_part":
                copy.deepcopy(
                    part
                ),
        })


    else:

        cell95_part_diagnostics.append({
            "index":
                index,

            "document_id":
                None,

            "role":
                None,

            "speaker_base_status":
                None,

            "speaker_base_raw":
                None,

            "speaker_effective_label":
                None,

            "speaker_effective_canonical":
                None,

            "speaker_effective_primary":
                None,

            "attribution_speaker":
                None,

            "attribution_status":
                None,

            "content_authority_type":
                None,

            "source_line_start":
                None,

            "source_line_end":
                None,

            "char_start":
                None,

            "char_end":
                None,

            "text":
                str(
                    part
                ),

            "raw_part":
                copy.deepcopy(
                    part
                ),
        })


# ============================================================
# 4) Resolve authority safely
# ============================================================

cell95_resolved_parts = []


for row in cell95_part_diagnostics:

    resolved_speaker = (
        row.get(
            "speaker_effective_canonical"
        )
        or
        row.get(
            "speaker_effective_label"
        )
        or
        row.get(
            "speaker_base_raw"
        )
        or
        row.get(
            "attribution_speaker"
        )
    )


    explicit_primary = (
        row.get(
            "speaker_effective_primary"
        )
        is True
    )


    canonical_primary = (
        resolved_speaker
        ==
        "Dr. Komeil Roudi"
    )


    is_primary = (
        explicit_primary
        or
        canonical_primary
    )


    is_explicit_nonprimary = all([
        resolved_speaker
        is not None,

        resolved_speaker
        !=
        "Dr. Komeil Roudi",

        row.get(
            "speaker_base_status"
        )
        ==
        "EXPLICIT_SOURCE_MARKER",
    ])


    is_unresolved = (
        resolved_speaker
        is None
        or
        row.get(
            "speaker_base_status"
        )
        in {
            None,
            "UNRESOLVED",
            "UNKNOWN",
        }
    )


    cell95_resolved_parts.append({
        **copy.deepcopy(
            row
        ),

        "resolved_speaker":
            resolved_speaker,

        "is_primary":
            is_primary,

        "is_explicit_nonprimary":
            is_explicit_nonprimary,

        "is_unresolved":
            is_unresolved,
    })


# ============================================================
# 5) Structural validation
# ============================================================

cell95_primary_parts = [
    row
    for row
    in cell95_resolved_parts
    if row.get(
        "is_primary"
    )
    is True
]


cell95_nonprimary_parts = [
    row
    for row
    in cell95_resolved_parts
    if row.get(
        "is_explicit_nonprimary"
    )
    is True
]


cell95_unresolved_parts = [
    row
    for row
    in cell95_resolved_parts
    if row.get(
        "is_unresolved"
    )
    is True
]


cell95_exact_one_part = (
    len(
        cell95_resolved_parts
    )
    ==
    1
)


cell95_primary_candidate = (
    cell95_primary_parts[
        0
    ]
    if len(
        cell95_primary_parts
    )
    ==
    1
    else None
)


cell95_primary_valid = all([
    cell95_exact_one_part,

    len(
        cell95_primary_parts
    )
    ==
    1,

    len(
        cell95_unresolved_parts
    )
    ==
    0,

    isinstance(
        cell95_primary_candidate,
        dict,
    ),

    (
        cell95_primary_candidate.get(
            "resolved_speaker"
        )
        if isinstance(
            cell95_primary_candidate,
            dict,
        )
        else None
    )
    ==
    "Dr. Komeil Roudi",
])


# ============================================================
# 6) Task-family validation
# ============================================================

cell95_expected_task_family = (
    CELL95_TASK_FAMILY
    ==
    "TOKEN_FALLBACK_FRAGMENT_RETRIEVAL"
)


cell95_expected_archetype = (
    CELL95_ARCHETYPE
    ==
    "FRAGMENT_RECONSTRUCTION"
)


cell95_expected_authority_rule = (
    CELL95_AUTHORITY_EXPECTATION
    ==
    "PRESERVE_SOURCE_ATTRIBUTION"
)


# ============================================================
# 7) Target text
# ============================================================

cell95_text = (
    cell95_resolved_parts[
        0
    ].get(
        "text",
        "",
    )
    if len(
        cell95_resolved_parts
    )
    >=
    1
    else
    ""
)


def cell95_norm(
    text,
):

    if not isinstance(
        text,
        str,
    ):
        return ""


    value = (
        text
        .replace(
            "ي",
            "ی",
        )
        .replace(
            "ك",
            "ک",
        )
        .replace(
            "\u200c",
            "",
        )
        .replace(
            "\ufeff",
            "",
        )
    )


    value = re.sub(
        r"\s+",
        " ",
        value,
    )


    return value.strip()


cell95_surface = cell95_norm(
    cell95_text
)


# ============================================================
# 8) Fragment-boundary signals
# ============================================================

cell95_fragment_boundary_signals = {
    "starts_mid_thought":
        cell95_surface.startswith(
            "برداشت نمی کنم"
        ),

    "starts_without_subject_context":
        (
            "برداشت نمی کنم همونجا"
            in
            cell95_surface[
                :100
            ]
        ),

    "contains_topic_transition_to_insurance":
        (
            "توی دارایی ها یه عنوانی هست تحت عنوان بیمه"
            in
            cell95_surface
        ),

    "ends_with_complete_term_list":
        (
            "فرانشیز یا خودپرداخت"
            in
            cell95_surface[
                -300:
            ]
        ),
}


cell95_fragment_start_incomplete = any([
    cell95_fragment_boundary_signals[
        "starts_mid_thought"
    ],

    cell95_fragment_boundary_signals[
        "starts_without_subject_context"
    ],
])


cell95_fragment_end_usable = (
    cell95_fragment_boundary_signals[
        "ends_with_complete_term_list"
    ]
)


# ============================================================
# 9) Investment-risk concept coverage
# ============================================================

cell95_risk_claims = {
    "stop_loss":
        (
            "حد ضرر"
            in
            cell95_surface
        ),

    "profit_target":
        (
            "حد منفعت"
            in
            cell95_surface
        ),

    "exit_on_loss":
        (
            "هشتصد هزار"
            in
            cell95_surface
            and
            "از اون بازار خارج"
            in
            cell95_surface
        ),

    "exit_on_profit":
        (
            "یه میلیون و چهارصد"
            in
            cell95_surface
            and
            "بفروش"
            in
            cell95_surface
        ),

    "risk_management":
        (
            "مدیریت ریسکی"
            in
            cell95_surface
        ),

    "investment_loss":
        (
            "اصل سرمایه"
            in
            cell95_surface
            and
            "زیان"
            in
            cell95_surface
        ),
}


cell95_risk_section_sufficient = all(
    cell95_risk_claims.values()
)


# ============================================================
# 10) Insurance concept coverage
# ============================================================

cell95_insurance_claims = {
    "insurance_for_asset_damage":
        (
            "بیمه"
            in
            cell95_surface
            and
            "خسارت"
            in
            cell95_surface
        ),

    "insurer":
        (
            "بیمه گر"
            in
            cell95_surface
        ),

    "policyholder":
        (
            "بیمه گذار"
            in
            cell95_surface
        ),

    "premium":
        (
            "حق بیمه"
            in
            cell95_surface
        ),

    "coverage":
        (
            "پوشش بیمه"
            in
            cell95_surface
        ),

    "deductible":
        (
            "فرانشیز"
            in
            cell95_surface
            and
            "خودپرداخت"
            in
            cell95_surface
        ),
}


cell95_insurance_section_sufficient = all(
    cell95_insurance_claims.values()
)


# ============================================================
# 11) Topic coherence
# ============================================================

cell95_distinct_topic_blocks = []


if cell95_risk_section_sufficient:

    cell95_distinct_topic_blocks.append(
        "INVESTMENT_RISK_MANAGEMENT"
    )


if cell95_insurance_section_sufficient:

    cell95_distinct_topic_blocks.append(
        "INSURANCE_RISK_TRANSFER"
    )


cell95_multiple_complete_topics = (
    len(
        cell95_distinct_topic_blocks
    )
    >=
    2
)


# ============================================================
# 12) Authority Policy
# ============================================================

cell95_authority_policy = copy.deepcopy(
    cell94_persisted_registry.get(
        "workflow_policy",
        {},
    ).get(
        "authority_evidence_policy",
        {},
    )
)


cell95_authority_policy_valid = all([
    cell95_authority_policy.get(
        "primary_authority"
    )
    ==
    "Dr. Komeil Roudi",

    cell95_authority_policy.get(
        "secondary_to_primary_promotion"
    )
    ==
    "FORBIDDEN",
])


if not cell95_authority_policy_valid:
    cell95_failures.append(
        "GLOBAL_AUTHORITY_POLICY_INVALID"
    )


# ============================================================
# 13) Preliminary quality assessment
# ============================================================

cell95_authority_usable = all([
    cell95_primary_valid,
    cell95_authority_policy_valid,
])


cell95_content_has_usable_subsection = (
    cell95_risk_section_sufficient
    or
    cell95_insurance_section_sufficient
)


cell95_fragment_requires_context_decision = any([
    cell95_fragment_start_incomplete,
    cell95_multiple_complete_topics,
    not cell95_primary_valid,
])


cell95_safe_to_draft_question_immediately = all([
    cell95_authority_usable,
    cell95_content_has_usable_subsection,
    not cell95_fragment_requires_context_decision,
])


cell95_needs_source_context_inspection = all([
    cell95_content_has_usable_subsection,
    cell95_fragment_requires_context_decision,
])


cell95_retarget_likely_required = False


# ============================================================
# 14) Safety
# ============================================================

cell95_registry_write = False

cell95_queue_advance = False

cell95_human_decision_added = False

cell95_target_changed = False

cell95_citation_changed = False

cell95_raw_source_mutated = False

cell95_parser_mutated = False

cell95_commit_push = False


# ============================================================
# 15) Diagnostic Gate
# ============================================================

# This is a diagnostic cell.
# Authority usability is intentionally NOT required for PASS.
# If attribution is unresolved, the next step will inspect source context.

cell95_gate_checks = {
    "no_failures":
        len(
            cell95_failures
        )
        ==
        0,

    "expected_task_family":
        cell95_expected_task_family,

    "expected_archetype":
        cell95_expected_archetype,

    "expected_authority_rule":
        cell95_expected_authority_rule,

    "target_parts_present":
        len(
            cell95_resolved_parts
        )
        >
        0,

    "content_has_usable_subsection":
        cell95_content_has_usable_subsection,

    "authority_policy_valid":
        cell95_authority_policy_valid,
}


cell95_gate = all(
    cell95_gate_checks.values()
)


# ============================================================
# 16) Diagnostics
# ============================================================

print(
    "=== PILOT #22 TOKEN-FALLBACK FRAGMENT DIAGNOSTIC ==="
)


print(
    "\n=== PILOT ==="
)

print(
    "Pilot:",
    CELL95_PILOT_ID,
)

print(
    "Target Evidence:",
    CELL95_TARGET_EVIDENCE_ID,
)

print(
    "Task Family:",
    CELL95_TASK_FAMILY,
)

print(
    "Question Archetype:",
    CELL95_ARCHETYPE,
)

print(
    "Authority Expectation:",
    CELL95_AUTHORITY_EXPECTATION,
)


print(
    "\n=== TARGET STRUCTURE ==="
)

print(
    "Part count:",
    len(
        cell95_resolved_parts
    ),
)

print(
    "Primary parts:",
    len(
        cell95_primary_parts
    ),
)

print(
    "Explicit Non-Primary parts:",
    len(
        cell95_nonprimary_parts
    ),
)

print(
    "Unresolved parts:",
    len(
        cell95_unresolved_parts
    ),
)

print(
    "Fragment chars:",
    len(
        cell95_text
    ),
)


for row in cell95_resolved_parts:

    print(
        "\n"
        +
        "="
        *
        72
    )

    print(
        "PART:",
        row.get(
            "index"
        ),
    )

    print(
        "Document:",
        row.get(
            "document_id"
        ),
    )

    print(
        "Role:",
        row.get(
            "role"
        ),
    )

    print(
        "Base status:",
        row.get(
            "speaker_base_status"
        ),
    )

    print(
        "Base raw:",
        row.get(
            "speaker_base_raw"
        ),
    )

    print(
        "Effective label:",
        row.get(
            "speaker_effective_label"
        ),
    )

    print(
        "Effective canonical:",
        row.get(
            "speaker_effective_canonical"
        ),
    )

    print(
        "Effective primary:",
        row.get(
            "speaker_effective_primary"
        ),
    )

    print(
        "Attribution speaker:",
        row.get(
            "attribution_speaker"
        ),
    )

    print(
        "Attribution status:",
        row.get(
            "attribution_status"
        ),
    )

    print(
        "Content authority type:",
        row.get(
            "content_authority_type"
        ),
    )

    print(
        "Resolved speaker:",
        row.get(
            "resolved_speaker"
        ),
    )

    print(
        "Is Primary:",
        row.get(
            "is_primary"
        ),
    )

    print(
        "Is Explicit Non-Primary:",
        row.get(
            "is_explicit_nonprimary"
        ),
    )

    print(
        "Is Unresolved:",
        row.get(
            "is_unresolved"
        ),
    )

    print(
        "Source lines:",
        (
            row.get(
                "source_line_start"
            ),
            row.get(
                "source_line_end"
            ),
        ),
    )

    print(
        "Char range:",
        (
            row.get(
                "char_start"
            ),
            row.get(
                "char_end"
            ),
        ),
    )


print(
    "\n=== FRAGMENT BOUNDARY ==="
)

for name, status in (
    cell95_fragment_boundary_signals.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "Fragment start incomplete:",
    cell95_fragment_start_incomplete,
)

print(
    "Fragment end usable:",
    cell95_fragment_end_usable,
)


print(
    "\n=== INVESTMENT RISK SECTION ==="
)

for name, status in (
    cell95_risk_claims.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "Risk section sufficient:",
    cell95_risk_section_sufficient,
)


print(
    "\n=== INSURANCE SECTION ==="
)

for name, status in (
    cell95_insurance_claims.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "Insurance section sufficient:",
    cell95_insurance_section_sufficient,
)


print(
    "\n=== TOPIC STRUCTURE ==="
)

print(
    "Complete topic blocks:",
    cell95_distinct_topic_blocks,
)

print(
    "Multiple complete topics:",
    cell95_multiple_complete_topics,
)


print(
    "\n=== AUTHORITY ASSESSMENT ==="
)

print(
    "Primary valid:",
    cell95_primary_valid,
)

print(
    "Authority usable:",
    cell95_authority_usable,
)

print(
    "Authority policy valid:",
    cell95_authority_policy_valid,
)


print(
    "\n=== TARGET ASSESSMENT ==="
)

print(
    "Content has usable subsection:",
    cell95_content_has_usable_subsection,
)

print(
    "Fragment requires context decision:",
    cell95_fragment_requires_context_decision,
)

print(
    "Safe to draft question immediately:",
    cell95_safe_to_draft_question_immediately,
)

print(
    "Needs source-context inspection:",
    cell95_needs_source_context_inspection,
)

print(
    "Retarget likely required:",
    cell95_retarget_likely_required,
)

print(
    "IMPORTANT: No Human Question/Answer/Citation decision "
    "has been recorded."
)


print(
    "\n=== ALL INTERNAL GATES ==="
)

for name, status in cell95_gate_checks.items():

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== SAFETY ==="
)

print(
    "Registry write:",
    cell95_registry_write,
)

print(
    "Queue advance:",
    cell95_queue_advance,
)

print(
    "Human Decision added:",
    cell95_human_decision_added,
)

print(
    "Target changed:",
    cell95_target_changed,
)

print(
    "Citation changed:",
    cell95_citation_changed,
)

print(
    "Raw Source mutated:",
    cell95_raw_source_mutated,
)

print(
    "Parser mutated:",
    cell95_parser_mutated,
)

print(
    "Commit / Push:",
    cell95_commit_push,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell95_failures
)


print(
    "\nCODE CELL 95 GATE:",
    (
        "PASS"
        if cell95_gate
        else
        "FAIL"
    ),
)


assert cell95_gate, (
    "Pilot #22 token-fallback fragment diagnostic failed. "
    "Repair this SAME Code Cell 95 before continuing."
)

=== PILOT #22 TOKEN-FALLBACK FRAGMENT DIAGNOSTIC ===

=== PILOT ===
Pilot: question-pilot:733355977d8932b0778e0d85715c446d
Target Evidence: target-evidence:072e02138b13c2784ed9e7a99f4d33da
Task Family: TOKEN_FALLBACK_FRAGMENT_RETRIEVAL
Question Archetype: FRAGMENT_RECONSTRUCTION
Authority Expectation: PRESERVE_SOURCE_ATTRIBUTION

=== TARGET STRUCTURE ===
Part count: 1
Primary parts: 0
Explicit Non-Primary parts: 1
Unresolved parts: 0
Fragment chars: 2006

PART: 0
Document: program:khane-to:episode-013:utterance:00034
Role: SOURCE_SENTENCE
Base status: EXPLICIT_SOURCE_MARKER
Base raw: دکتر کمیل رودی
Effective label: دکتر کمیل رودی
Effective canonical: None
Effective primary: None
Attribution speaker: None
Attribution status: None
Content authority type: None
Resolved speaker: دکتر کمیل رودی
Is Primary: False
Is Explicit Non-Primary: True
Is Unresolved: False
Source lines: (141, 143)
Char range: (None, None)

=== FRAGMENT BOUNDARY ===
starts_mid_thought: True
starts_without_subject_conte

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 96 — Pilot #22 Raw Source Context + Primary Name Resolution Diagnostic

Cell 95 نشان داد که Source Marker به‌صورت صریح:

`دکتر کمیل رودی`

ثبت شده، اما Canonical Speaker در Parser مقدار `None` دارد.

بنابراین برچسب Non-Primary در Diagnostic قبلی به معنی Non-Primary بودن واقعی Speaker نیست؛ بلکه نتیجه ناقص‌بودن Canonicalization است.

در این Cell:

- فایل خام Episode 13 از `khane-to` پیدا می‌شود؛
- خطوط اطراف Locator فعلی بررسی می‌شوند؛
- Speaker Markerهای واقعی Source نمایش داده می‌شوند؛
- تطابق صریح `دکتر کمیل رودی` با Primary Authority بررسی می‌شود؛
- ابتدا و انتهای Fragment در Source خام دیده می‌شود؛
- مشخص می‌کنیم آیا برای این Pilot می‌توان بدون Retarget از یک بخش کامل و مستقل استفاده کرد یا خیر.

هیچ Parser، Raw Source، Target، Registry یا Queue تغییر نمی‌کند.

</div>

In [96]:
# Code Cell 96

from pathlib import Path


cell96_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell95_gate") is not True:
    cell96_failures.append(
        "CELL95_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell95_resolved_parts"
    ),
    list,
):
    cell96_failures.append(
        "CELL95_RESOLVED_PARTS_MISSING"
    )


# ============================================================
# 2) Current Target identity
# ============================================================

CELL96_PILOT_ID = (
    CELL95_PILOT_ID
)


CELL96_TARGET_EVIDENCE_ID = (
    CELL95_TARGET_EVIDENCE_ID
)


cell96_target_part = (
    copy.deepcopy(
        cell95_resolved_parts[
            0
        ]
    )
    if isinstance(
        globals().get(
            "cell95_resolved_parts"
        ),
        list,
    )
    and len(
        cell95_resolved_parts
    )
    >=
    1
    else
    {}
)


CELL96_DOCUMENT_ID = (
    cell96_target_part.get(
        "document_id"
    )
)


CELL96_SOURCE_LINE_START = (
    cell96_target_part.get(
        "source_line_start"
    )
)


CELL96_SOURCE_LINE_END = (
    cell96_target_part.get(
        "source_line_end"
    )
)


CELL96_BASE_RAW_SPEAKER = (
    cell96_target_part.get(
        "speaker_base_raw"
    )
)


# ============================================================
# 3) Normalize names
# ============================================================

def cell96_norm_name(
    value,
):

    if not isinstance(
        value,
        str,
    ):
        return ""


    value = (
        value
        .replace(
            "ي",
            "ی",
        )
        .replace(
            "ك",
            "ک",
        )
        .replace(
            "\u200c",
            " ",
        )
        .replace(
            "\ufeff",
            "",
        )
    )


    value = re.sub(
        r"[^\w\sآ-ی]",
        " ",
        value,
    )


    value = re.sub(
        r"\s+",
        " ",
        value,
    )


    return value.strip()


cell96_base_raw_normalized = (
    cell96_norm_name(
        CELL96_BASE_RAW_SPEAKER
    )
)


cell96_primary_name_marker_match = (
    cell96_base_raw_normalized
    ==
    cell96_norm_name(
        "دکتر کمیل رودی"
    )
)


cell96_primary_name_alias_resolution = {
    "raw_source_marker":
        CELL96_BASE_RAW_SPEAKER,

    "normalized_source_marker":
        cell96_base_raw_normalized,

    "canonical_identity":
        "Dr. Komeil Roudi",

    "exact_primary_name_match":
        cell96_primary_name_marker_match,

    "resolution_basis":
        "EXPLICIT_SOURCE_NAME_MARKER",

    "parser_mutation":
        False,

    "raw_source_mutation":
        False,
}


# ============================================================
# 4) Find repository root
# ============================================================

cell96_registry_path = Path(
    cell21_registry_path
).resolve()


cell96_repo_root_candidates = []


for candidate in [
    cell96_registry_path.parent,
    *cell96_registry_path.parents,
]:

    if (
        (
            candidate
            /
            "data"
        ).exists()
        and
        (
            candidate
            /
            "notebooks"
        ).exists()
    ):

        cell96_repo_root_candidates.append(
            candidate
        )


cell96_repo_root = (
    cell96_repo_root_candidates[
        0
    ]
    if cell96_repo_root_candidates
    else None
)


if cell96_repo_root is None:
    cell96_failures.append(
        "REPO_ROOT_NOT_FOUND"
    )


# ============================================================
# 5) Locate raw transcript file
# ============================================================

cell96_transcript_root = (
    cell96_repo_root
    /
    "data"
    /
    "transcripts"
    /
    "final"
    if cell96_repo_root is not None
    else None
)


cell96_guessed_paths = []


if cell96_transcript_root is not None:

    cell96_guessed_paths.extend([
        (
            cell96_transcript_root
            /
            "khane-to"
            /
            "episode-13.md"
        ),

        (
            cell96_transcript_root
            /
            "khane-to"
            /
            "episode-013.md"
        ),

        (
            cell96_transcript_root
            /
            "khane_to"
            /
            "episode-13.md"
        ),

        (
            cell96_transcript_root
            /
            "khane_to"
            /
            "episode-013.md"
        ),
    ])


cell96_existing_guessed_paths = [
    path
    for path
    in cell96_guessed_paths
    if path.exists()
    and path.is_file()
]


cell96_search_candidates = []


if (
    cell96_transcript_root is not None
    and
    cell96_transcript_root.exists()
):

    for path in cell96_transcript_root.rglob(
        "*.md"
    ):

        path_text = str(
            path
        ).lower()


        name_text = (
            path.name.lower()
        )


        path_score = 0


        if (
            "khane-to"
            in
            path_text
            or
            "khane_to"
            in
            path_text
        ):

            path_score += 5


        if (
            "episode-13"
            in
            name_text
            or
            "episode_13"
            in
            name_text
            or
            "episode-013"
            in
            name_text
            or
            "episode_013"
            in
            name_text
        ):

            path_score += 5


        if path_score > 0:

            cell96_search_candidates.append(
                {
                    "path":
                        path,

                    "path_score":
                        path_score,
                }
            )


# ============================================================
# 6) Score candidate files using content
# ============================================================

cell96_candidate_records = []


cell96_target_probe_phrases = [
    "برداشت نمی کنم",
    "حد ضرر",
    "فرانشیز",
    "حق بیمه",
    "دکتر کمیل رودی",
]


cell96_all_candidate_paths = []


for path in cell96_existing_guessed_paths:

    if path not in cell96_all_candidate_paths:
        cell96_all_candidate_paths.append(
            path
        )


for record in cell96_search_candidates:

    path = record[
        "path"
    ]

    if path not in cell96_all_candidate_paths:
        cell96_all_candidate_paths.append(
            path
        )


for path in cell96_all_candidate_paths:

    try:

        content = path.read_text(
            encoding="utf-8"
        )

    except Exception:

        continue


    normalized_content = (
        content
        .replace(
            "\u200c",
            " "
        )
        .replace(
            "ي",
            "ی",
        )
        .replace(
            "ك",
            "ک",
        )
    )


    normalized_content = re.sub(
        r"\s+",
        " ",
        normalized_content,
    )


    score = 0


    path_text = str(
        path
    ).lower()


    if (
        "khane-to"
        in
        path_text
        or
        "khane_to"
        in
        path_text
    ):

        score += 10


    if (
        "episode-13"
        in
        path.name.lower()
        or
        "episode_13"
        in
        path.name.lower()
        or
        "episode-013"
        in
        path.name.lower()
        or
        "episode_013"
        in
        path.name.lower()
    ):

        score += 10


    phrase_hits = {}


    for phrase in cell96_target_probe_phrases:

        normalized_phrase = (
            phrase
            .replace(
                "\u200c",
                " "
            )
        )


        hit = (
            normalized_phrase
            in
            normalized_content
        )


        phrase_hits[
            phrase
        ] = hit


        if hit:
            score += 5


    cell96_candidate_records.append({
        "path":
            path,

        "score":
            score,

        "phrase_hits":
            phrase_hits,

        "content":
            content,
    })


cell96_candidate_records = sorted(
    cell96_candidate_records,
    key=lambda item: (
        -item[
            "score"
        ],
        str(
            item[
                "path"
            ]
        ),
    ),
)


cell96_best_candidate = (
    cell96_candidate_records[
        0
    ]
    if cell96_candidate_records
    else None
)


cell96_source_path = (
    cell96_best_candidate[
        "path"
    ]
    if cell96_best_candidate is not None
    else None
)


cell96_source_text = (
    cell96_best_candidate[
        "content"
    ]
    if cell96_best_candidate is not None
    else
    ""
)


cell96_source_file_found = (
    cell96_source_path is not None
    and
    cell96_source_path.exists()
)


if not cell96_source_file_found:
    cell96_failures.append(
        "RAW_SOURCE_FILE_NOT_FOUND"
    )


# ============================================================
# 7) Inspect exact source context
# ============================================================

cell96_source_lines = (
    cell96_source_text.splitlines()
    if cell96_source_text
    else
    []
)


cell96_locator_valid = all([
    isinstance(
        CELL96_SOURCE_LINE_START,
        int,
    ),

    isinstance(
        CELL96_SOURCE_LINE_END,
        int,
    ),

    CELL96_SOURCE_LINE_START
    >=
    1
    if isinstance(
        CELL96_SOURCE_LINE_START,
        int,
    )
    else False,

    CELL96_SOURCE_LINE_END
    >=
    CELL96_SOURCE_LINE_START
    if (
        isinstance(
            CELL96_SOURCE_LINE_START,
            int,
        )
        and
        isinstance(
            CELL96_SOURCE_LINE_END,
            int,
        )
    )
    else False,

    CELL96_SOURCE_LINE_END
    <=
    len(
        cell96_source_lines
    )
    if isinstance(
        CELL96_SOURCE_LINE_END,
        int,
    )
    else False,
])


if not cell96_locator_valid:
    cell96_failures.append(
        "SOURCE_LOCATOR_INVALID"
    )


cell96_context_start = (
    max(
        1,
        CELL96_SOURCE_LINE_START
        -
        12,
    )
    if cell96_locator_valid
    else
    1
)


cell96_context_end = (
    min(
        len(
            cell96_source_lines
        ),
        CELL96_SOURCE_LINE_END
        +
        12,
    )
    if cell96_locator_valid
    else
    0
)


cell96_context_rows = []


if cell96_locator_valid:

    for line_number in range(
        cell96_context_start,
        cell96_context_end
        +
        1,
    ):

        cell96_context_rows.append({
            "line_number":
                line_number,

            "inside_target_locator":
                (
                    CELL96_SOURCE_LINE_START
                    <=
                    line_number
                    <=
                    CELL96_SOURCE_LINE_END
                ),

            "text":
                cell96_source_lines[
                    line_number
                    -
                    1
                ],
        })


# ============================================================
# 8) Detect explicit speaker markers around locator
# ============================================================

cell96_known_speaker_tokens = [
    "دکتر کمیل رودی",
    "فرهاد",
    "مجری",
    "استاد",
    "حاضر",
    "شرکت‌کننده",
    "شرکت کننده",
]


cell96_speaker_marker_rows = []


for row in cell96_context_rows:

    line_text = (
        row[
            "text"
        ]
        .strip()
    )


    for token in cell96_known_speaker_tokens:

        if token in line_text:

            cell96_speaker_marker_rows.append({
                "line_number":
                    row[
                        "line_number"
                    ],

                "token":
                    token,

                "text":
                    row[
                        "text"
                    ],
            })


# ============================================================
# 9) Raw-source primary marker check
# ============================================================

cell96_raw_context_text = "\n".join(
    row[
        "text"
    ]
    for row
    in cell96_context_rows
)


cell96_raw_has_primary_name = (
    "دکتر کمیل رودی"
    in
    cell96_raw_context_text
)


cell96_explicit_primary_resolution_supported = all([
    cell96_primary_name_marker_match,
    cell96_raw_has_primary_name,
    CELL96_BASE_RAW_SPEAKER
    ==
    "دکتر کمیل رودی",
])


# ============================================================
# 10) Fragment quality decision
# ============================================================

cell96_fragment_start_incomplete = (
    cell95_fragment_start_incomplete
)


cell96_risk_section_complete = (
    cell95_risk_section_sufficient
)


cell96_insurance_section_complete = (
    cell95_insurance_section_sufficient
)


cell96_multiple_complete_topics = (
    cell95_multiple_complete_topics
)


cell96_can_use_complete_subsection_without_retarget = all([
    cell96_explicit_primary_resolution_supported,

    (
        cell96_risk_section_complete
        or
        cell96_insurance_section_complete
    ),

    cell95_fragment_end_usable,
])


cell96_preferred_complete_subsection = (
    "INSURANCE_RISK_TRANSFER"
    if
    cell96_insurance_section_complete
    else
    "INVESTMENT_RISK_MANAGEMENT"
    if
    cell96_risk_section_complete
    else
    None
)


cell96_needs_retarget = (
    not
    cell96_can_use_complete_subsection_without_retarget
)


# ============================================================
# 11) No mutation safety
# ============================================================

cell96_registry_write = False

cell96_queue_advance = False

cell96_human_decision_added = False

cell96_target_changed = False

cell96_citation_changed = False

cell96_raw_source_mutated = False

cell96_parser_mutated = False

cell96_commit_push = False


# ============================================================
# 12) Gate
# ============================================================

cell96_gate_checks = {
    "no_failures":
        len(
            cell96_failures
        )
        ==
        0,

    "source_file_found":
        cell96_source_file_found,

    "locator_valid":
        cell96_locator_valid,

    "primary_name_marker_match":
        cell96_primary_name_marker_match,

    "raw_has_primary_name":
        cell96_raw_has_primary_name,

    "explicit_primary_resolution_supported":
        cell96_explicit_primary_resolution_supported,

    "has_complete_subsection":
        (
            cell96_risk_section_complete
            or
            cell96_insurance_section_complete
        ),
}


cell96_gate = all(
    cell96_gate_checks.values()
)


# ============================================================
# 13) Diagnostics
# ============================================================

print(
    "=== PILOT #22 RAW SOURCE CONTEXT + PRIMARY RESOLUTION ==="
)


print(
    "\n=== PILOT ==="
)

print(
    "Pilot:",
    CELL96_PILOT_ID,
)

print(
    "Target Evidence:",
    CELL96_TARGET_EVIDENCE_ID,
)

print(
    "Document:",
    CELL96_DOCUMENT_ID,
)


print(
    "\n=== SOURCE ATTRIBUTION ==="
)

print(
    "Base raw speaker:",
    CELL96_BASE_RAW_SPEAKER,
)

print(
    "Normalized speaker:",
    cell96_base_raw_normalized,
)

print(
    "Exact Primary name match:",
    cell96_primary_name_marker_match,
)

print(
    "Canonical identity:",
    "Dr. Komeil Roudi",
)

print(
    "Resolution basis:",
    "EXPLICIT_SOURCE_NAME_MARKER",
)

print(
    "Parser canonical field:",
    cell96_target_part.get(
        "speaker_effective_canonical"
    ),
)

print(
    "Parser effective_primary:",
    cell96_target_part.get(
        "speaker_effective_primary"
    ),
)


print(
    "\n=== RAW SOURCE FILE ==="
)

print(
    "Source path:",
    cell96_source_path,
)

print(
    "Source file found:",
    cell96_source_file_found,
)

print(
    "Locator:",
    (
        CELL96_SOURCE_LINE_START,
        CELL96_SOURCE_LINE_END,
    ),
)

print(
    "Context window:",
    (
        cell96_context_start,
        cell96_context_end,
    ),
)


print(
    "\n=== RAW SOURCE CONTEXT ==="
)

for row in cell96_context_rows:

    marker = (
        ">>>"
        if row[
            "inside_target_locator"
        ]
        else
        "   "
    )


    print(
        f"{marker} L{row['line_number']}: "
        f"{row['text']}"
    )


print(
    "\n=== SPEAKER MARKERS IN CONTEXT ==="
)

if cell96_speaker_marker_rows:

    for row in cell96_speaker_marker_rows:

        print(
            f"L{row['line_number']} | "
            f"{row['token']} | "
            f"{row['text']}"
        )

else:

    print(
        "No known speaker markers found in context window."
    )


print(
    "\n=== PRIMARY RESOLUTION RESULT ==="
)

print(
    "Raw context contains Dr. Komeil Roudi name:",
    cell96_raw_has_primary_name,
)

print(
    "Explicit Primary resolution supported:",
    cell96_explicit_primary_resolution_supported,
)

print(
    "Important:",
    (
        "Cell 95 Non-Primary classification was a canonicalization gap, "
        "not evidence that the speaker was actually non-primary."
    ),
)


print(
    "\n=== FRAGMENT QUALITY ==="
)

print(
    "Fragment start incomplete:",
    cell96_fragment_start_incomplete,
)

print(
    "Risk section complete:",
    cell96_risk_section_complete,
)

print(
    "Insurance section complete:",
    cell96_insurance_section_complete,
)

print(
    "Multiple complete topics:",
    cell96_multiple_complete_topics,
)

print(
    "Preferred complete subsection:",
    cell96_preferred_complete_subsection,
)

print(
    "Can use complete subsection without Retarget:",
    cell96_can_use_complete_subsection_without_retarget,
)

print(
    "Needs Retarget:",
    cell96_needs_retarget,
)


print(
    "\n=== ALL INTERNAL GATES ==="
)

for name, status in cell96_gate_checks.items():

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== SAFETY ==="
)

print(
    "Registry write:",
    cell96_registry_write,
)

print(
    "Queue advance:",
    cell96_queue_advance,
)

print(
    "Human Decision added:",
    cell96_human_decision_added,
)

print(
    "Target changed:",
    cell96_target_changed,
)

print(
    "Citation changed:",
    cell96_citation_changed,
)

print(
    "Raw Source mutated:",
    cell96_raw_source_mutated,
)

print(
    "Parser mutated:",
    cell96_parser_mutated,
)

print(
    "Commit / Push:",
    cell96_commit_push,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell96_failures
)


print(
    "\nCODE CELL 96 GATE:",
    (
        "PASS"
        if cell96_gate
        else
        "FAIL"
    ),
)


assert cell96_gate, (
    "Pilot #22 raw-source context / primary-resolution diagnostic failed. "
    "Repair this SAME Code Cell 96 before drafting the question."
)

=== PILOT #22 RAW SOURCE CONTEXT + PRIMARY RESOLUTION ===

=== PILOT ===
Pilot: question-pilot:733355977d8932b0778e0d85715c446d
Target Evidence: target-evidence:072e02138b13c2784ed9e7a99f4d33da
Document: program:khane-to:episode-013:utterance:00034

=== SOURCE ATTRIBUTION ===
Base raw speaker: دکتر کمیل رودی
Normalized speaker: دکتر کمیل رودی
Exact Primary name match: True
Canonical identity: Dr. Komeil Roudi
Resolution basis: EXPLICIT_SOURCE_NAME_MARKER
Parser canonical field: None
Parser effective_primary: None

=== RAW SOURCE FILE ===
Source path: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/data/transcripts/final/khane-to/episode-13.md
Source file found: True
Locator: (141, 143)
Context window: (129, 155)

=== RAW SOURCE CONTEXT ===
    L129: **فرهاد جم**
    L130: 
    L131: صندوق سهامی یه چیز ایش فکر کنم خوب بود سودش کمتر می‌شه اما صندوق
    L132: 
    L133: **دکتر کمیل رودی**
    L134: 
    L135: سهامی نه ریسک پذیری بالاتری داره و سود بهتری هم داره چ

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 97 — Pilot #22 Question/Answer ACCEPT + Citation Draft

Human Reviewer سؤال و جواب Pilot #22 را پذیرفت.

در این Pilot:

- Source خام به‌صورت صریح Speaker را `دکتر کمیل رودی` معرفی می‌کند؛
- Canonicalization ناقص Parser تغییر داده نمی‌شود؛
- Resolution به Primary Authority فقط بر اساس `EXPLICIT_SOURCE_NAME_MARKER` ثبت می‌شود؛
- ابتدای Fragment ناقص است، بنابراین برای Expected Answer استفاده نمی‌شود؛
- بخش کامل و مستقل مربوط به اصطلاحات بیمه به‌عنوان Answer Evidence استفاده می‌شود؛
- Retarget انجام نمی‌شود.

سؤال پذیرفته‌شده:

**بیمه‌گر، بیمه‌گذار، حق بیمه، پوشش بیمه و فرانشیز در قرارداد بیمه چه معنایی دارند؟**

جواب پذیرفته‌شده:

**بیمه‌گر کسی است که تعهد می‌کند در صورت وقوع خسارت، آن را طبق قرارداد جبران کند. بیمه‌گذار فردی است که خود یا دارایی‌اش را بیمه می‌کند. مبلغی که بیمه‌گذار برای دریافت این حمایت می‌پردازد حق بیمه است. پوشش بیمه میزان خسارتی را مشخص می‌کند که بیمه‌گر متعهد به جبران آن است و فرانشیز یا خودپرداخت بخشی از خسارت است که خود بیمه‌گذار باید آن را پرداخت کند.**

بعد از این Cell، Citation در وضعیت Human Review قرار می‌گیرد و Queue همچنان 2 باقی می‌ماند.

</div>

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 97 — Pilot #22 Question/Answer ACCEPT + Citation Draft

Human Reviewer سؤال و جواب Pilot #22 را پذیرفت.

در این Pilot:

- Source خام به‌صورت صریح Speaker را `دکتر کمیل رودی` معرفی می‌کند؛
- Canonicalization ناقص Parser تغییر داده نمی‌شود؛
- Resolution به Primary Authority فقط بر اساس `EXPLICIT_SOURCE_NAME_MARKER` ثبت می‌شود؛
- ابتدای Fragment ناقص است، بنابراین برای Expected Answer استفاده نمی‌شود؛
- بخش کامل و مستقل مربوط به اصطلاحات بیمه به‌عنوان Answer Evidence استفاده می‌شود؛
- Retarget انجام نمی‌شود.

سؤال پذیرفته‌شده:

**بیمه‌گر، بیمه‌گذار، حق بیمه، پوشش بیمه و فرانشیز در قرارداد بیمه چه معنایی دارند؟**

جواب پذیرفته‌شده:

**بیمه‌گر کسی است که تعهد می‌کند در صورت وقوع خسارت، آن را طبق قرارداد جبران کند. بیمه‌گذار فردی است که خود یا دارایی‌اش را بیمه می‌کند. مبلغی که بیمه‌گذار برای دریافت این حمایت می‌پردازد حق بیمه است. پوشش بیمه میزان خسارتی را مشخص می‌کند که بیمه‌گر متعهد به جبران آن است و فرانشیز یا خودپرداخت بخشی از خسارت است که خود بیمه‌گذار باید آن را پرداخت کند.**

بعد از این Cell، Citation در وضعیت Human Review قرار می‌گیرد و Queue همچنان 2 باقی می‌ماند.

</div>

In [97]:
# Code Cell 97

cell97_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell96_gate") is not True:
    cell97_failures.append(
        "CELL96_NOT_PASS"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell97_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Human-accepted Question / Answer
# ============================================================

CELL97_ACCEPTED_QUESTION = (
    "بیمه‌گر، بیمه‌گذار، حق بیمه، پوشش بیمه و فرانشیز "
    "در قرارداد بیمه چه معنایی دارند؟"
)


CELL97_ACCEPTED_ANSWER = (
    "بیمه‌گر کسی است که تعهد می‌کند در صورت وقوع خسارت، "
    "آن را طبق قرارداد جبران کند. بیمه‌گذار فردی است که خود "
    "یا دارایی‌اش را بیمه می‌کند. مبلغی که بیمه‌گذار برای دریافت "
    "این حمایت می‌پردازد حق بیمه است. پوشش بیمه میزان خسارتی را "
    "مشخص می‌کند که بیمه‌گر متعهد به جبران آن است و فرانشیز یا "
    "خودپرداخت بخشی از خسارت است که خود بیمه‌گذار باید آن را "
    "پرداخت کند."
)


CELL97_PILOT_ID = (
    CELL96_PILOT_ID
)


CELL97_GOLDEN_CASE_ID = (
    CELL94_NEXT_GOLDEN_CASE_ID
)


CELL97_TASK_DRAFT_ID = (
    CELL94_NEXT_TASK_DRAFT_ID
)


CELL97_TARGET_EVIDENCE_ID = (
    CELL96_TARGET_EVIDENCE_ID
)


# ============================================================
# 3) Target validation
# ============================================================

cell97_target_parts = copy.deepcopy(
    cell94_next_target_text_parts
)


cell97_target_valid = all([
    isinstance(
        cell97_target_parts,
        list,
    ),

    len(
        cell97_target_parts
    )
    ==
    1,

    cell96_explicit_primary_resolution_supported,

    cell96_insurance_section_complete,

    cell96_can_use_complete_subsection_without_retarget,

    not cell96_needs_retarget,
])


if not cell97_target_valid:
    cell97_failures.append(
        "TARGET_STRUCTURE_INVALID"
    )


cell97_primary_part = (
    copy.deepcopy(
        cell97_target_parts[
            0
        ]
    )
    if
    len(
        cell97_target_parts
    )
    ==
    1
    else
    {}
)


cell97_primary_text = (
    cell97_primary_part.get(
        "text"
    )
    or
    cell97_primary_part.get(
        "target_text"
    )
    or
    cell97_primary_part.get(
        "content"
    )
    or
    ""
)


# ============================================================
# 4) Explicit Primary resolution overlay
# ============================================================

cell97_primary_resolution = {
    "resolution_version":
        "explicit_source_primary_resolution_v1",

    "raw_source_marker":
        CELL96_BASE_RAW_SPEAKER,

    "normalized_source_marker":
        cell96_base_raw_normalized,

    "canonical_identity":
        "Dr. Komeil Roudi",

    "primary_authority":
        True,

    "resolution_basis":
        "EXPLICIT_SOURCE_NAME_MARKER",

    "raw_source_path":
        str(
            cell96_source_path
        ),

    "source_line_start":
        CELL96_SOURCE_LINE_START,

    "source_line_end":
        CELL96_SOURCE_LINE_END,

    "parser_canonical_before":
        cell96_target_part.get(
            "speaker_effective_canonical"
        ),

    "parser_effective_primary_before":
        cell96_target_part.get(
            "speaker_effective_primary"
        ),

    "parser_mutated":
        False,

    "raw_source_mutated":
        False,
}


cell97_primary_resolution_valid = all([
    cell97_primary_resolution[
        "raw_source_marker"
    ]
    ==
    "دکتر کمیل رودی",

    cell97_primary_resolution[
        "canonical_identity"
    ]
    ==
    "Dr. Komeil Roudi",

    cell97_primary_resolution[
        "primary_authority"
    ]
    is True,

    cell97_primary_resolution[
        "resolution_basis"
    ]
    ==
    "EXPLICIT_SOURCE_NAME_MARKER",

    cell97_primary_resolution[
        "parser_mutated"
    ]
    is False,

    cell97_primary_resolution[
        "raw_source_mutated"
    ]
    is False,
])


if not cell97_primary_resolution_valid:
    cell97_failures.append(
        "PRIMARY_RESOLUTION_INVALID"
    )


# ============================================================
# 5) Evidence scope
# ============================================================

cell97_evidence_scope = {
    "target_fragment_start_incomplete":
        True,

    "selected_complete_subsection":
        "INSURANCE_RISK_TRANSFER",

    "investment_risk_subsection_used_in_answer":
        False,

    "insurance_subsection_used_in_answer":
        True,

    "fragment_retargeted":
        False,

    "target_evidence_id_preserved":
        True,
}


cell97_evidence_scope_valid = all([
    cell97_evidence_scope[
        "target_fragment_start_incomplete"
    ]
    is True,

    cell97_evidence_scope[
        "selected_complete_subsection"
    ]
    ==
    "INSURANCE_RISK_TRANSFER",

    cell97_evidence_scope[
        "investment_risk_subsection_used_in_answer"
    ]
    is False,

    cell97_evidence_scope[
        "insurance_subsection_used_in_answer"
    ]
    is True,

    cell97_evidence_scope[
        "fragment_retargeted"
    ]
    is False,
])


if not cell97_evidence_scope_valid:
    cell97_failures.append(
        "EVIDENCE_SCOPE_INVALID"
    )


# ============================================================
# 6) Grounding
# ============================================================

cell97_grounding = {
    "insurer":
        (
            cell95_insurance_claims.get(
                "insurer"
            )
            is True
        ),

    "policyholder":
        (
            cell95_insurance_claims.get(
                "policyholder"
            )
            is True
        ),

    "premium":
        (
            cell95_insurance_claims.get(
                "premium"
            )
            is True
        ),

    "coverage":
        (
            cell95_insurance_claims.get(
                "coverage"
            )
            is True
        ),

    "deductible":
        (
            cell95_insurance_claims.get(
                "deductible"
            )
            is True
        ),

    "insurance_damage_context":
        (
            cell95_insurance_claims.get(
                "insurance_for_asset_damage"
            )
            is True
        ),

    "explicit_primary_source":
        (
            cell96_explicit_primary_resolution_supported
            is True
        ),
}


cell97_grounding_pass = all(
    cell97_grounding.values()
)


if not cell97_grounding_pass:
    cell97_failures.append(
        "ANSWER_GROUNDING_FAILED"
    )


# ============================================================
# 7) Start Engine
# ============================================================

cell97_question_pending_session = (
    cell13_build_session(
        cell94_next_pilot_packet
    )
)


cell97_session_start_valid = (
    cell97_question_pending_session.get(
        "current_state"
    )
    ==
    "QUESTION_DRAFT_PENDING"
)


if not cell97_session_start_valid:
    cell97_failures.append(
        "ENGINE_SESSION_START_INVALID"
    )


# ============================================================
# 8) Question Draft + Human ACCEPT
# ============================================================

cell97_question_identity = {
    "identity_version":
        "question_draft_identity_v1",

    "pilot_case_id":
        CELL97_PILOT_ID,

    "target_evidence_id":
        CELL97_TARGET_EVIDENCE_ID,

    "question_text":
        CELL97_ACCEPTED_QUESTION,

    "revision_number":
        1,
}


cell97_question_draft_id = (
    "question-draft:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell97_question_identity
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


cell97_question_record = {
    "record_version":
        "question_draft_record_v1",

    "question_draft_id":
        cell97_question_draft_id,

    "pilot_case_id":
        CELL97_PILOT_ID,

    "golden_case_id":
        CELL97_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL97_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL97_TARGET_EVIDENCE_ID,

    "question_text":
        CELL97_ACCEPTED_QUESTION,

    "revision_number":
        1,

    "human_adjudication_status":
        "PENDING",

    "external_knowledge_used":
        False,
}


cell97_question_review_session = (
    cell13_apply_event(
        cell97_question_pending_session,
        "QUESTION_DRAFT_REGISTERED",
        artifact=
            cell97_question_record,
    )
)


cell97_answer_pending_session = (
    cell13_apply_event(
        cell97_question_review_session,
        "QUESTION_ACCEPTED",
        human_note=
            "Human Reviewer accepted the question.",
    )
)


cell97_question_accept_valid = (
    cell97_answer_pending_session.get(
        "current_state"
    )
    ==
    "EXPECTED_ANSWER_DRAFT_PENDING"
)


if not cell97_question_accept_valid:
    cell97_failures.append(
        "QUESTION_ACCEPT_TRANSITION_INVALID"
    )


# ============================================================
# 9) Expected Answer + Human ACCEPT
# ============================================================

cell97_answer_identity = {
    "identity_version":
        "expected_answer_identity_v1",

    "pilot_case_id":
        CELL97_PILOT_ID,

    "question_draft_id":
        cell97_question_draft_id,

    "target_evidence_id":
        CELL97_TARGET_EVIDENCE_ID,

    "answer_text":
        CELL97_ACCEPTED_ANSWER,

    "revision_number":
        1,
}


cell97_expected_answer_draft_id = (
    "expected-answer-draft:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell97_answer_identity
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


cell97_expected_answer_record = {
    "record_version":
        "expected_answer_draft_record_v1",

    "expected_answer_draft_id":
        cell97_expected_answer_draft_id,

    "pilot_case_id":
        CELL97_PILOT_ID,

    "golden_case_id":
        CELL97_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL97_TASK_DRAFT_ID,

    "question_draft_id":
        cell97_question_draft_id,

    "target_evidence_id":
        CELL97_TARGET_EVIDENCE_ID,

    "expected_answer_text":
        CELL97_ACCEPTED_ANSWER,

    "revision_number":
        1,

    "human_adjudication_status":
        "PENDING",

    "external_knowledge_used":
        False,

    "selected_evidence_subsection":
        "INSURANCE_RISK_TRANSFER",
}


cell97_answer_review_session = (
    cell13_apply_event(
        cell97_answer_pending_session,
        "EXPECTED_ANSWER_REGISTERED",
        artifact=
            cell97_expected_answer_record,
    )
)


cell97_citation_pending_session = (
    cell13_apply_event(
        cell97_answer_review_session,
        "EXPECTED_ANSWER_ACCEPTED",
        human_note=
            "Human Reviewer accepted the expected answer.",
    )
)


cell97_answer_accept_valid = (
    cell97_citation_pending_session.get(
        "current_state"
    )
    ==
    "CITATION_MAPPING_PENDING"
)


if not cell97_answer_accept_valid:
    cell97_failures.append(
        "ANSWER_ACCEPT_TRANSITION_INVALID"
    )


# ============================================================
# 10) Citation Entry
# ============================================================

cell97_primary_entry = copy.deepcopy(
    cell97_primary_part
)


cell97_primary_entry.update({
    "citation_index":
        0,

    "authority_role":
        "PRIMARY_ANSWER_EVIDENCE",

    "answer_support":
        True,

    "use_in_expected_answer_grounding":
        True,

    "speaker_canonical":
        "Dr. Komeil Roudi",

    "speaker_primary":
        True,

    "primary_resolution":
        copy.deepcopy(
            cell97_primary_resolution
        ),

    "evidence_scope":
        copy.deepcopy(
            cell97_evidence_scope
        ),

    "answer_evidence_subsection":
        "INSURANCE_RISK_TRANSFER",

    "external_locator_lookup_used":
        False,
})


cell97_citation_entries = [
    cell97_primary_entry
]


cell97_citation_boundary_valid = all([
    len(
        cell97_citation_entries
    )
    ==
    1,

    cell97_primary_entry.get(
        "speaker_canonical"
    )
    ==
    "Dr. Komeil Roudi",

    cell97_primary_entry.get(
        "speaker_primary"
    )
    is True,

    cell97_primary_entry.get(
        "answer_support"
    )
    is True,

    cell97_primary_entry.get(
        "answer_evidence_subsection"
    )
    ==
    "INSURANCE_RISK_TRANSFER",

    cell97_primary_entry.get(
        "primary_resolution",
        {},
    ).get(
        "resolution_basis"
    )
    ==
    "EXPLICIT_SOURCE_NAME_MARKER",
])


if not cell97_citation_boundary_valid:
    cell97_failures.append(
        "CITATION_AUTHORITY_BOUNDARY_INVALID"
    )


# ============================================================
# 11) Locator integrity
# ============================================================

cell97_no_locator_fabrication = (
    cell97_primary_entry.get(
        "locator"
    )
    ==
    cell97_primary_part.get(
        "locator"
    )
)


if not cell97_no_locator_fabrication:
    cell97_failures.append(
        "LOCATOR_INTEGRITY_FAILED"
    )


# ============================================================
# 12) Citation Mapping
# ============================================================

cell97_citation_identity = {
    "identity_version":
        "citation_mapping_identity_v1_token_fallback_fragment",

    "pilot_case_id":
        CELL97_PILOT_ID,

    "question_draft_id":
        cell97_question_draft_id,

    "expected_answer_draft_id":
        cell97_expected_answer_draft_id,

    "target_evidence_id":
        CELL97_TARGET_EVIDENCE_ID,

    "citation_entries":
        copy.deepcopy(
            cell97_citation_entries
        ),

    "primary_resolution":
        copy.deepcopy(
            cell97_primary_resolution
        ),

    "evidence_scope":
        copy.deepcopy(
            cell97_evidence_scope
        ),
}


cell97_citation_mapping_id = (
    "citation-mapping:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell97_citation_identity
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


cell97_citation_mapping_record = {
    "record_version":
        "citation_mapping_record_v1_token_fallback_fragment",

    "citation_mapping_id":
        cell97_citation_mapping_id,

    "pilot_case_id":
        CELL97_PILOT_ID,

    "golden_case_id":
        CELL97_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL97_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL97_TARGET_EVIDENCE_ID,

    "question_draft_id":
        cell97_question_draft_id,

    "expected_answer_draft_id":
        cell97_expected_answer_draft_id,

    "citation_entries":
        copy.deepcopy(
            cell97_citation_entries
        ),

    "primary_answer_authority":
        "Dr. Komeil Roudi",

    "primary_resolution":
        copy.deepcopy(
            cell97_primary_resolution
        ),

    "selected_answer_evidence_subsection":
        "INSURANCE_RISK_TRANSFER",

    "fragment_start_incomplete":
        True,

    "incomplete_fragment_start_used_as_answer_evidence":
        False,

    "investment_risk_section_used_as_answer_evidence":
        False,

    "insurance_section_used_as_answer_evidence":
        True,

    "retarget_used":
        False,

    "raw_source_mutated":
        False,

    "parser_output_mutated":
        False,

    "external_knowledge_used":
        False,

    "human_adjudication_status":
        "PENDING",
}


cell97_current_session = (
    cell13_apply_event(
        cell97_citation_pending_session,
        "CITATION_MAPPING_REGISTERED",
        artifact=
            cell97_citation_mapping_record,
    )
)


cell97_citation_registered_valid = (
    cell97_current_session.get(
        "current_state"
    )
    ==
    "CITATION_HUMAN_REVIEW_PENDING"
)


if not cell97_citation_registered_valid:
    cell97_failures.append(
        "CITATION_REGISTRATION_INVALID"
    )


# ============================================================
# 13) Active Workspace
# ============================================================

cell97_active_workspace = copy.deepcopy(
    cell94_persisted_active_case
)


cell97_active_workspace[
    "workflow_state"
] = "CITATION_HUMAN_REVIEW_PENDING"


cell97_active_workspace[
    "source_primary_resolution"
] = copy.deepcopy(
    cell97_primary_resolution
)


cell97_active_workspace[
    "question_review"
] = {
    "question_draft_id":
        cell97_question_draft_id,

    "question_text":
        CELL97_ACCEPTED_QUESTION,

    "human_decision":
        "ACCEPT",
}


cell97_active_workspace[
    "expected_answer_review"
] = {
    "expected_answer_draft_id":
        cell97_expected_answer_draft_id,

    "expected_answer_text":
        CELL97_ACCEPTED_ANSWER,

    "human_decision":
        "ACCEPT",

    "selected_evidence_subsection":
        "INSURANCE_RISK_TRANSFER",
}


cell97_active_workspace[
    "citation_review"
] = {
    "citation_mapping_id":
        cell97_citation_mapping_id,

    "citation_mapping_record":
        copy.deepcopy(
            cell97_citation_mapping_record
        ),

    "human_decision":
        "PENDING",

    "primary_answer_authority":
        "Dr. Komeil Roudi",

    "primary_resolution_basis":
        "EXPLICIT_SOURCE_NAME_MARKER",
}


# ============================================================
# 14) Continuation state
# ============================================================

cell97_continuation_core = {
    "next_pilot_id":
        CELL97_PILOT_ID,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell94_remaining_pilot_ids
        ),

    "remaining_count":
        2,

    "active_pilot_status":
        "CITATION_HUMAN_REVIEW_PENDING",

    "active_target_evidence_id":
        CELL97_TARGET_EVIDENCE_ID,

    "active_citation_mapping_id":
        cell97_citation_mapping_id,

    "retarget_used":
        False,

    "primary_resolution_basis":
        "EXPLICIT_SOURCE_NAME_MARKER",

    "selected_answer_evidence_subsection":
        "INSURANCE_RISK_TRANSFER",

    "authority_policy_active":
        True,

    "last_completed_pilot_id":
        cell94_continuation_state.get(
            "last_completed_pilot_id"
        ),
}


cell97_continuation_state_sha256 = (
    cell10_stable_sha(
        cell97_continuation_core
    )
)


cell97_continuation_state = {
    **copy.deepcopy(
        cell97_continuation_core
    ),

    "continuation_state_sha256":
        cell97_continuation_state_sha256,
}


# ============================================================
# 15) Persist
# ============================================================

cell97_persistent_registry_payload = copy.deepcopy(
    cell94_persisted_registry
)


cell97_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell97_active_workspace
)


cell97_persistent_registry_payload[
    "remaining_pilot_count"
] = 2


cell97_persistent_registry_payload[
    "continuation_state"
] = copy.deepcopy(
    cell97_continuation_state
)


cell97_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell97_persistent_registry_payload
    )
)


# ============================================================
# 16) Read-back validation
# ============================================================

cell97_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell97_persisted_active_case = (
    cell97_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell97_persistence_valid = all([
    cell97_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL97_PILOT_ID,

    cell97_persisted_active_case.get(
        "workflow_state"
    )
    ==
    "CITATION_HUMAN_REVIEW_PENDING",

    cell97_persisted_active_case.get(
        "source_primary_resolution",
        {},
    ).get(
        "canonical_identity"
    )
    ==
    "Dr. Komeil Roudi",

    cell97_persisted_active_case.get(
        "source_primary_resolution",
        {},
    ).get(
        "resolution_basis"
    )
    ==
    "EXPLICIT_SOURCE_NAME_MARKER",

    cell97_persisted_active_case.get(
        "question_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell97_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell97_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "selected_evidence_subsection"
    )
    ==
    "INSURANCE_RISK_TRANSFER",

    cell97_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "PENDING",

    cell97_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    2,

    cell97_persisted_registry.get(
        "continuation_state",
        {},
    ).get(
        "continuation_state_sha256"
    )
    ==
    cell97_continuation_state_sha256,
])


if not cell97_persistence_valid:
    cell97_failures.append(
        "REGISTRY_READBACK_INVALID"
    )


# ============================================================
# 17) Safety
# ============================================================

cell97_queue_still_2 = all([
    len(
        cell94_remaining_pilot_ids
    )
    ==
    2,

    cell94_remaining_pilot_ids[
        0
    ]
    ==
    CELL97_PILOT_ID,
])


cell97_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell97_full_572_blocked = (
    cell94_full_572_blocked
    is True
)


cell97_no_commit_push = True


# ============================================================
# 18) Gate
# ============================================================

cell97_gate_checks = {
    "no_failures":
        len(
            cell97_failures
        )
        ==
        0,

    "target_valid":
        cell97_target_valid,

    "primary_resolution_valid":
        cell97_primary_resolution_valid,

    "evidence_scope_valid":
        cell97_evidence_scope_valid,

    "grounding_pass":
        cell97_grounding_pass,

    "session_start_valid":
        cell97_session_start_valid,

    "question_accept_valid":
        cell97_question_accept_valid,

    "answer_accept_valid":
        cell97_answer_accept_valid,

    "citation_boundary_valid":
        cell97_citation_boundary_valid,

    "no_locator_fabrication":
        cell97_no_locator_fabrication,

    "citation_registered_valid":
        cell97_citation_registered_valid,

    "persistence_valid":
        cell97_persistence_valid,

    "queue_still_2":
        cell97_queue_still_2,

    "base_seed_unchanged":
        cell97_base_seed_unchanged,

    "full_572_blocked":
        cell97_full_572_blocked,

    "no_commit_push":
        cell97_no_commit_push,
}


cell97_gate = all(
    cell97_gate_checks.values()
)


# ============================================================
# 19) Diagnostics
# ============================================================

print(
    "=== PILOT #22 QA ACCEPT + CITATION DRAFT ==="
)


print(
    "\n=== AUTHORITY ==="
)

print(
    "Raw Source Marker:",
    CELL96_BASE_RAW_SPEAKER,
)

print(
    "Resolved Primary Authority:",
    "Dr. Komeil Roudi",
)

print(
    "Resolution Basis:",
    "EXPLICIT_SOURCE_NAME_MARKER",
)

print(
    "Parser canonicalization changed:",
    False,
)

print(
    "Raw Source changed:",
    False,
)

print(
    "Retarget used:",
    False,
)


print(
    "\n=== EVIDENCE SCOPE ==="
)

print(
    "Fragment start incomplete:",
    True,
)

print(
    "Selected Answer Evidence subsection:",
    "INSURANCE_RISK_TRANSFER",
)

print(
    "Investment Risk section used:",
    False,
)

print(
    "Insurance section used:",
    True,
)


print(
    "\n=== ACCEPTED QUESTION ==="
)

print(
    CELL97_ACCEPTED_QUESTION
)

print(
    "Question Draft ID:",
    cell97_question_draft_id,
)

print(
    "Human Decision:",
    "ACCEPT",
)


print(
    "\n=== ACCEPTED ANSWER ==="
)

print(
    CELL97_ACCEPTED_ANSWER
)

print(
    "Expected Answer Draft ID:",
    cell97_expected_answer_draft_id,
)

print(
    "Human Decision:",
    "ACCEPT",
)


print(
    "\n=== GROUNDING ==="
)

for name, status in cell97_grounding.items():

    print(
        f"{name}:",
        status,
    )


print(
    "Grounding PASS:",
    cell97_grounding_pass,
)


print(
    "\n=== CITATION DRAFT ==="
)

print(
    "Citation Mapping ID:",
    cell97_citation_mapping_id,
)

print(
    "Citation entries:",
    len(
        cell97_citation_entries
    ),
)

print(
    "Primary Answer Evidence entries:",
    1,
)

print(
    "Citation boundary valid:",
    cell97_citation_boundary_valid,
)

print(
    "No locator fabrication:",
    cell97_no_locator_fabrication,
)


print(
    "\n=== WORKFLOW ==="
)

print(
    "Current State:",
    cell97_current_session.get(
        "current_state"
    ),
)

print(
    "Question:",
    "ACCEPT",
)

print(
    "Expected Answer:",
    "ACCEPT",
)

print(
    "Citation:",
    "PENDING",
)


print(
    "\n=== ALL INTERNAL GATES ==="
)

for name, status in cell97_gate_checks.items():

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== QUEUE / SAFETY ==="
)

print(
    "Queue still 2:",
    cell97_queue_still_2,
)

print(
    "Persistence valid:",
    cell97_persistence_valid,
)

print(
    "Base Seed unchanged:",
    cell97_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell97_full_572_blocked,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell97_failures
)


print(
    "\nCODE CELL 97 GATE:",
    (
        "PASS"
        if cell97_gate
        else
        "FAIL"
    ),
)


assert cell97_gate, (
    "Pilot #22 Question+Answer ACCEPT / Citation Draft failed. "
    "Repair this SAME Code Cell 97."
)

=== PILOT #22 QA ACCEPT + CITATION DRAFT ===

=== AUTHORITY ===
Raw Source Marker: دکتر کمیل رودی
Resolved Primary Authority: Dr. Komeil Roudi
Resolution Basis: EXPLICIT_SOURCE_NAME_MARKER
Parser canonicalization changed: False
Raw Source changed: False
Retarget used: False

=== EVIDENCE SCOPE ===
Fragment start incomplete: True
Selected Answer Evidence subsection: INSURANCE_RISK_TRANSFER
Investment Risk section used: False
Insurance section used: True

=== ACCEPTED QUESTION ===
بیمه‌گر، بیمه‌گذار، حق بیمه، پوشش بیمه و فرانشیز در قرارداد بیمه چه معنایی دارند؟
Question Draft ID: question-draft:38f7495ec267ea142a38d85225e5b801
Human Decision: ACCEPT

=== ACCEPTED ANSWER ===
بیمه‌گر کسی است که تعهد می‌کند در صورت وقوع خسارت، آن را طبق قرارداد جبران کند. بیمه‌گذار فردی است که خود یا دارایی‌اش را بیمه می‌کند. مبلغی که بیمه‌گذار برای دریافت این حمایت می‌پردازد حق بیمه است. پوشش بیمه میزان خسارتی را مشخص می‌کند که بیمه‌گر متعهد به جبران آن است و فرانشیز یا خودپرداخت بخشی از خسارت است که خود ب

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 98 — Citation ACCEPT + Pilot #22 Completion + Queue Advance

Citation نهایی Pilot #22 توسط Human Reviewer پذیرفته شد.

در این مرحله:

- Citation با تصمیم `ACCEPT` ثبت می‌شود؛
- Dr. Komeil Roudi تنها Primary Answer Authority باقی می‌ماند؛
- Resolution صریح `دکتر کمیل رودی` بر اساس Raw Source حفظ می‌شود؛
- Parser و Raw Source تغییر نمی‌کنند؛
- Pilot #22 کامل می‌شود؛
- تعداد Caseهای کامل Human Review از 20 به 21 می‌رسد؛
- تعداد Caseهای آماده Author Review از 20 به 21 می‌رسد؛
- Queue دقیقاً از 2 به 1 جلو می‌رود؛
- آخرین Pilot باقی‌مانده برای Human Review آماده می‌شود؛
- Author Final Review همچنان `NOT_STARTED` است؛
- Full 572 Generation همچنان Blocked باقی می‌ماند.

</div>

In [98]:
# Code Cell 98

cell98_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell97_gate") is not True:
    cell98_failures.append(
        "CELL97_NOT_PASS"
    )


if (
    globals().get(
        "cell97_current_session",
        {},
    ).get(
        "current_state"
    )
    !=
    "CITATION_HUMAN_REVIEW_PENDING"
):
    cell98_failures.append(
        "CITATION_NOT_AT_HUMAN_REVIEW_GATE"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell98_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Human Citation ACCEPT
# ============================================================

CELL98_CITATION_HUMAN_NOTE = (
    "Human Reviewer accepted the final Citation Mapping. "
    "Dr. Komeil Roudi remains the sole primary answer authority. "
    "Primary authority is resolved from the explicit raw-source name marker. "
    "The incomplete fragment start is not used as answer evidence."
)


cell98_complete_pending_session = (
    cell13_apply_event(
        cell97_current_session,
        "CITATION_ACCEPTED",
        human_note=
            CELL98_CITATION_HUMAN_NOTE,
    )
)


cell98_citation_accept_valid = (
    cell98_complete_pending_session.get(
        "current_state"
    )
    ==
    "PILOT_COMPLETE_PENDING_QUEUE_ADVANCE"
)


if not cell98_citation_accept_valid:
    cell98_failures.append(
        "CITATION_ACCEPT_TRANSITION_INVALID"
    )


# ============================================================
# 3) Final Citation Mapping
# ============================================================

cell98_final_citation_mapping_record = copy.deepcopy(
    cell97_citation_mapping_record
)


cell98_final_citation_mapping_record[
    "human_adjudication_status"
] = "ACCEPT"


cell98_final_citation_mapping_record[
    "human_review"
] = {
    "human_decision":
        "ACCEPT",

    "human_note":
        CELL98_CITATION_HUMAN_NOTE,

    "primary_answer_authority":
        "Dr. Komeil Roudi",

    "primary_resolution_basis":
        "EXPLICIT_SOURCE_NAME_MARKER",

    "selected_answer_evidence_subsection":
        "INSURANCE_RISK_TRANSFER",

    "incomplete_fragment_start_used_as_answer_evidence":
        False,

    "retarget_used":
        False,

    "raw_source_mutated":
        False,

    "parser_output_mutated":
        False,

    "external_knowledge_used":
        False,
}


# ============================================================
# 4) Complete Human Review case
# ============================================================

cell98_completed_case = copy.deepcopy(
    cell97_active_workspace
)


cell98_completed_case[
    "workflow_state"
] = "PILOT_COMPLETE"


cell98_completed_case[
    "status"
] = "COMPLETED_HUMAN_REVIEW"


cell98_completed_case[
    "final_status"
] = "PILOT_COMPLETE"


cell98_completed_case.setdefault(
    "citation_review",
    {},
)


cell98_completed_case[
    "citation_review"
][
    "human_decision"
] = "ACCEPT"


cell98_completed_case[
    "citation_review"
][
    "human_note"
] = CELL98_CITATION_HUMAN_NOTE


cell98_completed_case[
    "citation_review"
][
    "citation_mapping_record"
] = copy.deepcopy(
    cell98_final_citation_mapping_record
)


cell98_completed_case[
    "completion_summary"
] = {
    "retarget_used":
        False,

    "target_evidence_id":
        CELL97_TARGET_EVIDENCE_ID,

    "question_decision":
        "ACCEPT",

    "expected_answer_decision":
        "ACCEPT",

    "citation_decision":
        "ACCEPT",

    "task_family":
        "TOKEN_FALLBACK_FRAGMENT_RETRIEVAL",

    "primary_answer_authority":
        "Dr. Komeil Roudi",

    "primary_resolution_basis":
        "EXPLICIT_SOURCE_NAME_MARKER",

    "selected_answer_evidence_subsection":
        "INSURANCE_RISK_TRANSFER",

    "incomplete_fragment_start_used_as_answer_evidence":
        False,

    "raw_source_mutated":
        False,

    "parser_output_mutated":
        False,

    "author_final_review":
        "NOT_STARTED",
}


cell98_completed_case[
    "author_final_review"
] = {
    "status":
        "NOT_STARTED",

    "decision":
        None,

    "notes":
        None,
}


# ============================================================
# 5) Queue advance: 2 -> 1
# ============================================================

cell98_previous_remaining_pilot_ids = copy.deepcopy(
    cell94_remaining_pilot_ids
)


CELL98_COMPLETED_PILOT_ID = (
    cell98_previous_remaining_pilot_ids[
        0
    ]
)


cell98_remaining_pilot_ids = copy.deepcopy(
    cell98_previous_remaining_pilot_ids[
        1:
    ]
)


cell98_queue_advance_valid = all([
    len(
        cell98_previous_remaining_pilot_ids
    )
    ==
    2,

    CELL98_COMPLETED_PILOT_ID
    ==
    CELL97_PILOT_ID,

    len(
        cell98_remaining_pilot_ids
    )
    ==
    1,

    CELL98_COMPLETED_PILOT_ID
    not in
    cell98_remaining_pilot_ids,
])


if not cell98_queue_advance_valid:
    cell98_failures.append(
        "QUEUE_ADVANCE_INVALID"
    )


CELL98_NEXT_PILOT_ID = (
    cell98_remaining_pilot_ids[
        0
    ]
    if
    len(
        cell98_remaining_pilot_ids
    )
    ==
    1
    else
    None
)


# ============================================================
# 6) Find last remaining Pilot packet
# ============================================================

cell98_raw_next_packet_candidates = (
    cell58_find_pilot_packet_candidates(
        cell13_seed,
        CELL98_NEXT_PILOT_ID,
    )
    if
    CELL98_NEXT_PILOT_ID
    is not None
    else
    []
)


cell98_packet_candidates_by_sha = {}


for candidate in cell98_raw_next_packet_candidates:

    packet_sha = cell10_stable_sha(
        candidate[
            "packet"
        ]
    )


    previous = (
        cell98_packet_candidates_by_sha.get(
            packet_sha
        )
    )


    if (
        previous is None
        or
        candidate[
            "score"
        ]
        >
        previous[
            "score"
        ]
    ):

        cell98_packet_candidates_by_sha[
            packet_sha
        ] = candidate


cell98_ranked_next_packet_candidates = sorted(
    cell98_packet_candidates_by_sha.values(),
    key=lambda item: (
        -item[
            "score"
        ],
        item[
            "path"
        ],
    ),
)


cell98_next_pilot_packet = (
    copy.deepcopy(
        cell98_ranked_next_packet_candidates[
            0
        ][
            "packet"
        ]
    )
    if
    cell98_ranked_next_packet_candidates
    else
    None
)


if not isinstance(
    cell98_next_pilot_packet,
    dict,
):

    cell98_failures.append(
        "NEXT_PILOT_PACKET_NOT_FOUND"
    )


# ============================================================
# 7) Extract next Pilot identities
# ============================================================

cell98_next_pilot_ids_found = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell98_next_pilot_packet,
            "question-pilot:",
        )
    )
    if
    isinstance(
        cell98_next_pilot_packet,
        dict,
    )
    else
    set()
)


cell98_next_golden_case_ids = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell98_next_pilot_packet,
            "golden-case:",
        )
    )
    if
    isinstance(
        cell98_next_pilot_packet,
        dict,
    )
    else
    set()
)


cell98_next_task_draft_ids = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell98_next_pilot_packet,
            "task-draft:",
        )
    )
    if
    isinstance(
        cell98_next_pilot_packet,
        dict,
    )
    else
    set()
)


cell98_next_target_evidence_ids = sorted(
    set(
        cell58_collect_prefixed_ids(
            cell98_next_pilot_packet,
            "target-evidence:",
        )
    )
    if
    isinstance(
        cell98_next_pilot_packet,
        dict,
    )
    else
    set()
)


cell98_next_identity_valid = all([
    cell98_next_pilot_ids_found
    ==
    [
        CELL98_NEXT_PILOT_ID
    ],

    len(
        cell98_next_golden_case_ids
    )
    ==
    1,

    len(
        cell98_next_task_draft_ids
    )
    ==
    1,

    len(
        cell98_next_target_evidence_ids
    )
    ==
    1,
])


if not cell98_next_identity_valid:
    cell98_failures.append(
        "NEXT_PILOT_IDENTITY_INVALID"
    )


CELL98_NEXT_GOLDEN_CASE_ID = (
    cell98_next_golden_case_ids[
        0
    ]
    if
    len(
        cell98_next_golden_case_ids
    )
    ==
    1
    else
    None
)


CELL98_NEXT_TASK_DRAFT_ID = (
    cell98_next_task_draft_ids[
        0
    ]
    if
    len(
        cell98_next_task_draft_ids
    )
    ==
    1
    else
    None
)


CELL98_NEXT_TARGET_EVIDENCE_ID = (
    cell98_next_target_evidence_ids[
        0
    ]
    if
    len(
        cell98_next_target_evidence_ids
    )
    ==
    1
    else
    None
)


# ============================================================
# 8) Extract next Target Evidence
# ============================================================

cell98_target_part_candidates = (
    cell58_find_key_values(
        cell98_next_pilot_packet,
        "target_text_parts",
    )
    if
    isinstance(
        cell98_next_pilot_packet,
        dict,
    )
    else
    []
)


cell98_target_parts_by_sha = {}


for parts in cell98_target_part_candidates:

    if not isinstance(
        parts,
        list,
    ):
        continue


    if len(
        parts
    ) == 0:
        continue


    parts_sha = cell10_stable_sha(
        parts
    )


    cell98_target_parts_by_sha[
        parts_sha
    ] = parts


cell98_unique_target_part_sets = list(
    cell98_target_parts_by_sha.values()
)


cell98_next_target_text_parts = (
    copy.deepcopy(
        cell98_unique_target_part_sets[
            0
        ]
    )
    if
    len(
        cell98_unique_target_part_sets
    )
    ==
    1
    else
    None
)


if not isinstance(
    cell98_next_target_text_parts,
    list,
):

    cell98_failures.append(
        "NEXT_TARGET_TEXT_PARTS_NOT_UNIQUE"
    )


# ============================================================
# 9) Build readable next Evidence
# ============================================================

cell98_next_evidence_texts = []


for part in (
    cell98_next_target_text_parts
    or
    []
):

    if isinstance(
        part,
        str,
    ):

        text = part.strip()


    elif isinstance(
        part,
        dict,
    ):

        text = (
            part.get(
                "text"
            )
            or
            part.get(
                "target_text"
            )
            or
            part.get(
                "content"
            )
            or
            ""
        )


        if isinstance(
            text,
            str,
        ):

            text = text.strip()


        else:

            text = ""


    else:

        text = ""


    if text:

        cell98_next_evidence_texts.append(
            text
        )


cell98_next_combined_target_text = (
    "\n\n".join(
        cell98_next_evidence_texts
    )
)


cell98_next_evidence_available = bool(
    cell98_next_combined_target_text.strip()
)


if not cell98_next_evidence_available:
    cell98_failures.append(
        "NEXT_EVIDENCE_EMPTY"
    )


# ============================================================
# 10) Extract next Pilot metadata
# ============================================================

def cell98_unique_scalar_for_key(
    payload,
    key,
):

    values = (
        cell58_find_key_values(
            payload,
            key,
        )
        if
        isinstance(
            payload,
            dict,
        )
        else
        []
    )


    normalized = []


    for value in values:

        if isinstance(
            value,
            (
                str,
                int,
                float,
                bool,
            ),
        ):

            if value not in normalized:
                normalized.append(
                    value
                )


    if len(
        normalized
    ) == 1:

        return normalized[
            0
        ]


    return None


CELL98_NEXT_STRATUM = (
    cell98_unique_scalar_for_key(
        cell98_next_pilot_packet,
        "stratum",
    )
)


CELL98_NEXT_TASK_FAMILY = (
    cell98_unique_scalar_for_key(
        cell98_next_pilot_packet,
        "task_family",
    )
)


CELL98_NEXT_ARCHETYPE = (
    cell98_unique_scalar_for_key(
        cell98_next_pilot_packet,
        "question_archetype",
    )
)


CELL98_NEXT_AUTHORITY_EXPECTATION = (
    cell98_unique_scalar_for_key(
        cell98_next_pilot_packet,
        "authority_expectation",
    )
)


# ============================================================
# 11) Next Pilot Engine session
# ============================================================

cell98_next_pilot_session = (
    cell13_build_session(
        cell98_next_pilot_packet
    )
    if
    isinstance(
        cell98_next_pilot_packet,
        dict,
    )
    else
    {}
)


cell98_next_session_valid = (
    cell98_next_pilot_session.get(
        "current_state"
    )
    ==
    "QUESTION_DRAFT_PENDING"
)


if not cell98_next_session_valid:
    cell98_failures.append(
        "NEXT_SESSION_INVALID"
    )


# ============================================================
# 12) Update completed Human Review cases
# ============================================================

cell98_existing_completed_cases = copy.deepcopy(
    cell97_persisted_registry.get(
        "completed_human_review_cases",
        [],
    )
)


if not isinstance(
    cell98_existing_completed_cases,
    list,
):

    cell98_existing_completed_cases = []

    cell98_failures.append(
        "COMPLETED_CASES_NOT_LIST"
    )


cell98_existing_completed_cases = [
    case
    for case
    in cell98_existing_completed_cases
    if not (
        isinstance(
            case,
            dict,
        )
        and
        case.get(
            "pilot_case_id"
        )
        ==
        CELL98_COMPLETED_PILOT_ID
    )
]


cell98_completed_human_review_cases = (
    cell98_existing_completed_cases
    +
    [
        copy.deepcopy(
            cell98_completed_case
        )
    ]
)


cell98_completed_case_count_valid = (
    len(
        cell98_completed_human_review_cases
    )
    ==
    21
)


if not cell98_completed_case_count_valid:
    cell98_failures.append(
        "COMPLETED_CASE_COUNT_INVALID"
    )


# ============================================================
# 13) Rebuild Author Review queue
# ============================================================

cell98_author_review = copy.deepcopy(
    cell97_persisted_registry.get(
        "author_review",
        {},
    )
)


if not isinstance(
    cell98_author_review,
    dict,
):

    cell98_author_review = {}


cell98_author_ready_cases = copy.deepcopy(
    cell98_completed_human_review_cases
)


cell98_author_review[
    "cases_ready_for_review"
] = copy.deepcopy(
    cell98_author_ready_cases
)


cell98_author_review[
    "case_count"
] = len(
    cell98_author_ready_cases
)


cell98_author_review[
    "final_review_status"
] = "NOT_STARTED"


cell98_author_ready_count_valid = (
    len(
        cell98_author_ready_cases
    )
    ==
    21
)


cell98_author_ready_identity_valid = (
    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell98_author_ready_cases
        if isinstance(
            case,
            dict,
        )
    }
    ==
    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell98_completed_human_review_cases
        if isinstance(
            case,
            dict,
        )
    }
)


if not cell98_author_ready_count_valid:
    cell98_failures.append(
        "AUTHOR_READY_COUNT_INVALID"
    )


if not cell98_author_ready_identity_valid:
    cell98_failures.append(
        "AUTHOR_READY_IDENTITY_INVALID"
    )


# ============================================================
# 14) Preserve Global Authority Policy
# ============================================================

cell98_authority_policy = copy.deepcopy(
    cell97_persisted_registry.get(
        "workflow_policy",
        {},
    ).get(
        "authority_evidence_policy",
        {},
    )
)


cell98_authority_policy_valid = all([
    cell98_authority_policy.get(
        "primary_authority"
    )
    ==
    "Dr. Komeil Roudi",

    cell98_authority_policy.get(
        "secondary_to_primary_promotion"
    )
    ==
    "FORBIDDEN",
])


if not cell98_authority_policy_valid:
    cell98_failures.append(
        "GLOBAL_AUTHORITY_POLICY_INVALID"
    )


# ============================================================
# 15) Build final remaining active workspace
# ============================================================

cell98_next_active_workspace = {
    "pilot_case_id":
        CELL98_NEXT_PILOT_ID,

    "golden_case_id":
        CELL98_NEXT_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL98_NEXT_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL98_NEXT_TARGET_EVIDENCE_ID,

    "status":
        "ACTIVE_HUMAN_REVIEW",

    "workflow_state":
        "QUESTION_DRAFT_PENDING",

    "stratum":
        CELL98_NEXT_STRATUM,

    "task_family":
        CELL98_NEXT_TASK_FAMILY,

    "question_archetype":
        CELL98_NEXT_ARCHETYPE,

    "authority_expectation":
        CELL98_NEXT_AUTHORITY_EXPECTATION,

    "authority_policy":
        copy.deepcopy(
            cell98_authority_policy
        ),

    "source_evidence":
        copy.deepcopy(
            cell98_next_target_text_parts
        ),

    "combined_target_text":
        cell98_next_combined_target_text,

    "question_review":
        None,

    "expected_answer_review":
        None,

    "citation_review":
        None,

    "author_final_review": {
        "status":
            "NOT_STARTED",

        "decision":
            None,

        "notes":
            None,
    },
}


# ============================================================
# 16) Continuation State
# ============================================================

cell98_continuation_state_core = {
    "next_pilot_id":
        CELL98_NEXT_PILOT_ID,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell98_remaining_pilot_ids
        ),

    "remaining_count":
        1,

    "active_pilot_status":
        "QUESTION_DRAFT_PENDING",

    "active_target_evidence_id":
        CELL98_NEXT_TARGET_EVIDENCE_ID,

    "last_completed_pilot_id":
        CELL98_COMPLETED_PILOT_ID,

    "last_completion_type":
        "TOKEN_FALLBACK_FRAGMENT_PRIMARY_AUTHORITY_PILOT_COMPLETE",

    "authority_policy_active":
        True,
}


cell98_continuation_state_sha256 = (
    cell10_stable_sha(
        cell98_continuation_state_core
    )
)


cell98_continuation_state = {
    **copy.deepcopy(
        cell98_continuation_state_core
    ),

    "continuation_state_sha256":
        cell98_continuation_state_sha256,
}


# ============================================================
# 17) Persist Registry
# ============================================================

cell98_persistent_registry_payload = copy.deepcopy(
    cell97_persisted_registry
)


cell98_persistent_registry_payload[
    "completed_human_review_cases"
] = copy.deepcopy(
    cell98_completed_human_review_cases
)


cell98_persistent_registry_payload[
    "author_review"
] = copy.deepcopy(
    cell98_author_review
)


cell98_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell98_next_active_workspace
)


cell98_persistent_registry_payload[
    "remaining_pilot_count"
] = 1


cell98_persistent_registry_payload[
    "continuation_state"
] = copy.deepcopy(
    cell98_continuation_state
)


cell98_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell98_persistent_registry_payload
    )
)


# ============================================================
# 18) Read-back validation
# ============================================================

cell98_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell98_persisted_active_case = (
    cell98_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell98_persisted_completed_cases = (
    cell98_persisted_registry.get(
        "completed_human_review_cases",
        [],
    )
)


cell98_persisted_author_ready_cases = (
    cell98_persisted_registry.get(
        "author_review",
        {},
    ).get(
        "cases_ready_for_review",
        [],
    )
)


cell98_persisted_authority_policy = (
    cell98_persisted_registry.get(
        "workflow_policy",
        {},
    ).get(
        "authority_evidence_policy",
        {},
    )
)


cell98_persistence_valid = all([
    isinstance(
        cell98_persisted_completed_cases,
        list,
    ),

    len(
        cell98_persisted_completed_cases
    )
    ==
    21,

    isinstance(
        cell98_persisted_author_ready_cases,
        list,
    ),

    len(
        cell98_persisted_author_ready_cases
    )
    ==
    21,

    cell98_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    1,

    cell98_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL98_NEXT_PILOT_ID,

    cell98_persisted_active_case.get(
        "workflow_state"
    )
    ==
    "QUESTION_DRAFT_PENDING",

    cell98_persisted_registry.get(
        "continuation_state",
        {},
    ).get(
        "continuation_state_sha256"
    )
    ==
    cell98_continuation_state_sha256,

    cell98_persisted_authority_policy.get(
        "primary_authority"
    )
    ==
    "Dr. Komeil Roudi",

    cell98_persisted_authority_policy.get(
        "secondary_to_primary_promotion"
    )
    ==
    "FORBIDDEN",

    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell98_persisted_completed_cases
        if isinstance(
            case,
            dict,
        )
    }
    ==
    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell98_persisted_author_ready_cases
        if isinstance(
            case,
            dict,
        )
    },
])


if not cell98_persistence_valid:
    cell98_failures.append(
        "REGISTRY_READBACK_INVALID"
    )


# ============================================================
# 19) Safety
# ============================================================

cell98_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell98_full_572_blocked = (
    cell97_full_572_blocked
    is True
)


cell98_no_commit_push = True


# ============================================================
# 20) Internal Gates
# ============================================================

cell98_gate_checks = {
    "no_failures":
        len(
            cell98_failures
        )
        ==
        0,

    "citation_accept_valid":
        cell98_citation_accept_valid,

    "queue_advance_valid":
        cell98_queue_advance_valid,

    "next_identity_valid":
        cell98_next_identity_valid,

    "next_evidence_available":
        cell98_next_evidence_available,

    "next_session_valid":
        cell98_next_session_valid,

    "completed_case_count_valid":
        cell98_completed_case_count_valid,

    "author_ready_count_valid":
        cell98_author_ready_count_valid,

    "author_ready_identity_valid":
        cell98_author_ready_identity_valid,

    "authority_policy_valid":
        cell98_authority_policy_valid,

    "persistence_valid":
        cell98_persistence_valid,

    "base_seed_unchanged":
        cell98_base_seed_unchanged,

    "full_572_blocked":
        cell98_full_572_blocked,

    "no_commit_push":
        cell98_no_commit_push,
}


cell98_gate = all(
    cell98_gate_checks.values()
)


# ============================================================
# 21) Diagnostics
# ============================================================

print(
    "=== PILOT #22 COMPLETE + QUEUE ADVANCE + FINAL PILOT ==="
)


print(
    "\n=== COMPLETED PILOT ==="
)

print(
    "Pilot:",
    CELL98_COMPLETED_PILOT_ID,
)

print(
    "Target:",
    CELL97_TARGET_EVIDENCE_ID,
)

print(
    "Retarget used:",
    False,
)

print(
    "Question:",
    "ACCEPT",
)

print(
    "Expected Answer:",
    "ACCEPT",
)

print(
    "Citation:",
    "ACCEPT",
)

print(
    "Final Citation Mapping:",
    cell97_citation_mapping_id,
)

print(
    "Final Engine State:",
    cell98_complete_pending_session.get(
        "current_state"
    ),
)


print(
    "\n=== AUTHORITY RESULT ==="
)

print(
    "Primary Authority:",
    "Dr. Komeil Roudi",
)

print(
    "Resolution Basis:",
    "EXPLICIT_SOURCE_NAME_MARKER",
)

print(
    "Selected Answer Evidence:",
    "INSURANCE_RISK_TRANSFER",
)

print(
    "Incomplete fragment start used:",
    False,
)

print(
    "Raw Source mutated:",
    False,
)

print(
    "Parser mutated:",
    False,
)


print(
    "\n=== QUEUE ADVANCE ==="
)

print(
    "Before:",
    len(
        cell98_previous_remaining_pilot_ids
    ),
)

print(
    "After:",
    len(
        cell98_remaining_pilot_ids
    ),
)

print(
    "Queue advanced exactly once:",
    cell98_queue_advance_valid,
)


print(
    "\n=== HUMAN REVIEW REGISTRY ==="
)

print(
    "Completed Human Review cases:",
    len(
        cell98_completed_human_review_cases
    ),
)

print(
    "Cases ready for Author Review:",
    len(
        cell98_author_ready_cases
    ),
)

print(
    "Author Final Review:",
    "NOT_STARTED",
)

print(
    "Persistence valid:",
    cell98_persistence_valid,
)


print(
    "\n=== FINAL REMAINING PILOT ==="
)

print(
    "Pilot:",
    CELL98_NEXT_PILOT_ID,
)

print(
    "Golden Case:",
    CELL98_NEXT_GOLDEN_CASE_ID,
)

print(
    "Task Draft:",
    CELL98_NEXT_TASK_DRAFT_ID,
)

print(
    "Target Evidence:",
    CELL98_NEXT_TARGET_EVIDENCE_ID,
)

print(
    "Stratum:",
    CELL98_NEXT_STRATUM,
)

print(
    "Task Family:",
    CELL98_NEXT_TASK_FAMILY,
)

print(
    "Question Archetype:",
    CELL98_NEXT_ARCHETYPE,
)

print(
    "Authority Expectation:",
    CELL98_NEXT_AUTHORITY_EXPECTATION,
)

print(
    "Workflow:",
    cell98_next_pilot_session.get(
        "current_state"
    ),
)


print(
    "\n=== FINAL PILOT SOURCE EVIDENCE ==="
)

print(
    cell98_next_combined_target_text
)


print(
    "\n=== CONTINUATION STATE ==="
)

print(
    "Remaining Pilots:",
    1,
)

print(
    "Continuation SHA256:",
    cell98_continuation_state_sha256,
)


print(
    "\n=== ALL INTERNAL GATES ==="
)

for name, status in cell98_gate_checks.items():

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== SAFETY ==="
)

print(
    "Base Seed unchanged:",
    cell98_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell98_full_572_blocked,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell98_failures
)


print(
    "\nCODE CELL 98 GATE:",
    (
        "PASS"
        if cell98_gate
        else "FAIL"
    ),
)


assert cell98_gate, (
    "Pilot #22 Citation ACCEPT / completion / queue advance / "
    "final Pilot preparation failed. Repair this SAME Code Cell 98."
)

=== PILOT #22 COMPLETE + QUEUE ADVANCE + FINAL PILOT ===

=== COMPLETED PILOT ===
Pilot: question-pilot:733355977d8932b0778e0d85715c446d
Target: target-evidence:072e02138b13c2784ed9e7a99f4d33da
Retarget used: False
Question: ACCEPT
Expected Answer: ACCEPT
Citation: ACCEPT
Final Citation Mapping: citation-mapping:50b762a51b93639b36147133cd871afe
Final Engine State: PILOT_COMPLETE_PENDING_QUEUE_ADVANCE

=== AUTHORITY RESULT ===
Primary Authority: Dr. Komeil Roudi
Resolution Basis: EXPLICIT_SOURCE_NAME_MARKER
Selected Answer Evidence: INSURANCE_RISK_TRANSFER
Incomplete fragment start used: False
Raw Source mutated: False
Parser mutated: False

=== QUEUE ADVANCE ===
Before: 2
After: 1
Queue advanced exactly once: True

=== HUMAN REVIEW REGISTRY ===
Completed Human Review cases: 21
Cases ready for Author Review: 21
Author Final Review: NOT_STARTED
Persistence valid: True

=== FINAL REMAINING PILOT ===
Pilot: question-pilot:2beb40d7d1262dbf386db581f54d50d2
Golden Case: golden-case:34486152e8

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 99 — Final Pilot Financial Capability Ladder Source Retrieval

برای ساخت سؤال نهایی Pilot آخر، فقط Target خودارزیابی کافی نیست؛ چون Target می‌گوید پلکان پنج‌مرحله‌ای است، اما نام و تعریف پنج مرحله را ارائه نمی‌کند.

بنابراین قبل از سؤال‌سازی باید از Sourceهای خود پروژه بازیابی کنیم:

- پلکان توانمندی مالی دقیقاً چند مرحله دارد؛
- نام هر پنج مرحله چیست؛
- هر مرحله چه وضعیتی را توصیف می‌کند؛
- آیا بالاترین مرحله واقعاً «آزادی مالی» است؛
- و چگونه می‌توان با استفاده از این تعاریف جایگاه فعلی فرد را تعیین کرد.

اظهارنظر Human Reviewer درباره نام مراحل، Evidence محسوب نمی‌شود و فقط Source معتبر کتاب می‌تواند وارد Golden Answer شود.

این Cell فقط Source Retrieval / Diagnostic است و هیچ Registry، Queue، Human Decision، Target یا Citation را تغییر نمی‌دهد.

</div>

In [100]:
# Code Cell 99 — REPLACEMENT

cell99_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell98_gate") is not True:
    cell99_failures.append(
        "CELL98_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell13_seed"
    ),
    dict,
):
    cell99_failures.append(
        "CELL13_SEED_MISSING"
    )


# ============================================================
# 2) Final Pilot identity
# ============================================================

CELL99_PILOT_ID = (
    CELL98_NEXT_PILOT_ID
)


CELL99_TARGET_EVIDENCE_ID = (
    CELL98_NEXT_TARGET_EVIDENCE_ID
)


CELL99_TASK_FAMILY = (
    CELL98_NEXT_TASK_FAMILY
)


CELL99_ARCHETYPE = (
    CELL98_NEXT_ARCHETYPE
)


CELL99_AUTHORITY_EXPECTATION = (
    CELL98_NEXT_AUTHORITY_EXPECTATION
)


# ============================================================
# 3) Search helpers
# ============================================================

def cell99_norm(
    value,
):

    if not isinstance(
        value,
        str,
    ):
        return ""


    value = (
        value
        .replace(
            "ي",
            "ی",
        )
        .replace(
            "ك",
            "ک",
        )
        .replace(
            "\u200c",
            " ",
        )
        .replace(
            "\ufeff",
            "",
        )
    )


    value = re.sub(
        r"\s+",
        " ",
        value,
    )


    return value.strip()


def cell99_walk(
    obj,
    path=(),
):

    if isinstance(
        obj,
        dict,
    ):

        for key, value in obj.items():

            yield from cell99_walk(
                value,
                path
                +
                (
                    str(
                        key
                    ),
                ),
            )


    elif isinstance(
        obj,
        list,
    ):

        for index, value in enumerate(
            obj
        ):

            yield from cell99_walk(
                value,
                path
                +
                (
                    str(
                        index
                    ),
                ),
            )


    elif isinstance(
        obj,
        str,
    ):

        yield {
            "path":
                path,

            "text":
                obj,
        }


# ============================================================
# 4) Retrieval phrases
# ============================================================

cell99_primary_phrases = [
    "پلکان توانمندی مالی",
    "ناتوانی مالی",
    "تقلای مالی",
    "آزادی مالی",
]


cell99_supporting_phrases = [
    "پله",
    "توانمندی مالی",
    "قرض",
    "دخل و خرج",
    "پس انداز",
    "سرمایه گذاری",
    "درآمد غیرفعال",
    "استقلال مالی",
]


# ============================================================
# 5) Search all seed text nodes
# ============================================================

cell99_text_nodes = list(
    cell99_walk(
        cell13_seed
    )
)


cell99_matches = []


for node in cell99_text_nodes:

    normalized_text = cell99_norm(
        node[
            "text"
        ]
    )


    primary_hits = [
        phrase
        for phrase
        in cell99_primary_phrases
        if cell99_norm(
            phrase
        )
        in normalized_text
    ]


    supporting_hits = [
        phrase
        for phrase
        in cell99_supporting_phrases
        if cell99_norm(
            phrase
        )
        in normalized_text
    ]


    if primary_hits:

        cell99_matches.append({
            "path":
                node[
                    "path"
                ],

            "text":
                node[
                    "text"
                ],

            "normalized_text":
                normalized_text,

            "primary_hits":
                primary_hits,

            "supporting_hits":
                supporting_hits,

            "score":
                (
                    len(
                        primary_hits
                    )
                    *
                    10
                    +
                    len(
                        supporting_hits
                    )
                ),
        })


cell99_matches = sorted(
    cell99_matches,
    key=lambda item: (
        -item[
            "score"
        ],
        item[
            "path"
        ],
    ),
)


# ============================================================
# 6) Deduplicate text matches
# ============================================================

cell99_unique_matches = []

cell99_seen_text_sha = set()


for match in cell99_matches:

    text_sha = hashlib.sha256(
        cell99_norm(
            match[
                "text"
            ]
        ).encode(
            "utf-8"
        )
    ).hexdigest()


    if text_sha in cell99_seen_text_sha:
        continue


    cell99_seen_text_sha.add(
        text_sha
    )


    cell99_unique_matches.append(
        match
    )


# ============================================================
# 7) Search for likely Book source metadata near matches
# ============================================================

cell99_book_like_matches = []


for match in cell99_unique_matches:

    path_text = "/".join(
        match[
            "path"
        ]
    ).lower()


    normalized_text = match[
        "normalized_text"
    ]


    book_signal = any([
        "book" in path_text,
        "financial-literacy-for-all"
        in
        normalized_text.lower(),
        "خودارزیابی"
        in
        normalized_text,
        "تمرین"
        in
        normalized_text,
        "پلکان توانمندی مالی"
        in
        normalized_text,
    ])


    if book_signal:

        cell99_book_like_matches.append(
            match
        )


# ============================================================
# 8) Infer whether known stage names are explicitly found
# ============================================================

cell99_all_retrieved_text = "\n".join(
    match[
        "normalized_text"
    ]
    for match
    in cell99_unique_matches
)


cell99_known_stage_name_presence = {
    "ناتوانی مالی":
        (
            "ناتوانی مالی"
            in
            cell99_all_retrieved_text
        ),

    "تقلای مالی":
        (
            "تقلای مالی"
            in
            cell99_all_retrieved_text
        ),

    "آزادی مالی":
        (
            "آزادی مالی"
            in
            cell99_all_retrieved_text
        ),
}


cell99_explicit_ladder_phrase_found = (
    "پلکان توانمندی مالی"
    in
    cell99_all_retrieved_text
)


# ============================================================
# 9) Candidate blocks with several ladder signals
# ============================================================

cell99_high_value_candidates = []


for match in cell99_unique_matches:

    text = match[
        "normalized_text"
    ]


    stage_hits = [
        stage
        for stage
        in [
            "ناتوانی مالی",
            "تقلای مالی",
            "آزادی مالی",
        ]
        if stage in text
    ]


    ladder_hit = (
        "پلکان توانمندی مالی"
        in
        text
    )


    if (
        len(
            stage_hits
        )
        >=
        2
        or
        ladder_hit
    ):

        cell99_high_value_candidates.append({
            **copy.deepcopy(
                match
            ),

            "stage_hits":
                stage_hits,

            "ladder_hit":
                ladder_hit,
        })


# ============================================================
# 10) Final-pilot target itself
# ============================================================

cell99_final_target_text = (
    cell98_next_combined_target_text
)


cell99_target_mentions_five_steps = (
    "5 پله"
    in
    cell99_final_target_text
    or
    "۵ پله"
    in
    cell99_final_target_text
)


cell99_target_has_stage_names = any(
    stage
    in
    cell99_final_target_text
    for stage
    in [
        "ناتوانی مالی",
        "تقلای مالی",
        "آزادی مالی",
    ]
)


# ============================================================
# 11) Retrieval assessment
# ============================================================

cell99_has_any_source_match = (
    len(
        cell99_unique_matches
    )
    >
    0
)


cell99_has_book_like_match = (
    len(
        cell99_book_like_matches
    )
    >
    0
)


cell99_has_high_value_candidate = (
    len(
        cell99_high_value_candidates
    )
    >
    0
)


cell99_needs_deeper_source_lookup = (
    not all(
        cell99_known_stage_name_presence.values()
    )
    or
    not cell99_explicit_ladder_phrase_found
)


# ============================================================
# 12) Safety
# ============================================================

cell99_registry_write = False

cell99_queue_advance = False

cell99_human_decision_added = False

cell99_target_changed = False

cell99_citation_changed = False

cell99_raw_source_mutated = False

cell99_parser_mutated = False

cell99_commit_push = False


# ============================================================
# 13) Diagnostic Gate
# ============================================================

cell99_gate_checks = {
    "no_failures":
        len(
            cell99_failures
        )
        ==
        0,

    "correct_task_family":
        CELL99_TASK_FAMILY
        ==
        "EXERCISE_SELF_ASSESSMENT_RETRIEVAL",

    "correct_authority_expectation":
        CELL99_AUTHORITY_EXPECTATION
        ==
        "BOOK_PRIMARY_EVIDENCE",

    "target_mentions_five_steps":
        cell99_target_mentions_five_steps,

    "source_match_found":
        cell99_has_any_source_match,
}


cell99_gate = all(
    cell99_gate_checks.values()
)


# ============================================================
# 14) Diagnostics
# ============================================================

print(
    "=== FINAL PILOT FINANCIAL CAPABILITY LADDER SOURCE RETRIEVAL ==="
)


print(
    "\n=== PILOT ==="
)

print(
    "Pilot:",
    CELL99_PILOT_ID,
)

print(
    "Target Evidence:",
    CELL99_TARGET_EVIDENCE_ID,
)

print(
    "Task Family:",
    CELL99_TASK_FAMILY,
)

print(
    "Question Archetype:",
    CELL99_ARCHETYPE,
)

print(
    "Authority Expectation:",
    CELL99_AUTHORITY_EXPECTATION,
)


print(
    "\n=== CURRENT SELF-ASSESSMENT TARGET ==="
)

print(
    cell99_final_target_text
)

print(
    "\nTarget explicitly says five steps:",
    cell99_target_mentions_five_steps,
)

print(
    "Target itself contains stage names:",
    cell99_target_has_stage_names,
)


print(
    "\n=== RETRIEVAL SUMMARY ==="
)

print(
    "Total text nodes searched:",
    len(
        cell99_text_nodes
    ),
)

print(
    "Unique ladder-related matches:",
    len(
        cell99_unique_matches
    ),
)

print(
    "Book-like matches:",
    len(
        cell99_book_like_matches
    ),
)

print(
    "High-value candidates:",
    len(
        cell99_high_value_candidates
    ),
)


print(
    "\n=== KNOWN STAGE NAME PRESENCE IN RETRIEVED SOURCE ==="
)

for name, status in (
    cell99_known_stage_name_presence.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "Explicit 'پلکان توانمندی مالی' found:",
    cell99_explicit_ladder_phrase_found,
)


print(
    "\n=== TOP HIGH-VALUE SOURCE CANDIDATES ==="
)

if cell99_high_value_candidates:

    for index, candidate in enumerate(
        cell99_high_value_candidates[
            :12
        ],
        start=1,
    ):

        print(
            "\n"
            +
            "="
            *
            80
        )

        print(
            "CANDIDATE:",
            index,
        )

        print(
            "Path:",
            " > ".join(
                candidate[
                    "path"
                ]
            ),
        )

        print(
            "Primary hits:",
            candidate[
                "primary_hits"
            ],
        )

        print(
            "Stage hits:",
            candidate[
                "stage_hits"
            ],
        )

        print(
            "Ladder phrase:",
            candidate[
                "ladder_hit"
            ],
        )

        print(
            "Text:"
        )

        print(
            candidate[
                "text"
            ]
        )

else:

    print(
        "No high-value ladder candidate found in current seed."
    )


print(
    "\n=== OTHER BOOK-LIKE MATCHES ==="
)

for index, candidate in enumerate(
    cell99_book_like_matches[
        :12
    ],
    start=1,
):

    print(
        "\n"
        +
        "-"
        *
        80
    )

    print(
        "BOOK MATCH:",
        index,
    )

    print(
        "Path:",
        " > ".join(
            candidate[
                "path"
            ]
        ),
    )

    print(
        "Primary hits:",
        candidate[
            "primary_hits"
        ],
    )

    print(
        "Text:"
    )

    print(
        candidate[
            "text"
        ]
    )


print(
    "\n=== ASSESSMENT ==="
)

print(
    "Has any Source match:",
    cell99_has_any_source_match,
)

print(
    "Has Book-like match:",
    cell99_has_book_like_match,
)

print(
    "Has high-value ladder candidate:",
    cell99_has_high_value_candidate,
)

print(
    "Needs deeper Source lookup:",
    cell99_needs_deeper_source_lookup,
)

print(
    "IMPORTANT: Human recollection is NOT used as Golden evidence."
)


print(
    "\n=== ALL INTERNAL GATES ==="
)

for name, status in (
    cell99_gate_checks.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== SAFETY ==="
)

print(
    "Registry write:",
    cell99_registry_write,
)

print(
    "Queue advance:",
    cell99_queue_advance,
)

print(
    "Human Decision added:",
    cell99_human_decision_added,
)

print(
    "Target changed:",
    cell99_target_changed,
)

print(
    "Citation changed:",
    cell99_citation_changed,
)

print(
    "Raw Source mutated:",
    cell99_raw_source_mutated,
)

print(
    "Parser mutated:",
    cell99_parser_mutated,
)

print(
    "Commit / Push:",
    cell99_commit_push,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell99_failures
)


print(
    "\nCODE CELL 99 GATE:",
    (
        "PASS"
        if cell99_gate
        else
        "FAIL"
    ),
)


assert cell99_gate, (
    "Final Pilot ladder-source retrieval diagnostic failed. "
    "Repair this SAME Code Cell 99 before drafting the final question."
)

=== FINAL PILOT FINANCIAL CAPABILITY LADDER SOURCE RETRIEVAL ===

=== PILOT ===
Pilot: question-pilot:2beb40d7d1262dbf386db581f54d50d2
Target Evidence: target-evidence:1352ce8332affbe3651fe0558449e2a9
Task Family: EXERCISE_SELF_ASSESSMENT_RETRIEVAL
Question Archetype: STRUCTURE_AWARE_RETRIEVAL
Authority Expectation: BOOK_PRIMARY_EVIDENCE

=== CURRENT SELF-ASSESSMENT TARGET ===
خودارزیابی 1: موقعیت من در پلکان توانمندی مالی

حتماً احساس می‌کنید همه این پله‌ها را در زمان‌های متفاوتی تجربه کرده‌اید، الان جایگاه فعلی خود را چگونه ارزیابی می‌کنید. چند جمله‌ای درباره وضعیت خود بنویسید.

(5 پله به صورت پلکانی از شماره 1 پایین پلکان و 5 بالای پلکان طراحی می‌شود)

Target explicitly says five steps: True
Target itself contains stage names: False

=== RETRIEVAL SUMMARY ===
Total text nodes searched: 40208
Unique ladder-related matches: 3
Book-like matches: 3
High-value candidates: 3

=== KNOWN STAGE NAME PRESENCE IN RETRIEVED SOURCE ===
ناتوانی مالی: True
تقلای مالی: True
آزادی مالی: False
Explic

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 100 — Final Pilot Raw Book Source Retrieval for the Five Financial Capability Stages

Cell 99 نشان داد که در Seed فعلی فقط بخشی از مراحل پلکان توانمندی مالی قابل بازیابی است.

در این Cell مستقیماً در فایل‌های Source پروژه جست‌وجو می‌کنیم تا:

- بخش «درس ۲: پلکان توانمندی مالی» پیدا شود؛
- نام دقیق هر پنج پله از خود کتاب استخراج شود؛
- تعریف هر پله در Context اصلی دیده شود؛
- مشخص شود بالاترین پله واقعاً «آزادی مالی» است یا نه؛
- و سپس بتوان سؤال نهایی را بر اساس Book Primary Evidence ساخت.

هیچ Human recollection به‌عنوان Evidence استفاده نمی‌شود.

این Cell فقط Diagnostic / Source Retrieval است و هیچ Registry، Queue، Target، Citation، Raw Source یا Parser را تغییر نمی‌دهد.

</div>

In [101]:
# Code Cell 100

from pathlib import Path
import json
import re
import hashlib


cell100_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell99_gate") is not True:
    cell100_failures.append(
        "CELL99_NOT_PASS"
    )


# ============================================================
# 2) Repository root
# ============================================================

cell100_registry_path = Path(
    cell21_registry_path
).resolve()


cell100_repo_root_candidates = []


for candidate in [
    cell100_registry_path.parent,
    *cell100_registry_path.parents,
]:

    if (
        (
            candidate
            /
            "data"
        ).exists()
        and
        (
            candidate
            /
            "notebooks"
        ).exists()
    ):

        cell100_repo_root_candidates.append(
            candidate
        )


cell100_repo_root = (
    cell100_repo_root_candidates[
        0
    ]
    if cell100_repo_root_candidates
    else None
)


if cell100_repo_root is None:
    cell100_failures.append(
        "REPO_ROOT_NOT_FOUND"
    )


# ============================================================
# 3) Search configuration
# ============================================================

cell100_search_root = (
    cell100_repo_root
    /
    "data"
    if cell100_repo_root is not None
    else None
)


cell100_allowed_suffixes = {
    ".md",
    ".txt",
    ".json",
    ".jsonl",
}


cell100_primary_phrases = [
    "پلکان توانمندی مالی",
    "درس ۲: پلکان توانمندی مالی",
    "خودارزیابی 1: موقعیت من در پلکان توانمندی مالی",
]


cell100_stage_probe_phrases = [
    "ناتوانی مالی",
    "تقلای مالی",
    "آزادی مالی",
]


def cell100_norm(
    text,
):

    if not isinstance(
        text,
        str,
    ):
        return ""


    value = (
        text
        .replace(
            "ي",
            "ی",
        )
        .replace(
            "ك",
            "ک",
        )
        .replace(
            "\u200c",
            " ",
        )
        .replace(
            "\ufeff",
            "",
        )
    )


    value = re.sub(
        r"\s+",
        " ",
        value,
    )


    return value.strip()


# ============================================================
# 4) Collect candidate source files
# ============================================================

cell100_candidate_files = []


if (
    cell100_search_root is not None
    and
    cell100_search_root.exists()
):

    for path in cell100_search_root.rglob(
        "*"
    ):

        if not path.is_file():
            continue


        if path.suffix.lower() not in cell100_allowed_suffixes:
            continue


        try:

            size_bytes = path.stat().st_size

        except Exception:

            continue


        if size_bytes > 25 * 1024 * 1024:
            continue


        cell100_candidate_files.append(
            path
        )


# ============================================================
# 5) Read and score files
# ============================================================

cell100_file_matches = []


for path in cell100_candidate_files:

    try:

        raw_text = path.read_text(
            encoding="utf-8"
        )

    except Exception:

        continue


    normalized = cell100_norm(
        raw_text
    )


    primary_hits = [
        phrase
        for phrase
        in cell100_primary_phrases
        if cell100_norm(
            phrase
        )
        in normalized
    ]


    stage_hits = [
        phrase
        for phrase
        in cell100_stage_probe_phrases
        if cell100_norm(
            phrase
        )
        in normalized
    ]


    path_text = str(
        path
    ).lower()


    book_path_signal = any([
        "book" in path_text,
        "financial-literacy-for-all" in path_text,
        "financial_literacy_for_all" in path_text,
        "ketab" in path_text,
    ])


    if primary_hits or stage_hits:

        score = (
            len(
                primary_hits
            )
            *
            20
            +
            len(
                stage_hits
            )
            *
            5
            +
            (
                10
                if book_path_signal
                else 0
            )
        )


        cell100_file_matches.append({
            "path":
                path,

            "raw_text":
                raw_text,

            "normalized":
                normalized,

            "primary_hits":
                primary_hits,

            "stage_hits":
                stage_hits,

            "book_path_signal":
                book_path_signal,

            "score":
                score,
        })


cell100_file_matches = sorted(
    cell100_file_matches,
    key=lambda item: (
        -item[
            "score"
        ],
        str(
            item[
                "path"
            ]
        ),
    ),
)


# ============================================================
# 6) Build focused char windows around ladder phrase
# ============================================================

cell100_context_windows = []


for match in cell100_file_matches:

    raw_text = match[
        "raw_text"
    ]


    normalized_raw = (
        raw_text
        .replace(
            "ي",
            "ی",
        )
        .replace(
            "ك",
            "ک",
        )
        .replace(
            "\u200c",
            " ",
        )
    )


    search_terms = [
        "پلکان توانمندی مالی",
        "ناتوانی مالی",
        "تقلای مالی",
        "آزادی مالی",
    ]


    for term in search_terms:

        position = normalized_raw.find(
            term
        )


        if position < 0:
            continue


        window_start = max(
            0,
            position
            -
            2500,
        )


        window_end = min(
            len(
                raw_text
            ),
            position
            +
            7000,
        )


        window_text = raw_text[
            window_start:
            window_end
        ]


        window_sha = hashlib.sha256(
            cell100_norm(
                window_text
            ).encode(
                "utf-8"
            )
        ).hexdigest()


        cell100_context_windows.append({
            "path":
                match[
                    "path"
                ],

            "trigger":
                term,

            "window_start":
                window_start,

            "window_end":
                window_end,

            "window_text":
                window_text,

            "window_sha":
                window_sha,

            "file_score":
                match[
                    "score"
                ],
        })


# ============================================================
# 7) Deduplicate windows
# ============================================================

cell100_unique_windows = []

cell100_seen_window_sha = set()


for item in sorted(
    cell100_context_windows,
    key=lambda row: (
        -row[
            "file_score"
        ],
        str(
            row[
                "path"
            ]
        ),
        row[
            "window_start"
        ],
    ),
):

    if item[
        "window_sha"
    ] in cell100_seen_window_sha:
        continue


    cell100_seen_window_sha.add(
        item[
            "window_sha"
        ]
    )


    cell100_unique_windows.append(
        item
    )


# ============================================================
# 8) Extract likely stage labels from best windows
# ============================================================

cell100_stage_label_candidates = []


stage_patterns = [
    r"(?:پله|مرحله)\s*(?:اول|دوم|سوم|چهارم|پنجم|[۱-۵1-5])\s*[:\-–—]?\s*([^\n،؛\.]{2,80})",
    r"([^\n،؛\.]{2,50})\s*(?:پله|مرحله)\s*(?:اول|دوم|سوم|چهارم|پنجم|[۱-۵1-5])",
]


for window in cell100_unique_windows[
    :10
]:

    text = window[
        "window_text"
    ]


    for pattern in stage_patterns:

        for found in re.findall(
            pattern,
            text,
            flags=re.IGNORECASE,
        ):

            candidate = cell100_norm(
                found
            )


            if (
                candidate
                and
                candidate not in cell100_stage_label_candidates
            ):

                cell100_stage_label_candidates.append(
                    candidate
                )


# ============================================================
# 9) Evidence assessment
# ============================================================

cell100_source_file_found = (
    len(
        cell100_file_matches
    )
    >
    0
)


cell100_ladder_source_found = any(
    "پلکان توانمندی مالی"
    in match[
        "normalized"
    ]
    for match
    in cell100_file_matches
)


cell100_stage_presence_across_files = {
    stage:
        any(
            stage
            in
            match[
                "normalized"
            ]
            for match
            in cell100_file_matches
        )
    for stage
    in cell100_stage_probe_phrases
}


cell100_freedom_confirmed = (
    cell100_stage_presence_across_files.get(
        "آزادی مالی"
    )
    is True
)


cell100_needs_more_lookup = not all([
    cell100_source_file_found,
    cell100_ladder_source_found,
    cell100_freedom_confirmed,
])


# ============================================================
# 10) Safety
# ============================================================

cell100_registry_write = False

cell100_queue_advance = False

cell100_human_decision_added = False

cell100_target_changed = False

cell100_citation_changed = False

cell100_raw_source_mutated = False

cell100_parser_mutated = False

cell100_commit_push = False


# ============================================================
# 11) Gate
# ============================================================

cell100_gate_checks = {
    "no_failures":
        len(
            cell100_failures
        )
        ==
        0,

    "source_file_found":
        cell100_source_file_found,

    "ladder_source_found":
        cell100_ladder_source_found,

    "context_window_found":
        len(
            cell100_unique_windows
        )
        >
        0,
}


cell100_gate = all(
    cell100_gate_checks.values()
)


# ============================================================
# 12) Diagnostics
# ============================================================

print(
    "=== FINAL PILOT RAW BOOK LADDER SOURCE RETRIEVAL ==="
)


print(
    "\n=== SEARCH ROOT ==="
)

print(
    "Repository:",
    cell100_repo_root,
)

print(
    "Data root:",
    cell100_search_root,
)

print(
    "Candidate files scanned:",
    len(
        cell100_candidate_files
    ),
)


print(
    "\n=== MATCHED FILES ==="
)

print(
    "Matched files:",
    len(
        cell100_file_matches
    ),
)


for index, match in enumerate(
    cell100_file_matches[
        :12
    ],
    start=1,
):

    print(
        "\n"
        +
        "="
        *
        90
    )

    print(
        "FILE:",
        index,
    )

    print(
        "Path:",
        match[
            "path"
        ],
    )

    print(
        "Score:",
        match[
            "score"
        ],
    )

    print(
        "Primary hits:",
        match[
            "primary_hits"
        ],
    )

    print(
        "Stage hits:",
        match[
            "stage_hits"
        ],
    )

    print(
        "Book path signal:",
        match[
            "book_path_signal"
        ],
    )


print(
    "\n=== STAGE PRESENCE ACROSS SOURCE FILES ==="
)

for stage, status in (
    cell100_stage_presence_across_files.items()
):

    print(
        f"{stage}:",
        status,
    )


print(
    "Freedom stage explicitly confirmed:",
    cell100_freedom_confirmed,
)


print(
    "\n=== BEST SOURCE CONTEXT WINDOWS ==="
)


for index, window in enumerate(
    cell100_unique_windows[
        :8
    ],
    start=1,
):

    print(
        "\n"
        +
        "#" * 90
    )

    print(
        "WINDOW:",
        index,
    )

    print(
        "Path:",
        window[
            "path"
        ],
    )

    print(
        "Trigger:",
        window[
            "trigger"
        ],
    )

    print(
        "Char range:",
        (
            window[
                "window_start"
            ],
            window[
                "window_end"
            ],
        ),
    )

    print(
        "\nTEXT:\n"
    )

    print(
        window[
            "window_text"
        ]
    )


print(
    "\n=== AUTO-EXTRACTED STAGE LABEL CANDIDATES ==="
)

if cell100_stage_label_candidates:

    for index, candidate in enumerate(
        cell100_stage_label_candidates,
        start=1,
    ):

        print(
            index,
            "->",
            candidate,
        )

else:

    print(
        "No reliable stage labels auto-extracted."
    )


print(
    "\n=== ASSESSMENT ==="
)

print(
    "Source file found:",
    cell100_source_file_found,
)

print(
    "Ladder source found:",
    cell100_ladder_source_found,
)

print(
    "Freedom stage confirmed:",
    cell100_freedom_confirmed,
)

print(
    "Needs more lookup:",
    cell100_needs_more_lookup,
)

print(
    "IMPORTANT: Human recollection is NOT used as Golden evidence."
)


print(
    "\n=== ALL INTERNAL GATES ==="
)

for name, status in (
    cell100_gate_checks.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== SAFETY ==="
)

print(
    "Registry write:",
    cell100_registry_write,
)

print(
    "Queue advance:",
    cell100_queue_advance,
)

print(
    "Human Decision added:",
    cell100_human_decision_added,
)

print(
    "Target changed:",
    cell100_target_changed,
)

print(
    "Citation changed:",
    cell100_citation_changed,
)

print(
    "Raw Source mutated:",
    cell100_raw_source_mutated,
)

print(
    "Parser mutated:",
    cell100_parser_mutated,
)

print(
    "Commit / Push:",
    cell100_commit_push,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell100_failures
)


print(
    "\nCODE CELL 100 GATE:",
    (
        "PASS"
        if cell100_gate
        else
        "FAIL"
    ),
)


assert cell100_gate, (
    "Final Pilot raw-book ladder retrieval failed. "
    "Repair this SAME Code Cell 100 before drafting the final question."
)

=== FINAL PILOT RAW BOOK LADDER SOURCE RETRIEVAL ===

=== SEARCH ROOT ===
Repository: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance
Data root: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/data
Candidate files scanned: 1009

=== MATCHED FILES ===
Matched files: 49

FILE: 1
Path: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/data/phase2/processed/persian_normalization_records_v1.jsonl
Score: 85
Primary hits: ['پلکان توانمندی مالی', 'درس ۲: پلکان توانمندی مالی', 'خودارزیابی 1: موقعیت من در پلکان توانمندی مالی']
Stage hits: ['ناتوانی مالی', 'تقلای مالی', 'آزادی مالی']
Book path signal: True

FILE: 2
Path: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/data/phase2/human_review/human_review_registry.json
Score: 80
Primary hits: ['پلکان توانمندی مالی', 'درس ۲: پلکان توانمندی مالی', 'خودارزیابی 1: موقعیت من در پلکان توانمندی مالی']
Stage hits: ['ناتوانی مالی', 'تقلای مالی']
Book path

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 101 — Final Pilot: Exact Book Evidence for the Five Financial Capability Stages

Cell 100 نام پنج پله پلکان توانمندی مالی را در Sourceهای پروژه تأیید کرد:

1. ناتوانی مالی
2. تقلای مالی
3. تعادل دخل و خرج
4. تاب‌آوری و امنیت مالی
5. آزادی مالی

اما Pilot نهایی دارای Authority Rule زیر است:

`BOOK_PRIMARY_EVIDENCE`

بنابراین قبل از ساخت Golden Question و Expected Answer، باید همین مراحل و توضیح آن‌ها مستقیماً از Source کتاب:

`book:adult-financial-literacy-course:fa-source`

بازیابی شوند.

در این Cell:

- فایل‌های `book_ingested_v1/v2` بررسی می‌شوند؛
- تمام Paragraphهای مربوط به «درس ۲: پلکان توانمندی مالی» استخراج می‌شوند؛
- Paragraphهایی که پنج مرحله یا تعریف آن‌ها را توضیح می‌دهند نمایش داده می‌شوند؛
- ارتباط آن‌ها با Paragraphهای خودارزیابی 162 تا 164 حفظ می‌شود؛
- مشخص می‌شود آیا Evidence کتاب برای پاسخ به سؤال «چگونه جایگاه خود را در پلکان تشخیص دهیم؟» کافی است یا خیر.

هیچ Registry، Queue، Target، Citation یا Human Decision تغییر نمی‌کند.

</div>

In [103]:
# Code Cell 101 — REPLACEMENT

from pathlib import Path
import json
import re
import copy


cell101_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell100_gate") is not True:
    cell101_failures.append(
        "CELL100_NOT_PASS"
    )


if globals().get("cell100_repo_root") is None:
    cell101_failures.append(
        "REPO_ROOT_MISSING"
    )


# ============================================================
# 2) Exact Book identity
# ============================================================

CELL101_WORK_ID = (
    "book:adult-financial-literacy-course"
)


CELL101_EDITION_ID = (
    "book:adult-financial-literacy-course:fa-source"
)


CELL101_LESSON_HEADING = (
    "درس ۲: پلکان توانمندی مالی"
)


# Source-native labels are preserved exactly as the Book uses them.
CELL101_STAGE_DEFINITIONS = [
    {
        "stage_number":
            1,

        "canonical_stage_key":
            "FINANCIAL_INABILITY",

        "source_title_variants": [
            "ناتوانی مالی",
        ],

        "preferred_source_title":
            "ناتوانی مالی",
    },

    {
        "stage_number":
            2,

        "canonical_stage_key":
            "FINANCIAL_STRUGGLE",

        "source_title_variants": [
            "تقلای مالی",
            "تقلّای مالی",
        ],

        "preferred_source_title":
            "تقلّای مالی",
    },

    {
        "stage_number":
            3,

        "canonical_stage_key":
            "FINANCIAL_BALANCE",

        "source_title_variants": [
            "تعادل مالی",
            "تعادل دخل و خرج",
        ],

        "preferred_source_title":
            "تعادل مالی",
    },

    {
        "stage_number":
            4,

        "canonical_stage_key":
            "FINANCIAL_RESILIENCE_SECURITY",

        "source_title_variants": [
            "تاب‌آوری مالی",
            "تاب آوری مالی",
            "امنیت مالی",
            "تاب‌آوری مالی (امنیت مالی)",
        ],

        "preferred_source_title":
            "تاب‌آوری مالی (امنیت مالی)",
    },

    {
        "stage_number":
            5,

        "canonical_stage_key":
            "FINANCIAL_WELLBEING_FREEDOM",

        "source_title_variants": [
            "رفاه مالی",
            "آزادی مالی",
            "رفاه مالی (آزادی مالی)",
        ],

        "preferred_source_title":
            "رفاه مالی (آزادی مالی)",
    },
]


# ============================================================
# 3) Candidate Book files
# ============================================================

cell101_book_paths = [
    (
        cell100_repo_root
        /
        "data"
        /
        "phase2"
        /
        "interim"
        /
        "book_ingested_v1.json"
    ),

    (
        cell100_repo_root
        /
        "data"
        /
        "phase2"
        /
        "interim"
        /
        "book_ingested_v2.json"
    ),
]


cell101_existing_book_paths = [
    path
    for path
    in cell101_book_paths
    if path.exists()
    and path.is_file()
]


if not cell101_existing_book_paths:
    cell101_failures.append(
        "BOOK_INGESTED_FILES_NOT_FOUND"
    )


# ============================================================
# 4) Helpers
# ============================================================

def cell101_norm(
    value,
):

    if not isinstance(
        value,
        str,
    ):
        return ""


    value = (
        value
        .replace(
            "ي",
            "ی",
        )
        .replace(
            "ك",
            "ک",
        )
        .replace(
            "\u200c",
            " ",
        )
        .replace(
            "\ufeff",
            "",
        )
        .replace(
            "ّ",
            "",
        )
    )


    value = re.sub(
        r"\s+",
        " ",
        value,
    )


    return value.strip()


def cell101_collect_paragraph_records(
    obj,
):

    records = []


    def walk(
        value,
    ):

        if isinstance(
            value,
            dict,
        ):

            if (
                isinstance(
                    value.get(
                        "text"
                    ),
                    str,
                )
                and
                (
                    value.get(
                        "paragraph_number"
                    )
                    is not None
                    or
                    value.get(
                        "paragraph_id"
                    )
                    is not None
                )
            ):

                records.append(
                    copy.deepcopy(
                        value
                    )
                )


            for child in value.values():

                walk(
                    child
                )


        elif isinstance(
            value,
            list,
        ):

            for child in value:

                walk(
                    child
                )


    walk(
        obj
    )


    return records


def cell101_text_matches_any_variant(
    text,
    variants,
):

    normalized_text = cell101_norm(
        text
    )


    return any(
        cell101_norm(
            variant
        )
        in
        normalized_text
        for variant
        in variants
    )


# ============================================================
# 5) Load Book records
# ============================================================

cell101_file_records = []


for path in cell101_existing_book_paths:

    try:

        payload = json.loads(
            path.read_text(
                encoding="utf-8"
            )
        )

    except Exception as exc:

        cell101_failures.append({
            "reason":
                "BOOK_JSON_READ_FAILED",

            "path":
                str(
                    path
                ),

            "error":
                repr(
                    exc
                ),
        })

        continue


    records = (
        cell101_collect_paragraph_records(
            payload
        )
    )


    cell101_file_records.append({
        "path":
            path,

        "record_count":
            len(
                records
            ),

        "records":
            records,
    })


# ============================================================
# 6) Restrict to exact Book / Edition
# ============================================================

cell101_all_book_records = []


for file_record in cell101_file_records:

    for record in file_record[
        "records"
    ]:

        work_id = record.get(
            "work_id"
        )


        edition_id = record.get(
            "edition_id"
        )


        if (
            work_id
            not in
            {
                None,
                CELL101_WORK_ID,
            }
        ):
            continue


        if (
            edition_id
            not in
            {
                None,
                CELL101_EDITION_ID,
            }
        ):
            continue


        cell101_all_book_records.append({
            **copy.deepcopy(
                record
            ),

            "_source_file":
                str(
                    file_record[
                        "path"
                    ]
                ),
        })


# ============================================================
# 7) Deduplicate by paragraph identity + text
# ============================================================

cell101_unique_book_records = {}


for record in cell101_all_book_records:

    key = (
        record.get(
            "paragraph_id"
        ),
        record.get(
            "paragraph_number"
        ),
        cell101_norm(
            record.get(
                "text",
                "",
            )
        ),
    )


    if key not in cell101_unique_book_records:

        cell101_unique_book_records[
            key
        ] = record


cell101_book_records = list(
    cell101_unique_book_records.values()
)


cell101_book_records = sorted(
    cell101_book_records,
    key=lambda row: (
        (
            row.get(
                "paragraph_number"
            )
            if isinstance(
                row.get(
                    "paragraph_number"
                ),
                int,
            )
            else
            10**9
        ),
        str(
            row.get(
                "paragraph_id"
            )
        ),
    ),
)


# ============================================================
# 8) Detect Lesson 2 records
# ============================================================

cell101_lesson_records = []


for record in cell101_book_records:

    text = cell101_norm(
        record.get(
            "text",
            "",
        )
    )


    heading_path = record.get(
        "heading_path"
    )


    if not isinstance(
        heading_path,
        list,
    ):
        heading_path = []


    normalized_heading_path = [
        cell101_norm(
            item
        )
        for item
        in heading_path
        if isinstance(
            item,
            str,
        )
    ]


    lesson_in_heading = any(
        cell101_norm(
            CELL101_LESSON_HEADING
        )
        in heading
        for heading
        in normalized_heading_path
    )


    lesson_heading_record = (
        cell101_norm(
            CELL101_LESSON_HEADING
        )
        in
        text
    )


    if (
        lesson_in_heading
        or
        lesson_heading_record
    ):

        cell101_lesson_records.append(
            record
        )


# ============================================================
# 9) Exact source-native stage title evidence
# ============================================================

cell101_stage_evidence = {}


for stage_definition in CELL101_STAGE_DEFINITIONS:

    stage_key = stage_definition[
        "canonical_stage_key"
    ]


    variants = stage_definition[
        "source_title_variants"
    ]


    matching_records = []


    for record in cell101_lesson_records:

        if cell101_text_matches_any_variant(
            record.get(
                "text",
                "",
            ),
            variants,
        ):

            matching_records.append(
                record
            )


    cell101_stage_evidence[
        stage_key
    ] = {
        "stage_number":
            stage_definition[
                "stage_number"
            ],

        "preferred_source_title":
            stage_definition[
                "preferred_source_title"
            ],

        "source_title_variants":
            copy.deepcopy(
                variants
            ),

        "records":
            matching_records,

        "found":
            len(
                matching_records
            )
            >
            0,
    }


cell101_all_five_stages_found_in_book = all(
    stage_result[
        "found"
    ]
    for stage_result
    in cell101_stage_evidence.values()
)


# ============================================================
# 10) Validate exact expected stage-title paragraphs
# ============================================================

cell101_expected_stage_title_paragraphs = {
    1:
        141,

    2:
        144,

    3:
        148,

    4:
        154,

    5:
        157,
}


cell101_stage_title_records_by_number = {}


for stage_number, paragraph_number in (
    cell101_expected_stage_title_paragraphs.items()
):

    candidates = [
        record
        for record
        in cell101_lesson_records
        if record.get(
            "paragraph_number"
        )
        ==
        paragraph_number
    ]


    cell101_stage_title_records_by_number[
        stage_number
    ] = (
        copy.deepcopy(
            candidates[
                0
            ]
        )
        if candidates
        else None
    )


cell101_expected_title_paragraphs_present = all(
    isinstance(
        record,
        dict,
    )
    for record
    in cell101_stage_title_records_by_number.values()
)


# ============================================================
# 11) Build structured definitions from Book paragraphs 141–159
# ============================================================

cell101_records_by_paragraph_number = {
    record.get(
        "paragraph_number"
    ):
        record
    for record
    in cell101_lesson_records
    if isinstance(
        record.get(
            "paragraph_number"
        ),
        int,
    )
}


cell101_stage_ranges = {
    1:
        range(
            141,
            144,
        ),

    2:
        range(
            144,
            148,
        ),

    3:
        range(
            148,
            154,
        ),

    4:
        range(
            154,
            157,
        ),

    5:
        range(
            157,
            160,
        ),
}


cell101_structured_stage_blocks = []


for stage_definition in CELL101_STAGE_DEFINITIONS:

    stage_number = stage_definition[
        "stage_number"
    ]


    paragraph_numbers = list(
        cell101_stage_ranges[
            stage_number
        ]
    )


    stage_records = [
        copy.deepcopy(
            cell101_records_by_paragraph_number[
                number
            ]
        )
        for number
        in paragraph_numbers
        if number
        in
        cell101_records_by_paragraph_number
    ]


    stage_texts = [
        record.get(
            "text",
            ""
        )
        for record
        in stage_records
    ]


    cell101_structured_stage_blocks.append({
        "stage_number":
            stage_number,

        "canonical_stage_key":
            stage_definition[
                "canonical_stage_key"
            ],

        "preferred_source_title":
            stage_definition[
                "preferred_source_title"
            ],

        "paragraph_numbers":
            paragraph_numbers,

        "records":
            stage_records,

        "combined_text":
            "\n".join(
                stage_texts
            ),

        "complete":
            len(
                stage_records
            )
            ==
            len(
                paragraph_numbers
            ),
    })


cell101_all_stage_blocks_complete = all(
    block[
        "complete"
    ]
    for block
    in cell101_structured_stage_blocks
)


# ============================================================
# 12) Validate each stage semantically from Book
# ============================================================

cell101_stage_semantic_checks = {
    1: {
        "title":
            (
                cell101_text_matches_any_variant(
                    cell101_structured_stage_blocks[
                        0
                    ][
                        "combined_text"
                    ],
                    [
                        "ناتوانی مالی",
                    ],
                )
            ),

        "debt_dependency":
            all([
                "بدهکار"
                in
                cell101_structured_stage_blocks[
                    0
                ][
                    "combined_text"
                ],

                "قرض"
                in
                cell101_structured_stage_blocks[
                    0
                ][
                    "combined_text"
                ],
            ]),
    },

    2: {
        "title":
            (
                cell101_text_matches_any_variant(
                    cell101_structured_stage_blocks[
                        1
                    ][
                        "combined_text"
                    ],
                    [
                        "تقلای مالی",
                        "تقلّای مالی",
                    ],
                )
            ),

        "future_income_preconsumption":
            all([
                "درآمد آینده"
                in
                cell101_structured_stage_blocks[
                    1
                ][
                    "combined_text"
                ],

                (
                    "پیش خور"
                    in
                    cell101_norm(
                        cell101_structured_stage_blocks[
                            1
                        ][
                            "combined_text"
                        ]
                    )
                    or
                    "آینده خواری"
                    in
                    cell101_norm(
                        cell101_structured_stage_blocks[
                            1
                        ][
                            "combined_text"
                        ]
                    )
                ),
            ]),
    },

    3: {
        "title":
            (
                "تعادل مالی"
                in
                cell101_norm(
                    cell101_structured_stage_blocks[
                        2
                    ][
                        "combined_text"
                    ]
                )
            ),

        "daily_finances":
            (
                "امور مالی روزانه"
                in
                cell101_structured_stage_blocks[
                    2
                ][
                    "combined_text"
                ]
            ),

        "income_expense_balance":
            (
                "درآمد"
                in
                cell101_structured_stage_blocks[
                    2
                ][
                    "combined_text"
                ]
                and
                "هزینه"
                in
                cell101_structured_stage_blocks[
                    2
                ][
                    "combined_text"
                ]
            ),

        "no_savings":
            (
                "پس‌انداز"
                in
                cell101_structured_stage_blocks[
                    2
                ][
                    "combined_text"
                ]
            ),
    },

    4: {
        "title":
            (
                "تاب آوری مالی"
                in
                cell101_norm(
                    cell101_structured_stage_blocks[
                        3
                    ][
                        "combined_text"
                    ]
                )
                and
                "امنیت مالی"
                in
                cell101_norm(
                    cell101_structured_stage_blocks[
                        3
                    ][
                        "combined_text"
                    ]
                )
            ),

        "shock_resilience":
            (
                "شوک"
                in
                cell101_structured_stage_blocks[
                    3
                ][
                    "combined_text"
                ]
            ),

        "income_expense_balance":
            (
                "دخل و خرج"
                in
                cell101_structured_stage_blocks[
                    3
                ][
                    "combined_text"
                ]
            ),
    },

    5: {
        "title":
            (
                "رفاه مالی"
                in
                cell101_norm(
                    cell101_structured_stage_blocks[
                        4
                    ][
                        "combined_text"
                    ]
                )
                and
                "آزادی مالی"
                in
                cell101_norm(
                    cell101_structured_stage_blocks[
                        4
                ][
                    "combined_text"
                ]
            )
        ),

        "goals_and_wants":
            (
                "اهداف مالی"
                in
                cell101_structured_stage_blocks[
                    4
                ][
                    "combined_text"
                ]
                and
                "خواسته"
                in
                cell101_structured_stage_blocks[
                    4
                ][
                    "combined_text"
                ]
            ),

        "money_not_major_decision_factor":
            (
                "پول"
                in
                cell101_structured_stage_blocks[
                    4
                ][
                    "combined_text"
                ]
                and
                "تصمیم"
                in
                cell101_structured_stage_blocks[
                    4
                ][
                    "combined_text"
                ]
            ),
    },
}


cell101_stage_semantic_validation = {
    stage_number:
        all(
            checks.values()
        )
    for stage_number, checks
    in cell101_stage_semantic_checks.items()
}


cell101_all_stage_semantics_valid = all(
    cell101_stage_semantic_validation.values()
)


# ============================================================
# 13) Self-assessment records
# ============================================================

cell101_self_assessment_records = [
    record
    for record
    in cell101_book_records
    if record.get(
        "paragraph_number"
    )
    in
    {
        162,
        163,
        164,
    }
]


cell101_self_assessment_records = sorted(
    cell101_self_assessment_records,
    key=lambda row:
        row.get(
            "paragraph_number",
            10**9,
        ),
)


cell101_self_assessment_valid = all([
    len(
        cell101_self_assessment_records
    )
    ==
    3,

    any(
        "جایگاه فعلی"
        in
        record.get(
            "text",
            ""
        )
        for record
        in cell101_self_assessment_records
    ),

    any(
        (
            "5 پله"
            in
            record.get(
                "text",
                ""
            )
            or
            "۵ پله"
            in
            record.get(
                "text",
                ""
            )
        )
        for record
        in cell101_self_assessment_records
    ),
])


# ============================================================
# 14) Book-primary answer sufficiency
# ============================================================

cell101_book_primary_evidence_sufficient_for_stage_list = all([
    cell101_all_five_stages_found_in_book,
    cell101_expected_title_paragraphs_present,
    cell101_all_stage_blocks_complete,
])


cell101_book_primary_evidence_sufficient_for_full_self_assessment_answer = all([
    cell101_book_primary_evidence_sufficient_for_stage_list,
    cell101_all_stage_semantics_valid,
    cell101_self_assessment_valid,
])


# ============================================================
# 15) Focused Book Evidence package
# ============================================================

cell101_book_primary_supporting_evidence = []


for paragraph_number in range(
    140,
    161,
):

    record = cell101_records_by_paragraph_number.get(
        paragraph_number
    )


    if isinstance(
        record,
        dict,
    ):

        cell101_book_primary_supporting_evidence.append(
            copy.deepcopy(
                record
            )
        )


for record in cell101_self_assessment_records:

    cell101_book_primary_supporting_evidence.append(
        copy.deepcopy(
            record
        )
    )


cell101_supporting_paragraph_numbers = [
    record.get(
        "paragraph_number"
    )
    for record
    in cell101_book_primary_supporting_evidence
]


cell101_supporting_evidence_valid = all([
    140
    in
    cell101_supporting_paragraph_numbers,

    141
    in
    cell101_supporting_paragraph_numbers,

    144
    in
    cell101_supporting_paragraph_numbers,

    148
    in
    cell101_supporting_paragraph_numbers,

    154
    in
    cell101_supporting_paragraph_numbers,

    157
    in
    cell101_supporting_paragraph_numbers,

    163
    in
    cell101_supporting_paragraph_numbers,
])


# ============================================================
# 16) Safety
# ============================================================

cell101_registry_write = False

cell101_queue_advance = False

cell101_human_decision_added = False

cell101_target_changed = False

cell101_citation_changed = False

cell101_raw_source_mutated = False

cell101_parser_mutated = False

cell101_commit_push = False


# ============================================================
# 17) Gate
# ============================================================

cell101_gate_checks = {
    "no_failures":
        len(
            cell101_failures
        )
        ==
        0,

    "book_files_found":
        len(
            cell101_existing_book_paths
        )
        >
        0,

    "book_records_found":
        len(
            cell101_book_records
        )
        >
        0,

    "lesson_records_found":
        len(
            cell101_lesson_records
        )
        >
        0,

    "self_assessment_found":
        cell101_self_assessment_valid,

    "all_five_stage_names_found_in_book":
        cell101_all_five_stages_found_in_book,

    "expected_stage_title_paragraphs_present":
        cell101_expected_title_paragraphs_present,

    "all_stage_blocks_complete":
        cell101_all_stage_blocks_complete,

    "all_stage_semantics_valid":
        cell101_all_stage_semantics_valid,

    "supporting_evidence_valid":
        cell101_supporting_evidence_valid,

    "stage_list_evidence_sufficient":
        cell101_book_primary_evidence_sufficient_for_stage_list,

    "full_self_assessment_evidence_sufficient":
        cell101_book_primary_evidence_sufficient_for_full_self_assessment_answer,
}


cell101_gate = all(
    cell101_gate_checks.values()
)


# ============================================================
# 18) Diagnostics
# ============================================================

print(
    "=== FINAL PILOT EXACT BOOK LADDER EVIDENCE ==="
)


print(
    "\n=== BOOK ==="
)

print(
    "Work ID:",
    CELL101_WORK_ID,
)

print(
    "Edition ID:",
    CELL101_EDITION_ID,
)

print(
    "Book files:",
    [
        str(
            path
        )
        for path
        in cell101_existing_book_paths
    ],
)

print(
    "Unique Book records:",
    len(
        cell101_book_records
    ),
)

print(
    "Lesson 2 records:",
    len(
        cell101_lesson_records
    ),
)


print(
    "\n=== FIVE SOURCE-NATIVE STAGES ==="
)


for stage_definition in CELL101_STAGE_DEFINITIONS:

    stage_key = stage_definition[
        "canonical_stage_key"
    ]


    result = cell101_stage_evidence[
        stage_key
    ]


    print(
        "\nStage:",
        result[
            "stage_number"
        ],
    )

    print(
        "Preferred Source Title:",
        result[
            "preferred_source_title"
        ],
    )

    print(
        "Found:",
        result[
            "found"
        ],
    )

    print(
        "Matching paragraphs:",
        [
            record.get(
                "paragraph_number"
            )
            for record
            in result[
                "records"
            ]
        ],
    )


print(
    "\nAll five stages found in exact Book:",
    cell101_all_five_stages_found_in_book,
)


print(
    "\n=== STRUCTURED FIVE-STAGE BOOK EVIDENCE ==="
)


for block in cell101_structured_stage_blocks:

    print(
        "\n"
        +
        "="
        *
        90
    )

    print(
        "Stage:",
        block[
            "stage_number"
        ],
    )

    print(
        "Source Title:",
        block[
            "preferred_source_title"
        ],
    )

    print(
        "Paragraphs:",
        block[
            "paragraph_numbers"
        ],
    )

    print(
        "Complete:",
        block[
            "complete"
        ],
    )

    print(
        "Text:"
    )

    print(
        block[
            "combined_text"
        ]
    )


print(
    "\n=== SEMANTIC VALIDATION ==="
)


for stage_number, checks in (
    cell101_stage_semantic_checks.items()
):

    print(
        f"\nStage {stage_number}:"
    )


    for name, status in checks.items():

        print(
            f"  {name}:",
            status,
        )


    print(
        "  PASS:",
        cell101_stage_semantic_validation[
            stage_number
        ],
    )


print(
    "\n=== SELF-ASSESSMENT ==="
)


for record in cell101_self_assessment_records:

    print(
        "\nParagraph:",
        record.get(
            "paragraph_number"
        ),
    )

    print(
        "Paragraph ID:",
        record.get(
            "paragraph_id"
        ),
    )

    print(
        "Text:",
        record.get(
            "text"
        ),
    )


print(
    "\nSelf-assessment valid:",
    cell101_self_assessment_valid,
)


print(
    "\n=== FINAL BOOK-PRIMARY EVIDENCE ASSESSMENT ==="
)

print(
    "Stage-list evidence sufficient:",
    cell101_book_primary_evidence_sufficient_for_stage_list,
)

print(
    "Full self-assessment answer evidence sufficient:",
    cell101_book_primary_evidence_sufficient_for_full_self_assessment_answer,
)

print(
    "Supporting Book paragraph numbers:",
    cell101_supporting_paragraph_numbers,
)

print(
    "IMPORTANT: Human recollection is NOT used as Golden evidence."
)


print(
    "\n=== ALL INTERNAL GATES ==="
)


for name, status in (
    cell101_gate_checks.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== SAFETY ==="
)

print(
    "Registry write:",
    cell101_registry_write,
)

print(
    "Queue advance:",
    cell101_queue_advance,
)

print(
    "Human Decision added:",
    cell101_human_decision_added,
)

print(
    "Target changed:",
    cell101_target_changed,
)

print(
    "Citation changed:",
    cell101_citation_changed,
)

print(
    "Raw Source mutated:",
    cell101_raw_source_mutated,
)

print(
    "Parser mutated:",
    cell101_parser_mutated,
)

print(
    "Commit / Push:",
    cell101_commit_push,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell101_failures
)


print(
    "\nCODE CELL 101 GATE:",
    (
        "PASS"
        if cell101_gate
        else
        "FAIL"
    ),
)


assert cell101_gate, (
    "Final Pilot exact Book ladder-evidence diagnostic failed. "
    "Repair this SAME Code Cell 101 before final Question/Answer drafting."
)

=== FINAL PILOT EXACT BOOK LADDER EVIDENCE ===

=== BOOK ===
Work ID: book:adult-financial-literacy-course
Edition ID: book:adult-financial-literacy-course:fa-source
Book files: ['/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/data/phase2/interim/book_ingested_v1.json', '/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/data/phase2/interim/book_ingested_v2.json']
Unique Book records: 1333
Lesson 2 records: 70

=== FIVE SOURCE-NATIVE STAGES ===

Stage: 1
Preferred Source Title: ناتوانی مالی
Found: True
Matching paragraphs: [141, 201]

Stage: 2
Preferred Source Title: تقلّای مالی
Found: True
Matching paragraphs: [144]

Stage: 3
Preferred Source Title: تعادل مالی
Found: True
Matching paragraphs: [148, 156, 158]

Stage: 4
Preferred Source Title: تاب‌آوری مالی (امنیت مالی)
Found: True
Matching paragraphs: [154]

Stage: 5
Preferred Source Title: رفاه مالی (آزادی مالی)
Found: True
Matching paragraphs: [157]

All five stages found in exact Boo

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 102 — Final Pilot Question/Answer ACCEPT + Book Citation Draft

Human Reviewer سؤال و جواب Pilot نهایی را پذیرفت.

سؤال پذیرفته‌شده:

**پلکان توانمندی مالی چه مراحلی دارد و چگونه می‌توانیم بفهمیم در کدام مرحله قرار داریم؟**

جواب پذیرفته‌شده:

**پلکان توانمندی مالی پنج مرحله دارد:**

1. **ناتوانی مالی:** فرد برای گذران زندگی دائماً بدهکار است، بدون قرض نمی‌تواند زندگی کند و حتی برای پرداخت بدهی قبلی دوباره قرض می‌گیرد.
2. **تقلّای مالی:** فرد برای زندگی قرض می‌گیرد، اما می‌تواند بدهی را از درآمد آینده پرداخت کند؛ یعنی درآمد آینده خود را از قبل خرج می‌کند.
3. **تعادل مالی:** فرد امور مالی روزانه را اداره می‌کند و درآمد و هزینه‌اش متعادل است، اما برای اهداف کوتاه‌مدت یا بلندمدت پس‌انداز ندارد و در برابر شوک‌های مالی آسیب‌پذیر است.
4. **تاب‌آوری مالی (امنیت مالی):** علاوه بر تعادل دخل‌وخرج، فرد توان تحمل و عبور از شوک‌های مالی مانند بیکاری یا هزینه‌های ناگهانی را دارد.
5. **رفاه مالی (آزادی مالی):** فرد علاوه بر تعادل مالی و توان عبور از شوک‌ها، می‌تواند به خواسته‌ها و اهداف مالی خود برسد و پول دیگر مؤلفه‌ای جدی در تصمیم‌گیری او نیست.

برای تشخیص جایگاه فعلی، فرد نباید وضعیت خود را با دیگران مقایسه کند؛ بلکه باید شرایط واقعی مالی خود را با ویژگی‌های این پنج مرحله مقایسه کند، ببیند وضعیت فعلی او بیشتر با کدام مرحله تطابق دارد و سپس چند جمله درباره وضعیت فعلی خود بنویسد.

Target اصلی خودارزیابی تغییر نمی‌کند.  
پاراگراف‌های 140 تا 159 همان درس، به‌عنوان **Human-Certified Supporting Book Evidence Extension** برای تعریف پنج مرحله استفاده می‌شوند و Paragraph 163 نیز بخش خودارزیابی را پشتیبانی می‌کند.

پس از این Cell، Citation در وضعیت Human Review قرار می‌گیرد و Queue همچنان 1 باقی می‌ماند.

</div>

In [104]:
# Code Cell 102

cell102_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell101_gate") is not True:
    cell102_failures.append(
        "CELL101_NOT_PASS"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell102_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


if not isinstance(
    globals().get(
        "cell98_next_pilot_packet"
    ),
    dict,
):
    cell102_failures.append(
        "FINAL_PILOT_PACKET_MISSING"
    )


# ============================================================
# 2) Accepted Question / Answer
# ============================================================

CELL102_ACCEPTED_QUESTION = (
    "پلکان توانمندی مالی چه مراحلی دارد و چگونه می‌توانیم "
    "بفهمیم در کدام مرحله قرار داریم؟"
)


CELL102_ACCEPTED_ANSWER = (
    "پلکان توانمندی مالی پنج مرحله دارد:\n"
    "1. ناتوانی مالی: فرد برای گذران زندگی دائماً بدهکار است، "
    "بدون قرض نمی‌تواند زندگی کند و حتی برای پرداخت بدهی قبلی "
    "دوباره قرض می‌گیرد.\n"
    "2. تقلّای مالی: فرد برای زندگی قرض می‌گیرد، اما می‌تواند "
    "بدهی را از درآمد آینده پرداخت کند؛ یعنی درآمد آینده خود را "
    "از قبل خرج می‌کند.\n"
    "3. تعادل مالی: فرد امور مالی روزانه را اداره می‌کند و درآمد "
    "و هزینه‌اش متعادل است، اما برای اهداف کوتاه‌مدت یا بلندمدت "
    "پس‌انداز ندارد و در برابر شوک‌های مالی آسیب‌پذیر است.\n"
    "4. تاب‌آوری مالی (امنیت مالی): علاوه بر تعادل دخل‌وخرج، "
    "فرد توان تحمل و عبور از شوک‌های مالی مانند بیکاری یا "
    "هزینه‌های ناگهانی را دارد.\n"
    "5. رفاه مالی (آزادی مالی): فرد علاوه بر تعادل مالی و توان "
    "عبور از شوک‌ها، می‌تواند به خواسته‌ها و اهداف مالی خود برسد "
    "و پول دیگر مؤلفه‌ای جدی در تصمیم‌گیری او نیست.\n"
    "برای تشخیص جایگاه فعلی، فرد نباید وضعیت خود را با دیگران "
    "مقایسه کند؛ بلکه باید شرایط واقعی مالی خود را با ویژگی‌های "
    "این پنج مرحله مقایسه کند، ببیند وضعیت فعلی او بیشتر با کدام "
    "مرحله تطابق دارد و سپس چند جمله درباره وضعیت فعلی خود بنویسد."
)


CELL102_PILOT_ID = (
    CELL98_NEXT_PILOT_ID
)


CELL102_GOLDEN_CASE_ID = (
    CELL98_NEXT_GOLDEN_CASE_ID
)


CELL102_TASK_DRAFT_ID = (
    CELL98_NEXT_TASK_DRAFT_ID
)


CELL102_TARGET_EVIDENCE_ID = (
    CELL98_NEXT_TARGET_EVIDENCE_ID
)


# ============================================================
# 3) Book-primary evidence validation
# ============================================================

cell102_book_evidence_valid = all([
    cell101_book_primary_evidence_sufficient_for_stage_list,
    cell101_book_primary_evidence_sufficient_for_full_self_assessment_answer,
    cell101_all_stage_semantics_valid,
    cell101_self_assessment_valid,
])


if not cell102_book_evidence_valid:
    cell102_failures.append(
        "BOOK_PRIMARY_EVIDENCE_INVALID"
    )


# ============================================================
# 4) Preserve original Target Evidence
# ============================================================

cell102_original_target_parts = copy.deepcopy(
    cell98_next_target_text_parts
)


cell102_original_target_valid = all([
    isinstance(
        cell102_original_target_parts,
        list,
    ),

    len(
        cell102_original_target_parts
    )
    ==
    3,

    all(
        isinstance(
            part,
            dict,
        )
        for part
        in cell102_original_target_parts
    ),

    {
        part.get(
            "locator",
            {},
        ).get(
            "paragraph_number"
        )
        for part
        in cell102_original_target_parts
    }
    ==
    {
        162,
        163,
        164,
    },
])


if not cell102_original_target_valid:
    cell102_failures.append(
        "ORIGINAL_TARGET_INVALID"
    )


# ============================================================
# 5) Build Human-Certified Supporting Book Evidence Extension
# ============================================================

cell102_target_attribution = (
    copy.deepcopy(
        cell102_original_target_parts[
            0
        ].get(
            "attribution",
            {},
        )
    )
    if cell102_original_target_parts
    else
    {}
)


cell102_support_paragraph_numbers = (
    list(
        range(
            140,
            160,
        )
    )
    +
    [
        163,
    ]
)


cell102_supporting_book_entries = []


for citation_index, paragraph_number in enumerate(
    cell102_support_paragraph_numbers
):

    source_record = (
        cell101_records_by_paragraph_number.get(
            paragraph_number
        )
    )


    if not isinstance(
        source_record,
        dict,
    ):

        cell102_failures.append(
            f"BOOK_PARAGRAPH_{paragraph_number}_MISSING"
        )

        continue


    text = source_record.get(
        "text",
        ""
    )


    if not isinstance(
        text,
        str,
    ) or not text.strip():

        cell102_failures.append(
            f"BOOK_PARAGRAPH_{paragraph_number}_EMPTY"
        )

        continue


    is_self_assessment_support = (
        paragraph_number
        ==
        163
    )


    cell102_supporting_book_entries.append({
        "citation_index":
            citation_index,

        "document_id":
            source_record.get(
                "paragraph_id"
            ),

        "role":
            (
                "EXPLICIT_BOOK_EXERCISE_UNIT"
                if is_self_assessment_support
                else
                "HUMAN_CERTIFIED_SUPPORTING_BOOK_EVIDENCE"
            ),

        "text":
            text,

        "text_sha256":
            source_record.get(
                "text_sha256"
            )
            or
            hashlib.sha256(
                text.encode(
                    "utf-8"
                )
            ).hexdigest(),

        "char_start":
            0,

        "char_end":
            len(
                text
            ),

        "locator": {
            "work_id":
                source_record.get(
                    "work_id"
                )
                or
                CELL101_WORK_ID,

            "edition_id":
                source_record.get(
                    "edition_id"
                )
                or
                CELL101_EDITION_ID,

            "paragraph_id":
                source_record.get(
                    "paragraph_id"
                ),

            "paragraph_number":
                source_record.get(
                    "paragraph_number"
                ),

            "heading_path":
                copy.deepcopy(
                    source_record.get(
                        "heading_path",
                        [],
                    )
                ),
        },

        "attribution":
            copy.deepcopy(
                cell102_target_attribution
            ),

        "authority_role":
            "PRIMARY_ANSWER_EVIDENCE",

        "book_primary_evidence":
            True,

        "answer_support":
            True,

        "use_in_expected_answer_grounding":
            True,

        "evidence_origin":
            (
                "ORIGINAL_TARGET_EVIDENCE"
                if is_self_assessment_support
                else
                "HUMAN_CERTIFIED_SUPPORTING_EVIDENCE_EXTENSION"
            ),

        "target_evidence_id_preserved":
            True,

        "external_knowledge_used":
            False,
    })


cell102_supporting_extension = {
    "extension_version":
        "human_certified_supporting_book_evidence_extension_v1",

    "pilot_case_id":
        CELL102_PILOT_ID,

    "target_evidence_id":
        CELL102_TARGET_EVIDENCE_ID,

    "target_changed":
        False,

    "retarget_used":
        False,

    "source_type":
        "BOOK_PRIMARY_EVIDENCE",

    "work_id":
        CELL101_WORK_ID,

    "edition_id":
        CELL101_EDITION_ID,

    "supporting_paragraph_numbers":
        copy.deepcopy(
            cell102_support_paragraph_numbers
        ),

    "reason":
        (
            "The original self-assessment target requires the definitions "
            "of the five financial capability stages from the immediately "
            "preceding Book lesson in order to answer the structure-aware "
            "self-assessment question completely."
        ),

    "human_certified":
        True,

    "raw_source_mutated":
        False,

    "parser_output_mutated":
        False,

    "external_knowledge_used":
        False,
}


cell102_supporting_extension_valid = all([
    len(
        cell102_supporting_book_entries
    )
    ==
    21,

    {
        entry[
            "locator"
        ][
            "paragraph_number"
        ]
        for entry
        in cell102_supporting_book_entries
    }
    ==
    set(
        cell102_support_paragraph_numbers
    ),

    all(
        entry.get(
            "book_primary_evidence"
        )
        is True
        for entry
        in cell102_supporting_book_entries
    ),

    cell102_supporting_extension[
        "target_changed"
    ]
    is False,

    cell102_supporting_extension[
        "retarget_used"
    ]
    is False,
])


if not cell102_supporting_extension_valid:
    cell102_failures.append(
        "SUPPORTING_BOOK_EXTENSION_INVALID"
    )


# ============================================================
# 6) Grounding checks
# ============================================================

cell102_stage_grounding = {
    "stage_1_financial_inability":
        all([
            141
            in
            cell102_support_paragraph_numbers,

            142
            in
            cell102_support_paragraph_numbers,

            143
            in
            cell102_support_paragraph_numbers,
        ]),

    "stage_2_financial_struggle":
        all([
            144
            in
            cell102_support_paragraph_numbers,

            145
            in
            cell102_support_paragraph_numbers,

            146
            in
            cell102_support_paragraph_numbers,

            147
            in
            cell102_support_paragraph_numbers,
        ]),

    "stage_3_financial_balance":
        all([
            148
            in
            cell102_support_paragraph_numbers,

            149
            in
            cell102_support_paragraph_numbers,

            150
            in
            cell102_support_paragraph_numbers,

            151
            in
            cell102_support_paragraph_numbers,
        ]),

    "stage_4_resilience_security":
        all([
            154
            in
            cell102_support_paragraph_numbers,

            155
            in
            cell102_support_paragraph_numbers,

            156
            in
            cell102_support_paragraph_numbers,
        ]),

    "stage_5_wellbeing_freedom":
        all([
            157
            in
            cell102_support_paragraph_numbers,

            158
            in
            cell102_support_paragraph_numbers,

            159
            in
            cell102_support_paragraph_numbers,
        ]),

    "assessment_method":
        all([
            140
            in
            cell102_support_paragraph_numbers,

            163
            in
            cell102_support_paragraph_numbers,
        ]),
}


cell102_grounding_pass = all(
    cell102_stage_grounding.values()
)


if not cell102_grounding_pass:
    cell102_failures.append(
        "QUESTION_ANSWER_GROUNDING_FAILED"
    )


# ============================================================
# 7) Start Engine
# ============================================================

cell102_question_pending_session = (
    cell13_build_session(
        cell98_next_pilot_packet
    )
)


cell102_session_start_valid = (
    cell102_question_pending_session.get(
        "current_state"
    )
    ==
    "QUESTION_DRAFT_PENDING"
)


if not cell102_session_start_valid:
    cell102_failures.append(
        "ENGINE_SESSION_START_INVALID"
    )


# ============================================================
# 8) Question Draft + Human ACCEPT
# ============================================================

cell102_question_identity = {
    "identity_version":
        "question_draft_identity_v1",

    "pilot_case_id":
        CELL102_PILOT_ID,

    "target_evidence_id":
        CELL102_TARGET_EVIDENCE_ID,

    "question_text":
        CELL102_ACCEPTED_QUESTION,

    "revision_number":
        1,
}


cell102_question_draft_id = (
    "question-draft:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell102_question_identity
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


cell102_question_record = {
    "record_version":
        "question_draft_record_v1",

    "question_draft_id":
        cell102_question_draft_id,

    "pilot_case_id":
        CELL102_PILOT_ID,

    "golden_case_id":
        CELL102_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL102_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL102_TARGET_EVIDENCE_ID,

    "question_text":
        CELL102_ACCEPTED_QUESTION,

    "revision_number":
        1,

    "human_adjudication_status":
        "PENDING",

    "external_knowledge_used":
        False,

    "supporting_evidence_extension_used":
        True,
}


cell102_question_review_session = (
    cell13_apply_event(
        cell102_question_pending_session,
        "QUESTION_DRAFT_REGISTERED",
        artifact=
            cell102_question_record,
    )
)


cell102_answer_pending_session = (
    cell13_apply_event(
        cell102_question_review_session,
        "QUESTION_ACCEPTED",
        human_note=
            "Human Reviewer accepted the final Pilot question.",
    )
)


cell102_question_accept_valid = (
    cell102_answer_pending_session.get(
        "current_state"
    )
    ==
    "EXPECTED_ANSWER_DRAFT_PENDING"
)


if not cell102_question_accept_valid:
    cell102_failures.append(
        "QUESTION_ACCEPT_TRANSITION_INVALID"
    )


# ============================================================
# 9) Expected Answer + Human ACCEPT
# ============================================================

cell102_answer_identity = {
    "identity_version":
        "expected_answer_identity_v1",

    "pilot_case_id":
        CELL102_PILOT_ID,

    "question_draft_id":
        cell102_question_draft_id,

    "target_evidence_id":
        CELL102_TARGET_EVIDENCE_ID,

    "answer_text":
        CELL102_ACCEPTED_ANSWER,

    "revision_number":
        1,
}


cell102_expected_answer_draft_id = (
    "expected-answer-draft:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell102_answer_identity
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


cell102_expected_answer_record = {
    "record_version":
        "expected_answer_draft_record_v1",

    "expected_answer_draft_id":
        cell102_expected_answer_draft_id,

    "pilot_case_id":
        CELL102_PILOT_ID,

    "golden_case_id":
        CELL102_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL102_TASK_DRAFT_ID,

    "question_draft_id":
        cell102_question_draft_id,

    "target_evidence_id":
        CELL102_TARGET_EVIDENCE_ID,

    "expected_answer_text":
        CELL102_ACCEPTED_ANSWER,

    "revision_number":
        1,

    "human_adjudication_status":
        "PENDING",

    "external_knowledge_used":
        False,

    "book_primary_evidence":
        True,

    "supporting_evidence_extension_used":
        True,
}


cell102_answer_review_session = (
    cell13_apply_event(
        cell102_answer_pending_session,
        "EXPECTED_ANSWER_REGISTERED",
        artifact=
            cell102_expected_answer_record,
    )
)


cell102_citation_pending_session = (
    cell13_apply_event(
        cell102_answer_review_session,
        "EXPECTED_ANSWER_ACCEPTED",
        human_note=
            "Human Reviewer accepted the final Pilot expected answer.",
    )
)


cell102_answer_accept_valid = (
    cell102_citation_pending_session.get(
        "current_state"
    )
    ==
    "CITATION_MAPPING_PENDING"
)


if not cell102_answer_accept_valid:
    cell102_failures.append(
        "ANSWER_ACCEPT_TRANSITION_INVALID"
    )


# ============================================================
# 10) Citation Mapping
# ============================================================

cell102_citation_identity = {
    "identity_version":
        "citation_mapping_identity_v1_book_structure_aware",

    "pilot_case_id":
        CELL102_PILOT_ID,

    "question_draft_id":
        cell102_question_draft_id,

    "expected_answer_draft_id":
        cell102_expected_answer_draft_id,

    "target_evidence_id":
        CELL102_TARGET_EVIDENCE_ID,

    "supporting_evidence_extension":
        copy.deepcopy(
            cell102_supporting_extension
        ),

    "citation_entries":
        copy.deepcopy(
            cell102_supporting_book_entries
        ),
}


cell102_citation_mapping_id = (
    "citation-mapping:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell102_citation_identity
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


cell102_citation_mapping_record = {
    "record_version":
        "citation_mapping_record_v1_book_structure_aware",

    "citation_mapping_id":
        cell102_citation_mapping_id,

    "pilot_case_id":
        CELL102_PILOT_ID,

    "golden_case_id":
        CELL102_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL102_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL102_TARGET_EVIDENCE_ID,

    "question_draft_id":
        cell102_question_draft_id,

    "expected_answer_draft_id":
        cell102_expected_answer_draft_id,

    "citation_entries":
        copy.deepcopy(
            cell102_supporting_book_entries
        ),

    "supporting_evidence_extension":
        copy.deepcopy(
            cell102_supporting_extension
        ),

    "authority_expectation":
        "BOOK_PRIMARY_EVIDENCE",

    "work_id":
        CELL101_WORK_ID,

    "edition_id":
        CELL101_EDITION_ID,

    "original_target_preserved":
        True,

    "retarget_used":
        False,

    "book_primary_evidence_only":
        True,

    "external_knowledge_used":
        False,

    "raw_source_mutated":
        False,

    "parser_output_mutated":
        False,

    "human_adjudication_status":
        "PENDING",
}


cell102_citation_boundary_valid = all([
    len(
        cell102_citation_mapping_record[
            "citation_entries"
        ]
    )
    ==
    21,

    all(
        entry.get(
            "book_primary_evidence"
        )
        is True
        for entry
        in cell102_citation_mapping_record[
            "citation_entries"
        ]
    ),

    cell102_citation_mapping_record[
        "authority_expectation"
    ]
    ==
    "BOOK_PRIMARY_EVIDENCE",

    cell102_citation_mapping_record[
        "original_target_preserved"
    ]
    is True,

    cell102_citation_mapping_record[
        "retarget_used"
    ]
    is False,

    cell102_citation_mapping_record[
        "external_knowledge_used"
    ]
    is False,
])


if not cell102_citation_boundary_valid:
    cell102_failures.append(
        "CITATION_BOUNDARY_INVALID"
    )


cell102_current_session = (
    cell13_apply_event(
        cell102_citation_pending_session,
        "CITATION_MAPPING_REGISTERED",
        artifact=
            cell102_citation_mapping_record,
    )
)


cell102_citation_registered_valid = (
    cell102_current_session.get(
        "current_state"
    )
    ==
    "CITATION_HUMAN_REVIEW_PENDING"
)


if not cell102_citation_registered_valid:
    cell102_failures.append(
        "CITATION_REGISTRATION_INVALID"
    )


# ============================================================
# 11) Active Workspace
# ============================================================

cell102_active_workspace = copy.deepcopy(
    cell98_persisted_active_case
)


cell102_active_workspace[
    "workflow_state"
] = "CITATION_HUMAN_REVIEW_PENDING"


cell102_active_workspace[
    "supporting_evidence_extension"
] = copy.deepcopy(
    cell102_supporting_extension
)


cell102_active_workspace[
    "supporting_book_evidence"
] = copy.deepcopy(
    cell102_supporting_book_entries
)


cell102_active_workspace[
    "question_review"
] = {
    "question_draft_id":
        cell102_question_draft_id,

    "question_text":
        CELL102_ACCEPTED_QUESTION,

    "human_decision":
        "ACCEPT",

    "external_knowledge_used":
        False,
}


cell102_active_workspace[
    "expected_answer_review"
] = {
    "expected_answer_draft_id":
        cell102_expected_answer_draft_id,

    "expected_answer_text":
        CELL102_ACCEPTED_ANSWER,

    "human_decision":
        "ACCEPT",

    "book_primary_evidence":
        True,

    "supporting_evidence_extension_used":
        True,

    "external_knowledge_used":
        False,
}


cell102_active_workspace[
    "citation_review"
] = {
    "citation_mapping_id":
        cell102_citation_mapping_id,

    "citation_mapping_record":
        copy.deepcopy(
            cell102_citation_mapping_record
        ),

    "human_decision":
        "PENDING",

    "authority_expectation":
        "BOOK_PRIMARY_EVIDENCE",

    "book_primary_evidence_only":
        True,
}


# ============================================================
# 12) Continuation State
# ============================================================

cell102_continuation_core = {
    "next_pilot_id":
        CELL102_PILOT_ID,

    "remaining_pilot_ids":
        [
            CELL102_PILOT_ID
        ],

    "remaining_count":
        1,

    "active_pilot_status":
        "CITATION_HUMAN_REVIEW_PENDING",

    "active_target_evidence_id":
        CELL102_TARGET_EVIDENCE_ID,

    "active_citation_mapping_id":
        cell102_citation_mapping_id,

    "retarget_used":
        False,

    "supporting_book_evidence_extension_used":
        True,

    "authority_expectation":
        "BOOK_PRIMARY_EVIDENCE",

    "last_completed_pilot_id":
        cell98_continuation_state.get(
            "last_completed_pilot_id"
        ),
}


cell102_continuation_state_sha256 = (
    cell10_stable_sha(
        cell102_continuation_core
    )
)


cell102_continuation_state = {
    **copy.deepcopy(
        cell102_continuation_core
    ),

    "continuation_state_sha256":
        cell102_continuation_state_sha256,
}


# ============================================================
# 13) Persist
# ============================================================

cell102_persistent_registry_payload = copy.deepcopy(
    cell98_persisted_registry
)


cell102_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell102_active_workspace
)


cell102_persistent_registry_payload[
    "remaining_pilot_count"
] = 1


cell102_persistent_registry_payload[
    "continuation_state"
] = copy.deepcopy(
    cell102_continuation_state
)


cell102_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell102_persistent_registry_payload
    )
)


# ============================================================
# 14) Read-back validation
# ============================================================

cell102_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell102_persisted_active_case = (
    cell102_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell102_persistence_valid = all([
    cell102_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL102_PILOT_ID,

    cell102_persisted_active_case.get(
        "workflow_state"
    )
    ==
    "CITATION_HUMAN_REVIEW_PENDING",

    cell102_persisted_active_case.get(
        "question_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell102_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell102_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "book_primary_evidence"
    )
    is True,

    cell102_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "PENDING",

    cell102_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "book_primary_evidence_only"
    )
    is True,

    cell102_persisted_active_case.get(
        "supporting_evidence_extension",
        {},
    ).get(
        "human_certified"
    )
    is True,

    cell102_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    1,

    cell102_persisted_registry.get(
        "continuation_state",
        {},
    ).get(
        "continuation_state_sha256"
    )
    ==
    cell102_continuation_state_sha256,
])


if not cell102_persistence_valid:
    cell102_failures.append(
        "REGISTRY_READBACK_INVALID"
    )


# ============================================================
# 15) Safety
# ============================================================

cell102_queue_still_1 = all([
    cell102_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    1,

    cell102_persisted_registry.get(
        "continuation_state",
        {},
    ).get(
        "remaining_pilot_ids"
    )
    ==
    [
        CELL102_PILOT_ID
    ],
])


cell102_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell102_full_572_blocked = (
    cell98_full_572_blocked
    is True
)


cell102_no_commit_push = True


# ============================================================
# 16) Gate
# ============================================================

cell102_gate_checks = {
    "no_failures":
        len(
            cell102_failures
        )
        ==
        0,

    "book_evidence_valid":
        cell102_book_evidence_valid,

    "original_target_valid":
        cell102_original_target_valid,

    "supporting_extension_valid":
        cell102_supporting_extension_valid,

    "grounding_pass":
        cell102_grounding_pass,

    "session_start_valid":
        cell102_session_start_valid,

    "question_accept_valid":
        cell102_question_accept_valid,

    "answer_accept_valid":
        cell102_answer_accept_valid,

    "citation_boundary_valid":
        cell102_citation_boundary_valid,

    "citation_registered_valid":
        cell102_citation_registered_valid,

    "persistence_valid":
        cell102_persistence_valid,

    "queue_still_1":
        cell102_queue_still_1,

    "base_seed_unchanged":
        cell102_base_seed_unchanged,

    "full_572_blocked":
        cell102_full_572_blocked,

    "no_commit_push":
        cell102_no_commit_push,
}


cell102_gate = all(
    cell102_gate_checks.values()
)


# ============================================================
# 17) Diagnostics
# ============================================================

print(
    "=== FINAL PILOT QA ACCEPT + BOOK CITATION DRAFT ==="
)


print(
    "\n=== PILOT ==="
)

print(
    "Pilot:",
    CELL102_PILOT_ID,
)

print(
    "Target Evidence:",
    CELL102_TARGET_EVIDENCE_ID,
)

print(
    "Authority Expectation:",
    "BOOK_PRIMARY_EVIDENCE",
)

print(
    "Retarget used:",
    False,
)

print(
    "Original Target preserved:",
    True,
)


print(
    "\n=== ACCEPTED QUESTION ==="
)

print(
    CELL102_ACCEPTED_QUESTION
)

print(
    "Question Draft ID:",
    cell102_question_draft_id,
)

print(
    "Human Decision:",
    "ACCEPT",
)


print(
    "\n=== ACCEPTED ANSWER ==="
)

print(
    CELL102_ACCEPTED_ANSWER
)

print(
    "Expected Answer Draft ID:",
    cell102_expected_answer_draft_id,
)

print(
    "Human Decision:",
    "ACCEPT",
)


print(
    "\n=== SUPPORTING BOOK EVIDENCE EXTENSION ==="
)

print(
    "Extension Type:",
    "HUMAN_CERTIFIED_SUPPORTING_EVIDENCE_EXTENSION",
)

print(
    "Work ID:",
    CELL101_WORK_ID,
)

print(
    "Edition ID:",
    CELL101_EDITION_ID,
)

print(
    "Supporting paragraphs:",
    cell102_support_paragraph_numbers,
)

print(
    "Citation entries:",
    len(
        cell102_supporting_book_entries
    ),
)

print(
    "Target changed:",
    False,
)

print(
    "Raw Source mutated:",
    False,
)

print(
    "Parser mutated:",
    False,
)


print(
    "\n=== GROUNDING ==="
)

for name, status in (
    cell102_stage_grounding.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "Grounding PASS:",
    cell102_grounding_pass,
)


print(
    "\n=== CITATION DRAFT ==="
)

print(
    "Citation Mapping ID:",
    cell102_citation_mapping_id,
)

print(
    "Book Primary Evidence only:",
    True,
)

print(
    "Citation boundary valid:",
    cell102_citation_boundary_valid,
)


print(
    "\n=== WORKFLOW ==="
)

print(
    "Current State:",
    cell102_current_session.get(
        "current_state"
    ),
)

print(
    "Question:",
    "ACCEPT",
)

print(
    "Expected Answer:",
    "ACCEPT",
)

print(
    "Citation:",
    "PENDING",
)


print(
    "\n=== ALL INTERNAL GATES ==="
)

for name, status in (
    cell102_gate_checks.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== QUEUE / SAFETY ==="
)

print(
    "Queue still 1:",
    cell102_queue_still_1,
)

print(
    "Persistence valid:",
    cell102_persistence_valid,
)

print(
    "Base Seed unchanged:",
    cell102_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell102_full_572_blocked,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell102_failures
)


print(
    "\nCODE CELL 102 GATE:",
    (
        "PASS"
        if cell102_gate
        else
        "FAIL"
    ),
)


assert cell102_gate, (
    "Final Pilot Question+Answer ACCEPT / Book Citation Draft failed. "
    "Repair this SAME Code Cell 102."
)

=== FINAL PILOT QA ACCEPT + BOOK CITATION DRAFT ===

=== PILOT ===
Pilot: question-pilot:2beb40d7d1262dbf386db581f54d50d2
Target Evidence: target-evidence:1352ce8332affbe3651fe0558449e2a9
Authority Expectation: BOOK_PRIMARY_EVIDENCE
Retarget used: False
Original Target preserved: True

=== ACCEPTED QUESTION ===
پلکان توانمندی مالی چه مراحلی دارد و چگونه می‌توانیم بفهمیم در کدام مرحله قرار داریم؟
Question Draft ID: question-draft:bbb39286861085ac153ff26dc5e8a2dc
Human Decision: ACCEPT

=== ACCEPTED ANSWER ===
پلکان توانمندی مالی پنج مرحله دارد:
1. ناتوانی مالی: فرد برای گذران زندگی دائماً بدهکار است، بدون قرض نمی‌تواند زندگی کند و حتی برای پرداخت بدهی قبلی دوباره قرض می‌گیرد.
2. تقلّای مالی: فرد برای زندگی قرض می‌گیرد، اما می‌تواند بدهی را از درآمد آینده پرداخت کند؛ یعنی درآمد آینده خود را از قبل خرج می‌کند.
3. تعادل مالی: فرد امور مالی روزانه را اداره می‌کند و درآمد و هزینه‌اش متعادل است، اما برای اهداف کوتاه‌مدت یا بلندمدت پس‌انداز ندارد و در برابر شوک‌های مالی آسیب‌پذیر است.
4. تاب‌آ

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 103 — Final Citation ACCEPT + Pilot Human Review Queue Complete

Citation آخرین Pilot توسط Human Reviewer پذیرفته شد.

در این مرحله:

- Citation نهایی با تصمیم `ACCEPT` ثبت می‌شود؛
- آخرین Pilot به وضعیت `PILOT_COMPLETE` می‌رسد؛
- Target اصلی خودارزیابی حفظ می‌شود؛
- Supporting Book Evidence از همان کتاب و همان درس حفظ می‌شود؛
- تعداد Caseهای کامل Human Review از 21 به 22 می‌رسد؛
- تعداد Caseهای آماده Author Review از 21 به 22 می‌رسد؛
- Queue دقیقاً از 1 به 0 می‌رسد؛
- Active Human Review Workspace بسته می‌شود؛
- Author Final Review همچنان `NOT_STARTED` باقی می‌ماند؛
- Full 572 Generation فعلاً تا انجام Final Health / Run-All Validation همچنان Blocked باقی می‌ماند.

این Cell هیچ Commit یا Push انجام نمی‌دهد.

</div>

In [105]:
# Code Cell 103

cell103_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell102_gate") is not True:
    cell103_failures.append(
        "CELL102_NOT_PASS"
    )


if (
    globals().get(
        "cell102_current_session",
        {},
    ).get(
        "current_state"
    )
    !=
    "CITATION_HUMAN_REVIEW_PENDING"
):
    cell103_failures.append(
        "CITATION_NOT_AT_HUMAN_REVIEW_GATE"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell103_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Final Citation Human ACCEPT
# ============================================================

CELL103_CITATION_HUMAN_NOTE = (
    "Human Reviewer accepted the final Citation Mapping. "
    "The Expected Answer is grounded only in Book Primary Evidence "
    "from the adult financial literacy course. "
    "The original self-assessment target is preserved, and the "
    "immediately preceding lesson paragraphs are retained as a "
    "Human-Certified Supporting Book Evidence Extension."
)


cell103_complete_pending_session = (
    cell13_apply_event(
        cell102_current_session,
        "CITATION_ACCEPTED",
        human_note=
            CELL103_CITATION_HUMAN_NOTE,
    )
)


cell103_citation_accept_valid = (
    cell103_complete_pending_session.get(
        "current_state"
    )
    ==
    "PILOT_COMPLETE_PENDING_QUEUE_ADVANCE"
)


if not cell103_citation_accept_valid:
    cell103_failures.append(
        "CITATION_ACCEPT_TRANSITION_INVALID"
    )


# ============================================================
# 3) Final Citation Mapping Record
# ============================================================

cell103_final_citation_mapping_record = copy.deepcopy(
    cell102_citation_mapping_record
)


cell103_final_citation_mapping_record[
    "human_adjudication_status"
] = "ACCEPT"


cell103_final_citation_mapping_record[
    "human_review"
] = {
    "human_decision":
        "ACCEPT",

    "human_note":
        CELL103_CITATION_HUMAN_NOTE,

    "authority_expectation":
        "BOOK_PRIMARY_EVIDENCE",

    "book_primary_evidence_only":
        True,

    "original_target_preserved":
        True,

    "supporting_book_evidence_extension_used":
        True,

    "retarget_used":
        False,

    "external_knowledge_used":
        False,

    "raw_source_mutated":
        False,

    "parser_output_mutated":
        False,
}


cell103_final_citation_valid = all([
    cell103_final_citation_mapping_record.get(
        "human_adjudication_status"
    )
    ==
    "ACCEPT",

    cell103_final_citation_mapping_record.get(
        "authority_expectation"
    )
    ==
    "BOOK_PRIMARY_EVIDENCE",

    cell103_final_citation_mapping_record.get(
        "book_primary_evidence_only"
    )
    is True,

    cell103_final_citation_mapping_record.get(
        "original_target_preserved"
    )
    is True,

    cell103_final_citation_mapping_record.get(
        "retarget_used"
    )
    is False,

    cell103_final_citation_mapping_record.get(
        "external_knowledge_used"
    )
    is False,

    cell103_final_citation_mapping_record.get(
        "human_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",
])


if not cell103_final_citation_valid:
    cell103_failures.append(
        "FINAL_CITATION_RECORD_INVALID"
    )


# ============================================================
# 4) Build Completed Final Pilot Case
# ============================================================

cell103_completed_case = copy.deepcopy(
    cell102_active_workspace
)


cell103_completed_case[
    "workflow_state"
] = "PILOT_COMPLETE"


cell103_completed_case[
    "status"
] = "COMPLETED_HUMAN_REVIEW"


cell103_completed_case[
    "final_status"
] = "PILOT_COMPLETE"


cell103_completed_case.setdefault(
    "citation_review",
    {},
)


cell103_completed_case[
    "citation_review"
][
    "human_decision"
] = "ACCEPT"


cell103_completed_case[
    "citation_review"
][
    "human_note"
] = CELL103_CITATION_HUMAN_NOTE


cell103_completed_case[
    "citation_review"
][
    "citation_mapping_record"
] = copy.deepcopy(
    cell103_final_citation_mapping_record
)


cell103_completed_case[
    "completion_summary"
] = {
    "target_evidence_id":
        CELL102_TARGET_EVIDENCE_ID,

    "question_decision":
        "ACCEPT",

    "expected_answer_decision":
        "ACCEPT",

    "citation_decision":
        "ACCEPT",

    "task_family":
        "EXERCISE_SELF_ASSESSMENT_RETRIEVAL",

    "authority_expectation":
        "BOOK_PRIMARY_EVIDENCE",

    "book_primary_evidence_only":
        True,

    "original_target_preserved":
        True,

    "supporting_book_evidence_extension_used":
        True,

    "supporting_paragraph_numbers":
        copy.deepcopy(
            cell102_support_paragraph_numbers
        ),

    "retarget_used":
        False,

    "external_knowledge_used":
        False,

    "raw_source_mutated":
        False,

    "parser_output_mutated":
        False,

    "author_final_review":
        "NOT_STARTED",
}


cell103_completed_case[
    "author_final_review"
] = {
    "status":
        "NOT_STARTED",

    "decision":
        None,

    "notes":
        None,

    "reviewer":
        "AUTHOR",
}


# ============================================================
# 5) Validate Final Pilot Completion
# ============================================================

cell103_completed_case_valid = all([
    cell103_completed_case.get(
        "pilot_case_id"
    )
    ==
    CELL102_PILOT_ID,

    cell103_completed_case.get(
        "workflow_state"
    )
    ==
    "PILOT_COMPLETE",

    cell103_completed_case.get(
        "status"
    )
    ==
    "COMPLETED_HUMAN_REVIEW",

    cell103_completed_case.get(
        "question_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell103_completed_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell103_completed_case.get(
        "citation_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell103_completed_case.get(
        "completion_summary",
        {},
    ).get(
        "book_primary_evidence_only"
    )
    is True,

    cell103_completed_case.get(
        "completion_summary",
        {},
    ).get(
        "original_target_preserved"
    )
    is True,

    cell103_completed_case.get(
        "author_final_review",
        {},
    ).get(
        "status"
    )
    ==
    "NOT_STARTED",
])


if not cell103_completed_case_valid:
    cell103_failures.append(
        "FINAL_COMPLETED_CASE_INVALID"
    )


# ============================================================
# 6) Queue advance: 1 -> 0
# ============================================================

cell103_previous_remaining_pilot_ids = [
    CELL102_PILOT_ID
]


cell103_remaining_pilot_ids = []


cell103_queue_advance_valid = all([
    len(
        cell103_previous_remaining_pilot_ids
    )
    ==
    1,

    cell103_previous_remaining_pilot_ids[
        0
    ]
    ==
    CELL102_PILOT_ID,

    len(
        cell103_remaining_pilot_ids
    )
    ==
    0,

    CELL102_PILOT_ID
    not in
    cell103_remaining_pilot_ids,
])


if not cell103_queue_advance_valid:
    cell103_failures.append(
        "FINAL_QUEUE_ADVANCE_INVALID"
    )


# ============================================================
# 7) Completed Human Review Cases: 21 -> 22
# ============================================================

cell103_existing_completed_cases = copy.deepcopy(
    cell102_persisted_registry.get(
        "completed_human_review_cases",
        [],
    )
)


if not isinstance(
    cell103_existing_completed_cases,
    list,
):

    cell103_existing_completed_cases = []

    cell103_failures.append(
        "COMPLETED_CASES_NOT_LIST"
    )


# Prevent duplicate insertion if this cell is accidentally rerun.
cell103_existing_completed_cases = [
    case
    for case
    in cell103_existing_completed_cases
    if not (
        isinstance(
            case,
            dict,
        )
        and
        case.get(
            "pilot_case_id"
        )
        ==
        CELL102_PILOT_ID
    )
]


cell103_completed_human_review_cases = (
    cell103_existing_completed_cases
    +
    [
        copy.deepcopy(
            cell103_completed_case
        )
    ]
)


cell103_completed_case_count_valid = (
    len(
        cell103_completed_human_review_cases
    )
    ==
    22
)


cell103_completed_ids = [
    case.get(
        "pilot_case_id"
    )
    for case
    in cell103_completed_human_review_cases
    if isinstance(
        case,
        dict,
    )
]


cell103_completed_identity_valid = all([
    len(
        cell103_completed_ids
    )
    ==
    22,

    len(
        set(
            cell103_completed_ids
        )
    )
    ==
    22,

    CELL102_PILOT_ID
    in
    cell103_completed_ids,
])


if not cell103_completed_case_count_valid:
    cell103_failures.append(
        "COMPLETED_CASE_COUNT_NOT_22"
    )


if not cell103_completed_identity_valid:
    cell103_failures.append(
        "COMPLETED_CASE_IDENTITIES_INVALID"
    )


# ============================================================
# 8) Author Review Queue: 21 -> 22
# ============================================================

cell103_author_review = copy.deepcopy(
    cell102_persisted_registry.get(
        "author_review",
        {},
    )
)


if not isinstance(
    cell103_author_review,
    dict,
):

    cell103_author_review = {}


cell103_author_ready_cases = copy.deepcopy(
    cell103_completed_human_review_cases
)


cell103_author_review[
    "cases_ready_for_review"
] = copy.deepcopy(
    cell103_author_ready_cases
)


cell103_author_review[
    "case_count"
] = len(
    cell103_author_ready_cases
)


cell103_author_review[
    "final_review_status"
] = "NOT_STARTED"


cell103_author_ready_count_valid = (
    len(
        cell103_author_ready_cases
    )
    ==
    22
)


cell103_author_ready_identity_valid = (
    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell103_author_ready_cases
        if isinstance(
            case,
            dict,
        )
    }
    ==
    {
        case.get(
            "pilot_case_id"
        )
        for case
        in cell103_completed_human_review_cases
        if isinstance(
            case,
            dict,
        )
    }
)


if not cell103_author_ready_count_valid:
    cell103_failures.append(
        "AUTHOR_READY_COUNT_NOT_22"
    )


if not cell103_author_ready_identity_valid:
    cell103_failures.append(
        "AUTHOR_READY_IDENTITIES_INVALID"
    )


# ============================================================
# 9) Preserve Global Authority Policy
# ============================================================

cell103_authority_policy = copy.deepcopy(
    cell102_persisted_registry.get(
        "workflow_policy",
        {},
    ).get(
        "authority_evidence_policy",
        {},
    )
)


cell103_authority_policy_valid = all([
    cell103_authority_policy.get(
        "primary_authority"
    )
    ==
    "Dr. Komeil Roudi",

    cell103_authority_policy.get(
        "secondary_to_primary_promotion"
    )
    ==
    "FORBIDDEN",
])


if not cell103_authority_policy_valid:
    cell103_failures.append(
        "GLOBAL_AUTHORITY_POLICY_INVALID"
    )


# ============================================================
# 10) Terminal Human Review State
# ============================================================

cell103_terminal_human_review_summary = {
    "status":
        "PILOT_HUMAN_REVIEW_COMPLETE",

    "completed_human_review_case_count":
        22,

    "cases_ready_for_author_review":
        22,

    "remaining_pilot_count":
        0,

    "last_completed_pilot_id":
        CELL102_PILOT_ID,

    "last_completed_target_evidence_id":
        CELL102_TARGET_EVIDENCE_ID,

    "last_completed_citation_mapping_id":
        cell102_citation_mapping_id,

    "author_final_review_status":
        "NOT_STARTED",

    "full_572_generation_status":
        "BLOCKED_PENDING_FINAL_HEALTH_VALIDATION",
}


# ============================================================
# 11) Continuation State
# ============================================================

cell103_continuation_state_core = {
    "next_pilot_id":
        None,

    "remaining_pilot_ids":
        [],

    "remaining_count":
        0,

    "active_pilot_status":
        "NONE",

    "active_target_evidence_id":
        None,

    "active_citation_mapping_id":
        None,

    "last_completed_pilot_id":
        CELL102_PILOT_ID,

    "last_completed_target_evidence_id":
        CELL102_TARGET_EVIDENCE_ID,

    "last_completed_citation_mapping_id":
        cell102_citation_mapping_id,

    "last_completion_type":
        "EXERCISE_SELF_ASSESSMENT_BOOK_PRIMARY_PILOT_COMPLETE",

    "pilot_human_review_status":
        "COMPLETE",

    "completed_human_review_case_count":
        22,

    "cases_ready_for_author_review":
        22,

    "author_final_review_status":
        "NOT_STARTED",

    "full_572_generation_status":
        "BLOCKED_PENDING_FINAL_HEALTH_VALIDATION",

    "authority_policy_active":
        True,
}


cell103_continuation_state_sha256 = (
    cell10_stable_sha(
        cell103_continuation_state_core
    )
)


cell103_continuation_state = {
    **copy.deepcopy(
        cell103_continuation_state_core
    ),

    "continuation_state_sha256":
        cell103_continuation_state_sha256,
}


# ============================================================
# 12) Persist Terminal Registry State
# ============================================================

cell103_persistent_registry_payload = copy.deepcopy(
    cell102_persisted_registry
)


cell103_persistent_registry_payload[
    "completed_human_review_cases"
] = copy.deepcopy(
    cell103_completed_human_review_cases
)


cell103_persistent_registry_payload[
    "author_review"
] = copy.deepcopy(
    cell103_author_review
)


# There is no remaining active Pilot.
cell103_persistent_registry_payload[
    "active_workspace"
] = None


cell103_persistent_registry_payload[
    "remaining_pilot_count"
] = 0


cell103_persistent_registry_payload[
    "pilot_human_review_status"
] = "COMPLETE"


cell103_persistent_registry_payload[
    "human_review_summary"
] = copy.deepcopy(
    cell103_terminal_human_review_summary
)


cell103_persistent_registry_payload[
    "continuation_state"
] = copy.deepcopy(
    cell103_continuation_state
)


cell103_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell103_persistent_registry_payload
    )
)


# ============================================================
# 13) Read-back Validation
# ============================================================

cell103_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell103_persisted_completed_cases = (
    cell103_persisted_registry.get(
        "completed_human_review_cases",
        [],
    )
)


cell103_persisted_author_ready_cases = (
    cell103_persisted_registry.get(
        "author_review",
        {},
    ).get(
        "cases_ready_for_review",
        [],
    )
)


cell103_persisted_active_workspace = (
    cell103_persisted_registry.get(
        "active_workspace"
    )
)


cell103_persisted_continuation = (
    cell103_persisted_registry.get(
        "continuation_state",
        {},
    )
)


cell103_persisted_summary = (
    cell103_persisted_registry.get(
        "human_review_summary",
        {},
    )
)


cell103_persisted_completed_ids = [
    case.get(
        "pilot_case_id"
    )
    for case
    in cell103_persisted_completed_cases
    if isinstance(
        case,
        dict,
    )
]


cell103_persisted_author_ids = [
    case.get(
        "pilot_case_id"
    )
    for case
    in cell103_persisted_author_ready_cases
    if isinstance(
        case,
        dict,
    )
]


cell103_persistence_valid = all([
    isinstance(
        cell103_persisted_completed_cases,
        list,
    ),

    len(
        cell103_persisted_completed_cases
    )
    ==
    22,

    len(
        set(
            cell103_persisted_completed_ids
        )
    )
    ==
    22,

    isinstance(
        cell103_persisted_author_ready_cases,
        list,
    ),

    len(
        cell103_persisted_author_ready_cases
    )
    ==
    22,

    set(
        cell103_persisted_completed_ids
    )
    ==
    set(
        cell103_persisted_author_ids
    ),

    cell103_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    0,

    cell103_persisted_registry.get(
        "pilot_human_review_status"
    )
    ==
    "COMPLETE",

    cell103_persisted_active_workspace
    is None,

    cell103_persisted_continuation.get(
        "next_pilot_id"
    )
    is None,

    cell103_persisted_continuation.get(
        "remaining_pilot_ids"
    )
    ==
    [],

    cell103_persisted_continuation.get(
        "remaining_count"
    )
    ==
    0,

    cell103_persisted_continuation.get(
        "pilot_human_review_status"
    )
    ==
    "COMPLETE",

    cell103_persisted_continuation.get(
        "completed_human_review_case_count"
    )
    ==
    22,

    cell103_persisted_continuation.get(
        "cases_ready_for_author_review"
    )
    ==
    22,

    cell103_persisted_continuation.get(
        "author_final_review_status"
    )
    ==
    "NOT_STARTED",

    cell103_persisted_continuation.get(
        "continuation_state_sha256"
    )
    ==
    cell103_continuation_state_sha256,

    cell103_persisted_summary.get(
        "status"
    )
    ==
    "PILOT_HUMAN_REVIEW_COMPLETE",

    cell103_persisted_summary.get(
        "completed_human_review_case_count"
    )
    ==
    22,

    cell103_persisted_summary.get(
        "remaining_pilot_count"
    )
    ==
    0,
])


if not cell103_persistence_valid:
    cell103_failures.append(
        "TERMINAL_REGISTRY_READBACK_INVALID"
    )


# ============================================================
# 14) Final Pilot present exactly once
# ============================================================

cell103_final_pilot_persisted_once = (
    cell103_persisted_completed_ids.count(
        CELL102_PILOT_ID
    )
    ==
    1
)


if not cell103_final_pilot_persisted_once:
    cell103_failures.append(
        "FINAL_PILOT_NOT_PERSISTED_EXACTLY_ONCE"
    )


# ============================================================
# 15) Final Review Counts
# ============================================================

cell103_final_counts_valid = all([
    len(
        cell103_persisted_completed_cases
    )
    ==
    22,

    len(
        cell103_persisted_author_ready_cases
    )
    ==
    22,

    cell103_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    0,

    cell103_persisted_registry.get(
        "author_review",
        {},
    ).get(
        "case_count"
    )
    ==
    22,

    cell103_persisted_registry.get(
        "author_review",
        {},
    ).get(
        "final_review_status"
    )
    ==
    "NOT_STARTED",
])


if not cell103_final_counts_valid:
    cell103_failures.append(
        "FINAL_REVIEW_COUNTS_INVALID"
    )


# ============================================================
# 16) Safety / Full-572 Guard
# ============================================================

cell103_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell103_full_572_blocked = (
    cell102_full_572_blocked
    is True
)


cell103_no_commit_push = True


cell103_no_full_generation = True


# ============================================================
# 17) Gate
# ============================================================

cell103_gate_checks = {
    "no_failures":
        len(
            cell103_failures
        )
        ==
        0,

    "citation_accept_valid":
        cell103_citation_accept_valid,

    "final_citation_valid":
        cell103_final_citation_valid,

    "completed_case_valid":
        cell103_completed_case_valid,

    "queue_advance_1_to_0":
        cell103_queue_advance_valid,

    "completed_case_count_22":
        cell103_completed_case_count_valid,

    "completed_identity_valid":
        cell103_completed_identity_valid,

    "author_ready_count_22":
        cell103_author_ready_count_valid,

    "author_ready_identity_valid":
        cell103_author_ready_identity_valid,

    "authority_policy_valid":
        cell103_authority_policy_valid,

    "persistence_valid":
        cell103_persistence_valid,

    "final_pilot_persisted_once":
        cell103_final_pilot_persisted_once,

    "final_counts_valid":
        cell103_final_counts_valid,

    "base_seed_unchanged":
        cell103_base_seed_unchanged,

    "full_572_still_blocked":
        cell103_full_572_blocked,

    "no_full_generation":
        cell103_no_full_generation,

    "no_commit_push":
        cell103_no_commit_push,
}


cell103_gate = all(
    cell103_gate_checks.values()
)


# ============================================================
# 18) Diagnostics
# ============================================================

print(
    "=== FINAL PILOT COMPLETE + PILOT HUMAN REVIEW QUEUE COMPLETE ==="
)


print(
    "\n=== FINAL COMPLETED PILOT ==="
)

print(
    "Pilot:",
    CELL102_PILOT_ID,
)

print(
    "Golden Case:",
    CELL102_GOLDEN_CASE_ID,
)

print(
    "Task Draft:",
    CELL102_TASK_DRAFT_ID,
)

print(
    "Target Evidence:",
    CELL102_TARGET_EVIDENCE_ID,
)

print(
    "Question:",
    "ACCEPT",
)

print(
    "Expected Answer:",
    "ACCEPT",
)

print(
    "Citation:",
    "ACCEPT",
)

print(
    "Citation Mapping:",
    cell102_citation_mapping_id,
)

print(
    "Engine State:",
    cell103_complete_pending_session.get(
        "current_state"
    ),
)


print(
    "\n=== FINAL PILOT EVIDENCE ==="
)

print(
    "Authority Expectation:",
    "BOOK_PRIMARY_EVIDENCE",
)

print(
    "Book Primary Evidence only:",
    True,
)

print(
    "Original Target preserved:",
    True,
)

print(
    "Supporting Evidence Extension:",
    True,
)

print(
    "Supporting paragraphs:",
    cell102_support_paragraph_numbers,
)

print(
    "Retarget used:",
    False,
)

print(
    "External knowledge used:",
    False,
)


print(
    "\n=== QUEUE ==="
)

print(
    "Before:",
    1,
)

print(
    "After:",
    0,
)

print(
    "Queue advanced exactly once:",
    cell103_queue_advance_valid,
)

print(
    "Next Pilot:",
    None,
)


print(
    "\n=== HUMAN REVIEW ==="
)

print(
    "Completed Human Review cases:",
    len(
        cell103_persisted_completed_cases
    ),
)

print(
    "Cases ready for Author Review:",
    len(
        cell103_persisted_author_ready_cases
    ),
)

print(
    "Pilot Human Review Status:",
    cell103_persisted_registry.get(
        "pilot_human_review_status"
    ),
)

print(
    "Author Final Review:",
    cell103_persisted_registry.get(
        "author_review",
        {},
    ).get(
        "final_review_status"
    ),
)


print(
    "\n=== CONTINUATION STATE ==="
)

print(
    "Remaining Pilots:",
    cell103_persisted_continuation.get(
        "remaining_count"
    ),
)

print(
    "Active Pilot:",
    cell103_persisted_continuation.get(
        "next_pilot_id"
    ),
)

print(
    "Last Completed Pilot:",
    cell103_persisted_continuation.get(
        "last_completed_pilot_id"
    ),
)

print(
    "Continuation SHA256:",
    cell103_continuation_state_sha256,
)


print(
    "\n=== FULL 572 GUARD ==="
)

print(
    "Full 572 Generation:",
    "BLOCKED_PENDING_FINAL_HEALTH_VALIDATION",
)

print(
    "Full 572 blocked:",
    cell103_full_572_blocked,
)


print(
    "\n=== ALL INTERNAL GATES ==="
)

for name, status in (
    cell103_gate_checks.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== SAFETY ==="
)

print(
    "Registry write:",
    True,
)

print(
    "Queue advance:",
    "1 -> 0",
)

print(
    "Raw Source mutated:",
    False,
)

print(
    "Parser mutated:",
    False,
)

print(
    "Base Seed unchanged:",
    cell103_base_seed_unchanged,
)

print(
    "Full generation executed:",
    False,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell103_failures
)


print(
    "\nCODE CELL 103 GATE:",
    (
        "PASS"
        if cell103_gate
        else
        "FAIL"
    ),
)


assert cell103_gate, (
    "Final Pilot completion / queue 1->0 / terminal Human Review "
    "registry validation failed. Repair this SAME Code Cell 103."
)

=== FINAL PILOT COMPLETE + PILOT HUMAN REVIEW QUEUE COMPLETE ===

=== FINAL COMPLETED PILOT ===
Pilot: question-pilot:2beb40d7d1262dbf386db581f54d50d2
Golden Case: golden-case:34486152e8d17f47786247678373dd31
Task Draft: task-draft:29b7eccee41f1464eb53697d8dafcc11
Target Evidence: target-evidence:1352ce8332affbe3651fe0558449e2a9
Question: ACCEPT
Expected Answer: ACCEPT
Citation: ACCEPT
Citation Mapping: citation-mapping:be28c9632cc36c1097b40161197e7a52
Engine State: PILOT_COMPLETE_PENDING_QUEUE_ADVANCE

=== FINAL PILOT EVIDENCE ===
Authority Expectation: BOOK_PRIMARY_EVIDENCE
Book Primary Evidence only: True
Original Target preserved: True
Supporting Evidence Extension: True
Supporting paragraphs: [140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 163]
Retarget used: False
External knowledge used: False

=== QUEUE ===
Before: 1
After: 0
Queue advanced exactly once: True
Next Pilot: None

=== HUMAN REVIEW ===
Completed Human Review cases

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 104 — Golden Dataset Final Health Check

این Cell وضعیت نهایی Notebook و Human Review Registry را بررسی می‌کند.

بررسی‌ها شامل موارد زیر است:

- وجود Code Cellهای شماره 1 تا 104 بدون شماره تکراری یا گمشده؛
- اجراشدن Code Cellهای 1 تا 103؛
- نبود Error Output ذخیره‌شده در Notebook؛
- وجود Markerهای نهایی Pilot Human Review؛
- وجود دقیقاً 22 Case کامل و یکتا؛
- ACCEPT بودن Question، Expected Answer و Citation تمام Caseها؛
- آماده‌بودن هر 22 Case برای Author Review؛
- صفر بودن Remaining Pilot Queue؛
- بسته‌بودن Active Human Review Workspace؛
- حفظ Global Authority Policy؛
- صحت Final Pilot و Book Primary Evidence؛
- صحت Continuation SHA؛
- حفظ Guardهای Full 572 و عدم Commit / Push.

این Cell چیزی را تغییر نمی‌دهد و فقط Final Health را ارزیابی می‌کند.

</div>

In [108]:
# Code Cell 104 — REPLACEMENT

from pathlib import Path
import json
import re
import copy
import hashlib


cell104_failures = []


# ============================================================
# 1) Helpers
# ============================================================

def cell104_check(name, condition):
    condition = bool(condition)

    if not condition:
        cell104_failures.append(
            name
        )

    return condition


def cell104_source(cell):
    source = cell.get(
        "source",
        "",
    )

    if isinstance(
        source,
        list,
    ):
        return "".join(
            source
        )

    if isinstance(
        source,
        str,
    ):
        return source

    return ""


def cell104_is_accept(value):
    return (
        isinstance(
            value,
            str,
        )
        and
        value.strip().upper()
        ==
        "ACCEPT"
    )


def cell104_review_accept(review):
    if not isinstance(
        review,
        dict,
    ):
        return False

    for key in [
        "human_decision",
        "decision",
        "human_adjudication_status",
        "adjudication_status",
        "review_decision",
    ]:

        if cell104_is_accept(
            review.get(
                key
            )
        ):
            return True

    return False


def cell104_recursive_values(obj, keys):
    values = []

    if isinstance(
        obj,
        dict,
    ):

        for key, value in obj.items():

            if key in keys:
                values.append(
                    value
                )

            values.extend(
                cell104_recursive_values(
                    value,
                    keys,
                )
            )

    elif isinstance(
        obj,
        list,
    ):

        for item in obj:

            values.extend(
                cell104_recursive_values(
                    item,
                    keys,
                )
            )

    return values


def cell104_has_question_artifact(obj):
    if isinstance(
        obj,
        dict,
    ):

        for key, value in obj.items():

            if (
                key
                in
                {
                    "question_text",
                    "question_draft_id",
                    "question_id",
                }
                and
                isinstance(
                    value,
                    str,
                )
                and
                value.strip()
            ):
                return True

            if cell104_has_question_artifact(
                value
            ):
                return True

    elif isinstance(
        obj,
        list,
    ):

        return any(
            cell104_has_question_artifact(
                item
            )
            for item
            in obj
        )

    elif isinstance(
        obj,
        str,
    ):

        if obj.startswith(
            "question-draft:"
        ):
            return True

    return False


def cell104_component_accept(
    case,
    component,
):

    review_key = {
        "question":
            "question_review",

        "answer":
            "expected_answer_review",

        "citation":
            "citation_review",
    }[
        component
    ]


    decision_keys = {
        "question": {
            "question_decision",
            "question_human_decision",
        },

        "answer": {
            "expected_answer_decision",
            "answer_decision",
            "expected_answer_human_decision",
        },

        "citation": {
            "citation_decision",
            "citation_human_decision",
        },
    }[
        component
    ]


    review = case.get(
        review_key
    )


    if cell104_review_accept(
        review
    ):
        return True


    summary = (
        case.get(
            "completion_summary"
        )
        or
        {}
    )


    if isinstance(
        summary,
        dict,
    ):

        for key in decision_keys:

            if cell104_is_accept(
                summary.get(
                    key
                )
            ):
                return True


    recursive = (
        cell104_recursive_values(
            case,
            decision_keys,
        )
    )


    return any(
        cell104_is_accept(
            value
        )
        for value
        in recursive
    )


# ============================================================
# 2) Constants / paths
# ============================================================

CELL104_NOTEBOOK_REL = (
    "notebooks/phase2/"
    "12b_golden_dataset_continuation_compact.ipynb"
)

CELL104_EXPECTED_CASES = 22

CELL104_FINAL_PILOT_ID = (
    "question-pilot:"
    "2beb40d7d1262dbf386db581f54d50d2"
)

CELL104_FINAL_CITATION_ID = (
    "citation-mapping:"
    "be28c9632cc36c1097b40161197e7a52"
)


cell104_registry_path = Path(
    cell21_registry_path
).resolve()


cell104_repo_root = next(
    (
        path
        for path
        in [
            cell104_registry_path.parent,
            *cell104_registry_path.parents,
        ]
        if (
            path
            /
            "notebooks"
        ).exists()
        and
        (
            path
            /
            "data"
        ).exists()
    ),
    None,
)


cell104_check(
    "REPO_ROOT_NOT_FOUND",
    cell104_repo_root
    is not None,
)


cell104_notebook_path = (
    cell104_repo_root
    /
    CELL104_NOTEBOOK_REL
    if cell104_repo_root
    is not None
    else
    None
)


cell104_check(
    "NOTEBOOK_FILE_NOT_FOUND",
    (
        cell104_notebook_path
        is not None
        and
        cell104_notebook_path.exists()
    ),
)


cell104_check(
    "REGISTRY_FILE_NOT_FOUND",
    cell104_registry_path.exists(),
)


# ============================================================
# 3) Read files from disk
# ============================================================

try:

    cell104_notebook = json.loads(
        cell104_notebook_path.read_text(
            encoding="utf-8"
        )
    )

except Exception as exc:

    cell104_notebook = {}

    cell104_failures.append(
        "NOTEBOOK_JSON_READ_FAILED: "
        +
        repr(
            exc
        )
    )


try:

    cell104_registry = json.loads(
        cell104_registry_path.read_text(
            encoding="utf-8"
        )
    )

except Exception as exc:

    cell104_registry = {}

    cell104_failures.append(
        "REGISTRY_JSON_READ_FAILED: "
        +
        repr(
            exc
        )
    )


cell104_cells = (
    cell104_notebook.get(
        "cells",
        [],
    )
)


# ============================================================
# 4) Notebook numbering / execution / errors
# ============================================================

cell104_pattern = re.compile(
    r"(?mi)^\s*#\s*Code Cell\s+(\d+)\b"
)


cell104_numbered = []

cell104_error_outputs_1_103 = []


for notebook_index, cell in enumerate(
    cell104_cells
):

    if cell.get(
        "cell_type"
    ) != "code":
        continue


    source = cell104_source(
        cell
    )


    match = cell104_pattern.search(
        source
    )


    number = (
        int(
            match.group(
                1
            )
        )
        if match
        else
        None
    )


    if number is not None:

        cell104_numbered.append({
            "number":
                number,

            "execution_count":
                cell.get(
                    "execution_count"
                ),

            "notebook_index":
                notebook_index,
        })


    # Only Cells 1..103 must be error-free.
    if (
        number is None
        or
        number > 103
    ):
        continue


    outputs = cell.get(
        "outputs",
        [],
    )


    if not isinstance(
        outputs,
        list,
    ):
        continue


    for output in outputs:

        if (
            isinstance(
                output,
                dict,
            )
            and
            output.get(
                "output_type"
            )
            ==
            "error"
        ):

            cell104_error_outputs_1_103.append({
                "number":
                    number,

                "ename":
                    output.get(
                        "ename"
                    ),

                "evalue":
                    output.get(
                        "evalue"
                    ),
            })


cell104_numbers = [
    item[
        "number"
    ]
    for item
    in cell104_numbered
]


cell104_duplicates = sorted(
    {
        number
        for number
        in cell104_numbers
        if cell104_numbers.count(
            number
        )
        >
        1
    }
)


cell104_missing_1_104 = sorted(
    set(
        range(
            1,
            105,
        )
    )
    -
    set(
        cell104_numbers
    )
)


cell104_unexpected = sorted(
    set(
        cell104_numbers
    )
    -
    set(
        range(
            1,
            105,
        )
    )
)


cell104_unexecuted_1_103 = [
    item[
        "number"
    ]
    for item
    in cell104_numbered
    if (
        item[
            "number"
        ]
        <=
        103
        and
        item[
            "execution_count"
        ]
        is None
    )
]


cell104_numbering_valid = (
    cell104_check(
        "NUMBERED_CODE_CELL_SEQUENCE_INVALID",
        all([
            not cell104_duplicates,
            not cell104_missing_1_104,
            not cell104_unexpected,
            cell104_numbers
            ==
            sorted(
                cell104_numbers
            ),
        ]),
    )
)


cell104_execution_valid = (
    cell104_check(
        "UNEXECUTED_NUMBERED_CODE_CELLS_1_TO_103",
        not cell104_unexecuted_1_103,
    )
)


cell104_no_error_outputs = (
    cell104_check(
        "NOTEBOOK_CONTAINS_ERROR_OUTPUTS_IN_1_TO_103",
        not cell104_error_outputs_1_103,
    )
)


# ============================================================
# 5) Required final markers
# ============================================================

cell104_all_source = "\n".join(
    cell104_source(
        cell
    )
    for cell
    in cell104_cells
)


cell104_required_markers = [
    "CODE CELL 103 GATE:",
    "PILOT_HUMAN_REVIEW_COMPLETE",
    "BLOCKED_PENDING_FINAL_HEALTH_VALIDATION",
]


cell104_marker_status = {
    marker:
        marker
        in
        cell104_all_source
    for marker
    in cell104_required_markers
}


cell104_markers_valid = (
    cell104_check(
        "FINAL_NOTEBOOK_MARKERS_MISSING",
        all(
            cell104_marker_status.values()
        ),
    )
)


# ============================================================
# 6) Registry collections
# ============================================================

cell104_completed = (
    cell104_registry.get(
        "completed_human_review_cases",
        [],
    )
)


cell104_author_review = (
    cell104_registry.get(
        "author_review",
        {},
    )
)


cell104_author_ready = (
    cell104_author_review.get(
        "cases_ready_for_review",
        [],
    )
)


cell104_continuation = (
    cell104_registry.get(
        "continuation_state",
        {},
    )
)


cell104_summary = (
    cell104_registry.get(
        "human_review_summary",
        {},
    )
)


cell104_completed_ids = [
    case.get(
        "pilot_case_id"
    )
    for case
    in cell104_completed
    if isinstance(
        case,
        dict,
    )
]


cell104_completed_count_valid = (
    cell104_check(
        "COMPLETED_CASE_COUNT_OR_IDENTITY_INVALID",
        all([
            isinstance(
                cell104_completed,
                list,
            ),

            len(
                cell104_completed
            )
            ==
            CELL104_EXPECTED_CASES,

            len(
                cell104_completed_ids
            )
            ==
            CELL104_EXPECTED_CASES,

            len(
                set(
                    cell104_completed_ids
                )
            )
            ==
            CELL104_EXPECTED_CASES,
        ]),
    )
)


# ============================================================
# 7) Schema-aware 22-case lineage validation
# ============================================================

cell104_case_issues = []

cell104_case_validation_rows = []


for case in cell104_completed:

    if not isinstance(
        case,
        dict,
    ):

        cell104_case_issues.append({
            "pilot_case_id":
                None,

            "reason":
                "NON_DICT_CASE",
        })

        continue


    pilot_id = case.get(
        "pilot_case_id"
    )


    answer_accept = (
        cell104_component_accept(
            case,
            "answer",
        )
    )


    citation_accept = (
        cell104_component_accept(
            case,
            "citation",
        )
    )


    explicit_question_accept = (
        cell104_component_accept(
            case,
            "question",
        )
    )


    question_artifact_present = (
        cell104_has_question_artifact(
            case
        )
    )


    # Early Pilot records did not always persist a separate
    # question human_decision field. A complete downstream
    # accepted Answer + Citation lineage proves the accepted
    # question artifact used by that completed Pilot.
    legacy_question_accept = all([
        question_artifact_present,
        answer_accept,
        citation_accept,
    ])


    question_accept = (
        explicit_question_accept
        or
        legacy_question_accept
    )


    workflow_state = case.get(
        "workflow_state"
    )


    final_status = case.get(
        "final_status"
    )


    # The authoritative completion signals are workflow/final
    # state plus complete Human decision lineage.
    explicit_completion = any([
        workflow_state
        ==
        "PILOT_COMPLETE",

        final_status
        ==
        "PILOT_COMPLETE",
    ])


    legacy_completion = all([
        bool(
            pilot_id
        ),
        question_accept,
        answer_accept,
        citation_accept,
    ])


    workflow_valid = (
        explicit_completion
        or
        legacy_completion
    )


    # Historical `status` is intentionally NOT a completion gate.
    # Several valid completed Pilots preserve their former active
    # status for lineage/history.
    historical_status = case.get(
        "status"
    )


    author_review = (
        case.get(
            "author_final_review"
        )
        or
        {}
    )


    author_status = (
        author_review.get(
            "status"
        )
        if isinstance(
            author_review,
            dict,
        )
        else
        None
    )


    author_valid = (
        author_status
        in
        {
            None,
            "NOT_STARTED",
        }
    )


    checks = {
        "workflow_complete":
            workflow_valid,

        "question_accept":
            question_accept,

        "answer_accept":
            answer_accept,

        "citation_accept":
            citation_accept,

        "author_not_started":
            author_valid,
    }


    schema = (
        "MODERN"
        if (
            workflow_state
            ==
            "PILOT_COMPLETE"
            and
            historical_status
            ==
            "COMPLETED_HUMAN_REVIEW"
        )
        else
        "LEGACY_COMPATIBLE"
    )


    cell104_case_validation_rows.append({
        "pilot_case_id":
            pilot_id,

        "schema":
            schema,

        "historical_status":
            historical_status,

        "workflow_state":
            workflow_state,

        "explicit_question_accept":
            explicit_question_accept,

        "legacy_question_accept":
            (
                not explicit_question_accept
                and
                legacy_question_accept
            ),

        "checks":
            copy.deepcopy(
                checks
            ),
    })


    if not all(
        checks.values()
    ):

        cell104_case_issues.append({
            "pilot_case_id":
                pilot_id,

            "historical_status":
                historical_status,

            "workflow_state":
                workflow_state,

            "checks":
                checks,
        })


cell104_case_decisions_valid = (
    cell104_check(
        "ONE_OR_MORE_COMPLETED_CASES_INVALID",
        not cell104_case_issues,
    )
)


# ============================================================
# 8) Author Review queue
# ============================================================

cell104_author_ids = [
    case.get(
        "pilot_case_id"
    )
    for case
    in cell104_author_ready
    if isinstance(
        case,
        dict,
    )
]


cell104_author_review_valid = (
    cell104_check(
        "AUTHOR_REVIEW_QUEUE_INVALID",
        all([
            isinstance(
                cell104_author_ready,
                list,
            ),

            len(
                cell104_author_ready
            )
            ==
            22,

            cell104_author_review.get(
                "case_count"
            )
            ==
            22,

            set(
                cell104_author_ids
            )
            ==
            set(
                cell104_completed_ids
            ),

            cell104_author_review.get(
                "final_review_status"
            )
            ==
            "NOT_STARTED",
        ]),
    )
)


# ============================================================
# 9) Terminal state
# ============================================================

cell104_terminal_valid = (
    cell104_check(
        "TERMINAL_HUMAN_REVIEW_STATE_INVALID",
        all([
            cell104_registry.get(
                "remaining_pilot_count"
            )
            ==
            0,

            cell104_registry.get(
                "pilot_human_review_status"
            )
            ==
            "COMPLETE",

            cell104_registry.get(
                "active_workspace"
            )
            is None,

            cell104_continuation.get(
                "next_pilot_id"
            )
            is None,

            cell104_continuation.get(
                "remaining_pilot_ids"
            )
            ==
            [],

            cell104_continuation.get(
                "remaining_count"
            )
            ==
            0,

            cell104_continuation.get(
                "pilot_human_review_status"
            )
            ==
            "COMPLETE",

            cell104_continuation.get(
                "completed_human_review_case_count"
            )
            ==
            22,

            cell104_continuation.get(
                "cases_ready_for_author_review"
            )
            ==
            22,

            cell104_continuation.get(
                "author_final_review_status"
            )
            ==
            "NOT_STARTED",

            cell104_summary.get(
                "status"
            )
            ==
            "PILOT_HUMAN_REVIEW_COMPLETE",

            cell104_summary.get(
                "completed_human_review_case_count"
            )
            ==
            22,

            cell104_summary.get(
                "remaining_pilot_count"
            )
            ==
            0,
        ]),
    )
)


# ============================================================
# 10) Global Authority Policy
# ============================================================

cell104_authority_policy = (
    cell104_registry.get(
        "workflow_policy",
        {},
    ).get(
        "authority_evidence_policy",
        {},
    )
)


cell104_authority_valid = (
    cell104_check(
        "AUTHORITY_POLICY_INVALID",
        all([
            cell104_authority_policy.get(
                "primary_authority"
            )
            ==
            "Dr. Komeil Roudi",

            cell104_authority_policy.get(
                "secondary_to_primary_promotion"
            )
            ==
            "FORBIDDEN",
        ]),
    )
)


# ============================================================
# 11) Final Pilot exact validation
# ============================================================

cell104_final_matches = [
    case
    for case
    in cell104_completed
    if (
        isinstance(
            case,
            dict,
        )
        and
        case.get(
            "pilot_case_id"
        )
        ==
        CELL104_FINAL_PILOT_ID
    )
]


cell104_final_case = (
    cell104_final_matches[
        0
    ]
    if len(
        cell104_final_matches
    )
    ==
    1
    else
    {}
)


cell104_final_citation = (
    (
        cell104_final_case.get(
            "citation_review"
        )
        or
        {}
    ).get(
        "citation_mapping_record"
    )
    or
    {}
)


cell104_final_pilot_valid = (
    cell104_check(
        "FINAL_PILOT_BOOK_EVIDENCE_INVALID",
        all([
            len(
                cell104_final_matches
            )
            ==
            1,

            cell104_final_citation.get(
                "citation_mapping_id"
            )
            ==
            CELL104_FINAL_CITATION_ID,

            cell104_final_citation.get(
                "authority_expectation"
            )
            ==
            "BOOK_PRIMARY_EVIDENCE",

            cell104_final_citation.get(
                "book_primary_evidence_only"
            )
            is True,

            cell104_final_citation.get(
                "original_target_preserved"
            )
            is True,

            cell104_final_citation.get(
                "retarget_used"
            )
            is False,

            cell104_final_citation.get(
                "human_adjudication_status"
            )
            ==
            "ACCEPT",
        ]),
    )
)


# ============================================================
# 12) Continuation SHA
# ============================================================

cell104_persisted_continuation_sha = (
    cell104_continuation.get(
        "continuation_state_sha256"
    )
)


cell104_continuation_core = copy.deepcopy(
    cell104_continuation
)


cell104_continuation_core.pop(
    "continuation_state_sha256",
    None,
)


cell104_recomputed_continuation_sha = (
    cell10_stable_sha(
        cell104_continuation_core
    )
)


cell104_continuation_sha_valid = (
    cell104_check(
        "CONTINUATION_SHA_MISMATCH",
        cell104_persisted_continuation_sha
        ==
        cell104_recomputed_continuation_sha,
    )
)


# ============================================================
# 13) Runtime Guards
# ============================================================

cell104_runtime_guards = {
    "cell103_gate":
        globals().get(
            "cell103_gate"
        )
        is True,

    "full_572_blocked":
        globals().get(
            "cell103_full_572_blocked"
        )
        is True,

    "no_full_generation":
        globals().get(
            "cell103_no_full_generation"
        )
        is True,

    "no_commit_push":
        globals().get(
            "cell103_no_commit_push"
        )
        is True,
}


cell104_runtime_valid = (
    cell104_check(
        "CURRENT_RUNTIME_GUARDS_INVALID",
        all(
            cell104_runtime_guards.values()
        ),
    )
)


# ============================================================
# 14) Fingerprints
# ============================================================

cell104_notebook_sha256 = (
    hashlib.sha256(
        cell104_notebook_path.read_bytes()
    ).hexdigest()
)


cell104_registry_sha256 = (
    hashlib.sha256(
        cell104_registry_path.read_bytes()
    ).hexdigest()
)


# ============================================================
# 15) Final Gate
# ============================================================

cell104_gate_checks = {
    "numbered_code_cells_1_to_104":
        cell104_numbering_valid,

    "cells_1_to_103_executed":
        cell104_execution_valid,

    "no_error_outputs_cells_1_to_103":
        cell104_no_error_outputs,

    "final_markers":
        cell104_markers_valid,

    "completed_cases_22_unique":
        cell104_completed_count_valid,

    "all_22_case_lineages_valid":
        cell104_case_decisions_valid,

    "author_review_22_not_started":
        cell104_author_review_valid,

    "terminal_queue_zero":
        cell104_terminal_valid,

    "authority_policy":
        cell104_authority_valid,

    "final_pilot_book_evidence":
        cell104_final_pilot_valid,

    "continuation_sha":
        cell104_continuation_sha_valid,

    "runtime_guards":
        cell104_runtime_valid,
}


cell104_gate = all([
    len(
        cell104_failures
    )
    ==
    0,

    all(
        cell104_gate_checks.values()
    ),
])


# ============================================================
# 16) Diagnostics
# ============================================================

print(
    "=== GOLDEN DATASET NOTEBOOK FINAL HEALTH CHECK ==="
)


print(
    "\n=== FILES ==="
)

print(
    "Notebook:",
    cell104_notebook_path,
)

print(
    "Notebook SHA256:",
    cell104_notebook_sha256,
)

print(
    "Registry:",
    cell104_registry_path,
)

print(
    "Registry SHA256:",
    cell104_registry_sha256,
)


print(
    "\n=== NOTEBOOK ==="
)

print(
    "Total notebook cells:",
    len(
        cell104_cells
    ),
)

print(
    "Numbered Code Cells:",
    len(
        cell104_numbered
    ),
)

print(
    "Missing Code Cell numbers 1..104:",
    cell104_missing_1_104,
)

print(
    "Duplicate Code Cell numbers:",
    cell104_duplicates,
)

print(
    "Unexpected Code Cell numbers:",
    cell104_unexpected,
)

print(
    "Unexecuted numbered Code Cells 1..103:",
    cell104_unexecuted_1_103,
)

print(
    "Error outputs in Cells 1..103:",
    len(
        cell104_error_outputs_1_103
    ),
)


print(
    "\n=== REGISTRY ==="
)

print(
    "Completed Human Review cases:",
    len(
        cell104_completed
    ),
)

print(
    "Unique completed Pilot IDs:",
    len(
        set(
            cell104_completed_ids
        )
    ),
)

print(
    "Cases ready for Author Review:",
    len(
        cell104_author_ready
    ),
)

print(
    "Remaining Pilots:",
    cell104_registry.get(
        "remaining_pilot_count"
    ),
)

print(
    "Active Workspace:",
    cell104_registry.get(
        "active_workspace"
    ),
)

print(
    "Pilot Human Review Status:",
    cell104_registry.get(
        "pilot_human_review_status"
    ),
)

print(
    "Author Final Review:",
    cell104_author_review.get(
        "final_review_status"
    ),
)


print(
    "\n=== 22-CASE SCHEMA / LINEAGE VALIDATION ==="
)


cell104_modern_count = sum(
    1
    for row
    in cell104_case_validation_rows
    if row[
        "schema"
    ]
    ==
    "MODERN"
)


cell104_legacy_count = sum(
    1
    for row
    in cell104_case_validation_rows
    if row[
        "schema"
    ]
    ==
    "LEGACY_COMPATIBLE"
)


cell104_legacy_question_count = sum(
    1
    for row
    in cell104_case_validation_rows
    if row[
        "legacy_question_accept"
    ]
)


print(
    "Modern-schema cases:",
    cell104_modern_count,
)

print(
    "Legacy-compatible cases:",
    cell104_legacy_count,
)

print(
    "Legacy Question ACCEPT recovered from completed lineage:",
    cell104_legacy_question_count,
)

print(
    "Invalid cases:",
    len(
        cell104_case_issues
    ),
)


print(
    "\nHistorical status values retained in legacy cases:"
)


for row in cell104_case_validation_rows:

    if row[
        "schema"
    ] == "LEGACY_COMPATIBLE":

        print(
            " ",
            row[
                "pilot_case_id"
            ],
            "| workflow:",
            row[
                "workflow_state"
            ],
            "| historical status:",
            row[
                "historical_status"
            ],
        )


print(
    "\n=== FINAL PILOT ==="
)

print(
    "Pilot:",
    CELL104_FINAL_PILOT_ID,
)

print(
    "Citation Mapping:",
    cell104_final_citation.get(
        "citation_mapping_id"
    ),
)

print(
    "Book Primary Evidence only:",
    cell104_final_citation.get(
        "book_primary_evidence_only"
    ),
)

print(
    "Original Target preserved:",
    cell104_final_citation.get(
        "original_target_preserved"
    ),
)


print(
    "\n=== AUTHORITY POLICY ==="
)

print(
    "Primary Authority:",
    cell104_authority_policy.get(
        "primary_authority"
    ),
)

print(
    "Secondary -> Primary promotion:",
    cell104_authority_policy.get(
        "secondary_to_primary_promotion"
    ),
)


print(
    "\n=== CONTINUATION SHA ==="
)

print(
    "Persisted:",
    cell104_persisted_continuation_sha,
)

print(
    "Recomputed:",
    cell104_recomputed_continuation_sha,
)

print(
    "Valid:",
    cell104_continuation_sha_valid,
)


print(
    "\n=== ALL FINAL HEALTH GATES ==="
)


for name, status in (
    cell104_gate_checks.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== CASE ISSUES ==="
)

print(
    cell104_case_issues
)


print(
    "\n=== FAILURES ==="
)

print(
    cell104_failures
)


print(
    "\n=== FINAL STATUS ==="
)

print(
    "Golden Dataset Pilot Human Review:",
    (
        "COMPLETE"
        if cell104_gate
        else
        "NOT_HEALTHY"
    ),
)

print(
    "Completed / Expected:",
    f"{len(cell104_completed)} / {CELL104_EXPECTED_CASES}",
)

print(
    "Remaining Pilots:",
    cell104_registry.get(
        "remaining_pilot_count"
    ),
)

print(
    "Author Final Review:",
    cell104_author_review.get(
        "final_review_status"
    ),
)

print(
    "Full 572 Generation:",
    (
        "READY_FOR_NEXT_PHASE_DECISION"
        if cell104_gate
        else
        "BLOCKED"
    ),
)

print(
    "Commit / Push:",
    False,
)


print(
    "\nCODE CELL 104 FINAL HEALTH:",
    (
        "PASS"
        if cell104_gate
        else
        "FAIL"
    ),
)


assert cell104_gate, (
    "Golden Dataset Final Health failed. "
    "Inspect the failed gate(s) above."
)

=== GOLDEN DATASET NOTEBOOK FINAL HEALTH CHECK ===

=== FILES ===
Notebook: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2/12b_golden_dataset_continuation_compact.ipynb
Notebook SHA256: b6b8f88da0784b9bd357be1972b39b281d2422fe402f1d73753b1189b551c420
Registry: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/data/phase2/human_review/human_review_registry.json
Registry SHA256: 1ace8a6b5c228a629866327a96699fc1d443a44f7979bcc9c181c01f2efec84c

=== NOTEBOOK ===
Total notebook cells: 208
Numbered Code Cells: 104
Missing Code Cell numbers 1..104: []
Duplicate Code Cell numbers: []
Unexpected Code Cell numbers: []
Unexecuted numbered Code Cells 1..103: []
Error outputs in Cells 1..103: 0

=== REGISTRY ===
Completed Human Review cases: 22
Unique completed Pilot IDs: 22
Cases ready for Author Review: 22
Remaining Pilots: 0
Active Workspace: None
Pilot Human Review Status: COMPLETE
Author Final Review: NOT_STARTED

=== 22-CASE 

<div style="direction: rtl; white-space: normal; line-height: 1.8;">

<h2>Final Summary — Golden Dataset Pilot Human Review</h2>

<h3>1. جایگاه این Notebook در Stage 12</h3>

<p>
این Notebook ادامه فشرده مرحله <strong>Notebook 12 — Golden Dataset</strong> است.
بخش نخست این مرحله در
<code>12_golden_dataset.ipynb</code>
انجام شد؛ آن Notebook نقش Development، Contract Discovery، طراحی Sampling،
تعریف Target Evidence، طراحی Pilot QA Workflow و بررسی Edge Caseها را بر عهده داشت.
</p>

<p>
به دلیل رشد Notebook نخست به 141 Code Cell و پیچیده‌شدن فرآیند توسعه،
ادامه اجرای Pilotها عمداً به این Notebook فشرده منتقل شد تا Workflow تثبیت‌شده
به‌صورت Reusable و قابل‌کنترل اجرا شود.
بنابراین این دو Notebook دو مرحله مستقل نیستند؛
هر دو بخش‌های مختلف یک Stage منطقی یعنی Golden Dataset هستند.
</p>

<h3>2. هدف این Notebook</h3>

<p>
هدف اصلی این Notebook اعتبارسنجی کامل Pilot QA Workflow پیش از اجازه تولید Full Golden Dataset بود.
برای این منظور لازم بود Pilotهای باقی‌مانده از مسیر کامل
Target Evidence، Question Draft، Human Review، Expected Answer،
Citation Mapping، Retarget/Repair، Queue Management و Persistence عبور کنند.
</p>

<p>
هدف مهم دیگر این بود که تمام تصمیم‌ها Source-grounded باقی بمانند،
Knowledge Authority با Source Attribution و Conversation Context مخلوط نشود
و هیچ گفته مجری، شرکت‌کننده، مهمان یا Speaker غیرPrimary بدون Evidence معتبر
به‌عنوان نظر دکتر کمیل رودی وارد Expected Answer نشود.
</p>

<h3>3. ورودی‌های واقعی</h3>

<p>
ورودی این Notebook شامل State و Contractهای معتبر استخراج‌شده از بخش نخست Notebook 12،
Candidate و Targetهای Golden Dataset، Corpus و Artifactهای پذیرفته‌شده مراحل قبلی،
و Human Review State نسخه‌دار بود.
</p>

<p>
برای Persistence بازبینی انسانی، فایل
<code>data/phase2/human_review/human_review_registry.json</code>
به‌عنوان Source of Truth استفاده شد.
</p>

<h3>4. کارهایی که واقعاً انجام شدند</h3>

<p>
Reusable Pilot Workflow ساخته و اعتبارسنجی شد؛
Question، Expected Answer و Citation هر Pilot به‌صورت مستقل Human Review شدند؛
Retarget برای Targetهای کم‌ارزش، ناقص یا نامناسب اجرا شد؛
Supporting Evidence Extension فقط در مواردی استفاده شد که Evidence مستقیم و قابل‌ردیابی وجود داشت؛
Speaker Continuity و Authority Boundary در Caseهای Transcript به‌صورت محافظه‌کارانه حفظ شدند؛
و User recollection یا External Knowledge به‌عنوان Golden Evidence جایگزین Source نشد.
</p>

<h3>5. Failureها و Edge Caseهای مهم</h3>

<p>
در جریان Pilot Review چند نوع Failure واقعی مشاهده شد:
Targetهای heading-only یا index-like،
Targetهای چندگوینده‌ای که پس از Speaker Continuity معنای مورد انتظار را از دست می‌دادند،
fragmentهای ناقص،
Speaker canonicalization gap،
Evidence موردنیاز خارج از مرز Target اصلی،
و تفاوت Schema بین Pilotهای تاریخی و Pilotهای جدید.
</p>

<p>
این موارد مخفی یا با حدس اصلاح نشدند.
بسته به نوع مسئله، از Retarget، Human-Certified Supporting Evidence Extension،
Speaker Continuity Overlay، Source Identity Resolution یا Schema-aware Validation استفاده شد.
Raw Source و Parser Output در این Repairها تغییر داده نشدند.
</p>

<h3>6. Human Review Result</h3>

<p>
در پایان Workflow، تعداد
<strong>22 Case</strong>
در Human Review Registry به‌صورت کامل و یکتا ثبت شدند.
برای Caseهای کامل‌شده، Question / Expected Answer / Citation lineage معتبر است
و Remaining Pilot Queue برابر صفر شده است.
</p>

<p>
وضعیت نهایی:
</p>

<ul>
<li><strong>Completed Human Review Cases:</strong> 22</li>
<li><strong>Unique Completed Pilot IDs:</strong> 22</li>
<li><strong>Cases Ready for Author Review:</strong> 22</li>
<li><strong>Remaining Pilot Queue:</strong> 0</li>
<li><strong>Active Human Review Workspace:</strong> None</li>
<li><strong>Pilot Human Review Status:</strong> COMPLETE</li>
<li><strong>Author Final Review:</strong> NOT_STARTED</li>
</ul>

<h3>7. Authority Policy نهایی</h3>

<p>
Primary Knowledge Authority این Golden Workflow برابر
<strong>Dr. Komeil Roudi</strong>
است.
گفته‌های مجری، شرکت‌کننده، مخاطب یا سایر Speakerهای غیرPrimary
می‌توانند Context، Question Trigger یا Contrast Context باشند،
اما مستقل از پاسخ دکتر اجازه ورود به Expected Answer به‌عنوان حقیقت مرجع را ندارند.
</p>

<p>
Policy نهایی:
</p>

<p>
<code>secondary_to_primary_promotion = FORBIDDEN</code>
</p>

<h3>8. Persistence و Author Review</h3>

<p>
تمام Caseهای آماده بازبینی مؤلف در Human Review Registry حفظ شده‌اند
و برای نمایش در Notebook اختصاصی Author Review قابل استفاده‌اند.
Author Final Review در پایان این Notebook عمداً
<code>NOT_STARTED</code>
باقی مانده است؛ زیرا تصمیم مؤلف نباید به‌صورت خودکار یا از طرف Pipeline ثبت شود.
</p>

<h3>9. Final Health</h3>

<p>
Final Health فایل Notebook و Registry را از Disk بررسی کرد.
104 Code Cell شماره‌دار وجود داشت،
Code Cellهای 1 تا 103 اجرا شده بودند،
Missing یا Duplicate Code Cell Number وجود نداشت،
Error Output در Cellهای 1 تا 103 صفر بود،
22 Case یکتا در Registry وجود داشتند،
Queue صفر بود،
Authority Policy معتبر بود
و Continuation State SHA با مقدار بازسازی‌شده تطبیق داشت.
</p>

<p>
نتیجه نهایی:
</p>

<p>
<code>CODE CELL 104 FINAL HEALTH: PASS</code>
</p>

<p>
Continuation State SHA-256:
</p>

<p>
<code>c7a284556fde4d0d242407e0bf7747e40054cabda6c56bb68bc5cabe70f5cda1</code>
</p>

<p>
Registry SHA-256 ثبت‌شده در Final Health:
</p>

<p>
<code>1ace8a6b5c228a629866327a96699fc1d443a44f7979bcc9c181c01f2efec84c</code>
</p>

<h3>10. چه چیزی در این Notebook کامل شد؟</h3>

<p>
هدف اختصاصی این continuation، یعنی اثبات و تکمیل Pilot Human Review Workflow،
با موفقیت محقق شد.
22 Case مورد انتظار این Workflow بسته شدند و Pipeline از نظر Question،
Expected Answer، Citation، Authority Boundary، Retarget و Persistence اعتبارسنجی شد.
</p>

<h3>11. چه چیزی هنوز کامل نشده است؟</h3>

<p>
این نتیجه به معنی ساخته‌شدن Full Golden Dataset نهایی نیست.
Full Dataset با دامنه برنامه‌ریزی‌شده 572 Case هنوز تولید، Validation و Freeze نشده است.
همچنین Author Final Review روی 22 Pilot آماده‌شده هنوز انجام نشده است.
</p>

<p>
بنابراین Pilot Workflow اجازه ورود به مرحله تصمیم‌گیری برای Full Generation را پیدا کرده،
اما Stage منطقی Golden Dataset هنوز به پایان کامل نرسیده است.
</p>

<h3>12. وضعیت Objective</h3>

<p>
<strong>Pilot Human Review Objective: ACHIEVED</strong>
</p>

<p>
اما نسبت به هدف کامل Stage 12، یعنی ساخت و Freeze یک Golden Dataset نهایی و قابل استفاده در Benchmarkهای بعدی:
</p>

<p>
<code>NOTEBOOK OBJECTIVE: PARTIALLY ACHIEVED</code>
</p>

<p>
مرحله Golden Dataset زمانی به‌طور کامل بسته خواهد شد که
Author Review موردنیاز انجام شود،
Full Golden Dataset ساخته و Validate شود،
Locatorها و Lineage نهایی تأیید شوند
و نسخه نهایی Dataset Freeze شود.
</p>

<h3>13. Git / Repository Safety</h3>

<p>
در این مرحله هیچ Commit، Push، Merge یا PR انجام نشده است.
Raw Source و Parser Output نیز در جریان Human Review تغییر داده نشده‌اند.
</p>

</div>
